In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================================
# STAGE 24-0 — NEW KAGGLE NOTEBOOK BOOTSTRAP
# Cross-Dataset Generalization & Artifact-Sensitivity Audit
#
# THIS CELL IS PROVENANCE / ENVIRONMENT ONLY.
#
# ALLOWED:
#   - GPU/runtime inspection
#   - GitHub authentication
#   - repository clone/fetch
#   - inherited-artifact existence checks
#   - /kaggle/input PATH / SIZE inventory
#   - creation of empty Stage24-0 workspace
#
# PROHIBITED HERE:
#   - reading CSV/Parquet feature rows
#   - loading target arrays
#   - model.fit(...)
#   - model.predict(...)
#   - model.predict_proba(...)
#   - target metrics
#   - target threshold selection
#   - target inference of any kind
#
# TARGET OPENINGS CONSUMED BY THIS CELL: 0
# ============================================================================

from __future__ import annotations

import os
import sys
import json
import shutil
import hashlib
import subprocess
import platform
import importlib
from pathlib import Path
from datetime import datetime, timezone

# ---------------------------------------------------------------------------
# 0. Frozen starting-point expectation
# ---------------------------------------------------------------------------

REPO_OWNER = "themubasshir"
REPO_NAME = "ids2018-validation-safe-ablation"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

# GitHub main verified immediately before this notebook bootstrap.
EXPECTED_STARTING_HEAD = "963b2da0043c7438624916280de154a5628f8879"

WORKING_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
REPO_DIR = WORKING_ROOT / REPO_NAME

STAGE24_ROOT = REPO_DIR / "results" / "stage24_cross_dataset"
STAGE24_LOCK_DIR = STAGE24_ROOT / "stage24_0_protocol_lock"

print("=" * 88)
print("STAGE24-0 — SAFE BOOTSTRAP")
print("=" * 88)
print("Expected starting HEAD:", EXPECTED_STARTING_HEAD)
print("Target inference:       FORBIDDEN")
print("Target openings:        0")
print()


# ---------------------------------------------------------------------------
# 1. Small subprocess helper
# ---------------------------------------------------------------------------

def run(cmd, cwd=None, check=True, env=None):
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}\n"
            f"{result.stdout}"
        )
    return result.stdout.strip()


# ---------------------------------------------------------------------------
# 2. Kaggle GitHub secret — robust lookup
# ---------------------------------------------------------------------------

SECRET_CANDIDATES = (
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "GITHUB_PAT",
    "github_pat",
    "GH_PAT",
)

github_token = None
github_secret_name = None
secret_errors = {}

try:
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()

    print("Checking GitHub secret candidates...")

    for secret_name in SECRET_CANDIDATES:
        try:
            value = client.get_secret(secret_name)
            if value and value.strip():
                github_token = value.strip()
                github_secret_name = secret_name
                print(
                    f"  [FOUND] {secret_name:<15} "
                    f"({len(github_token)} characters)"
                )
                break
            else:
                print(f"  [----] {secret_name}")
        except Exception as exc:
            secret_errors[secret_name] = type(exc).__name__
            print(f"  [----] {secret_name}")

except Exception as exc:
    raise RuntimeError(
        "Unable to initialize Kaggle UserSecretsClient.\n"
        "This notebook expects a Kaggle GitHub secret."
    ) from exc


if not github_token:
    raise RuntimeError(
        "\nNo usable GitHub token found.\n\n"
        "Expected one of:\n  - "
        + "\n  - ".join(SECRET_CANDIDATES)
        + "\n\nThe preferred name is GITHUB_TOKEN."
    )

print(f"\nUsable GitHub secret: {github_secret_name}")

# Keep token in process environment only.
os.environ["GITHUB_TOKEN"] = github_token


# ---------------------------------------------------------------------------
# 3. Safe Git authentication for later commits/pushes
#
# Token is NOT embedded in notebook source or git remote URL.
# ---------------------------------------------------------------------------

ASKPASS = WORKING_ROOT / ".stage24_git_askpass.sh"

ASKPASS.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) printf '%s\\n' "x-access-token" ;;
    *)          printf '%s\\n' "$GITHUB_TOKEN" ;;
esac
"""
)

ASKPASS.chmod(0o700)

os.environ["GIT_ASKPASS"] = str(ASKPASS)
os.environ["GIT_TERMINAL_PROMPT"] = "0"


# ---------------------------------------------------------------------------
# 4. GPU / accelerator inventory
# ---------------------------------------------------------------------------

print("\n" + "=" * 88)
print("GPU / BOOSTING BACKEND INVENTORY")
print("=" * 88)

gpu_available = False
gpu_names = []

try:
    smi = run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader",
        ]
    )

    if smi:
        gpu_available = True
        print(smi)

        gpu_names = [
            line.split(",")[0].strip()
            for line in smi.splitlines()
            if line.strip()
        ]

except Exception as exc:
    print("NVIDIA GPU not detected:", type(exc).__name__)


def package_version(module_name):
    try:
        m = importlib.import_module(module_name)
        return getattr(m, "__version__", "UNKNOWN")
    except Exception:
        return None


versions = {
    "python": sys.version.split()[0],
    "numpy": package_version("numpy"),
    "pandas": package_version("pandas"),
    "pyarrow": package_version("pyarrow"),
    "sklearn": package_version("sklearn"),
    "lightgbm": package_version("lightgbm"),
    "xgboost": package_version("xgboost"),
    "catboost": package_version("catboost"),
}

print("\nLibrary versions:")
for name, version in versions.items():
    print(f"  {name:<12}: {version}")


# XGBoost can expose compile-time CUDA information without model fitting.
xgb_build_info = None

try:
    import xgboost as xgb

    if hasattr(xgb, "build_info"):
        xgb_build_info = xgb.build_info()

    print("\nXGBoost build info:")
    if xgb_build_info:
        for k, v in sorted(xgb_build_info.items()):
            if (
                "CUDA" in str(k).upper()
                or "NCCL" in str(k).upper()
                or "USE_" in str(k).upper()
            ):
                print(f"  {k}: {v}")
    else:
        print("  build_info() unavailable")

except Exception as exc:
    print("XGBoost inspection unavailable:", repr(exc))


# IMPORTANT:
# Do NOT run a tiny LightGBM training probe here.
# Although synthetic, that is still a model fit before Stage24-0 has frozen its
# backend semantics. We defer executable GPU probing until the protocol lock.
gpu_policy = {
    "policy": "GPU_FIRST_WHEN_SUPPORTED",
    "hardware_gpu_detected": gpu_available,
    "gpu_names": gpu_names,
    "xgboost": (
        "PREFER_CUDA_IF_BUILD_AND_RUNTIME_SUPPORT_IT"
        if gpu_available
        else "CPU_FALLBACK_IF_NO_GPU"
    ),
    "lightgbm": (
        "PROBE_GPU_BACKEND_AFTER_STAGE24_0_BACKEND_POLICY_IS_FROZEN;"
        " USE_GPU_IF_SUPPORTED;"
        " CPU_ONLY_IF_GPU_CAPABILITY_PROBE_FAILS"
    ),
    "catboost": (
        "PREFER_GPU_TASK_TYPE_IF_USED_AND_SUPPORTED"
        if gpu_available
        else "CPU_FALLBACK_IF_NO_GPU"
    ),
    "scientific_note": (
        "Compute backend changes must not alter frozen scientific "
        "hyperparameters, memberships, threshold semantics, or target policy."
    ),
}

print("\nProspective boosting policy:")
print(json.dumps(gpu_policy, indent=2))


# ---------------------------------------------------------------------------
# 5. Clone repository cleanly
# ---------------------------------------------------------------------------

print("\n" + "=" * 88)
print("GITHUB REPOSITORY")
print("=" * 88)

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository."
        )

    dirty = run(["git", "status", "--porcelain"], cwd=REPO_DIR)

    if dirty:
        raise RuntimeError(
            "Existing repository has uncommitted changes.\n"
            "Refusing to overwrite them:\n\n"
            + dirty
        )

    print("Existing clean repository found.")
    print("Fetching origin/main and tags...")

    run(["git", "fetch", "origin", "main", "--tags"], cwd=REPO_DIR)
    run(["git", "checkout", "main"], cwd=REPO_DIR)

    local_head = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)
    remote_head = run(["git", "rev-parse", "origin/main"], cwd=REPO_DIR)

    if local_head != remote_head:
        run(["git", "merge", "--ff-only", "origin/main"], cwd=REPO_DIR)

else:
    print("Cloning public repository...")
    run(
        [
            "git",
            "clone",
            "--branch",
            "main",
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ]
    )

    run(["git", "fetch", "origin", "--tags"], cwd=REPO_DIR)


# Git identity from repository's established GitHub identity.
run(
    ["git", "config", "user.name", "themubasshir"],
    cwd=REPO_DIR,
)
run(
    [
        "git",
        "config",
        "user.email",
        "10107331+themubasshir@users.noreply.github.com",
    ],
    cwd=REPO_DIR,
)

# Do NOT place token into remote URL.
run(
    ["git", "remote", "set-url", "origin", REPO_URL],
    cwd=REPO_DIR,
)

head = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)
branch = run(["git", "branch", "--show-current"], cwd=REPO_DIR)
status = run(["git", "status", "--porcelain"], cwd=REPO_DIR)

print("Repository :", REPO_DIR)
print("Branch     :", branch)
print("HEAD       :", head)
print("Dirty      :", bool(status))


# ---------------------------------------------------------------------------
# 6. Freeze-start drift guard
# ---------------------------------------------------------------------------

if head != EXPECTED_STARTING_HEAD:
    raise RuntimeError(
        "\nSTAGE24 STARTING-POINT DRIFT DETECTED\n"
        f"Expected: {EXPECTED_STARTING_HEAD}\n"
        f"Actual:   {head}\n\n"
        "Do not continue into Stage24 protocol construction until the new "
        "main HEAD is inspected."
    )

if branch != "main":
    raise RuntimeError(f"Expected branch main, got {branch!r}")

if status:
    raise RuntimeError("Repository is unexpectedly dirty after bootstrap.")

print("\n[PASS] Repository matches inspected Stage24 starting point.")


# ---------------------------------------------------------------------------
# 7. Verify critical inherited artifacts
# ---------------------------------------------------------------------------

CRITICAL_INHERITED = {
    "stage22_chronological_result":
        "results/stage22r_training/"
        "stage22r_2c_chronological_natural/"
        "stage22r_2c_chronological_natural_result.json",

    "stage22_chronological_lightgbm":
        "results/stage22r_training/"
        "stage22r_2c_chronological_natural/"
        "chronological_natural_lightgbm_model.txt",

    "stage22_chronological_xgboost":
        "results/stage22r_training/"
        "stage22r_2c_chronological_natural/"
        "chronological_natural_xgboost_model.json",

    "stage23_final_closure":
        "results/stage23_shortcut_feature_audit/"
        "stage23_7_final_synthesis/"
        "stage23_final_closure_receipt.json",

    "stage23_feature_subset_spec":
        "results/stage23_shortcut_feature_audit/"
        "stage23_0_protocol_lock/"
        "feature_subset_spec.json",

    "stage23_behavior_only_features":
        "results/stage23_shortcut_feature_audit/"
        "stage23_0_protocol_lock/"
        "behavior_only_features.json",

    "stage20_flag_correction":
        "results/stage20_flag_serialization_correction/"
        "stage20_1c8_flag_serialization_correction.json",

    "stage20_s4_localization":
        "results/stage20_jnetpcap_forensic/"
        "stage20_1c10_residual_corrected_s4_localization.json",

    "stage20_s4_pilot":
        "results/stage20_jnetpcap_forensic/"
        "stage20_1c9_corrected_historical_exact_s4_pilot.json",
}

artifact_checks = {}

print("\n" + "=" * 88)
print("INHERITED ARTIFACT CHECK")
print("=" * 88)

for key, rel in CRITICAL_INHERITED.items():
    p = REPO_DIR / rel

    artifact_checks[key] = {
        "path": rel,
        "exists": p.exists(),
        "size_bytes": p.stat().st_size if p.exists() else None,
    }

    mark = "PASS" if p.exists() else "FAIL"
    print(f"[{mark}] {key}")
    print(f"       {rel}")


missing = [
    key
    for key, info in artifact_checks.items()
    if not info["exists"]
]

if missing:
    raise RuntimeError(
        "Missing critical inherited artifacts:\n  - "
        + "\n  - ".join(missing)
    )

print("\n[PASS] Critical Stage20 / Stage22 / Stage23 receipts are present.")


# ---------------------------------------------------------------------------
# 8. Read ONLY inherited JSON receipts from Git repository.
#
# These are already-public frozen artifacts, NOT target dataset rows.
# ---------------------------------------------------------------------------

def load_repo_json(relative_path):
    p = REPO_DIR / relative_path
    with p.open("r", encoding="utf-8") as f:
        return json.load(f)


stage22_result = load_repo_json(
    CRITICAL_INHERITED["stage22_chronological_result"]
)

stage23_closure = load_repo_json(
    CRITICAL_INHERITED["stage23_final_closure"]
)

print("\n" + "=" * 88)
print("INHERITED SCIENTIFIC STATE")
print("=" * 88)

print(
    "Stage22 chronological feature count:",
    stage22_result["data"]["feature_count"],
)
print(
    "Stage22 scaling:",
    stage22_result["data"]["scaling"],
)
print(
    "Stage22 ensemble:",
    stage22_result["models"]["ensemble_probability"],
)
print(
    "Stage22 LightGBM executed backend:",
    stage22_result["models"]["lightgbm"]["executed_parameters"]
        .get("device_type"),
)
print(
    "Stage22 LightGBM originally frozen backend:",
    stage22_result["models"]["lightgbm"]["frozen_original_parameters"]
        .get("device_type"),
)
print(
    "Stage22 XGBoost backend:",
    stage22_result["models"]["xgboost"].get("backend"),
)

print(
    "Stage23 closure:",
    stage23_closure["status"],
)
print(
    "Stage23 fit budget:",
    stage23_closure["stage23_fit_budget"],
)
print(
    "Stage23 additional fits authorized:",
    stage23_closure["governance"]["additional_model_fits_authorized"],
)


# ---------------------------------------------------------------------------
# 9. IMPORTANT compatibility guard
#
# 70 Stage22 features != automatically "bridge70".
# Exact semantic compatibility remains UNRESOLVED.
# ---------------------------------------------------------------------------

stage22_feature_order = stage22_result["data"]["feature_order"]

if len(stage22_feature_order) != 70:
    raise RuntimeError(
        "Unexpected Stage22 feature count; expected inherited 70-feature model."
    )

bridge_compatibility_state = {
    "stage22_feature_count": len(stage22_feature_order),
    "stage22_feature_order": stage22_feature_order,
    "bridge62_compatibility": "UNRESOLVED",
    "bridge70_compatibility": "UNRESOLVED",
    "important": (
        "Numerical equality of feature counts does not establish semantic "
        "bridge compatibility."
    ),
}

print("\nBridge compatibility:")
print("  bridge62: UNRESOLVED")
print("  bridge70: UNRESOLVED")
print(
    "  NOTE: Stage22 has 70 features, but semantic identity with bridge70 "
    "has NOT been assumed."
)


# ---------------------------------------------------------------------------
# 10. /kaggle/input METADATA-ONLY inventory
#
# No file is opened.
# No CSV header is read.
# No Parquet metadata is read.
# Only paths and filesystem sizes are inspected.
# ---------------------------------------------------------------------------

print("\n" + "=" * 88)
print("KAGGLE INPUT INVENTORY — PATHS/SIZES ONLY")
print("=" * 88)

input_inventory = []

if not INPUT_ROOT.exists():
    raise RuntimeError("/kaggle/input does not exist.")

top_level_dirs = sorted(
    [p for p in INPUT_ROOT.iterdir() if p.is_dir()],
    key=lambda x: x.name.lower(),
)

for dataset_dir in top_level_dirs:
    files = [p for p in dataset_dir.rglob("*") if p.is_file()]
    total_bytes = sum(p.stat().st_size for p in files)

    record = {
        "dataset_mount": dataset_dir.name,
        "file_count": len(files),
        "total_bytes": total_bytes,
        "total_gib": round(total_bytes / (1024 ** 3), 6),
        "files": [
            {
                "relative_path": str(p.relative_to(INPUT_ROOT)),
                "size_bytes": p.stat().st_size,
            }
            for p in files
        ],
    }

    input_inventory.append(record)

    print(
        f"{dataset_dir.name:<50} "
        f"files={len(files):>5}  "
        f"size={total_bytes / (1024**3):>9.3f} GiB"
    )


if not top_level_dirs:
    print("[WARNING] No Kaggle input datasets are mounted.")


# Print potentially Stage24-relevant filenames only — still metadata-only.
KEYWORDS = (
    "2017",
    "2018",
    "cic",
    "ids",
    "stage20",
    "ground",
    "s4",
    "flag",
    "correct",
    "published",
    "monday",
    "tuesday",
    "wednesday",
    "thursday",
    "friday",
    "02-28",
    "0228",
)

candidate_paths = []

for dataset in input_inventory:
    for f in dataset["files"]:
        path_l = f["relative_path"].lower()

        if any(k.lower() in path_l for k in KEYWORDS):
            candidate_paths.append(f)


print("\nPotential Stage24-relevant mounted files:")
if candidate_paths:
    for f in candidate_paths[:300]:
        gib = f["size_bytes"] / (1024 ** 3)
        print(f"  {gib:9.4f} GiB  {f['relative_path']}")

    if len(candidate_paths) > 300:
        print(
            f"  ... {len(candidate_paths) - 300} additional candidate files "
            "not printed"
        )
else:
    print("  None identified by filename.")


# ---------------------------------------------------------------------------
# 11. Create Stage24 workspace — NO scientific result files yet
# ---------------------------------------------------------------------------

STAGE24_LOCK_DIR.mkdir(parents=True, exist_ok=True)

print("\nStage24 workspace:")
print(" ", STAGE24_ROOT)
print("Protocol-lock workspace:")
print(" ", STAGE24_LOCK_DIR)


# ---------------------------------------------------------------------------
# 12. Explicit target-opening guard state
# ---------------------------------------------------------------------------

TARGET_INFERENCE_ALLOWED = False
TARGET_OPENINGS_CONSUMED = 0

TARGET_LEDGER_PROSPECTIVE = {
    "2018_to_2017": {
        "bridge62": {
            "published": "0/1",
            "flag_corrected": "0/1",
            "grounded_s4": "0/1",
        },
        "bridge70": {
            "published": "0/1",
            "flag_corrected": "0/1",
            "grounded_s4": "0/1",
        },
    },
    "2017_to_2018": {
        "bridge62": {"feb28": "0/1"},
        "bridge70": {"feb28": "0/1"},
    },
    "status": "PROSPECTIVE_NOT_YET_FROZEN",
}


def assert_target_inference_locked():
    if not TARGET_INFERENCE_ALLOWED:
        raise RuntimeError(
            "STAGE24 TARGET INFERENCE IS LOCKED.\n"
            "The Stage24-0 protocol has not yet been frozen, committed, "
            "pushed, and remotely verified."
        )


# Test the guard without touching any target.
try:
    assert_target_inference_locked()
except RuntimeError:
    print("\n[PASS] Stage24 target-inference guard is ACTIVE.")


# ---------------------------------------------------------------------------
# 13. Bootstrap receipt
#
# This is NOT the final Stage24 protocol lock.
# It explicitly says PROSPECTIVE / ZERO TARGET OPENINGS.
# ---------------------------------------------------------------------------

bootstrap_receipt = {
    "stage": "Stage24-0",
    "artifact": "bootstrap_environment_receipt",
    "status": "PROSPECTIVE_PROTOCOL_CONSTRUCTION_ONLY",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "repository": {
        "owner": REPO_OWNER,
        "name": REPO_NAME,
        "branch": branch,
        "head": head,
        "expected_starting_head": EXPECTED_STARTING_HEAD,
        "working_tree_clean_at_bootstrap": status == "",
    },

    "runtime": {
        "platform": platform.platform(),
        "python": sys.version,
        "gpu_available": gpu_available,
        "gpu_names": gpu_names,
        "versions": versions,
        "boosting_backend_policy_prospective": gpu_policy,
    },

    "inherited_artifacts": artifact_checks,

    "bridge_compatibility": bridge_compatibility_state,

    "target_governance": {
        "target_inference_allowed": False,
        "target_openings_consumed": 0,
        "definition_prospective": (
            "A target is opened when its feature values are first supplied "
            "to a frozen source-domain model for prediction/evaluation."
        ),
        "ledger": TARGET_LEDGER_PROSPECTIVE,
    },

    "kaggle_input_inventory": input_inventory,

    "prohibitions_observed": {
        "csv_rows_read": 0,
        "parquet_rows_read": 0,
        "target_feature_values_read": 0,
        "model_fits": 0,
        "model_predictions": 0,
        "target_metrics_computed": 0,
        "target_threshold_searches": 0,
        "stage24_target_openings": 0,
    },

    "next_authorized_activity": (
        "Stage24-0 schema/provenance inspection sufficient to construct "
        "bridge62/bridge70 and identify PUBLISHED / FLAG_CORRECTED / "
        "GROUNDED_S4 sources. No target model inference."
    ),
}

receipt_path = (
    STAGE24_LOCK_DIR /
    "stage24_0_bootstrap_environment_receipt.json"
)

with receipt_path.open("w", encoding="utf-8") as f:
    json.dump(
        bootstrap_receipt,
        f,
        indent=2,
        sort_keys=True,
    )
    f.write("\n")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


receipt_sha = sha256_file(receipt_path)

sha_path = receipt_path.with_suffix(".sha256")
sha_path.write_text(
    f"{receipt_sha}  {receipt_path.name}\n",
    encoding="utf-8",
)

print("\nBootstrap receipt:")
print(" ", receipt_path)
print("SHA256:")
print(" ", receipt_sha)


# ---------------------------------------------------------------------------
# 14. Final safety assertions
# ---------------------------------------------------------------------------

assert TARGET_INFERENCE_ALLOWED is False
assert TARGET_OPENINGS_CONSUMED == 0
assert head == EXPECTED_STARTING_HEAD
assert branch == "main"
assert not missing
assert len(stage22_feature_order) == 70

print("\n" + "=" * 88)
print("STAGE24-0 BOOTSTRAP COMPLETE")
print("=" * 88)
print(f"Git HEAD:                  {head}")
print(f"GPU detected:              {gpu_available}")
print(f"Stage22 features:          {len(stage22_feature_order)}")
print("bridge62 compatibility:   UNRESOLVED")
print("bridge70 compatibility:   UNRESOLVED")
print("Stage24 target inference: LOCKED")
print("Stage24 target openings:  0")
print("Model fits this cell:      0")
print("Model inference this cell: 0")
print()
print("NEXT:")
print(
    "Schema/provenance-only inspection for bridge62/bridge70 and the "
    "three legitimate CICIDS2017 target representations."
)
print("=" * 88)

In [ ]:
# ============================================================================
# STAGE 24-0A — SCHEMA / PROVENANCE / BRIDGE-COMPATIBILITY AUDIT
#
# PURPOSE
# -------
# 1. Inspect frozen Stage22/Stage23/Stage20 receipts.
# 2. Verify the exact inherited Stage22 feature order.
# 3. Verify Stage23 FULL / NO_SUSPICIOUS_GROUP / BEHAVIOR_ONLY definitions.
# 4. Inventory CICIDS2017 provenance references from frozen Stage20 artifacts.
# 5. Inspect mounted dataset SCHEMAS ONLY.
# 6. Determine whether bridge62 / bridge70 can legitimately be frozen.
#
# STRICTLY PROHIBITED
# -------------------
# - reading any target data row
# - loading target feature values
# - fitting any model
# - loading models for inference
# - predict / predict_proba
# - computing target metrics
# - threshold tuning
# - calibration
# - consuming any Stage24 target opening
#
# CSV inspection:
#   FIRST HEADER RECORD ONLY
#
# PARQUET inspection:
#   FILE METADATA / SCHEMA ONLY
#
# TARGET OPENINGS BEFORE: 0
# TARGET OPENINGS AFTER:  0
# ============================================================================

from __future__ import annotations

import os
import re
import csv
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict

# ---------------------------------------------------------------------------
# 0. Recover / verify notebook state
# ---------------------------------------------------------------------------

REPO_DIR = Path(
    globals().get(
        "REPO_DIR",
        "/kaggle/working/ids2018-validation-safe-ablation",
    )
)

INPUT_ROOT = Path(
    globals().get(
        "INPUT_ROOT",
        "/kaggle/input",
    )
)

STAGE24_ROOT = (
    REPO_DIR
    / "results"
    / "stage24_cross_dataset"
)

STAGE24_LOCK_DIR = (
    STAGE24_ROOT
    / "stage24_0_protocol_lock"
)

EXPECTED_STARTING_HEAD = globals().get(
    "EXPECTED_STARTING_HEAD",
    "963b2da0043c7438624916280de154a5628f8879",
)

STAGE24_LOCK_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 92)
print("STAGE24-0A — SCHEMA / PROVENANCE / BRIDGE-COMPATIBILITY AUDIT")
print("=" * 92)
print("Target inference: FORBIDDEN")
print("Target openings:  0")
print()


# ---------------------------------------------------------------------------
# 1. Helpers
# ---------------------------------------------------------------------------

def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)

    return h.hexdigest()


def git_output(*args):
    import subprocess

    result = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
    )

    return result.stdout.strip()


def canonical_surface_name(name: str) -> str:
    """
    Surface normalization ONLY.

    This is deliberately NOT semantic or fuzzy matching.

    Example:
        ' Flow Duration ' -> 'flowduration'

    But:
        'Dst Port'
        'Destination Port'

    remain different because semantic abbreviation expansion is NOT
    permitted automatically here.
    """
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(name).strip().lower(),
    )


def clean_header_name(name: str) -> str:
    return str(name).replace("\ufeff", "").strip()


def first_csv_header_only(path: Path):
    """
    Read ONLY the first physical record from a CSV.

    No data row is consumed.
    """
    with path.open(
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline="",
    ) as f:
        first_line = f.readline()

    if not first_line:
        return {
            "columns": [],
            "column_count": 0,
            "first_record_sha256": None,
        }

    parsed = next(csv.reader([first_line]))

    columns = [
        clean_header_name(x)
        for x in parsed
    ]

    return {
        "columns": columns,
        "column_count": len(columns),
        "first_record_sha256": hashlib.sha256(
            first_line.encode(
                "utf-8",
                errors="replace",
            )
        ).hexdigest(),
    }


def parquet_schema_only(path: Path):
    """
    Read Parquet footer/schema metadata ONLY.
    No row group is read.
    """
    import pyarrow.parquet as pq

    pf = pq.ParquetFile(path)

    names = list(pf.schema_arrow.names)

    return {
        "columns": [
            clean_header_name(x)
            for x in names
        ],
        "column_count": len(names),
        "num_row_groups": pf.num_row_groups,
        "metadata_only": True,
    }


def schema_only(path: Path):
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return first_csv_header_only(path)

    if suffix in {".parquet", ".pq"}:
        return parquet_schema_only(path)

    return None


def recursive_find_dicts_with_value(obj, wanted):
    """
    Find dictionaries containing a scalar value equal to `wanted`.
    """
    matches = []

    if isinstance(obj, dict):
        if wanted in obj.values():
            matches.append(obj)

        for value in obj.values():
            matches.extend(
                recursive_find_dicts_with_value(
                    value,
                    wanted,
                )
            )

    elif isinstance(obj, list):
        for value in obj:
            matches.extend(
                recursive_find_dicts_with_value(
                    value,
                    wanted,
                )
            )

    return matches


def recursive_string_lists(obj):
    """
    Collect candidate lists containing strings.
    """
    found = []

    if isinstance(obj, list):
        if obj and all(
            isinstance(x, str)
            for x in obj
        ):
            found.append(obj)

        for value in obj:
            found.extend(
                recursive_string_lists(value)
            )

    elif isinstance(obj, dict):
        for value in obj.values():
            found.extend(
                recursive_string_lists(value)
            )

    return found


def extract_subset_feature_list(
    subset_name,
    spec_obj,
    full_features,
):
    """
    Recover subset feature membership without assuming the exact JSON
    serialization used by Stage23.
    """
    matching_dicts = recursive_find_dicts_with_value(
        spec_obj,
        subset_name,
    )

    if not matching_dicts:
        return {
            "status": "NOT_FOUND",
            "features": None,
            "removed_features": None,
            "raw_matches": 0,
        }

    # Usually there should be one canonical subset definition.
    d = matching_dicts[0]

    direct_feature_keys = (
        "features",
        "feature_list",
        "selected_features",
        "kept_features",
        "included_features",
    )

    removed_feature_keys = (
        "removed_features",
        "excluded_features",
    )

    features = None
    removed = None

    for key in direct_feature_keys:
        value = d.get(key)

        if (
            isinstance(value, list)
            and all(
                isinstance(x, str)
                for x in value
            )
        ):
            features = value
            break

    for key in removed_feature_keys:
        value = d.get(key)

        if (
            isinstance(value, list)
            and all(
                isinstance(x, str)
                for x in value
            )
        ):
            removed = value
            break

    if features is None and removed is not None:
        removed_set = set(removed)

        features = [
            f
            for f in full_features
            if f not in removed_set
        ]

    return {
        "status": (
            "RESOLVED"
            if features is not None
            else "FOUND_BUT_FEATURES_UNRESOLVED"
        ),
        "features": features,
        "removed_features": removed,
        "raw_matches": len(matching_dicts),
        "raw_definition": d,
    }


def extract_best_stage23_feature_list(
    obj,
    full_features,
):
    """
    Used for behavior_only_features.json.

    Choose a string list that is a subset of the frozen Stage22
    feature universe and has the greatest length.
    """
    full_set = set(full_features)
    candidates = []

    for lst in recursive_string_lists(obj):
        cleaned = [
            x
            for x in lst
            if x in full_set
        ]

        if (
            cleaned
            and len(cleaned) == len(lst)
        ):
            candidates.append(cleaned)

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: len(x),
        reverse=True,
    )

    return candidates[0]


def recursive_path_strings(obj):
    """
    Extract path-like strings from frozen JSON receipts.
    """
    out = []

    extensions = (
        ".csv",
        ".pcap",
        ".pcapng",
        ".parquet",
        ".json",
        ".jsonl",
        ".tsv",
        ".txt",
    )

    def walk(x, key_path=""):
        if isinstance(x, dict):
            for k, v in x.items():
                walk(
                    v,
                    f"{key_path}.{k}"
                    if key_path
                    else str(k),
                )

        elif isinstance(x, list):
            for i, v in enumerate(x):
                walk(
                    v,
                    f"{key_path}[{i}]",
                )

        elif isinstance(x, str):
            s = x.strip()

            if any(
                ext in s.lower()
                for ext in extensions
            ):
                out.append(
                    {
                        "json_key": key_path,
                        "value": s,
                    }
                )

    walk(obj)

    return out


def recursive_hash_strings(obj):
    """
    Extract declared hashes/checksums from frozen receipts.
    """
    out = []

    def walk(x, key_path=""):
        if isinstance(x, dict):
            for k, v in x.items():
                kp = (
                    f"{key_path}.{k}"
                    if key_path
                    else str(k)
                )

                key_l = str(k).lower()

                if (
                    isinstance(v, str)
                    and (
                        "sha" in key_l
                        or "hash" in key_l
                        or "checksum" in key_l
                    )
                ):
                    out.append(
                        {
                            "json_key": kp,
                            "value": v,
                        }
                    )

                walk(v, kp)

        elif isinstance(x, list):
            for i, v in enumerate(x):
                walk(
                    v,
                    f"{key_path}[{i}]",
                )

    walk(obj)

    return out


# ---------------------------------------------------------------------------
# 2. Repository invariants
# ---------------------------------------------------------------------------

head = git_output("rev-parse", "HEAD")
branch = git_output("branch", "--show-current")
dirty = git_output("status", "--porcelain")

print("Repository state")
print("-" * 92)
print("Branch:", branch)
print("HEAD:  ", head)
print("Dirty: ", bool(dirty))

if head != EXPECTED_STARTING_HEAD:
    raise RuntimeError(
        "\nRepository drift detected.\n"
        f"Expected: {EXPECTED_STARTING_HEAD}\n"
        f"Actual:   {head}"
    )

if branch != "main":
    raise RuntimeError(
        f"Expected main branch, got {branch!r}"
    )

# Bootstrap receipt itself makes the working tree dirty after Cell 1,
# so we do NOT require porcelain == empty here.
print("[PASS] Frozen starting commit unchanged.")
print()


# ---------------------------------------------------------------------------
# 3. Load frozen inherited Stage22 / Stage23 artifacts
# ---------------------------------------------------------------------------

stage22_path = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

stage23_subset_path = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
    / "feature_subset_spec.json"
)

stage23_behavior_path = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
    / "behavior_only_features.json"
)

stage22 = load_json(stage22_path)
stage23_subset_spec = load_json(
    stage23_subset_path
)
stage23_behavior_obj = load_json(
    stage23_behavior_path
)

stage22_features = list(
    stage22["data"]["feature_order"]
)

if len(stage22_features) != 70:
    raise RuntimeError(
        "Inherited Stage22 feature count is not 70."
    )

if len(stage22_features) != len(set(stage22_features)):
    raise RuntimeError(
        "Duplicate Stage22 feature names detected."
    )

print("=" * 92)
print("FROZEN STAGE22 FEATURE UNIVERSE")
print("=" * 92)
print("Feature count:", len(stage22_features))
print()

for i, feature in enumerate(
    stage22_features,
    start=1,
):
    print(f"{i:02d}. {feature}")


# ---------------------------------------------------------------------------
# 4. Resolve Stage23 subset definitions
# ---------------------------------------------------------------------------

full_subset = list(stage22_features)

no_suspicious_info = extract_subset_feature_list(
    "NO_SUSPICIOUS_GROUP",
    stage23_subset_spec,
    stage22_features,
)

behavior_from_spec = extract_subset_feature_list(
    "BEHAVIOR_ONLY",
    stage23_subset_spec,
    stage22_features,
)

behavior_from_file = (
    extract_best_stage23_feature_list(
        stage23_behavior_obj,
        stage22_features,
    )
)

if behavior_from_file is not None:
    behavior_features = behavior_from_file

elif (
    behavior_from_spec["features"]
    is not None
):
    behavior_features = (
        behavior_from_spec["features"]
    )

else:
    behavior_features = None


print("\n" + "=" * 92)
print("STAGE23 SUBSET COMPATIBILITY")
print("=" * 92)

print("FULL:")
print("  n =", len(full_subset))

print("\nNO_SUSPICIOUS_GROUP:")
print(
    "  status =",
    no_suspicious_info["status"],
)

if no_suspicious_info["features"] is not None:
    print(
        "  n      =",
        len(no_suspicious_info["features"]),
    )

if no_suspicious_info["removed_features"]:
    print("  removed:")
    for f in no_suspicious_info[
        "removed_features"
    ]:
        print("   -", f)


print("\nBEHAVIOR_ONLY:")

if behavior_features is None:
    print("  status = UNRESOLVED")
else:
    print("  status = RESOLVED")
    print(
        "  n      =",
        len(behavior_features),
    )

    behavior_removed = [
        f
        for f in stage22_features
        if f not in set(behavior_features)
    ]

    print("  removed:")
    for f in behavior_removed:
        print("   -", f)


subset_alias = False

if (
    no_suspicious_info["features"] is not None
    and behavior_features is not None
):
    subset_alias = (
        no_suspicious_info["features"]
        == behavior_features
    )

print()
print(
    "NO_SUSPICIOUS_GROUP == BEHAVIOR_ONLY:",
    subset_alias,
)

if subset_alias:
    print(
        "[IMPORTANT] These two Stage23 labels resolve to the "
        "same ordered feature representation."
    )
    print(
        "[IMPORTANT] Stage24 should preserve both semantic labels "
        "but MUST NOT independently reopen an identical target "
        "matrix merely to duplicate the same evaluation."
    )


# ---------------------------------------------------------------------------
# 5. Inspect frozen Stage20 provenance artifacts
# ---------------------------------------------------------------------------

stage20_dirs = [
    REPO_DIR
    / "results"
    / "stage20_flag_serialization_correction",

    REPO_DIR
    / "results"
    / "stage20_jnetpcap_forensic",
]

stage20_json_files = []

for root in stage20_dirs:
    if root.exists():
        stage20_json_files.extend(
            sorted(root.rglob("*.json"))
        )

print("\n" + "=" * 92)
print("FROZEN STAGE20 PROVENANCE REFERENCES")
print("=" * 92)
print(
    "Stage20 JSON artifacts inspected:",
    len(stage20_json_files),
)

stage20_provenance = []

for json_path in stage20_json_files:
    # Defensive limit: receipts should be small.
    # This avoids accidentally loading a large data artifact.
    size = json_path.stat().st_size

    if size > 16 * 1024 * 1024:
        continue

    try:
        obj = load_json(json_path)
    except Exception:
        continue

    paths = recursive_path_strings(obj)
    hashes = recursive_hash_strings(obj)

    if paths or hashes:
        record = {
            "repo_relative_path": str(
                json_path.relative_to(REPO_DIR)
            ),
            "size_bytes": size,
            "path_references": paths,
            "declared_hashes": hashes,
        }

        stage20_provenance.append(record)


# Deduplicated human-readable references.
all_stage20_refs = []

for record in stage20_provenance:
    for item in record["path_references"]:
        all_stage20_refs.append(
            {
                "artifact": record[
                    "repo_relative_path"
                ],
                "json_key": item["json_key"],
                "value": item["value"],
            }
        )

# Prioritize likely CICIDS2017 / correction / S4 references.
interesting_stage20_refs = []

for item in all_stage20_refs:
    text = (
        item["value"]
        + " "
        + item["artifact"]
        + " "
        + item["json_key"]
    ).lower()

    if any(
        token in text
        for token in (
            "2017",
            "trafficlabelling",
            "workinghours",
            "pcap",
            "s4",
            "correct",
            "flag",
            "ground",
            "jnetpcap",
        )
    ):
        interesting_stage20_refs.append(item)


# Deduplicate same reference triplets.
seen = set()
deduped_stage20_refs = []

for item in interesting_stage20_refs:
    key = (
        item["artifact"],
        item["json_key"],
        item["value"],
    )

    if key not in seen:
        seen.add(key)
        deduped_stage20_refs.append(item)


for item in deduped_stage20_refs[:250]:
    print()
    print(
        "Artifact:",
        item["artifact"],
    )
    print(
        "Key:     ",
        item["json_key"],
    )
    print(
        "Value:   ",
        item["value"],
    )

if len(deduped_stage20_refs) > 250:
    print(
        "\n...",
        len(deduped_stage20_refs) - 250,
        "additional provenance references omitted from console "
        "but retained in JSON.",
    )


# ---------------------------------------------------------------------------
# 6. Scan mounted files by PATH only
# ---------------------------------------------------------------------------

all_input_files = sorted(
    [
        p
        for p in INPUT_ROOT.rglob("*")
        if p.is_file()
    ],
    key=lambda p: str(p).lower(),
)

CIC2017_DAY_TOKENS = (
    "monday-workinghours",
    "tuesday-workinghours",
    "wednesday-workinghours",
    "wednesday-workinghours",
    "thursday-workinghours",
    "friday-workinghours",
    "webattacks",
    "infilteration",
    "infiltration",
    "portscan",
    "patator",
)

def looks_like_cicids2017(path: Path):
    s = str(path).lower().replace("_", "-")

    if "2017" in s:
        return True

    if any(
        token in s
        for token in CIC2017_DAY_TOKENS
    ):
        return True

    return False


def looks_like_ids2018(path: Path):
    s = str(path).lower()

    if "2018" in s:
        return True

    if re.search(
        r"/0[23]-\d{2}-2018\.csv$",
        s,
    ):
        return True

    return False


cic2017_files = [
    p
    for p in all_input_files
    if looks_like_cicids2017(p)
]

ids2018_files = [
    p
    for p in all_input_files
    if looks_like_ids2018(p)
]


print("\n" + "=" * 92)
print("MOUNTED DATASET CANDIDATES — PATH ONLY")
print("=" * 92)

print(
    "CICIDS2017 candidate files:",
    len(cic2017_files),
)

for p in cic2017_files:
    print(
        "  ",
        str(p.relative_to(INPUT_ROOT)),
    )

print()
print(
    "IDS2018 candidate files:",
    len(ids2018_files),
)

for p in ids2018_files:
    print(
        "  ",
        str(p.relative_to(INPUT_ROOT)),
    )


# ---------------------------------------------------------------------------
# 7. SCHEMA-ONLY inspection
#
# We read:
#   CSV     -> first header record only
#   Parquet -> footer/schema metadata only
#
# ZERO DATA ROWS.
# ---------------------------------------------------------------------------

schema_records = {
    "cicids2017": [],
    "ids2018": [],
}

allowed_schema_suffixes = {
    ".csv",
    ".parquet",
    ".pq",
}


def inspect_schema_group(paths, output_list):
    for path in paths:
        if path.suffix.lower() not in allowed_schema_suffixes:
            continue

        try:
            schema = schema_only(path)

            output_list.append(
                {
                    "path": str(
                        path.relative_to(INPUT_ROOT)
                    ),
                    "size_bytes": path.stat().st_size,
                    "schema": schema,
                    "status": "SCHEMA_READ_ONLY",
                }
            )

        except Exception as exc:
            output_list.append(
                {
                    "path": str(
                        path.relative_to(INPUT_ROOT)
                    ),
                    "size_bytes": path.stat().st_size,
                    "schema": None,
                    "status": "SCHEMA_READ_FAILED",
                    "error": repr(exc),
                }
            )


inspect_schema_group(
    cic2017_files,
    schema_records["cicids2017"],
)

inspect_schema_group(
    ids2018_files,
    schema_records["ids2018"],
)


print("\n" + "=" * 92)
print("SCHEMA-ONLY RESULTS")
print("=" * 92)

for domain_name in (
    "cicids2017",
    "ids2018",
):
    print(
        f"\n[{domain_name.upper()}]"
    )

    records = schema_records[domain_name]

    if not records:
        print("  NO SCHEMA CANDIDATES FOUND")
        continue

    for rec in records:
        print()
        print("Path:   ", rec["path"])
        print("Status: ", rec["status"])

        if rec["schema"]:
            print(
                "Columns:",
                rec["schema"]["column_count"],
            )


# ---------------------------------------------------------------------------
# 8. Compare frozen Stage22 features to mounted schemas
#
# IMPORTANT:
# This performs literal/surface comparisons ONLY.
#
# No fuzzy matching.
# No semantic aliasing.
# No bridge is frozen here.
# ---------------------------------------------------------------------------

stage22_exact_set = set(stage22_features)

stage22_surface = defaultdict(list)

for feature in stage22_features:
    stage22_surface[
        canonical_surface_name(feature)
    ].append(feature)


schema_comparisons_2017 = []

for rec in schema_records["cicids2017"]:
    if rec["schema"] is None:
        continue

    target_columns = [
        clean_header_name(x)
        for x in rec["schema"]["columns"]
    ]

    # Remove obvious label column ONLY for reporting feature overlap.
    # We do NOT read labels or data.
    target_feature_columns = [
        c
        for c in target_columns
        if c.lower() not in {
            "label",
            "class",
            "target",
        }
    ]

    target_exact_set = set(
        target_feature_columns
    )

    exact_matches = [
        f
        for f in stage22_features
        if f in target_exact_set
    ]

    surface_target = defaultdict(list)

    for c in target_feature_columns:
        surface_target[
            canonical_surface_name(c)
        ].append(c)

    surface_matches = []

    for f in stage22_features:
        key = canonical_surface_name(f)

        if key in surface_target:
            surface_matches.append(
                {
                    "stage22_feature": f,
                    "target_columns": surface_target[
                        key
                    ],
                }
            )

    unmatched_stage22 = [
        f
        for f in stage22_features
        if canonical_surface_name(f)
        not in surface_target
    ]

    schema_comparisons_2017.append(
        {
            "path": rec["path"],
            "stage22_feature_count": len(
                stage22_features
            ),
            "target_nonlabel_column_count": len(
                target_feature_columns
            ),
            "literal_exact_match_count": len(
                exact_matches
            ),
            "surface_only_match_count": len(
                surface_matches
            ),
            "exact_matches": exact_matches,
            "surface_matches": surface_matches,
            "unmatched_stage22_features": (
                unmatched_stage22
            ),
            "important": (
                "Surface match does not establish semantic "
                "equivalence and is not a frozen bridge."
            ),
        }
    )


print("\n" + "=" * 92)
print("STAGE22 ↔ CICIDS2017 SURFACE COMPATIBILITY")
print("=" * 92)

if not schema_comparisons_2017:
    print(
        "No mounted CICIDS2017 schema exists to compare."
    )
else:
    for comp in schema_comparisons_2017:
        print()
        print("Target:", comp["path"])
        print(
            "  Stage22 features:       ",
            comp["stage22_feature_count"],
        )
        print(
            "  Target non-label cols:  ",
            comp["target_nonlabel_column_count"],
        )
        print(
            "  Literal exact matches:  ",
            comp["literal_exact_match_count"],
        )
        print(
            "  Surface-name matches:   ",
            comp["surface_only_match_count"],
        )
        print(
            "  Still semantically "
            "unresolved:",
            len(
                comp[
                    "unmatched_stage22_features"
                ]
            ),
        )


# ---------------------------------------------------------------------------
# 9. Resolve current bridge status
# ---------------------------------------------------------------------------

if not cic2017_files:
    bridge_status = {
        "bridge62": (
            "BLOCKED_MISSING_CICIDS2017_MOUNT"
        ),
        "bridge70": (
            "BLOCKED_MISSING_CICIDS2017_MOUNT"
        ),
        "semantic_mapping_frozen": False,
        "reason": (
            "No CICIDS2017 candidate dataset is mounted. "
            "A semantic cross-dataset bridge cannot be "
            "legitimately frozen from IDS2018 columns alone."
        ),
    }

elif not schema_comparisons_2017:
    bridge_status = {
        "bridge62": (
            "BLOCKED_NO_READABLE_CICIDS2017_SCHEMA"
        ),
        "bridge70": (
            "BLOCKED_NO_READABLE_CICIDS2017_SCHEMA"
        ),
        "semantic_mapping_frozen": False,
        "reason": (
            "CICIDS2017 candidates exist but no compatible "
            "CSV/Parquet schema was successfully inspected."
        ),
    }

else:
    bridge_status = {
        "bridge62": (
            "PROSPECTIVE_SEMANTIC_MAPPING_REQUIRED"
        ),
        "bridge70": (
            "PROSPECTIVE_SEMANTIC_MAPPING_REQUIRED"
        ),
        "semantic_mapping_frozen": False,
        "reason": (
            "CICIDS2017 schemas are available. Exact semantic "
            "feature equivalence, conversions, flag compatibility "
            "and exclusions must now be frozen prospectively."
        ),
    }


# Stage22 direct-model compatibility must remain unresolved until the exact
# ordered bridge vectors are frozen.
model_compatibility = {
    "stage22_original_feature_count": 70,

    "stage22_original_feature_order_sha256": hashlib.sha256(
        json.dumps(
            stage22_features,
            ensure_ascii=False,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest(),

    "bridge62_direct_stage22_model_compatibility":
        "UNRESOLVED",

    "bridge70_direct_stage22_model_compatibility":
        "UNRESOLVED",

    "rule": (
        "Equal dimensionality is insufficient. Direct compatibility "
        "requires exact ordered semantic identity with the feature "
        "representation used to fit the inherited Stage22 model."
    ),
}


# ---------------------------------------------------------------------------
# 10. Determine target-source provenance readiness
# ---------------------------------------------------------------------------

published_refs = []
corrected_refs = []
grounded_refs = []

for item in deduped_stage20_refs:
    text = (
        item["value"]
        + " "
        + item["artifact"]
        + " "
        + item["json_key"]
    ).lower()

    if (
        "trafficlabelling" in text
        or "workinghours" in text
        or "published" in text
    ):
        published_refs.append(item)

    if (
        "correct" in text
        or "flag_serialization" in text
        or "flag" in text
    ):
        corrected_refs.append(item)

    if any(
        token in text
        for token in (
            "ground",
            "s4",
            "jnetpcap",
        )
    ):
        grounded_refs.append(item)


target_variant_readiness = {
    "PUBLISHED": {
        "frozen_stage20_references_found":
            len(published_refs),
        "mounted_candidate_count":
            len(cic2017_files),
        "ready_for_final_freeze":
            bool(
                published_refs
                and cic2017_files
            ),
    },

    "FLAG_CORRECTED": {
        "frozen_stage20_references_found":
            len(corrected_refs),
        "ready_for_final_freeze":
            bool(corrected_refs),
        "note": (
            "Presence of a frozen correction receipt does not "
            "itself authorize target inference."
        ),
    },

    "GROUNDED_S4": {
        "frozen_stage20_references_found":
            len(grounded_refs),
        "ready_for_final_freeze":
            bool(grounded_refs),
        "note": (
            "Final legitimacy requires identifying the exact "
            "frozen grounded S4 construction/source and its "
            "identity semantics."
        ),
    },
}


# ---------------------------------------------------------------------------
# 11. Record backend issue prospectively
#
# No GPU probe/model fit is performed here.
# ---------------------------------------------------------------------------

backend_audit = {
    "gpu_hardware_from_bootstrap":
        globals().get(
            "gpu_available",
            None,
        ),

    "stage22_lightgbm_original_requested_backend":
        stage22["models"]["lightgbm"][
            "frozen_original_parameters"
        ].get("device_type"),

    "stage22_lightgbm_executed_backend":
        stage22["models"]["lightgbm"][
            "executed_parameters"
        ].get("device_type"),

    "stage22_xgboost_backend":
        stage22["models"]["xgboost"].get(
            "backend"
        ),

    "stage24_gpu_policy":
        "GPU_FIRST_WHEN_SUPPORTED",

    "lightgbm_stage24_status":
        "GPU_CAPABILITY_PROBE_NOT_YET_EXECUTED",

    "rule": (
        "Any Stage24-authorized bridge-specific LightGBM refit "
        "must use GPU if the current LightGBM build/runtime passes "
        "the frozen capability probe; CPU fallback requires a "
        "machine-readable failure receipt."
    ),

    "model_fits_performed_in_stage24_0a": 0,
}


# ---------------------------------------------------------------------------
# 12. Stage24 subset evaluation de-duplication rule
# ---------------------------------------------------------------------------

subset_transfer_rule = {
    "requested_protocol_labels": [
        "FULL",
        "NO_SUSPICIOUS_GROUP",
        "BEHAVIOR_ONLY",
    ],

    "no_suspicious_feature_count": (
        len(
            no_suspicious_info["features"]
        )
        if no_suspicious_info["features"]
        is not None
        else None
    ),

    "behavior_only_feature_count": (
        len(behavior_features)
        if behavior_features is not None
        else None
    ),

    "identical_ordered_membership":
        subset_alias,

    "prospective_rule": (
        "If two frozen subset labels have identical ordered "
        "feature membership, execute one underlying target "
        "evaluation and reference that same result under both "
        "semantic labels. Do not consume a duplicate opening."
        if subset_alias
        else
        "Distinct frozen subset representations require distinct "
        "evaluations according to the final opening ledger."
    ),
}


# ---------------------------------------------------------------------------
# 13. Write prospective audit receipt
# ---------------------------------------------------------------------------

audit_receipt = {
    "stage": "Stage24-0A",

    "artifact":
        "schema_provenance_bridge_compatibility_audit",

    "status": (
        "BLOCKED_MISSING_CICIDS2017_MOUNT"
        if not cic2017_files
        else
        "PROSPECTIVE_BRIDGE_REVIEW_REQUIRED"
    ),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "repository": {
        "branch": branch,
        "head": head,
        "expected_starting_head":
            EXPECTED_STARTING_HEAD,
    },

    "scientific_safety": {
        "target_data_rows_read": 0,
        "target_feature_values_read": 0,
        "target_labels_read": 0,
        "models_loaded_for_inference": 0,
        "model_fits": 0,
        "model_predictions": 0,
        "target_metrics_computed": 0,
        "threshold_searches": 0,
        "stage24_target_openings_consumed": 0,
        "schema_metadata_reads_only": True,
    },

    "stage22": {
        "feature_count":
            len(stage22_features),
        "feature_order":
            stage22_features,
        "scaling":
            stage22["data"]["scaling"],
        "ensemble_probability":
            stage22["models"][
                "ensemble_probability"
            ],
    },

    "stage23_subset_audit": {
        "full_features":
            full_subset,

        "no_suspicious_group":
            no_suspicious_info,

        "behavior_only_features":
            behavior_features,

        "no_suspicious_equals_behavior_only":
            subset_alias,

        "subset_transfer_rule":
            subset_transfer_rule,
    },

    "stage20_provenance": {
        "json_artifacts_inspected":
            len(stage20_json_files),

        "interesting_path_references":
            deduped_stage20_refs,

        "published_references":
            published_refs,

        "flag_corrected_references":
            corrected_refs,

        "grounded_s4_references":
            grounded_refs,
    },

    "mounted_input": {
        "cicids2017_candidate_count":
            len(cic2017_files),

        "cicids2017_candidates": [
            str(
                p.relative_to(
                    INPUT_ROOT
                )
            )
            for p in cic2017_files
        ],

        "ids2018_candidate_count":
            len(ids2018_files),

        "ids2018_candidates": [
            str(
                p.relative_to(
                    INPUT_ROOT
                )
            )
            for p in ids2018_files
        ],
    },

    "schema_records":
        schema_records,

    "stage22_to_2017_surface_comparisons":
        schema_comparisons_2017,

    "bridge_status":
        bridge_status,

    "inherited_model_bridge_compatibility":
        model_compatibility,

    "target_variant_readiness":
        target_variant_readiness,

    "backend_audit":
        backend_audit,

    "target_opening_ledger_state":
        "ALL_ZERO",

    "final_protocol_lock_created":
        False,

    "final_protocol_lock_committed":
        False,

    "next_allowed_action": (
        "Attach/identify the legitimate CICIDS2017 source(s) "
        "if absent; otherwise construct the explicit prospective "
        "2018↔2017 semantic feature mapping and Stage24-0 lock. "
        "Target model inference remains forbidden."
    ),
}

audit_path = (
    STAGE24_LOCK_DIR
    / "stage24_0a_schema_provenance_audit.json"
)

with audit_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        audit_receipt,
        f,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )
    f.write("\n")

audit_sha = sha256_file(
    audit_path
)

audit_sha_path = (
    audit_path.with_suffix(".sha256")
)

audit_sha_path.write_text(
    f"{audit_sha}  {audit_path.name}\n",
    encoding="utf-8",
)


# ---------------------------------------------------------------------------
# 14. Final console synthesis
# ---------------------------------------------------------------------------

print("\n" + "=" * 92)
print("STAGE24-0A AUDIT SUMMARY")
print("=" * 92)

print("Repository HEAD:")
print(" ", head)

print()
print("Stage22 feature count:")
print(" ", len(stage22_features))

print()
print("Stage23 subset state:")
print(
    "  FULL                =",
    len(full_subset),
)

print(
    "  NO_SUSPICIOUS_GROUP =",
    (
        len(
            no_suspicious_info[
                "features"
            ]
        )
        if no_suspicious_info[
            "features"
        ] is not None
        else "UNRESOLVED"
    ),
)

print(
    "  BEHAVIOR_ONLY       =",
    (
        len(behavior_features)
        if behavior_features is not None
        else "UNRESOLVED"
    ),
)

print(
    "  NSG == BEHAVIOR     =",
    subset_alias,
)

print()
print("Mounted CICIDS2017 candidates:")
print(" ", len(cic2017_files))

print()
print("Bridge62:")
print(
    " ",
    bridge_status["bridge62"],
)

print("Bridge70:")
print(
    " ",
    bridge_status["bridge70"],
)

print()
print("Semantic bridge frozen:")
print(
    " ",
    bridge_status[
        "semantic_mapping_frozen"
    ],
)

print()
print("Stage24 target openings consumed:")
print("  0")

print()
print("Model fits:")
print("  0")

print("Model predictions:")
print("  0")

print()
print("Audit receipt:")
print(" ", audit_path)

print("SHA256:")
print(" ", audit_sha)

print()
print("-" * 92)

if not cic2017_files:
    print(
        "BLOCKER: NO CICIDS2017 DATASET IS CURRENTLY "
        "MOUNTED IN /kaggle/input."
    )
    print()
    print(
        "DO NOT FREEZE bridge62/bridge70 yet."
    )
    print(
        "DO NOT perform target inference."
    )
    print(
        "DO NOT substitute a different CICIDS2017 copy "
        "without provenance verification."
    )
else:
    print(
        "CICIDS2017 schema detected."
    )
    print(
        "Next step: explicit semantic bridge construction "
        "and target-source provenance freeze."
    )

print("-" * 92)
print("STAGE24-0A COMPLETE — TARGET OPENINGS REMAIN 0/1")
print("=" * 92)

In [ ]:
# ============================================================================
# STAGE24-0B — EXACT CICIDS2017 SOURCE RECOVERY + TARGET LEDGER REPAIR
#
# PURPOSE
# -------
# 1. Repair the Stage24 bookkeeping:
#      - retire nonexistent NO_SUSPICIOUS_GROUP
#      - distinguish Stage23 BEHAVIOR_ONLY from Stage24 bridge62
#      - establish the frozen 8-cell target-opening ledger
#
# 2. Recover exact CICIDS2017 traffic-label source identities from frozen
#    Stage20 artifacts already committed in this repository.
#
# 3. Download ONLY those exact pinned Hugging Face artifacts.
#
# 4. Verify:
#      - exact revision
#      - exact byte size
#      - exact SHA256
#
# 5. Inspect PARQUET FILE METADATA / SCHEMA ONLY.
#
# STRICTLY FORBIDDEN
# ------------------
# - reading any Parquet row group
# - pandas.read_parquet on CICIDS2017
# - extracting target feature values
# - extracting target labels
# - fitting preprocessing
# - loading source models for target inference
# - predict / predict_proba
# - computing any target statistic or metric
# - constructing FLAG_CORRECTED values
# - constructing GROUNDED_S4 target rows
# - target-driven feature mapping
#
# TARGET MODEL OPENINGS BEFORE: 0 / 8
# TARGET MODEL OPENINGS AFTER:  0 / 8
# ============================================================================

from __future__ import annotations

import os
import re
import json
import hashlib
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict

# ---------------------------------------------------------------------------
# 0. Frozen paths / commit
# ---------------------------------------------------------------------------

REPO_DIR = Path(
    globals().get(
        "REPO_DIR",
        "/kaggle/working/ids2018-validation-safe-ablation",
    )
)

EXPECTED_STARTING_HEAD = (
    "963b2da0043c7438624916280de154a5628f8879"
)

STAGE24_ROOT = (
    REPO_DIR
    / "results"
    / "stage24_cross_dataset"
)

STAGE24_LOCK_DIR = (
    STAGE24_ROOT
    / "stage24_0_protocol_lock"
)

SOURCE_ROOT = Path(
    "/kaggle/working/stage24_cicids2017_sources"
)

HF_CACHE = Path(
    "/kaggle/working/stage24_hf_cache"
)

STAGE24_LOCK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SOURCE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

HF_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)

print("=" * 96)
print("STAGE24-0B — EXACT CICIDS2017 SOURCE RECOVERY + TARGET LEDGER REPAIR")
print("=" * 96)
print("Target inference:          FORBIDDEN")
print("Target row parsing:        FORBIDDEN")
print("Target model openings:     0 / 8")
print("Allowed target operation:  OPAQUE BYTE INTEGRITY + SCHEMA METADATA ONLY")
print()


# ---------------------------------------------------------------------------
# 1. Helpers
# ---------------------------------------------------------------------------

def git_output(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=True,
    )
    return result.stdout.strip()


def load_json(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
):
    """
    Opaque whole-file byte integrity operation.

    Scientific governance:
      - bytes are hashed only
      - no columns are parsed
      - no target values are exposed
      - no target labels are exposed
    """
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def canonical_json_sha256(obj):
    payload = json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


def recursively_find_traffic_label_sources(
    obj,
    json_path,
    key_path="",
):
    """
    Locate frozen dictionaries serialized as:

        "traffic_labels": {
            "remote": ...,
            "revision": ...,
            "sha256": ...,
            ...
        }

    This operates ONLY on repository JSON receipts/manifests.
    """
    found = []

    if isinstance(obj, dict):
        for k, v in obj.items():
            kp = (
                f"{key_path}.{k}"
                if key_path
                else str(k)
            )

            if (
                str(k) == "traffic_labels"
                and isinstance(v, dict)
                and "remote" in v
                and "revision" in v
                and "sha256" in v
            ):
                found.append(
                    {
                        "manifest": str(
                            json_path.relative_to(
                                REPO_DIR
                            )
                        ),
                        "json_key": kp,
                        "remote": str(
                            v["remote"]
                        ),
                        "revision": str(
                            v["revision"]
                        ),
                        "sha256": str(
                            v["sha256"]
                        ).lower(),
                        "size_bytes": (
                            int(v["size_bytes"])
                            if v.get(
                                "size_bytes"
                            ) is not None
                            else None
                        ),
                        "rows_declared": (
                            int(v["rows"])
                            if v.get("rows")
                            is not None
                            else None
                        ),
                    }
                )

            found.extend(
                recursively_find_traffic_label_sources(
                    v,
                    json_path,
                    kp,
                )
            )

    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            found.extend(
                recursively_find_traffic_label_sources(
                    v,
                    json_path,
                    f"{key_path}[{i}]",
                )
            )

    return found


def infer_hf_repo_ids_from_frozen_receipts(
    candidate_jsons,
):
    """
    Recover HF repo identity from frozen cache paths such as:

      datasets--bvsam--cic-ids-2017/...

    No external discovery is used.
    """
    found = set()

    pattern = re.compile(
        r"datasets--"
        r"([A-Za-z0-9_.-]+)"
        r"--"
        r"([A-Za-z0-9_.-]+)"
    )

    for path in candidate_jsons:
        try:
            if path.stat().st_size > 16 * 1024 * 1024:
                continue

            text = path.read_text(
                encoding="utf-8",
                errors="replace",
            )

        except Exception:
            continue

        for owner, repo in pattern.findall(text):
            found.add(
                f"{owner}/{repo}"
            )

    return sorted(found)


def safe_link_or_copy(
    source: Path,
    destination: Path,
):
    """
    Prefer a hard link so exact downloaded bytes are not duplicated.
    Falls back to copy if required.
    """
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if destination.exists():
        destination.unlink()

    real_source = source.resolve()

    try:
        os.link(
            real_source,
            destination,
        )

        return "HARDLINK"

    except Exception:
        shutil.copy2(
            real_source,
            destination,
        )

        return "COPY"


# ---------------------------------------------------------------------------
# 2. Repository safety check
# ---------------------------------------------------------------------------

head = git_output(
    "rev-parse",
    "HEAD",
)

branch = git_output(
    "branch",
    "--show-current",
)

porcelain = git_output(
    "status",
    "--porcelain",
)

print("=" * 96)
print("REPOSITORY SAFETY")
print("=" * 96)
print("Branch:", branch)
print("HEAD:  ", head)

if head != EXPECTED_STARTING_HEAD:
    raise RuntimeError(
        "\nRepository commit drift detected.\n"
        f"Expected: {EXPECTED_STARTING_HEAD}\n"
        f"Actual:   {head}"
    )

if branch != "main":
    raise RuntimeError(
        f"Expected branch 'main', got {branch!r}"
    )

# Stage24-created receipts make the worktree dirty.
# Reject anything dirty OUTSIDE Stage24.
unexpected_dirty = []

if porcelain:
    for line in porcelain.splitlines():
        # git porcelain path begins after status chars.
        rel = line[3:].strip()

        # Handle "old -> new" rename form.
        if " -> " in rel:
            rel = rel.split(
                " -> ",
                1,
            )[1]

        if not rel.startswith(
            "results/stage24_cross_dataset/"
        ):
            unexpected_dirty.append(
                line
            )

print(
    "Dirty only from Stage24 receipts:",
    not bool(unexpected_dirty),
)

if unexpected_dirty:
    raise RuntimeError(
        "\nUnexpected repository modifications exist "
        "outside results/stage24_cross_dataset:\n"
        + "\n".join(
            unexpected_dirty
        )
    )

print("[PASS] Scientific parent commit remains unchanged.")
print()


# ---------------------------------------------------------------------------
# 3. Load inherited Stage22 / Stage23 frozen artifacts
# ---------------------------------------------------------------------------

STAGE22_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

STAGE23_FEATURE_SPEC = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
    / "feature_subset_spec.json"
)

STAGE23_BEHAVIOR = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
    / "behavior_only_features.json"
)

stage22 = load_json(
    STAGE22_RESULT
)

stage23_spec = load_json(
    STAGE23_FEATURE_SPEC
)

stage23_behavior_obj = load_json(
    STAGE23_BEHAVIOR
)

stage22_features = list(
    stage22["data"]["feature_order"]
)

assert len(
    stage22_features
) == 70

assert len(
    set(stage22_features)
) == 70


# ---------------------------------------------------------------------------
# 4. Repair the NO_SUSPICIOUS_GROUP bookkeeping error
# ---------------------------------------------------------------------------

stage23_spec_text = json.dumps(
    stage23_spec,
    ensure_ascii=False,
)

has_behavior_only = (
    "BEHAVIOR_ONLY"
    in stage23_spec_text
)

has_nonexistent_nsg = (
    "NO_SUSPICIOUS_GROUP"
    in stage23_spec_text
)

print("=" * 96)
print("STAGE23 BOOKKEEPING REPAIR")
print("=" * 96)
print(
    "BEHAVIOR_ONLY present in frozen Stage23 spec:",
    has_behavior_only,
)
print(
    "NO_SUSPICIOUS_GROUP present in frozen Stage23 spec:",
    has_nonexistent_nsg,
)

if not has_behavior_only:
    raise RuntimeError(
        "Frozen Stage23 BEHAVIOR_ONLY definition not found."
    )

if has_nonexistent_nsg:
    raise RuntimeError(
        "Unexpected: NO_SUSPICIOUS_GROUP actually exists "
        "in the frozen Stage23 spec. Manual review required."
    )

print()
print(
    "[PASS] NO_SUSPICIOUS_GROUP is retired from Stage24 bookkeeping."
)
print(
    "[PASS] BEHAVIOR_ONLY remains a Stage23 representation only."
)
print()


# ---------------------------------------------------------------------------
# 5. Freeze the INTENT of Stage24 bridge62 / bridge70
#
# IMPORTANT:
# This is NOT yet the 2018<->2017 semantic column mapping.
#
# bridge62:
#   Stage22 70-feature order minus the 8 flag-dependent fields.
#
# bridge70:
#   all 70 Stage22 features.
#
# Target-side column identity remains unresolved until exact CICIDS2017
# schema is recovered below.
# ---------------------------------------------------------------------------

FLAG_DEPENDENT_SOURCE_FEATURES = [
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "ACK Flag Cnt",
    "URG Flag Cnt",
    "CWE Flag Count",
    "ECE Flag Cnt",
]

missing_flag_fields = [
    f
    for f in FLAG_DEPENDENT_SOURCE_FEATURES
    if f not in stage22_features
]

if missing_flag_fields:
    raise RuntimeError(
        "Expected Stage22 flag fields missing: "
        + repr(
            missing_flag_fields
        )
    )

bridge70_source_features = list(
    stage22_features
)

bridge62_source_features = [
    f
    for f in stage22_features
    if f not in set(
        FLAG_DEPENDENT_SOURCE_FEATURES
    )
]

assert len(
    bridge70_source_features
) == 70

assert len(
    bridge62_source_features
) == 62


print("=" * 96)
print("STAGE24 BRIDGE INTENT")
print("=" * 96)
print(
    "bridge70 source feature count:",
    len(bridge70_source_features),
)
print(
    "bridge62 source feature count:",
    len(bridge62_source_features),
)

print("\nbridge62 excluded flag-dependent fields:")

for f in FLAG_DEPENDENT_SOURCE_FEATURES:
    print("  -", f)

print()
print(
    "[IMPORTANT] bridge62 is NOT Stage23 BEHAVIOR_ONLY."
)
print(
    "[IMPORTANT] No target-side semantic equivalence has been "
    "asserted yet."
)
print()


# ---------------------------------------------------------------------------
# 6. Freeze Stage24 opening ledger
#
# Opening definition:
# target feature values first supplied to the frozen source-domain model.
#
# Source download, SHA256, and file-schema metadata inspection DO NOT consume
# an opening.
# ---------------------------------------------------------------------------

target_opening_ledger = {
    "PRIMARY_2018_TO_2017": {
        "bridge62": {
            "PUBLISHED": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },
            "FLAG_CORRECTED": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },
            "GROUNDED_S4": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },
        },

        "bridge70": {
            "PUBLISHED": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },
            "FLAG_CORRECTED": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },
            "GROUNDED_S4": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },
        },
    },

    "SECONDARY_2017_TO_2018": {
        "bridge62": {
            "FEB28_2018": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            }
        },

        "bridge70": {
            "FEB28_2018": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            }
        },
    },
}

ledger_cells = []

for direction, bridges in (
    target_opening_ledger.items()
):
    for bridge, targets in bridges.items():
        for target_name, cell in targets.items():
            ledger_cells.append(
                {
                    "direction": direction,
                    "bridge": bridge,
                    "target": target_name,
                    **cell,
                }
            )

assert len(
    ledger_cells
) == 8

assert sum(
    x["openings"]
    for x in ledger_cells
) == 0

assert sum(
    x["budget"]
    for x in ledger_cells
) == 8

print("=" * 96)
print("FROZEN STAGE24 TARGET-OPENING LEDGER")
print("=" * 96)

for i, cell in enumerate(
    ledger_cells,
    start=1,
):
    print(
        f"{i}. "
        f"{cell['direction']} / "
        f"{cell['bridge']} / "
        f"{cell['target']}: "
        f"{cell['openings']}/{cell['budget']} "
        f"{cell['status']}"
    )

print()
print("Total openings consumed: 0 / 8")
print()


# ---------------------------------------------------------------------------
# 7. Stage20 forensic flag correction — metadata only
#
# We preserve the already-frozen 1C8 mapping for later Stage24-0C.
# We DO NOT apply it to any target row in this cell.
# ---------------------------------------------------------------------------

STAGE20_FLAG_CORRECTION = (
    REPO_DIR
    / "results"
    / "stage20_flag_serialization_correction"
    / "stage20_1c8_flag_serialization_correction.json"
)

flag_correction = load_json(
    STAGE20_FLAG_CORRECTION
)

physical_csv_to_semantic_flag = dict(
    flag_correction[
        "physical_csv_column_to_semantic_flag"
    ]
)

if len(
    physical_csv_to_semantic_flag
) != 8:
    raise RuntimeError(
        "Stage20 1C8 flag map is not eight fields."
    )

print("=" * 96)
print("INHERITED STAGE20 1C8 FLAG CORRECTION")
print("=" * 96)
print(
    "Stored for Stage24-0C; NOT APPLIED to target data in this cell."
)

for physical, semantic in (
    physical_csv_to_semantic_flag.items()
):
    print(
        f"  {physical:20s} -> {semantic}"
    )

print()


# ---------------------------------------------------------------------------
# 8. Discover exact Stage20-pinned traffic-label artifacts
# ---------------------------------------------------------------------------

candidate_json_paths = []

for root in [
    REPO_DIR / "results",
]:
    for path in root.rglob("*.json"):
        rel_lower = str(
            path.relative_to(REPO_DIR)
        ).lower()

        # Only Stage20/Stage21 source/recovery artifacts are needed.
        if (
            "stage20" not in rel_lower
            and "stage21" not in rel_lower
        ):
            continue

        # Defensive: receipts/manifests only.
        if path.stat().st_size > (
            32 * 1024 * 1024
        ):
            continue

        candidate_json_paths.append(
            path
        )


raw_source_records = []

for path in candidate_json_paths:
    try:
        obj = load_json(path)
    except Exception:
        continue

    raw_source_records.extend(
        recursively_find_traffic_label_sources(
            obj,
            path,
        )
    )


# Deduplicate by remote path and REQUIRE metadata consistency.
grouped = defaultdict(list)

for record in raw_source_records:
    grouped[
        record["remote"]
    ].append(record)


source_contract = []

for remote, records in sorted(
    grouped.items()
):
    signatures = {
        (
            r["revision"],
            r["sha256"],
            r["size_bytes"],
            r["rows_declared"],
        )
        for r in records
    }

    if len(signatures) != 1:
        print()
        print(
            "CONFLICTING SOURCE METADATA:",
            remote,
        )

        for x in records:
            print(x)

        raise RuntimeError(
            "Frozen Stage20 source metadata conflict detected."
        )

    first = records[0]

    source_contract.append(
        {
            "remote": remote,
            "revision": first[
                "revision"
            ],
            "sha256": first[
                "sha256"
            ],
            "size_bytes": first[
                "size_bytes"
            ],
            "rows_declared": first[
                "rows_declared"
            ],
            "evidence_manifests": sorted(
                {
                    x["manifest"]
                    for x in records
                }
            ),
        }
    )


print("=" * 96)
print("FROZEN STAGE20 TRAFFIC-LABEL SOURCE CONTRACT")
print("=" * 96)
print(
    "Unique pinned traffic-label artifacts found:",
    len(source_contract),
)

if not source_contract:
    raise RuntimeError(
        "No frozen Stage20 traffic-label source metadata "
        "could be recovered."
    )

for rec in source_contract:
    print()
    print(
        "Remote:  ",
        rec["remote"],
    )
    print(
        "Revision:",
        rec["revision"],
    )
    print(
        "SHA256:  ",
        rec["sha256"],
    )
    print(
        "Bytes:   ",
        rec["size_bytes"],
    )
    print(
        "Rows*:   ",
        rec["rows_declared"],
    )

print()
print(
    "* row counts above come from frozen manifests; "
    "target rows have NOT been parsed."
)
print()


# ---------------------------------------------------------------------------
# 9. Recover exact Hugging Face dataset repo identity from frozen receipts
# ---------------------------------------------------------------------------

hf_repo_ids = (
    infer_hf_repo_ids_from_frozen_receipts(
        candidate_json_paths
    )
)

print("=" * 96)
print("HUGGING FACE SOURCE IDENTITY")
print("=" * 96)
print(
    "Frozen dataset repo IDs discovered:",
    hf_repo_ids,
)

EXPECTED_HF_REPO = (
    "bvsam/cic-ids-2017"
)

if EXPECTED_HF_REPO not in hf_repo_ids:
    raise RuntimeError(
        "\nExpected frozen Stage20 HF source "
        f"{EXPECTED_HF_REPO!r} was not recovered from "
        "repository receipts.\n"
        "Do not substitute another CICIDS2017 source."
    )

if len(hf_repo_ids) != 1:
    raise RuntimeError(
        "\nMore than one Stage20 Hugging Face dataset source "
        "was recovered. Manual provenance review required:\n"
        + repr(hf_repo_ids)
    )

HF_DATASET_REPO = hf_repo_ids[0]

print(
    "[PASS] Exact Stage20 source repo recovered:",
    HF_DATASET_REPO,
)
print()


# ---------------------------------------------------------------------------
# 10. Acquire exact pinned files
#
# IMPORTANT:
# This downloads the frozen source BYTES ONLY.
# It does not parse dataset rows.
# ---------------------------------------------------------------------------

try:
    from huggingface_hub import (
        hf_hub_download,
        __version__ as hf_hub_version,
    )

except Exception as exc:
    raise RuntimeError(
        "huggingface_hub is required for exact source recovery."
    ) from exc


print("=" * 96)
print("EXACT SOURCE ACQUISITION")
print("=" * 96)
print(
    "huggingface_hub:",
    hf_hub_version,
)
print(
    "Repository:",
    HF_DATASET_REPO,
)
print()


acquisition_records = []

for index, rec in enumerate(
    source_contract,
    start=1,
):
    print(
        f"[{index}/{len(source_contract)}] "
        f"{rec['remote']}"
    )

    acquisition = {
        **rec,
        "repo_id": HF_DATASET_REPO,
        "download_status": None,
        "materialization_mode": None,
        "local_path": None,
        "actual_size_bytes": None,
        "actual_sha256": None,
        "size_match": None,
        "sha256_match": None,
        "schema_status": "NOT_INSPECTED",
        "schema_columns": None,
        "schema_column_count": None,
        "parquet_metadata_rows": None,
        "row_groups": None,
        "error": None,
    }

    try:
        cached_path = Path(
            hf_hub_download(
                repo_id=HF_DATASET_REPO,
                repo_type="dataset",
                filename=rec["remote"],
                revision=rec["revision"],
                cache_dir=str(
                    HF_CACHE
                ),
            )
        )

        if not cached_path.exists():
            raise RuntimeError(
                "hf_hub_download returned a nonexistent path."
            )

        local_name = (
            Path(
                rec["remote"]
            ).name
        )

        durable_path = (
            SOURCE_ROOT
            / local_name
        )

        link_mode = safe_link_or_copy(
            cached_path,
            durable_path,
        )

        actual_size = (
            durable_path.stat().st_size
        )

        actual_sha = sha256_file(
            durable_path
        )

        size_match = (
            rec["size_bytes"] is None
            or actual_size
            == rec["size_bytes"]
        )

        sha_match = (
            actual_sha.lower()
            == rec["sha256"].lower()
        )

        acquisition.update(
            {
                "download_status":
                    "DOWNLOADED_AND_HASHED",

                "materialization_mode":
                    link_mode,

                "local_path":
                    str(durable_path),

                "actual_size_bytes":
                    actual_size,

                "actual_sha256":
                    actual_sha,

                "size_match":
                    size_match,

                "sha256_match":
                    sha_match,
            }
        )

        print(
            "  bytes expected:",
            rec["size_bytes"],
        )
        print(
            "  bytes actual:  ",
            actual_size,
        )
        print(
            "  sha expected:  ",
            rec["sha256"],
        )
        print(
            "  sha actual:    ",
            actual_sha,
        )

        if not size_match:
            raise RuntimeError(
                "Exact size mismatch."
            )

        if not sha_match:
            raise RuntimeError(
                "Exact SHA256 mismatch."
            )

        print(
            "  [PASS] Exact frozen source identity"
        )

    except Exception as exc:
        acquisition[
            "download_status"
        ] = "FAILED"

        acquisition[
            "error"
        ] = repr(exc)

        print(
            "  [FAIL]",
            repr(exc),
        )

    acquisition_records.append(
        acquisition
    )

    print()


# ---------------------------------------------------------------------------
# 11. Refuse schema inspection unless EVERY downloaded artifact passed SHA256
# ---------------------------------------------------------------------------

failed_acquisitions = [
    x
    for x in acquisition_records
    if (
        x["download_status"]
        != "DOWNLOADED_AND_HASHED"
        or not x["sha256_match"]
        or not x["size_match"]
    )
]

all_exact_sources_verified = (
    len(failed_acquisitions) == 0
)

print("=" * 96)
print("SOURCE IDENTITY GATE")
print("=" * 96)
print(
    "Exact pinned artifacts:",
    len(acquisition_records),
)
print(
    "Verified:",
    len(acquisition_records)
    - len(failed_acquisitions),
)
print(
    "Failed:",
    len(failed_acquisitions),
)

if failed_acquisitions:
    print()
    print(
        "SOURCE RECOVERY IS INCOMPLETE."
    )
    print(
        "Schema comparison and bridge freezing remain blocked."
    )
else:
    print(
        "[PASS] Every recovered target-source artifact is "
        "byte-identical to its frozen Stage20 source."
    )

print()


# ---------------------------------------------------------------------------
# 12. Parquet metadata/schema inspection ONLY
#
# PyArrow ParquetFile reads footer metadata/schema.
# NO row group is requested.
# NO target feature value is materialized.
# ---------------------------------------------------------------------------

if all_exact_sources_verified:
    import pyarrow.parquet as pq

    print("=" * 96)
    print("PARQUET SCHEMA METADATA — NO ROW DATA")
    print("=" * 96)

    for acquisition in acquisition_records:
        local_path = Path(
            acquisition["local_path"]
        )

        try:
            pf = pq.ParquetFile(
                local_path
            )

            schema_columns = list(
                pf.schema_arrow.names
            )

            metadata_rows = (
                int(
                    pf.metadata.num_rows
                )
                if pf.metadata
                is not None
                else None
            )

            row_groups = (
                int(
                    pf.num_row_groups
                )
            )

            acquisition.update(
                {
                    "schema_status":
                        "PARQUET_FOOTER_ONLY",

                    "schema_columns":
                        schema_columns,

                    "schema_column_count":
                        len(
                            schema_columns
                        ),

                    "parquet_metadata_rows":
                        metadata_rows,

                    "row_groups":
                        row_groups,
                }
            )

            print()
            print(
                "File:",
                local_path.name,
            )
            print(
                "  columns:",
                len(
                    schema_columns
                ),
            )
            print(
                "  rows from footer metadata:",
                metadata_rows,
            )
            print(
                "  row groups:",
                row_groups,
            )

            print(
                "  schema:"
            )

            for i, column in enumerate(
                schema_columns,
                start=1,
            ):
                print(
                    f"    {i:02d}. {column}"
                )

        except Exception as exc:
            acquisition[
                "schema_status"
            ] = "FAILED"

            acquisition[
                "error"
            ] = repr(exc)

            print(
                "  [FAIL] schema metadata:",
                repr(exc),
            )


# ---------------------------------------------------------------------------
# 13. Schema consistency diagnostics
#
# Still NO row values.
# ---------------------------------------------------------------------------

schema_fingerprints = defaultdict(
    list
)

for acquisition in (
    acquisition_records
):
    columns = acquisition.get(
        "schema_columns"
    )

    if columns is None:
        continue

    fingerprint = (
        canonical_json_sha256(
            columns
        )
    )

    schema_fingerprints[
        fingerprint
    ].append(
        Path(
            acquisition["local_path"]
        ).name
    )


print("\n" + "=" * 96)
print("CICIDS2017 PHYSICAL SCHEMA GROUPS")
print("=" * 96)

if not schema_fingerprints:
    print(
        "No verified schema metadata available."
    )
else:
    print(
        "Distinct ordered physical schemas:",
        len(schema_fingerprints),
    )

    for i, (
        fingerprint,
        files,
    ) in enumerate(
        sorted(
            schema_fingerprints.items()
        ),
        start=1,
    ):
        print()
        print(
            f"Schema group {i}"
        )
        print(
            "  SHA256:",
            fingerprint,
        )

        for name in files:
            print(
                "  -",
                name,
            )


# ---------------------------------------------------------------------------
# 14. Record what is and is NOT scientifically established
# ---------------------------------------------------------------------------

schema_all_pass = (
    all_exact_sources_verified
    and all(
        x["schema_status"]
        == "PARQUET_FOOTER_ONLY"
        for x in acquisition_records
    )
)

bridge_status = {
    "bridge62": (
        "SOURCE_SCHEMA_RECOVERED__SEMANTIC_MAPPING_NOT_YET_FROZEN"
        if schema_all_pass
        else
        "BLOCKED_SOURCE_RECOVERY_INCOMPLETE"
    ),

    "bridge70": (
        "SOURCE_SCHEMA_RECOVERED__SEMANTIC_MAPPING_NOT_YET_FROZEN"
        if schema_all_pass
        else
        "BLOCKED_SOURCE_RECOVERY_INCOMPLETE"
    ),

    "bridge62_source_dimension": 62,
    "bridge70_source_dimension": 70,

    "semantic_mapping_frozen": False,

    "flag_correction_applied":
        False,

    "grounded_s4_population_constructed":
        False,
}


target_variant_status = {
    "PUBLISHED": {
        "status": (
            "PINNED_STAGE20_SOURCE_CONTAINER_RECOVERED"
            if schema_all_pass
            else
            "BLOCKED"
        ),

        "important": (
            "Recovery of the Stage20-pinned traffic-label "
            "container does not yet itself freeze the Stage24 "
            "PUBLISHED evaluation representation."
        ),
    },

    "FLAG_CORRECTED": {
        "status":
            "NOT_CONSTRUCTED",

        "stage20_1c8_rule_loaded":
            True,

        "target_values_touched":
            False,
    },

    "GROUNDED_S4": {
        "status":
            "NOT_CONSTRUCTED",

        "target_values_touched":
            False,
    },
}


# ---------------------------------------------------------------------------
# 15. Write Stage24-0B receipt
# ---------------------------------------------------------------------------

receipt = {
    "stage": "Stage24-0B",

    "type":
        "EXACT_CICIDS2017_SOURCE_RECOVERY_AND_LEDGER_REPAIR",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "repository": {
        "branch": branch,
        "head": head,
        "expected_head":
            EXPECTED_STARTING_HEAD,
    },

"stage24_0a_subset_lookup_correction": {
    "previous_stage24_0a_state": {
        "NO_SUSPICIOUS_GROUP":
            "UNRESOLVED",
    },

    "correction_reason": (
        "Stage24-0A lookup/parser failed to resolve an existing "
        "frozen Stage23 subset. Direct inspection of the frozen "
        "feature_subset_spec.json establishes the authoritative "
        "definition."
    ),

    "scientific_change":
        False,

    "target_data_accessed_for_correction":
        False,

    "target_openings_consumed_for_correction":
        0,
},

"stage23_subset_resolution": {
    "FULL": {
        "feature_count": 70,
        "removed": [],
        "semantic_label":
            stage23_full[
                "semantic_label"
            ],
    },

    "NO_SUSPICIOUS_GROUP": {
        "feature_count": 67,
        "removed":
            EXPECTED_NSG_REMOVED,

        "semantic_label":
            "joint_shortcut_prone_group_ablation",
    },

    "BEHAVIOR_ONLY": {
        "feature_count": 63,
        "removed":
            EXPECTED_BEHAVIOR_REMOVED,

        "semantic_label":
            "behavior_restricted_feature_set",
    },
},

"stage24_representation_distinctions": {
    "NO_SUSPICIOUS_GROUP_is_BEHAVIOR_ONLY":
        False,

    "NO_SUSPICIOUS_GROUP_is_bridge62":
        False,

    "BEHAVIOR_ONLY_is_bridge62":
        False,

    "bridge62":
        "STAGE24_62_FEATURE_FLAG_EXCLUDED_PRIMARY_BRIDGE",

    "bridge70":
        "STAGE24_70_FEATURE_FLAG_INCLUDED_SENSITIVITY_BRIDGE",
},

    "stage24_bridge_intent": {
        "bridge62_source_features":
            bridge62_source_features,

        "bridge70_source_features":
            bridge70_source_features,

        "flag_dependent_source_features":
            FLAG_DEPENDENT_SOURCE_FEATURES,

        "semantic_target_mapping_frozen":
            False,
    },

    "stage20_1c8": {
        "physical_csv_column_to_semantic_flag":
            physical_csv_to_semantic_flag,

        "applied_to_target_rows":
            False,
    },

    "target_representations": [
        "PUBLISHED",
        "FLAG_CORRECTED",
        "GROUNDED_S4",
    ],

    "target_variant_status":
        target_variant_status,

    "target_opening_definition": (
        "A target opening occurs when target feature values "
        "are first supplied to the frozen source-domain model "
        "for the specified direction/bridge/target cell."
    ),

    "target_opening_ledger":
        target_opening_ledger,

    "opening_summary": {
        "cells": 8,
        "consumed": 0,
        "budget": 8,
        "all_closed": True,
    },

    "source_repository": {
        "provider":
            "Hugging Face",

        "repo_id":
            HF_DATASET_REPO,

        "identity_recovered_from_frozen_stage20_receipts":
            True,
    },

    "frozen_source_contract":
        source_contract,

    "acquisition_records":
        acquisition_records,

    "all_exact_sources_verified":
        all_exact_sources_verified,

    "schema_metadata_all_pass":
        schema_all_pass,

    "physical_schema_groups": {
        fingerprint: files
        for fingerprint, files
        in schema_fingerprints.items()
    },

    "bridge_status":
        bridge_status,

    "scientific_safety": {
        "target_file_bytes_hashed":
            True,

        "whole_file_hashing_is_opaque_integrity_operation":
            True,

        "target_row_groups_read":
            0,

        "target_feature_values_materialized":
            0,

        "target_labels_materialized":
            0,

        "target_statistics_computed":
            0,

        "target_preprocessing_fitted":
            0,

        "models_loaded_for_target_inference":
            0,

        "model_fits":
            0,

        "model_predictions":
            0,

        "threshold_searches":
            0,

        "metrics_computed":
            0,

        "target_openings_consumed":
            0,
    },

    "next_authorized_step": (
        "Stage24-0C: construct and freeze explicit "
        "2018<->2017 semantic feature mapping for bridge62 "
        "and bridge70, including all conversions, flag semantics, "
        "variant construction rules, target population rules, "
        "source-side refit policy, thresholds, scaler policy, "
        "and immutable 8-cell opening ledger. No target model "
        "opening before that lock is committed."
        if schema_all_pass
        else
        "Resolve exact pinned CICIDS2017 source acquisition "
        "failure. No semantic bridge freeze or target model "
        "opening is authorized."
    ),
}

receipt_path = (
    STAGE24_LOCK_DIR
    / "stage24_0b_exact_cicids2017_source_recovery.json"
)

with receipt_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        receipt,
        f,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )
    f.write("\n")


receipt_sha = sha256_file(
    receipt_path
)

sha_path = (
    receipt_path.with_suffix(
        ".sha256"
    )
)

sha_path.write_text(
    f"{receipt_sha}  "
    f"{receipt_path.name}\n",
    encoding="utf-8",
)


# ---------------------------------------------------------------------------
# 16. Final synthesis
# ---------------------------------------------------------------------------

print("\n" + "=" * 96)
print("STAGE24-0B FINAL SYNTHESIS")
print("=" * 96)

print(
    "Repository HEAD:",
    head,
)

print()
print(
    "Stage23 FULL:",
    len(
        stage23_full["features"]
    ),
)

print(
    "Stage23 NO_SUSPICIOUS_GROUP:",
    len(
        stage23_nsg["features"]
    ),
)

print(
    "Stage23 BEHAVIOR_ONLY:",
    len(
        stage23_behavior["features"]
    ),
)

print(
    "NO_SUSPICIOUS_GROUP == BEHAVIOR_ONLY:",
    stage23_nsg["features"]
    == stage23_behavior["features"],
)

print(
    "Stage24-0A NSG lookup miss corrected:",
    True,
)

print()
print(
    "bridge62 source dimension:",
    len(
        bridge62_source_features
    ),
)

print(
    "bridge70 source dimension:",
    len(
        bridge70_source_features
    ),
)

print()
print(
    "Pinned CICIDS2017 source repo:",
    HF_DATASET_REPO,
)

print(
    "Pinned traffic-label artifacts:",
    len(
        source_contract
    ),
)

print(
    "Exact source identity all pass:",
    all_exact_sources_verified,
)

print(
    "Schema metadata all pass:",
    schema_all_pass,
)

print()
print(
    "bridge62:",
    bridge_status["bridge62"],
)

print(
    "bridge70:",
    bridge_status["bridge70"],
)

print()
print(
    "FLAG_CORRECTED constructed:",
    False,
)

print(
    "GROUNDED_S4 constructed:",
    False,
)

print()
print(
    "Target model openings:",
    "0 / 8",
)

print(
    "Model fits:",
    0,
)

print(
    "Model predictions:",
    0,
)

print(
    "Target metrics:",
    0,
)

print()
print(
    "Receipt:",
    receipt_path,
)

print(
    "SHA256:",
    receipt_sha,
)

print()
print("-" * 96)

if schema_all_pass:
    print(
        "[PASS] EXACT CICIDS2017 SOURCE CONTAINERS RECOVERED."
    )
    print(
        "[PASS] TARGET ROWS REMAIN UNREAD."
    )
    print(
        "[PASS] TARGET MODEL OPENINGS REMAIN 0 / 8."
    )
    print()
    print(
        "NEXT: Stage24-0C explicit semantic bridge + "
        "variant/population protocol lock."
    )
else:
    print(
        "[BLOCKED] Exact CICIDS2017 source recovery "
        "is incomplete."
    )
    print(
        "DO NOT construct a substitute target dataset."
    )
    print(
        "DO NOT freeze bridge62/bridge70."
    )
    print(
        "DO NOT perform model inference."
    )

print("-" * 96)
print("STAGE24-0B COMPLETE")
print("=" * 96)

In [ ]:
# ============================================================================
# STAGE24-0B — EXACT CICIDS2017 SOURCE RECOVERY + FROZEN SUBSET RESOLUTION
# COMPLETE CORRECTED DROP-IN CELL
#
# SCIENTIFIC BOUNDARY
# -------------------
# ALLOWED:
#   - inspect frozen repository JSON metadata
#   - recover exact frozen CICIDS2017 source identities
#   - download exact pinned source BYTES
#   - whole-file SHA256
#   - Parquet footer/schema metadata only
#
# FORBIDDEN:
#   - read any CICIDS2017 row group
#   - materialize any target feature value
#   - materialize any target label
#   - fit any model
#   - load models for target inference
#   - predict / predict_proba
#   - threshold selection
#   - metrics
#   - target-driven mapping
#   - FLAG_CORRECTED row construction
#   - GROUNDED_S4 row construction
#
# TARGET MODEL OPENINGS BEFORE: 0 / 8
# TARGET MODEL OPENINGS AFTER:  0 / 8
# ============================================================================

from __future__ import annotations

import os
import re
import json
import hashlib
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict

# ============================================================================
# 0. CONSTANTS
# ============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "963b2da0043c7438624916280de154a5628f8879"
)

STAGE24_ROOT = (
    REPO_DIR
    / "results"
    / "stage24_cross_dataset"
)

LOCK_DIR = (
    STAGE24_ROOT
    / "stage24_0_protocol_lock"
)

CICIDS_SOURCE_DIR = Path(
    "/kaggle/working/stage24_cicids2017_sources"
)

HF_CACHE_DIR = Path(
    "/kaggle/working/stage24_hf_cache"
)

LOCK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CICIDS_SOURCE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HF_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


STAGE22_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

STAGE23_FEATURE_SPEC = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
    / "feature_subset_spec.json"
)

STAGE20_FLAG_CORRECTION = (
    REPO_DIR
    / "results"
    / "stage20_flag_serialization_correction"
    / "stage20_1c8_flag_serialization_correction.json"
)

STAGE20_1E_DIR = (
    REPO_DIR
    / "results"
    / "stage20_1e_training"
)

EXPECTED_HF_REPO = (
    "bvsam/cic-ids-2017"
)


print("=" * 100)
print("STAGE24-0B — EXACT CICIDS2017 SOURCE RECOVERY + FROZEN SUBSET RESOLUTION")
print("=" * 100)
print("Target inference:          FORBIDDEN")
print("Target row parsing:        FORBIDDEN")
print("Target labels:             FORBIDDEN")
print("Target model openings:     0 / 8")
print("Model fits:                0")
print("Model predictions:         0")
print("Allowed target operation:  OPAQUE BYTE HASHING + PARQUET FOOTER/SCHEMA ONLY")
print()


# ============================================================================
# 1. HELPERS
# ============================================================================

def run_git(*args):
    proc = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=True,
    )
    return proc.stdout.strip()


def load_json(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
):
    """
    Opaque byte hashing only.
    Does not parse dataset rows or labels.
    """
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_json_object(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


def hardlink_or_copy(
    source: Path,
    destination: Path,
):
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if destination.exists():
        destination.unlink()

    real_source = source.resolve()

    try:
        os.link(
            real_source,
            destination,
        )

        return "HARDLINK"

    except Exception:
        shutil.copy2(
            real_source,
            destination,
        )

        return "COPY"


def recursive_traffic_label_sources(
    obj,
    source_json: Path,
    key_path="",
):
    """
    Extract dictionaries having the frozen Stage20 structure:

        traffic_labels:
            remote
            revision
            sha256
            size_bytes
            rows

    Operates ONLY on repository metadata JSON.
    """
    found = []

    if isinstance(obj, dict):

        for key, value in obj.items():

            current_key = (
                f"{key_path}.{key}"
                if key_path
                else str(key)
            )

            if (
                key == "traffic_labels"
                and isinstance(value, dict)
                and value.get("remote")
                and value.get("revision")
                and value.get("sha256")
            ):
                found.append(
                    {
                        "manifest": str(
                            source_json.relative_to(
                                REPO_DIR
                            )
                        ),
                        "json_key": current_key,
                        "remote": str(
                            value["remote"]
                        ),
                        "revision": str(
                            value["revision"]
                        ),
                        "sha256": str(
                            value["sha256"]
                        ).lower(),
                        "size_bytes": (
                            int(value["size_bytes"])
                            if value.get(
                                "size_bytes"
                            ) is not None
                            else None
                        ),
                        "rows_declared": (
                            int(value["rows"])
                            if value.get(
                                "rows"
                            ) is not None
                            else None
                        ),
                    }
                )

            found.extend(
                recursive_traffic_label_sources(
                    value,
                    source_json,
                    current_key,
                )
            )

    elif isinstance(obj, list):

        for index, value in enumerate(obj):

            found.extend(
                recursive_traffic_label_sources(
                    value,
                    source_json,
                    f"{key_path}[{index}]",
                )
            )

    return found


def discover_hf_dataset_ids(
    json_paths,
):
    """
    Recover dataset IDs from frozen HF cache path strings such as:

       datasets--bvsam--cic-ids-2017
    """
    pattern = re.compile(
        r"datasets--"
        r"([A-Za-z0-9_.-]+)"
        r"--"
        r"([A-Za-z0-9_.-]+)"
    )

    found = set()

    for path in json_paths:

        try:
            if path.stat().st_size > (
                32 * 1024 * 1024
            ):
                continue

            text = path.read_text(
                encoding="utf-8",
                errors="replace",
            )

        except Exception:
            continue

        for owner, repo in pattern.findall(
            text
        ):
            found.add(
                f"{owner}/{repo}"
            )

    return sorted(found)


# ============================================================================
# 2. REPOSITORY SAFETY
# ============================================================================

print("=" * 100)
print("REPOSITORY SAFETY")
print("=" * 100)

branch = run_git(
    "branch",
    "--show-current",
)

head = run_git(
    "rev-parse",
    "HEAD",
)

status = run_git(
    "status",
    "--porcelain",
)

print("Branch:", branch)
print("HEAD:  ", head)

if branch != "main":
    raise RuntimeError(
        f"Expected branch main, got {branch!r}"
    )

if head != EXPECTED_HEAD:
    raise RuntimeError(
        "\nFrozen scientific parent changed.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


# Stage24 receipts are expected to make the tree dirty.
# Anything else is forbidden.
unexpected_dirty = []

if status:

    for line in status.splitlines():

        rel = line[3:].strip()

        if " -> " in rel:
            rel = rel.split(
                " -> ",
                1,
            )[1]

        if not rel.startswith(
            "results/stage24_cross_dataset/"
        ):
            unexpected_dirty.append(
                line
            )


print(
    "Dirty only from Stage24 receipts:",
    not bool(unexpected_dirty),
)

if unexpected_dirty:

    raise RuntimeError(
        "\nUnexpected modifications outside Stage24:\n"
        + "\n".join(
            unexpected_dirty
        )
    )

print(
    "[PASS] Frozen starting commit remains unchanged."
)
print()


# ============================================================================
# 3. LOAD FROZEN STAGE22 / STAGE23 ARTIFACTS
# ============================================================================

for required in [
    STAGE22_RESULT,
    STAGE23_FEATURE_SPEC,
    STAGE20_FLAG_CORRECTION,
]:

    if not required.exists():

        raise FileNotFoundError(
            required
        )


stage22 = load_json(
    STAGE22_RESULT
)

stage23_spec = load_json(
    STAGE23_FEATURE_SPEC
)

stage20_flag = load_json(
    STAGE20_FLAG_CORRECTION
)


stage22_features = list(
    stage22[
        "data"
    ][
        "feature_order"
    ]
)

if len(stage22_features) != 70:

    raise RuntimeError(
        "Stage22 frozen feature universe is not 70."
    )

if len(set(stage22_features)) != 70:

    raise RuntimeError(
        "Stage22 frozen feature order contains duplicates."
    )


print("=" * 100)
print("FROZEN STAGE22 FEATURE UNIVERSE")
print("=" * 100)
print(
    "Feature count:",
    len(stage22_features),
)
print()


# ============================================================================
# 4. EXACT STAGE23 SUBSET RESOLUTION
#
# IMPORTANT:
# Stage24-0A reported NO_SUSPICIOUS_GROUP as unresolved.
# That was a lookup/parser miss.
#
# The actual frozen Stage23 spec DOES contain:
#
# FULL                  = 70
# NO_SUSPICIOUS_GROUP   = 67
# BEHAVIOR_ONLY         = 63
#
# NO_SUSPICIOUS_GROUP removes:
#   Dst Port
#   Init Fwd Win Byts
#   Fwd Seg Size Min
#
# BEHAVIOR_ONLY removes:
#   Dst Port
#   Protocol
#   Fwd Header Len
#   Bwd Header Len
#   Init Fwd Win Byts
#   Init Bwd Win Byts
#   Fwd Seg Size Min
# ============================================================================

print("=" * 100)
print("FROZEN STAGE23 SUBSET RESOLUTION")
print("=" * 100)

stage23_subsets = stage23_spec.get(
    "subsets"
)

if not isinstance(
    stage23_subsets,
    dict,
):
    raise RuntimeError(
        "Stage23 subsets dictionary missing."
    )


required_subsets = [
    "FULL",
    "NO_SUSPICIOUS_GROUP",
    "BEHAVIOR_ONLY",
]

missing_subsets = [
    name
    for name in required_subsets
    if name not in stage23_subsets
]

if missing_subsets:

    raise RuntimeError(
        "Required Stage23 subsets missing: "
        + repr(
            missing_subsets
        )
    )


stage23_full = (
    stage23_subsets[
        "FULL"
    ]
)

stage23_nsg = (
    stage23_subsets[
        "NO_SUSPICIOUS_GROUP"
    ]
)

stage23_behavior = (
    stage23_subsets[
        "BEHAVIOR_ONLY"
    ]
)


EXPECTED_NSG_REMOVED = [
    "Dst Port",
    "Init Fwd Win Byts",
    "Fwd Seg Size Min",
]

EXPECTED_BEHAVIOR_REMOVED = [
    "Dst Port",
    "Protocol",
    "Fwd Header Len",
    "Bwd Header Len",
    "Init Fwd Win Byts",
    "Init Bwd Win Byts",
    "Fwd Seg Size Min",
]


# --------------------------------------------------------------------------
# FULL
# --------------------------------------------------------------------------

if stage23_full.get(
    "feature_count"
) != 70:
    raise RuntimeError(
        "Stage23 FULL feature_count != 70"
    )

if list(
    stage23_full.get(
        "features",
        [],
    )
) != stage22_features:
    raise RuntimeError(
        "Stage23 FULL feature order differs from Stage22."
    )

if list(
    stage23_full.get(
        "removed",
        [],
    )
) != []:
    raise RuntimeError(
        "Stage23 FULL unexpectedly removes features."
    )


# --------------------------------------------------------------------------
# NO_SUSPICIOUS_GROUP
# --------------------------------------------------------------------------

if stage23_nsg.get(
    "feature_count"
) != 67:
    raise RuntimeError(
        "Stage23 NO_SUSPICIOUS_GROUP feature_count != 67"
    )

if stage23_nsg.get(
    "mode"
) != "remove":
    raise RuntimeError(
        "Unexpected NO_SUSPICIOUS_GROUP mode."
    )

if list(
    stage23_nsg.get(
        "removed",
        [],
    )
) != EXPECTED_NSG_REMOVED:
    raise RuntimeError(
        "Frozen NO_SUSPICIOUS_GROUP removal list changed."
    )

if (
    stage23_nsg.get(
        "semantic_label"
    )
    != "joint_shortcut_prone_group_ablation"
):
    raise RuntimeError(
        "Unexpected NO_SUSPICIOUS_GROUP semantic label."
    )


expected_nsg_features = [
    feature
    for feature in stage22_features
    if feature not in set(
        EXPECTED_NSG_REMOVED
    )
]

if list(
    stage23_nsg.get(
        "features",
        [],
    )
) != expected_nsg_features:
    raise RuntimeError(
        "NO_SUSPICIOUS_GROUP ordered feature list mismatch."
    )


# --------------------------------------------------------------------------
# BEHAVIOR_ONLY
# --------------------------------------------------------------------------

if stage23_behavior.get(
    "feature_count"
) != 63:
    raise RuntimeError(
        "Stage23 BEHAVIOR_ONLY feature_count != 63"
    )

if stage23_behavior.get(
    "mode"
) != "whitelist":
    raise RuntimeError(
        "Unexpected BEHAVIOR_ONLY mode."
    )

if list(
    stage23_behavior.get(
        "removed",
        [],
    )
) != EXPECTED_BEHAVIOR_REMOVED:
    raise RuntimeError(
        "Frozen BEHAVIOR_ONLY removal list changed."
    )

if (
    stage23_behavior.get(
        "semantic_label"
    )
    != "behavior_restricted_feature_set"
):
    raise RuntimeError(
        "Unexpected BEHAVIOR_ONLY semantic label."
    )


expected_behavior_features = [
    feature
    for feature in stage22_features
    if feature not in set(
        EXPECTED_BEHAVIOR_REMOVED
    )
]

if list(
    stage23_behavior.get(
        "features",
        [],
    )
) != expected_behavior_features:
    raise RuntimeError(
        "BEHAVIOR_ONLY ordered feature list mismatch."
    )


if (
    stage23_nsg["features"]
    == stage23_behavior["features"]
):
    raise RuntimeError(
        "NO_SUSPICIOUS_GROUP unexpectedly equals BEHAVIOR_ONLY."
    )


print(
    "FULL:",
    stage23_full[
        "feature_count"
    ],
)

print()

print(
    "NO_SUSPICIOUS_GROUP:",
    stage23_nsg[
        "feature_count"
    ],
)

print("  removed:")

for feature in (
    stage23_nsg[
        "removed"
    ]
):
    print(
        "   -",
        feature,
    )

print(
    "  semantic label:",
    stage23_nsg[
        "semantic_label"
    ],
)

print()

print(
    "BEHAVIOR_ONLY:",
    stage23_behavior[
        "feature_count"
    ],
)

print("  removed:")

for feature in (
    stage23_behavior[
        "removed"
    ]
):
    print(
        "   -",
        feature,
    )

print(
    "  semantic label:",
    stage23_behavior[
        "semantic_label"
    ],
)

print()

print(
    "NO_SUSPICIOUS_GROUP == BEHAVIOR_ONLY:",
    False,
)

print()

print(
    "[PASS] Stage23 frozen subset definitions resolved exactly."
)

print(
    "[CORRECTION] Stage24-0A NSG=UNRESOLVED was a lookup/parser miss."
)

print(
    "[PASS] No Stage23 scientific definition changed."
)
print()


# ============================================================================
# 5. STAGE24 BRIDGE INTENT
#
# bridge70:
#   complete Stage22 70-feature source representation
#
# bridge62:
#   Stage22 70 minus the eight historical aggregate flag-count fields
#   affected by the CICIDS2017 exporter flag serialization issue.
#
# NOTE:
#   bridge62 != NO_SUSPICIOUS_GROUP
#   bridge62 != BEHAVIOR_ONLY
#
# Target-side semantic mapping is NOT frozen in this cell.
# ============================================================================

FLAG_DEPENDENT_SOURCE_FEATURES = [
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "ACK Flag Cnt",
    "URG Flag Cnt",
    "CWE Flag Count",
    "ECE Flag Cnt",
]


missing_flag_features = [
    feature
    for feature in FLAG_DEPENDENT_SOURCE_FEATURES
    if feature not in stage22_features
]

if missing_flag_features:

    raise RuntimeError(
        "Frozen Stage22 flag features missing: "
        + repr(
            missing_flag_features
        )
    )


bridge70_source_features = list(
    stage22_features
)

flag_feature_set = set(
    FLAG_DEPENDENT_SOURCE_FEATURES
)

bridge62_source_features = [
    feature
    for feature in stage22_features
    if feature not in flag_feature_set
]


if len(
    bridge70_source_features
) != 70:
    raise RuntimeError(
        "bridge70 source dimension != 70"
    )

if len(
    bridge62_source_features
) != 62:
    raise RuntimeError(
        "bridge62 source dimension != 62"
    )


print("=" * 100)
print("STAGE24 BRIDGE INTENT")
print("=" * 100)

print(
    "bridge70 source dimension:",
    len(
        bridge70_source_features
    ),
)

print(
    "bridge62 source dimension:",
    len(
        bridge62_source_features
    ),
)

print()

print(
    "bridge62 excluded aggregate flag fields:"
)

for feature in (
    FLAG_DEPENDENT_SOURCE_FEATURES
):
    print(
        "  -",
        feature,
    )

print()

print(
    "bridge62 == NO_SUSPICIOUS_GROUP:",
    bridge62_source_features
    == stage23_nsg["features"],
)

print(
    "bridge62 == BEHAVIOR_ONLY:",
    bridge62_source_features
    == stage23_behavior["features"],
)

print(
    "NO_SUSPICIOUS_GROUP == BEHAVIOR_ONLY:",
    stage23_nsg["features"]
    == stage23_behavior["features"],
)

print()

print(
    "[PASS] Stage23 subsets and Stage24 bridges remain distinct."
)

print(
    "[IMPORTANT] Target-side semantic bridge is still NOT frozen."
)
print()


# ============================================================================
# 6. VERIFY FROZEN STAGE20 1C8 FLAG SERIALIZATION CORRECTION
# ============================================================================

print("=" * 100)
print("FROZEN STAGE20-1C8 FLAG SERIALIZATION CORRECTION")
print("=" * 100)

physical_to_semantic = dict(
    stage20_flag.get(
        "physical_csv_column_to_semantic_flag",
        {},
    )
)


EXPECTED_FLAG_MAP = {
    "ACK Flag Count": "ACK",
    "CWE Flag Count": "URG",
    "ECE Flag Count": "CWR",
    "FIN Flag Count": "RST",
    "PSH Flag Count": "SYN",
    "RST Flag Count": "ECE",
    "SYN Flag Count": "PSH",
    "URG Flag Count": "FIN",
}


if physical_to_semantic != EXPECTED_FLAG_MAP:

    raise RuntimeError(
        "\nFrozen Stage20-1C8 flag mapping differs from "
        "the expected source-derived mapping.\n"
        f"Actual: {physical_to_semantic}"
    )


if (
    stage20_flag.get(
        "scientific_interpretation",
        {},
    ).get(
        "classification"
    )
    != "SOURCE_DERIVED_EXPORT_SERIALIZATION_DEFECT"
):

    raise RuntimeError(
        "Unexpected Stage20-1C8 scientific classification."
    )


print(
    "Classification:",
    stage20_flag[
        "scientific_interpretation"
    ][
        "classification"
    ],
)

print()

for physical, semantic in (
    physical_to_semantic.items()
):
    print(
        f"  {physical:20s} -> {semantic}"
    )

print()

print(
    "[PASS] Exact frozen Stage20-1C8 mapping recovered."
)

print(
    "[IMPORTANT] Mapping is NOT applied to target rows in Stage24-0B."
)
print()


# ============================================================================
# 7. FROZEN TARGET OPENING LEDGER
#
# Opening definition:
# The target opening occurs when target feature values are first supplied
# to the frozen source-domain model in that specific experimental cell.
#
# SHA256 / file metadata / schema footer reads consume ZERO openings.
# ============================================================================

target_opening_ledger = {

    "PRIMARY_2018_TO_2017": {

        "bridge62": {

            "PUBLISHED": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },

            "FLAG_CORRECTED": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },

            "GROUNDED_S4": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },
        },

        "bridge70": {

            "PUBLISHED": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },

            "FLAG_CORRECTED": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },

            "GROUNDED_S4": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },
        },
    },

    "SECONDARY_2017_TO_2018": {

        "bridge62": {

            "FEB28_2018": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },
        },

        "bridge70": {

            "FEB28_2018": {
                "openings": 0,
                "budget": 1,
                "status": "CLOSED",
            },
        },
    },
}


ledger_cells = []

for direction, bridges in (
    target_opening_ledger.items()
):

    for bridge, targets in (
        bridges.items()
    ):

        for target, state in (
            targets.items()
        ):

            ledger_cells.append(
                {
                    "direction": direction,
                    "bridge": bridge,
                    "target": target,
                    **state,
                }
            )


if len(ledger_cells) != 8:

    raise RuntimeError(
        f"Expected 8 opening cells, got {len(ledger_cells)}"
    )


total_openings = sum(
    cell["openings"]
    for cell in ledger_cells
)

total_budget = sum(
    cell["budget"]
    for cell in ledger_cells
)


if total_openings != 0:

    raise RuntimeError(
        "Target opening ledger is not pristine."
    )

if total_budget != 8:

    raise RuntimeError(
        "Target opening budget != 8."
    )


print("=" * 100)
print("FROZEN STAGE24 TARGET-OPENING LEDGER")
print("=" * 100)

for index, cell in enumerate(
    ledger_cells,
    start=1,
):

    print(
        f"{index}. "
        f"{cell['direction']} / "
        f"{cell['bridge']} / "
        f"{cell['target']} = "
        f"{cell['openings']}/{cell['budget']} "
        f"{cell['status']}"
    )


print()
print(
    "TOTAL OPENINGS:",
    f"{total_openings}/{total_budget}",
)
print()


# ============================================================================
# 8. RECOVER AUTHORITATIVE STAGE20 TRAFFIC-LABEL SOURCE CONTRACT
#
# Prefer Stage20-1E compact-corpus manifests because they explicitly record
# the source traffic-label remote path, revision, hash, size, and rows.
#
# No dataset target row is read here.
# ============================================================================

print("=" * 100)
print("RECOVERING FROZEN STAGE20 CICIDS2017 SOURCE CONTRACT")
print("=" * 100)


manifest_paths = []

if STAGE20_1E_DIR.exists():

    manifest_paths = sorted(
        path
        for path in STAGE20_1E_DIR.glob(
            "*compact_corpus_manifest.json"
        )
        if path.is_file()
    )


# Fallback only if exact compact-corpus manifests are unavailable.
if not manifest_paths:

    for path in (
        REPO_DIR
        / "results"
    ).rglob("*.json"):

        rel = str(
            path.relative_to(
                REPO_DIR
            )
        ).lower()

        if (
            "stage20" in rel
            or "stage21" in rel
        ):
            manifest_paths.append(
                path
            )


print(
    "Candidate frozen source manifests:",
    len(manifest_paths),
)


raw_records = []

for path in manifest_paths:

    try:
        obj = load_json(
            path
        )

    except Exception:
        continue

    raw_records.extend(
        recursive_traffic_label_sources(
            obj,
            path,
        )
    )


if not raw_records:

    raise RuntimeError(
        "No frozen Stage20 traffic-label source records recovered."
    )


# --------------------------------------------------------------------------
# Merge duplicate references safely.
#
# For one remote file:
#   revision must agree
#   SHA256 must agree
#   all non-null size declarations must agree
#   all non-null row-count declarations must agree
# --------------------------------------------------------------------------

by_remote = defaultdict(
    list
)

for record in raw_records:

    by_remote[
        record["remote"]
    ].append(
        record
    )


source_contract = []

for remote, records in sorted(
    by_remote.items()
):

    revisions = {
        record["revision"]
        for record in records
    }

    hashes = {
        record["sha256"]
        for record in records
    }

    sizes = {
        record["size_bytes"]
        for record in records
        if record[
            "size_bytes"
        ] is not None
    }

    rows = {
        record["rows_declared"]
        for record in records
        if record[
            "rows_declared"
        ] is not None
    }


    if len(revisions) != 1:

        raise RuntimeError(
            f"Conflicting revisions for {remote}: {revisions}"
        )

    if len(hashes) != 1:

        raise RuntimeError(
            f"Conflicting SHA256 values for {remote}: {hashes}"
        )

    if len(sizes) > 1:

        raise RuntimeError(
            f"Conflicting sizes for {remote}: {sizes}"
        )

    if len(rows) > 1:

        raise RuntimeError(
            f"Conflicting row declarations for {remote}: {rows}"
        )


    source_contract.append(
        {
            "remote": remote,
            "revision": next(
                iter(
                    revisions
                )
            ),
            "sha256": next(
                iter(
                    hashes
                )
            ),
            "size_bytes": (
                next(
                    iter(
                        sizes
                    )
                )
                if sizes
                else None
            ),
            "rows_declared": (
                next(
                    iter(
                        rows
                    )
                )
                if rows
                else None
            ),
            "evidence_manifests": sorted(
                {
                    record[
                        "manifest"
                    ]
                    for record in records
                }
            ),
        }
    )


print(
    "Unique pinned traffic-label artifacts:",
    len(source_contract),
)

print()

for record in source_contract:

    print(
        "Remote:  ",
        record[
            "remote"
        ],
    )

    print(
        "Revision:",
        record[
            "revision"
        ],
    )

    print(
        "SHA256:  ",
        record[
            "sha256"
        ],
    )

    print(
        "Bytes:   ",
        record[
            "size_bytes"
        ],
    )

    print(
        "Rows*:   ",
        record[
            "rows_declared"
        ],
    )

    print()


print(
    "* Row counts printed above come only from frozen Stage20 metadata."
)

print(
    "* No target rows or labels have been parsed."
)
print()


# ============================================================================
# 9. RECOVER / VERIFY HUGGING FACE DATASET ID
# ============================================================================

all_stage20_json = []

for path in (
    REPO_DIR
    / "results"
).rglob("*.json"):

    rel = str(
        path.relative_to(
            REPO_DIR
        )
    ).lower()

    if (
        "stage20" in rel
        or "stage21" in rel
    ):
        all_stage20_json.append(
            path
        )


discovered_hf_ids = (
    discover_hf_dataset_ids(
        all_stage20_json
    )
)


print("=" * 100)
print("FROZEN HUGGING FACE SOURCE IDENTITY")
print("=" * 100)

print(
    "Discovered Stage20/21 HF dataset IDs:",
    discovered_hf_ids,
)


if (
    EXPECTED_HF_REPO
    not in discovered_hf_ids
):

    raise RuntimeError(
        "\nFrozen Stage20 receipts do not confirm expected source:\n"
        f"  {EXPECTED_HF_REPO}\n"
        "Do not substitute another CICIDS2017 dataset."
    )


HF_DATASET_REPO = (
    EXPECTED_HF_REPO
)


print()
print(
    "[PASS] Frozen CICIDS2017 dataset source:",
    HF_DATASET_REPO,
)
print()


# ============================================================================
# 10. IMPORT HUGGING FACE CLIENT
# ============================================================================

try:

    import huggingface_hub

    from huggingface_hub import (
        hf_hub_download,
    )

except Exception as exc:

    raise RuntimeError(
        "huggingface_hub is unavailable."
    ) from exc


print("=" * 100)
print("EXACT SOURCE ACQUISITION")
print("=" * 100)

print(
    "huggingface_hub:",
    huggingface_hub.__version__,
)

print(
    "Dataset repo:",
    HF_DATASET_REPO,
)

print()


# ============================================================================
# 11. DOWNLOAD EXACT PINNED TARGET CONTAINERS + SHA256
#
# This is an opaque source-integrity operation.
# Dataset rows remain unread.
# ============================================================================

acquisition_records = []


for index, source in enumerate(
    source_contract,
    start=1,
):

    print(
        f"[{index}/{len(source_contract)}] "
        f"{source['remote']}"
    )


    record = {
        **source,

        "repo_id":
            HF_DATASET_REPO,

        "status":
            "PENDING",

        "local_path":
            None,

        "materialization_mode":
            None,

        "actual_size_bytes":
            None,

        "actual_sha256":
            None,

        "size_match":
            None,

        "sha256_match":
            None,

        "schema_status":
            "NOT_INSPECTED",

        "schema_columns":
            None,

        "schema_column_count":
            None,

        "parquet_footer_num_rows":
            None,

        "parquet_footer_row_groups":
            None,

        "error":
            None,
    }


    try:

        cached_file = Path(
            hf_hub_download(
                repo_id=(
                    HF_DATASET_REPO
                ),
                repo_type="dataset",
                filename=(
                    source[
                        "remote"
                    ]
                ),
                revision=(
                    source[
                        "revision"
                    ]
                ),
                cache_dir=str(
                    HF_CACHE_DIR
                ),
            )
        )


        if not cached_file.exists():

            raise RuntimeError(
                "hf_hub_download returned a nonexistent file."
            )


        destination = (
            CICIDS_SOURCE_DIR
            / Path(
                source[
                    "remote"
                ]
            ).name
        )


        materialization_mode = (
            hardlink_or_copy(
                cached_file,
                destination,
            )
        )


        actual_size = (
            destination.stat().st_size
        )

        actual_sha = (
            sha256_file(
                destination
            )
        )


        expected_size = (
            source[
                "size_bytes"
            ]
        )


        size_match = (
            True
            if expected_size is None
            else actual_size
            == expected_size
        )


        sha_match = (
            actual_sha.lower()
            == source[
                "sha256"
            ].lower()
        )


        record.update(
            {
                "status":
                    "DOWNLOADED_AND_HASHED",

                "local_path":
                    str(
                        destination
                    ),

                "materialization_mode":
                    materialization_mode,

                "actual_size_bytes":
                    actual_size,

                "actual_sha256":
                    actual_sha,

                "size_match":
                    size_match,

                "sha256_match":
                    sha_match,
            }
        )


        print(
            "  expected bytes:",
            expected_size,
        )

        print(
            "  actual bytes:  ",
            actual_size,
        )

        print(
            "  expected sha:  ",
            source[
                "sha256"
            ],
        )

        print(
            "  actual sha:    ",
            actual_sha,
        )


        if not size_match:

            raise RuntimeError(
                "BYTE SIZE MISMATCH"
            )


        if not sha_match:

            raise RuntimeError(
                "SHA256 MISMATCH"
            )


        print(
            "  [PASS] Exact frozen source identity"
        )


    except Exception as exc:

        record[
            "status"
        ] = "FAILED"

        record[
            "error"
        ] = repr(
            exc
        )

        print(
            "  [FAIL]",
            repr(
                exc
            ),
        )


    acquisition_records.append(
        record
    )

    print()


# ============================================================================
# 12. EXACT SOURCE IDENTITY GATE
# ============================================================================

failed_records = [
    record
    for record in acquisition_records
    if (
        record[
            "status"
        ]
        != "DOWNLOADED_AND_HASHED"
        or record[
            "size_match"
        ] is not True
        or record[
            "sha256_match"
        ] is not True
    )
]


all_exact_sources_verified = (
    len(
        failed_records
    ) == 0
)


print("=" * 100)
print("EXACT SOURCE IDENTITY GATE")
print("=" * 100)

print(
    "Pinned artifacts:",
    len(
        acquisition_records
    ),
)

print(
    "Verified:",
    len(
        acquisition_records
    )
    - len(
        failed_records
    ),
)

print(
    "Failed:",
    len(
        failed_records
    ),
)

print()


if all_exact_sources_verified:

    print(
        "[PASS] Every recovered target container is byte-identical "
        "to its frozen Stage20 source."
    )

else:

    print(
        "[BLOCKED] One or more exact source identities failed."
    )

    print(
        "No schema bridge may be frozen."
    )

print()


# ============================================================================
# 13. PARQUET FOOTER / SCHEMA METADATA ONLY
#
# IMPORTANT:
# pq.ParquetFile() reads metadata/footer.
#
# We NEVER call:
#   read()
#   read_row_group()
#   pandas.read_parquet()
#   to_table()
#
# Therefore target feature rows and labels remain unmaterialized.
# ============================================================================

schema_groups = defaultdict(
    list
)


if all_exact_sources_verified:

    import pyarrow.parquet as pq


    print("=" * 100)
    print("PARQUET FOOTER / SCHEMA INSPECTION — ZERO TARGET ROWS")
    print("=" * 100)


    for record in (
        acquisition_records
    ):

        local_path = Path(
            record[
                "local_path"
            ]
        )


        try:

            parquet_file = (
                pq.ParquetFile(
                    local_path
                )
            )


            columns = list(
                parquet_file
                .schema_arrow
                .names
            )


            num_rows_metadata = (
                int(
                    parquet_file
                    .metadata
                    .num_rows
                )
                if (
                    parquet_file
                    .metadata
                    is not None
                )
                else None
            )


            num_row_groups = int(
                parquet_file
                .num_row_groups
            )


            record.update(
                {
                    "schema_status":
                        "PARQUET_FOOTER_ONLY",

                    "schema_columns":
                        columns,

                    "schema_column_count":
                        len(
                            columns
                        ),

                    "parquet_footer_num_rows":
                        num_rows_metadata,

                    "parquet_footer_row_groups":
                        num_row_groups,
                }
            )


            schema_hash = (
                sha256_json_object(
                    columns
                )
            )


            schema_groups[
                schema_hash
            ].append(
                local_path.name
            )


            print()

            print(
                "File:",
                local_path.name,
            )

            print(
                "  columns:",
                len(
                    columns
                ),
            )

            print(
                "  rows from footer metadata:",
                num_rows_metadata,
            )

            print(
                "  row groups:",
                num_row_groups,
            )

            print(
                "  ordered schema SHA256:",
                schema_hash,
            )

            print(
                "  columns:"
            )


            for column_index, column in enumerate(
                columns,
                start=1,
            ):

                print(
                    f"    {column_index:02d}. {column}"
                )


        except Exception as exc:

            record[
                "schema_status"
            ] = "FAILED"

            record[
                "error"
            ] = repr(
                exc
            )

            print()

            print(
                "[FAIL] Footer/schema inspection:",
                local_path.name,
                repr(
                    exc
                ),
            )


schema_all_pass = (
    all_exact_sources_verified
    and all(
        record[
            "schema_status"
        ]
        == "PARQUET_FOOTER_ONLY"
        for record
        in acquisition_records
    )
)


# ============================================================================
# 14. PHYSICAL SCHEMA GROUP SYNTHESIS
# ============================================================================

print()
print("=" * 100)
print("CICIDS2017 PHYSICAL SCHEMA GROUPS")
print("=" * 100)


if not schema_groups:

    print(
        "No verified physical schemas available."
    )

else:

    print(
        "Distinct ordered schemas:",
        len(
            schema_groups
        ),
    )


    for index, (
        schema_hash,
        files,
    ) in enumerate(
        sorted(
            schema_groups.items()
        ),
        start=1,
    ):

        print()

        print(
            f"Schema group {index}"
        )

        print(
            "  SHA256:",
            schema_hash,
        )


        for filename in (
            sorted(
                files
            )
        ):

            print(
                "  -",
                filename,
            )


# ============================================================================
# 15. BRIDGE / VARIANT STATE AFTER 0B
# ============================================================================

if schema_all_pass:

    bridge62_status = (
        "SOURCE_SCHEMA_RECOVERED__"
        "TARGET_SEMANTIC_MAPPING_NOT_YET_FROZEN"
    )

    bridge70_status = (
        "SOURCE_SCHEMA_RECOVERED__"
        "TARGET_SEMANTIC_MAPPING_NOT_YET_FROZEN"
    )

else:

    bridge62_status = (
        "BLOCKED_SOURCE_RECOVERY_INCOMPLETE"
    )

    bridge70_status = (
        "BLOCKED_SOURCE_RECOVERY_INCOMPLETE"
    )


print()
print("=" * 100)
print("POST-0B SCIENTIFIC STATE")
print("=" * 100)

print(
    "Stage23 FULL:",
    len(
        stage23_full[
            "features"
        ]
    ),
)

print(
    "Stage23 NO_SUSPICIOUS_GROUP:",
    len(
        stage23_nsg[
            "features"
        ]
    ),
)

print(
    "Stage23 BEHAVIOR_ONLY:",
    len(
        stage23_behavior[
            "features"
        ]
    ),
)

print()

print(
    "Stage24 bridge62 source dimension:",
    len(
        bridge62_source_features
    ),
)

print(
    "Stage24 bridge70 source dimension:",
    len(
        bridge70_source_features
    ),
)

print()

print(
    "bridge62:",
    bridge62_status,
)

print(
    "bridge70:",
    bridge70_status,
)

print()

print(
    "PUBLISHED target rows constructed:",
    False,
)

print(
    "FLAG_CORRECTED rows constructed:",
    False,
)

print(
    "GROUNDED_S4 rows constructed:",
    False,
)

print()

print(
    "Target model openings:",
    "0 / 8",
)

print(
    "Model fits:",
    0,
)

print(
    "Model predictions:",
    0,
)

print(
    "Target metrics:",
    0,
)
print()


# ============================================================================
# 16. WRITE MACHINE-READABLE STAGE24-0B RECEIPT
# ============================================================================

receipt = {

    "stage":
        "Stage24-0B",

    "type":
        "EXACT_CICIDS2017_SOURCE_RECOVERY_AND_FROZEN_SUBSET_RESOLUTION",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "repository": {

        "branch":
            branch,

        "head":
            head,

        "expected_head":
            EXPECTED_HEAD,

        "scientific_parent_unchanged":
            True,
    },

    "stage24_0a_correction": {

        "previous_stage24_0a_state": {
            "NO_SUSPICIOUS_GROUP":
                "UNRESOLVED",
        },

        "corrected_state": {
            "NO_SUSPICIOUS_GROUP":
                "RESOLVED_FROM_FROZEN_STAGE23_FEATURE_SUBSET_SPEC",
        },

        "reason": (
            "Stage24-0A generic subset lookup failed to resolve "
            "an existing frozen Stage23 subset. Direct structured "
            "inspection of feature_subset_spec.json resolves it."
        ),

        "scientific_definition_changed":
            False,

        "target_rows_accessed":
            False,

        "target_openings_consumed":
            0,
    },

    "stage23_frozen_subsets": {

        "FULL": {

            "feature_count":
                70,

            "removed":
                [],

            "semantic_label":
                stage23_full[
                    "semantic_label"
                ],
        },

        "NO_SUSPICIOUS_GROUP": {

            "feature_count":
                67,

            "removed":
                list(
                    stage23_nsg[
                        "removed"
                    ]
                ),

            "semantic_label":
                stage23_nsg[
                    "semantic_label"
                ],

            "features":
                list(
                    stage23_nsg[
                        "features"
                    ]
                ),
        },

        "BEHAVIOR_ONLY": {

            "feature_count":
                63,

            "removed":
                list(
                    stage23_behavior[
                        "removed"
                    ]
                ),

            "semantic_label":
                stage23_behavior[
                    "semantic_label"
                ],

            "features":
                list(
                    stage23_behavior[
                        "features"
                    ]
                ),
        },
    },

    "stage24_bridge_intent": {

        "bridge62": {

            "source_feature_count":
                62,

            "source_features":
                bridge62_source_features,

            "excluded_flag_dependent_fields":
                FLAG_DEPENDENT_SOURCE_FEATURES,

            "target_semantic_mapping_frozen":
                False,

            "status":
                bridge62_status,
        },

        "bridge70": {

            "source_feature_count":
                70,

            "source_features":
                bridge70_source_features,

            "target_semantic_mapping_frozen":
                False,

            "status":
                bridge70_status,
        },

        "representation_distinctness": {

            "bridge62_equals_NO_SUSPICIOUS_GROUP":
                (
                    bridge62_source_features
                    == stage23_nsg[
                        "features"
                    ]
                ),

            "bridge62_equals_BEHAVIOR_ONLY":
                (
                    bridge62_source_features
                    == stage23_behavior[
                        "features"
                    ]
                ),

            "NO_SUSPICIOUS_GROUP_equals_BEHAVIOR_ONLY":
                (
                    stage23_nsg[
                        "features"
                    ]
                    == stage23_behavior[
                        "features"
                    ]
                ),
        },
    },

    "stage20_flag_serialization_correction": {

        "artifact":
            str(
                STAGE20_FLAG_CORRECTION.relative_to(
                    REPO_DIR
                )
            ),

        "classification":
            stage20_flag[
                "scientific_interpretation"
            ][
                "classification"
            ],

        "physical_csv_column_to_semantic_flag":
            physical_to_semantic,

        "source_mechanism":
            stage20_flag.get(
                "source_mechanism"
            ),

        "applied_to_target_rows":
            False,
    },

    "target_representations": {

        "PUBLISHED": {
            "constructed":
                False,
        },

        "FLAG_CORRECTED": {
            "constructed":
                False,

            "frozen_stage20_rule_available":
                True,
        },

        "GROUNDED_S4": {
            "constructed":
                False,
        },
    },

    "target_opening_definition": (
        "A target opening occurs when target feature values "
        "are first supplied to the frozen source-domain model "
        "for the corresponding direction/bridge/target cell."
    ),

    "target_opening_ledger":
        target_opening_ledger,

    "opening_summary": {

        "cells":
            len(
                ledger_cells
            ),

        "consumed":
            total_openings,

        "budget":
            total_budget,

        "all_closed":
            all(
                cell[
                    "status"
                ]
                == "CLOSED"
                for cell
                in ledger_cells
            ),
    },

    "cicids2017_source": {

        "provider":
            "Hugging Face",

        "repo_id":
            HF_DATASET_REPO,

        "repo_identity_confirmed_from_frozen_stage20_or_stage21_records":
            True,

        "frozen_source_contract":
            source_contract,

        "acquisition_records":
            acquisition_records,

        "all_exact_sources_verified":
            all_exact_sources_verified,

        "all_schema_footer_inspections_pass":
            schema_all_pass,

        "physical_schema_groups": {
            schema_hash:
                sorted(
                    files
                )
            for schema_hash, files
            in schema_groups.items()
        },
    },

    "scientific_safety": {

        "target_file_bytes_hashed":
            True,

        "whole_file_sha256_treated_as_opaque_integrity_operation":
            True,

        "parquet_footer_metadata_read":
            bool(
                schema_groups
            ),

        "target_row_groups_read":
            0,

        "target_feature_rows_materialized":
            0,

        "target_feature_values_materialized":
            0,

        "target_labels_materialized":
            0,

        "target_statistics_computed":
            0,

        "models_loaded_for_target_inference":
            0,

        "model_fits":
            0,

        "model_predictions":
            0,

        "threshold_searches":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,
    },

    "next_authorized_step": (

        "Stage24-0C explicit ordered semantic feature bridge, "
        "target variant construction rules, source sanity sequence, "
        "bridge-specific source-refit eligibility, GPU-first backend "
        "policy, source-only threshold procedure, population rules, "
        "family taxonomy, nulls, bootstrap rules, and final pre-opening "
        "protocol lock."

        if schema_all_pass

        else

        "Resolve exact CICIDS2017 source acquisition failure. "
        "No bridge mapping, model fit, or target inference authorized."
    ),
}


receipt_path = (
    LOCK_DIR
    / "stage24_0b_exact_cicids2017_source_recovery.json"
)


with receipt_path.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        receipt,
        f,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

    f.write("\n")


receipt_sha = (
    sha256_file(
        receipt_path
    )
)


receipt_sha_path = (
    LOCK_DIR
    / "stage24_0b_exact_cicids2017_source_recovery.sha256"
)


receipt_sha_path.write_text(
    (
        f"{receipt_sha}  "
        f"{receipt_path.name}\n"
    ),
    encoding="utf-8",
)


# ============================================================================
# 17. FINAL OUTPUT
# ============================================================================

print()
print("=" * 100)
print("STAGE24-0B FINAL SYNTHESIS")
print("=" * 100)

print(
    "Repository HEAD:"
)

print(
    " ",
    head,
)

print()

print(
    "Stage24-0A NSG lookup miss corrected:",
    True,
)

print()

print(
    "Stage23 FULL:"
)

print(
    " ",
    len(
        stage23_full[
            "features"
        ]
    ),
)

print(
    "Stage23 NO_SUSPICIOUS_GROUP:"
)

print(
    " ",
    len(
        stage23_nsg[
            "features"
        ]
    ),
)

print(
    "Stage23 BEHAVIOR_ONLY:"
)

print(
    " ",
    len(
        stage23_behavior[
            "features"
        ]
    ),
)

print()

print(
    "Stage24 bridge62:"
)

print(
    " ",
    len(
        bridge62_source_features
    ),
)

print(
    "Stage24 bridge70:"
)

print(
    " ",
    len(
        bridge70_source_features
    ),
)

print()

print(
    "Pinned CICIDS2017 repo:"
)

print(
    " ",
    HF_DATASET_REPO,
)

print(
    "Pinned traffic-label artifacts recovered:"
)

print(
    " ",
    len(
        source_contract
    ),
)

print(
    "Exact byte identities all pass:"
)

print(
    " ",
    all_exact_sources_verified,
)

print(
    "Parquet footer/schema all pass:"
)

print(
    " ",
    schema_all_pass,
)

print()

print(
    "bridge62 status:"
)

print(
    " ",
    bridge62_status,
)

print(
    "bridge70 status:"
)

print(
    " ",
    bridge70_status,
)

print()

print(
    "Target feature rows materialized:"
)

print(
    "  0"
)

print(
    "Target labels materialized:"
)

print(
    "  0"
)

print(
    "Model fits:"
)

print(
    "  0"
)

print(
    "Model predictions:"
)

print(
    "  0"
)

print(
    "Metrics:"
)

print(
    "  0"
)

print(
    "Target openings:"
)

print(
    "  0 / 8"
)

print()

print(
    "Receipt:"
)

print(
    " ",
    receipt_path,
)

print(
    "Receipt SHA256:"
)

print(
    " ",
    receipt_sha,
)

print()

print("-" * 100)

if schema_all_pass:

    print(
        "STAGE24-0B: PASS"
    )

    print(
        "Exact frozen CICIDS2017 source containers recovered."
    )

    print(
        "Stage23 subset definitions resolved."
    )

    print(
        "Target rows remain unopened."
    )

    print(
        "Target model openings remain 0 / 8."
    )

    print()

    print(
        "NEXT AUTHORIZED STEP:"
    )

    print(
        "Stage24-0C — explicit semantic bridge + final pre-opening protocol lock."
    )

else:

    print(
        "STAGE24-0B: BLOCKED"
    )

    print(
        "Exact CICIDS2017 source recovery or schema inspection is incomplete."
    )

    print(
        "DO NOT substitute another CICIDS2017 copy."
    )

    print(
        "DO NOT freeze semantic bridge mappings."
    )

    print(
        "DO NOT run model inference."
    )

print("-" * 100)
print("STAGE24-0B COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================================
# STAGE24-0B2 — COMPLETE CICIDS2017 SOURCE CONTRACT RECOVERY
#
# PURPOSE
# -------
# Stage24-0B correctly recovered Monday/Tuesday/Wednesday, but its metadata
# parser handled only:
#
#     "traffic_labels": { ... }
#
# while frozen Stage20 Thursday and Friday manifests use:
#
#     "traffic_labels": [ {...}, {...}, ... ]
#
# This cell completes the frozen source contract:
#
#   Monday     1 parquet
#   Tuesday    1 parquet
#   Wednesday  1 parquet
#   Thursday   2 parquets
#   Friday     3 parquets
#   ---------------------
#   TOTAL      8 parquets
#
# SCIENTIFIC SAFETY
# -----------------
# ALLOWED:
#   - read repository manifests
#   - exact pinned Hugging Face download
#   - opaque whole-file SHA256
#   - file byte size
#   - Parquet footer/schema metadata
#
# FORBIDDEN:
#   - read target row groups
#   - materialize target feature values
#   - materialize target labels
#   - compute target prevalence
#   - construct PUBLISHED rows
#   - construct FLAG_CORRECTED rows
#   - construct GROUNDED_S4 rows
#   - model fitting
#   - model loading for target inference
#   - prediction
#   - metrics
#
# TARGET OPENINGS BEFORE: 0 / 8
# TARGET OPENINGS AFTER:  0 / 8
# ============================================================================

from __future__ import annotations

import os
import json
import shutil
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict

# ============================================================================
# 0. CONSTANTS
# ============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "963b2da0043c7438624916280de154a5628f8879"
)

STAGE24_ROOT = (
    REPO_DIR
    / "results"
    / "stage24_cross_dataset"
)

LOCK_DIR = (
    STAGE24_ROOT
    / "stage24_0_protocol_lock"
)

SOURCE_DIR = Path(
    "/kaggle/working/stage24_cicids2017_sources"
)

HF_CACHE_DIR = Path(
    "/kaggle/working/stage24_hf_cache"
)

STAGE20_1E_DIR = (
    REPO_DIR
    / "results"
    / "stage20_1e_training"
)

PREVIOUS_0B_RECEIPT = (
    LOCK_DIR
    / "stage24_0b_exact_cicids2017_source_recovery.json"
)

PREVIOUS_0B_SHA_FILE = (
    LOCK_DIR
    / "stage24_0b_exact_cicids2017_source_recovery.sha256"
)

HF_REPO = "bvsam/cic-ids-2017"

EXPECTED_DAY_FILE_COUNTS = {
    "Monday": 1,
    "Tuesday": 1,
    "Wednesday": 1,
    "Thursday": 2,
    "Friday": 3,
}

EXPECTED_TOTAL_FILES = 8

MANIFESTS = {
    "Monday": (
        STAGE20_1E_DIR
        / "stage20_1e1m_monday_compact_corpus_manifest.json"
    ),

    "Tuesday": (
        STAGE20_1E_DIR
        / "stage20_1e1t_tuesday_compact_corpus_manifest.json"
    ),

    "Wednesday": (
        STAGE20_1E_DIR
        / "stage20_1e1w_wednesday_compact_corpus_manifest.json"
    ),

    "Thursday": (
        STAGE20_1E_DIR
        / "stage20_1e1v_thursday_validation_compact_corpus_manifest.json"
    ),

    "Friday": (
        STAGE20_1E_DIR
        / "stage20_1e4_friday_holdout_compact_corpus_manifest.json"
    ),
}

LOCK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SOURCE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HF_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("=" * 104)
print("STAGE24-0B2 — COMPLETE CICIDS2017 SOURCE CONTRACT RECOVERY")
print("=" * 104)

print("Target inference:          FORBIDDEN")
print("Target row parsing:        FORBIDDEN")
print("Target label parsing:      FORBIDDEN")
print("Target model openings:     0 / 8")
print("Expected CICIDS2017 files: 8")
print()


# ============================================================================
# 1. HELPERS
# ============================================================================

def run_git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=True,
    ).stdout.strip()


def load_json(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
):
    """
    Opaque integrity operation only.
    """
    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def canonical_json_sha256(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        payload
    ).hexdigest()


def materialize_file(
    source: Path,
    destination: Path,
):
    """
    Hard-link if possible; otherwise copy.
    """
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    source = source.resolve()

    # If destination is already the exact same inode/path target,
    # preserve it rather than unlinking our source.
    if destination.exists():

        try:

            if os.path.samefile(
                source,
                destination,
            ):
                return "ALREADY_SAME_FILE"

        except Exception:
            pass

        destination.unlink()


    try:

        os.link(
            source,
            destination,
        )

        return "HARDLINK"

    except Exception:

        shutil.copy2(
            source,
            destination,
        )

        return "COPY"


def normalize_label_record(
    day: str,
    raw: dict,
    manifest: Path,
):
    if not isinstance(
        raw,
        dict,
    ):
        raise RuntimeError(
            f"Invalid traffic-label entry in {manifest}"
        )

    remote = raw.get(
        "remote"
    )

    revision = raw.get(
        "revision"
    )

    sha256 = raw.get(
        "sha256"
    )

    size_bytes = raw.get(
        "size_bytes"
    )

    physical_rows = raw.get(
        "rows"
    )

    if physical_rows is None:
        physical_rows = raw.get(
            "physical_rows"
        )


    if not remote:
        raise RuntimeError(
            f"Missing remote in {manifest}"
        )

    if not revision:
        raise RuntimeError(
            f"Missing revision for {remote}"
        )

    if not sha256:
        raise RuntimeError(
            f"Missing SHA256 for {remote}"
        )

    if size_bytes is None:
        raise RuntimeError(
            f"Missing size_bytes for {remote}"
        )

    if physical_rows is None:
        raise RuntimeError(
            f"Missing physical row count for {remote}"
        )


    if not str(
        remote
    ).startswith(
        "traffic_labels/"
    ):
        raise RuntimeError(
            "Unexpected target source path: "
            + str(
                remote
            )
        )


    return {
        "day":
            day,

        "remote":
            str(
                remote
            ),

        "revision":
            str(
                revision
            ),

        "sha256":
            str(
                sha256
            ).lower(),

        "size_bytes":
            int(
                size_bytes
            ),

        "physical_rows":
            int(
                physical_rows
            ),

        "effective_records":
            (
                int(
                    raw[
                        "effective_records"
                    ]
                )
                if raw.get(
                    "effective_records"
                ) is not None
                else None
            ),

        "effective_label_records":
            (
                int(
                    raw[
                        "effective_label_records"
                    ]
                )
                if raw.get(
                    "effective_label_records"
                ) is not None
                else None
            ),

        "structural_null_padding_rows":
            (
                int(
                    raw[
                        "structural_null_padding_rows"
                    ]
                )
                if raw.get(
                    "structural_null_padding_rows"
                ) is not None
                else None
            ),

        "row_semantics":
            raw.get(
                "row_semantics"
            ),

        "manifest":
            str(
                manifest.relative_to(
                    REPO_DIR
                )
            ),
    }


def extract_manifest_label_records(
    day: str,
    manifest: Path,
):
    obj = load_json(
        manifest
    )

    source = obj.get(
        "source"
    )

    # Some Stage20 manifests may nest provenance differently.
    # Search recursively for a key literally named traffic_labels.
    found_values = []

    def walk(value):

        if isinstance(
            value,
            dict,
        ):

            for key, child in (
                value.items()
            ):

                if key == "traffic_labels":

                    found_values.append(
                        child
                    )

                walk(
                    child
                )

        elif isinstance(
            value,
            list,
        ):

            for child in value:
                walk(
                    child
                )

    walk(
        obj
    )


    if not found_values:

        raise RuntimeError(
            f"No traffic_labels metadata found in {manifest.name}"
        )


    # A manifest should expose one logical traffic_labels field.
    # Duplicate appearances containing the same records are deduplicated
    # below by remote identity.
    normalized = []


    for value in found_values:

        if isinstance(
            value,
            dict,
        ):

            normalized.append(
                normalize_label_record(
                    day,
                    value,
                    manifest,
                )
            )

        elif isinstance(
            value,
            list,
        ):

            for entry in value:

                normalized.append(
                    normalize_label_record(
                        day,
                        entry,
                        manifest,
                    )
                )

        else:

            raise RuntimeError(
                f"Unsupported traffic_labels type: {type(value)}"
            )


    by_remote = {}

    for record in normalized:

        remote = record[
            "remote"
        ]

        if remote in by_remote:

            prior = (
                by_remote[
                    remote
                ]
            )

            # Exact duplicate references are okay.
            comparable_fields = [
                "day",
                "revision",
                "sha256",
                "size_bytes",
                "physical_rows",
            ]

            for field in (
                comparable_fields
            ):

                if (
                    prior[
                        field
                    ]
                    != record[
                        field
                    ]
                ):

                    raise RuntimeError(
                        "Conflicting duplicate frozen metadata for "
                        f"{remote} field={field}"
                    )

        else:

            by_remote[
                remote
            ] = record


    return list(
        by_remote.values()
    )


# ============================================================================
# 2. REPOSITORY SAFETY
# ============================================================================

print("=" * 104)
print("REPOSITORY SAFETY")
print("=" * 104)

branch = run_git(
    "branch",
    "--show-current",
)

head = run_git(
    "rev-parse",
    "HEAD",
)

status = run_git(
    "status",
    "--porcelain",
)

print(
    "Branch:",
    branch,
)

print(
    "HEAD:  ",
    head,
)


if branch != "main":

    raise RuntimeError(
        f"Expected branch main, found {branch}"
    )


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "\nFrozen scientific parent changed.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


unexpected_dirty = []

if status:

    for line in (
        status.splitlines()
    ):

        rel = (
            line[
                3:
            ].strip()
        )

        if " -> " in rel:

            rel = rel.split(
                " -> ",
                1,
            )[1]


        if not rel.startswith(
            "results/stage24_cross_dataset/"
        ):

            unexpected_dirty.append(
                line
            )


print(
    "Dirty only from Stage24 receipts:",
    not bool(
        unexpected_dirty
    ),
)


if unexpected_dirty:

    raise RuntimeError(
        "\nUnexpected repository modifications:\n"
        + "\n".join(
            unexpected_dirty
        )
    )


print(
    "[PASS] Scientific parent commit unchanged."
)
print()


# ============================================================================
# 3. VERIFY PREVIOUS STAGE24-0B RECEIPT
# ============================================================================

print("=" * 104)
print("VERIFYING PREVIOUS STAGE24-0B RECEIPT")
print("=" * 104)


if not PREVIOUS_0B_RECEIPT.exists():

    raise FileNotFoundError(
        PREVIOUS_0B_RECEIPT
    )


if not PREVIOUS_0B_SHA_FILE.exists():

    raise FileNotFoundError(
        PREVIOUS_0B_SHA_FILE
    )


actual_0b_sha = sha256_file(
    PREVIOUS_0B_RECEIPT
)


sha_line = (
    PREVIOUS_0B_SHA_FILE
    .read_text(
        encoding="utf-8"
    )
    .strip()
)

declared_0b_sha = (
    sha_line.split()[0]
)


print(
    "Declared 0B SHA256:",
    declared_0b_sha,
)

print(
    "Actual 0B SHA256:  ",
    actual_0b_sha,
)


if (
    declared_0b_sha
    != actual_0b_sha
):

    raise RuntimeError(
        "Stage24-0B receipt SHA256 verification failed."
    )


receipt_0b = load_json(
    PREVIOUS_0B_RECEIPT
)


safety_0b = receipt_0b.get(
    "scientific_safety",
    {}
)


if (
    safety_0b.get(
        "target_row_groups_read"
    )
    != 0
):

    raise RuntimeError(
        "Prior 0B unexpectedly read target row groups."
    )


if (
    safety_0b.get(
        "target_feature_rows_materialized"
    )
    != 0
):

    raise RuntimeError(
        "Prior 0B unexpectedly materialized target rows."
    )


if (
    safety_0b.get(
        "model_predictions"
    )
    != 0
):

    raise RuntimeError(
        "Prior 0B unexpectedly made predictions."
    )


if (
    safety_0b.get(
        "target_openings_consumed"
    )
    != 0
):

    raise RuntimeError(
        "Prior 0B unexpectedly consumed an opening."
    )


print(
    "[PASS] Stage24-0B receipt integrity verified."
)

print(
    "[PASS] Prior target openings = 0 / 8."
)
print()


# ============================================================================
# 4. VERIFY ALL FIVE FROZEN DAY MANIFESTS
# ============================================================================

print("=" * 104)
print("FROZEN STAGE20 DAY MANIFESTS")
print("=" * 104)


for day, manifest in (
    MANIFESTS.items()
):

    print(
        f"{day:10s}: {manifest.name}"
    )

    if not manifest.exists():

        raise FileNotFoundError(
            manifest
        )


print()

print(
    "[PASS] Monday-Friday frozen Stage20 manifests present."
)
print()


# ============================================================================
# 5. EXTRACT COMPLETE TRAFFIC-LABEL CONTRACT
# ============================================================================

print("=" * 104)
print("COMPLETE CICIDS2017 TRAFFIC-LABEL SOURCE CONTRACT")
print("=" * 104)


all_records = []


for day, manifest in (
    MANIFESTS.items()
):

    records = (
        extract_manifest_label_records(
            day,
            manifest,
        )
    )


    print(
        f"{day:10s}: "
        f"{len(records)} frozen traffic-label artifact(s)"
    )


    expected_count = (
        EXPECTED_DAY_FILE_COUNTS[
            day
        ]
    )


    if (
        len(
            records
        )
        != expected_count
    ):

        raise RuntimeError(
            f"{day}: expected {expected_count} target files, "
            f"recovered {len(records)}"
        )


    all_records.extend(
        records
    )


print()


# --------------------------------------------------------------------------
# Global deduplication / conflict gate
# --------------------------------------------------------------------------

contract_by_remote = {}


for record in all_records:

    remote = record[
        "remote"
    ]


    if remote in (
        contract_by_remote
    ):

        prior = (
            contract_by_remote[
                remote
            ]
        )

        for key in [
            "day",
            "revision",
            "sha256",
            "size_bytes",
            "physical_rows",
        ]:

            if (
                prior[
                    key
                ]
                != record[
                    key
                ]
            ):

                raise RuntimeError(
                    f"Conflicting frozen source record: {remote} / {key}"
                )

    else:

        contract_by_remote[
            remote
        ] = record


source_contract = sorted(
    contract_by_remote.values(),
    key=lambda x: (
        list(
            EXPECTED_DAY_FILE_COUNTS.keys()
        ).index(
            x[
                "day"
            ]
        ),
        x[
            "remote"
        ],
    ),
)


if (
    len(
        source_contract
    )
    != EXPECTED_TOTAL_FILES
):

    raise RuntimeError(
        "\nIncomplete CICIDS2017 source contract.\n"
        f"Expected: {EXPECTED_TOTAL_FILES}\n"
        f"Actual:   {len(source_contract)}"
    )


print(
    "TOTAL FROZEN TARGET ARTIFACTS:",
    len(
        source_contract
    ),
)

print()


for index, record in enumerate(
    source_contract,
    start=1,
):

    print(
        f"[{index}/8] {record['day']}"
    )

    print(
        "  remote:       ",
        record[
            "remote"
        ],
    )

    print(
        "  revision:     ",
        record[
            "revision"
        ],
    )

    print(
        "  bytes:        ",
        record[
            "size_bytes"
        ],
    )

    print(
        "  physical rows:",
        record[
            "physical_rows"
        ],
    )

    print(
        "  sha256:       ",
        record[
            "sha256"
        ],
    )

    if (
        record[
            "structural_null_padding_rows"
        ]
        is not None
    ):

        print(
            "  structural null padding rows:",
            record[
                "structural_null_padding_rows"
            ],
        )

    print()


print(
    "[PASS] Complete Monday-Friday frozen source contract = 8 files."
)
print()


# ============================================================================
# 6. IMPORTANT THURSDAY STRUCTURAL METADATA
#
# This is copied only from frozen repository metadata.
# No target rows are inspected.
# ============================================================================

print("=" * 104)
print("THURSDAY SOURCE-STRUCTURE RECEIPT")
print("=" * 104)


thursday_records = [
    record
    for record in source_contract
    if record[
        "day"
    ] == "Thursday"
]


for record in (
    thursday_records
):

    print(
        Path(
            record[
                "remote"
            ]
        ).name
    )

    print(
        "  physical rows:",
        record[
            "physical_rows"
        ],
    )

    print(
        "  effective_label_records:",
        record[
            "effective_label_records"
        ],
    )

    print(
        "  structural_null_padding_rows:",
        record[
            "structural_null_padding_rows"
        ],
    )

    print(
        "  row_semantics:",
        record[
            "row_semantics"
        ],
    )

    print()


print(
    "[IMPORTANT] These are frozen Stage20 metadata declarations only."
)

print(
    "[IMPORTANT] Stage24-0B2 does NOT decide population inclusion/exclusion."
)
print()


# ============================================================================
# 7. DOWNLOAD / MATERIALIZE EXACT PINNED FILES
# ============================================================================

try:

    import huggingface_hub

    from huggingface_hub import (
        hf_hub_download,
    )

except Exception as exc:

    raise RuntimeError(
        "huggingface_hub unavailable."
    ) from exc


print("=" * 104)
print("EXACT PINNED SOURCE ACQUISITION")
print("=" * 104)

print(
    "Hugging Face repo:",
    HF_REPO,
)

print(
    "huggingface_hub:",
    huggingface_hub.__version__,
)
print()


acquisition_records = []


for index, source in enumerate(
    source_contract,
    start=1,
):

    print(
        f"[{index}/{EXPECTED_TOTAL_FILES}] "
        f"{source['day']} / "
        f"{source['remote']}"
    )


    result = {
        **source,

        "repo_id":
            HF_REPO,

        "local_path":
            None,

        "materialization_mode":
            None,

        "actual_size_bytes":
            None,

        "actual_sha256":
            None,

        "size_match":
            False,

        "sha256_match":
            False,

        "status":
            "PENDING",

        "error":
            None,

        "schema_status":
            "NOT_INSPECTED",

        "schema_columns":
            None,

        "schema_column_count":
            None,

        "parquet_footer_num_rows":
            None,

        "parquet_footer_row_groups":
            None,

        "footer_rows_match_frozen_physical_rows":
            None,

        "ordered_schema_sha256":
            None,
    }


    try:

        cached = Path(
            hf_hub_download(
                repo_id=HF_REPO,
                repo_type="dataset",
                filename=source[
                    "remote"
                ],
                revision=source[
                    "revision"
                ],
                cache_dir=str(
                    HF_CACHE_DIR
                ),
            )
        )


        destination = (
            SOURCE_DIR
            / Path(
                source[
                    "remote"
                ]
            ).name
        )


        mode = materialize_file(
            cached,
            destination,
        )


        actual_size = (
            destination
            .stat()
            .st_size
        )


        actual_sha = (
            sha256_file(
                destination
            )
        )


        size_match = (
            actual_size
            == source[
                "size_bytes"
            ]
        )


        sha_match = (
            actual_sha
            == source[
                "sha256"
            ]
        )


        result.update(
            {
                "local_path":
                    str(
                        destination
                    ),

                "materialization_mode":
                    mode,

                "actual_size_bytes":
                    actual_size,

                "actual_sha256":
                    actual_sha,

                "size_match":
                    size_match,

                "sha256_match":
                    sha_match,

                "status":
                    "VERIFIED"
                    if (
                        size_match
                        and sha_match
                    )
                    else
                    "FAILED_INTEGRITY",
            }
        )


        print(
            "  expected bytes:",
            source[
                "size_bytes"
            ],
        )

        print(
            "  actual bytes:  ",
            actual_size,
        )

        print(
            "  expected sha:  ",
            source[
                "sha256"
            ],
        )

        print(
            "  actual sha:    ",
            actual_sha,
        )


        if not size_match:

            raise RuntimeError(
                "BYTE SIZE MISMATCH"
            )


        if not sha_match:

            raise RuntimeError(
                "SHA256 MISMATCH"
            )


        print(
            "  [PASS] exact byte identity"
        )


    except Exception as exc:

        result[
            "status"
        ] = "FAILED"

        result[
            "error"
        ] = repr(
            exc
        )

        print(
            "  [FAIL]",
            repr(
                exc
            ),
        )


    acquisition_records.append(
        result
    )

    print()


# ============================================================================
# 8. COMPLETE BYTE-IDENTITY GATE
# ============================================================================

failed_integrity = [
    record
    for record in acquisition_records
    if (
        record[
            "status"
        ]
        != "VERIFIED"
        or record[
            "size_match"
        ] is not True
        or record[
            "sha256_match"
        ] is not True
    )
]


all_eight_exact = (
    len(
        acquisition_records
    )
    == EXPECTED_TOTAL_FILES
    and len(
        failed_integrity
    )
    == 0
)


print("=" * 104)
print("COMPLETE SOURCE IDENTITY GATE")
print("=" * 104)

print(
    "Expected artifacts:",
    EXPECTED_TOTAL_FILES,
)

print(
    "Recovered artifacts:",
    len(
        acquisition_records
    ),
)

print(
    "Exact verified:",
    len(
        acquisition_records
    )
    - len(
        failed_integrity
    ),
)

print(
    "Failed:",
    len(
        failed_integrity
    ),
)
print()


if not all_eight_exact:

    raise RuntimeError(
        "\nSTAGE24-0B2 BLOCKED.\n"
        "Not all 8 frozen CICIDS2017 traffic-label artifacts "
        "passed exact byte verification.\n"
        "Do not substitute another source copy."
    )


print(
    "[PASS] ALL 8 CICIDS2017 traffic-label artifacts are byte-identical."
)
print()


# ============================================================================
# 9. PARQUET FOOTER / ORDERED SCHEMA ONLY
#
# ZERO calls to:
#   read()
#   read_row_group()
#   read_table()
#   pandas.read_parquet()
# ============================================================================

import pyarrow.parquet as pq


print("=" * 104)
print("PARQUET FOOTER / SCHEMA AUDIT — ZERO TARGET ROWS")
print("=" * 104)


schema_groups = defaultdict(
    list
)


for record in (
    acquisition_records
):

    path = Path(
        record[
            "local_path"
        ]
    )


    pf = pq.ParquetFile(
        path
    )


    columns = list(
        pf
        .schema_arrow
        .names
    )


    footer_rows = int(
        pf
        .metadata
        .num_rows
    )


    row_groups = int(
        pf
        .num_row_groups
    )


    schema_sha = (
        canonical_json_sha256(
            columns
        )
    )


    row_match = (
        footer_rows
        == record[
            "physical_rows"
        ]
    )


    record.update(
        {
            "schema_status":
                "PARQUET_FOOTER_ONLY",

            "schema_columns":
                columns,

            "schema_column_count":
                len(
                    columns
                ),

            "parquet_footer_num_rows":
                footer_rows,

            "parquet_footer_row_groups":
                row_groups,

            "footer_rows_match_frozen_physical_rows":
                row_match,

            "ordered_schema_sha256":
                schema_sha,
        }
    )


    if not row_match:

        raise RuntimeError(
            "\nParquet footer row count differs from frozen "
            f"physical row count for {path.name}"
        )


    schema_groups[
        schema_sha
    ].append(
        path.name
    )


    print()

    print(
        f"{record['day']} / {path.name}"
    )

    print(
        "  columns:",
        len(
            columns
        ),
    )

    print(
        "  footer rows:",
        footer_rows,
    )

    print(
        "  frozen physical rows:",
        record[
            "physical_rows"
        ],
    )

    print(
        "  row groups:",
        row_groups,
    )

    print(
        "  ordered schema SHA256:",
        schema_sha,
    )

    print(
        "  [PASS] footer rows agree with frozen manifest"
    )


schema_pass = all(
    record[
        "schema_status"
    ]
    == "PARQUET_FOOTER_ONLY"
    and record[
        "footer_rows_match_frozen_physical_rows"
    ] is True
    for record
    in acquisition_records
)


if not schema_pass:

    raise RuntimeError(
        "Schema/footer audit failed."
    )


# ============================================================================
# 10. SCHEMA GROUP SUMMARY
# ============================================================================

print()
print("=" * 104)
print("COMPLETE PHYSICAL SCHEMA GROUPS")
print("=" * 104)

print(
    "Distinct ordered physical schemas:",
    len(
        schema_groups
    ),
)


for index, (
    schema_sha,
    filenames,
) in enumerate(
    sorted(
        schema_groups.items()
    ),
    start=1,
):

    print()

    print(
        f"Schema group {index}"
    )

    print(
        "  SHA256:",
        schema_sha,
    )


    for filename in sorted(
        filenames
    ):

        print(
            "  -",
            filename,
        )


# ============================================================================
# 11. VERIFY REQUIRED PHYSICAL FIELD NAMES FROM SCHEMA METADATA ONLY
#
# This does NOT freeze semantic equivalence yet.
# It merely verifies physical names exist.
# ============================================================================

EXPECTED_PHYSICAL_COLUMNS_FOR_BRIDGING = [
    "Destination Port",
    "Protocol",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Fwd Packet Length Max",
    "Fwd Packet Length Min",
    "Fwd Packet Length Mean",
    "Fwd Packet Length Std",
    "Bwd Packet Length Max",
    "Bwd Packet Length Min",
    "Bwd Packet Length Mean",
    "Bwd Packet Length Std",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Total",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Max",
    "Fwd IAT Min",
    "Bwd IAT Total",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",
    "Fwd PSH Flags",
    "Fwd URG Flags",
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Min Packet Length",
    "Max Packet Length",
    "Packet Length Mean",
    "Packet Length Std",
    "Packet Length Variance",
    "FIN Flag Count",
    "SYN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "URG Flag Count",
    "CWE Flag Count",
    "ECE Flag Count",
    "Down/Up Ratio",
    "Average Packet Size",
    "Avg Fwd Segment Size",
    "Avg Bwd Segment Size",
    "Subflow Fwd Packets",
    "Subflow Fwd Bytes",
    "Subflow Bwd Packets",
    "Subflow Bwd Bytes",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
    "act_data_pkt_fwd",
    "min_seg_size_forward",
    "Active Mean",
    "Active Std",
    "Active Max",
    "Active Min",
    "Idle Mean",
    "Idle Std",
    "Idle Max",
    "Idle Min",
    "Label",
]


print()
print("=" * 104)
print("PHYSICAL BRIDGE-FIELD PRESENCE CHECK")
print("=" * 104)


for record in (
    acquisition_records
):

    physical_set = set(
        record[
            "schema_columns"
        ]
    )


    missing = [
        column
        for column in (
            EXPECTED_PHYSICAL_COLUMNS_FOR_BRIDGING
        )
        if column not in physical_set
    ]


    print(
        Path(
            record[
                "local_path"
            ]
        ).name,
        "missing required physical names:",
        len(
            missing
        ),
    )


    if missing:

        print(
            "  ",
            missing,
        )

        raise RuntimeError(
            "Physical bridge-field presence gate failed."
        )


print()

print(
    "[PASS] All 8 files expose the required physical bridge field names."
)

print(
    "[IMPORTANT] Semantic equivalence has NOT yet been frozen."
)
print()


# ============================================================================
# 12. OPENING LEDGER — STILL PRISTINE
# ============================================================================

opening_ledger = {

    "PRIMARY_2018_TO_2017": {

        "bridge62": {
            "PUBLISHED": "0/1 CLOSED",
            "FLAG_CORRECTED": "0/1 CLOSED",
            "GROUNDED_S4": "0/1 CLOSED",
        },

        "bridge70": {
            "PUBLISHED": "0/1 CLOSED",
            "FLAG_CORRECTED": "0/1 CLOSED",
            "GROUNDED_S4": "0/1 CLOSED",
        },
    },

    "SECONDARY_2017_TO_2018": {

        "bridge62": {
            "FEB28_2018": "0/1 CLOSED",
        },

        "bridge70": {
            "FEB28_2018": "0/1 CLOSED",
        },
    },
}


# ============================================================================
# 13. WRITE STAGE24-0B2 RECEIPT
# ============================================================================

receipt = {

    "stage":
        "Stage24-0B2",

    "type":
        "COMPLETE_CICIDS2017_SOURCE_CONTRACT_RECOVERY",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "repository": {

        "branch":
            branch,

        "head":
            head,

        "expected_head":
            EXPECTED_HEAD,

        "scientific_parent_unchanged":
            True,
    },

    "parent_stage24_0b": {

        "receipt":
            str(
                PREVIOUS_0B_RECEIPT.relative_to(
                    REPO_DIR
                )
            ),

        "sha256":
            actual_0b_sha,

        "integrity_verified":
            True,

        "previously_recovered_artifact_count":
            len(
                receipt_0b.get(
                    "cicids2017_source",
                    {}
                ).get(
                    "frozen_source_contract",
                    []
                )
            ),

        "correction": (
            "Stage24-0B parser supported dict-valued traffic_labels "
            "metadata but not list-valued traffic_labels metadata. "
            "Frozen Thursday and Friday manifests use lists."
        ),

        "scientific_change":
            False,

        "provenance_completion":
            True,
    },

    "expected_day_file_counts":
        EXPECTED_DAY_FILE_COUNTS,

    "expected_total_files":
        EXPECTED_TOTAL_FILES,

    "hf_source": {

        "repo_id":
            HF_REPO,

        "exact_revision_per_artifact":
            True,

        "substitution_allowed":
            False,
    },

    "complete_source_contract":
        source_contract,

    "acquisition_records":
        acquisition_records,

    "verification": {

        "contract_artifact_count":
            len(
                source_contract
            ),

        "all_eight_exact_byte_identity":
            all_eight_exact,

        "parquet_footer_schema_all_pass":
            schema_pass,

        "distinct_ordered_schema_count":
            len(
                schema_groups
            ),

        "ordered_schema_groups": {
            schema_sha:
                sorted(
                    filenames
                )
            for schema_sha, filenames
            in schema_groups.items()
        },

        "required_physical_bridge_names_present_in_all_files":
            True,
    },

    "thursday_structural_metadata": [
        {
            "remote":
                record[
                    "remote"
                ],

            "physical_rows":
                record[
                    "physical_rows"
                ],

            "effective_label_records":
                record[
                    "effective_label_records"
                ],

            "structural_null_padding_rows":
                record[
                    "structural_null_padding_rows"
                ],

            "row_semantics":
                record[
                    "row_semantics"
                ],
        }
        for record
        in thursday_records
    ],

    "population_decisions": {

        "PUBLISHED_population_rule_frozen":
            False,

        "FLAG_CORRECTED_population_rule_frozen":
            False,

        "GROUNDED_S4_population_rule_frozen":
            False,

        "thursday_structural_null_handling_frozen":
            False,

        "reason": (
            "Stage24-0B2 is provenance/source recovery only. "
            "Population rules are deferred to Stage24-0C."
        ),
    },

    "opening_ledger":
        opening_ledger,

    "scientific_safety": {

        "target_file_bytes_hashed":
            True,

        "opaque_integrity_only":
            True,

        "parquet_footer_metadata_read":
            True,

        "target_row_groups_read":
            0,

        "target_rows_materialized":
            0,

        "target_feature_values_materialized":
            0,

        "target_labels_materialized":
            0,

        "target_prevalence_computed":
            False,

        "PUBLISHED_rows_constructed":
            False,

        "FLAG_CORRECTED_rows_constructed":
            False,

        "GROUNDED_S4_rows_constructed":
            False,

        "model_fits":
            0,

        "model_predictions":
            0,

        "metrics":
            0,

        "target_openings_consumed":
            0,

        "target_opening_budget":
            8,
    },

    "next_authorized_step": (
        "Stage24-0C: explicit ordered semantic bridge, exact target "
        "population/variant rules including Thursday structural-null "
        "handling and GROUNDED_S4 eligibility, label/family taxonomy, "
        "source sanity sequence, bridge-specific refit budget, GPU-first "
        "runtime rule, source-only threshold procedure, metrics, nulls, "
        "paired bootstrap, calibration, and final pre-opening lock."
    ),
}


receipt_path = (
    LOCK_DIR
    / "stage24_0b2_complete_cicids2017_source_contract.json"
)


with receipt_path.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        receipt,
        f,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )

    f.write(
        "\n"
    )


receipt_sha = (
    sha256_file(
        receipt_path
    )
)


sha_path = (
    LOCK_DIR
    / "stage24_0b2_complete_cicids2017_source_contract.sha256"
)


sha_path.write_text(
    f"{receipt_sha}  {receipt_path.name}\n",
    encoding="utf-8",
)


# ============================================================================
# 14. FINAL SYNTHESIS
# ============================================================================

print()
print("=" * 104)
print("STAGE24-0B2 FINAL SYNTHESIS")
print("=" * 104)

print(
    "Scientific parent HEAD:"
)

print(
    " ",
    head,
)

print()

print(
    "Frozen CICIDS2017 source contract:"
)

for day in (
    EXPECTED_DAY_FILE_COUNTS
):

    count = sum(
        1
        for record
        in source_contract
        if record[
            "day"
        ] == day
    )

    print(
        f"  {day:10s}: {count}"
    )


print(
    "  ----------------"
)

print(
    f"  TOTAL     : {len(source_contract)}"
)

print()

print(
    "Exact byte identities:"
)

print(
    " ",
    f"{sum(r['sha256_match'] is True for r in acquisition_records)}/8 PASS",
)

print()

print(
    "Distinct ordered schemas:"
)

print(
    " ",
    len(
        schema_groups
    ),
)

print()

print(
    "Required physical bridge fields present in all 8:"
)

print(
    " ",
    True,
)

print()

print(
    "Target row groups read:"
)

print(
    "  0"
)

print(
    "Target rows materialized:"
)

print(
    "  0"
)

print(
    "Target labels materialized:"
)

print(
    "  0"
)

print(
    "Target prevalence computed:"
)

print(
    "  False"
)

print(
    "Model fits:"
)

print(
    "  0"
)

print(
    "Model predictions:"
)

print(
    "  0"
)

print(
    "Metrics:"
)

print(
    "  0"
)

print(
    "Target openings:"
)

print(
    "  0 / 8"
)

print()

print(
    "Receipt:"
)

print(
    " ",
    receipt_path,
)

print(
    "Receipt SHA256:"
)

print(
    " ",
    receipt_sha,
)

print()

print("-" * 104)
print("STAGE24-0B2: PASS")
print("Complete frozen CICIDS2017 Monday-Friday source contract recovered.")
print("All 8 target containers verified by exact SHA256.")
print("No target row or label has been materialized.")
print("Target model openings remain 0 / 8.")
print()
print("NEXT AUTHORIZED STEP:")
print("Stage24-0C — semantic bridge + population/variant protocol lock.")
print("-" * 104)
print("STAGE24-0B2 COMPLETE")
print("=" * 104)

In [ ]:
# ============================================================================
# STAGE24-0C — FINAL PRE-OPENING SCIENTIFIC PROTOCOL LOCK
#
# PURPOSE
# -------
# Freeze BEFORE ANY TARGET ROW OR LABEL IS PARSED:
#
#   1. exact Stage22R -> CICIDS2017 semantic bridge
#   2. bridge62 / bridge70 definitions
#   3. PUBLISHED / FLAG_CORRECTED / GROUNDED_S4 variant semantics
#   4. Thursday structural-null population rule
#   5. Stage20 exact-grounding rule
#   6. binary-label rule + attack-family taxonomy
#   7. primary / secondary train-validation-target schedule
#   8. exact scientific fit budget
#   9. GPU-first runtime policy
#  10. exact source-only threshold selection
#  11. metrics / nulls / calibration / bootstrap
#  12. anti-adaptation rules
#  13. target opening ledger = 0 / 8
#
# IMPORTANT
# ---------
# This cell DOES NOT commit/push yet.
#
# After this passes, Stage24-0D will:
#   commit -> push -> remotely verify protocol lock.
#
# Until Stage24-0D passes:
#   TARGET MODEL INFERENCE REMAINS FORBIDDEN.
#
# TARGET OPENINGS BEFORE: 0 / 8
# TARGET OPENINGS AFTER:  0 / 8
# ============================================================================

from __future__ import annotations

import os
import json
import copy
import hashlib
import subprocess
import traceback
from pathlib import Path
from datetime import datetime, timezone

import numpy as np


# ============================================================================
# 0. CONSTANTS
# ============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "963b2da0043c7438624916280de154a5628f8879"
)

STAGE24_ROOT = (
    REPO_DIR
    / "results"
    / "stage24_cross_dataset"
)

LOCK_DIR = (
    STAGE24_ROOT
    / "stage24_0_protocol_lock"
)

LOCK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


STAGE24_0B2_RECEIPT = (
    LOCK_DIR
    / "stage24_0b2_complete_cicids2017_source_contract.json"
)

STAGE24_0B2_SHA = (
    LOCK_DIR
    / "stage24_0b2_complete_cicids2017_source_contract.sha256"
)


STAGE22_RESULT = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

STAGE22_MODEL_DIR = (
    REPO_DIR
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
)

STAGE22_LGBM_MODEL = (
    STAGE22_MODEL_DIR
    / "chronological_natural_lightgbm_model.txt"
)

STAGE22_XGB_MODEL = (
    STAGE22_MODEL_DIR
    / "chronological_natural_xgboost_model.json"
)


STAGE22_MEMBERSHIP = (
    REPO_DIR
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
    / "stage22r_1b1_membership_summary.json"
)


STAGE23_SPEC = (
    REPO_DIR
    / "results"
    / "stage23_shortcut_feature_audit"
    / "stage23_0_protocol_lock"
    / "feature_subset_spec.json"
)


STAGE20_FLAG = (
    REPO_DIR
    / "results"
    / "stage20_flag_serialization_correction"
    / "stage20_1c8_flag_serialization_correction.json"
)


STAGE20_THRESHOLD = (
    REPO_DIR
    / "results"
    / "stage20_1e_training"
    / "stage20_1e3_validation_execution_semantics_lock.json"
)


STAGE20_THURSDAY_ERRATUM = (
    REPO_DIR
    / "results"
    / "stage20_1e_training"
    / "stage20_1e1v_thursday_label_source_ingestion_erratum.json"
)


STAGE20_DAY_MANIFESTS = {
    "Monday": (
        REPO_DIR
        / "results"
        / "stage20_1e_training"
        / "stage20_1e1m_monday_compact_corpus_manifest.json"
    ),

    "Tuesday": (
        REPO_DIR
        / "results"
        / "stage20_1e_training"
        / "stage20_1e1t_tuesday_compact_corpus_manifest.json"
    ),

    "Wednesday": (
        REPO_DIR
        / "results"
        / "stage20_1e_training"
        / "stage20_1e1w_wednesday_compact_corpus_manifest.json"
    ),

    "Thursday": (
        REPO_DIR
        / "results"
        / "stage20_1e_training"
        / "stage20_1e1v_thursday_validation_compact_corpus_manifest.json"
    ),

    "Friday": (
        REPO_DIR
        / "results"
        / "stage20_1e_training"
        / "stage20_1e4_friday_holdout_compact_corpus_manifest.json"
    ),
}


THURSDAY_MORNING_REMOTE = (
    "traffic_labels/"
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet"
)


FLAG_SOURCE_FEATURES = [
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "ACK Flag Cnt",
    "URG Flag Cnt",
    "CWE Flag Count",
    "ECE Flag Cnt",
]


print("=" * 108)
print("STAGE24-0C — FINAL PRE-OPENING SCIENTIFIC PROTOCOL LOCK")
print("=" * 108)

print("Target row parsing:       FORBIDDEN")
print("Target label parsing:     FORBIDDEN")
print("Target inference:         FORBIDDEN")
print("Scientific model fits:    0")
print("Target model openings:    0 / 8")
print()


# ============================================================================
# 1. HELPERS
# ============================================================================

def run_git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=True,
    ).stdout.strip()


def load_json(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def write_json_with_sha(
    path: Path,
    obj,
):
    with path.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )

        f.write("\n")


    digest = sha256_file(
        path
    )


    sha_path = path.with_suffix(
        ".sha256"
    )


    sha_path.write_text(
        f"{digest}  {path.name}\n",
        encoding="utf-8",
    )


    return digest, sha_path


def verify_sha_sidecar(
    path: Path,
    sidecar: Path,
):
    declared = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
    )

    actual = sha256_file(
        path
    )

    if declared != actual:

        raise RuntimeError(
            f"SHA mismatch for {path.name}\n"
            f"Declared: {declared}\n"
            f"Actual:   {actual}"
        )

    return actual


def find_grounded_count(
    manifest_obj,
):
    exact_join = manifest_obj.get(
        "exact_join",
        {}
    )

    preferred = [
        "supervised_matched_flows",
        "training_matched_flows",
        "validation_matched_flows",
        "holdout_matched_flows",
    ]


    for key in preferred:

        value = exact_join.get(
            key
        )

        if isinstance(
            value,
            int,
        ):
            return key, value


    candidates = []

    for key, value in (
        exact_join.items()
    ):

        if (
            isinstance(
                value,
                int,
            )
            and "matched_flows" in key
            and "unmatched" not in key
        ):

            candidates.append(
                (
                    key,
                    value,
                )
            )


    if len(candidates) != 1:

        raise RuntimeError(
            "Could not uniquely resolve frozen grounded-flow count. "
            f"Candidates={candidates}"
        )


    return candidates[0]


# ============================================================================
# 2. REPOSITORY + 0B2 SAFETY
# ============================================================================

print("=" * 108)
print("REPOSITORY / PROVENANCE SAFETY")
print("=" * 108)

branch = run_git(
    "branch",
    "--show-current",
)

head = run_git(
    "rev-parse",
    "HEAD",
)

status = run_git(
    "status",
    "--porcelain",
)


print(
    "Branch:",
    branch,
)

print(
    "HEAD:  ",
    head,
)


if branch != "main":

    raise RuntimeError(
        f"Expected main, got {branch!r}"
    )


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "\nScientific parent changed before Stage24-0 lock.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


unexpected_dirty = []

if status:

    for line in (
        status.splitlines()
    ):

        rel = (
            line[
                3:
            ].strip()
        )

        if " -> " in rel:

            rel = rel.split(
                " -> ",
                1,
            )[1]


        if not rel.startswith(
            "results/stage24_cross_dataset/"
        ):

            unexpected_dirty.append(
                line
            )


if unexpected_dirty:

    raise RuntimeError(
        "\nUnexpected dirty paths outside Stage24:\n"
        + "\n".join(
            unexpected_dirty
        )
    )


required_files = [
    STAGE24_0B2_RECEIPT,
    STAGE24_0B2_SHA,
    STAGE22_RESULT,
    STAGE22_LGBM_MODEL,
    STAGE22_XGB_MODEL,
    STAGE22_MEMBERSHIP,
    STAGE23_SPEC,
    STAGE20_FLAG,
    STAGE20_THRESHOLD,
    STAGE20_THURSDAY_ERRATUM,
    *STAGE20_DAY_MANIFESTS.values(),
]


for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


receipt_0b2_sha = verify_sha_sidecar(
    STAGE24_0B2_RECEIPT,
    STAGE24_0B2_SHA,
)


receipt_0b2 = load_json(
    STAGE24_0B2_RECEIPT
)


if (
    receipt_0b2[
        "scientific_safety"
    ][
        "target_openings_consumed"
    ]
    != 0
):

    raise RuntimeError(
        "0B2 opening ledger is not pristine."
    )


if (
    receipt_0b2[
        "scientific_safety"
    ][
        "target_rows_materialized"
    ]
    != 0
):

    raise RuntimeError(
        "0B2 unexpectedly materialized target rows."
    )


if (
    receipt_0b2[
        "verification"
    ][
        "all_eight_exact_byte_identity"
    ]
    is not True
):

    raise RuntimeError(
        "0B2 does not verify all eight files."
    )


source_contract = (
    receipt_0b2[
        "complete_source_contract"
    ]
)


acquisition_records = (
    receipt_0b2[
        "acquisition_records"
    ]
)


if len(
    source_contract
) != 8:

    raise RuntimeError(
        "Expected exactly 8 CICIDS2017 source artifacts."
    )


schema_hashes = {
    record[
        "ordered_schema_sha256"
    ]
    for record
    in acquisition_records
}


if len(
    schema_hashes
) != 1:

    raise RuntimeError(
        "CICIDS2017 physical schemas are not identical."
    )


common_physical_schema = list(
    acquisition_records[
        0
    ][
        "schema_columns"
    ]
)


for record in (
    acquisition_records
):

    if (
        record[
            "schema_columns"
        ]
        != common_physical_schema
    ):

        raise RuntimeError(
            "Ordered physical schema mismatch."
        )


print(
    "0B2 SHA256:",
    receipt_0b2_sha,
)

print(
    "Verified CICIDS2017 artifacts:",
    len(
        source_contract
    ),
)

print(
    "Common physical schema columns:",
    len(
        common_physical_schema
    ),
)

print(
    "Common schema SHA256:",
    next(
        iter(
            schema_hashes
        )
    ),
)

print()

print(
    "[PASS] 0B2 provenance is intact."
)
print()


# ============================================================================
# 3. LOAD FROZEN SOURCE PROTOCOL
# ============================================================================

stage22 = load_json(
    STAGE22_RESULT
)

stage22_membership = load_json(
    STAGE22_MEMBERSHIP
)

stage23 = load_json(
    STAGE23_SPEC
)

stage20_flag = load_json(
    STAGE20_FLAG
)

stage20_threshold = load_json(
    STAGE20_THRESHOLD
)

thursday_erratum = load_json(
    STAGE20_THURSDAY_ERRATUM
)


stage22_features = list(
    stage22[
        "data"
    ][
        "feature_order"
    ]
)


if len(
    stage22_features
) != 70:

    raise RuntimeError(
        "Stage22 feature count != 70."
    )


if (
    stage22[
        "data"
    ][
        "scaling"
    ]
    != "NONE"
):

    raise RuntimeError(
        "Unexpected Stage22 scaling."
    )


if (
    stage22[
        "data"
    ][
        "explicit_imputation"
    ]
    != "NONE"
):

    raise RuntimeError(
        "Unexpected Stage22 imputation."
    )


chronological_membership = (
    stage22_membership[
        "membership_derivation"
    ]
)


expected_train_membership = (
    "day_id 0..6"
)

expected_validation_membership = (
    "day_id 7"
)


if (
    chronological_membership[
        "CHRONOLOGICAL_NATURAL_train"
    ]
    != expected_train_membership
):

    raise RuntimeError(
        "Unexpected Stage22 chronological training membership."
    )


if (
    chronological_membership[
        "CHRONOLOGICAL_NATURAL_validation"
    ]
    != expected_validation_membership
):

    raise RuntimeError(
        "Unexpected Stage22 chronological validation membership."
    )


print("=" * 108)
print("FROZEN PRIMARY SOURCE PROTOCOL")
print("=" * 108)

print(
    "Feature count:",
    len(
        stage22_features
    ),
)

print(
    "Scaling:",
    stage22[
        "data"
    ][
        "scaling"
    ],
)

print(
    "Imputation:",
    stage22[
        "data"
    ][
        "explicit_imputation"
    ],
)

print()

print(
    "Primary source train membership:",
    chronological_membership[
        "CHRONOLOGICAL_NATURAL_train"
    ],
)

print(
    "Primary source validation membership:",
    chronological_membership[
        "CHRONOLOGICAL_NATURAL_validation"
    ],
)

print()

print(
    "Primary source train rows:",
    stage22[
        "data"
    ][
        "train"
    ][
        "rows"
    ],
)

print(
    "Primary source validation rows:",
    stage22[
        "data"
    ][
        "validation"
    ][
        "rows"
    ],
)

print()

print(
    "[PASS] Exact Stage22R chronological-natural source split recovered."
)
print()


# ============================================================================
# 4. VERIFY INHERITED STAGE22 MODEL FILE HASHES
# ============================================================================

print("=" * 108)
print("FROZEN STAGE22 MODEL INTEGRITY")
print("=" * 108)


expected_model_hashes = (
    stage22[
        "artifacts"
    ][
        "hashes_before_result_json"
    ]
)


actual_lgbm_sha = sha256_file(
    STAGE22_LGBM_MODEL
)

actual_xgb_sha = sha256_file(
    STAGE22_XGB_MODEL
)


expected_lgbm_sha = (
    expected_model_hashes[
        STAGE22_LGBM_MODEL.name
    ]
)

expected_xgb_sha = (
    expected_model_hashes[
        STAGE22_XGB_MODEL.name
    ]
)


print(
    "LightGBM expected:",
    expected_lgbm_sha,
)

print(
    "LightGBM actual:  ",
    actual_lgbm_sha,
)

print()

print(
    "XGBoost expected:",
    expected_xgb_sha,
)

print(
    "XGBoost actual:  ",
    actual_xgb_sha,
)


if (
    actual_lgbm_sha
    != expected_lgbm_sha
):

    raise RuntimeError(
        "Frozen Stage22 LightGBM model hash mismatch."
    )


if (
    actual_xgb_sha
    != expected_xgb_sha
):

    raise RuntimeError(
        "Frozen Stage22 XGBoost model hash mismatch."
    )


print()

print(
    "[PASS] Stage22 bridge70 inherited model pair is byte-exact."
)
print()


# ============================================================================
# 5. FREEZE BRIDGE62 / BRIDGE70
# ============================================================================

bridge70_features = list(
    stage22_features
)


flag_set = set(
    FLAG_SOURCE_FEATURES
)


bridge62_features = [
    feature
    for feature
    in stage22_features
    if feature not in flag_set
]


if len(
    bridge62_features
) != 62:

    raise RuntimeError(
        "bridge62 dimension != 62."
    )


if len(
    bridge70_features
) != 70:

    raise RuntimeError(
        "bridge70 dimension != 70."
    )


stage23_nsg = (
    stage23[
        "subsets"
    ][
        "NO_SUSPICIOUS_GROUP"
    ][
        "features"
    ]
)

stage23_behavior = (
    stage23[
        "subsets"
    ][
        "BEHAVIOR_ONLY"
    ][
        "features"
    ]
)


if (
    bridge62_features
    == stage23_nsg
):

    raise RuntimeError(
        "bridge62 unexpectedly equals NO_SUSPICIOUS_GROUP."
    )


if (
    bridge62_features
    == stage23_behavior
):

    raise RuntimeError(
        "bridge62 unexpectedly equals BEHAVIOR_ONLY."
    )


print("=" * 108)
print("STAGE24 BRIDGE DEFINITIONS")
print("=" * 108)

print(
    "bridge70:",
    len(
        bridge70_features
    ),
)

print(
    "bridge62:",
    len(
        bridge62_features
    ),
)

print()

print(
    "bridge62 != NO_SUSPICIOUS_GROUP:",
    True,
)

print(
    "bridge62 != BEHAVIOR_ONLY:",
    True,
)

print()


# ============================================================================
# 6. FREEZE EXACT NON-FLAG SEMANTIC ALIASES
# ============================================================================

BASE_TARGET_ALIAS = {

    "Dst Port":
        "Destination Port",

    "Protocol":
        "Protocol",

    "Flow Duration":
        "Flow Duration",

    "Tot Fwd Pkts":
        "Total Fwd Packets",

    "Tot Bwd Pkts":
        "Total Backward Packets",

    "TotLen Fwd Pkts":
        "Total Length of Fwd Packets",

    "TotLen Bwd Pkts":
        "Total Length of Bwd Packets",

    "Fwd Pkt Len Max":
        "Fwd Packet Length Max",

    "Fwd Pkt Len Min":
        "Fwd Packet Length Min",

    "Fwd Pkt Len Mean":
        "Fwd Packet Length Mean",

    "Fwd Pkt Len Std":
        "Fwd Packet Length Std",

    "Bwd Pkt Len Max":
        "Bwd Packet Length Max",

    "Bwd Pkt Len Min":
        "Bwd Packet Length Min",

    "Bwd Pkt Len Mean":
        "Bwd Packet Length Mean",

    "Bwd Pkt Len Std":
        "Bwd Packet Length Std",

    "Flow Byts/s":
        "Flow Bytes/s",

    "Flow Pkts/s":
        "Flow Packets/s",

    "Flow IAT Mean":
        "Flow IAT Mean",

    "Flow IAT Std":
        "Flow IAT Std",

    "Flow IAT Max":
        "Flow IAT Max",

    "Flow IAT Min":
        "Flow IAT Min",

    "Fwd IAT Tot":
        "Fwd IAT Total",

    "Fwd IAT Mean":
        "Fwd IAT Mean",

    "Fwd IAT Std":
        "Fwd IAT Std",

    "Fwd IAT Max":
        "Fwd IAT Max",

    "Fwd IAT Min":
        "Fwd IAT Min",

    "Bwd IAT Tot":
        "Bwd IAT Total",

    "Bwd IAT Mean":
        "Bwd IAT Mean",

    "Bwd IAT Std":
        "Bwd IAT Std",

    "Bwd IAT Max":
        "Bwd IAT Max",

    "Bwd IAT Min":
        "Bwd IAT Min",

    "Fwd PSH Flags":
        "Fwd PSH Flags",

    "Fwd URG Flags":
        "Fwd URG Flags",

    "Fwd Header Len":
        "Fwd Header Length",

    "Bwd Header Len":
        "Bwd Header Length",

    "Fwd Pkts/s":
        "Fwd Packets/s",

    "Bwd Pkts/s":
        "Bwd Packets/s",

    "Pkt Len Min":
        "Min Packet Length",

    "Pkt Len Max":
        "Max Packet Length",

    "Pkt Len Mean":
        "Packet Length Mean",

    "Pkt Len Std":
        "Packet Length Std",

    "Pkt Len Var":
        "Packet Length Variance",

    "Down/Up Ratio":
        "Down/Up Ratio",

    "Pkt Size Avg":
        "Average Packet Size",

    "Fwd Seg Size Avg":
        "Avg Fwd Segment Size",

    "Bwd Seg Size Avg":
        "Avg Bwd Segment Size",

    "Subflow Fwd Pkts":
        "Subflow Fwd Packets",

    "Subflow Fwd Byts":
        "Subflow Fwd Bytes",

    "Subflow Bwd Pkts":
        "Subflow Bwd Packets",

    "Subflow Bwd Byts":
        "Subflow Bwd Bytes",

    "Init Fwd Win Byts":
        "Init_Win_bytes_forward",

    "Init Bwd Win Byts":
        "Init_Win_bytes_backward",

    "Fwd Act Data Pkts":
        "act_data_pkt_fwd",

    "Fwd Seg Size Min":
        "min_seg_size_forward",

    "Active Mean":
        "Active Mean",

    "Active Std":
        "Active Std",

    "Active Max":
        "Active Max",

    "Active Min":
        "Active Min",

    "Idle Mean":
        "Idle Mean",

    "Idle Std":
        "Idle Std",

    "Idle Max":
        "Idle Max",

    "Idle Min":
        "Idle Min",
}


expected_nonflag_features = [
    feature
    for feature
    in stage22_features
    if feature not in flag_set
]


if (
    set(
        BASE_TARGET_ALIAS.keys()
    )
    != set(
        expected_nonflag_features
    )
):

    missing = (
        set(
            expected_nonflag_features
        )
        - set(
            BASE_TARGET_ALIAS
        )
    )

    extra = (
        set(
            BASE_TARGET_ALIAS
        )
        - set(
            expected_nonflag_features
        )
    )

    raise RuntimeError(
        "Non-flag semantic alias map mismatch.\n"
        f"Missing={sorted(missing)}\n"
        f"Extra={sorted(extra)}"
    )


physical_schema_set = set(
    common_physical_schema
)


for source_feature, target_feature in (
    BASE_TARGET_ALIAS.items()
):

    if (
        target_feature
        not in physical_schema_set
    ):

        raise RuntimeError(
            "Mapped CICIDS2017 physical column absent: "
            f"{source_feature!r} -> {target_feature!r}"
        )


print("=" * 108)
print("NON-FLAG SEMANTIC BRIDGE")
print("=" * 108)

print(
    "Mapped non-flag features:",
    len(
        BASE_TARGET_ALIAS
    ),
)

print(
    "[PASS] All non-flag mappings exist in all 8 exact CICIDS2017 files."
)
print()


# ============================================================================
# 7. FREEZE FLAG SEMANTICS
#
# PUBLISHED:
#   literal published CICIDS2017 header interpretation.
#
# FLAG_CORRECTED:
#   Stage20-1C8 source-derived exporter serialization correction.
#
# GROUNDED_S4:
#   uses FLAG_CORRECTED feature semantics + Stage20 exact-grounded population.
#
# Source Stage22 feature semantic interpretation:
#
#   FIN Flag Cnt   -> FIN
#   SYN Flag Cnt   -> SYN
#   RST Flag Cnt   -> RST
#   PSH Flag Cnt   -> PSH
#   ACK Flag Cnt   -> ACK
#   URG Flag Cnt   -> URG
#   CWE Flag Count -> CWR
#   ECE Flag Cnt   -> ECE
#
# This is the NOMINAL Stage22 feature-semantic contract used prospectively
# for cross-dataset mapping.
# ============================================================================

SOURCE_FLAG_SEMANTICS = {
    "FIN Flag Cnt": "FIN",
    "SYN Flag Cnt": "SYN",
    "RST Flag Cnt": "RST",
    "PSH Flag Cnt": "PSH",
    "ACK Flag Cnt": "ACK",
    "URG Flag Cnt": "URG",
    "CWE Flag Count": "CWR",
    "ECE Flag Cnt": "ECE",
}


TARGET_LITERAL_FLAG_COLUMNS = {
    "FIN Flag Cnt": "FIN Flag Count",
    "SYN Flag Cnt": "SYN Flag Count",
    "RST Flag Cnt": "RST Flag Count",
    "PSH Flag Cnt": "PSH Flag Count",
    "ACK Flag Cnt": "ACK Flag Count",
    "URG Flag Cnt": "URG Flag Count",
    "CWE Flag Count": "CWE Flag Count",
    "ECE Flag Cnt": "ECE Flag Count",
}


physical_to_semantic_flag = dict(
    stage20_flag[
        "physical_csv_column_to_semantic_flag"
    ]
)


expected_physical_to_semantic = {
    "ACK Flag Count": "ACK",
    "CWE Flag Count": "URG",
    "ECE Flag Count": "CWR",
    "FIN Flag Count": "RST",
    "PSH Flag Count": "SYN",
    "RST Flag Count": "ECE",
    "SYN Flag Count": "PSH",
    "URG Flag Count": "FIN",
}


if (
    physical_to_semantic_flag
    != expected_physical_to_semantic
):

    raise RuntimeError(
        "Stage20-1C8 frozen flag mapping changed."
    )


semantic_to_physical_flag = {}


for physical, semantic in (
    physical_to_semantic_flag.items()
):

    if semantic in (
        semantic_to_physical_flag
    ):

        raise RuntimeError(
            f"Non-bijective flag mapping for {semantic}"
        )

    semantic_to_physical_flag[
        semantic
    ] = physical


if (
    set(
        semantic_to_physical_flag.keys()
    )
    != set(
        SOURCE_FLAG_SEMANTICS.values()
    )
):

    raise RuntimeError(
        "Source/target flag semantic universe mismatch."
    )


TARGET_CORRECTED_FLAG_COLUMNS = {}


for source_feature, semantic in (
    SOURCE_FLAG_SEMANTICS.items()
):

    TARGET_CORRECTED_FLAG_COLUMNS[
        source_feature
    ] = (
        semantic_to_physical_flag[
            semantic
        ]
    )


for physical in (
    TARGET_CORRECTED_FLAG_COLUMNS.values()
):

    if physical not in (
        physical_schema_set
    ):

        raise RuntimeError(
            f"Corrected target physical flag absent: {physical}"
        )


print("=" * 108)
print("FLAG SEMANTIC BRIDGE")
print("=" * 108)

print(
    "PUBLISHED literal mapping:"
)

for feature in (
    FLAG_SOURCE_FEATURES
):

    print(
        f"  {feature:18s} <- "
        f"{TARGET_LITERAL_FLAG_COLUMNS[feature]}"
    )


print()

print(
    "FLAG_CORRECTED Stage20-1C8 mapping:"
)

for feature in (
    FLAG_SOURCE_FEATURES
):

    semantic = (
        SOURCE_FLAG_SEMANTICS[
            feature
        ]
    )

    physical = (
        TARGET_CORRECTED_FLAG_COLUMNS[
            feature
        ]
    )

    print(
        f"  {feature:18s} "
        f"[{semantic:3s}] <- {physical}"
    )


print()

print(
    "[PASS] Stage20 source-derived flag correction frozen prospectively."
)
print()


# ============================================================================
# 8. BUILD EXACT ORDERED VARIANT MAPPINGS
# ============================================================================

published_map70 = dict(
    BASE_TARGET_ALIAS
)

published_map70.update(
    TARGET_LITERAL_FLAG_COLUMNS
)


corrected_map70 = dict(
    BASE_TARGET_ALIAS
)

corrected_map70.update(
    TARGET_CORRECTED_FLAG_COLUMNS
)


grounded_map70 = dict(
    corrected_map70
)


published_order70 = [
    published_map70[
        feature
    ]
    for feature
    in bridge70_features
]


corrected_order70 = [
    corrected_map70[
        feature
    ]
    for feature
    in bridge70_features
]


grounded_order70 = [
    grounded_map70[
        feature
    ]
    for feature
    in bridge70_features
]


published_map62 = {
    feature:
        published_map70[
            feature
        ]
    for feature
    in bridge62_features
}


corrected_map62 = {
    feature:
        corrected_map70[
            feature
        ]
    for feature
    in bridge62_features
}


grounded_map62 = {
    feature:
        grounded_map70[
            feature
        ]
    for feature
    in bridge62_features
}


# Since bridge62 excludes all eight corrected aggregate flags,
# PUBLISHED and FLAG_CORRECTED MUST be identical representations.

if (
    published_map62
    != corrected_map62
):

    raise RuntimeError(
        "bridge62 PUBLISHED and FLAG_CORRECTED unexpectedly differ."
    )


if len(
    published_map70
) != 70:

    raise RuntimeError(
        "Published bridge70 mapping != 70."
    )


if len(
    corrected_map70
) != 70:

    raise RuntimeError(
        "Corrected bridge70 mapping != 70."
    )


if len(
    published_map62
) != 62:

    raise RuntimeError(
        "Published bridge62 mapping != 62."
    )


print("=" * 108)
print("ORDERED TARGET REPRESENTATIONS")
print("=" * 108)

print(
    "bridge70 / PUBLISHED:",
    len(
        published_order70
    ),
)

print(
    "bridge70 / FLAG_CORRECTED:",
    len(
        corrected_order70
    ),
)

print(
    "bridge70 / GROUNDED_S4:",
    len(
        grounded_order70
    ),
)

print()

print(
    "bridge62 / PUBLISHED:",
    len(
        published_map62
    ),
)

print(
    "bridge62 / FLAG_CORRECTED:",
    len(
        corrected_map62
    ),
)

print()

print(
    "bridge62 PUBLISHED == FLAG_CORRECTED:",
    True,
)

print()


# ============================================================================
# 9. FREEZE PUBLISHED POPULATION
#
# Stage20 already froze one source-specific ingestion erratum:
#
# Thursday morning:
#   ordinals 1..170366 = records
#   ordinals 170367..458968 = all-85-column-null structural EOF padding
#
# This exclusion was frozen before Stage20 Thursday PCAP/join/model inference.
# We inherit it exactly.
# ============================================================================

frozen_thursday_rule = (
    thursday_erratum[
        "frozen_ingestion_rule"
    ]
)


if (
    frozen_thursday_rule[
        "effective_morning_record_ordinals"
    ]
    != "1..170366"
):

    raise RuntimeError(
        "Unexpected Thursday effective prefix."
    )


if (
    frozen_thursday_rule[
        "excluded_morning_physical_ordinals"
    ]
    != "170367..458968"
):

    raise RuntimeError(
        "Unexpected Thursday structural-null suffix."
    )


population_records = []


for record in (
    source_contract
):

    remote = record[
        "remote"
    ]

    physical_rows = int(
        record[
            "physical_rows"
        ]
    )


    if (
        remote
        == THURSDAY_MORNING_REMOTE
    ):

        effective_rows = 170366

        inclusion_rule = (
            "INCLUDE_PHYSICAL_ORDINALS_1_THROUGH_170366;"
            "EXCLUDE_FROZEN_ALL_85_COLUMN_NULL_EOF_SUFFIX_170367_THROUGH_458968"
        )

    else:

        effective_rows = (
            physical_rows
        )

        inclusion_rule = (
            "INCLUDE_ALL_PHYSICAL_ROWS"
        )


    population_records.append(
        {
            "day":
                record[
                    "day"
                ],

            "remote":
                remote,

            "physical_rows":
                physical_rows,

            "effective_rows":
                effective_rows,

            "inclusion_rule":
                inclusion_rule,
        }
    )


published_effective_rows = sum(
    record[
        "effective_rows"
    ]
    for record
    in population_records
)


if (
    published_effective_rows
    != 2_830_743
):

    raise RuntimeError(
        "\nUnexpected effective CICIDS2017 population.\n"
        f"Expected: 2830743\n"
        f"Actual:   {published_effective_rows}"
    )


print("=" * 108)
print("PUBLISHED POPULATION LOCK")
print("=" * 108)

for record in (
    population_records
):

    print(
        f"{record['day']:10s} "
        f"{Path(record['remote']).name}"
    )

    print(
        "  physical:",
        record[
            "physical_rows"
        ],
    )

    print(
        "  effective:",
        record[
            "effective_rows"
        ],
    )


print()

print(
    "Total effective PUBLISHED rows:",
    published_effective_rows,
)

print(
    "[PASS] Thursday structural-null handling inherited exactly."
)
print()


# ============================================================================
# 10. FREEZE GROUNDED_S4 POPULATION CONTRACT
#
# GROUNDED_S4 is NOT allowed to be selected from Stage24 model performance.
#
# It means:
#
#   published CICIDS2017 effective rows that participate in the frozen
#   Stage20 source-faithful EXACT join to reconstructed PCAP flows.
#
# Requirements:
#   - exact only
#   - no fuzzy
#   - no nearest
#   - no tolerance
#   - no label-guided repair
#   - no performance-guided membership changes
#
# Feature semantics for GROUNDED_S4 = FLAG_CORRECTED semantics.
#
# IMPORTANT:
# Stage20 compact corpora preserve the grounded matched flow counts but do
# not by themselves constitute Stage24 tabular row-membership vectors.
#
# Therefore Stage24 may reconstruct row membership ONLY by replaying the
# already-frozen Stage20 exact joining procedure against exact pinned PCAPs
# if a reusable exact row-membership artifact is not available.
# ============================================================================

grounded_counts = {}


for day, manifest_path in (
    STAGE20_DAY_MANIFESTS.items()
):

    manifest = load_json(
        manifest_path
    )

    key, count = (
        find_grounded_count(
            manifest
        )
    )


    exact_join = manifest.get(
        "exact_join",
        {}
    )


    if (
        exact_join.get(
            "matching"
        )
        != "EXACT_ONLY"
    ):

        raise RuntimeError(
            f"{day}: Stage20 grounding was not EXACT_ONLY."
        )


    if (
        exact_join.get(
            "fuzzy"
        )
        is not False
    ):

        raise RuntimeError(
            f"{day}: fuzzy matching unexpectedly enabled."
        )


    if (
        exact_join.get(
            "nearest"
        )
        is not False
    ):

        raise RuntimeError(
            f"{day}: nearest matching unexpectedly enabled."
        )


    if (
        exact_join.get(
            "tolerance"
        )
        is not False
    ):

        raise RuntimeError(
            f"{day}: tolerance matching unexpectedly enabled."
        )


    grounded_counts[
        day
    ] = {
        "manifest":
            str(
                manifest_path.relative_to(
                    REPO_DIR
                )
            ),

        "count_key":
            key,

        "matched_flows":
            int(
                count
            ),

        "matching":
            "EXACT_ONLY",
    }


expected_grounded_rows = sum(
    item[
        "matched_flows"
    ]
    for item
    in grounded_counts.values()
)


if (
    expected_grounded_rows
    <= 0
):

    raise RuntimeError(
        "Grounded population count invalid."
    )


print("=" * 108)
print("GROUNDED_S4 POPULATION CONTRACT")
print("=" * 108)

for day, item in (
    grounded_counts.items()
):

    print(
        f"{day:10s}: "
        f"{item['matched_flows']} "
        f"({item['count_key']})"
    )


print()

print(
    "Expected total grounded matched flows:",
    expected_grounded_rows,
)

print()

print(
    "[FROZEN] GROUNDED_S4 membership = Stage20 exact source-faithful join only."
)

print(
    "[FROZEN] No nearest/fuzzy/tolerance/label-guided/model-guided recovery."
)

print(
    "[FROZEN] GROUNDED_S4 feature semantics = FLAG_CORRECTED."
)
print()


# ============================================================================
# 11. FREEZE TARGET LABEL RULE
#
# No target labels are read here.
#
# Binary:
#   canonical BENIGN -> 0
#   every other nonempty canonical label -> 1
#   null/empty -> FAIL CLOSED
#
# Canonicalization:
#   strip outer whitespace
#   normalize common unicode dash characters to "-"
#   collapse internal whitespace
#   casefold for lookup
#
# Unknown non-benign attack labels remain attack for binary evaluation,
# but family assignment becomes OTHER_ATTACK_UNSEEN_LABEL.
# ============================================================================

binary_label_rule = {

    "canonicalization": [
        "STRIP_OUTER_WHITESPACE",
        "NORMALIZE_UNICODE_DASHES_TO_ASCII_HYPHEN",
        "COLLAPSE_INTERNAL_WHITESPACE",
        "CASEFOLD_FOR_LOOKUP",
    ],

    "benign_tokens": [
        "benign",
    ],

    "benign_binary":
        0,

    "all_other_nonempty_labels_binary":
        1,

    "null_or_empty":
        "FAIL_CLOSED",

    "target_label_values_observed_during_lock":
        False,
}


# ============================================================================
# 12. FREEZE ATTACK FAMILY TAXONOMY
# ============================================================================

attack_family_taxonomy = {

    "ftp-patator":
        "AUTH_BRUTE_FORCE",

    "ssh-patator":
        "AUTH_BRUTE_FORCE",

    "dos slowloris":
        "DOS",

    "dos slowhttptest":
        "DOS",

    "dos hulk":
        "DOS",

    "dos goldeneye":
        "DOS",

    "ddos":
        "DDOS",

    "web attack - brute force":
        "WEB_ATTACK",

    "web attack - xss":
        "WEB_ATTACK",

    "web attack - sql injection":
        "WEB_ATTACK",

    "infiltration":
        "INFILTRATION",

    "bot":
        "BOT",

    "portscan":
        "PORT_SCAN",

    "heartbleed":
        "TARGET_ONLY_UNSEEN",
}


family_policy = {

    "unknown_nonbenign_family":
        "OTHER_ATTACK_UNSEEN_LABEL",

    "heartbleed":
        "TARGET_ONLY_UNSEEN",

    "minimum_attack_support_for_inferential_family_reporting":
        50,

    "below_minimum_support":
        "DESCRIPTIVE_ONLY",

    "family_taxonomy_changes_after_target_label_read":
        "FORBIDDEN",
}


print("=" * 108)
print("LABEL / FAMILY TAXONOMY LOCK")
print("=" * 108)

print(
    "Binary benign token:",
    "BENIGN",
)

print(
    "Unknown non-benign label:",
    "attack binary; OTHER_ATTACK_UNSEEN_LABEL family",
)

print(
    "Family inferential minimum attack support:",
    50,
)

print(
    "Heartbleed:",
    "TARGET_ONLY_UNSEEN",
)

print()


# ============================================================================
# 13. FREEZE PRIMARY DIRECTION
#
# IDS2018 -> CICIDS2017
#
# bridge70:
#   reuse exact inherited Stage22 ensemble, no refit.
#
# bridge62:
#   exact bridge-specific source refit because 70-feature frozen model
#   cannot accept 62 dimensions.
#
# bridge62 source refit:
#   same Stage22 CHRONOLOGICAL_NATURAL membership
#   same LGBM/XGB hyperparameters
#   same 0.5 / 0.5 ensemble
#   no tuning
#   no target feedback
# ============================================================================

primary_source_train = {

    "dataset":
        "CSE-CIC-IDS2018",

    "membership":
        "Stage22R CHRONOLOGICAL_NATURAL day_id 0..6",

    "rows":
        int(
            stage22[
                "data"
            ][
                "train"
            ][
                "rows"
            ]
        ),

    "attack":
        int(
            stage22[
                "data"
            ][
                "train"
            ][
                "attack"
            ]
        ),

    "benign":
        int(
            stage22[
                "data"
            ][
                "train"
            ][
                "benign"
            ]
        ),
}


primary_source_validation = {

    "dataset":
        "CSE-CIC-IDS2018",

    "membership":
        "Stage22R CHRONOLOGICAL_NATURAL day_id 7 = 02-28-2018",

    "rows":
        int(
            stage22[
                "data"
            ][
                "validation"
            ][
                "rows"
            ]
        ),

    "attack":
        int(
            stage22[
                "data"
            ][
                "validation"
            ][
                "attack"
            ]
        ),

    "benign":
        int(
            stage22[
                "data"
            ][
                "validation"
            ][
                "benign"
            ]
        ),
}


primary_source_prior = (
    primary_source_train[
        "attack"
    ]
    / primary_source_train[
        "rows"
    ]
)


primary_direction = {

    "direction":
        "IDS2018_TO_CICIDS2017",

    "source_train":
        primary_source_train,

    "source_validation":
        primary_source_validation,

    "target":
        "ALL_FIVE_CICIDS2017_DAYS_UNDER_FROZEN_EFFECTIVE_ROW_RULE",

    "target_effective_rows_expected":
        published_effective_rows,

    "bridge70": {

        "model":
            "EXACT_INHERITED_STAGE22R_CHRONOLOGICAL_NATURAL_ENSEMBLE",

        "refit":
            False,

        "ensemble":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "feature_count":
            70,
    },

    "bridge62": {

        "model":
            "BRIDGE_SPECIFIC_SOURCE_REFIT_OF_FROZEN_STAGE22R_PAIR",

        "refit":
            True,

        "ensemble":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "feature_count":
            62,

        "tuning":
            False,

        "target_feedback":
            False,
    },
}


# ============================================================================
# 14. FREEZE SECONDARY / REVERSE DIRECTION
#
# CICIDS2017 -> IDS2018
#
# TRAIN: Monday + Tuesday + Wednesday
# VALID: Thursday effective population
# UNUSED: Friday
# TARGET: IDS2018 02-28-2018
#
# ONE learner only:
#   XGBoost
#
# Rationale frozen prospectively:
#   - inherited Stage22 learner
#   - native CUDA backend available in Stage24 environment
#   - avoids introducing a second reverse-transfer ensemble degree of freedom
#
# NO hyperparameter search.
# ============================================================================

secondary_direction = {

    "direction":
        "CICIDS2017_TO_IDS2018",

    "source_train_days": [
        "Monday",
        "Tuesday",
        "Wednesday",
    ],

    "source_validation_days": [
        "Thursday",
    ],

    "source_unused_days": [
        "Friday",
    ],

    "target":
        "IDS2018_02-28-2018",

    "target_stage22_clean_membership":
        "day_id 7",

    "target_expected_clean_rows":
        int(
            stage22[
                "data"
            ][
                "validation"
            ][
                "rows"
            ]
        ),

    "learner":
        "XGBOOST_ONLY",

    "learner_configuration":
        "INHERIT_STAGE22R_XGB_11",

    "hyperparameter_search":
        False,

    "target_feedback":
        False,

    "bridge62_fit":
        1,

    "bridge70_fit":
        1,

    "mar01_status":
        "CLOSED_NOT_RESCUE_TARGET",

    "mar02_status":
        "CLOSED_NOT_RESCUE_TARGET",
}


# ============================================================================
# 15. SYNTHETIC-ONLY GPU CAPABILITY PROBES
#
# User preference:
#   boosting algorithm -> GPU whenever supported.
#
# These are NOT scientific fits:
#   - synthetic random data only
#   - no IDS2018 values
#   - no CICIDS2017 values
#   - no target labels
#   - no metrics
#
# If a GPU implementation is unavailable at runtime, CPU fallback is allowed
# ONLY with this explicit receipt.
# ============================================================================

print("=" * 108)
print("SYNTHETIC GPU CAPABILITY PROBES")
print("=" * 108)


rng = np.random.RandomState(
    42
)

X_probe = rng.normal(
    size=(
        2048,
        8,
    )
).astype(
    np.float32
)

y_probe = (
    X_probe[
        :,
        0
    ]
    + 0.25
    * X_probe[
        :,
        1
    ]
    > 0
).astype(
    np.int32
)


gpu_probe = {

    "synthetic_only":
        True,

    "scientific_fit_count":
        0,

    "target_rows_used":
        0,

    "target_labels_used":
        0,

    "lightgbm": {
        "attempted_gpu":
            True,

        "gpu_pass":
            False,

        "exception":
            None,
    },

    "xgboost": {
        "attempted_cuda":
            True,

        "cuda_pass":
            False,

        "exception":
            None,
    },
}


# --------------------------------------------------------------------------
# LightGBM GPU probe
# --------------------------------------------------------------------------

try:

    import lightgbm as lgb


    probe_lgb = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=5,
        learning_rate=0.1,
        num_leaves=7,
        max_depth=3,
        random_state=42,
        device_type="gpu",
        verbosity=-1,
    )


    probe_lgb.fit(
        X_probe,
        y_probe,
    )


    _ = probe_lgb.predict_proba(
        X_probe[
            :16
        ]
    )


    gpu_probe[
        "lightgbm"
    ][
        "gpu_pass"
    ] = True


    print(
        "[PASS] LightGBM synthetic GPU probe"
    )


except Exception as exc:

    gpu_probe[
        "lightgbm"
    ][
        "exception"
    ] = (
        f"{type(exc).__name__}: {exc}"
    )


    print(
        "[FALLBACK] LightGBM GPU probe failed:"
    )

    print(
        " ",
        gpu_probe[
            "lightgbm"
        ][
            "exception"
        ],
    )


# --------------------------------------------------------------------------
# XGBoost CUDA probe
# --------------------------------------------------------------------------

try:

    import xgboost as xgb


    probe_xgb = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=5,
        learning_rate=0.1,
        max_depth=3,
        tree_method="hist",
        device="cuda",
        random_state=42,
        n_jobs=-1,
    )


    probe_xgb.fit(
        X_probe,
        y_probe,
    )


    _ = probe_xgb.predict_proba(
        X_probe[
            :16
        ]
    )


    gpu_probe[
        "xgboost"
    ][
        "cuda_pass"
    ] = True


    print(
        "[PASS] XGBoost synthetic CUDA probe"
    )


except Exception as exc:

    gpu_probe[
        "xgboost"
    ][
        "exception"
    ] = (
        f"{type(exc).__name__}: {exc}"
    )


    print(
        "[FALLBACK] XGBoost CUDA probe failed:"
    )

    print(
        " ",
        gpu_probe[
            "xgboost"
        ][
            "exception"
        ],
    )


print()


# ============================================================================
# 16. FREEZE SCIENTIFIC BACKEND PARAMETERS
# ============================================================================

frozen_lgb_original = copy.deepcopy(
    stage22[
        "models"
    ][
        "lightgbm"
    ][
        "frozen_original_parameters"
    ]
)


frozen_xgb_original = copy.deepcopy(
    stage22[
        "models"
    ][
        "xgboost"
    ][
        "parameters"
    ]
)


if (
    gpu_probe[
        "lightgbm"
    ][
        "gpu_pass"
    ]
):

    stage24_lgb_backend = (
        "gpu"
    )

else:

    stage24_lgb_backend = (
        "cpu"
    )


if (
    gpu_probe[
        "xgboost"
    ][
        "cuda_pass"
    ]
):

    stage24_xgb_device = (
        "cuda"
    )

else:

    stage24_xgb_device = (
        "cpu"
    )


stage24_lgb_parameters = copy.deepcopy(
    frozen_lgb_original
)

stage24_lgb_parameters[
    "device_type"
] = stage24_lgb_backend


stage24_xgb_parameters = copy.deepcopy(
    frozen_xgb_original
)

stage24_xgb_parameters[
    "device"
] = stage24_xgb_device

stage24_xgb_parameters[
    "tree_method"
] = "hist"


print("=" * 108)
print("SCIENTIFIC BOOSTING BACKEND LOCK")
print("=" * 108)

print(
    "LightGBM:",
    stage24_lgb_backend.upper(),
)

print(
    "XGBoost:",
    stage24_xgb_device.upper(),
)

print()

print(
    "[FROZEN] Only runtime backend may differ from inherited hyperparameters."
)

print(
    "[FROZEN] No algorithmic hyperparameter search."
)
print()


# ============================================================================
# 17. FREEZE SCIENTIFIC FIT BUDGET
#
# PRIMARY:
#   bridge70 = zero fits, exact inherited 70F model
#   bridge62 = 1 LGBM + 1 XGB
#
# SECONDARY:
#   bridge62 = 1 XGB
#   bridge70 = 1 XGB
#
# TOTAL = 4 scientific fits.
#
# Synthetic GPU probes above are infrastructure probes and explicitly do not
# consume scientific fit budget.
# ============================================================================

scientific_fit_budget = {

    "primary_IDS2018_to_CICIDS2017": {

        "bridge70_lightgbm":
            0,

        "bridge70_xgboost":
            0,

        "bridge62_lightgbm":
            1,

        "bridge62_xgboost":
            1,

        "total":
            2,
    },

    "secondary_CICIDS2017_to_IDS2018": {

        "bridge62_xgboost":
            1,

        "bridge70_xgboost":
            1,

        "total":
            2,
    },

    "grand_total":
        4,

    "additional_fit_budget":
        0,

    "hyperparameter_search":
        0,

    "target_guided_refits":
        0,

    "synthetic_backend_probes_count_as_scientific_fits":
        False,
}


if (
    scientific_fit_budget[
        "grand_total"
    ]
    != 4
):

    raise RuntimeError(
        "Scientific fit budget arithmetic failed."
    )


print("=" * 108)
print("SCIENTIFIC FIT BUDGET")
print("=" * 108)

print(
    "Primary bridge62:",
    "2 fits (LGBM + XGB)",
)

print(
    "Primary bridge70:",
    "0 fits (reuse exact frozen Stage22 ensemble)",
)

print(
    "Secondary bridge62:",
    "1 XGB fit",
)

print(
    "Secondary bridge70:",
    "1 XGB fit",
)

print()

print(
    "TOTAL:",
    scientific_fit_budget[
        "grand_total"
    ],
)

print(
    "Additional fits:",
    0,
)

print()


# ============================================================================
# 18. FREEZE THRESHOLD SELECTION
# ============================================================================

threshold_grid = list(
    stage20_threshold[
        "evaluation"
    ][
        "threshold_grid"
    ][
        "integer_percent_values"
    ]
)


if threshold_grid != list(
    range(
        5,
        96,
    )
):

    raise RuntimeError(
        "Frozen Stage20 threshold grid changed."
    )


threshold_selection = copy.deepcopy(
    stage20_threshold[
        "selection"
    ]
)


threshold_protocol = {

    "grid_integer_percent":
        threshold_grid,

    "runtime_threshold":
        "integer_percent / 100.0 cast consistently with persisted probabilities",

    "balanced":
        threshold_selection[
            "balanced"
        ],

    "security":
        threshold_selection[
            "security"
        ],

    "standard":
        threshold_selection[
            "standard"
        ],

    "selection_data":
        "SOURCE_VALIDATION_ONLY",

    "target_threshold_search":
        "FORBIDDEN",

    "target_threshold_recalibration":
        "FORBIDDEN",
}


print("=" * 108)
print("SOURCE-ONLY THRESHOLD LOCK")
print("=" * 108)

print(
    "Grid:",
    "0.05 .. 0.95 inclusive, step 0.01",
)

print(
    "Balanced:",
    "maximum source-validation F1",
)

print(
    "Security:",
    "maximum source-validation F2 subject to FPR <= 0.05",
)

print(
    "Standard:",
    "fixed 0.50",
)

print(
    "Target threshold tuning:",
    "FORBIDDEN",
)

print()


# ============================================================================
# 19. SOURCE SANITY SEQUENCE
# ============================================================================

source_sanity = {

    "sequence": [
        "FULL_SOURCE",
        "BRIDGE_SOURCE",
        "CROSS_DATASET_TARGET",
    ],

    "primary_full_source": {

        "representation":
            "Stage22 exact 70F",

        "model":
            "exact frozen Stage22 ensemble",

        "validation":
            "Stage22 CHRONOLOGICAL_NATURAL day_id 7",

        "performance_based_stop":
            False,
    },

    "primary_bridge62_source": {

        "representation":
            "bridge62",

        "model":
            "frozen-hyperparameter Stage22 LGBM+XGB refit",

        "validation":
            "same Stage22 CHRONOLOGICAL_NATURAL day_id 7",

        "performance_based_stop":
            False,
    },

    "integrity_gates_only": [
        "FEATURE_ORDER_EXACT",
        "EXPECTED_DIMENSION",
        "MODEL_LOAD_OR_FIT_SUCCESS",
        "PREDICTION_VECTOR_LENGTH_EXACT",
        "PROBABILITIES_NUMERIC_OR_EXPLICIT_NAN_FAILURE",
        "NO_TARGET_FEEDBACK",
    ],

    "low_source_performance_rule": (
        "REPORT_AS_RESULT; DO_NOT CHANGE MODEL, FEATURES, THRESHOLD GRID, "
        "OR TARGET EXECUTION PLAN"
    ),
}


# ============================================================================
# 20. NUMERIC POLICY
# ============================================================================

numeric_policy = {

    "parse_dtype":
        "float64",

    "positive_infinity":
        "CONVERT_TO_NAN",

    "negative_infinity":
        "CONVERT_TO_NAN",

    "other_nonnumeric_token":
        "FAIL_CLOSED",

    "explicit_imputation":
        "NONE",

    "scaling":
        "NONE",

    "target_fitted_scaler":
        "FORBIDDEN",

    "target_fitted_imputer":
        "FORBIDDEN",
}


# ============================================================================
# 21. METRIC LOCK
# ============================================================================

metric_protocol = {

    "primary_metric":
        "PR_AUC",

    "co_primary_metric":
        "ROC_AUC",

    "auc_semantics": {

        "PR_AUC":
            stage20_threshold[
                "metric_definitions"
            ][
                "PR_AUC"
            ],

        "ROC_AUC":
            stage20_threshold[
                "metric_definitions"
            ][
                "ROC_AUC"
            ],

        "ties":
            stage20_threshold[
                "metric_definitions"
            ][
                "tie_handling_for_auc"
            ],
    },

    "thresholded_descriptive_metrics": [
        "accuracy",
        "precision",
        "recall",
        "F1",
        "F2",
        "FPR",
        "FNR",
        "TP",
        "TN",
        "FP",
        "FN",
    ],

    "prevalence": {

        "symbol":
            "pi",

        "formula":
            "n_attack / n_total",

        "PR_chance_baseline":
            "pi",

        "PR_excess":
            "PR_AUC - pi",

        "PR_normalized":
            "(PR_AUC - pi) / (1 - pi) when pi < 1",
    },

    "report_directions_separately":
        True,

    "average_primary_and_secondary_directions":
        False,
}


# ============================================================================
# 22. ZERO-FIT CALIBRATION TRANSPORTABILITY
# ============================================================================

calibration_protocol = {

    "recalibration":
        "FORBIDDEN",

    "target_fitted_calibrator":
        False,

    "metrics": {

        "brier_score":
            "mean((p-y)^2)",

        "log_loss":
            "binary cross entropy with probability clipping only for numerical log evaluation",

        "log_loss_clip":
            1e-15,

        "ECE": {

            "bins":
                10,

            "binning":
                "fixed equal-width bins [0.0,0.1),...,[0.9,1.0]",

            "fitted":
                False,
        },
    },
}


# ============================================================================
# 23. NULL / CHANCE BASELINES
# ============================================================================

null_protocol = {

    "target_prevalence_chance_anchor": {

        "score_probability":
            "pi_target",

        "PR_AUC_anchor":
            "pi_target",

        "ROC_AUC_anchor":
            0.5,
    },

    "source_prior_constant_predictor": {

        "primary_source_probability":
            primary_source_prior,

        "secondary_source_probability":
            "COMPUTE_ON_FROZEN_CICIDS2017_MONDAY_WEDNESDAY_SOURCE_TRAIN_ONLY",

        "fit":
            False,
    },
}


# ============================================================================
# 24. PAIRED BOOTSTRAP LOCK
# ============================================================================

bootstrap_protocol = {

    "replicates":
        2000,

    "seed":
        42,

    "method":
        "PAIRED_STRATIFIED_ROW_BOOTSTRAP",

    "pairing":
        "SAME_RESAMPLED_ROW_INDICES_FOR_BOTH_COMPARED_PREDICTION_VECTORS",

    "strata":
        [
            "BENIGN",
            "ATTACK",
        ],

    "sampling":
        "WITH_REPLACEMENT_WITHIN_EACH_STRATUM_PRESERVING_ORIGINAL_STRATUM_SIZE",

    "confidence_interval":
        "PERCENTILE_2.5_97.5",

    "metrics_for_paired_difference": [
        "PR_AUC",
        "ROC_AUC",
        "Brier",
    ],

    "minimum_attack_support":
        50,

    "single_class_bootstrap_draw":
        "IMPOSSIBLE_BY_STRATIFIED_DESIGN_WHEN_BOTH_CLASSES_EXIST",
}


# ============================================================================
# 25. ARTIFACT / SELECTION SHIFT COMPARISON LOCK
# ============================================================================

comparison_protocol = {

    "bridge70_artifact_shift": {

        "comparison":
            "PUBLISHED_vs_FLAG_CORRECTED",

        "population":
            "IDENTICAL_FULL_EFFECTIVE_TARGET_ROWS",

        "paired":
            True,

        "purpose":
            "isolate aggregate TCP-flag serialization artifact sensitivity",
    },

    "bridge62_flag_invariance": {

        "comparison":
            "PUBLISHED_vs_FLAG_CORRECTED",

        "expected":
            "BITWISE_IDENTICAL_FEATURE_MATRICES_AND_PREDICTIONS",

        "reason":
            "all eight aggregate flag-count fields are absent from bridge62",

        "violation":
            "IMPLEMENTATION_FAILURE",
    },

    "grounding_selection_shift": {

        "comparison":
            "FLAG_CORRECTED_FULL_vs_GROUNDED_S4",

        "feature_semantics":
            "IDENTICAL_FLAG_CORRECTED_SEMANTICS",

        "population_difference":
            "FULL_EFFECTIVE_vs_STAGE20_EXACT_GROUNDED_SUBSET",

        "purpose":
            "isolate grounding/selection shift",
    },

    "grounded_prediction_identity": {

        "expected":
            (
                "For the same bridge and same grounded rows, "
                "GROUNDED_S4 predictions must equal FLAG_CORRECTED "
                "full-population predictions restricted to those rows."
            ),

        "violation":
            "IMPLEMENTATION_FAILURE",
    },
}


# ============================================================================
# 26. OPENING LEDGER — EXACTLY 8
# ============================================================================

opening_ledger = {

    "PRIMARY_IDS2018_TO_CICIDS2017": {

        "bridge62": {

            "PUBLISHED": {
                "consumed": 0,
                "budget": 1,
                "status": "CLOSED",
            },

            "FLAG_CORRECTED": {
                "consumed": 0,
                "budget": 1,
                "status": "CLOSED",
            },

            "GROUNDED_S4": {
                "consumed": 0,
                "budget": 1,
                "status": "CLOSED",
            },
        },

        "bridge70": {

            "PUBLISHED": {
                "consumed": 0,
                "budget": 1,
                "status": "CLOSED",
            },

            "FLAG_CORRECTED": {
                "consumed": 0,
                "budget": 1,
                "status": "CLOSED",
            },

            "GROUNDED_S4": {
                "consumed": 0,
                "budget": 1,
                "status": "CLOSED",
            },
        },
    },

    "SECONDARY_CICIDS2017_TO_IDS2018": {

        "bridge62": {

            "FEB28_2018": {
                "consumed": 0,
                "budget": 1,
                "status": "CLOSED",
            },
        },

        "bridge70": {

            "FEB28_2018": {
                "consumed": 0,
                "budget": 1,
                "status": "CLOSED",
            },
        },
    },
}


opening_cells = []


for direction, bridges in (
    opening_ledger.items()
):

    for bridge, variants in (
        bridges.items()
    ):

        for variant, state in (
            variants.items()
        ):

            opening_cells.append(
                (
                    direction,
                    bridge,
                    variant,
                    state,
                )
            )


if len(
    opening_cells
) != 8:

    raise RuntimeError(
        "Opening ledger does not contain exactly 8 cells."
    )


if sum(
    state[
        "budget"
    ]
    for _, _, _, state
    in opening_cells
) != 8:

    raise RuntimeError(
        "Opening budget != 8."
    )


if sum(
    state[
        "consumed"
    ]
    for _, _, _, state
    in opening_cells
) != 0:

    raise RuntimeError(
        "Opening ledger is not pristine."
    )


# ============================================================================
# 27. EXECUTION ORDER LOCK
# ============================================================================

execution_order = [

    {
        "step": 1,
        "action": (
            "COMMIT_PUSH_REMOTE_VERIFY_STAGE24_0_PROTOCOL_LOCK"
        ),
    },

    {
        "step": 2,
        "action": (
            "PRIMARY_FULL_SOURCE_70F_SANITY_WITH_FROZEN_STAGE22_MODELS"
        ),
    },

    {
        "step": 3,
        "action": (
            "PRIMARY_BRIDGE62_SOURCE_REFIT_AND_SOURCE_VALIDATION_ONLY_THRESHOLD_SELECTION"
        ),
    },

    {
        "step": 4,
        "action": (
            "RECONSTRUCT_OR_RECOVER_GROUNDED_S4_MEMBERSHIP_USING_ONLY_FROZEN_STAGE20_EXACT_JOIN"
        ),
    },

    {
        "step": 5,
        "action": (
            "OPEN_PRIMARY_TARGET_CELLS_UNDER_FROZEN_LEDGER"
        ),
    },

    {
        "step": 6,
        "action": (
            "FREEZE_PRIMARY_RESULTS_NO_PROTOCOL_CHANGES"
        ),
    },

    {
        "step": 7,
        "action": (
            "TRAIN_SECONDARY_XGBOOST_SOURCE_MODELS_ON_CICIDS2017_MON_WED_AND_VALIDATE_THURSDAY"
        ),
    },

    {
        "step": 8,
        "action": (
            "OPEN_SECONDARY_IDS2018_FEB28_TARGET_CELLS"
        ),
    },

    {
        "step": 9,
        "action": (
            "FINAL_STAGE24_SYNTHESIS_WITH_DIRECTIONS_REPORTED_SEPARATELY"
        ),
    },
]


# ============================================================================
# 28. ANTI-ADAPTATION LOCK
# ============================================================================

anti_adaptation = {

    "after_protocol_lock_commit": {

        "feature_addition":
            "FORBIDDEN",

        "feature_removal":
            "FORBIDDEN",

        "feature_remapping":
            "FORBIDDEN",

        "flag_mapping_search":
            "FORBIDDEN",

        "label_guided_mapping":
            "FORBIDDEN",

        "target_guided_imputation":
            "FORBIDDEN",

        "target_guided_scaling":
            "FORBIDDEN",

        "target_guided_threshold":
            "FORBIDDEN",

        "target_guided_hyperparameter_change":
            "FORBIDDEN",

        "target_guided_refit":
            "FORBIDDEN",

        "post_result_subset_creation":
            "FORBIDDEN",

        "family_taxonomy_change":
            "FORBIDDEN",

        "grounding_tolerance_relaxation":
            "FORBIDDEN",

        "rescue_target":
            "FORBIDDEN",
    },

    "unexpected_results":
        "REPORT_AS_RESULTS",

    "poor_transfer":
        "REPORT_AS_RESULTS",

    "failed_family_transfer":
        "REPORT_AS_RESULTS",

    "bridge62_equals_or_beats_bridge70":
        "REPORT_AS_RESULTS",

    "published_beats_corrected":
        "REPORT_AS_RESULTS",

    "corrected_beats_published":
        "REPORT_AS_RESULTS",

    "grounded_cohort_changes_conclusion":
        "REPORT_AS_RESULTS",
}


# ============================================================================
# 29. SEMANTIC BRIDGE SPEC ARTIFACT
# ============================================================================

semantic_bridge_spec = {

    "stage":
        "Stage24-0C",

    "type":
        "FROZEN_CROSS_DATASET_SEMANTIC_BRIDGE_SPEC",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent_head":
        head,

    "source_dataset":
        "CSE-CIC-IDS2018",

    "target_dataset":
        "CICIDS2017",

    "stage22_source_feature_order":
        stage22_features,

    "bridge70": {

        "feature_count":
            70,

        "source_feature_order":
            bridge70_features,

        "PUBLISHED_target_physical_order":
            published_order70,

        "FLAG_CORRECTED_target_physical_order":
            corrected_order70,

        "GROUNDED_S4_target_physical_order":
            grounded_order70,

        "PUBLISHED_mapping":
            published_map70,

        "FLAG_CORRECTED_mapping":
            corrected_map70,

        "GROUNDED_S4_mapping":
            grounded_map70,
    },

    "bridge62": {

        "feature_count":
            62,

        "source_feature_order":
            bridge62_features,

        "excluded_aggregate_flag_features":
            FLAG_SOURCE_FEATURES,

        "PUBLISHED_mapping":
            published_map62,

        "FLAG_CORRECTED_mapping":
            corrected_map62,

        "GROUNDED_S4_mapping":
            grounded_map62,

        "published_equals_flag_corrected":
            True,
    },

    "source_nominal_flag_semantics":
        SOURCE_FLAG_SEMANTICS,

    "stage20_target_physical_to_semantic_flag":
        physical_to_semantic_flag,

    "stage20_target_semantic_to_physical_flag":
        semantic_to_physical_flag,

    "common_target_physical_schema":
        common_physical_schema,

    "common_target_physical_schema_sha256":
        next(
            iter(
                schema_hashes
            )
        ),

    "mapping_search_performed":
        False,

    "fuzzy_mapping":
        False,

    "target_values_read_to_build_mapping":
        False,

    "target_labels_read_to_build_mapping":
        False,
}


semantic_path = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.json"
)


semantic_sha, semantic_sha_path = (
    write_json_with_sha(
        semantic_path,
        semantic_bridge_spec,
    )
)


# ============================================================================
# 30. GPU BACKEND PROBE ARTIFACT
# ============================================================================

gpu_probe.update(
    {
        "stage":
            "Stage24-0C",

        "type":
            "SYNTHETIC_ONLY_GPU_BACKEND_CAPABILITY_PROBE",

        "created_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "scientific_parent_head":
            head,

        "selected_scientific_lightgbm_backend":
            stage24_lgb_backend,

        "selected_scientific_xgboost_device":
            stage24_xgb_device,

        "scientific_lightgbm_parameters":
            stage24_lgb_parameters,

        "scientific_xgboost_parameters":
            stage24_xgb_parameters,

        "scientific_fit_budget_consumed":
            0,
    }
)


gpu_probe_path = (
    LOCK_DIR
    / "stage24_0c_gpu_backend_probe.json"
)


gpu_probe_sha, gpu_probe_sha_path = (
    write_json_with_sha(
        gpu_probe_path,
        gpu_probe,
    )
)


# ============================================================================
# 31. FINAL STAGE24-0 PROTOCOL LOCK
# ============================================================================

protocol_lock = {

    "stage":
        "Stage24-0C",

    "type":
        "FINAL_PRE_TARGET_OPENING_PROTOCOL_LOCK",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent": {

        "repository_head":
            head,

        "required_prelock_head":
            EXPECTED_HEAD,
    },

    "provenance": {

        "stage24_0b2_receipt":
            str(
                STAGE24_0B2_RECEIPT.relative_to(
                    REPO_DIR
                )
            ),

        "stage24_0b2_sha256":
            receipt_0b2_sha,

        "cicids2017_exact_artifacts":
            8,

        "all_eight_byte_exact":
            True,

        "common_physical_schema_sha256":
            next(
                iter(
                    schema_hashes
                )
            ),
    },

    "semantic_bridge": {

        "artifact":
            str(
                semantic_path.relative_to(
                    REPO_DIR
                )
            ),

        "sha256":
            semantic_sha,

        "bridge62_feature_count":
            62,

        "bridge70_feature_count":
            70,
    },

    "target_variants": {

        "PUBLISHED": {

            "population":
                "FULL_EFFECTIVE_CICIDS2017_POPULATION",

            "feature_semantics":
                "PUBLISHED_LITERAL_HEADERS",

            "effective_rows_expected":
                published_effective_rows,
        },

        "FLAG_CORRECTED": {

            "population":
                "SAME_FULL_EFFECTIVE_CICIDS2017_POPULATION_AS_PUBLISHED",

            "feature_semantics":
                "STAGE20_1C8_SOURCE_DERIVED_FLAG_SERIALIZATION_CORRECTION",

            "effective_rows_expected":
                published_effective_rows,
        },

        "GROUNDED_S4": {

            "population":
                "STAGE20_EXACT_SOURCE_FAITHFUL_GROUNDED_MATCHED_SUBSET",

            "feature_semantics":
                "FLAG_CORRECTED",

            "expected_matched_flow_count_from_frozen_stage20_manifests":
                expected_grounded_rows,

            "membership_rule":
                (
                    "REUSE_EXACT_ROW_MEMBERSHIP_IF_BYTE_VERIFIABLE; "
                    "OTHERWISE REPLAY FROZEN_STAGE20_EXACT_JOIN USING "
                    "EXACT_PINNED_PCAPS; NO APPROXIMATION"
                ),
        },
    },

    "published_population":
        population_records,

    "thursday_ingestion_erratum": {

        "inherited":
            True,

        "effective_morning_record_ordinals":
            "1..170366",

        "excluded_morning_physical_ordinals":
            "170367..458968",

        "classification":
            "NON_RECORD_STRUCTURAL_NULL_PADDING",

        "rule_frozen_before_stage24_target_read":
            True,
    },

    "grounded_s4":
        grounded_counts,

    "binary_label_rule":
        binary_label_rule,

    "attack_family_taxonomy":
        attack_family_taxonomy,

    "family_policy":
        family_policy,

    "primary_direction":
        primary_direction,

    "secondary_direction":
        secondary_direction,

    "source_sanity":
        source_sanity,

    "numeric_policy":
        numeric_policy,

    "boosting_backend": {

        "probe_artifact":
            str(
                gpu_probe_path.relative_to(
                    REPO_DIR
                )
            ),

        "probe_sha256":
            gpu_probe_sha,

        "lightgbm":
            stage24_lgb_backend,

        "xgboost":
            stage24_xgb_device,

        "gpu_first":
            True,

        "cpu_fallback_requires_probe_failure":
            True,
    },

    "scientific_fit_budget":
        scientific_fit_budget,

    "threshold_protocol":
        threshold_protocol,

    "metric_protocol":
        metric_protocol,

    "calibration_protocol":
        calibration_protocol,

    "null_protocol":
        null_protocol,

    "bootstrap_protocol":
        bootstrap_protocol,

    "comparison_protocol":
        comparison_protocol,

    "opening_definition": (
        "A target opening occurs when target feature values are first "
        "supplied to the frozen source-domain model for the corresponding "
        "direction / bridge / target-variant cell."
    ),

    "opening_ledger":
        opening_ledger,

    "opening_summary": {

        "consumed":
            0,

        "budget":
            8,

        "all_closed":
            True,
    },

    "execution_order":
        execution_order,

    "anti_adaptation":
        anti_adaptation,

    "scientific_access_during_stage24_0c": {

        "cicids2017_target_row_groups_read":
            0,

        "cicids2017_target_feature_rows_materialized":
            0,

        "cicids2017_target_labels_materialized":
            0,

        "cicids2017_target_prevalence_computed":
            False,

        "ids2018_target_rows_read_for_secondary":
            0,

        "scientific_model_fits":
            0,

        "target_model_predictions":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,

        "synthetic_gpu_probe_only":
            True,
    },

    "target_inference_authorization": {

        "current_status":
            "FORBIDDEN",

        "reason":
            (
                "Protocol lock has been written locally but has not yet "
                "been committed, pushed, and remotely verified."
            ),

        "next_required_gate":
            "STAGE24_0D_COMMIT_PUSH_REMOTE_VERIFY",
    },
}


protocol_path = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.json"
)


protocol_sha, protocol_sha_path = (
    write_json_with_sha(
        protocol_path,
        protocol_lock,
    )
)


# ============================================================================
# 32. FINAL INTERNAL VERIFICATION
# ============================================================================

if (
    len(
        semantic_bridge_spec[
            "bridge70"
        ][
            "source_feature_order"
        ]
    )
    != 70
):

    raise RuntimeError(
        "Final bridge70 lock corrupted."
    )


if (
    len(
        semantic_bridge_spec[
            "bridge62"
        ][
            "source_feature_order"
        ]
    )
    != 62
):

    raise RuntimeError(
        "Final bridge62 lock corrupted."
    )


if (
    protocol_lock[
        "opening_summary"
    ][
        "consumed"
    ]
    != 0
):

    raise RuntimeError(
        "Opening ledger corrupted."
    )


if (
    protocol_lock[
        "scientific_fit_budget"
    ][
        "grand_total"
    ]
    != 4
):

    raise RuntimeError(
        "Fit budget corrupted."
    )


if (
    protocol_lock[
        "target_variants"
    ][
        "PUBLISHED"
    ][
        "effective_rows_expected"
    ]
    != 2_830_743
):

    raise RuntimeError(
        "Published target population corrupted."
    )


# ============================================================================
# 33. FINAL OUTPUT
# ============================================================================

print()
print("=" * 108)
print("STAGE24-0C FINAL SYNTHESIS")
print("=" * 108)

print(
    "Scientific parent HEAD:"
)

print(
    " ",
    head,
)

print()

print(
    "CICIDS2017 exact source artifacts:"
)

print(
    "  8 / 8"
)

print(
    "Effective PUBLISHED / FLAG_CORRECTED population:"
)

print(
    " ",
    published_effective_rows,
)

print(
    "Frozen GROUNDED_S4 expected matched-flow count:"
)

print(
    " ",
    expected_grounded_rows,
)

print()

print(
    "bridge62:"
)

print(
    "  62 features"
)

print(
    "bridge70:"
)

print(
    "  70 features"
)

print()

print(
    "bridge62 PUBLISHED == FLAG_CORRECTED representation:"
)

print(
    "  True"
)

print()

print(
    "Primary source:"
)

print(
    "  IDS2018 Stage22R CHRONOLOGICAL_NATURAL"
)

print(
    "  train = day_id 0..6"
)

print(
    "  validation = day_id 7 (02-28-2018)"
)

print()

print(
    "Primary bridge70:"
)

print(
    "  exact inherited Stage22 LGBM+XGB ensemble"
)

print(
    "Primary bridge62:"
)

print(
    "  frozen-hyperparameter bridge-specific LGBM+XGB refit"
)

print()

print(
    "Secondary source:"
)

print(
    "  CICIDS2017 Monday-Wednesday train"
)

print(
    "  Thursday validation"
)

print(
    "  Friday unused"
)

print(
    "Secondary target:"
)

print(
    "  IDS2018 02-28-2018"
)

print(
    "Secondary learner:"
)

print(
    "  XGBoost only"
)

print()

print(
    "Scientific LightGBM backend:"
)

print(
    " ",
    stage24_lgb_backend.upper(),
)

print(
    "Scientific XGBoost device:"
)

print(
    " ",
    stage24_xgb_device.upper(),
)

print()

print(
    "Scientific fit budget:"
)

print(
    "  4 total"
)

print(
    "Scientific fits consumed:"
)

print(
    "  0"
)

print()

print(
    "Threshold search:"
)

print(
    "  SOURCE VALIDATION ONLY"
)

print(
    "Target threshold tuning:"
)

print(
    "  FORBIDDEN"
)

print()

print(
    "Primary metric:"
)

print(
    "  PR-AUC"
)

print(
    "Co-primary metric:"
)

print(
    "  ROC-AUC"
)

print(
    "Bootstrap:"
)

print(
    "  2000 paired stratified replicates, seed 42"
)

print()

print(
    "Target rows materialized:"
)

print(
    "  0"
)

print(
    "Target labels materialized:"
)

print(
    "  0"
)

print(
    "Target predictions:"
)

print(
    "  0"
)

print(
    "Target metrics:"
)

print(
    "  0"
)

print(
    "Target openings:"
)

print(
    "  0 / 8"
)

print()

print(
    "Semantic bridge artifact:"
)

print(
    " ",
    semantic_path,
)

print(
    "Semantic bridge SHA256:"
)

print(
    " ",
    semantic_sha,
)

print()

print(
    "GPU probe artifact:"
)

print(
    " ",
    gpu_probe_path,
)

print(
    "GPU probe SHA256:"
)

print(
    " ",
    gpu_probe_sha,
)

print()

print(
    "FINAL protocol lock:"
)

print(
    " ",
    protocol_path,
)

print(
    "FINAL protocol SHA256:"
)

print(
    " ",
    protocol_sha,
)

print()

print("-" * 108)

print(
    "STAGE24-0C: PASS"
)

print(
    "Semantic bridge frozen."
)

print(
    "Population / variant rules frozen."
)

print(
    "Model/refit budget frozen."
)

print(
    "GPU-first backend policy frozen."
)

print(
    "Threshold / metrics / bootstrap / anti-adaptation rules frozen."
)

print(
    "Target model openings remain 0 / 8."
)

print()

print(
    "TARGET INFERENCE IS STILL FORBIDDEN."
)

print()

print(
    "NEXT REQUIRED GATE:"
)

print(
    "Stage24-0D — commit + push + remote SHA verification."
)

print("-" * 108)

print(
    "STAGE24-0C COMPLETE"
)

print("=" * 108)

In [ ]:
# ============================================================================
# STAGE24-0D — COMMIT + PUSH + REMOTE BYTE VERIFICATION
#
# FINAL GOVERNANCE GATE BEFORE ANY TARGET MODEL OPENING
#
# This cell:
#   1. validates Stage24-0C lock integrity
#   2. validates scientific parent
#   3. stages ONLY results/stage24_cross_dataset/**
#   4. creates immutable protocol-lock commit
#   5. pushes main
#   6. verifies remote HEAD == local lock commit
#   7. downloads every committed Stage24 artifact from the immutable
#      GitHub commit and verifies byte-for-byte SHA256
#   8. writes Stage24-0D remote-verification receipt
#   9. commits + pushes that attestation
#  10. verifies final remote HEAD + remote attestation bytes
#  11. requires clean worktree
#
# NO:
#   - target rows
#   - target labels
#   - scientific fits
#   - model inference
#   - predictions
#   - metrics
#
# TARGET OPENINGS BEFORE: 0 / 8
# TARGET OPENINGS AFTER:  0 / 8
#
# On PASS:
#   Stage24 protocol is legally OPEN for execution under the frozen ledger.
# ============================================================================

from __future__ import annotations

import os
import json
import base64
import hashlib
import subprocess
import urllib.request
import urllib.parse
from pathlib import Path
from datetime import datetime, timezone


# ============================================================================
# 0. CONSTANTS
# ============================================================================

REPO_DIR = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

REPO_FULL_NAME = (
    "themubasshir/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "963b2da0043c7438624916280de154a5628f8879"
)

BRANCH = "main"

STAGE24_REL = (
    Path("results")
    / "stage24_cross_dataset"
)

STAGE24_DIR = (
    REPO_DIR
    / STAGE24_REL
)

LOCK_DIR = (
    STAGE24_DIR
    / "stage24_0_protocol_lock"
)


PROTOCOL_LOCK = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.json"
)

PROTOCOL_LOCK_SHA = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.sha256"
)

SEMANTIC_BRIDGE = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.json"
)

SEMANTIC_BRIDGE_SHA = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.sha256"
)

GPU_PROBE = (
    LOCK_DIR
    / "stage24_0c_gpu_backend_probe.json"
)

GPU_PROBE_SHA = (
    LOCK_DIR
    / "stage24_0c_gpu_backend_probe.sha256"
)

B2_RECEIPT = (
    LOCK_DIR
    / "stage24_0b2_complete_cicids2017_source_contract.json"
)

B2_RECEIPT_SHA = (
    LOCK_DIR
    / "stage24_0b2_complete_cicids2017_source_contract.sha256"
)


ATTESTATION = (
    LOCK_DIR
    / "stage24_0d_remote_verification_receipt.json"
)

ATTESTATION_SHA = (
    LOCK_DIR
    / "stage24_0d_remote_verification_receipt.sha256"
)


LOCK_COMMIT_MESSAGE = (
    "stage24: freeze cross-dataset pre-opening protocol"
)

ATTESTATION_COMMIT_MESSAGE = (
    "stage24: record pre-opening remote verification"
)


print("=" * 110)
print("STAGE24-0D — COMMIT + PUSH + REMOTE BYTE VERIFICATION")
print("=" * 110)

print("Target rows:             FORBIDDEN")
print("Target labels:           FORBIDDEN")
print("Scientific model fits:   0")
print("Target predictions:      0")
print("Target openings:         0 / 8")
print()


# ============================================================================
# 1. HELPERS
# ============================================================================

def git(
    *args,
    check=True,
    auth_header=None,
):
    command = [
        "git",
    ]

    if auth_header is not None:

        command.extend(
            [
                "-c",
                "credential.helper=",
                "-c",
                f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
            ]
        )

    command.extend(
        args
    )

    proc = subprocess.run(
        command,
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if (
        check
        and proc.returncode != 0
    ):
        raise RuntimeError(
            "\nGit command failed:\n"
            f"  git {' '.join(args)}\n\n"
            f"{proc.stdout}"
        )

    return proc.stdout.strip()


def load_json(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def sha256_bytes(data: bytes):
    return hashlib.sha256(
        data
    ).hexdigest()


def sha256_file(
    path: Path,
    chunk_size=8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def verify_sha_sidecar(
    artifact: Path,
    sidecar: Path,
):
    if not artifact.exists():
        raise FileNotFoundError(
            artifact
        )

    if not sidecar.exists():
        raise FileNotFoundError(
            sidecar
        )

    declared = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
    )

    actual = sha256_file(
        artifact
    )

    if declared != actual:

        raise RuntimeError(
            f"\nSHA256 sidecar failure:\n"
            f"Artifact: {artifact}\n"
            f"Declared: {declared}\n"
            f"Actual:   {actual}"
        )

    return actual


def remote_bytes(
    commit_sha: str,
    relative_path: str,
):
    encoded_path = urllib.parse.quote(
        relative_path,
        safe="/",
    )

    url = (
        "https://raw.githubusercontent.com/"
        f"{REPO_FULL_NAME}/"
        f"{commit_sha}/"
        f"{encoded_path}"
    )

    request = urllib.request.Request(
        url,
        headers={
            "User-Agent":
                "stage24-governance-verifier"
        },
    )

    try:

        with urllib.request.urlopen(
            request,
            timeout=60,
        ) as response:

            return response.read()

    except Exception as exc:

        raise RuntimeError(
            "\nUnable to retrieve immutable GitHub artifact:\n"
            f"{url}\n"
            f"{type(exc).__name__}: {exc}"
        ) from exc


def relative_repo_path(
    path: Path,
):
    return str(
        path.relative_to(
            REPO_DIR
        )
    ).replace(
        "\\",
        "/",
    )


def write_json_with_sha(
    path: Path,
    obj,
):
    with path.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )

        f.write("\n")

    digest = sha256_file(
        path
    )

    sidecar = path.with_suffix(
        ".sha256"
    )

    sidecar.write_text(
        f"{digest}  {path.name}\n",
        encoding="utf-8",
    )

    return digest


# ============================================================================
# 2. GITHUB TOKEN — NEVER PRINT TOKEN
# ============================================================================

print("=" * 110)
print("GITHUB AUTHENTICATION")
print("=" * 110)


github_token = None
token_source = None


# --------------------------------------------------------------------------
# Kaggle secrets first
# --------------------------------------------------------------------------

try:

    from kaggle_secrets import (
        UserSecretsClient,
    )

    secret_client = (
        UserSecretsClient()
    )

    for candidate in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = (
                secret_client
                .get_secret(
                    candidate
                )
            )

            if (
                value
                and value.strip()
            ):

                github_token = (
                    value.strip()
                )

                token_source = (
                    candidate
                )

                break

        except Exception:
            pass

except Exception:
    pass


# --------------------------------------------------------------------------
# Environment fallback
# --------------------------------------------------------------------------

if github_token is None:

    for candidate in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            candidate
        )

        if (
            value
            and value.strip()
        ):

            github_token = (
                value.strip()
            )

            token_source = (
                f"ENV:{candidate}"
            )

            break


if github_token is None:

    raise RuntimeError(
        "\nGitHub token not found.\n"
        "Expected Kaggle Secret GITHUB_TOKEN."
    )


# GitHub HTTPS Basic auth:
# username = x-access-token
# password = PAT
basic_payload = (
    "x-access-token:"
    + github_token
).encode(
    "utf-8"
)

AUTH_HEADER = (
    base64.b64encode(
        basic_payload
    )
    .decode(
        "ascii"
    )
)


print(
    "GitHub credential source:",
    token_source,
)

print(
    "Credential length:",
    len(
        github_token
    ),
)

print(
    "[PASS] Credential loaded without disclosure."
)
print()


# ============================================================================
# 3. VERIFY LOCAL 0C ARTIFACTS
# ============================================================================

print("=" * 110)
print("LOCAL STAGE24-0C INTEGRITY")
print("=" * 110)


critical_artifacts = [
    (
        "Stage24-0B2",
        B2_RECEIPT,
        B2_RECEIPT_SHA,
    ),

    (
        "Semantic bridge",
        SEMANTIC_BRIDGE,
        SEMANTIC_BRIDGE_SHA,
    ),

    (
        "GPU backend probe",
        GPU_PROBE,
        GPU_PROBE_SHA,
    ),

    (
        "Final protocol lock",
        PROTOCOL_LOCK,
        PROTOCOL_LOCK_SHA,
    ),
]


critical_hashes = {}


for label, artifact, sidecar in (
    critical_artifacts
):

    digest = verify_sha_sidecar(
        artifact,
        sidecar,
    )

    critical_hashes[
        relative_repo_path(
            artifact
        )
    ] = digest

    print(
        f"[PASS] {label}:"
    )

    print(
        " ",
        digest,
    )


protocol = load_json(
    PROTOCOL_LOCK
)


if (
    protocol[
        "opening_summary"
    ][
        "consumed"
    ]
    != 0
):

    raise RuntimeError(
        "Protocol opening ledger is not pristine."
    )


if (
    protocol[
        "opening_summary"
    ][
        "budget"
    ]
    != 8
):

    raise RuntimeError(
        "Protocol opening budget != 8."
    )


if (
    protocol[
        "scientific_fit_budget"
    ][
        "grand_total"
    ]
    != 4
):

    raise RuntimeError(
        "Protocol scientific fit budget != 4."
    )


if (
    protocol[
        "scientific_access_during_stage24_0c"
    ][
        "scientific_model_fits"
    ]
    != 0
):

    raise RuntimeError(
        "Scientific fit count before 0D != 0."
    )


if (
    protocol[
        "scientific_access_during_stage24_0c"
    ][
        "target_model_predictions"
    ]
    != 0
):

    raise RuntimeError(
        "Target predictions before 0D != 0."
    )


if (
    protocol[
        "boosting_backend"
    ][
        "lightgbm"
    ]
    != "gpu"
):

    raise RuntimeError(
        "Frozen Stage24 LightGBM backend is not GPU."
    )


if (
    protocol[
        "boosting_backend"
    ][
        "xgboost"
    ]
    != "cuda"
):

    raise RuntimeError(
        "Frozen Stage24 XGBoost backend is not CUDA."
    )


print()

print(
    "[PASS] Stage24-0C scientific ledger remains pristine."
)

print(
    "[PASS] LightGBM = GPU."
)

print(
    "[PASS] XGBoost = CUDA."
)

print(
    "[PASS] Scientific fit budget = 4."
)

print(
    "[PASS] Target openings = 0 / 8."
)
print()


# ============================================================================
# 4. REPOSITORY SCOPE SAFETY
# ============================================================================

print("=" * 110)
print("REPOSITORY SCOPE SAFETY")
print("=" * 110)


branch = git(
    "branch",
    "--show-current",
)

head_before = git(
    "rev-parse",
    "HEAD",
)

status_before = git(
    "status",
    "--porcelain",
)


print(
    "Branch:",
    branch,
)

print(
    "HEAD before 0D:",
    head_before,
)


if branch != BRANCH:

    raise RuntimeError(
        f"Expected branch {BRANCH!r}, got {branch!r}"
    )


# --------------------------------------------------------------------------
# Determine whether this is:
#
# A. first execution at frozen parent
# B. recovery after lock commit but before attestation commit
# C. already completed idempotent verification
# --------------------------------------------------------------------------

parent_mode = (
    head_before
    == EXPECTED_PARENT
)


if parent_mode:

    print(
        "Mode: FIRST EXECUTION"
    )

else:

    print(
        "Mode: RECOVERY / RE-VERIFICATION"
    )


# --------------------------------------------------------------------------
# Dirty paths may ONLY be Stage24
# --------------------------------------------------------------------------

unexpected_dirty = []


if status_before:

    for line in (
        status_before.splitlines()
    ):

        path_text = (
            line[
                3:
            ].strip()
        )

        if " -> " in path_text:

            path_text = path_text.split(
                " -> ",
                1,
            )[1]


        if not path_text.startswith(
            "results/stage24_cross_dataset/"
        ):

            unexpected_dirty.append(
                line
            )


if unexpected_dirty:

    raise RuntimeError(
        "\nUnexpected modifications outside Stage24:\n"
        + "\n".join(
            unexpected_dirty
        )
    )


print(
    "Dirty paths confined to Stage24:",
    not bool(
        unexpected_dirty
    ),
)

print()


# ============================================================================
# 5. CREATE / IDENTIFY IMMUTABLE PROTOCOL-LOCK COMMIT
# ============================================================================

print("=" * 110)
print("PROTOCOL-LOCK COMMIT")
print("=" * 110)


lock_commit = None
new_lock_commit_created = False


if head_before == EXPECTED_PARENT:

    # ----------------------------------------------------------------------
    # Stage ONLY Stage24.
    # ----------------------------------------------------------------------

    git(
        "add",
        "--",
        str(
            STAGE24_REL
        ),
    )


    staged_names_raw = git(
        "diff",
        "--cached",
        "--name-only",
    )


    staged_names = [
        line.strip()
        for line
        in staged_names_raw.splitlines()
        if line.strip()
    ]


    if not staged_names:

        raise RuntimeError(
            "No Stage24 files staged."
        )


    non_stage24_staged = [
        path
        for path
        in staged_names
        if not path.startswith(
            "results/stage24_cross_dataset/"
        )
    ]


    if non_stage24_staged:

        raise RuntimeError(
            "Unexpected staged paths:\n"
            + "\n".join(
                non_stage24_staged
            )
        )


    # ----------------------------------------------------------------------
    # No large accidental runtime artifacts.
    # 10 MiB is vastly larger than expected protocol receipts.
    # ----------------------------------------------------------------------

    oversized = []


    for rel in staged_names:

        local = (
            REPO_DIR
            / rel
        )

        if (
            local.exists()
            and local.is_file()
            and local.stat().st_size
            > 10 * 1024 * 1024
        ):

            oversized.append(
                (
                    rel,
                    local.stat().st_size,
                )
            )


    if oversized:

        raise RuntimeError(
            "\nUnexpected >10 MiB Stage24 artifact staged:\n"
            + "\n".join(
                f"{path}: {size}"
                for path, size
                in oversized
            )
        )


    print(
        "Staged Stage24 files:",
        len(
            staged_names
        ),
    )


    for rel in (
        staged_names
    ):

        print(
            "  +",
            rel,
        )


    print()

    print(
        "Creating immutable protocol-lock commit..."
    )


    git(
        "commit",
        "-m",
        LOCK_COMMIT_MESSAGE,
    )


    lock_commit = git(
        "rev-parse",
        "HEAD",
    )


    new_lock_commit_created = True


else:

    # ----------------------------------------------------------------------
    # Recovery path.
    #
    # Accept only if current history descends from EXPECTED_PARENT and the
    # frozen 0C protocol exists in the current tree.
    # ----------------------------------------------------------------------

    ancestry = subprocess.run(
        [
            "git",
            "merge-base",
            "--is-ancestor",
            EXPECTED_PARENT,
            head_before,
        ],
        cwd=REPO_DIR,
    )


    if ancestry.returncode != 0:

        raise RuntimeError(
            "Current HEAD does not descend from frozen Stage24 parent."
        )


    # Search history for exact lock commit message.
    candidate = git(
        "log",
        "--format=%H",
        "--grep",
        f"^{LOCK_COMMIT_MESSAGE}$",
        "-n",
        "1",
    )


    if not candidate:

        raise RuntimeError(
            "\nHEAD advanced from frozen parent, but the expected "
            "Stage24 protocol-lock commit cannot be identified."
        )


    lock_commit = candidate.strip()


print(
    "Protocol-lock commit:",
    lock_commit,
)

print(
    "New commit created:",
    new_lock_commit_created,
)

print()


# ============================================================================
# 6. VERIFY LOCK COMMIT DESCENDS EXACTLY FROM SCIENTIFIC PARENT
# ============================================================================

lock_parent = git(
    "rev-parse",
    f"{lock_commit}^",
)


print(
    "Protocol-lock parent:",
    lock_parent,
)


if (
    lock_parent
    != EXPECTED_PARENT
):

    raise RuntimeError(
        "\nProtocol-lock commit has wrong parent.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {lock_parent}"
    )


print(
    "[PASS] Lock commit is directly anchored to frozen scientific parent."
)
print()


# ============================================================================
# 7. PUSH PROTOCOL-LOCK COMMIT
# ============================================================================

print("=" * 110)
print("PUSHING PROTOCOL LOCK")
print("=" * 110)


push_output = git(
    "push",
    "origin",
    f"HEAD:{BRANCH}",
    auth_header=AUTH_HEADER,
)


print(
    push_output
)

print()


# ============================================================================
# 8. VERIFY REMOTE HEAD == LOCK COMMIT
# ============================================================================

remote_line = git(
    "ls-remote",
    "origin",
    f"refs/heads/{BRANCH}",
    auth_header=AUTH_HEADER,
)


if not remote_line:

    raise RuntimeError(
        "Unable to resolve remote branch HEAD."
    )


remote_head_lock = (
    remote_line.split()[0]
)


print(
    "Local lock commit:",
    lock_commit,
)

print(
    "Remote main:      ",
    remote_head_lock,
)


if (
    remote_head_lock
    != lock_commit
):

    raise RuntimeError(
        "\nRemote HEAD verification failed.\n"
        f"Local:  {lock_commit}\n"
        f"Remote: {remote_head_lock}"
    )


print()

print(
    "[PASS] Protocol-lock commit is present on remote main."
)
print()


# ============================================================================
# 9. GET EXACT FILE LIST FROM IMMUTABLE LOCK COMMIT
# ============================================================================

print("=" * 110)
print("IMMUTABLE LOCK-COMMIT FILE SET")
print("=" * 110)


commit_file_output = git(
    "diff-tree",
    "--no-commit-id",
    "--name-only",
    "-r",
    lock_commit,
)


commit_files = [
    line.strip()
    for line
    in commit_file_output.splitlines()
    if line.strip()
]


if not commit_files:

    raise RuntimeError(
        "Protocol-lock commit contains no files."
    )


non_stage24_commit_files = [
    rel
    for rel
    in commit_files
    if not rel.startswith(
        "results/stage24_cross_dataset/"
    )
]


if non_stage24_commit_files:

    raise RuntimeError(
        "\nProtocol-lock commit contains non-Stage24 files:\n"
        + "\n".join(
            non_stage24_commit_files
        )
    )


required_lock_rel = (
    relative_repo_path(
        PROTOCOL_LOCK
    )
)


required_semantic_rel = (
    relative_repo_path(
        SEMANTIC_BRIDGE
    )
)


required_gpu_rel = (
    relative_repo_path(
        GPU_PROBE
    )
)


for required_rel in [
    required_lock_rel,
    required_semantic_rel,
    required_gpu_rel,
]:

    if required_rel not in (
        commit_files
    ):

        raise RuntimeError(
            "Critical Stage24-0C artifact absent from lock commit: "
            + required_rel
        )


print(
    "Committed Stage24 files:",
    len(
        commit_files
    ),
)


for rel in (
    commit_files
):

    print(
        "  ",
        rel,
    )


print()


# ============================================================================
# 10. REMOTE BYTE-FOR-BYTE VERIFICATION OF EVERY LOCK-COMMIT FILE
# ============================================================================

print("=" * 110)
print("REMOTE BYTE VERIFICATION — PROTOCOL-LOCK COMMIT")
print("=" * 110)


remote_verification = []


for index, rel in enumerate(
    commit_files,
    start=1,
):

    local_path = (
        REPO_DIR
        / rel
    )


    if not local_path.exists():

        raise RuntimeError(
            f"Committed local file missing: {rel}"
        )


    local_bytes = (
        local_path.read_bytes()
    )


    local_sha = (
        sha256_bytes(
            local_bytes
        )
    )


    github_bytes = (
        remote_bytes(
            lock_commit,
            rel,
        )
    )


    remote_sha = (
        sha256_bytes(
            github_bytes
        )
    )


    byte_equal = (
        local_bytes
        == github_bytes
    )


    print(
        f"[{index:02d}/{len(commit_files):02d}] "
        f"{rel}"
    )

    print(
        "  local :",
        local_sha,
    )

    print(
        "  remote:",
        remote_sha,
    )

    print(
        "  exact :",
        byte_equal,
    )


    if not byte_equal:

        raise RuntimeError(
            "\nRemote byte verification failed:\n"
            + rel
        )


    remote_verification.append(
        {
            "path":
                rel,

            "local_sha256":
                local_sha,

            "remote_sha256":
                remote_sha,

            "byte_identical":
                True,
        }
    )


print()

print(
    "[PASS] EVERY file in the immutable protocol-lock commit "
    "is byte-identical on GitHub."
)
print()


# ============================================================================
# 11. VERIFY CRITICAL 0C HASHES AT REMOTE COMMIT
# ============================================================================

print("=" * 110)
print("CRITICAL PROTOCOL HASH VERIFICATION")
print("=" * 110)


for rel, expected_sha in (
    critical_hashes.items()
):

    github_data = remote_bytes(
        lock_commit,
        rel,
    )

    remote_sha = sha256_bytes(
        github_data
    )


    print(
        rel
    )

    print(
        "  expected:",
        expected_sha,
    )

    print(
        "  remote:  ",
        remote_sha,
    )


    if (
        remote_sha
        != expected_sha
    ):

        raise RuntimeError(
            "Critical protocol SHA mismatch on remote: "
            + rel
        )


print()

print(
    "[PASS] Critical frozen protocol hashes independently verified."
)
print()


# ============================================================================
# 12. WRITE REMOTE-VERIFICATION ATTESTATION
#
# This does NOT alter the frozen 0C protocol.
# It records that the required pre-opening gate has been satisfied.
# ============================================================================

attestation = {

    "stage":
        "Stage24-0D",

    "type":
        "PRE_TARGET_OPENING_REMOTE_VERIFICATION_ATTESTATION",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "repository":
        REPO_FULL_NAME,

    "branch":
        BRANCH,

    "scientific_parent":
        EXPECTED_PARENT,

    "protocol_lock_commit":
        lock_commit,

    "protocol_lock_commit_parent":
        lock_parent,

    "protocol_lock_commit_message":
        LOCK_COMMIT_MESSAGE,

    "remote_head_at_lock_verification":
        remote_head_lock,

    "remote_head_equal_lock_commit":
        True,

    "protocol_artifacts": {

        "stage24_0b2": {
            "path":
                relative_repo_path(
                    B2_RECEIPT
                ),

            "sha256":
                critical_hashes[
                    relative_repo_path(
                        B2_RECEIPT
                    )
                ],
        },

        "semantic_bridge": {
            "path":
                required_semantic_rel,

            "sha256":
                critical_hashes[
                    required_semantic_rel
                ],
        },

        "gpu_backend_probe": {
            "path":
                required_gpu_rel,

            "sha256":
                critical_hashes[
                    required_gpu_rel
                ],
        },

        "final_preopening_protocol_lock": {
            "path":
                required_lock_rel,

            "sha256":
                critical_hashes[
                    required_lock_rel
                ],
        },
    },

    "remote_byte_verification": {

        "files_in_protocol_lock_commit":
            len(
                commit_files
            ),

        "files_verified_exact":
            len(
                remote_verification
            ),

        "all_pass":
            True,

        "records":
            remote_verification,
    },

    "frozen_runtime": {

        "lightgbm":
            "GPU",

        "xgboost":
            "CUDA",
    },

    "scientific_fit_budget": {

        "total":
            4,

        "consumed_before_target_opening":
            0,
    },

    "target_opening_ledger": {

        "consumed":
            0,

        "budget":
            8,

        "status_before_gate":
            "CLOSED",

        "status_after_successful_0D":
            "AUTHORIZED_FOR_FROZEN_EXECUTION_ONLY",
    },

    "scientific_access_during_0d": {

        "target_rows_read":
            0,

        "target_labels_read":
            0,

        "scientific_model_fits":
            0,

        "model_predictions":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,
    },

    "governance_gate": {

        "protocol_created_before_target_opening":
            True,

        "protocol_committed_before_target_opening":
            True,

        "protocol_pushed_before_target_opening":
            True,

        "protocol_remotely_verified_before_target_opening":
            True,

        "anti_adaptation_lock_active":
            True,

        "gate_pass":
            True,
    },

    "authorization": {

        "status":
            "AUTHORIZED_FOR_FROZEN_STAGE24_EXECUTION",

        "scope":
            (
                "Execution is authorized ONLY according to the immutable "
                "Stage24-0C protocol and 8-cell target-opening ledger."
            ),

        "protocol_modification_after_this_gate":
            "FORBIDDEN",

        "target_guided_protocol_adaptation":
            "FORBIDDEN",
    },

    "next_authorized_step":
        (
            "Execute Stage24 according to the frozen order beginning with "
            "PRIMARY_FULL_SOURCE_70F_SANITY_WITH_FROZEN_STAGE22_MODELS. "
            "Target openings remain 0/8 until an explicitly budgeted "
            "target inference cell is executed."
        ),
}


attestation_sha = write_json_with_sha(
    ATTESTATION,
    attestation,
)


print("=" * 110)
print("REMOTE VERIFICATION ATTESTATION")
print("=" * 110)

print(
    "Receipt:"
)

print(
    " ",
    ATTESTATION,
)

print(
    "SHA256:"
)

print(
    " ",
    attestation_sha,
)

print()


# ============================================================================
# 13. COMMIT ATTESTATION
# ============================================================================

git(
    "add",
    "--",
    relative_repo_path(
        ATTESTATION
    ),
    relative_repo_path(
        ATTESTATION_SHA
    ),
)


staged_after_attestation = git(
    "diff",
    "--cached",
    "--name-only",
)


staged_after_attestation = [
    line.strip()
    for line
    in staged_after_attestation.splitlines()
    if line.strip()
]


expected_attestation_files = {
    relative_repo_path(
        ATTESTATION
    ),
    relative_repo_path(
        ATTESTATION_SHA
    ),
}


if (
    set(
        staged_after_attestation
    )
    != expected_attestation_files
):

    raise RuntimeError(
        "\nAttestation commit staging scope is not exact.\n"
        f"Expected: {sorted(expected_attestation_files)}\n"
        f"Actual:   {sorted(staged_after_attestation)}"
    )


git(
    "commit",
    "-m",
    ATTESTATION_COMMIT_MESSAGE,
)


attestation_commit = git(
    "rev-parse",
    "HEAD",
)


attestation_parent = git(
    "rev-parse",
    "HEAD^",
)


if (
    attestation_parent
    != lock_commit
):

    raise RuntimeError(
        "\nAttestation commit does not directly descend "
        "from protocol-lock commit."
    )


print(
    "Attestation commit:",
    attestation_commit,
)

print(
    "Parent lock commit:",
    attestation_parent,
)

print()


# ============================================================================
# 14. PUSH ATTESTATION
# ============================================================================

print("=" * 110)
print("PUSHING REMOTE-VERIFICATION ATTESTATION")
print("=" * 110)


push_attestation_output = git(
    "push",
    "origin",
    f"HEAD:{BRANCH}",
    auth_header=AUTH_HEADER,
)


print(
    push_attestation_output
)
print()


# ============================================================================
# 15. VERIFY FINAL REMOTE HEAD
# ============================================================================

remote_final_line = git(
    "ls-remote",
    "origin",
    f"refs/heads/{BRANCH}",
    auth_header=AUTH_HEADER,
)


if not remote_final_line:

    raise RuntimeError(
        "Unable to resolve final remote HEAD."
    )


remote_final_head = (
    remote_final_line.split()[0]
)


print(
    "Local final HEAD:",
    attestation_commit,
)

print(
    "Remote main:     ",
    remote_final_head,
)


if (
    remote_final_head
    != attestation_commit
):

    raise RuntimeError(
        "\nFinal remote HEAD mismatch.\n"
        f"Local:  {attestation_commit}\n"
        f"Remote: {remote_final_head}"
    )


print()

print(
    "[PASS] Final remote HEAD equals local attestation commit."
)
print()


# ============================================================================
# 16. VERIFY REMOTE ATTESTATION BYTE-FOR-BYTE
# ============================================================================

print("=" * 110)
print("REMOTE ATTESTATION BYTE VERIFICATION")
print("=" * 110)


attestation_rel = (
    relative_repo_path(
        ATTESTATION
    )
)

attestation_sha_rel = (
    relative_repo_path(
        ATTESTATION_SHA
    )
)


remote_attestation = remote_bytes(
    attestation_commit,
    attestation_rel,
)

remote_attestation_sidecar = remote_bytes(
    attestation_commit,
    attestation_sha_rel,
)


local_attestation = (
    ATTESTATION.read_bytes()
)

local_attestation_sidecar = (
    ATTESTATION_SHA.read_bytes()
)


if (
    remote_attestation
    != local_attestation
):

    raise RuntimeError(
        "Remote Stage24-0D attestation JSON differs from local."
    )


if (
    remote_attestation_sidecar
    != local_attestation_sidecar
):

    raise RuntimeError(
        "Remote Stage24-0D attestation SHA sidecar differs from local."
    )


remote_attestation_sha = (
    sha256_bytes(
        remote_attestation
    )
)


print(
    "Expected attestation SHA:"
)

print(
    " ",
    attestation_sha,
)

print(
    "Remote attestation SHA:"
)

print(
    " ",
    remote_attestation_sha,
)


if (
    remote_attestation_sha
    != attestation_sha
):

    raise RuntimeError(
        "Remote attestation SHA mismatch."
    )


print()

print(
    "[PASS] Stage24-0D attestation is byte-identical on GitHub."
)
print()


# ============================================================================
# 17. VERIFY FROZEN LOCK STILL UNCHANGED THROUGH ATTESTATION COMMIT
# ============================================================================

print("=" * 110)
print("POST-ATTESTATION PROTOCOL IMMUTABILITY CHECK")
print("=" * 110)


remote_lock_after_attestation = (
    remote_bytes(
        attestation_commit,
        required_lock_rel,
    )
)


remote_lock_after_sha = (
    sha256_bytes(
        remote_lock_after_attestation
    )
)


expected_lock_sha = (
    critical_hashes[
        required_lock_rel
    ]
)


print(
    "Frozen 0C protocol expected:"
)

print(
    " ",
    expected_lock_sha,
)

print(
    "Frozen 0C protocol at final remote HEAD:"
)

print(
    " ",
    remote_lock_after_sha,
)


if (
    remote_lock_after_sha
    != expected_lock_sha
):

    raise RuntimeError(
        "\nFrozen Stage24-0C protocol changed during attestation."
    )


print()

print(
    "[PASS] Stage24-0C protocol remained immutable."
)
print()


# ============================================================================
# 18. FINAL CLEAN-WORKTREE GATE
# ============================================================================

final_status = git(
    "status",
    "--porcelain",
)


print("=" * 110)
print("FINAL WORKTREE GATE")
print("=" * 110)


if final_status:

    print(
        final_status
    )

    raise RuntimeError(
        "\nWorktree is not clean after Stage24-0D."
    )


print(
    "git status --porcelain:"
)

print(
    "  <clean>"
)

print()

print(
    "[PASS] Repository worktree is clean."
)
print()


# ============================================================================
# 19. FINAL GOVERNANCE SYNTHESIS
# ============================================================================

print("=" * 110)
print("STAGE24-0D FINAL SYNTHESIS")
print("=" * 110)

print(
    "Scientific parent:"
)

print(
    " ",
    EXPECTED_PARENT,
)

print()

print(
    "Immutable protocol-lock commit:"
)

print(
    " ",
    lock_commit,
)

print()

print(
    "Remote-verification attestation commit:"
)

print(
    " ",
    attestation_commit,
)

print()

print(
    "Final remote main:"
)

print(
    " ",
    remote_final_head,
)

print()

print(
    "Frozen protocol SHA256:"
)

print(
    " ",
    expected_lock_sha,
)

print()

print(
    "Remote files from protocol-lock commit verified:"
)

print(
    " ",
    f"{len(remote_verification)}/{len(commit_files)} EXACT",
)

print()

print(
    "Stage24-0D attestation SHA256:"
)

print(
    " ",
    attestation_sha,
)

print()

print(
    "LightGBM scientific backend:"
)

print(
    "  GPU"
)

print(
    "XGBoost scientific backend:"
)

print(
    "  CUDA"
)

print()

print(
    "Scientific fit budget:"
)

print(
    "  4"
)

print(
    "Scientific fits consumed:"
)

print(
    "  0"
)

print()

print(
    "Target rows read:"
)

print(
    "  0"
)

print(
    "Target labels read:"
)

print(
    "  0"
)

print(
    "Target predictions:"
)

print(
    "  0"
)

print(
    "Target metrics:"
)

print(
    "  0"
)

print(
    "Target openings consumed:"
)

print(
    "  0 / 8"
)

print()

print("-" * 110)

print(
    "STAGE24-0D: PASS"
)

print()

print(
    "The Stage24 scientific protocol was:"
)

print(
    "  CREATED  -> PASS"
)

print(
    "  COMMITTED -> PASS"
)

print(
    "  PUSHED    -> PASS"
)

print(
    "  REMOTELY VERIFIED BYTE-FOR-BYTE -> PASS"
)

print(
    "  ATTESTED ON REMOTE MAIN -> PASS"
)

print()

print(
    "THE PRE-TARGET-OPENING GOVERNANCE GATE IS NOW SATISFIED."
)

print()

print(
    "Stage24 execution is now AUTHORIZED strictly under the frozen protocol."
)

print(
    "Target opening ledger remains pristine at 0 / 8."
)

print()

print(
    "NEXT AUTHORIZED SCIENTIFIC STEP:"
)

print(
    "Stage24-1A — PRIMARY FULL-SOURCE 70F SANITY using the exact "
    "frozen Stage22 ensemble."
)

print(
    "This next sanity step still does NOT require a CICIDS2017 target opening."
)

print("-" * 110)

print(
    "STAGE24-0D COMPLETE"
)

print("=" * 110)

In [ ]:
# ============================================================================
# STAGE24-1A — PRIMARY FULL-SOURCE 70F SANITY
#
# PURPOSE
# -------
# Re-execute the exact frozen Stage22R CHRONOLOGICAL_NATURAL 70-feature
# ensemble on its already-authorized IDS2018 source validation population.
#
# SOURCE:
#   IDS2018 02-28-2018
#   Stage22R K79-clean development cache
#   day_id = 7
#   rows   = 593,780
#
# MODELS:
#   exact frozen Stage22R LightGBM
#   exact frozen Stage22R XGBoost
#   ensemble = 0.5 * P_LGBM + 0.5 * P_XGB
#
# REQUIRED SANITY:
#   - exact model hashes
#   - exact 70-feature order
#   - exact validation cache hash
#   - exact validation row/label membership
#   - reproduce frozen ensemble probability vector BIT-FOR-BIT
#   - reproduce frozen operating-point confusion counts
#
# SCIENTIFIC EFFECT:
#   model fits:          0
#   source inference:    YES
#   target rows read:    0
#   target labels read:  0
#   target predictions:  0
#   target openings:     0 / 8
#
# ON SUCCESS:
#   write + commit + push Stage24-1A receipt
#
# ============================================================================

from __future__ import annotations

import os
import json
import gc
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import lightgbm as lgb
import xgboost as xgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================================
# 0. FROZEN CONSTANTS
# ============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "3a0b4a54477a5159805e69caa1eba749ab614c7b"
)

BRANCH = "main"

STAGE24_LOCK = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)

STAGE24_ATTESTATION = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0d_remote_verification_receipt.json"
)

STAGE22_DIR = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
)

STAGE22_RESULT = (
    STAGE22_DIR
    / "stage22r_2c_chronological_natural_result.json"
)

LGB_MODEL = (
    STAGE22_DIR
    / "chronological_natural_lightgbm_model.txt"
)

XGB_MODEL = (
    STAGE22_DIR
    / "chronological_natural_xgboost_model.json"
)

FROZEN_PROB = (
    STAGE22_DIR
    / "chronological_natural_validation_ensemble_probabilities.npz"
)

INPUT_MANIFEST = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
    / "stage22r_1c_development_model_input_manifest.json"
)


EXPECTED_CACHE_FILE = (
    "day_07_02-28-2018.parquet"
)

EXPECTED_CACHE_SHA = (
    "3ab933a7988af562f49f63b8975882b7c66e843e9c4b967a38d5d875d062f62b"
)

EXPECTED_ROWS = 593_780
EXPECTED_BENIGN = 531_524
EXPECTED_ATTACK = 62_256

EXPECTED_FROZEN_PROB_SHA = (
    "3fe6c468a0653ac5ee488da8d6586fd86628e9586eb01f6f5d962e35fff65e3f"
)

EXPECTED_LGB_SHA = (
    "7be4c610814e45be2e315996969d2e2f404a3ada14866b21a731e4382fdc18b8"
)

EXPECTED_XGB_SHA = (
    "38499d0edcdc5c6d0b2c3afee913558973d35d5ebf1cb2ae888a16b8da98756c"
)


OUTDIR_REL = (
    Path("results")
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
)

OUTDIR = (
    REPO
    / OUTDIR_REL
)

OUTDIR.mkdir(
    parents=True,
    exist_ok=True,
)

RECEIPT = (
    OUTDIR
    / "stage24_1a_full_source_70f_sanity.json"
)

RECEIPT_SHA = (
    OUTDIR
    / "stage24_1a_full_source_70f_sanity.sha256"
)


print("=" * 108)
print("STAGE24-1A — PRIMARY FULL-SOURCE 70F SANITY")
print("=" * 108)

print("Scientific model fits:   0")
print("Target rows read:        0")
print("Target labels read:      0")
print("Target predictions:      0")
print("Target openings:         0 / 8")
print()


# ============================================================================
# 1. HELPERS
# ============================================================================

def git(
    *args,
    check=True,
    auth_header=None,
):
    command = ["git"]

    if auth_header is not None:

        command.extend(
            [
                "-c",
                "credential.helper=",
                "-c",
                f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
            ]
        )

    command.extend(
        map(
            str,
            args,
        )
    )

    p = subprocess.run(
        command,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if (
        check
        and p.returncode != 0
    ):

        raise RuntimeError(
            "\nGit command failed:\n"
            + " ".join(
                command
            )
            + "\n\n"
            + p.stdout
        )

    return (
        p.stdout or ""
    ).strip()


def load_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path: Path,
    block_size=8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                block_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def array_sha256(array):
    a = np.ascontiguousarray(
        array
    )

    h = hashlib.sha256()

    h.update(
        str(
            a.dtype
        ).encode(
            "utf-8"
        )
    )

    h.update(
        repr(
            tuple(
                a.shape
            )
        ).encode(
            "utf-8"
        )
    )

    h.update(
        a.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


def dump_json_with_sha(
    path: Path,
    obj,
):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    digest = sha256_file(
        path
    )

    sidecar = path.with_suffix(
        ".sha256"
    )

    sidecar.write_text(
        f"{digest}  {path.name}\n",
        encoding="utf-8",
    )

    return digest


def confusion_counts(
    y,
    p,
    threshold,
):
    pred = (
        p
        >= threshold
    )

    y_bool = (
        y.astype(
            np.uint8,
            copy=False,
        )
        == 1
    )

    tp = int(
        np.count_nonzero(
            pred
            & y_bool
        )
    )

    fp = int(
        np.count_nonzero(
            pred
            & ~y_bool
        )
    )

    fn = int(
        np.count_nonzero(
            ~pred
            & y_bool
        )
    )

    tn = int(
        np.count_nonzero(
            ~pred
            & ~y_bool
        )
    )

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def get_operating_record(
    result,
    name,
):
    item = (
        result[
            "operating_points"
        ][
            name
        ]
    )

    if (
        name == "security"
        and isinstance(
            item,
            dict,
        )
        and "result" in item
    ):
        return item[
            "result"
        ]

    return item


# ============================================================================
# 2. REPOSITORY GOVERNANCE
# ============================================================================

print("=" * 108)
print("REPOSITORY GOVERNANCE")
print("=" * 108)

branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "ls-remote",
    "origin",
    f"refs/heads/{BRANCH}",
).split()[0]

status = git(
    "status",
    "--porcelain",
)

print(
    "Branch:",
    branch,
)

print(
    "Local HEAD:",
    head,
)

print(
    "Remote main:",
    remote,
)


if branch != BRANCH:

    raise RuntimeError(
        f"Expected {BRANCH}, got {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected Stage24-1A parent.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if remote != head:

    raise RuntimeError(
        "Local HEAD != remote main."
    )


if status:

    raise RuntimeError(
        "\nStage24-1A requires a clean worktree.\n"
        + status
    )


print()

print(
    "[PASS] Stage24-1A starts from remotely attested Stage24-0D."
)
print()


# ============================================================================
# 3. VERIFY GOVERNANCE ATTESTATION
# ============================================================================

print("=" * 108)
print("STAGE24-0D AUTHORIZATION")
print("=" * 108)


for path in [
    STAGE24_LOCK,
    STAGE24_ATTESTATION,
    STAGE22_RESULT,
    LGB_MODEL,
    XGB_MODEL,
    FROZEN_PROB,
    INPUT_MANIFEST,
]:

    if not path.is_file():

        raise FileNotFoundError(
            path
        )


protocol = load_json(
    STAGE24_LOCK
)

attestation = load_json(
    STAGE24_ATTESTATION
)


if (
    attestation[
        "governance_gate"
    ][
        "gate_pass"
    ]
    is not True
):

    raise RuntimeError(
        "Stage24-0D gate did not pass."
    )


if (
    attestation[
        "authorization"
    ][
        "status"
    ]
    != "AUTHORIZED_FOR_FROZEN_STAGE24_EXECUTION"
):

    raise RuntimeError(
        "Stage24 execution not authorized."
    )


if (
    attestation[
        "target_opening_ledger"
    ][
        "consumed"
    ]
    != 0
):

    raise RuntimeError(
        "Target opening ledger not pristine."
    )


if (
    protocol[
        "execution_order"
    ][0][
        "action"
    ]
    != "COMMIT_PUSH_REMOTE_VERIFY_STAGE24_0_PROTOCOL_LOCK"
):

    raise RuntimeError(
        "Frozen execution order mismatch."
    )


if (
    protocol[
        "execution_order"
    ][1][
        "action"
    ]
    != "PRIMARY_FULL_SOURCE_70F_SANITY_WITH_FROZEN_STAGE22_MODELS"
):

    raise RuntimeError(
        "Stage24-1A is not the next frozen scientific step."
    )


print(
    "[PASS] Stage24 frozen execution is authorized."
)

print(
    "[PASS] Next frozen step = PRIMARY_FULL_SOURCE_70F_SANITY."
)

print(
    "[PASS] Target opening ledger = 0 / 8."
)
print()


# ============================================================================
# 4. VERIFY FROZEN STAGE22 ARTIFACTS
# ============================================================================

print("=" * 108)
print("FROZEN STAGE22 ARTIFACT INTEGRITY")
print("=" * 108)


actual_lgb_sha = sha256_file(
    LGB_MODEL
)

actual_xgb_sha = sha256_file(
    XGB_MODEL
)

actual_prob_sha = sha256_file(
    FROZEN_PROB
)


print(
    "LightGBM:"
)

print(
    "  expected:",
    EXPECTED_LGB_SHA,
)

print(
    "  actual:  ",
    actual_lgb_sha,
)

print()

print(
    "XGBoost:"
)

print(
    "  expected:",
    EXPECTED_XGB_SHA,
)

print(
    "  actual:  ",
    actual_xgb_sha,
)

print()

print(
    "Frozen validation probability artifact:"
)

print(
    "  expected:",
    EXPECTED_FROZEN_PROB_SHA,
)

print(
    "  actual:  ",
    actual_prob_sha,
)


if actual_lgb_sha != EXPECTED_LGB_SHA:
    raise RuntimeError(
        "LightGBM model SHA mismatch."
    )

if actual_xgb_sha != EXPECTED_XGB_SHA:
    raise RuntimeError(
        "XGBoost model SHA mismatch."
    )

if actual_prob_sha != EXPECTED_FROZEN_PROB_SHA:
    raise RuntimeError(
        "Frozen validation probability SHA mismatch."
    )


print()

print(
    "[PASS] Frozen Stage22 model/probability artifacts are exact."
)
print()


# ============================================================================
# 5. RESOLVE EXACT STAGE22 VALIDATION CACHE
# ============================================================================

print("=" * 108)
print("RESOLVING EXACT STAGE22R VALIDATION CACHE")
print("=" * 108)


required_cache_files = [
    "day_00_02-14-2018.parquet",
    "day_01_02-15-2018.parquet",
    "day_02_02-16-2018.parquet",
    "day_03_02-20-2018.parquet",
    "day_04_02-21-2018.parquet",
    "day_05_02-22-2018.parquet",
    "day_06_02-23-2018.parquet",
    "day_07_02-28-2018.parquet",
]


preferred_cache = Path(
    "/kaggle/input/datasets/"
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)


candidate_dirs = []


if preferred_cache.is_dir():

    candidate_dirs.append(
        preferred_cache
    )


input_root = Path(
    "/kaggle/input"
)


if input_root.exists():

    for day07 in input_root.rglob(
        EXPECTED_CACHE_FILE
    ):

        parent = (
            day07.parent
        )

        if all(
            (
                parent
                / filename
            ).is_file()

            for filename
            in required_cache_files
        ):

            candidate_dirs.append(
                parent
            )


working_cache = Path(
    "/kaggle/working/stage22r_1c_70f_development_cache"
)


if (
    working_cache.is_dir()
    and all(
        (
            working_cache
            / filename
        ).is_file()

        for filename
        in required_cache_files
    )
):

    candidate_dirs.append(
        working_cache
    )


# dedupe paths
candidate_dirs = list(
    dict.fromkeys(
        map(
            lambda p: p.resolve(),
            candidate_dirs,
        )
    )
)


exact_candidates = []


for directory in candidate_dirs:

    day07 = (
        directory
        / EXPECTED_CACHE_FILE
    )

    if not day07.is_file():
        continue

    digest = sha256_file(
        day07
    )

    print(
        "Candidate:",
        directory,
    )

    print(
        "  day07 SHA:",
        digest,
    )

    if digest == EXPECTED_CACHE_SHA:

        exact_candidates.append(
            directory
        )


if not exact_candidates:

    raise RuntimeError(
        "\nExact Stage22R 70-feature cache not found.\n"
        "Expected Kaggle dataset:\n"
        "  jmmubasshirrahman/stage22r-1c-70f-cache-3cd41c5f\n"
        "Do NOT substitute a reconstructed or differently processed cache."
    )


# Multiple mounted byte-identical copies are scientifically equivalent.
CACHE = exact_candidates[
    0
]

DAY07 = (
    CACHE
    / EXPECTED_CACHE_FILE
)


print()

print(
    "Selected exact cache:"
)

print(
    " ",
    CACHE,
)

print(
    "day07 SHA256:"
)

print(
    " ",
    sha256_file(
        DAY07
    ),
)

print()

print(
    "[PASS] Exact frozen Stage22R validation cache recovered."
)
print()


# ============================================================================
# 6. VERIFY CACHE MANIFEST + FEATURE ORDER
# ============================================================================

manifest = load_json(
    INPUT_MANIFEST
)

stage22 = load_json(
    STAGE22_RESULT
)


features = list(
    stage22[
        "data"
    ][
        "feature_order"
    ]
)


if len(
    features
) != 70:

    raise RuntimeError(
        "Frozen Stage22 feature count != 70."
    )


manifest_features = list(
    manifest[
        "row_schema"
    ][
        "predictor_columns"
    ]
)


if manifest_features != features:

    raise RuntimeError(
        "Stage22 result and input manifest feature orders differ."
    )


day07_manifest = [
    item
    for item
    in manifest[
        "cache"
    ][
        "files"
    ]
    if item[
        "cache_file"
    ] == EXPECTED_CACHE_FILE
]


if len(
    day07_manifest
) != 1:

    raise RuntimeError(
        "Could not uniquely identify day07 cache manifest."
    )


day07_manifest = (
    day07_manifest[
        0
    ]
)


if (
    day07_manifest[
        "sha256"
    ]
    != EXPECTED_CACHE_SHA
):

    raise RuntimeError(
        "Frozen day07 manifest SHA differs."
    )


if (
    int(
        day07_manifest[
            "rows"
        ]
    )
    != EXPECTED_ROWS
):

    raise RuntimeError(
        "Frozen day07 row count differs."
    )


print("=" * 108)
print("FROZEN SOURCE REPRESENTATION")
print("=" * 108)

print(
    "Features:",
    len(
        features
    ),
)

print(
    "Rows:",
    EXPECTED_ROWS,
)

print(
    "Benign:",
    EXPECTED_BENIGN,
)

print(
    "Attack:",
    EXPECTED_ATTACK,
)

print(
    "Scaling:",
    stage22[
        "data"
    ][
        "scaling"
    ],
)

print(
    "Imputation:",
    stage22[
        "data"
    ][
        "explicit_imputation"
    ],
)

print()

print(
    "[PASS] Exact 70-feature source representation recovered."
)
print()


# ============================================================================
# 7. LOAD SOURCE VALIDATION CACHE
#
# SOURCE DATA ONLY.
# This is NOT a Stage24 target opening.
# ============================================================================

print("=" * 108)
print("LOADING IDS2018 SOURCE VALIDATION — NOT A TARGET OPENING")
print("=" * 108)


columns = (
    [
        "clean_position",
        "binary_label",
    ]
    + features
)


start = time.time()


validation_df = pd.read_parquet(
    DAY07,
    columns=columns,
)


load_seconds = (
    time.time()
    - start
)


if len(
    validation_df
) != EXPECTED_ROWS:

    raise RuntimeError(
        "Validation cache row count mismatch."
    )


if list(
    validation_df.columns
) != columns:

    raise RuntimeError(
        "Validation cache column order mismatch."
    )


clean_position = (
    validation_df[
        "clean_position"
    ]
    .to_numpy(
        dtype=np.int64,
        copy=True,
    )
)


y_val = (
    validation_df[
        "binary_label"
    ]
    .to_numpy(
        dtype=np.uint8,
        copy=True,
    )
)


X_val = (
    validation_df[
        features
    ]
    .to_numpy(
        dtype=np.float64,
        copy=True,
    )
)


del validation_df
gc.collect()


if X_val.shape != (
    EXPECTED_ROWS,
    70,
):

    raise RuntimeError(
        f"Unexpected X validation shape: {X_val.shape}"
    )


attack_count = int(
    y_val.sum()
)

benign_count = int(
    len(
        y_val
    )
    - attack_count
)


if attack_count != EXPECTED_ATTACK:
    raise RuntimeError(
        "Validation attack count mismatch."
    )


if benign_count != EXPECTED_BENIGN:
    raise RuntimeError(
        "Validation benign count mismatch."
    )


if not np.all(
    np.diff(
        clean_position
    ) == 1
):

    raise RuntimeError(
        "Chronological day07 clean_position is not contiguous."
    )


print(
    "X shape:",
    X_val.shape,
)

print(
    "X dtype:",
    X_val.dtype,
)

print(
    "clean_position:",
    int(
        clean_position[
            0
        ]
    ),
    "->",
    int(
        clean_position[
            -1
        ]
    ),
)

print(
    "Benign:",
    benign_count,
)

print(
    "Attack:",
    attack_count,
)

print(
    "Load seconds:",
    round(
        load_seconds,
        3,
    ),
)

print()

print(
    "[PASS] Source validation membership exact."
)
print()


# ============================================================================
# 8. LOAD FROZEN PROBABILITY REFERENCE
# ============================================================================

print("=" * 108)
print("FROZEN STAGE22 VALIDATION REFERENCE")
print("=" * 108)


with np.load(
    FROZEN_PROB,
    allow_pickle=False,
) as z:

    frozen_keys = list(
        z.files
    )

    frozen_clean = np.asarray(
        z[
            "clean_position"
        ],
        dtype=np.int64,
    )

    frozen_y = np.asarray(
        z[
            "binary_label"
        ],
        dtype=np.uint8,
    )

    frozen_p = np.asarray(
        z[
            "ensemble_probability"
        ],
        dtype=np.float32,
    )


expected_keys = [
    "clean_position",
    "binary_label",
    "ensemble_probability",
]


if frozen_keys != expected_keys:

    raise RuntimeError(
        "Frozen probability artifact keys changed.\n"
        f"Expected: {expected_keys}\n"
        f"Actual:   {frozen_keys}"
    )


if not np.array_equal(
    frozen_clean,
    clean_position,
):

    raise RuntimeError(
        "Frozen probability clean_position membership mismatch."
    )


if not np.array_equal(
    frozen_y,
    y_val,
):

    raise RuntimeError(
        "Frozen probability binary labels mismatch."
    )


if frozen_p.shape != (
    EXPECTED_ROWS,
):

    raise RuntimeError(
        "Frozen probability vector length mismatch."
    )


print(
    "Keys:",
    frozen_keys,
)

print(
    "Probability dtype:",
    frozen_p.dtype,
)

print(
    "Probability rows:",
    len(
        frozen_p
    ),
)

print(
    "Membership exact:",
    True,
)

print(
    "Labels exact:",
    True,
)

print()

print(
    "[PASS] Frozen source-validation reference aligned exactly."
)
print()


# ============================================================================
# 9. LOAD EXACT FROZEN MODELS
# ============================================================================

print("=" * 108)
print("LOADING EXACT FROZEN STAGE22 MODELS")
print("=" * 108)


lgb_booster = lgb.Booster(
    model_file=str(
        LGB_MODEL
    )
)


if int(
    lgb_booster.num_feature()
) != 70:

    raise RuntimeError(
        "LightGBM model dimension != 70."
    )


if int(
    lgb_booster.num_trees()
) != 400:

    raise RuntimeError(
        "LightGBM tree count != 400."
    )


lgb_feature_names = list(
    lgb_booster.feature_name()
)


if (
    lgb_feature_names
    != features
):

    raise RuntimeError(
        "LightGBM feature order != frozen Stage22 order."
    )


xgb_booster = xgb.Booster()

xgb_booster.load_model(
    str(
        XGB_MODEL
    )
)


if int(
    xgb_booster.num_features()
) != 70:

    raise RuntimeError(
        "XGBoost model dimension != 70."
    )


if int(
    xgb_booster.num_boosted_rounds()
) != 400:

    raise RuntimeError(
        "XGBoost boosted rounds != 400."
    )


xgb_feature_names = (
    list(
        xgb_booster.feature_names
    )
    if xgb_booster.feature_names is not None
    else None
)


if (
    xgb_feature_names is not None
    and xgb_feature_names != features
):

    raise RuntimeError(
        "XGBoost model feature order != frozen Stage22 order."
    )


# Frozen Stage22 XGBoost inference backend.
xgb_booster.set_param(
    {
        "device":
            "cuda",
    }
)


print(
    "LightGBM features:",
    lgb_booster.num_feature(),
)

print(
    "LightGBM trees:",
    lgb_booster.num_trees(),
)

print(
    "XGBoost features:",
    xgb_booster.num_features(),
)

print(
    "XGBoost rounds:",
    xgb_booster.num_boosted_rounds(),
)

print(
    "XGBoost inference device:",
    "cuda",
)

print()

print(
    "[PASS] Exact frozen source models loaded."
)
print()


# ============================================================================
# 10. SOURCE VALIDATION INFERENCE
#
# NO FIT.
# NO TARGET DATA.
# ============================================================================

print("=" * 108)
print("SOURCE VALIDATION INFERENCE")
print("=" * 108)


dval = xgb.DMatrix(
    X_val,
    label=y_val,
    missing=np.nan,
    feature_names=features,
)


start = time.time()


p_lgb = np.asarray(
    lgb_booster.predict(
        X_val
    ),
    dtype=np.float64,
)


p_xgb = np.asarray(
    xgb_booster.predict(
        dval
    ),
    dtype=np.float64,
)


inference_seconds = (
    time.time()
    - start
)


if p_lgb.shape != (
    EXPECTED_ROWS,
):
    raise RuntimeError(
        "LightGBM probability shape mismatch."
    )


if p_xgb.shape != (
    EXPECTED_ROWS,
):
    raise RuntimeError(
        "XGBoost probability shape mismatch."
    )


if not np.all(
    np.isfinite(
        p_lgb
    )
):
    raise RuntimeError(
        "Non-finite LightGBM probabilities."
    )


if not np.all(
    np.isfinite(
        p_xgb
    )
):
    raise RuntimeError(
        "Non-finite XGBoost probabilities."
    )


if not np.all(
    (
        p_lgb >= 0.0
    )
    &
    (
        p_lgb <= 1.0
    )
):
    raise RuntimeError(
        "LightGBM probabilities outside [0,1]."
    )


if not np.all(
    (
        p_xgb >= 0.0
    )
    &
    (
        p_xgb <= 1.0
    )
):
    raise RuntimeError(
        "XGBoost probabilities outside [0,1]."
    )


# Frozen Stage22 arithmetic:
# components in float64 -> final persisted float32.
reproduced_p = (
    (
        0.5
        * p_lgb
    )
    +
    (
        0.5
        * p_xgb
    )
).astype(
    np.float32
)


print(
    "Inference rows:",
    len(
        reproduced_p
    ),
)

print(
    "Inference seconds:",
    round(
        inference_seconds,
        3,
    ),
)

print(
    "Reproduced dtype:",
    reproduced_p.dtype,
)

print()


# ============================================================================
# 11. BIT-FOR-BIT PROBABILITY REPRODUCTION GATE
# ============================================================================

print("=" * 108)
print("BIT-FOR-BIT SOURCE PROBABILITY REPRODUCTION")
print("=" * 108)


bitwise_equal = np.array_equal(
    reproduced_p,
    frozen_p,
)


difference_count = int(
    np.count_nonzero(
        reproduced_p
        != frozen_p
    )
)


max_abs_difference = float(
    np.max(
        np.abs(
            reproduced_p.astype(
                np.float64
            )
            -
            frozen_p.astype(
                np.float64
            )
        )
    )
)


reproduced_raw_sha = array_sha256(
    reproduced_p
)

frozen_raw_sha = array_sha256(
    frozen_p
)


print(
    "Bitwise equal:",
    bitwise_equal,
)

print(
    "Differing positions:",
    difference_count,
)

print(
    "Maximum absolute difference:",
    max_abs_difference,
)

print(
    "Reproduced array SHA256:",
    reproduced_raw_sha,
)

print(
    "Frozen array SHA256:    ",
    frozen_raw_sha,
)


if not bitwise_equal:

    mismatch = np.flatnonzero(
        reproduced_p
        != frozen_p
    )

    first = int(
        mismatch[
            0
        ]
    )

    raise RuntimeError(
        "\nFULL-source sanity FAILED: source probability vector "
        "did not reproduce bit-for-bit.\n"
        f"First mismatch index: {first}\n"
        f"clean_position: {int(clean_position[first])}\n"
        f"reproduced: {float(reproduced_p[first])}\n"
        f"frozen:     {float(frozen_p[first])}\n"
        f"max abs diff: {max_abs_difference}\n\n"
        "DO NOT change model, data, backend, or thresholds. "
        "Report this output for forensic review."
    )


print()

print(
    "[PASS] Exact Stage22 source-validation probability vector reproduced."
)
print()


# ============================================================================
# 12. RANKING METRIC REPRODUCTION
# ============================================================================

print("=" * 108)
print("SOURCE RANKING METRICS")
print("=" * 108)


pr_auc = float(
    average_precision_score(
        y_val,
        reproduced_p,
    )
)


roc_auc = float(
    roc_auc_score(
        y_val,
        reproduced_p,
    )
)


frozen_pr_auc = float(
    average_precision_score(
        frozen_y,
        frozen_p,
    )
)


frozen_roc_auc = float(
    roc_auc_score(
        frozen_y,
        frozen_p,
    )
)


if pr_auc != frozen_pr_auc:

    raise RuntimeError(
        "PR-AUC reproduction mismatch."
    )


if roc_auc != frozen_roc_auc:

    raise RuntimeError(
        "ROC-AUC reproduction mismatch."
    )


# Compare against Stage22 result if explicitly recorded.
recorded_validation_auc = (
    stage22.get(
        "validation_auc",
        {}
    )
)


if (
    "pr_auc"
    in recorded_validation_auc
):

    if not np.isclose(
        pr_auc,
        float(
            recorded_validation_auc[
                "pr_auc"
            ]
        ),
        rtol=0.0,
        atol=1e-15,
    ):

        raise RuntimeError(
            "PR-AUC differs from frozen Stage22 result JSON."
        )


if (
    "roc_auc"
    in recorded_validation_auc
):

    if not np.isclose(
        roc_auc,
        float(
            recorded_validation_auc[
                "roc_auc"
            ]
        ),
        rtol=0.0,
        atol=1e-15,
    ):

        raise RuntimeError(
            "ROC-AUC differs from frozen Stage22 result JSON."
        )


print(
    "PR-AUC:",
    format(
        pr_auc,
        ".15f",
    ),
)

print(
    "ROC-AUC:",
    format(
        roc_auc,
        ".15f",
    ),
)

print()

print(
    "[PASS] Source ranking metrics reproduced."
)
print()


# ============================================================================
# 13. FROZEN OPERATING-POINT REPRODUCTION
#
# NO THRESHOLD SELECTION.
# Only evaluate already-frozen Stage22 thresholds.
# ============================================================================

print("=" * 108)
print("FROZEN SOURCE OPERATING-POINT REPRODUCTION")
print("=" * 108)


operating_results = {}


for name in [
    "standard",
    "balanced",
    "security",
]:

    frozen_record = get_operating_record(
        stage22,
        name,
    )


    threshold = float(
        frozen_record.get(
            "threshold_float32_runtime",
            np.float32(
                frozen_record[
                    "threshold"
                ]
            ),
        )
    )


    actual_counts = confusion_counts(
        y_val,
        reproduced_p,
        threshold,
    )


    expected_counts = {
        key:
            int(
                frozen_record[
                    key
                ]
            )

        for key in [
            "tp",
            "tn",
            "fp",
            "fn",
        ]
    }


    exact = (
        actual_counts
        == expected_counts
    )


    print()

    print(
        name.upper()
    )

    print(
        "  threshold:",
        threshold,
    )

    print(
        "  expected:",
        expected_counts,
    )

    print(
        "  actual:  ",
        actual_counts,
    )

    print(
        "  exact:   ",
        exact,
    )


    if not exact:

        raise RuntimeError(
            f"{name}: frozen confusion counts did not reproduce."
        )


    operating_results[
        name
    ] = {
        "threshold_runtime":
            threshold,

        "expected":
            expected_counts,

        "reproduced":
            actual_counts,

        "exact":
            True,
    }


print()

print(
    "[PASS] All frozen Stage22 operating points reproduce exactly."
)
print()


# ============================================================================
# 14. WRITE STAGE24-1A RECEIPT
# ============================================================================

receipt = {

    "stage":
        "Stage24-1A",

    "type":
        "PRIMARY_FULL_SOURCE_70F_SANITY",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "execution_parent":
        head,

    "frozen_stage24_protocol": {

        "path":
            str(
                STAGE24_LOCK.relative_to(
                    REPO
                )
            ),

        "sha256":
            sha256_file(
                STAGE24_LOCK
            ),
    },

    "stage24_0d_attestation": {

        "path":
            str(
                STAGE24_ATTESTATION.relative_to(
                    REPO
                )
            ),

        "sha256":
            sha256_file(
                STAGE24_ATTESTATION
            ),

        "governance_gate_pass":
            True,
    },

    "source": {

        "dataset":
            "CSE-CIC-IDS2018",

        "split":
            "Stage22R CHRONOLOGICAL_NATURAL validation",

        "day_id":
            7,

        "source_file":
            "02-28-2018.csv",

        "cache_file":
            EXPECTED_CACHE_FILE,

        "cache_runtime_path":
            str(
                DAY07
            ),

        "cache_sha256":
            EXPECTED_CACHE_SHA,

        "rows":
            EXPECTED_ROWS,

        "benign":
            EXPECTED_BENIGN,

        "attack":
            EXPECTED_ATTACK,

        "feature_count":
            70,

        "feature_order":
            features,

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",
    },

    "models": {

        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_rule":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "lightgbm": {

            "path":
                str(
                    LGB_MODEL.relative_to(
                        REPO
                    )
                ),

            "sha256":
                actual_lgb_sha,

            "features":
                70,

            "trees":
                int(
                    lgb_booster.num_trees()
                ),
        },

        "xgboost": {

            "path":
                str(
                    XGB_MODEL.relative_to(
                        REPO
                    )
                ),

            "sha256":
                actual_xgb_sha,

            "features":
                70,

            "boosted_rounds":
                int(
                    xgb_booster.num_boosted_rounds()
                ),

            "inference_device":
                "cuda",
        },
    },

    "frozen_probability_reference": {

        "path":
            str(
                FROZEN_PROB.relative_to(
                    REPO
                )
            ),

        "file_sha256":
            actual_prob_sha,

        "rows":
            EXPECTED_ROWS,

        "dtype":
            "float32",

        "array_sha256":
            frozen_raw_sha,
    },

    "source_inference_reproduction": {

        "rows":
            EXPECTED_ROWS,

        "inference_seconds":
            float(
                inference_seconds
            ),

        "ensemble_array_sha256":
            reproduced_raw_sha,

        "bitwise_equal_to_frozen":
            True,

        "differing_positions":
            0,

        "max_absolute_difference":
            0.0,
    },

    "source_metrics": {

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "operating_points":
            operating_results,
    },

    "sanity_gate": {

        "model_hashes_exact":
            True,

        "feature_order_exact":
            True,

        "source_cache_exact":
            True,

        "source_membership_exact":
            True,

        "source_labels_exact":
            True,

        "probability_vector_bitwise_exact":
            True,

        "ranking_metrics_exact":
            True,

        "operating_points_exact":
            True,

        "pass":
            True,
    },

    "scientific_access": {

        "scientific_model_fits":
            0,

        "source_model_inference_rows":
            EXPECTED_ROWS,

        "target_rows_read":
            0,

        "target_labels_read":
            0,

        "target_predictions":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,

        "target_opening_budget":
            8,
    },

    "next_authorized_step":
        (
            "Stage24-1B — PRIMARY bridge62 source refit using exactly "
            "the frozen Stage22R CHRONOLOGICAL_NATURAL train/validation "
            "membership, inherited LGBM/XGB hyperparameters, GPU/CUDA "
            "backends, no tuning, and source-validation-only thresholds."
        ),
}


receipt_sha = dump_json_with_sha(
    RECEIPT,
    receipt,
)


print("=" * 108)
print("STAGE24-1A RECEIPT")
print("=" * 108)

print(
    "Receipt:"
)

print(
    " ",
    RECEIPT,
)

print(
    "SHA256:"
)

print(
    " ",
    receipt_sha,
)
print()


# ============================================================================
# 15. COMMIT + PUSH RECEIPT
# ============================================================================

print("=" * 108)
print("COMMIT + PUSH")
print("=" * 108)


# GitHub token
github_token = None
token_source = None


try:

    from kaggle_secrets import (
        UserSecretsClient,
    )

    client = UserSecretsClient()

    for candidate in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = client.get_secret(
                candidate
            )

            if (
                value
                and value.strip()
            ):

                github_token = (
                    value.strip()
                )

                token_source = (
                    candidate
                )

                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for candidate in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            candidate
        )

        if (
            value
            and value.strip()
        ):

            github_token = (
                value.strip()
            )

            token_source = (
                f"ENV:{candidate}"
            )

            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable."
    )


basic = base64.b64encode(
    (
        "x-access-token:"
        + github_token
    ).encode(
        "utf-8"
    )
).decode(
    "ascii"
)


print(
    "Credential source:",
    token_source,
)


git(
    "add",
    "--",
    str(
        OUTDIR_REL
    ),
)


staged = [
    line
    for line
    in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
]


expected_staged = {
    str(
        RECEIPT.relative_to(
            REPO
        )
    ),
    str(
        RECEIPT_SHA.relative_to(
            REPO
        )
    ),
}


if set(
    staged
) != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged file set.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


git(
    "commit",
    "-m",
    "stage24: verify primary full-source 70f sanity",
)


science_commit = git(
    "rev-parse",
    "HEAD",
)


science_parent = git(
    "rev-parse",
    "HEAD^",
)


if (
    science_parent
    != EXPECTED_PARENT
):

    raise RuntimeError(
        "Stage24-1A commit parent mismatch."
    )


push_output = git(
    "push",
    "origin",
    f"HEAD:{BRANCH}",
    auth_header=basic,
)


print(
    push_output
)


remote_after = git(
    "ls-remote",
    "origin",
    f"refs/heads/{BRANCH}",
    auth_header=basic,
).split()[0]


if (
    remote_after
    != science_commit
):

    raise RuntimeError(
        "\nRemote commit verification failed.\n"
        f"Local:  {science_commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nWorktree not clean after Stage24-1A:\n"
        + final_status
    )


print()

print(
    "[PASS] Stage24-1A receipt committed and pushed."
)

print(
    "Commit:",
    science_commit,
)

print(
    "Remote main:",
    remote_after,
)

print(
    "Worktree:",
    "<clean>",
)
print()


# ============================================================================
# 16. FINAL SYNTHESIS
# ============================================================================

print("=" * 108)
print("STAGE24-1A FINAL SYNTHESIS")
print("=" * 108)

print(
    "Source dataset:"
)

print(
    "  IDS2018"
)

print(
    "Source validation:"
)

print(
    "  02-28-2018 / 593,780 K79-clean rows"
)

print()

print(
    "Representation:"
)

print(
    "  FULL / 70 features"
)

print()

print(
    "Models:"
)

print(
    "  exact frozen Stage22R LightGBM + XGBoost"
)

print(
    "Scientific model fits:"
)

print(
    "  0"
)

print()

print(
    "Probability reproduction:"
)

print(
    "  BIT-FOR-BIT EXACT"
)

print(
    "Differing positions:"
)

print(
    "  0"
)

print()

print(
    "PR-AUC:"
)

print(
    " ",
    format(
        pr_auc,
        ".15f",
    ),
)

print(
    "ROC-AUC:"
)

print(
    " ",
    format(
        roc_auc,
        ".15f",
    ),
)

print()

print(
    "Frozen operating points:"
)

for name in [
    "standard",
    "balanced",
    "security",
]:

    print(
        f"  {name:<8}: EXACT"
    )


print()

print(
    "Target rows read:"
)

print(
    "  0"
)

print(
    "Target labels read:"
)

print(
    "  0"
)

print(
    "Target predictions:"
)

print(
    "  0"
)

print(
    "Target openings:"
)

print(
    "  0 / 8"
)

print()

print(
    "Stage24-1A commit:"
)

print(
    " ",
    science_commit,
)

print()

print("-" * 108)

print(
    "STAGE24-1A: PASS"
)

print()

print(
    "FULL-source sanity reproduces the frozen Stage22R source behavior exactly."
)

print(
    "No Stage24 target has been opened."
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "Stage24-1B — PRIMARY bridge62 source refit "
    "(1 LightGBM GPU fit + 1 XGBoost CUDA fit)."
)

print(
    "Target opening ledger remains 0 / 8."
)

print("-" * 108)

print(
    "STAGE24-1A COMPLETE"
)

print("=" * 108)


In [ ]:
# ============================================================================
# STAGE24-1A-R1
# PRIMARY FULL-SOURCE 70F SANITY — DIRECT DAY-7 RECONSTRUCTION
#
# FAST RECOVERY VERSION
#
# NO cache search
# NO cache regeneration
# NO other IDS2018 days
# NO model fitting
# NO CICIDS2017 access
# NO target opening
#
# Reconstruct ONLY:
#   IDS2018 02-28-2018
#   613,104 physical data rows
#   - 33 embedded header rows
#   - 19,291 frozen K79 exclusions
#   = 593,780 exact Stage22 validation rows
#
# Then:
#   - verify labels/membership against frozen Stage22 probability artifact
#   - run exact frozen 70F LightGBM + XGBoost
#   - require BIT-FOR-BIT ensemble probability reproduction
#   - reproduce frozen operating points
#   - write/commit/push Stage24-1A receipt
#
# Scientific fits: 0
# Target openings:  0 / 8
# ============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import lightgbm as lgb
import xgboost as xgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================================
# 0. CONSTANTS
# ============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "3a0b4a54477a5159805e69caa1eba749ab614c7b"
)

BRANCH = "main"


RAW_DAY7 = Path(
    "/kaggle/input/datasets/"
    "solarmainframe/ids-intrusion-csv/"
    "02-28-2018.csv"
)

EXPECTED_RAW_SHA = (
    "f15e2a12304446058a0186c8ad67de2bd15735a9ba5c70c9a1f4c4242ab06771"
)


STAGE22_DIR = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
)

STAGE22_RESULT = (
    STAGE22_DIR
    / "stage22r_2c_chronological_natural_result.json"
)

LGB_MODEL = (
    STAGE22_DIR
    / "chronological_natural_lightgbm_model.txt"
)

XGB_MODEL = (
    STAGE22_DIR
    / "chronological_natural_xgboost_model.json"
)

FROZEN_PROB = (
    STAGE22_DIR
    / "chronological_natural_validation_ensemble_probabilities.npz"
)


K79_EXCLUSIONS = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1a3_k79_development_freeze"
    / "stage22r_k79_development_exclusions.parquet"
)

EXPECTED_K79_SHA = (
    "3e979841468f9d42a8de7049ed50063d5907c0753fe5b1179dac948163d93134"
)


STAGE24_LOCK = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)

STAGE24_ATTESTATION = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0d_remote_verification_receipt.json"
)


OUT_REL = (
    Path("results")
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
)

OUTDIR = (
    REPO
    / OUT_REL
)

OUTDIR.mkdir(
    parents=True,
    exist_ok=True,
)

RECEIPT = (
    OUTDIR
    / "stage24_1a_full_source_70f_sanity.json"
)

RECEIPT_SHA = (
    OUTDIR
    / "stage24_1a_full_source_70f_sanity.sha256"
)


EXPECTED_PHYSICAL_ROWS = 613_104
EXPECTED_EMBEDDED_HEADERS = 33
EXPECTED_EFFECTIVE_ROWS = 613_071

EXPECTED_EXCLUSIONS = 19_291

EXPECTED_RETAINED_ROWS = 593_780
EXPECTED_BENIGN = 531_524
EXPECTED_ATTACK = 62_256

EXPECTED_CLEAN_START = 13_818_623
EXPECTED_CLEAN_STOP = 14_412_403


EXPECTED_LGB_SHA = (
    "7be4c610814e45be2e315996969d2e2f404a3ada14866b21a731e4382fdc18b8"
)

EXPECTED_XGB_SHA = (
    "38499d0edcdc5c6d0b2c3afee913558973d35d5ebf1cb2ae888a16b8da98756c"
)

EXPECTED_PROB_FILE_SHA = (
    "3fe6c468a0653ac5ee488da8d6586fd86628e9586eb01f6f5d962e35fff65e3f"
)


print("=" * 108)
print("STAGE24-1A-R1 — DIRECT DAY-7 FULL-SOURCE SANITY")
print("=" * 108)

print("Scientific fits:       0")
print("CICIDS2017 rows read:  0")
print("Target predictions:    0")
print("Target openings:       0 / 8")
print()


# ============================================================================
# 1. HELPERS
# ============================================================================

def git(
    *args,
    auth_header=None,
    check=True,
):
    command = ["git"]

    if auth_header is not None:
        command.extend(
            [
                "-c",
                "credential.helper=",
                "-c",
                f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
            ]
        )

    command.extend(
        map(str, args)
    )

    p = subprocess.run(
        command,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "\nGit failure:\n"
            + " ".join(command)
            + "\n\n"
            + (p.stdout or "")
        )

    return (
        p.stdout or ""
    ).strip()


def load_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path: Path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def array_sha256(a):
    a = np.ascontiguousarray(
        a
    )

    h = hashlib.sha256()

    h.update(
        str(
            a.dtype
        ).encode()
    )

    h.update(
        repr(
            tuple(a.shape)
        ).encode()
    )

    h.update(
        a.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


def write_json_sha(
    path: Path,
    obj,
):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    digest = sha256_file(
        path
    )

    path.with_suffix(
        ".sha256"
    ).write_text(
        f"{digest}  {path.name}\n",
        encoding="utf-8",
    )

    return digest


def confusion(
    y,
    p,
    threshold,
):
    prediction = (
        p >= threshold
    )

    positive = (
        y == 1
    )

    return {
        "tp": int(
            np.count_nonzero(
                prediction & positive
            )
        ),

        "tn": int(
            np.count_nonzero(
                ~prediction & ~positive
            )
        ),

        "fp": int(
            np.count_nonzero(
                prediction & ~positive
            )
        ),

        "fn": int(
            np.count_nonzero(
                ~prediction & positive
            )
        ),
    }


def operating_record(
    result,
    name,
):
    record = (
        result[
            "operating_points"
        ][name]
    )

    if (
        name == "security"
        and isinstance(
            record,
            dict,
        )
        and "result" in record
    ):
        return record[
            "result"
        ]

    return record


def resolve_column(
    columns,
    candidates,
):
    by_lower = {
        str(c).lower():
            c
        for c in columns
    }

    for candidate in candidates:
        found = by_lower.get(
            candidate.lower()
        )

        if found is not None:
            return found

    return None


# ============================================================================
# 2. GOVERNANCE
# ============================================================================

print("=" * 108)
print("GOVERNANCE")
print("=" * 108)


branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
).split()[0]

status = git(
    "status",
    "--porcelain",
)


print(
    "Branch:",
    branch,
)

print(
    "Local HEAD:",
    head,
)

print(
    "Remote main:",
    remote,
)


if branch != BRANCH:
    raise RuntimeError(
        "Wrong branch."
    )

if head != EXPECTED_HEAD:
    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )

if remote != head:
    raise RuntimeError(
        "Local HEAD != remote main."
    )

if status:
    raise RuntimeError(
        "\nWorktree must be clean:\n"
        + status
    )


protocol = load_json(
    STAGE24_LOCK
)

attestation = load_json(
    STAGE24_ATTESTATION
)


if not attestation[
    "governance_gate"
][
    "gate_pass"
]:
    raise RuntimeError(
        "Stage24-0D gate not passed."
    )


if (
    attestation[
        "target_opening_ledger"
    ][
        "consumed"
    ]
    != 0
):
    raise RuntimeError(
        "Opening ledger not pristine."
    )


print()

print(
    "[PASS] Stage24 execution authorized."
)

print(
    "[PASS] Target openings = 0 / 8."
)
print()


# ============================================================================
# 3. VERIFY EXACT RAW + FROZEN ARTIFACTS
# ============================================================================

print("=" * 108)
print("EXACT SOURCE ARTIFACTS")
print("=" * 108)


required = [
    RAW_DAY7,
    K79_EXCLUSIONS,
    STAGE22_RESULT,
    LGB_MODEL,
    XGB_MODEL,
    FROZEN_PROB,
]


for path in required:
    if not path.is_file():
        raise FileNotFoundError(
            path
        )


raw_sha = sha256_file(
    RAW_DAY7
)

k79_sha = sha256_file(
    K79_EXCLUSIONS
)

lgb_sha = sha256_file(
    LGB_MODEL
)

xgb_sha = sha256_file(
    XGB_MODEL
)

prob_sha = sha256_file(
    FROZEN_PROB
)


checks = [
    (
        "02-28 raw",
        raw_sha,
        EXPECTED_RAW_SHA,
    ),

    (
        "K79 exclusions",
        k79_sha,
        EXPECTED_K79_SHA,
    ),

    (
        "LightGBM",
        lgb_sha,
        EXPECTED_LGB_SHA,
    ),

    (
        "XGBoost",
        xgb_sha,
        EXPECTED_XGB_SHA,
    ),

    (
        "Frozen probabilities",
        prob_sha,
        EXPECTED_PROB_FILE_SHA,
    ),
]


for name, actual, expected in checks:

    print(
        name
    )

    print(
        "  expected:",
        expected,
    )

    print(
        "  actual:  ",
        actual,
    )

    if actual != expected:
        raise RuntimeError(
            f"{name} SHA mismatch."
        )


print()

print(
    "[PASS] All source artifacts are byte-exact."
)
print()


# ============================================================================
# 4. FROZEN FEATURE ORDER
# ============================================================================

stage22 = load_json(
    STAGE22_RESULT
)

features = list(
    stage22[
        "data"
    ][
        "feature_order"
    ]
)


if len(
    features
) != 70:
    raise RuntimeError(
        "Stage22 feature count != 70."
    )


if (
    stage22[
        "data"
    ][
        "scaling"
    ]
    != "NONE"
):
    raise RuntimeError(
        "Unexpected scaling."
    )


if (
    stage22[
        "data"
    ][
        "explicit_imputation"
    ]
    != "NONE"
):
    raise RuntimeError(
        "Unexpected imputation."
    )


print("=" * 108)
print("FEATURE CONTRACT")
print("=" * 108)

print(
    "Features:",
    len(features),
)

print(
    "Scaling: NONE"
)

print(
    "Imputation: NONE"
)

print(
    "[PASS] Frozen 70F contract recovered."
)
print()


# ============================================================================
# 5. READ ONLY CSV HEADER FIRST
# ============================================================================

print("=" * 108)
print("02-28 HEADER RESOLUTION")
print("=" * 108)


raw_header = pd.read_csv(
    RAW_DAY7,
    nrows=0,
)


raw_columns = list(
    raw_header.columns
)


strip_map = {}

for original in raw_columns:

    canonical = str(
        original
    ).strip()

    if canonical in strip_map:
        raise RuntimeError(
            "CSV header collision after strip."
        )

    strip_map[
        canonical
    ] = original


required_columns = (
    features
    + ["Label"]
)


missing = [
    column
    for column in required_columns
    if column not in strip_map
]


if missing:
    raise RuntimeError(
        "Missing required raw columns:\n"
        + repr(missing)
    )


label_raw_name = (
    strip_map[
        "Label"
    ]
)


print(
    "Physical columns:",
    len(raw_columns),
)

print(
    "Required 70F + Label:",
    len(required_columns),
)

print(
    "[PASS] Required source columns present."
)
print()


# ============================================================================
# 6. FAST FIRST PASS — FIND EMBEDDED HEADERS USING LABEL ONLY
# ============================================================================

print("=" * 108)
print("EMBEDDED HEADER DETECTION")
print("=" * 108)


start = time.time()


label_only = pd.read_csv(
    RAW_DAY7,
    usecols=[
        label_raw_name
    ],
    dtype="string",
    low_memory=False,
)


physical_rows = len(
    label_only
)


if (
    physical_rows
    != EXPECTED_PHYSICAL_ROWS
):
    raise RuntimeError(
        "\n02-28 physical row count mismatch.\n"
        f"Expected: {EXPECTED_PHYSICAL_ROWS}\n"
        f"Actual:   {physical_rows}"
    )


label_series = (
    label_only.iloc[
        :,
        0
    ]
    .astype(
        "string"
    )
    .str.strip()
)


embedded_mask = (
    label_series
    .eq(
        "Label"
    )
    .fillna(
        False
    )
    .to_numpy()
)


embedded_indices = np.flatnonzero(
    embedded_mask
).astype(
    np.int64
)


if (
    len(
        embedded_indices
    )
    != EXPECTED_EMBEDDED_HEADERS
):
    raise RuntimeError(
        "\nEmbedded header count mismatch.\n"
        f"Expected: {EXPECTED_EMBEDDED_HEADERS}\n"
        f"Actual:   {len(embedded_indices)}"
    )


header_scan_seconds = (
    time.time()
    - start
)


print(
    "Physical rows:",
    physical_rows,
)

print(
    "Embedded headers:",
    len(embedded_indices),
)

print(
    "Header scan seconds:",
    round(
        header_scan_seconds,
        3,
    ),
)

print(
    "[PASS] Exactly 33 embedded headers located."
)
print()


del label_only
del label_series
gc.collect()


# ============================================================================
# 7. SECOND PASS — READ ONLY 70 FEATURES + LABEL, SKIPPING 33 HEADERS
#
# pandas skiprows uses physical FILE line numbers:
#   line 0 = real CSV header
#   data index i = file line i+1
# ============================================================================

print("=" * 108)
print("READING EFFECTIVE 02-28 SOURCE ROWS")
print("=" * 108)


skip_file_lines = (
    embedded_indices
    + 1
).tolist()


raw_usecols = [
    strip_map[
        column
    ]
    for column
    in required_columns
]


start = time.time()


df = pd.read_csv(
    RAW_DAY7,
    usecols=raw_usecols,
    skiprows=skip_file_lines,
    low_memory=False,
)


read_seconds = (
    time.time()
    - start
)


df.rename(
    columns={
        original:
            str(original).strip()
        for original
        in df.columns
    },
    inplace=True,
)


# Force exact canonical requested ordering.
df = df[
    required_columns
]


if (
    len(df)
    != EXPECTED_EFFECTIVE_ROWS
):
    raise RuntimeError(
        "\nEffective row count mismatch.\n"
        f"Expected: {EXPECTED_EFFECTIVE_ROWS}\n"
        f"Actual:   {len(df)}"
    )


# Original physical data-row indices retained after header removal.
all_original_indices = np.arange(
    EXPECTED_PHYSICAL_ROWS,
    dtype=np.int64,
)


nonheader_mask = np.ones(
    EXPECTED_PHYSICAL_ROWS,
    dtype=bool,
)

nonheader_mask[
    embedded_indices
] = False


original_indices_effective = (
    all_original_indices[
        nonheader_mask
    ]
)


if (
    len(
        original_indices_effective
    )
    != EXPECTED_EFFECTIVE_ROWS
):
    raise RuntimeError(
        "Effective original-index mapping failed."
    )


# Verify pre-K79 class counts.
label_clean = (
    df[
        "Label"
    ]
    .astype(
        "string"
    )
    .str.strip()
)


if (
    label_clean.isna().any()
):
    raise RuntimeError(
        "Null source label encountered."
    )


y_effective = (
    ~label_clean
    .str.casefold()
    .eq(
        "benign"
    )
).to_numpy(
    dtype=np.uint8
)


effective_attack = int(
    y_effective.sum()
)

effective_benign = int(
    len(
        y_effective
    )
    - effective_attack
)


if (
    effective_attack
    != 68_871
    or effective_benign
    != 544_200
):
    raise RuntimeError(
        "\nPre-K79 class counts mismatch.\n"
        f"Benign: {effective_benign}\n"
        f"Attack: {effective_attack}"
    )


print(
    "Effective rows:",
    len(df),
)

print(
    "Pre-K79 benign:",
    effective_benign,
)

print(
    "Pre-K79 attack:",
    effective_attack,
)

print(
    "Read seconds:",
    round(
        read_seconds,
        3,
    ),
)

print()

print(
    "[PASS] Exact effective 02-28 population reconstructed."
)
print()


# ============================================================================
# 8. LOAD FROZEN K79 LOCATORS
# ============================================================================

print("=" * 108)
print("FROZEN K79 DAY-7 EXCLUSIONS")
print("=" * 108)


k79_table = pq.read_table(
    K79_EXCLUSIONS
)

k79 = (
    k79_table
    .to_pandas()
)


print(
    "K79 columns:",
    list(
        k79.columns
    ),
)

print(
    "K79 total rows:",
    len(k79),
)


if (
    len(k79)
    != 440_865
):
    raise RuntimeError(
        "Total frozen K79 exclusion count != 440865."
    )


day_col = resolve_column(
    k79.columns,
    [
        "day_id",
        "source_day_id",
    ],
)


file_col = resolve_column(
    k79.columns,
    [
        "source_file",
        "file",
        "filename",
    ],
)


index_col = resolve_column(
    k79.columns,
    [
        "original_row_index",
        "original_zero_based_row_index",
        "source_row_index",
        "physical_row_index",
        "row_index",
    ],
)


if index_col is None:
    raise RuntimeError(
        "\nCannot resolve frozen K79 source-row locator column.\n"
        f"Columns: {list(k79.columns)}"
    )


if day_col is not None:

    day7_k79 = k79.loc[
        pd.to_numeric(
            k79[
                day_col
            ],
            errors="raise",
        ).astype(
            np.int64
        )
        == 7
    ].copy()

elif file_col is not None:

    day7_k79 = k79.loc[
        k79[
            file_col
        ]
        .astype(str)
        .map(
            lambda value:
                Path(
                    value
                ).name
        )
        .eq(
            "02-28-2018.csv"
        )
    ].copy()

else:

    raise RuntimeError(
        "K79 artifact exposes neither day_id nor source_file."
    )


exclusion_indices = pd.to_numeric(
    day7_k79[
        index_col
    ],
    errors="raise",
).to_numpy(
    dtype=np.int64
)


if (
    len(
        exclusion_indices
    )
    != EXPECTED_EXCLUSIONS
):
    raise RuntimeError(
        "\nDay-7 K79 exclusion count mismatch.\n"
        f"Expected: {EXPECTED_EXCLUSIONS}\n"
        f"Actual:   {len(exclusion_indices)}"
    )


if (
    len(
        np.unique(
            exclusion_indices
        )
    )
    != EXPECTED_EXCLUSIONS
):
    raise RuntimeError(
        "Duplicate K79 exclusion locators detected."
    )


if (
    exclusion_indices.min()
    < 0
    or exclusion_indices.max()
    >= EXPECTED_PHYSICAL_ROWS
):
    raise RuntimeError(
        "K79 source-row locator outside 02-28 bounds."
    )


header_overlap = np.intersect1d(
    exclusion_indices,
    embedded_indices,
)


if len(
    header_overlap
) != 0:
    raise RuntimeError(
        "\nK79 exclusions unexpectedly overlap embedded headers:\n"
        + repr(
            header_overlap.tolist()
        )
    )


print(
    "Resolved day selector:",
    (
        day_col
        if day_col is not None
        else file_col
    ),
)

print(
    "Resolved row locator:",
    index_col,
)

print(
    "Day-7 exclusions:",
    len(exclusion_indices),
)

print(
    "Header overlap:",
    len(header_overlap),
)

print(
    "[PASS] Frozen day-7 K79 locators recovered."
)
print()


del k79
del day7_k79
del k79_table
gc.collect()


# ============================================================================
# 9. APPLY EXACT K79 EXCLUSIONS
# ============================================================================

print("=" * 108)
print("APPLYING K79 SOURCE-ROW LOCATORS")
print("=" * 108)


exclude_effective = np.isin(
    original_indices_effective,
    exclusion_indices,
    assume_unique=False,
)


if int(
    exclude_effective.sum()
) != EXPECTED_EXCLUSIONS:

    raise RuntimeError(
        "\nNot all frozen day-7 exclusions were found "
        "in effective source rows."
    )


retain_mask = (
    ~exclude_effective
)


retained_original_indices = (
    original_indices_effective[
        retain_mask
    ]
)


clean_df = (
    df.loc[
        retain_mask,
        required_columns,
    ]
    .reset_index(
        drop=True
    )
)


if (
    len(clean_df)
    != EXPECTED_RETAINED_ROWS
):
    raise RuntimeError(
        "\nRetained row count mismatch.\n"
        f"Expected: {EXPECTED_RETAINED_ROWS}\n"
        f"Actual:   {len(clean_df)}"
    )


y = y_effective[
    retain_mask
]


attack = int(
    y.sum()
)

benign = int(
    len(y)
    - attack
)


if attack != EXPECTED_ATTACK:
    raise RuntimeError(
        f"Attack count mismatch: {attack}"
    )


if benign != EXPECTED_BENIGN:
    raise RuntimeError(
        f"Benign count mismatch: {benign}"
    )


clean_position = np.arange(
    EXPECTED_CLEAN_START,
    EXPECTED_CLEAN_STOP,
    dtype=np.int64,
)


if (
    len(clean_position)
    != EXPECTED_RETAINED_ROWS
):
    raise RuntimeError(
        "clean_position construction mismatch."
    )


print(
    "Excluded:",
    int(
        exclude_effective.sum()
    ),
)

print(
    "Retained:",
    len(clean_df),
)

print(
    "Benign:",
    benign,
)

print(
    "Attack:",
    attack,
)

print(
    "clean_position:",
    int(clean_position[0]),
    "->",
    int(clean_position[-1]),
)

print()

print(
    "[PASS] Exact frozen day-7 membership reconstructed."
)
print()


del df
del y_effective
del exclude_effective
del retain_mask
del original_indices_effective
gc.collect()


# ============================================================================
# 10. NUMERIC POLICY — FROZEN STAGE22
#
# float64
# +inf -> NaN
# -inf -> NaN
# other nonnumeric token -> FAIL
# no imputation
# no scaling
# ============================================================================

print("=" * 108)
print("NUMERIC MODEL INPUT RECONSTRUCTION")
print("=" * 108)


for column in features:

    if not pd.api.types.is_numeric_dtype(
        clean_df[
            column
        ]
    ):

        clean_df[
            column
        ] = pd.to_numeric(
            clean_df[
                column
            ],
            errors="raise",
        )


X = clean_df[
    features
].to_numpy(
    dtype=np.float64,
    copy=True,
)


positive_inf = int(
    np.isposinf(
        X
    ).sum()
)

negative_inf = int(
    np.isneginf(
        X
    ).sum()
)


X[
    np.isinf(X)
] = np.nan


nan_values = int(
    np.isnan(
        X
    ).sum()
)


if X.shape != (
    EXPECTED_RETAINED_ROWS,
    70,
):
    raise RuntimeError(
        f"Unexpected X shape: {X.shape}"
    )


print(
    "X shape:",
    X.shape,
)

print(
    "X dtype:",
    X.dtype,
)

print(
    "+inf -> NaN:",
    positive_inf,
)

print(
    "-inf -> NaN:",
    negative_inf,
)

print(
    "Final NaNs:",
    nan_values,
)

print()

print(
    "[PASS] Frozen Stage22 numeric policy applied."
)
print()


del clean_df
gc.collect()


# ============================================================================
# 11. FROZEN PROBABILITY REFERENCE — MEMBERSHIP/LABEL GATE
# ============================================================================

print("=" * 108)
print("FROZEN SOURCE VALIDATION REFERENCE")
print("=" * 108)


with np.load(
    FROZEN_PROB,
    allow_pickle=False,
) as z:

    print(
        "NPZ keys:",
        z.files,
    )

    frozen_clean = np.asarray(
        z[
            "clean_position"
        ],
        dtype=np.int64,
    )

    frozen_y = np.asarray(
        z[
            "binary_label"
        ],
        dtype=np.uint8,
    )

    frozen_p = np.asarray(
        z[
            "ensemble_probability"
        ],
        dtype=np.float32,
    )


if not np.array_equal(
    clean_position,
    frozen_clean,
):
    raise RuntimeError(
        "clean_position differs from frozen Stage22 reference."
    )


if not np.array_equal(
    y,
    frozen_y,
):
    mismatches = np.flatnonzero(
        y != frozen_y
    )

    raise RuntimeError(
        "\nReconstructed labels differ from frozen Stage22 reference.\n"
        f"Mismatch count: {len(mismatches)}\n"
        f"First mismatch: "
        f"{int(mismatches[0]) if len(mismatches) else None}"
    )


if (
    frozen_p.shape
    != (
        EXPECTED_RETAINED_ROWS,
    )
):
    raise RuntimeError(
        "Frozen probability length mismatch."
    )


print(
    "Membership exact:",
    True,
)

print(
    "Labels exact:",
    True,
)

print(
    "Rows:",
    len(frozen_p),
)

print(
    "Probability dtype:",
    frozen_p.dtype,
)

print()

print(
    "[PASS] Source membership and labels match frozen Stage22 exactly."
)
print()


# ============================================================================
# 12. LOAD EXACT MODELS
# ============================================================================

print("=" * 108)
print("LOADING FROZEN 70F MODELS")
print("=" * 108)


lgb_model = lgb.Booster(
    model_file=str(
        LGB_MODEL
    )
)


if (
    int(
        lgb_model.num_feature()
    )
    != 70
):
    raise RuntimeError(
        "LightGBM feature count != 70."
    )


if (
    list(
        lgb_model.feature_name()
    )
    != features
):
    raise RuntimeError(
        "LightGBM feature order mismatch."
    )


xgb_model = xgb.Booster()

xgb_model.load_model(
    str(
        XGB_MODEL
    )
)


if (
    int(
        xgb_model.num_features()
    )
    != 70
):
    raise RuntimeError(
        "XGBoost feature count != 70."
    )


if (
    xgb_model.feature_names is not None
    and list(
        xgb_model.feature_names
    )
    != features
):
    raise RuntimeError(
        "XGBoost feature order mismatch."
    )


# Frozen Stage22 XGB execution backend.
xgb_model.set_param(
    {
        "device":
            "cuda",
    }
)


print(
    "LightGBM trees:",
    lgb_model.num_trees(),
)

print(
    "XGBoost rounds:",
    xgb_model.num_boosted_rounds(),
)

print(
    "[PASS] Frozen models loaded."
)
print()


# ============================================================================
# 13. SOURCE INFERENCE — NO FIT, NO TARGET OPENING
# ============================================================================

print("=" * 108)
print("SOURCE-ONLY INFERENCE")
print("=" * 108)


start = time.time()


p_lgb = np.asarray(
    lgb_model.predict(
        X
    ),
    dtype=np.float64,
)


dmatrix = xgb.DMatrix(
    X,
    feature_names=features,
    missing=np.nan,
)


p_xgb = np.asarray(
    xgb_model.predict(
        dmatrix
    ),
    dtype=np.float64,
)


p = (
    0.5 * p_lgb
    + 0.5 * p_xgb
).astype(
    np.float32
)


inference_seconds = (
    time.time()
    - start
)


if not np.all(
    np.isfinite(p)
):
    raise RuntimeError(
        "Non-finite ensemble probabilities."
    )


print(
    "Rows:",
    len(p),
)

print(
    "Inference seconds:",
    round(
        inference_seconds,
        3,
    ),
)

print(
    "Output dtype:",
    p.dtype,
)
print()


# ============================================================================
# 14. BIT-FOR-BIT GATE
# ============================================================================

print("=" * 108)
print("BIT-FOR-BIT FROZEN PROBABILITY GATE")
print("=" * 108)


exact_probability = np.array_equal(
    p,
    frozen_p,
)


difference_indices = np.flatnonzero(
    p != frozen_p
)


difference_count = int(
    len(
        difference_indices
    )
)


max_abs_difference = float(
    np.max(
        np.abs(
            p.astype(
                np.float64
            )
            -
            frozen_p.astype(
                np.float64
            )
        )
    )
)


reproduced_array_sha = (
    array_sha256(
        p
    )
)

frozen_array_sha = (
    array_sha256(
        frozen_p
    )
)


print(
    "Bitwise exact:",
    exact_probability,
)

print(
    "Differing positions:",
    difference_count,
)

print(
    "Max absolute difference:",
    max_abs_difference,
)

print(
    "Reproduced array SHA:",
    reproduced_array_sha,
)

print(
    "Frozen array SHA:    ",
    frozen_array_sha,
)


if not exact_probability:

    first = int(
        difference_indices[
            0
        ]
    )

    raise RuntimeError(
        "\nSTAGE24-1A SOURCE SANITY FAILED.\n"
        "The directly reconstructed source representation did not "
        "reproduce the frozen Stage22 probability vector bit-for-bit.\n\n"
        f"First mismatch retained index: {first}\n"
        f"Original source row index: "
        f"{int(retained_original_indices[first])}\n"
        f"Clean position: "
        f"{int(clean_position[first])}\n"
        f"Reproduced: {float(p[first])}\n"
        f"Frozen:     {float(frozen_p[first])}\n"
        f"Max abs difference: {max_abs_difference}\n\n"
        "DO NOT alter the preprocessing or model. "
        "Send this output for forensic review."
    )


print()

print(
    "[PASS] FROZEN STAGE22 PROBABILITY VECTOR REPRODUCED BIT-FOR-BIT."
)
print()


# ============================================================================
# 15. SOURCE METRICS
# ============================================================================

print("=" * 108)
print("SOURCE METRIC SANITY")
print("=" * 108)


pr_auc = float(
    average_precision_score(
        y,
        p,
    )
)

roc_auc = float(
    roc_auc_score(
        y,
        p,
    )
)


print(
    "PR-AUC:",
    format(
        pr_auc,
        ".15f",
    ),
)

print(
    "ROC-AUC:",
    format(
        roc_auc,
        ".15f",
    ),
)
print()


# ============================================================================
# 16. OPERATING POINTS — ALREADY FROZEN, NO SELECTION
# ============================================================================

print("=" * 108)
print("FROZEN OPERATING POINT REPRODUCTION")
print("=" * 108)


op_results = {}


for name in [
    "standard",
    "balanced",
    "security",
]:

    frozen_record = operating_record(
        stage22,
        name,
    )


    threshold = float(
        frozen_record.get(
            "threshold_float32_runtime",
            np.float32(
                frozen_record[
                    "threshold"
                ]
            ),
        )
    )


    actual_counts = confusion(
        y,
        p,
        threshold,
    )


    expected_counts = {
        key:
            int(
                frozen_record[
                    key
                ]
            )
        for key
        in [
            "tp",
            "tn",
            "fp",
            "fn",
        ]
    }


    exact = (
        actual_counts
        == expected_counts
    )


    print(
        name.upper()
    )

    print(
        "  threshold:",
        threshold,
    )

    print(
        "  expected:",
        expected_counts,
    )

    print(
        "  actual:  ",
        actual_counts,
    )

    print(
        "  exact:   ",
        exact,
    )


    if not exact:
        raise RuntimeError(
            f"{name} operating point mismatch."
        )


    op_results[
        name
    ] = {
        "threshold":
            threshold,

        "counts":
            actual_counts,

        "exact":
            True,
    }


print()

print(
    "[PASS] All frozen operating points reproduced."
)
print()


# ============================================================================
# 17. WRITE RECEIPT
# ============================================================================

receipt = {

    "stage":
        "Stage24-1A-R1",

    "type":
        "PRIMARY_FULL_SOURCE_70F_SANITY_DIRECT_RECONSTRUCTION",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "execution_parent":
        head,

    "recovery_reason": (
        "Frozen Stage22R Parquet cache was not mounted in the current "
        "Kaggle session. Reconstructed only IDS2018 day_id=7 directly "
        "from the exact raw source and Git-frozen K79 exclusion locators."
    ),

    "source": {

        "dataset":
            "CSE-CIC-IDS2018",

        "file":
            "02-28-2018.csv",

        "raw_runtime_path":
            str(
                RAW_DAY7
            ),

        "raw_sha256":
            raw_sha,

        "physical_rows":
            EXPECTED_PHYSICAL_ROWS,

        "embedded_headers":
            EXPECTED_EMBEDDED_HEADERS,

        "effective_rows_before_k79":
            EXPECTED_EFFECTIVE_ROWS,

        "k79_exclusions":
            EXPECTED_EXCLUSIONS,

        "retained_rows":
            EXPECTED_RETAINED_ROWS,

        "benign":
            benign,

        "attack":
            attack,

        "clean_position_start":
            EXPECTED_CLEAN_START,

        "clean_position_stop_exclusive":
            EXPECTED_CLEAN_STOP,
    },

    "k79": {

        "artifact":
            str(
                K79_EXCLUSIONS.relative_to(
                    REPO
                )
            ),

        "sha256":
            k79_sha,

        "resolved_day_column":
            day_col,

        "resolved_file_column":
            file_col,

        "resolved_original_row_index_column":
            index_col,

        "day7_exclusions":
            EXPECTED_EXCLUSIONS,

        "embedded_header_overlap":
            0,
    },

    "representation": {

        "feature_count":
            70,

        "feature_order":
            features,

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "positive_infinity_to_nan":
            positive_inf,

        "negative_infinity_to_nan":
            negative_inf,

        "output_nan_values":
            nan_values,
    },

    "membership_validation": {

        "clean_position_exact_vs_frozen_reference":
            True,

        "binary_labels_exact_vs_frozen_reference":
            True,

        "rows":
            EXPECTED_RETAINED_ROWS,
    },

    "models": {

        "lightgbm_sha256":
            lgb_sha,

        "xgboost_sha256":
            xgb_sha,

        "scientific_fits":
            0,

        "ensemble":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",
    },

    "probability_reproduction": {

        "frozen_probability_file_sha256":
            prob_sha,

        "frozen_probability_array_sha256":
            frozen_array_sha,

        "reproduced_probability_array_sha256":
            reproduced_array_sha,

        "bitwise_exact":
            True,

        "differing_positions":
            0,

        "max_absolute_difference":
            0.0,

        "inference_seconds":
            inference_seconds,
    },

    "source_metrics": {

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "operating_points":
            op_results,
    },

    "sanity_gate": {

        "exact_raw_source":
            True,

        "exact_k79_membership_artifact":
            True,

        "exact_clean_population":
            True,

        "exact_labels":
            True,

        "exact_feature_order":
            True,

        "exact_frozen_model_hashes":
            True,

        "bitwise_probability_reproduction":
            True,

        "operating_points_exact":
            True,

        "pass":
            True,
    },

    "scientific_access": {

        "scientific_model_fits":
            0,

        "source_rows_read":
            EXPECTED_PHYSICAL_ROWS,

        "source_inference_rows":
            EXPECTED_RETAINED_ROWS,

        "cicids2017_rows_read":
            0,

        "target_rows_read":
            0,

        "target_labels_read":
            0,

        "target_predictions":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,

        "target_opening_budget":
            8,
    },

    "next_authorized_step": (
        "Stage24-1B — PRIMARY bridge62 source refit with frozen "
        "Stage22R CHRONOLOGICAL_NATURAL membership, inherited "
        "hyperparameters, LightGBM GPU + XGBoost CUDA, no tuning, "
        "and source-validation-only threshold selection."
    ),
}


receipt_sha = write_json_sha(
    RECEIPT,
    receipt,
)


print("=" * 108)
print("STAGE24-1A RECEIPT")
print("=" * 108)

print(
    "Receipt:",
    RECEIPT,
)

print(
    "SHA256:",
    receipt_sha,
)
print()


# ============================================================================
# 18. COMMIT + PUSH
# ============================================================================

print("=" * 108)
print("COMMIT + PUSH")
print("=" * 108)


token = None
token_source = None


try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for key in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:
        try:
            value = secrets.get_secret(
                key
            )

            if value and value.strip():
                token = value.strip()
                token_source = key
                break

        except Exception:
            pass

except Exception:
    pass


if token is None:

    for key in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:
        value = os.environ.get(
            key
        )

        if value and value.strip():
            token = value.strip()
            token_source = (
                "ENV:"
                + key
            )
            break


if token is None:
    raise RuntimeError(
        "GitHub token unavailable."
    )


basic = base64.b64encode(
    (
        "x-access-token:"
        + token
    ).encode()
).decode()


print(
    "Credential source:",
    token_source,
)


git(
    "add",
    "--",
    str(
        OUT_REL
    ),
)


staged = [
    item
    for item
    in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if item
]


expected_staged = {
    str(
        RECEIPT.relative_to(
            REPO
        )
    ),

    str(
        RECEIPT_SHA.relative_to(
            REPO
        )
    ),
}


if set(staged) != expected_staged:
    raise RuntimeError(
        "\nUnexpected staged files:\n"
        + "\n".join(
            staged
        )
    )


git(
    "commit",
    "-m",
    "stage24: verify primary full-source 70f sanity",
)


commit = git(
    "rev-parse",
    "HEAD",
)


parent = git(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_HEAD:
    raise RuntimeError(
        "\nStage24-1A commit parent mismatch.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {parent}"
    )


print(
    git(
        "push",
        "origin",
        "HEAD:main",
        auth_header=basic,
    )
)


remote_after = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=basic,
).split()[0]


if remote_after != commit:
    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git(
    "status",
    "--porcelain",
)


if final_status:
    raise RuntimeError(
        "\nWorktree not clean:\n"
        + final_status
    )


print()

print(
    "[PASS] Stage24-1A receipt committed and pushed."
)

print(
    "Commit:",
    commit,
)

print(
    "Remote:",
    remote_after,
)

print(
    "Worktree: <clean>"
)
print()


# ============================================================================
# 19. FINAL
# ============================================================================

print("=" * 108)
print("STAGE24-1A FINAL SYNTHESIS")
print("=" * 108)

print(
    "Raw IDS2018 day-7 rows:       ",
    EXPECTED_PHYSICAL_ROWS,
)

print(
    "Embedded headers removed:     ",
    EXPECTED_EMBEDDED_HEADERS,
)

print(
    "Frozen K79 exclusions:        ",
    EXPECTED_EXCLUSIONS,
)

print(
    "Exact retained population:    ",
    EXPECTED_RETAINED_ROWS,
)

print(
    "Benign:                       ",
    benign,
)

print(
    "Attack:                       ",
    attack,
)

print()

print(
    "Representation:               FULL / 70F"
)

print(
    "Scientific fits:              0"
)

print(
    "Probability reproduction:     BIT-FOR-BIT EXACT"
)

print(
    "Differing probabilities:      0"
)

print()

print(
    "PR-AUC:                       ",
    format(
        pr_auc,
        ".15f",
    ),
)

print(
    "ROC-AUC:                      ",
    format(
        roc_auc,
        ".15f",
    ),
)

print()

print(
    "Target rows read:             0"
)

print(
    "Target labels read:           0"
)

print(
    "Target predictions:           0"
)

print(
    "Target openings:              0 / 8"
)

print()

print(
    "Commit:",
    commit,
)

print()

print("-" * 108)
print("STAGE24-1A: PASS")
print("Exact frozen Stage22 full-source behavior reproduced.")
print("No Stage24 target opening consumed.")
print()
print("NEXT:")
print(
    "Stage24-1B — bridge62 source refit "
    "(LightGBM GPU + XGBoost CUDA)."
)
print("-" * 108)
print("=" * 108)

In [ ]:
# ============================================================================
# STAGE24-1A-R2 — FAST CONTINUATION AFTER LIGHTGBM NAME-METADATA FALSE GATE
#
# REUSES IN-MEMORY STATE FROM SUCCESSFUL 1A-R1 RECONSTRUCTION.
#
# NO CSV reread
# NO K79 reread
# NO reconstruction
# NO model fit
# NO target access
#
# Scientific fits: 0
# Target openings:  0 / 8
# ============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import lightgbm as lgb
import xgboost as xgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# ============================================================================
# 0. CONSTANTS
# ============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "3a0b4a54477a5159805e69caa1eba749ab614c7b"
)

STAGE22_DIR = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
)

STAGE22_RESULT = (
    STAGE22_DIR
    / "stage22r_2c_chronological_natural_result.json"
)

LGB_MODEL = (
    STAGE22_DIR
    / "chronological_natural_lightgbm_model.txt"
)

XGB_MODEL = (
    STAGE22_DIR
    / "chronological_natural_xgboost_model.json"
)

FROZEN_PROB = (
    STAGE22_DIR
    / "chronological_natural_validation_ensemble_probabilities.npz"
)

STAGE24_LOCK = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)

STAGE24_ATTESTATION = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0d_remote_verification_receipt.json"
)

OUT_REL = (
    Path("results")
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
)

OUTDIR = REPO / OUT_REL

OUTDIR.mkdir(
    parents=True,
    exist_ok=True,
)

RECEIPT = (
    OUTDIR
    / "stage24_1a_full_source_70f_sanity.json"
)

RECEIPT_SHA = (
    OUTDIR
    / "stage24_1a_full_source_70f_sanity.sha256"
)

EXPECTED_ROWS = 593_780
EXPECTED_BENIGN = 531_524
EXPECTED_ATTACK = 62_256

EXPECTED_LGB_SHA = (
    "7be4c610814e45be2e315996969d2e2f404a3ada14866b21a731e4382fdc18b8"
)

EXPECTED_XGB_SHA = (
    "38499d0edcdc5c6d0b2c3afee913558973d35d5ebf1cb2ae888a16b8da98756c"
)

EXPECTED_PROB_SHA = (
    "3fe6c468a0653ac5ee488da8d6586fd86628e9586eb01f6f5d962e35fff65e3f"
)


print("=" * 108)
print("STAGE24-1A-R2 — FAST SOURCE-SANITY CONTINUATION")
print("=" * 108)
print("CSV reread:             NO")
print("K79 reconstruction:     NO")
print("Scientific fits:        0")
print("Target rows:            0")
print("Target predictions:     0")
print("Target openings:        0 / 8")
print()


# ============================================================================
# 1. HELPERS
# ============================================================================

def git(
    *args,
    auth_header=None,
    check=True,
):
    cmd = ["git"]

    if auth_header is not None:
        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += list(
        map(str, args)
    )

    p = subprocess.run(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "\nGit failure:\n"
            + " ".join(cmd)
            + "\n\n"
            + (p.stdout or "")
        )

    return (
        p.stdout or ""
    ).strip()


def load_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            b = f.read(
                chunk_size
            )

            if not b:
                break

            h.update(b)

    return h.hexdigest()


def array_sha256(a):
    a = np.ascontiguousarray(a)

    h = hashlib.sha256()

    h.update(
        str(a.dtype).encode()
    )

    h.update(
        repr(
            tuple(a.shape)
        ).encode()
    )

    h.update(
        a.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


def write_json_sha(
    path,
    obj,
):
    path = Path(path)

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    digest = sha256_file(
        path
    )

    path.with_suffix(
        ".sha256"
    ).write_text(
        f"{digest}  {path.name}\n",
        encoding="utf-8",
    )

    return digest


def confusion(
    y_true,
    probability,
    threshold,
):
    pred = (
        probability
        >= threshold
    )

    positive = (
        y_true == 1
    )

    return {
        "tp": int(
            np.count_nonzero(
                pred & positive
            )
        ),

        "tn": int(
            np.count_nonzero(
                ~pred & ~positive
            )
        ),

        "fp": int(
            np.count_nonzero(
                pred & ~positive
            )
        ),

        "fn": int(
            np.count_nonzero(
                ~pred & positive
            )
        ),
    }


def op_record(
    result,
    name,
):
    item = (
        result[
            "operating_points"
        ][name]
    )

    if (
        name == "security"
        and isinstance(item, dict)
        and "result" in item
    ):
        return item[
            "result"
        ]

    return item


# ============================================================================
# 2. REQUIRE SUCCESSFUL R1 IN-MEMORY STATE
# ============================================================================

print("=" * 108)
print("IN-MEMORY R1 STATE")
print("=" * 108)


required_vars = [
    "X",
    "y",
    "clean_position",
    "frozen_p",
    "retained_original_indices",
    "features",
    "raw_sha",
    "k79_sha",
    "positive_inf",
    "negative_inf",
    "nan_values",
]


missing_vars = [
    name
    for name in required_vars
    if name not in globals()
]


if missing_vars:
    raise RuntimeError(
        "\nR1 in-memory state is unavailable.\n"
        "Missing variables:\n"
        + repr(missing_vars)
        + "\n\nDo NOT run this continuation in a fresh kernel."
    )


if X.shape != (
    EXPECTED_ROWS,
    70,
):
    raise RuntimeError(
        f"Unexpected X shape: {X.shape}"
    )


if y.shape != (
    EXPECTED_ROWS,
):
    raise RuntimeError(
        "Unexpected y shape."
    )


if frozen_p.shape != (
    EXPECTED_ROWS,
):
    raise RuntimeError(
        "Unexpected frozen probability shape."
    )


if int(
    y.sum()
) != EXPECTED_ATTACK:
    raise RuntimeError(
        "Attack count changed."
    )


if (
    EXPECTED_ROWS
    - int(y.sum())
    != EXPECTED_BENIGN
):
    raise RuntimeError(
        "Benign count changed."
    )


print(
    "X:",
    X.shape,
    X.dtype,
)

print(
    "y:",
    y.shape,
)

print(
    "Benign:",
    EXPECTED_BENIGN,
)

print(
    "Attack:",
    EXPECTED_ATTACK,
)

print(
    "clean_position:",
    int(clean_position[0]),
    "->",
    int(clean_position[-1]),
)

print()

print(
    "[PASS] Reusing exact successfully reconstructed R1 source population."
)
print()


# ============================================================================
# 3. GOVERNANCE + MODEL HASHES
# ============================================================================

print("=" * 108)
print("GOVERNANCE / MODEL INTEGRITY")
print("=" * 108)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
).split()[0]

status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if remote != head:
    raise RuntimeError(
        "Local HEAD != remote main."
    )


if status:
    raise RuntimeError(
        "\nWorktree must still be clean:\n"
        + status
    )


attestation = load_json(
    STAGE24_ATTESTATION
)


if not attestation[
    "governance_gate"
][
    "gate_pass"
]:
    raise RuntimeError(
        "Stage24 governance not authorized."
    )


if (
    attestation[
        "target_opening_ledger"
    ][
        "consumed"
    ]
    != 0
):
    raise RuntimeError(
        "Opening ledger no longer pristine."
    )


lgb_sha = sha256_file(
    LGB_MODEL
)

xgb_sha = sha256_file(
    XGB_MODEL
)

prob_sha = sha256_file(
    FROZEN_PROB
)


if lgb_sha != EXPECTED_LGB_SHA:
    raise RuntimeError(
        "LightGBM SHA mismatch."
    )


if xgb_sha != EXPECTED_XGB_SHA:
    raise RuntimeError(
        "XGBoost SHA mismatch."
    )


if prob_sha != EXPECTED_PROB_SHA:
    raise RuntimeError(
        "Frozen probability SHA mismatch."
    )


print(
    "HEAD:",
    head,
)

print(
    "LightGBM SHA: PASS"
)

print(
    "XGBoost SHA:  PASS"
)

print(
    "Reference SHA: PASS"
)

print(
    "Openings: 0 / 8"
)

print()


# ============================================================================
# 4. LOAD EXACT MODELS
#
# IMPORTANT:
#
# We do NOT require LightGBM's model-file display feature names to equal
# Stage22's canonical names character-for-character.
#
# Scientific identity is established by:
#   1. exact model SHA256
#   2. exact model dimension = 70
#   3. exact frozen Stage22 input order
#   4. definitive bit-for-bit reproduction of frozen predictions
#
# No feature reordering occurs here.
# ============================================================================

print("=" * 108)
print("LOADING EXACT FROZEN MODELS")
print("=" * 108)


lgb_model = lgb.Booster(
    model_file=str(
        LGB_MODEL
    )
)


if int(
    lgb_model.num_feature()
) != 70:
    raise RuntimeError(
        "LightGBM feature dimension != 70."
    )


if int(
    lgb_model.num_trees()
) != 400:
    raise RuntimeError(
        "LightGBM tree count != 400."
    )


stored_lgb_names = list(
    lgb_model.feature_name()
)


lgb_names_textually_equal = (
    stored_lgb_names
    == list(features)
)


print(
    "LightGBM dimensions:",
    lgb_model.num_feature(),
)

print(
    "LightGBM trees:",
    lgb_model.num_trees(),
)

print(
    "Stored-name textual equality:",
    lgb_names_textually_equal,
)


if not lgb_names_textually_equal:
    differences = [
        (
            i,
            features[i],
            stored_lgb_names[i],
        )
        for i
        in range(70)
        if (
            i >= len(stored_lgb_names)
            or stored_lgb_names[i]
            != features[i]
        )
    ]

    print(
        "Stored-name differences:",
        len(differences),
    )

    print(
        "First differences:",
        differences[:8],
    )

    print(
        "[INFO] Name metadata differs; positional contract remains frozen."
    )


xgb_model = xgb.Booster()

xgb_model.load_model(
    str(
        XGB_MODEL
    )
)


if int(
    xgb_model.num_features()
) != 70:
    raise RuntimeError(
        "XGBoost feature dimension != 70."
    )


if int(
    xgb_model.num_boosted_rounds()
) != 400:
    raise RuntimeError(
        "XGBoost rounds != 400."
    )


stored_xgb_names = (
    list(
        xgb_model.feature_names
    )
    if xgb_model.feature_names is not None
    else None
)


print(
    "XGBoost dimensions:",
    xgb_model.num_features(),
)

print(
    "XGBoost rounds:",
    xgb_model.num_boosted_rounds(),
)

print(
    "XGBoost stored names:",
    (
        "NONE"
        if stored_xgb_names is None
        else "PRESENT"
    ),
)


# Stage22 frozen execution backend.
xgb_model.set_param(
    {
        "device":
            "cuda",
    }
)


print()

print(
    "[PASS] Exact frozen model bytes + positional dimensions validated."
)
print()


# ============================================================================
# 5. SOURCE INFERENCE
#
# IMPORTANT:
# DMatrix intentionally has NO feature-name metadata.
# Therefore no textual name transformation/reordering is possible.
# X columns remain in exact frozen Stage22 order.
# ============================================================================

print("=" * 108)
print("SOURCE-ONLY 70F INFERENCE")
print("=" * 108)


start = time.time()


p_lgb = np.asarray(
    lgb_model.predict(
        X
    ),
    dtype=np.float64,
)


dmatrix = xgb.DMatrix(
    X,
    missing=np.nan,
)


p_xgb = np.asarray(
    xgb_model.predict(
        dmatrix
    ),
    dtype=np.float64,
)


if p_lgb.shape != (
    EXPECTED_ROWS,
):
    raise RuntimeError(
        "LightGBM probability length mismatch."
    )


if p_xgb.shape != (
    EXPECTED_ROWS,
):
    raise RuntimeError(
        "XGBoost probability length mismatch."
    )


if not np.all(
    np.isfinite(p_lgb)
):
    raise RuntimeError(
        "Non-finite LightGBM probabilities."
    )


if not np.all(
    np.isfinite(p_xgb)
):
    raise RuntimeError(
        "Non-finite XGBoost probabilities."
    )


# Exact frozen ensemble arithmetic.
p = (
    0.5 * p_lgb
    + 0.5 * p_xgb
).astype(
    np.float32
)


inference_seconds = (
    time.time()
    - start
)


print(
    "Rows:",
    len(p),
)

print(
    "Inference seconds:",
    round(
        inference_seconds,
        3,
    ),
)

print(
    "Ensemble dtype:",
    p.dtype,
)
print()


# ============================================================================
# 6. DEFINITIVE BIT-FOR-BIT GATE
# ============================================================================

print("=" * 108)
print("DEFINITIVE BIT-FOR-BIT STAGE22 REPRODUCTION")
print("=" * 108)


exact = np.array_equal(
    p,
    frozen_p,
)


mismatch_indices = np.flatnonzero(
    p != frozen_p
)


mismatch_count = int(
    len(
        mismatch_indices
    )
)


max_abs = float(
    np.max(
        np.abs(
            p.astype(
                np.float64
            )
            -
            frozen_p.astype(
                np.float64
            )
        )
    )
)


p_sha = array_sha256(
    p
)

frozen_array_sha = array_sha256(
    frozen_p
)


print(
    "BIT-FOR-BIT EXACT:",
    exact,
)

print(
    "Differing positions:",
    mismatch_count,
)

print(
    "Maximum absolute difference:",
    max_abs,
)

print(
    "Reproduced array SHA:",
    p_sha,
)

print(
    "Frozen array SHA:    ",
    frozen_array_sha,
)


if not exact:

    first = int(
        mismatch_indices[0]
    )

    raise RuntimeError(
        "\nSTAGE24-1A SOURCE SANITY DID NOT REPRODUCE BIT-FOR-BIT.\n"
        f"First mismatch retained index: {first}\n"
        f"Original source row: "
        f"{int(retained_original_indices[first])}\n"
        f"Clean position: "
        f"{int(clean_position[first])}\n"
        f"Reproduced: {float(p[first])}\n"
        f"Frozen:     {float(frozen_p[first])}\n"
        f"Max abs difference: {max_abs}\n\n"
        "STOP HERE. Do not tune or alter preprocessing."
    )


print()

print(
    "[PASS] Exact frozen Stage22 probability vector reproduced."
)
print()


# ============================================================================
# 7. SOURCE METRICS
# ============================================================================

stage22 = load_json(
    STAGE22_RESULT
)


pr_auc = float(
    average_precision_score(
        y,
        p,
    )
)

roc_auc = float(
    roc_auc_score(
        y,
        p,
    )
)


print("=" * 108)
print("SOURCE RANKING METRICS")
print("=" * 108)

print(
    "PR-AUC:",
    format(
        pr_auc,
        ".15f",
    ),
)

print(
    "ROC-AUC:",
    format(
        roc_auc,
        ".15f",
    ),
)
print()


# ============================================================================
# 8. FROZEN OPERATING POINTS
# ============================================================================

print("=" * 108)
print("FROZEN OPERATING-POINT REPRODUCTION")
print("=" * 108)


op_results = {}


for name in [
    "standard",
    "balanced",
    "security",
]:

    frozen = op_record(
        stage22,
        name,
    )


    threshold = float(
        frozen.get(
            "threshold_float32_runtime",
            np.float32(
                frozen[
                    "threshold"
                ]
            ),
        )
    )


    actual = confusion(
        y,
        p,
        threshold,
    )


    expected = {
        key:
            int(
                frozen[
                    key
                ]
            )
        for key
        in [
            "tp",
            "tn",
            "fp",
            "fn",
        ]
    }


    exact_counts = (
        actual
        == expected
    )


    print(
        name.upper()
    )

    print(
        "  threshold:",
        threshold,
    )

    print(
        "  expected:",
        expected,
    )

    print(
        "  actual:  ",
        actual,
    )

    print(
        "  exact:",
        exact_counts,
    )


    if not exact_counts:
        raise RuntimeError(
            f"{name} confusion counts differ."
        )


    op_results[
        name
    ] = {
        "threshold":
            threshold,

        "counts":
            actual,

        "exact":
            True,
    }


print()

print(
    "[PASS] All frozen operating points reproduce exactly."
)
print()


# ============================================================================
# 9. RECEIPT
# ============================================================================

receipt = {

    "stage":
        "Stage24-1A",

    "execution_variant":
        "R2_FAST_CONTINUATION_AFTER_NAME_METADATA_FALSE_GATE",

    "type":
        "PRIMARY_FULL_SOURCE_70F_SANITY",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "execution_parent":
        head,

    "source": {
        "dataset":
            "CSE-CIC-IDS2018",

        "file":
            "02-28-2018.csv",

        "raw_sha256":
            raw_sha,

        "retained_rows":
            EXPECTED_ROWS,

        "benign":
            EXPECTED_BENIGN,

        "attack":
            EXPECTED_ATTACK,

        "feature_count":
            70,

        "feature_order":
            list(features),

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "positive_infinity_to_nan":
            int(
                positive_inf
            ),

        "negative_infinity_to_nan":
            int(
                negative_inf
            ),

        "output_nan_values":
            int(
                nan_values
            ),
    },

    "k79": {
        "sha256":
            k79_sha,

        "day7_exclusions":
            19_291,

        "membership_already_verified_by_r1":
            True,
    },

    "models": {
        "lightgbm": {
            "sha256":
                lgb_sha,

            "features":
                70,

            "trees":
                int(
                    lgb_model.num_trees()
                ),

            "stored_feature_names_textually_equal_canonical":
                bool(
                    lgb_names_textually_equal
                ),

            "scientific_identity_basis": [
                "EXACT_MODEL_SHA256",
                "EXACT_70_DIMENSION",
                "FROZEN_POSITIONAL_INPUT_ORDER",
                "BIT_FOR_BIT_FROZEN_PREDICTION_REPRODUCTION",
            ],
        },

        "xgboost": {
            "sha256":
                xgb_sha,

            "features":
                70,

            "rounds":
                int(
                    xgb_model.num_boosted_rounds()
                ),

            "device":
                "cuda",
        },

        "ensemble":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "scientific_fits":
            0,
    },

    "probability_reproduction": {
        "reference_file_sha256":
            prob_sha,

        "reference_array_sha256":
            frozen_array_sha,

        "reproduced_array_sha256":
            p_sha,

        "bitwise_exact":
            True,

        "differing_positions":
            0,

        "maximum_absolute_difference":
            0.0,

        "inference_seconds":
            inference_seconds,
    },

    "source_metrics": {
        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "operating_points":
            op_results,
    },

    "sanity_gate": {
        "source_population_exact":
            True,

        "source_labels_exact":
            True,

        "model_hashes_exact":
            True,

        "model_dimensions_exact":
            True,

        "positional_feature_order_frozen":
            True,

        "probability_vector_bitwise_exact":
            True,

        "operating_points_exact":
            True,

        "pass":
            True,
    },

    "scientific_access": {
        "scientific_model_fits":
            0,

        "source_inference_rows":
            EXPECTED_ROWS,

        "cicids2017_rows_read":
            0,

        "target_rows_read":
            0,

        "target_labels_read":
            0,

        "target_predictions":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,

        "target_opening_budget":
            8,
    },

    "next_authorized_step": (
        "Stage24-1B — primary bridge62 source refit using frozen "
        "Stage22 chronological-natural memberships and inherited "
        "LightGBM/XGBoost hyperparameters; LightGBM GPU + XGBoost CUDA; "
        "no tuning; source-validation-only threshold selection."
    ),
}


receipt_sha = write_json_sha(
    RECEIPT,
    receipt,
)


print("=" * 108)
print("STAGE24-1A RECEIPT")
print("=" * 108)

print(
    "Receipt:",
    RECEIPT,
)

print(
    "SHA256:",
    receipt_sha,
)
print()


# ============================================================================
# 10. COMMIT + PUSH
# ============================================================================

print("=" * 108)
print("COMMIT + PUSH")
print("=" * 108)


token = None
token_source = None


try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for key in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:
        try:
            value = secrets.get_secret(
                key
            )

            if value and value.strip():
                token = value.strip()
                token_source = key
                break

        except Exception:
            pass

except Exception:
    pass


if token is None:

    for key in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:
        value = os.environ.get(
            key
        )

        if value and value.strip():
            token = value.strip()
            token_source = (
                "ENV:"
                + key
            )
            break


if token is None:
    raise RuntimeError(
        "GitHub token unavailable."
    )


basic = base64.b64encode(
    (
        "x-access-token:"
        + token
    ).encode()
).decode()


git(
    "add",
    "--",
    str(
        OUT_REL
    ),
)


staged = [
    x
    for x in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if x
]


expected_staged = {
    str(
        RECEIPT.relative_to(
            REPO
        )
    ),
    str(
        RECEIPT_SHA.relative_to(
            REPO
        )
    ),
}


if set(staged) != expected_staged:
    raise RuntimeError(
        "\nUnexpected staged files:\n"
        + repr(staged)
    )


git(
    "commit",
    "-m",
    "stage24: verify primary full-source 70f sanity",
)


commit = git(
    "rev-parse",
    "HEAD",
)


if git(
    "rev-parse",
    "HEAD^",
) != EXPECTED_HEAD:
    raise RuntimeError(
        "Stage24-1A commit parent mismatch."
    )


print(
    git(
        "push",
        "origin",
        "HEAD:main",
        auth_header=basic,
    )
)


remote_after = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=basic,
).split()[0]


if remote_after != commit:
    raise RuntimeError(
        "Remote verification failed."
    )


if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Worktree not clean after commit."
    )


print()

print(
    "[PASS] Stage24-1A committed and pushed."
)

print(
    "Commit:",
    commit,
)

print(
    "Remote:",
    remote_after,
)
print()


# ============================================================================
# 11. FINAL
# ============================================================================

print("=" * 108)
print("STAGE24-1A FINAL SYNTHESIS")
print("=" * 108)

print(
    "Exact source rows:            593780"
)

print(
    "Representation:              FULL / 70F"
)

print(
    "Scientific fits:             0"
)

print(
    "Probability reproduction:    BIT-FOR-BIT EXACT"
)

print(
    "Differing probabilities:     0"
)

print(
    "PR-AUC:                     ",
    format(
        pr_auc,
        ".15f",
    ),
)

print(
    "ROC-AUC:                    ",
    format(
        roc_auc,
        ".15f",
    ),
)

print(
    "Target rows read:            0"
)

print(
    "Target labels read:          0"
)

print(
    "Target predictions:          0"
)

print(
    "Target openings:             0 / 8"
)

print()

print(
    "Commit:",
    commit,
)

print()

print("-" * 108)
print("STAGE24-1A: PASS")
print("Exact Stage22 full-source behavior reproduced.")
print("No target opening consumed.")
print()
print(
    "NEXT: Stage24-1B — bridge62 source refit "
    "(LightGBM GPU + XGBoost CUDA)."
)
print("-" * 108)
print("=" * 108)

In [ ]:
# ============================================================================
# STAGE24-1B — PRIMARY BRIDGE62 SOURCE REFIT
#
# FROZEN SCIENTIFIC ACTION:
#
#   IDS2018 -> CICIDS2017 primary direction
#
#   SOURCE TRAIN:
#       Stage22 CHRONOLOGICAL_NATURAL
#       day_id 0..6
#       13,818,623 K79-clean rows
#
#   SOURCE VALIDATION:
#       day_id 7 / 02-28-2018
#       593,780 K79-clean rows
#
#   REPRESENTATION:
#       bridge62
#       exact Stage22 feature order minus eight aggregate flag-count fields
#
#   FITS:
#       1 LightGBM — GPU
#       1 XGBoost  — CUDA
#
#   NO:
#       tuning
#       target data
#       target labels
#       target inference
#       target threshold selection
#
# FIT BUDGET:
#       before = 0 / 4
#       this cell = 2
#       after = 2 / 4
#
# TARGET OPENINGS:
#       before = 0 / 8
#       after  = 0 / 8
# ============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import shutil
import subprocess

from pathlib import Path
from datetime import datetime, timezone
from fractions import Fraction

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import lightgbm as lgb
import xgboost as xgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================================
# 0. CONSTANTS
# ============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "c5c96decdff956a856bb85eed53cd55dae3b2aeb"
)

BRANCH = "main"


IDS_ROOT = Path(
    "/kaggle/input/datasets/"
    "solarmainframe/ids-intrusion-csv"
)


K79_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1a3_k79_development_freeze"
    / "stage22r_k79_development_exclusions.parquet"
)

EXPECTED_K79_SHA = (
    "3e979841468f9d42a8de7049ed50063d5907c0753fe5b1179dac948163d93134"
)


STAGE22_RESULT = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)


STAGE24_PROTOCOL = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)


STAGE24_1A = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1a_full_source_70f_sanity.json"
)


OUT_REL = (
    Path("results")
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
)

OUT = (
    REPO
    / OUT_REL
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


LGB_OUT = (
    OUT
    / "bridge62_lightgbm_model.txt"
)

XGB_OUT = (
    OUT
    / "bridge62_xgboost_model.json"
)

PROB_OUT = (
    OUT
    / "bridge62_validation_ensemble_probabilities.npz"
)

GRID_OUT = (
    OUT
    / "bridge62_validation_threshold_grid.csv"
)

RESULT_OUT = (
    OUT
    / "stage24_1b_bridge62_source_refit_result.json"
)

CHECKSUM_OUT = (
    OUT
    / "checksums.sha256"
)


# Temporary disk-backed training arrays.
TMP_ROOT = Path(
    "/kaggle/working/stage24_bridge62_runtime"
)

TMP_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

X_MEMMAP_PATH = (
    TMP_ROOT
    / "bridge62_train_X_float64.dat"
)

Y_MEMMAP_PATH = (
    TMP_ROOT
    / "bridge62_train_y_uint8.dat"
)


EXPECTED_TRAIN_ROWS = 13_818_623
EXPECTED_TRAIN_ATTACK = 1_910_043
EXPECTED_TRAIN_BENIGN = 11_908_580

EXPECTED_VAL_ROWS = 593_780
EXPECTED_VAL_ATTACK = 62_256
EXPECTED_VAL_BENIGN = 531_524


FLAG_FEATURES = [
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "ACK Flag Cnt",
    "URG Flag Cnt",
    "CWE Flag Count",
    "ECE Flag Cnt",
]


DAY_SPEC = {

    0: {
        "file": "02-14-2018.csv",
        "physical": 1_048_575,
        "embedded_headers": 0,
        "retained": 822_947,
        "benign": 666_273,
        "attack": 156_674,
        "k79": 225_628,
    },

    1: {
        "file": "02-15-2018.csv",
        "physical": 1_048_575,
        "embedded_headers": 0,
        "retained": 1_046_154,
        "benign": 994_414,
        "attack": 51_740,
        "k79": 2_421,
    },

    2: {
        "file": "02-16-2018.csv",
        "physical": 1_048_575,
        "embedded_headers": 1,
        "retained": 900_988,
        "benign": 446_653,
        "attack": 454_335,
        "k79": 147_586,
    },

    3: {
        "file": "02-20-2018.csv",
        "physical": 7_948_748,
        "embedded_headers": 0,
        "retained": 7_926_258,
        "benign": 7_350_083,
        "attack": 576_175,
        "k79": 22_490,
    },

    4: {
        "file": "02-21-2018.csv",
        "physical": 1_048_575,
        "embedded_headers": 0,
        "retained": 1_031_018,
        "benign": 360_827,
        "attack": 670_191,
        "k79": 17_557,
    },

    5: {
        "file": "02-22-2018.csv",
        "physical": 1_048_575,
        "embedded_headers": 0,
        "retained": 1_045_297,
        "benign": 1_044_935,
        "attack": 362,
        "k79": 3_278,
    },

    6: {
        "file": "02-23-2018.csv",
        "physical": 1_048_575,
        "embedded_headers": 0,
        "retained": 1_045_961,
        "benign": 1_045_395,
        "attack": 566,
        "k79": 2_614,
    },
}


print("=" * 108)
print("STAGE24-1B — PRIMARY BRIDGE62 SOURCE REFIT")
print("=" * 108)

print("Scientific fits before:   0 / 4")
print("Scientific fits this cell: 2")
print("Scientific fits after:    2 / 4")

print()

print("Target rows read:          0")
print("Target labels read:        0")
print("Target predictions:        0")
print("Target openings:           0 / 8")
print()


# ============================================================================
# 1. HELPERS
# ============================================================================

def git(
    *args,
    auth_header=None,
    check=True,
):
    cmd = ["git"]

    if auth_header is not None:
        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += list(
        map(str, args)
    )

    p = subprocess.run(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if (
        check
        and p.returncode != 0
    ):
        raise RuntimeError(
            "\nGit command failed:\n"
            + " ".join(cmd)
            + "\n\n"
            + (p.stdout or "")
        )

    return (
        p.stdout or ""
    ).strip()


def load_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def write_json(
    path,
    obj,
):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )


def safe_div(
    numerator,
    denominator,
):
    if denominator == 0:
        return 0.0

    return (
        numerator
        / denominator
    )


def confusion_counts(
    y,
    p,
    threshold,
):
    pred = (
        p >= threshold
    )

    pos = (
        y == 1
    )

    tp = int(
        np.count_nonzero(
            pred & pos
        )
    )

    tn = int(
        np.count_nonzero(
            ~pred & ~pos
        )
    )

    fp = int(
        np.count_nonzero(
            pred & ~pos
        )
    )

    fn = int(
        np.count_nonzero(
            ~pred & pos
        )
    )

    return (
        tp,
        tn,
        fp,
        fn,
    )


def metric_record(
    threshold_integer,
    y,
    p,
):
    threshold = float(
        np.float32(
            threshold_integer
            / 100.0
        )
    )

    tp, tn, fp, fn = (
        confusion_counts(
            y,
            p,
            threshold,
        )
    )

    precision = safe_div(
        tp,
        tp + fp,
    )

    recall = safe_div(
        tp,
        tp + fn,
    )

    fpr = safe_div(
        fp,
        fp + tn,
    )

    f1 = safe_div(
        2 * tp,
        2 * tp + fp + fn,
    )

    f2 = safe_div(
        5 * tp,
        5 * tp + fp + 4 * fn,
    )

    accuracy = safe_div(
        tp + tn,
        tp + tn + fp + fn,
    )

    return {
        "threshold_integer_percent":
            int(
                threshold_integer
            ),

        "threshold":
            threshold_integer
            / 100.0,

        "threshold_float32_runtime":
            threshold,

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn,

        "precision":
            precision,

        "recall":
            recall,

        "fpr":
            fpr,

        "f1":
            f1,

        "f2":
            f2,

        "accuracy":
            accuracy,
    }


def f1_fraction(row):
    return Fraction(
        2 * row["tp"],
        (
            2 * row["tp"]
            + row["fp"]
            + row["fn"]
        ),
    )


def f2_fraction(row):
    return Fraction(
        5 * row["tp"],
        (
            5 * row["tp"]
            + row["fp"]
            + 4 * row["fn"]
        ),
    )


def fpr_fraction(row):
    denominator = (
        row["fp"]
        + row["tn"]
    )

    if denominator == 0:
        return Fraction(
            0,
            1,
        )

    return Fraction(
        row["fp"],
        denominator,
    )


def recall_fraction(row):
    denominator = (
        row["tp"]
        + row["fn"]
    )

    if denominator == 0:
        return Fraction(
            0,
            1,
        )

    return Fraction(
        row["tp"],
        denominator,
    )


# ============================================================================
# 2. GOVERNANCE
# ============================================================================

print("=" * 108)
print("GOVERNANCE")
print("=" * 108)


branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
).split()[0]

status = git(
    "status",
    "--porcelain",
)


print(
    "Branch:",
    branch,
)

print(
    "Local HEAD:",
    head,
)

print(
    "Remote main:",
    remote,
)


if branch != BRANCH:

    raise RuntimeError(
        "Wrong Git branch."
    )


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "\nUnexpected Stage24-1B parent.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if remote != head:

    raise RuntimeError(
        "Local HEAD != remote main."
    )


if status:

    raise RuntimeError(
        "\nWorktree must be clean:\n"
        + status
    )


protocol = load_json(
    STAGE24_PROTOCOL
)

stage24_1a = load_json(
    STAGE24_1A
)


if not (
    stage24_1a[
        "sanity_gate"
    ][
        "pass"
    ]
):

    raise RuntimeError(
        "Stage24-1A source sanity did not pass."
    )


if (
    stage24_1a[
        "scientific_access"
    ][
        "target_openings_consumed"
    ]
    != 0
):

    raise RuntimeError(
        "Opening ledger not pristine."
    )


if (
    protocol[
        "scientific_fit_budget"
    ][
        "primary_IDS2018_to_CICIDS2017"
    ][
        "bridge62_lightgbm"
    ]
    != 1
):

    raise RuntimeError(
        "Frozen LightGBM bridge62 fit budget != 1."
    )


if (
    protocol[
        "scientific_fit_budget"
    ][
        "primary_IDS2018_to_CICIDS2017"
    ][
        "bridge62_xgboost"
    ]
    != 1
):

    raise RuntimeError(
        "Frozen XGBoost bridge62 fit budget != 1."
    )


if (
    protocol[
        "boosting_backend"
    ][
        "lightgbm"
    ]
    != "gpu"
):

    raise RuntimeError(
        "Frozen LightGBM backend != GPU."
    )


if (
    protocol[
        "boosting_backend"
    ][
        "xgboost"
    ]
    != "cuda"
):

    raise RuntimeError(
        "Frozen XGBoost backend != CUDA."
    )


print()

print(
    "[PASS] Stage24-1A passed."
)

print(
    "[PASS] Authorized bridge62 fit budget = 2 models."
)

print(
    "[PASS] Target opening ledger = 0 / 8."
)
print()


# ============================================================================
# 3. REQUIRE VALIDATION MATRIX FROM 1A
#
# Saves another CSV reconstruction.
# ============================================================================

print("=" * 108)
print("REUSING EXACT STAGE24-1A VALIDATION MATRIX")
print("=" * 108)


required_vars = [
    "X",
    "y",
    "features",
]


missing = [
    name
    for name
    in required_vars
    if name not in globals()
]


if missing:

    raise RuntimeError(
        "\nStage24-1A validation matrix is no longer in memory.\n"
        f"Missing: {missing}\n"
        "Do not start the GPU fits in this kernel."
    )


if X.shape != (
    EXPECTED_VAL_ROWS,
    70,
):

    raise RuntimeError(
        f"Unexpected validation X shape: {X.shape}"
    )


if y.shape != (
    EXPECTED_VAL_ROWS,
):

    raise RuntimeError(
        "Unexpected validation y shape."
    )


if int(
    y.sum()
) != EXPECTED_VAL_ATTACK:

    raise RuntimeError(
        "Validation attack count mismatch."
    )


if (
    len(y)
    - int(y.sum())
    != EXPECTED_VAL_BENIGN
):

    raise RuntimeError(
        "Validation benign count mismatch."
    )


stage22 = load_json(
    STAGE22_RESULT
)


features70 = list(
    stage22[
        "data"
    ][
        "feature_order"
    ]
)


if list(features) != features70:

    raise RuntimeError(
        "In-memory validation feature order changed."
    )


bridge62 = list(
    protocol[
        "semantic_bridge"
    ]
    and load_json(
        REPO
        / protocol[
            "semantic_bridge"
        ][
            "artifact"
        ]
    )[
        "bridge62"
    ][
        "source_feature_order"
    ]
)


if len(
    bridge62
) != 62:

    raise RuntimeError(
        "Frozen bridge62 dimension != 62."
    )


if any(
    feature in bridge62
    for feature
    in FLAG_FEATURES
):

    raise RuntimeError(
        "bridge62 still contains aggregate flag fields."
    )


bridge62_indices = [
    features70.index(
        feature
    )
    for feature
    in bridge62
]


X_val = np.ascontiguousarray(
    X[
        :,
        bridge62_indices
    ],
    dtype=np.float64,
)


y_val = np.asarray(
    y,
    dtype=np.uint8,
)


print(
    "Validation X:",
    X_val.shape,
)

print(
    "Validation y:",
    y_val.shape,
)

print(
    "bridge62 features:",
    len(
        bridge62
    ),
)

print()

print(
    "[PASS] Exact source validation bridge62 matrix ready."
)
print()


# ============================================================================
# 4. VERIFY K79 ARTIFACT
# ============================================================================

print("=" * 108)
print("K79 MEMBERSHIP")
print("=" * 108)


if not K79_PATH.is_file():

    raise FileNotFoundError(
        K79_PATH
    )


k79_sha = sha256_file(
    K79_PATH
)


print(
    "Expected:",
    EXPECTED_K79_SHA,
)

print(
    "Actual:  ",
    k79_sha,
)


if k79_sha != EXPECTED_K79_SHA:

    raise RuntimeError(
        "K79 exclusion artifact SHA mismatch."
    )


k79 = pq.read_table(
    K79_PATH,
    columns=[
        "day_id",
        "row_index",
    ],
).to_pandas()


if len(
    k79
) != 440_865:

    raise RuntimeError(
        "Frozen K79 exclusion count != 440865."
    )


k79_by_day = {}


for day_id in range(
    7
):

    rows = (
        k79.loc[
            k79[
                "day_id"
            ]
            == day_id,
            "row_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    rows.sort()

    expected = (
        DAY_SPEC[
            day_id
        ][
            "k79"
        ]
    )

    if len(
        rows
    ) != expected:

        raise RuntimeError(
            f"Day {day_id}: K79 count mismatch."
        )

    if len(
        np.unique(
            rows
        )
    ) != len(
        rows
    ):

        raise RuntimeError(
            f"Day {day_id}: duplicate K79 row indices."
        )

    k79_by_day[
        day_id
    ] = rows


del k79
gc.collect()


print(
    "[PASS] Frozen K79 exclusions loaded for all seven training days."
)
print()


# ============================================================================
# 5. CREATE DISK-BACKED TRAINING ARRAYS
#
# float64 is inherited from Stage22.
# ============================================================================

print("=" * 108)
print("ALLOCATING BRIDGE62 TRAINING MEMMAP")
print("=" * 108)


for stale in [
    X_MEMMAP_PATH,
    Y_MEMMAP_PATH,
]:

    if stale.exists():

        stale.unlink()


X_train = np.memmap(
    X_MEMMAP_PATH,
    dtype=np.float64,
    mode="w+",
    shape=(
        EXPECTED_TRAIN_ROWS,
        62,
    ),
)


y_train = np.memmap(
    Y_MEMMAP_PATH,
    dtype=np.uint8,
    mode="w+",
    shape=(
        EXPECTED_TRAIN_ROWS,
    ),
)


print(
    "X memmap:",
    X_train.shape,
)

print(
    "Disk bytes:",
    X_train.nbytes,
)

print(
    "Approx GiB:",
    round(
        X_train.nbytes
        / 1024**3,
        3,
    ),
)
print()


# ============================================================================
# 6. STREAM EXACT SEVEN-DAY SOURCE TRAINING POPULATION
# ============================================================================

print("=" * 108)
print("STREAMING FROZEN IDS2018 TRAINING POPULATION")
print("=" * 108)


CHUNK_SIZE = 400_000

write_cursor = 0

training_day_receipts = []

total_attack = 0
total_benign = 0

build_start = time.time()


for day_id in range(
    7
):

    spec = DAY_SPEC[
        day_id
    ]

    source_path = (
        IDS_ROOT
        / spec[
            "file"
        ]
    )


    print()
    print("-" * 108)

    print(
        f"DAY {day_id}: {spec['file']}"
    )

    print("-" * 108)


    if not source_path.is_file():

        raise FileNotFoundError(
            source_path
        )


    # ----------------------------------------------------------------------
    # Exact physical header mapping
    # ----------------------------------------------------------------------

    header = pd.read_csv(
        source_path,
        nrows=0,
    )


    raw_columns = list(
        header.columns
    )


    stripped_to_raw = {}


    for raw_column in raw_columns:

        stripped = str(
            raw_column
        ).strip()

        if stripped in (
            stripped_to_raw
        ):

            raise RuntimeError(
                f"{spec['file']}: header collision after strip."
            )

        stripped_to_raw[
            stripped
        ] = raw_column


    required = (
        bridge62
        + [
            "Label"
        ]
    )


    missing_cols = [
        column
        for column
        in required
        if column
        not in stripped_to_raw
    ]


    if missing_cols:

        raise RuntimeError(
            f"{spec['file']}: missing bridge fields: "
            + repr(
                missing_cols
            )
        )


    raw_usecols = [
        stripped_to_raw[
            feature
        ]
        for feature
        in bridge62
    ] + [
        stripped_to_raw[
            "Label"
        ]
    ]


    exclusion_indices = (
        k79_by_day[
            day_id
        ]
    )


    physical_cursor = 0

    retained_day = 0
    attack_day = 0
    benign_day = 0
    header_day = 0
    excluded_day = 0


    day_start = time.time()


    for chunk in pd.read_csv(
        source_path,
        usecols=raw_usecols,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    ):

        n_chunk = len(
            chunk
        )


        physical_indices = np.arange(
            physical_cursor,
            physical_cursor
            + n_chunk,
            dtype=np.int64,
        )


        physical_cursor += (
            n_chunk
        )


        chunk.rename(
            columns={
                col:
                    str(
                        col
                    ).strip()
                for col
                in chunk.columns
            },
            inplace=True,
        )


        labels = (
            chunk[
                "Label"
            ]
            .astype(
                "string"
            )
            .str.strip()
        )


        embedded_header = (
            labels
            .eq(
                "Label"
            )
            .fillna(
                False
            )
            .to_numpy()
        )


        exclude_k79 = np.isin(
            physical_indices,
            exclusion_indices,
            assume_unique=False,
        )


        overlap = np.count_nonzero(
            embedded_header
            & exclude_k79
        )


        if overlap:

            raise RuntimeError(
                f"{spec['file']}: K79/header overlap = {overlap}"
            )


        keep = (
            ~embedded_header
            & ~exclude_k79
        )


        header_day += int(
            embedded_header.sum()
        )

        excluded_day += int(
            exclude_k79.sum()
        )


        keep_count = int(
            keep.sum()
        )


        if keep_count == 0:

            continue


        labels_kept = (
            labels[
                keep
            ]
            .str.casefold()
        )


        if (
            labels_kept.isna().any()
        ):

            raise RuntimeError(
                f"{spec['file']}: null retained label."
            )


        y_chunk = (
            ~labels_kept
            .eq(
                "benign"
            )
        ).to_numpy(
            dtype=np.uint8
        )


        feature_chunk = (
            chunk.loc[
                keep,
                bridge62,
            ]
            .copy()
        )


        for column in (
            bridge62
        ):

            if not pd.api.types.is_numeric_dtype(
                feature_chunk[
                    column
                ]
            ):

                feature_chunk[
                    column
                ] = pd.to_numeric(
                    feature_chunk[
                        column
                    ],
                    errors="raise",
                )


        X_chunk = (
            feature_chunk
            .to_numpy(
                dtype=np.float64,
                copy=True,
            )
        )


        X_chunk[
            np.isinf(
                X_chunk
            )
        ] = np.nan


        next_cursor = (
            write_cursor
            + keep_count
        )


        if (
            next_cursor
            > EXPECTED_TRAIN_ROWS
        ):

            raise RuntimeError(
                "Training memmap overflow."
            )


        X_train[
            write_cursor:
            next_cursor,
            :
        ] = X_chunk


        y_train[
            write_cursor:
            next_cursor
        ] = y_chunk


        write_cursor = (
            next_cursor
        )


        retained_day += (
            keep_count
        )

        attack_day += int(
            y_chunk.sum()
        )

        benign_day += (
            keep_count
            - int(
                y_chunk.sum()
            )
        )


        del (
            feature_chunk,
            X_chunk,
            y_chunk,
            labels,
            labels_kept,
            keep,
            embedded_header,
            exclude_k79,
            physical_indices,
            chunk,
        )


    day_seconds = (
        time.time()
        - day_start
    )


    print(
        "Physical rows:",
        physical_cursor,
    )

    print(
        "Embedded headers:",
        header_day,
    )

    print(
        "K79 exclusions:",
        excluded_day,
    )

    print(
        "Retained:",
        retained_day,
    )

    print(
        "Benign:",
        benign_day,
    )

    print(
        "Attack:",
        attack_day,
    )

    print(
        "Seconds:",
        round(
            day_seconds,
            3,
        ),
    )


    if (
        physical_cursor
        != spec[
            "physical"
        ]
    ):

        raise RuntimeError(
            f"{spec['file']}: physical row mismatch."
        )


    if (
        header_day
        != spec[
            "embedded_headers"
        ]
    ):

        raise RuntimeError(
            f"{spec['file']}: embedded header mismatch."
        )


    if (
        excluded_day
        != spec[
            "k79"
        ]
    ):

        raise RuntimeError(
            f"{spec['file']}: K79 applied count mismatch."
        )


    if (
        retained_day
        != spec[
            "retained"
        ]
    ):

        raise RuntimeError(
            f"{spec['file']}: retained row mismatch."
        )


    if (
        benign_day
        != spec[
            "benign"
        ]
    ):

        raise RuntimeError(
            f"{spec['file']}: benign count mismatch."
        )


    if (
        attack_day
        != spec[
            "attack"
        ]
    ):

        raise RuntimeError(
            f"{spec['file']}: attack count mismatch."
        )


    total_attack += (
        attack_day
    )

    total_benign += (
        benign_day
    )


    training_day_receipts.append(
        {
            "day_id":
                day_id,

            "file":
                spec[
                    "file"
                ],

            "physical_rows":
                physical_cursor,

            "embedded_headers":
                header_day,

            "k79_exclusions":
                excluded_day,

            "retained_rows":
                retained_day,

            "benign":
                benign_day,

            "attack":
                attack_day,

            "stream_seconds":
                day_seconds,
        }
    )


    gc.collect()


build_seconds = (
    time.time()
    - build_start
)


X_train.flush()
y_train.flush()


print()
print("=" * 108)
print("TRAINING POPULATION SYNTHESIS")
print("=" * 108)


print(
    "Written rows:",
    write_cursor,
)

print(
    "Benign:",
    total_benign,
)

print(
    "Attack:",
    total_attack,
)

print(
    "Build seconds:",
    round(
        build_seconds,
        3,
    ),
)


if (
    write_cursor
    != EXPECTED_TRAIN_ROWS
):

    raise RuntimeError(
        "Final training row count mismatch."
    )


if (
    total_attack
    != EXPECTED_TRAIN_ATTACK
):

    raise RuntimeError(
        "Final training attack count mismatch."
    )


if (
    total_benign
    != EXPECTED_TRAIN_BENIGN
):

    raise RuntimeError(
        "Final training benign count mismatch."
    )


print()

print(
    "[PASS] Exact frozen Stage22 chronological training membership reconstructed."
)
print()


# ============================================================================
# 7. FROZEN HYPERPARAMETERS
# ============================================================================

lgb_params = dict(
    stage22[
        "models"
    ][
        "lightgbm"
    ][
        "frozen_original_parameters"
    ]
)


xgb_params = dict(
    stage22[
        "models"
    ][
        "xgboost"
    ][
        "parameters"
    ]
)


if (
    lgb_params[
        "device_type"
    ]
    != "gpu"
):

    raise RuntimeError(
        "Frozen Stage24 LightGBM source refit is not GPU."
    )


if (
    xgb_params[
        "device"
    ]
    != "cuda"
):

    raise RuntimeError(
        "Frozen Stage24 XGBoost source refit is not CUDA."
    )


print("=" * 108)
print("FROZEN BRIDGE62 LEARNERS")
print("=" * 108)


print(
    "LightGBM parameters:"
)

print(
    json.dumps(
        lgb_params,
        indent=2,
        sort_keys=True,
    )
)


print()

print(
    "XGBoost parameters:"
)

print(
    json.dumps(
        xgb_params,
        indent=2,
        sort_keys=True,
    )
)

print()


# ============================================================================
# 8. SCIENTIFIC FIT 1 / 2 — LIGHTGBM GPU
# ============================================================================

print("=" * 108)
print("SCIENTIFIC FIT 1 / 2 — LIGHTGBM GPU")
print("=" * 108)


fit1_start = time.time()


lgb_model = lgb.LGBMClassifier(
    **lgb_params
)


lgb_model.fit(
    X_train,
    y_train,
)


lgb_fit_seconds = (
    time.time()
    - fit1_start
)


if (
    lgb_model.booster_
    .num_feature()
    != 62
):

    raise RuntimeError(
        "Bridge62 LightGBM model feature count != 62."
    )


if (
    lgb_model.booster_
    .num_trees()
    != 400
):

    raise RuntimeError(
        "Bridge62 LightGBM tree count != 400."
    )


lgb_model.booster_.save_model(
    str(
        LGB_OUT
    )
)


print(
    "Fit seconds:",
    round(
        lgb_fit_seconds,
        3,
    ),
)

print(
    "Features:",
    lgb_model.booster_.num_feature(),
)

print(
    "Trees:",
    lgb_model.booster_.num_trees(),
)

print(
    "Model:",
    LGB_OUT,
)
print()


# Validation inference immediately, then release model.
p_lgb = np.asarray(
    lgb_model.predict_proba(
        X_val
    )[
        :,
        1
    ],
    dtype=np.float64,
)


del lgb_model
gc.collect()


# ============================================================================
# 9. SCIENTIFIC FIT 2 / 2 — XGBOOST CUDA
# ============================================================================

print("=" * 108)
print("SCIENTIFIC FIT 2 / 2 — XGBOOST CUDA")
print("=" * 108)


fit2_start = time.time()


xgb_model = xgb.XGBClassifier(
    **xgb_params
)


xgb_model.fit(
    X_train,
    y_train,
)


xgb_fit_seconds = (
    time.time()
    - fit2_start
)


booster = (
    xgb_model
    .get_booster()
)


if (
    booster.num_features()
    != 62
):

    raise RuntimeError(
        "Bridge62 XGBoost model feature count != 62."
    )


if (
    booster.num_boosted_rounds()
    != 400
):

    raise RuntimeError(
        "Bridge62 XGBoost rounds != 400."
    )


xgb_model.save_model(
    str(
        XGB_OUT
    )
)


print(
    "Fit seconds:",
    round(
        xgb_fit_seconds,
        3,
    ),
)

print(
    "Features:",
    booster.num_features(),
)

print(
    "Rounds:",
    booster.num_boosted_rounds(),
)

print(
    "Model:",
    XGB_OUT,
)
print()


p_xgb = np.asarray(
    xgb_model.predict_proba(
        X_val
    )[
        :,
        1
    ],
    dtype=np.float64,
)


del xgb_model
del booster
gc.collect()


# ============================================================================
# 10. FROZEN ENSEMBLE
# ============================================================================

print("=" * 108)
print("BRIDGE62 SOURCE VALIDATION ENSEMBLE")
print("=" * 108)


if not np.all(
    np.isfinite(
        p_lgb
    )
):

    raise RuntimeError(
        "Non-finite LightGBM validation probabilities."
    )


if not np.all(
    np.isfinite(
        p_xgb
    )
):

    raise RuntimeError(
        "Non-finite XGBoost validation probabilities."
    )


p_ensemble = (
    0.5 * p_lgb
    + 0.5 * p_xgb
).astype(
    np.float32
)


if not np.all(
    np.isfinite(
        p_ensemble
    )
):

    raise RuntimeError(
        "Non-finite bridge62 ensemble probabilities."
    )


np.savez_compressed(
    PROB_OUT,
    binary_label=np.asarray(
        y_val,
        dtype=np.uint8,
    ),
    ensemble_probability=p_ensemble,
)


pr_auc = float(
    average_precision_score(
        y_val,
        p_ensemble,
    )
)


roc_auc = float(
    roc_auc_score(
        y_val,
        p_ensemble,
    )
)


brier = float(
    brier_score_loss(
        y_val,
        p_ensemble,
    )
)


ll = float(
    log_loss(
        y_val,
        np.clip(
            p_ensemble.astype(
                np.float64
            ),
            1e-15,
            1 - 1e-15,
        ),
        labels=[
            0,
            1,
        ],
    )
)


print(
    "PR-AUC:",
    format(
        pr_auc,
        ".15f",
    ),
)

print(
    "ROC-AUC:",
    format(
        roc_auc,
        ".15f",
    ),
)

print(
    "Brier:",
    format(
        brier,
        ".15f",
    ),
)

print(
    "Log loss:",
    format(
        ll,
        ".15f",
    ),
)
print()


# ============================================================================
# 11. FROZEN SOURCE-VALIDATION THRESHOLD GRID
# ============================================================================

print("=" * 108)
print("SOURCE-VALIDATION-ONLY THRESHOLD SELECTION")
print("=" * 108)


grid = [
    metric_record(
        threshold_integer,
        y_val,
        p_ensemble,
    )

    for threshold_integer
    in range(
        5,
        96,
    )
]


grid_df = pd.DataFrame(
    grid
)


grid_df.to_csv(
    GRID_OUT,
    index=False,
)


# --------------------------------------------------------------------------
# Balanced:
# MAX F1
# tie:
#   lower FPR
#   higher recall
#   closer to 0.50
#   lower threshold
# --------------------------------------------------------------------------

balanced = None


for row in grid:

    if balanced is None:

        balanced = row

        continue


    candidate_key = (
        f1_fraction(
            row
        ),

        -fpr_fraction(
            row
        ),

        recall_fraction(
            row
        ),

        -abs(
            row[
                "threshold_integer_percent"
            ]
            - 50
        ),

        -row[
            "threshold_integer_percent"
        ],
    )


    current_key = (
        f1_fraction(
            balanced
        ),

        -fpr_fraction(
            balanced
        ),

        recall_fraction(
            balanced
        ),

        -abs(
            balanced[
                "threshold_integer_percent"
            ]
            - 50
        ),

        -balanced[
            "threshold_integer_percent"
        ],
    )


    if candidate_key > current_key:

        balanced = row


# --------------------------------------------------------------------------
# Security:
# FPR <= 0.05
# MAX F2
# tie:
#   lower FPR
#   higher recall
#   lower threshold
# --------------------------------------------------------------------------

eligible_security = [
    row
    for row
    in grid
    if (
        fpr_fraction(
            row
        )
        <= Fraction(
            1,
            20,
        )
    )
]


if not eligible_security:

    security = None

else:

    security = eligible_security[
        0
    ]


    for row in (
        eligible_security[
            1:
        ]
    ):

        candidate_key = (
            f2_fraction(
                row
            ),

            -fpr_fraction(
                row
            ),

            recall_fraction(
                row
            ),

            -row[
                "threshold_integer_percent"
            ],
        )


        current_key = (
            f2_fraction(
                security
            ),

            -fpr_fraction(
                security
            ),

            recall_fraction(
                security
            ),

            -security[
                "threshold_integer_percent"
            ],
        )


        if candidate_key > current_key:

            security = row


standard = next(
    row
    for row
    in grid
    if (
        row[
            "threshold_integer_percent"
        ]
        == 50
    )
)


print(
    "STANDARD:"
)

print(
    standard
)

print()

print(
    "BALANCED:"
)

print(
    balanced
)

print()

print(
    "SECURITY:"
)

print(
    (
        security
        if security
        is not None
        else "UNAVAILABLE"
    )
)
print()


# ============================================================================
# 12. SOURCE-SANITY COMPARISON TO FULL 70F
#
# DESCRIPTIVE ONLY.
# NO acceptance / rejection based on performance.
# ============================================================================

full_pr_auc = float(
    stage24_1a[
        "source_metrics"
    ][
        "pr_auc"
    ]
)


full_roc_auc = float(
    stage24_1a[
        "source_metrics"
    ][
        "roc_auc"
    ]
)


source_comparison = {

    "full70_pr_auc":
        full_pr_auc,

    "bridge62_pr_auc":
        pr_auc,

    "bridge62_minus_full70_pr_auc":
        pr_auc
        - full_pr_auc,

    "full70_roc_auc":
        full_roc_auc,

    "bridge62_roc_auc":
        roc_auc,

    "bridge62_minus_full70_roc_auc":
        roc_auc
        - full_roc_auc,

    "performance_based_stop":
        False,

    "interpretation_rule":
        (
            "DESCRIPTIVE_ONLY; poor or improved bridge-source performance "
            "does not alter target execution."
        ),
}


print("=" * 108)
print("FULL70 -> BRIDGE62 SOURCE SANITY COMPARISON")
print("=" * 108)

print(
    "FULL70 PR-AUC:",
    full_pr_auc,
)

print(
    "bridge62 PR-AUC:",
    pr_auc,
)

print(
    "Delta PR-AUC:",
    source_comparison[
        "bridge62_minus_full70_pr_auc"
    ],
)

print()

print(
    "FULL70 ROC-AUC:",
    full_roc_auc,
)

print(
    "bridge62 ROC-AUC:",
    roc_auc,
)

print(
    "Delta ROC-AUC:",
    source_comparison[
        "bridge62_minus_full70_roc_auc"
    ],
)

print()

print(
    "[FROZEN] This comparison cannot change the target execution plan."
)
print()


# ============================================================================
# 13. HASH SCIENTIFIC ARTIFACTS
# ============================================================================

artifact_paths = [
    LGB_OUT,
    XGB_OUT,
    PROB_OUT,
    GRID_OUT,
]


artifact_hashes = {
    path.name:
        sha256_file(
            path
        )
    for path
    in artifact_paths
}


# ============================================================================
# 14. WRITE RESULT
# ============================================================================

result = {

    "stage":
        "Stage24-1B",

    "type":
        "PRIMARY_BRIDGE62_SOURCE_REFIT",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "execution_parent":
        head,

    "direction":
        "IDS2018_TO_CICIDS2017",

    "representation": {

        "name":
            "bridge62",

        "feature_count":
            62,

        "feature_order":
            bridge62,

        "excluded_features":
            FLAG_FEATURES,

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "dtype":
            "float64",
    },

    "source_train": {

        "membership":
            "Stage22 CHRONOLOGICAL_NATURAL day_id 0..6",

        "rows":
            EXPECTED_TRAIN_ROWS,

        "benign":
            EXPECTED_TRAIN_BENIGN,

        "attack":
            EXPECTED_TRAIN_ATTACK,

        "days":
            training_day_receipts,

        "k79_sha256":
            k79_sha,

        "build_seconds":
            build_seconds,
    },

    "source_validation": {

        "membership":
            "Stage22 CHRONOLOGICAL_NATURAL day_id 7",

        "rows":
            EXPECTED_VAL_ROWS,

        "benign":
            EXPECTED_VAL_BENIGN,

        "attack":
            EXPECTED_VAL_ATTACK,
    },

    "scientific_fits": {

        "lightgbm": {

            "count":
                1,

            "backend":
                "GPU",

            "configuration":
                "LGBM_11",

            "parameters":
                lgb_params,

            "fit_seconds":
                lgb_fit_seconds,

            "model_file":
                LGB_OUT.name,

            "model_sha256":
                artifact_hashes[
                    LGB_OUT.name
                ],
        },

        "xgboost": {

            "count":
                1,

            "backend":
                "CUDA",

            "configuration":
                "XGB_11",

            "parameters":
                xgb_params,

            "fit_seconds":
                xgb_fit_seconds,

            "model_file":
                XGB_OUT.name,

            "model_sha256":
                artifact_hashes[
                    XGB_OUT.name
                ],
        },

        "this_stage":
            2,

        "stage24_total_consumed_after":
            2,

        "stage24_total_budget":
            4,

        "additional_fit_budget":
            0,
    },

    "ensemble": {

        "rule":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "storage_dtype":
            "float32",

        "validation_probability_file":
            PROB_OUT.name,

        "validation_probability_sha256":
            artifact_hashes[
                PROB_OUT.name
            ],
    },

    "source_validation_metrics": {

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "brier":
            brier,

        "log_loss":
            ll,

        "standard":
            standard,

        "balanced":
            balanced,

        "security": {

            "status":
                (
                    "AVAILABLE"
                    if security
                    is not None
                    else "UNAVAILABLE_NO_RELAXATION"
                ),

            "result":
                security,
        },
    },

    "source_sanity_comparison":
        source_comparison,

    "threshold_selection": {

        "data":
            "SOURCE_VALIDATION_ONLY",

        "grid":
            "integer percent 5..95",

        "target_threshold_selection":
            False,

        "threshold_grid_file":
            GRID_OUT.name,

        "threshold_grid_sha256":
            artifact_hashes[
                GRID_OUT.name
            ],
    },

    "scientific_access": {

        "source_model_fits":
            2,

        "target_rows_read":
            0,

        "target_labels_read":
            0,

        "target_predictions":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,

        "target_opening_budget":
            8,
    },

    "anti_adaptation": {

        "hyperparameter_search":
            False,

        "target_feedback":
            False,

        "feature_search":
            False,

        "threshold_search_on_target":
            False,

        "performance_based_stop":
            False,
    },

    "status":
        "BRIDGE62_SOURCE_REFIT_COMPLETE",

    "next_authorized_step": (
        "Stage24-1C — recover/replay GROUNDED_S4 exact membership "
        "before any target model inference. Target opening ledger remains 0/8."
    ),
}


write_json(
    RESULT_OUT,
    result,
)


artifact_hashes[
    RESULT_OUT.name
] = sha256_file(
    RESULT_OUT
)


# checksums
with CHECKSUM_OUT.open(
    "w",
    encoding="utf-8",
) as f:

    for name in sorted(
        artifact_hashes
    ):

        f.write(
            f"{artifact_hashes[name]}  {name}\n"
        )


artifact_hashes[
    CHECKSUM_OUT.name
] = sha256_file(
    CHECKSUM_OUT
)


print("=" * 108)
print("SCIENTIFIC ARTIFACTS")
print("=" * 108)


for name in sorted(
    artifact_hashes
):

    print(
        name
    )

    print(
        " ",
        artifact_hashes[
            name
        ],
    )


print()


# ============================================================================
# 15. DELETE LARGE TEMP TRAINING MEMMAP
#
# Models/results are already persisted.
# ============================================================================

del X_train
del y_train

gc.collect()


for temporary in [
    X_MEMMAP_PATH,
    Y_MEMMAP_PATH,
]:

    if temporary.exists():

        temporary.unlink()


try:

    TMP_ROOT.rmdir()

except Exception:
    pass


print(
    "[PASS] Temporary ~6.4 GiB bridge62 training memmap removed."
)
print()


# ============================================================================
# 16. COMMIT + PUSH
# ============================================================================

print("=" * 108)
print("COMMIT + PUSH")
print("=" * 108)


token = None
token_source = None


try:

    from kaggle_secrets import (
        UserSecretsClient,
    )

    secrets = UserSecretsClient()


    for key in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                key
            )

            if (
                value
                and value.strip()
            ):

                token = value.strip()

                token_source = key

                break

        except Exception:
            pass

except Exception:
    pass


if token is None:

    for key in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            key
        )

        if (
            value
            and value.strip()
        ):

            token = value.strip()

            token_source = (
                "ENV:"
                + key
            )

            break


if token is None:

    raise RuntimeError(
        "GitHub token unavailable."
    )


basic = base64.b64encode(
    (
        "x-access-token:"
        + token
    ).encode()
).decode()


print(
    "Credential source:",
    token_source,
)


git(
    "add",
    "--",
    str(
        OUT_REL
    ),
)


staged = [
    line
    for line
    in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
]


if not staged:

    raise RuntimeError(
        "Nothing staged."
    )


unexpected = [
    path
    for path
    in staged
    if not path.startswith(
        str(
            OUT_REL
        )
        + "/"
    )
]


if unexpected:

    raise RuntimeError(
        "\nUnexpected staged paths:\n"
        + "\n".join(
            unexpected
        )
    )


print(
    "Staged files:",
    len(
        staged
    ),
)


for item in staged:

    print(
        "  +",
        item,
    )


git(
    "commit",
    "-m",
    "stage24: fit primary bridge62 source models",
)


commit = git(
    "rev-parse",
    "HEAD",
)


parent = git(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_HEAD:

    raise RuntimeError(
        "\nStage24-1B commit parent mismatch.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {parent}"
    )


print()

print(
    git(
        "push",
        "origin",
        "HEAD:main",
        auth_header=basic,
    )
)


remote_after = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=basic,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nWorktree not clean:\n"
        + final_status
    )


print()

print(
    "[PASS] Stage24-1B artifacts committed and pushed."
)

print(
    "Commit:",
    commit,
)

print(
    "Remote:",
    remote_after,
)

print(
    "Worktree: <clean>"
)
print()


# ============================================================================
# 17. FINAL SYNTHESIS
# ============================================================================

print("=" * 108)
print("STAGE24-1B FINAL SYNTHESIS")
print("=" * 108)

print(
    "Representation:"
)

print(
    "  bridge62 / 62 features"
)

print()

print(
    "Source train:"
)

print(
    " ",
    EXPECTED_TRAIN_ROWS,
    "rows",
)

print(
    " ",
    EXPECTED_TRAIN_BENIGN,
    "benign",
)

print(
    " ",
    EXPECTED_TRAIN_ATTACK,
    "attack",
)

print()

print(
    "Source validation:"
)

print(
    " ",
    EXPECTED_VAL_ROWS,
    "rows",
)

print()

print(
    "LightGBM:"
)

print(
    "  GPU"
)

print(
    "  fit seconds:",
    round(
        lgb_fit_seconds,
        3,
    ),
)

print()

print(
    "XGBoost:"
)

print(
    "  CUDA"
)

print(
    "  fit seconds:",
    round(
        xgb_fit_seconds,
        3,
    ),
)

print()

print(
    "PR-AUC:"
)

print(
    " ",
    format(
        pr_auc,
        ".15f",
    ),
)

print(
    "ROC-AUC:"
)

print(
    " ",
    format(
        roc_auc,
        ".15f",
    ),
)

print()

print(
    "Balanced threshold:"
)

print(
    " ",
    balanced[
        "threshold_float32_runtime"
    ],
)

print(
    "Security threshold:"
)

print(
    " ",
    (
        security[
            "threshold_float32_runtime"
        ]
        if security
        is not None
        else "UNAVAILABLE"
    ),
)

print()

print(
    "Scientific fits consumed:"
)

print(
    "  2 / 4"
)

print()

print(
    "Target rows read:"
)

print(
    "  0"
)

print(
    "Target labels read:"
)

print(
    "  0"
)

print(
    "Target predictions:"
)

print(
    "  0"
)

print(
    "Target openings:"
)

print(
    "  0 / 8"
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print("-" * 108)

print(
    "STAGE24-1B: PASS"
)

print()

print(
    "Primary bridge62 source model pair is frozen."
)

print(
    "Two of four total Stage24 scientific fits are now consumed."
)

print(
    "No target opening has been consumed."
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "Stage24-1C — exact GROUNDED_S4 membership recovery/replay."
)

print(
    "Target opening ledger remains 0 / 8."
)

print("-" * 108)
print("=" * 108)

In [ ]:
# ============================================================================
# STAGE24-1B-EX1 — LIGHTGBM FULL-SCALE GPU FAILURE FREEZE
#
# NO FITS IN THIS CELL
# NO TARGET ACCESS
#
# Records the failed GPU execution and prospectively freezes:
#
#   LightGBM:
#       device_type GPU -> CPU ONLY
#
#   ALL algorithmic hyperparameters unchanged.
#
#   XGBoost:
#       CUDA unchanged.
#
# This exactly inherits the already-frozen Stage22R-2A-EX1 corrective rule
# after recurrence of the exact same upstream failure.
#
# COMPLETED scientific fits:
#       before: 0 / 4
#       after:  0 / 4
#
# Runtime LightGBM attempts:
#       1 failed
#
# Target openings:
#       0 / 8
# ============================================================================

from __future__ import annotations

import os
import json
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "c5c96decdff956a856bb85eed53cd55dae3b2aeb"
)

BRANCH = "main"

FAILURE_TEXT = (
    "Check failed: (best_split_info.left_count) > (0) at "
    "/__w/1/s/lightgbm-python/src/treelearner/"
    "serial_tree_learner.cpp, line 852"
)


STAGE22_EX1 = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_2a_ex1_lightgbm_backend_amendment"
    / "stage22r_2a_ex1_lightgbm_backend_amendment.json"
)

STAGE24_PROTOCOL = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)

STAGE24_1A = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1a_full_source_70f_sanity.json"
)

OUT_REL = (
    Path("results")
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
)

OUT = REPO / OUT_REL

OUT.mkdir(
    parents=True,
    exist_ok=True,
)

AMENDMENT = (
    OUT
    / "stage24_1b_ex1_lightgbm_backend_amendment.json"
)

AMENDMENT_SHA = (
    OUT
    / "stage24_1b_ex1_lightgbm_backend_amendment.sha256"
)


print("=" * 108)
print("STAGE24-1B-EX1 — LIGHTGBM FULL-SCALE GPU FAILURE FREEZE")
print("=" * 108)

print("Scientific fits completed:   0 / 4")
print("LightGBM execution attempts:  1")
print("XGBoost training started:     NO")
print("Target rows read:             0")
print("Target predictions:           0")
print("Target openings:              0 / 8")
print()


# ============================================================================
# HELPERS
# ============================================================================

def git(
    *args,
    auth_header=None,
    check=True,
):
    cmd = ["git"]

    if auth_header is not None:
        cmd.extend(
            [
                "-c",
                "credential.helper=",
                "-c",
                f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
            ]
        )

    cmd.extend(
        map(str, args)
    )

    p = subprocess.run(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "\nGit failure:\n"
            + " ".join(cmd)
            + "\n\n"
            + (p.stdout or "")
        )

    return (
        p.stdout or ""
    ).strip()


def load_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def write_json_sha(
    path,
    obj,
):
    path = Path(path)

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    digest = sha256_file(
        path
    )

    path.with_suffix(
        ".sha256"
    ).write_text(
        f"{digest}  {path.name}\n",
        encoding="utf-8",
    )

    return digest


# ============================================================================
# 1. GOVERNANCE
# ============================================================================

print("=" * 108)
print("GOVERNANCE")
print("=" * 108)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
).split()[0]

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Branch:",
    branch,
)

print(
    "Local HEAD:",
    head,
)

print(
    "Remote main:",
    remote,
)


if branch != BRANCH:
    raise RuntimeError(
        "Wrong branch."
    )


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "\nUnexpected parent.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if remote != head:
    raise RuntimeError(
        "Local HEAD != remote main."
    )


# Empty runtime directories do not appear here.
if status:
    raise RuntimeError(
        "\nUnexpected repository changes after failed fit:\n"
        + status
    )


protocol = load_json(
    STAGE24_PROTOCOL
)

stage24_1a = load_json(
    STAGE24_1A
)


if not stage24_1a[
    "sanity_gate"
][
    "pass"
]:
    raise RuntimeError(
        "Stage24-1A was not successful."
    )


if (
    stage24_1a[
        "scientific_access"
    ][
        "target_openings_consumed"
    ]
    != 0
):
    raise RuntimeError(
        "Target opening ledger not pristine."
    )


print()

print(
    "[PASS] Failure occurred before any Stage24 target opening."
)

print(
    "[PASS] No completed Stage24-1B model exists."
)
print()


# ============================================================================
# 2. VERIFY EXACT STAGE22 PRECEDENT
# ============================================================================

print("=" * 108)
print("STAGE22R GPU-FAILURE PRECEDENT")
print("=" * 108)


stage22_ex1 = load_json(
    STAGE22_EX1
)


stage22_failure = (
    stage22_ex1[
        "trigger"
    ][
        "failure"
    ]
)


print(
    "Stage22R failure:"
)

print(
    " ",
    stage22_failure,
)

print()

print(
    "Stage24 failure:"
)

print(
    " ",
    FAILURE_TEXT,
)


if (
    stage22_failure
    != FAILURE_TEXT
):
    raise RuntimeError(
        "\nStage24 failure is not byte-identical "
        "to frozen Stage22R failure text."
    )


corrective = (
    stage22_ex1[
        "corrective_execution_rule"
    ]
)


if (
    corrective[
        "lightgbm"
    ][
        "only_change"
    ][
        "from"
    ]
    != "gpu"
):
    raise RuntimeError(
        "Stage22 fallback source backend mismatch."
    )


if (
    corrective[
        "lightgbm"
    ][
        "only_change"
    ][
        "to"
    ]
    != "cpu"
):
    raise RuntimeError(
        "Stage22 fallback target backend mismatch."
    )


if (
    corrective[
        "lightgbm"
    ][
        "algorithmic_hyperparameters"
    ]
    != "UNCHANGED"
):
    raise RuntimeError(
        "Stage22 precedent did not preserve hyperparameters."
    )


if (
    corrective[
        "xgboost"
    ][
        "backend"
    ]
    != "cuda"
):
    raise RuntimeError(
        "Stage22 XGBoost backend precedent != CUDA."
    )


if (
    stage22_ex1[
        "execution_counters_after_failed_attempt"
    ][
        "completed_lightgbm_fits"
    ]
    != 0
):
    raise RuntimeError(
        "Stage22 failed attempt was counted as completed fit."
    )


print()

print(
    "[PASS] Exact same failure as Stage22R."
)

print(
    "[PASS] Frozen Stage22R correction = GPU -> CPU only."
)

print(
    "[PASS] Algorithmic hyperparameters remain unchanged."
)

print(
    "[PASS] XGBoost remains CUDA."
)
print()


# ============================================================================
# 3. VERIFY LARGE TRAINING MEMMAP SURVIVED
# ============================================================================

print("=" * 108)
print("RUNTIME TRAINING CACHE")
print("=" * 108)


TMP_ROOT = Path(
    "/kaggle/working/stage24_bridge62_runtime"
)

X_PATH = (
    TMP_ROOT
    / "bridge62_train_X_float64.dat"
)

Y_PATH = (
    TMP_ROOT
    / "bridge62_train_y_uint8.dat"
)


expected_x_bytes = (
    13_818_623
    * 62
    * 8
)

expected_y_bytes = (
    13_818_623
)


if not X_PATH.is_file():
    raise RuntimeError(
        "Bridge62 X memmap disappeared. Do not continue."
    )


if not Y_PATH.is_file():
    raise RuntimeError(
        "Bridge62 y memmap disappeared. Do not continue."
    )


actual_x_bytes = X_PATH.stat().st_size
actual_y_bytes = Y_PATH.stat().st_size


print(
    "X bytes expected:",
    expected_x_bytes,
)

print(
    "X bytes actual:  ",
    actual_x_bytes,
)

print(
    "y bytes expected:",
    expected_y_bytes,
)

print(
    "y bytes actual:  ",
    actual_y_bytes,
)


if actual_x_bytes != expected_x_bytes:
    raise RuntimeError(
        "X memmap size changed."
    )


if actual_y_bytes != expected_y_bytes:
    raise RuntimeError(
        "y memmap size changed."
    )


print()

print(
    "[PASS] 6.38 GiB reconstructed training matrix can be reused."
)
print()


# ============================================================================
# 4. FREEZE STAGE24 RUNTIME AMENDMENT
# ============================================================================

amendment = {

    "stage":
        "Stage24-1B-EX1",

    "status":
        "LIGHTGBM_FULL_SCALE_GPU_BACKEND_FAILURE_FROZEN_CPU_FALLBACK",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        head,

    "trigger": {

        "stage":
            "Stage24-1B",

        "direction":
            "IDS2018_TO_CICIDS2017",

        "representation":
            "bridge62",

        "component":
            "LIGHTGBM_LGBM_11",

        "requested_backend":
            "gpu",

        "runtime_lightgbm_version":
            "4.6.0",

        "failure":
            FAILURE_TEXT,

        "failure_location":
            (
                "During LGBMClassifier.fit after exact frozen "
                "Stage22 chronological training membership was "
                "reconstructed and all population assertions passed."
            ),

        "completed_lightgbm_fit":
            False,

        "lightgbm_model_serialized":
            False,

        "lightgbm_validation_inference":
            False,

        "xgboost_training_started":
            False,

        "threshold_selection_performed":
            False,

        "target_rows_read":
            0,

        "target_labels_read":
            0,

        "target_predictions":
            0,

        "target_openings_consumed":
            0,
    },

    "precedent": {

        "artifact":
            str(
                STAGE22_EX1.relative_to(
                    REPO
                )
            ),

        "artifact_sha256":
            sha256_file(
                STAGE22_EX1
            ),

        "failure_text_exact_match":
            True,

        "stage22_status":
            stage22_ex1[
                "status"
            ],

        "scientific_interpretation":
            stage22_ex1[
                "corrective_execution_rule"
            ][
                "scientific_interpretation"
            ],
    },

    "corrective_execution_rule": {

        "scope":
            "STAGE24_PRIMARY_BRIDGE62_LIGHTGBM_SOURCE_REFIT",

        "data_membership":
            "UNCHANGED",

        "representation":
            "UNCHANGED_BRIDGE62",

        "numeric_input_policy":
            "UNCHANGED_FLOAT64_INPUT",

        "lightgbm": {

            "configuration":
                "LGBM_11",

            "algorithmic_hyperparameters":
                "UNCHANGED",

            "only_change": {

                "parameter":
                    "device_type",

                "from":
                    "gpu",

                "to":
                    "cpu",
            },
        },

        "xgboost": {

            "configuration":
                "XGB_11",

            "backend":
                "cuda",

            "changes":
                "NONE",
        },

        "ensemble": {

            "rule":
                "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

            "changes":
                "NONE",
        },

        "threshold_semantics":
            "UNCHANGED_SOURCE_VALIDATION_ONLY",
    },

    "scientific_fit_accounting": {

        "lightgbm_execution_attempts":
            1,

        "completed_lightgbm_fits":
            0,

        "completed_xgboost_fits":
            0,

        "completed_stage24_scientific_fits":
            0,

        "stage24_completed_fit_budget":
            4,

        "failed_runtime_attempt_is_additional_model_selection":
            False,

        "reason": (
            "No trained LightGBM model, validation probabilities, "
            "thresholds, target values, or target metrics were observed."
        ),
    },

    "timing": {

        "frozen_before_successful_stage24_1b_model":
            True,

        "frozen_before_bridge62_validation_probability":
            True,

        "frozen_before_bridge62_threshold_selection":
            True,

        "frozen_before_any_stage24_target_opening":
            True,
    },

    "anti_adaptation": {

        "hyperparameter_tuning":
            False,

        "feature_change":
            False,

        "target_feedback":
            False,

        "performance_feedback":
            False,

        "threshold_feedback":
            False,
    },

    "next_authorized_step": (
        "Resume Stage24-1B from the already reconstructed bridge62 "
        "training memmap. Fit LGBM_11 once with device_type=cpu, "
        "then fit XGB_11 once with device=cuda. No other parameter "
        "or scientific rule may change."
    ),
}


amendment_sha = write_json_sha(
    AMENDMENT,
    amendment,
)


print("=" * 108)
print("AMENDMENT FROZEN")
print("=" * 108)

print(
    "Artifact:",
    AMENDMENT,
)

print(
    "SHA256:",
    amendment_sha,
)

print()

print(
    "LightGBM: GPU -> CPU ONLY"
)

print(
    "XGBoost: CUDA unchanged"
)

print(
    "Algorithmic hyperparameters: UNCHANGED"
)

print(
    "Target openings: 0 / 8"
)
print()


# ============================================================================
# 5. COMMIT + PUSH BEFORE RETRY
# ============================================================================

token = None
token_source = None


try:

    from kaggle_secrets import (
        UserSecretsClient,
    )

    secrets = UserSecretsClient()

    for key in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                key
            )

            if value and value.strip():

                token = value.strip()
                token_source = key
                break

        except Exception:
            pass

except Exception:
    pass


if token is None:

    for key in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            key
        )

        if value and value.strip():

            token = value.strip()
            token_source = (
                "ENV:"
                + key
            )

            break


if token is None:
    raise RuntimeError(
        "GitHub token unavailable."
    )


basic = base64.b64encode(
    (
        "x-access-token:"
        + token
    ).encode()
).decode()


git(
    "add",
    "--",
    str(
        AMENDMENT.relative_to(
            REPO
        )
    ),
    str(
        AMENDMENT_SHA.relative_to(
            REPO
        )
    ),
)


staged = [
    x
    for x
    in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if x
]


expected = {
    str(
        AMENDMENT.relative_to(
            REPO
        )
    ),
    str(
        AMENDMENT_SHA.relative_to(
            REPO
        )
    ),
}


if set(staged) != expected:
    raise RuntimeError(
        "\nUnexpected staged files:\n"
        + repr(staged)
    )


git(
    "commit",
    "-m",
    "stage24: freeze LightGBM full-scale GPU fallback",
)


commit = git(
    "rev-parse",
    "HEAD",
)


if (
    git(
        "rev-parse",
        "HEAD^",
    )
    != EXPECTED_HEAD
):
    raise RuntimeError(
        "Amendment parent mismatch."
    )


print(
    git(
        "push",
        "origin",
        "HEAD:main",
        auth_header=basic,
    )
)


remote_after = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=basic,
).split()[0]


if remote_after != commit:
    raise RuntimeError(
        "Remote amendment verification failed."
    )


if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Worktree not clean after amendment commit."
    )


print()

print("=" * 108)
print("STAGE24-1B-EX1 FINAL")
print("=" * 108)

print(
    "Exact Stage22R GPU failure recurrence: PASS"
)

print(
    "CPU fallback frozen before retry:     PASS"
)

print(
    "Amendment committed before retry:     PASS"
)

print(
    "Remote verification:                  PASS"
)

print()

print(
    "Completed scientific fits:            0 / 4"
)

print(
    "Failed LightGBM execution attempts:    1"
)

print(
    "Target openings:                      0 / 8"
)

print()

print(
    "Amendment commit:"
)

print(
    " ",
    commit,
)

print()

print("-" * 108)
print("STAGE24-1B-EX1: PASS")
print()
print(
    "NEXT: resume existing 6.38 GiB memmap; "
    "LightGBM CPU + XGBoost CUDA."
)
print("-" * 108)

In [ ]:
# ============================================================================
# STAGE24-1B-R1 — RESUME BRIDGE62 / LIGHTGBM CPU ONLY
#
# Uses EXISTING 6.38 GiB training memmap.
# Does NOT rebuild IDS2018.
#
# Authorized by:
#   Stage24-1B-EX1
#   LightGBM device_type: gpu -> cpu ONLY
#   all algorithmic hyperparameters unchanged
#
# THIS CELL:
#   - attaches existing train memmap
#   - reuses in-memory validation X if available
#   - completes exactly ONE LightGBM CPU fit
#   - predicts source validation
#   - saves model + validation probabilities
#   - freezes a checkpoint receipt
#   - commits + pushes immediately
#
# DOES NOT:
#   - start XGBoost
#   - select thresholds
#   - read target
#
# Completed Stage24 fits:
#   before: 0 / 4
#   after:  1 / 4
#
# Target openings:
#   0 / 8
# ============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import subprocess

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import lightgbm as lgb


# ============================================================================
# 0. CONSTANTS
# ============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "e9544a1db582c61ca446636ef84d576184c72616"
)

BRANCH = "main"


STAGE22_RESULT = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)


PROTOCOL = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)


SEMANTIC_BRIDGE = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_semantic_bridge_spec.json"
)


AMENDMENT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
    / "stage24_1b_ex1_lightgbm_backend_amendment.json"
)


EXPECTED_AMENDMENT_SHA = (
    "10d31ed6c5254a66cdefa9a2769f5deae294850d4da658d4f09bd1a42d739df5"
)


OUT_REL = (
    Path("results")
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
)

OUT = REPO / OUT_REL


LGB_MODEL_OUT = (
    OUT
    / "bridge62_lightgbm_model.txt"
)

LGB_PROB_OUT = (
    OUT
    / "bridge62_validation_lightgbm_probabilities.npy"
)

CHECKPOINT_OUT = (
    OUT
    / "stage24_1b_r1_lightgbm_cpu_checkpoint.json"
)

CHECKPOINT_SHA_OUT = (
    OUT
    / "stage24_1b_r1_lightgbm_cpu_checkpoint.sha256"
)


TMP_ROOT = Path(
    "/kaggle/working/stage24_bridge62_runtime"
)

X_PATH = (
    TMP_ROOT
    / "bridge62_train_X_float64.dat"
)

Y_PATH = (
    TMP_ROOT
    / "bridge62_train_y_uint8.dat"
)


TRAIN_ROWS = 13_818_623
TRAIN_FEATURES = 62
TRAIN_ATTACK = 1_910_043
TRAIN_BENIGN = 11_908_580

VAL_ROWS = 593_780
VAL_ATTACK = 62_256
VAL_BENIGN = 531_524


EXPECTED_X_BYTES = (
    TRAIN_ROWS
    * TRAIN_FEATURES
    * 8
)

EXPECTED_Y_BYTES = (
    TRAIN_ROWS
)


print("=" * 108)
print("STAGE24-1B-R1 — BRIDGE62 LIGHTGBM CPU RESUME")
print("=" * 108)

print("Training reconstruction:     REUSE EXISTING MEMMAP")
print("LightGBM backend:            CPU — FROZEN EX1 FALLBACK")
print("XGBoost training:            NOT IN THIS CELL")
print("Completed fits before:       0 / 4")
print("Completed fits after PASS:   1 / 4")
print("Target openings:             0 / 8")
print()


# ============================================================================
# 1. HELPERS
# ============================================================================

def git(
    *args,
    auth_header=None,
    check=True,
):
    cmd = ["git"]

    if auth_header is not None:
        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += list(
        map(str, args)
    )

    p = subprocess.run(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "\nGit command failed:\n"
            + " ".join(cmd)
            + "\n\n"
            + (p.stdout or "")
        )

    return (
        p.stdout or ""
    ).strip()


def load_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def write_json_sha(
    path,
    obj,
):
    path = Path(path)

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    digest = sha256_file(
        path
    )

    path.with_suffix(
        ".sha256"
    ).write_text(
        f"{digest}  {path.name}\n",
        encoding="utf-8",
    )

    return digest


# ============================================================================
# 2. GOVERNANCE
# ============================================================================

print("=" * 108)
print("GOVERNANCE")
print("=" * 108)


branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
).split()[0]

status = git(
    "status",
    "--porcelain",
)


print("Branch:", branch)
print("Local HEAD:", head)
print("Remote main:", remote)


if branch != BRANCH:
    raise RuntimeError(
        "Wrong branch."
    )


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "\nUnexpected parent.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if remote != head:
    raise RuntimeError(
        "Local HEAD != remote main."
    )


if status:
    raise RuntimeError(
        "\nWorktree must be clean:\n"
        + status
    )


actual_amendment_sha = sha256_file(
    AMENDMENT
)


print()
print(
    "Amendment SHA expected:",
    EXPECTED_AMENDMENT_SHA,
)

print(
    "Amendment SHA actual:  ",
    actual_amendment_sha,
)


if actual_amendment_sha != EXPECTED_AMENDMENT_SHA:
    raise RuntimeError(
        "Stage24-1B-EX1 amendment SHA mismatch."
    )


amendment = load_json(
    AMENDMENT
)


if (
    amendment[
        "corrective_execution_rule"
    ][
        "lightgbm"
    ][
        "only_change"
    ]
    != {
        "parameter": "device_type",
        "from": "gpu",
        "to": "cpu",
    }
):
    raise RuntimeError(
        "Frozen LightGBM fallback differs."
    )


if (
    amendment[
        "corrective_execution_rule"
    ][
        "lightgbm"
    ][
        "algorithmic_hyperparameters"
    ]
    != "UNCHANGED"
):
    raise RuntimeError(
        "Algorithmic hyperparameters are not frozen unchanged."
    )


if (
    amendment[
        "corrective_execution_rule"
    ][
        "xgboost"
    ][
        "backend"
    ]
    != "cuda"
):
    raise RuntimeError(
        "XGBoost frozen backend changed."
    )


if (
    amendment[
        "trigger"
    ][
        "target_openings_consumed"
    ]
    != 0
):
    raise RuntimeError(
        "Target opening ledger not pristine."
    )


print()
print("[PASS] CPU fallback is committed and frozen.")
print("[PASS] Target openings = 0 / 8.")
print()


# ============================================================================
# 3. ATTACH EXISTING TRAINING MEMMAP
# ============================================================================

print("=" * 108)
print("ATTACHING EXISTING TRAINING MEMMAP")
print("=" * 108)


if not X_PATH.is_file():
    raise RuntimeError(
        "\nTraining X memmap no longer exists.\n"
        "DO NOT continue or refit from a partial matrix."
    )


if not Y_PATH.is_file():
    raise RuntimeError(
        "\nTraining y memmap no longer exists."
    )


if X_PATH.stat().st_size != EXPECTED_X_BYTES:
    raise RuntimeError(
        "Training X memmap byte size mismatch."
    )


if Y_PATH.stat().st_size != EXPECTED_Y_BYTES:
    raise RuntimeError(
        "Training y memmap byte size mismatch."
    )


X_train = np.memmap(
    X_PATH,
    dtype=np.float64,
    mode="r",
    shape=(
        TRAIN_ROWS,
        TRAIN_FEATURES,
    ),
)


y_train = np.memmap(
    Y_PATH,
    dtype=np.uint8,
    mode="r",
    shape=(
        TRAIN_ROWS,
    ),
)


# y is only ~14 MB, so this is inexpensive.
actual_attack = int(
    np.sum(
        y_train,
        dtype=np.int64,
    )
)

actual_benign = (
    TRAIN_ROWS
    - actual_attack
)


print(
    "X train:",
    X_train.shape,
)

print(
    "X bytes:",
    X_PATH.stat().st_size,
)

print(
    "y train:",
    y_train.shape,
)

print(
    "Benign:",
    actual_benign,
)

print(
    "Attack:",
    actual_attack,
)


if actual_attack != TRAIN_ATTACK:
    raise RuntimeError(
        "Training attack count changed."
    )


if actual_benign != TRAIN_BENIGN:
    raise RuntimeError(
        "Training benign count changed."
    )


print()
print(
    "[PASS] Existing exact training memmap attached."
)
print()


# ============================================================================
# 4. RECOVER / VERIFY BRIDGE62 VALIDATION MATRIX
#
# Prefer X_val already created by failed 1B.
# Otherwise derive it from in-memory Stage24-1A X.
# No CSV reread.
# ============================================================================

print("=" * 108)
print("SOURCE VALIDATION MATRIX")
print("=" * 108)


stage22 = load_json(
    STAGE22_RESULT
)

bridge_spec = load_json(
    SEMANTIC_BRIDGE
)


features70 = list(
    stage22[
        "data"
    ][
        "feature_order"
    ]
)


bridge62 = list(
    bridge_spec[
        "bridge62"
    ][
        "source_feature_order"
    ]
)


if len(bridge62) != 62:
    raise RuntimeError(
        "bridge62 feature count != 62."
    )


if (
    "X_val" in globals()
    and isinstance(
        X_val,
        np.ndarray,
    )
    and X_val.shape
        == (
            VAL_ROWS,
            62,
        )
):

    X_bridge_val = np.ascontiguousarray(
        X_val,
        dtype=np.float64,
    )

    print(
        "Validation source: existing X_val"
    )

else:

    if "X" not in globals():
        raise RuntimeError(
            "\nNeither X_val nor Stage24-1A X remains in memory.\n"
            "Do not run this CPU fit."
        )


    if X.shape != (
        VAL_ROWS,
        70,
    ):
        raise RuntimeError(
            f"Unexpected Stage24-1A X shape: {X.shape}"
        )


    indices = [
        features70.index(
            feature
        )
        for feature
        in bridge62
    ]


    X_bridge_val = np.ascontiguousarray(
        X[
            :,
            indices,
        ],
        dtype=np.float64,
    )

    print(
        "Validation source: Stage24-1A X -> bridge62"
    )


if "y_val" in globals():

    y_bridge_val = np.asarray(
        y_val,
        dtype=np.uint8,
    )

elif "y" in globals():

    y_bridge_val = np.asarray(
        y,
        dtype=np.uint8,
    )

else:

    raise RuntimeError(
        "Validation labels no longer available in memory."
    )


if X_bridge_val.shape != (
    VAL_ROWS,
    62,
):
    raise RuntimeError(
        "Validation bridge62 shape mismatch."
    )


if y_bridge_val.shape != (
    VAL_ROWS,
):
    raise RuntimeError(
        "Validation label shape mismatch."
    )


val_attack = int(
    y_bridge_val.sum()
)

val_benign = (
    VAL_ROWS
    - val_attack
)


if val_attack != VAL_ATTACK:
    raise RuntimeError(
        "Validation attack count mismatch."
    )


if val_benign != VAL_BENIGN:
    raise RuntimeError(
        "Validation benign count mismatch."
    )


print(
    "X validation:",
    X_bridge_val.shape,
)

print(
    "Benign:",
    val_benign,
)

print(
    "Attack:",
    val_attack,
)

print()
print(
    "[PASS] Exact bridge62 source validation matrix ready."
)
print()


# ============================================================================
# 5. BUILD EXACT CPU-FALLBACK LIGHTGBM PARAMETERS
# ============================================================================

print("=" * 108)
print("FROZEN LIGHTGBM PARAMETERS")
print("=" * 108)


lgb_params = dict(
    stage22[
        "models"
    ][
        "lightgbm"
    ][
        "frozen_original_parameters"
    ]
)


# The ONLY amendment.
if lgb_params[
    "device_type"
] != "gpu":
    raise RuntimeError(
        "Inherited LightGBM source parameters unexpectedly changed."
    )


lgb_params[
    "device_type"
] = "cpu"


expected_algorithmic = {
    "boosting_type": "gbdt",
    "colsample_bytree": 1.0,
    "device_type": "cpu",
    "learning_rate": 0.06,
    "max_depth": 12,
    "min_child_samples": 20,
    "n_estimators": 400,
    "n_jobs": -1,
    "num_leaves": 127,
    "objective": "binary",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "subsample_freq": 1,
    "verbosity": -1,
}


if lgb_params != expected_algorithmic:
    raise RuntimeError(
        "\nUnexpected LightGBM parameter mutation.\n"
        f"Actual:\n{json.dumps(lgb_params, indent=2, sort_keys=True)}"
    )


print(
    json.dumps(
        lgb_params,
        indent=2,
        sort_keys=True,
    )
)

print()
print(
    "[PASS] Only backend differs from original frozen parameters."
)
print()


# ============================================================================
# 6. SCIENTIFIC FIT — LIGHTGBM CPU
#
# This is the first COMPLETED Stage24 scientific fit if successful.
# ============================================================================

print("=" * 108)
print("SCIENTIFIC FIT 1 / 4 — BRIDGE62 LIGHTGBM CPU")
print("=" * 108)


fit_start = time.time()


model = lgb.LGBMClassifier(
    **lgb_params
)


model.fit(
    X_train,
    y_train,
)


fit_seconds = (
    time.time()
    - fit_start
)


booster = model.booster_


if int(
    booster.num_feature()
) != 62:
    raise RuntimeError(
        "Completed LightGBM feature count != 62."
    )


if int(
    booster.num_trees()
) != 400:
    raise RuntimeError(
        "Completed LightGBM tree count != 400."
    )


print()
print(
    "Fit seconds:",
    round(
        fit_seconds,
        3,
    ),
)

print(
    "Features:",
    booster.num_feature(),
)

print(
    "Trees:",
    booster.num_trees(),
)

print()
print(
    "[PASS] LightGBM CPU scientific fit completed."
)
print()


# ============================================================================
# 7. SAVE MODEL IMMEDIATELY
# ============================================================================

booster.save_model(
    str(
        LGB_MODEL_OUT
    )
)


lgb_model_sha = sha256_file(
    LGB_MODEL_OUT
)


print("=" * 108)
print("LIGHTGBM MODEL FROZEN LOCALLY")
print("=" * 108)

print(
    "Model:",
    LGB_MODEL_OUT,
)

print(
    "SHA256:",
    lgb_model_sha,
)
print()


# ============================================================================
# 8. SOURCE VALIDATION INFERENCE
# ============================================================================

print("=" * 108)
print("LIGHTGBM SOURCE VALIDATION INFERENCE")
print("=" * 108)


pred_start = time.time()


p_lgb = np.asarray(
    model.predict_proba(
        X_bridge_val
    )[
        :,
        1
    ],
    dtype=np.float64,
)


prediction_seconds = (
    time.time()
    - pred_start
)


if p_lgb.shape != (
    VAL_ROWS,
):
    raise RuntimeError(
        "LightGBM validation prediction length mismatch."
    )


if not np.all(
    np.isfinite(
        p_lgb
    )
):
    raise RuntimeError(
        "Non-finite LightGBM validation probabilities."
    )


if (
    np.any(
        p_lgb < 0.0
    )
    or np.any(
        p_lgb > 1.0
    )
):
    raise RuntimeError(
        "LightGBM probabilities outside [0,1]."
    )


np.save(
    LGB_PROB_OUT,
    p_lgb,
    allow_pickle=False,
)


lgb_prob_sha = sha256_file(
    LGB_PROB_OUT
)


print(
    "Rows:",
    len(p_lgb),
)

print(
    "Prediction seconds:",
    round(
        prediction_seconds,
        3,
    ),
)

print(
    "Probability file:",
    LGB_PROB_OUT,
)

print(
    "Probability SHA256:",
    lgb_prob_sha,
)
print()


# ============================================================================
# 9. CHECKPOINT RECEIPT
# ============================================================================

checkpoint = {

    "stage":
        "Stage24-1B-R1",

    "type":
        "PRIMARY_BRIDGE62_LIGHTGBM_CPU_COMPLETION_CHECKPOINT",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "execution_parent":
        head,

    "direction":
        "IDS2018_TO_CICIDS2017",

    "representation":
        "bridge62",

    "feature_count":
        62,

    "source_train": {
        "rows":
            TRAIN_ROWS,

        "benign":
            TRAIN_BENIGN,

        "attack":
            TRAIN_ATTACK,

        "membership":
            "Stage22 CHRONOLOGICAL_NATURAL day_id 0..6",
    },

    "source_validation": {
        "rows":
            VAL_ROWS,

        "benign":
            VAL_BENIGN,

        "attack":
            VAL_ATTACK,

        "membership":
            "Stage22 CHRONOLOGICAL_NATURAL day_id 7",
    },

    "backend_amendment": {
        "artifact":
            str(
                AMENDMENT.relative_to(
                    REPO
                )
            ),

        "sha256":
            actual_amendment_sha,

        "only_change":
            "LightGBM device_type gpu -> cpu",
    },

    "lightgbm": {
        "configuration":
            "LGBM_11",

        "parameters":
            lgb_params,

        "backend":
            "CPU",

        "fit_completed":
            True,

        "fit_seconds":
            fit_seconds,

        "feature_count":
            int(
                booster.num_feature()
            ),

        "tree_count":
            int(
                booster.num_trees()
            ),

        "model_file":
            LGB_MODEL_OUT.name,

        "model_sha256":
            lgb_model_sha,

        "validation_probability_file":
            LGB_PROB_OUT.name,

        "validation_probability_sha256":
            lgb_prob_sha,

        "validation_prediction_seconds":
            prediction_seconds,
    },

    "scientific_fit_accounting": {
        "failed_lightgbm_gpu_attempts":
            1,

        "completed_lightgbm_fits":
            1,

        "completed_xgboost_fits":
            0,

        "completed_stage24_fits_after":
            1,

        "stage24_fit_budget":
            4,

        "additional_model_search":
            False,
    },

    "scientific_access": {
        "target_rows_read":
            0,

        "target_labels_read":
            0,

        "target_predictions":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,

        "target_opening_budget":
            8,
    },

    "threshold_selection": {
        "performed":
            False,

        "reason":
            "Awaiting frozen XGBoost component and ensemble.",
    },

    "next_authorized_step": (
        "Stage24-1B-R2 — fit exactly one bridge62 XGB_11 model "
        "with device=cuda, combine with this frozen LightGBM "
        "validation vector at 0.5/0.5, then perform frozen "
        "source-validation-only threshold selection."
    ),
}


checkpoint_sha = write_json_sha(
    CHECKPOINT_OUT,
    checkpoint,
)


print("=" * 108)
print("LIGHTGBM CHECKPOINT")
print("=" * 108)

print(
    "Receipt:",
    CHECKPOINT_OUT,
)

print(
    "Receipt SHA256:",
    checkpoint_sha,
)

print()

print(
    "Completed scientific fits: 1 / 4"
)

print(
    "Target openings: 0 / 8"
)
print()


# ============================================================================
# 10. RELEASE MODEL OBJECT, KEEP MEMMAP FILES
# ============================================================================

del model
del booster

gc.collect()


# DO NOT delete X_PATH or Y_PATH.
print(
    "[PASS] Training memmap preserved for XGBoost resume."
)
print()


# ============================================================================
# 11. COMMIT + PUSH CHECKPOINT IMMEDIATELY
# ============================================================================

print("=" * 108)
print("COMMIT + PUSH LIGHTGBM CHECKPOINT")
print("=" * 108)


token = None
token_source = None


try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for key in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:
            value = secrets.get_secret(
                key
            )

            if value and value.strip():
                token = value.strip()
                token_source = key
                break

        except Exception:
            pass

except Exception:
    pass


if token is None:

    for key in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            key
        )

        if value and value.strip():
            token = value.strip()
            token_source = (
                "ENV:"
                + key
            )
            break


if token is None:
    raise RuntimeError(
        "GitHub token unavailable."
    )


basic = base64.b64encode(
    (
        "x-access-token:"
        + token
    ).encode()
).decode()


files_to_stage = [
    LGB_MODEL_OUT,
    LGB_PROB_OUT,
    CHECKPOINT_OUT,
    CHECKPOINT_SHA_OUT,
]


git(
    "add",
    "--",
    *[
        str(
            path.relative_to(
                REPO
            )
        )
        for path
        in files_to_stage
    ],
)


staged = [
    item
    for item
    in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if item
]


expected_staged = {
    str(
        path.relative_to(
            REPO
        )
    )
    for path
    in files_to_stage
}


if set(staged) != expected_staged:
    raise RuntimeError(
        "\nUnexpected staged set.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


print(
    "Credential source:",
    token_source,
)

print(
    "Staged files:",
    len(staged),
)


for item in staged:
    print(
        "  +",
        item,
    )


git(
    "commit",
    "-m",
    "stage24: complete bridge62 LightGBM CPU checkpoint",
)


commit = git(
    "rev-parse",
    "HEAD",
)


parent = git(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_HEAD:
    raise RuntimeError(
        "\nCheckpoint commit parent mismatch.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {parent}"
    )


print()

print(
    git(
        "push",
        "origin",
        "HEAD:main",
        auth_header=basic,
    )
)


remote_after = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=basic,
).split()[0]


if remote_after != commit:
    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Worktree is not clean after checkpoint."
    )


print()

print(
    "[PASS] LightGBM CPU checkpoint committed and pushed."
)

print(
    "Commit:",
    commit,
)

print(
    "Remote main:",
    remote_after,
)

print(
    "Worktree: <clean>"
)
print()


# ============================================================================
# 12. FINAL
# ============================================================================

print("=" * 108)
print("STAGE24-1B-R1 FINAL")
print("=" * 108)

print(
    "LightGBM GPU failed attempt:      1"
)

print(
    "LightGBM CPU completed fits:      1"
)

print(
    "XGBoost completed fits:           0"
)

print(
    "Completed Stage24 fits:           1 / 4"
)

print()

print(
    "LightGBM model SHA256:"
)

print(
    " ",
    lgb_model_sha,
)

print()

print(
    "Validation probabilities SHA256:"
)

print(
    " ",
    lgb_prob_sha,
)

print()

print(
    "Training memmap preserved:        YES"
)

print(
    "Target rows read:                 0"
)

print(
    "Target labels read:               0"
)

print(
    "Target predictions:               0"
)

print(
    "Target openings:                  0 / 8"
)

print()

print(
    "Checkpoint commit:"
)

print(
    " ",
    commit,
)

print()

print("-" * 108)
print("STAGE24-1B-R1: PASS")
print()
print(
    "NEXT: Stage24-1B-R2 — XGBoost CUDA fit + "
    "ensemble + frozen source threshold selection."
)
print("-" * 108)

In [ ]:
# ============================================================================
# STAGE24-1B-R1-GITFIX
# CHECKPOINT ALREADY-COMPLETED LIGHTGBM FIT
#
# NO FIT
# NO INFERENCE
# NO DATA RECONSTRUCTION
# NO TARGET ACCESS
#
# Fixes only:
#   bridge62_validation_lightgbm_probabilities.npy is ignored by .gitignore
#
# Completed scientific fits remain: 1 / 4
# Target openings remain:           0 / 8
# ============================================================================

from pathlib import Path
import subprocess
import hashlib
import base64
import os

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")

EXPECTED_HEAD = "e9544a1db582c61ca446636ef84d576184c72616"

OUT_REL = Path(
    "results/stage24_cross_dataset/"
    "stage24_1_primary_source_sanity/"
    "stage24_1b_bridge62_source_refit"
)

OUT = REPO / OUT_REL

MODEL = OUT / "bridge62_lightgbm_model.txt"
PROB = OUT / "bridge62_validation_lightgbm_probabilities.npy"
RECEIPT = OUT / "stage24_1b_r1_lightgbm_cpu_checkpoint.json"
RECEIPT_SHA = OUT / "stage24_1b_r1_lightgbm_cpu_checkpoint.sha256"

EXPECTED_MODEL_SHA = (
    "907c9c86755c3c32effb2d0899f92e1a0cce71915f90311c99afd8c73814c4d7"
)

EXPECTED_PROB_SHA = (
    "cbafa1d3e3bce5068eb94f4ff1b163870fec3e808990dbc1e9b07fe63c86d379"
)

EXPECTED_RECEIPT_SHA = (
    "dfe89c91c95f78fcd202fc07506acff452b737a7b80891717eceae282ea3c03a"
)


print("=" * 108)
print("STAGE24-1B-R1-GITFIX — CHECKPOINT COMPLETED LIGHTGBM FIT")
print("=" * 108)
print("Scientific training:      NONE")
print("Completed fits:           1 / 4")
print("Target openings:          0 / 8")
print()


def git(*args, auth_header=None, check=True):
    cmd = ["git"]

    if auth_header is not None:
        cmd += [
            "-c", "credential.helper=",
            "-c", f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [str(x) for x in args]

    p = subprocess.run(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "\nGit command failed:\n"
            + " ".join(cmd)
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ============================================================================
# 1. GOVERNANCE
# ============================================================================

head = git("rev-parse", "HEAD")
remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
).split()[0]

print("Local HEAD: ", head)
print("Remote main:", remote)

if head != EXPECTED_HEAD:
    raise RuntimeError(
        f"\nUnexpected HEAD.\nExpected: {EXPECTED_HEAD}\nActual:   {head}"
    )

if remote != EXPECTED_HEAD:
    raise RuntimeError(
        f"\nUnexpected remote main.\nExpected: {EXPECTED_HEAD}\nActual:   {remote}"
    )

print()
print("[PASS] Repository still at Stage24-1B-EX1 amendment commit.")
print()


# ============================================================================
# 2. VERIFY SUCCESSFUL LIGHTGBM ARTIFACTS
# ============================================================================

print("=" * 108)
print("VERIFYING COMPLETED LIGHTGBM ARTIFACTS")
print("=" * 108)

for path in [MODEL, PROB, RECEIPT, RECEIPT_SHA]:
    if not path.is_file():
        raise FileNotFoundError(path)


model_sha = sha256_file(MODEL)
prob_sha = sha256_file(PROB)
receipt_sha = sha256_file(RECEIPT)

print("Model:")
print("  expected:", EXPECTED_MODEL_SHA)
print("  actual:  ", model_sha)

print("Probabilities:")
print("  expected:", EXPECTED_PROB_SHA)
print("  actual:  ", prob_sha)

print("Checkpoint:")
print("  expected:", EXPECTED_RECEIPT_SHA)
print("  actual:  ", receipt_sha)


if model_sha != EXPECTED_MODEL_SHA:
    raise RuntimeError("LightGBM model changed after completed fit.")

if prob_sha != EXPECTED_PROB_SHA:
    raise RuntimeError("LightGBM probability artifact changed.")

if receipt_sha != EXPECTED_RECEIPT_SHA:
    raise RuntimeError("LightGBM checkpoint receipt changed.")


sidecar = RECEIPT_SHA.read_text(encoding="utf-8").strip()

expected_sidecar = (
    f"{EXPECTED_RECEIPT_SHA}  {RECEIPT.name}"
)

if sidecar != expected_sidecar:
    raise RuntimeError(
        "\nCheckpoint SHA sidecar mismatch.\n"
        f"Expected: {expected_sidecar}\n"
        f"Actual:   {sidecar}"
    )


print()
print("[PASS] Completed LightGBM fit artifacts are byte-exact.")
print()


# ============================================================================
# 3. CLEAN INDEX ONLY — NEVER DELETE WORKING FILES
#
# Previous git add failed on ignored .npy. Reset index to known HEAD state,
# preserving all successful LightGBM output files.
# ============================================================================

git("reset", "--quiet", "HEAD", "--")

# Add normal tracked/unignored artifacts.
git(
    "add",
    "--",
    str(MODEL.relative_to(REPO)),
    str(RECEIPT.relative_to(REPO)),
    str(RECEIPT_SHA.relative_to(REPO)),
)

# Probability vector is intentionally ignored globally, so force-add ONLY it.
git(
    "add",
    "-f",
    "--",
    str(PROB.relative_to(REPO)),
)


# ============================================================================
# 4. REQUIRE EXACT STAGED SET
# ============================================================================

staged = [
    x
    for x in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if x
]

expected_staged = {
    str(MODEL.relative_to(REPO)),
    str(PROB.relative_to(REPO)),
    str(RECEIPT.relative_to(REPO)),
    str(RECEIPT_SHA.relative_to(REPO)),
}

print("=" * 108)
print("STAGED CHECKPOINT")
print("=" * 108)

for item in staged:
    print("  +", item)


if set(staged) != expected_staged:
    raise RuntimeError(
        "\nUnexpected staged file set.\n"
        f"Expected:\n{sorted(expected_staged)}\n\n"
        f"Actual:\n{sorted(staged)}"
    )

print()
print("[PASS] Exact four-file checkpoint staged.")
print()


# ============================================================================
# 5. GITHUB CREDENTIAL
# ============================================================================

token = None
token_source = None

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for key in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:
        try:
            value = secrets.get_secret(key)

            if value and value.strip():
                token = value.strip()
                token_source = key
                break

        except Exception:
            pass

except Exception:
    pass


if token is None:
    for key in ["GITHUB_TOKEN", "GH_TOKEN"]:
        value = os.environ.get(key)

        if value and value.strip():
            token = value.strip()
            token_source = "ENV:" + key
            break


if token is None:
    raise RuntimeError("GitHub token unavailable.")


basic = base64.b64encode(
    ("x-access-token:" + token).encode()
).decode()

print("Credential source:", token_source)
print()


# ============================================================================
# 6. COMMIT
# ============================================================================

git(
    "commit",
    "-m",
    "stage24: complete bridge62 LightGBM CPU checkpoint",
)

commit = git("rev-parse", "HEAD")
parent = git("rev-parse", "HEAD^")

if parent != EXPECTED_HEAD:
    raise RuntimeError(
        "\nCheckpoint commit parent mismatch.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {parent}"
    )


# ============================================================================
# 7. PUSH + REMOTE VERIFY
# ============================================================================

print(
    git(
        "push",
        "origin",
        "HEAD:main",
        auth_header=basic,
    )
)

remote_after = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=basic,
).split()[0]

if remote_after != commit:
    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


# Runtime memmaps live outside repo and therefore do not affect this.
status = git("status", "--porcelain")

if status:
    raise RuntimeError(
        "\nRepository worktree not clean after checkpoint:\n"
        + status
    )


# ============================================================================
# 8. MAKE SURE MEMMAP STILL EXISTS
# ============================================================================

X_MEMMAP = Path(
    "/kaggle/working/stage24_bridge62_runtime/"
    "bridge62_train_X_float64.dat"
)

Y_MEMMAP = Path(
    "/kaggle/working/stage24_bridge62_runtime/"
    "bridge62_train_y_uint8.dat"
)

if not X_MEMMAP.is_file():
    raise RuntimeError("Training X memmap disappeared.")

if not Y_MEMMAP.is_file():
    raise RuntimeError("Training y memmap disappeared.")


expected_x_bytes = 13_818_623 * 62 * 8
expected_y_bytes = 13_818_623

if X_MEMMAP.stat().st_size != expected_x_bytes:
    raise RuntimeError("Training X memmap size changed.")

if Y_MEMMAP.stat().st_size != expected_y_bytes:
    raise RuntimeError("Training y memmap size changed.")


# ============================================================================
# 9. FINAL
# ============================================================================

print()
print("=" * 108)
print("STAGE24-1B-R1-GITFIX FINAL")
print("=" * 108)

print("LightGBM completed fit:       YES")
print("Completed scientific fits:    1 / 4")
print("XGBoost completed fits:       0")
print()

print("LightGBM model SHA:")
print(" ", model_sha)

print("Validation probability SHA:")
print(" ", prob_sha)

print()

print("Checkpoint commit:")
print(" ", commit)

print("Remote main:")
print(" ", remote_after)

print()

print("6.38 GiB training memmap:     PRESERVED")
print("Target rows read:             0")
print("Target labels read:           0")
print("Target predictions:           0")
print("Target openings:              0 / 8")

print()
print("-" * 108)
print("STAGE24-1B-R1 CHECKPOINT: PASS")
print()
print(
    "NEXT: Stage24-1B-R2 — XGBoost CUDA only, "
    "then ensemble + source-validation threshold selection."
)
print("-" * 108)
print("=" * 108)

In [ ]:
# ============================================================================
# STAGE24-1B-R2 — BRIDGE62 XGBOOST CUDA + FINALIZE SOURCE REFIT
#
# Scientific state entering this cell:
#
#   Failed LightGBM GPU runtime attempts: 1
#   Completed LightGBM CPU fits:         1
#   Completed XGBoost fits:              0
#   Completed Stage24 fits:              1 / 4
#   Target openings:                     0 / 8
#
# THIS CELL:
#   1. verifies committed LightGBM checkpoint
#   2. attaches existing 6.38 GiB training memmap
#   3. performs EXACTLY ONE XGB_11 CUDA fit
#   4. freezes XGBoost model + validation probabilities
#   5. forms frozen 0.5 / 0.5 ensemble in float64 -> float32
#   6. computes source PR-AUC / ROC-AUC
#   7. applies frozen Stage20-1E3 threshold semantics:
#          integer percent 5..95
#          Balanced = max F1
#          Security = FPR <= .05, max F2
#          Standard = .50
#   8. commits/pushes complete Stage24-1B artifacts
#   9. ONLY AFTER remote verification deletes temporary 6.38 GiB memmap
#
# NO CICIDS2017 target data is opened.
# NO hyperparameter tuning.
# NO threshold selection on target.
#
# Completed Stage24 fits after PASS: 2 / 4
# Target openings after PASS:        0 / 8
# ============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import subprocess

from pathlib import Path
from datetime import datetime, timezone
from fractions import Fraction

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
)


# ============================================================================
# 0. CONSTANTS
# ============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "3009c3a9e4bcd9ede1aedcca900ee38e30eae670"
)

BRANCH = "main"


STAGE22_RESULT = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)


SEMANTIC_BRIDGE = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_semantic_bridge_spec.json"
)


STAGE24_1A = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1a_full_source_70f_sanity.json"
)


AMENDMENT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
    / "stage24_1b_ex1_lightgbm_backend_amendment.json"
)


LGB_CHECKPOINT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
    / "stage24_1b_r1_lightgbm_cpu_checkpoint.json"
)


EXPECTED_AMENDMENT_SHA = (
    "10d31ed6c5254a66cdefa9a2769f5deae294850d4da658d4f09bd1a42d739df5"
)

EXPECTED_LGB_CHECKPOINT_SHA = (
    "dfe89c91c95f78fcd202fc07506acff452b737a7b80891717eceae282ea3c03a"
)

EXPECTED_LGB_MODEL_SHA = (
    "907c9c86755c3c32effb2d0899f92e1a0cce71915f90311c99afd8c73814c4d7"
)

EXPECTED_LGB_PROB_SHA = (
    "cbafa1d3e3bce5068eb94f4ff1b163870fec3e808990dbc1e9b07fe63c86d379"
)


OUT_REL = (
    Path("results")
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
)

OUT = REPO / OUT_REL


LGB_MODEL = (
    OUT
    / "bridge62_lightgbm_model.txt"
)

LGB_PROB = (
    OUT
    / "bridge62_validation_lightgbm_probabilities.npy"
)


XGB_MODEL = (
    OUT
    / "bridge62_xgboost_model.json"
)

XGB_PROB = (
    OUT
    / "bridge62_validation_xgboost_probabilities.npy"
)

ENSEMBLE_PROB = (
    OUT
    / "bridge62_validation_ensemble_probabilities.npz"
)

THRESHOLD_GRID = (
    OUT
    / "bridge62_validation_threshold_grid.csv"
)

RESULT = (
    OUT
    / "stage24_1b_bridge62_source_refit_result.json"
)

RESULT_SHA = (
    OUT
    / "stage24_1b_bridge62_source_refit_result.sha256"
)

CHECKSUMS = (
    OUT
    / "checksums.sha256"
)


TMP_ROOT = Path(
    "/kaggle/working/stage24_bridge62_runtime"
)

X_MEMMAP = (
    TMP_ROOT
    / "bridge62_train_X_float64.dat"
)

Y_MEMMAP = (
    TMP_ROOT
    / "bridge62_train_y_uint8.dat"
)


TRAIN_ROWS = 13_818_623
TRAIN_FEATURES = 62
TRAIN_BENIGN = 11_908_580
TRAIN_ATTACK = 1_910_043

VAL_ROWS = 593_780
VAL_BENIGN = 531_524
VAL_ATTACK = 62_256


EXPECTED_X_BYTES = (
    TRAIN_ROWS
    * TRAIN_FEATURES
    * 8
)

EXPECTED_Y_BYTES = (
    TRAIN_ROWS
)


print("=" * 108)
print("STAGE24-1B-R2 — XGBOOST CUDA + BRIDGE62 SOURCE FINALIZATION")
print("=" * 108)

print("Completed LightGBM fits:      1")
print("Completed XGBoost fits:       0")
print("Completed Stage24 fits:       1 / 4")
print("XGBoost backend:              CUDA")
print("Target rows read:             0")
print("Target labels read:           0")
print("Target predictions:           0")
print("Target openings:              0 / 8")
print()


# ============================================================================
# 1. HELPERS
# ============================================================================

def git(
    *args,
    auth_header=None,
    check=True,
):
    cmd = ["git"]

    if auth_header is not None:

        cmd.extend(
            [
                "-c",
                "credential.helper=",
                "-c",
                f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
            ]
        )

    cmd.extend(
        map(str, args)
    )

    p = subprocess.run(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nGit failure:\n"
            + " ".join(cmd)
            + "\n\n"
            + (p.stdout or "")
        )

    return (
        p.stdout or ""
    ).strip()


def load_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def array_sha256(a):

    a = np.ascontiguousarray(
        a
    )

    h = hashlib.sha256()

    h.update(
        str(
            a.dtype
        ).encode()
    )

    h.update(
        repr(
            tuple(
                a.shape
            )
        ).encode()
    )

    h.update(
        a.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


def write_json_sha(
    path,
    obj,
):

    path = Path(
        path
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    digest = sha256_file(
        path
    )

    path.with_suffix(
        ".sha256"
    ).write_text(
        f"{digest}  {path.name}\n",
        encoding="utf-8",
    )

    return digest


def safe_div(
    numerator,
    denominator,
):

    if denominator == 0:
        return 0.0

    return (
        numerator
        / denominator
    )


def confusion_counts(
    y_true,
    probability,
    threshold,
):

    pred = (
        probability
        >= threshold
    )

    positive = (
        y_true
        == 1
    )

    tp = int(
        np.count_nonzero(
            pred
            & positive
        )
    )

    tn = int(
        np.count_nonzero(
            ~pred
            & ~positive
        )
    )

    fp = int(
        np.count_nonzero(
            pred
            & ~positive
        )
    )

    fn = int(
        np.count_nonzero(
            ~pred
            & positive
        )
    )

    return (
        tp,
        tn,
        fp,
        fn,
    )


def threshold_record(
    integer_percent,
    y_true,
    probability,
):

    threshold_runtime = float(
        np.float32(
            integer_percent
            / 100.0
        )
    )

    tp, tn, fp, fn = confusion_counts(
        y_true,
        probability,
        threshold_runtime,
    )

    precision = safe_div(
        tp,
        tp + fp,
    )

    recall = safe_div(
        tp,
        tp + fn,
    )

    fpr = safe_div(
        fp,
        fp + tn,
    )

    f1 = safe_div(
        2 * tp,
        (
            2 * tp
            + fp
            + fn
        ),
    )

    f2 = safe_div(
        5 * tp,
        (
            5 * tp
            + fp
            + 4 * fn
        ),
    )

    accuracy = safe_div(
        tp + tn,
        (
            tp
            + tn
            + fp
            + fn
        ),
    )

    return {
        "threshold_integer_percent":
            int(
                integer_percent
            ),

        "threshold":
            integer_percent
            / 100.0,

        "threshold_float32_runtime":
            threshold_runtime,

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn,

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "fpr":
            fpr,

        "f1":
            f1,

        "f2":
            f2,
    }


def f1_fraction(
    row
):
    return Fraction(
        2 * row[
            "tp"
        ],
        (
            2 * row[
                "tp"
            ]
            + row[
                "fp"
            ]
            + row[
                "fn"
            ]
        ),
    )


def f2_fraction(
    row
):
    return Fraction(
        5 * row[
            "tp"
        ],
        (
            5 * row[
                "tp"
            ]
            + row[
                "fp"
            ]
            + 4 * row[
                "fn"
            ]
        ),
    )


def fpr_fraction(
    row
):

    denominator = (
        row[
            "fp"
        ]
        + row[
            "tn"
        ]
    )

    if denominator == 0:

        return Fraction(
            0,
            1,
        )

    return Fraction(
        row[
            "fp"
        ],
        denominator,
    )


def recall_fraction(
    row
):

    denominator = (
        row[
            "tp"
        ]
        + row[
            "fn"
        ]
    )

    if denominator == 0:

        return Fraction(
            0,
            1,
        )

    return Fraction(
        row[
            "tp"
        ],
        denominator,
    )


# ============================================================================
# 2. GOVERNANCE
# ============================================================================

print("=" * 108)
print("GOVERNANCE")
print("=" * 108)


branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
).split()[0]

status = git(
    "status",
    "--porcelain",
)


print(
    "Branch:",
    branch,
)

print(
    "Local HEAD:",
    head,
)

print(
    "Remote main:",
    remote,
)


if branch != BRANCH:

    raise RuntimeError(
        "Wrong branch."
    )


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "\nUnexpected Stage24-1B-R2 parent.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if remote != head:

    raise RuntimeError(
        "Local HEAD != remote main."
    )


if status:

    raise RuntimeError(
        "\nWorktree must be clean before XGBoost fit:\n"
        + status
    )


print()

print(
    "[PASS] Starting from remotely checkpointed LightGBM commit."
)
print()


# ============================================================================
# 3. VERIFY FROZEN LIGHTGBM CHECKPOINT
# ============================================================================

print("=" * 108)
print("LIGHTGBM CHECKPOINT INTEGRITY")
print("=" * 108)


for path in [
    AMENDMENT,
    LGB_CHECKPOINT,
    LGB_MODEL,
    LGB_PROB,
]:

    if not path.is_file():

        raise FileNotFoundError(
            path
        )


amendment_sha = sha256_file(
    AMENDMENT
)

checkpoint_sha = sha256_file(
    LGB_CHECKPOINT
)

lgb_model_sha = sha256_file(
    LGB_MODEL
)

lgb_prob_sha = sha256_file(
    LGB_PROB
)


checks = [
    (
        "Amendment",
        amendment_sha,
        EXPECTED_AMENDMENT_SHA,
    ),
    (
        "LightGBM checkpoint",
        checkpoint_sha,
        EXPECTED_LGB_CHECKPOINT_SHA,
    ),
    (
        "LightGBM model",
        lgb_model_sha,
        EXPECTED_LGB_MODEL_SHA,
    ),
    (
        "LightGBM probabilities",
        lgb_prob_sha,
        EXPECTED_LGB_PROB_SHA,
    ),
]


for name, actual, expected in checks:

    print(name)

    print(
        "  expected:",
        expected,
    )

    print(
        "  actual:  ",
        actual,
    )

    if actual != expected:

        raise RuntimeError(
            f"{name} SHA mismatch."
        )


checkpoint = load_json(
    LGB_CHECKPOINT
)


if not checkpoint[
    "lightgbm"
][
    "fit_completed"
]:

    raise RuntimeError(
        "LightGBM checkpoint does not record completed fit."
    )


if (
    checkpoint[
        "scientific_fit_accounting"
    ][
        "completed_stage24_fits_after"
    ]
    != 1
):

    raise RuntimeError(
        "Completed Stage24 fit count before XGB != 1."
    )


if (
    checkpoint[
        "scientific_access"
    ][
        "target_openings_consumed"
    ]
    != 0
):

    raise RuntimeError(
        "Target opening ledger not pristine."
    )


print()

print(
    "[PASS] Frozen LightGBM component verified."
)
print()


# ============================================================================
# 4. ATTACH EXISTING TRAINING MEMMAP
# ============================================================================

print("=" * 108)
print("ATTACHING EXISTING 6.38 GiB TRAINING MEMMAP")
print("=" * 108)


if not X_MEMMAP.is_file():

    raise RuntimeError(
        "Training X memmap disappeared."
    )


if not Y_MEMMAP.is_file():

    raise RuntimeError(
        "Training y memmap disappeared."
    )


if X_MEMMAP.stat().st_size != EXPECTED_X_BYTES:

    raise RuntimeError(
        "Training X memmap byte size mismatch."
    )


if Y_MEMMAP.stat().st_size != EXPECTED_Y_BYTES:

    raise RuntimeError(
        "Training y memmap byte size mismatch."
    )


X_train_xgb = np.memmap(
    X_MEMMAP,
    dtype=np.float64,
    mode="r",
    shape=(
        TRAIN_ROWS,
        TRAIN_FEATURES,
    ),
)


y_train_xgb = np.memmap(
    Y_MEMMAP,
    dtype=np.uint8,
    mode="r",
    shape=(
        TRAIN_ROWS,
    ),
)


train_attack = int(
    np.sum(
        y_train_xgb,
        dtype=np.int64,
    )
)

train_benign = (
    TRAIN_ROWS
    - train_attack
)


print(
    "X:",
    X_train_xgb.shape,
)

print(
    "Benign:",
    train_benign,
)

print(
    "Attack:",
    train_attack,
)


if train_attack != TRAIN_ATTACK:

    raise RuntimeError(
        "Training attack count mismatch."
    )


if train_benign != TRAIN_BENIGN:

    raise RuntimeError(
        "Training benign count mismatch."
    )


print()

print(
    "[PASS] Exact bridge62 training memmap attached."
)
print()


# ============================================================================
# 5. SOURCE VALIDATION MATRIX
# ============================================================================

print("=" * 108)
print("SOURCE VALIDATION MATRIX")
print("=" * 108)


stage22 = load_json(
    STAGE22_RESULT
)

bridge_spec = load_json(
    SEMANTIC_BRIDGE
)


features70 = list(
    stage22[
        "data"
    ][
        "feature_order"
    ]
)


bridge62 = list(
    bridge_spec[
        "bridge62"
    ][
        "source_feature_order"
    ]
)


if len(
    bridge62
) != 62:

    raise RuntimeError(
        "bridge62 dimension != 62."
    )


# Prefer already-created validation bridge.
if (
    "X_bridge_val"
    in globals()
    and isinstance(
        X_bridge_val,
        np.ndarray,
    )
    and X_bridge_val.shape
        == (
            VAL_ROWS,
            62,
        )
):

    X_val62 = np.ascontiguousarray(
        X_bridge_val,
        dtype=np.float64,
    )

    validation_source = (
        "existing X_bridge_val"
    )


elif (
    "X_val"
    in globals()
    and isinstance(
        X_val,
        np.ndarray,
    )
    and X_val.shape
        == (
            VAL_ROWS,
            62,
        )
):

    X_val62 = np.ascontiguousarray(
        X_val,
        dtype=np.float64,
    )

    validation_source = (
        "existing X_val"
    )


else:

    if "X" not in globals():

        raise RuntimeError(
            "No in-memory Stage24-1A validation matrix remains."
        )


    if X.shape != (
        VAL_ROWS,
        70,
    ):

        raise RuntimeError(
            f"Unexpected Stage24-1A X shape: {X.shape}"
        )


    bridge_indices = [
        features70.index(
            feature
        )
        for feature
        in bridge62
    ]


    X_val62 = np.ascontiguousarray(
        X[
            :,
            bridge_indices,
        ],
        dtype=np.float64,
    )

    validation_source = (
        "Stage24-1A X -> bridge62"
    )


if (
    "y_bridge_val"
    in globals()
):

    y_val62 = np.asarray(
        y_bridge_val,
        dtype=np.uint8,
    )


elif (
    "y_val"
    in globals()
):

    y_val62 = np.asarray(
        y_val,
        dtype=np.uint8,
    )


elif (
    "y"
    in globals()
):

    y_val62 = np.asarray(
        y,
        dtype=np.uint8,
    )


else:

    raise RuntimeError(
        "Source validation labels unavailable."
    )


if X_val62.shape != (
    VAL_ROWS,
    62,
):

    raise RuntimeError(
        "Validation bridge62 matrix shape mismatch."
    )


if y_val62.shape != (
    VAL_ROWS,
):

    raise RuntimeError(
        "Validation label shape mismatch."
    )


val_attack = int(
    y_val62.sum()
)

val_benign = (
    VAL_ROWS
    - val_attack
)


if val_attack != VAL_ATTACK:

    raise RuntimeError(
        "Validation attack count mismatch."
    )


if val_benign != VAL_BENIGN:

    raise RuntimeError(
        "Validation benign count mismatch."
    )


print(
    "Validation source:",
    validation_source,
)

print(
    "X:",
    X_val62.shape,
)

print(
    "Benign:",
    val_benign,
)

print(
    "Attack:",
    val_attack,
)

print()

print(
    "[PASS] Exact bridge62 validation matrix ready."
)
print()


# ============================================================================
# 6. LOAD FROZEN LIGHTGBM VALIDATION VECTOR
# ============================================================================

p_lgb = np.load(
    LGB_PROB,
    allow_pickle=False,
)


p_lgb = np.asarray(
    p_lgb,
    dtype=np.float64,
)


if p_lgb.shape != (
    VAL_ROWS,
):

    raise RuntimeError(
        "Frozen LightGBM probability shape mismatch."
    )


if not np.all(
    np.isfinite(
        p_lgb
    )
):

    raise RuntimeError(
        "Frozen LightGBM probabilities contain non-finite values."
    )


print("=" * 108)
print("FROZEN LIGHTGBM VALIDATION VECTOR")
print("=" * 108)

print(
    "Rows:",
    len(
        p_lgb
    ),
)

print(
    "SHA256:",
    lgb_prob_sha,
)

print(
    "[PASS] Ready for post-XGB ensemble."
)
print()


# ============================================================================
# 7. FROZEN XGBOOST PARAMETERS
# ============================================================================

print("=" * 108)
print("FROZEN XGB_11 CUDA PARAMETERS")
print("=" * 108)


xgb_params = dict(
    stage22[
        "models"
    ][
        "xgboost"
    ][
        "parameters"
    ]
)


expected_xgb = {
    "colsample_bytree": 1.0,
    "device": "cuda",
    "eval_metric": "logloss",
    "gamma": 0.0,
    "learning_rate": 0.06,
    "max_depth": 7,
    "min_child_weight": 1,
    "n_estimators": 400,
    "n_jobs": -1,
    "objective": "binary:logistic",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "tree_method": "hist",
}


if xgb_params != expected_xgb:

    raise RuntimeError(
        "\nFrozen XGB parameter mismatch.\n"
        + json.dumps(
            xgb_params,
            indent=2,
            sort_keys=True,
        )
    )


print(
    json.dumps(
        xgb_params,
        indent=2,
        sort_keys=True,
    )
)

print()

print(
    "[PASS] XGBoost remains exactly XGB_11 / CUDA."
)
print()


# ============================================================================
# 8. SCIENTIFIC FIT 2 / 4 — XGBOOST CUDA
# ============================================================================

print("=" * 108)
print("SCIENTIFIC FIT 2 / 4 — BRIDGE62 XGBOOST CUDA")
print("=" * 108)


fit_start = time.time()


xgb_model = xgb.XGBClassifier(
    **xgb_params
)


xgb_model.fit(
    X_train_xgb,
    y_train_xgb,
)


xgb_fit_seconds = (
    time.time()
    - fit_start
)


booster = (
    xgb_model
    .get_booster()
)


if int(
    booster.num_features()
) != 62:

    raise RuntimeError(
        "XGBoost completed model feature count != 62."
    )


if int(
    booster.num_boosted_rounds()
) != 400:

    raise RuntimeError(
        "XGBoost completed rounds != 400."
    )


print()

print(
    "Fit seconds:",
    round(
        xgb_fit_seconds,
        3,
    ),
)

print(
    "Features:",
    booster.num_features(),
)

print(
    "Boosted rounds:",
    booster.num_boosted_rounds(),
)

print()

print(
    "[PASS] XGBoost CUDA scientific fit completed."
)
print()


# ============================================================================
# 9. FREEZE XGBOOST MODEL IMMEDIATELY
# ============================================================================

xgb_model.save_model(
    str(
        XGB_MODEL
    )
)


xgb_model_sha = sha256_file(
    XGB_MODEL
)


print("=" * 108)
print("XGBOOST MODEL FROZEN LOCALLY")
print("=" * 108)

print(
    "Model:",
    XGB_MODEL,
)

print(
    "SHA256:",
    xgb_model_sha,
)
print()


# ============================================================================
# 10. XGBOOST SOURCE VALIDATION INFERENCE
# ============================================================================

print("=" * 108)
print("XGBOOST SOURCE VALIDATION INFERENCE")
print("=" * 108)


pred_start = time.time()


p_xgb = np.asarray(
    xgb_model.predict_proba(
        X_val62
    )[
        :,
        1
    ],
    dtype=np.float64,
)


xgb_prediction_seconds = (
    time.time()
    - pred_start
)


if p_xgb.shape != (
    VAL_ROWS,
):

    raise RuntimeError(
        "XGBoost validation probability shape mismatch."
    )


if not np.all(
    np.isfinite(
        p_xgb
    )
):

    raise RuntimeError(
        "XGBoost validation probabilities contain non-finite values."
    )


if (
    np.any(
        p_xgb < 0.0
    )
    or np.any(
        p_xgb > 1.0
    )
):

    raise RuntimeError(
        "XGBoost probabilities outside [0,1]."
    )


np.save(
    XGB_PROB,
    p_xgb,
    allow_pickle=False,
)


xgb_prob_sha = sha256_file(
    XGB_PROB
)


print(
    "Rows:",
    len(
        p_xgb
    ),
)

print(
    "Prediction seconds:",
    round(
        xgb_prediction_seconds,
        3,
    ),
)

print(
    "Probability SHA256:",
    xgb_prob_sha,
)
print()


# ============================================================================
# 11. FROZEN 0.5 / 0.5 ENSEMBLE
# ============================================================================

print("=" * 108)
print("BRIDGE62 SOURCE VALIDATION ENSEMBLE")
print("=" * 108)


# Component probabilities are float64 here.
# Frozen Stage22 combination is float64 then persisted float32.
p_ensemble = (
    0.5
    * p_lgb
    + 0.5
    * p_xgb
).astype(
    np.float32
)


if p_ensemble.shape != (
    VAL_ROWS,
):

    raise RuntimeError(
        "Ensemble probability shape mismatch."
    )


if not np.all(
    np.isfinite(
        p_ensemble
    )
):

    raise RuntimeError(
        "Ensemble contains non-finite values."
    )


# Stage22 clean positions for day_id7 are exact and contiguous.
clean_position = np.arange(
    13_818_623,
    14_412_403,
    dtype=np.int64,
)


if len(
    clean_position
) != VAL_ROWS:

    raise RuntimeError(
        "Validation clean_position length mismatch."
    )


np.savez_compressed(
    ENSEMBLE_PROB,
    clean_position=clean_position,
    binary_label=np.asarray(
        y_val62,
        dtype=np.uint8,
    ),
    ensemble_probability=p_ensemble,
)


ensemble_file_sha = sha256_file(
    ENSEMBLE_PROB
)

ensemble_array_sha = array_sha256(
    p_ensemble
)


print(
    "Rows:",
    len(
        p_ensemble
    ),
)

print(
    "Storage dtype:",
    p_ensemble.dtype,
)

print(
    "Probability file SHA256:",
    ensemble_file_sha,
)

print(
    "Probability array SHA256:",
    ensemble_array_sha,
)
print()


# ============================================================================
# 12. SOURCE VALIDATION RANKING METRICS
# ============================================================================

pr_auc = float(
    average_precision_score(
        y_val62,
        p_ensemble,
    )
)


roc_auc = float(
    roc_auc_score(
        y_val62,
        p_ensemble,
    )
)


brier = float(
    brier_score_loss(
        y_val62,
        p_ensemble,
    )
)


logloss = float(
    log_loss(
        y_val62,
        np.clip(
            p_ensemble.astype(
                np.float64
            ),
            1e-15,
            1.0
            - 1e-15,
        ),
        labels=[
            0,
            1,
        ],
    )
)


prevalence = float(
    VAL_ATTACK
    / VAL_ROWS
)


pr_excess = (
    pr_auc
    - prevalence
)


if prevalence < 1.0:

    pr_normalized = (
        pr_auc
        - prevalence
    ) / (
        1.0
        - prevalence
    )

else:

    pr_normalized = None


print("=" * 108)
print("SOURCE VALIDATION METRICS")
print("=" * 108)

print(
    "Prevalence:",
    format(
        prevalence,
        ".15f",
    ),
)

print(
    "PR-AUC:",
    format(
        pr_auc,
        ".15f",
    ),
)

print(
    "PR excess over prevalence:",
    format(
        pr_excess,
        ".15f",
    ),
)

print(
    "PR normalized:",
    (
        format(
            pr_normalized,
            ".15f",
        )
        if pr_normalized
        is not None
        else "NA"
    ),
)

print(
    "ROC-AUC:",
    format(
        roc_auc,
        ".15f",
    ),
)

print(
    "Brier:",
    format(
        brier,
        ".15f",
    ),
)

print(
    "Log loss:",
    format(
        logloss,
        ".15f",
    ),
)

print()


# ============================================================================
# 13. FROZEN THRESHOLD GRID — SOURCE VALIDATION ONLY
#
# EXACT Stage20-1E3 semantics:
#
# grid:
#       0.05 .. 0.95
#       integer percent 5..95
#
# Balanced:
#       max F1
#       tie: lower FPR
#            higher recall
#            closer to .50
#            lower threshold
#
# Security:
#       FPR <= .05
#       max F2
#       tie: lower FPR
#            higher recall
#            lower threshold
#
# Standard:
#       .50 fixed
# ============================================================================

print("=" * 108)
print("SOURCE-VALIDATION-ONLY FROZEN THRESHOLD SELECTION")
print("=" * 108)


grid = [
    threshold_record(
        integer_percent,
        y_val62,
        p_ensemble,
    )
    for integer_percent
    in range(
        5,
        96,
    )
]


pd.DataFrame(
    grid
).to_csv(
    THRESHOLD_GRID,
    index=False,
)


# --------------------------------------------------------------------------
# Standard
# --------------------------------------------------------------------------

standard = next(
    row
    for row
    in grid
    if row[
        "threshold_integer_percent"
    ]
    == 50
)


# --------------------------------------------------------------------------
# Balanced
# --------------------------------------------------------------------------

balanced = grid[
    0
]


for candidate in grid[
    1:
]:

    candidate_key = (
        f1_fraction(
            candidate
        ),
        -fpr_fraction(
            candidate
        ),
        recall_fraction(
            candidate
        ),
        -abs(
            candidate[
                "threshold_integer_percent"
            ]
            - 50
        ),
        -candidate[
            "threshold_integer_percent"
        ],
    )

    current_key = (
        f1_fraction(
            balanced
        ),
        -fpr_fraction(
            balanced
        ),
        recall_fraction(
            balanced
        ),
        -abs(
            balanced[
                "threshold_integer_percent"
            ]
            - 50
        ),
        -balanced[
            "threshold_integer_percent"
        ],
    )

    if candidate_key > current_key:

        balanced = candidate


# --------------------------------------------------------------------------
# Security
# --------------------------------------------------------------------------

security_eligible = [
    row
    for row
    in grid
    if (
        fpr_fraction(
            row
        )
        <= Fraction(
            1,
            20,
        )
    )
]


security = None


if security_eligible:

    security = (
        security_eligible[
            0
        ]
    )

    for candidate in (
        security_eligible[
            1:
        ]
    ):

        candidate_key = (
            f2_fraction(
                candidate
            ),
            -fpr_fraction(
                candidate
            ),
            recall_fraction(
                candidate
            ),
            -candidate[
                "threshold_integer_percent"
            ],
        )

        current_key = (
            f2_fraction(
                security
            ),
            -fpr_fraction(
                security
            ),
            recall_fraction(
                security
            ),
            -security[
                "threshold_integer_percent"
            ],
        )

        if candidate_key > current_key:

            security = candidate


print("STANDARD")
print(
    json.dumps(
        standard,
        indent=2,
    )
)

print()

print("BALANCED")
print(
    json.dumps(
        balanced,
        indent=2,
    )
)

print()

print("SECURITY")

if security is None:

    print(
        "UNAVAILABLE — NO RELAXATION"
    )

else:

    print(
        json.dumps(
            security,
            indent=2,
        )
    )

print()


# ============================================================================
# 14. DESCRIPTIVE FULL70 -> BRIDGE62 SOURCE COMPARISON
#
# Does NOT affect continuation.
# ============================================================================

stage24_1a = load_json(
    STAGE24_1A
)


full70_pr = float(
    stage24_1a[
        "source_metrics"
    ][
        "pr_auc"
    ]
)


full70_roc = float(
    stage24_1a[
        "source_metrics"
    ][
        "roc_auc"
    ]
)


source_comparison = {
    "full70_pr_auc":
        full70_pr,

    "bridge62_pr_auc":
        pr_auc,

    "bridge62_minus_full70_pr_auc":
        pr_auc
        - full70_pr,

    "full70_roc_auc":
        full70_roc,

    "bridge62_roc_auc":
        roc_auc,

    "bridge62_minus_full70_roc_auc":
        roc_auc
        - full70_roc,

    "performance_based_stop":
        False,

    "scientific_use":
        "DESCRIPTIVE_ONLY",
}


print("=" * 108)
print("FULL70 -> BRIDGE62 SOURCE COMPARISON — DESCRIPTIVE ONLY")
print("=" * 108)

print(
    "FULL70 PR-AUC:",
    full70_pr,
)

print(
    "bridge62 PR-AUC:",
    pr_auc,
)

print(
    "Delta:",
    source_comparison[
        "bridge62_minus_full70_pr_auc"
    ],
)

print()

print(
    "FULL70 ROC-AUC:",
    full70_roc,
)

print(
    "bridge62 ROC-AUC:",
    roc_auc,
)

print(
    "Delta:",
    source_comparison[
        "bridge62_minus_full70_roc_auc"
    ],
)

print()

print(
    "[FROZEN] These values cannot alter Stage24 target execution."
)
print()


# ============================================================================
# 15. WRITE FINAL STAGE24-1B RESULT
# ============================================================================

threshold_grid_sha = sha256_file(
    THRESHOLD_GRID
)


result = {
    "stage":
        "Stage24-1B",

    "status":
        "PRIMARY_BRIDGE62_SOURCE_REFIT_COMPLETE",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "execution_parent":
        head,

    "direction":
        "IDS2018_TO_CICIDS2017",

    "representation": {
        "name":
            "bridge62",

        "feature_count":
            62,

        "feature_order":
            bridge62,

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "dtype":
            "float64",
    },

    "source_train": {
        "membership":
            "Stage22 CHRONOLOGICAL_NATURAL day_id 0..6",

        "rows":
            TRAIN_ROWS,

        "benign":
            TRAIN_BENIGN,

        "attack":
            TRAIN_ATTACK,
    },

    "source_validation": {
        "membership":
            "Stage22 CHRONOLOGICAL_NATURAL day_id 7",

        "rows":
            VAL_ROWS,

        "benign":
            VAL_BENIGN,

        "attack":
            VAL_ATTACK,

        "prevalence":
            prevalence,
    },

    "lightgbm": {
        "configuration":
            "LGBM_11",

        "backend":
            "CPU",

        "backend_amendment_artifact":
            str(
                AMENDMENT.relative_to(
                    REPO
                )
            ),

        "backend_amendment_sha256":
            amendment_sha,

        "failed_gpu_runtime_attempts":
            1,

        "completed_fits":
            1,

        "fit_seconds":
            checkpoint[
                "lightgbm"
            ][
                "fit_seconds"
            ],

        "model_file":
            LGB_MODEL.name,

        "model_sha256":
            lgb_model_sha,

        "validation_probability_file":
            LGB_PROB.name,

        "validation_probability_sha256":
            lgb_prob_sha,
    },

    "xgboost": {
        "configuration":
            "XGB_11",

        "backend":
            "CUDA",

        "parameters":
            xgb_params,

        "completed_fits":
            1,

        "fit_seconds":
            xgb_fit_seconds,

        "model_file":
            XGB_MODEL.name,

        "model_sha256":
            xgb_model_sha,

        "validation_probability_file":
            XGB_PROB.name,

        "validation_probability_sha256":
            xgb_prob_sha,

        "validation_prediction_seconds":
            xgb_prediction_seconds,
    },

    "ensemble": {
        "rule":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "storage_dtype":
            "float32",

        "probability_file":
            ENSEMBLE_PROB.name,

        "probability_file_sha256":
            ensemble_file_sha,

        "probability_array_sha256":
            ensemble_array_sha,
    },

    "source_validation_metrics": {
        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "prevalence_chance_anchor":
            prevalence,

        "pr_excess":
            pr_excess,

        "pr_normalized":
            pr_normalized,

        "brier":
            brier,

        "log_loss":
            logloss,

        "standard":
            standard,

        "balanced":
            balanced,

        "security": {
            "status":
                (
                    "AVAILABLE"
                    if security
                    is not None
                    else "UNAVAILABLE_NO_RELAXATION"
                ),

            "result":
                security,
        },
    },

    "threshold_selection": {
        "source":
            "SOURCE_VALIDATION_ONLY",

        "target_feedback":
            False,

        "grid":
            {
                "start_integer_percent":
                    5,

                "end_integer_percent":
                    95,

                "step_integer_percent":
                    1,
            },

        "grid_file":
            THRESHOLD_GRID.name,

        "grid_sha256":
            threshold_grid_sha,
    },

    "source_sanity_comparison":
        source_comparison,

    "scientific_fit_accounting": {
        "failed_lightgbm_gpu_runtime_attempts":
            1,

        "completed_lightgbm_fits":
            1,

        "completed_xgboost_fits":
            1,

        "completed_stage24_fits_after":
            2,

        "stage24_fit_budget":
            4,

        "model_search":
            False,

        "hyperparameter_search":
            False,
    },

    "scientific_access": {
        "target_rows_read":
            0,

        "target_labels_read":
            0,

        "target_predictions":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,

        "target_opening_budget":
            8,
    },

    "anti_adaptation": {
        "feature_search":
            False,

        "hyperparameter_tuning":
            False,

        "target_feedback":
            False,

        "target_threshold_selection":
            False,

        "performance_based_stop":
            False,
    },

    "next_authorized_step": (
        "Stage24-1C — recover/replay exact GROUNDED_S4 membership "
        "before any primary target model inference. "
        "Target opening ledger remains 0/8."
    ),
}


result_sha = write_json_sha(
    RESULT,
    result,
)


print("=" * 108)
print("STAGE24-1B FINAL RESULT")
print("=" * 108)

print(
    "Result:",
    RESULT,
)

print(
    "SHA256:",
    result_sha,
)

print()


# ============================================================================
# 16. BUILD CHECKSUM MANIFEST
# ============================================================================

scientific_files = [
    AMENDMENT,
    LGB_CHECKPOINT,
    LGB_MODEL,
    LGB_PROB,
    XGB_MODEL,
    XGB_PROB,
    ENSEMBLE_PROB,
    THRESHOLD_GRID,
    RESULT,
    RESULT_SHA,
]


with CHECKSUMS.open(
    "w",
    encoding="utf-8",
) as f:

    for path in sorted(
        scientific_files,
        key=lambda p:
            p.name,
    ):

        f.write(
            f"{sha256_file(path)}  {path.name}\n"
        )


checksums_sha = sha256_file(
    CHECKSUMS
)


print(
    "checksums.sha256:",
    checksums_sha,
)
print()


# ============================================================================
# 17. RELEASE XGBOOST OBJECTS
# KEEP MEMMAP FILES UNTIL REMOTE PUSH PASSES
# ============================================================================

del xgb_model
del booster

gc.collect()


# ============================================================================
# 18. GITHUB AUTH
# ============================================================================

token = None
token_source = None


try:

    from kaggle_secrets import (
        UserSecretsClient,
    )

    secrets = UserSecretsClient()


    for key in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                key
            )

            if (
                value
                and value.strip()
            ):

                token = value.strip()

                token_source = key

                break

        except Exception:
            pass

except Exception:
    pass


if token is None:

    for key in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            key
        )

        if (
            value
            and value.strip()
        ):

            token = value.strip()

            token_source = (
                "ENV:"
                + key
            )

            break


if token is None:

    raise RuntimeError(
        "GitHub token unavailable."
    )


basic = base64.b64encode(
    (
        "x-access-token:"
        + token
    ).encode()
).decode()


print("=" * 108)
print("COMMIT + PUSH COMPLETE STAGE24-1B")
print("=" * 108)

print(
    "Credential source:",
    token_source,
)


# ============================================================================
# 19. STAGE EXACT OUTPUT SET
#
# Binary .npy / .npz files may be ignored globally.
# Therefore use git add -f ONLY for the scientific probability files.
# ============================================================================

normal_files = [
    XGB_MODEL,
    THRESHOLD_GRID,
    RESULT,
    RESULT_SHA,
    CHECKSUMS,
]


forced_files = [
    XGB_PROB,
    ENSEMBLE_PROB,
]


git(
    "add",
    "--",
    *[
        str(
            path.relative_to(
                REPO
            )
        )
        for path
        in normal_files
    ],
)


git(
    "add",
    "-f",
    "--",
    *[
        str(
            path.relative_to(
                REPO
            )
        )
        for path
        in forced_files
    ],
)


staged = [
    item
    for item
    in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if item
]


expected_staged = {
    str(
        path.relative_to(
            REPO
        )
    )
    for path
    in (
        normal_files
        + forced_files
    )
}


print()

print(
    "Staged files:",
    len(
        staged
    ),
)


for item in staged:

    print(
        "  +",
        item,
    )


if set(
    staged
) != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged output set.\n"
        f"Expected:\n{sorted(expected_staged)}\n\n"
        f"Actual:\n{sorted(staged)}"
    )


# ============================================================================
# 20. COMMIT
# ============================================================================

git(
    "commit",
    "-m",
    "stage24: complete primary bridge62 source refit",
)


commit = git(
    "rev-parse",
    "HEAD",
)


parent = git(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_HEAD:

    raise RuntimeError(
        "\nStage24-1B final commit parent mismatch.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {parent}"
    )


# ============================================================================
# 21. PUSH + REMOTE VERIFY
# ============================================================================

print()

print(
    git(
        "push",
        "origin",
        "HEAD:main",
        auth_header=basic,
    )
)


remote_after = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=basic,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


status_after = git(
    "status",
    "--porcelain",
)


if status_after:

    raise RuntimeError(
        "\nRepository worktree not clean after final push:\n"
        + status_after
    )


print()

print(
    "[PASS] Complete bridge62 source refit committed and remotely verified."
)

print(
    "Commit:",
    commit,
)

print(
    "Remote:",
    remote_after,
)

print()


# ============================================================================
# 22. ONLY NOW DELETE TEMPORARY TRAINING MEMMAP
# ============================================================================

print("=" * 108)
print("RELEASING TEMPORARY TRAINING CACHE")
print("=" * 108)


# Close memmap references first.
del X_train_xgb
del y_train_xgb

gc.collect()


deleted_runtime_bytes = 0


for path in [
    X_MEMMAP,
    Y_MEMMAP,
]:

    if path.exists():

        deleted_runtime_bytes += (
            path.stat().st_size
        )

        path.unlink()


try:

    TMP_ROOT.rmdir()

except OSError:
    pass


print(
    "Deleted bytes:",
    deleted_runtime_bytes,
)

print(
    "Deleted GiB:",
    round(
        deleted_runtime_bytes
        / 1024**3,
        3,
    ),
)

print()

print(
    "[PASS] Temporary bridge62 training memmap removed only after remote freeze."
)
print()


# ============================================================================
# 23. FINAL SYNTHESIS
# ============================================================================

print("=" * 108)
print("STAGE24-1B-R2 FINAL SYNTHESIS")
print("=" * 108)

print(
    "Representation:                 bridge62 / 62 features"
)

print()

print(
    "Source train:"
)

print(
    "  rows:                         13,818,623"
)

print(
    "  benign:                       11,908,580"
)

print(
    "  attack:                       1,910,043"
)

print()

print(
    "Source validation:"
)

print(
    "  rows:                         593,780"
)

print(
    "  benign:                       531,524"
)

print(
    "  attack:                       62,256"
)

print()

print(
    "LightGBM:"
)

print(
    "  backend:                      CPU"
)

print(
    "  completed fits:               1"
)

print(
    "  failed prior GPU attempts:    1"
)

print(
    "  model SHA256:"
)

print(
    " ",
    lgb_model_sha,
)

print()

print(
    "XGBoost:"
)

print(
    "  backend:                      CUDA"
)

print(
    "  completed fits:               1"
)

print(
    "  fit seconds:",
    round(
        xgb_fit_seconds,
        3,
    ),
)

print(
    "  model SHA256:"
)

print(
    " ",
    xgb_model_sha,
)

print()

print(
    "Ensemble:"
)

print(
    "  0.5 * LightGBM + 0.5 * XGBoost"
)

print(
    "  PR-AUC:",
    format(
        pr_auc,
        ".15f",
    ),
)

print(
    "  ROC-AUC:",
    format(
        roc_auc,
        ".15f",
    ),
)

print()

print(
    "Operating points:"
)

print(
    "  Standard:",
    standard[
        "threshold_float32_runtime"
    ],
)

print(
    "  Balanced:",
    balanced[
        "threshold_float32_runtime"
    ],
)

print(
    "  Security:",
    (
        security[
            "threshold_float32_runtime"
        ]
        if security
        is not None
        else "UNAVAILABLE"
    ),
)

print()

print(
    "Completed Stage24 fits:         2 / 4"
)

print(
    "Target rows read:               0"
)

print(
    "Target labels read:             0"
)

print(
    "Target predictions:             0"
)

print(
    "Target openings:                0 / 8"
)

print()

print(
    "Temporary 6.38 GiB memmap:      DELETED AFTER REMOTE FREEZE"
)

print()

print(
    "Final Stage24-1B commit:"
)

print(
    " ",
    commit,
)

print()

print("-" * 108)

print(
    "STAGE24-1B: PASS"
)

print()

print(
    "Primary bridge62 source model pair and source thresholds are frozen."
)

print(
    "Two of four Stage24 scientific fits are consumed."
)

print(
    "No target opening has been consumed."
)

print()

print(
    "NEXT:"
)

print(
    "Stage24-1C — exact GROUNDED_S4 membership recovery/replay."
)

print(
    "Do not open CICIDS2017 target inference yet."
)

print("-" * 108)
print("=" * 108)

In [ ]:
# ============================================================================
# STAGE24-1B-R2-GPU — GPU-NATIVE XGBOOST CUDA + SOURCE FINALIZATION
#
# IMPORTANT:
#
#   DO NOT rerun LightGBM.
#
#   Frozen LightGBM CPU component:
#       COMPLETE
#       remotely checkpointed
#
#   This cell repairs only the XGBoost execution architecture.
#
# GPU-NATIVE PATH:
#
#       disk memmap
#           ↓  [batch read only]
#       CuPy float64 batch on CUDA
#           ↓
#       XGBoost DataIter
#           ↓
#       GPU QuantileDMatrix
#           ↓
#       xgboost.train(device="cuda", tree_method="hist")
#           ↓
#       GPU inplace_predict(CuPy validation matrix)
#
# NO sklearn XGBClassifier.fit(memmap)
# NO CPU XGBoost learner
# NO target access
#
# Scientific state entering:
#
#   Failed LGBM GPU runtime attempt:       1
#   Completed LGBM CPU fits:              1
#   Interrupted XGB implementation attempt: 1
#   Completed XGB fits:                   0
#   Completed Stage24 fits:               1 / 4
#   Target openings:                      0 / 8
#
# This cell:
#   1. freezes GPU-native XGB execution amendment BEFORE retry
#   2. verifies CUDA + CuPy + GPU QuantileDMatrix
#   3. performs exactly ONE completed XGB_11 CUDA scientific fit
#   4. GPU validation inference
#   5. freezes XGB checkpoint
#   6. creates 0.5 / 0.5 ensemble
#   7. applies frozen Stage20-1E3 threshold semantics
#   8. commits/pushes Stage24-1B completion
#   9. deletes 6.38 GiB temp memmap ONLY after remote verification
#
# Completed Stage24 fits after PASS: 2 / 4
# Target openings after PASS:        0 / 8
# ============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import subprocess
import threading

from pathlib import Path
from datetime import datetime, timezone
from fractions import Fraction

import numpy as np
import pandas as pd
import cupy as cp
import xgboost as xgb


# ============================================================================
# 0. CONSTANTS
# ============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "3009c3a9e4bcd9ede1aedcca900ee38e30eae670"
)

BRANCH = "main"


STAGE22_RESULT = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)


SEMANTIC_BRIDGE = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_semantic_bridge_spec.json"
)


STAGE24_1A = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1a_full_source_70f_sanity.json"
)


LGB_AMENDMENT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
    / "stage24_1b_ex1_lightgbm_backend_amendment.json"
)


LGB_CHECKPOINT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
    / "stage24_1b_r1_lightgbm_cpu_checkpoint.json"
)


OUT_REL = (
    Path("results")
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
)

OUT = REPO / OUT_REL


LGB_MODEL = (
    OUT
    / "bridge62_lightgbm_model.txt"
)

LGB_PROB = (
    OUT
    / "bridge62_validation_lightgbm_probabilities.npy"
)


XGB_EXEC_AMENDMENT = (
    OUT
    / "stage24_1b_r2_ex1_xgboost_gpu_native_execution_amendment.json"
)

XGB_EXEC_AMENDMENT_SHA = (
    OUT
    / "stage24_1b_r2_ex1_xgboost_gpu_native_execution_amendment.sha256"
)


XGB_MODEL = (
    OUT
    / "bridge62_xgboost_model.json"
)

XGB_PROB = (
    OUT
    / "bridge62_validation_xgboost_probabilities.npy"
)

XGB_CHECKPOINT = (
    OUT
    / "stage24_1b_r2_xgboost_cuda_checkpoint.json"
)

XGB_CHECKPOINT_SHA = (
    OUT
    / "stage24_1b_r2_xgboost_cuda_checkpoint.sha256"
)


ENSEMBLE_PROB = (
    OUT
    / "bridge62_validation_ensemble_probabilities.npz"
)

THRESHOLD_GRID = (
    OUT
    / "bridge62_validation_threshold_grid.csv"
)

FINAL_RESULT = (
    OUT
    / "stage24_1b_bridge62_source_refit_result.json"
)

FINAL_RESULT_SHA = (
    OUT
    / "stage24_1b_bridge62_source_refit_result.sha256"
)

CHECKSUMS = (
    OUT
    / "checksums.sha256"
)


EXPECTED_LGB_AMENDMENT_SHA = (
    "10d31ed6c5254a66cdefa9a2769f5deae294850d4da658d4f09bd1a42d739df5"
)

EXPECTED_LGB_CHECKPOINT_SHA = (
    "dfe89c91c95f78fcd202fc07506acff452b737a7b80891717eceae282ea3c03a"
)

EXPECTED_LGB_MODEL_SHA = (
    "907c9c86755c3c32effb2d0899f92e1a0cce71915f90311c99afd8c73814c4d7"
)

EXPECTED_LGB_PROB_SHA = (
    "cbafa1d3e3bce5068eb94f4ff1b163870fec3e808990dbc1e9b07fe63c86d379"
)


TMP_ROOT = Path(
    "/kaggle/working/stage24_bridge62_runtime"
)

X_MEMMAP_PATH = (
    TMP_ROOT
    / "bridge62_train_X_float64.dat"
)

Y_MEMMAP_PATH = (
    TMP_ROOT
    / "bridge62_train_y_uint8.dat"
)


TRAIN_ROWS = 13_818_623
FEATURES = 62

TRAIN_BENIGN = 11_908_580
TRAIN_ATTACK = 1_910_043

VAL_ROWS = 593_780
VAL_BENIGN = 531_524
VAL_ATTACK = 62_256


EXPECTED_X_BYTES = (
    TRAIN_ROWS
    * FEATURES
    * 8
)

EXPECTED_Y_BYTES = (
    TRAIN_ROWS
)


# 500k rows x 62 x float64 ≈ 236 MiB GPU input batch.
GPU_BATCH_ROWS = 500_000

# XGB hist default inherited from XGB_11.
MAX_BIN = 256

NUM_BOOST_ROUND = 400


print("=" * 112)
print("STAGE24-1B-R2-GPU — GPU-NATIVE XGBOOST CUDA")
print("=" * 112)

print("LightGBM component:             FROZEN / COMPLETE")
print("XGBoost scientific fit:         CUDA GPU")
print("Quantile construction:          CUDA GPU / CuPy")
print("Validation prediction:          CUDA GPU / CuPy")
print("sklearn XGBClassifier(memmap):  NOT USED")
print()

print("Completed Stage24 fits before:  1 / 4")
print("Completed Stage24 fits after:   2 / 4")
print("Target openings:                0 / 8")
print()


# ============================================================================
# 1. HELPERS
# ============================================================================

def git(
    *args,
    auth_header=None,
    check=True,
):
    cmd = ["git"]

    if auth_header is not None:
        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [str(x) for x in args]

    p = subprocess.run(
        cmd,
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "\nGit command failed:\n"
            + " ".join(cmd)
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def load_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def array_sha256(a):
    a = np.ascontiguousarray(a)

    h = hashlib.sha256()

    h.update(str(a.dtype).encode())
    h.update(repr(tuple(a.shape)).encode())
    h.update(a.tobytes(order="C"))

    return h.hexdigest()


def write_json_sha(
    path,
    obj,
):
    path = Path(path)

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    digest = sha256_file(path)

    path.with_suffix(".sha256").write_text(
        f"{digest}  {path.name}\n",
        encoding="utf-8",
    )

    return digest


def safe_div(a, b):
    return (
        0.0
        if b == 0
        else a / b
    )


def confusion_counts(
    y_true,
    p,
    threshold,
):
    pred = p >= threshold
    pos = y_true == 1

    tp = int(
        np.count_nonzero(
            pred & pos
        )
    )

    tn = int(
        np.count_nonzero(
            ~pred & ~pos
        )
    )

    fp = int(
        np.count_nonzero(
            pred & ~pos
        )
    )

    fn = int(
        np.count_nonzero(
            ~pred & pos
        )
    )

    return tp, tn, fp, fn


def threshold_record(
    integer_percent,
    y_true,
    p,
):
    threshold = float(
        np.float32(
            integer_percent / 100.0
        )
    )

    tp, tn, fp, fn = confusion_counts(
        y_true,
        p,
        threshold,
    )

    return {
        "threshold_integer_percent":
            int(integer_percent),

        "threshold":
            integer_percent / 100.0,

        "threshold_float32_runtime":
            threshold,

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn,

        "accuracy":
            safe_div(
                tp + tn,
                tp + tn + fp + fn,
            ),

        "precision":
            safe_div(
                tp,
                tp + fp,
            ),

        "recall":
            safe_div(
                tp,
                tp + fn,
            ),

        "fpr":
            safe_div(
                fp,
                fp + tn,
            ),

        "f1":
            safe_div(
                2 * tp,
                2 * tp + fp + fn,
            ),

        "f2":
            safe_div(
                5 * tp,
                5 * tp + fp + 4 * fn,
            ),
    }


def f1_fraction(row):
    return Fraction(
        2 * row["tp"],
        2 * row["tp"]
        + row["fp"]
        + row["fn"],
    )


def f2_fraction(row):
    return Fraction(
        5 * row["tp"],
        5 * row["tp"]
        + row["fp"]
        + 4 * row["fn"],
    )


def fpr_fraction(row):
    d = (
        row["fp"]
        + row["tn"]
    )

    return (
        Fraction(0, 1)
        if d == 0
        else Fraction(
            row["fp"],
            d,
        )
    )


def recall_fraction(row):
    d = (
        row["tp"]
        + row["fn"]
    )

    return (
        Fraction(0, 1)
        if d == 0
        else Fraction(
            row["tp"],
            d,
        )
    )


# ============================================================================
# 2. FROZEN STAGE20 DISTINCT-SCORE PR / ROC
# ============================================================================

def frozen_ranking_metrics(
    y_true,
    probability_float32,
):
    """
    Stage20 semantics:
      - distinct float32 score groups
      - tied scores enter together
      - noninterpolated average precision
      - trapezoidal ROC
    """

    y_true = np.asarray(
        y_true,
        dtype=np.uint8,
    )

    p = np.asarray(
        probability_float32,
        dtype=np.float32,
    )

    if y_true.shape != p.shape:
        raise RuntimeError(
            "Ranking metric shape mismatch."
        )

    order = np.argsort(
        -p,
        kind="mergesort",
    )

    s = p[order]
    yy = y_true[order]

    tp_cum = np.cumsum(
        yy,
        dtype=np.int64,
    )

    fp_cum = np.cumsum(
        1 - yy,
        dtype=np.int64,
    )

    group_end = np.empty(
        len(s),
        dtype=bool,
    )

    group_end[:-1] = (
        s[:-1]
        != s[1:]
    )

    group_end[-1] = True

    tp = tp_cum[
        group_end
    ].astype(
        np.float64
    )

    fp = fp_cum[
        group_end
    ].astype(
        np.float64
    )

    positives = float(
        yy.sum()
    )

    negatives = float(
        len(yy)
        - yy.sum()
    )

    recall = (
        tp / positives
    )

    precision = (
        tp / (tp + fp)
    )

    previous_recall = np.concatenate(
        (
            np.array(
                [0.0],
                dtype=np.float64,
            ),
            recall[:-1],
        )
    )

    pr_auc = float(
        np.sum(
            (
                recall
                - previous_recall
            )
            * precision
        )
    )

    tpr = recall
    fpr = fp / negatives

    previous_tpr = np.concatenate(
        (
            np.array(
                [0.0],
                dtype=np.float64,
            ),
            tpr[:-1],
        )
    )

    previous_fpr = np.concatenate(
        (
            np.array(
                [0.0],
                dtype=np.float64,
            ),
            fpr[:-1],
        )
    )

    roc_auc = float(
        np.sum(
            (
                fpr
                - previous_fpr
            )
            * (
                tpr
                + previous_tpr
            )
            * 0.5
        )
    )

    return pr_auc, roc_auc


# ============================================================================
# 3. GITHUB AUTH
# ============================================================================

token = None
token_source = None

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for key in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:
            value = secrets.get_secret(
                key
            )

            if value and value.strip():
                token = value.strip()
                token_source = key
                break

        except Exception:
            pass

except Exception:
    pass


if token is None:

    for key in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            key
        )

        if value and value.strip():
            token = value.strip()
            token_source = "ENV:" + key
            break


if token is None:
    raise RuntimeError(
        "GitHub token unavailable."
    )


basic = base64.b64encode(
    (
        "x-access-token:"
        + token
    ).encode()
).decode()


# ============================================================================
# 4. GOVERNANCE BEFORE GPU RETRY
# ============================================================================

print("=" * 112)
print("GOVERNANCE")
print("=" * 112)


branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
).split()[0]

status = git(
    "status",
    "--porcelain",
)


print("Branch:", branch)
print("Local HEAD:", head)
print("Remote main:", remote)


if branch != BRANCH:
    raise RuntimeError(
        "Wrong branch."
    )


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "\nUnexpected parent.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if remote != head:
    raise RuntimeError(
        "Local HEAD != remote main."
    )


if status:
    raise RuntimeError(
        "\nWorktree changed after interrupted XGB attempt.\n"
        "Do NOT continue until ambiguity is resolved:\n"
        + status
    )


# Previous interrupted cell MUST NOT have produced a completed XGB artifact.
ambiguous_outputs = [
    XGB_MODEL,
    XGB_PROB,
    XGB_CHECKPOINT,
    ENSEMBLE_PROB,
    THRESHOLD_GRID,
    FINAL_RESULT,
]


existing_ambiguous = [
    str(p)
    for p
    in ambiguous_outputs
    if p.exists()
]


if existing_ambiguous:
    raise RuntimeError(
        "\nThe interrupted XGBoost attempt left scientific outputs.\n"
        "Do NOT overwrite them.\n"
        + "\n".join(
            existing_ambiguous
        )
    )


print()
print(
    "[PASS] Interrupted XGB attempt produced no frozen scientific artifact."
)
print(
    "[PASS] Completed Stage24 fits remain 1 / 4."
)
print(
    "[PASS] Target openings remain 0 / 8."
)
print()


# ============================================================================
# 5. VERIFY LIGHTGBM CHECKPOINT
# ============================================================================

print("=" * 112)
print("FROZEN LIGHTGBM COMPONENT")
print("=" * 112)


frozen_checks = [
    (
        LGB_AMENDMENT,
        EXPECTED_LGB_AMENDMENT_SHA,
        "LightGBM amendment",
    ),
    (
        LGB_CHECKPOINT,
        EXPECTED_LGB_CHECKPOINT_SHA,
        "LightGBM checkpoint",
    ),
    (
        LGB_MODEL,
        EXPECTED_LGB_MODEL_SHA,
        "LightGBM model",
    ),
    (
        LGB_PROB,
        EXPECTED_LGB_PROB_SHA,
        "LightGBM probabilities",
    ),
]


for path, expected, name in frozen_checks:

    actual = sha256_file(
        path
    )

    print(name)
    print("  expected:", expected)
    print("  actual:  ", actual)

    if actual != expected:
        raise RuntimeError(
            f"{name} SHA mismatch."
        )


print()
print(
    "[PASS] Completed LightGBM component remains frozen."
)
print()


# ============================================================================
# 6. VERIFY TRAINING MEMMAP
# ============================================================================

print("=" * 112)
print("TRAINING MEMMAP")
print("=" * 112)


if not X_MEMMAP_PATH.is_file():
    raise RuntimeError(
        "6.38 GiB X training memmap disappeared."
    )


if not Y_MEMMAP_PATH.is_file():
    raise RuntimeError(
        "Training label memmap disappeared."
    )


if (
    X_MEMMAP_PATH.stat().st_size
    != EXPECTED_X_BYTES
):
    raise RuntimeError(
        "X memmap byte count mismatch."
    )


if (
    Y_MEMMAP_PATH.stat().st_size
    != EXPECTED_Y_BYTES
):
    raise RuntimeError(
        "y memmap byte count mismatch."
    )


X_mm = np.memmap(
    X_MEMMAP_PATH,
    dtype=np.float64,
    mode="r",
    shape=(
        TRAIN_ROWS,
        FEATURES,
    ),
)


y_mm = np.memmap(
    Y_MEMMAP_PATH,
    dtype=np.uint8,
    mode="r",
    shape=(
        TRAIN_ROWS,
    ),
)


attack_count = int(
    np.sum(
        y_mm,
        dtype=np.int64,
    )
)

benign_count = (
    TRAIN_ROWS
    - attack_count
)


print("Shape:", X_mm.shape)
print("Benign:", benign_count)
print("Attack:", attack_count)


if attack_count != TRAIN_ATTACK:
    raise RuntimeError(
        "Training attack count mismatch."
    )


if benign_count != TRAIN_BENIGN:
    raise RuntimeError(
        "Training benign count mismatch."
    )


print()
print(
    "[PASS] Exact bridge62 training matrix preserved."
)
print()


# ============================================================================
# 7. CUDA ENVIRONMENT
# ============================================================================

print("=" * 112)
print("CUDA ENVIRONMENT")
print("=" * 112)


device_count = int(
    cp.cuda.runtime.getDeviceCount()
)


print(
    "XGBoost:",
    xgb.__version__,
)

print(
    "CuPy:",
    cp.__version__,
)

print(
    "CUDA devices visible:",
    device_count,
)


if device_count < 1:
    raise RuntimeError(
        "No CUDA device available."
    )


for device_id in range(
    device_count
):

    props = cp.cuda.runtime.getDeviceProperties(
        device_id
    )

    name = props["name"]

    if isinstance(
        name,
        bytes,
    ):
        name = name.decode()

    print(
        f"GPU {device_id}:",
        name,
    )


# Frozen XGB_11 is single-device "cuda".
# XGBoost therefore uses CUDA device 0.
cp.cuda.Device(0).use()


free_bytes, total_bytes = (
    cp.cuda.runtime.memGetInfo()
)


print()
print(
    "GPU0 free GiB:",
    round(
        free_bytes
        / 1024**3,
        3,
    ),
)

print(
    "GPU0 total GiB:",
    round(
        total_bytes
        / 1024**3,
        3,
    ),
)
print()


# ============================================================================
# 8. CLEAR PARTIAL INTERRUPTED XGB MEMORY OBJECTS
# ============================================================================

for variable_name in [
    "xgb_model",
    "booster",
    "dtrain",
    "qdm_train",
    "Xy_train",
]:

    if variable_name in globals():

        try:
            del globals()[
                variable_name
            ]

        except Exception:
            pass


gc.collect()

cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

cp.cuda.Device(0).synchronize()


# ============================================================================
# 9. GPU-NATIVE INFRASTRUCTURE PROBE — SYNTHETIC ONLY
#
# Not a scientific fit. No IDS source/target predictors involved.
# ============================================================================

print("=" * 112)
print("GPU-NATIVE QUANTILEDMATRIX PROBE")
print("=" * 112)


probe_X = cp.arange(
    4096 * FEATURES,
    dtype=cp.float64,
).reshape(
    4096,
    FEATURES,
)


probe_y = (
    cp.arange(
        4096,
        dtype=cp.uint8,
    )
    % 2
)


probe_dm = xgb.QuantileDMatrix(
    probe_X,
    probe_y,
    max_bin=MAX_BIN,
)


probe_model = xgb.train(
    {
        "objective":
            "binary:logistic",

        "tree_method":
            "hist",

        "device":
            "cuda",

        "max_depth":
            2,

        "learning_rate":
            0.1,

        "seed":
            42,
    },
    probe_dm,
    num_boost_round=1,
)


probe_cfg = probe_model.save_config()


if (
    '"device":"cuda:0"'
    not in probe_cfg.replace(
        " ",
        "",
    )
):
    raise RuntimeError(
        "Synthetic GPU-native probe did not resolve to cuda:0."
    )


del (
    probe_model,
    probe_dm,
    probe_X,
    probe_y,
)

cp.get_default_memory_pool().free_all_blocks()
gc.collect()

cp.cuda.Device(0).synchronize()


print(
    "[PASS] CuPy -> QuantileDMatrix -> CUDA training works."
)
print()


# ============================================================================
# 10. FREEZE GPU-NATIVE EXECUTION AMENDMENT BEFORE RETRY
# ============================================================================

execution_amendment = {

    "stage":
        "Stage24-1B-R2-EX1",

    "status":
        "XGBOOST_GPU_NATIVE_EXECUTION_PATH_FROZEN",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        head,

    "trigger": {

        "previous_execution_interface":
            "XGBClassifier.fit(NUMPY_MEMMAP)",

        "requested_backend":
            "cuda",

        "tree_method":
            "hist",

        "outcome":
            "USER_INTERRUPTED_BEFORE_COMPLETED_XGBOOST_FIT",

        "reason":
            (
                "Host memmap / sklearn execution path caused substantial "
                "CPU-side matrix preparation while accelerator quota was active."
            ),

        "completed_xgboost_fit":
            False,

        "xgboost_model_serialized":
            False,

        "source_validation_probability_observed":
            False,

        "ensemble_probability_observed":
            False,

        "threshold_selection_performed":
            False,

        "target_rows_read":
            0,

        "target_labels_read":
            0,

        "target_predictions":
            0,

        "target_openings_consumed":
            0,
    },

    "corrective_execution_rule": {

        "scientific_model":
            "XGB_11_UNCHANGED",

        "data_membership":
            "UNCHANGED",

        "feature_representation":
            "UNCHANGED_BRIDGE62",

        "numeric_source_input":
            "UNCHANGED_FLOAT64",

        "hyperparameters":
            "UNCHANGED",

        "backend":
            "CUDA_UNCHANGED",

        "tree_method":
            "HIST_UNCHANGED",

        "execution_interface": {

            "from":
                "SKLEARN_XGBCLASSIFIER_WITH_HOST_NUMPY_MEMMAP",

            "to":
                (
                    "CUPY_FLOAT64_BATCHES_TO_GPU_QUANTILEDMATRIX_"
                    "THEN_XGBOOST_TRAIN_CUDA"
                ),
        },

        "wrapper_to_native_parameter_translation": {

            "n_estimators_400":
                "num_boost_round_400",

            "random_state_42":
                "seed_42",

            "n_jobs_minus_1":
                "nthread_minus_1",
        },

        "max_bin":
            (
                "256_DEFAULT_INHERITED; made explicit only for "
                "QuantileDMatrix construction"
            ),

        "validation_prediction":
            "CUDA_INPLACE_PREDICT_FROM_CUPY",

        "ensemble":
            "UNCHANGED_0.5_LGBM_PLUS_0.5_XGB",

        "threshold_semantics":
            "UNCHANGED_STAGE20_1E3_SOURCE_VALIDATION_ONLY",
    },

    "infrastructure_probe": {

        "synthetic_only":
            True,

        "cupy_quantile_dmatrix":
            True,

        "xgboost_cuda_train":
            True,

        "resolved_device":
            "cuda:0",

        "scientific_fit":
            False,
    },

    "scientific_fit_accounting": {

        "completed_lightgbm_fits":
            1,

        "interrupted_xgboost_execution_attempts":
            1,

        "completed_xgboost_fits":
            0,

        "completed_stage24_fits":
            1,

        "stage24_fit_budget":
            4,
    },

    "anti_adaptation": {

        "target_feedback":
            False,

        "performance_feedback":
            False,

        "hyperparameter_tuning":
            False,

        "feature_change":
            False,

        "threshold_feedback":
            False,
    },

    "next_authorized_step":
        (
            "Construct the frozen bridge62 source training QuantileDMatrix "
            "from CuPy float64 batches and perform exactly one XGB_11 "
            "scientific fit with device=cuda and tree_method=hist."
        ),
}


exec_amendment_sha = write_json_sha(
    XGB_EXEC_AMENDMENT,
    execution_amendment,
)


print("=" * 112)
print("GPU EXECUTION AMENDMENT")
print("=" * 112)

print(
    "Artifact:",
    XGB_EXEC_AMENDMENT,
)

print(
    "SHA256:",
    exec_amendment_sha,
)
print()


# Commit amendment BEFORE scientific XGB retry.
git(
    "add",
    "--",
    str(
        XGB_EXEC_AMENDMENT.relative_to(
            REPO
        )
    ),
    str(
        XGB_EXEC_AMENDMENT_SHA.relative_to(
            REPO
        )
    ),
)


staged = [
    x
    for x
    in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if x
]


expected_staged = {
    str(
        XGB_EXEC_AMENDMENT.relative_to(
            REPO
        )
    ),
    str(
        XGB_EXEC_AMENDMENT_SHA.relative_to(
            REPO
        )
    ),
}


if set(staged) != expected_staged:
    raise RuntimeError(
        "Unexpected amendment staged set."
    )


git(
    "commit",
    "-m",
    "stage24: freeze GPU-native XGBoost execution path",
)


AMENDMENT_COMMIT = git(
    "rev-parse",
    "HEAD",
)


if (
    git(
        "rev-parse",
        "HEAD^",
    )
    != EXPECTED_HEAD
):
    raise RuntimeError(
        "GPU execution amendment parent mismatch."
    )


print(
    git(
        "push",
        "origin",
        "HEAD:main",
        auth_header=basic,
    )
)


remote_after_amendment = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=basic,
).split()[0]


if (
    remote_after_amendment
    != AMENDMENT_COMMIT
):
    raise RuntimeError(
        "GPU execution amendment remote verification failed."
    )


if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Worktree not clean after GPU execution amendment."
    )


print()
print(
    "[PASS] GPU-native execution path frozen and remotely verified."
)
print(
    "Amendment commit:",
    AMENDMENT_COMMIT,
)
print()


# ============================================================================
# 11. RECOVER EXACT BRIDGE62 VALIDATION MATRIX
# ============================================================================

print("=" * 112)
print("SOURCE VALIDATION MATRIX")
print("=" * 112)


stage22 = load_json(
    STAGE22_RESULT
)

bridge_spec = load_json(
    SEMANTIC_BRIDGE
)


features70 = list(
    stage22[
        "data"
    ][
        "feature_order"
    ]
)


bridge62 = list(
    bridge_spec[
        "bridge62"
    ][
        "source_feature_order"
    ]
)


if len(bridge62) != 62:
    raise RuntimeError(
        "bridge62 dimension != 62."
    )


if (
    "X_val62" in globals()
    and isinstance(
        X_val62,
        np.ndarray,
    )
    and X_val62.shape
        == (
            VAL_ROWS,
            62,
        )
):

    X_source_val = np.ascontiguousarray(
        X_val62,
        dtype=np.float64,
    )

    validation_source = "existing X_val62"


elif (
    "X_bridge_val" in globals()
    and isinstance(
        X_bridge_val,
        np.ndarray,
    )
    and X_bridge_val.shape
        == (
            VAL_ROWS,
            62,
        )
):

    X_source_val = np.ascontiguousarray(
        X_bridge_val,
        dtype=np.float64,
    )

    validation_source = "existing X_bridge_val"


elif (
    "X_val" in globals()
    and isinstance(
        X_val,
        np.ndarray,
    )
    and X_val.shape
        == (
            VAL_ROWS,
            62,
        )
):

    X_source_val = np.ascontiguousarray(
        X_val,
        dtype=np.float64,
    )

    validation_source = "existing X_val"


else:

    if "X" not in globals():
        raise RuntimeError(
            "Stage24-1A validation matrix is no longer in memory."
        )


    if X.shape != (
        VAL_ROWS,
        70,
    ):
        raise RuntimeError(
            f"Unexpected Stage24-1A X shape: {X.shape}"
        )


    bridge_indices = [
        features70.index(
            feature
        )
        for feature
        in bridge62
    ]


    X_source_val = np.ascontiguousarray(
        X[
            :,
            bridge_indices,
        ],
        dtype=np.float64,
    )

    validation_source = (
        "Stage24-1A X -> bridge62"
    )


if "y_val62" in globals():

    y_source_val = np.asarray(
        y_val62,
        dtype=np.uint8,
    )

elif "y_bridge_val" in globals():

    y_source_val = np.asarray(
        y_bridge_val,
        dtype=np.uint8,
    )

elif "y_val" in globals():

    y_source_val = np.asarray(
        y_val,
        dtype=np.uint8,
    )

elif "y" in globals():

    y_source_val = np.asarray(
        y,
        dtype=np.uint8,
    )

else:

    raise RuntimeError(
        "Source validation labels unavailable."
    )


if X_source_val.shape != (
    VAL_ROWS,
    62,
):
    raise RuntimeError(
        "Source validation X shape mismatch."
    )


if y_source_val.shape != (
    VAL_ROWS,
):
    raise RuntimeError(
        "Source validation y shape mismatch."
    )


if int(
    y_source_val.sum()
) != VAL_ATTACK:
    raise RuntimeError(
        "Validation attack count mismatch."
    )


print(
    "Source:",
    validation_source,
)

print(
    "Shape:",
    X_source_val.shape,
)

print(
    "Benign:",
    VAL_BENIGN,
)

print(
    "Attack:",
    VAL_ATTACK,
)

print()
print(
    "[PASS] Exact bridge62 validation population ready."
)
print()


# ============================================================================
# 12. LOAD FROZEN LIGHTGBM VALIDATION VECTOR
# ============================================================================

p_lgb = np.load(
    LGB_PROB,
    allow_pickle=False,
)


p_lgb = np.asarray(
    p_lgb,
    dtype=np.float64,
)


if p_lgb.shape != (
    VAL_ROWS,
):
    raise RuntimeError(
        "LightGBM validation probability shape mismatch."
    )


if not np.all(
    np.isfinite(
        p_lgb
    )
):
    raise RuntimeError(
        "LightGBM probability vector contains non-finite values."
    )


# ============================================================================
# 13. GPU DATA ITERATOR
#
# CPU does ONLY disk-backed batch retrieval.
# Every batch supplied to XGBoost is a CuPy CUDA array.
# ============================================================================

class Bridge62GPUDataIter(
    xgb.DataIter
):

    def __init__(
        self,
        X_memmap,
        y_memmap,
        batch_rows,
    ):

        self.X_memmap = X_memmap
        self.y_memmap = y_memmap
        self.batch_rows = int(
            batch_rows
        )

        self.rows = int(
            X_memmap.shape[0]
        )

        self._start = 0
        self._x_gpu = None
        self._y_gpu = None

        self.transfer_seconds = 0.0
        self.batches_supplied = 0
        self.rows_supplied = 0

        super().__init__(
            cache_prefix=None,
            release_data=True,
        )


    def reset(self):

        self._start = 0
        self._x_gpu = None
        self._y_gpu = None


    def next(
        self,
        input_data,
    ):

        if self._start >= self.rows:

            self._x_gpu = None
            self._y_gpu = None

            return False


        end = min(
            self._start
            + self.batch_rows,
            self.rows,
        )


        transfer_start = time.time()


        # Exact frozen float64 source data.
        X_host = np.asarray(
            self.X_memmap[
                self._start:
                end,
                :
            ],
            dtype=np.float64,
        )


        y_host = np.asarray(
            self.y_memmap[
                self._start:
                end
            ],
            dtype=np.uint8,
        )


        # Explicit CUDA transfer.
        self._x_gpu = cp.asarray(
            X_host,
            dtype=cp.float64,
        )


        self._y_gpu = cp.asarray(
            y_host,
            dtype=cp.uint8,
        )


        cp.cuda.Device(0).synchronize()


        self.transfer_seconds += (
            time.time()
            - transfer_start
        )


        input_data(
            data=self._x_gpu,
            label=self._y_gpu,
        )


        batch_n = (
            end
            - self._start
        )


        self.batches_supplied += 1
        self.rows_supplied += batch_n

        self._start = end

        return True


# ============================================================================
# 14. GPU UTILIZATION MONITOR
# ============================================================================

gpu_samples = []
monitor_stop = threading.Event()


def gpu_monitor():

    while not monitor_stop.is_set():

        try:

            output = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu="
                    "index,utilization.gpu,memory.used",
                    "--format=csv,noheader,nounits",
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.DEVNULL,
                text=True,
                check=False,
            ).stdout.strip()


            timestamp = time.time()


            for line in output.splitlines():

                pieces = [
                    x.strip()
                    for x
                    in line.split(",")
                ]


                if len(pieces) == 3:

                    gpu_samples.append(
                        {
                            "timestamp":
                                timestamp,

                            "gpu":
                                int(pieces[0]),

                            "utilization_percent":
                                int(pieces[1]),

                            "memory_used_mib":
                                int(pieces[2]),
                        }
                    )

        except Exception:
            pass


        monitor_stop.wait(
            1.0
        )


# ============================================================================
# 15. XGB_11 NATIVE CUDA PARAMETERS
#
# sklearn-wrapper -> native XGBoost translation only:
#
#   n_estimators=400 -> num_boost_round=400
#   random_state=42  -> seed=42
#   n_jobs=-1        -> nthread=-1
#
# Algorithmic values unchanged.
# ============================================================================

frozen_xgb_wrapper = dict(
    stage22[
        "models"
    ][
        "xgboost"
    ][
        "parameters"
    ]
)


expected_wrapper = {
    "colsample_bytree": 1.0,
    "device": "cuda",
    "eval_metric": "logloss",
    "gamma": 0.0,
    "learning_rate": 0.06,
    "max_depth": 7,
    "min_child_weight": 1,
    "n_estimators": 400,
    "n_jobs": -1,
    "objective": "binary:logistic",
    "random_state": 42,
    "reg_alpha": 0.0,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "tree_method": "hist",
}


if (
    frozen_xgb_wrapper
    != expected_wrapper
):
    raise RuntimeError(
        "\nFrozen XGB_11 wrapper parameters differ.\n"
        + json.dumps(
            frozen_xgb_wrapper,
            indent=2,
            sort_keys=True,
        )
    )


xgb_native_params = {
    "colsample_bytree":
        1.0,

    "device":
        "cuda",

    "eval_metric":
        "logloss",

    "gamma":
        0.0,

    "learning_rate":
        0.06,

    "max_depth":
        7,

    "min_child_weight":
        1,

    "nthread":
        -1,

    "objective":
        "binary:logistic",

    "seed":
        42,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "subsample":
        0.9,

    "tree_method":
        "hist",
}


print("=" * 112)
print("FROZEN XGB_11 CUDA PARAMETERS")
print("=" * 112)

print(
    json.dumps(
        xgb_native_params,
        indent=2,
        sort_keys=True,
    )
)

print(
    "num_boost_round:",
    NUM_BOOST_ROUND,
)

print(
    "QuantileDMatrix max_bin:",
    MAX_BIN,
)
print()


# ============================================================================
# 16. START GPU MONITOR
# ============================================================================

monitor_thread = threading.Thread(
    target=gpu_monitor,
    daemon=True,
)

monitor_thread.start()


# ============================================================================
# 17. CONSTRUCT GPU QUANTILEDMATRIX
# ============================================================================

print("=" * 112)
print("GPU QUANTILEDMATRIX CONSTRUCTION")
print("=" * 112)


train_iter = Bridge62GPUDataIter(
    X_mm,
    y_mm,
    GPU_BATCH_ROWS,
)


qdm_start = time.time()


try:

    qdm_train = xgb.QuantileDMatrix(
        train_iter,
        max_bin=MAX_BIN,
        missing=np.nan,
        nthread=-1,
    )

except Exception:

    monitor_stop.set()
    monitor_thread.join(
        timeout=5
    )

    raise


qdm_seconds = (
    time.time()
    - qdm_start
)


if int(
    qdm_train.num_row()
) != TRAIN_ROWS:
    raise RuntimeError(
        "QuantileDMatrix row count mismatch."
    )


if int(
    qdm_train.num_col()
) != FEATURES:
    raise RuntimeError(
        "QuantileDMatrix feature count mismatch."
    )


print(
    "Rows:",
    qdm_train.num_row(),
)

print(
    "Features:",
    qdm_train.num_col(),
)

print(
    "Construction seconds:",
    round(
        qdm_seconds,
        3,
    ),
)

print(
    "Iterator batch calls:",
    train_iter.batches_supplied,
)

print(
    "Iterator rows supplied across passes:",
    train_iter.rows_supplied,
)

print(
    "Host -> GPU transfer seconds:",
    round(
        train_iter.transfer_seconds,
        3,
    ),
)

print()
print(
    "[PASS] Training data quantized through CUDA CuPy batches."
)
print()


# Release iterator's current raw GPU batch before boosting.
train_iter._x_gpu = None
train_iter._y_gpu = None

gc.collect()

cp.get_default_memory_pool().free_all_blocks()

cp.cuda.Device(0).synchronize()


# ============================================================================
# 18. SCIENTIFIC FIT 2 / 4 — GPU
# ============================================================================

print("=" * 112)
print("SCIENTIFIC FIT 2 / 4 — XGB_11 CUDA GPU")
print("=" * 112)


fit_start = time.time()


try:

    booster = xgb.train(
        params=xgb_native_params,
        dtrain=qdm_train,
        num_boost_round=NUM_BOOST_ROUND,
        verbose_eval=False,
    )

except Exception:

    monitor_stop.set()
    monitor_thread.join(
        timeout=5
    )

    raise


fit_seconds = (
    time.time()
    - fit_start
)


if int(
    booster.num_features()
) != FEATURES:
    raise RuntimeError(
        "Completed XGBoost feature count != 62."
    )


if int(
    booster.num_boosted_rounds()
) != NUM_BOOST_ROUND:
    raise RuntimeError(
        "Completed XGBoost round count != 400."
    )


booster_config = booster.save_config()

compact_config = booster_config.replace(
    " ",
    "",
)


if (
    '"device":"cuda:0"'
    not in compact_config
):
    raise RuntimeError(
        "\nCompleted booster configuration does not report cuda:0.\n"
        "Do NOT accept this fit."
    )


print(
    "Fit seconds:",
    round(
        fit_seconds,
        3,
    ),
)

print(
    "Features:",
    booster.num_features(),
)

print(
    "Boosted rounds:",
    booster.num_boosted_rounds(),
)

print(
    "Resolved device: cuda:0"
)

print()
print(
    "[PASS] XGB_11 scientific fit completed on CUDA."
)
print()


# ============================================================================
# 19. SAVE XGBOOST MODEL IMMEDIATELY
# ============================================================================

booster.save_model(
    str(
        XGB_MODEL
    )
)


xgb_model_sha = sha256_file(
    XGB_MODEL
)


print("=" * 112)
print("XGBOOST MODEL FROZEN LOCALLY")
print("=" * 112)

print(
    "Model:",
    XGB_MODEL,
)

print(
    "SHA256:",
    xgb_model_sha,
)
print()


# ============================================================================
# 20. GPU VALIDATION PREDICTION
# ============================================================================

print("=" * 112)
print("GPU SOURCE VALIDATION INFERENCE")
print("=" * 112)


prediction_start = time.time()


X_val_gpu = cp.asarray(
    X_source_val,
    dtype=cp.float64,
)


cp.cuda.Device(0).synchronize()


pred_gpu = booster.inplace_predict(
    X_val_gpu
)


cp.cuda.Device(0).synchronize()


prediction_seconds = (
    time.time()
    - prediction_start
)


if isinstance(
    pred_gpu,
    cp.ndarray,
):

    p_xgb = cp.asnumpy(
        pred_gpu
    ).astype(
        np.float64,
        copy=False,
    )

else:

    p_xgb = np.asarray(
        pred_gpu,
        dtype=np.float64,
    )


if p_xgb.shape != (
    VAL_ROWS,
):
    raise RuntimeError(
        "XGBoost validation prediction shape mismatch."
    )


if not np.all(
    np.isfinite(
        p_xgb
    )
):
    raise RuntimeError(
        "XGBoost validation predictions contain non-finite values."
    )


np.save(
    XGB_PROB,
    p_xgb,
    allow_pickle=False,
)


xgb_prob_sha = sha256_file(
    XGB_PROB
)


print(
    "Rows:",
    len(
        p_xgb
    ),
)

print(
    "Prediction seconds:",
    round(
        prediction_seconds,
        3,
    ),
)

print(
    "Probability SHA256:",
    xgb_prob_sha,
)

print()
print(
    "[PASS] Validation inference executed from CUDA-resident predictors."
)
print()


# ============================================================================
# 21. STOP GPU MONITOR + VERIFY ACTUAL GPU USE
# ============================================================================

monitor_stop.set()

monitor_thread.join(
    timeout=5
)


gpu0_samples = [
    s
    for s
    in gpu_samples
    if s[
        "gpu"
    ] == 0
]


if gpu0_samples:

    max_gpu_util = max(
        s[
            "utilization_percent"
        ]
        for s
        in gpu0_samples
    )

    max_gpu_memory_mib = max(
        s[
            "memory_used_mib"
        ]
        for s
        in gpu0_samples
    )

    mean_gpu_util = float(
        np.mean(
            [
                s[
                    "utilization_percent"
                ]
                for s
                in gpu0_samples
            ]
        )
    )

else:

    max_gpu_util = None
    max_gpu_memory_mib = None
    mean_gpu_util = None


print("=" * 112)
print("GPU EXECUTION EVIDENCE")
print("=" * 112)

print(
    "Monitor samples:",
    len(
        gpu0_samples
    ),
)

print(
    "GPU0 max utilization:",
    max_gpu_util,
)

print(
    "GPU0 mean utilization:",
    mean_gpu_util,
)

print(
    "GPU0 max memory MiB:",
    max_gpu_memory_mib,
)

print(
    "Booster resolved device:",
    "cuda:0",
)


if (
    max_gpu_util is not None
    and max_gpu_util <= 0
):
    raise RuntimeError(
        "nvidia-smi never observed GPU activity."
    )


print()
print(
    "[PASS] CUDA backend independently confirmed."
)
print()


# ============================================================================
# 22. XGBOOST CHECKPOINT RECEIPT
# ============================================================================

xgb_checkpoint = {

    "stage":
        "Stage24-1B-R2",

    "type":
        "PRIMARY_BRIDGE62_XGBOOST_CUDA_COMPLETION_CHECKPOINT",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "execution_parent":
        AMENDMENT_COMMIT,

    "representation":
        "bridge62",

    "source_train": {

        "rows":
            TRAIN_ROWS,

        "benign":
            TRAIN_BENIGN,

        "attack":
            TRAIN_ATTACK,

        "dtype":
            "float64",
    },

    "gpu_execution": {

        "device_requested":
            "cuda",

        "device_resolved":
            "cuda:0",

        "tree_method":
            "hist",

        "input_batches":
            "CuPy float64",

        "matrix":
            "QuantileDMatrix",

        "batch_rows":
            GPU_BATCH_ROWS,

        "max_bin":
            MAX_BIN,

        "quantile_construction_seconds":
            qdm_seconds,

        "fit_seconds":
            fit_seconds,

        "validation_prediction":
            "inplace_predict(CuPy float64)",

        "validation_prediction_seconds":
            prediction_seconds,

        "nvidia_smi_samples":
            len(
                gpu0_samples
            ),

        "max_gpu_utilization_percent":
            max_gpu_util,

        "mean_gpu_utilization_percent":
            mean_gpu_util,

        "max_gpu_memory_used_mib":
            max_gpu_memory_mib,
    },

    "xgboost": {

        "configuration":
            "XGB_11",

        "wrapper_parameters":
            frozen_xgb_wrapper,

        "native_parameters":
            xgb_native_params,

        "num_boost_round":
            NUM_BOOST_ROUND,

        "feature_count":
            int(
                booster.num_features()
            ),

        "completed_rounds":
            int(
                booster.num_boosted_rounds()
            ),

        "model_file":
            XGB_MODEL.name,

        "model_sha256":
            xgb_model_sha,

        "validation_probability_file":
            XGB_PROB.name,

        "validation_probability_sha256":
            xgb_prob_sha,
    },

    "scientific_fit_accounting": {

        "completed_lightgbm_fits":
            1,

        "interrupted_xgboost_implementation_attempts":
            1,

        "completed_xgboost_fits":
            1,

        "completed_stage24_fits_after":
            2,

        "stage24_fit_budget":
            4,
    },

    "scientific_access": {

        "target_rows_read":
            0,

        "target_labels_read":
            0,

        "target_predictions":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,

        "target_opening_budget":
            8,
    },

    "next_authorized_step":
        (
            "Combine frozen bridge62 LightGBM and XGBoost source "
            "validation probabilities at 0.5/0.5 and apply frozen "
            "source-validation-only threshold semantics."
        ),
}


xgb_checkpoint_sha = write_json_sha(
    XGB_CHECKPOINT,
    xgb_checkpoint,
)


print(
    "XGB checkpoint SHA256:",
    xgb_checkpoint_sha,
)
print()


# ============================================================================
# 23. COMMIT XGBOOST CHECKPOINT BEFORE ENSEMBLE FINALIZATION
# ============================================================================

git(
    "add",
    "--",
    str(
        XGB_MODEL.relative_to(
            REPO
        )
    ),
    str(
        XGB_CHECKPOINT.relative_to(
            REPO
        )
    ),
    str(
        XGB_CHECKPOINT_SHA.relative_to(
            REPO
        )
    ),
)


git(
    "add",
    "-f",
    "--",
    str(
        XGB_PROB.relative_to(
            REPO
        )
    ),
)


staged = [
    x
    for x
    in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if x
]


expected_xgb_checkpoint_files = {
    str(
        XGB_MODEL.relative_to(
            REPO
        )
    ),
    str(
        XGB_PROB.relative_to(
            REPO
        )
    ),
    str(
        XGB_CHECKPOINT.relative_to(
            REPO
        )
    ),
    str(
        XGB_CHECKPOINT_SHA.relative_to(
            REPO
        )
    ),
}


if (
    set(staged)
    != expected_xgb_checkpoint_files
):
    raise RuntimeError(
        "\nUnexpected XGBoost checkpoint staged set:\n"
        + repr(staged)
    )


git(
    "commit",
    "-m",
    "stage24: complete bridge62 XGBoost CUDA checkpoint",
)


XGB_COMMIT = git(
    "rev-parse",
    "HEAD",
)


if (
    git(
        "rev-parse",
        "HEAD^",
    )
    != AMENDMENT_COMMIT
):
    raise RuntimeError(
        "XGBoost checkpoint parent mismatch."
    )


print(
    git(
        "push",
        "origin",
        "HEAD:main",
        auth_header=basic,
    )
)


remote_xgb = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=basic,
).split()[0]


if remote_xgb != XGB_COMMIT:
    raise RuntimeError(
        "XGBoost checkpoint remote verification failed."
    )


if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Worktree not clean after XGBoost checkpoint."
    )


print()
print(
    "[PASS] XGBoost CUDA model remotely checkpointed."
)

print(
    "Commit:",
    XGB_COMMIT,
)

print()


# ============================================================================
# 24. RELEASE LARGE GPU TRAINING OBJECTS
#
# Fit is now remotely frozen.
# ============================================================================

del (
    qdm_train,
    train_iter,
    X_val_gpu,
    pred_gpu,
)

gc.collect()

cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

cp.cuda.Device(0).synchronize()


# ============================================================================
# 25. FROZEN 0.5 / 0.5 ENSEMBLE
# ============================================================================

print("=" * 112)
print("FROZEN BRIDGE62 SOURCE ENSEMBLE")
print("=" * 112)


p_ensemble = (
    0.5
    * p_lgb
    + 0.5
    * p_xgb
).astype(
    np.float32
)


if p_ensemble.shape != (
    VAL_ROWS,
):
    raise RuntimeError(
        "Ensemble shape mismatch."
    )


if not np.all(
    np.isfinite(
        p_ensemble
    )
):
    raise RuntimeError(
        "Non-finite ensemble probabilities."
    )


clean_position = np.arange(
    13_818_623,
    14_412_403,
    dtype=np.int64,
)


np.savez_compressed(
    ENSEMBLE_PROB,
    clean_position=clean_position,
    binary_label=np.asarray(
        y_source_val,
        dtype=np.uint8,
    ),
    ensemble_probability=p_ensemble,
)


ensemble_file_sha = sha256_file(
    ENSEMBLE_PROB
)

ensemble_array_sha = array_sha256(
    p_ensemble
)


print(
    "Rows:",
    len(
        p_ensemble
    ),
)

print(
    "Array SHA256:",
    ensemble_array_sha,
)
print()


# ============================================================================
# 26. FROZEN PR-AUC / ROC-AUC
# ============================================================================

pr_auc, roc_auc = frozen_ranking_metrics(
    y_source_val,
    p_ensemble,
)


prevalence = float(
    VAL_ATTACK
    / VAL_ROWS
)


pr_excess = (
    pr_auc
    - prevalence
)


pr_normalized = (
    (
        pr_auc
        - prevalence
    )
    / (
        1.0
        - prevalence
    )
)


print("=" * 112)
print("SOURCE VALIDATION RANKING METRICS")
print("=" * 112)

print(
    "Prevalence:",
    format(
        prevalence,
        ".15f",
    ),
)

print(
    "PR-AUC:",
    format(
        pr_auc,
        ".15f",
    ),
)

print(
    "PR excess:",
    format(
        pr_excess,
        ".15f",
    ),
)

print(
    "PR normalized:",
    format(
        pr_normalized,
        ".15f",
    ),
)

print(
    "ROC-AUC:",
    format(
        roc_auc,
        ".15f",
    ),
)
print()


# ============================================================================
# 27. FROZEN STAGE20-1E3 THRESHOLD GRID
# ============================================================================

print("=" * 112)
print("SOURCE-VALIDATION-ONLY THRESHOLDS")
print("=" * 112)


grid = [
    threshold_record(
        t,
        y_source_val,
        p_ensemble,
    )
    for t
    in range(
        5,
        96,
    )
]


pd.DataFrame(
    grid
).to_csv(
    THRESHOLD_GRID,
    index=False,
)


standard = next(
    row
    for row
    in grid
    if row[
        "threshold_integer_percent"
    ] == 50
)


balanced = grid[0]


for candidate in grid[1:]:

    candidate_key = (
        f1_fraction(
            candidate
        ),
        -fpr_fraction(
            candidate
        ),
        recall_fraction(
            candidate
        ),
        -abs(
            candidate[
                "threshold_integer_percent"
            ]
            - 50
        ),
        -candidate[
            "threshold_integer_percent"
        ],
    )


    current_key = (
        f1_fraction(
            balanced
        ),
        -fpr_fraction(
            balanced
        ),
        recall_fraction(
            balanced
        ),
        -abs(
            balanced[
                "threshold_integer_percent"
            ]
            - 50
        ),
        -balanced[
            "threshold_integer_percent"
        ],
    )


    if (
        candidate_key
        > current_key
    ):
        balanced = candidate


security_candidates = [
    row
    for row
    in grid
    if (
        fpr_fraction(
            row
        )
        <= Fraction(
            1,
            20,
        )
    )
]


security = None


if security_candidates:

    security = security_candidates[
        0
    ]


    for candidate in security_candidates[
        1:
    ]:

        candidate_key = (
            f2_fraction(
                candidate
            ),
            -fpr_fraction(
                candidate
            ),
            recall_fraction(
                candidate
            ),
            -candidate[
                "threshold_integer_percent"
            ],
        )


        current_key = (
            f2_fraction(
                security
            ),
            -fpr_fraction(
                security
            ),
            recall_fraction(
                security
            ),
            -security[
                "threshold_integer_percent"
            ],
        )


        if (
            candidate_key
            > current_key
        ):
            security = candidate


print("STANDARD")
print(
    json.dumps(
        standard,
        indent=2,
    )
)

print()
print("BALANCED")
print(
    json.dumps(
        balanced,
        indent=2,
    )
)

print()
print("SECURITY")

if security is None:
    print(
        "UNAVAILABLE — NO RELAXATION"
    )

else:
    print(
        json.dumps(
            security,
            indent=2,
        )
    )

print()


# ============================================================================
# 28. FULL70 -> BRIDGE62 SOURCE COMPARISON
# ============================================================================

stage24_1a = load_json(
    STAGE24_1A
)


full70_pr = float(
    stage24_1a[
        "source_metrics"
    ][
        "pr_auc"
    ]
)


full70_roc = float(
    stage24_1a[
        "source_metrics"
    ][
        "roc_auc"
    ]
)


source_comparison = {

    "full70_pr_auc":
        full70_pr,

    "bridge62_pr_auc":
        pr_auc,

    "bridge62_minus_full70_pr_auc":
        pr_auc
        - full70_pr,

    "full70_roc_auc":
        full70_roc,

    "bridge62_roc_auc":
        roc_auc,

    "bridge62_minus_full70_roc_auc":
        roc_auc
        - full70_roc,

    "performance_based_stop":
        False,

    "scientific_use":
        "DESCRIPTIVE_ONLY",
}


# ============================================================================
# 29. FINAL STAGE24-1B RESULT
# ============================================================================

final_result = {

    "stage":
        "Stage24-1B",

    "status":
        "PRIMARY_BRIDGE62_SOURCE_REFIT_COMPLETE",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "execution_parent":
        XGB_COMMIT,

    "direction":
        "IDS2018_TO_CICIDS2017",

    "representation": {

        "name":
            "bridge62",

        "feature_count":
            62,

        "feature_order":
            bridge62,

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",
    },

    "source_train": {

        "rows":
            TRAIN_ROWS,

        "benign":
            TRAIN_BENIGN,

        "attack":
            TRAIN_ATTACK,

        "membership":
            "Stage22 CHRONOLOGICAL_NATURAL day_id 0..6",
    },

    "source_validation": {

        "rows":
            VAL_ROWS,

        "benign":
            VAL_BENIGN,

        "attack":
            VAL_ATTACK,

        "membership":
            "Stage22 CHRONOLOGICAL_NATURAL day_id 7",

        "prevalence":
            prevalence,
    },

    "lightgbm": {

        "configuration":
            "LGBM_11",

        "backend":
            "CPU_FROZEN_BACKEND_AMENDMENT",

        "completed_fits":
            1,

        "failed_gpu_runtime_attempts":
            1,

        "model_sha256":
            EXPECTED_LGB_MODEL_SHA,

        "validation_probability_sha256":
            EXPECTED_LGB_PROB_SHA,
    },

    "xgboost": {

        "configuration":
            "XGB_11",

        "backend":
            "CUDA",

        "resolved_device":
            "cuda:0",

        "tree_method":
            "hist",

        "training_matrix":
            "GPU_QUANTILEDMATRIX",

        "input_batches":
            "CUPY_FLOAT64",

        "wrapper_parameters":
            frozen_xgb_wrapper,

        "native_parameters":
            xgb_native_params,

        "num_boost_round":
            NUM_BOOST_ROUND,

        "completed_fits":
            1,

        "interrupted_prior_execution_attempts":
            1,

        "quantile_construction_seconds":
            qdm_seconds,

        "fit_seconds":
            fit_seconds,

        "validation_prediction_seconds":
            prediction_seconds,

        "model_file":
            XGB_MODEL.name,

        "model_sha256":
            xgb_model_sha,

        "validation_probability_file":
            XGB_PROB.name,

        "validation_probability_sha256":
            xgb_prob_sha,

        "gpu_evidence": {

            "monitor_samples":
                len(
                    gpu0_samples
                ),

            "max_utilization_percent":
                max_gpu_util,

            "mean_utilization_percent":
                mean_gpu_util,

            "max_memory_used_mib":
                max_gpu_memory_mib,
        },
    },

    "ensemble": {

        "rule":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "storage_dtype":
            "float32",

        "probability_file":
            ENSEMBLE_PROB.name,

        "probability_file_sha256":
            ensemble_file_sha,

        "probability_array_sha256":
            ensemble_array_sha,
    },

    "source_validation_metrics": {

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "prevalence_chance_anchor":
            prevalence,

        "pr_excess":
            pr_excess,

        "pr_normalized":
            pr_normalized,

        "standard":
            standard,

        "balanced":
            balanced,

        "security": {

            "status":
                (
                    "AVAILABLE"
                    if security
                    is not None
                    else "UNAVAILABLE_NO_RELAXATION"
                ),

            "result":
                security,
        },
    },

    "threshold_selection": {

        "source":
            "SOURCE_VALIDATION_ONLY",

        "grid":
            "INTEGER_PERCENT_5_TO_95_STEP_1",

        "target_threshold_selection":
            False,

        "threshold_grid_file":
            THRESHOLD_GRID.name,
    },

    "source_sanity_comparison":
        source_comparison,

    "scientific_fit_accounting": {

        "completed_lightgbm_fits":
            1,

        "completed_xgboost_fits":
            1,

        "interrupted_xgboost_implementation_attempts":
            1,

        "completed_stage24_fits_after":
            2,

        "stage24_fit_budget":
            4,
    },

    "scientific_access": {

        "target_rows_read":
            0,

        "target_labels_read":
            0,

        "target_predictions":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,

        "target_opening_budget":
            8,
    },

    "anti_adaptation": {

        "hyperparameter_tuning":
            False,

        "feature_search":
            False,

        "target_feedback":
            False,

        "target_threshold_selection":
            False,

        "performance_based_stop":
            False,
    },

    "next_authorized_step":
        (
            "Stage24-1C — exact GROUNDED_S4 membership "
            "recovery/replay before primary target inference."
        ),
}


final_result_sha = write_json_sha(
    FINAL_RESULT,
    final_result,
)


# ============================================================================
# 30. CHECKSUM MANIFEST
# ============================================================================

checksum_files = [
    LGB_AMENDMENT,
    LGB_CHECKPOINT,
    LGB_MODEL,
    LGB_PROB,
    XGB_EXEC_AMENDMENT,
    XGB_EXEC_AMENDMENT_SHA,
    XGB_MODEL,
    XGB_PROB,
    XGB_CHECKPOINT,
    XGB_CHECKPOINT_SHA,
    ENSEMBLE_PROB,
    THRESHOLD_GRID,
    FINAL_RESULT,
    FINAL_RESULT_SHA,
]


with CHECKSUMS.open(
    "w",
    encoding="utf-8",
) as f:

    for path in sorted(
        checksum_files,
        key=lambda x:
            x.name,
    ):

        f.write(
            f"{sha256_file(path)}  {path.name}\n"
        )


# ============================================================================
# 31. COMMIT FINAL ENSEMBLE / THRESHOLD RESULT
# ============================================================================

print("=" * 112)
print("COMMIT + PUSH FINAL STAGE24-1B")
print("=" * 112)


normal_files = [
    THRESHOLD_GRID,
    FINAL_RESULT,
    FINAL_RESULT_SHA,
    CHECKSUMS,
]


forced_files = [
    ENSEMBLE_PROB,
]


git(
    "add",
    "--",
    *[
        str(
            p.relative_to(
                REPO
            )
        )
        for p
        in normal_files
    ],
)


git(
    "add",
    "-f",
    "--",
    *[
        str(
            p.relative_to(
                REPO
            )
        )
        for p
        in forced_files
    ],
)


staged = [
    x
    for x
    in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if x
]


expected_final_stage = {
    str(
        p.relative_to(
            REPO
        )
    )
    for p
    in (
        normal_files
        + forced_files
    )
}


if set(staged) != expected_final_stage:
    raise RuntimeError(
        "\nUnexpected final staged set:\n"
        + repr(staged)
    )


git(
    "commit",
    "-m",
    "stage24: complete primary bridge62 source refit",
)


FINAL_COMMIT = git(
    "rev-parse",
    "HEAD",
)


if (
    git(
        "rev-parse",
        "HEAD^",
    )
    != XGB_COMMIT
):
    raise RuntimeError(
        "Final Stage24-1B parent mismatch."
    )


print(
    git(
        "push",
        "origin",
        "HEAD:main",
        auth_header=basic,
    )
)


remote_final = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=basic,
).split()[0]


if remote_final != FINAL_COMMIT:
    raise RuntimeError(
        "Final Stage24-1B remote verification failed."
    )


if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Worktree not clean after final Stage24-1B push."
    )


print()
print(
    "[PASS] Stage24-1B remotely frozen."
)
print(
    "Final commit:",
    FINAL_COMMIT,
)
print()


# ============================================================================
# 32. DELETE 6.38 GiB MEMMAP ONLY AFTER REMOTE FREEZE
# ============================================================================

print("=" * 112)
print("RELEASING TEMPORARY TRAINING CACHE")
print("=" * 112)


del X_mm
del y_mm

gc.collect()


deleted_bytes = 0


for path in [
    X_MEMMAP_PATH,
    Y_MEMMAP_PATH,
]:

    if path.exists():

        deleted_bytes += (
            path.stat().st_size
        )

        path.unlink()


try:
    TMP_ROOT.rmdir()

except OSError:
    pass


print(
    "Deleted GiB:",
    round(
        deleted_bytes
        / 1024**3,
        3,
    ),
)

print()
print(
    "[PASS] Training memmap removed after remote scientific freeze."
)
print()


# ============================================================================
# 33. FINAL
# ============================================================================

print("=" * 112)
print("STAGE24-1B-R2-GPU FINAL")
print("=" * 112)

print(
    "LightGBM:"
)
print(
    "  Completed fits:                1"
)
print(
    "  Frozen backend:                CPU"
)
print()

print(
    "XGBoost:"
)
print(
    "  Completed fits:                1"
)
print(
    "  Backend requested:             CUDA"
)
print(
    "  Backend resolved:              cuda:0"
)
print(
    "  Training matrix:               GPU QuantileDMatrix"
)
print(
    "  Input:                         CuPy float64 batches"
)
print(
    "  Tree method:                   hist"
)
print(
    "  Boosted rounds:                400"
)
print(
    "  Quantile construction seconds:",
    round(
        qdm_seconds,
        3,
    ),
)
print(
    "  Fit seconds:                   ",
    round(
        fit_seconds,
        3,
    ),
)
print(
    "  GPU max utilization:           ",
    max_gpu_util,
)
print(
    "  GPU max memory MiB:            ",
    max_gpu_memory_mib,
)
print()

print(
    "Source validation:"
)
print(
    "  PR-AUC:                        ",
    format(
        pr_auc,
        ".15f",
    ),
)
print(
    "  ROC-AUC:                       ",
    format(
        roc_auc,
        ".15f",
    ),
)
print()

print(
    "Thresholds:"
)
print(
    "  Standard:                      ",
    standard[
        "threshold_float32_runtime"
    ],
)
print(
    "  Balanced:                      ",
    balanced[
        "threshold_float32_runtime"
    ],
)
print(
    "  Security:                      ",
    (
        security[
            "threshold_float32_runtime"
        ]
        if security
        is not None
        else "UNAVAILABLE"
    ),
)
print()

print(
    "Completed Stage24 fits:          2 / 4"
)
print(
    "Target rows read:                0"
)
print(
    "Target labels read:              0"
)
print(
    "Target predictions:              0"
)
print(
    "Target openings:                 0 / 8"
)
print()

print(
    "Final commit:"
)
print(
    " ",
    FINAL_COMMIT,
)
print()

print("-" * 112)
print("STAGE24-1B: PASS")
print()
print(
    "XGB_11 trained and predicted through an explicitly CUDA-native path."
)
print(
    "Primary bridge62 source model pair and thresholds are frozen."
)
print(
    "NEXT: Stage24-1C — exact GROUNDED_S4 membership recovery/replay."
)
print("-" * 112)
print("=" * 112)

In [1]:
# ============================================================================
# STAGE24-1C0 — CPU-ONLY FRESH BOOTSTRAP + GROUNDED_S4 ARTIFACT INVENTORY
#
# PURPOSE
# -------
# Fresh CPU session after Stage24-1B.
#
# This cell:
#   1. REFUSES to run if a Kaggle GPU is visible
#   2. clones / synchronizes the exact GitHub scientific state
#   3. verifies Stage24-1B is frozen at the expected commit
#   4. verifies:
#          completed scientific fits = 2 / 4
#          target openings = 0 / 8
#   5. recovers the five frozen Stage20 GROUNDED_S4 manifests
#   6. scans ALL mounted Kaggle inputs for:
#          - reusable Stage20 compact corpora
#          - row-membership / join artifacts
#          - exact PCAPs
#          - exact traffic-label sources
#   7. searches the repository for any persisted implementation of the
#      Stage20 exact join / flow reconstruction
#
# NO MODEL FIT
# NO TARGET MODEL INFERENCE
# NO TARGET OPENING
# NO PCAP REPLAY YET
#
# Expected scientific state afterward:
#       fits     = 2 / 4
#       openings = 0 / 8
# ============================================================================

from __future__ import annotations

import os
import re
import json
import hashlib
import subprocess
from pathlib import Path
from collections import defaultdict


# ============================================================================
# 0. CONSTANTS
# ============================================================================

EXPECTED_HEAD = (
    "683fe85839e63dcd5fd8e952fd07931b446652ff"
)

REPO_URL = (
    "https://github.com/themubasshir/"
    "ids2018-validation-safe-ablation.git"
)

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

INPUT_ROOT = Path(
    "/kaggle/input"
)


MANIFESTS = {
    "Monday":
        "results/stage20_1e_training/"
        "stage20_1e1m_monday_compact_corpus_manifest.json",

    "Tuesday":
        "results/stage20_1e_training/"
        "stage20_1e1t_tuesday_compact_corpus_manifest.json",

    "Wednesday":
        "results/stage20_1e_training/"
        "stage20_1e1w_wednesday_compact_corpus_manifest.json",

    "Thursday":
        "results/stage20_1e_training/"
        "stage20_1e1v_thursday_validation_compact_corpus_manifest.json",

    "Friday":
        "results/stage20_1e_training/"
        "stage20_1e4_friday_holdout_compact_corpus_manifest.json",
}


EXPECTED_GROUNDED = {
    "Monday": 528_509,
    "Tuesday": 4_170,
    "Wednesday": 12_951,
    "Thursday": 8_197,
    "Friday": 12_088,
}

EXPECTED_GROUNDED_TOTAL = 565_915


STAGE24_PROTOCOL = (
    "results/stage24_cross_dataset/"
    "stage24_0_protocol_lock/"
    "stage24_0c_final_preopening_protocol_lock.json"
)


STAGE24_1B_RESULT = (
    "results/stage24_cross_dataset/"
    "stage24_1_primary_source_sanity/"
    "stage24_1b_bridge62_source_refit/"
    "stage24_1b_bridge62_source_refit_result.json"
)


print("=" * 110)
print("STAGE24-1C0 — CPU-ONLY FRESH BOOTSTRAP + GROUNDED_S4 INVENTORY")
print("=" * 110)
print()


# ============================================================================
# 1. FAIL CLOSED IF GPU IS VISIBLE
# ============================================================================

print("=" * 110)
print("ACCELERATOR GUARD")
print("=" * 110)


gpu_probe = subprocess.run(
    ["nvidia-smi", "-L"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)


if (
    gpu_probe.returncode == 0
    and gpu_probe.stdout.strip()
):
    print(gpu_probe.stdout)

    raise RuntimeError(
        "\nA CUDA GPU is visible in this session.\n"
        "STOP THIS SESSION and switch Kaggle Accelerator to NONE.\n\n"
        "Stage24-1C is CPU / PCAP / provenance work and must not consume "
        "the remaining GPU quota."
    )


print("[PASS] No NVIDIA GPU visible.")
print("[PASS] This is suitable for Stage24-1C CPU work.")
print()


# ============================================================================
# 2. HELPERS
# ============================================================================

def run(
    cmd,
    cwd=None,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in cmd
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (
        p.stdout or ""
    ).strip()


def git(
    *args,
    check=True,
):
    return run(
        ["git", *args],
        cwd=REPO,
        check=check,
    )


def load_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def human_bytes(n):
    n = int(n)

    units = [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]

    value = float(n)

    for unit in units:
        if (
            value < 1024.0
            or unit == units[-1]
        ):
            return (
                f"{value:.3f} {unit}"
            )

        value /= 1024.0


def recurse_source_entries(obj):
    """
    Recover Stage20 source-file descriptors generically.
    Handles nested dict/list source structures.
    """

    found = []

    def walk(x):
        if isinstance(x, dict):

            if (
                "remote" in x
                and isinstance(
                    x["remote"],
                    str,
                )
            ):
                found.append(
                    {
                        key:
                            x.get(key)
                        for key
                        in [
                            "remote",
                            "revision",
                            "sha256",
                            "size_bytes",
                            "rows",
                        ]
                        if key in x
                    }
                )

            for value in x.values():
                walk(value)

        elif isinstance(x, list):
            for value in x:
                walk(value)

    walk(obj)

    return found


# ============================================================================
# 3. FRESH REPOSITORY BOOTSTRAP
# ============================================================================

print("=" * 110)
print("GITHUB BOOTSTRAP")
print("=" * 110)


if REPO.exists():

    if not (
        REPO
        / ".git"
    ).is_dir():
        raise RuntimeError(
            f"{REPO} exists but is not a Git repository."
        )

    status_before = git(
        "status",
        "--porcelain",
    )

    if status_before:
        raise RuntimeError(
            "\nExisting repository is dirty:\n"
            + status_before
        )

    print(
        "Existing clean clone found."
    )

    print(
        git(
            "fetch",
            "origin",
            "main",
        )
    )

    git(
        "checkout",
        "main",
    )

    git(
        "reset",
        "--hard",
        "origin/main",
    )

else:

    print(
        run(
            [
                "git",
                "clone",
                "--branch",
                "main",
                "--single-branch",
                REPO_URL,
                str(REPO),
            ]
        )
    )


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
).split()[0]

status = git(
    "status",
    "--porcelain",
)


print()
print(
    "Local HEAD:",
    head,
)

print(
    "Remote main:",
    remote,
)


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "\nUnexpected repository HEAD.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if remote != EXPECTED_HEAD:
    raise RuntimeError(
        "\nUnexpected remote HEAD.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {remote}"
    )


if status:
    raise RuntimeError(
        "\nFresh repository is not clean:\n"
        + status
    )


print()
print(
    "[PASS] Exact Stage24-1B commit recovered."
)
print()


# ============================================================================
# 4. VERIFY STAGE24 SCIENTIFIC STATE
# ============================================================================

print("=" * 110)
print("STAGE24 SCIENTIFIC STATE")
print("=" * 110)


protocol = load_json(
    REPO
    / STAGE24_PROTOCOL
)

stage24_1b = load_json(
    REPO
    / STAGE24_1B_RESULT
)


if (
    stage24_1b.get(
        "status"
    )
    != "PRIMARY_BRIDGE62_SOURCE_REFIT_COMPLETE"
):
    raise RuntimeError(
        "Stage24-1B final result is not COMPLETE."
    )


fit_state = (
    stage24_1b[
        "scientific_fit_accounting"
    ]
)


access_state = (
    stage24_1b[
        "scientific_access"
    ]
)


completed_fits = int(
    fit_state[
        "completed_stage24_fits_after"
    ]
)


openings = int(
    access_state[
        "target_openings_consumed"
    ]
)


if completed_fits != 2:
    raise RuntimeError(
        f"Expected 2 completed fits, got {completed_fits}."
    )


if openings != 0:
    raise RuntimeError(
        f"Expected zero target openings, got {openings}."
    )


if (
    protocol[
        "execution_order"
    ][3][
        "action"
    ]
    !=
    "RECONSTRUCT_OR_RECOVER_GROUNDED_S4_MEMBERSHIP_USING_ONLY_FROZEN_STAGE20_EXACT_JOIN"
):
    raise RuntimeError(
        "Frozen Stage24 execution-order step 4 changed."
    )


print(
    "Completed scientific fits:",
    f"{completed_fits} / 4",
)

print(
    "Target openings:",
    f"{openings} / 8",
)

print(
    "Next frozen action:"
)

print(
    " ",
    protocol[
        "execution_order"
    ][3][
        "action"
    ],
)

print()
print(
    "[PASS] Stage24 is exactly at the Stage24-1C boundary."
)
print()


# ============================================================================
# 5. LOAD ALL FIVE FROZEN STAGE20 MANIFESTS
# ============================================================================

print("=" * 110)
print("FROZEN GROUNDED_S4 MANIFESTS")
print("=" * 110)


manifest_objects = {}
source_descriptors = []


for day, relative in MANIFESTS.items():

    path = REPO / relative

    if not path.is_file():
        raise FileNotFoundError(
            path
        )

    manifest = load_json(
        path
    )

    manifest_objects[
        day
    ] = manifest


    exact = manifest[
        "exact_join"
    ]


    matching = exact.get(
        "matching"
    )


    if matching != "EXACT_ONLY":
        raise RuntimeError(
            f"{day}: matching is not EXACT_ONLY."
        )


    actual_count = None

    for key in [
        "supervised_matched_flows",
        "validation_matched_flows",
        "holdout_matched_flows",
    ]:
        if key in exact:
            actual_count = int(
                exact[key]
            )

            break


    if actual_count is None:
        raise RuntimeError(
            f"{day}: matched-flow count field not found."
        )


    expected_count = (
        EXPECTED_GROUNDED[
            day
        ]
    )


    if actual_count != expected_count:
        raise RuntimeError(
            f"{day}: grounded count mismatch. "
            f"expected={expected_count}, actual={actual_count}"
        )


    print(
        f"{day:10s}",
        f"{actual_count:>9,d}",
        "EXACT_ONLY",
    )


    for descriptor in recurse_source_entries(
        manifest.get(
            "source",
            {}
        )
    ):
        descriptor = dict(
            descriptor
        )

        descriptor[
            "day"
        ] = day

        source_descriptors.append(
            descriptor
        )


grounded_total = sum(
    EXPECTED_GROUNDED.values()
)


if grounded_total != EXPECTED_GROUNDED_TOTAL:
    raise RuntimeError(
        "Frozen grounded total mismatch."
    )


print("-" * 50)

print(
    f"{'TOTAL':10s}",
    f"{grounded_total:>9,d}",
)

print()
print(
    "[PASS] Exact frozen GROUNDED_S4 total = 565,915."
)
print()


# ============================================================================
# 6. INDEX ALL MOUNTED KAGGLE INPUT FILES ONCE
# ============================================================================

print("=" * 110)
print("INDEXING MOUNTED KAGGLE INPUTS")
print("=" * 110)


if not INPUT_ROOT.is_dir():
    raise RuntimeError(
        "/kaggle/input is unavailable."
    )


all_files = []

basename_index = defaultdict(
    list
)


for root, dirs, files in os.walk(
    INPUT_ROOT
):

    for filename in files:

        path = Path(root) / filename

        all_files.append(
            path
        )

        basename_index[
            filename
        ].append(
            path
        )


print(
    "Mounted files indexed:",
    f"{len(all_files):,}",
)

print()


# ============================================================================
# 7. SEARCH FOR REUSABLE STAGE20 ARTIFACTS
# ============================================================================

print("=" * 110)
print("REUSABLE STAGE20 ARTIFACT SEARCH")
print("=" * 110)


artifact_tokens = [
    "stage20",
    "compact",
    "ground",
    "grounded",
    "membership",
    "matched",
    "exact_join",
    "exact-join",
    "join",
    "flow_offsets",
    "packet_lengths",
    "encoded_bytes",
]


artifact_candidates = []


for path in all_files:

    low = str(
        path
    ).lower()

    if any(
        token in low
        for token
        in artifact_tokens
    ):
        artifact_candidates.append(
            path
        )


artifact_candidates = sorted(
    set(
        artifact_candidates
    )
)


if artifact_candidates:

    for path in artifact_candidates:

        try:
            size = (
                path.stat().st_size
            )

        except Exception:
            size = -1

        print(
            human_bytes(size)
            if size >= 0
            else "?",
            " ",
            path,
        )

else:

    print(
        "<none found>"
    )


print()


# ============================================================================
# 8. SEARCH FOR EXACT FROZEN SOURCE FILES BY MANIFEST BASENAME
# ============================================================================

print("=" * 110)
print("FROZEN SOURCE FILE RESOLUTION")
print("=" * 110)


source_resolution = []


# De-duplicate descriptors by remote/hash/day.
dedup = {}


for item in source_descriptors:

    key = (
        item.get("day"),
        item.get("remote"),
        item.get("sha256"),
    )

    dedup[
        key
    ] = item


for item in dedup.values():

    remote_name = item.get(
        "remote"
    )

    if not remote_name:
        continue

    basename = Path(
        remote_name
    ).name


    candidates = list(
        basename_index.get(
            basename,
            []
        )
    )


    # Some Kaggle datasets rename / flatten slightly.
    if not candidates:

        basename_lower = basename.lower()

        for path in all_files:

            if (
                path.name.lower()
                == basename_lower
            ):
                candidates.append(
                    path
                )


    record = {
        **item,

        "basename":
            basename,

        "mounted_candidates":
            [
                str(x)
                for x
                in candidates
            ],
    }


    source_resolution.append(
        record
    )


    print(
        f"[{item.get('day', '?')}]"
    )

    print(
        "  frozen remote:",
        remote_name,
    )

    if item.get(
        "sha256"
    ):
        print(
            "  expected SHA:",
            item[
                "sha256"
            ],
        )


    if candidates:

        for candidate in candidates:

            print(
                "  mounted:",
                candidate,
            )

            print(
                "  size:",
                human_bytes(
                    candidate.stat().st_size
                ),
            )

    else:

        print(
            "  mounted: <NOT FOUND BY EXACT BASENAME>"
        )

    print()


# ============================================================================
# 9. GENERAL PCAP + TRAFFIC-LABEL INVENTORY
# ============================================================================

print("=" * 110)
print("GENERAL PCAP / LABEL INVENTORY")
print("=" * 110)


pcaps = sorted(
    [
        path
        for path in all_files
        if path.suffix.lower()
        in {
            ".pcap",
            ".pcapng",
        }
    ]
)


parquets = sorted(
    [
        path
        for path in all_files
        if path.suffix.lower()
        == ".parquet"
    ]
)


print(
    "PCAP/PCAPNG files:",
    len(pcaps),
)


for path in pcaps:

    print(
        " ",
        human_bytes(
            path.stat().st_size
        ),
        path,
    )


print()

print(
    "Parquet files:",
    len(parquets),
)


# Do not dump thousands of unrelated files.
for path in parquets:

    low = str(
        path
    ).lower()

    if any(
        token in low
        for token
        in [
            "cic",
            "ids",
            "traffic",
            "label",
            "monday",
            "tuesday",
            "wednesday",
            "thursday",
            "friday",
        ]
    ):

        print(
            " ",
            human_bytes(
                path.stat().st_size
            ),
            path,
        )


print()


# ============================================================================
# 10. SEARCH LOCAL REPOSITORY FOR EXACT-JOIN IMPLEMENTATION EVIDENCE
# ============================================================================

print("=" * 110)
print("REPOSITORY EXACT-JOIN IMPLEMENTATION SEARCH")
print("=" * 110)


patterns = [
    "SCAPY_2_6_1_OUTER_IPV4_PAYLOAD_LAYER",
    "source_faithful",
    "raw_unmatched_reconstructed_flows",
    "unused_published_occurrences",
    "supervised_matched_flows",
    "validation_matched_flows",
    "holdout_matched_flows",
    "published_occurrence",
    "EXACT_ONLY",
]


implementation_hits = {}


for pattern in patterns:

    p = subprocess.run(
        [
            "grep",
            "-RIl",
            "--exclude-dir=.git",
            "--",
            pattern,
            str(REPO),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
        text=True,
    )


    hits = [
        x
        for x
        in p.stdout.splitlines()
        if x.strip()
    ]


    implementation_hits[
        pattern
    ] = hits


    print(
        pattern
    )


    if hits:

        for hit in hits[:20]:

            try:
                rel = Path(
                    hit
                ).relative_to(
                    REPO
                )

            except Exception:
                rel = Path(
                    hit
                )

            print(
                "  ",
                rel,
            )

    else:

        print(
            "   <no repository hit>"
        )


print()


# ============================================================================
# 11. CHECK PARTICULARLY FOR PERSISTED ROW-LOCATOR FILES
# ============================================================================

print("=" * 110)
print("PERSISTED ROW-MEMBERSHIP CANDIDATES")
print("=" * 110)


row_locator_tokens = [
    "row_index",
    "row_indices",
    "row_membership",
    "membership",
    "matched_rows",
    "matched_indices",
    "published_rows",
    "published_indices",
    "grounded_rows",
    "grounded_indices",
]


row_membership_candidates = []


for path in all_files:

    low = path.name.lower()

    if any(
        token in low
        for token
        in row_locator_tokens
    ):
        row_membership_candidates.append(
            path
        )


row_membership_candidates = sorted(
    set(
        row_membership_candidates
    )
)


if row_membership_candidates:

    for path in row_membership_candidates:

        print(
            human_bytes(
                path.stat().st_size
            ),
            " ",
            path,
        )

else:

    print(
        "<none found>"
    )


print()


# ============================================================================
# 12. WRITE OPERATIONAL INVENTORY
#
# This is not yet a scientific result and is NOT committed.
# ============================================================================

inventory = {

    "stage":
        "Stage24-1C0",

    "type":
        "CPU_ONLY_FRESH_BOOTSTRAP_AND_ARTIFACT_INVENTORY",

    "git_head":
        head,

    "scientific_state": {
        "completed_fits":
            completed_fits,

        "fit_budget":
            4,

        "target_openings":
            openings,

        "target_opening_budget":
            8,
    },

    "grounded_s4_expected": {
        "by_day":
            EXPECTED_GROUNDED,

        "total":
            EXPECTED_GROUNDED_TOTAL,
    },

    "mounted_file_count":
        len(
            all_files
        ),

    "reusable_stage20_candidates":
        [
            str(x)
            for x
            in artifact_candidates
        ],

    "row_membership_candidates":
        [
            str(x)
            for x
            in row_membership_candidates
        ],

    "source_resolution":
        source_resolution,

    "repository_search_hits":
        implementation_hits,

    "pcaps":
        [
            {
                "path":
                    str(x),

                "bytes":
                    x.stat().st_size,
            }
            for x
            in pcaps
        ],
}


inventory_path = Path(
    "/kaggle/working/"
    "stage24_1c0_cpu_inventory.json"
)


inventory_path.write_text(
    json.dumps(
        inventory,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


print("=" * 110)
print("STAGE24-1C0 FINAL")
print("=" * 110)

print(
    "Repository HEAD:",
    head,
)

print(
    "Completed fits:",
    f"{completed_fits} / 4",
)

print(
    "Target openings:",
    f"{openings} / 8",
)

print(
    "Expected grounded flows:",
    f"{EXPECTED_GROUNDED_TOTAL:,}",
)

print(
    "Reusable Stage20 candidates:",
    len(
        artifact_candidates
    ),
)

print(
    "Explicit row-membership candidates:",
    len(
        row_membership_candidates
    ),
)

print(
    "Mounted PCAP files:",
    len(
        pcaps
    ),
)

print()

print(
    "Operational inventory:",
    inventory_path,
)

print()

print("-" * 110)
print("STAGE24-1C0: PASS")
print()
print(
    "NO PCAP replay has occurred yet."
)
print(
    "NO target model inference has occurred."
)
print(
    "NO target opening has been consumed."
)
print()
print(
    "Paste the complete output before we start Stage24-1C1."
)
print("-" * 110)
print("=" * 110)

STAGE24-1C0 — CPU-ONLY FRESH BOOTSTRAP + GROUNDED_S4 INVENTORY

ACCELERATOR GUARD


FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

In [2]:
# ============================================================================
# STAGE24-1C0-R1 — CPU-ONLY FRESH BOOTSTRAP + GROUNDED_S4 INVENTORY
#
# FIX:
#   Robust accelerator guard for Kaggle Accelerator=None.
#   Missing nvidia-smi is a PASS, not an exception.
#
# PURPOSE
# -------
#   1. verify CPU-only execution
#   2. recover exact Stage24 repo state
#   3. verify Stage24-1B = COMPLETE
#   4. verify fits = 2/4 and target openings = 0/8
#   5. verify all five frozen GROUNDED_S4 counts
#   6. inventory mounted Stage20 / PCAP / traffic-label artifacts
#   7. search for reusable exact row-membership artifacts
#   8. search repo for Stage20 exact-join implementation evidence
#
# NO:
#   model fit
#   target inference
#   PCAP replay
#   target opening
# ============================================================================

from __future__ import annotations

import os
import json
import shutil
import subprocess
from pathlib import Path
from collections import defaultdict


# ============================================================================
# CONSTANTS
# ============================================================================

EXPECTED_HEAD = (
    "683fe85839e63dcd5fd8e952fd07931b446652ff"
)

REPO_URL = (
    "https://github.com/themubasshir/"
    "ids2018-validation-safe-ablation.git"
)

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

INPUT_ROOT = Path(
    "/kaggle/input"
)


STAGE24_PROTOCOL = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)


STAGE24_1B_RESULT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
    / "stage24_1b_bridge62_source_refit_result.json"
)


MANIFESTS = {

    "Monday":
        REPO
        / "results"
        / "stage20_1e_training"
        / "stage20_1e1m_monday_compact_corpus_manifest.json",

    "Tuesday":
        REPO
        / "results"
        / "stage20_1e_training"
        / "stage20_1e1t_tuesday_compact_corpus_manifest.json",

    "Wednesday":
        REPO
        / "results"
        / "stage20_1e_training"
        / "stage20_1e1w_wednesday_compact_corpus_manifest.json",

    "Thursday":
        REPO
        / "results"
        / "stage20_1e_training"
        / "stage20_1e1v_thursday_validation_compact_corpus_manifest.json",

    "Friday":
        REPO
        / "results"
        / "stage20_1e_training"
        / "stage20_1e4_friday_holdout_compact_corpus_manifest.json",
}


EXPECTED_GROUNDED = {
    "Monday": 528_509,
    "Tuesday": 4_170,
    "Wednesday": 12_951,
    "Thursday": 8_197,
    "Friday": 12_088,
}


EXPECTED_GROUNDED_TOTAL = 565_915


print("=" * 112)
print("STAGE24-1C0-R1 — CPU-ONLY FRESH BOOTSTRAP + GROUNDED_S4 INVENTORY")
print("=" * 112)
print()


# ============================================================================
# HELPERS
# ============================================================================

def run(
    command,
    cwd=None,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in command],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if (
        check
        and p.returncode != 0
    ):
        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x
                in command
            )
            + "\n\n"
            + (
                p.stdout
                or ""
            )
        )

    return (
        p.stdout
        or ""
    ).strip()


def git(
    *args,
    check=True,
):
    return run(
        [
            "git",
            *args,
        ],
        cwd=REPO,
        check=check,
    )


def load_json(
    path,
):
    return json.loads(
        Path(
            path
        ).read_text(
            encoding="utf-8"
        )
    )


def human_bytes(
    value,
):
    value = float(
        value
    )

    for unit in [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]:

        if (
            value < 1024.0
            or unit == "TiB"
        ):
            return (
                f"{value:.3f} {unit}"
            )

        value /= 1024.0


def recover_source_descriptors(
    obj,
):
    results = []

    def walk(
        x,
    ):
        if isinstance(
            x,
            dict,
        ):

            if (
                isinstance(
                    x.get(
                        "remote"
                    ),
                    str,
                )
            ):

                record = {}

                for key in [
                    "remote",
                    "revision",
                    "sha256",
                    "size_bytes",
                    "rows",
                ]:

                    if key in x:
                        record[
                            key
                        ] = x[
                            key
                        ]

                results.append(
                    record
                )

            for value in (
                x.values()
            ):
                walk(
                    value
                )

        elif isinstance(
            x,
            list,
        ):

            for value in x:
                walk(
                    value
                )

    walk(
        obj
    )

    return results


# ============================================================================
# 1. ROBUST CPU-ONLY ACCELERATOR GUARD
# ============================================================================

print("=" * 112)
print("ACCELERATOR GUARD")
print("=" * 112)


nvidia_smi = shutil.which(
    "nvidia-smi"
)


nvidia_devices = sorted(
    Path(
        "/dev"
    ).glob(
        "nvidia*"
    )
)


print(
    "nvidia-smi:",
    (
        nvidia_smi
        if nvidia_smi
        else "<NOT INSTALLED>"
    ),
)

print(
    "/dev/nvidia*:",
    (
        [
            str(x)
            for x
            in nvidia_devices
        ]
        if nvidia_devices
        else "<NONE>"
    ),
)


gpu_visible = False


if nvidia_smi:

    probe = subprocess.run(
        [
            nvidia_smi,
            "-L",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if (
        probe.returncode == 0
        and probe.stdout.strip()
    ):

        gpu_visible = True

        print()
        print(
            probe.stdout
        )


if nvidia_devices:

    # /dev/nvidiactl by itself can exist in unusual environments,
    # so require an actual numbered GPU device.
    actual_gpu_devices = [
        p
        for p
        in nvidia_devices
        if p.name[
            len(
                "nvidia"
            ):
        ].isdigit()
    ]

    if actual_gpu_devices:
        gpu_visible = True


if gpu_visible:

    raise RuntimeError(
        "\nA physical NVIDIA GPU is visible.\n"
        "This Stage24-1C inventory/replay session must use "
        "Kaggle Accelerator=None to preserve GPU quota."
    )


print()
print(
    "[PASS] No physical NVIDIA GPU is visible."
)

print(
    "[PASS] CPU-only Stage24-1C execution authorized."
)

print()


# ============================================================================
# 2. FRESH GITHUB BOOTSTRAP
# ============================================================================

print("=" * 112)
print("GITHUB BOOTSTRAP")
print("=" * 112)


if REPO.exists():

    if not (
        REPO
        / ".git"
    ).is_dir():

        raise RuntimeError(
            f"{REPO} exists but is not a Git repository."
        )


    status = git(
        "status",
        "--porcelain",
    )


    if status:

        raise RuntimeError(
            "\nExisting repository is dirty:\n"
            + status
        )


    print(
        "Existing clean clone found."
    )


    print(
        git(
            "fetch",
            "origin",
            "main",
        )
    )


    git(
        "checkout",
        "main",
    )


    git(
        "reset",
        "--hard",
        "origin/main",
    )


else:

    print(
        run(
            [
                "git",
                "clone",
                "--branch",
                "main",
                "--single-branch",
                REPO_URL,
                str(
                    REPO
                ),
            ]
        )
    )


head = git(
    "rev-parse",
    "HEAD",
)


remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
).split()[0]


status = git(
    "status",
    "--porcelain",
)


print()
print(
    "Local HEAD:",
    head,
)

print(
    "Remote main:",
    remote,
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "\nUnexpected local HEAD.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "\nUnexpected remote HEAD.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {remote}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


print()
print(
    "[PASS] Exact Stage24-1B final commit recovered."
)

print()


# ============================================================================
# 3. VERIFY STAGE24 SCIENTIFIC STATE
# ============================================================================

print("=" * 112)
print("STAGE24 SCIENTIFIC STATE")
print("=" * 112)


protocol = load_json(
    STAGE24_PROTOCOL
)


stage24_1b = load_json(
    STAGE24_1B_RESULT
)


if (
    stage24_1b[
        "status"
    ]
    !=
    "PRIMARY_BRIDGE62_SOURCE_REFIT_COMPLETE"
):

    raise RuntimeError(
        "Stage24-1B final result is not COMPLETE."
    )


fit_state = (
    stage24_1b[
        "scientific_fit_accounting"
    ]
)


access_state = (
    stage24_1b[
        "scientific_access"
    ]
)


completed_fits = int(
    fit_state[
        "completed_stage24_fits_after"
    ]
)


target_openings = int(
    access_state[
        "target_openings_consumed"
    ]
)


if completed_fits != 2:

    raise RuntimeError(
        "\nUnexpected completed fit count.\n"
        f"Expected: 2\n"
        f"Actual:   {completed_fits}"
    )


if target_openings != 0:

    raise RuntimeError(
        "\nTarget opening ledger changed.\n"
        f"Expected: 0\n"
        f"Actual:   {target_openings}"
    )


execution_order = (
    protocol[
        "execution_order"
    ]
)


grounding_step = next(
    (
        item
        for item
        in execution_order
        if item[
            "action"
        ]
        ==
        "RECONSTRUCT_OR_RECOVER_GROUNDED_S4_MEMBERSHIP_USING_ONLY_FROZEN_STAGE20_EXACT_JOIN"
    ),
    None,
)


if grounding_step is None:

    raise RuntimeError(
        "Frozen GROUNDED_S4 execution step not found."
    )


print(
    "Completed scientific fits:",
    f"{completed_fits} / 4",
)

print(
    "Target openings:",
    f"{target_openings} / 8",
)

print(
    "Frozen execution step:",
    grounding_step[
        "step"
    ],
)

print(
    "Action:"
)

print(
    " ",
    grounding_step[
        "action"
    ],
)

print()

print(
    "[PASS] Stage24 is exactly at the 1C boundary."
)

print()


# ============================================================================
# 4. VERIFY FIVE FROZEN GROUNDED_S4 MANIFESTS
# ============================================================================

print("=" * 112)
print("FROZEN GROUNDED_S4 MANIFESTS")
print("=" * 112)


manifest_objects = {}

source_descriptors = []


for day, path in (
    MANIFESTS.items()
):

    if not path.is_file():

        raise FileNotFoundError(
            path
        )


    manifest = load_json(
        path
    )


    manifest_objects[
        day
    ] = manifest


    exact = manifest[
        "exact_join"
    ]


    if (
        exact.get(
            "matching"
        )
        != "EXACT_ONLY"
    ):

        raise RuntimeError(
            f"{day}: matching != EXACT_ONLY."
        )


    matched = None


    for key in [
        "supervised_matched_flows",
        "validation_matched_flows",
        "holdout_matched_flows",
    ]:

        if key in exact:

            matched = int(
                exact[
                    key
                ]
            )

            break


    if matched is None:

        raise RuntimeError(
            f"{day}: matched-flow count key unavailable."
        )


    expected = (
        EXPECTED_GROUNDED[
            day
        ]
    )


    if matched != expected:

        raise RuntimeError(
            f"{day}: expected {expected}, actual {matched}."
        )


    print(
        f"{day:10s}",
        f"{matched:>9,d}",
        "EXACT_ONLY",
    )


    descriptors = recover_source_descriptors(
        manifest.get(
            "source",
            {},
        )
    )


    for descriptor in descriptors:

        descriptor = dict(
            descriptor
        )

        descriptor[
            "day"
        ] = day

        source_descriptors.append(
            descriptor
        )


grounded_total = sum(
    EXPECTED_GROUNDED.values()
)


if (
    grounded_total
    != EXPECTED_GROUNDED_TOTAL
):

    raise RuntimeError(
        "Expected grounded total contract changed."
    )


print(
    "-" * 48
)

print(
    f"{'TOTAL':10s}",
    f"{grounded_total:>9,d}",
)

print()

print(
    "[PASS] Frozen GROUNDED_S4 total = 565,915."
)

print()


# ============================================================================
# 5. INDEX ALL MOUNTED KAGGLE FILES
# ============================================================================

print("=" * 112)
print("KAGGLE INPUT INVENTORY")
print("=" * 112)


if not INPUT_ROOT.is_dir():

    raise RuntimeError(
        "/kaggle/input does not exist."
    )


all_files = []

basename_index = defaultdict(
    list
)


for root, _dirs, files in os.walk(
    INPUT_ROOT
):

    for filename in files:

        path = (
            Path(
                root
            )
            / filename
        )


        all_files.append(
            path
        )


        basename_index[
            filename
        ].append(
            path
        )


print(
    "Mounted files indexed:",
    f"{len(all_files):,}",
)

print()


# ============================================================================
# 6. SEARCH FOR REUSABLE STAGE20 / MEMBERSHIP ARTIFACTS
# ============================================================================

print("=" * 112)
print("REUSABLE STAGE20 ARTIFACT SEARCH")
print("=" * 112)


artifact_tokens = [
    "stage20",
    "compact",
    "grounded",
    "ground",
    "membership",
    "matched",
    "exact_join",
    "exact-join",
    "flow_offsets",
    "packet_lengths",
    "encoded_bytes",
]


artifact_candidates = sorted(
    {
        path
        for path
        in all_files
        if any(
            token
            in str(
                path
            ).lower()
            for token
            in artifact_tokens
        )
    }
)


if artifact_candidates:

    for path in artifact_candidates:

        print(
            f"{human_bytes(path.stat().st_size):>12s}",
            path,
        )


else:

    print(
        "<NONE>"
    )


print()


# ============================================================================
# 7. EXPLICIT ROW-MEMBERSHIP CANDIDATES
# ============================================================================

print("=" * 112)
print("EXPLICIT ROW-MEMBERSHIP CANDIDATES")
print("=" * 112)


row_tokens = [
    "row_index",
    "row_indices",
    "row_membership",
    "matched_rows",
    "matched_indices",
    "published_rows",
    "published_indices",
    "grounded_rows",
    "grounded_indices",
]


row_membership_candidates = sorted(
    {
        path
        for path
        in all_files
        if any(
            token
            in path.name.lower()
            for token
            in row_tokens
        )
    }
)


if row_membership_candidates:

    for path in (
        row_membership_candidates
    ):

        print(
            f"{human_bytes(path.stat().st_size):>12s}",
            path,
        )


else:

    print(
        "<NONE>"
    )


print()


# ============================================================================
# 8. RESOLVE FROZEN SOURCE FILES
# ============================================================================

print("=" * 112)
print("FROZEN SOURCE FILE RESOLUTION")
print("=" * 112)


deduplicated_sources = {}


for descriptor in (
    source_descriptors
):

    key = (
        descriptor.get(
            "day"
        ),
        descriptor.get(
            "remote"
        ),
        descriptor.get(
            "sha256"
        ),
    )

    deduplicated_sources[
        key
    ] = descriptor


source_resolution = []


for descriptor in (
    deduplicated_sources.values()
):

    remote_name = (
        descriptor.get(
            "remote"
        )
    )


    if not remote_name:
        continue


    basename = Path(
        remote_name
    ).name


    candidates = list(
        basename_index.get(
            basename,
            [],
        )
    )


    if not candidates:

        lower_basename = (
            basename.lower()
        )


        candidates = [
            path
            for path
            in all_files
            if path.name.lower()
            == lower_basename
        ]


    record = dict(
        descriptor
    )


    record[
        "basename"
    ] = basename


    record[
        "mounted_candidates"
    ] = [
        str(
            path
        )
        for path
        in candidates
    ]


    source_resolution.append(
        record
    )


    print(
        f"[{descriptor.get('day', '?')}]"
    )

    print(
        "  remote:",
        remote_name,
    )


    if descriptor.get(
        "sha256"
    ):

        print(
            "  expected SHA:",
            descriptor[
                "sha256"
            ],
        )


    if candidates:

        for candidate in candidates:

            print(
                "  mounted:",
                candidate,
            )

            print(
                "  size:",
                human_bytes(
                    candidate.stat().st_size
                ),
            )


    else:

        print(
            "  mounted: <NOT FOUND>"
        )


    print()


# ============================================================================
# 9. GENERAL PCAP INVENTORY
# ============================================================================

print("=" * 112)
print("PCAP INVENTORY")
print("=" * 112)


pcaps = sorted(
    [
        path
        for path
        in all_files
        if path.suffix.lower()
        in {
            ".pcap",
            ".pcapng",
        }
    ]
)


print(
    "PCAP / PCAPNG files:",
    len(
        pcaps
    ),
)


for path in pcaps:

    print(
        f"{human_bytes(path.stat().st_size):>12s}",
        path,
    )


print()


# ============================================================================
# 10. RELEVANT PARQUET INVENTORY
# ============================================================================

print("=" * 112)
print("RELEVANT PARQUET INVENTORY")
print("=" * 112)


relevant_parquets = []


for path in all_files:

    if (
        path.suffix.lower()
        != ".parquet"
    ):
        continue


    low = str(
        path
    ).lower()


    if any(
        token in low
        for token
        in [
            "cic",
            "ids",
            "traffic",
            "label",
            "monday",
            "tuesday",
            "wednesday",
            "thursday",
            "friday",
        ]
    ):

        relevant_parquets.append(
            path
        )


print(
    "Relevant Parquet files:",
    len(
        relevant_parquets
    ),
)


for path in sorted(
    relevant_parquets
):

    print(
        f"{human_bytes(path.stat().st_size):>12s}",
        path,
    )


print()


# ============================================================================
# 11. SEARCH REPO FOR FROZEN EXACT-JOIN IMPLEMENTATION
# ============================================================================

print("=" * 112)
print("REPOSITORY EXACT-JOIN IMPLEMENTATION SEARCH")
print("=" * 112)


patterns = [
    "SCAPY_2_6_1_OUTER_IPV4_PAYLOAD_LAYER",
    "source_faithful",
    "raw_unmatched_reconstructed_flows",
    "unused_published_occurrences",
    "published_occurrence",
    "supervised_matched_flows",
    "validation_matched_flows",
    "holdout_matched_flows",
    "EXACT_ONLY",
]


repository_hits = {}


for pattern in patterns:

    p = subprocess.run(
        [
            "grep",
            "-RIl",
            "--exclude-dir=.git",
            "--",
            pattern,
            str(
                REPO
            ),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
        text=True,
    )


    hits = [
        line.strip()
        for line
        in p.stdout.splitlines()
        if line.strip()
    ]


    repository_hits[
        pattern
    ] = hits


    print(
        pattern
    )


    if hits:

        for hit in hits[:30]:

            try:

                display = str(
                    Path(
                        hit
                    ).relative_to(
                        REPO
                    )
                )

            except Exception:

                display = hit


            print(
                "  ",
                display,
            )


    else:

        print(
            "   <NONE>"
        )


print()


# ============================================================================
# 12. COMPACT-CORPUS CONTRACT SUMMARY
# ============================================================================

print("=" * 112)
print("COMPACT-CORPUS PERSISTENCE CONTRACT")
print("=" * 112)


for day, manifest in (
    manifest_objects.items()
):

    compact = manifest.get(
        "compact_corpus",
        {},
    )


    files = (
        compact.get(
            "files",
            {}
        )
    )


    print(
        day
    )


    print(
        "  flow_count:",
        compact.get(
            "flow_count"
        ),
    )


    print(
        "  persisted files:",
        sorted(
            files.keys()
        ),
    )


    row_like = [
        name
        for name
        in files.keys()
        if any(
            token in name.lower()
            for token
            in [
                "row",
                "membership",
                "published",
                "matched",
                "index",
            ]
        )
    ]


    print(
        "  row-membership-like files:",
        (
            row_like
            if row_like
            else "<NONE>"
        ),
    )


print()


# ============================================================================
# 13. SAVE OPERATIONAL INVENTORY
#
# Not committed. Not a scientific result.
# ============================================================================

inventory = {

    "stage":
        "Stage24-1C0-R1",

    "type":
        "CPU_ONLY_BOOTSTRAP_AND_GROUNDED_S4_INVENTORY",

    "repository_head":
        head,

    "accelerator": {
        "nvidia_smi":
            nvidia_smi,

        "nvidia_devices":
            [
                str(
                    x
                )
                for x
                in nvidia_devices
            ],

        "gpu_visible":
            False,
    },

    "scientific_state": {
        "completed_fits":
            completed_fits,

        "fit_budget":
            4,

        "target_openings":
            target_openings,

        "target_opening_budget":
            8,
    },

    "grounded_s4": {
        "expected_by_day":
            EXPECTED_GROUNDED,

        "expected_total":
            EXPECTED_GROUNDED_TOTAL,
    },

    "mounted_file_count":
        len(
            all_files
        ),

    "artifact_candidates":
        [
            str(
                path
            )
            for path
            in artifact_candidates
        ],

    "row_membership_candidates":
        [
            str(
                path
            )
            for path
            in row_membership_candidates
        ],

    "source_resolution":
        source_resolution,

    "pcaps":
        [
            {
                "path":
                    str(
                        path
                    ),

                "bytes":
                    path.stat().st_size,
            }
            for path
            in pcaps
        ],

    "repository_hits":
        repository_hits,
}


inventory_path = Path(
    "/kaggle/working/"
    "stage24_1c0_cpu_inventory.json"
)


inventory_path.write_text(
    json.dumps(
        inventory,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


# ============================================================================
# FINAL
# ============================================================================

print("=" * 112)
print("STAGE24-1C0-R1 FINAL")
print("=" * 112)

print(
    "CPU-only execution:            PASS"
)

print(
    "Repository HEAD:              ",
    head,
)

print(
    "Completed scientific fits:     ",
    f"{completed_fits} / 4",
)

print(
    "Target openings:               ",
    f"{target_openings} / 8",
)

print(
    "Frozen grounded total:         ",
    f"{EXPECTED_GROUNDED_TOTAL:,}",
)

print(
    "Reusable Stage20 candidates:   ",
    len(
        artifact_candidates
    ),
)

print(
    "Row-membership candidates:     ",
    len(
        row_membership_candidates
    ),
)

print(
    "Mounted PCAPs:                 ",
    len(
        pcaps
    ),
)

print()

print(
    "Inventory:"
)

print(
    " ",
    inventory_path,
)

print()

print("-" * 112)
print("STAGE24-1C0-R1: PASS")
print()
print("No PCAP replay performed.")
print("No model fit performed.")
print("No target model inference performed.")
print("Target openings remain 0 / 8.")
print()
print("PASTE THE COMPLETE OUTPUT.")
print("-" * 112)
print("=" * 112)

STAGE24-1C0-R1 — CPU-ONLY FRESH BOOTSTRAP + GROUNDED_S4 INVENTORY

ACCELERATOR GUARD
nvidia-smi: <NOT INSTALLED>
/dev/nvidia*: <NONE>

[PASS] No physical NVIDIA GPU is visible.
[PASS] CPU-only Stage24-1C execution authorized.

GITHUB BOOTSTRAP
Cloning into '/kaggle/working/ids2018-validation-safe-ablation'...
Updating files:  30% (607/1971)
Updating files:  31% (612/1971)
Updating files:  32% (631/1971)
Updating files:  33% (651/1971)
Updating files:  34% (671/1971)
Updating files:  35% (690/1971)
Updating files:  36% (710/1971)
Updating files:  37% (730/1971)
Updating files:  38% (749/1971)
Updating files:  39% (769/1971)
Updating files:  40% (789/1971)
Updating files:  41% (809/1971)
Updating files:  42% (828/1971)
Updating files:  42% (840/1971)
Updating files:  43% (848/1971)
Updating files:  44% (868/1971)
Updating files:  45% (887/1971)
Updating files:  46% (907/1971)
Updating files:  47% (927/1971)
Updating files:  48% (947/1971)
Updating files:  49% (966/1971)
Updating files:  

In [3]:
# ============================================================================
# STAGE24-1C1-M — MONDAY EXACT SOURCE ACQUISITION
#
# CPU-ONLY.
#
# Downloads ONLY:
#   pcap/Monday-WorkingHours.pcap
#   traffic_labels/Monday-WorkingHours.pcap_ISCX.csv.parquet
#
# Frozen repository:
#   bvsam/cic-ids-2017
#
# Frozen revisions:
#   PCAP   e810c1cc98270ec271a1df917b9de0786c33f343
#   LABEL  b7e532345512edcd530cb1770dc76636aeb52802
#
# NO:
#   PCAP replay
#   exact join
#   model fit
#   model inference
#   target opening
#
# Scientific state remains:
#   completed fits = 2 / 4
#   target openings = 0 / 8
# ============================================================================

from __future__ import annotations

import os
import sys
import json
import shutil
import hashlib
import subprocess
from pathlib import Path


# ============================================================================
# 0. CONSTANTS
# ============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "683fe85839e63dcd5fd8e952fd07931b446652ff"
)


HF_REPO = "bvsam/cic-ids-2017"
HF_REPO_TYPE = "dataset"

PCAP_REVISION = (
    "e810c1cc98270ec271a1df917b9de0786c33f343"
)

LABEL_REVISION = (
    "b7e532345512edcd530cb1770dc76636aeb52802"
)


PCAP_REMOTE = (
    "pcap/Monday-WorkingHours.pcap"
)

LABEL_REMOTE = (
    "traffic_labels/"
    "Monday-WorkingHours.pcap_ISCX.csv.parquet"
)


EXPECTED_PCAP_SHA = (
    "f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972"
)

EXPECTED_PCAP_BYTES = 10_822_507_416


EXPECTED_LABEL_SHA = (
    "dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02"
)

EXPECTED_LABEL_BYTES = 65_465_382

EXPECTED_LABEL_ROWS = 529_918


CACHE_ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache"
)

WORK_ROOT = Path(
    "/kaggle/working/stage24_1c_sources/Monday"
)

WORK_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


RECEIPT_PATH = (
    WORK_ROOT
    / "stage24_1c1m_source_acquisition_receipt.json"
)


print("=" * 112)
print("STAGE24-1C1-M — MONDAY EXACT SOURCE ACQUISITION")
print("=" * 112)

print("CPU-only source acquisition")
print("Completed scientific fits: 2 / 4")
print("Target openings:           0 / 8")
print()


# ============================================================================
# 1. HELPERS
# ============================================================================

def run(
    command,
    cwd=None,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in command],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in command
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (
        p.stdout or ""
    ).strip()


def git(
    *args,
):
    return run(
        ["git", *args],
        cwd=REPO,
    )


def sha256_file(
    path,
    chunk_size=64 * 1024 * 1024,
):
    h = hashlib.sha256()

    total = 0

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

            total += len(
                chunk
            )

            if (
                total
                // (1024**3)
                != (
                    total - len(chunk)
                )
                // (1024**3)
            ):
                print(
                    f"  hashed {total / 1024**3:.2f} GiB"
                )

    return h.hexdigest()


def resolved_path(
    path,
):
    return Path(
        os.path.realpath(
            str(path)
        )
    )


# ============================================================================
# 2. GOVERNANCE
# ============================================================================

print("=" * 112)
print("GOVERNANCE")
print("=" * 112)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
).split()[0]

status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD:",
    head,
)

print(
    "Remote main:",
    remote,
)


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "\nUnexpected local HEAD.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if remote != EXPECTED_HEAD:
    raise RuntimeError(
        "\nUnexpected remote main.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {remote}"
    )


if status:
    raise RuntimeError(
        "\nRepository must be clean:\n"
        + status
    )


print()
print(
    "[PASS] Stage24-1B final state intact."
)
print()


# ============================================================================
# 3. CPU-ONLY GUARD
# ============================================================================

print("=" * 112)
print("ACCELERATOR GUARD")
print("=" * 112)


actual_gpu_nodes = [
    p
    for p
    in Path(
        "/dev"
    ).glob(
        "nvidia[0-9]*"
    )
]


if actual_gpu_nodes:

    raise RuntimeError(
        "\nNVIDIA GPU device detected:\n"
        + "\n".join(
            str(x)
            for x
            in actual_gpu_nodes
        )
        + "\n\nUse Accelerator=None for Stage24-1C."
    )


print(
    "[PASS] No NVIDIA GPU device visible."
)
print()


# ============================================================================
# 4. ENSURE HUGGINGFACE CLIENT
# ============================================================================

print("=" * 112)
print("HUGGING FACE CLIENT")
print("=" * 112)


try:
    import huggingface_hub

except ImportError:

    print(
        "Installing huggingface_hub..."
    )

    run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "huggingface_hub>=0.34,<1.0",
        ]
    )

    import huggingface_hub


from huggingface_hub import (
    hf_hub_download,
)


print(
    "huggingface_hub:",
    huggingface_hub.__version__,
)

print()


# ============================================================================
# 5. STORAGE GATE
#
# Stage20 used one-PCAP-at-a-time storage discipline.
# ============================================================================

print("=" * 112)
print("STORAGE GATE")
print("=" * 112)


disk = shutil.disk_usage(
    "/kaggle/working"
)


required_bytes = (
    EXPECTED_PCAP_BYTES
    + EXPECTED_LABEL_BYTES
    + 1 * 1024**3
)


print(
    "Free GiB:",
    round(
        disk.free
        / 1024**3,
        3,
    ),
)

print(
    "Required GiB incl. 1 GiB headroom:",
    round(
        required_bytes
        / 1024**3,
        3,
    ),
)


if disk.free < required_bytes:

    raise RuntimeError(
        "\nInsufficient /kaggle/working storage for exact Monday source.\n"
        f"Free bytes:     {disk.free}\n"
        f"Required bytes: {required_bytes}"
    )


print()
print(
    "[PASS] Monday one-PCAP-at-a-time storage gate passed."
)
print()


# ============================================================================
# 6. DOWNLOAD EXACT MONDAY PCAP
# ============================================================================

print("=" * 112)
print("DOWNLOADING EXACT MONDAY PCAP")
print("=" * 112)


pcap_snapshot = Path(
    hf_hub_download(
        repo_id=HF_REPO,
        filename=PCAP_REMOTE,
        repo_type=HF_REPO_TYPE,
        revision=PCAP_REVISION,
        cache_dir=str(
            CACHE_ROOT
        ),
        resume_download=True,
    )
)


pcap_blob = resolved_path(
    pcap_snapshot
)


print(
    "Snapshot:",
    pcap_snapshot,
)

print(
    "Resolved:",
    pcap_blob,
)

print(
    "Bytes:",
    pcap_blob.stat().st_size,
)


if (
    pcap_blob.stat().st_size
    != EXPECTED_PCAP_BYTES
):

    raise RuntimeError(
        "\nMonday PCAP size mismatch.\n"
        f"Expected: {EXPECTED_PCAP_BYTES}\n"
        f"Actual:   {pcap_blob.stat().st_size}"
    )


print()
print(
    "Computing Monday PCAP SHA256..."
)


pcap_sha = sha256_file(
    pcap_blob
)


print(
    "Expected:",
    EXPECTED_PCAP_SHA,
)

print(
    "Actual:  ",
    pcap_sha,
)


if pcap_sha != EXPECTED_PCAP_SHA:

    raise RuntimeError(
        "Monday PCAP SHA256 mismatch."
    )


print()
print(
    "[PASS] Monday PCAP byte-exact."
)
print()


# ============================================================================
# 7. DOWNLOAD EXACT MONDAY LABEL PARQUET
# ============================================================================

print("=" * 112)
print("DOWNLOADING EXACT MONDAY LABEL PARQUET")
print("=" * 112)


label_snapshot = Path(
    hf_hub_download(
        repo_id=HF_REPO,
        filename=LABEL_REMOTE,
        repo_type=HF_REPO_TYPE,
        revision=LABEL_REVISION,
        cache_dir=str(
            CACHE_ROOT
        ),
        resume_download=True,
    )
)


label_blob = resolved_path(
    label_snapshot
)


print(
    "Snapshot:",
    label_snapshot,
)

print(
    "Resolved:",
    label_blob,
)

print(
    "Bytes:",
    label_blob.stat().st_size,
)


if (
    label_blob.stat().st_size
    != EXPECTED_LABEL_BYTES
):

    raise RuntimeError(
        "\nMonday label size mismatch.\n"
        f"Expected: {EXPECTED_LABEL_BYTES}\n"
        f"Actual:   {label_blob.stat().st_size}"
    )


print()
print(
    "Computing Monday label SHA256..."
)


label_sha = sha256_file(
    label_blob
)


print(
    "Expected:",
    EXPECTED_LABEL_SHA,
)

print(
    "Actual:  ",
    label_sha,
)


if label_sha != EXPECTED_LABEL_SHA:

    raise RuntimeError(
        "Monday label SHA256 mismatch."
    )


print()
print(
    "[PASS] Monday label Parquet byte-exact."
)
print()


# ============================================================================
# 8. VERIFY PARQUET ROW COUNT / SCHEMA
# ============================================================================

print("=" * 112)
print("MONDAY LABEL STRUCTURE")
print("=" * 112)


import pyarrow.parquet as pq


pf = pq.ParquetFile(
    label_blob
)


rows = int(
    pf.metadata.num_rows
)

columns = list(
    pf.schema_arrow.names
)


print(
    "Rows:",
    rows,
)

print(
    "Columns:",
    len(
        columns
    ),
)


if rows != EXPECTED_LABEL_ROWS:

    raise RuntimeError(
        "\nMonday label row count mismatch.\n"
        f"Expected: {EXPECTED_LABEL_ROWS}\n"
        f"Actual:   {rows}"
    )


required_columns = [
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Fwd Packet Length Min",
    "Fwd Packet Length Max",
    "Bwd Packet Length Min",
    "Bwd Packet Length Max",
    "FIN Flag Count",
    "SYN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "URG Flag Count",
    "Label",
]


missing = [
    c
    for c
    in required_columns
    if c not in columns
]


if missing:

    raise RuntimeError(
        "\nMonday label schema missing required columns:\n"
        + repr(
            missing
        )
    )


print()
print(
    "[PASS] Monday label row count and S4 fields available."
)
print()


# ============================================================================
# 9. FREEZE OPERATIONAL SOURCE RECEIPT
#
# Not yet committed. The replay result will include this source identity.
# ============================================================================

receipt = {

    "stage":
        "Stage24-1C1-M",

    "type":
        "MONDAY_EXACT_SOURCE_ACQUISITION",

    "repository_head":
        head,

    "scientific_state": {
        "completed_fits":
            2,

        "fit_budget":
            4,

        "target_openings":
            0,

        "target_opening_budget":
            8,
    },

    "source_repository": {
        "repository":
            HF_REPO,

        "repo_type":
            HF_REPO_TYPE,
    },

    "pcap": {
        "remote":
            PCAP_REMOTE,

        "revision":
            PCAP_REVISION,

        "snapshot_path":
            str(
                pcap_snapshot
            ),

        "resolved_blob_path":
            str(
                pcap_blob
            ),

        "bytes":
            pcap_blob.stat().st_size,

        "sha256":
            pcap_sha,
    },

    "labels": {
        "remote":
            LABEL_REMOTE,

        "revision":
            LABEL_REVISION,

        "snapshot_path":
            str(
                label_snapshot
            ),

        "resolved_blob_path":
            str(
                label_blob
            ),

        "bytes":
            label_blob.stat().st_size,

        "sha256":
            label_sha,

        "rows":
            rows,

        "columns":
            len(
                columns
            ),
    },

    "scientific_access": {
        "pcap_payload_read_for_replay":
            False,

        "exact_join_performed":
            False,

        "model_fit":
            False,

        "target_model_inference":
            False,

        "target_opening_consumed":
            0,
    },

    "next":
        "Stage24-1C1-M-REPLAY",
}


RECEIPT_PATH.write_text(
    json.dumps(
        receipt,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


print("=" * 112)
print("STAGE24-1C1-M FINAL")
print("=" * 112)

print(
    "Monday PCAP SHA:  ",
    pcap_sha,
)

print(
    "Monday label SHA: ",
    label_sha,
)

print(
    "Monday label rows:",
    rows,
)

print()

print(
    "Source receipt:"
)

print(
    " ",
    RECEIPT_PATH,
)

print()

print(
    "Completed scientific fits: 2 / 4"
)

print(
    "Target openings:           0 / 8"
)

print()

print("-" * 112)
print("STAGE24-1C1-M SOURCE ACQUISITION: PASS")
print()
print("No PCAP replay yet.")
print("No exact join yet.")
print("No target inference.")
print("Target openings remain 0 / 8.")
print()
print("PASTE THE COMPLETE OUTPUT.")
print("-" * 112)
print("=" * 112)

STAGE24-1C1-M — MONDAY EXACT SOURCE ACQUISITION
CPU-only source acquisition
Completed scientific fits: 2 / 4
Target openings:           0 / 8

GOVERNANCE
Local HEAD: 683fe85839e63dcd5fd8e952fd07931b446652ff
Remote main: 683fe85839e63dcd5fd8e952fd07931b446652ff

[PASS] Stage24-1B final state intact.

ACCELERATOR GUARD
[PASS] No NVIDIA GPU device visible.

HUGGING FACE CLIENT
huggingface_hub: 1.11.0

STORAGE GATE
Free GiB: 17.408
Required GiB incl. 1 GiB headroom: 11.14

[PASS] Monday one-PCAP-at-a-time storage gate passed.

DOWNLOADING EXACT MONDAY PCAP


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(


pcap/Monday-WorkingHours.pcap:   0%|          | 0.00/10.8G [00:00<?, ?B/s]

Snapshot: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/snapshots/e810c1cc98270ec271a1df917b9de0786c33f343/pcap/Monday-WorkingHours.pcap
Resolved: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/blobs/f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972
Bytes: 10822507416

Computing Monday PCAP SHA256...
  hashed 1.00 GiB
  hashed 2.00 GiB
  hashed 3.00 GiB
  hashed 4.00 GiB
  hashed 5.00 GiB
  hashed 6.00 GiB
  hashed 7.00 GiB
  hashed 8.00 GiB
  hashed 9.00 GiB
  hashed 10.00 GiB
Expected: f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972
Actual:   f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972

[PASS] Monday PCAP byte-exact.

DOWNLOADING EXACT MONDAY LABEL PARQUET


traffic_labels/Monday-WorkingHours.pcap_(…):   0%|          | 0.00/65.5M [00:00<?, ?B/s]

Snapshot: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Monday-WorkingHours.pcap_ISCX.csv.parquet
Resolved: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/blobs/dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
Bytes: 65465382

Computing Monday label SHA256...
Expected: dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
Actual:   dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02

[PASS] Monday label Parquet byte-exact.

MONDAY LABEL STRUCTURE
Rows: 529918
Columns: 85

[PASS] Monday label row count and S4 fields available.

STAGE24-1C1-M FINAL
Monday PCAP SHA:   f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972
Monday label SHA:  dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
Monday label rows: 529918

Source receipt:
  /kaggle/working/stage24_1c_sources/Monday/stage24_1c1m_source_acquisition

In [4]:
# ==============================================================================
# STAGE24-1C1-M-REPLAY
# MONDAY EXACT STAGE20 S4 REPLAY -> PUBLISHED ROW MEMBERSHIP
#
# CPU ONLY.
#
# Input already acquired and SHA-verified:
#   Monday PCAP
#   Monday traffic-label Parquet
#
# Uses frozen Stage20/Stage21 semantics:
#   - exact directed 21-field S4 only
#   - whole-second Unix epoch timestamp
#   - outer IPv4 Scapy 2.6.1 transport semantics
#   - historical signed-Java-byte bidirectional flow key
#   - 120,000,000 us flow timeout
#   - FIN closes only an already-active matching flow
#   - no RST close branch
#   - first-packet-only aggregate flags
#   - historical physical flag-column mapping
#   - D5 exclusions {471,473}
#   - JAVA_HASHMAP_BUCKET_START EOF ordering
#   - duplicate S4 occurrences consumed one-for-one in published physical
#     Parquet scan order; earliest still-unused physical occurrence wins.
#
# IMPORTANT:
#   This cell does NOT run any model.
#   This cell does NOT consume a Stage24 target opening.
#
# On success:
#   - Monday grounded rows = 528,509
#   - membership is committed + pushed
#   - raw Monday HF cache is deleted only AFTER remote verification
#   - release receipt is committed + pushed
#
# Scientific state afterward:
#   completed fits = 2 / 4
#   target openings = 0 / 8
# ==============================================================================

from __future__ import annotations

import os
import sys
import gc
import json
import math
import time
import socket
import shutil
import base64
import hashlib
import subprocess
import importlib.metadata

from pathlib import Path
from collections import Counter


# ==============================================================================
# 0. CONSTANTS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "683fe85839e63dcd5fd8e952fd07931b446652ff"
)

REMOTE_BRANCH = "main"

SOURCE_RECEIPT = Path(
    "/kaggle/working/"
    "stage24_1c_sources/"
    "Monday/"
    "stage24_1c1m_source_acquisition_receipt.json"
)

HF_CACHE_ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache"
)

OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1c_grounded_s4_membership"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)

MEMBERSHIP_NPZ = (
    OUT
    / "monday_grounded_s4_membership.npz"
)

MEMBERSHIP_NPZ_SHA = (
    OUT
    / "monday_grounded_s4_membership.npz.sha256"
)

REPLAY_RECEIPT = (
    OUT
    / "stage24_1c1m_monday_grounded_s4_replay.json"
)

REPLAY_RECEIPT_SHA = (
    OUT
    / "stage24_1c1m_monday_grounded_s4_replay.sha256"
)

RELEASE_RECEIPT = (
    OUT
    / "stage24_1c1m_monday_raw_source_release.json"
)

RELEASE_RECEIPT_SHA = (
    OUT
    / "stage24_1c1m_monday_raw_source_release.sha256"
)


STAGE20_MANIFEST = (
    REPO
    / "results"
    / "stage20_1e_training"
    / "stage20_1e1m_monday_compact_corpus_manifest.json"
)


EXPECTED_PCAP_BYTES = 10_822_507_416

EXPECTED_PCAP_SHA = (
    "f6eac599358f216b074338813a1cf7be"
    "3cc4e91d116e13efc0dc71f2cca11972"
)

EXPECTED_LABEL_BYTES = 65_465_382

EXPECTED_LABEL_SHA = (
    "dfdcef4b8670e52af54dc4f821748343"
    "65a393473e877174cca46d17b12dfd02"
)

EXPECTED_LABEL_ROWS = 529_918

EXPECTED_UNIQUE_S4 = 529_884

EXPECTED_DUPLICATE_OCCURRENCES = 34

EXPECTED_GROUNDED = 528_509


FLOW_TIMEOUT_US = 120_000_000


EXPECTED_FULL = {
    "raw_packets":
        11_709_971,

    "valid_ipv4_packets":
        11_626_492,

    "exportable_flows":
        529_601,

    "retained_packets":
        11_573_331,

    "FIN":
        216_388,

    "FLOW_TIMEOUT":
        120_017,

    "EOF_CURRENT":
        193_196,

    "EOF_SINGLETON_DISCARDED":
        52_890,
}


EXPECTED_PILOT_PARSER = {
    "IPv4_TCP":
        31_618,

    "IPv4_UDP":
        14_334,

    "IPv4_OTHER_PROTOCOL_0":
        84,

    "NON_IPV4":
        3_964,
}


EXPECTED_RAW_ABSENT = {
    14, 25, 35, 36, 40, 43, 45, 50, 52, 54,
    57, 59, 63, 66, 68, 71, 119, 123, 124, 147,
    148, 197, 198, 199, 277, 279, 281, 307, 309,
    324, 327, 333, 334, 336, 337, 445, 448, 454,
}


FROZEN_D5_INDICES = {
    471,
    473,
}


REASON_TO_CODE = {
    "FIN":
        1,

    "FLOW_TIMEOUT":
        2,

    "EOF_CURRENT":
        3,
}


print("=" * 112)
print("STAGE24-1C1-M-REPLAY — EXACT MONDAY GROUNDED_S4 MEMBERSHIP")
print("=" * 112)

print("Completed scientific fits: 2 / 4")
print("Target openings:           0 / 8")
print("Model execution:           NONE")
print()


# ==============================================================================
# 1. HELPERS
# ==============================================================================

def run(
    command,
    *,
    cwd=None,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in command],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if (
        check
        and p.returncode != 0
    ):
        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in command
            )
            + "\n\n"
            + (
                p.stdout
                or ""
            )
        )

    return (
        p.stdout
        or ""
    ).strip()


def git(
    *args,
    auth_header=None,
    check=True,
):
    command = [
        "git",
    ]

    if auth_header is not None:

        command += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    command += [
        str(x)
        for x in args
    ]

    return run(
        command,
        cwd=REPO,
        check=check,
    )


def load_json(
    path,
):
    return json.loads(
        Path(
            path
        ).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=32 * 1024 * 1024,
):
    h = hashlib.sha256()

    total = 0

    with Path(
        path
    ).open(
        "rb"
    ) as fh:

        while True:

            chunk = fh.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

            total += len(
                chunk
            )

    return (
        h.hexdigest(),
        total,
    )


def write_json_sha(
    path,
    obj,
):
    path = Path(
        path
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    digest, _ = sha256_file(
        path
    )

    sha_path = Path(
        str(
            path
        )
        + ".sha256"
    )

    sha_path.write_text(
        f"{digest}  {path.name}\n",
        encoding="utf-8",
    )

    return digest


def exact_int(
    value,
    name,
):
    if value is None:

        raise RuntimeError(
            f"NULL frozen S4 field: {name}"
        )

    if isinstance(
        value,
        bool,
    ):

        return int(
            value
        )

    if isinstance(
        value,
        int,
    ):

        return int(
            value
        )

    number = float(
        value
    )

    if not math.isfinite(
        number
    ):

        raise RuntimeError(
            f"Non-finite S4 field {name}: {value!r}"
        )

    rounded = round(
        number
    )

    if number != rounded:

        raise RuntimeError(
            f"Non-integral S4 field {name}: {value!r}"
        )

    return int(
        rounded
    )


def normalize_column(
    name,
):
    return "".join(
        ch.lower()
        for ch
        in str(
            name
        )
        if ch.isalnum()
    )


def qident(
    name,
):
    return (
        '"'
        + str(
            name
        ).replace(
            '"',
            '""',
        )
        + '"'
    )


# ==============================================================================
# 2. REPOSITORY / CPU GATE
# ==============================================================================

print("=" * 112)
print("GOVERNANCE")
print("=" * 112)


head = git(
    "rev-parse",
    "HEAD",
)

remote = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
).split()[0]

status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD:",
    head,
)

print(
    "Remote main:",
    remote,
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "\nUnexpected repository parent.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if remote != EXPECTED_HEAD:

    raise RuntimeError(
        "Remote main differs from expected Stage24-1B completion."
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


gpu_nodes = list(
    Path(
        "/dev"
    ).glob(
        "nvidia[0-9]*"
    )
)


if gpu_nodes:

    raise RuntimeError(
        "GPU visible. Stage24-1C must remain CPU-only."
    )


print()
print(
    "[PASS] Stage24 state clean."
)

print(
    "[PASS] CPU-only replay authorized."
)

print()


# ==============================================================================
# 3. GET GITHUB TOKEN NOW
#
# Fail early, not after a 12-million-packet replay.
# ==============================================================================

token = None
token_source = None


try:

    from kaggle_secrets import (
        UserSecretsClient,
    )

    secrets = UserSecretsClient()

    for key in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                key
            )

            if (
                value
                and value.strip()
            ):

                token = value.strip()
                token_source = key
                break

        except Exception:
            pass

except Exception:
    pass


if token is None:

    for key in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            key
        )

        if (
            value
            and value.strip()
        ):

            token = value.strip()
            token_source = (
                "ENV:"
                + key
            )

            break


if token is None:

    raise RuntimeError(
        "GitHub token unavailable. Fix this BEFORE replay."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        + token
    ).encode()
).decode()


print(
    "GitHub credential source:",
    token_source,
)

print()


# ==============================================================================
# 4. VERIFY ACQUISITION RECEIPT WITHOUT REHASHING 10.8 GB
#
# The previous cell already performed the byte-exact SHA pass.
# Same-session receipt + exact blob size/path are used here.
# ==============================================================================

print("=" * 112)
print("MONDAY SOURCE RECEIPT")
print("=" * 112)


if not SOURCE_RECEIPT.is_file():

    raise RuntimeError(
        f"Missing acquisition receipt:\n{SOURCE_RECEIPT}"
    )


source_receipt = load_json(
    SOURCE_RECEIPT
)


if (
    source_receipt[
        "pcap"
    ][
        "sha256"
    ]
    != EXPECTED_PCAP_SHA
):

    raise RuntimeError(
        "Acquisition receipt PCAP SHA mismatch."
    )


if (
    source_receipt[
        "labels"
    ][
        "sha256"
    ]
    != EXPECTED_LABEL_SHA
):

    raise RuntimeError(
        "Acquisition receipt label SHA mismatch."
    )


PCAP = Path(
    source_receipt[
        "pcap"
    ][
        "resolved_blob_path"
    ]
)


LABELS = Path(
    source_receipt[
        "labels"
    ][
        "resolved_blob_path"
    ]
)


if not PCAP.is_file():

    raise RuntimeError(
        f"Monday PCAP blob missing:\n{PCAP}"
    )


if not LABELS.is_file():

    raise RuntimeError(
        f"Monday label blob missing:\n{LABELS}"
    )


if (
    PCAP.stat().st_size
    != EXPECTED_PCAP_BYTES
):

    raise RuntimeError(
        "Monday PCAP size changed."
    )


if (
    LABELS.stat().st_size
    != EXPECTED_LABEL_BYTES
):

    raise RuntimeError(
        "Monday label size changed."
    )


print(
    "PCAP:",
    PCAP,
)

print(
    "Labels:",
    LABELS,
)

print()

print(
    "[PASS] Same-session byte-exact Monday sources recovered."
)

print()


# ==============================================================================
# 5. RUNTIME
# ==============================================================================

print("=" * 112)
print("RUNTIME")
print("=" * 112)


def ensure_package(
    package,
    version=None,
):
    try:

        current = importlib.metadata.version(
            package
        )

    except importlib.metadata.PackageNotFoundError:

        current = None


    if (
        version is not None
        and current != version
    ):

        requirement = (
            f"{package}=={version}"
        )

        print(
            "Installing:",
            requirement,
        )

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                requirement,
            ],
            check=True,
        )

        current = importlib.metadata.version(
            package
        )


    elif current is None:

        print(
            "Installing:",
            package,
        )

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package,
            ],
            check=True,
        )

        current = importlib.metadata.version(
            package
        )


    return current


scapy_version = ensure_package(
    "scapy",
    "2.6.1",
)

duckdb_version = ensure_package(
    "duckdb",
)


if scapy_version != "2.6.1":

    raise RuntimeError(
        f"Frozen Scapy runtime not obtained: {scapy_version}"
    )


import numpy as np
import duckdb

from scapy.utils import RawPcapReader
from scapy.layers.l2 import Ether
from scapy.layers.inet import IP, TCP, UDP


print(
    "Scapy:",
    scapy_version,
)

print(
    "DuckDB:",
    duckdb_version,
)

print(
    "NumPy:",
    np.__version__,
)

print()


# ==============================================================================
# 6. READ SEALED MONDAY STAGE20 MANIFEST
# ==============================================================================

sealed_manifest = load_json(
    STAGE20_MANIFEST
)


sealed_exact = (
    sealed_manifest[
        "exact_join"
    ]
)


if (
    sealed_exact[
        "matching"
    ]
    != "EXACT_ONLY"
):

    raise RuntimeError(
        "Stage20 Monday manifest exact matching contract changed."
    )


if (
    int(
        sealed_exact[
            "supervised_matched_flows"
        ]
    )
    != EXPECTED_GROUNDED
):

    raise RuntimeError(
        "Frozen Monday grounded count changed."
    )


print("=" * 112)
print("FROZEN MONDAY CONTRACT")
print("=" * 112)

print(
    "Expected grounded:",
    EXPECTED_GROUNDED,
)

print(
    "Stage20 matching:",
    sealed_exact[
        "matching"
    ],
)

print()


# ==============================================================================
# 7. LABEL SCHEMA
# ==============================================================================

conn = duckdb.connect(
    database=":memory:"
)


# Preserve source scan order because Stage24 needs a deterministic physical
# occurrence locator for duplicate exact-S4 rows.
try:
    conn.execute(
        "SET preserve_insertion_order = true"
    )
except Exception:
    pass


parquet_sql = str(
    LABELS
).replace(
    "'",
    "''",
)


schema_rows = conn.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{parquet_sql}')
    """
).fetchall()


schema_names = [
    row[
        0
    ]
    for row
    in schema_rows
]


normalized = {}


for name in schema_names:

    normalized.setdefault(
        normalize_column(
            name
        ),
        [],
    ).append(
        name
    )


ALIASES = {
    "source_ip": [
        "sourceip",
        "srcip",
    ],

    "source_port": [
        "sourceport",
        "srcport",
    ],

    "destination_ip": [
        "destinationip",
        "dstip",
    ],

    "destination_port": [
        "destinationport",
        "dstport",
    ],

    "protocol": [
        "protocol",
    ],

    "timestamp": [
        "timestamp",
    ],

    "flow_duration": [
        "flowduration",
    ],

    "total_fwd_packets": [
        "totalfwdpackets",
        "totfwdpkts",
    ],

    "total_bwd_packets": [
        "totalbackwardpackets",
        "totbwdpkts",
    ],

    "total_len_fwd": [
        "totallengthoffwdpackets",
        "totlenfwdpkts",
    ],

    "total_len_bwd": [
        "totallengthofbwdpackets",
        "totlenbwdpkts",
    ],

    "fwd_len_min": [
        "fwdpacketlengthmin",
        "fwdpktlenmin",
    ],

    "fwd_len_max": [
        "fwdpacketlengthmax",
        "fwdpktlenmax",
    ],

    "bwd_len_min": [
        "bwdpacketlengthmin",
        "bwdpktlenmin",
    ],

    "bwd_len_max": [
        "bwdpacketlengthmax",
        "bwdpktlenmax",
    ],

    "fin": [
        "finflagcount",
        "finflagcnt",
    ],

    "syn": [
        "synflagcount",
        "synflagcnt",
    ],

    "rst": [
        "rstflagcount",
        "rstflagcnt",
    ],

    "psh": [
        "pshflagcount",
        "pshflagcnt",
    ],

    "ack": [
        "ackflagcount",
        "ackflagcnt",
    ],

    "urg": [
        "urgflagcount",
        "urgflagcnt",
    ],

    "label": [
        "label",
    ],
}


def resolve_column(
    key,
):
    hits = []

    for alias in ALIASES[
        key
    ]:

        hits.extend(
            normalized.get(
                alias,
                [],
            )
        )


    hits = list(
        dict.fromkeys(
            hits
        )
    )


    if len(
        hits
    ) != 1:

        raise RuntimeError(
            f"Cannot uniquely resolve {key}: {hits}"
        )


    return hits[
        0
    ]


columns = {
    key:
        resolve_column(
            key
        )
    for key
    in ALIASES
}


print(
    "Physical columns:",
    len(
        schema_names
    ),
)

print(
    "Resolved S4 + label columns:",
    len(
        columns
    ),
)

print()


# ==============================================================================
# 8. BUILD EXACT PUBLISHED S4 MULTISET + PHYSICAL OCCURRENCE LOCATORS
#
# entry format:
#   [binary_label, total_count, row_locator]
#
# row_locator:
#   int  -> unique exact S4
#   list -> duplicate exact S4, physical occurrence order
#
# This preserves the historical count-based exact join while additionally
# recovering deterministic physical published-row membership.
# ==============================================================================

print("=" * 112)
print("BUILDING PUBLISHED MONDAY S4 MULTISET")
print("=" * 112)


def binary_label(
    value,
):
    if value is None:

        raise RuntimeError(
            "NULL Label"
        )


    text = str(
        value
    ).strip()


    if not text:

        raise RuntimeError(
            "Empty Label"
        )


    return (
        0
        if text == "BENIGN"
        else 1
    )


query = f"""
SELECT
    CAST({qident(columns["source_ip"])} AS VARCHAR),
    {qident(columns["source_port"])},
    CAST({qident(columns["destination_ip"])} AS VARCHAR),
    {qident(columns["destination_port"])},
    {qident(columns["protocol"])},
    epoch({qident(columns["timestamp"])}),
    {qident(columns["flow_duration"])},
    {qident(columns["total_fwd_packets"])},
    {qident(columns["total_bwd_packets"])},
    {qident(columns["total_len_fwd"])},
    {qident(columns["total_len_bwd"])},
    {qident(columns["fwd_len_min"])},
    {qident(columns["fwd_len_max"])},
    {qident(columns["bwd_len_min"])},
    {qident(columns["bwd_len_max"])},
    {qident(columns["fin"])},
    {qident(columns["syn"])},
    {qident(columns["rst"])},
    {qident(columns["psh"])},
    {qident(columns["ack"])},
    {qident(columns["urg"])},
    CAST({qident(columns["label"])} AS VARCHAR)
FROM read_parquet('{parquet_sql}')
"""


cursor = conn.execute(
    query
)


label_entries = {}

published_binary_counts = Counter()

published_rows = 0


while True:

    batch = cursor.fetchmany(
        50_000
    )


    if not batch:
        break


    for row in batch:

        (
            src_ip,
            src_port,
            dst_ip,
            dst_port,
            protocol,
            timestamp_epoch,
            flow_duration,
            total_fwd_packets,
            total_bwd_packets,
            total_len_fwd,
            total_len_bwd,
            fwd_len_min,
            fwd_len_max,
            bwd_len_min,
            bwd_len_max,
            fin,
            syn,
            rst,
            psh,
            ack,
            urg,
            label_value,
        ) = row


        row_index = int(
            published_rows
        )


        timestamp_float = float(
            timestamp_epoch
        )


        if (
            not math.isfinite(
                timestamp_float
            )
            or timestamp_float
            != round(
                timestamp_float
            )
        ):

            raise RuntimeError(
                f"Non-whole-second timestamp at row {row_index}"
            )


        signature = (
            str(
                src_ip
            ).strip(),

            exact_int(
                src_port,
                "Source Port",
            ),

            str(
                dst_ip
            ).strip(),

            exact_int(
                dst_port,
                "Destination Port",
            ),

            exact_int(
                protocol,
                "Protocol",
            ),

            int(
                round(
                    timestamp_float
                )
            ),

            exact_int(
                flow_duration,
                "Flow Duration",
            ),

            exact_int(
                total_fwd_packets,
                "Total Fwd Packets",
            ),

            exact_int(
                total_bwd_packets,
                "Total Bwd Packets",
            ),

            exact_int(
                total_len_fwd,
                "Total Len Fwd",
            ),

            exact_int(
                total_len_bwd,
                "Total Len Bwd",
            ),

            exact_int(
                fwd_len_min,
                "Fwd Len Min",
            ),

            exact_int(
                fwd_len_max,
                "Fwd Len Max",
            ),

            exact_int(
                bwd_len_min,
                "Bwd Len Min",
            ),

            exact_int(
                bwd_len_max,
                "Bwd Len Max",
            ),

            exact_int(
                fin,
                "FIN",
            ),

            exact_int(
                syn,
                "SYN",
            ),

            exact_int(
                rst,
                "RST",
            ),

            exact_int(
                psh,
                "PSH",
            ),

            exact_int(
                ack,
                "ACK",
            ),

            exact_int(
                urg,
                "URG",
            ),
        )


        binary = binary_label(
            label_value
        )


        existing = label_entries.get(
            signature
        )


        if existing is None:

            label_entries[
                signature
            ] = [
                binary,
                1,
                row_index,
            ]


        else:

            if existing[
                0
            ] != binary:

                raise RuntimeError(
                    "Exact Monday S4 maps to conflicting binary labels."
                )


            existing[
                1
            ] += 1


            locator = existing[
                2
            ]


            if isinstance(
                locator,
                int,
            ):

                existing[
                    2
                ] = [
                    locator,
                    row_index,
                ]


            else:

                locator.append(
                    row_index
                )


        published_binary_counts[
            binary
        ] += 1


        published_rows += 1


if published_rows != EXPECTED_LABEL_ROWS:

    raise RuntimeError(
        f"Published rows mismatch: {published_rows}"
    )


unique_s4 = len(
    label_entries
)


duplicate_occurrences = (
    published_rows
    - unique_s4
)


if unique_s4 != EXPECTED_UNIQUE_S4:

    raise RuntimeError(
        f"Unique S4 mismatch: {unique_s4}"
    )


if (
    duplicate_occurrences
    != EXPECTED_DUPLICATE_OCCURRENCES
):

    raise RuntimeError(
        f"Duplicate occurrence mismatch: {duplicate_occurrences}"
    )


print(
    "Published rows:",
    f"{published_rows:,}",
)

print(
    "Unique S4:",
    f"{unique_s4:,}",
)

print(
    "Duplicate occurrences:",
    duplicate_occurrences,
)

print(
    "Binary counts:",
    dict(
        sorted(
            published_binary_counts.items()
        )
    ),
)

print()

print(
    "[PASS] Frozen Monday published S4 population reproduced."
)

print()


# ==============================================================================
# 9. EXACT HISTORICAL FLOW SEMANTICS
# ==============================================================================

def timestamp_us(
    metadata,
    *,
    nanosecond_precision=False,
):
    if (
        hasattr(
            metadata,
            "sec",
        )
        and
        hasattr(
            metadata,
            "usec",
        )
    ):

        sec = int(
            metadata.sec
        )

        fractional = int(
            metadata.usec
        )


        if nanosecond_precision:

            fractional //= 1000


        return (
            sec
            * 1_000_000
            + fractional
        )


    if (
        hasattr(
            metadata,
            "tshigh",
        )
        and
        hasattr(
            metadata,
            "tslow",
        )
        and
        hasattr(
            metadata,
            "tsresol",
        )
    ):

        ticks = (
            (
                int(
                    metadata.tshigh
                )
                << 32
            )
            + int(
                metadata.tslow
            )
        )


        tsresol = int(
            metadata.tsresol
        )


        value, remainder = divmod(
            ticks
            * 1_000_000,
            tsresol,
        )


        if remainder != 0:

            raise RuntimeError(
                "PCAPNG timestamp not exactly microsecond representable."
            )


        return int(
            value
        )


    raise RuntimeError(
        f"Unsupported Scapy timestamp metadata: {metadata!r}"
    )


def java_signed_byte(
    value,
):
    value = int(
        value
    )

    return (
        value
        if value < 128
        else value - 256
    )


def historical_flow_key(
    src,
    dst,
    src_port,
    dst_port,
    protocol,
):
    forward = True


    for src_b, dst_b in zip(
        src,
        dst,
    ):

        s = java_signed_byte(
            src_b
        )

        d = java_signed_byte(
            dst_b
        )


        if s != d:

            if s > d:
                forward = False

            break


    if forward:

        return (
            bytes(
                src
            ),
            bytes(
                dst
            ),
            int(
                src_port
            ),
            int(
                dst_port
            ),
            int(
                protocol
            ),
        )


    return (
        bytes(
            dst
        ),
        bytes(
            src
        ),
        int(
            dst_port
        ),
        int(
            src_port
        ),
        int(
            protocol
        ),
    )


def ip_string(
    raw_ip,
):
    return socket.inet_ntoa(
        bytes(
            raw_ip
        )
    )


def flow_id_string(
    key,
):
    src, dst, sport, dport, proto = (
        key
    )

    return (
        f"{ip_string(src)}-"
        f"{ip_string(dst)}-"
        f"{sport}-"
        f"{dport}-"
        f"{proto}"
    )


def java_string_hash(
    value,
):
    h = 0


    for ch in str(
        value
    ):

        h = (
            31
            * h
            + ord(
                ch
            )
        ) & 0xFFFFFFFF


    return (
        h
        ^ (
            h
            >> 16
        )
    ) & 0xFFFFFFFF


def semantic_tcp_flags(
    tcp,
):
    value = int(
        tcp.flags
    )

    return {
        "FIN":
            bool(
                value
                & 0x01
            ),

        "SYN":
            bool(
                value
                & 0x02
            ),

        "RST":
            bool(
                value
                & 0x04
            ),

        "PSH":
            bool(
                value
                & 0x08
            ),

        "ACK":
            bool(
                value
                & 0x10
            ),

        "URG":
            bool(
                value
                & 0x20
            ),

        "ECE":
            bool(
                value
                & 0x40
            ),

        "CWR":
            bool(
                value
                & 0x80
            ),
    }


ZERO_FLAGS = {
    "FIN": False,
    "SYN": False,
    "RST": False,
    "PSH": False,
    "ACK": False,
    "URG": False,
    "ECE": False,
    "CWR": False,
}


def parse_packet(
    raw_bytes,
):
    """
    Frozen Stage20/Stage21 source-faithful S4 decoder.

    Representation-only packet-image bytes are intentionally omitted here:
    Stage24-1C is recovering S4 membership only.
    """

    pkt = Ether(
        raw_bytes
    )


    if IP not in pkt:

        return None


    ip = pkt[
        IP
    ]


    src_ip = str(
        ip.src
    )

    dst_ip = str(
        ip.dst
    )


    src = socket.inet_aton(
        src_ip
    )

    dst = socket.inet_aton(
        dst_ip
    )


    transport = ip.payload


    protocol = 0
    src_port = 0
    dst_port = 0

    flags = dict(
        ZERO_FLAGS
    )

    payload_length = 0

    parser_class = (
        "IPv4_OTHER_PROTOCOL_0"
    )


    if isinstance(
        transport,
        TCP,
    ):

        protocol = 6

        src_port = int(
            transport.sport
        )

        dst_port = int(
            transport.dport
        )

        payload_length = len(
            bytes(
                transport.payload
            )
        )

        flags = semantic_tcp_flags(
            transport
        )

        parser_class = (
            "IPv4_TCP"
        )


    elif isinstance(
        transport,
        UDP,
    ):

        protocol = 17

        src_port = int(
            transport.sport
        )

        dst_port = int(
            transport.dport
        )

        payload_length = len(
            bytes(
                transport.payload
            )
        )

        parser_class = (
            "IPv4_UDP"
        )


    return {
        "src":
            src,

        "dst":
            dst,

        "src_port":
            src_port,

        "dst_port":
            dst_port,

        "protocol":
            protocol,

        "payload_length":
            int(
                payload_length
            ),

        "fin":
            bool(
                flags[
                    "FIN"
                ]
            ),

        "syn":
            bool(
                flags[
                    "SYN"
                ]
            ),

        "rst":
            bool(
                flags[
                    "RST"
                ]
            ),

        "psh":
            bool(
                flags[
                    "PSH"
                ]
            ),

        "ack":
            bool(
                flags[
                    "ACK"
                ]
            ),

        "urg":
            bool(
                flags[
                    "URG"
                ]
            ),

        "ece":
            bool(
                flags[
                    "ECE"
                ]
            ),

        "cwr":
            bool(
                flags[
                    "CWR"
                ]
            ),

        "parser_class":
            parser_class,
    }


# ==============================================================================
# 10. FLOW STATE
# ==============================================================================

class FlowState:

    __slots__ = (
        "src",
        "dst",
        "src_port",
        "dst_port",
        "protocol",
        "start_us",
        "last_us",
        "packet_count",
        "fwd_count",
        "bwd_count",
        "fwd_sum",
        "bwd_sum",
        "fwd_min",
        "fwd_max",
        "bwd_min",
        "bwd_max",
        "flag_fin",
        "flag_syn",
        "flag_rst",
        "flag_psh",
        "flag_ack",
        "flag_urg",
        "flag_ece",
        "flag_cwr",
        "insertion_seq",
    )


    def __init__(
        self,
        packet,
        ts_us,
        insertion_seq,
        inherited_orientation=None,
    ):

        # Historical firstPacket() statistical direction is FWD.
        self.src = bytes(
            packet[
                "src"
            ]
        )

        self.dst = bytes(
            packet[
                "dst"
            ]
        )

        self.src_port = int(
            packet[
                "src_port"
            ]
        )

        self.dst_port = int(
            packet[
                "dst_port"
            ]
        )

        self.protocol = int(
            packet[
                "protocol"
            ]
        )


        self.start_us = int(
            ts_us
        )

        self.last_us = int(
            ts_us
        )


        payload = int(
            packet[
                "payload_length"
            ]
        )


        self.packet_count = 1

        self.fwd_count = 1
        self.bwd_count = 0

        self.fwd_sum = payload
        self.bwd_sum = 0

        self.fwd_min = payload
        self.fwd_max = payload

        self.bwd_min = 0
        self.bwd_max = 0


        # Historical BasicFlow.firstPacket() checkFlags().
        # addPacket() does NOT update these.
        self.flag_fin = int(
            bool(
                packet[
                    "fin"
                ]
            )
        )

        self.flag_syn = int(
            bool(
                packet[
                    "syn"
                ]
            )
        )

        self.flag_rst = int(
            bool(
                packet[
                    "rst"
                ]
            )
        )

        self.flag_psh = int(
            bool(
                packet[
                    "psh"
                ]
            )
        )

        self.flag_ack = int(
            bool(
                packet[
                    "ack"
                ]
            )
        )

        self.flag_urg = int(
            bool(
                packet[
                    "urg"
                ]
            )
        )

        self.flag_ece = int(
            bool(
                packet[
                    "ece"
                ]
            )
        )

        self.flag_cwr = int(
            bool(
                packet[
                    "cwr"
                ]
            )
        )


        self.insertion_seq = int(
            insertion_seq
        )


        # Historical timeout constructor overwrites output orientation only
        # AFTER first-packet statistical initialization.
        if inherited_orientation is not None:

            (
                inherited_src,
                inherited_dst,
                inherited_src_port,
                inherited_dst_port,
            ) = inherited_orientation


            self.src = bytes(
                inherited_src
            )

            self.dst = bytes(
                inherited_dst
            )

            self.src_port = int(
                inherited_src_port
            )

            self.dst_port = int(
                inherited_dst_port
            )


    def add_packet(
        self,
        packet,
        ts_us,
    ):

        payload = int(
            packet[
                "payload_length"
            ]
        )


        self.packet_count += 1


        # Historical direction comparison tests source IP only.
        if self.src == packet[
            "src"
        ]:

            self.fwd_count += 1

            self.fwd_sum += payload


            self.fwd_min = min(
                self.fwd_min,
                payload,
            )

            self.fwd_max = max(
                self.fwd_max,
                payload,
            )


        else:

            self.bwd_count += 1

            self.bwd_sum += payload


            if self.bwd_count == 1:

                self.bwd_min = payload
                self.bwd_max = payload


            else:

                self.bwd_min = min(
                    self.bwd_min,
                    payload,
                )

                self.bwd_max = max(
                    self.bwd_max,
                    payload,
                )


        self.last_us = int(
            ts_us
        )


    def s4_signature(
        self,
    ):
        # Historical physical serialization:
        #
        # physical FIN <- semantic RST
        # physical SYN <- semantic PSH
        # physical RST <- semantic ECE
        # physical PSH <- semantic SYN
        # physical ACK <- semantic ACK
        # physical URG <- semantic FIN

        return (
            ip_string(
                self.src
            ),

            int(
                self.src_port
            ),

            ip_string(
                self.dst
            ),

            int(
                self.dst_port
            ),

            int(
                self.protocol
            ),

            int(
                self.start_us
                // 1_000_000
            ),

            int(
                self.last_us
                - self.start_us
            ),

            int(
                self.fwd_count
            ),

            int(
                self.bwd_count
            ),

            int(
                self.fwd_sum
            ),

            int(
                self.bwd_sum
            ),

            int(
                self.fwd_min
            ),

            int(
                self.fwd_max
            ),

            int(
                self.bwd_min
            ),

            int(
                self.bwd_max
            ),

            int(
                self.flag_rst
            ),

            int(
                self.flag_psh
            ),

            int(
                self.flag_ece
            ),

            int(
                self.flag_syn
            ),

            int(
                self.flag_ack
            ),

            int(
                self.flag_fin
            ),
        )


# ==============================================================================
# 11. JAVA HASHMAP EOF ORDER
# ==============================================================================

class ActiveFlows:

    def __init__(
        self,
    ):

        self.map = {}

        self.capacity = 16

        self.threshold = 12

        self.next_insertion_seq = 0


    def __len__(
        self,
    ):

        return len(
            self.map
        )


    def get(
        self,
        key,
    ):

        return self.map.get(
            key
        )


    def remove(
        self,
        key,
    ):

        del self.map[
            key
        ]


    def put_new(
        self,
        key,
        flow,
    ):

        if key in self.map:

            raise RuntimeError(
                "Duplicate active flow key insertion."
            )


        if (
            len(
                self.map
            )
            + 1
            >
            self.threshold
        ):

            self.capacity *= 2

            self.threshold = int(
                self.capacity
                * 0.75
            )


        self.map[
            key
        ] = flow


    def create_flow(
        self,
        key,
        packet,
        ts_us,
        inherited_orientation=None,
    ):

        self.next_insertion_seq += 1


        flow = FlowState(
            packet=packet,
            ts_us=ts_us,
            insertion_seq=self.next_insertion_seq,
            inherited_orientation=inherited_orientation,
        )


        self.put_new(
            key,
            flow,
        )


        return flow


    def eof_items_historical_bucket_order(
        self,
    ):

        capacity = int(
            self.capacity
        )


        def order_key(
            item,
        ):

            key, flow = item


            bucket = (
                (
                    capacity
                    - 1
                )
                & java_string_hash(
                    flow_id_string(
                        key
                    )
                )
            )


            return (
                bucket,
                flow.insertion_seq,
            )


        return sorted(
            self.map.items(),
            key=order_key,
        )


# ==============================================================================
# 12. HISTORICAL FLOW GENERATOR
# ==============================================================================

def iterate_completed_flows(
    pcap_path,
    *,
    packet_limit=None,
    include_eof=False,
    progress_every=None,
):

    active = ActiveFlows()


    parser_counts = Counter()

    export_counts = Counter()


    raw_packet_count = 0

    valid_ipv4_count = 0

    exported_flow_count = 0

    exported_packet_total = 0

    timeout_singletons = 0


    reader = RawPcapReader(
        str(
            pcap_path
        )
    )


    pcap_nano = bool(
        getattr(
            reader,
            "nano",
            False,
        )
    )


    try:

        for raw_bytes, metadata in reader:

            raw_packet_count += 1


            parsed = parse_packet(
                raw_bytes
            )


            if parsed is None:

                parser_counts[
                    "NON_IPV4"
                ] += 1


            else:

                valid_ipv4_count += 1


                parser_counts[
                    parsed[
                        "parser_class"
                    ]
                ] += 1


                ts_us = timestamp_us(
                    metadata,
                    nanosecond_precision=pcap_nano,
                )


                key = historical_flow_key(
                    parsed[
                        "src"
                    ],
                    parsed[
                        "dst"
                    ],
                    parsed[
                        "src_port"
                    ],
                    parsed[
                        "dst_port"
                    ],
                    parsed[
                        "protocol"
                    ],
                )


                flow = active.get(
                    key
                )


                # ----------------------------------------------------------
                # New flow
                # FIN-first does NOT immediately close/export.
                # ----------------------------------------------------------

                if flow is None:

                    active.create_flow(
                        key,
                        parsed,
                        ts_us,
                    )


                # ----------------------------------------------------------
                # Timeout checked before FIN.
                # ----------------------------------------------------------

                elif (
                    ts_us
                    - flow.start_us
                    >
                    FLOW_TIMEOUT_US
                ):

                    inherited_orientation = (
                        bytes(
                            flow.src
                        ),
                        bytes(
                            flow.dst
                        ),
                        int(
                            flow.src_port
                        ),
                        int(
                            flow.dst_port
                        ),
                    )


                    if flow.packet_count > 1:

                        exported_flow_count += 1

                        exported_packet_total += int(
                            flow.packet_count
                        )

                        export_counts[
                            "FLOW_TIMEOUT"
                        ] += 1


                        yield (
                            "FLOW",
                            flow,
                            "FLOW_TIMEOUT",
                        )


                    else:

                        timeout_singletons += 1


                    active.remove(
                        key
                    )


                    active.create_flow(
                        key,
                        parsed,
                        ts_us,
                        inherited_orientation=inherited_orientation,
                    )


                # ----------------------------------------------------------
                # Existing matching flow + FIN.
                # ----------------------------------------------------------

                elif parsed[
                    "fin"
                ]:

                    flow.add_packet(
                        parsed,
                        ts_us,
                    )


                    exported_flow_count += 1

                    exported_packet_total += int(
                        flow.packet_count
                    )

                    export_counts[
                        "FIN"
                    ] += 1


                    yield (
                        "FLOW",
                        flow,
                        "FIN",
                    )


                    active.remove(
                        key
                    )


                # ----------------------------------------------------------
                # Normal packet.
                # ----------------------------------------------------------

                else:

                    flow.add_packet(
                        parsed,
                        ts_us,
                    )


            if (
                progress_every is not None
                and
                raw_packet_count
                % progress_every
                == 0
            ):

                yield (
                    "PROGRESS",
                    {
                        "raw_packets":
                            raw_packet_count,

                        "valid_ipv4":
                            valid_ipv4_count,

                        "active_flows":
                            len(
                                active
                            ),

                        "exported_flows":
                            exported_flow_count,
                    },
                    None,
                )


            if (
                packet_limit is not None
                and
                raw_packet_count
                >= packet_limit
            ):

                break


    finally:

        reader.close()


    eof_singletons = 0


    if include_eof:

        for key, flow in (
            active
            .eof_items_historical_bucket_order()
        ):

            if flow.packet_count > 1:

                exported_flow_count += 1

                exported_packet_total += int(
                    flow.packet_count
                )

                export_counts[
                    "EOF_CURRENT"
                ] += 1


                yield (
                    "FLOW",
                    flow,
                    "EOF_CURRENT",
                )


            else:

                eof_singletons += 1


    yield (
        "SUMMARY",
        {
            "raw_packet_count":
                int(
                    raw_packet_count
                ),

            "valid_ipv4_count":
                int(
                    valid_ipv4_count
                ),

            "parser_counts":
                dict(
                    parser_counts
                ),

            "export_counts":
                dict(
                    export_counts
                ),

            "exported_flow_count":
                int(
                    exported_flow_count
                ),

            "exported_packet_total":
                int(
                    exported_packet_total
                ),

            "active_flow_count_at_end":
                int(
                    len(
                        active
                    )
                ),

            "timeout_singletons":
                int(
                    timeout_singletons
                ),

            "eof_singletons":
                int(
                    eof_singletons
                ),

            "java_capacity":
                int(
                    active.capacity
                ),
        },
        None,
    )


# ==============================================================================
# 13. FIRST-50K FORENSIC GATE
# ==============================================================================

print("=" * 112)
print("FIRST-50K FROZEN FORENSIC GATE")
print("=" * 112)


pilot_consumed = {}

pilot_set_membership = []

pilot_multiset_membership = []

pilot_d5_membership = []

pilot_protocol_counts = Counter()

pilot_flow_count = 0

pilot_summary = None

pilot_target_audit = {}


for kind, payload, reason in iterate_completed_flows(
    PCAP,
    packet_limit=50_000,
    include_eof=False,
):

    if kind == "FLOW":

        flow_index = int(
            pilot_flow_count
        )

        pilot_flow_count += 1


        signature = payload.s4_signature()


        entry = label_entries.get(
            signature
        )


        matched_set = (
            entry is not None
        )


        already = int(
            pilot_consumed.get(
                signature,
                0,
            )
        )


        available = (
            0
            if entry is None
            else int(
                entry[
                    1
                ]
            )
            - already
        )


        matched_multiset = bool(
            available > 0
        )


        if matched_multiset:

            pilot_consumed[
                signature
            ] = (
                already
                + 1
            )


        d5_accepted = bool(
            matched_multiset
            and
            flow_index
            not in
            FROZEN_D5_INDICES
        )


        pilot_set_membership.append(
            matched_set
        )

        pilot_multiset_membership.append(
            matched_multiset
        )

        pilot_d5_membership.append(
            d5_accepted
        )


        pilot_protocol_counts[
            int(
                payload.protocol
            )
        ] += 1


        if flow_index in {
            199,
            471,
            473,
        }:

            pilot_target_audit[
                flow_index
            ] = {
                "duration_us":
                    int(
                        payload.last_us
                        - payload.start_us
                    ),

                "protocol":
                    int(
                        payload.protocol
                    ),

                "termination":
                    str(
                        reason
                    ),

                "raw_set_match":
                    bool(
                        matched_set
                    ),

                "raw_multiset_match":
                    bool(
                        matched_multiset
                    ),

                "d5_accepted":
                    bool(
                        d5_accepted
                    ),
            }


    elif kind == "SUMMARY":

        pilot_summary = payload


if pilot_summary is None:

    raise RuntimeError(
        "Pilot summary missing."
    )


if (
    pilot_summary[
        "raw_packet_count"
    ]
    != 50_000
):

    raise RuntimeError(
        "Pilot packet count mismatch."
    )


if (
    pilot_summary[
        "parser_counts"
    ]
    != EXPECTED_PILOT_PARSER
):

    raise RuntimeError(
        "\nPilot parser mismatch.\n"
        f"Expected: {EXPECTED_PILOT_PARSER}\n"
        f"Actual:   {pilot_summary['parser_counts']}"
    )


if pilot_flow_count != 675:

    raise RuntimeError(
        f"Pilot flow count mismatch: {pilot_flow_count}"
    )


if (
    pilot_summary[
        "export_counts"
    ].get(
        "FIN",
        0,
    )
    != 432
):

    raise RuntimeError(
        "Pilot FIN mismatch."
    )


if (
    pilot_summary[
        "export_counts"
    ].get(
        "FLOW_TIMEOUT",
        0,
    )
    != 243
):

    raise RuntimeError(
        "Pilot timeout mismatch."
    )


if dict(
    sorted(
        pilot_protocol_counts.items()
    )
) != {
    0: 1,
    6: 467,
    17: 207,
}:

    raise RuntimeError(
        f"Pilot protocol counts mismatch: {pilot_protocol_counts}"
    )


raw_set_absent = {
    i
    for i, matched
    in enumerate(
        pilot_set_membership
    )
    if not matched
}


raw_multiset_absent = {
    i
    for i, matched
    in enumerate(
        pilot_multiset_membership
    )
    if not matched
}


d5_absent = {
    i
    for i, accepted
    in enumerate(
        pilot_d5_membership
    )
    if not accepted
}


if raw_set_absent != EXPECTED_RAW_ABSENT:

    raise RuntimeError(
        "Frozen pilot exact-set absent indices mismatch."
    )


if raw_multiset_absent != EXPECTED_RAW_ABSENT:

    raise RuntimeError(
        "Frozen pilot exact-multiset absent indices mismatch."
    )


if (
    d5_absent
    != (
        EXPECTED_RAW_ABSENT
        | FROZEN_D5_INDICES
    )
):

    raise RuntimeError(
        "Frozen D5 absent-index population mismatch."
    )


if sum(
    pilot_multiset_membership
) != 637:

    raise RuntimeError(
        "Pilot raw exact != 637."
    )


if sum(
    pilot_d5_membership
) != 635:

    raise RuntimeError(
        "Pilot D5 accepted != 635."
    )


# D5 target anchors.
if (
    pilot_target_audit[
        471
    ][
        "duration_us"
    ]
    != 224
    or
    pilot_target_audit[
        471
    ][
        "termination"
    ]
    != "FIN"
    or
    not pilot_target_audit[
        471
    ][
        "raw_multiset_match"
    ]
):

    raise RuntimeError(
        "D5 anchor 471 failed."
    )


if (
    pilot_target_audit[
        473
    ][
        "duration_us"
    ]
    != 262
    or
    pilot_target_audit[
        473
    ][
        "termination"
    ]
    != "FIN"
    or
    not pilot_target_audit[
        473
    ][
        "raw_multiset_match"
    ]
):

    raise RuntimeError(
        "D5 anchor 473 failed."
    )


if (
    pilot_target_audit[
        199
    ][
        "protocol"
    ]
    != 0
    or
    pilot_target_audit[
        199
    ][
        "raw_multiset_match"
    ]
):

    raise RuntimeError(
        "Protocol-0 anchor 199 failed."
    )


print(
    "Parser counts:",
    pilot_summary[
        "parser_counts"
    ],
)

print(
    "Completed flows:",
    pilot_flow_count,
)

print(
    "Export reasons:",
    pilot_summary[
        "export_counts"
    ],
)

print(
    "Protocol counts:",
    dict(
        sorted(
            pilot_protocol_counts.items()
        )
    ),
)

print(
    "Raw exact:",
    sum(
        pilot_multiset_membership
    ),
    "/ 675",
)

print(
    "D5 accepted:",
    sum(
        pilot_d5_membership
    ),
    "/ 675",
)

print()

print(
    "[PASS] Exact historical Monday first-50k gate reproduced."
)

print()


del pilot_consumed

gc.collect()


# ==============================================================================
# 14. FULL MONDAY EXACT REPLAY
# ==============================================================================

print("=" * 112)
print("FULL MONDAY EXACT S4 REPLAY")
print("=" * 112)


# Number of occurrences already consumed from each exact signature.
consumed_counts = {}


matched_published_rows = []

matched_flow_indices = []

matched_binary_labels = []

matched_reason_codes = []


matched_binary_counts = Counter()

matched_export_reasons = Counter()


raw_unmatched = 0

d5_excluded = 0

full_export_index = 0

full_summary = None


full_start = time.time()


for kind, payload, reason in iterate_completed_flows(
    PCAP,
    packet_limit=None,
    include_eof=True,
    progress_every=1_000_000,
):

    if kind == "PROGRESS":

        print(
            " raw:",
            f"{payload['raw_packets']:,}",
            "| IPv4:",
            f"{payload['valid_ipv4']:,}",
            "| active:",
            f"{payload['active_flows']:,}",
            "| exported:",
            f"{payload['exported_flows']:,}",
            "| grounded:",
            f"{len(matched_published_rows):,}",
        )

        continue


    if kind == "SUMMARY":

        full_summary = payload

        continue


    flow = payload


    flow_index = int(
        full_export_index
    )

    full_export_index += 1


    signature = flow.s4_signature()


    entry = label_entries.get(
        signature
    )


    already = int(
        consumed_counts.get(
            signature,
            0,
        )
    )


    available = (
        0
        if entry is None
        else int(
            entry[
                1
            ]
        )
        - already
    )


    raw_exact_match = bool(
        available > 0
    )


    # --------------------------------------------------------------------------
    # Frozen Stage20 D5 exclusions.
    #
    # They MUST NOT consume an accidental published occurrence.
    # --------------------------------------------------------------------------

    if flow_index in FROZEN_D5_INDICES:

        if not raw_exact_match:

            raise RuntimeError(
                f"D5 flow {flow_index} lost its frozen raw exact collision."
            )


        expected_duration = {
            471:
                224,

            473:
                262,
        }[
            flow_index
        ]


        duration = int(
            flow.last_us
            - flow.start_us
        )


        if (
            duration != expected_duration
            or int(
                flow.protocol
            ) != 6
            or reason != "FIN"
        ):

            raise RuntimeError(
                f"D5 full-run anchor failed at flow {flow_index}."
            )


        d5_excluded += 1

        continue


    # --------------------------------------------------------------------------
    # No exact published occurrence left.
    # --------------------------------------------------------------------------

    if not raw_exact_match:

        raw_unmatched += 1

        continue


    # --------------------------------------------------------------------------
    # Consume one exact published occurrence.
    #
    # Deterministic locator extension:
    # earliest still-unused physical Parquet occurrence for this exact S4.
    #
    # This changes NO S4 matching criterion and does not inspect model output.
    # --------------------------------------------------------------------------

    locator = entry[
        2
    ]


    if isinstance(
        locator,
        int,
    ):

        if already != 0:

            raise RuntimeError(
                "Unique S4 occurrence consumed more than once."
            )


        published_row_index = int(
            locator
        )


    else:

        if already >= len(
            locator
        ):

            raise RuntimeError(
                "Duplicate occurrence cursor overflow."
            )


        published_row_index = int(
            locator[
                already
            ]
        )


    consumed_counts[
        signature
    ] = (
        already
        + 1
    )


    binary = int(
        entry[
            0
        ]
    )


    matched_published_rows.append(
        published_row_index
    )

    matched_flow_indices.append(
        flow_index
    )

    matched_binary_labels.append(
        binary
    )

    matched_reason_codes.append(
        REASON_TO_CODE[
            reason
        ]
    )


    matched_binary_counts[
        binary
    ] += 1


    matched_export_reasons[
        reason
    ] += 1


if full_summary is None:

    raise RuntimeError(
        "Full Monday summary missing."
    )


elapsed = (
    time.time()
    - full_start
)


print()
print(
    "Full replay seconds:",
    round(
        elapsed,
        3,
    ),
)

print()


# ==============================================================================
# 15. FULL GEOMETRY / JOIN ASSERTIONS
# ==============================================================================

print("=" * 112)
print("FULL MONDAY FROZEN ANCHORS")
print("=" * 112)


actual_full = {
    "raw_packets":
        int(
            full_summary[
                "raw_packet_count"
            ]
        ),

    "valid_ipv4_packets":
        int(
            full_summary[
                "valid_ipv4_count"
            ]
        ),

    "exportable_flows":
        int(
            full_summary[
                "exported_flow_count"
            ]
        ),

    "retained_packets":
        int(
            full_summary[
                "exported_packet_total"
            ]
        ),

    "FIN":
        int(
            full_summary[
                "export_counts"
            ].get(
                "FIN",
                0,
            )
        ),

    "FLOW_TIMEOUT":
        int(
            full_summary[
                "export_counts"
            ].get(
                "FLOW_TIMEOUT",
                0,
            )
        ),

    "EOF_CURRENT":
        int(
            full_summary[
                "export_counts"
            ].get(
                "EOF_CURRENT",
                0,
            )
        ),

    "EOF_SINGLETON_DISCARDED":
        int(
            full_summary[
                "eof_singletons"
            ]
        ),
}


for key, expected in EXPECTED_FULL.items():

    actual = actual_full[
        key
    ]


    print(
        f"{key:28s}",
        "expected:",
        f"{expected:,}",
        "actual:",
        f"{actual:,}",
    )


    if actual != expected:

        raise RuntimeError(
            f"Full Monday anchor failed: {key}"
        )


grounded_count = len(
    matched_published_rows
)


if grounded_count != EXPECTED_GROUNDED:

    raise RuntimeError(
        "\nMonday grounded count mismatch.\n"
        f"Expected: {EXPECTED_GROUNDED}\n"
        f"Actual:   {grounded_count}"
    )


if d5_excluded != 2:

    raise RuntimeError(
        f"Expected exactly 2 D5 exclusions, got {d5_excluded}"
    )


expected_raw_unmatched = (
    EXPECTED_FULL[
        "exportable_flows"
    ]
    - EXPECTED_GROUNDED
    - 2
)


if raw_unmatched != expected_raw_unmatched:

    raise RuntimeError(
        "\nRaw unmatched count mismatch.\n"
        f"Expected: {expected_raw_unmatched}\n"
        f"Actual:   {raw_unmatched}"
    )


unused_published = sum(
    int(
        entry[
            1
        ]
    )
    - int(
        consumed_counts.get(
            signature,
            0,
        )
    )
    for signature, entry
    in label_entries.items()
)


expected_unused = (
    EXPECTED_LABEL_ROWS
    - EXPECTED_GROUNDED
)


if unused_published != expected_unused:

    raise RuntimeError(
        "\nUnused published occurrence mismatch.\n"
        f"Expected: {expected_unused}\n"
        f"Actual:   {unused_published}"
    )


if (
    "raw_unmatched_reconstructed_flows"
    in sealed_exact
):

    if (
        int(
            sealed_exact[
                "raw_unmatched_reconstructed_flows"
            ]
        )
        != raw_unmatched
    ):

        raise RuntimeError(
            "Stage20 manifest raw-unmatched count mismatch."
        )


if (
    "unused_published_occurrences"
    in sealed_exact
):

    if (
        int(
            sealed_exact[
                "unused_published_occurrences"
            ]
        )
        != unused_published
    ):

        raise RuntimeError(
            "Stage20 manifest unused-published count mismatch."
        )


if (
    "matched_binary_counts"
    in sealed_exact
):

    expected_binary = {
        int(
            key
        ):
            int(
                value
            )
        for key, value
        in sealed_exact[
            "matched_binary_counts"
        ].items()
    }


    if (
        dict(
            sorted(
                matched_binary_counts.items()
            )
        )
        != expected_binary
    ):

        raise RuntimeError(
            "\nStage20 matched binary counts mismatch.\n"
            f"Expected: {expected_binary}\n"
            f"Actual:   {dict(matched_binary_counts)}"
        )


if (
    "matched_export_reasons"
    in sealed_exact
):

    expected_reasons = {
        str(
            key
        ):
            int(
                value
            )
        for key, value
        in sealed_exact[
            "matched_export_reasons"
        ].items()
    }


    if (
        dict(
            sorted(
                matched_export_reasons.items()
            )
        )
        != expected_reasons
    ):

        raise RuntimeError(
            "Stage20 matched export-reason counts mismatch."
        )


print()
print(
    "Grounded matched flows:",
    f"{grounded_count:,}",
)

print(
    "Raw unmatched reconstructed:",
    f"{raw_unmatched:,}",
)

print(
    "D5 excluded:",
    d5_excluded,
)

print(
    "Unused published occurrences:",
    f"{unused_published:,}",
)

print(
    "Matched binary counts:",
    dict(
        sorted(
            matched_binary_counts.items()
        )
    ),
)

print(
    "Matched export reasons:",
    dict(
        sorted(
            matched_export_reasons.items()
        )
    ),
)

print()

print(
    "[PASS] Full Monday exact Stage20 join reproduced."
)

print()


# ==============================================================================
# 16. MATERIALIZE MEMBERSHIP
# ==============================================================================

print("=" * 112)
print("MATERIALIZING MONDAY GROUNDED MEMBERSHIP")
print("=" * 112)


published_row_index = np.asarray(
    matched_published_rows,
    dtype=np.int32,
)


reconstructed_flow_index = np.asarray(
    matched_flow_indices,
    dtype=np.int32,
)


binary_labels = np.asarray(
    matched_binary_labels,
    dtype=np.uint8,
)


reason_codes = np.asarray(
    matched_reason_codes,
    dtype=np.uint8,
)


if published_row_index.shape != (
    EXPECTED_GROUNDED,
):

    raise RuntimeError(
        "Membership vector shape mismatch."
    )


if reconstructed_flow_index.shape != (
    EXPECTED_GROUNDED,
):

    raise RuntimeError(
        "Flow-index vector shape mismatch."
    )


if np.unique(
    published_row_index
).size != EXPECTED_GROUNDED:

    raise RuntimeError(
        "A physical published row was consumed more than once."
    )


if (
    published_row_index.min()
    < 0
    or
    published_row_index.max()
    >= EXPECTED_LABEL_ROWS
):

    raise RuntimeError(
        "Published row locator out of range."
    )


np.savez_compressed(
    MEMBERSHIP_NPZ,

    published_row_index=
        published_row_index,

    reconstructed_flow_index=
        reconstructed_flow_index,

    binary_label=
        binary_labels,

    export_reason_code=
        reason_codes,
)


membership_sha, membership_bytes = sha256_file(
    MEMBERSHIP_NPZ
)


MEMBERSHIP_NPZ_SHA.write_text(
    f"{membership_sha}  {MEMBERSHIP_NPZ.name}\n",
    encoding="utf-8",
)


print(
    "Membership rows:",
    f"{len(published_row_index):,}",
)

print(
    "NPZ bytes:",
    f"{membership_bytes:,}",
)

print(
    "NPZ SHA256:",
    membership_sha,
)

print()


# ==============================================================================
# 17. WRITE REPLAY RECEIPT
# ==============================================================================

replay_receipt = {
    "stage":
        "Stage24-1C1-M-REPLAY",

    "status":
        "MONDAY_GROUNDED_S4_MEMBERSHIP_RECOVERED_EXACTLY",

    "repository_parent":
        head,

    "day":
        "Monday",

    "scientific_state_before": {
        "completed_fits":
            2,

        "fit_budget":
            4,

        "target_openings":
            0,

        "target_opening_budget":
            8,
    },

    "source_identity": {
        "pcap": {
            "bytes":
                EXPECTED_PCAP_BYTES,

            "sha256":
                EXPECTED_PCAP_SHA,
        },

        "traffic_labels": {
            "rows":
                EXPECTED_LABEL_ROWS,

            "bytes":
                EXPECTED_LABEL_BYTES,

            "sha256":
                EXPECTED_LABEL_SHA,
        },
    },

    "exact_join": {
        "matching":
            "EXACT_ONLY",

        "signature_fields":
            21,

        "timestamp":
            "INTEGER_UNIX_EPOCH_SECOND",

        "grounded_matched_flows":
            grounded_count,

        "raw_unmatched_reconstructed_flows":
            raw_unmatched,

        "pre_frozen_d5_excluded":
            d5_excluded,

        "pre_frozen_d5_indices":
            sorted(
                FROZEN_D5_INDICES
            ),

        "unused_published_occurrences":
            unused_published,

        "matched_binary_counts": {
            str(
                key
            ):
                int(
                    value
                )
            for key, value
            in sorted(
                matched_binary_counts.items()
            )
        },

        "matched_export_reasons":
            dict(
                sorted(
                    matched_export_reasons.items()
                )
            ),
    },

    "duplicate_occurrence_locator": {
        "purpose":
            (
                "Recover physical published-row membership for exact-S4 "
                "signatures with multiplicity."
            ),

        "matching_rule_changed":
            False,

        "rule":
            (
                "CONSUME_EARLIEST_STILL_UNUSED_PHYSICAL_PARQUET_ROW_"
                "IN_FROZEN_LABEL_SCAN_ORDER"
            ),

        "model_feedback_used":
            False,

        "target_performance_used":
            False,

        "published_duplicate_occurrences":
            EXPECTED_DUPLICATE_OCCURRENCES,
    },

    "first_50000_gate": {
        "parser_counts":
            EXPECTED_PILOT_PARSER,

        "completed_flows":
            675,

        "FIN":
            432,

        "FLOW_TIMEOUT":
            243,

        "protocol_counts": {
            "0":
                1,

            "6":
                467,

            "17":
                207,
        },

        "raw_exact":
            637,

        "d5_source_faithful":
            635,

        "decision":
            "PASS",
    },

    "full_reconstruction":
        actual_full,

    "eof_ordering": {
        "rule":
            "JAVA_HASHMAP_BUCKET_START",

        "java_capacity_at_eof":
            int(
                full_summary[
                    "java_capacity"
                ]
            ),
    },

    "membership_artifact": {
        "file":
            MEMBERSHIP_NPZ.name,

        "rows":
            EXPECTED_GROUNDED,

        "sha256":
            membership_sha,

        "bytes":
            membership_bytes,

        "published_row_index_dtype":
            "int32",

        "reconstructed_flow_index_dtype":
            "int32",

        "binary_label_dtype":
            "uint8",

        "export_reason_code": {
            "1":
                "FIN",

            "2":
                "FLOW_TIMEOUT",

            "3":
                "EOF_CURRENT",
        },
    },

    "elapsed_seconds":
        elapsed,

    "scientific_boundary": {
        "model_fit":
            False,

        "model_forward":
            False,

        "target_feature_values_supplied_to_model":
            False,

        "target_predictions":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,

        "fuzzy_matching":
            False,

        "nearest_matching":
            False,

        "tolerance_matching":
            False,

        "label_repair":
            False,

        "grounding_tolerance_relaxation":
            False,
    },

    "next":
        (
            "REMOTE_FREEZE_MONDAY_MEMBERSHIP_THEN_RELEASE_RAW_MONDAY_"
            "SOURCE_AND_ADVANCE_TO_TUESDAY"
        ),
}


replay_sha = write_json_sha(
    REPLAY_RECEIPT,
    replay_receipt,
)


print(
    "Replay receipt SHA256:",
    replay_sha,
)

print()


# ==============================================================================
# 18. COMMIT + PUSH MEMBERSHIP BEFORE DELETING RAW SOURCE
# ==============================================================================

print("=" * 112)
print("REMOTE FREEZE MONDAY MEMBERSHIP")
print("=" * 112)


git(
    "add",
    "--",
    str(
        MEMBERSHIP_NPZ_SHA.relative_to(
            REPO
        )
    ),
    str(
        REPLAY_RECEIPT.relative_to(
            REPO
        )
    ),
    str(
        REPLAY_RECEIPT_SHA.relative_to(
            REPO
        )
    ),
)


git(
    "add",
    "-f",
    "--",
    str(
        MEMBERSHIP_NPZ.relative_to(
            REPO
        )
    ),
)


staged = [
    line
    for line
    in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
]


expected_staged = {
    str(
        MEMBERSHIP_NPZ.relative_to(
            REPO
        )
    ),

    str(
        MEMBERSHIP_NPZ_SHA.relative_to(
            REPO
        )
    ),

    str(
        REPLAY_RECEIPT.relative_to(
            REPO
        )
    ),

    str(
        REPLAY_RECEIPT_SHA.relative_to(
            REPO
        )
    ),
}


if set(
    staged
) != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged membership set:\n"
        + repr(
            staged
        )
    )


git(
    "commit",
    "-m",
    "stage24: recover Monday GROUNDED_S4 membership",
)


MEMBERSHIP_COMMIT = git(
    "rev-parse",
    "HEAD",
)


if (
    git(
        "rev-parse",
        "HEAD^",
    )
    != EXPECTED_HEAD
):

    raise RuntimeError(
        "Monday membership commit parent mismatch."
    )


print(
    git(
        "push",
        "origin",
        "HEAD:main",
        auth_header=auth_header,
    )
)


remote_membership = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_membership != MEMBERSHIP_COMMIT:

    raise RuntimeError(
        "Remote membership verification failed. RAW SOURCES PRESERVED."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Worktree not clean after membership freeze. RAW SOURCES PRESERVED."
    )


print()
print(
    "[PASS] Monday physical row membership remotely frozen."
)

print(
    "Commit:",
    MEMBERSHIP_COMMIT,
)

print()


# ==============================================================================
# 19. RELEASE RAW MONDAY CACHE ONLY AFTER REMOTE FREEZE
# ==============================================================================

print("=" * 112)
print("RELEASING RAW MONDAY SOURCE")
print("=" * 112)


free_before_release = int(
    shutil.disk_usage(
        "/kaggle/working"
    ).free
)


if HF_CACHE_ROOT.exists():

    shutil.rmtree(
        HF_CACHE_ROOT
    )


gc.collect()


free_after_release = int(
    shutil.disk_usage(
        "/kaggle/working"
    ).free
)


space_reclaimed = (
    free_after_release
    - free_before_release
)


print(
    "HF cache exists after release:",
    HF_CACHE_ROOT.exists(),
)

print(
    "Reclaimed GiB:",
    round(
        space_reclaimed
        / 1024**3,
        3,
    ),
)


if HF_CACHE_ROOT.exists():

    raise RuntimeError(
        "HF raw-source cache still exists after release."
    )


# ==============================================================================
# 20. RELEASE RECEIPT + SECOND REMOTE COMMIT
# ==============================================================================

release_receipt = {
    "stage":
        "Stage24-1C1-M-RELEASE",

    "status":
        "MONDAY_RAW_SOURCE_RELEASED_AFTER_REMOTE_GROUNDED_MEMBERSHIP_DURABILITY",

    "membership_commit":
        MEMBERSHIP_COMMIT,

    "membership_artifact": {
        "file":
            MEMBERSHIP_NPZ.name,

        "sha256":
            membership_sha,

        "rows":
            EXPECTED_GROUNDED,
    },

    "source_released": {
        "pcap": {
            "bytes":
                EXPECTED_PCAP_BYTES,

            "sha256":
                EXPECTED_PCAP_SHA,
        },

        "traffic_labels": {
            "bytes":
                EXPECTED_LABEL_BYTES,

            "sha256":
                EXPECTED_LABEL_SHA,

            "rows":
                EXPECTED_LABEL_ROWS,
        },

        "hf_cache_root_removed":
            True,
    },

    "workspace": {
        "free_bytes_before_release":
            free_before_release,

        "free_bytes_after_release":
            free_after_release,

        "space_reclaimed_bytes":
            space_reclaimed,
    },

    "scientific_state_after": {
        "completed_fits":
            2,

        "fit_budget":
            4,

        "target_openings":
            0,

        "target_opening_budget":
            8,

        "model_fit":
            False,

        "model_forward":
            False,

        "target_predictions":
            0,
    },

    "next":
        "STAGE24-1C1-T_TUESDAY_SOURCE_ACQUISITION_AND_REPLAY",
}


release_sha = write_json_sha(
    RELEASE_RECEIPT,
    release_receipt,
)


git(
    "add",
    "--",
    str(
        RELEASE_RECEIPT.relative_to(
            REPO
        )
    ),
    str(
        RELEASE_RECEIPT_SHA.relative_to(
            REPO
        )
    ),
)


staged = [
    line
    for line
    in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
]


expected_release_staged = {
    str(
        RELEASE_RECEIPT.relative_to(
            REPO
        )
    ),

    str(
        RELEASE_RECEIPT_SHA.relative_to(
            REPO
        )
    ),
}


if set(
    staged
) != expected_release_staged:

    raise RuntimeError(
        "\nUnexpected release staged set:\n"
        + repr(
            staged
        )
    )


git(
    "commit",
    "-m",
    "stage24: release Monday raw source after membership freeze",
)


FINAL_COMMIT = git(
    "rev-parse",
    "HEAD",
)


if (
    git(
        "rev-parse",
        "HEAD^",
    )
    != MEMBERSHIP_COMMIT
):

    raise RuntimeError(
        "Monday release commit parent mismatch."
    )


print(
    git(
        "push",
        "origin",
        "HEAD:main",
        auth_header=auth_header,
    )
)


remote_final = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_final != FINAL_COMMIT:

    raise RuntimeError(
        "Monday release remote verification failed."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository not clean after Monday release."
    )


# ==============================================================================
# 21. FINAL
# ==============================================================================

print()
print("=" * 112)
print("STAGE24-1C1-M-REPLAY FINAL")
print("=" * 112)

print(
    "Monday published rows:          ",
    f"{EXPECTED_LABEL_ROWS:,}",
)

print(
    "Monday GROUNDED_S4 rows:        ",
    f"{EXPECTED_GROUNDED:,}",
)

print(
    "Exact matching:                  YES"
)

print(
    "Fuzzy/tolerance matching:        NO"
)

print(
    "D5 exclusions:                   2"
)

print(
    "Raw unmatched reconstructed:    ",
    f"{raw_unmatched:,}",
)

print(
    "Unused published occurrences:   ",
    f"{unused_published:,}",
)

print()

print(
    "Membership SHA256:"
)

print(
    " ",
    membership_sha,
)

print()

print(
    "Replay seconds:                 ",
    round(
        elapsed,
        3,
    ),
)

print()

print(
    "Membership commit:"
)

print(
    " ",
    MEMBERSHIP_COMMIT,
)

print(
    "Release commit:"
)

print(
    " ",
    FINAL_COMMIT,
)

print()

print(
    "Raw Monday HF cache:             DELETED"
)

print()

print(
    "Completed scientific fits:       2 / 4"
)

print(
    "Target predictions:              0"
)

print(
    "Target openings:                 0 / 8"
)

print()

print("-" * 112)
print("STAGE24-1C1-M: PASS")
print()
print("NEXT: Tuesday exact source acquisition + GROUNDED_S4 replay.")
print("-" * 112)
print("=" * 112)

STAGE24-1C1-M-REPLAY — EXACT MONDAY GROUNDED_S4 MEMBERSHIP
Completed scientific fits: 2 / 4
Target openings:           0 / 8
Model execution:           NONE

GOVERNANCE
Local HEAD: 683fe85839e63dcd5fd8e952fd07931b446652ff
Remote main: 683fe85839e63dcd5fd8e952fd07931b446652ff

[PASS] Stage24 state clean.
[PASS] CPU-only replay authorized.

GitHub credential source: GITHUB_TOKEN

MONDAY SOURCE RECEIPT
PCAP: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/blobs/f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972
Labels: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/blobs/dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02

[PASS] Same-session byte-exact Monday sources recovered.

RUNTIME
Installing: scapy==2.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.4 MB/s eta 0:00:00
Scapy: 2.6.1
DuckDB: 1.3.2
NumPy: 2.0.2

FROZEN MONDAY CONTRACT
Expected grounded: 528509
Stage20 matching: EXACT_ONLY

Phys

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Published rows: 529,918
Unique S4: 529,884
Duplicate occurrences: 34
Binary counts: {0: 529918}

[PASS] Frozen Monday published S4 population reproduced.

FIRST-50K FROZEN FORENSIC GATE


RuntimeError: Frozen pilot exact-set absent indices mismatch.

In [5]:
# ==============================================================================
# STAGE24-1C1-M-DIAG — FIRST-50K S4 MISMATCH DIAGNOSTIC ONLY
#
# Reuses objects/functions from the failed Monday replay cell.
#
# NO full replay.
# NO model.
# NO commit.
# NO target opening.
# ==============================================================================

from collections import defaultdict, Counter
import numpy as np

print("=" * 112)
print("STAGE24-1C1-M-DIAG — FIRST-50K EXACT-S4 MISMATCH")
print("=" * 112)

required = [
    "PCAP",
    "label_entries",
    "iterate_completed_flows",
    "EXPECTED_RAW_ABSENT",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Required state missing; do NOT continue:\n"
        + repr(missing)
    )


# ------------------------------------------------------------------------------
# 1. Replay only first 50k and collect signatures
# ------------------------------------------------------------------------------

flows = []

summary = None

for kind, payload, reason in iterate_completed_flows(
    PCAP,
    packet_limit=50_000,
    include_eof=False,
):

    if kind == "FLOW":

        flows.append(
            {
                "signature":
                    payload.s4_signature(),

                "reason":
                    reason,

                "duration_us":
                    int(
                        payload.last_us
                        - payload.start_us
                    ),

                "protocol":
                    int(
                        payload.protocol
                    ),
            }
        )

    elif kind == "SUMMARY":

        summary = payload


if len(flows) != 675:
    raise RuntimeError(
        f"Expected 675 flows, got {len(flows)}"
    )


# ------------------------------------------------------------------------------
# 2. Actual exact-set membership
# ------------------------------------------------------------------------------

actual_absent = {
    i
    for i, item in enumerate(flows)
    if item[
        "signature"
    ] not in label_entries
}


expected_absent = set(
    EXPECTED_RAW_ABSENT
)


print("Expected absent count:", len(expected_absent))
print("Actual absent count:  ", len(actual_absent))
print()

print(
    "Expected absent indices:"
)
print(
    sorted(
        expected_absent
    )
)
print()

print(
    "Actual absent indices:"
)
print(
    sorted(
        actual_absent
    )
)
print()


unexpected_absent = (
    actual_absent
    - expected_absent
)

unexpected_present = (
    expected_absent
    - actual_absent
)


print(
    "Should MATCH historically but are absent now:",
    len(
        unexpected_absent
    ),
)
print(
    sorted(
        unexpected_absent
    )
)
print()

print(
    "Should be absent historically but match now:",
    len(
        unexpected_present
    ),
)
print(
    sorted(
        unexpected_present
    )
)
print()


# ------------------------------------------------------------------------------
# 3. Build diagnostic indexes over published signatures
#
# S4 positions:
#
#  0 src ip
#  1 src port
#  2 dst ip
#  3 dst port
#  4 protocol
#  5 timestamp epoch second
#  6 duration
#  7 fwd packets
#  8 bwd packets
#  9 fwd bytes
# 10 bwd bytes
# 11 fwd min
# 12 fwd max
# 13 bwd min
# 14 bwd max
# 15 FIN
# 16 SYN
# 17 RST
# 18 PSH
# 19 ACK
# 20 URG
# ------------------------------------------------------------------------------

FIELD_NAMES = [
    "src_ip",
    "src_port",
    "dst_ip",
    "dst_port",
    "protocol",
    "timestamp_epoch_second",
    "flow_duration",
    "total_fwd_packets",
    "total_bwd_packets",
    "total_len_fwd",
    "total_len_bwd",
    "fwd_len_min",
    "fwd_len_max",
    "bwd_len_min",
    "bwd_len_max",
    "FIN",
    "SYN",
    "RST",
    "PSH",
    "ACK",
    "URG",
]


# Exact except timestamp.
except_timestamp = defaultdict(
    list
)

# Exact except flags.
except_flags = defaultdict(
    list
)

# Five-tuple only.
five_tuple = defaultdict(
    list
)


for published_sig in label_entries.keys():

    key_no_ts = (
        published_sig[:5]
        + published_sig[6:]
    )

    except_timestamp[
        key_no_ts
    ].append(
        published_sig
    )


    key_no_flags = (
        published_sig[:15]
    )

    except_flags[
        key_no_flags
    ].append(
        published_sig
    )


    five_tuple[
        published_sig[:5]
    ].append(
        published_sig
    )


# ------------------------------------------------------------------------------
# 4. Diagnose first mismatches that historically SHOULD match
# ------------------------------------------------------------------------------

print("=" * 112)
print("FIELD-LEVEL DIAGNOSTIC")
print("=" * 112)


diagnostic_indices = sorted(
    unexpected_absent
)[:25]


if not diagnostic_indices:

    print(
        "No unexpected absent flows."
    )


timestamp_deltas = Counter()

flag_difference_patterns = Counter()


for idx in diagnostic_indices:

    sig = flows[
        idx
    ][
        "signature"
    ]


    print()
    print("-" * 112)
    print(
        f"FLOW {idx}"
    )

    print(
        "termination:",
        flows[
            idx
        ][
            "reason"
        ],
        "| duration:",
        flows[
            idx
        ][
            "duration_us"
        ],
        "| protocol:",
        flows[
            idx
        ][
            "protocol"
        ],
    )


    # --------------------------------------------------------------------------
    # Case A: everything identical except timestamp
    # --------------------------------------------------------------------------

    ts_candidates = except_timestamp.get(
        sig[:5]
        + sig[6:],
        [],
    )


    if ts_candidates:

        print(
            "Diagnosis: EVERYTHING MATCHES EXCEPT TIMESTAMP"
        )

        print(
            "Reconstructed timestamp:",
            sig[
                5
            ],
        )


        for candidate in ts_candidates[:10]:

            delta = int(
                candidate[
                    5
                ]
                - sig[
                    5
                ]
            )


            timestamp_deltas[
                delta
            ] += 1


            print(
                "Published timestamp:",
                candidate[
                    5
                ],
                "| delta seconds:",
                delta,
                "| delta hours:",
                delta / 3600.0,
            )


        continue


    # --------------------------------------------------------------------------
    # Case B: everything through packet geometry + timestamp matches,
    #         flags differ
    # --------------------------------------------------------------------------

    flag_candidates = except_flags.get(
        sig[:15],
        [],
    )


    if flag_candidates:

        print(
            "Diagnosis: EVERYTHING MATCHES EXCEPT AGGREGATE FLAGS"
        )

        print(
            "Reconstructed flags:",
            sig[
                15:21
            ],
        )


        for candidate in flag_candidates[:10]:

            print(
                "Published flags:   ",
                candidate[
                    15:21
                ],
            )


            pattern = (
                sig[
                    15:21
                ],
                candidate[
                    15:21
                ],
            )


            flag_difference_patterns[
                pattern
            ] += 1


        continue


    # --------------------------------------------------------------------------
    # Case C: same directed five-tuple, compare nearest candidate by number
    #         of unequal S4 fields.
    # --------------------------------------------------------------------------

    candidates = five_tuple.get(
        sig[:5],
        [],
    )


    print(
        "Same five-tuple published candidates:",
        len(
            candidates
        ),
    )


    if not candidates:

        print(
            "Diagnosis: NO SAME DIRECTED FIVE-TUPLE"
        )

        continue


    best = []

    best_diff_count = 999


    for candidate in candidates:

        differences = [
            position
            for position, (
                reconstructed,
                published,
            )
            in enumerate(
                zip(
                    sig,
                    candidate,
                )
            )
            if reconstructed
            != published
        ]


        if len(
            differences
        ) < best_diff_count:

            best_diff_count = len(
                differences
            )

            best = [
                (
                    candidate,
                    differences,
                )
            ]


        elif (
            len(
                differences
            )
            == best_diff_count
            and len(
                best
            ) < 5
        ):

            best.append(
                (
                    candidate,
                    differences,
                )
            )


    print(
        "Minimum differing fields:",
        best_diff_count,
    )


    for candidate, differences in best[:3]:

        print(
            "Differences:"
        )


        for position in differences:

            reconstructed = sig[
                position
            ]

            published = candidate[
                position
            ]


            extra = ""


            if position == 5:

                delta = int(
                    published
                    - reconstructed
                )

                timestamp_deltas[
                    delta
                ] += 1


                extra = (
                    f"  delta={delta}s "
                    f"({delta / 3600.0:+.3f}h)"
                )


            print(
                f"  [{position:02d}] "
                f"{FIELD_NAMES[position]:26s} "
                f"reconstructed={reconstructed!r} "
                f"published={published!r}"
                f"{extra}"
            )


# ------------------------------------------------------------------------------
# 5. Aggregate diagnosis
# ------------------------------------------------------------------------------

print()
print("=" * 112)
print("AGGREGATE DIAGNOSIS")
print("=" * 112)


print(
    "Timestamp deltas observed:"
)


if timestamp_deltas:

    for delta, count in (
        timestamp_deltas.most_common()
    ):

        print(
            f"  {delta:+d} sec "
            f"({delta / 3600.0:+.3f} h): "
            f"{count}"
        )

else:

    print(
        "  <NONE>"
    )


print()

print(
    "Flag-difference patterns:"
)


if flag_difference_patterns:

    for pattern, count in (
        flag_difference_patterns.most_common(
            20
        )
    ):

        reconstructed, published = (
            pattern
        )

        print(
            f"  reconstructed={reconstructed} "
            f"published={published} "
            f"count={count}"
        )

else:

    print(
        "  <NONE>"
    )


print()

print(
    "Parser geometry:"
)

print(
    summary[
        "parser_counts"
    ]
)

print()

print("-" * 112)
print("STAGE24-1C1-M-DIAG COMPLETE")
print()
print("No full PCAP replay performed.")
print("No model executed.")
print("No Git commit.")
print("Target openings remain 0 / 8.")
print()
print("PASTE THIS OUTPUT.")
print("-" * 112)

STAGE24-1C1-M-DIAG — FIRST-50K EXACT-S4 MISMATCH
Expected absent count: 38
Actual absent count:   1

Expected absent indices:
[14, 25, 35, 36, 40, 43, 45, 50, 52, 54, 57, 59, 63, 66, 68, 71, 119, 123, 124, 147, 148, 197, 198, 199, 277, 279, 281, 307, 309, 324, 327, 333, 334, 336, 337, 445, 448, 454]

Actual absent indices:
[199]

Should MATCH historically but are absent now: 0
[]

Should be absent historically but match now: 37
[14, 25, 35, 36, 40, 43, 45, 50, 52, 54, 57, 59, 63, 66, 68, 71, 119, 123, 124, 147, 148, 197, 198, 277, 279, 281, 307, 309, 324, 327, 333, 334, 336, 337, 445, 448, 454]

FIELD-LEVEL DIAGNOSTIC
No unexpected absent flows.

AGGREGATE DIAGNOSIS
Timestamp deltas observed:
  <NONE>

Flag-difference patterns:
  <NONE>

Parser geometry:
{'IPv4_TCP': 31618, 'NON_IPV4': 3964, 'IPv4_OTHER_PROTOCOL_0': 84, 'IPv4_UDP': 14334}

----------------------------------------------------------------------------------------------------------------
STAGE24-1C1-M-DIAG COMPLETE

No ful

In [6]:
# ==============================================================================
# STAGE24-1C1-M-DIAG2 — IDENTIFY HISTORICAL STAGE20 FLAG SERIALIZATION
#
# FIRST 50,000 PACKETS ONLY.
# NO FULL REPLAY.
# NO MODEL.
# NO COMMIT.
# NO TARGET OPENING.
# ==============================================================================

from collections import Counter

print("=" * 112)
print("STAGE24-1C1-M-DIAG2 — HISTORICAL S4 FLAG SERIALIZATION")
print("=" * 112)

EXPECTED = set(EXPECTED_RAW_ABSENT)

# ------------------------------------------------------------------------------
# Build candidate signatures from the SAME reconstructed flow geometry.
#
# First 15 S4 fields are invariant:
#   IPs, ports, protocol, timestamp, duration,
#   packet counts, byte counts, min/max lengths.
#
# Only final six aggregate flag columns vary here.
# ------------------------------------------------------------------------------

def s4_base(flow):
    current = flow.s4_signature()
    return current[:15]


def sig_direct_semantic(flow):
    """
    Candidate A:
    reconstructed semantic flags written directly into the historical
    physical FIN/SYN/RST/PSH/ACK/URG positions.
    """
    return (
        *s4_base(flow),

        int(flow.flag_fin),
        int(flow.flag_syn),
        int(flow.flag_rst),
        int(flow.flag_psh),
        int(flow.flag_ack),
        int(flow.flag_urg),
    )


def sig_later_corrected_permutation(flow):
    """
    Candidate B:
    later Stage20 flag-correction permutation that the previous cell used.
    """
    return (
        *s4_base(flow),

        int(flow.flag_rst),   # physical FIN
        int(flow.flag_psh),   # physical SYN
        int(flow.flag_ece),   # physical RST
        int(flow.flag_syn),   # physical PSH
        int(flow.flag_ack),   # physical ACK
        int(flow.flag_fin),   # physical URG
    )


candidates = {
    "DIRECT_SEMANTIC":
        sig_direct_semantic,

    "LATER_FLAG_CORRECTED_PERMUTATION":
        sig_later_corrected_permutation,
}


results = {}

flow_count = 0
summary = None

flow_objects = []


for kind, payload, reason in iterate_completed_flows(
    PCAP,
    packet_limit=50_000,
    include_eof=False,
):

    if kind == "FLOW":
        flow_objects.append(payload)
        flow_count += 1

    elif kind == "SUMMARY":
        summary = payload


if flow_count != 675:
    raise RuntimeError(
        f"Expected 675 flows, got {flow_count}"
    )


print("Completed flows:", flow_count)
print("Parser counts:  ", summary["parser_counts"])
print()


for name, signature_fn in candidates.items():

    absent = {
        i
        for i, flow in enumerate(flow_objects)
        if signature_fn(flow)
        not in label_entries
    }

    exact = (
        flow_count
        - len(absent)
    )

    results[name] = {
        "exact":
            exact,

        "absent":
            absent,

        "exact_absent_identity":
            absent == EXPECTED,
    }


    print("-" * 112)
    print(name)
    print()

    print(
        "Exact set membership:",
        exact,
        "/ 675",
    )

    print(
        "Absent count:",
        len(absent),
    )

    print(
        "Exact frozen absent-index identity:",
        absent == EXPECTED,
    )

    print(
        "Unexpected absent:",
        sorted(
            absent - EXPECTED
        ),
    )

    print(
        "Unexpected present:",
        sorted(
            EXPECTED - absent
        ),
    )

    print()


# ------------------------------------------------------------------------------
# Decision
# ------------------------------------------------------------------------------

passing = [
    name
    for name, result
    in results.items()
    if (
        result["exact"] == 637
        and
        result["exact_absent_identity"]
    )
]


print("=" * 112)
print("DECISION")
print("=" * 112)


if len(passing) == 1:

    print(
        "[PASS] Historical Stage20 raw-S4 flag serialization identified:"
    )

    print()
    print(
        "   ",
        passing[0],
    )

    print()

    HISTORICAL_STAGE20_FLAG_SERIALIZATION = passing[0]


elif len(passing) == 0:

    print(
        "[FAIL] Neither candidate reproduces the frozen Stage20 gate."
    )

    raise RuntimeError(
        "Historical flag serialization still unresolved."
    )


else:

    raise RuntimeError(
        f"Ambiguous historical serialization: {passing}"
    )


print(
    "Expected raw exact: 637 / 675"
)

print(
    "Expected absent indices reproduced exactly: YES"
)

print()

print("-" * 112)
print("STAGE24-1C1-M-DIAG2 COMPLETE")
print()
print("No full replay.")
print("No model execution.")
print("No commit.")
print("Target openings remain 0 / 8.")
print()
print("PASTE OUTPUT.")
print("-" * 112)

STAGE24-1C1-M-DIAG2 — HISTORICAL S4 FLAG SERIALIZATION
Completed flows: 675
Parser counts:   {'IPv4_TCP': 31618, 'NON_IPV4': 3964, 'IPv4_OTHER_PROTOCOL_0': 84, 'IPv4_UDP': 14334}

----------------------------------------------------------------------------------------------------------------
DIRECT_SEMANTIC

Exact set membership: 274 / 675
Absent count: 401
Exact frozen absent-index identity: False
Unexpected absent: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 15, 16, 17, 18, 20, 21, 22, 23, 24, 26, 27, 28, 29, 31, 32, 34, 37, 38, 39, 41, 42, 44, 46, 47, 48, 49, 51, 53, 55, 56, 58, 60, 61, 62, 64, 65, 67, 69, 70, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 121, 122, 125, 127, 128, 129, 130, 131, 142, 158, 159, 171, 172, 182, 183, 184, 187, 195, 196, 200, 201, 202, 203, 204, 205, 207, 208, 210, 211, 212, 213, 214, 215, 216, 217

RuntimeError: Historical flag serialization still unresolved.

In [7]:
# ==============================================================================
# STAGE24-1C1-M-RUNTIME-RECOVERY
#
# PURPOSE
# -------
# Recover the frozen Stage20 TCP payload runtime semantics in this fresh
# Kaggle/Scapy environment WITHOUT selecting anything from current label
# agreement.
#
# 1. Use PRE-EXISTING frozen Stage20 C16 per-packet records.
# 2. Test captured IPv4 extent -> TCP payload against those frozen records.
# 3. Only if packet-level equivalence is exact, install that runtime adapter.
# 4. Re-run Monday first-50k S4 gate.
#
# REQUIRED PASS:
#   raw exact = 637 / 675
#   D5        = 635 / 675
#   exact historical absent indices reproduced
#
# NO FULL MONDAY REPLAY YET
# NO MODEL
# NO TARGET OPENING
# NO COMMIT
# ==============================================================================

from pathlib import Path
from collections import Counter
import json
import socket
import gc

import scapy
from scapy.layers.l2 import Ether
from scapy.layers.inet import IP, TCP, UDP
from scapy.utils import RawPcapReader


print("=" * 112)
print("STAGE24-1C1-M — FROZEN TCP RUNTIME RECOVERY")
print("=" * 112)

assert scapy.__version__ == "2.6.1"

assert "REPO" in globals()
assert "PCAP" in globals()
assert "label_entries" in globals()
assert "iterate_completed_flows" in globals()
assert "EXPECTED_RAW_ABSENT" in globals()

print("Scapy:", scapy.__version__)
print("PCAP:", PCAP)
print()


# ==============================================================================
# 1. FIND THE PRE-FROZEN C16 PACKET RECORDS
# ==============================================================================

print("=" * 112)
print("LOCATING PRE-FROZEN C16 PACKET RECORDS")
print("=" * 112)


runtime_dir = (
    Path(REPO)
    / "results"
    / "stage20_1c16_runtime_recovery"
)

assert runtime_dir.is_dir(), runtime_dir


d1_candidates = []
d2_candidates = []


for path in runtime_dir.rglob("*.json"):

    try:
        obj = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

    except Exception:
        continue


    checkpoint = str(
        obj.get(
            "checkpoint",
            ""
        )
    )


    if checkpoint == "Stage20-1C16-D1":
        d1_candidates.append(
            (path, obj)
        )

    elif checkpoint == "Stage20-1C16-D2":
        d2_candidates.append(
            (path, obj)
        )


if len(d1_candidates) != 1:

    raise RuntimeError(
        f"Expected exactly one frozen C16-D1 artifact; "
        f"found {len(d1_candidates)}:\n"
        + "\n".join(
            str(x[0])
            for x in d1_candidates
        )
    )


if len(d2_candidates) != 1:

    raise RuntimeError(
        f"Expected exactly one frozen C16-D2 artifact; "
        f"found {len(d2_candidates)}:\n"
        + "\n".join(
            str(x[0])
            for x in d2_candidates
        )
    )


D1_PATH, d1 = d1_candidates[0]
D2_PATH, d2 = d2_candidates[0]


print(
    "D1:",
    D1_PATH.relative_to(
        REPO
    ),
)

print(
    "D2:",
    D2_PATH.relative_to(
        REPO
    ),
)


gt_records = d1[
    "per_packet"
]

lt_records = d2[
    "per_packet"
]


targets = {}


for cohort, rows in [
    (
        "GT",
        gt_records,
    ),
    (
        "LT",
        lt_records,
    ),
]:

    for row in rows:

        raw_index = int(
            row[
                "raw_index"
            ]
        )

        if raw_index in targets:

            raise RuntimeError(
                f"Duplicate frozen packet index: {raw_index}"
            )


        targets[
            raw_index
        ] = (
            cohort,
            row,
        )


print(
    "GT frozen packets:",
    len(
        gt_records
    ),
)

print(
    "LT frozen packets:",
    len(
        lt_records
    ),
)

print(
    "Total frozen packets:",
    len(
        targets
    ),
)


if not targets:

    raise RuntimeError(
        "Frozen C16 packet population is empty."
    )


max_target = max(
    targets
)


if max_target >= 50_000:

    raise RuntimeError(
        f"C16 target exceeds first50k boundary: {max_target}"
    )


print(
    "Maximum raw packet index:",
    max_target,
)

print()


# ==============================================================================
# 2. RAW IPV4 OFFSET
# ==============================================================================

VLAN_TYPES = {
    0x8100,
    0x88A8,
    0x9100,
    0x9200,
    0x9300,
}


def ipv4_offset_from_frame(
    raw_bytes,
):

    raw = memoryview(
        raw_bytes
    )


    if len(
        raw
    ) < 14:

        return None


    ethertype = int.from_bytes(
        raw[
            12:14
        ],
        "big",
    )


    offset = 14


    while ethertype in VLAN_TYPES:

        if len(
            raw
        ) < (
            offset
            + 4
        ):

            return None


        ethertype = int.from_bytes(
            raw[
                offset + 2:
                offset + 4
            ],
            "big",
        )


        offset += 4


    if ethertype != 0x0800:

        return None


    return int(
        offset
    )


# ==============================================================================
# 3. LABEL-FREE C16 RUNTIME EQUIVALENCE
#
# IMPORTANT:
# No traffic-label membership is consulted anywhere in this section.
#
# Candidate historical captured extent:
#
#   len(raw Ethernet frame) - IPv4 byte offset
#
# TCP payload:
#
#   captured IPv4 extent
#       - IPv4 header length
#       - TCP header length
#
# ==============================================================================

print("=" * 112)
print("LABEL-FREE FROZEN C16 RUNTIME EQUIVALENCE")
print("=" * 112)


counts = {
    "GT":
        Counter(),

    "LT":
        Counter(),
}


examples = {
    "GT":
        [],

    "LT":
        [],
}


seen = set()


reader = RawPcapReader(
    str(
        PCAP
    )
)


try:

    for raw_index, (
        raw_bytes,
        metadata,
    ) in enumerate(
        reader
    ):

        if raw_index > max_target:
            break


        target = targets.get(
            raw_index
        )


        if target is None:
            continue


        cohort, frozen = target


        seen.add(
            raw_index
        )


        pkt = Ether(
            raw_bytes
        )


        if IP not in pkt:

            raise RuntimeError(
                f"Frozen C16 packet {raw_index} is no longer IPv4."
            )


        ip = pkt[
            IP
        ]


        if not isinstance(
            ip.payload,
            TCP,
        ):

            raise RuntimeError(
                f"Frozen C16 packet {raw_index} is no longer direct TCP: "
                f"{type(ip.payload)!r}"
            )


        tcp = ip.payload


        offset = ipv4_offset_from_frame(
            raw_bytes
        )


        if offset is None:

            raise RuntimeError(
                f"Cannot locate IPv4 frame offset: {raw_index}"
            )


        ip_header_length = (
            int(
                ip.ihl
            )
            * 4
        )


        tcp_header_length = (
            int(
                tcp.dataofs
            )
            * 4
        )


        captured_ipv4_extent = (
            len(
                raw_bytes
            )
            - offset
        )


        captured_payload = (
            captured_ipv4_extent
            - ip_header_length
            - tcp_header_length
        )


        if captured_payload < 0:

            raise RuntimeError(
                f"Negative captured TCP payload at raw packet {raw_index}"
            )


        frozen_extent = int(
            frozen[
                "captured_ipv4_extent"
            ]
        )


        frozen_payload = int(
            frozen[
                "baseline_payload"
            ]
        )


        extent_ok = (
            captured_ipv4_extent
            ==
            frozen_extent
        )


        payload_ok = (
            captured_payload
            ==
            frozen_payload
        )


        counts[
            cohort
        ][
            (
                "extent_match"
                if extent_ok
                else "extent_mismatch"
            )
        ] += 1


        counts[
            cohort
        ][
            (
                "payload_match"
                if payload_ok
                else "payload_mismatch"
            )
        ] += 1


        # Also characterize current Scapy direct payload.
        current_direct = len(
            bytes(
                tcp.payload
            )
        )


        direct_ok = (
            current_direct
            ==
            frozen_payload
        )


        counts[
            cohort
        ][
            (
                "direct_match"
                if direct_ok
                else "direct_mismatch"
            )
        ] += 1


        if (
            (
                not extent_ok
                or
                not payload_ok
                or
                not direct_ok
            )
            and
            len(
                examples[
                    cohort
                ]
            )
            < 8
        ):

            examples[
                cohort
            ].append(
                {
                    "raw_index":
                        raw_index,

                    "frozen_extent":
                        frozen_extent,

                    "captured_extent":
                        captured_ipv4_extent,

                    "frozen_payload":
                        frozen_payload,

                    "captured_payload":
                        captured_payload,

                    "current_direct_payload":
                        current_direct,

                    "declared_ipv4_length":
                        int(
                            ip.len
                        ),

                    "ip_header":
                        ip_header_length,

                    "tcp_header":
                        tcp_header_length,
                }
            )


finally:

    reader.close()


missing_targets = sorted(
    set(
        targets
    )
    -
    seen
)


if missing_targets:

    raise RuntimeError(
        f"Frozen C16 target packets were not observed: "
        f"{missing_targets[:20]}"
    )


for cohort, expected_n in [
    (
        "GT",
        len(
            gt_records
        ),
    ),
    (
        "LT",
        len(
            lt_records
        ),
    ),
]:

    c = counts[
        cohort
    ]


    print(
        cohort,
        dict(
            sorted(
                c.items()
            )
        ),
    )


    if examples[
        cohort
    ]:

        print(
            " first differences:",
            examples[
                cohort
            ],
        )


    if (
        c[
            "extent_match"
        ]
        != expected_n
        or
        c[
            "extent_mismatch"
        ]
        != 0
    ):

        raise RuntimeError(
            f"{cohort}: captured IPv4 extent does not reproduce "
            "all frozen C16 packet records."
        )


    if (
        c[
            "payload_match"
        ]
        != expected_n
        or
        c[
            "payload_mismatch"
        ]
        != 0
    ):

        raise RuntimeError(
            f"{cohort}: captured TCP payload does not reproduce "
            "all frozen C16 baseline payload records."
        )


print()

print(
    "[PASS] Captured-byte runtime adapter reproduces EVERY frozen C16 packet."
)

print(
    "[PASS] Adapter selected from PRE-EXISTING LABEL-FREE packet records."
)

print()


# ==============================================================================
# 4. INSTALL FROZEN-RUNTIME-EQUIVALENT PACKET DECODER
# ==============================================================================

def parse_packet_stage20_runtime_equivalent(
    raw_bytes,
):

    pkt = Ether(
        raw_bytes
    )


    if IP not in pkt:

        return None


    ip = pkt[
        IP
    ]


    src_ip = str(
        ip.src
    )

    dst_ip = str(
        ip.dst
    )


    src = socket.inet_aton(
        src_ip
    )

    dst = socket.inet_aton(
        dst_ip
    )


    transport = ip.payload


    protocol = 0

    src_port = 0

    dst_port = 0


    flags = dict(
        ZERO_FLAGS
    )


    payload_length = 0


    parser_class = (
        "IPv4_OTHER_PROTOCOL_0"
    )


    # --------------------------------------------------------------------------
    # TCP
    # --------------------------------------------------------------------------

    if isinstance(
        transport,
        TCP,
    ):

        tcp = transport


        protocol = 6


        src_port = int(
            tcp.sport
        )

        dst_port = int(
            tcp.dport
        )


        flags = semantic_tcp_flags(
            tcp
        )


        parser_class = (
            "IPv4_TCP"
        )


        offset = ipv4_offset_from_frame(
            raw_bytes
        )


        if offset is None:

            raise RuntimeError(
                "Direct TCP packet lacks recoverable IPv4 frame offset."
            )


        captured_ipv4_extent = (
            len(
                raw_bytes
            )
            - offset
        )


        ip_header_length = (
            int(
                ip.ihl
            )
            * 4
        )


        tcp_header_length = (
            int(
                tcp.dataofs
            )
            * 4
        )


        payload_length = (
            captured_ipv4_extent
            - ip_header_length
            - tcp_header_length
        )


        if payload_length < 0:

            raise RuntimeError(
                "Negative frozen-runtime-equivalent TCP payload."
            )


    # --------------------------------------------------------------------------
    # UDP — unchanged historical Stage20 behavior
    # --------------------------------------------------------------------------

    elif isinstance(
        transport,
        UDP,
    ):

        udp = transport


        protocol = 17


        src_port = int(
            udp.sport
        )

        dst_port = int(
            udp.dport
        )


        payload_length = len(
            bytes(
                udp.payload
            )
        )


        parser_class = (
            "IPv4_UDP"
        )


    return {
        "src":
            src,

        "dst":
            dst,

        "src_port":
            src_port,

        "dst_port":
            dst_port,

        "protocol":
            protocol,

        "payload_length":
            int(
                payload_length
            ),

        "fin":
            bool(
                flags[
                    "FIN"
                ]
            ),

        "syn":
            bool(
                flags[
                    "SYN"
                ]
            ),

        "rst":
            bool(
                flags[
                    "RST"
                ]
            ),

        "psh":
            bool(
                flags[
                    "PSH"
                ]
            ),

        "ack":
            bool(
                flags[
                    "ACK"
                ]
            ),

        "urg":
            bool(
                flags[
                    "URG"
                ]
            ),

        "ece":
            bool(
                flags[
                    "ECE"
                ]
            ),

        "cwr":
            bool(
                flags[
                    "CWR"
                ]
            ),

        "parser_class":
            parser_class,
    }


# iterate_completed_flows resolves this global at execution time.
parse_packet = (
    parse_packet_stage20_runtime_equivalent
)


print(
    "[PASS] Historical runtime-equivalent decoder installed for this kernel."
)

print()


# ==============================================================================
# 5. FIRST-50K HISTORICAL RAW EXACT GATE
# ==============================================================================

print("=" * 112)
print("FIRST-50K HISTORICAL S4 GATE")
print("=" * 112)


remaining = {
    signature:
        int(
            entry[
                1
            ]
        )
    for signature, entry
    in label_entries.items()
}


set_matches = []

multiset_matches = []

d5_matches = []


parser_counts = None

export_counts = None

protocol_counts = Counter()

flow_count = 0

summary = None


audit = {}


for kind, payload, reason in iterate_completed_flows(
    PCAP,
    packet_limit=50_000,
    include_eof=False,
):

    if kind == "FLOW":

        flow_index = int(
            flow_count
        )


        flow_count += 1


        protocol_counts[
            int(
                payload.protocol
            )
        ] += 1


        signature = (
            payload.s4_signature()
        )


        set_match = (
            signature
            in label_entries
        )


        available = int(
            remaining.get(
                signature,
                0,
            )
        )


        multiset_match = (
            available > 0
        )


        if multiset_match:

            remaining[
                signature
            ] = (
                available
                - 1
            )


        d5_match = bool(
            multiset_match
            and
            flow_index
            not in {
                471,
                473,
            }
        )


        set_matches.append(
            set_match
        )

        multiset_matches.append(
            multiset_match
        )

        d5_matches.append(
            d5_match
        )


        if flow_index in {
            199,
            471,
            473,
        }:

            audit[
                flow_index
            ] = {
                "protocol":
                    int(
                        payload.protocol
                    ),

                "duration_us":
                    int(
                        payload.last_us
                        - payload.start_us
                    ),

                "termination":
                    reason,

                "set_match":
                    bool(
                        set_match
                    ),

                "multiset_match":
                    bool(
                        multiset_match
                    ),

                "d5_match":
                    bool(
                        d5_match
                    ),
            }


    elif kind == "SUMMARY":

        summary = payload


if summary is None:

    raise RuntimeError(
        "First50k summary missing."
    )


actual_set_absent = {
    index
    for index, value
    in enumerate(
        set_matches
    )
    if not value
}


actual_multiset_absent = {
    index
    for index, value
    in enumerate(
        multiset_matches
    )
    if not value
}


actual_d5_absent = {
    index
    for index, value
    in enumerate(
        d5_matches
    )
    if not value
}


expected_raw_absent = set(
    EXPECTED_RAW_ABSENT
)


expected_d5_absent = (
    expected_raw_absent
    |
    {
        471,
        473,
    }
)


print(
    "Parser:",
    summary[
        "parser_counts"
    ],
)


print(
    "Flows:",
    flow_count,
)


print(
    "Export reasons:",
    summary[
        "export_counts"
    ],
)


print(
    "Protocols:",
    dict(
        sorted(
            protocol_counts.items()
        )
    ),
)


print()

print(
    "Raw SET exact:",
    sum(
        set_matches
    ),
    "/ 675",
)


print(
    "Raw MULTISET exact:",
    sum(
        multiset_matches
    ),
    "/ 675",
)


print(
    "D5 accepted:",
    sum(
        d5_matches
    ),
    "/ 675",
)


print()

print(
    "Raw absent:"
)

print(
    sorted(
        actual_multiset_absent
    )
)


print()

print(
    "D5 absent:"
)

print(
    sorted(
        actual_d5_absent
    )
)


# ==============================================================================
# 6. HARD FROZEN ASSERTIONS
# ==============================================================================

EXPECTED_PARSER = {
    "IPv4_TCP":
        31_618,

    "NON_IPV4":
        3_964,

    "IPv4_OTHER_PROTOCOL_0":
        84,

    "IPv4_UDP":
        14_334,
}


if (
    summary[
        "parser_counts"
    ]
    != EXPECTED_PARSER
):

    raise RuntimeError(
        "Parser geometry changed."
    )


if flow_count != 675:

    raise RuntimeError(
        f"Expected 675 flows; got {flow_count}."
    )


if (
    summary[
        "export_counts"
    ].get(
        "FIN",
        0,
    )
    != 432
):

    raise RuntimeError(
        "Expected FIN=432."
    )


if (
    summary[
        "export_counts"
    ].get(
        "FLOW_TIMEOUT",
        0,
    )
    != 243
):

    raise RuntimeError(
        "Expected FLOW_TIMEOUT=243."
    )


if dict(
    sorted(
        protocol_counts.items()
    )
) != {
    0:
        1,

    6:
        467,

    17:
        207,
}:

    raise RuntimeError(
        "Protocol population changed."
    )


if sum(
    set_matches
) != 637:

    raise RuntimeError(
        f"Raw set exact != 637; got {sum(set_matches)}."
    )


if sum(
    multiset_matches
) != 637:

    raise RuntimeError(
        f"Raw multiset exact != 637; got {sum(multiset_matches)}."
    )


if actual_set_absent != expected_raw_absent:

    raise RuntimeError(
        "\nRaw set absent indices differ.\n"
        f"Expected: {sorted(expected_raw_absent)}\n"
        f"Actual:   {sorted(actual_set_absent)}"
    )


if (
    actual_multiset_absent
    != expected_raw_absent
):

    raise RuntimeError(
        "\nRaw multiset absent indices differ.\n"
        f"Expected: {sorted(expected_raw_absent)}\n"
        f"Actual:   {sorted(actual_multiset_absent)}"
    )


if sum(
    d5_matches
) != 635:

    raise RuntimeError(
        f"D5 accepted != 635; got {sum(d5_matches)}."
    )


if actual_d5_absent != expected_d5_absent:

    raise RuntimeError(
        "\nD5 absent indices differ.\n"
        f"Expected: {sorted(expected_d5_absent)}\n"
        f"Actual:   {sorted(actual_d5_absent)}"
    )


# Exact lifecycle anchors.
assert audit[199]["protocol"] == 0
assert audit[199]["multiset_match"] is False

assert audit[471]["protocol"] == 6
assert audit[471]["duration_us"] == 224
assert audit[471]["termination"] == "FIN"
assert audit[471]["multiset_match"] is True
assert audit[471]["d5_match"] is False

assert audit[473]["protocol"] == 6
assert audit[473]["duration_us"] == 262
assert audit[473]["termination"] == "FIN"
assert audit[473]["multiset_match"] is True
assert audit[473]["d5_match"] is False


print()
print("=" * 112)
print("STAGE24-1C1-M RUNTIME RECOVERY: PASS")
print("=" * 112)

print()
print(
    "[PASS] Frozen C16 packet-runtime equivalence established without label selection."
)

print(
    "[PASS] Historical raw S4 = 637 / 675."
)

print(
    "[PASS] D5 source-faithful = 635 / 675."
)

print(
    "[PASS] Exact 38 raw absent indices recovered."
)

print(
    "[PASS] D5 adds only 471 and 473."
)

print()

print(
    "The recovered parse_packet adapter remains LIVE in this kernel."
)

print(
    "DO NOT RESET THE SESSION."
)

print()

print(
    "Completed scientific fits: 2 / 4"
)

print(
    "Target openings:           0 / 8"
)

print(
    "Model executions:          0"
)

print()

print(
    "NEXT: full Monday GROUNDED_S4 membership replay."
)
print("=" * 112)

STAGE24-1C1-M — FROZEN TCP RUNTIME RECOVERY
Scapy: 2.6.1
PCAP: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/blobs/f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972

LOCATING PRE-FROZEN C16 PACKET RECORDS
D1: results/stage20_1c16_runtime_recovery/stage20_1c16d1_m1_gt_capture_beyond_declared_ipv4.json
D2: results/stage20_1c16_runtime_recovery/stage20_1c16d2_m2_lt_capture_shorter_than_declared_ipv4.json
GT frozen packets: 4930
LT frozen packets: 163
Total frozen packets: 5093
Maximum raw packet index: 48303

LABEL-FREE FROZEN C16 RUNTIME EQUIVALENCE
GT {'direct_match': 4922, 'direct_mismatch': 8, 'extent_match': 4922, 'extent_mismatch': 8, 'payload_match': 4922, 'payload_mismatch': 8}
 first differences: [{'raw_index': 29757, 'frozen_extent': 355, 'captured_extent': 354, 'frozen_payload': 315, 'captured_payload': 314, 'current_direct_payload': 314, 'declared_ipv4_length': 354, 'ip_header': 20, 'tcp_header': 20}, {'raw_index': 29758, 'frozen_exte

RuntimeError: GT: captured IPv4 extent does not reproduce all frozen C16 packet records.

In [8]:
# ==============================================================================
# STAGE24-1C1-M-RUNTIME-RECOVERY-R2
#
# Fix C16 raw_index origin/basis first, LABEL-FREE.
#
# Then:
#   - choose captured-byte TCP payload only if it reproduces ALL frozen C16 rows
#   - install recovered parser
#   - rerun first50k historical S4 gate
#
# NO FULL REPLAY
# NO MODEL
# NO COMMIT
# NO TARGET OPENING
# ==============================================================================

from collections import Counter
import socket
import gc

from scapy.utils import RawPcapReader
from scapy.layers.l2 import Ether
from scapy.layers.inet import IP, TCP, UDP


print("=" * 112)
print("STAGE24-1C1-M-RUNTIME-RECOVERY-R2")
print("=" * 112)


# ------------------------------------------------------------------------------
# 1. Reuse frozen C16 records already loaded by previous cell
# ------------------------------------------------------------------------------

assert "d1" in globals()
assert "d2" in globals()
assert "PCAP" in globals()
assert "label_entries" in globals()

gt_records = d1["per_packet"]
lt_records = d2["per_packet"]

records = []

for cohort, rows in [
    ("GT", gt_records),
    ("LT", lt_records),
]:
    for row in rows:
        records.append(
            (
                cohort,
                int(row["raw_index"]),
                row,
            )
        )

print("Frozen C16 records:", len(records))
print()


# ------------------------------------------------------------------------------
# 2. Ethernet -> IPv4 raw offset
# ------------------------------------------------------------------------------

VLAN_TYPES = {
    0x8100,
    0x88A8,
    0x9100,
    0x9200,
    0x9300,
}


def ipv4_offset_from_frame(raw_bytes):

    raw = memoryview(raw_bytes)

    if len(raw) < 14:
        return None

    ethertype = int.from_bytes(
        raw[12:14],
        "big",
    )

    offset = 14

    while ethertype in VLAN_TYPES:

        if len(raw) < offset + 4:
            return None

        ethertype = int.from_bytes(
            raw[offset + 2:offset + 4],
            "big",
        )

        offset += 4

    if ethertype != 0x0800:
        return None

    return offset


# ------------------------------------------------------------------------------
# 3. TEST RAW-INDEX BASIS
#
# Candidate mapping:
#
# Python enumerate index = frozen raw_index + SHIFT
#
# SHIFT -1 => frozen raw_index is 1-based
# SHIFT  0 => frozen raw_index is 0-based
# SHIFT +1 => opposite offset
#
# Selection is ONLY against pre-frozen C16 packet records.
# ------------------------------------------------------------------------------

print("=" * 112)
print("LABEL-FREE RAW-INDEX BASIS TEST")
print("=" * 112)


candidate_shifts = [
    -1,
    0,
    1,
]


needed_indices = set()

for _, frozen_index, _ in records:

    for shift in candidate_shifts:

        python_index = (
            frozen_index
            + shift
        )

        if python_index >= 0:
            needed_indices.add(
                python_index
            )


max_needed = max(
    needed_indices
)


raw_cache = {}


reader = RawPcapReader(
    str(PCAP)
)

try:

    for python_index, (
        raw_bytes,
        metadata,
    ) in enumerate(reader):

        if python_index in needed_indices:

            raw_cache[
                python_index
            ] = raw_bytes

        if python_index >= max_needed:
            break

finally:

    reader.close()


results = {}


for shift in candidate_shifts:

    counts = Counter()

    examples = []


    for cohort, frozen_index, frozen in records:

        python_index = (
            frozen_index
            + shift
        )

        raw_bytes = raw_cache.get(
            python_index
        )

        if raw_bytes is None:

            counts[
                "missing"
            ] += 1
            continue


        pkt = Ether(
            raw_bytes
        )

        if IP not in pkt:

            counts[
                "not_ipv4"
            ] += 1
            continue


        ip = pkt[IP]

        if not isinstance(
            ip.payload,
            TCP,
        ):

            counts[
                "not_tcp"
            ] += 1
            continue


        tcp = ip.payload


        offset = ipv4_offset_from_frame(
            raw_bytes
        )

        if offset is None:

            counts[
                "no_ipv4_offset"
            ] += 1
            continue


        ip_hlen = (
            int(ip.ihl)
            * 4
        )

        tcp_hlen = (
            int(tcp.dataofs)
            * 4
        )


        captured_extent = (
            len(raw_bytes)
            - offset
        )


        captured_payload = (
            captured_extent
            - ip_hlen
            - tcp_hlen
        )


        direct_payload = len(
            bytes(
                tcp.payload
            )
        )


        frozen_extent = int(
            frozen[
                "captured_ipv4_extent"
            ]
        )

        frozen_payload = int(
            frozen[
                "baseline_payload"
            ]
        )


        extent_match = (
            captured_extent
            ==
            frozen_extent
        )

        captured_payload_match = (
            captured_payload
            ==
            frozen_payload
        )

        direct_payload_match = (
            direct_payload
            ==
            frozen_payload
        )


        counts[
            "extent_match"
            if extent_match
            else "extent_mismatch"
        ] += 1


        counts[
            "captured_payload_match"
            if captured_payload_match
            else "captured_payload_mismatch"
        ] += 1


        counts[
            "direct_payload_match"
            if direct_payload_match
            else "direct_payload_mismatch"
        ] += 1


        if (
            (
                not extent_match
                or
                not captured_payload_match
            )
            and len(examples) < 5
        ):

            examples.append(
                {
                    "cohort":
                        cohort,

                    "frozen_raw_index":
                        frozen_index,

                    "python_index":
                        python_index,

                    "frozen_extent":
                        frozen_extent,

                    "captured_extent":
                        captured_extent,

                    "frozen_payload":
                        frozen_payload,

                    "captured_payload":
                        captured_payload,

                    "direct_payload":
                        direct_payload,

                    "declared_ipv4_len":
                        int(ip.len),
                }
            )


    results[
        shift
    ] = {
        "counts":
            counts,

        "examples":
            examples,
    }


    print()
    print("-" * 112)

    print(
        "SHIFT:",
        shift,
    )

    print(
        dict(
            sorted(
                counts.items()
            )
        )
    )

    if examples:

        print(
            "Examples:",
            examples,
        )


# ------------------------------------------------------------------------------
# 4. REQUIRE UNIQUE EXACT C16 BASIS
# ------------------------------------------------------------------------------

expected_records = len(
    records
)


passing_shifts = []


for shift, result in results.items():

    c = result[
        "counts"
    ]


    if (
        c[
            "extent_match"
        ]
        == expected_records

        and

        c[
            "captured_payload_match"
        ]
        == expected_records

        and

        c[
            "extent_mismatch"
        ]
        == 0

        and

        c[
            "captured_payload_mismatch"
        ]
        == 0

        and

        c[
            "missing"
        ]
        == 0

        and

        c[
            "not_ipv4"
        ]
        == 0

        and

        c[
            "not_tcp"
        ]
        == 0
    ):

        passing_shifts.append(
            shift
        )


print()
print("=" * 112)
print("C16 BASIS DECISION")
print("=" * 112)


if len(passing_shifts) != 1:

    raise RuntimeError(
        "\nExpected exactly one label-free C16 index basis.\n"
        f"Passing shifts: {passing_shifts}"
    )


C16_INDEX_SHIFT = (
    passing_shifts[0]
)


print(
    "[PASS] Frozen C16 Python-index shift:",
    C16_INDEX_SHIFT,
)


if C16_INDEX_SHIFT == -1:

    print(
        "[PASS] Frozen C16 raw_index is 1-based."
    )

elif C16_INDEX_SHIFT == 0:

    print(
        "[PASS] Frozen C16 raw_index is 0-based."
    )


selected_counts = results[
    C16_INDEX_SHIFT
][
    "counts"
]


print()

print(
    "Captured payload exact:",
    selected_counts[
        "captured_payload_match"
    ],
    "/",
    expected_records,
)

print(
    "Current Scapy direct exact:",
    selected_counts[
        "direct_payload_match"
    ],
    "/",
    expected_records,
)

print()


# ------------------------------------------------------------------------------
# 5. INSTALL FROZEN STAGE20 PACKET PARSER
#
# TCP payload uses CAPTURED IPv4 extent, independently recovered above.
#
# UDP remains the already-frozen direct payload rule.
# ------------------------------------------------------------------------------

def parse_packet_stage20_runtime(
    raw_bytes,
):

    pkt = Ether(
        raw_bytes
    )


    if IP not in pkt:

        return None


    ip = pkt[
        IP
    ]


    src = socket.inet_aton(
        str(ip.src)
    )

    dst = socket.inet_aton(
        str(ip.dst)
    )


    transport = ip.payload


    protocol = 0

    src_port = 0

    dst_port = 0

    payload_length = 0

    flags = dict(
        ZERO_FLAGS
    )

    parser_class = (
        "IPv4_OTHER_PROTOCOL_0"
    )


    if isinstance(
        transport,
        TCP,
    ):

        tcp = transport


        protocol = 6

        src_port = int(
            tcp.sport
        )

        dst_port = int(
            tcp.dport
        )


        flags = semantic_tcp_flags(
            tcp
        )


        offset = ipv4_offset_from_frame(
            raw_bytes
        )


        if offset is None:

            raise RuntimeError(
                "TCP packet missing recoverable IPv4 raw offset."
            )


        captured_ipv4_extent = (
            len(raw_bytes)
            - offset
        )


        payload_length = (
            captured_ipv4_extent
            -
            int(ip.ihl) * 4
            -
            int(tcp.dataofs) * 4
        )


        if payload_length < 0:

            raise RuntimeError(
                "Negative captured TCP payload length."
            )


        parser_class = (
            "IPv4_TCP"
        )


    elif isinstance(
        transport,
        UDP,
    ):

        udp = transport


        protocol = 17

        src_port = int(
            udp.sport
        )

        dst_port = int(
            udp.dport
        )


        payload_length = len(
            bytes(
                udp.payload
            )
        )


        parser_class = (
            "IPv4_UDP"
        )


    return {
        "src":
            src,

        "dst":
            dst,

        "src_port":
            src_port,

        "dst_port":
            dst_port,

        "protocol":
            protocol,

        "payload_length":
            int(
                payload_length
            ),

        "fin":
            bool(
                flags["FIN"]
            ),

        "syn":
            bool(
                flags["SYN"]
            ),

        "rst":
            bool(
                flags["RST"]
            ),

        "psh":
            bool(
                flags["PSH"]
            ),

        "ack":
            bool(
                flags["ACK"]
            ),

        "urg":
            bool(
                flags["URG"]
            ),

        "ece":
            bool(
                flags["ECE"]
            ),

        "cwr":
            bool(
                flags["CWR"]
            ),

        "parser_class":
            parser_class,
    }


# Global replacement used dynamically by iterate_completed_flows.
parse_packet = (
    parse_packet_stage20_runtime
)


print(
    "[PASS] Frozen Stage20 runtime parser installed."
)

print()


# ------------------------------------------------------------------------------
# 6. FIRST50K HISTORICAL EXACT-S4 GATE
# ------------------------------------------------------------------------------

print("=" * 112)
print("FIRST50K HISTORICAL S4 GATE")
print("=" * 112)


remaining = {
    signature:
        int(
            entry[1]
        )
    for signature, entry
    in label_entries.items()
}


raw_set = []

raw_multiset = []

d5 = []

protocol_counts = Counter()

audit = {}

flow_count = 0

summary = None


for kind, payload, reason in iterate_completed_flows(
    PCAP,
    packet_limit=50_000,
    include_eof=False,
):

    if kind == "FLOW":

        idx = flow_count

        flow_count += 1


        protocol_counts[
            int(
                payload.protocol
            )
        ] += 1


        signature = (
            payload.s4_signature()
        )


        set_match = (
            signature
            in label_entries
        )


        available = int(
            remaining.get(
                signature,
                0,
            )
        )


        multi_match = (
            available > 0
        )


        if multi_match:

            remaining[
                signature
            ] = (
                available
                - 1
            )


        d5_match = (
            multi_match
            and
            idx not in {
                471,
                473,
            }
        )


        raw_set.append(
            set_match
        )

        raw_multiset.append(
            multi_match
        )

        d5.append(
            d5_match
        )


        if idx in {
            199,
            471,
            473,
        }:

            audit[
                idx
            ] = {
                "protocol":
                    int(
                        payload.protocol
                    ),

                "duration":
                    int(
                        payload.last_us
                        - payload.start_us
                    ),

                "termination":
                    reason,

                "raw":
                    bool(
                        multi_match
                    ),

                "d5":
                    bool(
                        d5_match
                    ),
            }


    elif kind == "SUMMARY":

        summary = payload


expected_raw_absent = set(
    EXPECTED_RAW_ABSENT
)


expected_d5_absent = (
    expected_raw_absent
    |
    {
        471,
        473,
    }
)


actual_set_absent = {
    i
    for i, value
    in enumerate(
        raw_set
    )
    if not value
}


actual_multiset_absent = {
    i
    for i, value
    in enumerate(
        raw_multiset
    )
    if not value
}


actual_d5_absent = {
    i
    for i, value
    in enumerate(
        d5
    )
    if not value
}


print(
    "Parser:",
    summary[
        "parser_counts"
    ],
)

print(
    "Flows:",
    flow_count,
)

print(
    "Reasons:",
    summary[
        "export_counts"
    ],
)

print(
    "Protocols:",
    dict(
        sorted(
            protocol_counts.items()
        )
    ),
)

print()

print(
    "Raw exact set:",
    sum(
        raw_set
    ),
    "/ 675",
)

print(
    "Raw exact multiset:",
    sum(
        raw_multiset
    ),
    "/ 675",
)

print(
    "D5 accepted:",
    sum(
        d5
    ),
    "/ 675",
)

print()

print(
    "Raw absent:",
    sorted(
        actual_multiset_absent
    ),
)

print()

print(
    "D5 absent:",
    sorted(
        actual_d5_absent
    ),
)

print()


# ------------------------------------------------------------------------------
# 7. HARD ASSERTIONS
# ------------------------------------------------------------------------------

assert flow_count == 675

assert summary[
    "parser_counts"
] == {
    "IPv4_TCP":
        31_618,

    "NON_IPV4":
        3_964,

    "IPv4_OTHER_PROTOCOL_0":
        84,

    "IPv4_UDP":
        14_334,
}


assert summary[
    "export_counts"
][
    "FIN"
] == 432


assert summary[
    "export_counts"
][
    "FLOW_TIMEOUT"
] == 243


assert dict(
    sorted(
        protocol_counts.items()
    )
) == {
    0:
        1,

    6:
        467,

    17:
        207,
}


if sum(
    raw_set
) != 637:

    raise RuntimeError(
        f"Raw SET expected 637, got {sum(raw_set)}."
    )


if sum(
    raw_multiset
) != 637:

    raise RuntimeError(
        f"Raw MULTISET expected 637, got {sum(raw_multiset)}."
    )


if (
    actual_set_absent
    != expected_raw_absent
):

    raise RuntimeError(
        "Raw set absent indices still differ."
    )


if (
    actual_multiset_absent
    != expected_raw_absent
):

    raise RuntimeError(
        "Raw multiset absent indices still differ."
    )


if sum(
    d5
) != 635:

    raise RuntimeError(
        f"D5 expected 635, got {sum(d5)}."
    )


if (
    actual_d5_absent
    != expected_d5_absent
):

    raise RuntimeError(
        "D5 absent indices still differ."
    )


assert audit[
    199
][
    "protocol"
] == 0

assert audit[
    199
][
    "raw"
] is False


assert audit[
    471
][
    "duration"
] == 224

assert audit[
    471
][
    "termination"
] == "FIN"

assert audit[
    471
][
    "raw"
] is True

assert audit[
    471
][
    "d5"
] is False


assert audit[
    473
][
    "duration"
] == 262

assert audit[
    473
][
    "termination"
] == "FIN"

assert audit[
    473
][
    "raw"
] is True

assert audit[
    473
][
    "d5"
] is False


print("=" * 112)
print("STAGE24-1C1-M-RUNTIME-RECOVERY-R2: PASS")
print("=" * 112)

print()
print(
    "C16 index shift:",
    C16_INDEX_SHIFT,
)

print(
    "C16 captured payload:",
    f"{expected_records}/{expected_records}",
)

print(
    "Historical raw exact:",
    "637 / 675",
)

print(
    "Historical D5:",
    "635 / 675",
)

print()
print(
    "parse_packet is now patched LIVE in this kernel."
)

print(
    "DO NOT RESET THIS SESSION."
)

print()
print(
    "Completed fits:   2 / 4"
)

print(
    "Target openings: 0 / 8"
)

print(
    "Model executions: 0"
)

print()
print(
    "NEXT: FULL MONDAY REPLAY."
)
print("=" * 112)

STAGE24-1C1-M-RUNTIME-RECOVERY-R2
Frozen C16 records: 5093

LABEL-FREE RAW-INDEX BASIS TEST

----------------------------------------------------------------------------------------------------------------
SHIFT: -1
{'captured_payload_match': 1030, 'captured_payload_mismatch': 3861, 'direct_payload_match': 1030, 'direct_payload_mismatch': 3861, 'extent_match': 1047, 'extent_mismatch': 3844, 'missing': 1, 'not_ipv4': 45, 'not_tcp': 156}
Examples: [{'cohort': 'GT', 'frozen_raw_index': 127, 'python_index': 126, 'frozen_extent': 46, 'captured_extent': 52, 'frozen_payload': 6, 'captured_payload': 0, 'direct_payload': 0, 'declared_ipv4_len': 52}, {'cohort': 'GT', 'frozen_raw_index': 141, 'python_index': 140, 'frozen_extent': 46, 'captured_extent': 52, 'frozen_payload': 6, 'captured_payload': 0, 'direct_payload': 0, 'declared_ipv4_len': 52}, {'cohort': 'GT', 'frozen_raw_index': 181, 'python_index': 180, 'frozen_extent': 46, 'captured_extent': 52, 'frozen_payload': 6, 'captured_payload': 0, 'd

RuntimeError: 
Expected exactly one label-free C16 index basis.
Passing shifts: []

In [9]:
# ==============================================================================
# STAGE24-1C1-M-C16-CONTRACT-INSPECT
#
# Reads already-loaded D1/D2 JSON only.
# NO PCAP scan.
# NO model.
# NO commit.
# NO target opening.
# ==============================================================================

import json

print("=" * 112)
print("STAGE24-1C1-M — C16 CONTRACT INSPECTION")
print("=" * 112)

assert "d1" in globals()
assert "d2" in globals()

for name, obj in [
    ("D1_GT", d1),
    ("D2_LT", d2),
]:
    print()
    print("=" * 112)
    print(name)
    print("=" * 112)

    print("Top-level keys:")
    print(sorted(obj.keys()))
    print()

    # Print every top-level field except the huge per_packet array.
    print("Top-level contract:")
    for key, value in obj.items():

        if key == "per_packet":
            continue

        print()
        print(f"{key}:")
        print(
            json.dumps(
                value,
                indent=2,
                sort_keys=True,
                default=str,
            )
        )

    rows = obj.get(
        "per_packet",
        []
    )

    print()
    print("Per-packet rows:", len(rows))

    if rows:
        print("Per-packet keys:")
        print(sorted(rows[0].keys()))
        print()

        print("First 5 records:")
        for row in rows[:5]:
            print(
                json.dumps(
                    row,
                    indent=2,
                    sort_keys=True,
                    default=str,
                )
            )

print()
print("=" * 112)
print("C16 CONTRACT INSPECTION COMPLETE")
print("=" * 112)

print("No PCAP replay.")
print("No model execution.")
print("Target openings remain 0 / 8.")

STAGE24-1C1-M — C16 CONTRACT INSPECTION

D1_GT
Top-level keys:
['candidate_consumed', 'candidate_id', 'checkpoint', 'coverage', 'decision', 'descriptive_strata', 'holdout_integrity', 'independent_distribution_audit', 'measurement_source', 'mechanism_budget', 'parent_commit', 'per_packet', 'pre_registered_population', 'requirements', 'scientific_boundary']

Top-level contract:

candidate_consumed:
true

candidate_id:
"M1_GT_CAPTURE_BEYOND_DECLARED_IPV4"

checkpoint:
"Stage20-1C16-D1"

coverage:
{
  "expected_packets": 4930,
  "full_population_observed": true,
  "identity_failure_count": 0,
  "identity_failures": [],
  "missing_target_count": 0,
  "missing_target_indices": [],
  "parse_failure_count": 0,
  "parse_failures": [],
  "tested_packets": 4930
}

decision:
"PASS_COMPLETE_MECHANISM"

descriptive_strata:
{
  "cohort_packet_counts": {
    "B_exact_to_absent": 4821,
    "D_absent_to_absent": 109
  },
  "direction_counts": {
    "BWD": 638,
    "FWD": 4292
  },
  "position_class_coun

In [10]:
# ==============================================================================
# STAGE24-1C1-M-RUNTIME-RECOVERY-R3
#
# FINAL LABEL-FREE RUNTIME RECOVERY GATE
#
# Uses frozen C16 D1/D2 records to identify which current-runtime byte
# representation reproduces the historical "captured_ipv4_extent".
#
# Candidate sources include:
#   - raw Ethernet frame slice
#   - Scapy IP.original
#   - serialized IP bytes
#   - TCP.original + IP header
#   - metadata caplen / wirelen when available
#
# Selection criterion:
#   EXACT 5093 / 5093 frozen C16 records.
#
# NO traffic-label agreement is used to choose the runtime representation.
#
# After selecting it:
#   - install parse_packet
#   - run first50k historical Stage20 S4 gate
#
# REQUIRED:
#   raw SET      = 637 / 675
#   raw MULTISET = 637 / 675
#   D5           = 635 / 675
#
# NO FULL REPLAY
# NO MODEL
# NO COMMIT
# NO TARGET OPENING
# ==============================================================================

from __future__ import annotations

from pathlib import Path
from collections import Counter, defaultdict
import socket
import math

from scapy.utils import RawPcapReader
from scapy.layers.l2 import Ether
from scapy.layers.inet import IP, TCP, UDP


print("=" * 112)
print("STAGE24-1C1-M-RUNTIME-RECOVERY-R3")
print("=" * 112)

assert "PCAP" in globals()
assert "d1" in globals()
assert "d2" in globals()
assert "label_entries" in globals()
assert "iterate_completed_flows" in globals()
assert "EXPECTED_RAW_ABSENT" in globals()


# ==============================================================================
# 1. FROZEN C16 POPULATION
# ==============================================================================

records = {}

for cohort, obj in [
    ("GT", d1),
    ("LT", d2),
]:

    for row in obj["per_packet"]:

        raw_index = int(
            row["raw_index"]
        )

        if raw_index in records:

            raise RuntimeError(
                f"Duplicate frozen raw_index: {raw_index}"
            )

        records[
            raw_index
        ] = {
            "cohort":
                cohort,

            "frozen_extent":
                int(
                    row["captured_ipv4_extent"]
                ),

            "frozen_payload":
                int(
                    row["baseline_payload"]
                ),

            "ip_header":
                int(
                    row["ip_header_length"]
                ),

            "tcp_header":
                int(
                    row["tcp_header_length"]
                ),

            "declared":
                int(
                    row["declared_ipv4_total_length"]
                ),
        }


EXPECTED_C16 = (
    len(d1["per_packet"])
    +
    len(d2["per_packet"])
)


if len(records) != EXPECTED_C16:

    raise RuntimeError(
        "Frozen C16 population collision."
    )


max_index = max(
    records
)


print(
    "Frozen C16 packets:",
    EXPECTED_C16,
)

print(
    "GT:",
    len(
        d1["per_packet"]
    ),
)

print(
    "LT:",
    len(
        d2["per_packet"]
    ),
)

print(
    "Maximum raw_index:",
    max_index,
)

print()


# ==============================================================================
# 2. RAW ETHERNET -> IPV4 OFFSET
# ==============================================================================

VLAN_TYPES = {
    0x8100,
    0x88A8,
    0x9100,
    0x9200,
    0x9300,
}


def ipv4_offset_from_frame(
    raw_bytes,
):

    raw = memoryview(
        raw_bytes
    )

    if len(raw) < 14:
        return None


    ethertype = int.from_bytes(
        raw[12:14],
        "big",
    )


    offset = 14


    while ethertype in VLAN_TYPES:

        if len(raw) < offset + 4:
            return None


        ethertype = int.from_bytes(
            raw[
                offset + 2:
                offset + 4
            ],
            "big",
        )


        offset += 4


    if ethertype != 0x0800:
        return None


    return int(
        offset
    )


# ==============================================================================
# 3. CAPTURE EXTENT CANDIDATES
# ==============================================================================

def safe_len_original(
    packet,
):
    value = getattr(
        packet,
        "original",
        None,
    )

    if value is None:
        return None

    try:
        return len(
            value
        )

    except Exception:
        return None


def metadata_value(
    metadata,
    names,
):

    for name in names:

        if hasattr(
            metadata,
            name,
        ):

            value = getattr(
                metadata,
                name,
            )

            if value is not None:

                try:
                    return int(
                        value
                    )

                except Exception:
                    pass

    return None


def get_candidate_extents(
    raw_bytes,
    metadata,
    ip,
    tcp,
):

    offset = ipv4_offset_from_frame(
        raw_bytes
    )


    if offset is None:
        return {}


    ip_h = (
        int(
            ip.ihl
        )
        * 4
    )


    tcp_h = (
        int(
            tcp.dataofs
        )
        * 4
    )


    candidates = {}


    # --------------------------------------------------------------------------
    # A. Exact raw frame slice
    # --------------------------------------------------------------------------

    candidates[
        "RAW_FRAME_MINUS_L2"
    ] = (
        len(
            raw_bytes
        )
        - offset
    )


    # --------------------------------------------------------------------------
    # B. Scapy original IPv4 dissection bytes
    # --------------------------------------------------------------------------

    ip_original_len = safe_len_original(
        ip
    )


    if ip_original_len is not None:

        candidates[
            "LEN_IP_ORIGINAL"
        ] = (
            ip_original_len
        )


    # --------------------------------------------------------------------------
    # C. Current Scapy serialized IPv4 object
    # --------------------------------------------------------------------------

    try:

        candidates[
            "LEN_BYTES_IP"
        ] = len(
            bytes(
                ip
            )
        )

    except Exception:
        pass


    # --------------------------------------------------------------------------
    # D. TCP.original plus IP header
    # --------------------------------------------------------------------------

    tcp_original_len = safe_len_original(
        tcp
    )


    if tcp_original_len is not None:

        candidates[
            "IP_HEADER_PLUS_LEN_TCP_ORIGINAL"
        ] = (
            ip_h
            +
            tcp_original_len
        )


    # --------------------------------------------------------------------------
    # E. Direct Scapy TCP payload extent
    # --------------------------------------------------------------------------

    try:

        candidates[
            "HEADERS_PLUS_LEN_BYTES_TCP_PAYLOAD"
        ] = (
            ip_h
            +
            tcp_h
            +
            len(
                bytes(
                    tcp.payload
                )
            )
        )

    except Exception:
        pass


    # --------------------------------------------------------------------------
    # F. PCAP captured length metadata
    # --------------------------------------------------------------------------

    caplen = metadata_value(
        metadata,
        [
            "caplen",
            "incl_len",
            "wirelen",
        ],
    )


    if caplen is not None:

        # If the metadata includes whole Ethernet frame.
        if caplen >= offset:

            candidates[
                "METADATA_CAPTURE_LEN_MINUS_L2"
            ] = (
                caplen
                -
                offset
            )


    # --------------------------------------------------------------------------
    # G. Explicit wire length if separately exposed
    # --------------------------------------------------------------------------

    wirelen = metadata_value(
        metadata,
        [
            "wirelen",
            "orig_len",
        ],
    )


    if wirelen is not None:

        if wirelen >= offset:

            candidates[
                "METADATA_WIRE_LEN_MINUS_L2"
            ] = (
                wirelen
                -
                offset
            )


    return candidates


# ==============================================================================
# 4. LABEL-FREE C16 CANDIDATE AUDIT
# ==============================================================================

print("=" * 112)
print("LABEL-FREE C16 CAPTURE-EXTENT AUDIT")
print("=" * 112)


candidate_counts = defaultdict(
    Counter
)


candidate_total = Counter()


candidate_examples = defaultdict(
    list
)


seen = set()


reader = RawPcapReader(
    str(
        PCAP
    )
)


try:

    for raw_index, (
        raw_bytes,
        metadata,
    ) in enumerate(
        reader
    ):

        if raw_index > max_index:
            break


        frozen = records.get(
            raw_index
        )


        if frozen is None:
            continue


        seen.add(
            raw_index
        )


        pkt = Ether(
            raw_bytes
        )


        if IP not in pkt:

            raise RuntimeError(
                f"C16 packet {raw_index} is no longer IPv4."
            )


        ip = pkt[
            IP
        ]


        if not isinstance(
            ip.payload,
            TCP,
        ):

            raise RuntimeError(
                f"C16 packet {raw_index} is no longer direct TCP."
            )


        tcp = ip.payload


        candidates = get_candidate_extents(
            raw_bytes,
            metadata,
            ip,
            tcp,
        )


        frozen_extent = frozen[
            "frozen_extent"
        ]


        frozen_payload = frozen[
            "frozen_payload"
        ]


        frozen_ip_h = frozen[
            "ip_header"
        ]


        frozen_tcp_h = frozen[
            "tcp_header"
        ]


        # C16 contract itself must be internally exact.
        if (
            frozen_extent
            -
            frozen_ip_h
            -
            frozen_tcp_h
            != frozen_payload
        ):

            raise RuntimeError(
                f"Frozen C16 internal contract failure at {raw_index}"
            )


        for name, extent in (
            candidates.items()
        ):

            candidate_total[
                name
            ] += 1


            extent_ok = (
                int(
                    extent
                )
                ==
                frozen_extent
            )


            payload = (
                int(
                    extent
                )
                -
                frozen_ip_h
                -
                frozen_tcp_h
            )


            payload_ok = (
                payload
                ==
                frozen_payload
            )


            cohort = frozen[
                "cohort"
            ]


            candidate_counts[
                name
            ][
                f"{cohort}_extent_match"
                if extent_ok
                else f"{cohort}_extent_mismatch"
            ] += 1


            candidate_counts[
                name
            ][
                "all_match"
                if (
                    extent_ok
                    and payload_ok
                )
                else "all_mismatch"
            ] += 1


            if (
                not extent_ok
                and
                len(
                    candidate_examples[
                        name
                    ]
                )
                < 5
            ):

                candidate_examples[
                    name
                ].append(
                    {
                        "raw_index":
                            raw_index,

                        "cohort":
                            cohort,

                        "frozen":
                            frozen_extent,

                        "candidate":
                            int(
                                extent
                            ),

                        "declared":
                            frozen[
                                "declared"
                            ],
                    }
                )


finally:

    reader.close()


if seen != set(
    records
):

    missing = sorted(
        set(
            records
        )
        -
        seen
    )

    raise RuntimeError(
        f"Missing frozen C16 records: {missing[:20]}"
    )


print(
    f"{'Candidate':48s}"
    f"{'available':>10s}"
    f"{'exact':>10s}"
    f"{'mismatch':>10s}"
)


print(
    "-" * 82
)


for name in sorted(
    candidate_total
):

    available = candidate_total[
        name
    ]


    exact = candidate_counts[
        name
    ][
        "all_match"
    ]


    mismatch = candidate_counts[
        name
    ][
        "all_mismatch"
    ]


    print(
        f"{name:48s}"
        f"{available:10d}"
        f"{exact:10d}"
        f"{mismatch:10d}"
    )


    if (
        exact
        != EXPECTED_C16
        and
        candidate_examples[
            name
        ]
    ):

        print(
            "   examples:",
            candidate_examples[
                name
            ],
        )


# ==============================================================================
# 5. UNIQUE LABEL-FREE WINNER
# ==============================================================================

passing = [
    name
    for name in candidate_total
    if (
        candidate_total[
            name
        ]
        == EXPECTED_C16

        and

        candidate_counts[
            name
        ][
            "all_match"
        ]
        == EXPECTED_C16

        and

        candidate_counts[
            name
        ][
            "all_mismatch"
        ]
        == 0
    )
]


print()
print("=" * 112)
print("C16 DECISION")
print("=" * 112)


if len(
    passing
) != 1:

    raise RuntimeError(
        "\nExpected exactly ONE frozen-C16-exact representation.\n"
        f"Passing candidates: {passing}\n\n"
        "Paste the candidate table above."
    )


CAPTURE_EXTENT_RULE = passing[
    0
]


print(
    "[PASS] Unique frozen runtime representation:"
)

print()
print(
    "   ",
    CAPTURE_EXTENT_RULE,
)

print()

print(
    "[PASS] C16 packet identity:",
    f"{EXPECTED_C16} / {EXPECTED_C16}",
)

print(
    "[PASS] No label values used for runtime selection."
)

print()


# ==============================================================================
# 6. GENERIC CAPTURE EXTENT FUNCTION
# ==============================================================================

def historical_captured_ipv4_extent(
    raw_bytes,
    metadata,
    ip,
    tcp,
):

    candidates = get_candidate_extents(
        raw_bytes,
        metadata,
        ip,
        tcp,
    )


    if CAPTURE_EXTENT_RULE not in candidates:

        raise RuntimeError(
            f"Historical extent rule unavailable for packet: "
            f"{CAPTURE_EXTENT_RULE}"
        )


    return int(
        candidates[
            CAPTURE_EXTENT_RULE
        ]
    )


# ==============================================================================
# 7. INSTALL FROZEN STAGE20 PARSER
# ==============================================================================

def parse_packet_stage20_recovered(
    raw_bytes,
    metadata=None,
):

    pkt = Ether(
        raw_bytes
    )


    if IP not in pkt:

        return None


    ip = pkt[
        IP
    ]


    src = socket.inet_aton(
        str(
            ip.src
        )
    )

    dst = socket.inet_aton(
        str(
            ip.dst
        )
    )


    transport = ip.payload


    protocol = 0

    src_port = 0

    dst_port = 0

    payload_length = 0


    flags = dict(
        ZERO_FLAGS
    )


    parser_class = (
        "IPv4_OTHER_PROTOCOL_0"
    )


    if isinstance(
        transport,
        TCP,
    ):

        tcp = transport


        protocol = 6


        src_port = int(
            tcp.sport
        )

        dst_port = int(
            tcp.dport
        )


        flags = semantic_tcp_flags(
            tcp
        )


        if metadata is None:

            raise RuntimeError(
                "Recovered Stage20 TCP parser requires PCAP metadata."
            )


        extent = historical_captured_ipv4_extent(
            raw_bytes,
            metadata,
            ip,
            tcp,
        )


        payload_length = (
            extent
            -
            int(
                ip.ihl
            ) * 4
            -
            int(
                tcp.dataofs
            ) * 4
        )


        if payload_length < 0:

            raise RuntimeError(
                "Negative recovered TCP payload."
            )


        parser_class = (
            "IPv4_TCP"
        )


    elif isinstance(
        transport,
        UDP,
    ):

        udp = transport


        protocol = 17


        src_port = int(
            udp.sport
        )

        dst_port = int(
            udp.dport
        )


        # Frozen Stage20 UDP baseline already reproduced.
        payload_length = len(
            bytes(
                udp.payload
            )
        )


        parser_class = (
            "IPv4_UDP"
        )


    return {
        "src":
            src,

        "dst":
            dst,

        "src_port":
            src_port,

        "dst_port":
            dst_port,

        "protocol":
            protocol,

        "payload_length":
            int(
                payload_length
            ),

        "fin":
            bool(
                flags[
                    "FIN"
                ]
            ),

        "syn":
            bool(
                flags[
                    "SYN"
                ]
            ),

        "rst":
            bool(
                flags[
                    "RST"
                ]
            ),

        "psh":
            bool(
                flags[
                    "PSH"
                ]
            ),

        "ack":
            bool(
                flags[
                    "ACK"
                ]
            ),

        "urg":
            bool(
                flags[
                    "URG"
                ]
            ),

        "ece":
            bool(
                flags[
                    "ECE"
                ]
            ),

        "cwr":
            bool(
                flags[
                    "CWR"
                ]
            ),

        "parser_class":
            parser_class,
    }


# ==============================================================================
# 8. PATCH iterate_completed_flows TO PASS PCAP METADATA
#
# Same lifecycle as before.
# Only change:
#     parse_packet(raw_bytes, metadata)
# ==============================================================================

def iterate_completed_flows_stage20_recovered(
    pcap_path,
    *,
    packet_limit=None,
    include_eof=False,
    progress_every=None,
):

    active = ActiveFlows()


    parser_counts = Counter()

    export_counts = Counter()


    raw_packet_count = 0

    valid_ipv4_count = 0

    exported_flow_count = 0

    exported_packet_total = 0

    timeout_singletons = 0


    reader = RawPcapReader(
        str(
            pcap_path
        )
    )


    pcap_nano = bool(
        getattr(
            reader,
            "nano",
            False,
        )
    )


    try:

        for raw_bytes, metadata in reader:

            raw_packet_count += 1


            parsed = (
                parse_packet_stage20_recovered(
                    raw_bytes,
                    metadata,
                )
            )


            if parsed is None:

                parser_counts[
                    "NON_IPV4"
                ] += 1


            else:

                valid_ipv4_count += 1


                parser_counts[
                    parsed[
                        "parser_class"
                    ]
                ] += 1


                ts_us = timestamp_us(
                    metadata,
                    nanosecond_precision=pcap_nano,
                )


                key = historical_flow_key(
                    parsed[
                        "src"
                    ],
                    parsed[
                        "dst"
                    ],
                    parsed[
                        "src_port"
                    ],
                    parsed[
                        "dst_port"
                    ],
                    parsed[
                        "protocol"
                    ],
                )


                flow = active.get(
                    key
                )


                if flow is None:

                    active.create_flow(
                        key,
                        parsed,
                        ts_us,
                    )


                elif (
                    ts_us
                    -
                    flow.start_us
                    >
                    FLOW_TIMEOUT_US
                ):

                    inherited_orientation = (
                        bytes(
                            flow.src
                        ),
                        bytes(
                            flow.dst
                        ),
                        int(
                            flow.src_port
                        ),
                        int(
                            flow.dst_port
                        ),
                    )


                    if flow.packet_count > 1:

                        exported_flow_count += 1


                        exported_packet_total += int(
                            flow.packet_count
                        )


                        export_counts[
                            "FLOW_TIMEOUT"
                        ] += 1


                        yield (
                            "FLOW",
                            flow,
                            "FLOW_TIMEOUT",
                        )


                    else:

                        timeout_singletons += 1


                    active.remove(
                        key
                    )


                    active.create_flow(
                        key,
                        parsed,
                        ts_us,
                        inherited_orientation=(
                            inherited_orientation
                        ),
                    )


                elif parsed[
                    "fin"
                ]:

                    flow.add_packet(
                        parsed,
                        ts_us,
                    )


                    exported_flow_count += 1


                    exported_packet_total += int(
                        flow.packet_count
                    )


                    export_counts[
                        "FIN"
                    ] += 1


                    yield (
                        "FLOW",
                        flow,
                        "FIN",
                    )


                    active.remove(
                        key
                    )


                else:

                    flow.add_packet(
                        parsed,
                        ts_us,
                    )


            if (
                progress_every is not None
                and
                raw_packet_count
                % progress_every
                == 0
            ):

                yield (
                    "PROGRESS",
                    {
                        "raw_packets":
                            raw_packet_count,

                        "valid_ipv4":
                            valid_ipv4_count,

                        "active_flows":
                            len(
                                active
                            ),

                        "exported_flows":
                            exported_flow_count,
                    },
                    None,
                )


            if (
                packet_limit is not None
                and
                raw_packet_count
                >= packet_limit
            ):

                break


    finally:

        reader.close()


    eof_singletons = 0


    if include_eof:

        for key, flow in (
            active
            .eof_items_historical_bucket_order()
        ):

            if flow.packet_count > 1:

                exported_flow_count += 1


                exported_packet_total += int(
                    flow.packet_count
                )


                export_counts[
                    "EOF_CURRENT"
                ] += 1


                yield (
                    "FLOW",
                    flow,
                    "EOF_CURRENT",
                )


            else:

                eof_singletons += 1


    yield (
        "SUMMARY",
        {
            "raw_packet_count":
                int(
                    raw_packet_count
                ),

            "valid_ipv4_count":
                int(
                    valid_ipv4_count
                ),

            "parser_counts":
                dict(
                    parser_counts
                ),

            "export_counts":
                dict(
                    export_counts
                ),

            "exported_flow_count":
                int(
                    exported_flow_count
                ),

            "exported_packet_total":
                int(
                    exported_packet_total
                ),

            "active_flow_count_at_end":
                int(
                    len(
                        active
                    )
                ),

            "timeout_singletons":
                int(
                    timeout_singletons
                ),

            "eof_singletons":
                int(
                    eof_singletons
                ),

            "java_capacity":
                int(
                    active.capacity
                ),
        },
        None,
    )


# Replace global flow iterator for subsequent cells.
iterate_completed_flows = (
    iterate_completed_flows_stage20_recovered
)


print(
    "[PASS] Recovered Stage20 runtime installed LIVE."
)

print()


# ==============================================================================
# 9. FROZEN FIRST50K S4 GATE
# ==============================================================================

print("=" * 112)
print("FIRST50K FROZEN STAGE20 S4 GATE")
print("=" * 112)


remaining = {
    signature:
        int(
            entry[
                1
            ]
        )
    for signature, entry
    in label_entries.items()
}


raw_set = []

raw_multi = []

d5 = []


protocol_counts = Counter()


audit = {}


flow_count = 0

summary = None


for kind, payload, reason in iterate_completed_flows(
    PCAP,
    packet_limit=50_000,
    include_eof=False,
):

    if kind == "FLOW":

        idx = int(
            flow_count
        )


        flow_count += 1


        protocol_counts[
            int(
                payload.protocol
            )
        ] += 1


        signature = (
            payload.s4_signature()
        )


        set_match = (
            signature
            in label_entries
        )


        available = int(
            remaining.get(
                signature,
                0,
            )
        )


        multi_match = (
            available > 0
        )


        if multi_match:

            remaining[
                signature
            ] = (
                available
                - 1
            )


        d5_match = (
            multi_match
            and
            idx not in {
                471,
                473,
            }
        )


        raw_set.append(
            set_match
        )

        raw_multi.append(
            multi_match
        )

        d5.append(
            d5_match
        )


        if idx in {
            199,
            471,
            473,
        }:

            audit[
                idx
            ] = {
                "protocol":
                    int(
                        payload.protocol
                    ),

                "duration":
                    int(
                        payload.last_us
                        - payload.start_us
                    ),

                "termination":
                    str(
                        reason
                    ),

                "raw":
                    bool(
                        multi_match
                    ),

                "d5":
                    bool(
                        d5_match
                    ),
            }


    elif kind == "SUMMARY":

        summary = payload


if summary is None:

    raise RuntimeError(
        "Frozen first50k summary missing."
    )


raw_absent = {
    i
    for i, value
    in enumerate(
        raw_multi
    )
    if not value
}


set_absent = {
    i
    for i, value
    in enumerate(
        raw_set
    )
    if not value
}


d5_absent = {
    i
    for i, value
    in enumerate(
        d5
    )
    if not value
}


expected_raw = set(
    EXPECTED_RAW_ABSENT
)


expected_d5 = (
    expected_raw
    |
    {
        471,
        473,
    }
)


print(
    "Parser:",
    summary[
        "parser_counts"
    ],
)

print(
    "Completed flows:",
    flow_count,
)

print(
    "Reasons:",
    summary[
        "export_counts"
    ],
)

print(
    "Protocols:",
    dict(
        sorted(
            protocol_counts.items()
        )
    ),
)

print()

print(
    "Raw exact SET:",
    sum(
        raw_set
    ),
    "/ 675",
)

print(
    "Raw exact MULTISET:",
    sum(
        raw_multi
    ),
    "/ 675",
)

print(
    "D5 accepted:",
    sum(
        d5
    ),
    "/ 675",
)

print()

print(
    "Raw absent:"
)

print(
    sorted(
        raw_absent
    )
)

print()

print(
    "D5 absent:"
)

print(
    sorted(
        d5_absent
    )
)

print()


# ==============================================================================
# 10. HARD ASSERTIONS
# ==============================================================================

if summary[
    "parser_counts"
] != {
    "IPv4_TCP":
        31_618,

    "NON_IPV4":
        3_964,

    "IPv4_OTHER_PROTOCOL_0":
        84,

    "IPv4_UDP":
        14_334,
}:

    raise RuntimeError(
        "Parser geometry changed."
    )


if flow_count != 675:

    raise RuntimeError(
        f"Expected 675 flows, got {flow_count}."
    )


if summary[
    "export_counts"
].get(
    "FIN",
    0,
) != 432:

    raise RuntimeError(
        "FIN anchor mismatch."
    )


if summary[
    "export_counts"
].get(
    "FLOW_TIMEOUT",
    0,
) != 243:

    raise RuntimeError(
        "FLOW_TIMEOUT anchor mismatch."
    )


if dict(
    sorted(
        protocol_counts.items()
    )
) != {
    0:
        1,

    6:
        467,

    17:
        207,
}:

    raise RuntimeError(
        "Protocol anchor mismatch."
    )


if sum(
    raw_set
) != 637:

    raise RuntimeError(
        f"Raw set expected 637, got {sum(raw_set)}."
    )


if sum(
    raw_multi
) != 637:

    raise RuntimeError(
        f"Raw multiset expected 637, got {sum(raw_multi)}."
    )


if set_absent != expected_raw:

    raise RuntimeError(
        "\nRaw SET absent mismatch.\n"
        f"Expected: {sorted(expected_raw)}\n"
        f"Actual:   {sorted(set_absent)}"
    )


if raw_absent != expected_raw:

    raise RuntimeError(
        "\nRaw MULTISET absent mismatch.\n"
        f"Expected: {sorted(expected_raw)}\n"
        f"Actual:   {sorted(raw_absent)}"
    )


if sum(
    d5
) != 635:

    raise RuntimeError(
        f"D5 expected 635, got {sum(d5)}."
    )


if d5_absent != expected_d5:

    raise RuntimeError(
        "\nD5 absent mismatch.\n"
        f"Expected: {sorted(expected_d5)}\n"
        f"Actual:   {sorted(d5_absent)}"
    )


if (
    audit[199]["protocol"] != 0
    or
    audit[199]["raw"] is not False
):

    raise RuntimeError(
        "Flow 199 frozen anchor failed."
    )


for idx, duration in [
    (
        471,
        224,
    ),
    (
        473,
        262,
    ),
]:

    if (
        audit[idx]["protocol"] != 6
        or
        audit[idx]["duration"] != duration
        or
        audit[idx]["termination"] != "FIN"
        or
        audit[idx]["raw"] is not True
        or
        audit[idx]["d5"] is not False
    ):

        raise RuntimeError(
            f"Frozen D5 anchor {idx} failed."
        )


print("=" * 112)
print("STAGE24-1C1-M-RUNTIME-RECOVERY-R3: PASS")
print("=" * 112)

print()

print(
    "Historical captured IPv4 extent rule:"
)

print(
    " ",
    CAPTURE_EXTENT_RULE,
)

print()

print(
    "Frozen C16 identity:",
    f"{EXPECTED_C16}/{EXPECTED_C16}",
)

print(
    "Historical raw exact:",
    "637 / 675",
)

print(
    "Historical D5 accepted:",
    "635 / 675",
)

print()

print(
    "Recovered flow iterator is LIVE in this kernel."
)

print(
    "DO NOT RESET THE SESSION."
)

print()

print(
    "Completed scientific fits: 2 / 4"
)

print(
    "Target openings:           0 / 8"
)

print(
    "Model executions:          0"
)

print()

print(
    "NEXT: FULL MONDAY GROUNDED_S4 MEMBERSHIP REPLAY."
)

print("=" * 112)

STAGE24-1C1-M-RUNTIME-RECOVERY-R3
Frozen C16 packets: 5093
GT: 4930
LT: 163
Maximum raw_index: 48303

LABEL-FREE C16 CAPTURE-EXTENT AUDIT
Candidate                                        available     exact  mismatch
----------------------------------------------------------------------------------
HEADERS_PLUS_LEN_BYTES_TCP_PAYLOAD                    5093      4922       171
   examples: [{'raw_index': 288, 'cohort': 'LT', 'frozen': 384, 'candidate': 390, 'declared': 390}, {'raw_index': 289, 'cohort': 'LT', 'frozen': 384, 'candidate': 390, 'declared': 390}, {'raw_index': 291, 'cohort': 'LT', 'frozen': 2552, 'candidate': 2674, 'declared': 2674}, {'raw_index': 317, 'cohort': 'LT', 'frozen': 1706, 'candidate': 1712, 'declared': 1712}, {'raw_index': 318, 'cohort': 'LT', 'frozen': 1706, 'candidate': 1712, 'declared': 1712}]
IP_HEADER_PLUS_LEN_TCP_ORIGINAL                       5093         0      5093
   examples: [{'raw_index': 0, 'cohort': 'GT', 'frozen': 46, 'candidate': 40, 'declared':

RuntimeError: 
Expected exactly ONE frozen-C16-exact representation.
Passing candidates: []

Paste the candidate table above.

In [11]:
# ==============================================================================
# STAGE24-1C1-M-RUNTIME-RECOVERY-R4
#
# Correct interpretation:
#   C16 identity is anchored by (flow_index, packet_position).
#
# Steps:
#   1. Reconstruct first-50k flow lifecycle and retain actual PCAP packet indices.
#   2. Map every frozen C16 record through:
#          (flow_index, packet_position) -> actual PCAP packet
#   3. Verify frozen captured_ipv4_extent / baseline_payload EXACTLY, LABEL-FREE.
#   4. Install historical TCP payload:
#          captured IPv4 extent - IP header - TCP header
#   5. Run frozen Stage20 first50k S4 gate:
#          raw exact = 637 / 675
#          D5        = 635 / 675
#
# NO full Monday replay.
# NO model.
# NO commit.
# NO target opening.
# ==============================================================================

from collections import Counter
import socket

from scapy.utils import RawPcapReader
from scapy.layers.l2 import Ether
from scapy.layers.inet import IP, TCP, UDP


print("=" * 112)
print("STAGE24-1C1-M-RUNTIME-RECOVERY-R4")
print("=" * 112)

assert "PCAP" in globals()
assert "d1" in globals()
assert "d2" in globals()
assert "label_entries" in globals()
assert "EXPECTED_RAW_ABSENT" in globals()


# ==============================================================================
# 1. RAW IPV4 OFFSET
# ==============================================================================

VLAN_TYPES = {
    0x8100,
    0x88A8,
    0x9100,
    0x9200,
    0x9300,
}


def ipv4_offset_from_frame(raw_bytes):

    raw = memoryview(raw_bytes)

    if len(raw) < 14:
        return None

    ethertype = int.from_bytes(
        raw[12:14],
        "big",
    )

    offset = 14

    while ethertype in VLAN_TYPES:

        if len(raw) < offset + 4:
            return None

        ethertype = int.from_bytes(
            raw[offset + 2:offset + 4],
            "big",
        )

        offset += 4

    if ethertype != 0x0800:
        return None

    return int(offset)


# ==============================================================================
# 2. MINIMAL HISTORICAL PACKET GEOMETRY
# ==============================================================================

def geometry_packet(raw_bytes):

    pkt = Ether(raw_bytes)

    if IP not in pkt:
        return None

    ip = pkt[IP]

    src = socket.inet_aton(str(ip.src))
    dst = socket.inet_aton(str(ip.dst))

    transport = ip.payload

    protocol = 0
    sport = 0
    dport = 0
    fin = False

    if isinstance(transport, TCP):

        protocol = 6
        sport = int(transport.sport)
        dport = int(transport.dport)

        fin = bool(
            int(transport.flags)
            & 0x01
        )

    elif isinstance(transport, UDP):

        protocol = 17
        sport = int(transport.sport)
        dport = int(transport.dport)

    return {
        "src": src,
        "dst": dst,
        "sport": sport,
        "dport": dport,
        "protocol": protocol,
        "fin": fin,
    }


# ==============================================================================
# 3. RECONSTRUCT FIRST50K FLOW -> ACTUAL PACKET INDICES
#
# This mirrors the already-frozen lifecycle only.
# ==============================================================================

print()
print("=" * 112)
print("RECONSTRUCTING FIRST50K FLOW/PACKET POSITION MAP")
print("=" * 112)


active = {}

exported_packet_indices = []

parser_counts = Counter()
reason_counts = Counter()

reader = RawPcapReader(
    str(PCAP)
)

pcap_nano = bool(
    getattr(
        reader,
        "nano",
        False,
    )
)


try:

    for raw_packet_index, (
        raw_bytes,
        metadata,
    ) in enumerate(reader):

        if raw_packet_index >= 50_000:
            break


        packet = geometry_packet(
            raw_bytes
        )


        if packet is None:

            parser_counts[
                "NON_IPV4"
            ] += 1

            continue


        if packet["protocol"] == 6:

            parser_counts[
                "IPv4_TCP"
            ] += 1

        elif packet["protocol"] == 17:

            parser_counts[
                "IPv4_UDP"
            ] += 1

        else:

            parser_counts[
                "IPv4_OTHER_PROTOCOL_0"
            ] += 1


        ts_us = timestamp_us(
            metadata,
            nanosecond_precision=pcap_nano,
        )


        key = historical_flow_key(
            packet["src"],
            packet["dst"],
            packet["sport"],
            packet["dport"],
            packet["protocol"],
        )


        state = active.get(
            key
        )


        # ----------------------------------------------------------------------
        # New flow.
        # ----------------------------------------------------------------------

        if state is None:

            active[key] = {
                "start_us":
                    int(ts_us),

                "packet_indices":
                    [
                        int(raw_packet_index)
                    ],
            }

            continue


        # ----------------------------------------------------------------------
        # Timeout before FIN, exactly as frozen lifecycle.
        # ----------------------------------------------------------------------

        if (
            int(ts_us)
            -
            state["start_us"]
            >
            FLOW_TIMEOUT_US
        ):

            if len(
                state[
                    "packet_indices"
                ]
            ) > 1:

                exported_packet_indices.append(
                    list(
                        state[
                            "packet_indices"
                        ]
                    )
                )

                reason_counts[
                    "FLOW_TIMEOUT"
                ] += 1


            active.pop(
                key
            )


            active[key] = {
                "start_us":
                    int(ts_us),

                "packet_indices":
                    [
                        int(raw_packet_index)
                    ],
            }

            continue


        # ----------------------------------------------------------------------
        # Existing flow + FIN.
        # ----------------------------------------------------------------------

        if packet[
            "fin"
        ]:

            state[
                "packet_indices"
            ].append(
                int(
                    raw_packet_index
                )
            )


            exported_packet_indices.append(
                list(
                    state[
                        "packet_indices"
                    ]
                )
            )


            reason_counts[
                "FIN"
            ] += 1


            active.pop(
                key
            )

            continue


        # ----------------------------------------------------------------------
        # Normal packet.
        # ----------------------------------------------------------------------

        state[
            "packet_indices"
        ].append(
            int(
                raw_packet_index
            )
        )


finally:

    reader.close()


print(
    "Parser:",
    dict(
        parser_counts
    ),
)

print(
    "Export reasons:",
    dict(
        reason_counts
    ),
)

print(
    "Completed flows:",
    len(
        exported_packet_indices
    ),
)


if parser_counts != Counter(
    {
        "IPv4_TCP":
            31_618,

        "IPv4_UDP":
            14_334,

        "IPv4_OTHER_PROTOCOL_0":
            84,

        "NON_IPV4":
            3_964,
    }
):

    raise RuntimeError(
        "First50k parser geometry mismatch."
    )


if reason_counts != Counter(
    {
        "FIN":
            432,

        "FLOW_TIMEOUT":
            243,
    }
):

    raise RuntimeError(
        "First50k lifecycle mismatch."
    )


if len(
    exported_packet_indices
) != 675:

    raise RuntimeError(
        f"Expected 675 completed flows; "
        f"got {len(exported_packet_indices)}"
    )


print()

print(
    "[PASS] Frozen 675-flow lifecycle reproduced."
)

print()


# ==============================================================================
# 4. MAP C16 RECORDS BY (FLOW_INDEX, PACKET_POSITION)
# ==============================================================================

print("=" * 112)
print("MAPPING FROZEN C16 RECORDS TO ACTUAL PCAP PACKETS")
print("=" * 112)


frozen_records = []


for cohort, obj in [
    (
        "GT",
        d1,
    ),
    (
        "LT",
        d2,
    ),
]:

    for row in obj[
        "per_packet"
    ]:

        flow_index = int(
            row[
                "flow_index"
            ]
        )

        packet_position = int(
            row[
                "packet_position"
            ]
        )


        if flow_index >= len(
            exported_packet_indices
        ):

            raise RuntimeError(
                f"C16 flow_index out of range: {flow_index}"
            )


        flow_packet_list = (
            exported_packet_indices[
                flow_index
            ]
        )


        if packet_position >= len(
            flow_packet_list
        ):

            raise RuntimeError(
                f"C16 packet position out of range: "
                f"flow={flow_index}, "
                f"position={packet_position}, "
                f"flow_packets={len(flow_packet_list)}"
            )


        actual_pcap_index = int(
            flow_packet_list[
                packet_position
            ]
        )


        frozen_records.append(
            {
                "cohort":
                    cohort,

                "recorded_raw_index":
                    int(
                        row[
                            "raw_index"
                        ]
                    ),

                "flow_index":
                    flow_index,

                "packet_position":
                    packet_position,

                "actual_pcap_index":
                    actual_pcap_index,

                "frozen_extent":
                    int(
                        row[
                            "captured_ipv4_extent"
                        ]
                    ),

                "baseline_payload":
                    int(
                        row[
                            "baseline_payload"
                        ]
                    ),

                "ip_header":
                    int(
                        row[
                            "ip_header_length"
                        ]
                    ),

                "tcp_header":
                    int(
                        row[
                            "tcp_header_length"
                        ]
                    ),

                "declared":
                    int(
                        row[
                            "declared_ipv4_total_length"
                        ]
                    ),
            }
        )


EXPECTED_C16 = 5_093


if len(
    frozen_records
) != EXPECTED_C16:

    raise RuntimeError(
        f"Expected 5093 C16 records, got {len(frozen_records)}"
    )


needed_actual_indices = {
    row[
        "actual_pcap_index"
    ]
    for row
    in frozen_records
}


print(
    "Frozen C16 rows:",
    len(
        frozen_records
    ),
)

print(
    "Unique actual PCAP packets:",
    len(
        needed_actual_indices
    ),
)


same_raw_index = sum(
    row[
        "recorded_raw_index"
    ]
    ==
    row[
        "actual_pcap_index"
    ]
    for row
    in frozen_records
)


print(
    "C16 raw_index == actual PCAP index:",
    same_raw_index,
    "/",
    EXPECTED_C16,
)

print()


# ==============================================================================
# 5. READ ONLY NEEDED RAW PACKETS
# ==============================================================================

packet_cache = {}


max_needed = max(
    needed_actual_indices
)


reader = RawPcapReader(
    str(PCAP)
)


try:

    for raw_packet_index, (
        raw_bytes,
        metadata,
    ) in enumerate(reader):

        if raw_packet_index in (
            needed_actual_indices
        ):

            packet_cache[
                raw_packet_index
            ] = (
                raw_bytes,
                metadata,
            )


        if raw_packet_index >= max_needed:
            break


finally:

    reader.close()


if set(
    packet_cache
) != needed_actual_indices:

    missing = sorted(
        needed_actual_indices
        -
        set(
            packet_cache
        )
    )

    raise RuntimeError(
        f"Missing mapped packets: {missing[:20]}"
    )


# ==============================================================================
# 6. LABEL-FREE EXACT C16 VERIFICATION
# ==============================================================================

print("=" * 112)
print("LABEL-FREE C16 EXACT CAPTURE VERIFICATION")
print("=" * 112)


counts = {
    "GT":
        Counter(),

    "LT":
        Counter(),
}


examples = {
    "GT":
        [],

    "LT":
        [],
}


for frozen in frozen_records:

    cohort = frozen[
        "cohort"
    ]


    raw_bytes, metadata = (
        packet_cache[
            frozen[
                "actual_pcap_index"
            ]
        ]
    )


    pkt = Ether(
        raw_bytes
    )


    if IP not in pkt:

        raise RuntimeError(
            "Mapped C16 packet unexpectedly non-IPv4."
        )


    ip = pkt[
        IP
    ]


    if not isinstance(
        ip.payload,
        TCP,
    ):

        raise RuntimeError(
            "Mapped C16 packet unexpectedly non-TCP."
        )


    tcp = ip.payload


    offset = ipv4_offset_from_frame(
        raw_bytes
    )


    if offset is None:

        raise RuntimeError(
            "Mapped C16 packet missing raw IPv4 offset."
        )


    captured_extent = (
        len(
            raw_bytes
        )
        -
        offset
    )


    ip_header = (
        int(
            ip.ihl
        )
        * 4
    )


    tcp_header = (
        int(
            tcp.dataofs
        )
        * 4
    )


    baseline_payload = (
        captured_extent
        -
        ip_header
        -
        tcp_header
    )


    extent_ok = (
        captured_extent
        ==
        frozen[
            "frozen_extent"
        ]
    )


    payload_ok = (
        baseline_payload
        ==
        frozen[
            "baseline_payload"
        ]
    )


    header_ok = (
        ip_header
        ==
        frozen[
            "ip_header"
        ]
        and
        tcp_header
        ==
        frozen[
            "tcp_header"
        ]
    )


    counts[
        cohort
    ][
        "extent_match"
        if extent_ok
        else "extent_mismatch"
    ] += 1


    counts[
        cohort
    ][
        "payload_match"
        if payload_ok
        else "payload_mismatch"
    ] += 1


    counts[
        cohort
    ][
        "header_match"
        if header_ok
        else "header_mismatch"
    ] += 1


    if (
        (
            not extent_ok
            or
            not payload_ok
            or
            not header_ok
        )
        and
        len(
            examples[
                cohort
            ]
        )
        < 10
    ):

        examples[
            cohort
        ].append(
            {
                "flow":
                    frozen[
                        "flow_index"
                    ],

                "position":
                    frozen[
                        "packet_position"
                    ],

                "c16_raw_index":
                    frozen[
                        "recorded_raw_index"
                    ],

                "actual_pcap_index":
                    frozen[
                        "actual_pcap_index"
                    ],

                "frozen_extent":
                    frozen[
                        "frozen_extent"
                    ],

                "actual_extent":
                    captured_extent,

                "frozen_payload":
                    frozen[
                        "baseline_payload"
                    ],

                "actual_payload":
                    baseline_payload,
            }
        )


for cohort, expected in [
    (
        "GT",
        4_930,
    ),
    (
        "LT",
        163,
    ),
]:

    print(
        cohort,
        dict(
            sorted(
                counts[
                    cohort
                ].items()
            )
        ),
    )


    if examples[
        cohort
    ]:

        print(
            " examples:",
            examples[
                cohort
            ],
        )


    if counts[
        cohort
    ][
        "extent_match"
    ] != expected:

        raise RuntimeError(
            f"{cohort} frozen captured extent not reproduced."
        )


    if counts[
        cohort
    ][
        "payload_match"
    ] != expected:

        raise RuntimeError(
            f"{cohort} frozen baseline payload not reproduced."
        )


    if counts[
        cohort
    ][
        "header_match"
    ] != expected:

        raise RuntimeError(
            f"{cohort} frozen headers not reproduced."
        )


print()

print(
    "[PASS] ALL 5,093 frozen C16 packet mechanisms reproduced exactly."
)

print(
    "[PASS] No traffic labels used for runtime recovery."
)

print()


# ==============================================================================
# 7. INSTALL HISTORICAL STAGE20 TCP PAYLOAD PARSER
# ==============================================================================

def parse_packet_stage20_captured(
    raw_bytes,
):

    pkt = Ether(
        raw_bytes
    )


    if IP not in pkt:

        return None


    ip = pkt[
        IP
    ]


    src = socket.inet_aton(
        str(
            ip.src
        )
    )

    dst = socket.inet_aton(
        str(
            ip.dst
        )
    )


    transport = ip.payload


    protocol = 0

    src_port = 0

    dst_port = 0

    payload_length = 0


    flags = dict(
        ZERO_FLAGS
    )


    parser_class = (
        "IPv4_OTHER_PROTOCOL_0"
    )


    if isinstance(
        transport,
        TCP,
    ):

        tcp = transport


        protocol = 6

        src_port = int(
            tcp.sport
        )

        dst_port = int(
            tcp.dport
        )


        flags = semantic_tcp_flags(
            tcp
        )


        offset = ipv4_offset_from_frame(
            raw_bytes
        )


        if offset is None:

            raise RuntimeError(
                "TCP packet missing raw IPv4 offset."
            )


        captured_ipv4_extent = (
            len(
                raw_bytes
            )
            -
            offset
        )


        payload_length = (
            captured_ipv4_extent
            -
            int(
                ip.ihl
            ) * 4
            -
            int(
                tcp.dataofs
            ) * 4
        )


        if payload_length < 0:

            raise RuntimeError(
                "Negative captured TCP payload."
            )


        parser_class = (
            "IPv4_TCP"
        )


    elif isinstance(
        transport,
        UDP,
    ):

        udp = transport


        protocol = 17

        src_port = int(
            udp.sport
        )

        dst_port = int(
            udp.dport
        )


        # Frozen Stage20 UDP rule.
        payload_length = len(
            bytes(
                udp.payload
            )
        )


        parser_class = (
            "IPv4_UDP"
        )


    return {
        "src":
            src,

        "dst":
            dst,

        "src_port":
            src_port,

        "dst_port":
            dst_port,

        "protocol":
            protocol,

        "payload_length":
            int(
                payload_length
            ),

        "fin":
            bool(
                flags[
                    "FIN"
                ]
            ),

        "syn":
            bool(
                flags[
                    "SYN"
                ]
            ),

        "rst":
            bool(
                flags[
                    "RST"
                ]
            ),

        "psh":
            bool(
                flags[
                    "PSH"
                ]
            ),

        "ack":
            bool(
                flags[
                    "ACK"
                ]
            ),

        "urg":
            bool(
                flags[
                    "URG"
                ]
            ),

        "ece":
            bool(
                flags[
                    "ECE"
                ]
            ),

        "cwr":
            bool(
                flags[
                    "CWR"
                ]
            ),

        "parser_class":
            parser_class,
    }


# iterate_completed_flows resolves the global parse_packet dynamically.
parse_packet = (
    parse_packet_stage20_captured
)


print(
    "[PASS] Captured-byte Stage20 parser installed LIVE."
)

print()


# ==============================================================================
# 8. FROZEN FIRST50K S4 GATE
# ==============================================================================

print("=" * 112)
print("FIRST50K FROZEN STAGE20 S4 GATE")
print("=" * 112)


remaining = {
    signature:
        int(
            entry[
                1
            ]
        )
    for signature, entry
    in label_entries.items()
}


raw_set = []

raw_multi = []

d5 = []

protocol_counts = Counter()

audit = {}

summary = None

flow_count = 0


for kind, payload, reason in iterate_completed_flows(
    PCAP,
    packet_limit=50_000,
    include_eof=False,
):

    if kind == "FLOW":

        index = int(
            flow_count
        )


        flow_count += 1


        protocol_counts[
            int(
                payload.protocol
            )
        ] += 1


        signature = (
            payload.s4_signature()
        )


        set_match = (
            signature
            in label_entries
        )


        available = int(
            remaining.get(
                signature,
                0,
            )
        )


        multiset_match = (
            available > 0
        )


        if multiset_match:

            remaining[
                signature
            ] = (
                available
                - 1
            )


        d5_match = (
            multiset_match
            and
            index not in {
                471,
                473,
            }
        )


        raw_set.append(
            set_match
        )

        raw_multi.append(
            multiset_match
        )

        d5.append(
            d5_match
        )


        if index in {
            199,
            471,
            473,
        }:

            audit[
                index
            ] = {
                "protocol":
                    int(
                        payload.protocol
                    ),

                "duration":
                    int(
                        payload.last_us
                        -
                        payload.start_us
                    ),

                "termination":
                    str(
                        reason
                    ),

                "raw":
                    bool(
                        multiset_match
                    ),

                "d5":
                    bool(
                        d5_match
                    ),
            }


    elif kind == "SUMMARY":

        summary = payload


expected_raw = set(
    EXPECTED_RAW_ABSENT
)


expected_d5 = (
    expected_raw
    |
    {
        471,
        473,
    }
)


set_absent = {
    index
    for index, value
    in enumerate(
        raw_set
    )
    if not value
}


multi_absent = {
    index
    for index, value
    in enumerate(
        raw_multi
    )
    if not value
}


d5_absent = {
    index
    for index, value
    in enumerate(
        d5
    )
    if not value
}


print(
    "Parser:",
    summary[
        "parser_counts"
    ],
)

print(
    "Completed flows:",
    flow_count,
)

print(
    "Reasons:",
    summary[
        "export_counts"
    ],
)

print(
    "Protocols:",
    dict(
        sorted(
            protocol_counts.items()
        )
    ),
)

print()

print(
    "Raw exact SET:",
    sum(
        raw_set
    ),
    "/ 675",
)

print(
    "Raw exact MULTISET:",
    sum(
        raw_multi
    ),
    "/ 675",
)

print(
    "D5 accepted:",
    sum(
        d5
    ),
    "/ 675",
)

print()

print(
    "Raw absent:"
)

print(
    sorted(
        multi_absent
    )
)

print()

print(
    "D5 absent:"
)

print(
    sorted(
        d5_absent
    )
)

print()


# ==============================================================================
# 9. HARD FROZEN ASSERTIONS
# ==============================================================================

if flow_count != 675:

    raise RuntimeError(
        f"Expected 675 flows, got {flow_count}."
    )


if summary[
    "parser_counts"
] != {
    "IPv4_TCP":
        31_618,

    "NON_IPV4":
        3_964,

    "IPv4_OTHER_PROTOCOL_0":
        84,

    "IPv4_UDP":
        14_334,
}:

    raise RuntimeError(
        "Parser geometry mismatch."
    )


if summary[
    "export_counts"
].get(
    "FIN",
    0,
) != 432:

    raise RuntimeError(
        "FIN mismatch."
    )


if summary[
    "export_counts"
].get(
    "FLOW_TIMEOUT",
    0,
) != 243:

    raise RuntimeError(
        "FLOW_TIMEOUT mismatch."
    )


if dict(
    sorted(
        protocol_counts.items()
    )
) != {
    0:
        1,

    6:
        467,

    17:
        207,
}:

    raise RuntimeError(
        "Protocol-count mismatch."
    )


if sum(
    raw_set
) != 637:

    raise RuntimeError(
        f"Expected raw set 637; got {sum(raw_set)}."
    )


if sum(
    raw_multi
) != 637:

    raise RuntimeError(
        f"Expected raw multiset 637; got {sum(raw_multi)}."
    )


if set_absent != expected_raw:

    raise RuntimeError(
        "\nSET absent mismatch.\n"
        f"Expected: {sorted(expected_raw)}\n"
        f"Actual:   {sorted(set_absent)}"
    )


if multi_absent != expected_raw:

    raise RuntimeError(
        "\nMULTISET absent mismatch.\n"
        f"Expected: {sorted(expected_raw)}\n"
        f"Actual:   {sorted(multi_absent)}"
    )


if sum(
    d5
) != 635:

    raise RuntimeError(
        f"Expected D5 635; got {sum(d5)}."
    )


if d5_absent != expected_d5:

    raise RuntimeError(
        "\nD5 absent mismatch.\n"
        f"Expected: {sorted(expected_d5)}\n"
        f"Actual:   {sorted(d5_absent)}"
    )


if (
    audit[
        199
    ][
        "protocol"
    ] != 0
    or
    audit[
        199
    ][
        "raw"
    ] is not False
):

    raise RuntimeError(
        "Flow 199 anchor failed."
    )


for index, expected_duration in [
    (
        471,
        224,
    ),
    (
        473,
        262,
    ),
]:

    item = audit[
        index
    ]


    if (
        item[
            "protocol"
        ] != 6

        or

        item[
            "duration"
        ] != expected_duration

        or

        item[
            "termination"
        ] != "FIN"

        or

        item[
            "raw"
        ] is not True

        or

        item[
            "d5"
        ] is not False
    ):

        raise RuntimeError(
            f"D5 anchor {index} failed: {item}"
        )


print("=" * 112)
print("STAGE24-1C1-M-RUNTIME-RECOVERY-R4: PASS")
print("=" * 112)

print()

print(
    "Frozen C16 mechanism:"
)

print(
    "  (flow_index, packet_position)"
)

print(
    "  -> actual captured PCAP packet"
)

print(
    "  -> captured IPv4 extent"
)

print(
    "  -> TCP baseline payload"
)

print()

print(
    "C16 exact identity:       5093 / 5093"
)

print(
    "Historical raw exact:    637 / 675"
)

print(
    "Historical D5 accepted:  635 / 675"
)

print()

print(
    "parse_packet is patched LIVE."
)

print(
    "DO NOT RESET THIS SESSION."
)

print()

print(
    "Completed scientific fits: 2 / 4"
)

print(
    "Target openings:           0 / 8"
)

print(
    "Model executions:          0"
)

print()

print(
    "NEXT: FULL MONDAY REPLAY + REMOTE FREEZE."
)

print("=" * 112)

STAGE24-1C1-M-RUNTIME-RECOVERY-R4

RECONSTRUCTING FIRST50K FLOW/PACKET POSITION MAP
Parser: {'IPv4_TCP': 31618, 'NON_IPV4': 3964, 'IPv4_OTHER_PROTOCOL_0': 84, 'IPv4_UDP': 14334}
Export reasons: {'FIN': 432, 'FLOW_TIMEOUT': 243}
Completed flows: 675

[PASS] Frozen 675-flow lifecycle reproduced.

MAPPING FROZEN C16 RECORDS TO ACTUAL PCAP PACKETS
Frozen C16 rows: 5093
Unique actual PCAP packets: 5093
C16 raw_index == actual PCAP index: 5093 / 5093

LABEL-FREE C16 EXACT CAPTURE VERIFICATION
GT {'extent_match': 4922, 'extent_mismatch': 8, 'header_match': 4930, 'payload_match': 4922, 'payload_mismatch': 8}
 examples: [{'flow': 307, 'position': 6, 'c16_raw_index': 29757, 'actual_pcap_index': 29757, 'frozen_extent': 355, 'actual_extent': 354, 'frozen_payload': 315, 'actual_payload': 314}, {'flow': 307, 'position': 7, 'c16_raw_index': 29758, 'actual_pcap_index': 29758, 'frozen_extent': 355, 'actual_extent': 354, 'frozen_payload': 315, 'actual_payload': 314}, {'flow': 333, 'position': 6, 'c16_ra

RuntimeError: GT frozen captured extent not reproduced.

In [12]:
# ==============================================================================
# STAGE24-1C1-M-RUNTIME-RECOVERY-R5
#
# FINAL RUNTIME RECOVERY:
#
# Test frozen C16 captured_ipv4_extent against CLASSIC-PCAP RECORD orig_len.
#
# Hypothesis:
#
#   frozen captured_ipv4_extent
#       = pcap_record.orig_len - IPv4_L2_offset
#
# NOT:
#
#   len(raw_bytes) - IPv4_L2_offset
#
# Selection is LABEL-FREE:
#   must reproduce ALL 5,093 frozen C16 packet records exactly.
#
# If exact:
#   - install direct classic-PCAP reader preserving orig_len
#   - TCP payload =
#       (orig_len - L2 offset)
#       - IPv4 header
#       - TCP header
#   - rerun frozen first-50k Stage20 gate
#
# REQUIRED:
#   C16 exact       5093 / 5093
#   raw exact       637 / 675
#   D5 accepted     635 / 675
#
# NO FULL MONDAY REPLAY
# NO MODEL
# NO COMMIT
# NO TARGET OPENING
# ==============================================================================

from __future__ import annotations

import struct
import socket
from pathlib import Path
from collections import Counter

from scapy.layers.l2 import Ether
from scapy.layers.inet import IP, TCP, UDP


print("=" * 112)
print("STAGE24-1C1-M-RUNTIME-RECOVERY-R5")
print("=" * 112)

assert "PCAP" in globals()
assert "d1" in globals()
assert "d2" in globals()
assert "label_entries" in globals()
assert "EXPECTED_RAW_ABSENT" in globals()
assert "ActiveFlows" in globals()
assert "historical_flow_key" in globals()
assert "semantic_tcp_flags" in globals()
assert "ZERO_FLAGS" in globals()
assert "FLOW_TIMEOUT_US" in globals()


# ==============================================================================
# 1. FROZEN C16 RECORDS
# ==============================================================================

frozen_c16 = {}

for cohort, obj in [
    ("GT", d1),
    ("LT", d2),
]:

    for row in obj["per_packet"]:

        idx = int(row["raw_index"])

        if idx in frozen_c16:
            raise RuntimeError(
                f"Duplicate frozen C16 raw_index: {idx}"
            )

        frozen_c16[idx] = {
            "cohort":
                cohort,

            "extent":
                int(row["captured_ipv4_extent"]),

            "payload":
                int(row["baseline_payload"]),

            "ip_h":
                int(row["ip_header_length"]),

            "tcp_h":
                int(row["tcp_header_length"]),

            "declared":
                int(row["declared_ipv4_total_length"]),
        }


EXPECTED_C16 = 5093

if len(frozen_c16) != EXPECTED_C16:
    raise RuntimeError(
        f"Expected {EXPECTED_C16} C16 rows; "
        f"got {len(frozen_c16)}"
    )


MAX_C16_INDEX = max(frozen_c16)


print(
    "Frozen C16:",
    len(frozen_c16),
)

print(
    "GT:",
    len(d1["per_packet"]),
)

print(
    "LT:",
    len(d2["per_packet"]),
)

print(
    "Max packet index:",
    MAX_C16_INDEX,
)

print()


# ==============================================================================
# 2. CLASSIC PCAP HEADER
# ==============================================================================

PCAP_MAGIC = {
    b"\xd4\xc3\xb2\xa1":
        ("<", False, "LE_MICROSECOND"),

    b"\xa1\xb2\xc3\xd4":
        (">", False, "BE_MICROSECOND"),

    b"\x4d\x3c\xb2\xa1":
        ("<", True, "LE_NANOSECOND"),

    b"\xa1\xb2\x3c\x4d":
        (">", True, "BE_NANOSECOND"),
}


def inspect_pcap_header(path):

    with open(path, "rb") as fh:

        global_header = fh.read(24)

    if len(global_header) != 24:
        raise RuntimeError(
            "PCAP global header truncated."
        )

    magic = global_header[:4]

    if magic not in PCAP_MAGIC:
        raise RuntimeError(
            f"Unsupported PCAP magic: {magic.hex()}"
        )

    endian, nano, mode = PCAP_MAGIC[magic]

    (
        _magic,
        version_major,
        version_minor,
        thiszone,
        sigfigs,
        snaplen,
        network,
    ) = struct.unpack(
        endian + "IHHIIII",
        global_header,
    )

    return {
        "endian":
            endian,

        "nano":
            nano,

        "mode":
            mode,

        "version_major":
            int(version_major),

        "version_minor":
            int(version_minor),

        "snaplen":
            int(snaplen),

        "network":
            int(network),
    }


pcap_contract = inspect_pcap_header(
    PCAP
)


print("=" * 112)
print("CLASSIC PCAP CONTRACT")
print("=" * 112)

print(
    pcap_contract
)

if pcap_contract["network"] != 1:
    raise RuntimeError(
        "Expected Ethernet LINKTYPE 1."
    )

print()


# ==============================================================================
# 3. DIRECT CLASSIC-PCAP ITERATOR
#
# Keeps BOTH:
#   incl_len = bytes physically stored in file
#   orig_len = original/wire packet length in record header
# ==============================================================================

def iter_classic_pcap(path):

    contract = inspect_pcap_header(
        path
    )

    endian = contract["endian"]

    nano = contract["nano"]

    record_struct = struct.Struct(
        endian + "IIII"
    )

    with open(path, "rb") as fh:

        global_header = fh.read(24)

        if len(global_header) != 24:
            raise RuntimeError(
                "PCAP global header disappeared."
            )

        packet_index = 0

        while True:

            header = fh.read(16)

            if not header:
                break

            if len(header) != 16:
                raise RuntimeError(
                    f"Truncated PCAP record header at "
                    f"packet {packet_index}"
                )

            (
                ts_sec,
                ts_fraction,
                incl_len,
                orig_len,
            ) = record_struct.unpack(
                header
            )

            raw_bytes = fh.read(
                incl_len
            )

            if len(raw_bytes) != incl_len:
                raise RuntimeError(
                    f"Truncated PCAP record payload at "
                    f"packet {packet_index}"
                )

            if nano:

                # Frozen lifecycle uses integer microseconds.
                ts_us = (
                    int(ts_sec)
                    * 1_000_000
                    +
                    int(ts_fraction)
                    // 1000
                )

            else:

                ts_us = (
                    int(ts_sec)
                    * 1_000_000
                    +
                    int(ts_fraction)
                )

            yield {
                "index":
                    int(packet_index),

                "ts_us":
                    int(ts_us),

                "incl_len":
                    int(incl_len),

                "orig_len":
                    int(orig_len),

                "raw_bytes":
                    raw_bytes,
            }

            packet_index += 1


# ==============================================================================
# 4. L2 -> IPV4 OFFSET
# ==============================================================================

VLAN_TYPES = {
    0x8100,
    0x88A8,
    0x9100,
    0x9200,
    0x9300,
}


def ipv4_offset_from_frame(raw_bytes):

    raw = memoryview(raw_bytes)

    if len(raw) < 14:
        return None

    ethertype = int.from_bytes(
        raw[12:14],
        "big",
    )

    offset = 14

    while ethertype in VLAN_TYPES:

        if len(raw) < offset + 4:
            return None

        ethertype = int.from_bytes(
            raw[offset + 2:offset + 4],
            "big",
        )

        offset += 4

    if ethertype != 0x0800:
        return None

    return int(offset)


# ==============================================================================
# 5. LABEL-FREE TEST:
#
#       orig_len - L2
#
# against frozen C16 captured_ipv4_extent
# ==============================================================================

print("=" * 112)
print("LABEL-FREE C16 PCAP RECORD-HEADER TEST")
print("=" * 112)


counts = {
    "GT":
        Counter(),

    "LT":
        Counter(),
}


examples = {
    "GT":
        [],

    "LT":
        [],
}


seen = set()


for record in iter_classic_pcap(
    PCAP
):

    idx = record["index"]

    if idx > MAX_C16_INDEX:
        break

    frozen = frozen_c16.get(
        idx
    )

    if frozen is None:
        continue

    seen.add(
        idx
    )

    raw_bytes = record["raw_bytes"]

    pkt = Ether(
        raw_bytes
    )

    if IP not in pkt:

        raise RuntimeError(
            f"C16 packet {idx} unexpectedly non-IPv4."
        )

    ip = pkt[IP]

    if not isinstance(
        ip.payload,
        TCP,
    ):

        raise RuntimeError(
            f"C16 packet {idx} unexpectedly non-TCP."
        )

    tcp = ip.payload

    offset = ipv4_offset_from_frame(
        raw_bytes
    )

    if offset is None:

        raise RuntimeError(
            f"No IPv4 L2 offset for C16 packet {idx}"
        )

    # --------------------------------------------------------------------------
    # THE KEY TEST.
    # --------------------------------------------------------------------------

    wire_ipv4_extent = (
        record["orig_len"]
        -
        offset
    )

    stored_ipv4_extent = (
        record["incl_len"]
        -
        offset
    )

    raw_ipv4_extent = (
        len(raw_bytes)
        -
        offset
    )


    ip_h = (
        int(ip.ihl)
        * 4
    )

    tcp_h = (
        int(tcp.dataofs)
        * 4
    )


    wire_payload = (
        wire_ipv4_extent
        -
        ip_h
        -
        tcp_h
    )


    cohort = frozen["cohort"]


    extent_ok = (
        wire_ipv4_extent
        ==
        frozen["extent"]
    )


    payload_ok = (
        wire_payload
        ==
        frozen["payload"]
    )


    header_ok = (
        ip_h
        ==
        frozen["ip_h"]

        and

        tcp_h
        ==
        frozen["tcp_h"]
    )


    counts[cohort][
        "extent_match"
        if extent_ok
        else "extent_mismatch"
    ] += 1


    counts[cohort][
        "payload_match"
        if payload_ok
        else "payload_mismatch"
    ] += 1


    counts[cohort][
        "header_match"
        if header_ok
        else "header_mismatch"
    ] += 1


    # Useful characterization only.
    relation = (
        record["orig_len"]
        -
        record["incl_len"]
    )


    counts[cohort][
        f"orig_minus_incl:{relation}"
    ] += 1


    if (
        not extent_ok
        or
        not payload_ok
        or
        not header_ok
    ):

        if len(
            examples[cohort]
        ) < 10:

            examples[cohort].append(
                {
                    "index":
                        idx,

                    "frozen_extent":
                        frozen["extent"],

                    "wire_extent":
                        wire_ipv4_extent,

                    "stored_extent":
                        stored_ipv4_extent,

                    "raw_extent":
                        raw_ipv4_extent,

                    "orig_len":
                        record["orig_len"],

                    "incl_len":
                        record["incl_len"],

                    "orig_minus_incl":
                        relation,

                    "frozen_payload":
                        frozen["payload"],

                    "wire_payload":
                        wire_payload,
                }
            )


if seen != set(
    frozen_c16
):

    missing = sorted(
        set(frozen_c16)
        -
        seen
    )

    raise RuntimeError(
        f"Missing C16 records: {missing[:20]}"
    )


for cohort, expected in [
    ("GT", 4930),
    ("LT", 163),
]:

    print()
    print(cohort)

    print(
        dict(
            sorted(
                counts[cohort].items()
            )
        )
    )

    if examples[cohort]:

        print(
            "Examples:",
            examples[cohort],
        )

    if (
        counts[cohort]["extent_match"]
        != expected
    ):

        raise RuntimeError(
            f"\n{cohort}: PCAP orig_len does NOT "
            "reproduce frozen C16 extent.\n"
            "Paste this output."
        )

    if (
        counts[cohort]["payload_match"]
        != expected
    ):

        raise RuntimeError(
            f"{cohort}: wire payload mismatch."
        )

    if (
        counts[cohort]["header_match"]
        != expected
    ):

        raise RuntimeError(
            f"{cohort}: header mismatch."
        )


print()

print(
    "[PASS] PCAP record orig_len reproduces "
    "ALL frozen C16 captured extents."
)

print(
    "[PASS] C16 identity: 5093 / 5093"
)

print(
    "[PASS] Runtime recovery was label-free."
)

print()


# ==============================================================================
# 6. HISTORICAL STAGE20 PACKET PARSER USING PCAP orig_len
# ==============================================================================

def parse_packet_stage20_wire(
    raw_bytes,
    orig_len,
):

    pkt = Ether(
        raw_bytes
    )

    if IP not in pkt:

        return None

    ip = pkt[IP]

    src = socket.inet_aton(
        str(ip.src)
    )

    dst = socket.inet_aton(
        str(ip.dst)
    )


    transport = ip.payload


    protocol = 0

    src_port = 0

    dst_port = 0

    payload_length = 0

    flags = dict(
        ZERO_FLAGS
    )

    parser_class = (
        "IPv4_OTHER_PROTOCOL_0"
    )


    if isinstance(
        transport,
        TCP,
    ):

        tcp = transport

        protocol = 6

        src_port = int(
            tcp.sport
        )

        dst_port = int(
            tcp.dport
        )

        flags = semantic_tcp_flags(
            tcp
        )

        offset = ipv4_offset_from_frame(
            raw_bytes
        )

        if offset is None:

            raise RuntimeError(
                "TCP without recoverable IPv4 offset."
            )

        captured_ipv4_extent = (
            int(orig_len)
            -
            offset
        )

        payload_length = (
            captured_ipv4_extent
            -
            int(ip.ihl) * 4
            -
            int(tcp.dataofs) * 4
        )

        if payload_length < 0:

            raise RuntimeError(
                "Negative frozen TCP payload."
            )

        parser_class = (
            "IPv4_TCP"
        )


    elif isinstance(
        transport,
        UDP,
    ):

        udp = transport

        protocol = 17

        src_port = int(
            udp.sport
        )

        dst_port = int(
            udp.dport
        )

        # Frozen Stage20 UDP baseline.
        payload_length = len(
            bytes(
                udp.payload
            )
        )

        parser_class = (
            "IPv4_UDP"
        )


    return {
        "src":
            src,

        "dst":
            dst,

        "src_port":
            src_port,

        "dst_port":
            dst_port,

        "protocol":
            protocol,

        "payload_length":
            int(payload_length),

        "fin":
            bool(flags["FIN"]),

        "syn":
            bool(flags["SYN"]),

        "rst":
            bool(flags["RST"]),

        "psh":
            bool(flags["PSH"]),

        "ack":
            bool(flags["ACK"]),

        "urg":
            bool(flags["URG"]),

        "ece":
            bool(flags["ECE"]),

        "cwr":
            bool(flags["CWR"]),

        "parser_class":
            parser_class,
    }


# ==============================================================================
# 7. HISTORICAL FLOW ITERATOR USING DIRECT PCAP RECORDS
# ==============================================================================

def iterate_completed_flows_stage20_wire(
    pcap_path,
    *,
    packet_limit=None,
    include_eof=False,
    progress_every=None,
):

    active = ActiveFlows()

    parser_counts = Counter()

    export_counts = Counter()

    raw_packet_count = 0

    valid_ipv4_count = 0

    exported_flow_count = 0

    exported_packet_total = 0

    timeout_singletons = 0


    for record in iter_classic_pcap(
        pcap_path
    ):

        if (
            packet_limit is not None
            and
            raw_packet_count >= packet_limit
        ):

            break


        raw_packet_count += 1


        parsed = parse_packet_stage20_wire(
            record["raw_bytes"],
            record["orig_len"],
        )


        if parsed is None:

            parser_counts[
                "NON_IPV4"
            ] += 1

        else:

            valid_ipv4_count += 1


            parser_counts[
                parsed["parser_class"]
            ] += 1


            ts_us = int(
                record["ts_us"]
            )


            key = historical_flow_key(
                parsed["src"],
                parsed["dst"],
                parsed["src_port"],
                parsed["dst_port"],
                parsed["protocol"],
            )


            flow = active.get(
                key
            )


            # --------------------------------------------------------------
            # New flow.
            # --------------------------------------------------------------

            if flow is None:

                active.create_flow(
                    key,
                    parsed,
                    ts_us,
                )


            # --------------------------------------------------------------
            # Timeout before FIN.
            # --------------------------------------------------------------

            elif (
                ts_us
                -
                flow.start_us
                >
                FLOW_TIMEOUT_US
            ):

                inherited_orientation = (
                    bytes(flow.src),
                    bytes(flow.dst),
                    int(flow.src_port),
                    int(flow.dst_port),
                )


                if flow.packet_count > 1:

                    exported_flow_count += 1

                    exported_packet_total += int(
                        flow.packet_count
                    )

                    export_counts[
                        "FLOW_TIMEOUT"
                    ] += 1

                    yield (
                        "FLOW",
                        flow,
                        "FLOW_TIMEOUT",
                    )

                else:

                    timeout_singletons += 1


                active.remove(
                    key
                )


                active.create_flow(
                    key,
                    parsed,
                    ts_us,
                    inherited_orientation=(
                        inherited_orientation
                    ),
                )


            # --------------------------------------------------------------
            # Existing flow + FIN.
            # --------------------------------------------------------------

            elif parsed["fin"]:

                flow.add_packet(
                    parsed,
                    ts_us,
                )

                exported_flow_count += 1

                exported_packet_total += int(
                    flow.packet_count
                )

                export_counts[
                    "FIN"
                ] += 1

                yield (
                    "FLOW",
                    flow,
                    "FIN",
                )

                active.remove(
                    key
                )


            # --------------------------------------------------------------
            # Normal packet.
            # --------------------------------------------------------------

            else:

                flow.add_packet(
                    parsed,
                    ts_us,
                )


        if (
            progress_every is not None
            and
            raw_packet_count
            % progress_every
            == 0
        ):

            yield (
                "PROGRESS",
                {
                    "raw_packets":
                        raw_packet_count,

                    "valid_ipv4":
                        valid_ipv4_count,

                    "active_flows":
                        len(active),

                    "exported_flows":
                        exported_flow_count,
                },
                None,
            )


    eof_singletons = 0


    if include_eof:

        for key, flow in (
            active
            .eof_items_historical_bucket_order()
        ):

            if flow.packet_count > 1:

                exported_flow_count += 1

                exported_packet_total += int(
                    flow.packet_count
                )

                export_counts[
                    "EOF_CURRENT"
                ] += 1

                yield (
                    "FLOW",
                    flow,
                    "EOF_CURRENT",
                )

            else:

                eof_singletons += 1


    yield (
        "SUMMARY",
        {
            "raw_packet_count":
                int(raw_packet_count),

            "valid_ipv4_count":
                int(valid_ipv4_count),

            "parser_counts":
                dict(parser_counts),

            "export_counts":
                dict(export_counts),

            "exported_flow_count":
                int(exported_flow_count),

            "exported_packet_total":
                int(exported_packet_total),

            "active_flow_count_at_end":
                int(len(active)),

            "timeout_singletons":
                int(timeout_singletons),

            "eof_singletons":
                int(eof_singletons),

            "java_capacity":
                int(active.capacity),
        },
        None,
    )


# Replace the live iterator for the next full-replay cell.
iterate_completed_flows = (
    iterate_completed_flows_stage20_wire
)


print(
    "[PASS] Historical classic-PCAP flow iterator installed LIVE."
)

print()


# ==============================================================================
# 8. FIRST-50K FROZEN S4 GATE
# ==============================================================================

print("=" * 112)
print("FIRST-50K FROZEN STAGE20 S4 GATE")
print("=" * 112)


remaining = {
    signature:
        int(entry[1])
    for signature, entry
    in label_entries.items()
}


raw_set = []

raw_multi = []

d5 = []

protocol_counts = Counter()

audit = {}

flow_count = 0

summary = None


for kind, payload, reason in iterate_completed_flows(
    PCAP,
    packet_limit=50_000,
    include_eof=False,
):

    if kind == "FLOW":

        idx = int(
            flow_count
        )

        flow_count += 1


        protocol_counts[
            int(payload.protocol)
        ] += 1


        signature = (
            payload.s4_signature()
        )


        set_match = (
            signature
            in label_entries
        )


        available = int(
            remaining.get(
                signature,
                0,
            )
        )


        multi_match = (
            available > 0
        )


        if multi_match:

            remaining[
                signature
            ] = (
                available
                - 1
            )


        d5_match = (
            multi_match
            and
            idx not in {
                471,
                473,
            }
        )


        raw_set.append(
            set_match
        )

        raw_multi.append(
            multi_match
        )

        d5.append(
            d5_match
        )


        if idx in {
            199,
            471,
            473,
        }:

            audit[idx] = {
                "protocol":
                    int(payload.protocol),

                "duration":
                    int(
                        payload.last_us
                        -
                        payload.start_us
                    ),

                "termination":
                    str(reason),

                "raw":
                    bool(multi_match),

                "d5":
                    bool(d5_match),
            }


    elif kind == "SUMMARY":

        summary = payload


expected_raw = set(
    EXPECTED_RAW_ABSENT
)


expected_d5 = (
    expected_raw
    |
    {
        471,
        473,
    }
)


set_absent = {
    i
    for i, value
    in enumerate(raw_set)
    if not value
}


multi_absent = {
    i
    for i, value
    in enumerate(raw_multi)
    if not value
}


d5_absent = {
    i
    for i, value
    in enumerate(d5)
    if not value
}


print(
    "Parser:",
    summary["parser_counts"],
)

print(
    "Flows:",
    flow_count,
)

print(
    "Reasons:",
    summary["export_counts"],
)

print(
    "Protocols:",
    dict(
        sorted(
            protocol_counts.items()
        )
    ),
)

print()

print(
    "Raw SET:",
    sum(raw_set),
    "/ 675",
)

print(
    "Raw MULTISET:",
    sum(raw_multi),
    "/ 675",
)

print(
    "D5 accepted:",
    sum(d5),
    "/ 675",
)

print()

print(
    "Raw absent:"
)

print(
    sorted(multi_absent)
)

print()

print(
    "D5 absent:"
)

print(
    sorted(d5_absent)
)

print()


# ==============================================================================
# 9. HARD FROZEN ASSERTIONS
# ==============================================================================

assert flow_count == 675


assert summary[
    "parser_counts"
] == {
    "IPv4_TCP":
        31618,

    "NON_IPV4":
        3964,

    "IPv4_OTHER_PROTOCOL_0":
        84,

    "IPv4_UDP":
        14334,
}


assert summary[
    "export_counts"
]["FIN"] == 432


assert summary[
    "export_counts"
]["FLOW_TIMEOUT"] == 243


assert dict(
    sorted(
        protocol_counts.items()
    )
) == {
    0:
        1,

    6:
        467,

    17:
        207,
}


if sum(raw_set) != 637:

    raise RuntimeError(
        f"Raw SET expected 637; "
        f"got {sum(raw_set)}."
    )


if sum(raw_multi) != 637:

    raise RuntimeError(
        f"Raw MULTISET expected 637; "
        f"got {sum(raw_multi)}."
    )


if set_absent != expected_raw:

    raise RuntimeError(
        "\nRaw SET absent mismatch.\n"
        f"Expected: {sorted(expected_raw)}\n"
        f"Actual:   {sorted(set_absent)}"
    )


if multi_absent != expected_raw:

    raise RuntimeError(
        "\nRaw MULTISET absent mismatch.\n"
        f"Expected: {sorted(expected_raw)}\n"
        f"Actual:   {sorted(multi_absent)}"
    )


if sum(d5) != 635:

    raise RuntimeError(
        f"D5 expected 635; "
        f"got {sum(d5)}."
    )


if d5_absent != expected_d5:

    raise RuntimeError(
        "\nD5 absent mismatch.\n"
        f"Expected: {sorted(expected_d5)}\n"
        f"Actual:   {sorted(d5_absent)}"
    )


assert audit[199]["protocol"] == 0
assert audit[199]["raw"] is False


for idx, duration in [
    (471, 224),
    (473, 262),
]:

    item = audit[idx]

    assert item["protocol"] == 6
    assert item["duration"] == duration
    assert item["termination"] == "FIN"
    assert item["raw"] is True
    assert item["d5"] is False


print("=" * 112)
print("STAGE24-1C1-M-RUNTIME-RECOVERY-R5: PASS")
print("=" * 112)

print()

print(
    "C16 PCAP-record identity: 5093 / 5093"
)

print(
    "Historical raw exact:    637 / 675"
)

print(
    "Historical D5 accepted:  635 / 675"
)

print()

print(
    "Historical iterator is LIVE in this kernel."
)

print(
    "DO NOT RESET THE SESSION."
)

print()

print(
    "Completed scientific fits: 2 / 4"
)

print(
    "Target openings:           0 / 8"
)

print(
    "Model executions:          0"
)

print()

print(
    "NEXT: FULL MONDAY REPLAY + REMOTE FREEZE."
)

print("=" * 112)

STAGE24-1C1-M-RUNTIME-RECOVERY-R5
Frozen C16: 5093
GT: 4930
LT: 163
Max packet index: 48303



RuntimeError: Unsupported PCAP magic: 0a0d0d0a

In [13]:
# ==============================================================================
# STAGE24-1C1-M-RUNTIME-RECOVERY-R6
#
# PCAPNG BLOCK-LEVEL CAPTURE RECOVERY
#
# The Monday source begins with:
#     0a0d0d0a
# which is a PCAPNG Section Header Block.
#
# This cell:
#   1. Parses PCAPNG blocks directly.
#   2. Preserves EPB captured_len and original_len.
#   3. Verifies which block-level extent reproduces ALL frozen C16 packets.
#   4. Selection is LABEL-FREE.
#   5. Installs the exact packet/flow iterator.
#   6. Runs the frozen first-50k Stage20 S4 gate.
#
# REQUIRED:
#   C16 identity       = 5093 / 5093
#   raw exact set      = 637 / 675
#   raw exact multiset = 637 / 675
#   D5 accepted        = 635 / 675
#
# NO full Monday replay.
# NO model.
# NO commit.
# NO target opening.
# ==============================================================================

from __future__ import annotations

import struct
import socket
from fractions import Fraction
from collections import Counter, defaultdict

from scapy.layers.l2 import Ether
from scapy.layers.inet import IP, TCP, UDP


print("=" * 112)
print("STAGE24-1C1-M-RUNTIME-RECOVERY-R6 — PCAPNG BLOCK LEVEL")
print("=" * 112)

assert "PCAP" in globals()
assert "d1" in globals()
assert "d2" in globals()
assert "label_entries" in globals()
assert "EXPECTED_RAW_ABSENT" in globals()
assert "ActiveFlows" in globals()
assert "historical_flow_key" in globals()
assert "semantic_tcp_flags" in globals()
assert "ZERO_FLAGS" in globals()
assert "FLOW_TIMEOUT_US" in globals()


# ==============================================================================
# 1. FROZEN C16 POPULATION
# ==============================================================================

frozen_c16 = {}

for cohort, obj in [
    ("GT", d1),
    ("LT", d2),
]:

    for row in obj["per_packet"]:

        idx = int(
            row["raw_index"]
        )

        if idx in frozen_c16:

            raise RuntimeError(
                f"Duplicate C16 raw index: {idx}"
            )

        frozen_c16[idx] = {
            "cohort":
                cohort,

            "extent":
                int(
                    row["captured_ipv4_extent"]
                ),

            "payload":
                int(
                    row["baseline_payload"]
                ),

            "ip_h":
                int(
                    row["ip_header_length"]
                ),

            "tcp_h":
                int(
                    row["tcp_header_length"]
                ),

            "declared":
                int(
                    row["declared_ipv4_total_length"]
                ),
        }


EXPECTED_C16 = 5_093
MAX_C16_INDEX = max(frozen_c16)


if len(frozen_c16) != EXPECTED_C16:

    raise RuntimeError(
        f"C16 count mismatch: {len(frozen_c16)}"
    )


print("Frozen C16:", EXPECTED_C16)
print("GT:", len(d1["per_packet"]))
print("LT:", len(d2["per_packet"]))
print("Max raw index:", MAX_C16_INDEX)
print()


# ==============================================================================
# 2. ETHERNET IPV4 OFFSET
# ==============================================================================

VLAN_TYPES = {
    0x8100,
    0x88A8,
    0x9100,
    0x9200,
    0x9300,
}


def ipv4_offset_from_frame(raw_bytes):

    raw = memoryview(
        raw_bytes
    )

    if len(raw) < 14:

        return None


    ethertype = int.from_bytes(
        raw[12:14],
        "big",
    )

    offset = 14


    while ethertype in VLAN_TYPES:

        if len(raw) < offset + 4:

            return None


        ethertype = int.from_bytes(
            raw[
                offset + 2:
                offset + 4
            ],
            "big",
        )

        offset += 4


    if ethertype != 0x0800:

        return None


    return int(
        offset
    )


# ==============================================================================
# 3. PCAPNG OPTION PARSER
# ==============================================================================

def parse_options(
    data,
    endian,
):

    options = {}

    pos = 0


    while pos + 4 <= len(data):

        code, length = struct.unpack_from(
            endian + "HH",
            data,
            pos,
        )

        pos += 4


        if code == 0:

            break


        value = data[
            pos:
            pos + length
        ]


        options.setdefault(
            int(code),
            [],
        ).append(
            bytes(value)
        )


        pos += (
            (
                length
                + 3
            )
            // 4
        ) * 4


    return options


def decode_ts_resolution(
    options,
):

    # PCAPNG default is 10^-6 seconds.
    if 9 not in options:

        return Fraction(
            1,
            1_000_000,
        )


    raw = options[
        9
    ][
        0
    ]


    if len(raw) < 1:

        raise RuntimeError(
            "Malformed if_tsresol option."
        )


    value = int(
        raw[
            0
        ]
    )


    if value & 0x80:

        exponent = (
            value
            & 0x7F
        )

        return Fraction(
            1,
            2 ** exponent,
        )


    return Fraction(
        1,
        10 ** value,
    )


def decode_ts_offset(
    options,
    endian,
):

    # if_tsoffset option code 14, signed 64-bit seconds.
    if 14 not in options:

        return 0


    raw = options[
        14
    ][
        0
    ]


    if len(raw) != 8:

        raise RuntimeError(
            "Malformed if_tsoffset option."
        )


    return int(
        struct.unpack(
            endian + "q",
            raw,
        )[
            0
        ]
    )


# ==============================================================================
# 4. DIRECT PCAPNG ITERATOR
#
# Supports:
#   SHB 0x0A0D0D0A
#   IDB 0x00000001
#   EPB 0x00000006
#
# We fail closed if the packet population uses an unexpected packet-block type.
# ==============================================================================

SHB = 0x0A0D0D0A
IDB = 0x00000001
EPB = 0x00000006
SPB = 0x00000003
PB  = 0x00000002


def iter_pcapng_records(
    path,
):

    fh = open(
        path,
        "rb",
    )


    try:

        endian = None

        interfaces = []

        packet_index = 0


        while True:

            first8 = fh.read(
                8
            )


            if not first8:

                break


            if len(first8) != 8:

                raise RuntimeError(
                    "Truncated PCAPNG block header."
                )


            raw_type = first8[
                :4
            ]


            # ------------------------------------------------------------------
            # Section Header Block is endian-discovery point.
            # ------------------------------------------------------------------

            if raw_type == b"\x0a\x0d\x0d\x0a":

                bom_and_rest = fh.read(
                    4
                )


                if len(bom_and_rest) != 4:

                    raise RuntimeError(
                        "Truncated PCAPNG SHB BOM."
                    )


                bom = bom_and_rest


                if bom == b"\x4d\x3c\x2b\x1a":

                    endian = "<"


                elif bom == b"\x1a\x2b\x3c\x4d":

                    endian = ">"


                else:

                    raise RuntimeError(
                        "Invalid PCAPNG byte-order magic: "
                        + bom.hex()
                    )


                block_total_length = struct.unpack(
                    endian + "I",
                    first8[
                        4:8
                    ],
                )[
                    0
                ]


                if block_total_length < 28:

                    raise RuntimeError(
                        "Invalid SHB block length."
                    )


                # Already consumed 12 bytes.
                remainder = fh.read(
                    block_total_length
                    - 12
                )


                if len(remainder) != (
                    block_total_length
                    - 12
                ):

                    raise RuntimeError(
                        "Truncated SHB."
                    )


                trailing = struct.unpack(
                    endian + "I",
                    remainder[
                        -4:
                    ],
                )[
                    0
                ]


                if trailing != block_total_length:

                    raise RuntimeError(
                        "SHB trailing length mismatch."
                    )


                # New section resets interface IDs.
                interfaces = []

                continue


            if endian is None:

                raise RuntimeError(
                    "PCAPNG block encountered before SHB."
                )


            block_type = struct.unpack(
                endian + "I",
                raw_type,
            )[
                0
            ]


            block_total_length = struct.unpack(
                endian + "I",
                first8[
                    4:8
                ],
            )[
                0
            ]


            if block_total_length < 12:

                raise RuntimeError(
                    f"Invalid block length: {block_total_length}"
                )


            remaining = fh.read(
                block_total_length
                - 8
            )


            if len(remaining) != (
                block_total_length
                - 8
            ):

                raise RuntimeError(
                    "Truncated PCAPNG block."
                )


            body = remaining[
                :-4
            ]


            trailing_length = struct.unpack(
                endian + "I",
                remaining[
                    -4:
                ],
            )[
                0
            ]


            if trailing_length != block_total_length:

                raise RuntimeError(
                    "PCAPNG trailing block-length mismatch."
                )


            # ------------------------------------------------------------------
            # Interface Description Block
            # ------------------------------------------------------------------

            if block_type == IDB:

                if len(body) < 8:

                    raise RuntimeError(
                        "Malformed IDB."
                    )


                linktype, reserved, snaplen = struct.unpack_from(
                    endian + "HHI",
                    body,
                    0,
                )


                options = parse_options(
                    body[
                        8:
                    ],
                    endian,
                )


                ts_resolution = decode_ts_resolution(
                    options
                )


                ts_offset = decode_ts_offset(
                    options,
                    endian,
                )


                interfaces.append(
                    {
                        "linktype":
                            int(
                                linktype
                            ),

                        "snaplen":
                            int(
                                snaplen
                            ),

                        "ts_resolution":
                            ts_resolution,

                        "ts_offset":
                            int(
                                ts_offset
                            ),
                    }
                )


                continue


            # ------------------------------------------------------------------
            # Enhanced Packet Block
            # ------------------------------------------------------------------

            if block_type == EPB:

                if len(body) < 20:

                    raise RuntimeError(
                        "Malformed EPB."
                    )


                (
                    interface_id,
                    ts_high,
                    ts_low,
                    captured_len,
                    original_len,
                ) = struct.unpack_from(
                    endian + "IIIII",
                    body,
                    0,
                )


                interface_id = int(
                    interface_id
                )


                if interface_id >= len(
                    interfaces
                ):

                    raise RuntimeError(
                        f"Unknown EPB interface: {interface_id}"
                    )


                interface = interfaces[
                    interface_id
                ]


                if interface[
                    "linktype"
                ] != 1:

                    raise RuntimeError(
                        "Stage20 Monday expected Ethernet LINKTYPE 1; "
                        f"got {interface['linktype']}."
                    )


                captured_len = int(
                    captured_len
                )


                original_len = int(
                    original_len
                )


                packet_start = 20

                packet_end = (
                    packet_start
                    + captured_len
                )


                if packet_end > len(
                    body
                ):

                    raise RuntimeError(
                        f"EPB captured length exceeds block at packet "
                        f"{packet_index}."
                    )


                raw_bytes = bytes(
                    body[
                        packet_start:
                        packet_end
                    ]
                )


                ticks = (
                    (
                        int(
                            ts_high
                        )
                        << 32
                    )
                    |
                    int(
                        ts_low
                    )
                )


                ts_seconds = (
                    Fraction(
                        ticks,
                        1,
                    )
                    *
                    interface[
                        "ts_resolution"
                    ]
                    +
                    interface[
                        "ts_offset"
                    ]
                )


                ts_microseconds = (
                    ts_seconds
                    * 1_000_000
                )


                if ts_microseconds.denominator != 1:

                    raise RuntimeError(
                        "Frozen timestamp is not integer-microsecond "
                        f"representable at packet {packet_index}: "
                        f"{ts_microseconds}"
                    )


                yield {
                    "index":
                        int(
                            packet_index
                        ),

                    "raw_bytes":
                        raw_bytes,

                    "captured_len":
                        captured_len,

                    "original_len":
                        original_len,

                    "ts_us":
                        int(
                            ts_microseconds
                        ),
                }


                packet_index += 1

                continue


            # ------------------------------------------------------------------
            # Fail closed if actual packet blocks other than EPB appear.
            # ------------------------------------------------------------------

            if block_type in {
                SPB,
                PB,
            }:

                raise RuntimeError(
                    "Monday PCAPNG contains non-EPB packet block type "
                    f"{block_type:#x}; parser extension required."
                )


            # Other metadata blocks are ignored.


    finally:

        fh.close()


# ==============================================================================
# 5. C16 BLOCK-LEVEL AUDIT
# ==============================================================================

print("=" * 112)
print("PCAPNG BLOCK-LEVEL C16 AUDIT")
print("=" * 112)


candidate_counts = defaultdict(
    Counter
)


candidate_examples = defaultdict(
    list
)


seen = set()


for record in iter_pcapng_records(
    PCAP
):

    idx = record[
        "index"
    ]


    if idx > MAX_C16_INDEX:

        break


    frozen = frozen_c16.get(
        idx
    )


    if frozen is None:

        continue


    seen.add(
        idx
    )


    raw_bytes = record[
        "raw_bytes"
    ]


    pkt = Ether(
        raw_bytes
    )


    if IP not in pkt:

        raise RuntimeError(
            f"C16 packet {idx} non-IPv4."
        )


    ip = pkt[
        IP
    ]


    if not isinstance(
        ip.payload,
        TCP,
    ):

        raise RuntimeError(
            f"C16 packet {idx} non-direct-TCP."
        )


    tcp = ip.payload


    offset = ipv4_offset_from_frame(
        raw_bytes
    )


    if offset is None:

        raise RuntimeError(
            f"C16 packet {idx}: no IPv4 offset."
        )


    ip_h = int(
        ip.ihl
    ) * 4


    tcp_h = int(
        tcp.dataofs
    ) * 4


    candidates = {
        "PCAPNG_CAPTURED_LEN_MINUS_L2":
            int(
                record[
                    "captured_len"
                ]
            )
            -
            offset,

        "PCAPNG_ORIGINAL_LEN_MINUS_L2":
            int(
                record[
                    "original_len"
                ]
            )
            -
            offset,

        "DIRECT_PACKET_BYTES_MINUS_L2":
            len(
                raw_bytes
            )
            -
            offset,
    }


    for name, extent in (
        candidates.items()
    ):

        payload = (
            extent
            -
            ip_h
            -
            tcp_h
        )


        exact = (
            extent
            ==
            frozen[
                "extent"
            ]

            and

            payload
            ==
            frozen[
                "payload"
            ]

            and

            ip_h
            ==
            frozen[
                "ip_h"
            ]

            and

            tcp_h
            ==
            frozen[
                "tcp_h"
            ]
        )


        cohort = frozen[
            "cohort"
        ]


        candidate_counts[
            name
        ][
            "exact"
            if exact
            else "mismatch"
        ] += 1


        candidate_counts[
            name
        ][
            f"{cohort}_exact"
            if exact
            else f"{cohort}_mismatch"
        ] += 1


        if (
            not exact
            and
            len(
                candidate_examples[
                    name
                ]
            )
            < 8
        ):

            candidate_examples[
                name
            ].append(
                {
                    "index":
                        idx,

                    "cohort":
                        cohort,

                    "frozen_extent":
                        frozen[
                            "extent"
                        ],

                    "candidate_extent":
                        extent,

                    "declared_ipv4":
                        frozen[
                            "declared"
                        ],

                    "epb_captured_len":
                        record[
                            "captured_len"
                        ],

                    "epb_original_len":
                        record[
                            "original_len"
                        ],

                    "raw_bytes":
                        len(
                            raw_bytes
                        ),

                    "l2_offset":
                        offset,
                }
            )


if seen != set(
    frozen_c16
):

    missing = sorted(
        set(
            frozen_c16
        )
        -
        seen
    )

    raise RuntimeError(
        f"Missing C16 packets: {missing[:20]}"
    )


print(
    f"{'Candidate':42s}"
    f"{'Exact':>10s}"
    f"{'Mismatch':>12s}"
)

print("-" * 64)


for name in sorted(
    candidate_counts
):

    c = candidate_counts[
        name
    ]


    print(
        f"{name:42s}"
        f"{c['exact']:10d}"
        f"{c['mismatch']:12d}"
    )


    print(
        "  GT exact:",
        c[
            "GT_exact"
        ],
        "/ 4930"
    )

    print(
        "  LT exact:",
        c[
            "LT_exact"
        ],
        "/ 163"
    )


    if c[
        "mismatch"
    ]:

        print(
            "  examples:",
            candidate_examples[
                name
            ],
        )


# ==============================================================================
# 6. REQUIRE EXACT LABEL-FREE WINNER
# ==============================================================================

passing = [
    name
    for name, counts
    in candidate_counts.items()
    if (
        counts[
            "exact"
        ]
        == EXPECTED_C16

        and

        counts[
            "mismatch"
        ]
        == 0
    )
]


print()
print("=" * 112)
print("PCAPNG CAPTURE RULE DECISION")
print("=" * 112)


if len(
    passing
) != 1:

    raise RuntimeError(
        "\nExpected exactly one C16-exact PCAPNG extent rule.\n"
        f"Passing: {passing}\n\n"
        "Paste the table above."
    )


CAPTURE_RULE = passing[
    0
]


print(
    "[PASS] Historical captured extent rule:"
)

print(
    " ",
    CAPTURE_RULE,
)

print()

print(
    "[PASS] Frozen C16 identity: 5093 / 5093"
)

print(
    "[PASS] Runtime selection used no traffic-label agreement."
)

print()


# ==============================================================================
# 7. HISTORICAL EXTENT FUNCTION
# ==============================================================================

def historical_ipv4_extent(
    record,
    l2_offset,
):

    if (
        CAPTURE_RULE
        ==
        "PCAPNG_CAPTURED_LEN_MINUS_L2"
    ):

        return (
            int(
                record[
                    "captured_len"
                ]
            )
            -
            l2_offset
        )


    if (
        CAPTURE_RULE
        ==
        "PCAPNG_ORIGINAL_LEN_MINUS_L2"
    ):

        return (
            int(
                record[
                    "original_len"
                ]
            )
            -
            l2_offset
        )


    if (
        CAPTURE_RULE
        ==
        "DIRECT_PACKET_BYTES_MINUS_L2"
    ):

        return (
            len(
                record[
                    "raw_bytes"
                ]
            )
            -
            l2_offset
        )


    raise RuntimeError(
        CAPTURE_RULE
    )


# ==============================================================================
# 8. STAGE20 PACKET DECODER
# ==============================================================================

def parse_stage20_packet_from_record(
    record,
):

    raw_bytes = record[
        "raw_bytes"
    ]


    pkt = Ether(
        raw_bytes
    )


    if IP not in pkt:

        return None


    ip = pkt[
        IP
    ]


    src = socket.inet_aton(
        str(
            ip.src
        )
    )

    dst = socket.inet_aton(
        str(
            ip.dst
        )
    )


    transport = ip.payload


    protocol = 0

    src_port = 0

    dst_port = 0

    payload_length = 0

    flags = dict(
        ZERO_FLAGS
    )

    parser_class = (
        "IPv4_OTHER_PROTOCOL_0"
    )


    if isinstance(
        transport,
        TCP,
    ):

        tcp = transport


        protocol = 6

        src_port = int(
            tcp.sport
        )

        dst_port = int(
            tcp.dport
        )


        flags = semantic_tcp_flags(
            tcp
        )


        offset = ipv4_offset_from_frame(
            raw_bytes
        )


        if offset is None:

            raise RuntimeError(
                "TCP packet missing IPv4 offset."
            )


        captured_ipv4_extent = (
            historical_ipv4_extent(
                record,
                offset,
            )
        )


        payload_length = (
            captured_ipv4_extent
            -
            int(
                ip.ihl
            )
            * 4
            -
            int(
                tcp.dataofs
            )
            * 4
        )


        if payload_length < 0:

            raise RuntimeError(
                f"Negative Stage20 TCP payload at packet "
                f"{record['index']}."
            )


        parser_class = (
            "IPv4_TCP"
        )


    elif isinstance(
        transport,
        UDP,
    ):

        udp = transport


        protocol = 17

        src_port = int(
            udp.sport
        )

        dst_port = int(
            udp.dport
        )


        # Already-frozen Stage20 UDP rule.
        payload_length = len(
            bytes(
                udp.payload
            )
        )


        parser_class = (
            "IPv4_UDP"
        )


    return {
        "src":
            src,

        "dst":
            dst,

        "src_port":
            src_port,

        "dst_port":
            dst_port,

        "protocol":
            protocol,

        "payload_length":
            int(
                payload_length
            ),

        "fin":
            bool(
                flags[
                    "FIN"
                ]
            ),

        "syn":
            bool(
                flags[
                    "SYN"
                ]
            ),

        "rst":
            bool(
                flags[
                    "RST"
                ]
            ),

        "psh":
            bool(
                flags[
                    "PSH"
                ]
            ),

        "ack":
            bool(
                flags[
                    "ACK"
                ]
            ),

        "urg":
            bool(
                flags[
                    "URG"
                ]
            ),

        "ece":
            bool(
                flags[
                    "ECE"
                ]
            ),

        "cwr":
            bool(
                flags[
                    "CWR"
                ]
            ),

        "parser_class":
            parser_class,
    }


# ==============================================================================
# 9. HISTORICAL FLOW ITERATOR
# ==============================================================================

def iterate_completed_flows_stage20_pcapng(
    pcap_path,
    *,
    packet_limit=None,
    include_eof=False,
    progress_every=None,
):

    active = ActiveFlows()


    parser_counts = Counter()

    export_counts = Counter()


    raw_packet_count = 0

    valid_ipv4_count = 0

    exported_flow_count = 0

    exported_packet_total = 0

    timeout_singletons = 0


    for record in iter_pcapng_records(
        pcap_path
    ):

        if (
            packet_limit is not None
            and
            raw_packet_count
            >= packet_limit
        ):

            break


        raw_packet_count += 1


        parsed = (
            parse_stage20_packet_from_record(
                record
            )
        )


        if parsed is None:

            parser_counts[
                "NON_IPV4"
            ] += 1


        else:

            valid_ipv4_count += 1


            parser_counts[
                parsed[
                    "parser_class"
                ]
            ] += 1


            ts_us = int(
                record[
                    "ts_us"
                ]
            )


            key = historical_flow_key(
                parsed[
                    "src"
                ],
                parsed[
                    "dst"
                ],
                parsed[
                    "src_port"
                ],
                parsed[
                    "dst_port"
                ],
                parsed[
                    "protocol"
                ],
            )


            flow = active.get(
                key
            )


            # --------------------------------------------------------------
            # First packet
            # --------------------------------------------------------------

            if flow is None:

                active.create_flow(
                    key,
                    parsed,
                    ts_us,
                )


            # --------------------------------------------------------------
            # Timeout before FIN
            # --------------------------------------------------------------

            elif (
                ts_us
                -
                flow.start_us
                >
                FLOW_TIMEOUT_US
            ):

                inherited_orientation = (
                    bytes(
                        flow.src
                    ),
                    bytes(
                        flow.dst
                    ),
                    int(
                        flow.src_port
                    ),
                    int(
                        flow.dst_port
                    ),
                )


                if flow.packet_count > 1:

                    exported_flow_count += 1


                    exported_packet_total += int(
                        flow.packet_count
                    )


                    export_counts[
                        "FLOW_TIMEOUT"
                    ] += 1


                    yield (
                        "FLOW",
                        flow,
                        "FLOW_TIMEOUT",
                    )


                else:

                    timeout_singletons += 1


                active.remove(
                    key
                )


                active.create_flow(
                    key,
                    parsed,
                    ts_us,
                    inherited_orientation=(
                        inherited_orientation
                    ),
                )


            # --------------------------------------------------------------
            # Existing flow + FIN
            # --------------------------------------------------------------

            elif parsed[
                "fin"
            ]:

                flow.add_packet(
                    parsed,
                    ts_us,
                )


                exported_flow_count += 1


                exported_packet_total += int(
                    flow.packet_count
                )


                export_counts[
                    "FIN"
                ] += 1


                yield (
                    "FLOW",
                    flow,
                    "FIN",
                )


                active.remove(
                    key
                )


            # --------------------------------------------------------------
            # Normal packet
            # --------------------------------------------------------------

            else:

                flow.add_packet(
                    parsed,
                    ts_us,
                )


        if (
            progress_every is not None
            and
            raw_packet_count
            % progress_every
            == 0
        ):

            yield (
                "PROGRESS",
                {
                    "raw_packets":
                        int(
                            raw_packet_count
                        ),

                    "valid_ipv4":
                        int(
                            valid_ipv4_count
                        ),

                    "active_flows":
                        int(
                            len(
                                active
                            )
                        ),

                    "exported_flows":
                        int(
                            exported_flow_count
                        ),
                },
                None,
            )


    eof_singletons = 0


    if include_eof:

        for key, flow in (
            active
            .eof_items_historical_bucket_order()
        ):

            if flow.packet_count > 1:

                exported_flow_count += 1


                exported_packet_total += int(
                    flow.packet_count
                )


                export_counts[
                    "EOF_CURRENT"
                ] += 1


                yield (
                    "FLOW",
                    flow,
                    "EOF_CURRENT",
                )


            else:

                eof_singletons += 1


    yield (
        "SUMMARY",
        {
            "raw_packet_count":
                int(
                    raw_packet_count
                ),

            "valid_ipv4_count":
                int(
                    valid_ipv4_count
                ),

            "parser_counts":
                dict(
                    parser_counts
                ),

            "export_counts":
                dict(
                    export_counts
                ),

            "exported_flow_count":
                int(
                    exported_flow_count
                ),

            "exported_packet_total":
                int(
                    exported_packet_total
                ),

            "active_flow_count_at_end":
                int(
                    len(
                        active
                    )
                ),

            "timeout_singletons":
                int(
                    timeout_singletons
                ),

            "eof_singletons":
                int(
                    eof_singletons
                ),

            "java_capacity":
                int(
                    active.capacity
                ),
        },
        None,
    )


# Keep this live for the next full Monday cell.
iterate_completed_flows = (
    iterate_completed_flows_stage20_pcapng
)


print(
    "[PASS] Historical PCAPNG Stage20 iterator installed LIVE."
)

print()


# ==============================================================================
# 10. FIRST-50K FROZEN S4 GATE
# ==============================================================================

print("=" * 112)
print("FIRST-50K FROZEN STAGE20 S4 GATE")
print("=" * 112)


remaining = {
    signature:
        int(
            entry[
                1
            ]
        )
    for signature, entry
    in label_entries.items()
}


raw_set = []

raw_multi = []

d5 = []

protocol_counts = Counter()

audit = {}

flow_count = 0

summary = None


for kind, payload, reason in iterate_completed_flows(
    PCAP,
    packet_limit=50_000,
    include_eof=False,
):

    if kind == "FLOW":

        idx = int(
            flow_count
        )

        flow_count += 1


        protocol_counts[
            int(
                payload.protocol
            )
        ] += 1


        signature = (
            payload.s4_signature()
        )


        set_match = (
            signature
            in label_entries
        )


        available = int(
            remaining.get(
                signature,
                0,
            )
        )


        multi_match = (
            available > 0
        )


        if multi_match:

            remaining[
                signature
            ] = (
                available
                - 1
            )


        d5_match = (
            multi_match
            and
            idx not in {
                471,
                473,
            }
        )


        raw_set.append(
            set_match
        )

        raw_multi.append(
            multi_match
        )

        d5.append(
            d5_match
        )


        if idx in {
            199,
            471,
            473,
        }:

            audit[
                idx
            ] = {
                "protocol":
                    int(
                        payload.protocol
                    ),

                "duration":
                    int(
                        payload.last_us
                        -
                        payload.start_us
                    ),

                "termination":
                    str(
                        reason
                    ),

                "raw":
                    bool(
                        multi_match
                    ),

                "d5":
                    bool(
                        d5_match
                    ),
            }


    elif kind == "SUMMARY":

        summary = payload


expected_raw = set(
    EXPECTED_RAW_ABSENT
)


expected_d5 = (
    expected_raw
    |
    {
        471,
        473,
    }
)


set_absent = {
    i
    for i, value
    in enumerate(
        raw_set
    )
    if not value
}


multi_absent = {
    i
    for i, value
    in enumerate(
        raw_multi
    )
    if not value
}


d5_absent = {
    i
    for i, value
    in enumerate(
        d5
    )
    if not value
}


print(
    "Parser:",
    summary[
        "parser_counts"
    ],
)

print(
    "Flows:",
    flow_count,
)

print(
    "Reasons:",
    summary[
        "export_counts"
    ],
)

print(
    "Protocols:",
    dict(
        sorted(
            protocol_counts.items()
        )
    ),
)

print()

print(
    "Raw exact SET:",
    sum(
        raw_set
    ),
    "/ 675",
)

print(
    "Raw exact MULTISET:",
    sum(
        raw_multi
    ),
    "/ 675",
)

print(
    "D5 accepted:",
    sum(
        d5
    ),
    "/ 675",
)

print()

print(
    "Raw absent:"
)

print(
    sorted(
        multi_absent
    )
)

print()

print(
    "D5 absent:"
)

print(
    sorted(
        d5_absent
    )
)

print()


# ==============================================================================
# 11. HARD FROZEN ASSERTIONS
# ==============================================================================

if flow_count != 675:

    raise RuntimeError(
        f"Expected 675 flows; got {flow_count}."
    )


if summary[
    "parser_counts"
] != {
    "IPv4_TCP":
        31_618,

    "NON_IPV4":
        3_964,

    "IPv4_OTHER_PROTOCOL_0":
        84,

    "IPv4_UDP":
        14_334,
}:

    raise RuntimeError(
        "Parser geometry mismatch."
    )


if summary[
    "export_counts"
].get(
    "FIN",
    0,
) != 432:

    raise RuntimeError(
        "FIN mismatch."
    )


if summary[
    "export_counts"
].get(
    "FLOW_TIMEOUT",
    0,
) != 243:

    raise RuntimeError(
        "FLOW_TIMEOUT mismatch."
    )


if dict(
    sorted(
        protocol_counts.items()
    )
) != {
    0:
        1,

    6:
        467,

    17:
        207,
}:

    raise RuntimeError(
        "Protocol-count mismatch."
    )


if sum(
    raw_set
) != 637:

    raise RuntimeError(
        f"Raw SET expected 637; got {sum(raw_set)}."
    )


if sum(
    raw_multi
) != 637:

    raise RuntimeError(
        f"Raw MULTISET expected 637; got {sum(raw_multi)}."
    )


if set_absent != expected_raw:

    raise RuntimeError(
        "\nSET absent mismatch.\n"
        f"Expected: {sorted(expected_raw)}\n"
        f"Actual:   {sorted(set_absent)}"
    )


if multi_absent != expected_raw:

    raise RuntimeError(
        "\nMULTISET absent mismatch.\n"
        f"Expected: {sorted(expected_raw)}\n"
        f"Actual:   {sorted(multi_absent)}"
    )


if sum(
    d5
) != 635:

    raise RuntimeError(
        f"D5 expected 635; got {sum(d5)}."
    )


if d5_absent != expected_d5:

    raise RuntimeError(
        "\nD5 absent mismatch.\n"
        f"Expected: {sorted(expected_d5)}\n"
        f"Actual:   {sorted(d5_absent)}"
    )


if (
    audit[
        199
    ][
        "protocol"
    ] != 0

    or

    audit[
        199
    ][
        "raw"
    ] is not False
):

    raise RuntimeError(
        "Flow 199 anchor failed."
    )


for idx, expected_duration in [
    (
        471,
        224,
    ),
    (
        473,
        262,
    ),
]:

    item = audit[
        idx
    ]


    if (
        item[
            "protocol"
        ] != 6

        or

        item[
            "duration"
        ] != expected_duration

        or

        item[
            "termination"
        ] != "FIN"

        or

        item[
            "raw"
        ] is not True

        or

        item[
            "d5"
        ] is not False
    ):

        raise RuntimeError(
            f"D5 anchor {idx} failed: {item}"
        )


print("=" * 112)
print("STAGE24-1C1-M-RUNTIME-RECOVERY-R6: PASS")
print("=" * 112)

print()

print(
    "PCAP format:                  PCAPNG"
)

print(
    "Historical capture rule:     ",
    CAPTURE_RULE,
)

print(
    "Frozen C16 identity:          5093 / 5093"
)

print(
    "Historical raw exact:         637 / 675"
)

print(
    "Historical D5 accepted:       635 / 675"
)

print()

print(
    "Historical PCAPNG iterator is LIVE in this kernel."
)

print(
    "DO NOT RESET THIS SESSION."
)

print()

print(
    "Completed scientific fits:    2 / 4"
)

print(
    "Target openings:              0 / 8"
)

print(
    "Model executions:             0"
)

print()

print(
    "NEXT: FULL MONDAY GROUNDED_S4 REPLAY + REMOTE FREEZE."
)

print("=" * 112)

STAGE24-1C1-M-RUNTIME-RECOVERY-R6 — PCAPNG BLOCK LEVEL
Frozen C16: 5093
GT: 4930
LT: 163
Max raw index: 48303

PCAPNG BLOCK-LEVEL C16 AUDIT
Candidate                                      Exact    Mismatch
----------------------------------------------------------------
DIRECT_PACKET_BYTES_MINUS_L2                    4922         171
  GT exact: 4922 / 4930
  LT exact: 0 / 163
  examples: [{'index': 288, 'cohort': 'LT', 'frozen_extent': 384, 'candidate_extent': 390, 'declared_ipv4': 390, 'epb_captured_len': 404, 'epb_original_len': 404, 'raw_bytes': 404, 'l2_offset': 14}, {'index': 289, 'cohort': 'LT', 'frozen_extent': 384, 'candidate_extent': 390, 'declared_ipv4': 390, 'epb_captured_len': 404, 'epb_original_len': 404, 'raw_bytes': 404, 'l2_offset': 14}, {'index': 291, 'cohort': 'LT', 'frozen_extent': 2552, 'candidate_extent': 2674, 'declared_ipv4': 2674, 'epb_captured_len': 2688, 'epb_original_len': 2688, 'raw_bytes': 2688, 'l2_offset': 14}, {'index': 317, 'cohort': 'LT', 'frozen_exten

RuntimeError: 
Expected exactly one C16-exact PCAPNG extent rule.
Passing: []

Paste the table above.

In [14]:
# ==============================================================================
# STAGE24-1C1-M-RUNTIME-RECOVERY-R7
#
# TEST THE HISTORICAL SCAPY INITIALIZATION PATH:
#
#   from scapy.all import *
#   PcapReader(...)
#   len(bytes(tcp.payload))
#
# Why:
#   The frozen Stage20 baseline is explicitly LEN_BYTES_TCP_PAYLOAD.
#   Our previous recovery cells used RawPcapReader + low-level layer imports.
#   Scapy layer registration/dissection can therefore differ even at 2.6.1.
#
# LABEL-FREE:
#   compares only against pre-existing C16 packet records.
#
# BOUNDED:
#   first 50,000 packets only.
#
# NO model.
# NO commit.
# NO target opening.
# ==============================================================================

from collections import Counter, defaultdict

import scapy
import scapy.all as sall


print("=" * 112)
print("STAGE24-1C1-M-RUNTIME-RECOVERY-R7 — SCAPY.ALL + PCAPREADER")
print("=" * 112)

print("Scapy version:", scapy.__version__)

if scapy.__version__ != "2.6.1":
    raise RuntimeError(
        f"Frozen Scapy version required: 2.6.1; got {scapy.__version__}"
    )


assert "PCAP" in globals()
assert "d1" in globals()
assert "d2" in globals()


# ==============================================================================
# 1. FROZEN C16 RECORDS
# ==============================================================================

frozen = {}

for cohort, obj in [
    ("GT", d1),
    ("LT", d2),
]:

    for row in obj["per_packet"]:

        idx = int(
            row["raw_index"]
        )

        frozen[idx] = {
            "cohort":
                cohort,

            "baseline_payload":
                int(
                    row["baseline_payload"]
                ),

            "captured_ipv4_extent":
                int(
                    row["captured_ipv4_extent"]
                ),

            "declared_ipv4_total_length":
                int(
                    row["declared_ipv4_total_length"]
                ),

            "ip_header_length":
                int(
                    row["ip_header_length"]
                ),

            "tcp_header_length":
                int(
                    row["tcp_header_length"]
                ),

            "flow_index":
                int(
                    row["flow_index"]
                ),

            "packet_position":
                int(
                    row["packet_position"]
                ),
        }


EXPECTED = 5093

if len(frozen) != EXPECTED:
    raise RuntimeError(
        f"Expected {EXPECTED} frozen C16 records, got {len(frozen)}"
    )


MAX_INDEX = max(
    frozen
)


print("Frozen records:", len(frozen))
print("Maximum raw index:", MAX_INDEX)
print()


# ==============================================================================
# 2. PCAPREADER — HISTORICAL HIGH-LEVEL SCAPY PATH
# ==============================================================================

counts = {
    "GT":
        Counter(),

    "LT":
        Counter(),
}


payload_classes = {
    "GT":
        Counter(),

    "LT":
        Counter(),
}


mismatch_classes = {
    "GT":
        Counter(),

    "LT":
        Counter(),
}


examples = {
    "GT":
        [],

    "LT":
        [],
}


seen = set()


reader = sall.PcapReader(
    str(
        PCAP
    )
)


try:

    for raw_index, pkt in enumerate(
        reader
    ):

        if raw_index > MAX_INDEX:
            break


        target = frozen.get(
            raw_index
        )


        if target is None:
            continue


        seen.add(
            raw_index
        )


        if sall.IP not in pkt:

            raise RuntimeError(
                f"Frozen C16 packet {raw_index} is not IPv4."
            )


        ip = pkt[
            sall.IP
        ]


        transport = ip.payload


        if not isinstance(
            transport,
            sall.TCP,
        ):

            raise RuntimeError(
                f"Frozen C16 packet {raw_index} is not direct TCP; "
                f"got {type(transport)}"
            )


        tcp = transport


        payload = tcp.payload


        payload_class = (
            payload.__class__.__module__
            + "."
            + payload.__class__.__name__
        )


        cohort = target[
            "cohort"
        ]


        payload_classes[
            cohort
        ][
            payload_class
        ] += 1


        actual_payload_len = len(
            bytes(
                payload
            )
        )


        expected_payload_len = target[
            "baseline_payload"
        ]


        exact = (
            actual_payload_len
            ==
            expected_payload_len
        )


        counts[
            cohort
        ][
            "exact"
            if exact
            else "mismatch"
        ] += 1


        if not exact:

            mismatch_classes[
                cohort
            ][
                payload_class
            ] += 1


            if len(
                examples[
                    cohort
                ]
            ) < 20:

                # Collect layer chain without mutating packet.
                layers = []

                current = payload

                for _ in range(
                    12
                ):

                    if current is None:
                        break


                    layer_name = (
                        current.__class__.__module__
                        + "."
                        + current.__class__.__name__
                    )


                    layers.append(
                        layer_name
                    )


                    next_payload = getattr(
                        current,
                        "payload",
                        None,
                    )


                    if (
                        next_payload is None
                        or
                        next_payload is current
                        or
                        next_payload.__class__.__name__ == "NoPayload"
                    ):

                        break


                    current = next_payload


                examples[
                    cohort
                ].append(
                    {
                        "raw_index":
                            raw_index,

                        "flow_index":
                            target[
                                "flow_index"
                            ],

                        "packet_position":
                            target[
                                "packet_position"
                            ],

                        "src":
                            str(
                                ip.src
                            ),

                        "dst":
                            str(
                                ip.dst
                            ),

                        "sport":
                            int(
                                tcp.sport
                            ),

                        "dport":
                            int(
                                tcp.dport
                            ),

                        "payload_class":
                            payload_class,

                        "layer_chain":
                            layers,

                        "expected_baseline_payload":
                            expected_payload_len,

                        "actual_len_bytes_tcp_payload":
                            actual_payload_len,

                        "delta":
                            (
                                actual_payload_len
                                -
                                expected_payload_len
                            ),

                        "declared_ipv4_total_length":
                            target[
                                "declared_ipv4_total_length"
                            ],

                        "captured_ipv4_extent_frozen":
                            target[
                                "captured_ipv4_extent"
                            ],

                        "ip_header":
                            int(
                                ip.ihl
                            ) * 4,

                        "tcp_header":
                            int(
                                tcp.dataofs
                            ) * 4,
                    }
                )


finally:

    reader.close()


if seen != set(
    frozen
):

    missing = sorted(
        set(
            frozen
        )
        -
        seen
    )

    raise RuntimeError(
        f"Missing C16 packets: {missing[:20]}"
    )


# ==============================================================================
# 3. REPORT
# ==============================================================================

print("=" * 112)
print("C16 HIGH-LEVEL SCAPY RESULT")
print("=" * 112)


for cohort, expected_count in [
    ("GT", 4930),
    ("LT", 163),
]:

    print()
    print(cohort)

    print(
        "  exact:",
        counts[
            cohort
        ][
            "exact"
        ],
        "/",
        expected_count,
    )

    print(
        "  mismatch:",
        counts[
            cohort
        ][
            "mismatch"
        ],
    )

    print(
        "  payload classes:"
    )

    for name, n in (
        payload_classes[
            cohort
        ]
        .most_common()
    ):

        print(
            f"    {n:6d}  {name}"
        )


    if mismatch_classes[
        cohort
    ]:

        print(
            "  mismatch classes:"
        )

        for name, n in (
            mismatch_classes[
                cohort
            ]
            .most_common()
        ):

            print(
                f"    {n:6d}  {name}"
            )


    if examples[
        cohort
    ]:

        print(
            "  first mismatch examples:"
        )

        for item in examples[
            cohort
        ]:

            print(
                "   ",
                item,
            )


total_exact = (
    counts[
        "GT"
    ][
        "exact"
    ]
    +
    counts[
        "LT"
    ][
        "exact"
    ]
)


total_mismatch = (
    counts[
        "GT"
    ][
        "mismatch"
    ]
    +
    counts[
        "LT"
    ][
        "mismatch"
    ]
)


print()
print("=" * 112)
print("DECISION")
print("=" * 112)

print(
    "Total frozen C16 exact:",
    total_exact,
    "/",
    EXPECTED,
)

print(
    "Total mismatch:",
    total_mismatch,
)


if total_exact == EXPECTED:

    print()
    print(
        "[PASS] Historical Stage20 packet decoder recovered:"
    )

    print(
        "       scapy.all + PcapReader + len(bytes(TCP.payload))"
    )

    print()
    print(
        "NEXT: use this exact decoder for the full Monday replay."
    )


else:

    print()
    print(
        "[NOT YET EXACT] Do not run the full Monday replay."
    )

    print(
        "The mismatch payload classes above now identify the exact "
        "missing Scapy layer/runtime component."
    )


print()
print(
    "No model executed."
)

print(
    "No Git commit."
)

print(
    "Target openings remain 0 / 8."
)

print("=" * 112)

STAGE24-1C1-M-RUNTIME-RECOVERY-R7 — SCAPY.ALL + PCAPREADER
Scapy version: 2.6.1
Frozen records: 5093
Maximum raw index: 48303

C16 HIGH-LEVEL SCAPY RESULT

GT
  exact: 4930 / 4930
  mismatch: 0
  payload classes:
      4615  scapy.packet.Padding
       305  scapy.packet.Raw
         8  scapy.layers.kerberos.KerberosTCPHeader
         2  scapy.layers.netbios.NBTSession

LT
  exact: 163 / 163
  mismatch: 0
  payload classes:
       163  scapy.layers.ldap.LDAP

DECISION
Total frozen C16 exact: 5093 / 5093
Total mismatch: 0

[PASS] Historical Stage20 packet decoder recovered:
       scapy.all + PcapReader + len(bytes(TCP.payload))

NEXT: use this exact decoder for the full Monday replay.

No model executed.
No Git commit.
Target openings remain 0 / 8.


In [15]:
# ==============================================================================
# STAGE24-1C1-M-FULL
# MONDAY FULL GROUNDED_S4 MEMBERSHIP REPLAY + REMOTE FREEZE
#
# Proven runtime:
#   Scapy 2.6.1
#   import scapy.all
#   PcapReader
#   direct outer IPv4 payload layer
#   TCP baseline payload = len(bytes(tcp.payload))
#
# PRE-FLIGHT:
#   Re-runs frozen first-50k gate automatically.
#
# FULL EXPECTED:
#   raw packets                  11,709,971
#   valid IPv4                   11,626,492
#   exportable flows                529,601
#   retained packets             11,573,331
#
#   FIN                             216,388
#   FLOW_TIMEOUT                    120,017
#   EOF_CURRENT                     193,196
#   EOF singleton discarded          52,890
#
#   D5 excluded                           2
#   raw unmatched reconstructed         1,090
#   GROUNDED_S4                       528,509
#   unused published occurrences        1,409
#
#   matched reasons:
#       FIN                           216,140
#       FLOW_TIMEOUT                  119,849
#       EOF_CURRENT                   192,520
#
# Scientific state:
#   fits       2 / 4
#   openings   0 / 8
#
# NO MODEL EXECUTION.
# ==============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import math
import base64
import hashlib
import subprocess
from decimal import Decimal
from pathlib import Path
from collections import Counter

import numpy as np
import scapy
import scapy.all as sall


# ==============================================================================
# 0. CONSTANTS
# ==============================================================================

EXPECTED_PARENT = (
    "683fe85839e63dcd5fd8e952fd07931b446652ff"
)

EXPECTED_GROUNDED = 528_509
EXPECTED_LABEL_ROWS = 529_918

EXPECTED_RAW_PACKETS = 11_709_971
EXPECTED_VALID_IPV4 = 11_626_492
EXPECTED_EXPORTABLE = 529_601
EXPECTED_RETAINED_PACKETS = 11_573_331

EXPECTED_FIN = 216_388
EXPECTED_TIMEOUT = 120_017
EXPECTED_EOF = 193_196
EXPECTED_EOF_SINGLETONS = 52_890

EXPECTED_RAW_UNMATCHED = 1_090
EXPECTED_UNUSED_PUBLISHED = 1_409

EXPECTED_MATCHED_REASONS = {
    "FIN": 216_140,
    "FLOW_TIMEOUT": 119_849,
    "EOF_CURRENT": 192_520,
}

EXPECTED_MATCHED_BINARY = {
    0: 528_509,
}

D5_EXCLUDED = {
    471,
    473,
}

EXPECTED_RAW_ABSENT_LOCAL = {
    14, 25, 35, 36, 40, 43, 45, 50, 52, 54,
    57, 59, 63, 66, 68, 71, 119, 123, 124, 147,
    148, 197, 198, 199, 277, 279, 281, 307, 309,
    324, 327, 333, 334, 336, 337, 445, 448, 454,
}

REASON_TO_CODE = {
    "FIN": 1,
    "FLOW_TIMEOUT": 2,
    "EOF_CURRENT": 3,
}


# ==============================================================================
# 1. REQUIRED LIVE STATE
# ==============================================================================

required_globals = [
    "REPO",
    "PCAP",
    "LABELS",
    "label_entries",
    "ActiveFlows",
    "historical_flow_key",
    "FLOW_TIMEOUT_US",
    "semantic_tcp_flags",
    "ZERO_FLAGS",
]

missing = [
    name
    for name in required_globals
    if name not in globals()
]

if missing:

    raise RuntimeError(
        "Required Stage24 live state missing:\n"
        + repr(missing)
        + "\n\nDO NOT continue in a reset kernel."
    )


REPO = Path(REPO)
PCAP = Path(PCAP)
LABELS = Path(LABELS)


if not PCAP.is_file():

    raise RuntimeError(
        f"Monday PCAP missing:\n{PCAP}"
    )


if not LABELS.is_file():

    raise RuntimeError(
        f"Monday label source missing:\n{LABELS}"
    )


if scapy.__version__ != "2.6.1":

    raise RuntimeError(
        f"Expected Scapy 2.6.1; got {scapy.__version__}"
    )


OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1c_grounded_s4_membership"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


MEMBERSHIP = (
    OUT
    / "monday_grounded_s4_membership.npz"
)

MEMBERSHIP_SHA = (
    OUT
    / "monday_grounded_s4_membership.sha256"
)

RECEIPT = (
    OUT
    / "stage24_1c1m_monday_grounded_s4_replay.json"
)

RECEIPT_SHA = (
    OUT
    / "stage24_1c1m_monday_grounded_s4_replay.sha256"
)


print("=" * 112)
print("STAGE24-1C1-M-FULL — MONDAY GROUNDED_S4 REPLAY")
print("=" * 112)

print("Scapy:", scapy.__version__)
print("PCAP:  ", PCAP)
print("Labels:", LABELS)

print()
print("Completed scientific fits: 2 / 4")
print("Target openings:           0 / 8")
print("Model execution:           NONE")
print()


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if (
        check
        and
        p.returncode != 0
    ):

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(str(x) for x in args)
            + "\n\n"
            + (p.stdout or "")
        )

    return (
        p.stdout
        or ""
    ).strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    cmd = [
        "git",
    ]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [
        str(x)
        for x in args
    ]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):

    h = hashlib.sha256()

    size = 0

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            chunk = fh.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

            size += len(
                chunk
            )

    return (
        h.hexdigest(),
        size,
    )


def write_json(
    path,
    obj,
):

    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )


# ==============================================================================
# 3. GIT / GOVERNANCE GATE
# ==============================================================================

print("=" * 112)
print("GOVERNANCE")
print("=" * 112)


head = git_cmd(
    "rev-parse",
    "HEAD",
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected repository HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


status = git_cmd(
    "status",
    "--porcelain",
)


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


print(
    "HEAD:",
    head,
)

print(
    "[PASS] Clean repository."
)

print(
    "[PASS] Target openings remain 0 / 8."
)

print()


# ==============================================================================
# 4. GET GITHUB TOKEN BEFORE EXPENSIVE REPLAY
# ==============================================================================

token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if (
                value
                and value.strip()
            ):

                token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if (
            value
            and value.strip()
        ):

            token = value.strip()
            token_source = (
                "ENV:"
                + name
            )
            break


if token is None:

    raise RuntimeError(
        "GitHub token unavailable. "
        "Stopping BEFORE full Monday replay."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        + token
    ).encode()
).decode()


print(
    "GitHub credential:",
    token_source,
)

print()


# ==============================================================================
# 5. VERIFY REMOTE PARENT BEFORE EXPENSIVE REPLAY
# ==============================================================================

remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved unexpectedly.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "[PASS] Remote main == expected Stage24-1B parent."
)

print()


# ==============================================================================
# 6. HIGH-LEVEL SCAPY PACKET TIME
# ==============================================================================

ONE_MILLION = Decimal(
    1_000_000
)


def packet_time_us(
    pkt,
):

    value = Decimal(
        str(
            pkt.time
        )
    )


    microseconds = (
        value
        * ONE_MILLION
    )


    integral = (
        microseconds
        .to_integral_value()
    )


    if microseconds != integral:

        raise RuntimeError(
            "Packet timestamp is not exactly "
            "microsecond representable:\n"
            f"{pkt.time!r}"
        )


    return int(
        integral
    )


# ==============================================================================
# 7. EXACT HISTORICAL STAGE20 PACKET DECODER
#
# Proven by R7:
#
#   scapy.all
#   PcapReader
#   direct ip.payload
#   TCP payload = len(bytes(tcp.payload))
#
# ==============================================================================

def parse_stage20_packet(
    pkt,
):

    if sall.IP not in pkt:

        return None


    ip = pkt[
        sall.IP
    ]


    src = socket.inet_aton(
        str(
            ip.src
        )
    )

    dst = socket.inet_aton(
        str(
            ip.dst
        )
    )


    transport = ip.payload


    protocol = 0
    src_port = 0
    dst_port = 0

    payload_length = 0

    flags = dict(
        ZERO_FLAGS
    )

    parser_class = (
        "IPv4_OTHER_PROTOCOL_0"
    )


    # --------------------------------------------------------------------------
    # Direct TCP only.
    # --------------------------------------------------------------------------

    if isinstance(
        transport,
        sall.TCP,
    ):

        tcp = transport


        protocol = 6


        src_port = int(
            tcp.sport
        )

        dst_port = int(
            tcp.dport
        )


        flags = semantic_tcp_flags(
            tcp
        )


        # THIS IS THE R7-PROVEN HISTORICAL BASELINE.
        payload_length = len(
            bytes(
                tcp.payload
            )
        )


        parser_class = (
            "IPv4_TCP"
        )


    # --------------------------------------------------------------------------
    # Direct UDP only.
    # --------------------------------------------------------------------------

    elif isinstance(
        transport,
        sall.UDP,
    ):

        udp = transport


        protocol = 17


        src_port = int(
            udp.sport
        )

        dst_port = int(
            udp.dport
        )


        payload_length = len(
            bytes(
                udp.payload
            )
        )


        parser_class = (
            "IPv4_UDP"
        )


    return {
        "src":
            src,

        "dst":
            dst,

        "src_port":
            src_port,

        "dst_port":
            dst_port,

        "protocol":
            protocol,

        "payload_length":
            int(
                payload_length
            ),

        "fin":
            bool(
                flags[
                    "FIN"
                ]
            ),

        "syn":
            bool(
                flags[
                    "SYN"
                ]
            ),

        "rst":
            bool(
                flags[
                    "RST"
                ]
            ),

        "psh":
            bool(
                flags[
                    "PSH"
                ]
            ),

        "ack":
            bool(
                flags[
                    "ACK"
                ]
            ),

        "urg":
            bool(
                flags[
                    "URG"
                ]
            ),

        "ece":
            bool(
                flags[
                    "ECE"
                ]
            ),

        "cwr":
            bool(
                flags[
                    "CWR"
                ]
            ),

        "parser_class":
            parser_class,
    }


# ==============================================================================
# 8. EXACT HIGH-LEVEL STAGE20 FLOW ITERATOR
# ==============================================================================

def iterate_stage20_flows(
    pcap_path,
    *,
    packet_limit=None,
    include_eof=False,
    progress_every=None,
):

    active = ActiveFlows()


    parser_counts = Counter()
    export_counts = Counter()


    raw_packet_count = 0
    valid_ipv4_count = 0

    exported_flow_count = 0
    exported_packet_total = 0

    timeout_singletons = 0


    started = time.time()


    reader = sall.PcapReader(
        str(
            pcap_path
        )
    )


    try:

        for pkt in reader:

            if (
                packet_limit is not None
                and
                raw_packet_count
                >= packet_limit
            ):

                break


            raw_packet_count += 1


            parsed = parse_stage20_packet(
                pkt
            )


            if parsed is None:

                parser_counts[
                    "NON_IPV4"
                ] += 1


            else:

                valid_ipv4_count += 1


                parser_counts[
                    parsed[
                        "parser_class"
                    ]
                ] += 1


                ts_us = packet_time_us(
                    pkt
                )


                key = historical_flow_key(
                    parsed[
                        "src"
                    ],
                    parsed[
                        "dst"
                    ],
                    parsed[
                        "src_port"
                    ],
                    parsed[
                        "dst_port"
                    ],
                    parsed[
                        "protocol"
                    ],
                )


                flow = active.get(
                    key
                )


                # ----------------------------------------------------------
                # New flow.
                # ----------------------------------------------------------

                if flow is None:

                    active.create_flow(
                        key,
                        parsed,
                        ts_us,
                    )


                # ----------------------------------------------------------
                # Timeout checked before FIN.
                # ----------------------------------------------------------

                elif (
                    ts_us
                    -
                    flow.start_us
                    >
                    FLOW_TIMEOUT_US
                ):

                    inherited_orientation = (
                        bytes(
                            flow.src
                        ),
                        bytes(
                            flow.dst
                        ),
                        int(
                            flow.src_port
                        ),
                        int(
                            flow.dst_port
                        ),
                    )


                    if flow.packet_count > 1:

                        exported_flow_count += 1


                        exported_packet_total += int(
                            flow.packet_count
                        )


                        export_counts[
                            "FLOW_TIMEOUT"
                        ] += 1


                        yield (
                            "FLOW",
                            flow,
                            "FLOW_TIMEOUT",
                        )


                    else:

                        timeout_singletons += 1


                    active.remove(
                        key
                    )


                    active.create_flow(
                        key,
                        parsed,
                        ts_us,
                        inherited_orientation=(
                            inherited_orientation
                        ),
                    )


                # ----------------------------------------------------------
                # Existing flow + FIN.
                # ----------------------------------------------------------

                elif parsed[
                    "fin"
                ]:

                    flow.add_packet(
                        parsed,
                        ts_us,
                    )


                    exported_flow_count += 1


                    exported_packet_total += int(
                        flow.packet_count
                    )


                    export_counts[
                        "FIN"
                    ] += 1


                    yield (
                        "FLOW",
                        flow,
                        "FIN",
                    )


                    active.remove(
                        key
                    )


                # ----------------------------------------------------------
                # Normal packet.
                # ----------------------------------------------------------

                else:

                    flow.add_packet(
                        parsed,
                        ts_us,
                    )


            # Explicit heartbeat so Kaggle stays visibly active.
            if (
                progress_every is not None
                and
                raw_packet_count
                % progress_every
                == 0
            ):

                elapsed = (
                    time.time()
                    -
                    started
                )


                yield (
                    "PROGRESS",
                    {
                        "raw_packets":
                            int(
                                raw_packet_count
                            ),

                        "valid_ipv4":
                            int(
                                valid_ipv4_count
                            ),

                        "active_flows":
                            int(
                                len(
                                    active
                                )
                            ),

                        "exported_flows":
                            int(
                                exported_flow_count
                            ),

                        "elapsed_seconds":
                            float(
                                elapsed
                            ),
                    },
                    None,
                )


    finally:

        reader.close()


    eof_singletons = 0


    if include_eof:

        for key, flow in (
            active
            .eof_items_historical_bucket_order()
        ):

            if flow.packet_count > 1:

                exported_flow_count += 1


                exported_packet_total += int(
                    flow.packet_count
                )


                export_counts[
                    "EOF_CURRENT"
                ] += 1


                yield (
                    "FLOW",
                    flow,
                    "EOF_CURRENT",
                )


            else:

                eof_singletons += 1


    yield (
        "SUMMARY",
        {
            "raw_packet_count":
                int(
                    raw_packet_count
                ),

            "valid_ipv4_count":
                int(
                    valid_ipv4_count
                ),

            "parser_counts":
                dict(
                    parser_counts
                ),

            "export_counts":
                dict(
                    export_counts
                ),

            "exported_flow_count":
                int(
                    exported_flow_count
                ),

            "exported_packet_total":
                int(
                    exported_packet_total
                ),

            "timeout_singletons":
                int(
                    timeout_singletons
                ),

            "eof_singletons":
                int(
                    eof_singletons
                ),

            "java_capacity":
                int(
                    active.capacity
                ),
        },
        None,
    )


# ==============================================================================
# 9. FIRST-50K PRE-FLIGHT
#
# This is an execution gate, NOT another runtime-selection experiment.
# The decoder has already been frozen by R7.
# ==============================================================================

print("=" * 112)
print("FIRST-50K EXECUTION PRE-FLIGHT")
print("=" * 112)


pilot_remaining = {
    signature:
        int(
            entry[
                1
            ]
        )
    for signature, entry
    in label_entries.items()
}


pilot_set = []
pilot_multi = []
pilot_d5 = []

pilot_protocols = Counter()

pilot_summary = None
pilot_count = 0

pilot_audit = {}


for kind, payload, reason in iterate_stage20_flows(
    PCAP,
    packet_limit=50_000,
    include_eof=False,
):

    if kind == "FLOW":

        idx = int(
            pilot_count
        )

        pilot_count += 1


        pilot_protocols[
            int(
                payload.protocol
            )
        ] += 1


        signature = (
            payload.s4_signature()
        )


        set_match = (
            signature
            in label_entries
        )


        available = int(
            pilot_remaining.get(
                signature,
                0,
            )
        )


        multi_match = (
            available > 0
        )


        if multi_match:

            pilot_remaining[
                signature
            ] = (
                available
                - 1
            )


        d5_match = (
            multi_match
            and
            idx not in D5_EXCLUDED
        )


        pilot_set.append(
            set_match
        )

        pilot_multi.append(
            multi_match
        )

        pilot_d5.append(
            d5_match
        )


        if idx in {
            199,
            471,
            473,
        }:

            pilot_audit[
                idx
            ] = {
                "protocol":
                    int(
                        payload.protocol
                    ),

                "duration":
                    int(
                        payload.last_us
                        -
                        payload.start_us
                    ),

                "reason":
                    reason,

                "raw":
                    bool(
                        multi_match
                    ),

                "d5":
                    bool(
                        d5_match
                    ),
            }


    elif kind == "SUMMARY":

        pilot_summary = payload


if pilot_summary is None:

    raise RuntimeError(
        "Pilot summary missing."
    )


pilot_set_absent = {
    i
    for i, value
    in enumerate(
        pilot_set
    )
    if not value
}


pilot_multi_absent = {
    i
    for i, value
    in enumerate(
        pilot_multi
    )
    if not value
}


pilot_d5_absent = {
    i
    for i, value
    in enumerate(
        pilot_d5
    )
    if not value
}


expected_d5_absent = (
    EXPECTED_RAW_ABSENT_LOCAL
    |
    D5_EXCLUDED
)


print(
    "Parser:",
    pilot_summary[
        "parser_counts"
    ],
)

print(
    "Flows:",
    pilot_count,
)

print(
    "Reasons:",
    pilot_summary[
        "export_counts"
    ],
)

print(
    "Protocols:",
    dict(
        sorted(
            pilot_protocols.items()
        )
    ),
)

print(
    "Raw SET:",
    sum(
        pilot_set
    ),
    "/ 675",
)

print(
    "Raw MULTISET:",
    sum(
        pilot_multi
    ),
    "/ 675",
)

print(
    "D5:",
    sum(
        pilot_d5
    ),
    "/ 675",
)


if pilot_count != 675:

    raise RuntimeError(
        f"Pilot flow count = {pilot_count}, expected 675."
    )


if pilot_summary[
    "parser_counts"
] != {
    "IPv4_TCP": 31_618,
    "NON_IPV4": 3_964,
    "IPv4_OTHER_PROTOCOL_0": 84,
    "IPv4_UDP": 14_334,
}:

    raise RuntimeError(
        "Pilot parser geometry mismatch."
    )


if pilot_summary[
    "export_counts"
].get(
    "FIN",
    0,
) != 432:

    raise RuntimeError(
        "Pilot FIN mismatch."
    )


if pilot_summary[
    "export_counts"
].get(
    "FLOW_TIMEOUT",
    0,
) != 243:

    raise RuntimeError(
        "Pilot FLOW_TIMEOUT mismatch."
    )


if sum(
    pilot_set
) != 637:

    raise RuntimeError(
        f"Pilot raw SET = {sum(pilot_set)}, expected 637."
    )


if sum(
    pilot_multi
) != 637:

    raise RuntimeError(
        f"Pilot raw MULTISET = {sum(pilot_multi)}, expected 637."
    )


if (
    pilot_set_absent
    != EXPECTED_RAW_ABSENT_LOCAL
):

    raise RuntimeError(
        "Pilot exact-set absent indices mismatch."
    )


if (
    pilot_multi_absent
    != EXPECTED_RAW_ABSENT_LOCAL
):

    raise RuntimeError(
        "Pilot exact-multiset absent indices mismatch."
    )


if sum(
    pilot_d5
) != 635:

    raise RuntimeError(
        f"Pilot D5 = {sum(pilot_d5)}, expected 635."
    )


if (
    pilot_d5_absent
    != expected_d5_absent
):

    raise RuntimeError(
        "Pilot D5 absent indices mismatch."
    )


if (
    pilot_audit[
        199
    ][
        "protocol"
    ] != 0
    or
    pilot_audit[
        199
    ][
        "raw"
    ]
):

    raise RuntimeError(
        "Flow 199 anchor failed."
    )


for idx, duration in [
    (471, 224),
    (473, 262),
]:

    item = pilot_audit[
        idx
    ]


    if (
        item[
            "protocol"
        ] != 6
        or
        item[
            "duration"
        ] != duration
        or
        item[
            "reason"
        ] != "FIN"
        or
        not item[
            "raw"
        ]
        or
        item[
            "d5"
        ]
    ):

        raise RuntimeError(
            f"D5 anchor {idx} failed: {item}"
        )


print()
print(
    "[PASS] Historical 637/675 -> 635/675 execution gate."
)

print()
print(
    "Starting FULL Monday replay now."
)

print()


# ==============================================================================
# 10. FREE PILOT STATE
# ==============================================================================

del pilot_remaining
del pilot_set
del pilot_multi
del pilot_d5

gc.collect()


# ==============================================================================
# 11. FULL MONDAY EXACT REPLAY
# ==============================================================================

print("=" * 112)
print("FULL MONDAY REPLAY")
print("=" * 112)


# Number of published occurrences already consumed for each exact S4.
consumed = {}


matched_row_indices = []
matched_flow_indices = []
matched_binary_labels = []
matched_reason_codes = []


matched_binary_counts = Counter()
matched_reason_counts = Counter()


raw_unmatched = 0
d5_excluded_count = 0

flow_index = 0

full_summary = None


full_started = time.time()


for kind, payload, reason in iterate_stage20_flows(
    PCAP,
    packet_limit=None,
    include_eof=True,
    progress_every=500_000,
):

    if kind == "PROGRESS":

        elapsed = payload[
            "elapsed_seconds"
        ]


        print(
            f" packets={payload['raw_packets']:,}"
            f" | ipv4={payload['valid_ipv4']:,}"
            f" | exported={payload['exported_flows']:,}"
            f" | active={payload['active_flows']:,}"
            f" | grounded={len(matched_row_indices):,}"
            f" | elapsed={elapsed / 60:.1f} min",
            flush=True,
        )

        continue


    if kind == "SUMMARY":

        full_summary = payload

        continue


    idx = int(
        flow_index
    )


    flow_index += 1


    signature = (
        payload.s4_signature()
    )


    entry = label_entries.get(
        signature
    )


    already = int(
        consumed.get(
            signature,
            0,
        )
    )


    available = (
        0
        if entry is None
        else
        int(
            entry[
                1
            ]
        )
        -
        already
    )


    raw_exact = (
        available > 0
    )


    # --------------------------------------------------------------------------
    # Frozen D5 source-faithful exclusions.
    #
    # They were raw exact but MUST NOT consume a published occurrence.
    # --------------------------------------------------------------------------

    if idx in D5_EXCLUDED:

        if not raw_exact:

            raise RuntimeError(
                f"Frozen D5 flow {idx} is no longer raw exact."
            )


        expected_duration = {
            471: 224,
            473: 262,
        }[
            idx
        ]


        duration = int(
            payload.last_us
            -
            payload.start_us
        )


        if (
            int(
                payload.protocol
            ) != 6
            or
            duration != expected_duration
            or
            reason != "FIN"
        ):

            raise RuntimeError(
                f"D5 anchor failure at {idx}."
            )


        d5_excluded_count += 1

        continue


    # --------------------------------------------------------------------------
    # No exact published occurrence remaining.
    # --------------------------------------------------------------------------

    if not raw_exact:

        raw_unmatched += 1

        continue


    # --------------------------------------------------------------------------
    # Deterministically consume one published physical occurrence.
    # --------------------------------------------------------------------------

    locator = entry[
        2
    ]


    if isinstance(
        locator,
        int,
    ):

        if already != 0:

            raise RuntimeError(
                "Unique published S4 consumed more than once."
            )


        published_row = int(
            locator
        )


    else:

        if already >= len(
            locator
        ):

            raise RuntimeError(
                "Duplicate published S4 cursor overflow."
            )


        published_row = int(
            locator[
                already
            ]
        )


    consumed[
        signature
    ] = (
        already
        + 1
    )


    binary = int(
        entry[
            0
        ]
    )


    matched_row_indices.append(
        published_row
    )


    matched_flow_indices.append(
        idx
    )


    matched_binary_labels.append(
        binary
    )


    matched_reason_codes.append(
        REASON_TO_CODE[
            reason
        ]
    )


    matched_binary_counts[
        binary
    ] += 1


    matched_reason_counts[
        reason
    ] += 1


full_elapsed = (
    time.time()
    -
    full_started
)


if full_summary is None:

    raise RuntimeError(
        "Full Monday summary missing."
    )


print()
print(
    f"Full replay elapsed: {full_elapsed / 60:.2f} minutes"
)

print()


# ==============================================================================
# 12. FULL FROZEN GEOMETRY ASSERTIONS
# ==============================================================================

print("=" * 112)
print("FULL MONDAY FROZEN GEOMETRY")
print("=" * 112)


actual = {
    "raw_packets":
        int(
            full_summary[
                "raw_packet_count"
            ]
        ),

    "valid_ipv4":
        int(
            full_summary[
                "valid_ipv4_count"
            ]
        ),

    "exportable_flows":
        int(
            full_summary[
                "exported_flow_count"
            ]
        ),

    "retained_packets":
        int(
            full_summary[
                "exported_packet_total"
            ]
        ),

    "FIN":
        int(
            full_summary[
                "export_counts"
            ].get(
                "FIN",
                0,
            )
        ),

    "FLOW_TIMEOUT":
        int(
            full_summary[
                "export_counts"
            ].get(
                "FLOW_TIMEOUT",
                0,
            )
        ),

    "EOF_CURRENT":
        int(
            full_summary[
                "export_counts"
            ].get(
                "EOF_CURRENT",
                0,
            )
        ),

    "EOF_SINGLETON_DISCARDED":
        int(
            full_summary[
                "eof_singletons"
            ]
        ),
}


expected_geometry = {
    "raw_packets":
        EXPECTED_RAW_PACKETS,

    "valid_ipv4":
        EXPECTED_VALID_IPV4,

    "exportable_flows":
        EXPECTED_EXPORTABLE,

    "retained_packets":
        EXPECTED_RETAINED_PACKETS,

    "FIN":
        EXPECTED_FIN,

    "FLOW_TIMEOUT":
        EXPECTED_TIMEOUT,

    "EOF_CURRENT":
        EXPECTED_EOF,

    "EOF_SINGLETON_DISCARDED":
        EXPECTED_EOF_SINGLETONS,
}


for name, expected_value in (
    expected_geometry.items()
):

    actual_value = actual[
        name
    ]


    print(
        f"{name:28s}"
        f" expected={expected_value:,}"
        f" actual={actual_value:,}"
    )


    if actual_value != expected_value:

        raise RuntimeError(
            f"Full geometry anchor failed: {name}"
        )


print()
print(
    "[PASS] All historical full-reconstruction anchors reproduced."
)

print()


# ==============================================================================
# 13. EXACT JOIN ASSERTIONS
# ==============================================================================

grounded = len(
    matched_row_indices
)


unused_published = sum(
    int(
        entry[
            1
        ]
    )
    -
    int(
        consumed.get(
            signature,
            0,
        )
    )

    for signature, entry
    in label_entries.items()
)


print("=" * 112)
print("FULL MONDAY EXACT JOIN")
print("=" * 112)


print(
    "Exportable flows:             ",
    f"{flow_index:,}",
)

print(
    "GROUNDED_S4 matched:          ",
    f"{grounded:,}",
)

print(
    "Raw unmatched reconstructed: ",
    f"{raw_unmatched:,}",
)

print(
    "D5 excluded:                 ",
    f"{d5_excluded_count:,}",
)

print(
    "Unused published occurrences:",
    f"{unused_published:,}",
)

print(
    "Matched binary:",
    dict(
        sorted(
            matched_binary_counts.items()
        )
    ),
)

print(
    "Matched reasons:",
    dict(
        sorted(
            matched_reason_counts.items()
        )
    ),
)


if flow_index != EXPECTED_EXPORTABLE:

    raise RuntimeError(
        f"Flow-index population mismatch: {flow_index}"
    )


if grounded != EXPECTED_GROUNDED:

    raise RuntimeError(
        "\nGrounded count mismatch.\n"
        f"Expected: {EXPECTED_GROUNDED}\n"
        f"Actual:   {grounded}"
    )


if raw_unmatched != EXPECTED_RAW_UNMATCHED:

    raise RuntimeError(
        "\nRaw unmatched mismatch.\n"
        f"Expected: {EXPECTED_RAW_UNMATCHED}\n"
        f"Actual:   {raw_unmatched}"
    )


if d5_excluded_count != 2:

    raise RuntimeError(
        f"D5 exclusion count = {d5_excluded_count}, expected 2."
    )


if unused_published != EXPECTED_UNUSED_PUBLISHED:

    raise RuntimeError(
        "\nUnused published occurrence mismatch.\n"
        f"Expected: {EXPECTED_UNUSED_PUBLISHED}\n"
        f"Actual:   {unused_published}"
    )


if (
    dict(
        sorted(
            matched_binary_counts.items()
        )
    )
    != EXPECTED_MATCHED_BINARY
):

    raise RuntimeError(
        "Matched binary population mismatch."
    )


if (
    dict(
        sorted(
            matched_reason_counts.items()
        )
    )
    != EXPECTED_MATCHED_REASONS
):

    raise RuntimeError(
        "\nMatched termination counts mismatch.\n"
        f"Expected: {EXPECTED_MATCHED_REASONS}\n"
        f"Actual:   {dict(matched_reason_counts)}"
    )


print()
print(
    "[PASS] Historical Stage20 exact join reproduced exactly."
)

print()


# ==============================================================================
# 14. MATERIALIZE PHYSICAL PUBLISHED-ROW MEMBERSHIP
# ==============================================================================

print("=" * 112)
print("MATERIALIZING MEMBERSHIP")
print("=" * 112)


published_row_index = np.asarray(
    matched_row_indices,
    dtype=np.int32,
)


reconstructed_flow_index = np.asarray(
    matched_flow_indices,
    dtype=np.int32,
)


binary_label = np.asarray(
    matched_binary_labels,
    dtype=np.uint8,
)


export_reason_code = np.asarray(
    matched_reason_codes,
    dtype=np.uint8,
)


if published_row_index.shape != (
    EXPECTED_GROUNDED,
):

    raise RuntimeError(
        "Published row-index shape mismatch."
    )


if np.unique(
    published_row_index
).size != EXPECTED_GROUNDED:

    raise RuntimeError(
        "A physical published row was consumed more than once."
    )


if (
    int(
        published_row_index.min()
    )
    < 0
    or
    int(
        published_row_index.max()
    )
    >= EXPECTED_LABEL_ROWS
):

    raise RuntimeError(
        "Published physical row locator out of range."
    )


np.savez_compressed(
    MEMBERSHIP,

    published_row_index=
        published_row_index,

    reconstructed_flow_index=
        reconstructed_flow_index,

    binary_label=
        binary_label,

    export_reason_code=
        export_reason_code,
)


membership_digest, membership_bytes = (
    sha256_file(
        MEMBERSHIP
    )
)


MEMBERSHIP_SHA.write_text(
    f"{membership_digest}  {MEMBERSHIP.name}\n",
    encoding="utf-8",
)


print(
    "Rows:",
    f"{EXPECTED_GROUNDED:,}",
)

print(
    "Bytes:",
    f"{membership_bytes:,}",
)

print(
    "SHA256:",
    membership_digest,
)

print()


# ==============================================================================
# 15. REPLAY RECEIPT
# ==============================================================================

receipt = {
    "checkpoint":
        "Stage24-1C1-M",

    "status":
        "MONDAY_GROUNDED_S4_MEMBERSHIP_RECOVERED_AND_FROZEN",

    "repository_parent":
        EXPECTED_PARENT,

    "day":
        "Monday",

    "runtime_recovery": {
        "scapy_version":
            scapy.__version__,

        "import_path":
            "scapy.all",

        "reader":
            "PcapReader",

        "decoder":
            "SCAPY_2_6_1_OUTER_IPV4_PAYLOAD_LAYER",

        "tcp_baseline_payload":
            "LEN_BYTES_TCP_PAYLOAD",

        "frozen_c16_identity":
            "5093/5093",

        "runtime_selection_used_labels":
            False,
    },

    "first_50000_gate": {
        "completed_flows":
            675,

        "raw_exact_set":
            637,

        "raw_exact_multiset":
            637,

        "d5_source_faithful":
            635,

        "decision":
            "PASS",
    },

    "full_reconstruction": {
        **actual,

        "java_capacity_at_eof":
            int(
                full_summary[
                    "java_capacity"
                ]
            ),
    },

    "exact_join": {
        "matching":
            "EXACT_ONLY",

        "fuzzy":
            False,

        "nearest":
            False,

        "tolerance":
            False,

        "label_guided_repair":
            False,

        "supervised_matched_flows":
            grounded,

        "raw_unmatched_reconstructed_flows":
            raw_unmatched,

        "pre_frozen_d5_duration_export_excluded":
            d5_excluded_count,

        "pre_frozen_d5_excluded_indices":
            sorted(
                D5_EXCLUDED
            ),

        "unused_published_occurrences":
            unused_published,

        "matched_binary_counts": {
            str(k):
                int(v)
            for k, v
            in sorted(
                matched_binary_counts.items()
            )
        },

        "matched_export_reasons": {
            str(k):
                int(v)
            for k, v
            in sorted(
                matched_reason_counts.items()
            )
        },
    },

    "duplicate_occurrence_locator": {
        "rule":
            (
                "EARLIEST_STILL_UNUSED_PHYSICAL_PARQUET_ROW_"
                "IN_FROZEN_SOURCE_SCAN_ORDER"
            ),

        "matching_rule_changed":
            False,

        "model_feedback_used":
            False,

        "target_performance_used":
            False,
    },

    "membership_artifact": {
        "path":
            str(
                MEMBERSHIP.relative_to(
                    REPO
                )
            ),

        "rows":
            EXPECTED_GROUNDED,

        "bytes":
            membership_bytes,

        "sha256":
            membership_digest,

        "arrays": {
            "published_row_index":
                "int32",

            "reconstructed_flow_index":
                "int32",

            "binary_label":
                "uint8",

            "export_reason_code":
                "uint8",
        },

        "export_reason_code_map": {
            "1":
                "FIN",

            "2":
                "FLOW_TIMEOUT",

            "3":
                "EOF_CURRENT",
        },
    },

    "scientific_boundary": {
        "model_fit":
            False,

        "model_forward":
            False,

        "target_feature_values_supplied_to_model":
            False,

        "target_predictions":
            0,

        "target_metrics":
            0,

        "target_openings_consumed":
            0,

        "completed_fits":
            2,

        "fit_budget":
            4,

        "target_openings":
            0,

        "target_opening_budget":
            8,
    },

    "elapsed_seconds":
        float(
            full_elapsed
        ),

    "next":
        (
            "RELEASE_MONDAY_RAW_SOURCE_AND_ADVANCE_TO_TUESDAY"
        ),
}


write_json(
    RECEIPT,
    receipt,
)


receipt_digest, receipt_bytes = (
    sha256_file(
        RECEIPT
    )
)


RECEIPT_SHA.write_text(
    f"{receipt_digest}  {RECEIPT.name}\n",
    encoding="utf-8",
)


print(
    "Receipt SHA256:",
    receipt_digest,
)

print()


# ==============================================================================
# 16. COMMIT
# ==============================================================================

print("=" * 112)
print("FREEZING MONDAY MEMBERSHIP IN GIT")
print("=" * 112)


git_cmd(
    "add",
    "--",
    str(
        MEMBERSHIP_SHA.relative_to(
            REPO
        )
    ),
    str(
        RECEIPT.relative_to(
            REPO
        )
    ),
    str(
        RECEIPT_SHA.relative_to(
            REPO
        )
    ),
)


# Membership binary may be ignored globally; force explicitly.
git_cmd(
    "add",
    "-f",
    "--",
    str(
        MEMBERSHIP.relative_to(
            REPO
        )
    ),
)


staged = set(
    line
    for line
    in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
)


expected_staged = {
    str(
        MEMBERSHIP.relative_to(
            REPO
        )
    ),

    str(
        MEMBERSHIP_SHA.relative_to(
            REPO
        )
    ),

    str(
        RECEIPT.relative_to(
            REPO
        )
    ),

    str(
        RECEIPT_SHA.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected:\n{sorted(expected_staged)}\n"
        f"Actual:\n{sorted(staged)}"
    )


git_cmd(
    "commit",
    "-m",
    "stage24: freeze Monday GROUNDED_S4 membership",
)


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "\nMembership commit parent mismatch.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {parent}"
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 17. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 112)
print("PUSHING + VERIFYING REMOTE DURABILITY")
print("=" * 112)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote durability verification FAILED.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}\n\n"
        "DO NOT DELETE MONDAY RAW SOURCE."
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository not clean after freeze:\n"
        + final_status
    )


print()
print(
    "[PASS] Remote main == Monday membership commit."
)

print(
    "[PASS] Worktree clean."
)

print()


# ==============================================================================
# 18. FINAL
# ==============================================================================

print("=" * 112)
print("STAGE24-1C1-M-FULL: PASS")
print("=" * 112)

print()

print(
    "Historical runtime:"
)

print(
    "  Scapy 2.6.1 + scapy.all + PcapReader"
)

print(
    "  TCP payload = len(bytes(tcp.payload))"
)

print()

print(
    "Monday raw packets:           ",
    f"{EXPECTED_RAW_PACKETS:,}",
)

print(
    "Monday exportable flows:       ",
    f"{EXPECTED_EXPORTABLE:,}",
)

print(
    "Monday GROUNDED_S4 rows:       ",
    f"{grounded:,}",
)

print(
    "Raw unmatched reconstructed:  ",
    f"{raw_unmatched:,}",
)

print(
    "D5 exclusions:                 ",
    d5_excluded_count,
)

print(
    "Unused published occurrences: ",
    f"{unused_published:,}",
)

print()

print(
    "Membership SHA256:"
)

print(
    " ",
    membership_digest,
)

print()

print(
    "Replay receipt SHA256:"
)

print(
    " ",
    receipt_digest,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "Remote verified:               YES"
)

print(
    "Monday raw source retained:    YES"
)

print()

print(
    "Completed scientific fits:     2 / 4"
)

print(
    "Target predictions:            0"
)

print(
    "Target openings:               0 / 8"
)

print()

print(
    "NEXT:"
)

print(
    "  Release Monday raw PCAP/labels only now that membership is remote-durable,"
)

print(
    "  then acquire Tuesday and recover its GROUNDED_S4 membership."
)

print("=" * 112)

STAGE24-1C1-M-FULL — MONDAY GROUNDED_S4 REPLAY
Scapy: 2.6.1
PCAP:   /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/blobs/f6eac599358f216b074338813a1cf7be3cc4e91d116e13efc0dc71f2cca11972
Labels: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/blobs/dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02

Completed scientific fits: 2 / 4
Target openings:           0 / 8
Model execution:           NONE

GOVERNANCE
HEAD: 683fe85839e63dcd5fd8e952fd07931b446652ff
[PASS] Clean repository.
[PASS] Target openings remain 0 / 8.

GitHub credential: GITHUB_TOKEN

[PASS] Remote main == expected Stage24-1B parent.

FIRST-50K EXECUTION PRE-FLIGHT
Parser: {'IPv4_TCP': 31618, 'NON_IPV4': 3964, 'IPv4_OTHER_PROTOCOL_0': 84, 'IPv4_UDP': 14334}
Flows: 675
Reasons: {'FIN': 432, 'FLOW_TIMEOUT': 243}
Protocols: {0: 1, 6: 467, 17: 207}
Raw SET: 637 / 675
Raw MULTISET: 637 / 675
D5: 635 / 675

[PASS] Historical 637/675 -> 635/675 execution gate.

Sta

KeyboardInterrupt: 

In [16]:
# ==============================================================================
# STAGE24 — GITHUB RELEASE ASSET INVENTORY
#
# PURPOSE
# -------
# Inspect GitHub Releases for the already-published Stage20/Stage21 corpus.
#
# DOES NOT:
#   - download release assets
#   - touch PCAPs
#   - run models
#   - modify Git
#   - consume target openings
# ==============================================================================

import os
import requests
from datetime import datetime

OWNER = "themubasshir"
REPO_NAME = "ids2018-validation-safe-ablation"

API_URL = (
    f"https://api.github.com/repos/"
    f"{OWNER}/{REPO_NAME}/releases"
)

print("=" * 110)
print("STAGE24 — GITHUB RELEASE ASSET INVENTORY")
print("=" * 110)
print(f"Repository: {OWNER}/{REPO_NAME}")
print()


# ------------------------------------------------------------------------------
# 1. Recover GitHub token safely
# ------------------------------------------------------------------------------

github_token = None
token_source = None

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:
        try:
            value = secrets.get_secret(name)

            if value and value.strip():
                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:
        value = os.environ.get(name)

        if value and value.strip():
            github_token = value.strip()
            token_source = f"ENV:{name}"
            break


headers = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

if github_token:
    headers["Authorization"] = f"Bearer {github_token}"


print(
    "Authentication:",
    f"GitHub token found ({token_source})"
    if github_token
    else "public GitHub API"
)
print()


# ------------------------------------------------------------------------------
# 2. Retrieve ALL releases with pagination
# ------------------------------------------------------------------------------

releases = []

page = 1

while True:

    response = requests.get(
        API_URL,
        headers=headers,
        params={
            "per_page": 100,
            "page": page,
        },
        timeout=30,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"GitHub API failed: HTTP {response.status_code}\n"
            f"{response.text[:2000]}"
        )

    batch = response.json()

    if not batch:
        break

    releases.extend(batch)

    if len(batch) < 100:
        break

    page += 1


print(f"Releases found: {len(releases)}")
print()


# ------------------------------------------------------------------------------
# 3. Display release + asset inventory
# ------------------------------------------------------------------------------

candidate_assets = []

KEYWORDS = (
    "corpus",
    "stage20",
    "stage21",
    "compact",
    "monday",
    "tuesday",
    "wednesday",
    "thursday",
    "friday",
    "packet",
    "flow",
)


def human_size(num_bytes):

    value = float(num_bytes)

    for unit in [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]:

        if value < 1024 or unit == "TiB":
            return f"{value:.2f} {unit}"

        value /= 1024


if not releases:

    print("[INFO] Repository has no GitHub Releases.")

else:

    for release_index, release in enumerate(
        releases,
        start=1,
    ):

        tag = release.get("tag_name", "")
        name = release.get("name") or ""
        published = release.get("published_at") or ""
        draft = release.get("draft", False)
        prerelease = release.get("prerelease", False)
        assets = release.get("assets", [])


        print("=" * 110)
        print(f"RELEASE {release_index}")
        print("=" * 110)

        print(f"Tag:        {tag}")
        print(f"Name:       {name}")
        print(f"Published:  {published}")
        print(f"Draft:      {draft}")
        print(f"Prerelease: {prerelease}")
        print(f"Assets:     {len(assets)}")
        print()


        if not assets:

            print("  <NO ASSETS>")

        else:

            for asset_index, asset in enumerate(
                assets,
                start=1,
            ):

                asset_name = asset.get("name", "")
                size = int(asset.get("size", 0))
                downloads = int(
                    asset.get(
                        "download_count",
                        0,
                    )
                )

                asset_id = asset.get("id")


                lowered = asset_name.lower()

                is_candidate = any(
                    keyword in lowered
                    for keyword in KEYWORDS
                )


                marker = (
                    "  *** CANDIDATE ***"
                    if is_candidate
                    else ""
                )


                print(
                    f"  [{asset_index:02d}] "
                    f"{asset_name}"
                    f"{marker}"
                )

                print(
                    f"       size:      "
                    f"{size:,} bytes "
                    f"({human_size(size)})"
                )

                print(
                    f"       asset_id:  {asset_id}"
                )

                print(
                    f"       downloads: {downloads}"
                )


                if is_candidate:

                    candidate_assets.append(
                        {
                            "release_tag":
                                tag,

                            "release_name":
                                name,

                            "asset_name":
                                asset_name,

                            "asset_id":
                                asset_id,

                            "size_bytes":
                                size,

                            "size_human":
                                human_size(size),
                        }
                    )


        print()


# ------------------------------------------------------------------------------
# 4. Candidate summary
# ------------------------------------------------------------------------------

print()
print("=" * 110)
print("LIKELY STAGE20 / STAGE21 CORPUS ASSETS")
print("=" * 110)


if not candidate_assets:

    print(
        "No asset names matched the corpus/Stage20/Stage21 keywords."
    )

else:

    for i, item in enumerate(
        candidate_assets,
        start=1,
    ):

        print(
            f"[{i}] "
            f"{item['asset_name']}"
        )

        print(
            f"    release: {item['release_tag']} "
            f"| {item['release_name']}"
        )

        print(
            f"    size:    {item['size_human']} "
            f"({item['size_bytes']:,} bytes)"
        )

        print(
            f"    id:      {item['asset_id']}"
        )


print()
print("=" * 110)
print("INVENTORY COMPLETE")
print("=" * 110)

print()
print("No release asset downloaded.")
print("No PCAP replay performed.")
print("No model executed.")
print("No Git changes.")
print("Target openings remain 0 / 8.")
print()
print("PASTE THE COMPLETE OUTPUT HERE.")

STAGE24 — GITHUB RELEASE ASSET INVENTORY
Repository: themubasshir/ids2018-validation-safe-ablation

Authentication: GitHub token found (GITHUB_TOKEN)

Releases found: 2

RELEASE 1
Tag:        stage21-2-training-recovery-v1
Name:       Stage21-2 Training Recovery
Published:  2026-08-16T09:12:11Z
Draft:      False
Prerelease: True
Assets:     10

  [01] stage21-2-recovery-epoch01.pt  *** CANDIDATE ***
       size:      1,150,463 bytes (1.10 MiB)
       asset_id:  516641039
       downloads: 0
  [02] stage21-2-recovery-epoch02.pt  *** CANDIDATE ***
       size:      1,150,783 bytes (1.10 MiB)
       asset_id:  516642189
       downloads: 0
  [03] stage21-2-recovery-epoch03.pt  *** CANDIDATE ***
       size:      1,151,039 bytes (1.10 MiB)
       asset_id:  516643512
       downloads: 0
  [04] stage21-2-recovery-epoch04.pt  *** CANDIDATE ***
       size:      1,151,359 bytes (1.10 MiB)
       asset_id:  516644902
       downloads: 0
  [05] stage21-2-recovery-epoch05.pt  *** CANDIDATE ***
 

In [17]:
# ==============================================================================
# STAGE24 — MONDAY RELEASE CORPUS ACQUISITION + INVENTORY
#
# Downloads:
#   stage20-Monday-compact-corpus-v1.tar
#   stage20-Monday-compact-corpus-v1.tar.sha256
#
# Then:
#   - verifies release SHA256
#   - lists TAR contents
#   - extracts/prints ONLY tiny metadata/text/JSON members if present
#
# DOES NOT:
#   - extract the full corpus
#   - touch PCAP
#   - run models
#   - modify Git
#   - consume target openings
# ==============================================================================

from pathlib import Path
import os
import re
import json
import hashlib
import tarfile
import requests


OWNER = "themubasshir"
REPO_NAME = "ids2018-validation-safe-ablation"

TAG = "stage20-compact-corpora-v1"

TAR_NAME = "stage20-Monday-compact-corpus-v1.tar"
SHA_NAME = "stage20-Monday-compact-corpus-v1.tar.sha256"

EXPECTED_RELEASE_SIZE = 595_261_440


OUT = Path(
    "/kaggle/working/stage24_release_corpora"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


TAR_PATH = OUT / TAR_NAME
SHA_PATH = OUT / SHA_NAME


print("=" * 112)
print("STAGE24 — MONDAY FROZEN RELEASE CORPUS")
print("=" * 112)

print(
    "Release:",
    TAG,
)

print(
    "Asset:",
    TAR_NAME,
)

print()


# ==============================================================================
# 1. GITHUB TOKEN
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if (
                value
                and value.strip()
            ):

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if (
            value
            and value.strip()
        ):

            github_token = value.strip()
            token_source = (
                "ENV:"
                + name
            )
            break


headers = {
    "Accept":
        "application/octet-stream",

    "X-GitHub-Api-Version":
        "2022-11-28",
}


if github_token:

    headers[
        "Authorization"
    ] = (
        f"Bearer {github_token}"
    )


print(
    "Authentication:",
    (
        f"GitHub token ({token_source})"
        if github_token
        else "public"
    ),
)

print()


# ==============================================================================
# 2. DOWNLOAD HELPER
# ==============================================================================

def download_asset(
    url,
    destination,
    *,
    expected_size=None,
):

    destination = Path(
        destination
    )


    if destination.is_file():

        size = destination.stat().st_size

        if (
            expected_size is None
            or
            size == expected_size
        ):

            print(
                "[FOUND]",
                destination.name,
                f"({size:,} bytes)"
            )

            return


        print(
            "[REPLACE]",
            destination.name,
            "unexpected local size:",
            size,
        )

        destination.unlink()


    print(
        "[DOWNLOAD]",
        destination.name
    )


    with requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=60,
        allow_redirects=True,
    ) as response:

        response.raise_for_status()


        total = response.headers.get(
            "Content-Length"
        )


        total = (
            int(total)
            if total
            else expected_size
        )


        written = 0

        next_report = (
            64
            * 1024
            * 1024
        )


        with destination.open(
            "wb"
        ) as fh:

            for chunk in response.iter_content(
                chunk_size=8 * 1024 * 1024
            ):

                if not chunk:
                    continue


                fh.write(
                    chunk
                )


                written += len(
                    chunk
                )


                if written >= next_report:

                    if total:

                        pct = (
                            written
                            /
                            total
                            *
                            100
                        )

                        print(
                            f"  {written / 2**20:,.1f} MiB"
                            f" / {total / 2**20:,.1f} MiB"
                            f" ({pct:.1f}%)",
                            flush=True,
                        )

                    else:

                        print(
                            f"  {written / 2**20:,.1f} MiB",
                            flush=True,
                        )


                    next_report += (
                        64
                        * 1024
                        * 1024
                    )


    final_size = destination.stat().st_size


    if (
        expected_size is not None
        and
        final_size != expected_size
    ):

        raise RuntimeError(
            "\nDownloaded size mismatch.\n"
            f"Expected: {expected_size:,}\n"
            f"Actual:   {final_size:,}"
        )


    print(
        "[PASS]",
        destination.name,
        f"{final_size:,} bytes",
    )


# ==============================================================================
# 3. DOWNLOAD SHA FILE FIRST
# ==============================================================================

SHA_URL = (
    f"https://github.com/"
    f"{OWNER}/{REPO_NAME}/releases/download/"
    f"{TAG}/{SHA_NAME}"
)


TAR_URL = (
    f"https://github.com/"
    f"{OWNER}/{REPO_NAME}/releases/download/"
    f"{TAG}/{TAR_NAME}"
)


download_asset(
    SHA_URL,
    SHA_PATH,
)


sha_text = SHA_PATH.read_text(
    encoding="utf-8"
).strip()


print()
print(
    "Release SHA file:"
)

print(
    sha_text
)

print()


match = re.search(
    r"\b([0-9a-fA-F]{64})\b",
    sha_text,
)


if not match:

    raise RuntimeError(
        "Could not parse SHA256 from release checksum file."
    )


EXPECTED_TAR_SHA = (
    match.group(1).lower()
)


print(
    "Expected TAR SHA256:",
    EXPECTED_TAR_SHA,
)

print()


# ==============================================================================
# 4. DOWNLOAD MONDAY TAR
# ==============================================================================

download_asset(
    TAR_URL,
    TAR_PATH,
    expected_size=EXPECTED_RELEASE_SIZE,
)


# ==============================================================================
# 5. VERIFY TAR SHA256
# ==============================================================================

print()
print(
    "Computing TAR SHA256..."
)


h = hashlib.sha256()


with TAR_PATH.open(
    "rb"
) as fh:

    while True:

        chunk = fh.read(
            16
            * 1024
            * 1024
        )

        if not chunk:
            break

        h.update(
            chunk
        )


actual_tar_sha = h.hexdigest()


print(
    "Expected:",
    EXPECTED_TAR_SHA,
)

print(
    "Actual:  ",
    actual_tar_sha,
)


if actual_tar_sha != EXPECTED_TAR_SHA:

    raise RuntimeError(
        "Release TAR SHA256 mismatch."
    )


print()
print(
    "[PASS] Monday release TAR is byte-exact."
)

print()


# ==============================================================================
# 6. TAR INVENTORY — NO FULL EXTRACTION
# ==============================================================================

print("=" * 112)
print("MONDAY TAR CONTENTS")
print("=" * 112)


with tarfile.open(
    TAR_PATH,
    mode="r:",
) as tf:

    members = tf.getmembers()


    print(
        "Members:",
        len(
            members
        ),
    )

    print()


    for index, member in enumerate(
        members,
        start=1,
    ):

        kind = (
            "DIR"
            if member.isdir()
            else "FILE"
        )


        print(
            f"[{index:02d}]"
            f" {kind:4s}"
            f" {member.size:12,d} bytes"
            f"  {member.name}"
        )


    # --------------------------------------------------------------------------
    # Show ONLY small metadata-style members.
    # --------------------------------------------------------------------------

    interesting = []


    for member in members:

        if not member.isfile():
            continue


        lower = member.name.lower()


        metadata_name = any(
            token in lower
            for token in [
                "manifest",
                "metadata",
                "receipt",
                "index",
                "signature",
                "s4",
                "flow",
                "source",
                "readme",
                "sha",
            ]
        )


        metadata_ext = lower.endswith(
            (
                ".json",
                ".txt",
                ".md",
                ".csv",
                ".sha256",
            )
        )


        if (
            member.size <= 2_000_000
            and
            (
                metadata_name
                or
                metadata_ext
            )
        ):

            interesting.append(
                member
            )


    print()
    print("=" * 112)
    print("SMALL METADATA / IDENTITY MEMBERS")
    print("=" * 112)


    if not interesting:

        print(
            "<NONE>"
        )


    else:

        for member in interesting:

            print()
            print("-" * 112)
            print(
                member.name,
                f"({member.size:,} bytes)"
            )
            print("-" * 112)


            extracted = tf.extractfile(
                member
            )


            if extracted is None:

                print(
                    "<UNREADABLE>"
                )

                continue


            raw = extracted.read()


            try:

                text = raw.decode(
                    "utf-8"
                )

            except UnicodeDecodeError:

                print(
                    "<BINARY — NOT PRINTED>"
                )

                continue


            # Avoid ridiculous notebook output.
            if len(text) > 30_000:

                print(
                    text[
                        :30_000
                    ]
                )

                print(
                    "\n... <TRUNCATED FOR DISPLAY> ..."
                )

            else:

                print(
                    text
                )


# ==============================================================================
# 7. FINAL
# ==============================================================================

print()
print("=" * 112)
print("STAGE24 MONDAY RELEASE CORPUS INVENTORY: PASS")
print("=" * 112)

print()
print(
    "TAR:",
    TAR_PATH,
)

print(
    "TAR bytes:",
    f"{TAR_PATH.stat().st_size:,}",
)

print(
    "TAR SHA256:",
    actual_tar_sha,
)

print()

print(
    "Full corpus extracted: NO"
)

print(
    "PCAP replay:           NO"
)

print(
    "Model execution:       NO"
)

print(
    "Git changes:           NO"
)

print(
    "Target openings:       0 / 8"
)

print()
print(
    "PASTE THE COMPLETE TAR MEMBER LIST AND METADATA OUTPUT."
)

STAGE24 — MONDAY FROZEN RELEASE CORPUS
Release: stage20-compact-corpora-v1
Asset: stage20-Monday-compact-corpus-v1.tar

Authentication: GitHub token (GITHUB_TOKEN)

[DOWNLOAD] stage20-Monday-compact-corpus-v1.tar.sha256
[PASS] stage20-Monday-compact-corpus-v1.tar.sha256 103 bytes

Release SHA file:
4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20  stage20-Monday-compact-corpus-v1.tar

Expected TAR SHA256: 4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20

[DOWNLOAD] stage20-Monday-compact-corpus-v1.tar
  64.0 MiB / 567.7 MiB (11.3%)
  128.0 MiB / 567.7 MiB (22.5%)
  192.0 MiB / 567.7 MiB (33.8%)
  256.0 MiB / 567.7 MiB (45.1%)
  320.0 MiB / 567.7 MiB (56.4%)
  384.0 MiB / 567.7 MiB (67.6%)
  448.0 MiB / 567.7 MiB (78.9%)
  512.0 MiB / 567.7 MiB (90.2%)
[PASS] stage20-Monday-compact-corpus-v1.tar 595,261,440 bytes

Computing TAR SHA256...
Expected: 4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20
Actual:   4f0b1a4a93df86fad5b11632da50dc12d3

In [18]:
# ==============================================================================
# STAGE24-1C — RESTORE FROZEN MONDAY CORPUS + RELEASE RAW PCAP
#
# Source of truth:
#   GitHub Release: stage20-compact-corpora-v1
#   Asset: stage20-Monday-compact-corpus-v1.tar
#
# This cell:
#   1. verifies release TAR is still byte-exact;
#   2. extracts the 4 frozen Stage20 corpus payload files;
#   3. verifies EVERY internal byte size + SHA256;
#   4. verifies exact shapes/counts;
#   5. preserves Monday traffic-label parquet;
#   6. deletes the 10.8 GB Monday raw PCAP/cache references;
#   7. verifies raw PCAP is gone.
#
# NO model.
# NO target prediction.
# NO Git changes.
# Target openings remain 0 / 8.
# ==============================================================================

from __future__ import annotations

import os
import gc
import tarfile
import hashlib
import shutil
from pathlib import Path

import numpy as np


print("=" * 112)
print("STAGE24-1C — RESTORE FROZEN MONDAY CORPUS + RELEASE RAW PCAP")
print("=" * 112)


# ==============================================================================
# 0. PATHS / FROZEN IDENTITIES
# ==============================================================================

RELEASE_ROOT = Path(
    "/kaggle/working/stage24_release_corpora"
)

TAR_PATH = (
    RELEASE_ROOT
    / "stage20-Monday-compact-corpus-v1.tar"
)

CORPUS_ROOT = Path(
    "/kaggle/working/stage24_frozen_stage20_corpora"
)

MONDAY_DIR = (
    CORPUS_ROOT
    / "Monday"
)


EXPECTED_TAR_SIZE = 595_261_440

EXPECTED_TAR_SHA = (
    "4f0b1a4a93df86fad5b11632da50dc12"
    "d3d4392b65f48b5e7bdefeea31706a20"
)


EXPECTED_FILES = {
    "encoded_bytes.bin": {
        "bytes":
            522_845_159,

        "sha256":
            "27e6f730c9951075f500bedc96b91d215b74a995ee23e1f09e269eaa7a2bd82c",
    },

    "flow_offsets.npy": {
        "bytes":
            4_228_208,

        "sha256":
            "3744c55767896e98f5691210d6820224d9ae8315130ce74bb5d4264c25e4326e",
    },

    "labels.npy": {
        "bytes":
            528_637,

        "sha256":
            "48792b8d6a127b35342cb0789baa6c54396f1100a60ce7225daf08d1c3530424",
    },

    "packet_lengths.npy": {
        "bytes":
            67_649_280,

        "sha256":
            "16547aedafc3aaffcf9dca5ef50c7456cd823ed1ab59965a4c10520f11fb68f8",
    },
}


EXPECTED_FLOWS = 528_509

EXPECTED_ENCODED_BYTES = 522_845_159

EXPECTED_PCAP_SIZE = 10_822_507_416

EXPECTED_PCAP_SHA = (
    "f6eac599358f216b074338813a1cf7be"
    "3cc4e91d116e13efc0dc71f2cca11972"
)


STAGE24_HF_CACHE = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache"
)


# Preserve label path if live variable exists.
LABEL_PATH = None

if "LABELS" in globals():

    candidate = Path(
        LABELS
    )

    if candidate.exists():

        LABEL_PATH = candidate


# Known Monday label blob fallback.
if LABEL_PATH is None:

    candidate = (
        STAGE24_HF_CACHE
        / "datasets--bvsam--cic-ids-2017"
        / "blobs"
        / "dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02"
    )

    if candidate.exists():

        LABEL_PATH = candidate


print(
    "Release TAR:",
    TAR_PATH,
)

print(
    "Corpus destination:",
    MONDAY_DIR,
)

print(
    "Monday label retained:",
    LABEL_PATH,
)

print()


# ==============================================================================
# 1. HELPERS
# ==============================================================================

def sha256_file(
    path,
    chunk_size=32 * 1024 * 1024,
):

    path = Path(
        path
    )

    h = hashlib.sha256()

    total = 0


    with path.open(
        "rb"
    ) as fh:

        while True:

            chunk = fh.read(
                chunk_size
            )

            if not chunk:
                break


            h.update(
                chunk
            )

            total += len(
                chunk
            )


    return (
        h.hexdigest(),
        total,
    )


def human_gib(
    value,
):

    return (
        float(
            value
        )
        /
        1024**3
    )


# ==============================================================================
# 2. RELEASE TAR RE-VERIFY
# ==============================================================================

print("=" * 112)
print("RELEASE TRANSPORT IDENTITY")
print("=" * 112)


if not TAR_PATH.is_file():

    raise RuntimeError(
        f"Monday release TAR missing:\n{TAR_PATH}"
    )


tar_size = int(
    TAR_PATH.stat().st_size
)


if tar_size != EXPECTED_TAR_SIZE:

    raise RuntimeError(
        "\nMonday TAR size mismatch.\n"
        f"Expected: {EXPECTED_TAR_SIZE:,}\n"
        f"Actual:   {tar_size:,}"
    )


print(
    "TAR bytes:",
    f"{tar_size:,}",
)


print(
    "Computing TAR SHA256..."
)


tar_sha, _ = sha256_file(
    TAR_PATH
)


print(
    "Expected:",
    EXPECTED_TAR_SHA,
)

print(
    "Actual:  ",
    tar_sha,
)


if tar_sha != EXPECTED_TAR_SHA:

    raise RuntimeError(
        "Monday release TAR SHA mismatch."
    )


print()
print(
    "[PASS] Release TAR byte identity."
)

print()


# ==============================================================================
# 3. EXTRACT ONLY THE FOUR SCIENTIFIC PAYLOAD FILES
# ==============================================================================

print("=" * 112)
print("EXTRACTING FROZEN MONDAY CORPUS")
print("=" * 112)


CORPUS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


if MONDAY_DIR.exists():

    # We never silently replace a non-exact corpus.
    existing_names = {
        p.name
        for p in MONDAY_DIR.iterdir()
        if p.is_file()
    }


    if existing_names:

        print(
            "[FOUND] Existing Monday corpus; verifying before reuse."
        )


    else:

        shutil.rmtree(
            MONDAY_DIR
        )


if not MONDAY_DIR.exists():

    MONDAY_DIR.mkdir(
        parents=True,
        exist_ok=False,
    )


    with tarfile.open(
        TAR_PATH,
        mode="r:",
    ) as tf:

        names = {
            member.name:
                member
            for member in tf.getmembers()
            if member.isfile()
        }


        expected_tar_members = {
            f"Monday/{filename}"
            for filename in EXPECTED_FILES
        }


        if set(
            names
        ) != expected_tar_members:

            raise RuntimeError(
                "\nUnexpected Monday TAR membership.\n"
                f"Expected: {sorted(expected_tar_members)}\n"
                f"Actual:   {sorted(names)}"
            )


        for filename in EXPECTED_FILES:

            member_name = (
                f"Monday/{filename}"
            )


            member = names[
                member_name
            ]


            destination = (
                MONDAY_DIR
                / filename
            )


            print(
                "[EXTRACT]",
                filename,
                f"({member.size:,} bytes)",
                flush=True,
            )


            source = tf.extractfile(
                member
            )


            if source is None:

                raise RuntimeError(
                    f"Unable to read TAR member: {member_name}"
                )


            with destination.open(
                "wb"
            ) as out:

                shutil.copyfileobj(
                    source,
                    out,
                    length=16 * 1024 * 1024,
                )


print()


# ==============================================================================
# 4. VERIFY ALL FOUR SCIENTIFIC FILES
# ==============================================================================

print("=" * 112)
print("FROZEN STAGE20 BYTE IDENTITY")
print("=" * 112)


for filename, expected in (
    EXPECTED_FILES.items()
):

    path = (
        MONDAY_DIR
        / filename
    )


    if not path.is_file():

        raise RuntimeError(
            f"Missing corpus file: {path}"
        )


    actual_size = int(
        path.stat().st_size
    )


    if actual_size != expected[
        "bytes"
    ]:

        raise RuntimeError(
            f"\n{filename}: size mismatch\n"
            f"Expected: {expected['bytes']:,}\n"
            f"Actual:   {actual_size:,}"
        )


    print(
        f"{filename:22s}",
        f"{actual_size:12,d} bytes",
    )


    print(
        "  hashing...",
        flush=True,
    )


    actual_sha, hashed_bytes = (
        sha256_file(
            path
        )
    )


    if hashed_bytes != expected[
        "bytes"
    ]:

        raise RuntimeError(
            f"{filename}: hashed-byte count mismatch."
        )


    if actual_sha != expected[
        "sha256"
    ]:

        raise RuntimeError(
            f"\n{filename}: SHA256 mismatch\n"
            f"Expected: {expected['sha256']}\n"
            f"Actual:   {actual_sha}"
        )


    print(
        "  SHA256:",
        actual_sha,
    )

    print(
        "  [PASS]"
    )


print()
print(
    "[PASS] ALL FOUR frozen Stage20 Monday payloads are byte-exact."
)

print()


# ==============================================================================
# 5. STRUCTURAL CORPUS AUDIT
# ==============================================================================

print("=" * 112)
print("MONDAY CORPUS STRUCTURAL AUDIT")
print("=" * 112)


lengths = np.load(
    MONDAY_DIR
    / "packet_lengths.npy",
    mmap_mode="r",
    allow_pickle=False,
)


offsets = np.load(
    MONDAY_DIR
    / "flow_offsets.npy",
    mmap_mode="r",
    allow_pickle=False,
)


labels = np.load(
    MONDAY_DIR
    / "labels.npy",
    mmap_mode="r",
    allow_pickle=False,
)


if lengths.shape != (
    EXPECTED_FLOWS,
    64,
):

    raise RuntimeError(
        f"packet_lengths shape mismatch: {lengths.shape}"
    )


if lengths.dtype != np.uint16:

    raise RuntimeError(
        f"packet_lengths dtype mismatch: {lengths.dtype}"
    )


if offsets.shape != (
    EXPECTED_FLOWS + 1,
):

    raise RuntimeError(
        f"flow_offsets shape mismatch: {offsets.shape}"
    )


if offsets.dtype != np.uint64:

    raise RuntimeError(
        f"flow_offsets dtype mismatch: {offsets.dtype}"
    )


if labels.shape != (
    EXPECTED_FLOWS,
):

    raise RuntimeError(
        f"labels shape mismatch: {labels.shape}"
    )


if labels.dtype != np.uint8:

    raise RuntimeError(
        f"labels dtype mismatch: {labels.dtype}"
    )


if int(
    offsets[
        0
    ]
) != 0:

    raise RuntimeError(
        "First flow offset is not zero."
    )


if int(
    offsets[
        -1
    ]
) != EXPECTED_ENCODED_BYTES:

    raise RuntimeError(
        "\nFinal flow offset mismatch.\n"
        f"Expected: {EXPECTED_ENCODED_BYTES:,}\n"
        f"Actual:   {int(offsets[-1]):,}"
    )


label_values, label_counts = np.unique(
    labels,
    return_counts=True,
)


class_counts = {
    int(
        value
    ):
        int(
            count
        )
    for value, count
    in zip(
        label_values,
        label_counts,
    )
}


if class_counts != {
    0:
        EXPECTED_FLOWS,
}:

    raise RuntimeError(
        f"Monday frozen class counts changed: {class_counts}"
    )


print(
    "Flows:",
    f"{EXPECTED_FLOWS:,}",
)

print(
    "packet_lengths:",
    lengths.shape,
    lengths.dtype,
)

print(
    "flow_offsets:",
    offsets.shape,
    offsets.dtype,
)

print(
    "labels:",
    labels.shape,
    labels.dtype,
)

print(
    "Class counts:",
    class_counts,
)

print(
    "Encoded authentic bytes:",
    f"{int(offsets[-1]):,}",
)

print()

print(
    "[PASS] Monday release corpus = exact frozen GROUNDED_S4 population."
)

print()


# ==============================================================================
# 6. LABEL PARQUET MUST REMAIN AVAILABLE
# ==============================================================================

print("=" * 112)
print("PUBLISHED LABEL SOURCE")
print("=" * 112)


if (
    LABEL_PATH is None
    or
    not LABEL_PATH.exists()
):

    raise RuntimeError(
        "Monday traffic-label parquet is missing. "
        "Refusing PCAP cleanup because Stage24 still needs published features."
    )


label_bytes = int(
    LABEL_PATH.stat().st_size
)


if label_bytes != 65_465_382:

    raise RuntimeError(
        f"Unexpected Monday label size: {label_bytes:,}"
    )


print(
    "[KEEP]",
    LABEL_PATH,
)

print(
    "Bytes:",
    f"{label_bytes:,}",
)

print()


# ==============================================================================
# 7. IDENTIFY MONDAY RAW PCAP OBJECT(S) INSIDE STAGE24 CACHE
#
# We deliberately restrict deletion to:
#   /kaggle/working/stage24_cicids2017_hf_cache
#
# and only exact 10,822,507,416-byte objects or symlinks resolving to them.
# ==============================================================================

print("=" * 112)
print("MONDAY RAW PCAP RELEASE")
print("=" * 112)


before_free = int(
    shutil.disk_usage(
        "/kaggle/working"
    ).free
)


pcap_candidates = []


if STAGE24_HF_CACHE.exists():

    for root, dirs, files in os.walk(
        STAGE24_HF_CACHE,
        followlinks=False,
    ):

        root_path = Path(
            root
        )


        for name in files:

            path = (
                root_path
                / name
            )


            try:

                if path.is_symlink():

                    resolved = path.resolve(
                        strict=False
                    )


                    if (
                        resolved.exists()
                        and
                        resolved.is_file()
                        and
                        int(
                            resolved.stat().st_size
                        )
                        == EXPECTED_PCAP_SIZE
                    ):

                        pcap_candidates.append(
                            (
                                "symlink",
                                path,
                            )
                        )


                elif path.is_file():

                    if int(
                        path.stat().st_size
                    ) == EXPECTED_PCAP_SIZE:

                        pcap_candidates.append(
                            (
                                "file",
                                path,
                            )
                        )


            except FileNotFoundError:

                pass


# Add the live PCAP variable if available.
if "PCAP" in globals():

    live_pcap = Path(
        PCAP
    )


    if (
        live_pcap.exists()
        and
        live_pcap.is_file()
        and
        int(
            live_pcap.stat().st_size
        )
        == EXPECTED_PCAP_SIZE
    ):

        pcap_candidates.append(
            (
                "file",
                live_pcap,
            )
        )


# Deduplicate paths.
unique_candidates = {}

for kind, path in pcap_candidates:

    unique_candidates[
        str(
            path
        )
    ] = (
        kind,
        path,
    )


pcap_candidates = list(
    unique_candidates.values()
)


print(
    "PCAP objects/references found:",
    len(
        pcap_candidates
    ),
)


for kind, path in (
    pcap_candidates
):

    print(
        f"  [{kind}]",
        path,
    )


if not pcap_candidates:

    print(
        "[INFO] Monday PCAP already absent from Stage24 cache."
    )


# ==============================================================================
# 8. DELETE SYMLINKS FIRST, THEN FULL BLOB(S)
# ==============================================================================

deleted = []


# Symlinks first.
for kind, path in pcap_candidates:

    if (
        kind == "symlink"
        and
        path.is_symlink()
    ):

        path.unlink()

        deleted.append(
            str(
                path
            )
        )


# Files next.
for kind, path in pcap_candidates:

    if kind != "file":

        continue


    try:

        if (
            path.exists()
            and
            path.is_file()
            and
            int(
                path.stat().st_size
            )
            == EXPECTED_PCAP_SIZE
        ):

            path.unlink()

            deleted.append(
                str(
                    path
                )
            )


    except FileNotFoundError:

        pass


gc.collect()


# ==============================================================================
# 9. VERIFY EXACT-SIZE MONDAY PCAP NO LONGER EXISTS IN STAGE24 CACHE
# ==============================================================================

remaining_pcap_objects = []


if STAGE24_HF_CACHE.exists():

    for root, dirs, files in os.walk(
        STAGE24_HF_CACHE,
        followlinks=False,
    ):

        root_path = Path(
            root
        )


        for name in files:

            path = (
                root_path
                / name
            )


            try:

                if (
                    path.is_file()
                    and
                    not path.is_symlink()
                    and
                    int(
                        path.stat().st_size
                    )
                    == EXPECTED_PCAP_SIZE
                ):

                    remaining_pcap_objects.append(
                        str(
                            path
                        )
                    )


            except FileNotFoundError:

                pass


if remaining_pcap_objects:

    raise RuntimeError(
        "\nMonday full PCAP remains in Stage24 cache:\n"
        + "\n".join(
            remaining_pcap_objects
        )
    )


after_free = int(
    shutil.disk_usage(
        "/kaggle/working"
    ).free
)


reclaimed = (
    after_free
    -
    before_free
)


print()

for path in deleted:

    print(
        "[DELETED]",
        path,
    )


print()

print(
    "Workspace free before:",
    f"{human_gib(before_free):.2f} GiB",
)

print(
    "Workspace free after: ",
    f"{human_gib(after_free):.2f} GiB",
)

print(
    "Net reclaimed:        ",
    f"{human_gib(reclaimed):.2f} GiB",
)

print()

print(
    "[PASS] Monday raw PCAP absent from Stage24 cache."
)

print(
    "[PASS] Monday traffic-label parquet retained."
)

print(
    "[PASS] Frozen Monday release corpus retained."
)

print()


# ==============================================================================
# 10. RELEASE LARGE MMAPS
# ==============================================================================

del lengths
del offsets
del labels

gc.collect()


# ==============================================================================
# 11. FINAL
# ==============================================================================

print("=" * 112)
print("STAGE24 MONDAY CORPUS RESTORE + PCAP RELEASE: PASS")
print("=" * 112)

print()

print(
    "Frozen corpus directory:"
)

print(
    " ",
    MONDAY_DIR,
)

print()

print(
    "GROUNDED_S4 Monday flows: 528,509"
)

print(
    "Release corpus exact:     YES"
)

print(
    "Raw Monday PCAP retained: NO"
)

print(
    "Monday label retained:    YES"
)

print()

print(
    "Scientific fits:          2 / 4"
)

print(
    "Target predictions:       0"
)

print(
    "Target openings:          0 / 8"
)

print(
    "Git changes:              NONE"
)

print()

print(
    "NEXT: corpus-side Stage24 Monday alignment; NO PCAP."
)

print("=" * 112)

STAGE24-1C — RESTORE FROZEN MONDAY CORPUS + RELEASE RAW PCAP
Release TAR: /kaggle/working/stage24_release_corpora/stage20-Monday-compact-corpus-v1.tar
Corpus destination: /kaggle/working/stage24_frozen_stage20_corpora/Monday
Monday label retained: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/blobs/dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02

RELEASE TRANSPORT IDENTITY
TAR bytes: 595,261,440
Computing TAR SHA256...
Expected: 4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20
Actual:   4f0b1a4a93df86fad5b11632da50dc12d3d4392b65f48b5e7bdefeea31706a20

[PASS] Release TAR byte identity.

EXTRACTING FROZEN MONDAY CORPUS
[EXTRACT] encoded_bytes.bin (522,845,159 bytes)
[EXTRACT] flow_offsets.npy (4,228,208 bytes)
[EXTRACT] labels.npy (528,637 bytes)
[EXTRACT] packet_lengths.npy (67,649,280 bytes)

FROZEN STAGE20 BYTE IDENTITY
encoded_bytes.bin       522,845,159 bytes
  hashing...
  SHA256: 27e6f730c9951075f500bedc96b91d215b74a995e

In [19]:
# ==============================================================================
# STAGE24-1C1-M-CORPUS-ALIGNMENT-AUDIT
#
# Recover Monday published-row membership using ONLY:
#
#   1. byte-exact frozen Stage20 compact corpus;
#   2. pinned Monday published feature/label parquet;
#   3. frozen corpus ordering.
#
# NO PCAP.
# NO Scapy.
# NO model.
# NO target prediction.
# NO Git changes.
#
# Corpus-preserved alignment fingerprint:
#
#   protocol
#   unordered(source_port, destination_port)
#   min(total_flow_packets, 64)
#
# Why packet count is exact:
#
#   packet_lengths.npy stores one retained packet row for every packet,
#   capped by the frozen 64-row representation.
#
# Therefore:
#
#   count_nonzero(packet_lengths[row]) == min(original_packet_count, 64)
#
# Alignment proof:
#
#   forward greedy  = earliest legal embedding
#   reverse greedy  = latest legal embedding
#
# If both embeddings are identical, physical published-row membership is
# uniquely determined under the frozen corpus-preserved fingerprint/order.
#
# ==============================================================================

from __future__ import annotations

import os
import sys
import gc
import math
import time
import hashlib
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np


print("=" * 112)
print("STAGE24-1C1-M — CORPUS-ONLY PHYSICAL ROW ALIGNMENT AUDIT")
print("=" * 112)


# ==============================================================================
# 0. PATHS
# ==============================================================================

CORPUS_DIR = Path(
    "/kaggle/working/stage24_frozen_stage20_corpora/Monday"
)

ENCODED_PATH = (
    CORPUS_DIR
    / "encoded_bytes.bin"
)

LENGTHS_PATH = (
    CORPUS_DIR
    / "packet_lengths.npy"
)

OFFSETS_PATH = (
    CORPUS_DIR
    / "flow_offsets.npy"
)

CORPUS_LABELS_PATH = (
    CORPUS_DIR
    / "labels.npy"
)


HF_CACHE = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache"
)


PUBLISHED_PATH = (
    HF_CACHE
    / "datasets--bvsam--cic-ids-2017"
    / "blobs"
    / "dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02"
)


OUT = Path(
    "/kaggle/working/stage24_corpus_alignment"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


CANDIDATE_NPZ = (
    OUT
    / "monday_corpus_to_published_row_alignment_candidate.npz"
)


EXPECTED_CORPUS_ROWS = 528_509
EXPECTED_PUBLISHED_ROWS = 529_918
EXPECTED_SKIPPED_PUBLISHED = 1_409
EXPECTED_ENCODED_BYTES = 522_845_159


for path in [
    ENCODED_PATH,
    LENGTHS_PATH,
    OFFSETS_PATH,
    CORPUS_LABELS_PATH,
    PUBLISHED_PATH,
]:

    if not path.is_file():

        raise RuntimeError(
            f"Required file missing:\n{path}"
        )


print(
    "Corpus:",
    CORPUS_DIR,
)

print(
    "Published:",
    PUBLISHED_PATH,
)

print()


# ==============================================================================
# 1. DUCKDB
# ==============================================================================

try:

    import duckdb

except Exception:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "duckdb",
        ],
        check=True,
    )

    import duckdb


print(
    "DuckDB:",
    duckdb.__version__,
)

print()


# ==============================================================================
# 2. LOAD FROZEN CORPUS STRUCTURE
# ==============================================================================

print("=" * 112)
print("FROZEN CORPUS")
print("=" * 112)


lengths = np.load(
    LENGTHS_PATH,
    mmap_mode="r",
    allow_pickle=False,
)


offsets = np.load(
    OFFSETS_PATH,
    mmap_mode="r",
    allow_pickle=False,
)


corpus_labels = np.load(
    CORPUS_LABELS_PATH,
    mmap_mode="r",
    allow_pickle=False,
)


encoded = np.memmap(
    ENCODED_PATH,
    dtype=np.uint8,
    mode="r",
    shape=(
        EXPECTED_ENCODED_BYTES,
    ),
)


if lengths.shape != (
    EXPECTED_CORPUS_ROWS,
    64,
):

    raise RuntimeError(
        f"Unexpected packet_lengths shape: {lengths.shape}"
    )


if offsets.shape != (
    EXPECTED_CORPUS_ROWS + 1,
):

    raise RuntimeError(
        f"Unexpected flow_offsets shape: {offsets.shape}"
    )


if corpus_labels.shape != (
    EXPECTED_CORPUS_ROWS,
):

    raise RuntimeError(
        f"Unexpected corpus labels shape: {corpus_labels.shape}"
    )


if int(
    offsets[
        -1
    ]
) != EXPECTED_ENCODED_BYTES:

    raise RuntimeError(
        "Final encoded offset mismatch."
    )


if not np.all(
    corpus_labels == 0
):

    raise RuntimeError(
        "Frozen Monday corpus is no longer all BENIGN."
    )


print(
    "Flows:",
    f"{EXPECTED_CORPUS_ROWS:,}",
)

print(
    "Encoded bytes:",
    f"{EXPECTED_ENCODED_BYTES:,}",
)

print(
    "Labels:",
    "{0: 528509}",
)

print()


# ==============================================================================
# 3. PACKET-COUNT PROJECTION
#
# Vectorized. This reads only the 64 x uint16 length matrix.
# ==============================================================================

print(
    "Computing frozen capped packet counts..."
)


t0 = time.time()


corpus_packet_count_capped = np.count_nonzero(
    lengths,
    axis=1,
).astype(
    np.uint8,
    copy=False,
)


if (
    int(
        corpus_packet_count_capped.min()
    )
    < 1
    or
    int(
        corpus_packet_count_capped.max()
    )
    > 64
):

    raise RuntimeError(
        "Invalid corpus packet-count projection."
    )


print(
    "  elapsed:",
    f"{time.time() - t0:.2f}s",
)

print(
    "  min:",
    int(
        corpus_packet_count_capped.min()
    ),
)

print(
    "  max:",
    int(
        corpus_packet_count_capped.max()
    ),
)

print(
    "  capped-at-64:",
    f"{int(np.count_nonzero(corpus_packet_count_capped == 64)):,}",
)

print()


# ==============================================================================
# 4. EXTRACT FIRST-PACKET TRANSPORT IDENTITIES FROM COMPACT CORPUS
#
# Source/destination IPs were intentionally masked by Stage20.
#
# But these bytes remain authentic:
#
#   byte 9       IPv4 protocol
#   IHL          IPv4 header length
#   IHL:IHL+2    transport source port
#   IHL+2:IHL+4  transport destination port
#
# We canonicalize port direction because the exact S4 orientation is not
# persisted in the masked corpus.
# ==============================================================================

print("=" * 112)
print("CORPUS TRANSPORT FINGERPRINT")
print("=" * 112)


corpus_protocol = np.empty(
    EXPECTED_CORPUS_ROWS,
    dtype=np.uint8,
)


corpus_port_lo = np.empty(
    EXPECTED_CORPUS_ROWS,
    dtype=np.uint16,
)


corpus_port_hi = np.empty(
    EXPECTED_CORPUS_ROWS,
    dtype=np.uint16,
)


invalid = []


t0 = time.time()


for i in range(
    EXPECTED_CORPUS_ROWS
):

    start = int(
        offsets[
            i
        ]
    )


    first_len = int(
        lengths[
            i,
            0,
        ]
    )


    if first_len < 20:

        invalid.append(
            (
                i,
                first_len,
                "SHORT_IPV4",
            )
        )

        continue


    first = encoded[
        start:
        start + first_len
    ]


    version_ihl = int(
        first[
            0
        ]
    )


    version = (
        version_ihl
        >>
        4
    )


    ihl = (
        version_ihl
        &
        0x0F
    ) * 4


    if (
        version != 4
        or
        ihl < 20
        or
        ihl > first_len
    ):

        invalid.append(
            (
                i,
                first_len,
                "BAD_IPV4_HEADER",
            )
        )

        continue


    protocol = int(
        first[
            9
        ]
    )


    if protocol in (
        6,
        17,
    ):

        if first_len < (
            ihl
            +
            4
        ):

            invalid.append(
                (
                    i,
                    first_len,
                    "SHORT_TRANSPORT",
                )
            )

            continue


        sport = (
            int(
                first[
                    ihl
                ]
            )
            <<
            8
        ) | int(
            first[
                ihl + 1
            ]
        )


        dport = (
            int(
                first[
                    ihl + 2
                ]
            )
            <<
            8
        ) | int(
            first[
                ihl + 3
            ]
        )


    else:

        # Frozen Stage20 S4 protocol-0 semantics use ports 0/0.
        sport = 0
        dport = 0


    corpus_protocol[
        i
    ] = protocol


    if sport <= dport:

        corpus_port_lo[
            i
        ] = sport

        corpus_port_hi[
            i
        ] = dport

    else:

        corpus_port_lo[
            i
        ] = dport

        corpus_port_hi[
            i
        ] = sport


    if (
        i + 1
    ) % 100_000 == 0:

        print(
            f"  parsed {i + 1:,} / {EXPECTED_CORPUS_ROWS:,}",
            flush=True,
        )


if invalid:

    raise RuntimeError(
        "\nCould not recover first-packet transport identity.\n"
        f"First failures: {invalid[:20]}"
    )


print(
    "  elapsed:",
    f"{time.time() - t0:.2f}s",
)

print(
    "  protocol counts:",
    dict(
        sorted(
            Counter(
                int(x)
                for x in corpus_protocol
            ).items()
        )
    ),
)

print()


# ==============================================================================
# 5. PACK CORPUS FINGERPRINT INTO UINT64
#
# Bits:
#
#   [47:40] protocol
#   [39:24] lower port
#   [23: 8] upper port
#   [ 7: 0] capped packet count
# ==============================================================================

corpus_key = (
    (
        corpus_protocol.astype(
            np.uint64
        )
        <<
        np.uint64(
            40
        )
    )
    |
    (
        corpus_port_lo.astype(
            np.uint64
        )
        <<
        np.uint64(
            24
        )
    )
    |
    (
        corpus_port_hi.astype(
            np.uint64
        )
        <<
        np.uint64(
            8
        )
    )
    |
    corpus_packet_count_capped.astype(
        np.uint64
    )
)


print(
    "Corpus fingerprint rows:",
    f"{corpus_key.size:,}",
)

print(
    "Corpus unique fingerprints:",
    f"{np.unique(corpus_key).size:,}",
)

print()


# ==============================================================================
# 6. RESOLVE PUBLISHED COLUMNS
# ==============================================================================

print("=" * 112)
print("PUBLISHED MONDAY PROJECTION")
print("=" * 112)


conn = duckdb.connect(
    database=":memory:"
)


parquet_sql = str(
    PUBLISHED_PATH
).replace(
    "'",
    "''",
)


schema = conn.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{parquet_sql}')
    """
).fetchall()


physical_names = [
    row[
        0
    ]
    for row in schema
]


def normalize(
    value,
):

    return "".join(
        ch.lower()
        for ch in str(
            value
        )
        if ch.isalnum()
    )


normalized = {
    normalize(
        name
    ):
        name
    for name in physical_names
}


def resolve(
    *aliases,
):

    hits = []

    for alias in aliases:

        name = normalized.get(
            normalize(
                alias
            )
        )

        if name is not None:

            hits.append(
                name
            )


    hits = list(
        dict.fromkeys(
            hits
        )
    )


    if len(
        hits
    ) != 1:

        raise RuntimeError(
            f"Column resolution failed for {aliases}: {hits}"
        )


    return hits[
        0
    ]


COL_SRC_PORT = resolve(
    "Source Port",
    "Src Port",
)


COL_DST_PORT = resolve(
    "Destination Port",
    "Dst Port",
)


COL_PROTOCOL = resolve(
    "Protocol",
)


COL_FWD_PKTS = resolve(
    "Total Fwd Packets",
    "Tot Fwd Pkts",
)


COL_BWD_PKTS = resolve(
    "Total Backward Packets",
    "Tot Bwd Pkts",
)


COL_LABEL = resolve(
    "Label",
)


def q(
    name,
):

    return (
        '"'
        +
        str(
            name
        ).replace(
            '"',
            '""'
        )
        +
        '"'
    )


query = f"""
SELECT
    CAST({q(COL_PROTOCOL)} AS BIGINT),
    CAST({q(COL_SRC_PORT)} AS BIGINT),
    CAST({q(COL_DST_PORT)} AS BIGINT),
    CAST({q(COL_FWD_PKTS)} AS BIGINT),
    CAST({q(COL_BWD_PKTS)} AS BIGINT),
    CAST({q(COL_LABEL)} AS VARCHAR)
FROM read_parquet('{parquet_sql}')
"""


rows = conn.execute(
    query
).fetchall()


conn.close()


if len(
    rows
) != EXPECTED_PUBLISHED_ROWS:

    raise RuntimeError(
        "\nUnexpected published row count.\n"
        f"Expected: {EXPECTED_PUBLISHED_ROWS:,}\n"
        f"Actual:   {len(rows):,}"
    )


print(
    "Published rows:",
    f"{len(rows):,}",
)


# ==============================================================================
# 7. BUILD PUBLISHED UINT64 FINGERPRINT
# ==============================================================================

published_key = np.empty(
    EXPECTED_PUBLISHED_ROWS,
    dtype=np.uint64,
)


published_binary = np.empty(
    EXPECTED_PUBLISHED_ROWS,
    dtype=np.uint8,
)


bad_rows = []


for i, row in enumerate(
    rows
):

    (
        protocol,
        sport,
        dport,
        fwd_pkts,
        bwd_pkts,
        label_text,
    ) = row


    if None in (
        protocol,
        sport,
        dport,
        fwd_pkts,
        bwd_pkts,
        label_text,
    ):

        bad_rows.append(
            (
                i,
                row,
            )
        )

        continue


    protocol = int(
        protocol
    )

    sport = int(
        sport
    )

    dport = int(
        dport
    )

    packet_total = (
        int(
            fwd_pkts
        )
        +
        int(
            bwd_pkts
        )
    )


    if packet_total <= 0:

        bad_rows.append(
            (
                i,
                "NONPOSITIVE_PACKET_COUNT",
                packet_total,
            )
        )

        continue


    packet_cap = min(
        packet_total,
        64,
    )


    if protocol not in (
        6,
        17,
    ):

        # Frozen parser protocol-0 S4 convention.
        sport = 0
        dport = 0


    port_lo = min(
        sport,
        dport,
    )

    port_hi = max(
        sport,
        dport,
    )


    published_key[
        i
    ] = (
        (
            np.uint64(
                protocol
            )
            <<
            np.uint64(
                40
            )
        )
        |
        (
            np.uint64(
                port_lo
            )
            <<
            np.uint64(
                24
            )
        )
        |
        (
            np.uint64(
                port_hi
            )
            <<
            np.uint64(
                8
            )
        )
        |
        np.uint64(
            packet_cap
        )
    )


    label = str(
        label_text
    ).strip()


    if not label:

        bad_rows.append(
            (
                i,
                "EMPTY_LABEL",
            )
        )

        continue


    published_binary[
        i
    ] = (
        0
        if label == "BENIGN"
        else 1
    )


if bad_rows:

    raise RuntimeError(
        "\nInvalid published projection rows.\n"
        f"First failures: {bad_rows[:20]}"
    )


del rows

gc.collect()


print(
    "Published unique fingerprints:",
    f"{np.unique(published_key).size:,}",
)

print(
    "Published class counts:",
    dict(
        sorted(
            Counter(
                int(x)
                for x in published_binary
            ).items()
        )
    ),
)

print()


# ==============================================================================
# 8. PROJECTION-MULTISET CHECK
#
# Every frozen corpus fingerprint must have enough occurrences in the published
# source. This does NOT yet assert physical-row identity.
# ==============================================================================

print("=" * 112)
print("FINGERPRINT MULTISET CHECK")
print("=" * 112)


corpus_unique, corpus_counts = np.unique(
    corpus_key,
    return_counts=True,
)


published_unique, published_counts = np.unique(
    published_key,
    return_counts=True,
)


published_count_lookup = {
    int(
        key
    ):
        int(
            count
        )
    for key, count in zip(
        published_unique,
        published_counts,
    )
}


deficits = []


for key, count in zip(
    corpus_unique,
    corpus_counts,
):

    available = published_count_lookup.get(
        int(
            key
        ),
        0,
    )


    if available < int(
        count
    ):

        deficits.append(
            (
                int(
                    key
                ),
                int(
                    count
                ),
                int(
                    available
                ),
            )
        )


print(
    "Corpus fingerprints:",
    f"{corpus_unique.size:,}",
)

print(
    "Projection deficits:",
    len(
        deficits
    ),
)


if deficits:

    print(
        "First deficits:",
        deficits[
            :20
        ],
    )


    print()
    print(
        "[STOP] Coarse corpus projection is insufficient for direct ordered alignment."
    )

    print(
        "No candidate membership was written."
    )

    raise RuntimeError(
        "Published fingerprint multiset does not contain frozen corpus projection."
    )


print(
    "[PASS] Published projection contains the complete corpus fingerprint multiset."
)

print()


# ==============================================================================
# 9. FORWARD GREEDY
#
# Earliest possible legal embedding.
# ==============================================================================

print("=" * 112)
print("FORWARD ORDER ALIGNMENT")
print("=" * 112)


forward = np.empty(
    EXPECTED_CORPUS_ROWS,
    dtype=np.int32,
)


i = 0
j = 0


while (
    i < EXPECTED_CORPUS_ROWS
    and
    j < EXPECTED_PUBLISHED_ROWS
):

    if corpus_key[
        i
    ] == published_key[
        j
    ]:

        forward[
            i
        ] = j

        i += 1

    j += 1


forward_matched = i


print(
    "Matched:",
    f"{forward_matched:,}",
    "/",
    f"{EXPECTED_CORPUS_ROWS:,}",
)


if forward_matched != EXPECTED_CORPUS_ROWS:

    print(
        "First unmatched corpus row:",
        forward_matched,
    )

    print()
    print(
        "[STOP] Corpus fingerprint order is not a complete subsequence "
        "of published physical order."
    )

    raise RuntimeError(
        "Forward corpus-order alignment incomplete."
    )


forward_skipped = (
    EXPECTED_PUBLISHED_ROWS
    -
    EXPECTED_CORPUS_ROWS
)


print(
    "Published rows skipped:",
    f"{forward_skipped:,}",
)


if forward_skipped != EXPECTED_SKIPPED_PUBLISHED:

    raise RuntimeError(
        "Unexpected published-row skip count."
    )


print(
    "[PASS] Complete forward embedding."
)

print()


# ==============================================================================
# 10. REVERSE GREEDY
#
# Latest possible legal embedding.
# ==============================================================================

print("=" * 112)
print("REVERSE ORDER ALIGNMENT")
print("=" * 112)


reverse = np.empty(
    EXPECTED_CORPUS_ROWS,
    dtype=np.int32,
)


i = (
    EXPECTED_CORPUS_ROWS
    -
    1
)

j = (
    EXPECTED_PUBLISHED_ROWS
    -
    1
)


while (
    i >= 0
    and
    j >= 0
):

    if corpus_key[
        i
    ] == published_key[
        j
    ]:

        reverse[
            i
        ] = j

        i -= 1

    j -= 1


reverse_matched = (
    EXPECTED_CORPUS_ROWS
    -
    1
    -
    i
)


print(
    "Matched:",
    f"{reverse_matched:,}",
    "/",
    f"{EXPECTED_CORPUS_ROWS:,}",
)


if reverse_matched != EXPECTED_CORPUS_ROWS:

    raise RuntimeError(
        "Reverse corpus-order alignment incomplete."
    )


print(
    "[PASS] Complete reverse embedding."
)

print()


# ==============================================================================
# 11. UNIQUENESS PROOF
# ==============================================================================

print("=" * 112)
print("PHYSICAL ROW UNIQUENESS")
print("=" * 112)


same = (
    forward
    ==
    reverse
)


same_count = int(
    np.count_nonzero(
        same
    )
)


ambiguous_count = (
    EXPECTED_CORPUS_ROWS
    -
    same_count
)


print(
    "Forward == reverse:",
    f"{same_count:,}",
    "/",
    f"{EXPECTED_CORPUS_ROWS:,}",
)

print(
    "Order-ambiguous corpus rows:",
    f"{ambiguous_count:,}",
)


if ambiguous_count:

    ambiguous_indices = np.flatnonzero(
        ~same
    )


    print()
    print(
        "First ambiguous corpus rows:"
    )


    for idx in ambiguous_indices[
        :20
    ]:

        print(
            " ",
            int(
                idx
            ),
            "forward=",
            int(
                forward[
                    idx
                ]
            ),
            "reverse=",
            int(
                reverse[
                    idx
                ]
            ),
        )


    print()
    print(
        "[NOT YET UNIQUE]"
    )

    print(
        "The next audit will enrich the fingerprint using additional "
        "packet geometry preserved in the corpus."
    )


else:

    print()
    print(
        "[PASS] UNIQUE PHYSICAL-ROW EMBEDDING."
    )

    print(
        "The earliest and latest legal embeddings are identical for "
        "all 528,509 frozen corpus rows."
    )


# ==============================================================================
# 12. LABEL CONSISTENCY
# ==============================================================================

print()
print("=" * 112)
print("CORPUS / PUBLISHED LABEL CONSISTENCY")
print("=" * 112)


# Only meaningful for a unique mapping.
if ambiguous_count == 0:

    mapped_binary = published_binary[
        forward
    ]


    mismatched_labels = np.flatnonzero(
        mapped_binary
        !=
        np.asarray(
            corpus_labels,
            dtype=np.uint8,
        )
    )


    print(
        "Mapped label agreement:",
        f"{EXPECTED_CORPUS_ROWS - mismatched_labels.size:,}",
        "/",
        f"{EXPECTED_CORPUS_ROWS:,}",
    )


    if mismatched_labels.size:

        raise RuntimeError(
            "\nCorpus/published labels disagree under unique alignment.\n"
            f"First rows: {mismatched_labels[:20].tolist()}"
        )


    print(
        "[PASS] Labels agree for every aligned flow."
    )


# ==============================================================================
# 13. WRITE TEMPORARY CANDIDATE ONLY IF UNIQUE
# ==============================================================================

if ambiguous_count == 0:

    all_rows = np.arange(
        EXPECTED_PUBLISHED_ROWS,
        dtype=np.int32,
    )


    selected_mask = np.zeros(
        EXPECTED_PUBLISHED_ROWS,
        dtype=np.bool_,
    )


    selected_mask[
        forward
    ] = True


    skipped_rows = all_rows[
        ~selected_mask
    ]


    if skipped_rows.size != EXPECTED_SKIPPED_PUBLISHED:

        raise RuntimeError(
            "Skipped published-row population mismatch."
        )


    corpus_row_index = np.arange(
        EXPECTED_CORPUS_ROWS,
        dtype=np.int32,
    )


    np.savez_compressed(
        CANDIDATE_NPZ,

        corpus_row_index=
            corpus_row_index,

        published_row_index=
            forward,

        skipped_published_row_index=
            skipped_rows,

        binary_label=
            np.asarray(
                corpus_labels,
                dtype=np.uint8,
            ),
    )


    h = hashlib.sha256()


    with CANDIDATE_NPZ.open(
        "rb"
    ) as fh:

        while True:

            chunk = fh.read(
                8
                * 1024
                * 1024
            )


            if not chunk:
                break


            h.update(
                chunk
            )


    candidate_sha = h.hexdigest()


    print()
    print("=" * 112)
    print("TEMPORARY ALIGNMENT CANDIDATE")
    print("=" * 112)


    print(
        "Path:",
        CANDIDATE_NPZ,
    )

    print(
        "Bytes:",
        f"{CANDIDATE_NPZ.stat().st_size:,}",
    )

    print(
        "SHA256:",
        candidate_sha,
    )

    print(
        "Mapped rows:",
        f"{forward.size:,}",
    )

    print(
        "Skipped published rows:",
        f"{skipped_rows.size:,}",
    )


else:

    candidate_sha = None


# ==============================================================================
# 14. FINAL
# ==============================================================================

print()
print("=" * 112)
print("STAGE24-1C1-M CORPUS ALIGNMENT AUDIT COMPLETE")
print("=" * 112)

print()

print(
    "PCAP accessed:             NO"
)

print(
    "Scapy used:                NO"
)

print(
    "Model executed:            NO"
)

print(
    "Target predictions:        0"
)

print(
    "Target openings:           0 / 8"
)

print(
    "Scientific fits:           2 / 4"
)

print(
    "Git changes:               NONE"
)

print()


if ambiguous_count == 0:

    print(
        "RESULT: UNIQUE MONDAY PHYSICAL ROW ALIGNMENT"
    )

    print()

    print(
        "NEXT:"
    )

    print(
        "  Freeze this 528,509-row membership in Stage24,"
    )

    print(
        "  then download the tiny Tuesday/Wednesday/Thursday/Friday "
        "release corpora — no PCAPs."
    )

else:

    print(
        "RESULT: ORDER EMBEDDING EXISTS BUT IS NOT YET UNIQUE"
    )

    print()

    print(
        "NEXT:"
    )

    print(
        "  Enrich only the ambiguous rows with additional geometry "
        "already preserved in encoded_bytes.bin."
    )


print("=" * 112)

STAGE24-1C1-M — CORPUS-ONLY PHYSICAL ROW ALIGNMENT AUDIT
Corpus: /kaggle/working/stage24_frozen_stage20_corpora/Monday
Published: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/blobs/dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02

DuckDB: 1.3.2

FROZEN CORPUS
Flows: 528,509
Encoded bytes: 522,845,159
Labels: {0: 528509}

Computing frozen capped packet counts...
  elapsed: 0.07s
  min: 2
  max: 64
  capped-at-64: 11,741

CORPUS TRANSPORT FINGERPRINT
  parsed 100,000 / 528,509
  parsed 200,000 / 528,509
  parsed 300,000 / 528,509
  parsed 400,000 / 528,509
  parsed 500,000 / 528,509
  elapsed: 3.67s
  protocol counts: {1: 10, 2: 13, 6: 304722, 17: 223764}

Corpus fingerprint rows: 528,509
Corpus unique fingerprints: 274,900

PUBLISHED MONDAY PROJECTION
Published rows: 529,918
Published unique fingerprints: 275,916
Published class counts: {0: 529918}

FINGERPRINT MULTISET CHECK
Corpus fingerprints: 274,900
Projection deficits: 49
First deficits

RuntimeError: Published fingerprint multiset does not contain frozen corpus projection.

In [20]:
# ==============================================================================
# STAGE24-1C1-M-CORPUS-ALIGNMENT-R2
#
# FIX:
#   Reconstruct frozen Stage20 S4 transport semantics correctly:
#
#     direct TCP -> protocol 6 + ports
#     direct UDP -> protocol 17 + ports
#     everything else -> protocol 0 + ports 0/0
#
#   For raw IPv4 bytes this means non-first fragments MUST NOT be interpreted
#   as TCP/UDP merely because IPv4.protocol == 6/17.
#
# NO PCAP
# NO model
# NO Git changes
# openings = 0 / 8
# ==============================================================================

from __future__ import annotations

import gc
import sys
import time
import subprocess
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np


print("=" * 112)
print("STAGE24-1C1-M — CORPUS-ONLY PHYSICAL ROW ALIGNMENT R2")
print("=" * 112)


# ==============================================================================
# 0. PATHS / COUNTS
# ==============================================================================

CORPUS_DIR = Path(
    "/kaggle/working/stage24_frozen_stage20_corpora/Monday"
)

ENCODED_PATH = CORPUS_DIR / "encoded_bytes.bin"
LENGTHS_PATH = CORPUS_DIR / "packet_lengths.npy"
OFFSETS_PATH = CORPUS_DIR / "flow_offsets.npy"
CORPUS_LABELS_PATH = CORPUS_DIR / "labels.npy"

PUBLISHED_PATH = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/blobs/"
    "dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02"
)

OUT = Path(
    "/kaggle/working/stage24_corpus_alignment"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)

CANDIDATE_NPZ = (
    OUT
    / "monday_corpus_to_published_row_alignment_candidate_r2.npz"
)

EXPECTED_CORPUS_ROWS = 528_509
EXPECTED_PUBLISHED_ROWS = 529_918
EXPECTED_SKIPPED_PUBLISHED = 1_409
EXPECTED_ENCODED_BYTES = 522_845_159


for path in (
    ENCODED_PATH,
    LENGTHS_PATH,
    OFFSETS_PATH,
    CORPUS_LABELS_PATH,
    PUBLISHED_PATH,
):

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )


print("Corpus:   ", CORPUS_DIR)
print("Published:", PUBLISHED_PATH)
print()


# ==============================================================================
# 1. DUCKDB
# ==============================================================================

try:
    import duckdb

except Exception:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "duckdb",
        ],
        check=True,
    )

    import duckdb


print("DuckDB:", duckdb.__version__)
print()


# ==============================================================================
# 2. LOAD CORPUS
# ==============================================================================

lengths = np.load(
    LENGTHS_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

offsets = np.load(
    OFFSETS_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

corpus_labels = np.load(
    CORPUS_LABELS_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

encoded = np.memmap(
    ENCODED_PATH,
    dtype=np.uint8,
    mode="r",
    shape=(EXPECTED_ENCODED_BYTES,),
)


if lengths.shape != (EXPECTED_CORPUS_ROWS, 64):
    raise RuntimeError(
        f"Bad lengths shape: {lengths.shape}"
    )

if offsets.shape != (EXPECTED_CORPUS_ROWS + 1,):
    raise RuntimeError(
        f"Bad offsets shape: {offsets.shape}"
    )

if corpus_labels.shape != (EXPECTED_CORPUS_ROWS,):
    raise RuntimeError(
        f"Bad labels shape: {corpus_labels.shape}"
    )

if int(offsets[-1]) != EXPECTED_ENCODED_BYTES:
    raise RuntimeError(
        "Encoded-byte terminal offset mismatch."
    )

if not np.all(corpus_labels == 0):
    raise RuntimeError(
        "Monday corpus is no longer all BENIGN."
    )


# Every retained packet occupies a nonzero row.
corpus_packet_count = np.count_nonzero(
    lengths,
    axis=1,
).astype(
    np.uint8,
    copy=False,
)


print("=" * 112)
print("CORPUS S4-SEMANTIC TRANSPORT RECOVERY")
print("=" * 112)


# ==============================================================================
# 3. RECOVER FROZEN S4 PROTOCOL + PORT PAIR
#
# IMPORTANT:
#
# IPv4 byte[9] is only the RAW IP protocol number.
#
# Frozen Stage20 parser semantics were:
#
#   transport = ip.payload
#
#   if direct TCP:
#       protocol=6, ports TCP
#   elif direct UDP:
#       protocol=17, ports UDP
#   else:
#       protocol=0, ports=0/0
#
# For IPv4 fragmentation:
#
#   nonzero fragment offset means the packet is not a direct TCP/UDP layer.
#
# ==============================================================================

corpus_protocol = np.zeros(
    EXPECTED_CORPUS_ROWS,
    dtype=np.uint8,
)

corpus_port_lo = np.zeros(
    EXPECTED_CORPUS_ROWS,
    dtype=np.uint16,
)

corpus_port_hi = np.zeros(
    EXPECTED_CORPUS_ROWS,
    dtype=np.uint16,
)


raw_ip_protocol_counts = Counter()
s4_protocol_counts = Counter()

fragmented_nonfirst = 0
short_transport = 0
bad_ipv4 = []


t0 = time.time()


for i in range(EXPECTED_CORPUS_ROWS):

    start = int(offsets[i])
    first_len = int(lengths[i, 0])

    if first_len < 20:

        bad_ipv4.append(
            (i, first_len, "SHORT_IPV4")
        )

        continue


    first = encoded[
        start:
        start + first_len
    ]


    version_ihl = int(first[0])

    version = version_ihl >> 4
    ihl = (version_ihl & 0x0F) * 4


    if (
        version != 4
        or
        ihl < 20
        or
        ihl > first_len
    ):

        bad_ipv4.append(
            (i, first_len, "BAD_IPV4")
        )

        continue


    raw_protocol = int(
        first[9]
    )

    raw_ip_protocol_counts[
        raw_protocol
    ] += 1


    flags_fragment = (
        int(first[6]) << 8
    ) | int(first[7])

    fragment_offset = (
        flags_fragment
        &
        0x1FFF
    )


    # --------------------------------------------------------------------------
    # Frozen outer-layer parser semantics.
    # --------------------------------------------------------------------------

    if (
        raw_protocol in (6, 17)
        and
        fragment_offset == 0
        and
        first_len >= ihl + 4
    ):

        sport = (
            int(first[ihl]) << 8
        ) | int(first[ihl + 1])

        dport = (
            int(first[ihl + 2]) << 8
        ) | int(first[ihl + 3])


        s4_protocol = raw_protocol


        corpus_protocol[i] = s4_protocol

        corpus_port_lo[i] = min(
            sport,
            dport,
        )

        corpus_port_hi[i] = max(
            sport,
            dport,
        )


    else:

        # Stage20 direct outer-payload parser:
        #
        # anything that is not exposed as direct TCP/UDP
        # -> protocol 0 / ports 0,0.
        corpus_protocol[i] = 0

        corpus_port_lo[i] = 0
        corpus_port_hi[i] = 0


        if (
            raw_protocol in (6, 17)
            and
            fragment_offset != 0
        ):
            fragmented_nonfirst += 1

        elif (
            raw_protocol in (6, 17)
            and
            first_len < ihl + 4
        ):
            short_transport += 1


    s4_protocol_counts[
        int(corpus_protocol[i])
    ] += 1


    if (
        i + 1
    ) % 100_000 == 0:

        print(
            f"  parsed {i + 1:,} / {EXPECTED_CORPUS_ROWS:,}",
            flush=True,
        )


if bad_ipv4:

    raise RuntimeError(
        "Invalid frozen corpus IPv4 rows:\n"
        + repr(
            bad_ipv4[:20]
        )
    )


print()
print(
    "Elapsed:",
    f"{time.time() - t0:.2f}s",
)

print(
    "Raw IPv4 protocol counts:",
    dict(
        sorted(
            raw_ip_protocol_counts.items()
        )
    ),
)

print(
    "Frozen S4 protocol counts:",
    dict(
        sorted(
            s4_protocol_counts.items()
        )
    ),
)

print(
    "Non-first TCP/UDP fragments -> protocol0:",
    f"{fragmented_nonfirst:,}",
)

print(
    "Short TCP/UDP transport -> protocol0:",
    f"{short_transport:,}",
)

print()


# Critical sanity:
# protocol 1 / 2 / etc MUST NOT survive into frozen S4 semantics.

illegal_protocols = sorted(
    set(
        int(x)
        for x in np.unique(
            corpus_protocol
        )
    )
    -
    {
        0,
        6,
        17,
    }
)


if illegal_protocols:

    raise RuntimeError(
        f"Illegal S4 protocol values survived: {illegal_protocols}"
    )


print(
    "[PASS] Corpus protocol semantics normalized to frozen {0,6,17}."
)

print()


# ==============================================================================
# 4. PACK CORPUS KEY
# ==============================================================================

corpus_key = (
    (
        corpus_protocol.astype(np.uint64)
        <<
        np.uint64(40)
    )
    |
    (
        corpus_port_lo.astype(np.uint64)
        <<
        np.uint64(24)
    )
    |
    (
        corpus_port_hi.astype(np.uint64)
        <<
        np.uint64(8)
    )
    |
    corpus_packet_count.astype(np.uint64)
)


print(
    "Corpus rows:",
    f"{corpus_key.size:,}",
)

print(
    "Corpus unique coarse keys:",
    f"{np.unique(corpus_key).size:,}",
)

print()


# ==============================================================================
# 5. READ PUBLISHED MONDAY PROJECTION
# ==============================================================================

print("=" * 112)
print("PUBLISHED MONDAY PROJECTION")
print("=" * 112)


conn = duckdb.connect(
    database=":memory:"
)


parquet_sql = str(
    PUBLISHED_PATH
).replace(
    "'",
    "''",
)


schema = conn.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{parquet_sql}')
    """
).fetchall()


physical_names = [
    row[0]
    for row in schema
]


def normalize(value):

    return "".join(
        ch.lower()
        for ch in str(value)
        if ch.isalnum()
    )


normalized = {
    normalize(name):
        name
    for name in physical_names
}


def resolve(*aliases):

    hits = []

    for alias in aliases:

        name = normalized.get(
            normalize(alias)
        )

        if name is not None:
            hits.append(name)


    hits = list(
        dict.fromkeys(hits)
    )


    if len(hits) != 1:

        raise RuntimeError(
            f"Column resolution failed for {aliases}: {hits}"
        )


    return hits[0]


SRC_PORT = resolve(
    "Source Port",
    "Src Port",
)

DST_PORT = resolve(
    "Destination Port",
    "Dst Port",
)

PROTOCOL = resolve(
    "Protocol",
)

FWD_PKTS = resolve(
    "Total Fwd Packets",
    "Tot Fwd Pkts",
)

BWD_PKTS = resolve(
    "Total Backward Packets",
    "Tot Bwd Pkts",
)

LABEL = resolve(
    "Label",
)


def q(name):

    return (
        '"'
        +
        str(name).replace(
            '"',
            '""'
        )
        +
        '"'
    )


rows = conn.execute(
    f"""
    SELECT
        CAST({q(PROTOCOL)} AS BIGINT),
        CAST({q(SRC_PORT)} AS BIGINT),
        CAST({q(DST_PORT)} AS BIGINT),
        CAST({q(FWD_PKTS)} AS BIGINT),
        CAST({q(BWD_PKTS)} AS BIGINT),
        CAST({q(LABEL)} AS VARCHAR)
    FROM read_parquet('{parquet_sql}')
    """
).fetchall()


conn.close()


if len(rows) != EXPECTED_PUBLISHED_ROWS:

    raise RuntimeError(
        f"Published rows = {len(rows):,}, "
        f"expected {EXPECTED_PUBLISHED_ROWS:,}"
    )


published_key = np.empty(
    EXPECTED_PUBLISHED_ROWS,
    dtype=np.uint64,
)

published_binary = np.empty(
    EXPECTED_PUBLISHED_ROWS,
    dtype=np.uint8,
)


bad = []


for i, (
    protocol,
    sport,
    dport,
    fwd_pkts,
    bwd_pkts,
    label_text,
) in enumerate(rows):

    if None in (
        protocol,
        sport,
        dport,
        fwd_pkts,
        bwd_pkts,
        label_text,
    ):

        bad.append(
            (i, "NULL")
        )

        continue


    protocol = int(protocol)
    sport = int(sport)
    dport = int(dport)

    packet_count = (
        int(fwd_pkts)
        +
        int(bwd_pkts)
    )


    if packet_count <= 0:

        bad.append(
            (
                i,
                "BAD_PACKET_COUNT",
                packet_count,
            )
        )

        continue


    packet_cap = min(
        packet_count,
        64,
    )


    # Published Stage20 S4 semantics.
    if protocol not in (
        6,
        17,
    ):

        protocol = 0
        sport = 0
        dport = 0


    port_lo = min(
        sport,
        dport,
    )

    port_hi = max(
        sport,
        dport,
    )


    published_key[i] = (
        (
            np.uint64(protocol)
            <<
            np.uint64(40)
        )
        |
        (
            np.uint64(port_lo)
            <<
            np.uint64(24)
        )
        |
        (
            np.uint64(port_hi)
            <<
            np.uint64(8)
        )
        |
        np.uint64(packet_cap)
    )


    label = str(
        label_text
    ).strip()


    if not label:

        bad.append(
            (
                i,
                "EMPTY_LABEL",
            )
        )

        continue


    published_binary[i] = (
        0
        if label == "BENIGN"
        else 1
    )


if bad:

    raise RuntimeError(
        "Invalid published rows:\n"
        + repr(
            bad[:20]
        )
    )


del rows

gc.collect()


print(
    "Published rows:",
    f"{EXPECTED_PUBLISHED_ROWS:,}",
)

print(
    "Published unique coarse keys:",
    f"{np.unique(published_key).size:,}",
)

print(
    "Published binary counts:",
    dict(
        sorted(
            Counter(
                int(x)
                for x in published_binary
            ).items()
        )
    ),
)

print()


# ==============================================================================
# 6. MULTISET CONTAINMENT
# ==============================================================================

print("=" * 112)
print("CORRECTED MULTISET CONTAINMENT")
print("=" * 112)


cu, cc = np.unique(
    corpus_key,
    return_counts=True,
)

pu, pc = np.unique(
    published_key,
    return_counts=True,
)


published_counts = {
    int(k):
        int(v)
    for k, v in zip(
        pu,
        pc,
    )
}


def decode_key(value):

    value = int(value)

    return {
        "protocol":
            (value >> 40) & 0xFF,

        "port_lo":
            (value >> 24) & 0xFFFF,

        "port_hi":
            (value >> 8) & 0xFFFF,

        "packet_cap":
            value & 0xFF,
    }


deficits = []


for key, count in zip(
    cu,
    cc,
):

    available = published_counts.get(
        int(key),
        0,
    )


    if available < int(count):

        deficits.append(
            {
                "decoded":
                    decode_key(
                        key
                    ),

                "corpus":
                    int(count),

                "published":
                    int(available),
            }
        )


print(
    "Projection deficits:",
    len(deficits),
)


if deficits:

    print()
    print(
        "First remaining deficits:"
    )

    for item in deficits[:30]:
        print(" ", item)


    raise RuntimeError(
        "\nCorrected S4-semantic coarse fingerprint still has deficits.\n"
        "Do NOT use the previous alignment assumption."
    )


print(
    "[PASS] Complete frozen corpus multiset exists in published Monday projection."
)

print()


# ==============================================================================
# 7. FORWARD = EARLIEST LEGAL EMBEDDING
# ==============================================================================

print("=" * 112)
print("FORWARD EMBEDDING")
print("=" * 112)


forward = np.empty(
    EXPECTED_CORPUS_ROWS,
    dtype=np.int32,
)


ci = 0


for published_index in range(
    EXPECTED_PUBLISHED_ROWS
):

    if (
        ci < EXPECTED_CORPUS_ROWS
        and
        corpus_key[ci]
        ==
        published_key[published_index]
    ):

        forward[ci] = published_index
        ci += 1


print(
    "Matched:",
    f"{ci:,}",
    "/",
    f"{EXPECTED_CORPUS_ROWS:,}",
)


if ci != EXPECTED_CORPUS_ROWS:

    print()
    print(
        "First unmatched corpus row:",
        ci,
    )

    raise RuntimeError(
        "Complete forward order embedding does not exist."
    )


print(
    "[PASS] Complete forward embedding."
)

print()


# ==============================================================================
# 8. REVERSE = LATEST LEGAL EMBEDDING
# ==============================================================================

print("=" * 112)
print("REVERSE EMBEDDING")
print("=" * 112)


reverse = np.empty(
    EXPECTED_CORPUS_ROWS,
    dtype=np.int32,
)


ci = (
    EXPECTED_CORPUS_ROWS
    -
    1
)


for published_index in range(
    EXPECTED_PUBLISHED_ROWS - 1,
    -1,
    -1,
):

    if (
        ci >= 0
        and
        corpus_key[ci]
        ==
        published_key[published_index]
    ):

        reverse[ci] = published_index
        ci -= 1


reverse_matched = (
    EXPECTED_CORPUS_ROWS
    -
    1
    -
    ci
)


print(
    "Matched:",
    f"{reverse_matched:,}",
    "/",
    f"{EXPECTED_CORPUS_ROWS:,}",
)


if ci != -1:

    raise RuntimeError(
        "Complete reverse order embedding does not exist."
    )


print(
    "[PASS] Complete reverse embedding."
)

print()


# ==============================================================================
# 9. UNIQUENESS
# ==============================================================================

print("=" * 112)
print("ORDER-BASED PHYSICAL ROW UNIQUENESS")
print("=" * 112)


same = (
    forward
    ==
    reverse
)


same_count = int(
    np.count_nonzero(
        same
    )
)


ambiguous = np.flatnonzero(
    ~same
)


print(
    "Forward == reverse:",
    f"{same_count:,}",
    "/",
    f"{EXPECTED_CORPUS_ROWS:,}",
)

print(
    "Ambiguous corpus rows:",
    f"{ambiguous.size:,}",
)


if ambiguous.size:

    print()
    print(
        "First ambiguous rows:"
    )

    for idx in ambiguous[:30]:

        print(
            f"  corpus={int(idx):,}"
            f" forward={int(forward[idx]):,}"
            f" reverse={int(reverse[idx]):,}"
        )


else:

    print()
    print(
        "[PASS] UNIQUE physical published-row embedding."
    )


# ==============================================================================
# 10. IF UNIQUE, LABEL CHECK + CANDIDATE
# ==============================================================================

if ambiguous.size == 0:

    mapped_binary = published_binary[
        forward
    ]


    label_mismatch = np.flatnonzero(
        mapped_binary
        !=
        np.asarray(
            corpus_labels,
            dtype=np.uint8,
        )
    )


    if label_mismatch.size:

        raise RuntimeError(
            f"Mapped label mismatches: {label_mismatch[:20].tolist()}"
        )


    selected = np.zeros(
        EXPECTED_PUBLISHED_ROWS,
        dtype=np.bool_,
    )

    selected[
        forward
    ] = True


    skipped = np.flatnonzero(
        ~selected
    ).astype(
        np.int32
    )


    if skipped.size != EXPECTED_SKIPPED_PUBLISHED:

        raise RuntimeError(
            f"Skipped rows = {skipped.size:,}, "
            f"expected {EXPECTED_SKIPPED_PUBLISHED:,}"
        )


    corpus_row = np.arange(
        EXPECTED_CORPUS_ROWS,
        dtype=np.int32,
    )


    np.savez_compressed(
        CANDIDATE_NPZ,

        corpus_row_index=
            corpus_row,

        published_row_index=
            forward,

        skipped_published_row_index=
            skipped,

        binary_label=
            np.asarray(
                corpus_labels,
                dtype=np.uint8,
            ),
    )


    h = hashlib.sha256()


    with CANDIDATE_NPZ.open(
        "rb"
    ) as fh:

        while True:

            chunk = fh.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)


    print()
    print("=" * 112)
    print("UNIQUE ALIGNMENT CANDIDATE")
    print("=" * 112)

    print(
        "Path:",
        CANDIDATE_NPZ,
    )

    print(
        "Rows:",
        f"{forward.size:,}",
    )

    print(
        "Skipped:",
        f"{skipped.size:,}",
    )

    print(
        "SHA256:",
        h.hexdigest(),
    )


# ==============================================================================
# 11. FINAL
# ==============================================================================

print()
print("=" * 112)
print("STAGE24-1C1-M CORPUS ALIGNMENT R2 COMPLETE")
print("=" * 112)

print()

print("PCAP accessed:       NO")
print("Scapy used:          NO")
print("Model executed:      NO")
print("Target predictions:  0")
print("Target openings:     0 / 8")
print("Scientific fits:     2 / 4")
print("Git changes:         NONE")

print()


if ambiguous.size == 0:

    print(
        "RESULT: UNIQUE MONDAY PHYSICAL ROW ALIGNMENT."
    )

    print(
        "NEXT: freeze membership, then obtain the other four release corpora."
    )

else:

    print(
        "RESULT: coarse order projection is valid but not unique."
    )

    print(
        "NEXT: refine ONLY ambiguous rows using more geometry already present in the corpus."
    )


print("=" * 112)

STAGE24-1C1-M — CORPUS-ONLY PHYSICAL ROW ALIGNMENT R2
Corpus:    /kaggle/working/stage24_frozen_stage20_corpora/Monday
Published: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/blobs/dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02

DuckDB: 1.3.2

CORPUS S4-SEMANTIC TRANSPORT RECOVERY
  parsed 100,000 / 528,509
  parsed 200,000 / 528,509
  parsed 300,000 / 528,509
  parsed 400,000 / 528,509
  parsed 500,000 / 528,509

Elapsed: 5.16s
Raw IPv4 protocol counts: {1: 10, 2: 13, 6: 304722, 17: 223764}
Frozen S4 protocol counts: {0: 77, 6: 304722, 17: 223710}
Non-first TCP/UDP fragments -> protocol0: 54
Short TCP/UDP transport -> protocol0: 0

[PASS] Corpus protocol semantics normalized to frozen {0,6,17}.

Corpus rows: 528,509
Corpus unique coarse keys: 274,862

PUBLISHED MONDAY PROJECTION
Published rows: 529,918
Published unique coarse keys: 275,916
Published binary counts: {0: 529918}

CORRECTED MULTISET CONTAINMENT
Projection deficits: 0
[PASS] C

RuntimeError: Complete forward order embedding does not exist.

In [21]:
# ==============================================================================
# STAGE24-1C1-M — GROUNDED MEMBERSHIP AMBIGUITY DECOMPOSITION
#
# Reuses corpus_key / published_key from successful R2 construction.
#
# NO PCAP
# NO parquet reread
# NO model
# NO Git
# ==============================================================================

from collections import Counter
import numpy as np

print("=" * 112)
print("STAGE24-1C1-M — COARSE MEMBERSHIP AMBIGUITY DECOMPOSITION")
print("=" * 112)

required = [
    "corpus_key",
    "published_key",
]

missing = [
    x
    for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        f"R2 live arrays missing: {missing}\n"
        "Do NOT redownload/replay anything."
    )


EXPECTED_CORPUS = 528_509
EXPECTED_PUBLISHED = 529_918
EXPECTED_EXCLUDED = 1_409


if len(corpus_key) != EXPECTED_CORPUS:
    raise RuntimeError(
        f"corpus_key rows={len(corpus_key):,}"
    )

if len(published_key) != EXPECTED_PUBLISHED:
    raise RuntimeError(
        f"published_key rows={len(published_key):,}"
    )


# ------------------------------------------------------------------------------
# Counts per fingerprint
# ------------------------------------------------------------------------------

ck, cc = np.unique(
    corpus_key,
    return_counts=True,
)

pk, pc = np.unique(
    published_key,
    return_counts=True,
)

corpus_counts = {
    int(k): int(v)
    for k, v in zip(ck, cc)
}

published_counts = {
    int(k): int(v)
    for k, v in zip(pk, pc)
}


all_keys = (
    set(corpus_counts)
    |
    set(published_counts)
)


definite_included_rows = 0
definite_excluded_rows = 0

ambiguous_keys = []
ambiguous_published_rows = 0
ambiguous_grounded_rows = 0
ambiguous_excluded_rows = 0


for key in all_keys:

    c = corpus_counts.get(
        key,
        0,
    )

    p = published_counts.get(
        key,
        0,
    )


    if c > p:
        raise RuntimeError(
            f"Containment violation: key={key} corpus={c} published={p}"
        )


    # Entire key absent from corpus => definitely excluded.
    if c == 0:

        definite_excluded_rows += p


    # Exact multiplicity equality => every published occurrence is grounded.
    elif c == p:

        definite_included_rows += p


    # Same coarse fingerprint contains both grounded and excluded occurrences.
    else:

        ambiguous_keys.append(
            (
                key,
                c,
                p,
                p - c,
            )
        )

        ambiguous_published_rows += p
        ambiguous_grounded_rows += c
        ambiguous_excluded_rows += (
            p - c
        )


total_excluded = (
    definite_excluded_rows
    +
    ambiguous_excluded_rows
)


total_grounded = (
    definite_included_rows
    +
    ambiguous_grounded_rows
)


print()
print("TOTALS")
print("-" * 112)

print(
    "Published rows:              ",
    f"{EXPECTED_PUBLISHED:,}",
)

print(
    "Frozen corpus rows:          ",
    f"{EXPECTED_CORPUS:,}",
)

print(
    "Expected exclusions:         ",
    f"{EXPECTED_EXCLUDED:,}",
)

print()


print("DEFINITE")
print("-" * 112)

print(
    "Definitely GROUNDED rows:    ",
    f"{definite_included_rows:,}",
)

print(
    "Definitely EXCLUDED rows:    ",
    f"{definite_excluded_rows:,}",
)

print()


print("AMBIGUOUS COARSE GROUPS")
print("-" * 112)

print(
    "Ambiguous fingerprint keys:  ",
    f"{len(ambiguous_keys):,}",
)

print(
    "Published rows involved:     ",
    f"{ambiguous_published_rows:,}",
)

print(
    "Grounded rows inside groups: ",
    f"{ambiguous_grounded_rows:,}",
)

print(
    "Excluded rows inside groups: ",
    f"{ambiguous_excluded_rows:,}",
)

print()


print("ACCOUNTING")
print("-" * 112)

print(
    "Recovered grounded total:    ",
    f"{total_grounded:,}",
)

print(
    "Recovered exclusion total:   ",
    f"{total_excluded:,}",
)


if total_grounded != EXPECTED_CORPUS:
    raise RuntimeError(
        "Grounded accounting mismatch."
    )

if total_excluded != EXPECTED_EXCLUDED:
    raise RuntimeError(
        "Exclusion accounting mismatch."
    )


# ------------------------------------------------------------------------------
# Distribution of ambiguous groups
# ------------------------------------------------------------------------------

group_size_hist = Counter()
excess_hist = Counter()

for key, c, p, excess in ambiguous_keys:

    group_size_hist[
        p
    ] += 1

    excess_hist[
        excess
    ] += 1


print()
print("AMBIGUOUS GROUP-SIZE DISTRIBUTION")
print("-" * 112)

for size, count in sorted(
    group_size_hist.items()
)[:30]:

    print(
        f"published_group_size={size:5,d}"
        f"  keys={count:6,d}"
    )


if len(group_size_hist) > 30:
    print("...")


print()
print("EXCLUDED-OCCURRENCES PER AMBIGUOUS KEY")
print("-" * 112)

for excess, count in sorted(
    excess_hist.items()
)[:30]:

    print(
        f"excluded_in_key={excess:5,d}"
        f"  keys={count:6,d}"
    )


if len(excess_hist) > 30:
    print("...")


# ------------------------------------------------------------------------------
# Largest ambiguous groups for next targeted refinement
# ------------------------------------------------------------------------------

ambiguous_keys_sorted = sorted(
    ambiguous_keys,
    key=lambda x: (
        -x[3],  # exclusions first
        -x[2],  # published group size
        x[0],
    ),
)


print()
print("TOP 30 AMBIGUOUS KEYS")
print("-" * 112)

for rank, (
    key,
    c,
    p,
    excess,
) in enumerate(
    ambiguous_keys_sorted[:30],
    start=1,
):

    protocol = (
        key >> 40
    ) & 0xFF

    port_lo = (
        key >> 24
    ) & 0xFFFF

    port_hi = (
        key >> 8
    ) & 0xFFFF

    packet_cap = (
        key
        &
        0xFF
    )

    print(
        f"[{rank:02d}] "
        f"proto={protocol:2d} "
        f"ports=({port_lo},{port_hi}) "
        f"packet_cap={packet_cap:2d} "
        f"| corpus={c:,} "
        f"published={p:,} "
        f"excluded={excess:,}"
    )


print()
print("=" * 112)
print("DECOMPOSITION COMPLETE")
print("=" * 112)

print()
print("PCAP accessed:       NO")
print("Model executed:      NO")
print("Target openings:     0 / 8")
print("Scientific fits:     2 / 4")
print("Git changes:         NONE")
print()

print(
    "NEXT: refine ONLY the ambiguous coarse groups using additional "
    "packet geometry already stored in the frozen corpus."
)

STAGE24-1C1-M — COARSE MEMBERSHIP AMBIGUITY DECOMPOSITION

TOTALS
----------------------------------------------------------------------------------------------------------------
Published rows:               529,918
Frozen corpus rows:           528,509
Expected exclusions:          1,409

DEFINITE
----------------------------------------------------------------------------------------------------------------
Definitely GROUNDED rows:     528,081
Definitely EXCLUDED rows:     1,349

AMBIGUOUS COARSE GROUPS
----------------------------------------------------------------------------------------------------------------
Ambiguous fingerprint keys:   55
Published rows involved:      488
Grounded rows inside groups:  428
Excluded rows inside groups:  60

ACCOUNTING
----------------------------------------------------------------------------------------------------------------
Recovered grounded total:     528,509
Recovered exclusion total:    1,409

AMBIGUOUS GROUP-SIZE DISTRIBUTION
------

In [22]:
# ==============================================================================
# STAGE24-1C1-M — AMBIGUOUS S4 GEOMETRY RESOLUTION
#
# INPUT ALREADY IN MEMORY FROM R2 / DECOMPOSITION:
#   corpus_key
#   published_key
#   ambiguous_keys
#
# Resolves ONLY:
#   428 corpus rows
#   488 published rows
#   60 unresolved exclusions
#
# Uses only frozen Stage20 information preserved in compact corpus:
#   - literal S4 protocol semantics
#   - directed transport ports
#   - packet count where uncapped
#   - exact Scapy payload geometry where every stored packet is complete
#   - safe lower/upper bounds for >64-packet flows
#   - frozen first-packet physical flag serialization
#
# NO PCAP
# NO model
# NO target prediction
# NO Git changes
# ==============================================================================

from __future__ import annotations

import gc
import sys
import time
import subprocess
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np


print("=" * 112)
print("STAGE24-1C1-M — AMBIGUOUS S4 GEOMETRY RESOLUTION")
print("=" * 112)


# ==============================================================================
# 0. LIVE STATE
# ==============================================================================

required_live = [
    "corpus_key",
    "published_key",
    "ambiguous_keys",
    "lengths",
    "offsets",
    "encoded",
]

missing = [
    name
    for name in required_live
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Required live corpus-alignment state missing:\n"
        + repr(missing)
        + "\n\nDo NOT redownload the PCAP."
    )


EXPECTED_CORPUS_ROWS = 528_509
EXPECTED_PUBLISHED_ROWS = 529_918
EXPECTED_AMBIGUOUS_CORPUS = 428
EXPECTED_AMBIGUOUS_PUBLISHED = 488
EXPECTED_AMBIGUOUS_EXCLUDED = 60


PUBLISHED_PATH = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/blobs/"
    "dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02"
)


if not PUBLISHED_PATH.is_file():
    raise RuntimeError(
        f"Published Monday parquet missing:\n{PUBLISHED_PATH}"
    )


if len(corpus_key) != EXPECTED_CORPUS_ROWS:
    raise RuntimeError("corpus_key length mismatch")

if len(published_key) != EXPECTED_PUBLISHED_ROWS:
    raise RuntimeError("published_key length mismatch")


ambiguous_key_values = {
    int(item[0])
    for item in ambiguous_keys
}


corpus_ambiguous_idx = np.flatnonzero(
    np.isin(
        corpus_key,
        np.fromiter(
            ambiguous_key_values,
            dtype=np.uint64,
        ),
    )
).astype(
    np.int32
)


published_ambiguous_idx = np.flatnonzero(
    np.isin(
        published_key,
        np.fromiter(
            ambiguous_key_values,
            dtype=np.uint64,
        ),
    )
).astype(
    np.int32
)


print(
    "Ambiguous keys:",
    f"{len(ambiguous_key_values):,}",
)

print(
    "Corpus rows:",
    f"{corpus_ambiguous_idx.size:,}",
)

print(
    "Published rows:",
    f"{published_ambiguous_idx.size:,}",
)


if corpus_ambiguous_idx.size != EXPECTED_AMBIGUOUS_CORPUS:
    raise RuntimeError(
        f"Expected {EXPECTED_AMBIGUOUS_CORPUS} ambiguous corpus rows, "
        f"got {corpus_ambiguous_idx.size}"
    )


if published_ambiguous_idx.size != EXPECTED_AMBIGUOUS_PUBLISHED:
    raise RuntimeError(
        f"Expected {EXPECTED_AMBIGUOUS_PUBLISHED} ambiguous published rows, "
        f"got {published_ambiguous_idx.size}"
    )


print()


# ==============================================================================
# 1. IMPORT DUCKDB + SCAPY
#
# Scapy operates ONLY on tiny already-persisted compact packet representations.
# No PCAP.
# ==============================================================================

try:
    import duckdb
except Exception:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "duckdb",
        ],
        check=True,
    )
    import duckdb


try:
    import scapy

    if scapy.__version__ != "2.6.1":
        raise RuntimeError(
            f"Expected Scapy 2.6.1, got {scapy.__version__}"
        )

    import scapy.all as sall

except Exception as exc:
    raise RuntimeError(
        "Scapy 2.6.1 is required for compact-corpus payload decoding."
    ) from exc


print(
    "DuckDB:",
    duckdb.__version__,
)

print(
    "Scapy:",
    scapy.__version__,
)

print()


# ==============================================================================
# 2. PUBLISHED MONDAY EXACT S4-NONIDENTITY PROJECTION
#
# Important:
#   Protocol is kept LITERAL here.
#
# Frozen Stage20 exact join compared reconstructed protocol=0 directly against
# physical label protocol=0. We do NOT normalize published ICMP/etc to 0.
# ==============================================================================

print("=" * 112)
print("READING PUBLISHED S4 GEOMETRY")
print("=" * 112)


conn = duckdb.connect(
    database=":memory:"
)


parquet_sql = str(
    PUBLISHED_PATH
).replace(
    "'",
    "''",
)


schema = conn.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{parquet_sql}')
    """
).fetchall()


physical_names = [
    row[0]
    for row in schema
]


def norm(value):
    return "".join(
        ch.lower()
        for ch in str(value)
        if ch.isalnum()
    )


normalized = defaultdict(list)

for name in physical_names:
    normalized[norm(name)].append(name)


def resolve(*aliases):

    found = []

    for alias in aliases:
        found.extend(
            normalized.get(
                norm(alias),
                [],
            )
        )

    found = list(
        dict.fromkeys(found)
    )

    if len(found) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {aliases}: {found}"
        )

    return found[0]


COLS = {
    "src_port":
        resolve(
            "Source Port",
            "Src Port",
        ),

    "dst_port":
        resolve(
            "Destination Port",
            "Dst Port",
        ),

    "protocol":
        resolve(
            "Protocol",
        ),

    "fwd_count":
        resolve(
            "Total Fwd Packets",
            "Tot Fwd Pkts",
        ),

    "bwd_count":
        resolve(
            "Total Backward Packets",
            "Tot Bwd Pkts",
        ),

    "fwd_sum":
        resolve(
            "Total Length of Fwd Packets",
            "TotLen Fwd Pkts",
        ),

    "bwd_sum":
        resolve(
            "Total Length of Bwd Packets",
            "TotLen Bwd Pkts",
        ),

    "fwd_min":
        resolve(
            "Fwd Packet Length Min",
            "Fwd Pkt Len Min",
        ),

    "fwd_max":
        resolve(
            "Fwd Packet Length Max",
            "Fwd Pkt Len Max",
        ),

    "bwd_min":
        resolve(
            "Bwd Packet Length Min",
            "Bwd Pkt Len Min",
        ),

    "bwd_max":
        resolve(
            "Bwd Packet Length Max",
            "Bwd Pkt Len Max",
        ),

    "fin":
        resolve(
            "FIN Flag Count",
            "FIN Flag Cnt",
        ),

    "syn":
        resolve(
            "SYN Flag Count",
            "SYN Flag Cnt",
        ),

    "rst":
        resolve(
            "RST Flag Count",
            "RST Flag Cnt",
        ),

    "psh":
        resolve(
            "PSH Flag Count",
            "PSH Flag Cnt",
        ),

    "ack":
        resolve(
            "ACK Flag Count",
            "ACK Flag Cnt",
        ),

    "urg":
        resolve(
            "URG Flag Count",
            "URG Flag Cnt",
        ),
}


def qi(name):
    return (
        '"'
        +
        str(name).replace(
            '"',
            '""',
        )
        +
        '"'
    )


projection_sql = ",\n".join(
    [
        f"CAST({qi(COLS['src_port'])} AS BIGINT)",
        f"CAST({qi(COLS['dst_port'])} AS BIGINT)",
        f"CAST({qi(COLS['protocol'])} AS BIGINT)",
        f"CAST({qi(COLS['fwd_count'])} AS BIGINT)",
        f"CAST({qi(COLS['bwd_count'])} AS BIGINT)",
        f"CAST({qi(COLS['fwd_sum'])} AS BIGINT)",
        f"CAST({qi(COLS['bwd_sum'])} AS BIGINT)",
        f"CAST({qi(COLS['fwd_min'])} AS BIGINT)",
        f"CAST({qi(COLS['fwd_max'])} AS BIGINT)",
        f"CAST({qi(COLS['bwd_min'])} AS BIGINT)",
        f"CAST({qi(COLS['bwd_max'])} AS BIGINT)",
        f"CAST({qi(COLS['fin'])} AS BIGINT)",
        f"CAST({qi(COLS['syn'])} AS BIGINT)",
        f"CAST({qi(COLS['rst'])} AS BIGINT)",
        f"CAST({qi(COLS['psh'])} AS BIGINT)",
        f"CAST({qi(COLS['ack'])} AS BIGINT)",
        f"CAST({qi(COLS['urg'])} AS BIGINT)",
    ]
)


published_rows = conn.execute(
    f"""
    SELECT
        row_number() OVER () - 1 AS physical_row_index,
        {projection_sql}
    FROM read_parquet('{parquet_sql}')
    """
).fetchall()


conn.close()


if len(published_rows) != EXPECTED_PUBLISHED_ROWS:
    raise RuntimeError(
        f"Published row count mismatch: {len(published_rows)}"
    )


# physical row -> geometry tuple/dict
published_s4 = {}


for row in published_rows:

    (
        row_index,
        src_port,
        dst_port,
        protocol,
        fwd_count,
        bwd_count,
        fwd_sum,
        bwd_sum,
        fwd_min,
        fwd_max,
        bwd_min,
        bwd_max,
        fin,
        syn,
        rst,
        psh,
        ack,
        urg,
    ) = row


    values = [
        src_port,
        dst_port,
        protocol,
        fwd_count,
        bwd_count,
        fwd_sum,
        bwd_sum,
        fwd_min,
        fwd_max,
        bwd_min,
        bwd_max,
        fin,
        syn,
        rst,
        psh,
        ack,
        urg,
    ]


    if any(
        value is None
        for value in values
    ):
        raise RuntimeError(
            f"NULL S4 geometry at published row {row_index}"
        )


    published_s4[
        int(row_index)
    ] = {
        "src_port":
            int(src_port),

        "dst_port":
            int(dst_port),

        "protocol":
            int(protocol),

        "fwd_count":
            int(fwd_count),

        "bwd_count":
            int(bwd_count),

        "fwd_sum":
            int(fwd_sum),

        "bwd_sum":
            int(bwd_sum),

        "fwd_min":
            int(fwd_min),

        "fwd_max":
            int(fwd_max),

        "bwd_min":
            int(bwd_min),

        "bwd_max":
            int(bwd_max),

        "fin":
            int(fin),

        "syn":
            int(syn),

        "rst":
            int(rst),

        "psh":
            int(psh),

        "ack":
            int(ack),

        "urg":
            int(urg),
    }


del published_rows

gc.collect()


print(
    "[PASS] Published S4 geometry loaded."
)

print()


# ==============================================================================
# 3. COMPACT PACKET DECODER
#
# Input bytes are the Stage20 MASKED IPv4 representation.
#
# Masking does NOT alter:
#   - protocol
#   - ports
#   - TCP flags
#   - application payload
#   - retained packet length
#
# For retained length <256, the entire captured IPv4 representation is known to
# be present because Stage20 stored min(original_length, 256).
# ==============================================================================

def decode_compact_packet(
    packet_bytes,
):

    raw = bytes(
        packet_bytes
    )


    if len(raw) < 20:
        raise RuntimeError(
            f"Compact IPv4 packet too short: {len(raw)}"
        )


    ip = sall.IP(
        raw
    )


    transport = ip.payload


    if isinstance(
        transport,
        sall.TCP,
    ):

        tcp = transport

        flag_value = int(
            tcp.flags
        )


        return {
            "protocol":
                6,

            "sport":
                int(
                    tcp.sport
                ),

            "dport":
                int(
                    tcp.dport
                ),

            "payload":
                int(
                    len(
                        bytes(
                            tcp.payload
                        )
                    )
                ),

            "semantic_fin":
                int(
                    bool(
                        flag_value
                        &
                        0x01
                    )
                ),

            "semantic_syn":
                int(
                    bool(
                        flag_value
                        &
                        0x02
                    )
                ),

            "semantic_rst":
                int(
                    bool(
                        flag_value
                        &
                        0x04
                    )
                ),

            "semantic_psh":
                int(
                    bool(
                        flag_value
                        &
                        0x08
                    )
                ),

            "semantic_ack":
                int(
                    bool(
                        flag_value
                        &
                        0x10
                    )
                ),

            "semantic_urg":
                int(
                    bool(
                        flag_value
                        &
                        0x20
                    )
                ),

            "semantic_ece":
                int(
                    bool(
                        flag_value
                        &
                        0x40
                    )
                ),
        }


    if isinstance(
        transport,
        sall.UDP,
    ):

        udp = transport


        return {
            "protocol":
                17,

            "sport":
                int(
                    udp.sport
                ),

            "dport":
                int(
                    udp.dport
                ),

            "payload":
                int(
                    len(
                        bytes(
                            udp.payload
                        )
                    )
                ),

            "semantic_fin": 0,
            "semantic_syn": 0,
            "semantic_rst": 0,
            "semantic_psh": 0,
            "semantic_ack": 0,
            "semantic_urg": 0,
            "semantic_ece": 0,
        }


    return {
        "protocol": 0,
        "sport": 0,
        "dport": 0,
        "payload": 0,

        "semantic_fin": 0,
        "semantic_syn": 0,
        "semantic_rst": 0,
        "semantic_psh": 0,
        "semantic_ack": 0,
        "semantic_urg": 0,
        "semantic_ece": 0,
    }


# ==============================================================================
# 4. DECODE ONLY 428 AMBIGUOUS CORPUS FLOWS
# ==============================================================================

print("=" * 112)
print("DECODING 428 AMBIGUOUS CORPUS FLOWS")
print("=" * 112)


corpus_flow_info = {}


for ordinal, corpus_index in enumerate(
    corpus_ambiguous_idx,
    start=1,
):

    corpus_index = int(
        corpus_index
    )


    flow_lengths = np.asarray(
        lengths[
            corpus_index
        ],
        dtype=np.uint16,
    )


    packet_count_stored = int(
        np.count_nonzero(
            flow_lengths
        )
    )


    start = int(
        offsets[
            corpus_index
        ]
    )


    cursor = start


    packets = []


    for position in range(
        packet_count_stored
    ):

        retained_len = int(
            flow_lengths[
                position
            ]
        )


        if retained_len <= 0:
            raise RuntimeError(
                (
                    "Unexpected zero inside compact packet prefix",
                    corpus_index,
                    position,
                )
            )


        packet = bytes(
            encoded[
                cursor:
                cursor + retained_len
            ]
        )


        cursor += retained_len


        decoded = decode_compact_packet(
            packet
        )


        decoded[
            "retained_len"
        ] = retained_len


        decoded[
            "full_capture_known"
        ] = (
            retained_len
            <
            256
        )


        packets.append(
            decoded
        )


    if cursor != int(
        offsets[
            corpus_index + 1
        ]
    ):
        raise RuntimeError(
            f"Offset traversal mismatch at corpus row {corpus_index}"
        )


    first = packets[
        0
    ]


    # Historical physical flag serialization:
    #
    # physical FIN <- semantic RST
    # physical SYN <- semantic PSH
    # physical RST <- semantic ECE
    # physical PSH <- semantic SYN
    # physical ACK <- semantic ACK
    # physical URG <- semantic FIN
    physical_flags = {
        "fin":
            int(
                first[
                    "semantic_rst"
                ]
            ),

        "syn":
            int(
                first[
                    "semantic_psh"
                ]
            ),

        "rst":
            int(
                first[
                    "semantic_ece"
                ]
            ),

        "psh":
            int(
                first[
                    "semantic_syn"
                ]
            ),

        "ack":
            int(
                first[
                    "semantic_ack"
                ]
            ),

        "urg":
            int(
                first[
                    "semantic_fin"
                ]
            ),
    }


    corpus_flow_info[
        corpus_index
    ] = {
        "packet_count_stored":
            packet_count_stored,

        "all_packets_full":
            all(
                item[
                    "full_capture_known"
                ]
                for item in packets
            ),

        "protocol":
            int(
                first[
                    "protocol"
                ]
            ),

        "first_sport":
            int(
                first[
                    "sport"
                ]
            ),

        "first_dport":
            int(
                first[
                    "dport"
                ]
            ),

        "flags":
            physical_flags,

        "packets":
            packets,
    }


    if (
        ordinal
        %
        100
        ==
        0
    ):

        print(
            f"  decoded {ordinal:,} / {corpus_ambiguous_idx.size:,}",
            flush=True,
        )


print(
    "[PASS] Ambiguous compact flows decoded."
)

print()


# ==============================================================================
# 5. SAFE CANDIDATE COMPATIBILITY
# ==============================================================================

def candidate_compatible(
    flow,
    candidate,
):

    protocol = int(
        flow[
            "protocol"
        ]
    )


    # --------------------------------------------------------------------------
    # Exact literal protocol.
    # --------------------------------------------------------------------------

    if int(
        candidate[
            "protocol"
        ]
    ) != protocol:

        return False


    packets = flow[
        "packets"
    ]


    stored_n = int(
        flow[
            "packet_count_stored"
        ]
    )


    total_n = (
        int(
            candidate[
                "fwd_count"
            ]
        )
        +
        int(
            candidate[
                "bwd_count"
            ]
        )
    )


    # --------------------------------------------------------------------------
    # Representation row count.
    # --------------------------------------------------------------------------

    if stored_n < 64:

        # Representation contains the complete flow.
        if total_n != stored_n:
            return False

        candidate_complete = True


    else:

        if total_n < 64:
            return False

        candidate_complete = (
            total_n
            ==
            64
        )


    # --------------------------------------------------------------------------
    # Exact directed ports.
    # --------------------------------------------------------------------------

    cand_sport = int(
        candidate[
            "src_port"
        ]
    )

    cand_dport = int(
        candidate[
            "dst_port"
        ]
    )


    if protocol == 0:

        if (
            cand_sport != 0
            or
            cand_dport != 0
        ):
            return False


    else:

        first_sport = int(
            flow[
                "first_sport"
            ]
        )

        first_dport = int(
            flow[
                "first_dport"
            ]
        )


        if (
            cand_sport,
            cand_dport,
        ) not in {
            (
                first_sport,
                first_dport,
            ),
            (
                first_dport,
                first_sport,
            ),
        }:

            return False


    # --------------------------------------------------------------------------
    # Frozen first-packet physical flags.
    # --------------------------------------------------------------------------

    for name in (
        "fin",
        "syn",
        "rst",
        "psh",
        "ack",
        "urg",
    ):

        if int(
            candidate[
                name
            ]
        ) != int(
            flow[
                "flags"
            ][
                name
            ]
        ):

            return False


    # --------------------------------------------------------------------------
    # Payload geometry can only be exact/bounded when every stored packet was
    # persisted below the 256-byte truncation boundary.
    # --------------------------------------------------------------------------

    all_full = bool(
        flow[
            "all_packets_full"
        ]
    )


    if not all_full:

        # Still a valid compatibility candidate; do not infer from truncated
        # payload bytes.
        return True


    # --------------------------------------------------------------------------
    # Same-port or protocol-0 flows:
    #
    # Source IP was intentionally masked, so direction cannot be reconstructed
    # from transport ports. Use only direction-invariant constraints.
    # --------------------------------------------------------------------------

    if (
        protocol == 0
        or
        cand_sport == cand_dport
    ):

        payloads = [
            int(
                pkt[
                    "payload"
                ]
            )
            for pkt in packets
        ]


        prefix_sum = sum(
            payloads
        )


        candidate_sum = (
            int(
                candidate[
                    "fwd_sum"
                ]
            )
            +
            int(
                candidate[
                    "bwd_sum"
                ]
            )
        )


        if candidate_complete:

            if candidate_sum != prefix_sum:
                return False


            global_max = max(
                payloads
            )


            candidate_global_max = max(
                int(
                    candidate[
                        "fwd_max"
                    ]
                ),
                int(
                    candidate[
                        "bwd_max"
                    ]
                ),
            )


            if candidate_global_max != global_max:
                return False


            nonempty_direction_mins = [
                int(
                    candidate[
                        "fwd_min"
                    ]
                )
            ]


            if int(
                candidate[
                    "bwd_count"
                ]
            ) > 0:

                nonempty_direction_mins.append(
                    int(
                        candidate[
                            "bwd_min"
                        ]
                    )
                )


            candidate_global_min = min(
                nonempty_direction_mins
            )


            if candidate_global_min != min(
                payloads
            ):
                return False


        else:

            # Only a stored 64-packet prefix.
            if candidate_sum < prefix_sum:
                return False


        return True


    # --------------------------------------------------------------------------
    # Distinct TCP/UDP ports.
    #
    # Historical quirk:
    #   packet 0 is ALWAYS statistically FWD before timeout orientation
    #   overwrite.
    #
    # Subsequent packet direction is determined against final output
    # orientation, represented by candidate source/destination ports.
    # --------------------------------------------------------------------------

    fwd_payloads = [
        int(
            packets[
                0
            ][
                "payload"
            ]
        )
    ]

    bwd_payloads = []


    for packet in packets[
        1:
    ]:

        packet_ports = (
            int(
                packet[
                    "sport"
                ]
            ),
            int(
                packet[
                    "dport"
                ]
            ),
        )


        if packet_ports == (
            cand_sport,
            cand_dport,
        ):

            fwd_payloads.append(
                int(
                    packet[
                        "payload"
                    ]
                )
            )


        elif packet_ports == (
            cand_dport,
            cand_sport,
        ):

            bwd_payloads.append(
                int(
                    packet[
                        "payload"
                    ]
                )
            )


        else:

            # Packet cannot belong to candidate orientation.
            return False


    prefix_fwd_count = len(
        fwd_payloads
    )

    prefix_bwd_count = len(
        bwd_payloads
    )


    cand_fwd_count = int(
        candidate[
            "fwd_count"
        ]
    )

    cand_bwd_count = int(
        candidate[
            "bwd_count"
        ]
    )


    prefix_fwd_sum = sum(
        fwd_payloads
    )

    prefix_bwd_sum = sum(
        bwd_payloads
    )


    if candidate_complete:

        # Exact complete-flow geometry.
        if cand_fwd_count != prefix_fwd_count:
            return False

        if cand_bwd_count != prefix_bwd_count:
            return False

        if int(
            candidate[
                "fwd_sum"
            ]
        ) != prefix_fwd_sum:
            return False

        if int(
            candidate[
                "bwd_sum"
            ]
        ) != prefix_bwd_sum:
            return False


        if int(
            candidate[
                "fwd_min"
            ]
        ) != min(
            fwd_payloads
        ):
            return False

        if int(
            candidate[
                "fwd_max"
            ]
        ) != max(
            fwd_payloads
        ):
            return False


        if prefix_bwd_count == 0:

            if int(
                candidate[
                    "bwd_min"
                ]
            ) != 0:
                return False

            if int(
                candidate[
                    "bwd_max"
                ]
            ) != 0:
                return False


        else:

            if int(
                candidate[
                    "bwd_min"
                ]
            ) != min(
                bwd_payloads
            ):
                return False

            if int(
                candidate[
                    "bwd_max"
                ]
            ) != max(
                bwd_payloads
            ):
                return False


    else:

        # Safe bounds for a >64-packet flow.
        if cand_fwd_count < prefix_fwd_count:
            return False

        if cand_bwd_count < prefix_bwd_count:
            return False

        if int(
            candidate[
                "fwd_sum"
            ]
        ) < prefix_fwd_sum:
            return False

        if int(
            candidate[
                "bwd_sum"
            ]
        ) < prefix_bwd_sum:
            return False


        if int(
            candidate[
                "fwd_max"
            ]
        ) < max(
            fwd_payloads
        ):
            return False


        if int(
            candidate[
                "fwd_min"
            ]
        ) > min(
            fwd_payloads
        ):
            return False


        if prefix_bwd_count > 0:

            if int(
                candidate[
                    "bwd_max"
                ]
            ) < max(
                bwd_payloads
            ):
                return False

            if int(
                candidate[
                    "bwd_min"
                ]
            ) > min(
                bwd_payloads
            ):
                return False


    return True


# ==============================================================================
# 6. MAXIMUM BIPARTITE MATCHING
# ==============================================================================

def can_match_all(
    corpus_rows,
    published_rows,
    compatibility,
):

    corpus_rows = list(
        corpus_rows
    )

    published_rows = set(
        published_rows
    )


    matched_pub_to_corpus = {}


    def dfs(
        corpus_row,
        seen,
    ):

        for published_row in compatibility[
            corpus_row
        ]:

            if published_row not in published_rows:
                continue

            if published_row in seen:
                continue

            seen.add(
                published_row
            )


            previous = matched_pub_to_corpus.get(
                published_row
            )


            if (
                previous is None
                or
                dfs(
                    previous,
                    seen,
                )
            ):

                matched_pub_to_corpus[
                    published_row
                ] = corpus_row

                return True


        return False


    for corpus_row in sorted(
        corpus_rows,
        key=lambda x: len(
            compatibility[
                x
            ]
        ),
    ):

        if not dfs(
            corpus_row,
            set(),
        ):

            return False


    return True


# ==============================================================================
# 7. BUILD GROUP COMPATIBILITY + DETERMINE FORCED MEMBERSHIP
# ==============================================================================

print("=" * 112)
print("AMBIGUOUS GROUP MATCHING")
print("=" * 112)


corpus_by_key = defaultdict(
    list
)

published_by_key = defaultdict(
    list
)


for index in corpus_ambiguous_idx:

    corpus_by_key[
        int(
            corpus_key[
                int(index)
            ]
        )
    ].append(
        int(
            index
        )
    )


for index in published_ambiguous_idx:

    published_by_key[
        int(
            published_key[
                int(index)
            ]
        )
    ].append(
        int(
            index
        )
    )


forced_included = set()
forced_excluded = set()
still_ambiguous = set()


group_results = []


for group_number, key in enumerate(
    sorted(
        ambiguous_key_values
    ),
    start=1,
):

    C = corpus_by_key[
        key
    ]

    P = published_by_key[
        key
    ]


    compatibility = {}


    for corpus_row in C:

        flow = corpus_flow_info[
            corpus_row
        ]


        compatible = []


        for published_row in P:

            candidate = published_s4[
                published_row
            ]


            if candidate_compatible(
                flow,
                candidate,
            ):

                compatible.append(
                    published_row
                )


        compatibility[
            corpus_row
        ] = compatible


        if not compatible:

            raise RuntimeError(
                "\nNo compatible published candidate for corpus row "
                f"{corpus_row} in key {key}."
            )


    # Whole group must admit the known corpus multiplicity.
    if not can_match_all(
        C,
        P,
        compatibility,
    ):

        raise RuntimeError(
            "\nSafe corpus constraints over-constrained ambiguous group.\n"
            f"key={key}\n"
            f"corpus={len(C)} published={len(P)}"
        )


    possible_excluded = set()


    # A published row CAN be excluded iff all corpus rows can still be matched
    # after removing that physical row.
    for published_row in P:

        remaining = [
            row
            for row in P
            if row != published_row
        ]


        if can_match_all(
            C,
            remaining,
            compatibility,
        ):

            possible_excluded.add(
                published_row
            )


    # If removing a row breaks every full matching, that row is mandatory
    # GROUNDED.
    mandatory_included = (
        set(P)
        -
        possible_excluded
    )


    forced_included.update(
        mandatory_included
    )


    exclusions_needed = (
        len(P)
        -
        len(C)
    )


    # For the overwhelmingly common e=1 case, a single possible exclusion
    # identifies the physical excluded row exactly.
    uniquely_excluded = set()


    if (
        exclusions_needed == 1
        and
        len(
            possible_excluded
        ) == 1
    ):

        uniquely_excluded = set(
            possible_excluded
        )

        forced_excluded.update(
            uniquely_excluded
        )


    unresolved_rows = (
        set(P)
        -
        mandatory_included
        -
        uniquely_excluded
    )


    still_ambiguous.update(
        unresolved_rows
    )


    group_results.append(
        {
            "key":
                key,

            "corpus":
                len(C),

            "published":
                len(P),

            "exclusions_needed":
                exclusions_needed,

            "compat_edges":
                sum(
                    len(
                        compatibility[
                            row
                        ]
                    )
                    for row in C
                ),

            "mandatory_included":
                len(
                    mandatory_included
                ),

            "unique_excluded":
                len(
                    uniquely_excluded
                ),

            "possible_excluded":
                len(
                    possible_excluded
                ),

            "unresolved":
                len(
                    unresolved_rows
                ),
        }
    )


    print(
        f"[{group_number:02d}/{len(ambiguous_key_values):02d}] "
        f"C={len(C):3d} "
        f"P={len(P):3d} "
        f"exclude={exclusions_needed:2d} "
        f"edges={group_results[-1]['compat_edges']:5d} "
        f"mandatory_in={len(mandatory_included):3d} "
        f"unique_out={len(uniquely_excluded):2d} "
        f"possible_out={len(possible_excluded):3d} "
        f"unresolved={len(unresolved_rows):3d}",
        flush=True,
    )


print()


# ==============================================================================
# 8. ACCOUNTING
# ==============================================================================

print("=" * 112)
print("REFINEMENT RESULT")
print("=" * 112)


already_definite_included = 528_081
already_definite_excluded = 1_349


new_forced_included = len(
    forced_included
)

new_forced_excluded = len(
    forced_excluded
)


total_definite_included = (
    already_definite_included
    +
    new_forced_included
)


total_definite_excluded = (
    already_definite_excluded
    +
    new_forced_excluded
)


print(
    "Previously definite GROUNDED:",
    f"{already_definite_included:,}",
)

print(
    "New mandatory GROUNDED:     ",
    f"{new_forced_included:,}",
)

print(
    "Total definite GROUNDED:    ",
    f"{total_definite_included:,}",
)

print()

print(
    "Previously definite EXCLUDED:",
    f"{already_definite_excluded:,}",
)

print(
    "New uniquely EXCLUDED:       ",
    f"{new_forced_excluded:,}",
)

print(
    "Total definite EXCLUDED:     ",
    f"{total_definite_excluded:,}",
)

print()

print(
    "Published rows still involved in ambiguity:",
    f"{len(still_ambiguous):,}",
)


resolved_exclusion_count = (
    total_definite_excluded
)


remaining_exclusions = (
    1_409
    -
    resolved_exclusion_count
)


print(
    "Exclusions still unresolved:",
    f"{remaining_exclusions:,}",
)

print()


# ==============================================================================
# 9. SHOW ONLY REMAINING GROUPS
# ==============================================================================

remaining_groups = [
    item
    for item in group_results
    if item[
        "unresolved"
    ] > 0
]


print("=" * 112)
print("REMAINING AMBIGUOUS GROUPS")
print("=" * 112)


if not remaining_groups:

    print(
        "<NONE>"
    )


else:

    for item in remaining_groups:

        key = int(
            item[
                "key"
            ]
        )


        protocol = (
            key
            >>
            40
        ) & 0xFF

        port_lo = (
            key
            >>
            24
        ) & 0xFFFF

        port_hi = (
            key
            >>
            8
        ) & 0xFFFF

        packet_cap = (
            key
            &
            0xFF
        )


        print(
            f"proto={protocol:2d} "
            f"ports=({port_lo},{port_hi}) "
            f"cap={packet_cap:2d} "
            f"| C={item['corpus']} "
            f"P={item['published']} "
            f"need_out={item['exclusions_needed']} "
            f"possible_out={item['possible_excluded']} "
            f"unresolved_rows={item['unresolved']}"
        )


# ==============================================================================
# 10. SAVE DIAGNOSTIC STATE ONLY
#
# NOT a scientific membership freeze yet.
# ==============================================================================

OUT = Path(
    "/kaggle/working/stage24_corpus_alignment"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


DIAG = (
    OUT
    / "monday_ambiguous_s4_geometry_resolution.npz"
)


np.savez_compressed(
    DIAG,

    forced_included_published_row_index=
        np.asarray(
            sorted(
                forced_included
            ),
            dtype=np.int32,
        ),

    forced_excluded_published_row_index=
        np.asarray(
            sorted(
                forced_excluded
            ),
            dtype=np.int32,
        ),

    still_ambiguous_published_row_index=
        np.asarray(
            sorted(
                still_ambiguous
            ),
            dtype=np.int32,
        ),
)


print()
print(
    "Diagnostic state:",
    DIAG,
)

print(
    "Bytes:",
    f"{DIAG.stat().st_size:,}",
)

print()


# ==============================================================================
# 11. FINAL
# ==============================================================================

print("=" * 112)
print("STAGE24-1C1-M AMBIGUOUS S4 GEOMETRY RESOLUTION COMPLETE")
print("=" * 112)

print()

print(
    "PCAP accessed:       NO"
)

print(
    "Raw source present:  NO"
)

print(
    "Compact corpus used: YES"
)

print(
    "Model executed:      NO"
)

print(
    "Target predictions:  0"
)

print(
    "Target openings:     0 / 8"
)

print(
    "Scientific fits:     2 / 4"
)

print(
    "Git changes:         NONE"
)

print()


if remaining_exclusions == 0:

    print(
        "[PASS] ALL 1,409 Monday physical exclusions resolved corpus-side."
    )

    print(
        "NEXT: construct and freeze the exact 528,509-row Monday membership."
    )

else:

    print(
        f"[PARTIAL] {remaining_exclusions:,} exclusion(s) remain unresolved."
    )

    print(
        "NEXT: inspect ONLY those remaining equivalence classes; "
        "no PCAP reconstruction."
    )


print("=" * 112)

STAGE24-1C1-M — AMBIGUOUS S4 GEOMETRY RESOLUTION
Ambiguous keys: 55
Corpus rows: 428
Published rows: 488

DuckDB: 1.3.2
Scapy: 2.6.1

READING PUBLISHED S4 GEOMETRY
[PASS] Published S4 geometry loaded.

DECODING 428 AMBIGUOUS CORPUS FLOWS
  decoded 100 / 428
  decoded 200 / 428
  decoded 300 / 428
  decoded 400 / 428
[PASS] Ambiguous compact flows decoded.

AMBIGUOUS GROUP MATCHING
[01/55] C= 39 P= 40 exclude= 1 edges= 1560 mandatory_in=  0 unique_out= 0 possible_out= 40 unresolved= 40
[02/55] C=  1 P=  2 exclude= 1 edges=    2 mandatory_in=  0 unique_out= 0 possible_out=  2 unresolved=  2
[03/55] C=  1 P=  3 exclude= 2 edges=    3 mandatory_in=  0 unique_out= 0 possible_out=  3 unresolved=  3
[04/55] C=  1 P=  2 exclude= 1 edges=    2 mandatory_in=  0 unique_out= 0 possible_out=  2 unresolved=  2
[05/55] C=  1 P=  2 exclude= 1 edges=    2 mandatory_in=  0 unique_out= 0 possible_out=  2 unresolved=  2
[06/55] C=  1 P=  2 exclude= 1 edges=    1 mandatory_in=  1 unique_out= 1 possible_out

In [24]:
# ==============================================================================
# STAGE24-1C-EX1
# GROUNDED_S4 DURABLE-ARTIFACT INFEASIBILITY AMENDMENT + REMOTE FREEZE
#
# Scientific decision:
#
#   GROUNDED_S4 remains exactly as preregistered.
#
#   However, exact physical-row membership cannot be recovered from the
#   durable Stage20 compact corpora because the persisted representation
#   intentionally omits identity dimensions required by the frozen 21-field
#   exact S4 join.
#
#   No substitute / fuzzy / inferred membership is adopted.
#
# Consequence:
#
#   CANCELLED BEFORE TARGET OPENING:
#       primary bridge62 / GROUNDED_S4
#       primary bridge70 / GROUNDED_S4
#
#   CONTINUE:
#       primary bridge62 / PUBLISHED
#       primary bridge62 / FLAG_CORRECTED
#       primary bridge70 / PUBLISHED
#       primary bridge70 / FLAG_CORRECTED
#       secondary bridge62 / IDS2018-Feb28
#       secondary bridge70 / IDS2018-Feb28
#
# Opening cap remains 8.
# 2 openings are administratively unavailable and may NOT be reassigned.
# 6 frozen cells remain evaluable.
#
# NO MODEL.
# NO TARGET OPENING.
# ==============================================================================

from __future__ import annotations

import os
import json
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

EXPECTED_PARENT = (
    "683fe85839e63dcd5fd8e952fd07931b446652ff"
)

PROTOCOL_SHA256 = (
    "8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2"
)

BRIDGE_SEMANTIC_SHA256 = (
    "b7ff1d8563aafc69145be3acc827fbc59ae75d880b9804bad727f408299fce2f"
)

PREOPENING_ATTESTATION_COMMIT = (
    "3a0b4a54477a5159805e69caa1eba749ab614c7b"
)

EXPECTED_CORPUS_ROWS = 528_509
EXPECTED_PUBLISHED_ROWS = 529_918

EXPECTED_INITIAL_DEFINITE_GROUNDED = 528_081
EXPECTED_INITIAL_DEFINITE_EXCLUDED = 1_349

EXPECTED_INITIAL_AMBIGUOUS_KEYS = 55
EXPECTED_INITIAL_AMBIGUOUS_PUBLISHED = 488
EXPECTED_INITIAL_AMBIGUOUS_GROUNDED = 428
EXPECTED_INITIAL_AMBIGUOUS_EXCLUDED = 60

EXPECTED_NEW_MANDATORY_GROUNDED = 357
EXPECTED_NEW_UNIQUE_EXCLUDED = 35

EXPECTED_FINAL_DEFINITE_GROUNDED = 528_438
EXPECTED_FINAL_DEFINITE_EXCLUDED = 1_384

EXPECTED_REMAINING_ROWS = 96
EXPECTED_REMAINING_EXCLUSIONS = 25
EXPECTED_REMAINING_GROUPS = 20


# ==============================================================================
# 1. PATHS
# ==============================================================================

REPO = Path(
    globals().get(
        "REPO",
        "/kaggle/working/ids2018-validation-safe-ablation",
    )
)

if not REPO.is_dir():
    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )


OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1c_grounded_s4_membership"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


AMENDMENT = (
    OUT
    / "stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendment.json"
)

AMENDMENT_SHA = (
    OUT
    / "stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendment.sha256"
)


DIAG = Path(
    "/kaggle/working/stage24_corpus_alignment/"
    "monday_ambiguous_s4_geometry_resolution.npz"
)


print("=" * 112)
print("STAGE24-1C-EX1 — GROUNDED_S4 DURABLE-ARTIFACT INFEASIBILITY AMENDMENT")
print("=" * 112)
print()


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(args, *, cwd=None, check=True):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(str(x) for x in args)
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(*args, auth_header=None, check=True):

    command = ["git"]

    if auth_header is not None:
        command += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    command += [str(x) for x in args]

    return run_cmd(
        command,
        cwd=REPO,
        check=check,
    )


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as fh:
        while True:

            chunk = fh.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ==============================================================================
# 3. GOVERNANCE GATE
# ==============================================================================

print("=" * 112)
print("GOVERNANCE")
print("=" * 112)


head = git_cmd(
    "rev-parse",
    "HEAD",
)


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


status = git_cmd(
    "status",
    "--porcelain",
)


if status:
    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


print(
    "HEAD:",
    head,
)

print(
    "[PASS] Repository clean."
)

print(
    "[PASS] Target openings remain 0 / 8."
)

print(
    "[PASS] Scientific fits remain 2 / 4."
)

print()


# ==============================================================================
# 4. VERIFY THE LAST CORPUS-SIDE DIAGNOSTIC
# ==============================================================================

print("=" * 112)
print("CORPUS-SIDE IDENTIFIABILITY EVIDENCE")
print("=" * 112)


if not DIAG.is_file():
    raise RuntimeError(
        f"Expected diagnostic artifact missing:\n{DIAG}"
    )


diag = np.load(
    DIAG,
    allow_pickle=False,
)


required_arrays = {
    "forced_included_published_row_index",
    "forced_excluded_published_row_index",
    "still_ambiguous_published_row_index",
}


if set(diag.files) != required_arrays:
    raise RuntimeError(
        "\nUnexpected diagnostic arrays.\n"
        f"Expected: {sorted(required_arrays)}\n"
        f"Actual:   {sorted(diag.files)}"
    )


forced_included = np.asarray(
    diag[
        "forced_included_published_row_index"
    ],
    dtype=np.int32,
)


forced_excluded = np.asarray(
    diag[
        "forced_excluded_published_row_index"
    ],
    dtype=np.int32,
)


still_ambiguous = np.asarray(
    diag[
        "still_ambiguous_published_row_index"
    ],
    dtype=np.int32,
)


print(
    "New mandatory GROUNDED:",
    f"{forced_included.size:,}",
)

print(
    "New uniquely EXCLUDED:",
    f"{forced_excluded.size:,}",
)

print(
    "Rows still ambiguous:",
    f"{still_ambiguous.size:,}",
)


if forced_included.size != EXPECTED_NEW_MANDATORY_GROUNDED:
    raise RuntimeError(
        "Unexpected mandatory-grounded count."
    )


if forced_excluded.size != EXPECTED_NEW_UNIQUE_EXCLUDED:
    raise RuntimeError(
        "Unexpected uniquely-excluded count."
    )


if still_ambiguous.size != EXPECTED_REMAINING_ROWS:
    raise RuntimeError(
        "Unexpected remaining ambiguous-row count."
    )


# Use the live diagnostic state from the just-completed audit when available.
if "remaining_exclusions" in globals():

    observed_remaining_exclusions = int(
        remaining_exclusions
    )

else:

    observed_remaining_exclusions = (
        EXPECTED_REMAINING_EXCLUSIONS
    )


if observed_remaining_exclusions != EXPECTED_REMAINING_EXCLUSIONS:
    raise RuntimeError(
        "\nUnexpected unresolved exclusion count.\n"
        f"Expected: {EXPECTED_REMAINING_EXCLUSIONS}\n"
        f"Actual:   {observed_remaining_exclusions}"
    )


if "remaining_groups" in globals():

    observed_remaining_groups = len(
        remaining_groups
    )

else:

    observed_remaining_groups = (
        EXPECTED_REMAINING_GROUPS
    )


if observed_remaining_groups != EXPECTED_REMAINING_GROUPS:
    raise RuntimeError(
        "\nUnexpected remaining-group count.\n"
        f"Expected: {EXPECTED_REMAINING_GROUPS}\n"
        f"Actual:   {observed_remaining_groups}"
    )


diag_sha = sha256_file(
    DIAG
)


print(
    "Diagnostic SHA256:",
    diag_sha,
)

print()
print(
    "[PASS] Corpus-side feasibility evidence reproduced."
)

print()


# ==============================================================================
# 5. GITHUB CREDENTIAL BEFORE WRITE
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if value and value.strip():

            github_token = value.strip()
            token_source = f"ENV:{name}"
            break


if github_token is None:
    raise RuntimeError(
        "GitHub token unavailable."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        + github_token
    ).encode()
).decode()


print(
    "GitHub credential:",
    token_source,
)

print()


# ==============================================================================
# 6. REMOTE PARENT GATE
# ==============================================================================

remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:
    raise RuntimeError(
        "\nRemote main moved unexpectedly.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "[PASS] Remote main == expected Stage24-1B parent."
)

print()


# ==============================================================================
# 7. FORMAL AMENDMENT
# ==============================================================================

amendment = {
    "checkpoint":
        "Stage24-1C-EX1",

    "status":
        "GROUNDED_S4_DURABLE_ARTIFACT_INFEASIBILITY_FROZEN",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_PARENT,

    "frozen_protocol_sha256":
        PROTOCOL_SHA256,

    "bridge_semantic_sha256":
        BRIDGE_SEMANTIC_SHA256,

    "preopening_attestation_commit":
        PREOPENING_ATTESTATION_COMMIT,

    "scientific_reason": {
        "requested_variant":
            "GROUNDED_S4",

        "frozen_grounding_definition":
            "FROZEN_DIRECTED_S4_EXACT_21_FIELD_SIGNATURE",

        "definition_changed":
            False,

        "matching_rule_changed":
            False,

        "replacement_population_created":
            False,

        "fuzzy_matching_adopted":
            False,

        "nearest_matching_adopted":
            False,

        "tolerance_matching_adopted":
            False,

        "label_guided_repair_adopted":
            False,

        "inferred_membership_adopted":
            False,

        "reason_not_evaluable":
            (
                "The durable Stage20 compact corpus preserves the exact accepted "
                "flow population and packet-image representation but does not "
                "persist all identity dimensions required to recover the exact "
                "physical published-row occurrence of the frozen 21-field S4 "
                "signature. IPv4 source/destination identity was intentionally "
                "masked in the persisted representation and flow-start timestamp "
                "was not persisted. Corpus-side deterministic refinement therefore "
                "cannot uniquely identify every physical published occurrence."
            ),
    },

    "durable_stage20_population": {
        "published_monday_rows":
            EXPECTED_PUBLISHED_ROWS,

        "grounded_s4_monday_flows":
            EXPECTED_CORPUS_ROWS,

        "published_occurrences_not_consumed":
            1_409,

        "compact_corpus_byte_exact":
            True,

        "raw_pcap_required_for_full_identity_reconstruction":
            True,

        "raw_pcap_replay_performed_after_amendment_decision":
            False,
    },

    "corpus_side_feasibility_audit": {
        "coarse_stage": {
            "definite_grounded":
                EXPECTED_INITIAL_DEFINITE_GROUNDED,

            "definite_excluded":
                EXPECTED_INITIAL_DEFINITE_EXCLUDED,

            "ambiguous_keys":
                EXPECTED_INITIAL_AMBIGUOUS_KEYS,

            "ambiguous_published_rows":
                EXPECTED_INITIAL_AMBIGUOUS_PUBLISHED,

            "ambiguous_grounded_rows":
                EXPECTED_INITIAL_AMBIGUOUS_GROUNDED,

            "ambiguous_excluded_rows":
                EXPECTED_INITIAL_AMBIGUOUS_EXCLUDED,
        },

        "geometry_refinement": {
            "new_mandatory_grounded":
                EXPECTED_NEW_MANDATORY_GROUNDED,

            "new_uniquely_excluded":
                EXPECTED_NEW_UNIQUE_EXCLUDED,

            "final_definite_grounded":
                EXPECTED_FINAL_DEFINITE_GROUNDED,

            "final_definite_excluded":
                EXPECTED_FINAL_DEFINITE_EXCLUDED,

            "remaining_ambiguous_published_rows":
                EXPECTED_REMAINING_ROWS,

            "remaining_unresolved_exclusions":
                EXPECTED_REMAINING_EXCLUSIONS,

            "remaining_equivalence_classes":
                EXPECTED_REMAINING_GROUPS,

            "diagnostic_npz_sha256":
                diag_sha,
        },

        "decision":
            (
                "STOP_WITHOUT_ADOPTING_A_NEW_MEMBERSHIP_RULE"
            ),
    },

    "cell_disposition": {
        "administratively_cancelled_before_opening": [
            {
                "direction":
                    "IDS2018_TO_CICIDS2017",

                "bridge":
                    "bridge62",

                "target_variant":
                    "GROUNDED_S4",
            },
            {
                "direction":
                    "IDS2018_TO_CICIDS2017",

                "bridge":
                    "bridge70",

                "target_variant":
                    "GROUNDED_S4",
            },
        ],

        "still_evaluable": [
            {
                "direction":
                    "IDS2018_TO_CICIDS2017",

                "bridge":
                    "bridge62",

                "target_variant":
                    "PUBLISHED",
            },
            {
                "direction":
                    "IDS2018_TO_CICIDS2017",

                "bridge":
                    "bridge62",

                "target_variant":
                    "FLAG_CORRECTED",
            },
            {
                "direction":
                    "IDS2018_TO_CICIDS2017",

                "bridge":
                    "bridge70",

                "target_variant":
                    "PUBLISHED",
            },
            {
                "direction":
                    "IDS2018_TO_CICIDS2017",

                "bridge":
                    "bridge70",

                "target_variant":
                    "FLAG_CORRECTED",
            },
            {
                "direction":
                    "CICIDS2017_TO_IDS2018",

                "bridge":
                    "bridge62",

                "target_variant":
                    "IDS2018_FEB28",
            },
            {
                "direction":
                    "CICIDS2017_TO_IDS2018",

                "bridge":
                    "bridge70",

                "target_variant":
                    "IDS2018_FEB28",
            },
        ],

        "cancelled_cells_may_be_reallocated":
            False,

        "new_target_variants_may_replace_cancelled_cells":
            False,
    },

    "budget": {
        "original_target_opening_cap":
            8,

        "target_openings_consumed":
            0,

        "administratively_cancelled_before_opening":
            2,

        "remaining_evaluable_frozen_cells":
            6,

        "cancelled_openings_reassignable":
            False,

        "scientific_fit_budget":
            4,

        "scientific_fits_completed":
            2,

        "fit_budget_changed":
            False,
    },

    "anti_adaptation": {
        "cicids2017_target_rows_supplied_to_model":
            0,

        "target_predictions_created":
            0,

        "target_metrics_computed":
            0,

        "target_threshold_tuning":
            False,

        "target_feature_selection":
            False,

        "target_model_selection":
            False,

        "target_result_used_for_amendment":
            False,

        "amendment_basis":
            "DURABLE_ARTIFACT_IDENTIFIABILITY_ONLY",
    },

    "forward_plan": [
        "Do not acquire or replay CICIDS2017 PCAPs for GROUNDED_S4.",
        "Do not download additional Stage20 compact corpora for Stage24 grounding.",
        "Use published CICIDS2017 feature/label parquets for PUBLISHED and FLAG_CORRECTED.",
        "Continue the frozen primary bridge62 and bridge70 target cells.",
        "Continue the frozen secondary IDS2018-Feb28 cells.",
        "Report GROUNDED_S4 as unavailable from durable artifacts, not as a negative result.",
    ],
}


AMENDMENT.write_text(
    json.dumps(
        amendment,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


amendment_sha = sha256_file(
    AMENDMENT
)


AMENDMENT_SHA.write_text(
    f"{amendment_sha}  {AMENDMENT.name}\n",
    encoding="utf-8",
)


print("=" * 112)
print("AMENDMENT ARTIFACT")
print("=" * 112)

print(
    "Path:",
    AMENDMENT.relative_to(
        REPO
    ),
)

print(
    "SHA256:",
    amendment_sha,
)

print()


# ==============================================================================
# 8. COMMIT
# ==============================================================================

git_cmd(
    "add",
    "--",
    str(
        AMENDMENT.relative_to(
            REPO
        )
    ),
    str(
        AMENDMENT_SHA.relative_to(
            REPO
        )
    ),
)


staged = set(
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
)


expected_staged = {
    str(
        AMENDMENT.relative_to(
            REPO
        )
    ),
    str(
        AMENDMENT_SHA.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:
    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


git_cmd(
    "commit",
    "-m",
    "stage24: freeze GROUNDED_S4 artifact infeasibility amendment",
)


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:
    raise RuntimeError(
        "Amendment commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 9. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 112)
print("PUSH + REMOTE VERIFY")
print("=" * 112)


print(
    git_cmd(
        "push",
        "origin",
        "HEAD:main",
        auth_header=auth_header,
    )
)


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:
    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:
    raise RuntimeError(
        "\nRepository not clean:\n"
        + final_status
    )


print()
print(
    "[PASS] Amendment remote-durable."
)

print()


# ==============================================================================
# 10. FINAL
# ==============================================================================

print("=" * 112)
print("STAGE24-1C-EX1: PASS")
print("=" * 112)

print()

print(
    "GROUNDED_S4 definition changed:          NO"
)

print(
    "Inferred/fuzzy membership adopted:       NO"
)

print(
    "Primary GROUNDED_S4 cells cancelled:      2"
)

print(
    "Cancelled cells reassignable:             NO"
)

print()

print(
    "Original opening cap:                     8"
)

print(
    "Openings consumed:                        0"
)

print(
    "Frozen evaluable cells remaining:         6"
)

print()

print(
    "Scientific fits completed:                2 / 4"
)

print(
    "Target predictions:                       0"
)

print(
    "Target metrics:                           0"
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "NEXT:"
)

print(
    "  Acquire only the remaining CICIDS2017 published feature/label parquets."
)

print(
    "  NO PCAPs."
)

print(
    "  NO additional Stage20 compact corpora."
)

print(
    "  Then execute primary PUBLISHED / FLAG_CORRECTED target cells."
)

print("=" * 112)

STAGE24-1C-EX1 — GROUNDED_S4 DURABLE-ARTIFACT INFEASIBILITY AMENDMENT

GOVERNANCE


RuntimeError: 
Repository is not clean:
A  results/stage24_cross_dataset/stage24_1_primary_source_sanity/stage24_1c_grounded_s4_membership/stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendment.json
A  results/stage24_cross_dataset/stage24_1_primary_source_sanity/stage24_1c_grounded_s4_membership/stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendment.sha256

In [25]:
# ==============================================================================
# STAGE24-1C-EX1-R1
# RECOVER DOUBLE-RUN STAGED AMENDMENT -> VALIDATE -> COMMIT -> PUSH
#
# Situation:
#   Previous Stage24-1C-EX1 execution created/staged the amendment files.
#   A second accidental execution then stopped at the clean-repository gate.
#
# This recovery cell:
#   - accepts ONLY the two expected staged amendment files;
#   - validates the JSON scientifically;
#   - validates its .sha256 sidecar;
#   - validates the surviving corpus diagnostic;
#   - commits the existing files;
#   - pushes and remote-verifies main.
#
# NO model.
# NO target prediction.
# NO target opening.
# ==============================================================================

from __future__ import annotations

import os
import json
import base64
import hashlib
import subprocess
from pathlib import Path

import numpy as np


print("=" * 112)
print("STAGE24-1C-EX1-R1 — DOUBLE-RUN RECOVERY")
print("=" * 112)
print()


# ==============================================================================
# 0. FROZEN STATE
# ==============================================================================

EXPECTED_PARENT = (
    "683fe85839e63dcd5fd8e952fd07931b446652ff"
)

EXPECTED_UNRESOLVED_EXCLUSIONS = 25
EXPECTED_AMBIGUOUS_ROWS = 96

EXPECTED_NEW_MANDATORY_GROUNDED = 357
EXPECTED_NEW_UNIQUE_EXCLUDED = 35

EXPECTED_OPENINGS_CONSUMED = 0
EXPECTED_FITS_COMPLETED = 2


REPO = Path(
    globals().get(
        "REPO",
        "/kaggle/working/ids2018-validation-safe-ablation",
    )
)


REL_JSON = Path(
    "results/stage24_cross_dataset/"
    "stage24_1_primary_source_sanity/"
    "stage24_1c_grounded_s4_membership/"
    "stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendment.json"
)


REL_SHA = Path(
    "results/stage24_cross_dataset/"
    "stage24_1_primary_source_sanity/"
    "stage24_1c_grounded_s4_membership/"
    "stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendment.sha256"
)


JSON_PATH = REPO / REL_JSON
SHA_PATH = REPO / REL_SHA


DIAG = Path(
    "/kaggle/working/stage24_corpus_alignment/"
    "monday_ambiguous_s4_geometry_resolution.npz"
)


EXPECTED_DIRTY_PATHS = {
    str(REL_JSON),
    str(REL_SHA),
}


if not REPO.is_dir():
    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )


# ==============================================================================
# 1. HELPERS
# ==============================================================================

def run_cmd(args, *, cwd=None, check=True):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(str(x) for x in args)
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(*args, auth_header=None, check=True):

    command = ["git"]

    if auth_header is not None:

        command += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    command += [
        str(x)
        for x in args
    ]

    return run_cmd(
        command,
        cwd=REPO,
        check=check,
    )


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            chunk = fh.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ==============================================================================
# 2. HEAD MUST STILL BE STAGE24-1B
# ==============================================================================

print("=" * 112)
print("HEAD / DIRTY-STATE RECOVERY GATE")
print("=" * 112)


head = git_cmd(
    "rev-parse",
    "HEAD",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nHEAD has changed; refusing automatic recovery.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


status_lines = [
    line
    for line in git_cmd(
        "status",
        "--porcelain",
    ).splitlines()
    if line.strip()
]


print()
print(
    "Dirty entries:",
    len(status_lines),
)


for line in status_lines:
    print(
        " ",
        line,
    )


# Extract path from porcelain line.
dirty_paths = set()


for line in status_lines:

    # Standard porcelain:
    # XY<space>path
    path_text = line[
        3:
    ].strip()


    # Handle rename form old -> new defensively.
    if " -> " in path_text:
        path_text = path_text.split(
            " -> ",
            1,
        )[1]


    dirty_paths.add(
        path_text
    )


unexpected = (
    dirty_paths
    -
    EXPECTED_DIRTY_PATHS
)


if unexpected:

    raise RuntimeError(
        "\nUnexpected repository changes exist.\n"
        "Nothing was modified.\n\n"
        + "\n".join(
            sorted(
                unexpected
            )
        )
    )


if dirty_paths != EXPECTED_DIRTY_PATHS:

    raise RuntimeError(
        "\nExpected both amendment files from the first execution.\n"
        f"Expected: {sorted(EXPECTED_DIRTY_PATHS)}\n"
        f"Actual:   {sorted(dirty_paths)}"
    )


print()
print(
    "[PASS] Dirty state contains ONLY the two expected Stage24 amendment files."
)

print()


# ==============================================================================
# 3. BOTH FILES MUST EXIST
# ==============================================================================

for path in (
    JSON_PATH,
    SHA_PATH,
):

    if not path.is_file():

        raise RuntimeError(
            f"Expected staged artifact missing:\n{path}"
        )


print(
    "[PASS] Both amendment artifacts exist."
)

print()


# ==============================================================================
# 4. VALIDATE SURVIVING CORPUS DIAGNOSTIC
# ==============================================================================

print("=" * 112)
print("CORPUS DIAGNOSTIC VALIDATION")
print("=" * 112)


if not DIAG.is_file():

    raise RuntimeError(
        f"Corpus diagnostic missing:\n{DIAG}"
    )


diag = np.load(
    DIAG,
    allow_pickle=False,
)


expected_arrays = {
    "forced_included_published_row_index",
    "forced_excluded_published_row_index",
    "still_ambiguous_published_row_index",
}


if set(
    diag.files
) != expected_arrays:

    raise RuntimeError(
        "\nUnexpected diagnostic arrays.\n"
        f"Expected: {sorted(expected_arrays)}\n"
        f"Actual:   {sorted(diag.files)}"
    )


forced_included = np.asarray(
    diag[
        "forced_included_published_row_index"
    ],
    dtype=np.int32,
)


forced_excluded = np.asarray(
    diag[
        "forced_excluded_published_row_index"
    ],
    dtype=np.int32,
)


still_ambiguous = np.asarray(
    diag[
        "still_ambiguous_published_row_index"
    ],
    dtype=np.int32,
)


print(
    "Mandatory GROUNDED recovered:",
    f"{forced_included.size:,}",
)

print(
    "Unique EXCLUDED recovered:",
    f"{forced_excluded.size:,}",
)

print(
    "Rows still ambiguous:",
    f"{still_ambiguous.size:,}",
)


if forced_included.size != EXPECTED_NEW_MANDATORY_GROUNDED:
    raise RuntimeError(
        "Mandatory-grounded diagnostic count changed."
    )


if forced_excluded.size != EXPECTED_NEW_UNIQUE_EXCLUDED:
    raise RuntimeError(
        "Unique-excluded diagnostic count changed."
    )


if still_ambiguous.size != EXPECTED_AMBIGUOUS_ROWS:
    raise RuntimeError(
        "Remaining ambiguous-row count changed."
    )


diag_sha = sha256_file(
    DIAG
)


print(
    "Diagnostic SHA256:",
    diag_sha,
)

print()
print(
    "[PASS] Corpus diagnostic unchanged."
)

print()


# ==============================================================================
# 5. VALIDATE EXISTING AMENDMENT JSON
# ==============================================================================

print("=" * 112)
print("EXISTING AMENDMENT VALIDATION")
print("=" * 112)


try:

    amendment = json.loads(
        JSON_PATH.read_text(
            encoding="utf-8"
        )
    )

except Exception as exc:

    raise RuntimeError(
        "Existing amendment JSON is not valid JSON."
    ) from exc


checks = []


def require(
    condition,
    description,
):

    if not condition:
        raise RuntimeError(
            f"Amendment validation failed:\n{description}"
        )

    checks.append(
        description
    )


require(
    amendment.get(
        "checkpoint"
    )
    ==
    "Stage24-1C-EX1",

    "checkpoint == Stage24-1C-EX1",
)


require(
    amendment.get(
        "status"
    )
    ==
    "GROUNDED_S4_DURABLE_ARTIFACT_INFEASIBILITY_FROZEN",

    "status is frozen durable-artifact infeasibility",
)


require(
    amendment.get(
        "parent_commit"
    )
    ==
    EXPECTED_PARENT,

    "parent commit matches Stage24-1B",
)


scientific_reason = amendment.get(
    "scientific_reason",
    {},
)


require(
    scientific_reason.get(
        "definition_changed"
    )
    is False,

    "GROUNDED_S4 definition unchanged",
)


require(
    scientific_reason.get(
        "matching_rule_changed"
    )
    is False,

    "matching rule unchanged",
)


require(
    scientific_reason.get(
        "replacement_population_created"
    )
    is False,

    "no replacement population created",
)


require(
    scientific_reason.get(
        "inferred_membership_adopted"
    )
    is False,

    "no inferred membership adopted",
)


audit = (
    amendment
    .get(
        "corpus_side_feasibility_audit",
        {}
    )
    .get(
        "geometry_refinement",
        {}
    )
)


require(
    int(
        audit.get(
            "remaining_ambiguous_published_rows",
            -1,
        )
    )
    ==
    EXPECTED_AMBIGUOUS_ROWS,

    "remaining ambiguous rows == 96",
)


require(
    int(
        audit.get(
            "remaining_unresolved_exclusions",
            -1,
        )
    )
    ==
    EXPECTED_UNRESOLVED_EXCLUSIONS,

    "remaining unresolved exclusions == 25",
)


require(
    audit.get(
        "diagnostic_npz_sha256"
    )
    ==
    diag_sha,

    "diagnostic SHA matches current NPZ",
)


budget = amendment.get(
    "budget",
    {},
)


require(
    int(
        budget.get(
            "original_target_opening_cap",
            -1,
        )
    )
    ==
    8,

    "original opening cap remains 8",
)


require(
    int(
        budget.get(
            "target_openings_consumed",
            -1,
        )
    )
    ==
    EXPECTED_OPENINGS_CONSUMED,

    "target openings consumed == 0",
)


require(
    int(
        budget.get(
            "administratively_cancelled_before_opening",
            -1,
        )
    )
    ==
    2,

    "exactly two GROUNDED_S4 cells cancelled",
)


require(
    int(
        budget.get(
            "remaining_evaluable_frozen_cells",
            -1,
        )
    )
    ==
    6,

    "six frozen cells remain evaluable",
)


require(
    budget.get(
        "cancelled_openings_reassignable"
    )
    is False,

    "cancelled openings are not reassignable",
)


require(
    int(
        budget.get(
            "scientific_fits_completed",
            -1,
        )
    )
    ==
    EXPECTED_FITS_COMPLETED,

    "scientific fits remain 2 / 4",
)


anti = amendment.get(
    "anti_adaptation",
    {},
)


require(
    int(
        anti.get(
            "cicids2017_target_rows_supplied_to_model",
            -1,
        )
    )
    ==
    0,

    "zero CICIDS2017 target rows supplied to model",
)


require(
    int(
        anti.get(
            "target_predictions_created",
            -1,
        )
    )
    ==
    0,

    "zero target predictions",
)


require(
    int(
        anti.get(
            "target_metrics_computed",
            -1,
        )
    )
    ==
    0,

    "zero target metrics",
)


cancelled = (
    amendment
    .get(
        "cell_disposition",
        {}
    )
    .get(
        "administratively_cancelled_before_opening",
        []
    )
)


require(
    len(
        cancelled
    )
    ==
    2,

    "two cancelled cell records present",
)


cancelled_pairs = {
    (
        item.get(
            "bridge"
        ),
        item.get(
            "target_variant"
        ),
    )
    for item in cancelled
}


require(
    cancelled_pairs
    ==
    {
        (
            "bridge62",
            "GROUNDED_S4",
        ),
        (
            "bridge70",
            "GROUNDED_S4",
        ),
    },

    "cancelled cells are exactly bridge62/70 GROUNDED_S4",
)


print(
    f"[PASS] {len(checks)} amendment governance checks."
)

print()


# ==============================================================================
# 6. VALIDATE SHA256 SIDECAR
# ==============================================================================

print("=" * 112)
print("AMENDMENT BYTE IDENTITY")
print("=" * 112)


actual_json_sha = sha256_file(
    JSON_PATH
)


sidecar_text = SHA_PATH.read_text(
    encoding="utf-8"
).strip()


sidecar_parts = sidecar_text.split()


if not sidecar_parts:

    raise RuntimeError(
        "SHA256 sidecar is empty."
    )


sidecar_sha = sidecar_parts[
    0
].strip().lower()


print(
    "JSON SHA256:",
    actual_json_sha,
)

print(
    "Sidecar SHA:",
    sidecar_sha,
)


if sidecar_sha != actual_json_sha:

    raise RuntimeError(
        "\nExisting amendment SHA sidecar does not match JSON.\n"
        "Nothing was committed."
    )


print()
print(
    "[PASS] Existing staged amendment is byte-consistent."
)

print()


# ==============================================================================
# 7. GITHUB TOKEN
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if value and value.strip():

            github_token = value.strip()
            token_source = (
                "ENV:"
                + name
            )

            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


print(
    "GitHub credential:",
    token_source,
)

print()


# ==============================================================================
# 8. REMOTE MAIN MUST STILL BE PARENT
# ==============================================================================

print("=" * 112)
print("REMOTE PARENT GATE")
print("=" * 112)


remote_before_output = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
)


if not remote_before_output:

    raise RuntimeError(
        "Could not resolve remote main."
    )


remote_before = remote_before_output.split()[0]


print(
    "Remote main:",
    remote_before,
)


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main changed unexpectedly.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "[PASS] Remote main still equals Stage24-1B."
)

print()


# ==============================================================================
# 9. NORMALIZE STAGING
#
# Re-add the exact two files. This is harmless if already staged.
# ==============================================================================

git_cmd(
    "add",
    "--",
    str(
        REL_JSON
    ),
    str(
        REL_SHA
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


if staged != EXPECTED_DIRTY_PATHS:

    raise RuntimeError(
        "\nUnexpected staged file set.\n"
        f"Expected: {sorted(EXPECTED_DIRTY_PATHS)}\n"
        f"Actual:   {sorted(staged)}"
    )


print(
    "[PASS] Staging contains exactly two amendment artifacts."
)

print()


# ==============================================================================
# 10. COMMIT
# ==============================================================================

print("=" * 112)
print("COMMIT")
print("=" * 112)


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: freeze GROUNDED_S4 artifact infeasibility amendment",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "\nNew commit has wrong parent.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {parent}"
    )


print(
    "Commit:",
    commit,
)

print(
    "[PASS] Amendment committed."
)

print()


# ==============================================================================
# 11. PUSH
# ==============================================================================

print("=" * 112)
print("PUSH")
print("=" * 112)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


# ==============================================================================
# 12. REMOTE VERIFY
# ==============================================================================

remote_after_output = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
)


remote_after = remote_after_output.split()[0]


print(
    "Local HEAD: ",
    commit,
)

print(
    "Remote main:",
    remote_after,
)


if remote_after != commit:

    raise RuntimeError(
        "Remote commit verification failed."
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository is not clean after commit:\n"
        + final_status
    )


print()
print(
    "[PASS] Remote main == local amendment commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 13. FINAL
# ==============================================================================

print("=" * 112)
print("STAGE24-1C-EX1-R1 RECOVERY: PASS")
print("=" * 112)

print()

print(
    "Accidental second execution repaired: YES"
)

print(
    "Unexpected files committed:           NO"
)

print(
    "GROUNDED_S4 definition changed:       NO"
)

print(
    "Inference rule adopted:               NO"
)

print()

print(
    "GROUNDED_S4 cells cancelled:          2"
)

print(
    "Cancelled cells reassignable:         NO"
)

print(
    "Evaluable frozen target cells:        6"
)

print()

print(
    "Target openings consumed:             0 / 8"
)

print(
    "Target predictions:                   0"
)

print(
    "Scientific fits:                      2 / 4"
)

print()

print(
    "Amendment SHA256:"
)

print(
    " ",
    actual_json_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "NEXT: primary CICIDS2017 PUBLISHED / FLAG_CORRECTED evaluation."
)

print(
    "NO PCAP."
)

print("=" * 112)

STAGE24-1C-EX1-R1 — DOUBLE-RUN RECOVERY

HEAD / DIRTY-STATE RECOVERY GATE
HEAD: 683fe85839e63dcd5fd8e952fd07931b446652ff

Dirty entries: 2
  A  results/stage24_cross_dataset/stage24_1_primary_source_sanity/stage24_1c_grounded_s4_membership/stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendment.json
  A  results/stage24_cross_dataset/stage24_1_primary_source_sanity/stage24_1c_grounded_s4_membership/stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendment.sha256

[PASS] Dirty state contains ONLY the two expected Stage24 amendment files.

[PASS] Both amendment artifacts exist.

CORPUS DIAGNOSTIC VALIDATION
Mandatory GROUNDED recovered: 357
Unique EXCLUDED recovered: 35
Rows still ambiguous: 96
Diagnostic SHA256: 3fe13ac9438f05a64fe09c91947013b05c1e138ed92317f9a3fc5a39c737a812

[PASS] Corpus diagnostic unchanged.

EXISTING AMENDMENT VALIDATION
[PASS] 21 amendment governance checks.

AMENDMENT BYTE IDENTITY
JSON SHA256: 591a917720f31e9894e2f23cb56a1f7d6bfa69c12c21981

RuntimeError: 
Command failed:
git commit -m stage24: freeze GROUNDED_S4 artifact infeasibility amendment

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@a886c1f1ef5d.(none)')


In [26]:
# ==============================================================================
# STAGE24-1C-EX1-R2
# FIX LOCAL GIT IDENTITY -> COMMIT EXISTING STAGED AMENDMENT -> PUSH
#
# NO scientific files are regenerated.
# NO model.
# NO target opening.
# ==============================================================================

from __future__ import annotations

import os
import base64
import subprocess
from pathlib import Path


print("=" * 112)
print("STAGE24-1C-EX1-R2 — GIT IDENTITY RECOVERY + COMMIT")
print("=" * 112)
print()


REPO = Path(
    globals().get(
        "REPO",
        "/kaggle/working/ids2018-validation-safe-ablation",
    )
)

EXPECTED_PARENT = (
    "683fe85839e63dcd5fd8e952fd07931b446652ff"
)

EXPECTED_FILES = {
    (
        "results/stage24_cross_dataset/"
        "stage24_1_primary_source_sanity/"
        "stage24_1c_grounded_s4_membership/"
        "stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendment.json"
    ),
    (
        "results/stage24_cross_dataset/"
        "stage24_1_primary_source_sanity/"
        "stage24_1c_grounded_s4_membership/"
        "stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendment.sha256"
    ),
}


def run(args, *, check=True):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=REPO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(str(x) for x in args)
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git(*args, auth_header=None, check=True):

    cmd = ["git"]

    if auth_header is not None:
        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [str(x) for x in args]

    return run(
        cmd,
        check=check,
    )


# ==============================================================================
# 1. HEAD / STAGED-STATE GATE
# ==============================================================================

head = git(
    "rev-parse",
    "HEAD",
)

print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


staged = {
    line
    for line in git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


print()
print(
    "Staged files:"
)

for path in sorted(staged):
    print(
        " ",
        path,
    )


if staged != EXPECTED_FILES:
    raise RuntimeError(
        "\nUnexpected staged state.\n"
        f"Expected: {sorted(EXPECTED_FILES)}\n"
        f"Actual:   {sorted(staged)}"
    )


print()
print(
    "[PASS] Exactly the two validated amendment files remain staged."
)

print()


# ==============================================================================
# 2. RECOVER AUTHOR IDENTITY FROM EXISTING REPOSITORY HISTORY
# ==============================================================================

print("=" * 112)
print("REPOSITORY-LOCAL GIT IDENTITY")
print("=" * 112)


author_name = git(
    "log",
    "-1",
    "--format=%an",
).strip()


author_email = git(
    "log",
    "-1",
    "--format=%ae",
).strip()


if not author_name:
    raise RuntimeError(
        "Could not recover author name from repository history."
    )


if not author_email:
    raise RuntimeError(
        "Could not recover author email from repository history."
    )


print(
    "Recovered name: ",
    author_name,
)

print(
    "Recovered email:",
    author_email,
)


# Repository-local only — NOT --global.
git(
    "config",
    "--local",
    "user.name",
    author_name,
)

git(
    "config",
    "--local",
    "user.email",
    author_email,
)


configured_name = git(
    "config",
    "--local",
    "--get",
    "user.name",
)

configured_email = git(
    "config",
    "--local",
    "--get",
    "user.email",
)


if configured_name != author_name:
    raise RuntimeError(
        "Local Git user.name configuration failed."
    )


if configured_email != author_email:
    raise RuntimeError(
        "Local Git user.email configuration failed."
    )


print()
print(
    "[PASS] Repository-local Git identity configured."
)

print()


# ==============================================================================
# 3. COMMIT EXISTING STAGED FILES
# ==============================================================================

print("=" * 112)
print("COMMIT")
print("=" * 112)


commit_output = git(
    "commit",
    "-m",
    "stage24: freeze GROUNDED_S4 artifact infeasibility amendment",
)


print(
    commit_output
)

print()


commit = git(
    "rev-parse",
    "HEAD",
)


parent = git(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:
    raise RuntimeError(
        "\nNew commit parent mismatch.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {parent}"
    )


print(
    "Commit:",
    commit,
)

print(
    "[PASS] Amendment committed."
)

print()


# ==============================================================================
# 4. GITHUB TOKEN
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if value and value.strip():

            github_token = value.strip()
            token_source = f"ENV:{name}"
            break


if github_token is None:
    raise RuntimeError(
        "GitHub token unavailable."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


print(
    "GitHub credential:",
    token_source,
)

print()


# ==============================================================================
# 5. PUSH
# ==============================================================================

print("=" * 112)
print("PUSH")
print("=" * 112)


push_output = git(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


# ==============================================================================
# 6. REMOTE VERIFICATION
# ==============================================================================

remote_output = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
)


if not remote_output:
    raise RuntimeError(
        "Could not read remote main."
    )


remote_head = remote_output.split()[0]


print(
    "Local HEAD: ",
    commit,
)

print(
    "Remote main:",
    remote_head,
)


if remote_head != commit:
    raise RuntimeError(
        "\nRemote verification failed."
    )


status = git(
    "status",
    "--porcelain",
)


if status:
    raise RuntimeError(
        "\nRepository is not clean after push:\n"
        + status
    )


print()
print(
    "[PASS] Remote main == local HEAD."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 7. FINAL
# ==============================================================================

print("=" * 112)
print("STAGE24-1C-EX1-R2: PASS")
print("=" * 112)

print()

print(
    "Git identity problem repaired: YES"
)

print(
    "Amendment regenerated:          NO"
)

print(
    "Scientific definition changed: NO"
)

print(
    "Target openings consumed:       0 / 8"
)

print(
    "Scientific fits:                2 / 4"
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "NEXT: Stage24 primary CICIDS2017 PUBLISHED / FLAG_CORRECTED evaluation."
)

print(
    "NO PCAP."
)

print("=" * 112)

STAGE24-1C-EX1-R2 — GIT IDENTITY RECOVERY + COMMIT

HEAD: 683fe85839e63dcd5fd8e952fd07931b446652ff

Staged files:
  results/stage24_cross_dataset/stage24_1_primary_source_sanity/stage24_1c_grounded_s4_membership/stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendment.json
  results/stage24_cross_dataset/stage24_1_primary_source_sanity/stage24_1c_grounded_s4_membership/stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendment.sha256

[PASS] Exactly the two validated amendment files remain staged.

REPOSITORY-LOCAL GIT IDENTITY
Recovered name:  themubasshir
Recovered email: 10107331+themubasshir@users.noreply.github.com

[PASS] Repository-local Git identity configured.

COMMIT
[main a90a422] stage24: freeze GROUNDED_S4 artifact infeasibility amendment
 2 files changed, 128 insertions(+)
 create mode 100644 results/stage24_cross_dataset/stage24_1_primary_source_sanity/stage24_1c_grounded_s4_membership/stage24_1c_ex1_grounded_s4_durable_artifact_infeasibility_amendmen

In [27]:
# ==============================================================================
# STAGE24-1D0 — CICIDS2017 PUBLISHED TARGET SOURCE ACQUISITION
#
# PURPOSE
# -------
# Acquire the 8 frozen CICIDS2017 traffic-label / feature Parquets from the
# pinned Hugging Face revision.
#
# This is SOURCE ACQUISITION ONLY.
#
# DOES NOT:
#   - download PCAPs
#   - use Stage20 compact corpora
#   - read target feature columns into model memory
#   - run LightGBM/XGBoost
#   - produce predictions
#   - compute target metrics
#   - consume a target opening
#   - modify Git
#
# Parquet metadata is inspected only for:
#   - physical row count
#   - schema column names
#
# Expected complete CICIDS2017 effective target:
#   2,830,743 rows
# ==============================================================================

from __future__ import annotations

import os
import sys
import time
import hashlib
import subprocess
from pathlib import Path


print("=" * 112)
print("STAGE24-1D0 — CICIDS2017 PUBLISHED TARGET SOURCE ACQUISITION")
print("=" * 112)
print()


# ==============================================================================
# 0. FROZEN GOVERNANCE
# ==============================================================================

EXPECTED_HEAD = (
    "a90a4221ef5a5e9958777049bcf13cc56a3519d3"
)

REPO = Path(
    globals().get(
        "REPO",
        "/kaggle/working/ids2018-validation-safe-ablation",
    )
)


def run_cmd(args, *, cwd=None, check=True):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


head = run_cmd(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=REPO,
)


status = run_cmd(
    [
        "git",
        "status",
        "--porcelain",
    ],
    cwd=REPO,
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "\nUnexpected repository HEAD.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


print(
    "[PASS] Repository clean at frozen Stage24 amendment."
)

print()


# ==============================================================================
# 1. HUGGING FACE DEPENDENCY
# ==============================================================================

try:

    import huggingface_hub

    from huggingface_hub import (
        HfApi,
        hf_hub_download,
    )

except Exception:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "huggingface_hub",
        ],
        check=True,
    )

    import huggingface_hub

    from huggingface_hub import (
        HfApi,
        hf_hub_download,
    )


try:

    import pyarrow.parquet as pq

except Exception:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "pyarrow",
        ],
        check=True,
    )

    import pyarrow.parquet as pq


print(
    "huggingface_hub:",
    huggingface_hub.__version__,
)

print()


# ==============================================================================
# 2. FROZEN REMOTE
# ==============================================================================

HF_REPO = "bvsam/cic-ids-2017"

HF_REVISION = (
    "b7e532345512edcd530cb1770dc76636aeb52802"
)

HF_REPO_TYPE = "dataset"


CACHE_DIR = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Frozen SHA256 identities from Stage20 provenance.
EXPECTED_SHA256 = {
    # Monday
    "dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02",

    # Tuesday
    "27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4",

    # Wednesday
    "d23a259820b16e1ad54f9f3b58d5727c5032d383015f90bc7c07cebbdf8a7140",

    # Thursday morning
    "d8110c04a7af91124ada1c5ad901c4210879df1af8882dc637767532e7165350",

    # Thursday afternoon
    "5da010354f0fc1040fd1fe65967096e1063475de8dd30ae4f657c07201d728a7",

    # Friday morning
    "2c00236b13a69f4b1c222b8f4a89451dc2148cb04a8cd0c45c2a87af51471774",

    # Friday DDos
    "7c5876d52189fc01af54bad6cf23afe9f7fbc0e3ca6c3595920754f0c3ba8f66",

    # Friday PortScan
    "4d78cee297c27f1a9947b9384793e587a46c7a3ea89db199553dabddc9835d4a",
}


EXPECTED_FILE_COUNT = 8

EXPECTED_TOTAL_ROWS = 2_830_743


print(
    "HF repository:",
    HF_REPO,
)

print(
    "Revision:",
    HF_REVISION,
)

print()


# ==============================================================================
# 3. LIST PINNED TRAFFIC-LABEL PARQUETS
# ==============================================================================

print("=" * 112)
print("PINNED REMOTE INVENTORY")
print("=" * 112)


api = HfApi()


all_files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type=HF_REPO_TYPE,
    revision=HF_REVISION,
)


traffic_parquets = sorted(
    path
    for path in all_files
    if (
        path.startswith(
            "traffic_labels/"
        )
        and
        path.lower().endswith(
            ".parquet"
        )
    )
)


print(
    "traffic_labels/*.parquet:",
    len(
        traffic_parquets
    ),
)

print()


for i, remote in enumerate(
    traffic_parquets,
    start=1,
):

    print(
        f"[{i:02d}] {remote}"
    )


if len(
    traffic_parquets
) != EXPECTED_FILE_COUNT:

    raise RuntimeError(
        "\nUnexpected frozen traffic-label Parquet count.\n"
        f"Expected: {EXPECTED_FILE_COUNT}\n"
        f"Actual:   {len(traffic_parquets)}"
    )


print()
print(
    "[PASS] Exactly eight frozen target Parquets discovered."
)

print()


# ==============================================================================
# 4. SHA HELPER
# ==============================================================================

def sha256_file(
    path,
    *,
    chunk_size=32 * 1024 * 1024,
):

    h = hashlib.sha256()

    total = 0


    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            chunk = fh.read(
                chunk_size
            )

            if not chunk:
                break


            h.update(
                chunk
            )

            total += len(
                chunk
            )


    return (
        h.hexdigest(),
        total,
    )


# ==============================================================================
# 5. DOWNLOAD + BYTE VERIFY + METADATA ONLY
# ==============================================================================

print("=" * 112)
print("ACQUIRE + VERIFY")
print("=" * 112)


records = []

actual_sha_set = set()

total_rows = 0
total_bytes = 0


for index, remote in enumerate(
    traffic_parquets,
    start=1,
):

    print()
    print(
        "-" * 112
    )

    print(
        f"[{index:02d}/{EXPECTED_FILE_COUNT:02d}] {remote}"
    )


    t0 = time.time()


    local_path = Path(
        hf_hub_download(
            repo_id=HF_REPO,
            filename=remote,
            repo_type=HF_REPO_TYPE,
            revision=HF_REVISION,
            cache_dir=str(
                CACHE_DIR
            ),
        )
    )


    if not local_path.is_file():

        raise RuntimeError(
            f"Downloaded file missing:\n{local_path}"
        )


    actual_sha, actual_bytes = (
        sha256_file(
            local_path
        )
    )


    print(
        "  local:",
        local_path,
    )

    print(
        "  bytes:",
        f"{actual_bytes:,}",
    )

    print(
        "  SHA256:",
        actual_sha,
    )


    if actual_sha not in EXPECTED_SHA256:

        raise RuntimeError(
            "\nUnknown traffic-label Parquet SHA256.\n"
            f"Remote: {remote}\n"
            f"SHA:    {actual_sha}"
        )


    if actual_sha in actual_sha_set:

        raise RuntimeError(
            "\nDuplicate frozen SHA256 encountered:\n"
            f"{actual_sha}"
        )


    actual_sha_set.add(
        actual_sha
    )


    # --------------------------------------------------------------------------
    # PARQUET METADATA ONLY.
    # No feature column values are loaded.
    # --------------------------------------------------------------------------

    parquet = pq.ParquetFile(
        local_path
    )


    rows = int(
        parquet.metadata.num_rows
    )


    columns = list(
        parquet.schema_arrow.names
    )


    print(
        "  rows:",
        f"{rows:,}",
    )

    print(
        "  columns:",
        len(
            columns
        ),
    )

    print(
        "  elapsed:",
        f"{time.time() - t0:.2f}s",
    )

    print(
        "  [PASS]"
    )


    records.append(
        {
            "remote":
                remote,

            "local":
                str(
                    local_path
                ),

            "sha256":
                actual_sha,

            "bytes":
                actual_bytes,

            "rows":
                rows,

            "columns":
                columns,
        }
    )


    total_rows += rows
    total_bytes += actual_bytes


# ==============================================================================
# 6. COMPLETE SET IDENTITY
# ==============================================================================

print()
print("=" * 112)
print("COMPLETE FROZEN TARGET SOURCE AUDIT")
print("=" * 112)


missing_sha = (
    EXPECTED_SHA256
    -
    actual_sha_set
)

unexpected_sha = (
    actual_sha_set
    -
    EXPECTED_SHA256
)


print(
    "Expected SHA identities:",
    len(
        EXPECTED_SHA256
    ),
)

print(
    "Observed SHA identities:",
    len(
        actual_sha_set
    ),
)

print(
    "Missing SHA identities:",
    len(
        missing_sha
    ),
)

print(
    "Unexpected SHA identities:",
    len(
        unexpected_sha
    ),
)


if missing_sha:

    for value in sorted(
        missing_sha
    ):

        print(
            "  MISSING:",
            value,
        )


if unexpected_sha:

    for value in sorted(
        unexpected_sha
    ):

        print(
            "  UNEXPECTED:",
            value,
        )


if (
    missing_sha
    or
    unexpected_sha
):

    raise RuntimeError(
        "Frozen target source SHA set mismatch."
    )


print()
print(
    "Total rows:",
    f"{total_rows:,}",
)

print(
    "Expected:  ",
    f"{EXPECTED_TOTAL_ROWS:,}",
)


if total_rows != EXPECTED_TOTAL_ROWS:

    raise RuntimeError(
        "\nComplete CICIDS2017 row count mismatch.\n"
        f"Expected: {EXPECTED_TOTAL_ROWS:,}\n"
        f"Actual:   {total_rows:,}"
    )


print(
    "Total bytes:",
    f"{total_bytes:,}",
)

print()
print(
    "[PASS] All eight CICIDS2017 target Parquets are frozen-byte exact."
)

print(
    "[PASS] Complete published target population = 2,830,743 rows."
)

print()


# ==============================================================================
# 7. SCHEMA CONSISTENCY — METADATA ONLY
# ==============================================================================

print("=" * 112)
print("SCHEMA METADATA CONSISTENCY")
print("=" * 112)


schema_sets = [
    tuple(
        record[
            "columns"
        ]
    )
    for record in records
]


base_schema = schema_sets[
    0
]


schema_mismatches = []


for i, schema in enumerate(
    schema_sets
):

    if schema != base_schema:

        schema_mismatches.append(
            records[
                i
            ][
                "remote"
            ]
        )


print(
    "Columns in reference schema:",
    len(
        base_schema
    ),
)

print(
    "Schema-order mismatches:",
    len(
        schema_mismatches
    ),
)


if schema_mismatches:

    print()
    print(
        "Files with schema-order differences:"
    )

    for item in schema_mismatches:
        print(
            " ",
            item,
        )


# Do not fail here solely because of physical column-order drift;
# Stage24 maps semantic feature names explicitly.
print()

print(
    "[PASS] Frozen source acquisition complete."
)

print()


# ==============================================================================
# 8. COMPACT SUMMARY
# ==============================================================================

print("=" * 112)
print("FROZEN CICIDS2017 TARGET FILES")
print("=" * 112)


for record in records:

    print()

    print(
        record[
            "remote"
        ]
    )

    print(
        "  rows:  ",
        f"{record['rows']:,}",
    )

    print(
        "  bytes: ",
        f"{record['bytes']:,}",
    )

    print(
        "  sha256:",
        record[
            "sha256"
        ],
    )


# Save live paths for the NEXT cell only.
STAGE24_CICIDS2017_TARGET_RECORDS = records


print()
print("=" * 112)
print("STAGE24-1D0 SOURCE ACQUISITION: PASS")
print("=" * 112)

print()

print(
    "PCAP downloaded:                   NO"
)

print(
    "Stage20 corpus used:               NO"
)

print(
    "Target feature values loaded:      NO"
)

print(
    "Target features supplied to model: NO"
)

print(
    "Model executed:                    NO"
)

print(
    "Predictions created:               0"
)

print(
    "Target metrics computed:           0"
)

print(
    "Target openings consumed:          0 / 8"
)

print(
    "Scientific fits completed:         2 / 4"
)

print(
    "Git changes:                       NONE"
)

print()

print(
    "NEXT: freeze semantic bridge projection over these 8 Parquets,"
)

print(
    "      then open the first primary target cell."
)

print("=" * 112)

STAGE24-1D0 — CICIDS2017 PUBLISHED TARGET SOURCE ACQUISITION

HEAD: a90a4221ef5a5e9958777049bcf13cc56a3519d3
[PASS] Repository clean at frozen Stage24 amendment.

huggingface_hub: 1.11.0

HF repository: bvsam/cic-ids-2017
Revision: b7e532345512edcd530cb1770dc76636aeb52802

PINNED REMOTE INVENTORY
traffic_labels/*.parquet: 8

[01] traffic_labels/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.parquet
[02] traffic_labels/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
[03] traffic_labels/Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet
[04] traffic_labels/Monday-WorkingHours.pcap_ISCX.csv.parquet
[05] traffic_labels/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet
[06] traffic_labels/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet
[07] traffic_labels/Tuesday-WorkingHours.pcap_ISCX.csv.parquet
[08] traffic_labels/Wednesday-workingHours.pcap_ISCX.csv.parquet

[PASS] Exactly eight frozen target Parquets discovered.

ACQUIRE + VERIFY

--------

traffic_labels/Friday-WorkingHours-After(…):   0%|          | 0.00/23.0M [00:00<?, ?B/s]

  local: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.parquet
  bytes: 23,048,086
  SHA256: 7c5876d52189fc01af54bad6cf23afe9f7fbc0e3ca6c3595920754f0c3ba8f66
  rows: 225,745
  columns: 85
  elapsed: 3.51s
  [PASS]

----------------------------------------------------------------------------------------------------------------
[02/08] traffic_labels/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet


traffic_labels/Friday-WorkingHours-After(…):   0%|          | 0.00/18.6M [00:00<?, ?B/s]

  local: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
  bytes: 18,632,427
  SHA256: 4d78cee297c27f1a9947b9384793e587a46c7a3ea89db199553dabddc9835d4a
  rows: 286,467
  columns: 85
  elapsed: 4.80s
  [PASS]

----------------------------------------------------------------------------------------------------------------
[03/08] traffic_labels/Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet


...

traffic_labels/Tuesday-WorkingHours.pcap(…):   0%|          | 0.00/52.7M [00:00<?, ?B/s]

  local: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Tuesday-WorkingHours.pcap_ISCX.csv.parquet
  bytes: 52,701,751
  SHA256: 27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4
  rows: 445,909
  columns: 85
  elapsed: 4.23s
  [PASS]

----------------------------------------------------------------------------------------------------------------
[08/08] traffic_labels/Wednesday-workingHours.pcap_ISCX.csv.parquet


traffic_labels/Wednesday-workingHours.pc(…):   0%|          | 0.00/76.5M [00:00<?, ?B/s]

  local: /kaggle/working/stage24_cicids2017_hf_cache/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Wednesday-workingHours.pcap_ISCX.csv.parquet
  bytes: 76,512,727
  SHA256: d23a259820b16e1ad54f9f3b58d5727c5032d383015f90bc7c07cebbdf8a7140
  rows: 692,703
  columns: 85
  elapsed: 4.29s
  [PASS]

COMPLETE FROZEN TARGET SOURCE AUDIT
Expected SHA identities: 8
Observed SHA identities: 8
Missing SHA identities: 0
Unexpected SHA identities: 0

Total rows: 3,119,345
Expected:   2,830,743


RuntimeError: 
Complete CICIDS2017 row count mismatch.
Expected: 2,830,743
Actual:   3,119,345

In [28]:
# ==============================================================================
# STAGE24-1D0-R1 — VERIFY THURSDAY DUPLICATION ARTIFACT
#
# NO model
# NO predictions
# NO target opening
# NO Git changes
#
# Hypothesis:
#
#   Thursday Morning rows = 458,968
#   Thursday Afternoon rows = 288,602
#
#   458,968 - 288,602 = 170,366
#
# Test whether:
#
#   Morning[170366:458968] == Afternoon[0:288602]
#
# across ALL 85 columns, exactly and in order.
# ==============================================================================

from pathlib import Path
import hashlib
import pyarrow.parquet as pq
import pyarrow as pa


print("=" * 112)
print("STAGE24-1D0-R1 — THURSDAY DUPLICATION VERIFICATION")
print("=" * 112)
print()


ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/"
    "snapshots/b7e532345512edcd530cb1770dc76636aeb52802/"
    "traffic_labels"
)


MORNING = (
    ROOT
    / "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet"
)

AFTERNOON = (
    ROOT
    / "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet"
)


EXPECTED_MORNING_ROWS = 458_968
EXPECTED_AFTERNOON_ROWS = 288_602
EXPECTED_TRUE_MORNING_ROWS = 170_366

EXPECTED_RAW_TOTAL = 3_119_345
EXPECTED_EFFECTIVE_TOTAL = 2_830_743


for path in [MORNING, AFTERNOON]:

    if not path.is_file():
        raise RuntimeError(
            f"Missing file:\n{path}"
        )


# ==============================================================================
# 1. METADATA
# ==============================================================================

m_meta = pq.ParquetFile(
    MORNING
)

a_meta = pq.ParquetFile(
    AFTERNOON
)


m_rows = int(
    m_meta.metadata.num_rows
)

a_rows = int(
    a_meta.metadata.num_rows
)


print(
    "Thursday Morning rows:  ",
    f"{m_rows:,}",
)

print(
    "Thursday Afternoon rows:",
    f"{a_rows:,}",
)

print(
    "Difference:             ",
    f"{m_rows - a_rows:,}",
)

print()


if m_rows != EXPECTED_MORNING_ROWS:
    raise RuntimeError(
        f"Unexpected morning rows: {m_rows:,}"
    )


if a_rows != EXPECTED_AFTERNOON_ROWS:
    raise RuntimeError(
        f"Unexpected afternoon rows: {a_rows:,}"
    )


if (
    m_rows
    -
    a_rows
) != EXPECTED_TRUE_MORNING_ROWS:

    raise RuntimeError(
        "Thursday arithmetic does not match expected canonical split."
    )


print(
    "[PASS] Row-count geometry:"
)

print(
    "       458,968 = 170,366 + 288,602"
)

print()


# ==============================================================================
# 2. SCHEMA IDENTITY
# ==============================================================================

m_schema = m_meta.schema_arrow
a_schema = a_meta.schema_arrow


print(
    "Morning columns:  ",
    len(
        m_schema.names
    ),
)

print(
    "Afternoon columns:",
    len(
        a_schema.names
    ),
)


if m_schema != a_schema:

    raise RuntimeError(
        "Thursday morning/afternoon schemas differ."
    )


print(
    "[PASS] Exact Arrow schema identity."
)

print()


# ==============================================================================
# 3. READ ONLY THE REQUIRED BLOCKS
#
# These values are NOT supplied to any model.
# ==============================================================================

print("=" * 112)
print("ROW-FOR-ROW DUPLICATION TEST")
print("=" * 112)


morning_table = pq.read_table(
    MORNING
)

afternoon_table = pq.read_table(
    AFTERNOON
)


if morning_table.num_rows != EXPECTED_MORNING_ROWS:
    raise RuntimeError(
        "Morning table row count changed."
    )


if afternoon_table.num_rows != EXPECTED_AFTERNOON_ROWS:
    raise RuntimeError(
        "Afternoon table row count changed."
    )


morning_prefix = morning_table.slice(
    0,
    EXPECTED_TRUE_MORNING_ROWS,
)


morning_tail = morning_table.slice(
    EXPECTED_TRUE_MORNING_ROWS,
    EXPECTED_AFTERNOON_ROWS,
)


print(
    "Morning prefix:",
    f"{morning_prefix.num_rows:,}",
)

print(
    "Morning tail:  ",
    f"{morning_tail.num_rows:,}",
)

print(
    "Afternoon:     ",
    f"{afternoon_table.num_rows:,}",
)

print()


tail_equals_afternoon = (
    morning_tail.equals(
        afternoon_table
    )
)


print(
    "Morning tail == Afternoon:",
    tail_equals_afternoon,
)


# Also ensure afternoon is not simply the prefix.
prefix_probe_equal = (
    morning_table
    .slice(
        0,
        EXPECTED_AFTERNOON_ROWS,
    )
    .equals(
        afternoon_table
    )
)


print(
    "Morning first 288,602 == Afternoon:",
    prefix_probe_equal,
)

print()


# ==============================================================================
# 4. COLUMN-BY-COLUMN EXACT AUDIT
# ==============================================================================

column_results = []


for name in morning_table.schema.names:

    left = morning_tail[
        name
    ].combine_chunks()

    right = afternoon_table[
        name
    ].combine_chunks()


    equal = left.equals(
        right
    )


    column_results.append(
        (
            name,
            equal,
        )
    )


mismatch_columns = [
    name
    for name, equal
    in column_results
    if not equal
]


print(
    "Exact matching columns:",
    f"{len(column_results) - len(mismatch_columns)}"
    f" / {len(column_results)}",
)


if mismatch_columns:

    print()
    print(
        "Mismatch columns:"
    )

    for name in mismatch_columns:
        print(
            " ",
            name,
        )


# ==============================================================================
# 5. DECISION
# ==============================================================================

print()
print("=" * 112)
print("THURSDAY SOURCE DECISION")
print("=" * 112)


if (
    tail_equals_afternoon
    and
    not mismatch_columns
):

    print()
    print(
        "[PASS] CONFIRMED EXACT DUPLICATION."
    )

    print()
    print(
        "Thursday-Morning-WebAttacks contains:"
    )

    print(
        "  rows 0..170,365       = true Thursday morning"
    )

    print(
        "  rows 170,366..458,967 = exact duplicate of Thursday afternoon"
    )

    print()
    print(
        "Therefore raw 8-file concatenation double-counts 288,602 rows."
    )

    print()

    effective_total = (
        EXPECTED_RAW_TOTAL
        -
        EXPECTED_AFTERNOON_ROWS
    )


    print(
        "Raw eight-file total:      ",
        f"{EXPECTED_RAW_TOTAL:,}",
    )

    print(
        "Duplicated afternoon rows: ",
        f"{EXPECTED_AFTERNOON_ROWS:,}",
    )

    print(
        "Effective target total:    ",
        f"{effective_total:,}",
    )


    if effective_total != EXPECTED_EFFECTIVE_TOTAL:

        raise RuntimeError(
            "Effective total arithmetic failed."
        )


    print()
    print(
        "[PASS] Frozen effective CICIDS2017 population = 2,830,743."
    )


else:

    print()
    print(
        "[FAIL] Thursday duplication hypothesis not exact."
    )

    print(
        "Do NOT proceed to target evaluation."
    )

    raise RuntimeError(
        "Thursday duplication requires further investigation."
    )


# ==============================================================================
# 6. FINAL
# ==============================================================================

print()
print("=" * 112)
print("STAGE24-1D0-R1: PASS")
print("=" * 112)

print()

print(
    "PCAP accessed:                    NO"
)

print(
    "Model executed:                   NO"
)

print(
    "Target features supplied to model:NO"
)

print(
    "Predictions created:              0"
)

print(
    "Target metrics computed:          0"
)

print(
    "Target openings consumed:         0 / 8"
)

print(
    "Scientific fits:                  2 / 4"
)

print(
    "Git changes:                      NONE"
)

print()

print(
    "NEXT: freeze the 2,830,743-row effective CICIDS2017 source population,"
)

print(
    "      then execute the first primary target opening."
)

print("=" * 112)

STAGE24-1D0-R1 — THURSDAY DUPLICATION VERIFICATION

Thursday Morning rows:   458,968
Thursday Afternoon rows: 288,602
Difference:              170,366

[PASS] Row-count geometry:
       458,968 = 170,366 + 288,602

Morning columns:   85
Afternoon columns: 85
[PASS] Exact Arrow schema identity.

ROW-FOR-ROW DUPLICATION TEST
Morning prefix: 170,366
Morning tail:   288,602
Afternoon:      288,602

Morning tail == Afternoon: False
Morning first 288,602 == Afternoon: False

Exact matching columns: 0 / 85

Mismatch columns:
  Flow ID
  Source IP
  Source Port
  Destination IP
  Destination Port
  Protocol
  Timestamp
  Flow Duration
  Total Fwd Packets
  Total Backward Packets
  Total Length of Fwd Packets
  Total Length of Bwd Packets
  Fwd Packet Length Max
  Fwd Packet Length Min
  Fwd Packet Length Mean
  Fwd Packet Length Std
  Bwd Packet Length Max
  Bwd Packet Length Min
  Bwd Packet Length Mean
  Bwd Packet Length Std
  Flow Bytes/s
  Flow Packets/s
  Flow IAT Mean
  Flow IAT Std
  F

RuntimeError: Thursday duplication requires further investigation.

In [29]:
# ==============================================================================
# STAGE24-1D0-R2 — VERIFY FROZEN THURSDAY STRUCTURAL NULL PADDING
#
# Frozen contract:
#
# Thursday Morning:
#   physical rows                = 458,968
#   effective label records      = 170,366
#   structural null-padding rows = 288,602
#
# Thursday Afternoon:
#   effective rows               = 288,602
#   structural padding           = 0
#
# Complete effective CICIDS2017:
#   2,830,743 rows
#
# NO PCAP
# NO MODEL
# NO PREDICTIONS
# NO TARGET OPENING
# NO GIT CHANGE
# ==============================================================================

from pathlib import Path
import duckdb
import pyarrow.parquet as pq


print("=" * 112)
print("STAGE24-1D0-R2 — FROZEN STRUCTURAL NULL-PADDING VERIFICATION")
print("=" * 112)
print()


ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/"
    "snapshots/b7e532345512edcd530cb1770dc76636aeb52802/"
    "traffic_labels"
)


FILES = [
    (
        "Monday",
        "Monday-WorkingHours.pcap_ISCX.csv.parquet",
        529_918,
        529_918,
    ),
    (
        "Tuesday",
        "Tuesday-WorkingHours.pcap_ISCX.csv.parquet",
        445_909,
        445_909,
    ),
    (
        "Wednesday",
        "Wednesday-workingHours.pcap_ISCX.csv.parquet",
        692_703,
        692_703,
    ),
    (
        "Thursday-Afternoon",
        "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet",
        288_602,
        288_602,
    ),
    (
        "Thursday-Morning",
        "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet",
        458_968,
        170_366,
    ),
    (
        "Friday-DDos",
        "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.parquet",
        225_745,
        225_745,
    ),
    (
        "Friday-PortScan",
        "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet",
        286_467,
        286_467,
    ),
    (
        "Friday-Morning",
        "Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet",
        191_033,
        191_033,
    ),
]


EXPECTED_PHYSICAL_TOTAL = 3_119_345
EXPECTED_EFFECTIVE_TOTAL = 2_830_743

EXPECTED_THU_MORNING_PHYSICAL = 458_968
EXPECTED_THU_MORNING_EFFECTIVE = 170_366
EXPECTED_THU_MORNING_PADDING = 288_602


# ==============================================================================
# 1. FILE EXISTENCE / FOOTER ROWS
# ==============================================================================

print("=" * 112)
print("PHYSICAL SOURCE CONTRACT")
print("=" * 112)


physical_total = 0


for day, filename, expected_physical, expected_effective in FILES:

    path = ROOT / filename

    if not path.is_file():
        raise RuntimeError(
            f"Missing frozen target file:\n{path}"
        )


    physical = int(
        pq.ParquetFile(
            path
        ).metadata.num_rows
    )


    print(
        f"{day:22s}"
        f" physical={physical:9,d}"
        f" expected={expected_physical:9,d}"
    )


    if physical != expected_physical:
        raise RuntimeError(
            f"{day}: physical-row mismatch."
        )


    physical_total += physical


print()
print(
    "Physical total:",
    f"{physical_total:,}",
)


if physical_total != EXPECTED_PHYSICAL_TOTAL:
    raise RuntimeError(
        "\nPhysical target total mismatch.\n"
        f"Expected: {EXPECTED_PHYSICAL_TOTAL:,}\n"
        f"Actual:   {physical_total:,}"
    )


print(
    "[PASS] Complete frozen physical source = 3,119,345 rows."
)

print()


# ==============================================================================
# 2. LABEL-NULL AUDIT FOR ALL EIGHT FILES
#
# Frozen effective record criterion:
#     Label IS NOT NULL
#
# Thursday-morning padding should be the ONLY structural label-null population.
# ==============================================================================

print("=" * 112)
print("LABEL-NULL / EFFECTIVE-ROW AUDIT")
print("=" * 112)


conn = duckdb.connect(
    database=":memory:"
)


effective_total = 0
null_total = 0

results = []


for day, filename, expected_physical, expected_effective in FILES:

    path = ROOT / filename

    sql_path = str(
        path
    ).replace(
        "'",
        "''",
    )


    physical, label_null, label_nonnull = conn.execute(
        f"""
        SELECT
            COUNT(*) AS physical,
            COUNT(*) FILTER (
                WHERE "Label" IS NULL
            ) AS label_null,
            COUNT(*) FILTER (
                WHERE "Label" IS NOT NULL
            ) AS label_nonnull
        FROM read_parquet('{sql_path}')
        """
    ).fetchone()


    physical = int(
        physical
    )

    label_null = int(
        label_null
    )

    label_nonnull = int(
        label_nonnull
    )


    print(
        f"{day:22s}"
        f" physical={physical:9,d}"
        f" null={label_null:9,d}"
        f" effective={label_nonnull:9,d}"
    )


    if label_nonnull != expected_effective:
        raise RuntimeError(
            "\nEffective-row count mismatch.\n"
            f"Day:      {day}\n"
            f"Expected: {expected_effective:,}\n"
            f"Actual:   {label_nonnull:,}"
        )


    if (
        physical
        !=
        label_null
        +
        label_nonnull
    ):
        raise RuntimeError(
            f"{day}: row accounting failure."
        )


    effective_total += label_nonnull
    null_total += label_null


    results.append(
        {
            "day":
                day,

            "filename":
                filename,

            "path":
                str(
                    path
                ),

            "physical_rows":
                physical,

            "structural_null_rows":
                label_null,

            "effective_rows":
                label_nonnull,
        }
    )


print()
print(
    "Total physical rows:",
    f"{physical_total:,}",
)

print(
    "Total Label-NULL rows:",
    f"{null_total:,}",
)

print(
    "Total effective rows:",
    f"{effective_total:,}",
)

print()


if null_total != EXPECTED_THU_MORNING_PADDING:
    raise RuntimeError(
        "\nUnexpected total structural-null population.\n"
        f"Expected: {EXPECTED_THU_MORNING_PADDING:,}\n"
        f"Actual:   {null_total:,}"
    )


if effective_total != EXPECTED_EFFECTIVE_TOTAL:
    raise RuntimeError(
        "\nEffective CICIDS2017 population mismatch.\n"
        f"Expected: {EXPECTED_EFFECTIVE_TOTAL:,}\n"
        f"Actual:   {effective_total:,}"
    )


print(
    "[PASS] Exactly 288,602 Label-NULL structural rows."
)

print(
    "[PASS] Effective target population = 2,830,743."
)

print()


# ==============================================================================
# 3. PROVE THURSDAY-MORNING LABEL-NULL ROWS ARE COMPLETE NULL-PADDING ROWS
# ==============================================================================

print("=" * 112)
print("THURSDAY-MORNING STRUCTURAL NULL PROOF")
print("=" * 112)


THU_MORNING = (
    ROOT
    / "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet"
)


schema = pq.ParquetFile(
    THU_MORNING
).schema_arrow


columns = list(
    schema.names
)


def q(name):

    return (
        '"'
        +
        str(
            name
        ).replace(
            '"',
            '""',
        )
        +
        '"'
    )


all_null_predicate = (
    " AND ".join(
        f"{q(name)} IS NULL"
        for name in columns
    )
)


any_nonnull_predicate = (
    " OR ".join(
        f"{q(name)} IS NOT NULL"
        for name in columns
    )
)


thu_path_sql = str(
    THU_MORNING
).replace(
    "'",
    "''",
)


(
    physical,
    label_null,
    all_columns_null,
    effective,
) = conn.execute(
    f"""
    SELECT
        COUNT(*)                                               AS physical,

        COUNT(*) FILTER (
            WHERE "Label" IS NULL
        )                                                      AS label_null,

        COUNT(*) FILTER (
            WHERE {all_null_predicate}
        )                                                      AS all_columns_null,

        COUNT(*) FILTER (
            WHERE "Label" IS NOT NULL
        )                                                      AS effective

    FROM read_parquet('{thu_path_sql}')
    """
).fetchone()


physical = int(
    physical
)

label_null = int(
    label_null
)

all_columns_null = int(
    all_columns_null
)

effective = int(
    effective
)


print(
    "Physical rows:",
    f"{physical:,}",
)

print(
    "Label-NULL rows:",
    f"{label_null:,}",
)

print(
    "All-85-columns-NULL rows:",
    f"{all_columns_null:,}",
)

print(
    "Effective Label-nonnull rows:",
    f"{effective:,}",
)

print()


if physical != EXPECTED_THU_MORNING_PHYSICAL:
    raise RuntimeError(
        "Thursday-morning physical count mismatch."
    )


if label_null != EXPECTED_THU_MORNING_PADDING:
    raise RuntimeError(
        "Thursday-morning Label-null count mismatch."
    )


if all_columns_null != EXPECTED_THU_MORNING_PADDING:
    raise RuntimeError(
        "\nLabel-null rows are not exactly the frozen all-column-null padding population."
    )


if effective != EXPECTED_THU_MORNING_EFFECTIVE:
    raise RuntimeError(
        "Thursday-morning effective count mismatch."
    )


print(
    "[PASS] All 288,602 Thursday-morning padding rows are NULL in all 85 columns."
)

print(
    "[PASS] No duplication assumption is required."
)

print()


# ==============================================================================
# 4. FROZEN EFFECTIVE SOURCE SET
# ==============================================================================

print("=" * 112)
print("FROZEN EFFECTIVE CICIDS2017 SOURCE")
print("=" * 112)


for record in results:

    print(
        f"{record['day']:22s}"
        f" effective={record['effective_rows']:9,d}"
    )


print()
print(
    "TOTAL:",
    f"{effective_total:,}",
)


if effective_total != 2_830_743:
    raise RuntimeError(
        "Frozen effective total changed."
    )


# Save for next cell.
STAGE24_CICIDS2017_EFFECTIVE_SOURCE_RECORDS = results

STAGE24_CICIDS2017_EFFECTIVE_FILTER = (
    '"Label" IS NOT NULL'
)


conn.close()


# ==============================================================================
# 5. FINAL
# ==============================================================================

print()
print("=" * 112)
print("STAGE24-1D0-R2: PASS")
print("=" * 112)

print()

print(
    "Physical source rows:              3,119,345"
)

print(
    "Structural null-padding rows:        288,602"
)

print(
    "Effective frozen target rows:      2,830,743"
)

print()

print(
    "Effective-row rule: Label IS NOT NULL"
)

print()

print(
    "PCAP accessed:                     NO"
)

print(
    "Model executed:                    NO"
)

print(
    "Target features supplied to model: NO"
)

print(
    "Predictions created:               0"
)

print(
    "Target metrics computed:           0"
)

print(
    "Target openings consumed:          0 / 8"
)

print(
    "Scientific fits:                   2 / 4"
)

print(
    "Git changes:                       NONE"
)

print()

print(
    "NEXT: semantic bridge projection validation,"
)

print(
    "      then PRIMARY OPENING #1: bridge62 / PUBLISHED."
)

print("=" * 112)

STAGE24-1D0-R2 — FROZEN STRUCTURAL NULL-PADDING VERIFICATION

PHYSICAL SOURCE CONTRACT
Monday                 physical=  529,918 expected=  529,918
Tuesday                physical=  445,909 expected=  445,909
Wednesday              physical=  692,703 expected=  692,703
Thursday-Afternoon     physical=  288,602 expected=  288,602
Thursday-Morning       physical=  458,968 expected=  458,968
Friday-DDos            physical=  225,745 expected=  225,745
Friday-PortScan        physical=  286,467 expected=  286,467
Friday-Morning         physical=  191,033 expected=  191,033

Physical total: 3,119,345
[PASS] Complete frozen physical source = 3,119,345 rows.

LABEL-NULL / EFFECTIVE-ROW AUDIT
Monday                 physical=  529,918 null=        0 effective=  529,918
Tuesday                physical=  445,909 null=        0 effective=  445,909
Wednesday              physical=  692,703 null=        0 effective=  692,703
Thursday-Afternoon     physical=  288,602 null=        0 effective=  288,602

In [30]:
# ==============================================================================
# STAGE24-1D1 — FROZEN SEMANTIC BRIDGE PROJECTION GATE
#
# PURPOSE
# -------
# Validate the already-frozen Stage24 semantic bridge against the eight
# byte-exact CICIDS2017 Parquets BEFORE the first target opening.
#
# This cell validates:
#   - repository / protocol-lock identity
#   - semantic bridge JSON + SHA sidecar
#   - bridge62 = exactly 62 source-ordered features
#   - bridge70 = exactly 70 source-ordered features
#   - bridge62 excludes exactly the 8 aggregate flag-count fields
#   - bridge62 PUBLISHED == FLAG_CORRECTED mapping
#   - bridge70 PUBLISHED / FLAG_CORRECTED mappings are both complete
#   - every frozen physical target column exists in every target Parquet
#   - all mapped feature columns have numeric-compatible physical schemas
#   - effective target population remains exactly 2,830,743
#   - Thursday structural-null rule remains exact
#
# DOES NOT:
#   - create feature matrices
#   - supply target rows to a model
#   - execute LightGBM/XGBoost
#   - create predictions
#   - compute target metrics
#   - consume a target opening
#   - modify Git
# ==============================================================================

from __future__ import annotations

import json
import hashlib
import subprocess
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 112)
print("STAGE24-1D1 — FROZEN SEMANTIC BRIDGE PROJECTION GATE")
print("=" * 112)
print()


# ==============================================================================
# 0. GOVERNANCE
# ==============================================================================

REPO = Path(
    globals().get(
        "REPO",
        "/kaggle/working/ids2018-validation-safe-ablation",
    )
)


EXPECTED_HEAD = (
    "a90a4221ef5a5e9958777049bcf13cc56a3519d3"
)


BRIDGE_SPEC = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_semantic_bridge_spec.json"
)


BRIDGE_SPEC_SHA = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_semantic_bridge_spec.sha256"
)


FINAL_LOCK = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)


FINAL_LOCK_SHA = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.sha256"
)


SOURCE_CONTRACT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0b2_complete_cicids2017_source_contract.json"
)


def run_cmd(args, *, cwd=None):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def verify_sidecar(
    artifact,
    sidecar,
):

    if not artifact.is_file():
        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )

    if not sidecar.is_file():
        raise RuntimeError(
            f"Missing SHA sidecar:\n{sidecar}"
        )


    actual = sha256_file(
        artifact
    )


    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )


    if actual != expected:

        raise RuntimeError(
            "\nSHA sidecar mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


    return actual


head = run_cmd(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=REPO,
)


status = run_cmd(
    [
        "git",
        "status",
        "--porcelain",
    ],
    cwd=REPO,
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "\nUnexpected repository HEAD.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository not clean:\n"
        + status
    )


print(
    "[PASS] Repository clean."
)

print()


bridge_file_sha = verify_sidecar(
    BRIDGE_SPEC,
    BRIDGE_SPEC_SHA,
)


lock_file_sha = verify_sidecar(
    FINAL_LOCK,
    FINAL_LOCK_SHA,
)


print(
    "Semantic bridge file SHA256:",
    bridge_file_sha,
)

print(
    "Final protocol lock SHA256: ",
    lock_file_sha,
)

print()

print(
    "[PASS] Frozen protocol artifacts byte-verified."
)

print()


# ==============================================================================
# 1. LOAD FROZEN SPEC — NO NEW MAPPING LOGIC
# ==============================================================================

bridge_spec = json.loads(
    BRIDGE_SPEC.read_text(
        encoding="utf-8"
    )
)


final_lock = json.loads(
    FINAL_LOCK.read_text(
        encoding="utf-8"
    )
)


source_contract = json.loads(
    SOURCE_CONTRACT.read_text(
        encoding="utf-8"
    )
)


if "bridge62" not in bridge_spec:
    raise RuntimeError(
        "bridge62 missing from frozen semantic spec."
    )


if "bridge70" not in bridge_spec:
    raise RuntimeError(
        "bridge70 missing from frozen semantic spec."
    )


b62 = bridge_spec[
    "bridge62"
]

b70 = bridge_spec[
    "bridge70"
]


# ==============================================================================
# 2. BRIDGE62 STRUCTURE
# ==============================================================================

print("=" * 112)
print("BRIDGE62")
print("=" * 112)


b62_order = list(
    b62[
        "source_feature_order"
    ]
)


b62_pub = dict(
    b62[
        "PUBLISHED_mapping"
    ]
)


b62_corr = dict(
    b62[
        "FLAG_CORRECTED_mapping"
    ]
)


excluded_flags = list(
    b62[
        "excluded_aggregate_flag_features"
    ]
)


EXPECTED_EXCLUDED_FLAGS = [
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "ACK Flag Cnt",
    "URG Flag Cnt",
    "CWE Flag Count",
    "ECE Flag Cnt",
]


print(
    "Feature count:",
    len(
        b62_order
    ),
)

print(
    "PUBLISHED mapping keys:",
    len(
        b62_pub
    ),
)

print(
    "FLAG_CORRECTED mapping keys:",
    len(
        b62_corr
    ),
)


if len(
    b62_order
) != 62:

    raise RuntimeError(
        f"bridge62 source order has {len(b62_order)} features."
    )


if set(
    b62_pub
) != set(
    b62_order
):

    raise RuntimeError(
        "bridge62 PUBLISHED mapping is incomplete or has extra keys."
    )


if set(
    b62_corr
) != set(
    b62_order
):

    raise RuntimeError(
        "bridge62 FLAG_CORRECTED mapping is incomplete or has extra keys."
    )


if b62_pub != b62_corr:

    raise RuntimeError(
        "\nbridge62 PUBLISHED != FLAG_CORRECTED.\n"
        "This violates the frozen flag-invariance contract."
    )


if sorted(
    excluded_flags
) != sorted(
    EXPECTED_EXCLUDED_FLAGS
):

    raise RuntimeError(
        "\nbridge62 excluded flag set changed.\n"
        f"Expected: {EXPECTED_EXCLUDED_FLAGS}\n"
        f"Actual:   {excluded_flags}"
    )


if b62.get(
    "published_equals_flag_corrected"
) is not True:

    raise RuntimeError(
        "Frozen bridge62 invariance marker is not true."
    )


print()
print(
    "[PASS] bridge62 = exactly 62 ordered features."
)

print(
    "[PASS] Exactly eight aggregate flag fields excluded."
)

print(
    "[PASS] PUBLISHED == FLAG_CORRECTED mapping bit-for-bit at specification level."
)

print()


# ==============================================================================
# 3. BRIDGE70 STRUCTURE
# ==============================================================================

print("=" * 112)
print("BRIDGE70")
print("=" * 112)


b70_order = list(
    b70[
        "source_feature_order"
    ]
)


b70_pub = dict(
    b70[
        "PUBLISHED_mapping"
    ]
)


b70_corr = dict(
    b70[
        "FLAG_CORRECTED_mapping"
    ]
)


print(
    "Feature count:",
    len(
        b70_order
    ),
)

print(
    "PUBLISHED mapping keys:",
    len(
        b70_pub
    ),
)

print(
    "FLAG_CORRECTED mapping keys:",
    len(
        b70_corr
    ),
)


if len(
    b70_order
) != 70:

    raise RuntimeError(
        f"bridge70 source order has {len(b70_order)} features."
    )


if set(
    b70_pub
) != set(
    b70_order
):

    raise RuntimeError(
        "bridge70 PUBLISHED mapping incomplete."
    )


if set(
    b70_corr
) != set(
    b70_order
):

    raise RuntimeError(
        "bridge70 FLAG_CORRECTED mapping incomplete."
    )


# Non-flag features must remain unchanged between variants.
nonflag70 = [
    feature
    for feature in b70_order
    if feature not in EXPECTED_EXCLUDED_FLAGS
]


variant_differences = [
    feature
    for feature in b70_order
    if b70_pub[
        feature
    ] != b70_corr[
        feature
    ]
]


unexpected_variant_differences = [
    feature
    for feature in variant_differences
    if feature not in EXPECTED_EXCLUDED_FLAGS
]


if unexpected_variant_differences:

    raise RuntimeError(
        "\nFLAG_CORRECTED remaps non-flag features:\n"
        + "\n".join(
            unexpected_variant_differences
        )
    )


print()
print(
    "PUBLISHED/FLAG_CORRECTED differing mappings:",
    len(
        variant_differences
    ),
)


for feature in variant_differences:

    print(
        f"  {feature:18s}"
        f" published <- {b70_pub[feature]}"
        f" | corrected <- {b70_corr[feature]}"
    )


if set(
    variant_differences
) != set(
    EXPECTED_EXCLUDED_FLAGS
):

    raise RuntimeError(
        "\nbridge70 variants do not differ on exactly the eight frozen flag fields."
    )


print()
print(
    "[PASS] bridge70 = exactly 70 ordered features."
)

print(
    "[PASS] Only the eight aggregate flag-count mappings differ by target variant."
)

print()


# ==============================================================================
# 4. SOURCE / TARGET FEATURE ORDER RELATION
# ==============================================================================

print("=" * 112)
print("BRIDGE ORDER RELATION")
print("=" * 112)


b70_without_flags = [
    feature
    for feature in b70_order
    if feature not in EXPECTED_EXCLUDED_FLAGS
]


if b70_without_flags != b62_order:

    raise RuntimeError(
        "\nbridge62 is not the ordered bridge70-minus-eight-flags projection."
    )


print(
    "[PASS] bridge62 order == bridge70 order with exactly 8 aggregate flags removed."
)

print()


# ==============================================================================
# 5. TARGET FILES
# ==============================================================================

ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/"
    "snapshots/b7e532345512edcd530cb1770dc76636aeb52802/"
    "traffic_labels"
)


TARGETS = [
    (
        "Monday",
        "Monday-WorkingHours.pcap_ISCX.csv.parquet",
        529_918,
    ),
    (
        "Tuesday",
        "Tuesday-WorkingHours.pcap_ISCX.csv.parquet",
        445_909,
    ),
    (
        "Wednesday",
        "Wednesday-workingHours.pcap_ISCX.csv.parquet",
        692_703,
    ),
    (
        "Thursday-Afternoon",
        "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet",
        288_602,
    ),
    (
        "Thursday-Morning",
        "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet",
        170_366,
    ),
    (
        "Friday-DDos",
        "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.parquet",
        225_745,
    ),
    (
        "Friday-PortScan",
        "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet",
        286_467,
    ),
    (
        "Friday-Morning",
        "Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet",
        191_033,
    ),
]


EXPECTED_EFFECTIVE_TOTAL = (
    2_830_743
)


# All physical target columns ever needed for either bridge/variant.
required_target_columns = (
    set(
        b62_pub.values()
    )
    |
    set(
        b62_corr.values()
    )
    |
    set(
        b70_pub.values()
    )
    |
    set(
        b70_corr.values()
    )
    |
    {
        "Label"
    }
)


print("=" * 112)
print("TARGET SCHEMA / MAPPING COVERAGE")
print("=" * 112)


effective_total = 0

reference_schema_names = None


# Numeric-compatible Arrow families.
def numeric_compatible(
    dtype,
):

    return (
        pa.types.is_integer(
            dtype
        )
        or
        pa.types.is_floating(
            dtype
        )
        or
        pa.types.is_decimal(
            dtype
        )
    )


type_warnings = []


for day, filename, expected_effective in TARGETS:

    path = (
        ROOT
        / filename
    )


    if not path.is_file():

        raise RuntimeError(
            f"Missing target source:\n{path}"
        )


    pf = pq.ParquetFile(
        path
    )


    schema = pf.schema_arrow

    names = list(
        schema.names
    )


    if reference_schema_names is None:

        reference_schema_names = names


    if names != reference_schema_names:

        raise RuntimeError(
            f"{day}: physical schema/order differs from frozen common schema."
        )


    missing = sorted(
        required_target_columns
        -
        set(
            names
        )
    )


    if missing:

        raise RuntimeError(
            f"\n{day}: required frozen target columns missing:\n"
            + "\n".join(
                missing
            )
        )


    field_lookup = {
        field.name:
            field.type
        for field in schema
    }


    for physical_column in sorted(
        required_target_columns
        -
        {
            "Label"
        }
    ):

        dtype = field_lookup[
            physical_column
        ]


        if not numeric_compatible(
            dtype
        ):

            type_warnings.append(
                (
                    day,
                    physical_column,
                    str(
                        dtype
                    ),
                )
            )


    # Metadata/previous gate already established the effective rule.
    # Here we count only the frozen expected effective populations.
    effective_total += (
        expected_effective
    )


    print(
        f"{day:22s}"
        f" columns={len(names):3d}"
        f" required={len(required_target_columns):3d}"
        f" effective={expected_effective:9,d}"
        "  [PASS]"
    )


if effective_total != EXPECTED_EFFECTIVE_TOTAL:

    raise RuntimeError(
        "\nEffective population arithmetic changed.\n"
        f"Expected: {EXPECTED_EFFECTIVE_TOTAL:,}\n"
        f"Actual:   {effective_total:,}"
    )


print()
print(
    "Effective population:",
    f"{effective_total:,}",
)

print(
    "[PASS] All frozen physical bridge columns exist in all eight Parquets."
)

print()


# ==============================================================================
# 6. TYPE CONTRACT
#
# String-typed numeric physical fields would still be legal under the frozen
# float64 parse policy, but we surface them before opening.
# ==============================================================================

print("=" * 112)
print("PHYSICAL TYPE AUDIT")
print("=" * 112)


if type_warnings:

    unique_warnings = sorted(
        set(
            (
                column,
                dtype,
            )
            for _day, column, dtype
            in type_warnings
        )
    )


    print(
        "Mapped fields requiring runtime float64 parsing:",
        len(
            unique_warnings
        ),
    )


    for column, dtype in unique_warnings:

        print(
            f"  {column:36s} {dtype}"
        )


    print()
    print(
        "[INFO] Allowed by frozen numeric policy; parse/fail-closed check occurs at opening."
    )


else:

    print(
        "[PASS] Every mapped target feature has a numeric-compatible Arrow physical type."
    )


print()


# ==============================================================================
# 7. FINAL LOCK CROSS-CHECK
# ==============================================================================

print("=" * 112)
print("FINAL PRE-OPENING LOCK CROSS-CHECK")
print("=" * 112)


primary = final_lock[
    "primary_direction"
]


if int(
    primary[
        "target_effective_rows_expected"
    ]
) != EXPECTED_EFFECTIVE_TOTAL:

    raise RuntimeError(
        "Final protocol lock target population changed."
    )


if int(
    primary[
        "bridge62"
    ][
        "feature_count"
    ]
) != 62:

    raise RuntimeError(
        "Final lock bridge62 feature count changed."
    )


if int(
    primary[
        "bridge70"
    ][
        "feature_count"
    ]
) != 70:

    raise RuntimeError(
        "Final lock bridge70 feature count changed."
    )


opening_summary = final_lock[
    "opening_summary"
]


if int(
    opening_summary[
        "consumed"
    ]
) != 0:

    raise RuntimeError(
        "Frozen pre-opening ledger was not at zero."
    )


comparison = final_lock[
    "comparison_protocol"
]


if (
    comparison[
        "bridge62_flag_invariance"
    ][
        "expected"
    ]
    !=
    "BITWISE_IDENTICAL_FEATURE_MATRICES_AND_PREDICTIONS"
):

    raise RuntimeError(
        "Frozen bridge62 flag-invariance expectation changed."
    )


print(
    "[PASS] Target population frozen at 2,830,743."
)

print(
    "[PASS] bridge62 frozen at 62 features."
)

print(
    "[PASS] bridge70 frozen at 70 features."
)

print(
    "[PASS] Original pre-opening ledger was 0 / 8."
)

print(
    "[PASS] bridge62 PUBLISHED/FLAG_CORRECTED identity remains mandatory."
)

print()


# ==============================================================================
# 8. SAVE LIVE SPEC FOR NEXT CELL ONLY
# ==============================================================================

STAGE24_BRIDGE_SPEC = bridge_spec

STAGE24_BRIDGE62_ORDER = b62_order

STAGE24_BRIDGE62_PUBLISHED_MAPPING = b62_pub

STAGE24_BRIDGE62_FLAG_CORRECTED_MAPPING = b62_corr

STAGE24_BRIDGE70_ORDER = b70_order

STAGE24_BRIDGE70_PUBLISHED_MAPPING = b70_pub

STAGE24_BRIDGE70_FLAG_CORRECTED_MAPPING = b70_corr

STAGE24_TARGET_ROOT = ROOT

STAGE24_EFFECTIVE_TARGET_ROWS = EXPECTED_EFFECTIVE_TOTAL


# ==============================================================================
# 9. FINAL
# ==============================================================================

print("=" * 112)
print("STAGE24-1D1 SEMANTIC BRIDGE GATE: PASS")
print("=" * 112)

print()

print(
    "Frozen target rows:                 2,830,743"
)

print(
    "bridge62 features:                  62"
)

print(
    "bridge70 features:                  70"
)

print(
    "bridge62 PUBLISHED == CORRECTED:    YES"
)

print()

print(
    "Target feature matrix materialized: NO"
)

print(
    "Target rows supplied to model:      0"
)

print(
    "Model executed:                     NO"
)

print(
    "Predictions created:                0"
)

print(
    "Target metrics computed:            0"
)

print(
    "Target openings consumed:           0 / 8"
)

print(
    "Scientific fits completed:          2 / 4"
)

print(
    "Git changes:                        NONE"
)

print()

print(
    "NEXT: PRIMARY TARGET OPENING #1"
)

print(
    "      bridge62 / PUBLISHED"
)

print(
    "      (FLAG_CORRECTED must reproduce the same matrix/predictions)"
)

print("=" * 112)

STAGE24-1D1 — FROZEN SEMANTIC BRIDGE PROJECTION GATE

HEAD: a90a4221ef5a5e9958777049bcf13cc56a3519d3
[PASS] Repository clean.

Semantic bridge file SHA256: b7ff1d8563aafc69145be3acc827fbc59ae75d880b9804bad727f408299fce2f
Final protocol lock SHA256:  8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2

[PASS] Frozen protocol artifacts byte-verified.

BRIDGE62
Feature count: 62
PUBLISHED mapping keys: 62
FLAG_CORRECTED mapping keys: 62

[PASS] bridge62 = exactly 62 ordered features.
[PASS] Exactly eight aggregate flag fields excluded.
[PASS] PUBLISHED == FLAG_CORRECTED mapping bit-for-bit at specification level.

BRIDGE70
Feature count: 70
PUBLISHED mapping keys: 70
FLAG_CORRECTED mapping keys: 70

PUBLISHED/FLAG_CORRECTED differing mappings: 7
  FIN Flag Cnt       published <- FIN Flag Count | corrected <- URG Flag Count
  SYN Flag Cnt       published <- SYN Flag Count | corrected <- PSH Flag Count
  RST Flag Cnt       published <- RST Flag Count | corrected <- FIN Flag Cou

RuntimeError: 
bridge70 variants do not differ on exactly the eight frozen flag fields.

In [31]:
# ==============================================================================
# STAGE24-1D1-R1 — CORRECTED FROZEN SEMANTIC BRIDGE PROJECTION GATE
#
# Correction:
#   bridge70 contains 8 aggregate flag features,
#   but only 7 PUBLISHED vs FLAG_CORRECTED physical mappings differ,
#   because ACK remains ACK in both variants.
#
# Frozen corrected flag mapping:
#
#   FIN Flag Cnt   <- URG Flag Count
#   SYN Flag Cnt   <- PSH Flag Count
#   RST Flag Cnt   <- FIN Flag Count
#   PSH Flag Cnt   <- SYN Flag Count
#   ACK Flag Cnt   <- ACK Flag Count      # identity, therefore NOT a diff
#   URG Flag Cnt   <- CWE Flag Count
#   CWE Flag Count <- ECE Flag Count
#   ECE Flag Cnt   <- RST Flag Count
#
# NO TARGET MODEL EXECUTION.
# NO PREDICTIONS.
# NO TARGET OPENING.
# NO GIT CHANGE.
# ==============================================================================

from __future__ import annotations

import json
import hashlib
import subprocess
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 112)
print("STAGE24-1D1-R1 — CORRECTED FROZEN SEMANTIC BRIDGE PROJECTION GATE")
print("=" * 112)
print()


# ==============================================================================
# 0. PATHS / GOVERNANCE
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "a90a4221ef5a5e9958777049bcf13cc56a3519d3"
)


BRIDGE_SPEC_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_semantic_bridge_spec.json"
)


BRIDGE_SPEC_SHA_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_semantic_bridge_spec.sha256"
)


FINAL_LOCK_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)


FINAL_LOCK_SHA_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.sha256"
)


TARGET_ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/"
    "snapshots/b7e532345512edcd530cb1770dc76636aeb52802/"
    "traffic_labels"
)


EXPECTED_EFFECTIVE_ROWS = 2_830_743


# ==============================================================================
# 1. HELPERS
# ==============================================================================

def run_cmd(args, *, cwd=None):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            chunk = fh.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def verify_sidecar(
    artifact,
    sidecar,
):

    if not artifact.is_file():

        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )


    if not sidecar.is_file():

        raise RuntimeError(
            f"Missing SHA sidecar:\n{sidecar}"
        )


    actual = sha256_file(
        artifact
    )


    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )


    if actual != expected:

        raise RuntimeError(
            "\nSHA mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


    return actual


# ==============================================================================
# 2. REPOSITORY GATE
# ==============================================================================

print("=" * 112)
print("GOVERNANCE")
print("=" * 112)


head = run_cmd(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=REPO,
)


status = run_cmd(
    [
        "git",
        "status",
        "--porcelain",
    ],
    cwd=REPO,
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


print(
    "[PASS] Repository clean."
)

print()


bridge_sha = verify_sidecar(
    BRIDGE_SPEC_PATH,
    BRIDGE_SPEC_SHA_PATH,
)


lock_sha = verify_sidecar(
    FINAL_LOCK_PATH,
    FINAL_LOCK_SHA_PATH,
)


print(
    "Semantic bridge SHA256:",
    bridge_sha,
)

print(
    "Final lock SHA256:     ",
    lock_sha,
)

print()


# ==============================================================================
# 3. LOAD EXACT FROZEN ARTIFACTS
# ==============================================================================

bridge_spec = json.loads(
    BRIDGE_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)


final_lock = json.loads(
    FINAL_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


b62 = bridge_spec[
    "bridge62"
]

b70 = bridge_spec[
    "bridge70"
]


# ==============================================================================
# 4. FROZEN FLAG SEMANTICS
# ==============================================================================

FLAG_FEATURES = [
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "ACK Flag Cnt",
    "URG Flag Cnt",
    "CWE Flag Count",
    "ECE Flag Cnt",
]


EXPECTED_PUBLISHED_FLAGS = {
    "FIN Flag Cnt":
        "FIN Flag Count",

    "SYN Flag Cnt":
        "SYN Flag Count",

    "RST Flag Cnt":
        "RST Flag Count",

    "PSH Flag Cnt":
        "PSH Flag Count",

    "ACK Flag Cnt":
        "ACK Flag Count",

    "URG Flag Cnt":
        "URG Flag Count",

    "CWE Flag Count":
        "CWE Flag Count",

    "ECE Flag Cnt":
        "ECE Flag Count",
}


EXPECTED_CORRECTED_FLAGS = {
    "FIN Flag Cnt":
        "URG Flag Count",

    "SYN Flag Cnt":
        "PSH Flag Count",

    "RST Flag Cnt":
        "FIN Flag Count",

    "PSH Flag Cnt":
        "SYN Flag Count",

    "ACK Flag Cnt":
        "ACK Flag Count",

    "URG Flag Cnt":
        "CWE Flag Count",

    "CWE Flag Count":
        "ECE Flag Count",

    "ECE Flag Cnt":
        "RST Flag Count",
}


# ==============================================================================
# 5. BRIDGE62
# ==============================================================================

print("=" * 112)
print("BRIDGE62")
print("=" * 112)


b62_order = list(
    b62[
        "source_feature_order"
    ]
)


b62_pub = dict(
    b62[
        "PUBLISHED_mapping"
    ]
)


b62_corr = dict(
    b62[
        "FLAG_CORRECTED_mapping"
    ]
)


b62_excluded = list(
    b62[
        "excluded_aggregate_flag_features"
    ]
)


print(
    "Feature count:",
    len(
        b62_order
    ),
)


if len(
    b62_order
) != 62:

    raise RuntimeError(
        "bridge62 feature count != 62."
    )


if set(
    b62_pub
) != set(
    b62_order
):

    raise RuntimeError(
        "bridge62 PUBLISHED mapping incomplete."
    )


if set(
    b62_corr
) != set(
    b62_order
):

    raise RuntimeError(
        "bridge62 FLAG_CORRECTED mapping incomplete."
    )


if b62_pub != b62_corr:

    raise RuntimeError(
        "bridge62 PUBLISHED and FLAG_CORRECTED mappings differ."
    )


if set(
    b62_excluded
) != set(
    FLAG_FEATURES
):

    raise RuntimeError(
        "bridge62 does not exclude exactly the eight aggregate flag fields."
    )


for flag in FLAG_FEATURES:

    if flag in b62_order:

        raise RuntimeError(
            f"Excluded flag unexpectedly present in bridge62: {flag}"
        )


print(
    "[PASS] bridge62 = 62 features."
)

print(
    "[PASS] Exactly 8 aggregate flag fields excluded."
)

print(
    "[PASS] PUBLISHED == FLAG_CORRECTED at bridge62."
)

print()


# ==============================================================================
# 6. BRIDGE70
# ==============================================================================

print("=" * 112)
print("BRIDGE70")
print("=" * 112)


b70_order = list(
    b70[
        "source_feature_order"
    ]
)


b70_pub = dict(
    b70[
        "PUBLISHED_mapping"
    ]
)


b70_corr = dict(
    b70[
        "FLAG_CORRECTED_mapping"
    ]
)


if len(
    b70_order
) != 70:

    raise RuntimeError(
        "bridge70 feature count != 70."
    )


if set(
    b70_pub
) != set(
    b70_order
):

    raise RuntimeError(
        "bridge70 PUBLISHED mapping incomplete."
    )


if set(
    b70_corr
) != set(
    b70_order
):

    raise RuntimeError(
        "bridge70 FLAG_CORRECTED mapping incomplete."
    )


# ------------------------------------------------------------------------------
# Exact eight-field semantic validation.
# ------------------------------------------------------------------------------

for feature in FLAG_FEATURES:

    actual_pub = b70_pub[
        feature
    ]

    expected_pub = EXPECTED_PUBLISHED_FLAGS[
        feature
    ]


    if actual_pub != expected_pub:

        raise RuntimeError(
            "\nUnexpected PUBLISHED flag mapping.\n"
            f"Feature:  {feature}\n"
            f"Expected: {expected_pub}\n"
            f"Actual:   {actual_pub}"
        )


    actual_corr = b70_corr[
        feature
    ]

    expected_corr = EXPECTED_CORRECTED_FLAGS[
        feature
    ]


    if actual_corr != expected_corr:

        raise RuntimeError(
            "\nUnexpected FLAG_CORRECTED flag mapping.\n"
            f"Feature:  {feature}\n"
            f"Expected: {expected_corr}\n"
            f"Actual:   {actual_corr}"
        )


print(
    "[PASS] All 8 frozen aggregate-flag semantics match exactly."
)

print()


# ------------------------------------------------------------------------------
# Only seven mappings DIFFER because ACK is identity in both variants.
# ------------------------------------------------------------------------------

variant_differences = [
    feature
    for feature in b70_order
    if b70_pub[
        feature
    ] != b70_corr[
        feature
    ]
]


EXPECTED_DIFFERING_FLAGS = [
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "URG Flag Cnt",
    "CWE Flag Count",
    "ECE Flag Cnt",
]


print(
    "PUBLISHED/FLAG_CORRECTED differing mappings:",
    len(
        variant_differences
    ),
)


for feature in variant_differences:

    print(
        f"  {feature:18s}"
        f" published <- {b70_pub[feature]}"
        f" | corrected <- {b70_corr[feature]}"
    )


if set(
    variant_differences
) != set(
    EXPECTED_DIFFERING_FLAGS
):

    raise RuntimeError(
        "\nVariant-difference set is not the frozen 7-field permutation set.\n"
        f"Expected: {sorted(EXPECTED_DIFFERING_FLAGS)}\n"
        f"Actual:   {sorted(variant_differences)}"
    )


if (
    b70_pub[
        "ACK Flag Cnt"
    ]
    !=
    "ACK Flag Count"
):

    raise RuntimeError(
        "PUBLISHED ACK mapping changed."
    )


if (
    b70_corr[
        "ACK Flag Cnt"
    ]
    !=
    "ACK Flag Count"
):

    raise RuntimeError(
        "FLAG_CORRECTED ACK mapping changed."
    )


print()

print(
    "[PASS] Exactly 7 physical mappings differ."
)

print(
    "[PASS] ACK Flag Cnt remains ACK Flag Count in both variants."
)

print()


# ==============================================================================
# 7. NON-FLAG FEATURES MUST BE IDENTICAL BETWEEN VARIANTS
# ==============================================================================

nonflag_features = [
    feature
    for feature in b70_order
    if feature not in FLAG_FEATURES
]


nonflag_differences = [
    feature
    for feature in nonflag_features
    if b70_pub[
        feature
    ] != b70_corr[
        feature
    ]
]


if nonflag_differences:

    raise RuntimeError(
        "\nNon-flag target mappings differ between variants:\n"
        + "\n".join(
            nonflag_differences
        )
    )


print(
    "[PASS] All 62 non-flag feature mappings are invariant."
)

print()


# ==============================================================================
# 8. BRIDGE62 MUST BE ORDERED BRIDGE70 MINUS THE 8 FLAG FEATURES
# ==============================================================================

print("=" * 112)
print("BRIDGE ORDER RELATION")
print("=" * 112)


b70_without_flags = [
    feature
    for feature in b70_order
    if feature not in FLAG_FEATURES
]


if b70_without_flags != b62_order:

    raise RuntimeError(
        "bridge62 order != bridge70 minus 8 aggregate flags."
    )


print(
    "[PASS] bridge62 order = bridge70 order minus exactly 8 flag features."
)

print()


# ==============================================================================
# 9. TARGET SOURCE CONTRACT
# ==============================================================================

TARGETS = [
    (
        "Monday",
        "Monday-WorkingHours.pcap_ISCX.csv.parquet",
        529_918,
    ),
    (
        "Tuesday",
        "Tuesday-WorkingHours.pcap_ISCX.csv.parquet",
        445_909,
    ),
    (
        "Wednesday",
        "Wednesday-workingHours.pcap_ISCX.csv.parquet",
        692_703,
    ),
    (
        "Thursday-Afternoon",
        "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet",
        288_602,
    ),
    (
        "Thursday-Morning",
        "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet",
        170_366,
    ),
    (
        "Friday-DDos",
        "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.parquet",
        225_745,
    ),
    (
        "Friday-PortScan",
        "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet",
        286_467,
    ),
    (
        "Friday-Morning",
        "Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet",
        191_033,
    ),
]


required_target_columns = (
    set(
        b70_pub.values()
    )
    |
    set(
        b70_corr.values()
    )
    |
    {
        "Label"
    }
)


print("=" * 112)
print("TARGET PHYSICAL SCHEMA COVERAGE")
print("=" * 112)


effective_total = 0

reference_names = None


def numeric_type(
    dtype,
):

    return (
        pa.types.is_integer(
            dtype
        )
        or
        pa.types.is_floating(
            dtype
        )
        or
        pa.types.is_decimal(
            dtype
        )
    )


nonnumeric_mapped = set()


for day, filename, effective_rows in TARGETS:

    path = (
        TARGET_ROOT
        / filename
    )


    if not path.is_file():

        raise RuntimeError(
            f"Missing target file:\n{path}"
        )


    pf = pq.ParquetFile(
        path
    )


    schema = pf.schema_arrow

    names = list(
        schema.names
    )


    if reference_names is None:

        reference_names = names


    if names != reference_names:

        raise RuntimeError(
            f"{day}: target schema/order mismatch."
        )


    missing = sorted(
        required_target_columns
        -
        set(
            names
        )
    )


    if missing:

        raise RuntimeError(
            f"\n{day}: missing required target columns:\n"
            + "\n".join(
                missing
            )
        )


    type_lookup = {
        field.name:
            field.type
        for field in schema
    }


    for column in (
        required_target_columns
        -
        {
            "Label"
        }
    ):

        dtype = type_lookup[
            column
        ]


        if not numeric_type(
            dtype
        ):

            nonnumeric_mapped.add(
                (
                    column,
                    str(
                        dtype
                    ),
                )
            )


    effective_total += int(
        effective_rows
    )


    print(
        f"{day:22s}"
        f" effective={effective_rows:9,d}"
        f" schema={len(names):2d}"
        " [PASS]"
    )


if effective_total != EXPECTED_EFFECTIVE_ROWS:

    raise RuntimeError(
        "\nEffective target total mismatch.\n"
        f"Expected: {EXPECTED_EFFECTIVE_ROWS:,}\n"
        f"Actual:   {effective_total:,}"
    )


print()

print(
    "Effective target total:",
    f"{effective_total:,}",
)

print(
    "[PASS] All required physical target fields exist."
)

print()


if nonnumeric_mapped:

    print(
        "Mapped fields requiring frozen runtime float64 parser:"
    )

    for column, dtype in sorted(
        nonnumeric_mapped
    ):

        print(
            f"  {column:38s} {dtype}"
        )

    print()

else:

    print(
        "[PASS] All mapped feature fields have numeric Arrow types."
    )

    print()


# ==============================================================================
# 10. FINAL PROTOCOL LOCK CROSS-CHECK
# ==============================================================================

print("=" * 112)
print("FINAL LOCK CROSS-CHECK")
print("=" * 112)


primary = final_lock[
    "primary_direction"
]


if int(
    primary[
        "target_effective_rows_expected"
    ]
) != EXPECTED_EFFECTIVE_ROWS:

    raise RuntimeError(
        "Frozen target population changed."
    )


if int(
    primary[
        "bridge62"
    ][
        "feature_count"
    ]
) != 62:

    raise RuntimeError(
        "Frozen bridge62 count changed."
    )


if int(
    primary[
        "bridge70"
    ][
        "feature_count"
    ]
) != 70:

    raise RuntimeError(
        "Frozen bridge70 count changed."
    )


if int(
    final_lock[
        "opening_summary"
    ][
        "consumed"
    ]
) != 0:

    raise RuntimeError(
        "Original pre-opening ledger is not zero."
    )


bridge62_expectation = (
    final_lock[
        "comparison_protocol"
    ][
        "bridge62_flag_invariance"
    ][
        "expected"
    ]
)


if (
    bridge62_expectation
    !=
    "BITWISE_IDENTICAL_FEATURE_MATRICES_AND_PREDICTIONS"
):

    raise RuntimeError(
        "bridge62 identity expectation changed."
    )


print(
    "[PASS] Frozen target rows = 2,830,743."
)

print(
    "[PASS] Frozen bridge62 = 62 features."
)

print(
    "[PASS] Frozen bridge70 = 70 features."
)

print(
    "[PASS] Pre-opening ledger remains 0 consumed."
)

print()


# ==============================================================================
# 11. LIVE STATE FOR NEXT CELL
# ==============================================================================

STAGE24_BRIDGE_SPEC = (
    bridge_spec
)

STAGE24_BRIDGE62_ORDER = (
    b62_order
)

STAGE24_BRIDGE62_PUBLISHED_MAPPING = (
    b62_pub
)

STAGE24_BRIDGE62_FLAG_CORRECTED_MAPPING = (
    b62_corr
)

STAGE24_BRIDGE70_ORDER = (
    b70_order
)

STAGE24_BRIDGE70_PUBLISHED_MAPPING = (
    b70_pub
)

STAGE24_BRIDGE70_FLAG_CORRECTED_MAPPING = (
    b70_corr
)

STAGE24_TARGET_ROOT = (
    TARGET_ROOT
)

STAGE24_EFFECTIVE_TARGET_ROWS = (
    EXPECTED_EFFECTIVE_ROWS
)


# ==============================================================================
# 12. FINAL
# ==============================================================================

print("=" * 112)
print("STAGE24-1D1-R1 SEMANTIC BRIDGE GATE: PASS")
print("=" * 112)

print()

print(
    "Aggregate flag features:             8"
)

print(
    "bridge70 variant mapping differences:7"
)

print(
    "ACK mapping difference:              NO"
)

print(
    "bridge62 variant mapping differences:0"
)

print()

print(
    "Effective target rows:               2,830,743"
)

print(
    "Target matrix materialized:          NO"
)

print(
    "Target rows supplied to model:       0"
)

print(
    "Predictions created:                 0"
)

print(
    "Metrics computed:                    0"
)

print(
    "Target openings consumed:            0 / 8"
)

print(
    "Scientific fits:                     2 / 4"
)

print(
    "Git changes:                         NONE"
)

print()

print(
    "NEXT: PRIMARY TARGET OPENING #1"
)

print(
    "      bridge62 / PUBLISHED"
)

print("=" * 112)

STAGE24-1D1-R1 — CORRECTED FROZEN SEMANTIC BRIDGE PROJECTION GATE

GOVERNANCE
HEAD: a90a4221ef5a5e9958777049bcf13cc56a3519d3
[PASS] Repository clean.

Semantic bridge SHA256: b7ff1d8563aafc69145be3acc827fbc59ae75d880b9804bad727f408299fce2f
Final lock SHA256:      8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2

BRIDGE62
Feature count: 62
[PASS] bridge62 = 62 features.
[PASS] Exactly 8 aggregate flag fields excluded.
[PASS] PUBLISHED == FLAG_CORRECTED at bridge62.

BRIDGE70
[PASS] All 8 frozen aggregate-flag semantics match exactly.

PUBLISHED/FLAG_CORRECTED differing mappings: 7
  FIN Flag Cnt       published <- FIN Flag Count | corrected <- URG Flag Count
  SYN Flag Cnt       published <- SYN Flag Count | corrected <- PSH Flag Count
  RST Flag Cnt       published <- RST Flag Count | corrected <- FIN Flag Count
  PSH Flag Cnt       published <- PSH Flag Count | corrected <- SYN Flag Count
  URG Flag Cnt       published <- URG Flag Count | corrected <- CWE Flag Count
  

In [32]:
# ==============================================================================
# STAGE24-2A — PRIMARY TARGET OPENING #1
# IDS2018 -> CICIDS2017
# bridge62 / PUBLISHED
#
# THIS CELL CONSUMES TARGET OPENING #1.
#
# Frozen:
#   model      = Stage24-1B bridge62 LightGBM + XGBoost
#   ensemble   = 0.5 * P_LGBM + 0.5 * P_XGB
#   input      = float64
#   scaling    = NONE
#   imputation = NONE
#   +/-inf     = NaN
#
# Source-validation thresholds:
#   STANDARD = 0.50
#   BALANCED = 0.05
#   SECURITY = 0.05
#
# Target:
#   CICIDS2017 PUBLISHED
#   effective rows = 2,830,743
#
# Anti-adaptation:
#   NO fit
#   NO target tuning
#   NO threshold search
#   NO feature search
#   NO target-fitted preprocessing
#
# Persists:
#   - ensemble float32 predictions
#   - binary target labels
#   - file/day ids
#   - result JSON
#   - SHA256 receipts
#
# Then commits + pushes.
# ==============================================================================

from __future__ import annotations

import os
import re
import gc
import json
import math
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import duckdb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


print("=" * 118)
print("STAGE24-2A — PRIMARY TARGET OPENING #1 — bridge62 / PUBLISHED")
print("=" * 118)
print()


# ==============================================================================
# 0. FROZEN GOVERNANCE
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_HEAD = (
    "a90a4221ef5a5e9958777049bcf13cc56a3519d3"
)

EXPECTED_TARGET_ROWS = 2_830_743

EXPECTED_LGBM_MODEL_SHA = (
    "907c9c86755c3c32effb2d0899f92e1a0cce71915f90311c99afd8c73814c4d7"
)

EXPECTED_XGB_MODEL_SHA = (
    "b30966f34d6f0799657c528344ec540badef7f3ca3dd207abdaa4b45400e6231"
)

EXPECTED_BRIDGE_SPEC_SHA = (
    "b7ff1d8563aafc69145be3acc827fbc59ae75d880b9804bad727f408299fce2f"
)

EXPECTED_PROTOCOL_LOCK_SHA = (
    "8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2"
)


SOURCE_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
)

LGB_MODEL_PATH = (
    SOURCE_DIR
    / "bridge62_lightgbm_model.txt"
)

XGB_MODEL_PATH = (
    SOURCE_DIR
    / "bridge62_xgboost_model.json"
)

SOURCE_RESULT_PATH = (
    SOURCE_DIR
    / "stage24_1b_bridge62_source_refit_result.json"
)

SOURCE_RESULT_SHA_PATH = (
    SOURCE_DIR
    / "stage24_1b_bridge62_source_refit_result.sha256"
)


BRIDGE_SPEC_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_semantic_bridge_spec.json"
)

BRIDGE_SPEC_SHA_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_semantic_bridge_spec.sha256"
)

FINAL_LOCK_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)

FINAL_LOCK_SHA_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.sha256"
)


TARGET_ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/"
    "snapshots/b7e532345512edcd530cb1770dc76636aeb52802/"
    "traffic_labels"
)


OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_2_primary_target_openings"
    / "stage24_2a_bridge62_published"
)

RESULT_PATH = (
    OUT
    / "stage24_2a_bridge62_published_result.json"
)

RESULT_SHA_PATH = (
    OUT
    / "stage24_2a_bridge62_published_result.sha256"
)

PRED_PATH = (
    OUT
    / "bridge62_published_ensemble_predictions.npz"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


# Runtime crash marker.
# If inference starts and the cell later crashes, DO NOT rerun this opening cell.
RUNTIME_LEDGER = Path(
    "/kaggle/working/stage24_target_opening_runtime_ledger.json"
)


# ==============================================================================
# 1. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    command = [
        "git"
    ]

    if auth_header is not None:

        command += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    command += [
        str(x)
        for x in args
    ]

    return run_cmd(
        command,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_array(
    array,
):

    arr = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        arr.view(
            np.uint8
        )
    ).hexdigest()


def verify_sidecar(
    artifact,
    sidecar,
):

    actual = sha256_file(
        artifact
    )

    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )

    if actual != expected:

        raise RuntimeError(
            "\nSHA mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )

    return actual


def qident(
    value,
):

    return (
        '"'
        +
        str(
            value
        ).replace(
            '"',
            '""',
        )
        +
        '"'
    )


def sql_path(
    path,
):

    return str(
        path
    ).replace(
        "'",
        "''",
    )


# ==============================================================================
# 2. REPOSITORY / RERUN GATES
# ==============================================================================

print("=" * 118)
print("PRE-OPENING GOVERNANCE")
print("=" * 118)


if RUNTIME_LEDGER.exists():

    raise RuntimeError(
        "\nA Stage24 target-opening runtime ledger already exists:\n"
        f"{RUNTIME_LEDGER}\n\n"
        "DO NOT rerun this opening cell. Use a recovery cell."
    )


head = git_cmd(
    "rev-parse",
    "HEAD",
)

status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_HEAD:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


if OUT.exists() and any(
    OUT.iterdir()
):

    raise RuntimeError(
        "\nOpening output directory already contains files:\n"
        f"{OUT}\n\n"
        "Refusing accidental duplicate opening."
    )


print(
    "[PASS] Repository clean."
)

print(
    "[PASS] No prior runtime opening marker."
)

print(
    "[PASS] No prior Stage24-2A artifacts."
)

print()


# ==============================================================================
# 3. VERIFY FROZEN ARTIFACTS
# ==============================================================================

print("=" * 118)
print("FROZEN ARTIFACT VERIFICATION")
print("=" * 118)


for path in [
    LGB_MODEL_PATH,
    XGB_MODEL_PATH,
    SOURCE_RESULT_PATH,
    SOURCE_RESULT_SHA_PATH,
    BRIDGE_SPEC_PATH,
    BRIDGE_SPEC_SHA_PATH,
    FINAL_LOCK_PATH,
    FINAL_LOCK_SHA_PATH,
]:

    if not path.is_file():

        raise RuntimeError(
            f"Missing frozen artifact:\n{path}"
        )


source_result_sha = verify_sidecar(
    SOURCE_RESULT_PATH,
    SOURCE_RESULT_SHA_PATH,
)

bridge_spec_sha = verify_sidecar(
    BRIDGE_SPEC_PATH,
    BRIDGE_SPEC_SHA_PATH,
)

protocol_lock_sha = verify_sidecar(
    FINAL_LOCK_PATH,
    FINAL_LOCK_SHA_PATH,
)


if bridge_spec_sha != EXPECTED_BRIDGE_SPEC_SHA:

    raise RuntimeError(
        "Semantic bridge SHA changed."
    )


if protocol_lock_sha != EXPECTED_PROTOCOL_LOCK_SHA:

    raise RuntimeError(
        "Final protocol lock SHA changed."
    )


lgb_model_sha = sha256_file(
    LGB_MODEL_PATH
)

xgb_model_sha = sha256_file(
    XGB_MODEL_PATH
)


print(
    "LightGBM model SHA:",
    lgb_model_sha,
)

print(
    "XGBoost model SHA: ",
    xgb_model_sha,
)


if lgb_model_sha != EXPECTED_LGBM_MODEL_SHA:

    raise RuntimeError(
        "Frozen LightGBM model SHA mismatch."
    )


if xgb_model_sha != EXPECTED_XGB_MODEL_SHA:

    raise RuntimeError(
        "Frozen XGBoost model SHA mismatch."
    )


print(
    "[PASS] Both frozen bridge62 model identities exact."
)

print()


# ==============================================================================
# 4. LOAD FROZEN SCIENTIFIC CONTRACT
# ==============================================================================

source_result = json.loads(
    SOURCE_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)

bridge_spec = json.loads(
    BRIDGE_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)

final_lock = json.loads(
    FINAL_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


b62 = bridge_spec[
    "bridge62"
]

FEATURE_ORDER = list(
    b62[
        "source_feature_order"
    ]
)

PUBLISHED_MAPPING = dict(
    b62[
        "PUBLISHED_mapping"
    ]
)


if len(
    FEATURE_ORDER
) != 62:

    raise RuntimeError(
        "bridge62 feature count changed."
    )


if set(
    FEATURE_ORDER
) != set(
    PUBLISHED_MAPPING
):

    raise RuntimeError(
        "bridge62 PUBLISHED mapping incomplete."
    )


if (
    b62[
        "PUBLISHED_mapping"
    ]
    !=
    b62[
        "FLAG_CORRECTED_mapping"
    ]
):

    raise RuntimeError(
        "bridge62 PUBLISHED/FLAG_CORRECTED invariance broken."
    )


# ------------------------------------------------------------------------------
# Thresholds are inherited from SOURCE VALIDATION ONLY.
# ------------------------------------------------------------------------------

STANDARD_THRESHOLD = float(
    source_result[
        "source_validation_metrics"
    ][
        "standard"
    ][
        "threshold_float32_runtime"
    ]
)


BALANCED_THRESHOLD = float(
    source_result[
        "source_validation_metrics"
    ][
        "balanced"
    ][
        "threshold_float32_runtime"
    ]
)


SECURITY_THRESHOLD = float(
    source_result[
        "source_validation_metrics"
    ][
        "security"
    ][
        "result"
    ][
        "threshold_float32_runtime"
    ]
)


print(
    "Frozen STANDARD threshold:",
    repr(
        STANDARD_THRESHOLD
    ),
)

print(
    "Frozen BALANCED threshold:",
    repr(
        BALANCED_THRESHOLD
    ),
)

print(
    "Frozen SECURITY threshold:",
    repr(
        SECURITY_THRESHOLD
    ),
)


if STANDARD_THRESHOLD != 0.5:

    raise RuntimeError(
        "Standard threshold changed."
    )


if np.float32(
    BALANCED_THRESHOLD
) != np.float32(
    0.05
):

    raise RuntimeError(
        "Balanced threshold changed."
    )


if np.float32(
    SECURITY_THRESHOLD
) != np.float32(
    0.05
):

    raise RuntimeError(
        "Security threshold changed."
    )


if source_result[
    "anti_adaptation"
][
    "target_threshold_selection"
]:

    raise RuntimeError(
        "Source result unexpectedly reports target threshold selection."
    )


print(
    "[PASS] Thresholds frozen from IDS2018 source validation only."
)

print()


# ==============================================================================
# 5. TARGET POPULATION ORDER
#
# Read directly from frozen final protocol lock.
# ==============================================================================

population = list(
    final_lock[
        "published_population"
    ]
)


if len(
    population
) != 8:

    raise RuntimeError(
        f"Expected 8 frozen published population records, got {len(population)}."
    )


expected_sum = sum(
    int(
        record[
            "effective_rows"
        ]
    )
    for record in population
)


if expected_sum != EXPECTED_TARGET_ROWS:

    raise RuntimeError(
        "\nFrozen target population sum mismatch.\n"
        f"Expected: {EXPECTED_TARGET_ROWS:,}\n"
        f"Actual:   {expected_sum:,}"
    )


print("=" * 118)
print("FROZEN TARGET ORDER")
print("=" * 118)


for i, record in enumerate(
    population,
    start=1,
):

    print(
        f"[{i:02d}] "
        f"{record['day']:10s} "
        f"{Path(record['remote']).name} "
        f"effective={int(record['effective_rows']):,}"
    )


print()
print(
    "[PASS] Frozen effective target rows:",
    f"{expected_sum:,}",
)

print()


# ==============================================================================
# 6. GITHUB CREDENTIAL + REMOTE PARENT GATE
#
# Do this BEFORE consuming the target opening so durable persistence is ready.
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if value and value.strip():

            github_token = value.strip()
            token_source = f"ENV:{name}"
            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable BEFORE target opening."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_HEAD:

    raise RuntimeError(
        "\nRemote main moved before opening.\n"
        f"Expected: {EXPECTED_HEAD}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main == frozen pre-opening parent."
)

print()


# ==============================================================================
# 7. LOAD FROZEN MODELS
#
# Loading a model does NOT consume the target opening.
# ==============================================================================

print("=" * 118)
print("MODEL LOAD")
print("=" * 118)


import lightgbm as lgb
import xgboost as xgb


print(
    "LightGBM:",
    lgb.__version__,
)

print(
    "XGBoost: ",
    xgb.__version__,
)


lgb_model = lgb.Booster(
    model_file=str(
        LGB_MODEL_PATH
    )
)


xgb_model = xgb.Booster()

xgb_model.load_model(
    str(
        XGB_MODEL_PATH
    )
)


# Target inference may occur on CPU.
# This does NOT alter the frozen tree model.
try:

    xgb_model.set_param(
        {
            "device":
                "cpu",

            "nthread":
                -1,
        }
    )

except Exception:
    pass


if lgb_model.num_feature() != 62:

    raise RuntimeError(
        f"LightGBM expects {lgb_model.num_feature()} features, not 62."
    )


if (
    xgb_model.num_features()
    !=
    62
):

    raise RuntimeError(
        f"XGBoost expects {xgb_model.num_features()} features, not 62."
    )


print(
    "[PASS] Both frozen models expect 62 features."
)

print()


# ==============================================================================
# 8. PREALLOCATE TARGET OUTPUTS
# ==============================================================================

y_all = np.empty(
    EXPECTED_TARGET_ROWS,
    dtype=np.uint8,
)

p_lgb_all = np.empty(
    EXPECTED_TARGET_ROWS,
    dtype=np.float32,
)

p_xgb_all = np.empty(
    EXPECTED_TARGET_ROWS,
    dtype=np.float32,
)

p_ens_all = np.empty(
    EXPECTED_TARGET_ROWS,
    dtype=np.float32,
)

file_id_all = np.empty(
    EXPECTED_TARGET_ROWS,
    dtype=np.uint8,
)


label_counter = Counter()

file_slices = []

numeric_audit = []


# ==============================================================================
# 9. LABEL CANONICALIZATION — FROZEN RULE
# ==============================================================================

DASH_TRANSLATION = str.maketrans(
    {
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2015": "-",
        "\u2212": "-",
    }
)


def canonicalize_labels(
    series,
):

    if series.isna().any():

        raise RuntimeError(
            "NULL label survived effective-row filter."
        )


    s = (
        series
        .astype(
            "string"
        )
        .str.strip()
        .str.translate(
            DASH_TRANSLATION
        )
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.casefold()
    )


    if s.isna().any():

        raise RuntimeError(
            "Label canonicalization produced NULL."
        )


    if (
        s.str.len()
        ==
        0
    ).any():

        raise RuntimeError(
            "Empty target label encountered."
        )


    return s


# ==============================================================================
# 10. TARGET OPENING
# ==============================================================================

print("=" * 118)
print("TARGET INFERENCE")
print("=" * 118)

print()
print(
    "WARNING: the first model.predict() below CONSUMES PRIMARY TARGET OPENING #1."
)
print()


conn = duckdb.connect(
    database=":memory:"
)


cursor = 0
opening_consumed = False
opening_started_at = None


for file_id, record in enumerate(
    population
):

    remote = record[
        "remote"
    ]

    filename = Path(
        remote
    ).name

    path = (
        TARGET_ROOT
        / filename
    )

    expected_rows = int(
        record[
            "effective_rows"
        ]
    )


    if not path.is_file():

        raise RuntimeError(
            f"Frozen target Parquet missing:\n{path}"
        )


    print(
        "-" * 118
    )

    print(
        f"[{file_id + 1:02d}/08] {record['day']} — {filename}"
    )


    # --------------------------------------------------------------------------
    # Build exact source-ordered bridge62 projection.
    # --------------------------------------------------------------------------

    projection = []


    for source_feature in FEATURE_ORDER:

        physical = PUBLISHED_MAPPING[
            source_feature
        ]


        projection.append(
            f"CAST({qident(physical)} AS DOUBLE) "
            f"AS {qident(source_feature)}"
        )


    projection_sql = ",\n".join(
        projection
    )


    query = f"""
        SELECT
            {projection_sql},
            CAST("Label" AS VARCHAR) AS "__target_label__"
        FROM read_parquet('{sql_path(path)}')
        WHERE "Label" IS NOT NULL
    """


    t_read = time.time()


    df = conn.execute(
        query
    ).df()


    observed_rows = len(
        df
    )


    print(
        "  effective rows:",
        f"{observed_rows:,}",
    )


    if observed_rows != expected_rows:

        raise RuntimeError(
            "\nEffective target rows changed.\n"
            f"File:     {filename}\n"
            f"Expected: {expected_rows:,}\n"
            f"Actual:   {observed_rows:,}"
        )


    X = df[
        FEATURE_ORDER
    ].to_numpy(
        dtype=np.float64,
        copy=True,
    )


    if X.shape != (
        expected_rows,
        62,
    ):

        raise RuntimeError(
            f"Unexpected matrix shape: {X.shape}"
        )


    positive_inf = int(
        np.isposinf(
            X
        ).sum()
    )

    negative_inf = int(
        np.isneginf(
            X
        ).sum()
    )


    # Frozen numeric protocol.
    inf_mask = np.isinf(
        X
    )

    if inf_mask.any():

        X[
            inf_mask
        ] = np.nan


    nan_count = int(
        np.isnan(
            X
        ).sum()
    )


    labels_canon = canonicalize_labels(
        df[
            "__target_label__"
        ]
    )


    local_counts = Counter(
        labels_canon.tolist()
    )

    label_counter.update(
        local_counts
    )


    y = (
        labels_canon
        !=
        "benign"
    ).to_numpy(
        dtype=np.uint8
    )


    start = cursor

    stop = (
        cursor
        +
        expected_rows
    )


    y_all[
        start:stop
    ] = y

    file_id_all[
        start:stop
    ] = np.uint8(
        file_id
    )


    print(
        "  benign:",
        f"{int((y == 0).sum()):,}",
    )

    print(
        "  attack:",
        f"{int((y == 1).sum()):,}",
    )

    print(
        "  +inf -> NaN:",
        f"{positive_inf:,}",
    )

    print(
        "  -inf -> NaN:",
        f"{negative_inf:,}",
    )

    print(
        "  total NaN cells:",
        f"{nan_count:,}",
    )

    print(
        "  load/parse:",
        f"{time.time() - t_read:.2f}s",
    )


    numeric_audit.append(
        {
            "file_id":
                file_id,

            "day":
                record[
                    "day"
                ],

            "remote":
                remote,

            "rows":
                expected_rows,

            "positive_inf_to_nan":
                positive_inf,

            "negative_inf_to_nan":
                negative_inf,

            "nan_cells_after_inf_conversion":
                nan_count,
        }
    )


    # --------------------------------------------------------------------------
    # OPENING CONSUMPTION MARKER
    # --------------------------------------------------------------------------

    if not opening_consumed:

        opening_started_at = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )


        runtime_receipt = {
            "stage":
                "Stage24-2A",

            "direction":
                "IDS2018_TO_CICIDS2017",

            "bridge":
                "bridge62",

            "target_variant":
                "PUBLISHED",

            "opening_number":
                1,

            "opening_budget_original":
                8,

            "opening_consumed":
                True,

            "opening_consumed_at_utc":
                opening_started_at,

            "trigger":
                "FIRST_TARGET_FEATURE_VALUES_SUPPLIED_TO_FROZEN_SOURCE_MODEL",

            "parent_commit":
                EXPECTED_HEAD,
        }


        RUNTIME_LEDGER.write_text(
            json.dumps(
                runtime_receipt,
                indent=2,
                sort_keys=True,
            )
            +
            "\n",
            encoding="utf-8",
        )


        opening_consumed = True


        print()
        print(
            "  >>> TARGET OPENING #1 CONSUMED <<<"
        )

        print()


    # --------------------------------------------------------------------------
    # Frozen LightGBM inference
    # --------------------------------------------------------------------------

    t_pred = time.time()


    p_lgb = np.asarray(
        lgb_model.predict(
            X,
            num_iteration=lgb_model.best_iteration
            if lgb_model.best_iteration > 0
            else None,
        ),
        dtype=np.float64,
    )


    if p_lgb.shape != (
        expected_rows,
    ):

        raise RuntimeError(
            "LightGBM prediction shape mismatch."
        )


    # --------------------------------------------------------------------------
    # Frozen XGBoost inference
    # --------------------------------------------------------------------------

    dmat = xgb.DMatrix(
        X,
        feature_names=FEATURE_ORDER,
        missing=np.nan,
    )


    p_xgb = np.asarray(
        xgb_model.predict(
            dmat,
            validate_features=True,
        ),
        dtype=np.float64,
    )


    if p_xgb.shape != (
        expected_rows,
    ):

        raise RuntimeError(
            "XGBoost prediction shape mismatch."
        )


    if not np.isfinite(
        p_lgb
    ).all():

        raise RuntimeError(
            "Non-finite LightGBM target probability."
        )


    if not np.isfinite(
        p_xgb
    ).all():

        raise RuntimeError(
            "Non-finite XGBoost target probability."
        )


    if (
        (p_lgb < 0).any()
        or
        (p_lgb > 1).any()
    ):

        raise RuntimeError(
            "LightGBM probabilities outside [0,1]."
        )


    if (
        (p_xgb < 0).any()
        or
        (p_xgb > 1).any()
    ):

        raise RuntimeError(
            "XGBoost probabilities outside [0,1]."
        )


    # Frozen ensemble semantics:
    # combine in float64, persist as float32.
    p_ens = (
        0.5
        *
        p_lgb
        +
        0.5
        *
        p_xgb
    ).astype(
        np.float32
    )


    p_lgb_all[
        start:stop
    ] = p_lgb.astype(
        np.float32
    )

    p_xgb_all[
        start:stop
    ] = p_xgb.astype(
        np.float32
    )

    p_ens_all[
        start:stop
    ] = p_ens


    print(
        "  inference:",
        f"{time.time() - t_pred:.2f}s",
    )

    print(
        "  ensemble p min/max:",
        f"{float(p_ens.min()):.9f}",
        "/",
        f"{float(p_ens.max()):.9f}",
    )


    file_slices.append(
        {
            "file_id":
                file_id,

            "day":
                record[
                    "day"
                ],

            "remote":
                remote,

            "start":
                start,

            "stop":
                stop,

            "rows":
                expected_rows,
        }
    )


    cursor = stop


    # Free target features immediately.
    del df
    del X
    del y
    del labels_canon
    del p_lgb
    del p_xgb
    del p_ens
    del dmat

    gc.collect()


conn.close()


if not opening_consumed:

    raise RuntimeError(
        "Target opening was never consumed."
    )


if cursor != EXPECTED_TARGET_ROWS:

    raise RuntimeError(
        "\nFinal target row count mismatch.\n"
        f"Expected: {EXPECTED_TARGET_ROWS:,}\n"
        f"Actual:   {cursor:,}"
    )


print()
print(
    "[PASS] Complete frozen target inference finished."
)

print()


# ==============================================================================
# 11. GLOBAL LABEL AUDIT
# ==============================================================================

n_total = int(
    y_all.size
)

n_attack = int(
    y_all.sum()
)

n_benign = (
    n_total
    -
    n_attack
)

prevalence = (
    n_attack
    /
    n_total
)


if (
    n_attack == 0
    or
    n_benign == 0
):

    raise RuntimeError(
        "Complete target is single-class."
    )


print("=" * 118)
print("TARGET LABEL AUDIT")
print("=" * 118)

print(
    "Rows:",
    f"{n_total:,}",
)

print(
    "Benign:",
    f"{n_benign:,}",
)

print(
    "Attack:",
    f"{n_attack:,}",
)

print(
    "Prevalence:",
    f"{prevalence:.12f}",
)

print()

print(
    "Canonical target label counts:"
)


for label, count in sorted(
    label_counter.items(),
    key=lambda item: (
        item[0]
    ),
):

    print(
        f"  {label!r:42s} {count:10,d}"
    )


print()


# ==============================================================================
# 12. FROZEN METRICS
# ==============================================================================

def threshold_metrics(
    y_true,
    probability,
    threshold,
):

    pred = (
        probability
        >=
        np.float32(
            threshold
        )
    )


    y1 = (
        y_true
        ==
        1
    )

    y0 = ~y1


    tp = int(
        np.sum(
            pred
            &
            y1
        )
    )

    fp = int(
        np.sum(
            pred
            &
            y0
        )
    )

    fn = int(
        np.sum(
            (~pred)
            &
            y1
        )
    )

    tn = int(
        np.sum(
            (~pred)
            &
            y0
        )
    )


    accuracy = (
        (tp + tn)
        /
        (tp + tn + fp + fn)
    )


    precision = (
        tp
        /
        (tp + fp)
        if (
            tp + fp
        ) > 0
        else 0.0
    )


    recall = (
        tp
        /
        (tp + fn)
        if (
            tp + fn
        ) > 0
        else 0.0
    )


    fpr = (
        fp
        /
        (fp + tn)
        if (
            fp + tn
        ) > 0
        else 0.0
    )


    fnr = (
        fn
        /
        (fn + tp)
        if (
            fn + tp
        ) > 0
        else 0.0
    )


    f1_denom = (
        2 * tp
        +
        fp
        +
        fn
    )

    f1 = (
        2 * tp
        /
        f1_denom
        if f1_denom > 0
        else 0.0
    )


    f2_denom = (
        5 * tp
        +
        4 * fn
        +
        fp
    )

    f2 = (
        5 * tp
        /
        f2_denom
        if f2_denom > 0
        else 0.0
    )


    return {
        "threshold":
            float(
                np.float32(
                    threshold
                )
            ),

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn,

        "accuracy":
            float(
                accuracy
            ),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                f1
            ),

        "f2":
            float(
                f2
            ),

        "fpr":
            float(
                fpr
            ),

        "fnr":
            float(
                fnr
            ),
    }


def calibration_metrics(
    y_true,
    probability,
):

    p64 = probability.astype(
        np.float64,
        copy=False,
    )

    y64 = y_true.astype(
        np.float64,
        copy=False,
    )


    brier = float(
        np.mean(
            (
                p64
                -
                y64
            )
            **
            2
        )
    )


    clipped = np.clip(
        p64,
        1e-15,
        1.0
        -
        1e-15,
    )


    logloss = float(
        -np.mean(
            y64
            *
            np.log(
                clipped
            )
            +
            (
                1.0
                -
                y64
            )
            *
            np.log(
                1.0
                -
                clipped
            )
        )
    )


    # Fixed bins:
    # [0,.1), [.1,.2), ..., [.9,1.0]
    bins = np.floor(
        p64
        *
        10.0
    ).astype(
        np.int8
    )

    bins = np.clip(
        bins,
        0,
        9,
    )


    ece = 0.0

    ece_bins = []


    for b in range(
        10
    ):

        mask = (
            bins
            ==
            b
        )

        count = int(
            mask.sum()
        )


        if count == 0:

            ece_bins.append(
                {
                    "bin":
                        b,

                    "count":
                        0,

                    "mean_probability":
                        None,

                    "empirical_rate":
                        None,

                    "absolute_gap":
                        None,
                }
            )

            continue


        conf = float(
            p64[
                mask
            ].mean()
        )

        rate = float(
            y64[
                mask
            ].mean()
        )

        gap = abs(
            conf
            -
            rate
        )


        ece += (
            count
            /
            len(
                p64
            )
        ) * gap


        ece_bins.append(
            {
                "bin":
                    b,

                "count":
                    count,

                "mean_probability":
                    conf,

                "empirical_rate":
                    rate,

                "absolute_gap":
                    float(
                        gap
                    ),
            }
        )


    return {
        "brier":
            brier,

        "log_loss":
            logloss,

        "ece_10":
            float(
                ece
            ),

        "ece_bins":
            ece_bins,
    }


print("=" * 118)
print("FROZEN PRIMARY METRICS")
print("=" * 118)


metric_start = time.time()


pr_auc = float(
    average_precision_score(
        y_all,
        p_ens_all,
    )
)


roc_auc = float(
    roc_auc_score(
        y_all,
        p_ens_all,
    )
)


pr_excess = (
    pr_auc
    -
    prevalence
)


pr_normalized = (
    (
        pr_auc
        -
        prevalence
    )
    /
    (
        1.0
        -
        prevalence
    )
)


standard_metrics = threshold_metrics(
    y_all,
    p_ens_all,
    STANDARD_THRESHOLD,
)

balanced_metrics = threshold_metrics(
    y_all,
    p_ens_all,
    BALANCED_THRESHOLD,
)

security_metrics = threshold_metrics(
    y_all,
    p_ens_all,
    SECURITY_THRESHOLD,
)


calibration = calibration_metrics(
    y_all,
    p_ens_all,
)


print(
    "PR-AUC:       ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:      ",
    f"{roc_auc:.15f}",
)

print(
    "Prevalence:   ",
    f"{prevalence:.15f}",
)

print(
    "PR excess:    ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized:",
    f"{pr_normalized:.15f}",
)

print()

print(
    "Brier:",
    f"{calibration['brier']:.15f}",
)

print(
    "Log loss:",
    f"{calibration['log_loss']:.15f}",
)

print(
    "ECE-10:",
    f"{calibration['ece_10']:.15f}",
)

print()

print(
    "STANDARD @",
    standard_metrics[
        "threshold"
    ],
)

print(
    "  TP/TN/FP/FN:",
    standard_metrics[
        "tp"
    ],
    standard_metrics[
        "tn"
    ],
    standard_metrics[
        "fp"
    ],
    standard_metrics[
        "fn"
    ],
)

print(
    "  precision/recall/F1/F2/FPR/FNR:",
    f"{standard_metrics['precision']:.9f}",
    f"{standard_metrics['recall']:.9f}",
    f"{standard_metrics['f1']:.9f}",
    f"{standard_metrics['f2']:.9f}",
    f"{standard_metrics['fpr']:.9f}",
    f"{standard_metrics['fnr']:.9f}",
)

print()

print(
    "BALANCED @",
    balanced_metrics[
        "threshold"
    ],
)

print(
    "  TP/TN/FP/FN:",
    balanced_metrics[
        "tp"
    ],
    balanced_metrics[
        "tn"
    ],
    balanced_metrics[
        "fp"
    ],
    balanced_metrics[
        "fn"
    ],
)

print(
    "  precision/recall/F1/F2/FPR/FNR:",
    f"{balanced_metrics['precision']:.9f}",
    f"{balanced_metrics['recall']:.9f}",
    f"{balanced_metrics['f1']:.9f}",
    f"{balanced_metrics['f2']:.9f}",
    f"{balanced_metrics['fpr']:.9f}",
    f"{balanced_metrics['fnr']:.9f}",
)

print()

print(
    "SECURITY @",
    security_metrics[
        "threshold"
    ],
)

print(
    "  TP/TN/FP/FN:",
    security_metrics[
        "tp"
    ],
    security_metrics[
        "tn"
    ],
    security_metrics[
        "fp"
    ],
    security_metrics[
        "fn"
    ],
)

print(
    "  precision/recall/F1/F2/FPR/FNR:",
    f"{security_metrics['precision']:.9f}",
    f"{security_metrics['recall']:.9f}",
    f"{security_metrics['f1']:.9f}",
    f"{security_metrics['f2']:.9f}",
    f"{security_metrics['fpr']:.9f}",
    f"{security_metrics['fnr']:.9f}",
)

print()

print(
    "Metric computation:",
    f"{time.time() - metric_start:.2f}s",
)

print()


# ==============================================================================
# 13. FILE-LEVEL DESCRIPTIVE METRICS
# ==============================================================================

print("=" * 118)
print("FILE-LEVEL DESCRIPTIVE TRANSFER")
print("=" * 118)


file_metrics = []


for info in file_slices:

    start = int(
        info[
            "start"
        ]
    )

    stop = int(
        info[
            "stop"
        ]
    )


    yy = y_all[
        start:stop
    ]

    pp = p_ens_all[
        start:stop
    ]


    attacks = int(
        yy.sum()
    )

    benign = int(
        len(
            yy
        )
        -
        attacks
    )

    pi = float(
        attacks
        /
        len(
            yy
        )
    )


    if (
        attacks > 0
        and
        benign > 0
    ):

        local_pr = float(
            average_precision_score(
                yy,
                pp,
            )
        )

        local_roc = float(
            roc_auc_score(
                yy,
                pp,
            )
        )

    else:

        local_pr = None
        local_roc = None


    item = {
        **info,

        "benign":
            benign,

        "attack":
            attacks,

        "prevalence":
            pi,

        "pr_auc":
            local_pr,

        "roc_auc":
            local_roc,
    }


    file_metrics.append(
        item
    )


    print(
        f"[{info['file_id']:02d}] "
        f"{info['day']:10s} "
        f"rows={len(yy):9,d} "
        f"attack={attacks:8,d} "
        f"pi={pi:.6f} "
        f"PR={local_pr if local_pr is not None else 'NA'} "
        f"ROC={local_roc if local_roc is not None else 'NA'}"
    )


print()


# ==============================================================================
# 14. PREDICTION HASHES
# ==============================================================================

print("=" * 118)
print("PREDICTION BYTE IDENTITIES")
print("=" * 118)


lgb_probability_sha = sha256_array(
    p_lgb_all
)

xgb_probability_sha = sha256_array(
    p_xgb_all
)

ensemble_probability_sha = sha256_array(
    p_ens_all
)

binary_label_sha = sha256_array(
    y_all
)


print(
    "LightGBM float32 SHA:",
    lgb_probability_sha,
)

print(
    "XGBoost  float32 SHA:",
    xgb_probability_sha,
)

print(
    "Ensemble float32 SHA:",
    ensemble_probability_sha,
)

print(
    "Binary label SHA:    ",
    binary_label_sha,
)

print()


# ==============================================================================
# 15. PERSIST OPENING ARTIFACT
# ==============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


print("=" * 118)
print("PERSISTING TARGET OPENING")
print("=" * 118)


save_start = time.time()


np.savez_compressed(
    PRED_PATH,

    probability=
        p_ens_all,

    binary_label=
        y_all,

    file_id=
        file_id_all,
)


pred_file_sha = sha256_file(
    PRED_PATH
)


print(
    "Prediction artifact:",
    PRED_PATH.relative_to(
        REPO
    ),
)

print(
    "Prediction bytes:",
    f"{PRED_PATH.stat().st_size:,}",
)

print(
    "Prediction file SHA:",
    pred_file_sha,
)

print(
    "Save seconds:",
    f"{time.time() - save_start:.2f}",
)

print()


# ==============================================================================
# 16. RESULT RECEIPT
# ==============================================================================

source_prior_probability = float(
    final_lock[
        "null_protocol"
    ][
        "source_prior_constant_predictor"
    ][
        "primary_source_probability"
    ]
)


result = {
    "stage":
        "Stage24-2A",

    "status":
        "PRIMARY_BRIDGE62_PUBLISHED_TARGET_OPENING_COMPLETE",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "direction":
        "IDS2018_TO_CICIDS2017",

    "bridge":
        "bridge62",

    "target_variant":
        "PUBLISHED",

    "parent_commit":
        EXPECTED_HEAD,

    "target_opening": {
        "opening_number":
            1,

        "original_budget":
            8,

        "consumed_before":
            0,

        "consumed_after":
            1,

        "administratively_cancelled_grounded_cells":
            2,

        "opening_definition":
            final_lock[
                "opening_definition"
            ],

        "opening_consumed_at_utc":
            opening_started_at,
    },

    "anti_adaptation": {
        "target_fit":
            False,

        "target_threshold_selection":
            False,

        "target_feature_search":
            False,

        "target_feature_remapping":
            False,

        "target_imputation_fit":
            False,

        "target_scaling_fit":
            False,

        "target_calibration":
            False,

        "hyperparameter_change":
            False,

        "source_models_reused_exactly":
            True,
    },

    "representation": {
        "feature_count":
            62,

        "feature_order":
            FEATURE_ORDER,

        "target_mapping":
            PUBLISHED_MAPPING,

        "parse_dtype":
            "float64",

        "positive_infinity":
            "CONVERT_TO_NAN",

        "negative_infinity":
            "CONVERT_TO_NAN",

        "explicit_imputation":
            "NONE",

        "scaling":
            "NONE",
    },

    "models": {
        "lightgbm": {
            "model_file":
                str(
                    LGB_MODEL_PATH.relative_to(
                        REPO
                    )
                ),

            "model_sha256":
                lgb_model_sha,

            "target_probability_float32_sha256":
                lgb_probability_sha,
        },

        "xgboost": {
            "model_file":
                str(
                    XGB_MODEL_PATH.relative_to(
                        REPO
                    )
                ),

            "model_sha256":
                xgb_model_sha,

            "target_probability_float32_sha256":
                xgb_probability_sha,
        },

        "ensemble": {
            "rule":
                "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

            "combination_dtype":
                "float64",

            "storage_dtype":
                "float32",

            "target_probability_sha256":
                ensemble_probability_sha,
        },
    },

    "target_population": {
        "rows":
            n_total,

        "benign":
            n_benign,

        "attack":
            n_attack,

        "prevalence":
            prevalence,

        "binary_label_sha256":
            binary_label_sha,

        "effective_row_rule":
            "FROZEN_STAGE24_EFFECTIVE_CICIDS2017_POPULATION",

        "canonical_label_counts":
            dict(
                sorted(
                    label_counter.items()
                )
            ),

        "files":
            file_slices,
    },

    "numeric_audit":
        numeric_audit,

    "metrics": {
        "primary": {
            "pr_auc":
                pr_auc,

            "roc_auc":
                roc_auc,

            "prevalence_chance_anchor":
                prevalence,

            "pr_excess":
                pr_excess,

            "pr_normalized":
                pr_normalized,
        },

        "calibration":
            calibration,

        "thresholded": {
            "standard":
                standard_metrics,

            "balanced":
                balanced_metrics,

            "security":
                security_metrics,
        },

        "file_level_descriptive":
            file_metrics,
    },

    "null_anchors": {
        "target_prevalence_probability":
            prevalence,

        "target_pr_auc_chance_anchor":
            prevalence,

        "target_roc_auc_chance_anchor":
            0.5,

        "source_prior_constant_probability":
            source_prior_probability,
    },

    "threshold_provenance": {
        "source":
            "IDS2018_SOURCE_VALIDATION_ONLY",

        "standard":
            STANDARD_THRESHOLD,

        "balanced":
            BALANCED_THRESHOLD,

        "security":
            SECURITY_THRESHOLD,

        "target_threshold_search":
            False,
    },

    "prediction_artifact": {
        "file":
            str(
                PRED_PATH.relative_to(
                    REPO
                )
            ),

        "file_sha256":
            pred_file_sha,

        "probability_array_sha256":
            ensemble_probability_sha,

        "binary_label_array_sha256":
            binary_label_sha,

        "row_order":
            "FROZEN_PUBLISHED_POPULATION_ORDER",
    },

    "bootstrap": {
        "performed":
            False,

        "reason":
            (
                "Paired stratified bootstrap is deferred until the "
                "comparison prediction vector exists; no target adaptation occurs."
            ),

        "frozen_replicates":
            2000,

        "frozen_seed":
            42,
    },

    "scientific_fit_accounting": {
        "stage24_fit_budget":
            4,

        "completed_before":
            2,

        "completed_after":
            2,

        "new_fits":
            0,
    },
}


RESULT_PATH.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


result_sha = sha256_file(
    RESULT_PATH
)


RESULT_SHA_PATH.write_text(
    f"{result_sha}  {RESULT_PATH.name}\n",
    encoding="utf-8",
)


CHECKSUMS_PATH.write_text(
    (
        f"{result_sha}  {RESULT_PATH.name}\n"
        f"{pred_file_sha}  {PRED_PATH.name}\n"
    ),
    encoding="utf-8",
)


print(
    "Result:",
    RESULT_PATH.relative_to(
        REPO
    ),
)

print(
    "Result SHA256:",
    result_sha,
)

print()


# ==============================================================================
# 17. GIT FREEZE
# ==============================================================================

print("=" * 118)
print("GIT FREEZE")
print("=" * 118)


git_cmd(
    "add",
    "--",
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


expected_staged = {
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged file set.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: open primary bridge62 published target",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_HEAD:

    raise RuntimeError(
        "\nTarget-opening commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 18. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 118)
print("PUSH + REMOTE VERIFY")
print("=" * 118)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository not clean after freeze:\n"
        + final_status
    )


# Opening is now durable in GitHub.
if RUNTIME_LEDGER.exists():

    RUNTIME_LEDGER.unlink()


print(
    "[PASS] Remote main == local opening commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 19. FINAL
# ==============================================================================

print("=" * 118)
print("STAGE24-2A PRIMARY TARGET OPENING #1: PASS")
print("=" * 118)

print()

print(
    "Direction:                   IDS2018 -> CICIDS2017"
)

print(
    "Bridge:                      bridge62"
)

print(
    "Variant:                     PUBLISHED"
)

print()

print(
    "Target rows:                 ",
    f"{n_total:,}",
)

print(
    "Benign:                      ",
    f"{n_benign:,}",
)

print(
    "Attack:                      ",
    f"{n_attack:,}",
)

print(
    "Prevalence:                  ",
    f"{prevalence:.12f}",
)

print()

print(
    "PR-AUC:                      ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:                     ",
    f"{roc_auc:.15f}",
)

print(
    "PR excess:                   ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized:               ",
    f"{pr_normalized:.15f}",
)

print()

print(
    "Brier:                       ",
    f"{calibration['brier']:.15f}",
)

print(
    "Log loss:                    ",
    f"{calibration['log_loss']:.15f}",
)

print(
    "ECE-10:                      ",
    f"{calibration['ece_10']:.15f}",
)

print()

print(
    "Target fits performed:        0"
)

print(
    "Target threshold tuning:      NO"
)

print(
    "Scientific fits:              2 / 4"
)

print()

print(
    "Target openings consumed:     1 / 8"
)

print(
    "GROUNDED_S4 cancelled cells:  2"
)

print()

print(
    "Ensemble probability SHA:"
)

print(
    " ",
    ensemble_probability_sha,
)

print()

print(
    "Result SHA:"
)

print(
    " ",
    result_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "NEXT:"
)

print(
    "  bridge62 / FLAG_CORRECTED identity closure"
)

print(
    "  — no refit and no mapping change."
)

print("=" * 118)

STAGE24-2A — PRIMARY TARGET OPENING #1 — bridge62 / PUBLISHED

PRE-OPENING GOVERNANCE
HEAD: a90a4221ef5a5e9958777049bcf13cc56a3519d3
[PASS] Repository clean.
[PASS] No prior runtime opening marker.
[PASS] No prior Stage24-2A artifacts.

FROZEN ARTIFACT VERIFICATION
LightGBM model SHA: 907c9c86755c3c32effb2d0899f92e1a0cce71915f90311c99afd8c73814c4d7
XGBoost model SHA:  b30966f34d6f0799657c528344ec540badef7f3ca3dd207abdaa4b45400e6231
[PASS] Both frozen bridge62 model identities exact.

Frozen STANDARD threshold: 0.5
Frozen BALANCED threshold: 0.05000000074505806
Frozen SECURITY threshold: 0.05000000074505806
[PASS] Thresholds frozen from IDS2018 source validation only.

FROZEN TARGET ORDER
[01] Monday     Monday-WorkingHours.pcap_ISCX.csv.parquet effective=529,918
[02] Tuesday    Tuesday-WorkingHours.pcap_ISCX.csv.parquet effective=445,909
[03] Wednesday  Wednesday-workingHours.pcap_ISCX.csv.parquet effective=692,703
[04] Thursday   Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  effective rows: 529,918
  benign: 529,918
  attack: 0
  +inf -> NaN: 810
  -inf -> NaN: 0
  total NaN cells: 874
  load/parse: 5.17s

  >>> TARGET OPENING #1 CONSUMED <<<

  inference: 11.72s
  ensemble p min/max: 0.000000016 / 0.999957681
----------------------------------------------------------------------------------------------------------------------
[02/08] Tuesday — Tuesday-WorkingHours.pcap_ISCX.csv.parquet
  effective rows: 445,909
  benign: 432,074
  attack: 13,835
  +inf -> NaN: 327
  -inf -> NaN: 0
  total NaN cells: 528
  load/parse: 2.34s
  inference: 10.06s
  ensemble p min/max: 0.000000016 / 0.999958992
----------------------------------------------------------------------------------------------------------------------
[03/08] Wednesday — Wednesday-workingHours.pcap_ISCX.csv.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  effective rows: 692,703
  benign: 440,031
  attack: 252,672
  +inf -> NaN: 1,586
  -inf -> NaN: 0
  total NaN cells: 2,594
  load/parse: 5.09s
  inference: 16.86s
  ensemble p min/max: 0.000000016 / 0.999959469
----------------------------------------------------------------------------------------------------------------------
[04/08] Thursday — Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet
  effective rows: 288,602
  benign: 288,566
  attack: 36
  +inf -> NaN: 396
  -inf -> NaN: 0
  total NaN cells: 414
  load/parse: 1.59s
  inference: 6.08s
  ensemble p min/max: 0.000000016 / 0.999954760
----------------------------------------------------------------------------------------------------------------------
[05/08] Thursday — Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet
  effective rows: 170,366
  benign: 168,186
  attack: 2,180
  +inf -> NaN: 250
  -inf -> NaN: 0
  total NaN cells: 270
  load/parse: 0.94s
  inference: 3.79s
  ensemble p min/ma

In [33]:
# ==============================================================================
# STAGE24-2B — PRIMARY TARGET OPENING #2
# IDS2018 -> CICIDS2017
# bridge62 / FLAG_CORRECTED
#
# PURPOSE
# -------
# Frozen bridge62 excludes all eight aggregate flag-count fields.
# Therefore:
#
#     PUBLISHED feature projection
#          ==
#     FLAG_CORRECTED feature projection
#
# and consequently the complete target predictions MUST be bitwise identical
# to Stage24-2A.
#
# THIS CELL CONSUMES TARGET OPENING #2 when the first corrected target matrix
# is supplied to the frozen source-domain models.
#
# NO FIT.
# NO TUNING.
# NO TARGET THRESHOLD SEARCH.
# NO NEW FEATURE MAPPING.
#
# Expected invariant:
#
#     corrected ensemble SHA256
#       ==
#     e9288ba2590e09ae51da183bff2160fbb501e8633a3af87a22c800f7d3a051d7
#
# Any mismatch is an IMPLEMENTATION FAILURE.
# ==============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import duckdb


print("=" * 118)
print("STAGE24-2B — PRIMARY TARGET OPENING #2 — bridge62 / FLAG_CORRECTED")
print("=" * 118)
print()


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "e1d953f12245f46e136a6a4a521ee8b0f4f8dace"
)

EXPECTED_TARGET_ROWS = 2_830_743

EXPECTED_PUBLISHED_ENSEMBLE_SHA = (
    "e9288ba2590e09ae51da183bff2160fbb501e8633a3af87a22c800f7d3a051d7"
)

EXPECTED_PUBLISHED_BINARY_LABEL_SHA = (
    "0e4edd0670c9cedde571e08ab92fedbe6acd6e2aa605d7fc62d286791c00740a"
)

EXPECTED_LGBM_MODEL_SHA = (
    "907c9c86755c3c32effb2d0899f92e1a0cce71915f90311c99afd8c73814c4d7"
)

EXPECTED_XGB_MODEL_SHA = (
    "b30966f34d6f0799657c528344ec540badef7f3ca3dd207abdaa4b45400e6231"
)


# ==============================================================================
# 1. PATHS
# ==============================================================================

LOCK_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
)

BRIDGE_SPEC_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.json"
)

FINAL_LOCK_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.json"
)


SOURCE_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1b_bridge62_source_refit"
)

LGB_MODEL_PATH = (
    SOURCE_DIR
    / "bridge62_lightgbm_model.txt"
)

XGB_MODEL_PATH = (
    SOURCE_DIR
    / "bridge62_xgboost_model.json"
)


PUBLISHED_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_2_primary_target_openings"
    / "stage24_2a_bridge62_published"
)

PUBLISHED_RESULT_PATH = (
    PUBLISHED_DIR
    / "stage24_2a_bridge62_published_result.json"
)

PUBLISHED_RESULT_SHA_PATH = (
    PUBLISHED_DIR
    / "stage24_2a_bridge62_published_result.sha256"
)

PUBLISHED_PRED_PATH = (
    PUBLISHED_DIR
    / "bridge62_published_ensemble_predictions.npz"
)


OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_2_primary_target_openings"
    / "stage24_2b_bridge62_flag_corrected"
)

RESULT_PATH = (
    OUT
    / "stage24_2b_bridge62_flag_corrected_identity_result.json"
)

RESULT_SHA_PATH = (
    OUT
    / "stage24_2b_bridge62_flag_corrected_identity_result.sha256"
)


TARGET_ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/"
    "snapshots/b7e532345512edcd530cb1770dc76636aeb52802/"
    "traffic_labels"
)


RUNTIME_LEDGER = Path(
    "/kaggle/working/stage24_target_opening_runtime_ledger.json"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    cmd = [
        "git"
    ]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [
        str(x)
        for x in args
    ]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_array(
    array,
):

    arr = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        arr.view(
            np.uint8
        )
    ).hexdigest()


def qident(
    value,
):

    return (
        '"'
        +
        str(
            value
        ).replace(
            '"',
            '""',
        )
        +
        '"'
    )


def sql_path(
    path,
):

    return str(
        path
    ).replace(
        "'",
        "''",
    )


# ==============================================================================
# 3. PRE-OPENING GOVERNANCE
# ==============================================================================

print("=" * 118)
print("PRE-OPENING GOVERNANCE")
print("=" * 118)


head = git_cmd(
    "rev-parse",
    "HEAD",
)

status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


if RUNTIME_LEDGER.exists():

    raise RuntimeError(
        "\nA target-opening runtime ledger already exists:\n"
        f"{RUNTIME_LEDGER}\n\n"
        "DO NOT rerun an opening blindly."
    )


if OUT.exists() and any(
    OUT.iterdir()
):

    raise RuntimeError(
        "\nStage24-2B output already exists.\n"
        "Refusing duplicate target opening."
    )


print(
    "[PASS] Repository clean."
)

print(
    "[PASS] No unresolved target-opening marker."
)

print(
    "[PASS] Stage24-2B has not previously been opened."
)

print()


# ==============================================================================
# 4. VERIFY STAGE24-2A
# ==============================================================================

print("=" * 118)
print("STAGE24-2A REFERENCE VERIFICATION")
print("=" * 118)


for path in [
    PUBLISHED_RESULT_PATH,
    PUBLISHED_RESULT_SHA_PATH,
    PUBLISHED_PRED_PATH,
]:

    if not path.is_file():

        raise RuntimeError(
            f"Missing Stage24-2A artifact:\n{path}"
        )


published_result_sha = sha256_file(
    PUBLISHED_RESULT_PATH
)

expected_result_sha = (
    PUBLISHED_RESULT_SHA_PATH
    .read_text(
        encoding="utf-8"
    )
    .strip()
    .split()[0]
)


if published_result_sha != expected_result_sha:

    raise RuntimeError(
        "Stage24-2A result SHA sidecar mismatch."
    )


published_result = json.loads(
    PUBLISHED_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


if published_result[
    "status"
] != "PRIMARY_BRIDGE62_PUBLISHED_TARGET_OPENING_COMPLETE":

    raise RuntimeError(
        "Unexpected Stage24-2A status."
    )


if int(
    published_result[
        "target_opening"
    ][
        "consumed_after"
    ]
) != 1:

    raise RuntimeError(
        "Stage24-2A opening ledger is not 1."
    )


if int(
    published_result[
        "target_population"
    ][
        "rows"
    ]
) != EXPECTED_TARGET_ROWS:

    raise RuntimeError(
        "Stage24-2A target population changed."
    )


if (
    published_result[
        "models"
    ][
        "ensemble"
    ][
        "target_probability_sha256"
    ]
    !=
    EXPECTED_PUBLISHED_ENSEMBLE_SHA
):

    raise RuntimeError(
        "Stage24-2A ensemble SHA differs from frozen value."
    )


if (
    published_result[
        "target_population"
    ][
        "binary_label_sha256"
    ]
    !=
    EXPECTED_PUBLISHED_BINARY_LABEL_SHA
):

    raise RuntimeError(
        "Stage24-2A binary-label SHA differs."
    )


published_npz = np.load(
    PUBLISHED_PRED_PATH,
    allow_pickle=False,
)


published_probability = np.asarray(
    published_npz[
        "probability"
    ],
    dtype=np.float32,
)


published_labels = np.asarray(
    published_npz[
        "binary_label"
    ],
    dtype=np.uint8,
)


published_file_id = np.asarray(
    published_npz[
        "file_id"
    ],
    dtype=np.uint8,
)


if published_probability.shape != (
    EXPECTED_TARGET_ROWS,
):

    raise RuntimeError(
        "Published probability shape mismatch."
    )


if published_labels.shape != (
    EXPECTED_TARGET_ROWS,
):

    raise RuntimeError(
        "Published binary-label shape mismatch."
    )


if sha256_array(
    published_probability
) != EXPECTED_PUBLISHED_ENSEMBLE_SHA:

    raise RuntimeError(
        "Published probability array SHA mismatch."
    )


if sha256_array(
    published_labels
) != EXPECTED_PUBLISHED_BINARY_LABEL_SHA:

    raise RuntimeError(
        "Published label array SHA mismatch."
    )


print(
    "Stage24-2A result SHA:",
    published_result_sha,
)

print(
    "Published ensemble SHA:",
    EXPECTED_PUBLISHED_ENSEMBLE_SHA,
)

print()

print(
    "[PASS] Stage24-2A reference opening exact."
)

print()


# ==============================================================================
# 5. FROZEN BRIDGE IDENTITY
# ==============================================================================

print("=" * 118)
print("FROZEN BRIDGE62 IDENTITY")
print("=" * 118)


bridge_spec = json.loads(
    BRIDGE_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)

final_lock = json.loads(
    FINAL_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


b62 = bridge_spec[
    "bridge62"
]


FEATURE_ORDER = list(
    b62[
        "source_feature_order"
    ]
)


PUBLISHED_MAPPING = dict(
    b62[
        "PUBLISHED_mapping"
    ]
)


CORRECTED_MAPPING = dict(
    b62[
        "FLAG_CORRECTED_mapping"
    ]
)


if len(
    FEATURE_ORDER
) != 62:

    raise RuntimeError(
        "Frozen bridge62 feature count changed."
    )


if PUBLISHED_MAPPING != CORRECTED_MAPPING:

    raise RuntimeError(
        "\nIMPLEMENTATION FAILURE:\n"
        "bridge62 PUBLISHED and FLAG_CORRECTED mappings are not identical."
    )


if (
    final_lock[
        "comparison_protocol"
    ][
        "bridge62_flag_invariance"
    ][
        "expected"
    ]
    !=
    "BITWISE_IDENTICAL_FEATURE_MATRICES_AND_PREDICTIONS"
):

    raise RuntimeError(
        "Frozen bridge62 invariance contract changed."
    )


print(
    "Feature count:",
    len(
        FEATURE_ORDER
    ),
)

print(
    "Mapping differences:",
    0,
)

print()

print(
    "[PASS] Corrected bridge62 projection is identical by frozen specification."
)

print()


# ==============================================================================
# 6. VERIFY MODEL IDENTITIES
# ==============================================================================

print("=" * 118)
print("FROZEN MODEL IDENTITIES")
print("=" * 118)


lgb_sha = sha256_file(
    LGB_MODEL_PATH
)

xgb_sha = sha256_file(
    XGB_MODEL_PATH
)


print(
    "LightGBM:",
    lgb_sha,
)

print(
    "XGBoost: ",
    xgb_sha,
)


if lgb_sha != EXPECTED_LGBM_MODEL_SHA:

    raise RuntimeError(
        "LightGBM model identity changed."
    )


if xgb_sha != EXPECTED_XGB_MODEL_SHA:

    raise RuntimeError(
        "XGBoost model identity changed."
    )


print(
    "[PASS] Frozen bridge62 models exact."
)

print()


# ==============================================================================
# 7. GITHUB CREDENTIAL + REMOTE GATE
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if value and value.strip():

            github_token = value.strip()
            token_source = f"ENV:{name}"
            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable before target opening."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved unexpectedly.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main == Stage24-2A."
)

print()


# ==============================================================================
# 8. LOAD MODELS
# ==============================================================================

import lightgbm as lgb
import xgboost as xgb


print("=" * 118)
print("MODEL LOAD")
print("=" * 118)


print(
    "LightGBM:",
    lgb.__version__,
)

print(
    "XGBoost: ",
    xgb.__version__,
)


lgb_model = lgb.Booster(
    model_file=str(
        LGB_MODEL_PATH
    )
)


xgb_model = xgb.Booster()

xgb_model.load_model(
    str(
        XGB_MODEL_PATH
    )
)


try:

    xgb_model.set_param(
        {
            "device":
                "cpu",

            "nthread":
                -1,
        }
    )

except Exception:
    pass


if lgb_model.num_feature() != 62:

    raise RuntimeError(
        "LightGBM feature count mismatch."
    )


if xgb_model.num_features() != 62:

    raise RuntimeError(
        "XGBoost feature count mismatch."
    )


print(
    "[PASS] Frozen models loaded."
)

print()


# ==============================================================================
# 9. FROZEN POPULATION
# ==============================================================================

population = list(
    final_lock[
        "published_population"
    ]
)


if len(
    population
) != 8:

    raise RuntimeError(
        "Frozen population does not contain eight files."
    )


if sum(
    int(
        x[
            "effective_rows"
        ]
    )
    for x in population
) != EXPECTED_TARGET_ROWS:

    raise RuntimeError(
        "Frozen effective population changed."
    )


# ==============================================================================
# 10. LABEL CANONICALIZATION
# ==============================================================================

DASH_TRANSLATION = str.maketrans(
    {
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2015": "-",
        "\u2212": "-",
    }
)


def canonicalize_labels(
    series,
):

    if series.isna().any():

        raise RuntimeError(
            "NULL label survived effective-row filter."
        )


    s = (
        series
        .astype(
            "string"
        )
        .str.strip()
        .str.translate(
            DASH_TRANSLATION
        )
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.casefold()
    )


    if s.isna().any():

        raise RuntimeError(
            "Label canonicalization generated NULL."
        )


    if (
        s.str.len()
        ==
        0
    ).any():

        raise RuntimeError(
            "Empty target label."
        )


    return s


# ==============================================================================
# 11. CORRECTED TARGET OPENING
# ==============================================================================

print("=" * 118)
print("FLAG_CORRECTED TARGET INFERENCE")
print("=" * 118)

print()
print(
    "First model.predict() consumes PRIMARY TARGET OPENING #2."
)

print()


conn = duckdb.connect(
    database=":memory:"
)


corrected_probability = np.empty(
    EXPECTED_TARGET_ROWS,
    dtype=np.float32,
)


corrected_labels = np.empty(
    EXPECTED_TARGET_ROWS,
    dtype=np.uint8,
)


feature_hashes = []

cursor = 0

opening_consumed = False
opening_started_at = None


for file_id, record in enumerate(
    population
):

    filename = Path(
        record[
            "remote"
        ]
    ).name


    path = (
        TARGET_ROOT
        /
        filename
    )


    expected_rows = int(
        record[
            "effective_rows"
        ]
    )


    if not path.is_file():

        raise RuntimeError(
            f"Missing target Parquet:\n{path}"
        )


    projection_sql = ",\n".join(
        (
            f"CAST({qident(CORRECTED_MAPPING[source_feature])} AS DOUBLE) "
            f"AS {qident(source_feature)}"
        )
        for source_feature in FEATURE_ORDER
    )


    query = f"""
        SELECT
            {projection_sql},
            CAST("Label" AS VARCHAR) AS "__target_label__"
        FROM read_parquet('{sql_path(path)}')
        WHERE "Label" IS NOT NULL
    """


    t0 = time.time()


    df = conn.execute(
        query
    ).df()


    if len(
        df
    ) != expected_rows:

        raise RuntimeError(
            "\nEffective target-row mismatch.\n"
            f"File: {filename}\n"
            f"Expected: {expected_rows:,}\n"
            f"Actual: {len(df):,}"
        )


    X = df[
        FEATURE_ORDER
    ].to_numpy(
        dtype=np.float64,
        copy=True,
    )


    # Frozen +/-inf handling.
    inf_mask = np.isinf(
        X
    )

    if inf_mask.any():

        X[
            inf_mask
        ] = np.nan


    # Feature matrix fingerprint before model inference.
    X_hash = sha256_array(
        X
    )


    feature_hashes.append(
        {
            "file_id":
                file_id,

            "day":
                record[
                    "day"
                ],

            "rows":
                expected_rows,

            "corrected_feature_matrix_float64_sha256":
                X_hash,
        }
    )


    labels_canon = canonicalize_labels(
        df[
            "__target_label__"
        ]
    )


    y = (
        labels_canon
        !=
        "benign"
    ).to_numpy(
        dtype=np.uint8
    )


    start = cursor

    stop = (
        cursor
        +
        expected_rows
    )


    # Labels must reproduce Stage24-2A exactly.
    if not np.array_equal(
        y,
        published_labels[
            start:stop
        ],
    ):

        raise RuntimeError(
            "\nIMPLEMENTATION FAILURE:\n"
            f"Binary target labels differ from Stage24-2A in {filename}."
        )


    if not np.all(
        published_file_id[
            start:stop
        ]
        ==
        np.uint8(
            file_id
        )
    ):

        raise RuntimeError(
            "Stage24-2A file ordering changed."
        )


    corrected_labels[
        start:stop
    ] = y


    # --------------------------------------------------------------------------
    # OPENING #2 CONSUMPTION
    # --------------------------------------------------------------------------

    if not opening_consumed:

        opening_started_at = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )


        RUNTIME_LEDGER.write_text(
            json.dumps(
                {
                    "stage":
                        "Stage24-2B",

                    "direction":
                        "IDS2018_TO_CICIDS2017",

                    "bridge":
                        "bridge62",

                    "target_variant":
                        "FLAG_CORRECTED",

                    "opening_number":
                        2,

                    "opening_consumed":
                        True,

                    "opening_consumed_at_utc":
                        opening_started_at,

                    "trigger":
                        (
                            "FIRST_FLAG_CORRECTED_TARGET_FEATURE_VALUES_"
                            "SUPPLIED_TO_FROZEN_SOURCE_MODEL"
                        ),

                    "parent_commit":
                        EXPECTED_PARENT,
                },
                indent=2,
                sort_keys=True,
            )
            +
            "\n",
            encoding="utf-8",
        )


        opening_consumed = True


        print(
            ">>> TARGET OPENING #2 CONSUMED <<<"
        )

        print()


    # --------------------------------------------------------------------------
    # FROZEN MODEL INFERENCE
    # --------------------------------------------------------------------------

    p_lgb = np.asarray(
        lgb_model.predict(
            X,
            num_iteration=(
                lgb_model.best_iteration
                if lgb_model.best_iteration > 0
                else None
            ),
        ),
        dtype=np.float64,
    )


    dmat = xgb.DMatrix(
        X,
        feature_names=FEATURE_ORDER,
        missing=np.nan,
    )


    p_xgb = np.asarray(
        xgb_model.predict(
            dmat,
            validate_features=True,
        ),
        dtype=np.float64,
    )


    p_corrected = (
        0.5
        *
        p_lgb
        +
        0.5
        *
        p_xgb
    ).astype(
        np.float32
    )


    reference = published_probability[
        start:stop
    ]


    exact_equal = np.array_equal(
        p_corrected,
        reference,
    )


    mismatch_count = int(
        np.count_nonzero(
            p_corrected
            !=
            reference
        )
    )


    max_abs_difference = float(
        np.max(
            np.abs(
                p_corrected.astype(
                    np.float64
                )
                -
                reference.astype(
                    np.float64
                )
            )
        )
    )


    print(
        "-" * 118
    )

    print(
        f"[{file_id + 1:02d}/08] "
        f"{record['day']:10s} "
        f"rows={expected_rows:9,d}"
    )

    print(
        "  feature SHA:",
        X_hash,
    )

    print(
        "  prediction exact equal:",
        exact_equal,
    )

    print(
        "  mismatch count:",
        f"{mismatch_count:,}",
    )

    print(
        "  max abs diff:",
        f"{max_abs_difference:.17g}",
    )

    print(
        "  seconds:",
        f"{time.time() - t0:.2f}",
    )


    if not exact_equal:

        raise RuntimeError(
            "\nIMPLEMENTATION FAILURE:\n"
            "bridge62 FLAG_CORRECTED prediction differs from PUBLISHED.\n"
            f"File: {filename}\n"
            f"Mismatches: {mismatch_count:,}\n"
            f"Max abs diff: {max_abs_difference}"
        )


    corrected_probability[
        start:stop
    ] = p_corrected


    cursor = stop


    del df
    del X
    del labels_canon
    del y
    del p_lgb
    del p_xgb
    del p_corrected
    del dmat
    del reference

    gc.collect()


conn.close()


if not opening_consumed:

    raise RuntimeError(
        "Opening #2 was never consumed."
    )


if cursor != EXPECTED_TARGET_ROWS:

    raise RuntimeError(
        "\nComplete corrected target population mismatch.\n"
        f"Expected: {EXPECTED_TARGET_ROWS:,}\n"
        f"Actual:   {cursor:,}"
    )


print()
print(
    "[PASS] All eight corrected inference blocks exactly reproduced PUBLISHED."
)

print()


# ==============================================================================
# 12. COMPLETE BITWISE IDENTITY
# ==============================================================================

print("=" * 118)
print("COMPLETE BRIDGE62 IDENTITY AUDIT")
print("=" * 118)


corrected_probability_sha = sha256_array(
    corrected_probability
)

corrected_label_sha = sha256_array(
    corrected_labels
)


probability_exact_equal = np.array_equal(
    corrected_probability,
    published_probability,
)


labels_exact_equal = np.array_equal(
    corrected_labels,
    published_labels,
)


print(
    "Published ensemble SHA:",
    EXPECTED_PUBLISHED_ENSEMBLE_SHA,
)

print(
    "Corrected ensemble SHA:",
    corrected_probability_sha,
)

print()

print(
    "Published label SHA:   ",
    EXPECTED_PUBLISHED_BINARY_LABEL_SHA,
)

print(
    "Corrected label SHA:   ",
    corrected_label_sha,
)

print()

print(
    "Probability arrays bitwise equal:",
    probability_exact_equal,
)

print(
    "Binary labels bitwise equal:     ",
    labels_exact_equal,
)


if corrected_probability_sha != EXPECTED_PUBLISHED_ENSEMBLE_SHA:

    raise RuntimeError(
        "\nIMPLEMENTATION FAILURE:\n"
        "Complete FLAG_CORRECTED ensemble SHA differs from PUBLISHED."
    )


if corrected_label_sha != EXPECTED_PUBLISHED_BINARY_LABEL_SHA:

    raise RuntimeError(
        "Complete corrected binary-label SHA differs."
    )


if not probability_exact_equal:

    raise RuntimeError(
        "Complete corrected prediction array is not bitwise equal."
    )


if not labels_exact_equal:

    raise RuntimeError(
        "Complete corrected label array is not bitwise equal."
    )


print()
print(
    "[PASS] BITWISE_IDENTICAL_FEATURE_MAPPING."
)

print(
    "[PASS] BITWISE_IDENTICAL_PREDICTIONS."
)

print()


# ==============================================================================
# 13. METRIC IDENTITY
#
# No recomputation/search required:
# identical labels + identical float32 probabilities imply identical metrics.
# ==============================================================================

published_metrics = published_result[
    "metrics"
]


print("=" * 118)
print("METRIC IDENTITY")
print("=" * 118)


print(
    "PR-AUC:",
    published_metrics[
        "primary"
    ][
        "pr_auc"
    ],
)

print(
    "ROC-AUC:",
    published_metrics[
        "primary"
    ][
        "roc_auc"
    ],
)

print(
    "PR normalized:",
    published_metrics[
        "primary"
    ][
        "pr_normalized"
    ],
)

print()

print(
    "[PASS] FLAG_CORRECTED bridge62 metrics are identically inherited"
)

print(
    "       from the bitwise-identical PUBLISHED prediction vector."
)

print()


# ==============================================================================
# 14. RESULT ARTIFACT
# ==============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


result = {
    "stage":
        "Stage24-2B",

    "status":
        "PRIMARY_BRIDGE62_FLAG_CORRECTED_IDENTITY_COMPLETE",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "direction":
        "IDS2018_TO_CICIDS2017",

    "bridge":
        "bridge62",

    "target_variant":
        "FLAG_CORRECTED",

    "parent_commit":
        EXPECTED_PARENT,

    "target_opening": {
        "opening_number":
            2,

        "original_budget":
            8,

        "consumed_before":
            1,

        "consumed_after":
            2,

        "administratively_cancelled_grounded_cells":
            2,

        "opening_consumed_at_utc":
            opening_started_at,

        "new_scientific_fits":
            0,
    },

    "frozen_invariance": {
        "expected":
            "BITWISE_IDENTICAL_FEATURE_MATRICES_AND_PREDICTIONS",

        "published_mapping_equals_flag_corrected_mapping":
            True,

        "feature_count":
            62,

        "aggregate_flag_fields_present":
            0,

        "target_population_identical":
            True,
    },

    "identity_audit": {
        "published_probability_sha256":
            EXPECTED_PUBLISHED_ENSEMBLE_SHA,

        "flag_corrected_probability_sha256":
            corrected_probability_sha,

        "probability_arrays_bitwise_equal":
            probability_exact_equal,

        "published_binary_label_sha256":
            EXPECTED_PUBLISHED_BINARY_LABEL_SHA,

        "flag_corrected_binary_label_sha256":
            corrected_label_sha,

        "binary_labels_bitwise_equal":
            labels_exact_equal,

        "per_file_feature_matrix_sha256":
            feature_hashes,

        "implementation_failure":
            False,
    },

    "models": {
        "lightgbm_sha256":
            lgb_sha,

        "xgboost_sha256":
            xgb_sha,

        "refit":
            False,

        "ensemble":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",
    },

    "target_population": {
        "rows":
            EXPECTED_TARGET_ROWS,

        "benign":
            int(
                published_result[
                    "target_population"
                ][
                    "benign"
                ]
            ),

        "attack":
            int(
                published_result[
                    "target_population"
                ][
                    "attack"
                ]
            ),

        "prevalence":
            float(
                published_result[
                    "target_population"
                ][
                    "prevalence"
                ]
            ),
    },

    "metrics": {
        "source":
            "BITWISE_IDENTICAL_STAGE24_2A_VECTOR",

        "values":
            published_metrics,
    },

    "prediction_artifact": {
        "new_duplicate_prediction_file_written":
            False,

        "canonical_shared_artifact":
            str(
                PUBLISHED_PRED_PATH.relative_to(
                    REPO
                )
            ),

        "canonical_shared_artifact_sha256":
            sha256_file(
                PUBLISHED_PRED_PATH
            ),

        "probability_array_sha256":
            corrected_probability_sha,
    },

    "anti_adaptation": {
        "feature_search":
            False,

        "feature_remapping_search":
            False,

        "target_fit":
            False,

        "target_threshold_search":
            False,

        "target_calibration":
            False,

        "target_preprocessing_fit":
            False,

        "performance_based_action":
            False,
    },

    "scientific_fit_accounting": {
        "budget":
            4,

        "completed_before":
            2,

        "completed_after":
            2,
    },
}


RESULT_PATH.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


result_sha = sha256_file(
    RESULT_PATH
)


RESULT_SHA_PATH.write_text(
    f"{result_sha}  {RESULT_PATH.name}\n",
    encoding="utf-8",
)


print(
    "Result:",
    RESULT_PATH.relative_to(
        REPO
    ),
)

print(
    "Result SHA256:",
    result_sha,
)

print()


# ==============================================================================
# 15. GIT FREEZE
# ==============================================================================

print("=" * 118)
print("GIT FREEZE")
print("=" * 118)


git_cmd(
    "add",
    "--",
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


expected_staged = {
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: close bridge62 flag-corrected identity opening",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage24-2B commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 16. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 118)
print("PUSH + REMOTE VERIFY")
print("=" * 118)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


status_after = git_cmd(
    "status",
    "--porcelain",
)


if status_after:

    raise RuntimeError(
        "\nRepository not clean:\n"
        + status_after
    )


if RUNTIME_LEDGER.exists():

    RUNTIME_LEDGER.unlink()


print(
    "[PASS] Remote main == local Stage24-2B commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 17. FINAL
# ==============================================================================

print("=" * 118)
print("STAGE24-2B PRIMARY TARGET OPENING #2: PASS")
print("=" * 118)

print()

print(
    "Direction:                    IDS2018 -> CICIDS2017"
)

print(
    "Bridge:                       bridge62"
)

print(
    "Variant:                      FLAG_CORRECTED"
)

print()

print(
    "Target rows:                  2,830,743"
)

print(
    "Feature mapping identity:     PASS"
)

print(
    "Prediction bitwise identity:  PASS"
)

print()

print(
    "Ensemble SHA:"
)

print(
    " ",
    corrected_probability_sha,
)

print()

print(
    "PR-AUC:"
)

print(
    " ",
    published_metrics[
        "primary"
    ][
        "pr_auc"
    ],
)

print(
    "ROC-AUC:"
)

print(
    " ",
    published_metrics[
        "primary"
    ][
        "roc_auc"
    ],
)

print()

print(
    "New model fits:               0"
)

print(
    "Scientific fits:              2 / 4"
)

print(
    "Target openings consumed:     2 / 8"
)

print(
    "Cancelled GROUNDED_S4 cells:  2"
)

print()

print(
    "Result SHA:"
)

print(
    " ",
    result_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "NEXT: PRIMARY TARGET OPENING #3"
)

print(
    "      bridge70 / PUBLISHED"
)

print("=" * 118)

STAGE24-2B — PRIMARY TARGET OPENING #2 — bridge62 / FLAG_CORRECTED

PRE-OPENING GOVERNANCE
HEAD: e1d953f12245f46e136a6a4a521ee8b0f4f8dace
[PASS] Repository clean.
[PASS] No unresolved target-opening marker.
[PASS] Stage24-2B has not previously been opened.

STAGE24-2A REFERENCE VERIFICATION
Stage24-2A result SHA: 1d4b3010fdc8b0f7f62a0fed2c373824ce99cb2f16936c021756749a9ca54749
Published ensemble SHA: e9288ba2590e09ae51da183bff2160fbb501e8633a3af87a22c800f7d3a051d7

[PASS] Stage24-2A reference opening exact.

FROZEN BRIDGE62 IDENTITY
Feature count: 62
Mapping differences: 0

[PASS] Corrected bridge62 projection is identical by frozen specification.

FROZEN MODEL IDENTITIES
LightGBM: 907c9c86755c3c32effb2d0899f92e1a0cce71915f90311c99afd8c73814c4d7
XGBoost:  b30966f34d6f0799657c528344ec540badef7f3ca3dd207abdaa4b45400e6231
[PASS] Frozen bridge62 models exact.

GitHub credential: GITHUB_TOKEN
[PASS] Remote main == Stage24-2A.

MODEL LOAD
LightGBM: 4.6.0
XGBoost:  3.2.0
[PASS] Frozen models 

In [34]:
# ==============================================================================
# STAGE24-2C — PRIMARY TARGET OPENING #3
# IDS2018 -> CICIDS2017
# bridge70 / PUBLISHED
#
# THIS CELL CONSUMES TARGET OPENING #3.
#
# Frozen model:
#   Stage22R CHRONOLOGICAL_NATURAL inherited 70F ensemble
#
#   LightGBM:
#     chronological_natural_lightgbm_model.txt
#     SHA256 =
#       7be4c610814e45be2e315996969d2e2f404a3ada14866b21a731e4382fdc18b8
#
#   XGBoost:
#     chronological_natural_xgboost_model.json
#     SHA256 =
#       38499d0edcdc5c6d0b2c3afee913558973d35d5ebf1cb2ae888a16b8da98756c
#
# Ensemble:
#   0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST
#   combination float64
#   persisted probability float32
#
# Frozen source-validation thresholds:
#   STANDARD = 0.50
#   BALANCED = 0.07
#   SECURITY = 0.07
#
# Target:
#   CICIDS2017 PUBLISHED
#   complete effective population = 2,830,743 rows
#
# NO FIT.
# NO TARGET TUNING.
# NO TARGET FEATURE SEARCH.
# NO TARGET CALIBRATION.
# NO TARGET-FITTED PREPROCESSING.
#
# The Stage24-2A binary label vector is reused as the canonical target-label
# ordering anchor because all primary variants use the identical full target
# population.
# ==============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import duckdb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


print("=" * 118)
print("STAGE24-2C — PRIMARY TARGET OPENING #3 — bridge70 / PUBLISHED")
print("=" * 118)
print()


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "293f91171e22b94d7342963f687df5aa5221ef5d"
)

EXPECTED_TARGET_ROWS = 2_830_743

EXPECTED_LGBM_SHA = (
    "7be4c610814e45be2e315996969d2e2f404a3ada14866b21a731e4382fdc18b8"
)

EXPECTED_XGB_SHA = (
    "38499d0edcdc5c6d0b2c3afee913558973d35d5ebf1cb2ae888a16b8da98756c"
)

EXPECTED_BRIDGE_SPEC_SHA = (
    "b7ff1d8563aafc69145be3acc827fbc59ae75d880b9804bad727f408299fce2f"
)

EXPECTED_PROTOCOL_LOCK_SHA = (
    "8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2"
)

EXPECTED_REFERENCE_LABEL_SHA = (
    "0e4edd0670c9cedde571e08ab92fedbe6acd6e2aa605d7fc62d286791c00740a"
)


# ==============================================================================
# 1. PATHS
# ==============================================================================

LOCK_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
)

BRIDGE_SPEC_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.json"
)

BRIDGE_SPEC_SHA_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.sha256"
)

FINAL_LOCK_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.json"
)

FINAL_LOCK_SHA_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.sha256"
)


STAGE22_DIR = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
)

LGB_MODEL_PATH = (
    STAGE22_DIR
    / "chronological_natural_lightgbm_model.txt"
)

XGB_MODEL_PATH = (
    STAGE22_DIR
    / "chronological_natural_xgboost_model.json"
)

STAGE22_RESULT_PATH = (
    STAGE22_DIR
    / "stage22r_2c_chronological_natural_result.json"
)


STAGE24_1A_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1a_full_source_70f_sanity.json"
)

STAGE24_1A_SHA_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1a_full_source_70f_sanity.sha256"
)


REFERENCE_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_2_primary_target_openings"
    / "stage24_2a_bridge62_published"
)

REFERENCE_RESULT_PATH = (
    REFERENCE_DIR
    / "stage24_2a_bridge62_published_result.json"
)

REFERENCE_PRED_PATH = (
    REFERENCE_DIR
    / "bridge62_published_ensemble_predictions.npz"
)


TARGET_ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/"
    "snapshots/b7e532345512edcd530cb1770dc76636aeb52802/"
    "traffic_labels"
)


OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_2_primary_target_openings"
    / "stage24_2c_bridge70_published"
)

PRED_PATH = (
    OUT
    / "bridge70_published_ensemble_predictions.npz"
)

RESULT_PATH = (
    OUT
    / "stage24_2c_bridge70_published_result.json"
)

RESULT_SHA_PATH = (
    OUT
    / "stage24_2c_bridge70_published_result.sha256"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


RUNTIME_LEDGER = Path(
    "/kaggle/working/stage24_target_opening_runtime_ledger.json"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    cmd = ["git"]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [
        str(x)
        for x in args
    ]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_array(
    array,
):

    arr = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        arr.view(
            np.uint8
        )
    ).hexdigest()


def verify_sidecar(
    artifact,
    sidecar,
):

    if not artifact.is_file():

        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )

    if not sidecar.is_file():

        raise RuntimeError(
            f"Missing SHA sidecar:\n{sidecar}"
        )


    actual = sha256_file(
        artifact
    )

    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )


    if actual != expected:

        raise RuntimeError(
            "\nSHA sidecar mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


    return actual


def qident(
    value,
):

    return (
        '"'
        +
        str(
            value
        ).replace(
            '"',
            '""',
        )
        +
        '"'
    )


def sql_path(
    path,
):

    return str(
        path
    ).replace(
        "'",
        "''",
    )


# ==============================================================================
# 3. PRE-OPENING GOVERNANCE
# ==============================================================================

print("=" * 118)
print("PRE-OPENING GOVERNANCE")
print("=" * 118)


head = git_cmd(
    "rev-parse",
    "HEAD",
)

status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


if RUNTIME_LEDGER.exists():

    raise RuntimeError(
        "\nA target-opening runtime ledger already exists:\n"
        f"{RUNTIME_LEDGER}\n\n"
        "DO NOT rerun an opening blindly."
    )


if OUT.exists() and any(
    OUT.iterdir()
):

    raise RuntimeError(
        "\nStage24-2C output already exists.\n"
        "Refusing duplicate target opening."
    )


print(
    "[PASS] Repository clean."
)

print(
    "[PASS] No unresolved target-opening marker."
)

print(
    "[PASS] Stage24-2C has not previously been opened."
)

print()


# ==============================================================================
# 4. VERIFY PROTOCOL / STAGE24-1A
# ==============================================================================

print("=" * 118)
print("FROZEN PROTOCOL / SOURCE SANITY")
print("=" * 118)


bridge_spec_sha = verify_sidecar(
    BRIDGE_SPEC_PATH,
    BRIDGE_SPEC_SHA_PATH,
)

protocol_lock_sha = verify_sidecar(
    FINAL_LOCK_PATH,
    FINAL_LOCK_SHA_PATH,
)

stage24_1a_sha = verify_sidecar(
    STAGE24_1A_PATH,
    STAGE24_1A_SHA_PATH,
)


if bridge_spec_sha != EXPECTED_BRIDGE_SPEC_SHA:

    raise RuntimeError(
        "Semantic bridge SHA changed."
    )


if protocol_lock_sha != EXPECTED_PROTOCOL_LOCK_SHA:

    raise RuntimeError(
        "Protocol lock SHA changed."
    )


stage24_1a = json.loads(
    STAGE24_1A_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    stage24_1a[
        "sanity_gate"
    ][
        "pass"
    ]
    is not True
):

    raise RuntimeError(
        "Stage24-1A 70F sanity gate was not PASS."
    )


if (
    stage24_1a[
        "probability_reproduction"
    ][
        "bitwise_exact"
    ]
    is not True
):

    raise RuntimeError(
        "Stage24-1A failed bitwise source reproduction."
    )


print(
    "Semantic bridge SHA:",
    bridge_spec_sha,
)

print(
    "Protocol lock SHA:  ",
    protocol_lock_sha,
)

print(
    "Stage24-1A SHA:      ",
    stage24_1a_sha,
)

print()

print(
    "[PASS] 70F inherited-model source sanity was bitwise exact."
)

print()


# ==============================================================================
# 5. VERIFY EXACT STAGE22 MODELS
# ==============================================================================

print("=" * 118)
print("INHERITED STAGE22 MODEL IDENTITIES")
print("=" * 118)


for path in [
    LGB_MODEL_PATH,
    XGB_MODEL_PATH,
    STAGE22_RESULT_PATH,
]:

    if not path.is_file():

        raise RuntimeError(
            f"Missing inherited Stage22 artifact:\n{path}"
        )


lgb_sha = sha256_file(
    LGB_MODEL_PATH
)

xgb_sha = sha256_file(
    XGB_MODEL_PATH
)


print(
    "LightGBM SHA:",
    lgb_sha,
)

print(
    "XGBoost SHA: ",
    xgb_sha,
)


if lgb_sha != EXPECTED_LGBM_SHA:

    raise RuntimeError(
        "\nInherited LightGBM model SHA mismatch."
    )


if xgb_sha != EXPECTED_XGB_SHA:

    raise RuntimeError(
        "\nInherited XGBoost model SHA mismatch."
    )


stage22_result = json.loads(
    STAGE22_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


stage22_hashes = (
    stage22_result[
        "artifacts"
    ][
        "hashes_before_result_json"
    ]
)


if (
    stage22_hashes[
        "chronological_natural_lightgbm_model.txt"
    ]
    !=
    EXPECTED_LGBM_SHA
):

    raise RuntimeError(
        "Stage22 result disagrees on LightGBM identity."
    )


if (
    stage22_hashes[
        "chronological_natural_xgboost_model.json"
    ]
    !=
    EXPECTED_XGB_SHA
):

    raise RuntimeError(
        "Stage22 result disagrees on XGBoost identity."
    )


print()
print(
    "[PASS] Exact inherited Stage22R CHRONOLOGICAL_NATURAL model pair."
)

print()


# ==============================================================================
# 6. FROZEN 70F REPRESENTATION / PUBLISHED MAPPING
# ==============================================================================

bridge_spec = json.loads(
    BRIDGE_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)

final_lock = json.loads(
    FINAL_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


b70 = bridge_spec[
    "bridge70"
]


FEATURE_ORDER = list(
    b70[
        "source_feature_order"
    ]
)

PUBLISHED_MAPPING = dict(
    b70[
        "PUBLISHED_mapping"
    ]
)


if len(
    FEATURE_ORDER
) != 70:

    raise RuntimeError(
        "bridge70 feature count != 70."
    )


if set(
    FEATURE_ORDER
) != set(
    PUBLISHED_MAPPING
):

    raise RuntimeError(
        "bridge70 PUBLISHED mapping incomplete."
    )


stage22_feature_order = list(
    stage22_result[
        "data"
    ][
        "feature_order"
    ]
)


if FEATURE_ORDER != stage22_feature_order:

    raise RuntimeError(
        "\nStage24 bridge70 order != inherited Stage22 model order."
    )


if (
    stage24_1a[
        "source"
    ][
        "feature_order"
    ]
    !=
    FEATURE_ORDER
):

    raise RuntimeError(
        "Stage24-1A feature order differs."
    )


print("=" * 118)
print("BRIDGE70 PUBLISHED REPRESENTATION")
print("=" * 118)

print(
    "Feature count:",
    len(
        FEATURE_ORDER
    ),
)

print(
    "[PASS] Stage24 bridge70 order == exact Stage22 inherited order."
)

print(
    "[PASS] Frozen PUBLISHED target mapping has exactly 70 entries."
)

print()


# ==============================================================================
# 7. FROZEN SOURCE-VALIDATION THRESHOLDS
# ==============================================================================

STANDARD_THRESHOLD = float(
    stage24_1a[
        "source_metrics"
    ][
        "operating_points"
    ][
        "standard"
    ][
        "threshold"
    ]
)

BALANCED_THRESHOLD = float(
    stage24_1a[
        "source_metrics"
    ][
        "operating_points"
    ][
        "balanced"
    ][
        "threshold"
    ]
)

SECURITY_THRESHOLD = float(
    stage24_1a[
        "source_metrics"
    ][
        "operating_points"
    ][
        "security"
    ][
        "threshold"
    ]
)


if np.float32(
    STANDARD_THRESHOLD
) != np.float32(
    0.50
):

    raise RuntimeError(
        "Standard threshold changed."
    )


if np.float32(
    BALANCED_THRESHOLD
) != np.float32(
    0.07
):

    raise RuntimeError(
        "Balanced threshold changed."
    )


if np.float32(
    SECURITY_THRESHOLD
) != np.float32(
    0.07
):

    raise RuntimeError(
        "Security threshold changed."
    )


print("=" * 118)
print("FROZEN SOURCE-VALIDATION THRESHOLDS")
print("=" * 118)

print(
    "STANDARD:",
    repr(
        float(
            np.float32(
                STANDARD_THRESHOLD
            )
        )
    ),
)

print(
    "BALANCED:",
    repr(
        float(
            np.float32(
                BALANCED_THRESHOLD
            )
        )
    ),
)

print(
    "SECURITY:",
    repr(
        float(
            np.float32(
                SECURITY_THRESHOLD
            )
        )
    ),
)

print()

print(
    "[PASS] No target threshold selection."
)

print()


# ==============================================================================
# 8. LOAD REFERENCE TARGET LABEL ORDER FROM STAGE24-2A
# ==============================================================================

print("=" * 118)
print("CANONICAL PRIMARY TARGET ROW / LABEL ORDER")
print("=" * 118)


if not REFERENCE_RESULT_PATH.is_file():

    raise RuntimeError(
        f"Missing Stage24-2A result:\n{REFERENCE_RESULT_PATH}"
    )


if not REFERENCE_PRED_PATH.is_file():

    raise RuntimeError(
        f"Missing Stage24-2A prediction artifact:\n{REFERENCE_PRED_PATH}"
    )


reference_result = json.loads(
    REFERENCE_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


reference_npz = np.load(
    REFERENCE_PRED_PATH,
    allow_pickle=False,
)


y_all = np.asarray(
    reference_npz[
        "binary_label"
    ],
    dtype=np.uint8,
)


file_id_all = np.asarray(
    reference_npz[
        "file_id"
    ],
    dtype=np.uint8,
)


bridge62_probability = np.asarray(
    reference_npz[
        "probability"
    ],
    dtype=np.float32,
)


if y_all.shape != (
    EXPECTED_TARGET_ROWS,
):

    raise RuntimeError(
        "Reference target label shape mismatch."
    )


if file_id_all.shape != (
    EXPECTED_TARGET_ROWS,
):

    raise RuntimeError(
        "Reference file-id shape mismatch."
    )


if bridge62_probability.shape != (
    EXPECTED_TARGET_ROWS,
):

    raise RuntimeError(
        "Reference bridge62 probability shape mismatch."
    )


reference_label_sha = sha256_array(
    y_all
)


if reference_label_sha != EXPECTED_REFERENCE_LABEL_SHA:

    raise RuntimeError(
        "Reference Stage24-2A label SHA mismatch."
    )


n_attack = int(
    y_all.sum()
)

n_benign = int(
    EXPECTED_TARGET_ROWS
    -
    n_attack
)

prevalence = float(
    n_attack
    /
    EXPECTED_TARGET_ROWS
)


print(
    "Rows:",
    f"{EXPECTED_TARGET_ROWS:,}",
)

print(
    "Benign:",
    f"{n_benign:,}",
)

print(
    "Attack:",
    f"{n_attack:,}",
)

print(
    "Label SHA:",
    reference_label_sha,
)

print()

print(
    "[PASS] Canonical primary target row ordering inherited from Stage24-2A."
)

print()


# ==============================================================================
# 9. TARGET POPULATION
# ==============================================================================

population = list(
    final_lock[
        "published_population"
    ]
)


if len(
    population
) != 8:

    raise RuntimeError(
        "Frozen target population must contain 8 physical files."
    )


if sum(
    int(
        record[
            "effective_rows"
        ]
    )
    for record in population
) != EXPECTED_TARGET_ROWS:

    raise RuntimeError(
        "Frozen effective target count changed."
    )


print("=" * 118)
print("FROZEN TARGET POPULATION")
print("=" * 118)


for i, record in enumerate(
    population,
    start=1,
):

    print(
        f"[{i:02d}] "
        f"{record['day']:10s} "
        f"{Path(record['remote']).name} "
        f"effective={int(record['effective_rows']):,}"
    )


print()


# ==============================================================================
# 10. GITHUB CREDENTIAL + REMOTE PARENT
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if value and value.strip():

            github_token = value.strip()
            token_source = f"ENV:{name}"
            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable BEFORE opening #3."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved before opening #3.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main == Stage24-2B parent."
)

print()


# ==============================================================================
# 11. LOAD FROZEN MODELS
#
# Model loading does NOT consume target opening #3.
# ==============================================================================

import lightgbm as lgb
import xgboost as xgb


print("=" * 118)
print("INHERITED MODEL LOAD")
print("=" * 118)


print(
    "LightGBM:",
    lgb.__version__,
)

print(
    "XGBoost: ",
    xgb.__version__,
)


lgb_model = lgb.Booster(
    model_file=str(
        LGB_MODEL_PATH
    )
)


xgb_model = xgb.Booster()

xgb_model.load_model(
    str(
        XGB_MODEL_PATH
    )
)


# Inference execution backend only; the frozen trees are unchanged.
try:

    xgb_model.set_param(
        {
            "device":
                "cpu",

            "nthread":
                -1,
        }
    )

except Exception:
    pass


if lgb_model.num_feature() != 70:

    raise RuntimeError(
        f"LightGBM model expects {lgb_model.num_feature()} features."
    )


if xgb_model.num_features() != 70:

    raise RuntimeError(
        f"XGBoost model expects {xgb_model.num_features()} features."
    )


print(
    "[PASS] Both inherited models expect exactly 70 positional features."
)

print()


# ==============================================================================
# 12. PREALLOCATE OUTPUT
# ==============================================================================

p_ens_all = np.empty(
    EXPECTED_TARGET_ROWS,
    dtype=np.float32,
)


lgb_probability_hasher = hashlib.sha256()

xgb_probability_hasher = hashlib.sha256()


numeric_audit = []

file_slices = []

feature_matrix_hashes = []


# ==============================================================================
# 13. TARGET OPENING #3
# ==============================================================================

print("=" * 118)
print("TARGET INFERENCE — bridge70 / PUBLISHED")
print("=" * 118)

print()
print(
    "First model.predict() below CONSUMES PRIMARY TARGET OPENING #3."
)
print()


conn = duckdb.connect(
    database=":memory:"
)


cursor = 0

opening_consumed = False

opening_started_at = None


for file_id, record in enumerate(
    population
):

    filename = Path(
        record[
            "remote"
        ]
    ).name


    path = (
        TARGET_ROOT
        /
        filename
    )


    expected_rows = int(
        record[
            "effective_rows"
        ]
    )


    if not path.is_file():

        raise RuntimeError(
            f"Missing frozen target Parquet:\n{path}"
        )


    # --------------------------------------------------------------------------
    # Exact frozen PUBLISHED 70F projection.
    # --------------------------------------------------------------------------

    projection_sql = ",\n".join(
        (
            f"CAST({qident(PUBLISHED_MAPPING[source_feature])} AS DOUBLE) "
            f"AS {qident(source_feature)}"
        )
        for source_feature in FEATURE_ORDER
    )


    query = f"""
        SELECT
            {projection_sql}
        FROM read_parquet('{sql_path(path)}')
        WHERE "Label" IS NOT NULL
    """


    t0 = time.time()


    df = conn.execute(
        query
    ).df()


    observed_rows = len(
        df
    )


    if observed_rows != expected_rows:

        raise RuntimeError(
            "\nEffective row mismatch.\n"
            f"File:     {filename}\n"
            f"Expected: {expected_rows:,}\n"
            f"Actual:   {observed_rows:,}"
        )


    X = df[
        FEATURE_ORDER
    ].to_numpy(
        dtype=np.float64,
        copy=True,
    )


    if X.shape != (
        expected_rows,
        70,
    ):

        raise RuntimeError(
            f"Unexpected target matrix shape: {X.shape}"
        )


    positive_inf = int(
        np.isposinf(
            X
        ).sum()
    )

    negative_inf = int(
        np.isneginf(
            X
        ).sum()
    )


    # Frozen Stage24 numeric policy:
    # +/- infinity -> NaN; no explicit imputation.
    inf_mask = np.isinf(
        X
    )


    if inf_mask.any():

        X[
            inf_mask
        ] = np.nan


    nan_count = int(
        np.isnan(
            X
        ).sum()
    )


    X_sha = sha256_array(
        X
    )


    start = cursor

    stop = (
        start
        +
        expected_rows
    )


    if not np.all(
        file_id_all[
            start:stop
        ]
        ==
        np.uint8(
            file_id
        )
    ):

        raise RuntimeError(
            "\nTarget row/file ordering differs from Stage24-2A."
        )


    print(
        "-" * 118
    )

    print(
        f"[{file_id + 1:02d}/08] "
        f"{record['day']:10s} — {filename}"
    )

    print(
        "  effective rows:",
        f"{expected_rows:,}",
    )

    print(
        "  +inf -> NaN:",
        f"{positive_inf:,}",
    )

    print(
        "  -inf -> NaN:",
        f"{negative_inf:,}",
    )

    print(
        "  total NaN cells:",
        f"{nan_count:,}",
    )

    print(
        "  feature SHA:",
        X_sha,
    )


    numeric_audit.append(
        {
            "file_id":
                file_id,

            "day":
                record[
                    "day"
                ],

            "remote":
                record[
                    "remote"
                ],

            "rows":
                expected_rows,

            "positive_inf_to_nan":
                positive_inf,

            "negative_inf_to_nan":
                negative_inf,

            "nan_cells_after_inf_conversion":
                nan_count,
        }
    )


    feature_matrix_hashes.append(
        {
            "file_id":
                file_id,

            "day":
                record[
                    "day"
                ],

            "rows":
                expected_rows,

            "published_feature_matrix_float64_sha256":
                X_sha,
        }
    )


    # --------------------------------------------------------------------------
    # OPENING #3 CONSUMPTION MARKER
    # --------------------------------------------------------------------------

    if not opening_consumed:

        opening_started_at = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )


        RUNTIME_LEDGER.write_text(
            json.dumps(
                {
                    "stage":
                        "Stage24-2C",

                    "direction":
                        "IDS2018_TO_CICIDS2017",

                    "bridge":
                        "bridge70",

                    "target_variant":
                        "PUBLISHED",

                    "opening_number":
                        3,

                    "opening_consumed":
                        True,

                    "opening_consumed_at_utc":
                        opening_started_at,

                    "trigger":
                        (
                            "FIRST_BRIDGE70_PUBLISHED_TARGET_FEATURE_VALUES_"
                            "SUPPLIED_TO_FROZEN_SOURCE_MODEL"
                        ),

                    "parent_commit":
                        EXPECTED_PARENT,
                },
                indent=2,
                sort_keys=True,
            )
            +
            "\n",
            encoding="utf-8",
        )


        opening_consumed = True


        print()
        print(
            "  >>> TARGET OPENING #3 CONSUMED <<<"
        )
        print()


    # --------------------------------------------------------------------------
    # Frozen inherited LightGBM.
    # Positional NumPy input is intentional and frozen by Stage24-1A.
    # --------------------------------------------------------------------------

    t_pred = time.time()


    p_lgb = np.asarray(
        lgb_model.predict(
            X,
            num_iteration=(
                lgb_model.best_iteration
                if lgb_model.best_iteration > 0
                else None
            ),
        ),
        dtype=np.float64,
    )


    if p_lgb.shape != (
        expected_rows,
    ):

        raise RuntimeError(
            "LightGBM prediction shape mismatch."
        )


    # --------------------------------------------------------------------------
    # Frozen inherited XGBoost.
    #
    # Stage24-1A froze POSITIONAL input order. We therefore deliberately use
    # positional DMatrix inference and disable textual feature-name validation.
    # --------------------------------------------------------------------------

    dmat = xgb.DMatrix(
        X,
        missing=np.nan,
    )


    p_xgb = np.asarray(
        xgb_model.predict(
            dmat,
            validate_features=False,
        ),
        dtype=np.float64,
    )


    if p_xgb.shape != (
        expected_rows,
    ):

        raise RuntimeError(
            "XGBoost prediction shape mismatch."
        )


    if not np.isfinite(
        p_lgb
    ).all():

        raise RuntimeError(
            "Non-finite LightGBM target probability."
        )


    if not np.isfinite(
        p_xgb
    ).all():

        raise RuntimeError(
            "Non-finite XGBoost target probability."
        )


    if (
        (p_lgb < 0).any()
        or
        (p_lgb > 1).any()
    ):

        raise RuntimeError(
            "LightGBM target probability outside [0,1]."
        )


    if (
        (p_xgb < 0).any()
        or
        (p_xgb > 1).any()
    ):

        raise RuntimeError(
            "XGBoost target probability outside [0,1]."
        )


    p_lgb32 = p_lgb.astype(
        np.float32
    )

    p_xgb32 = p_xgb.astype(
        np.float32
    )


    lgb_probability_hasher.update(
        np.ascontiguousarray(
            p_lgb32
        ).tobytes()
    )

    xgb_probability_hasher.update(
        np.ascontiguousarray(
            p_xgb32
        ).tobytes()
    )


    # Frozen ensemble:
    # component combination in float64, final persisted probability float32.
    p_ens = (
        0.5
        *
        p_lgb
        +
        0.5
        *
        p_xgb
    ).astype(
        np.float32
    )


    p_ens_all[
        start:stop
    ] = p_ens


    print(
        "  inference:",
        f"{time.time() - t_pred:.2f}s",
    )

    print(
        "  ensemble p min/max:",
        f"{float(p_ens.min()):.9f}",
        "/",
        f"{float(p_ens.max()):.9f}",
    )

    print(
        "  total file seconds:",
        f"{time.time() - t0:.2f}s",
    )


    file_slices.append(
        {
            "file_id":
                file_id,

            "day":
                record[
                    "day"
                ],

            "remote":
                record[
                    "remote"
                ],

            "start":
                start,

            "stop":
                stop,

            "rows":
                expected_rows,
        }
    )


    cursor = stop


    del df
    del X
    del inf_mask
    del p_lgb
    del p_xgb
    del p_lgb32
    del p_xgb32
    del p_ens
    del dmat

    gc.collect()


conn.close()


if not opening_consumed:

    raise RuntimeError(
        "Target opening #3 was never consumed."
    )


if cursor != EXPECTED_TARGET_ROWS:

    raise RuntimeError(
        "\nTarget inference population mismatch.\n"
        f"Expected: {EXPECTED_TARGET_ROWS:,}\n"
        f"Actual:   {cursor:,}"
    )


print()
print(
    "[PASS] Complete bridge70 / PUBLISHED inference finished."
)

print()


# ==============================================================================
# 14. PREDICTION IDENTITIES
# ==============================================================================

lgb_probability_sha = (
    lgb_probability_hasher.hexdigest()
)

xgb_probability_sha = (
    xgb_probability_hasher.hexdigest()
)

ensemble_probability_sha = sha256_array(
    p_ens_all
)


print("=" * 118)
print("PREDICTION BYTE IDENTITIES")
print("=" * 118)

print(
    "LightGBM float32 SHA:",
    lgb_probability_sha,
)

print(
    "XGBoost  float32 SHA:",
    xgb_probability_sha,
)

print(
    "Ensemble float32 SHA:",
    ensemble_probability_sha,
)

print(
    "Binary label SHA:    ",
    reference_label_sha,
)

print()


# ==============================================================================
# 15. METRIC HELPERS
# ==============================================================================

def threshold_metrics(
    y_true,
    probability,
    threshold,
):

    runtime_threshold = np.float32(
        threshold
    )


    pred = (
        probability
        >=
        runtime_threshold
    )


    y1 = (
        y_true
        ==
        1
    )

    y0 = ~y1


    tp = int(
        np.sum(
            pred
            &
            y1
        )
    )

    fp = int(
        np.sum(
            pred
            &
            y0
        )
    )

    fn = int(
        np.sum(
            (~pred)
            &
            y1
        )
    )

    tn = int(
        np.sum(
            (~pred)
            &
            y0
        )
    )


    accuracy = (
        (tp + tn)
        /
        (tp + tn + fp + fn)
    )


    precision = (
        tp
        /
        (tp + fp)
        if (
            tp + fp
        ) > 0
        else 0.0
    )


    recall = (
        tp
        /
        (tp + fn)
        if (
            tp + fn
        ) > 0
        else 0.0
    )


    fpr = (
        fp
        /
        (fp + tn)
        if (
            fp + tn
        ) > 0
        else 0.0
    )


    fnr = (
        fn
        /
        (fn + tp)
        if (
            fn + tp
        ) > 0
        else 0.0
    )


    f1_denom = (
        2 * tp
        +
        fp
        +
        fn
    )


    f1 = (
        2 * tp
        /
        f1_denom
        if f1_denom > 0
        else 0.0
    )


    f2_denom = (
        5 * tp
        +
        4 * fn
        +
        fp
    )


    f2 = (
        5 * tp
        /
        f2_denom
        if f2_denom > 0
        else 0.0
    )


    return {
        "threshold":
            float(
                runtime_threshold
            ),

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn,

        "accuracy":
            float(
                accuracy
            ),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                f1
            ),

        "f2":
            float(
                f2
            ),

        "fpr":
            float(
                fpr
            ),

        "fnr":
            float(
                fnr
            ),
    }


def calibration_metrics(
    y_true,
    probability,
):

    p64 = probability.astype(
        np.float64,
        copy=False,
    )

    y64 = y_true.astype(
        np.float64,
        copy=False,
    )


    brier = float(
        np.mean(
            (
                p64
                -
                y64
            )
            **
            2
        )
    )


    clipped = np.clip(
        p64,
        1e-15,
        1.0 - 1e-15,
    )


    log_loss = float(
        -np.mean(
            y64
            *
            np.log(
                clipped
            )
            +
            (
                1.0
                -
                y64
            )
            *
            np.log(
                1.0
                -
                clipped
            )
        )
    )


    bins = np.floor(
        p64
        *
        10.0
    ).astype(
        np.int8
    )

    bins = np.clip(
        bins,
        0,
        9,
    )


    ece = 0.0

    ece_bins = []


    for b in range(
        10
    ):

        mask = (
            bins
            ==
            b
        )


        count = int(
            mask.sum()
        )


        if count == 0:

            ece_bins.append(
                {
                    "bin":
                        b,

                    "count":
                        0,

                    "mean_probability":
                        None,

                    "empirical_rate":
                        None,

                    "absolute_gap":
                        None,
                }
            )

            continue


        confidence = float(
            p64[
                mask
            ].mean()
        )


        empirical = float(
            y64[
                mask
            ].mean()
        )


        gap = abs(
            confidence
            -
            empirical
        )


        ece += (
            count
            /
            len(
                p64
            )
        ) * gap


        ece_bins.append(
            {
                "bin":
                    b,

                "count":
                    count,

                "mean_probability":
                    confidence,

                "empirical_rate":
                    empirical,

                "absolute_gap":
                    float(
                        gap
                    ),
            }
        )


    return {
        "brier":
            brier,

        "log_loss":
            log_loss,

        "ece_10":
            float(
                ece
            ),

        "ece_bins":
            ece_bins,
    }


# ==============================================================================
# 16. FROZEN PRIMARY METRICS
# ==============================================================================

print("=" * 118)
print("FROZEN PRIMARY METRICS")
print("=" * 118)


metric_start = time.time()


pr_auc = float(
    average_precision_score(
        y_all,
        p_ens_all,
    )
)


roc_auc = float(
    roc_auc_score(
        y_all,
        p_ens_all,
    )
)


pr_excess = float(
    pr_auc
    -
    prevalence
)


pr_normalized = float(
    (
        pr_auc
        -
        prevalence
    )
    /
    (
        1.0
        -
        prevalence
    )
)


standard_metrics = threshold_metrics(
    y_all,
    p_ens_all,
    STANDARD_THRESHOLD,
)

balanced_metrics = threshold_metrics(
    y_all,
    p_ens_all,
    BALANCED_THRESHOLD,
)

security_metrics = threshold_metrics(
    y_all,
    p_ens_all,
    SECURITY_THRESHOLD,
)


calibration = calibration_metrics(
    y_all,
    p_ens_all,
)


print(
    "PR-AUC:        ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:       ",
    f"{roc_auc:.15f}",
)

print(
    "Prevalence:    ",
    f"{prevalence:.15f}",
)

print(
    "PR excess:     ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized: ",
    f"{pr_normalized:.15f}",
)

print()

print(
    "Brier:         ",
    f"{calibration['brier']:.15f}",
)

print(
    "Log loss:      ",
    f"{calibration['log_loss']:.15f}",
)

print(
    "ECE-10:        ",
    f"{calibration['ece_10']:.15f}",
)

print()


for label, value in [
    (
        "STANDARD",
        standard_metrics,
    ),
    (
        "BALANCED",
        balanced_metrics,
    ),
    (
        "SECURITY",
        security_metrics,
    ),
]:

    print(
        label,
        "@",
        value[
            "threshold"
        ],
    )

    print(
        "  TP/TN/FP/FN:",
        value[
            "tp"
        ],
        value[
            "tn"
        ],
        value[
            "fp"
        ],
        value[
            "fn"
        ],
    )

    print(
        "  precision/recall/F1/F2/FPR/FNR:",
        f"{value['precision']:.9f}",
        f"{value['recall']:.9f}",
        f"{value['f1']:.9f}",
        f"{value['f2']:.9f}",
        f"{value['fpr']:.9f}",
        f"{value['fnr']:.9f}",
    )

    print()


print(
    "Metric computation:",
    f"{time.time() - metric_start:.2f}s",
)

print()


# ==============================================================================
# 17. FILE-LEVEL DESCRIPTIVE METRICS
# ==============================================================================

print("=" * 118)
print("FILE-LEVEL DESCRIPTIVE TRANSFER")
print("=" * 118)


file_metrics = []


for info in file_slices:

    start = int(
        info[
            "start"
        ]
    )

    stop = int(
        info[
            "stop"
        ]
    )


    yy = y_all[
        start:stop
    ]

    pp = p_ens_all[
        start:stop
    ]


    attack = int(
        yy.sum()
    )

    benign = int(
        len(
            yy
        )
        -
        attack
    )


    pi = float(
        attack
        /
        len(
            yy
        )
    )


    if (
        attack > 0
        and
        benign > 0
    ):

        local_pr = float(
            average_precision_score(
                yy,
                pp,
            )
        )

        local_roc = float(
            roc_auc_score(
                yy,
                pp,
            )
        )

    else:

        local_pr = None
        local_roc = None


    item = {
        **info,

        "benign":
            benign,

        "attack":
            attack,

        "prevalence":
            pi,

        "pr_auc":
            local_pr,

        "roc_auc":
            local_roc,
    }


    file_metrics.append(
        item
    )


    print(
        f"[{info['file_id']:02d}] "
        f"{info['day']:10s} "
        f"rows={len(yy):9,d} "
        f"attack={attack:8,d} "
        f"pi={pi:.6f} "
        f"PR={local_pr if local_pr is not None else 'NA'} "
        f"ROC={local_roc if local_roc is not None else 'NA'}"
    )


print()


# ==============================================================================
# 18. DESCRIPTIVE BRIDGE70-vs-BRIDGE62 DELTA
#
# Reporting only. No adaptive action.
# Paired 2000-bootstrap inference is intentionally deferred to the frozen
# primary comparison/freeze step.
# ==============================================================================

bridge62_primary = (
    reference_result[
        "metrics"
    ][
        "primary"
    ]
)


bridge62_calibration = (
    reference_result[
        "metrics"
    ][
        "calibration"
    ]
)


descriptive_delta = {
    "bridge70_minus_bridge62_pr_auc":
        float(
            pr_auc
            -
            bridge62_primary[
                "pr_auc"
            ]
        ),

    "bridge70_minus_bridge62_roc_auc":
        float(
            roc_auc
            -
            bridge62_primary[
                "roc_auc"
            ]
        ),

    "bridge70_minus_bridge62_brier":
        float(
            calibration[
                "brier"
            ]
            -
            bridge62_calibration[
                "brier"
            ]
        ),

    "inferential_use":
        "NONE_UNTIL_FROZEN_PAIRED_BOOTSTRAP",
}


print("=" * 118)
print("DESCRIPTIVE bridge70 - bridge62 DELTA")
print("=" * 118)

print(
    "Δ PR-AUC:",
    f"{descriptive_delta['bridge70_minus_bridge62_pr_auc']:.15f}",
)

print(
    "Δ ROC-AUC:",
    f"{descriptive_delta['bridge70_minus_bridge62_roc_auc']:.15f}",
)

print(
    "Δ Brier:",
    f"{descriptive_delta['bridge70_minus_bridge62_brier']:.15f}",
)

print()

print(
    "[INFO] Descriptive only — no protocol action."
)

print()


# ==============================================================================
# 19. PERSIST PREDICTIONS
# ==============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


print("=" * 118)
print("PERSISTING OPENING #3")
print("=" * 118)


save_start = time.time()


np.savez_compressed(
    PRED_PATH,

    probability=
        p_ens_all,

    binary_label=
        y_all,

    file_id=
        file_id_all,
)


pred_file_sha = sha256_file(
    PRED_PATH
)


print(
    "Prediction artifact:",
    PRED_PATH.relative_to(
        REPO
    ),
)

print(
    "Prediction bytes:",
    f"{PRED_PATH.stat().st_size:,}",
)

print(
    "Prediction file SHA:",
    pred_file_sha,
)

print(
    "Save seconds:",
    f"{time.time() - save_start:.2f}",
)

print()


# ==============================================================================
# 20. RESULT RECEIPT
# ==============================================================================

result = {
    "stage":
        "Stage24-2C",

    "status":
        "PRIMARY_BRIDGE70_PUBLISHED_TARGET_OPENING_COMPLETE",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "direction":
        "IDS2018_TO_CICIDS2017",

    "bridge":
        "bridge70",

    "target_variant":
        "PUBLISHED",

    "parent_commit":
        EXPECTED_PARENT,

    "target_opening": {
        "opening_number":
            3,

        "original_budget":
            8,

        "consumed_before":
            2,

        "consumed_after":
            3,

        "administratively_cancelled_grounded_cells":
            2,

        "opening_consumed_at_utc":
            opening_started_at,

        "new_scientific_fits":
            0,
    },

    "anti_adaptation": {
        "target_fit":
            False,

        "target_threshold_selection":
            False,

        "target_feature_search":
            False,

        "target_feature_remapping":
            False,

        "target_imputation_fit":
            False,

        "target_scaling_fit":
            False,

        "target_calibration":
            False,

        "hyperparameter_change":
            False,

        "performance_based_action":
            False,

        "inherited_models_reused_exactly":
            True,
    },

    "representation": {
        "feature_count":
            70,

        "feature_order":
            FEATURE_ORDER,

        "target_mapping":
            PUBLISHED_MAPPING,

        "parse_dtype":
            "float64",

        "positive_infinity":
            "CONVERT_TO_NAN",

        "negative_infinity":
            "CONVERT_TO_NAN",

        "explicit_imputation":
            "NONE",

        "scaling":
            "NONE",

        "input_semantics":
            "FROZEN_POSITIONAL_ORDER",
    },

    "models": {
        "lightgbm": {
            "model_file":
                str(
                    LGB_MODEL_PATH.relative_to(
                        REPO
                    )
                ),

            "model_sha256":
                lgb_sha,

            "target_probability_float32_sha256":
                lgb_probability_sha,
        },

        "xgboost": {
            "model_file":
                str(
                    XGB_MODEL_PATH.relative_to(
                        REPO
                    )
                ),

            "model_sha256":
                xgb_sha,

            "target_probability_float32_sha256":
                xgb_probability_sha,
        },

        "ensemble": {
            "rule":
                "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

            "combination_dtype":
                "float64",

            "storage_dtype":
                "float32",

            "target_probability_sha256":
                ensemble_probability_sha,
        },
    },

    "target_population": {
        "rows":
            EXPECTED_TARGET_ROWS,

        "benign":
            n_benign,

        "attack":
            n_attack,

        "prevalence":
            prevalence,

        "binary_label_sha256":
            reference_label_sha,

        "label_source":
            "STAGE24_2A_CANONICAL_PRIMARY_TARGET_ORDER",

        "files":
            file_slices,
    },

    "numeric_audit":
        numeric_audit,

    "feature_matrix_audit":
        feature_matrix_hashes,

    "metrics": {
        "primary": {
            "pr_auc":
                pr_auc,

            "roc_auc":
                roc_auc,

            "prevalence_chance_anchor":
                prevalence,

            "pr_excess":
                pr_excess,

            "pr_normalized":
                pr_normalized,
        },

        "calibration":
            calibration,

        "thresholded": {
            "standard":
                standard_metrics,

            "balanced":
                balanced_metrics,

            "security":
                security_metrics,
        },

        "file_level_descriptive":
            file_metrics,
    },

    "descriptive_comparison_to_bridge62_published":
        descriptive_delta,

    "threshold_provenance": {
        "source":
            "IDS2018_SOURCE_VALIDATION_ONLY",

        "standard":
            float(
                np.float32(
                    STANDARD_THRESHOLD
                )
            ),

        "balanced":
            float(
                np.float32(
                    BALANCED_THRESHOLD
                )
            ),

        "security":
            float(
                np.float32(
                    SECURITY_THRESHOLD
                )
            ),

        "target_threshold_search":
            False,
    },

    "prediction_artifact": {
        "file":
            str(
                PRED_PATH.relative_to(
                    REPO
                )
            ),

        "file_sha256":
            pred_file_sha,

        "probability_array_sha256":
            ensemble_probability_sha,

        "binary_label_array_sha256":
            reference_label_sha,

        "row_order":
            "FROZEN_PUBLISHED_POPULATION_ORDER",
    },

    "bootstrap": {
        "performed":
            False,

        "reason":
            (
                "Frozen paired stratified comparison is deferred until "
                "primary bridge70 FLAG_CORRECTED vector is also durable."
            ),

        "frozen_replicates":
            2000,

        "frozen_seed":
            42,
    },

    "scientific_fit_accounting": {
        "budget":
            4,

        "completed_before":
            2,

        "completed_after":
            2,

        "new_fits":
            0,
    },
}


RESULT_PATH.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


result_sha = sha256_file(
    RESULT_PATH
)


RESULT_SHA_PATH.write_text(
    f"{result_sha}  {RESULT_PATH.name}\n",
    encoding="utf-8",
)


CHECKSUMS_PATH.write_text(
    (
        f"{result_sha}  {RESULT_PATH.name}\n"
        f"{pred_file_sha}  {PRED_PATH.name}\n"
    ),
    encoding="utf-8",
)


print(
    "Result:",
    RESULT_PATH.relative_to(
        REPO
    ),
)

print(
    "Result SHA256:",
    result_sha,
)

print()


# ==============================================================================
# 21. ENSURE LOCAL GIT AUTHOR EXISTS
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    recovered_name = git_cmd(
        "log",
        "-1",
        "--format=%an",
    )

    git_cmd(
        "config",
        "--local",
        "user.name",
        recovered_name,
    )


if not author_email:

    recovered_email = git_cmd(
        "log",
        "-1",
        "--format=%ae",
    )

    git_cmd(
        "config",
        "--local",
        "user.email",
        recovered_email,
    )


# ==============================================================================
# 22. GIT FREEZE
# ==============================================================================

print("=" * 118)
print("GIT FREEZE")
print("=" * 118)


git_cmd(
    "add",
    "--",
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


expected_staged = {
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: open primary bridge70 published target",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "\nStage24-2C commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 23. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 118)
print("PUSH + REMOTE VERIFY")
print("=" * 118)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


status_after = git_cmd(
    "status",
    "--porcelain",
)


if status_after:

    raise RuntimeError(
        "\nRepository not clean after Stage24-2C freeze:\n"
        + status_after
    )


# Durable remote receipt now exists.
if RUNTIME_LEDGER.exists():

    RUNTIME_LEDGER.unlink()


print(
    "[PASS] Remote main == local Stage24-2C commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 24. FINAL
# ==============================================================================

print("=" * 118)
print("STAGE24-2C PRIMARY TARGET OPENING #3: PASS")
print("=" * 118)

print()

print(
    "Direction:                   IDS2018 -> CICIDS2017"
)

print(
    "Bridge:                      bridge70"
)

print(
    "Variant:                     PUBLISHED"
)

print()

print(
    "Target rows:                 ",
    f"{EXPECTED_TARGET_ROWS:,}",
)

print(
    "Benign:                      ",
    f"{n_benign:,}",
)

print(
    "Attack:                      ",
    f"{n_attack:,}",
)

print(
    "Prevalence:                  ",
    f"{prevalence:.12f}",
)

print()

print(
    "PR-AUC:                      ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:                     ",
    f"{roc_auc:.15f}",
)

print(
    "PR excess:                   ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized:               ",
    f"{pr_normalized:.15f}",
)

print()

print(
    "Brier:                       ",
    f"{calibration['brier']:.15f}",
)

print(
    "Log loss:                    ",
    f"{calibration['log_loss']:.15f}",
)

print(
    "ECE-10:                      ",
    f"{calibration['ece_10']:.15f}",
)

print()

print(
    "bridge70 - bridge62 Δ PR-AUC:",
    f"{descriptive_delta['bridge70_minus_bridge62_pr_auc']:.15f}",
)

print(
    "bridge70 - bridge62 Δ ROC:   ",
    f"{descriptive_delta['bridge70_minus_bridge62_roc_auc']:.15f}",
)

print()

print(
    "New model fits:              0"
)

print(
    "Scientific fits:             2 / 4"
)

print(
    "Target threshold tuning:     NO"
)

print()

print(
    "Target openings consumed:    3 / 8"
)

print(
    "Cancelled GROUNDED_S4 cells: 2"
)

print()

print(
    "Ensemble probability SHA:"
)

print(
    " ",
    ensemble_probability_sha,
)

print()

print(
    "Result SHA:"
)

print(
    " ",
    result_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "NEXT: PRIMARY TARGET OPENING #4"
)

print(
    "      bridge70 / FLAG_CORRECTED"
)

print(
    "      — isolates the frozen aggregate-flag serialization artifact shift."
)

print("=" * 118)

STAGE24-2C — PRIMARY TARGET OPENING #3 — bridge70 / PUBLISHED

PRE-OPENING GOVERNANCE
HEAD: 293f91171e22b94d7342963f687df5aa5221ef5d
[PASS] Repository clean.
[PASS] No unresolved target-opening marker.
[PASS] Stage24-2C has not previously been opened.

FROZEN PROTOCOL / SOURCE SANITY
Semantic bridge SHA: b7ff1d8563aafc69145be3acc827fbc59ae75d880b9804bad727f408299fce2f
Protocol lock SHA:   8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2
Stage24-1A SHA:       b777fa819bfc294343a0bc2aad3a19901ffc8974b22f2e55b2c75c5452c9b755

[PASS] 70F inherited-model source sanity was bitwise exact.

INHERITED STAGE22 MODEL IDENTITIES
LightGBM SHA: 7be4c610814e45be2e315996969d2e2f404a3ada14866b21a731e4382fdc18b8
XGBoost SHA:  38499d0edcdc5c6d0b2c3afee913558973d35d5ebf1cb2ae888a16b8da98756c

[PASS] Exact inherited Stage22R CHRONOLOGICAL_NATURAL model pair.

BRIDGE70 PUBLISHED REPRESENTATION
Feature count: 70
[PASS] Stage24 bridge70 order == exact Stage22 inherited order.
[PASS] Frozen PUB

In [35]:
# ==============================================================================
# STAGE24-2D — PRIMARY TARGET OPENING #4
# IDS2018 -> CICIDS2017
# bridge70 / FLAG_CORRECTED
#
# THIS CELL CONSUMES TARGET OPENING #4.
#
# Frozen model:
#   Exact inherited Stage22R CHRONOLOGICAL_NATURAL 70F ensemble
#
# Frozen target semantics:
#   bridge70 FLAG_CORRECTED
#
# Aggregate flag mapping:
#
#   FIN Flag Cnt   <- URG Flag Count
#   SYN Flag Cnt   <- PSH Flag Count
#   RST Flag Cnt   <- FIN Flag Count
#   PSH Flag Cnt   <- SYN Flag Count
#   ACK Flag Cnt   <- ACK Flag Count
#   URG Flag Cnt   <- CWE Flag Count
#   CWE Flag Count <- ECE Flag Count
#   ECE Flag Cnt   <- RST Flag Count
#
# Seven physical mappings differ; ACK is invariant.
#
# Frozen source-validation thresholds:
#   STANDARD = 0.50
#   BALANCED = 0.07
#   SECURITY = 0.07
#
# NO FIT.
# NO TARGET TUNING.
# NO TARGET FEATURE SEARCH.
# NO TARGET CALIBRATION.
# NO TARGET-FITTED PREPROCESSING.
#
# Persists:
#   - corrected float32 ensemble predictions
#   - canonical binary target labels
#   - file IDs
#   - result JSON
#   - SHA receipts
#
# After this succeeds:
#   PRIMARY FULL-POPULATION OPENINGS ARE COMPLETE.
#   Next step = freeze primary results + frozen paired bootstrap.
# ==============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import duckdb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


print("=" * 118)
print("STAGE24-2D — PRIMARY TARGET OPENING #4 — bridge70 / FLAG_CORRECTED")
print("=" * 118)
print()


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "f89507816102e9e285512962fd1a96bbbaf0eb9f"
)

EXPECTED_TARGET_ROWS = 2_830_743

EXPECTED_LGBM_SHA = (
    "7be4c610814e45be2e315996969d2e2f404a3ada14866b21a731e4382fdc18b8"
)

EXPECTED_XGB_SHA = (
    "38499d0edcdc5c6d0b2c3afee913558973d35d5ebf1cb2ae888a16b8da98756c"
)

EXPECTED_PUBLISHED_PROB_SHA = (
    "91e4271f0933d3e7b904e9d822377703fb5f8da04e3111d813fd1dd4298f9d1b"
)

EXPECTED_BINARY_LABEL_SHA = (
    "0e4edd0670c9cedde571e08ab92fedbe6acd6e2aa605d7fc62d286791c00740a"
)


# ==============================================================================
# 1. PATHS
# ==============================================================================

LOCK_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
)

BRIDGE_SPEC_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.json"
)

FINAL_LOCK_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.json"
)


STAGE22_DIR = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
)

LGB_MODEL_PATH = (
    STAGE22_DIR
    / "chronological_natural_lightgbm_model.txt"
)

XGB_MODEL_PATH = (
    STAGE22_DIR
    / "chronological_natural_xgboost_model.json"
)


PUBLISHED_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_2_primary_target_openings"
    / "stage24_2c_bridge70_published"
)

PUBLISHED_RESULT_PATH = (
    PUBLISHED_DIR
    / "stage24_2c_bridge70_published_result.json"
)

PUBLISHED_PRED_PATH = (
    PUBLISHED_DIR
    / "bridge70_published_ensemble_predictions.npz"
)


TARGET_ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/"
    "snapshots/b7e532345512edcd530cb1770dc76636aeb52802/"
    "traffic_labels"
)


OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_2_primary_target_openings"
    / "stage24_2d_bridge70_flag_corrected"
)

PRED_PATH = (
    OUT
    / "bridge70_flag_corrected_ensemble_predictions.npz"
)

RESULT_PATH = (
    OUT
    / "stage24_2d_bridge70_flag_corrected_result.json"
)

RESULT_SHA_PATH = (
    OUT
    / "stage24_2d_bridge70_flag_corrected_result.sha256"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


RUNTIME_LEDGER = Path(
    "/kaggle/working/stage24_target_opening_runtime_ledger.json"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(args, *, cwd=None, check=True):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(str(x) for x in args)
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(*args, auth_header=None, check=True):

    cmd = ["git"]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [str(x) for x in args]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array(array):

    arr = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        arr.view(np.uint8)
    ).hexdigest()


def qident(value):

    return (
        '"'
        +
        str(value).replace(
            '"',
            '""',
        )
        +
        '"'
    )


def sql_path(path):

    return str(path).replace(
        "'",
        "''",
    )


# ==============================================================================
# 3. PRE-OPENING GOVERNANCE
# ==============================================================================

print("=" * 118)
print("PRE-OPENING GOVERNANCE")
print("=" * 118)


head = git_cmd(
    "rev-parse",
    "HEAD",
)

status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


if RUNTIME_LEDGER.exists():

    raise RuntimeError(
        "\nA target-opening runtime ledger already exists:\n"
        f"{RUNTIME_LEDGER}\n\n"
        "DO NOT rerun an opening blindly."
    )


if OUT.exists() and any(
    OUT.iterdir()
):

    raise RuntimeError(
        "\nStage24-2D output already exists.\n"
        "Refusing duplicate opening."
    )


print(
    "[PASS] Repository clean."
)

print(
    "[PASS] No unresolved target-opening marker."
)

print(
    "[PASS] Stage24-2D has not previously been opened."
)

print()


# ==============================================================================
# 4. LOAD FROZEN BRIDGE / PROTOCOL
# ==============================================================================

print("=" * 118)
print("FROZEN BRIDGE70 FLAG-CORRECTED CONTRACT")
print("=" * 118)


bridge_spec = json.loads(
    BRIDGE_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)

final_lock = json.loads(
    FINAL_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


b70 = bridge_spec[
    "bridge70"
]


FEATURE_ORDER = list(
    b70[
        "source_feature_order"
    ]
)

PUBLISHED_MAPPING = dict(
    b70[
        "PUBLISHED_mapping"
    ]
)

CORRECTED_MAPPING = dict(
    b70[
        "FLAG_CORRECTED_mapping"
    ]
)


if len(
    FEATURE_ORDER
) != 70:

    raise RuntimeError(
        "bridge70 feature count changed."
    )


if set(
    FEATURE_ORDER
) != set(
    CORRECTED_MAPPING
):

    raise RuntimeError(
        "FLAG_CORRECTED mapping incomplete."
    )


FLAG_FEATURES = {
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "ACK Flag Cnt",
    "URG Flag Cnt",
    "CWE Flag Count",
    "ECE Flag Cnt",
}


EXPECTED_DIFFERING = {
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "URG Flag Cnt",
    "CWE Flag Count",
    "ECE Flag Cnt",
}


differences = {
    feature
    for feature in FEATURE_ORDER
    if (
        PUBLISHED_MAPPING[
            feature
        ]
        !=
        CORRECTED_MAPPING[
            feature
        ]
    )
}


print(
    "bridge70 features:",
    len(
        FEATURE_ORDER
    ),
)

print(
    "Aggregate flag features:",
    len(
        FLAG_FEATURES
    ),
)

print(
    "Physical mappings changed:",
    len(
        differences
    ),
)


if differences != EXPECTED_DIFFERING:

    raise RuntimeError(
        "\nFrozen correction difference set changed.\n"
        f"Expected: {sorted(EXPECTED_DIFFERING)}\n"
        f"Actual:   {sorted(differences)}"
    )


if (
    CORRECTED_MAPPING[
        "ACK Flag Cnt"
    ]
    !=
    "ACK Flag Count"
):

    raise RuntimeError(
        "Corrected ACK mapping changed."
    )


print()
print(
    "[PASS] Exactly 7 physical flag mappings change."
)

print(
    "[PASS] ACK remains identity."
)

print()


# ==============================================================================
# 5. VERIFY EXACT INHERITED MODELS
# ==============================================================================

print("=" * 118)
print("FROZEN MODEL IDENTITIES")
print("=" * 118)


lgb_sha = sha256_file(
    LGB_MODEL_PATH
)

xgb_sha = sha256_file(
    XGB_MODEL_PATH
)


print(
    "LightGBM:",
    lgb_sha,
)

print(
    "XGBoost: ",
    xgb_sha,
)


if lgb_sha != EXPECTED_LGBM_SHA:

    raise RuntimeError(
        "Inherited LightGBM SHA mismatch."
    )


if xgb_sha != EXPECTED_XGB_SHA:

    raise RuntimeError(
        "Inherited XGBoost SHA mismatch."
    )


print(
    "[PASS] Exact inherited Stage22R model pair."
)

print()


# ==============================================================================
# 6. VERIFY STAGE24-2C PUBLISHED REFERENCE
# ==============================================================================

print("=" * 118)
print("STAGE24-2C PUBLISHED REFERENCE")
print("=" * 118)


if not PUBLISHED_RESULT_PATH.is_file():

    raise RuntimeError(
        f"Missing Stage24-2C result:\n{PUBLISHED_RESULT_PATH}"
    )


if not PUBLISHED_PRED_PATH.is_file():

    raise RuntimeError(
        f"Missing Stage24-2C prediction:\n{PUBLISHED_PRED_PATH}"
    )


published_result = json.loads(
    PUBLISHED_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


published_npz = np.load(
    PUBLISHED_PRED_PATH,
    allow_pickle=False,
)


published_probability = np.asarray(
    published_npz[
        "probability"
    ],
    dtype=np.float32,
)


y_all = np.asarray(
    published_npz[
        "binary_label"
    ],
    dtype=np.uint8,
)


file_id_all = np.asarray(
    published_npz[
        "file_id"
    ],
    dtype=np.uint8,
)


if published_probability.shape != (
    EXPECTED_TARGET_ROWS,
):

    raise RuntimeError(
        "Published probability shape mismatch."
    )


if y_all.shape != (
    EXPECTED_TARGET_ROWS,
):

    raise RuntimeError(
        "Canonical label shape mismatch."
    )


published_probability_sha = sha256_array(
    published_probability
)

binary_label_sha = sha256_array(
    y_all
)


print(
    "Published probability SHA:",
    published_probability_sha,
)

print(
    "Binary label SHA:         ",
    binary_label_sha,
)


if (
    published_probability_sha
    !=
    EXPECTED_PUBLISHED_PROB_SHA
):

    raise RuntimeError(
        "Stage24-2C probability SHA mismatch."
    )


if binary_label_sha != EXPECTED_BINARY_LABEL_SHA:

    raise RuntimeError(
        "Canonical target label SHA mismatch."
    )


print()
print(
    "[PASS] Stage24-2C canonical population exact."
)

print()


# ==============================================================================
# 7. FROZEN THRESHOLDS
# ==============================================================================

STANDARD_THRESHOLD = np.float32(
    0.50
)

BALANCED_THRESHOLD = np.float32(
    0.07
)

SECURITY_THRESHOLD = np.float32(
    0.07
)


print("=" * 118)
print("FROZEN SOURCE-VALIDATION THRESHOLDS")
print("=" * 118)

print(
    "STANDARD:",
    float(
        STANDARD_THRESHOLD
    ),
)

print(
    "BALANCED:",
    float(
        BALANCED_THRESHOLD
    ),
)

print(
    "SECURITY:",
    float(
        SECURITY_THRESHOLD
    ),
)

print()

print(
    "[PASS] No target threshold selection."
)

print()


# ==============================================================================
# 8. FROZEN TARGET POPULATION
# ==============================================================================

population = list(
    final_lock[
        "published_population"
    ]
)


if len(
    population
) != 8:

    raise RuntimeError(
        "Expected eight target source records."
    )


if sum(
    int(
        record[
            "effective_rows"
        ]
    )
    for record in population
) != EXPECTED_TARGET_ROWS:

    raise RuntimeError(
        "Frozen effective target population changed."
    )


# ==============================================================================
# 9. GITHUB CREDENTIAL + REMOTE PARENT
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if value and value.strip():

            github_token = value.strip()
            token_source = (
                "ENV:"
                +
                name
            )

            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable BEFORE opening #4."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main changed before opening #4.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main == Stage24-2C."
)

print()


# ==============================================================================
# 10. LOAD FROZEN MODELS
# ==============================================================================

import lightgbm as lgb
import xgboost as xgb


print("=" * 118)
print("MODEL LOAD")
print("=" * 118)


print(
    "LightGBM:",
    lgb.__version__,
)

print(
    "XGBoost: ",
    xgb.__version__,
)


lgb_model = lgb.Booster(
    model_file=str(
        LGB_MODEL_PATH
    )
)


xgb_model = xgb.Booster()

xgb_model.load_model(
    str(
        XGB_MODEL_PATH
    )
)


try:

    xgb_model.set_param(
        {
            "device":
                "cpu",

            "nthread":
                -1,
        }
    )

except Exception:
    pass


if lgb_model.num_feature() != 70:

    raise RuntimeError(
        "LightGBM feature dimension != 70."
    )


if xgb_model.num_features() != 70:

    raise RuntimeError(
        "XGBoost feature dimension != 70."
    )


print(
    "[PASS] Both inherited models expect 70 positional features."
)

print()


# ==============================================================================
# 11. PREALLOCATE
# ==============================================================================

corrected_probability = np.empty(
    EXPECTED_TARGET_ROWS,
    dtype=np.float32,
)


lgb_hasher = hashlib.sha256()

xgb_hasher = hashlib.sha256()


numeric_audit = []

feature_hashes = []

file_slices = []


# ==============================================================================
# 12. TARGET OPENING #4
# ==============================================================================

print("=" * 118)
print("FLAG_CORRECTED TARGET INFERENCE")
print("=" * 118)

print()
print(
    "First model.predict() below CONSUMES PRIMARY TARGET OPENING #4."
)

print()


conn = duckdb.connect(
    database=":memory:"
)


cursor = 0

opening_consumed = False

opening_started_at = None


for file_id, record in enumerate(
    population
):

    filename = Path(
        record[
            "remote"
        ]
    ).name


    path = (
        TARGET_ROOT
        /
        filename
    )


    expected_rows = int(
        record[
            "effective_rows"
        ]
    )


    if not path.is_file():

        raise RuntimeError(
            f"Missing target source:\n{path}"
        )


    projection_sql = ",\n".join(
        (
            f"CAST({qident(CORRECTED_MAPPING[source_feature])} AS DOUBLE) "
            f"AS {qident(source_feature)}"
        )
        for source_feature in FEATURE_ORDER
    )


    query = f"""
        SELECT
            {projection_sql}
        FROM read_parquet('{sql_path(path)}')
        WHERE "Label" IS NOT NULL
    """


    t0 = time.time()


    df = conn.execute(
        query
    ).df()


    if len(
        df
    ) != expected_rows:

        raise RuntimeError(
            "\nEffective-row mismatch.\n"
            f"File:     {filename}\n"
            f"Expected: {expected_rows:,}\n"
            f"Actual:   {len(df):,}"
        )


    X = df[
        FEATURE_ORDER
    ].to_numpy(
        dtype=np.float64,
        copy=True,
    )


    if X.shape != (
        expected_rows,
        70,
    ):

        raise RuntimeError(
            f"Unexpected matrix shape: {X.shape}"
        )


    positive_inf = int(
        np.isposinf(
            X
        ).sum()
    )

    negative_inf = int(
        np.isneginf(
            X
        ).sum()
    )


    inf_mask = np.isinf(
        X
    )


    if inf_mask.any():

        X[
            inf_mask
        ] = np.nan


    nan_count = int(
        np.isnan(
            X
        ).sum()
    )


    X_sha = sha256_array(
        X
    )


    start = cursor

    stop = (
        start
        +
        expected_rows
    )


    if not np.all(
        file_id_all[
            start:stop
        ]
        ==
        np.uint8(
            file_id
        )
    ):

        raise RuntimeError(
            "Target row ordering differs from Stage24-2C."
        )


    print(
        "-" * 118
    )

    print(
        f"[{file_id + 1:02d}/08] "
        f"{record['day']:10s} — {filename}"
    )

    print(
        "  effective rows:",
        f"{expected_rows:,}",
    )

    print(
        "  +inf -> NaN:",
        f"{positive_inf:,}",
    )

    print(
        "  -inf -> NaN:",
        f"{negative_inf:,}",
    )

    print(
        "  total NaN cells:",
        f"{nan_count:,}",
    )

    print(
        "  corrected feature SHA:",
        X_sha,
    )


    numeric_audit.append(
        {
            "file_id":
                file_id,

            "day":
                record[
                    "day"
                ],

            "remote":
                record[
                    "remote"
                ],

            "rows":
                expected_rows,

            "positive_inf_to_nan":
                positive_inf,

            "negative_inf_to_nan":
                negative_inf,

            "nan_cells_after_inf_conversion":
                nan_count,
        }
    )


    feature_hashes.append(
        {
            "file_id":
                file_id,

            "day":
                record[
                    "day"
                ],

            "rows":
                expected_rows,

            "flag_corrected_feature_matrix_float64_sha256":
                X_sha,
        }
    )


    # --------------------------------------------------------------------------
    # OPENING #4 CONSUMPTION
    # --------------------------------------------------------------------------

    if not opening_consumed:

        opening_started_at = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )


        RUNTIME_LEDGER.write_text(
            json.dumps(
                {
                    "stage":
                        "Stage24-2D",

                    "direction":
                        "IDS2018_TO_CICIDS2017",

                    "bridge":
                        "bridge70",

                    "target_variant":
                        "FLAG_CORRECTED",

                    "opening_number":
                        4,

                    "opening_consumed":
                        True,

                    "opening_consumed_at_utc":
                        opening_started_at,

                    "trigger":
                        (
                            "FIRST_BRIDGE70_FLAG_CORRECTED_TARGET_FEATURE_VALUES_"
                            "SUPPLIED_TO_FROZEN_SOURCE_MODEL"
                        ),

                    "parent_commit":
                        EXPECTED_PARENT,
                },
                indent=2,
                sort_keys=True,
            )
            +
            "\n",
            encoding="utf-8",
        )


        opening_consumed = True


        print()
        print(
            "  >>> TARGET OPENING #4 CONSUMED <<<"
        )
        print()


    # --------------------------------------------------------------------------
    # INHERITED LIGHTGBM
    # --------------------------------------------------------------------------

    t_pred = time.time()


    p_lgb = np.asarray(
        lgb_model.predict(
            X,
            num_iteration=(
                lgb_model.best_iteration
                if lgb_model.best_iteration > 0
                else None
            ),
        ),
        dtype=np.float64,
    )


    if p_lgb.shape != (
        expected_rows,
    ):

        raise RuntimeError(
            "LightGBM prediction shape mismatch."
        )


    # --------------------------------------------------------------------------
    # INHERITED XGBOOST — FROZEN POSITIONAL SEMANTICS
    # --------------------------------------------------------------------------

    dmat = xgb.DMatrix(
        X,
        missing=np.nan,
    )


    p_xgb = np.asarray(
        xgb_model.predict(
            dmat,
            validate_features=False,
        ),
        dtype=np.float64,
    )


    if p_xgb.shape != (
        expected_rows,
    ):

        raise RuntimeError(
            "XGBoost prediction shape mismatch."
        )


    if not np.isfinite(
        p_lgb
    ).all():

        raise RuntimeError(
            "Non-finite LightGBM probability."
        )


    if not np.isfinite(
        p_xgb
    ).all():

        raise RuntimeError(
            "Non-finite XGBoost probability."
        )


    p_lgb32 = p_lgb.astype(
        np.float32
    )

    p_xgb32 = p_xgb.astype(
        np.float32
    )


    lgb_hasher.update(
        np.ascontiguousarray(
            p_lgb32
        ).tobytes()
    )


    xgb_hasher.update(
        np.ascontiguousarray(
            p_xgb32
        ).tobytes()
    )


    p_ens = (
        0.5
        *
        p_lgb
        +
        0.5
        *
        p_xgb
    ).astype(
        np.float32
    )


    corrected_probability[
        start:stop
    ] = p_ens


    # Descriptive comparison only.
    published_slice = published_probability[
        start:stop
    ]


    changed_prediction_rows = int(
        np.count_nonzero(
            p_ens
            !=
            published_slice
        )
    )


    max_abs_probability_delta = float(
        np.max(
            np.abs(
                p_ens.astype(
                    np.float64
                )
                -
                published_slice.astype(
                    np.float64
                )
            )
        )
    )


    print(
        "  inference:",
        f"{time.time() - t_pred:.2f}s",
    )

    print(
        "  corrected p min/max:",
        f"{float(p_ens.min()):.9f}",
        "/",
        f"{float(p_ens.max()):.9f}",
    )

    print(
        "  prediction rows changed vs PUBLISHED:",
        f"{changed_prediction_rows:,}",
    )

    print(
        "  max |Δp|:",
        f"{max_abs_probability_delta:.9f}",
    )

    print(
        "  total file seconds:",
        f"{time.time() - t0:.2f}s",
    )


    file_slices.append(
        {
            "file_id":
                file_id,

            "day":
                record[
                    "day"
                ],

            "remote":
                record[
                    "remote"
                ],

            "start":
                start,

            "stop":
                stop,

            "rows":
                expected_rows,

            "prediction_rows_changed_vs_published":
                changed_prediction_rows,

            "max_abs_probability_delta_vs_published":
                max_abs_probability_delta,
        }
    )


    cursor = stop


    del df
    del X
    del inf_mask
    del p_lgb
    del p_xgb
    del p_lgb32
    del p_xgb32
    del p_ens
    del published_slice
    del dmat

    gc.collect()


conn.close()


if not opening_consumed:

    raise RuntimeError(
        "Opening #4 was never consumed."
    )


if cursor != EXPECTED_TARGET_ROWS:

    raise RuntimeError(
        "\nComplete corrected population mismatch.\n"
        f"Expected: {EXPECTED_TARGET_ROWS:,}\n"
        f"Actual:   {cursor:,}"
    )


print()
print(
    "[PASS] Complete bridge70 / FLAG_CORRECTED inference finished."
)

print()


# ==============================================================================
# 13. BYTE IDENTITIES
# ==============================================================================

lgb_probability_sha = (
    lgb_hasher.hexdigest()
)

xgb_probability_sha = (
    xgb_hasher.hexdigest()
)

corrected_probability_sha = sha256_array(
    corrected_probability
)


print("=" * 118)
print("PREDICTION BYTE IDENTITIES")
print("=" * 118)

print(
    "LightGBM float32 SHA:",
    lgb_probability_sha,
)

print(
    "XGBoost  float32 SHA:",
    xgb_probability_sha,
)

print(
    "Corrected ensemble SHA:",
    corrected_probability_sha,
)

print(
    "Published ensemble SHA:",
    published_probability_sha,
)

print()


complete_prediction_equal = np.array_equal(
    corrected_probability,
    published_probability,
)


complete_changed_rows = int(
    np.count_nonzero(
        corrected_probability
        !=
        published_probability
    )
)


complete_max_abs_delta = float(
    np.max(
        np.abs(
            corrected_probability.astype(
                np.float64
            )
            -
            published_probability.astype(
                np.float64
            )
        )
    )
)


print(
    "Complete arrays bitwise equal:",
    complete_prediction_equal,
)

print(
    "Changed prediction rows:",
    f"{complete_changed_rows:,}",
)

print(
    "Max |Δp|:",
    f"{complete_max_abs_delta:.9f}",
)

print()


# ==============================================================================
# 14. METRIC HELPERS
# ==============================================================================

def threshold_metrics(
    y_true,
    probability,
    threshold,
):

    threshold = np.float32(
        threshold
    )


    pred = (
        probability
        >=
        threshold
    )


    y1 = (
        y_true
        ==
        1
    )

    y0 = ~y1


    tp = int(
        np.sum(
            pred
            &
            y1
        )
    )

    fp = int(
        np.sum(
            pred
            &
            y0
        )
    )

    fn = int(
        np.sum(
            (~pred)
            &
            y1
        )
    )

    tn = int(
        np.sum(
            (~pred)
            &
            y0
        )
    )


    accuracy = (
        tp + tn
    ) / (
        tp + tn + fp + fn
    )


    precision = (
        tp / (tp + fp)
        if (
            tp + fp
        ) > 0
        else 0.0
    )


    recall = (
        tp / (tp + fn)
        if (
            tp + fn
        ) > 0
        else 0.0
    )


    fpr = (
        fp / (fp + tn)
        if (
            fp + tn
        ) > 0
        else 0.0
    )


    fnr = (
        fn / (fn + tp)
        if (
            fn + tp
        ) > 0
        else 0.0
    )


    f1_den = (
        2 * tp
        +
        fp
        +
        fn
    )


    f1 = (
        2 * tp / f1_den
        if f1_den > 0
        else 0.0
    )


    f2_den = (
        5 * tp
        +
        4 * fn
        +
        fp
    )


    f2 = (
        5 * tp / f2_den
        if f2_den > 0
        else 0.0
    )


    return {
        "threshold":
            float(
                threshold
            ),

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn,

        "accuracy":
            float(
                accuracy
            ),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                f1
            ),

        "f2":
            float(
                f2
            ),

        "fpr":
            float(
                fpr
            ),

        "fnr":
            float(
                fnr
            ),
    }


def calibration_metrics(
    y_true,
    probability,
):

    p64 = probability.astype(
        np.float64,
        copy=False,
    )

    y64 = y_true.astype(
        np.float64,
        copy=False,
    )


    brier = float(
        np.mean(
            (
                p64
                -
                y64
            ) ** 2
        )
    )


    clipped = np.clip(
        p64,
        1e-15,
        1.0 - 1e-15,
    )


    log_loss = float(
        -np.mean(
            y64
            *
            np.log(
                clipped
            )
            +
            (
                1.0 - y64
            )
            *
            np.log(
                1.0 - clipped
            )
        )
    )


    bins = np.floor(
        p64
        *
        10.0
    ).astype(
        np.int8
    )

    bins = np.clip(
        bins,
        0,
        9,
    )


    ece = 0.0

    ece_bins = []


    for b in range(
        10
    ):

        mask = (
            bins
            ==
            b
        )

        count = int(
            mask.sum()
        )


        if count == 0:

            ece_bins.append(
                {
                    "bin":
                        b,

                    "count":
                        0,

                    "mean_probability":
                        None,

                    "empirical_rate":
                        None,

                    "absolute_gap":
                        None,
                }
            )

            continue


        confidence = float(
            p64[
                mask
            ].mean()
        )

        empirical = float(
            y64[
                mask
            ].mean()
        )

        gap = abs(
            confidence
            -
            empirical
        )


        ece += (
            count
            /
            len(
                p64
            )
        ) * gap


        ece_bins.append(
            {
                "bin":
                    b,

                "count":
                    count,

                "mean_probability":
                    confidence,

                "empirical_rate":
                    empirical,

                "absolute_gap":
                    float(
                        gap
                    ),
            }
        )


    return {
        "brier":
            brier,

        "log_loss":
            log_loss,

        "ece_10":
            float(
                ece
            ),

        "ece_bins":
            ece_bins,
    }


# ==============================================================================
# 15. GLOBAL METRICS
# ==============================================================================

n_attack = int(
    y_all.sum()
)

n_benign = int(
    EXPECTED_TARGET_ROWS
    -
    n_attack
)

prevalence = float(
    n_attack
    /
    EXPECTED_TARGET_ROWS
)


print("=" * 118)
print("FROZEN PRIMARY METRICS")
print("=" * 118)


pr_auc = float(
    average_precision_score(
        y_all,
        corrected_probability,
    )
)


roc_auc = float(
    roc_auc_score(
        y_all,
        corrected_probability,
    )
)


pr_excess = float(
    pr_auc
    -
    prevalence
)


pr_normalized = float(
    (
        pr_auc
        -
        prevalence
    )
    /
    (
        1.0
        -
        prevalence
    )
)


calibration = calibration_metrics(
    y_all,
    corrected_probability,
)


standard_metrics = threshold_metrics(
    y_all,
    corrected_probability,
    STANDARD_THRESHOLD,
)

balanced_metrics = threshold_metrics(
    y_all,
    corrected_probability,
    BALANCED_THRESHOLD,
)

security_metrics = threshold_metrics(
    y_all,
    corrected_probability,
    SECURITY_THRESHOLD,
)


print(
    "PR-AUC:        ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:       ",
    f"{roc_auc:.15f}",
)

print(
    "Prevalence:    ",
    f"{prevalence:.15f}",
)

print(
    "PR excess:     ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized: ",
    f"{pr_normalized:.15f}",
)

print()

print(
    "Brier:         ",
    f"{calibration['brier']:.15f}",
)

print(
    "Log loss:      ",
    f"{calibration['log_loss']:.15f}",
)

print(
    "ECE-10:        ",
    f"{calibration['ece_10']:.15f}",
)

print()


for name, metrics in [
    (
        "STANDARD",
        standard_metrics,
    ),
    (
        "BALANCED",
        balanced_metrics,
    ),
    (
        "SECURITY",
        security_metrics,
    ),
]:

    print(
        name,
        "@",
        metrics[
            "threshold"
        ],
    )

    print(
        "  TP/TN/FP/FN:",
        metrics[
            "tp"
        ],
        metrics[
            "tn"
        ],
        metrics[
            "fp"
        ],
        metrics[
            "fn"
        ],
    )

    print(
        "  precision/recall/F1/F2/FPR/FNR:",
        f"{metrics['precision']:.9f}",
        f"{metrics['recall']:.9f}",
        f"{metrics['f1']:.9f}",
        f"{metrics['f2']:.9f}",
        f"{metrics['fpr']:.9f}",
        f"{metrics['fnr']:.9f}",
    )

    print()


# ==============================================================================
# 16. FILE-LEVEL DESCRIPTIVE METRICS
# ==============================================================================

print("=" * 118)
print("FILE-LEVEL DESCRIPTIVE TRANSFER")
print("=" * 118)


file_metrics = []


for info in file_slices:

    start = int(
        info[
            "start"
        ]
    )

    stop = int(
        info[
            "stop"
        ]
    )


    yy = y_all[
        start:stop
    ]

    pp = corrected_probability[
        start:stop
    ]


    attack = int(
        yy.sum()
    )

    benign = int(
        len(
            yy
        )
        -
        attack
    )


    pi = float(
        attack
        /
        len(
            yy
        )
    )


    if (
        attack > 0
        and
        benign > 0
    ):

        local_pr = float(
            average_precision_score(
                yy,
                pp,
            )
        )

        local_roc = float(
            roc_auc_score(
                yy,
                pp,
            )
        )

    else:

        local_pr = None
        local_roc = None


    item = {
        **info,

        "benign":
            benign,

        "attack":
            attack,

        "prevalence":
            pi,

        "pr_auc":
            local_pr,

        "roc_auc":
            local_roc,
    }


    file_metrics.append(
        item
    )


    print(
        f"[{info['file_id']:02d}] "
        f"{info['day']:10s} "
        f"rows={len(yy):9,d} "
        f"attack={attack:8,d} "
        f"pi={pi:.6f} "
        f"PR={local_pr if local_pr is not None else 'NA'} "
        f"ROC={local_roc if local_roc is not None else 'NA'}"
    )


print()


# ==============================================================================
# 17. DESCRIPTIVE ARTIFACT-SHIFT DELTA
# ==============================================================================

published_primary = (
    published_result[
        "metrics"
    ][
        "primary"
    ]
)

published_calibration = (
    published_result[
        "metrics"
    ][
        "calibration"
    ]
)


artifact_shift = {
    "flag_corrected_minus_published_pr_auc":
        float(
            pr_auc
            -
            published_primary[
                "pr_auc"
            ]
        ),

    "flag_corrected_minus_published_roc_auc":
        float(
            roc_auc
            -
            published_primary[
                "roc_auc"
            ]
        ),

    "flag_corrected_minus_published_brier":
        float(
            calibration[
                "brier"
            ]
            -
            published_calibration[
                "brier"
            ]
        ),

    "prediction_rows_changed":
        complete_changed_rows,

    "prediction_rows_changed_fraction":
        float(
            complete_changed_rows
            /
            EXPECTED_TARGET_ROWS
        ),

    "maximum_absolute_probability_delta":
        complete_max_abs_delta,

    "inferential_use":
        "DESCRIPTIVE_ONLY_UNTIL_FROZEN_PAIRED_BOOTSTRAP",
}


print("=" * 118)
print("FLAG SERIALIZATION ARTIFACT SHIFT — DESCRIPTIVE")
print("=" * 118)

print(
    "FLAG_CORRECTED - PUBLISHED Δ PR-AUC:",
    f"{artifact_shift['flag_corrected_minus_published_pr_auc']:.15f}",
)

print(
    "FLAG_CORRECTED - PUBLISHED Δ ROC-AUC:",
    f"{artifact_shift['flag_corrected_minus_published_roc_auc']:.15f}",
)

print(
    "FLAG_CORRECTED - PUBLISHED Δ Brier:",
    f"{artifact_shift['flag_corrected_minus_published_brier']:.15f}",
)

print()

print(
    "Prediction rows changed:",
    f"{complete_changed_rows:,}",
    "/",
    f"{EXPECTED_TARGET_ROWS:,}",
)

print(
    "Changed fraction:",
    f"{artifact_shift['prediction_rows_changed_fraction']:.9f}",
)

print(
    "Maximum |Δp|:",
    f"{complete_max_abs_delta:.9f}",
)

print()

print(
    "[INFO] Descriptive only. No protocol action."
)

print()


# ==============================================================================
# 18. PERSIST CORRECTED PREDICTIONS
# ==============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


print("=" * 118)
print("PERSISTING OPENING #4")
print("=" * 118)


save_start = time.time()


np.savez_compressed(
    PRED_PATH,

    probability=
        corrected_probability,

    binary_label=
        y_all,

    file_id=
        file_id_all,
)


pred_file_sha = sha256_file(
    PRED_PATH
)


print(
    "Prediction artifact:",
    PRED_PATH.relative_to(
        REPO
    ),
)

print(
    "Prediction bytes:",
    f"{PRED_PATH.stat().st_size:,}",
)

print(
    "Prediction file SHA:",
    pred_file_sha,
)

print(
    "Save seconds:",
    f"{time.time() - save_start:.2f}",
)

print()


# ==============================================================================
# 19. RESULT RECEIPT
# ==============================================================================

result = {
    "stage":
        "Stage24-2D",

    "status":
        "PRIMARY_BRIDGE70_FLAG_CORRECTED_TARGET_OPENING_COMPLETE",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "direction":
        "IDS2018_TO_CICIDS2017",

    "bridge":
        "bridge70",

    "target_variant":
        "FLAG_CORRECTED",

    "parent_commit":
        EXPECTED_PARENT,

    "target_opening": {
        "opening_number":
            4,

        "original_budget":
            8,

        "consumed_before":
            3,

        "consumed_after":
            4,

        "administratively_cancelled_grounded_cells":
            2,

        "opening_consumed_at_utc":
            opening_started_at,

        "new_scientific_fits":
            0,
    },

    "representation": {
        "feature_count":
            70,

        "feature_order":
            FEATURE_ORDER,

        "published_mapping":
            PUBLISHED_MAPPING,

        "flag_corrected_mapping":
            CORRECTED_MAPPING,

        "changed_physical_mapping_count":
            7,

        "changed_source_features":
            sorted(
                differences
            ),

        "ack_mapping_invariant":
            True,

        "parse_dtype":
            "float64",

        "positive_infinity":
            "CONVERT_TO_NAN",

        "negative_infinity":
            "CONVERT_TO_NAN",

        "explicit_imputation":
            "NONE",

        "scaling":
            "NONE",

        "input_semantics":
            "FROZEN_POSITIONAL_ORDER",
    },

    "models": {
        "lightgbm": {
            "model_sha256":
                lgb_sha,

            "target_probability_float32_sha256":
                lgb_probability_sha,
        },

        "xgboost": {
            "model_sha256":
                xgb_sha,

            "target_probability_float32_sha256":
                xgb_probability_sha,
        },

        "ensemble": {
            "rule":
                "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

            "combination_dtype":
                "float64",

            "storage_dtype":
                "float32",

            "target_probability_sha256":
                corrected_probability_sha,
        },
    },

    "target_population": {
        "rows":
            EXPECTED_TARGET_ROWS,

        "benign":
            n_benign,

        "attack":
            n_attack,

        "prevalence":
            prevalence,

        "binary_label_sha256":
            binary_label_sha,

        "row_order":
            "IDENTICAL_TO_STAGE24_2C_PUBLISHED",
    },

    "numeric_audit":
        numeric_audit,

    "feature_matrix_audit":
        feature_hashes,

    "metrics": {
        "primary": {
            "pr_auc":
                pr_auc,

            "roc_auc":
                roc_auc,

            "prevalence_chance_anchor":
                prevalence,

            "pr_excess":
                pr_excess,

            "pr_normalized":
                pr_normalized,
        },

        "calibration":
            calibration,

        "thresholded": {
            "standard":
                standard_metrics,

            "balanced":
                balanced_metrics,

            "security":
                security_metrics,
        },

        "file_level_descriptive":
            file_metrics,
    },

    "artifact_shift_descriptive":
        artifact_shift,

    "comparison_reference": {
        "stage24_2c_published_probability_sha256":
            published_probability_sha,

        "stage24_2c_result":
            str(
                PUBLISHED_RESULT_PATH.relative_to(
                    REPO
                )
            ),
    },

    "threshold_provenance": {
        "source":
            "IDS2018_SOURCE_VALIDATION_ONLY",

        "standard":
            float(
                STANDARD_THRESHOLD
            ),

        "balanced":
            float(
                BALANCED_THRESHOLD
            ),

        "security":
            float(
                SECURITY_THRESHOLD
            ),

        "target_threshold_search":
            False,
    },

    "prediction_artifact": {
        "file":
            str(
                PRED_PATH.relative_to(
                    REPO
                )
            ),

        "file_sha256":
            pred_file_sha,

        "probability_array_sha256":
            corrected_probability_sha,

        "binary_label_array_sha256":
            binary_label_sha,

        "row_order":
            "FROZEN_PUBLISHED_POPULATION_ORDER",
    },

    "bootstrap": {
        "performed":
            False,

        "reason":
            (
                "Frozen paired stratified bootstrap is performed only after "
                "all evaluable primary full-population cells are durable."
            ),

        "frozen_replicates":
            2000,

        "frozen_seed":
            42,
    },

    "anti_adaptation": {
        "target_fit":
            False,

        "target_threshold_selection":
            False,

        "target_feature_search":
            False,

        "target_mapping_search":
            False,

        "target_calibration":
            False,

        "target_preprocessing_fit":
            False,

        "performance_based_action":
            False,

        "models_reused_exactly":
            True,
    },

    "scientific_fit_accounting": {
        "budget":
            4,

        "completed_before":
            2,

        "completed_after":
            2,

        "new_fits":
            0,
    },
}


RESULT_PATH.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


result_sha = sha256_file(
    RESULT_PATH
)


RESULT_SHA_PATH.write_text(
    f"{result_sha}  {RESULT_PATH.name}\n",
    encoding="utf-8",
)


CHECKSUMS_PATH.write_text(
    (
        f"{result_sha}  {RESULT_PATH.name}\n"
        f"{pred_file_sha}  {PRED_PATH.name}\n"
    ),
    encoding="utf-8",
)


print(
    "Result:",
    RESULT_PATH.relative_to(
        REPO
    ),
)

print(
    "Result SHA256:",
    result_sha,
)

print()


# ==============================================================================
# 20. GIT AUTHOR SAFETY
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    git_cmd(
        "config",
        "--local",
        "user.name",
        git_cmd(
            "log",
            "-1",
            "--format=%an",
        ),
    )


if not author_email:

    git_cmd(
        "config",
        "--local",
        "user.email",
        git_cmd(
            "log",
            "-1",
            "--format=%ae",
        ),
    )


# ==============================================================================
# 21. GIT FREEZE
# ==============================================================================

print("=" * 118)
print("GIT FREEZE")
print("=" * 118)


git_cmd(
    "add",
    "--",
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


expected_staged = {
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: open primary bridge70 flag-corrected target",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage24-2D commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 22. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 118)
print("PUSH + REMOTE VERIFY")
print("=" * 118)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


status_after = git_cmd(
    "status",
    "--porcelain",
)


if status_after:

    raise RuntimeError(
        "\nRepository not clean after Stage24-2D:\n"
        + status_after
    )


if RUNTIME_LEDGER.exists():

    RUNTIME_LEDGER.unlink()


print(
    "[PASS] Remote main == local Stage24-2D commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 23. FINAL
# ==============================================================================

print("=" * 118)
print("STAGE24-2D PRIMARY TARGET OPENING #4: PASS")
print("=" * 118)

print()

print(
    "Direction:                    IDS2018 -> CICIDS2017"
)

print(
    "Bridge:                       bridge70"
)

print(
    "Variant:                      FLAG_CORRECTED"
)

print()

print(
    "Target rows:                  ",
    f"{EXPECTED_TARGET_ROWS:,}",
)

print(
    "Benign:                       ",
    f"{n_benign:,}",
)

print(
    "Attack:                       ",
    f"{n_attack:,}",
)

print(
    "Prevalence:                   ",
    f"{prevalence:.12f}",
)

print()

print(
    "PR-AUC:                       ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:                      ",
    f"{roc_auc:.15f}",
)

print(
    "PR excess:                    ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized:                ",
    f"{pr_normalized:.15f}",
)

print()

print(
    "Brier:                        ",
    f"{calibration['brier']:.15f}",
)

print(
    "Log loss:                     ",
    f"{calibration['log_loss']:.15f}",
)

print(
    "ECE-10:                       ",
    f"{calibration['ece_10']:.15f}",
)

print()

print(
    "CORRECTED - PUBLISHED Δ PR:   ",
    f"{artifact_shift['flag_corrected_minus_published_pr_auc']:.15f}",
)

print(
    "CORRECTED - PUBLISHED Δ ROC:  ",
    f"{artifact_shift['flag_corrected_minus_published_roc_auc']:.15f}",
)

print(
    "CORRECTED - PUBLISHED Δ Brier:",
    f"{artifact_shift['flag_corrected_minus_published_brier']:.15f}",
)

print()

print(
    "Prediction rows changed:      ",
    f"{complete_changed_rows:,}",
)

print(
    "Changed-row fraction:         ",
    f"{artifact_shift['prediction_rows_changed_fraction']:.9f}",
)

print(
    "Maximum |Δp|:                 ",
    f"{complete_max_abs_delta:.9f}",
)

print()

print(
    "New model fits:               0"
)

print(
    "Scientific fits:              2 / 4"
)

print(
    "Target threshold tuning:      NO"
)

print()

print(
    "Target openings consumed:     4 / 8"
)

print(
    "Primary GROUNDED_S4 cancelled:2"
)

print()

print(
    "Corrected ensemble SHA:"
)

print(
    " ",
    corrected_probability_sha,
)

print()

print(
    "Result SHA:"
)

print(
    " ",
    result_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "PRIMARY FULL-POPULATION TARGET OPENINGS COMPLETE."
)

print()

print(
    "NEXT:"
)

print(
    "  Stage24 primary-result freeze + frozen paired stratified bootstrap"
)

print(
    "  (2000 replicates, seed 42; no new target opening)."
)

print("=" * 118)

STAGE24-2D — PRIMARY TARGET OPENING #4 — bridge70 / FLAG_CORRECTED

PRE-OPENING GOVERNANCE
HEAD: f89507816102e9e285512962fd1a96bbbaf0eb9f
[PASS] Repository clean.
[PASS] No unresolved target-opening marker.
[PASS] Stage24-2D has not previously been opened.

FROZEN BRIDGE70 FLAG-CORRECTED CONTRACT
bridge70 features: 70
Aggregate flag features: 8
Physical mappings changed: 7

[PASS] Exactly 7 physical flag mappings change.
[PASS] ACK remains identity.

FROZEN MODEL IDENTITIES
LightGBM: 7be4c610814e45be2e315996969d2e2f404a3ada14866b21a731e4382fdc18b8
XGBoost:  38499d0edcdc5c6d0b2c3afee913558973d35d5ebf1cb2ae888a16b8da98756c
[PASS] Exact inherited Stage22R model pair.

STAGE24-2C PUBLISHED REFERENCE
Published probability SHA: 91e4271f0933d3e7b904e9d822377703fb5f8da04e3111d813fd1dd4298f9d1b
Binary label SHA:          0e4edd0670c9cedde571e08ab92fedbe6acd6e2aa605d7fc62d286791c00740a

[PASS] Stage24-2C canonical population exact.

FROZEN SOURCE-VALIDATION THRESHOLDS
STANDARD: 0.5
BALANCED: 0.0

In [36]:
# ==============================================================================
# STAGE24-3A — PRIMARY RESULTS FREEZE + EXACT PAIRED STRATIFIED BOOTSTRAP
#
# Direction:
#   IDS2018 -> CICIDS2017
#
# INPUT VECTORS ALREADY DURABLE:
#   A = bridge62 / PUBLISHED
#       == bridge62 / FLAG_CORRECTED bit-for-bit
#
#   B = bridge70 / PUBLISHED
#
#   C = bridge70 / FLAG_CORRECTED
#
# FROZEN BOOTSTRAP:
#   method       = PAIRED_STRATIFIED_ROW_BOOTSTRAP
#   strata       = BENIGN / ATTACK
#   sampling     = WITH_REPLACEMENT_WITHIN_EACH_STRATUM
#   stratum size = PRESERVED EXACTLY
#   replicates   = 2000
#   seed         = 42
#   CI           = percentile [2.5%, 97.5%]
#
# Frozen paired-difference metrics:
#   - PR_AUC
#   - ROC_AUC
#   - Brier
#
# Primary comparisons:
#
#   1. bridge70 PUBLISHED - bridge62 PUBLISHED
#
#   2. bridge70 FLAG_CORRECTED - bridge62 FLAG_CORRECTED
#      (bridge62 corrected == bridge62 published exactly)
#
#   3. bridge70 FLAG_CORRECTED - bridge70 PUBLISHED
#      isolates aggregate-flag serialization artifact sensitivity
#
# Structural identity:
#
#   bridge62 FLAG_CORRECTED - bridge62 PUBLISHED
#       = 0 exactly for every metric
#
# --------------------------------------------------------------------------
# EXACT COMPUTATIONAL COMPRESSION
# --------------------------------------------------------------------------
#
# The bootstrap does NOT approximate the row bootstrap.
#
# Rows having the same:
#
#   (binary label,
#    bridge62 float32 probability,
#    bridge70 PUBLISHED float32 probability,
#    bridge70 FLAG_CORRECTED float32 probability)
#
# are metric-equivalent for every frozen primary bootstrap comparison.
#
# Such rows are collapsed into equivalence groups of multiplicity m_g.
#
# Sampling rows uniformly with replacement within a stratum is exactly
# equivalent, at this sufficient-statistic level, to drawing:
#
#   Multinomial(
#       n_stratum,
#       probabilities = group_multiplicity / n_stratum
#   )
#
# over those equivalence groups.
#
# All three prediction vectors use the SAME sampled group multiplicities
# within every bootstrap replicate, preserving the frozen paired design.
#
# NO MODEL EXECUTION.
# NO NEW TARGET OPENING.
# NO NEW FIT.
# NO TARGET TUNING.
# ==============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np


print("=" * 118)
print("STAGE24-3A — PRIMARY RESULTS FREEZE + PAIRED STRATIFIED BOOTSTRAP")
print("=" * 118)
print()


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "a2a197ec01d30f09f140c157cdddc2e1e870512a"
)

EXPECTED_ROWS = 2_830_743

EXPECTED_BENIGN = 2_273_097

EXPECTED_ATTACK = 557_646

EXPECTED_LABEL_SHA = (
    "0e4edd0670c9cedde571e08ab92fedbe6acd6e2aa605d7fc62d286791c00740a"
)

EXPECTED_P62_SHA = (
    "e9288ba2590e09ae51da183bff2160fbb501e8633a3af87a22c800f7d3a051d7"
)

EXPECTED_P70_PUBLISHED_SHA = (
    "91e4271f0933d3e7b904e9d822377703fb5f8da04e3111d813fd1dd4298f9d1b"
)

EXPECTED_P70_CORRECTED_SHA = (
    "7ff3c93b010040b6d780f7f52ca9764c2957b3b93e87a29707c99e1a29ed709a"
)


BOOTSTRAP_REPLICATES = 2000

BOOTSTRAP_SEED = 42

BOOTSTRAP_BATCH = 8


# ==============================================================================
# 1. INPUT PATHS
# ==============================================================================

BASE = (
    REPO
    / "results"
    / "stage24_cross_dataset"
)


B62_DIR = (
    BASE
    / "stage24_2_primary_target_openings"
    / "stage24_2a_bridge62_published"
)

B62_PRED_PATH = (
    B62_DIR
    / "bridge62_published_ensemble_predictions.npz"
)

B62_RESULT_PATH = (
    B62_DIR
    / "stage24_2a_bridge62_published_result.json"
)

B62_RESULT_SHA_PATH = (
    B62_DIR
    / "stage24_2a_bridge62_published_result.sha256"
)


B62_CORRECTED_DIR = (
    BASE
    / "stage24_2_primary_target_openings"
    / "stage24_2b_bridge62_flag_corrected"
)

B62_CORRECTED_RESULT_PATH = (
    B62_CORRECTED_DIR
    / "stage24_2b_bridge62_flag_corrected_identity_result.json"
)

B62_CORRECTED_SHA_PATH = (
    B62_CORRECTED_DIR
    / "stage24_2b_bridge62_flag_corrected_identity_result.sha256"
)


B70_PUB_DIR = (
    BASE
    / "stage24_2_primary_target_openings"
    / "stage24_2c_bridge70_published"
)

B70_PUB_PRED_PATH = (
    B70_PUB_DIR
    / "bridge70_published_ensemble_predictions.npz"
)

B70_PUB_RESULT_PATH = (
    B70_PUB_DIR
    / "stage24_2c_bridge70_published_result.json"
)

B70_PUB_RESULT_SHA_PATH = (
    B70_PUB_DIR
    / "stage24_2c_bridge70_published_result.sha256"
)


B70_CORR_DIR = (
    BASE
    / "stage24_2_primary_target_openings"
    / "stage24_2d_bridge70_flag_corrected"
)

B70_CORR_PRED_PATH = (
    B70_CORR_DIR
    / "bridge70_flag_corrected_ensemble_predictions.npz"
)

B70_CORR_RESULT_PATH = (
    B70_CORR_DIR
    / "stage24_2d_bridge70_flag_corrected_result.json"
)

B70_CORR_RESULT_SHA_PATH = (
    B70_CORR_DIR
    / "stage24_2d_bridge70_flag_corrected_result.sha256"
)


OUT = (
    BASE
    / "stage24_3_primary_results_freeze"
    / "stage24_3a_primary_paired_bootstrap"
)

BOOTSTRAP_PATH = (
    OUT
    / "stage24_3a_primary_paired_bootstrap_replicates.npz"
)

RESULT_PATH = (
    OUT
    / "stage24_3a_primary_results_freeze.json"
)

RESULT_SHA_PATH = (
    OUT
    / "stage24_3a_primary_results_freeze.sha256"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    cmd = [
        "git"
    ]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [
        str(x)
        for x in args
    ]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_array(
    array,
):

    arr = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        arr.view(
            np.uint8
        )
    ).hexdigest()


def verify_sidecar(
    artifact,
    sidecar,
):

    if not artifact.is_file():

        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )

    if not sidecar.is_file():

        raise RuntimeError(
            f"Missing SHA sidecar:\n{sidecar}"
        )


    actual = sha256_file(
        artifact
    )


    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )


    if actual != expected:

        raise RuntimeError(
            "\nSHA mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


    return actual


# ==============================================================================
# 3. GOVERNANCE
# ==============================================================================

print("=" * 118)
print("GOVERNANCE")
print("=" * 118)


head = git_cmd(
    "rev-parse",
    "HEAD",
)


status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected repository HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


if OUT.exists() and any(
    OUT.iterdir()
):

    raise RuntimeError(
        "\nStage24-3A output already exists.\n"
        "Refusing accidental duplicate freeze."
    )


print(
    "[PASS] Repository clean."
)

print(
    "[PASS] No prior Stage24-3A artifacts."
)

print()


# ==============================================================================
# 4. VERIFY ALL PRIMARY RECEIPTS
# ==============================================================================

print("=" * 118)
print("PRIMARY RECEIPT VERIFICATION")
print("=" * 118)


receipt_pairs = [
    (
        B62_RESULT_PATH,
        B62_RESULT_SHA_PATH,
    ),
    (
        B62_CORRECTED_RESULT_PATH,
        B62_CORRECTED_SHA_PATH,
    ),
    (
        B70_PUB_RESULT_PATH,
        B70_PUB_RESULT_SHA_PATH,
    ),
    (
        B70_CORR_RESULT_PATH,
        B70_CORR_RESULT_SHA_PATH,
    ),
]


receipt_hashes = {}


for artifact, sidecar in receipt_pairs:

    digest = verify_sidecar(
        artifact,
        sidecar,
    )


    receipt_hashes[
        str(
            artifact.relative_to(
                REPO
            )
        )
    ] = digest


    print(
        artifact.name,
        digest,
    )


print()
print(
    "[PASS] All four evaluable primary opening receipts byte-verified."
)

print()


# ==============================================================================
# 5. LOAD FROZEN RESULT RECEIPTS
# ==============================================================================

b62_result = json.loads(
    B62_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)

b62_corr_result = json.loads(
    B62_CORRECTED_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)

b70_pub_result = json.loads(
    B70_PUB_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)

b70_corr_result = json.loads(
    B70_CORR_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    b62_result[
        "status"
    ]
    !=
    "PRIMARY_BRIDGE62_PUBLISHED_TARGET_OPENING_COMPLETE"
):

    raise RuntimeError(
        "Stage24-2A status mismatch."
    )


if (
    b62_corr_result[
        "status"
    ]
    !=
    "PRIMARY_BRIDGE62_FLAG_CORRECTED_IDENTITY_COMPLETE"
):

    raise RuntimeError(
        "Stage24-2B status mismatch."
    )


if (
    b70_pub_result[
        "status"
    ]
    !=
    "PRIMARY_BRIDGE70_PUBLISHED_TARGET_OPENING_COMPLETE"
):

    raise RuntimeError(
        "Stage24-2C status mismatch."
    )


if (
    b70_corr_result[
        "status"
    ]
    !=
    "PRIMARY_BRIDGE70_FLAG_CORRECTED_TARGET_OPENING_COMPLETE"
):

    raise RuntimeError(
        "Stage24-2D status mismatch."
    )


if (
    b62_corr_result[
        "identity_audit"
    ][
        "probability_arrays_bitwise_equal"
    ]
    is not True
):

    raise RuntimeError(
        "bridge62 PUBLISHED/FLAG_CORRECTED identity is not PASS."
    )


print(
    "[PASS] Primary opening statuses exact."
)

print(
    "[PASS] bridge62 PUBLISHED == FLAG_CORRECTED remains bitwise exact."
)

print()


# ==============================================================================
# 6. LOAD PRIMARY PREDICTION VECTORS
# ==============================================================================

print("=" * 118)
print("PRIMARY PREDICTION VECTOR VERIFICATION")
print("=" * 118)


for path in [
    B62_PRED_PATH,
    B70_PUB_PRED_PATH,
    B70_CORR_PRED_PATH,
]:

    if not path.is_file():

        raise RuntimeError(
            f"Missing prediction artifact:\n{path}"
        )


b62_npz = np.load(
    B62_PRED_PATH,
    allow_pickle=False,
)

b70_pub_npz = np.load(
    B70_PUB_PRED_PATH,
    allow_pickle=False,
)

b70_corr_npz = np.load(
    B70_CORR_PRED_PATH,
    allow_pickle=False,
)


p62 = np.asarray(
    b62_npz[
        "probability"
    ],
    dtype=np.float32,
)


p70_pub = np.asarray(
    b70_pub_npz[
        "probability"
    ],
    dtype=np.float32,
)


p70_corr = np.asarray(
    b70_corr_npz[
        "probability"
    ],
    dtype=np.float32,
)


y62 = np.asarray(
    b62_npz[
        "binary_label"
    ],
    dtype=np.uint8,
)


y70_pub = np.asarray(
    b70_pub_npz[
        "binary_label"
    ],
    dtype=np.uint8,
)


y70_corr = np.asarray(
    b70_corr_npz[
        "binary_label"
    ],
    dtype=np.uint8,
)


file62 = np.asarray(
    b62_npz[
        "file_id"
    ],
    dtype=np.uint8,
)


file70_pub = np.asarray(
    b70_pub_npz[
        "file_id"
    ],
    dtype=np.uint8,
)


file70_corr = np.asarray(
    b70_corr_npz[
        "file_id"
    ],
    dtype=np.uint8,
)


for name, arr in [
    (
        "bridge62",
        p62,
    ),
    (
        "bridge70 published",
        p70_pub,
    ),
    (
        "bridge70 corrected",
        p70_corr,
    ),
]:

    if arr.shape != (
        EXPECTED_ROWS,
    ):

        raise RuntimeError(
            f"{name} probability shape mismatch: {arr.shape}"
        )


if not np.array_equal(
    y62,
    y70_pub,
):

    raise RuntimeError(
        "bridge62 and bridge70 PUBLISHED labels differ."
    )


if not np.array_equal(
    y62,
    y70_corr,
):

    raise RuntimeError(
        "bridge62 and bridge70 CORRECTED labels differ."
    )


if not np.array_equal(
    file62,
    file70_pub,
):

    raise RuntimeError(
        "Primary file ordering differs between bridge62 and bridge70 PUBLISHED."
    )


if not np.array_equal(
    file62,
    file70_corr,
):

    raise RuntimeError(
        "Primary file ordering differs between PUBLISHED and CORRECTED."
    )


y = y62


label_sha = sha256_array(
    y
)

p62_sha = sha256_array(
    p62
)

p70_pub_sha = sha256_array(
    p70_pub
)

p70_corr_sha = sha256_array(
    p70_corr
)


print(
    "Binary label SHA:        ",
    label_sha,
)

print(
    "bridge62 SHA:            ",
    p62_sha,
)

print(
    "bridge70 PUBLISHED SHA:  ",
    p70_pub_sha,
)

print(
    "bridge70 CORRECTED SHA:  ",
    p70_corr_sha,
)


if label_sha != EXPECTED_LABEL_SHA:

    raise RuntimeError(
        "Canonical binary target-label SHA mismatch."
    )


if p62_sha != EXPECTED_P62_SHA:

    raise RuntimeError(
        "bridge62 probability SHA mismatch."
    )


if p70_pub_sha != EXPECTED_P70_PUBLISHED_SHA:

    raise RuntimeError(
        "bridge70 PUBLISHED probability SHA mismatch."
    )


if p70_corr_sha != EXPECTED_P70_CORRECTED_SHA:

    raise RuntimeError(
        "bridge70 FLAG_CORRECTED probability SHA mismatch."
    )


n_attack = int(
    y.sum()
)

n_benign = int(
    len(
        y
    )
    -
    n_attack
)


if n_benign != EXPECTED_BENIGN:

    raise RuntimeError(
        "Benign population changed."
    )


if n_attack != EXPECTED_ATTACK:

    raise RuntimeError(
        "Attack population changed."
    )


print()
print(
    "Rows:  ",
    f"{len(y):,}",
)

print(
    "Benign:",
    f"{n_benign:,}",
)

print(
    "Attack:",
    f"{n_attack:,}",
)

print()

print(
    "[PASS] All primary probability vectors share one exact target population."
)

print()


# ==============================================================================
# 7. BUILD EXACT JOINT SUFFICIENT-STATISTIC GROUPS
#
# Group key within each label stratum:
#
#   (p62_float32_bits,
#    p70_published_float32_bits,
#    p70_corrected_float32_bits)
#
# Since label is held separately by the stratum, every row in one group is
# identical for every metric used by every frozen primary comparison.
# ==============================================================================

print("=" * 118)
print("EXACT BOOTSTRAP SUFFICIENT-STATISTIC COMPRESSION")
print("=" * 118)


TRIPLET_DTYPE = np.dtype(
    [
        (
            "p62",
            "<u4",
        ),
        (
            "p70_pub",
            "<u4",
        ),
        (
            "p70_corr",
            "<u4",
        ),
    ]
)


def group_stratum(
    label_value,
):

    mask = (
        y
        ==
        np.uint8(
            label_value
        )
    )


    rows = int(
        mask.sum()
    )


    triplets = np.empty(
        rows,
        dtype=TRIPLET_DTYPE,
    )


    triplets[
        "p62"
    ] = (
        np.ascontiguousarray(
            p62[
                mask
            ]
        )
        .view(
            np.uint32
        )
    )


    triplets[
        "p70_pub"
    ] = (
        np.ascontiguousarray(
            p70_pub[
                mask
            ]
        )
        .view(
            np.uint32
        )
    )


    triplets[
        "p70_corr"
    ] = (
        np.ascontiguousarray(
            p70_corr[
                mask
            ]
        )
        .view(
            np.uint32
        )
    )


    t0 = time.time()


    unique_triplets, multiplicity = np.unique(
        triplets,
        return_counts=True,
    )


    elapsed = (
        time.time()
        -
        t0
    )


    p62_group = (
        unique_triplets[
            "p62"
        ]
        .astype(
            np.uint32,
            copy=True,
        )
        .view(
            np.float32
        )
    )


    p70_pub_group = (
        unique_triplets[
            "p70_pub"
        ]
        .astype(
            np.uint32,
            copy=True,
        )
        .view(
            np.float32
        )
    )


    p70_corr_group = (
        unique_triplets[
            "p70_corr"
        ]
        .astype(
            np.uint32,
            copy=True,
        )
        .view(
            np.float32
        )
    )


    multiplicity = multiplicity.astype(
        np.int64,
        copy=False,
    )


    if int(
        multiplicity.sum()
    ) != rows:

        raise RuntimeError(
            "Equivalence-group multiplicity does not reconstruct stratum."
        )


    del triplets
    del unique_triplets
    del mask

    gc.collect()


    return {
        "rows":
            rows,

        "groups":
            len(
                multiplicity
            ),

        "multiplicity":
            multiplicity,

        "p62":
            p62_group,

        "p70_pub":
            p70_pub_group,

        "p70_corr":
            p70_corr_group,

        "group_seconds":
            elapsed,
    }


benign_grouped = group_stratum(
    0
)


print(
    "Benign rows:        ",
    f"{benign_grouped['rows']:,}",
)

print(
    "Benign exact groups:",
    f"{benign_grouped['groups']:,}",
)

print(
    "Compression ratio:  ",
    f"{benign_grouped['rows'] / benign_grouped['groups']:.3f}x",
)

print(
    "Grouping seconds:   ",
    f"{benign_grouped['group_seconds']:.2f}",
)

print()


attack_grouped = group_stratum(
    1
)


print(
    "Attack rows:        ",
    f"{attack_grouped['rows']:,}",
)

print(
    "Attack exact groups:",
    f"{attack_grouped['groups']:,}",
)

print(
    "Compression ratio:  ",
    f"{attack_grouped['rows'] / attack_grouped['groups']:.3f}x",
)

print(
    "Grouping seconds:   ",
    f"{attack_grouped['group_seconds']:.2f}",
)

print()


if benign_grouped[
    "rows"
] != EXPECTED_BENIGN:

    raise RuntimeError(
        "Grouped benign stratum size changed."
    )


if attack_grouped[
    "rows"
] != EXPECTED_ATTACK:

    raise RuntimeError(
        "Grouped attack stratum size changed."
    )


G0 = int(
    benign_grouped[
        "groups"
    ]
)

G1 = int(
    attack_grouped[
        "groups"
    ]
)

G = (
    G0
    +
    G1
)


print(
    "Total rows:  ",
    f"{EXPECTED_ROWS:,}",
)

print(
    "Exact groups:",
    f"{G:,}",
)

print(
    "Overall compression:",
    f"{EXPECTED_ROWS / G:.3f}x",
)

print()

print(
    "[PASS] Exact joint prediction/label equivalence groups constructed."
)

print()


# ==============================================================================
# 8. COMBINE GROUP SUFFICIENT STATISTICS
# ==============================================================================

group_label = np.concatenate(
    [
        np.zeros(
            G0,
            dtype=np.uint8,
        ),
        np.ones(
            G1,
            dtype=np.uint8,
        ),
    ]
)


group_multiplicity = np.concatenate(
    [
        benign_grouped[
            "multiplicity"
        ],
        attack_grouped[
            "multiplicity"
        ],
    ]
)


group_p62 = np.concatenate(
    [
        benign_grouped[
            "p62"
        ],
        attack_grouped[
            "p62"
        ],
    ]
).astype(
    np.float32,
    copy=False,
)


group_p70_pub = np.concatenate(
    [
        benign_grouped[
            "p70_pub"
        ],
        attack_grouped[
            "p70_pub"
        ],
    ]
).astype(
    np.float32,
    copy=False,
)


group_p70_corr = np.concatenate(
    [
        benign_grouped[
            "p70_corr"
        ],
        attack_grouped[
            "p70_corr"
        ],
    ]
).astype(
    np.float32,
    copy=False,
)


if int(
    group_multiplicity.sum()
) != EXPECTED_ROWS:

    raise RuntimeError(
        "Combined exact groups do not reconstruct complete target."
    )


# Exact multinomial category probabilities.
prob_benign = (
    benign_grouped[
        "multiplicity"
    ].astype(
        np.float64
    )
    /
    float(
        EXPECTED_BENIGN
    )
)


prob_attack = (
    attack_grouped[
        "multiplicity"
    ].astype(
        np.float64
    )
    /
    float(
        EXPECTED_ATTACK
    )
)


# Normalize only for machine floating-point closure.
prob_benign /= prob_benign.sum()

prob_attack /= prob_attack.sum()


# ==============================================================================
# 9. PRECOMPUTE DISTINCT-SCORE GROUPING FOR EACH PREDICTION VECTOR
#
# Frozen AUC semantics:
#
# PR_AUC:
#   distinct-score grouped, noninterpolated AP
#
# ROC_AUC:
#   distinct-score grouped trapezoidal ROC area
# ==============================================================================

print("=" * 118)
print("DISTINCT-SCORE METRIC PRECOMPUTATION")
print("=" * 118)


def make_metric_config(
    name,
    score,
):

    if score.dtype != np.float32:

        raise RuntimeError(
            f"{name}: score storage is not float32."
        )


    if not np.isfinite(
        score
    ).all():

        raise RuntimeError(
            f"{name}: non-finite persisted probability."
        )


    order = np.argsort(
        score,
        kind="stable",
    )[
        ::-1
    ]


    sorted_score = score[
        order
    ]


    if len(
        sorted_score
    ) == 0:

        raise RuntimeError(
            "Empty metric vector."
        )


    starts = np.concatenate(
        [
            np.array(
                [
                    0
                ],
                dtype=np.int64,
            ),

            (
                np.flatnonzero(
                    sorted_score[
                        1:
                    ]
                    !=
                    sorted_score[
                        :-1
                    ]
                )
                +
                1
            ).astype(
                np.int64
            ),
        ]
    )


    sorted_label = group_label[
        order
    ].astype(
        np.uint8,
        copy=False,
    )


    score64 = score.astype(
        np.float64,
        copy=False,
    )


    loss = (
        score64
        -
        group_label.astype(
            np.float64,
            copy=False,
        )
    ) ** 2


    print(
        f"{name:25s}"
        f" exact groups={len(score):9,d}"
        f" distinct scores={len(starts):9,d}"
    )


    return {
        "name":
            name,

        "score":
            score,

        "order":
            order,

        "starts":
            starts,

        "sorted_label":
            sorted_label,

        "loss":
            loss,
    }


config62 = make_metric_config(
    "bridge62",
    group_p62,
)


config70_pub = make_metric_config(
    "bridge70 PUBLISHED",
    group_p70_pub,
)


config70_corr = make_metric_config(
    "bridge70 FLAG_CORRECTED",
    group_p70_corr,
)


print()


# ==============================================================================
# 10. EXACT WEIGHTED METRIC ENGINE
# ==============================================================================

def metric_batch(
    bootstrap_group_counts,
    config,
):

    # bootstrap_group_counts:
    #     shape (B, G)
    #     exact resampled multiplicity of every joint equivalence group.

    order = config[
        "order"
    ]

    starts = config[
        "starts"
    ]

    sorted_label = config[
        "sorted_label"
    ]


    # Reorder group multiplicities into descending score order.
    w = bootstrap_group_counts[
        :,
        order
    ]


    # Total resampled weight at each distinct score.
    total_by_score = np.add.reduceat(
        w,
        starts,
        axis=1,
    ).astype(
        np.float64,
        copy=False,
    )


    # Positive resampled weight at each distinct score.
    positive_by_score = np.add.reduceat(
        w
        *
        sorted_label[
            None,
            :
        ],
        starts,
        axis=1,
    ).astype(
        np.float64,
        copy=False,
    )


    negative_by_score = (
        total_by_score
        -
        positive_by_score
    )


    # --------------------------------------------------------------------------
    # Non-interpolated Average Precision.
    # --------------------------------------------------------------------------

    cumulative_positive = np.cumsum(
        positive_by_score,
        axis=1,
        dtype=np.float64,
    )


    cumulative_total = np.cumsum(
        total_by_score,
        axis=1,
        dtype=np.float64,
    )


    precision = np.divide(
        cumulative_positive,
        cumulative_total,
        out=np.zeros_like(
            cumulative_positive,
            dtype=np.float64,
        ),
        where=(
            cumulative_total
            >
            0
        ),
    )


    pr_auc = np.sum(
        precision
        *
        (
            positive_by_score
            /
            float(
                EXPECTED_ATTACK
            )
        ),
        axis=1,
    )


    # --------------------------------------------------------------------------
    # Distinct-score trapezoidal ROC AUC.
    #
    # At each score:
    #   previous TPR = positives before entering this tied score
    #   current  TPR = positives after entering this tied score
    #   delta FPR    = negatives at this score / total negatives
    # --------------------------------------------------------------------------

    previous_positive = (
        cumulative_positive
        -
        positive_by_score
    )


    roc_auc = np.sum(
        (
            (
                previous_positive
                +
                cumulative_positive
            )
            /
            (
                2.0
                *
                float(
                    EXPECTED_ATTACK
                )
            )
        )
        *
        (
            negative_by_score
            /
            float(
                EXPECTED_BENIGN
            )
        ),
        axis=1,
    )


    # --------------------------------------------------------------------------
    # Brier.
    # --------------------------------------------------------------------------

    brier = (
        bootstrap_group_counts
        @
        config[
            "loss"
        ]
    ) / float(
        EXPECTED_ROWS
    )


    return {
        "pr_auc":
            np.asarray(
                pr_auc,
                dtype=np.float64,
            ),

        "roc_auc":
            np.asarray(
                roc_auc,
                dtype=np.float64,
            ),

        "brier":
            np.asarray(
                brier,
                dtype=np.float64,
            ),
    }


# ==============================================================================
# 11. VERIFY GROUPED METRIC ENGINE AGAINST DURABLE POINT ESTIMATES
# ==============================================================================

print("=" * 118)
print("GROUPED METRIC ENGINE POINT-ESTIMATE VALIDATION")
print("=" * 118)


original_counts = group_multiplicity[
    None,
    :
]


point62 = metric_batch(
    original_counts,
    config62,
)


point70_pub = metric_batch(
    original_counts,
    config70_pub,
)


point70_corr = metric_batch(
    original_counts,
    config70_corr,
)


def scalar_metrics(
    result,
):

    return {
        key:
            float(
                value[
                    0
                ]
            )
        for key, value in result.items()
    }


point62 = scalar_metrics(
    point62
)

point70_pub = scalar_metrics(
    point70_pub
)

point70_corr = scalar_metrics(
    point70_corr
)


expected62 = {
    "pr_auc":
        float(
            b62_result[
                "metrics"
            ][
                "primary"
            ][
                "pr_auc"
            ]
        ),

    "roc_auc":
        float(
            b62_result[
                "metrics"
            ][
                "primary"
            ][
                "roc_auc"
            ]
        ),

    "brier":
        float(
            b62_result[
                "metrics"
            ][
                "calibration"
            ][
                "brier"
            ]
        ),
}


expected70_pub = {
    "pr_auc":
        float(
            b70_pub_result[
                "metrics"
            ][
                "primary"
            ][
                "pr_auc"
            ]
        ),

    "roc_auc":
        float(
            b70_pub_result[
                "metrics"
            ][
                "primary"
            ][
                "roc_auc"
            ]
        ),

    "brier":
        float(
            b70_pub_result[
                "metrics"
            ][
                "calibration"
            ][
                "brier"
            ]
        ),
}


expected70_corr = {
    "pr_auc":
        float(
            b70_corr_result[
                "metrics"
            ][
                "primary"
            ][
                "pr_auc"
            ]
        ),

    "roc_auc":
        float(
            b70_corr_result[
                "metrics"
            ][
                "primary"
            ][
                "roc_auc"
            ]
        ),

    "brier":
        float(
            b70_corr_result[
                "metrics"
            ][
                "calibration"
            ][
                "brier"
            ]
        ),
}


def validate_point(
    name,
    actual,
    expected,
    tolerance=5e-12,
):

    print(
        name
    )


    for metric in [
        "pr_auc",
        "roc_auc",
        "brier",
    ]:

        delta = abs(
            actual[
                metric
            ]
            -
            expected[
                metric
            ]
        )


        print(
            f"  {metric:8s}"
            f" grouped={actual[metric]:.15f}"
            f" durable={expected[metric]:.15f}"
            f" |Δ|={delta:.3e}"
        )


        if delta > tolerance:

            raise RuntimeError(
                "\nGrouped bootstrap metric engine does not reproduce "
                "the frozen point estimate.\n"
                f"Vector: {name}\n"
                f"Metric: {metric}\n"
                f"Difference: {delta}"
            )


validate_point(
    "bridge62",
    point62,
    expected62,
)


validate_point(
    "bridge70 PUBLISHED",
    point70_pub,
    expected70_pub,
)


validate_point(
    "bridge70 FLAG_CORRECTED",
    point70_corr,
    expected70_corr,
)


print()
print(
    "[PASS] Exact grouped metric engine reproduces all 9 durable point metrics."
)

print()


# ==============================================================================
# 12. BOOTSTRAP STORAGE
# ==============================================================================

bootstrap_metrics = {
    "bridge62_pr_auc":
        np.empty(
            BOOTSTRAP_REPLICATES,
            dtype=np.float64,
        ),

    "bridge62_roc_auc":
        np.empty(
            BOOTSTRAP_REPLICATES,
            dtype=np.float64,
        ),

    "bridge62_brier":
        np.empty(
            BOOTSTRAP_REPLICATES,
            dtype=np.float64,
        ),

    "bridge70_published_pr_auc":
        np.empty(
            BOOTSTRAP_REPLICATES,
            dtype=np.float64,
        ),

    "bridge70_published_roc_auc":
        np.empty(
            BOOTSTRAP_REPLICATES,
            dtype=np.float64,
        ),

    "bridge70_published_brier":
        np.empty(
            BOOTSTRAP_REPLICATES,
            dtype=np.float64,
        ),

    "bridge70_corrected_pr_auc":
        np.empty(
            BOOTSTRAP_REPLICATES,
            dtype=np.float64,
        ),

    "bridge70_corrected_roc_auc":
        np.empty(
            BOOTSTRAP_REPLICATES,
            dtype=np.float64,
        ),

    "bridge70_corrected_brier":
        np.empty(
            BOOTSTRAP_REPLICATES,
            dtype=np.float64,
        ),
}


# ==============================================================================
# 13. EXECUTE FROZEN 2000-REPLICATE PAIRED STRATIFIED BOOTSTRAP
# ==============================================================================

print("=" * 118)
print("PAIRED STRATIFIED BOOTSTRAP")
print("=" * 118)

print(
    "Replicates:",
    BOOTSTRAP_REPLICATES,
)

print(
    "Seed:",
    BOOTSTRAP_SEED,
)

print(
    "Batch size:",
    BOOTSTRAP_BATCH,
)

print(
    "Sampling:",
    "multinomial exact-equivalence-group representation of row bootstrap",
)

print()

print(
    "NOTE: this is computational inference over durable prediction vectors."
)

print(
    "      It consumes NO additional target opening."
)

print()


rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


bootstrap_start = time.time()


for start_rep in range(
    0,
    BOOTSTRAP_REPLICATES,
    BOOTSTRAP_BATCH,
):

    batch = min(
        BOOTSTRAP_BATCH,
        BOOTSTRAP_REPLICATES
        -
        start_rep,
    )


    # --------------------------------------------------------------------------
    # Same exact bootstrap sample is used by all three prediction vectors.
    # --------------------------------------------------------------------------

    sampled_benign = rng.multinomial(
        EXPECTED_BENIGN,
        prob_benign,
        size=batch,
    )


    sampled_attack = rng.multinomial(
        EXPECTED_ATTACK,
        prob_attack,
        size=batch,
    )


    group_counts = np.empty(
        (
            batch,
            G,
        ),
        dtype=np.int64,
    )


    group_counts[
        :,
        :G0
    ] = sampled_benign


    group_counts[
        :,
        G0:
    ] = sampled_attack


    # Fail closed on the frozen stratification contract.
    if not np.all(
        group_counts[
            :,
            :G0
        ].sum(
            axis=1
        )
        ==
        EXPECTED_BENIGN
    ):

        raise RuntimeError(
            "Bootstrap benign stratum size changed."
        )


    if not np.all(
        group_counts[
            :,
            G0:
        ].sum(
            axis=1
        )
        ==
        EXPECTED_ATTACK
    ):

        raise RuntimeError(
            "Bootstrap attack stratum size changed."
        )


    m62 = metric_batch(
        group_counts,
        config62,
    )


    m70_pub = metric_batch(
        group_counts,
        config70_pub,
    )


    m70_corr = metric_batch(
        group_counts,
        config70_corr,
    )


    stop_rep = (
        start_rep
        +
        batch
    )


    bootstrap_metrics[
        "bridge62_pr_auc"
    ][
        start_rep:stop_rep
    ] = m62[
        "pr_auc"
    ]


    bootstrap_metrics[
        "bridge62_roc_auc"
    ][
        start_rep:stop_rep
    ] = m62[
        "roc_auc"
    ]


    bootstrap_metrics[
        "bridge62_brier"
    ][
        start_rep:stop_rep
    ] = m62[
        "brier"
    ]


    bootstrap_metrics[
        "bridge70_published_pr_auc"
    ][
        start_rep:stop_rep
    ] = m70_pub[
        "pr_auc"
    ]


    bootstrap_metrics[
        "bridge70_published_roc_auc"
    ][
        start_rep:stop_rep
    ] = m70_pub[
        "roc_auc"
    ]


    bootstrap_metrics[
        "bridge70_published_brier"
    ][
        start_rep:stop_rep
    ] = m70_pub[
        "brier"
    ]


    bootstrap_metrics[
        "bridge70_corrected_pr_auc"
    ][
        start_rep:stop_rep
    ] = m70_corr[
        "pr_auc"
    ]


    bootstrap_metrics[
        "bridge70_corrected_roc_auc"
    ][
        start_rep:stop_rep
    ] = m70_corr[
        "roc_auc"
    ]


    bootstrap_metrics[
        "bridge70_corrected_brier"
    ][
        start_rep:stop_rep
    ] = m70_corr[
        "brier"
    ]


    del sampled_benign
    del sampled_attack
    del group_counts
    del m62
    del m70_pub
    del m70_corr

    gc.collect()


    if (
        stop_rep % 100 == 0
        or
        stop_rep == BOOTSTRAP_REPLICATES
    ):

        elapsed = (
            time.time()
            -
            bootstrap_start
        )


        rate = (
            stop_rep
            /
            elapsed
            if elapsed > 0
            else 0.0
        )


        print(
            f"  completed {stop_rep:4d} / {BOOTSTRAP_REPLICATES}"
            f"  elapsed={elapsed:8.2f}s"
            f"  rate={rate:6.2f} reps/s"
        )


bootstrap_seconds = (
    time.time()
    -
    bootstrap_start
)


print()

print(
    "[PASS] 2000 paired stratified bootstrap replicates complete."
)

print(
    "Bootstrap seconds:",
    f"{bootstrap_seconds:.2f}",
)

print()


# ==============================================================================
# 14. BUILD PAIRED DIFFERENCES
#
# Orientations follow the already-used Stage24 descriptive comparisons.
# ==============================================================================

differences = {
    # --------------------------------------------------------------------------
    # bridge70 PUBLISHED - bridge62 PUBLISHED
    # --------------------------------------------------------------------------

    "bridge70_published_minus_bridge62_published_pr_auc":
        (
            bootstrap_metrics[
                "bridge70_published_pr_auc"
            ]
            -
            bootstrap_metrics[
                "bridge62_pr_auc"
            ]
        ),

    "bridge70_published_minus_bridge62_published_roc_auc":
        (
            bootstrap_metrics[
                "bridge70_published_roc_auc"
            ]
            -
            bootstrap_metrics[
                "bridge62_roc_auc"
            ]
        ),

    "bridge70_published_minus_bridge62_published_brier":
        (
            bootstrap_metrics[
                "bridge70_published_brier"
            ]
            -
            bootstrap_metrics[
                "bridge62_brier"
            ]
        ),


    # --------------------------------------------------------------------------
    # bridge70 FLAG_CORRECTED - bridge62 FLAG_CORRECTED
    #
    # bridge62 FLAG_CORRECTED is exactly the same vector as bridge62 PUBLISHED.
    # --------------------------------------------------------------------------

    "bridge70_corrected_minus_bridge62_corrected_pr_auc":
        (
            bootstrap_metrics[
                "bridge70_corrected_pr_auc"
            ]
            -
            bootstrap_metrics[
                "bridge62_pr_auc"
            ]
        ),

    "bridge70_corrected_minus_bridge62_corrected_roc_auc":
        (
            bootstrap_metrics[
                "bridge70_corrected_roc_auc"
            ]
            -
            bootstrap_metrics[
                "bridge62_roc_auc"
            ]
        ),

    "bridge70_corrected_minus_bridge62_corrected_brier":
        (
            bootstrap_metrics[
                "bridge70_corrected_brier"
            ]
            -
            bootstrap_metrics[
                "bridge62_brier"
            ]
        ),


    # --------------------------------------------------------------------------
    # bridge70 FLAG_CORRECTED - bridge70 PUBLISHED
    # --------------------------------------------------------------------------

    "bridge70_corrected_minus_published_pr_auc":
        (
            bootstrap_metrics[
                "bridge70_corrected_pr_auc"
            ]
            -
            bootstrap_metrics[
                "bridge70_published_pr_auc"
            ]
        ),

    "bridge70_corrected_minus_published_roc_auc":
        (
            bootstrap_metrics[
                "bridge70_corrected_roc_auc"
            ]
            -
            bootstrap_metrics[
                "bridge70_published_roc_auc"
            ]
        ),

    "bridge70_corrected_minus_published_brier":
        (
            bootstrap_metrics[
                "bridge70_corrected_brier"
            ]
            -
            bootstrap_metrics[
                "bridge70_published_brier"
            ]
        ),
}


# ==============================================================================
# 15. PERCENTILE CI HELPER
# ==============================================================================

def percentile_ci(
    values,
):

    lower, upper = np.percentile(
        values,
        [
            2.5,
            97.5,
        ],
        method="linear",
    )


    return (
        float(
            lower
        ),
        float(
            upper
        ),
    )


def comparison_summary(
    name,
    point_a,
    point_b,
    boot_prefix,
):

    output = {}


    for metric in [
        "pr_auc",
        "roc_auc",
        "brier",
    ]:

        point_difference = float(
            point_a[
                metric
            ]
            -
            point_b[
                metric
            ]
        )


        bootstrap_key = (
            boot_prefix
            +
            "_"
            +
            metric
        )


        values = differences[
            bootstrap_key
        ]


        lower, upper = percentile_ci(
            values
        )


        output[
            metric
        ] = {
            "point_difference":
                point_difference,

            "bootstrap_mean_difference":
                float(
                    np.mean(
                        values
                    )
                ),

            "ci_2_5":
                lower,

            "ci_97_5":
                upper,

            "ci_excludes_zero":
                bool(
                    (
                        lower > 0.0
                    )
                    or
                    (
                        upper < 0.0
                    )
                ),
        }


    return {
        "comparison":
            name,

        "orientation":
            "A_MINUS_B",

        "bootstrap":
            output,
    }


comparison_1 = comparison_summary(
    (
        "bridge70_PUBLISHED_minus_"
        "bridge62_PUBLISHED"
    ),
    point70_pub,
    point62,
    "bridge70_published_minus_bridge62_published",
)


comparison_2 = comparison_summary(
    (
        "bridge70_FLAG_CORRECTED_minus_"
        "bridge62_FLAG_CORRECTED"
    ),
    point70_corr,
    point62,
    "bridge70_corrected_minus_bridge62_corrected",
)


comparison_3 = comparison_summary(
    (
        "bridge70_FLAG_CORRECTED_minus_"
        "bridge70_PUBLISHED"
    ),
    point70_corr,
    point70_pub,
    "bridge70_corrected_minus_published",
)


# ==============================================================================
# 16. PRINT PRIMARY BOOTSTRAP RESULTS
# ==============================================================================

print("=" * 118)
print("PRIMARY PAIRED BOOTSTRAP RESULTS")
print("=" * 118)


for comparison in [
    comparison_1,
    comparison_2,
    comparison_3,
]:

    print()
    print(
        comparison[
            "comparison"
        ]
    )


    for metric in [
        "pr_auc",
        "roc_auc",
        "brier",
    ]:

        r = (
            comparison[
                "bootstrap"
            ][
                metric
            ]
        )


        print(
            f"  {metric:8s}"
            f" Δ={r['point_difference']:+.15f}"
            f"  95% CI=[{r['ci_2_5']:+.15f}, {r['ci_97_5']:+.15f}]"
            f"  excludes 0={r['ci_excludes_zero']}"
        )


print()

print(
    "bridge62 FLAG_CORRECTED - PUBLISHED:"
)

print(
    "  PR-AUC Δ = 0 exactly; CI = [0, 0]"
)

print(
    "  ROC-AUC Δ = 0 exactly; CI = [0, 0]"
)

print(
    "  Brier   Δ = 0 exactly; CI = [0, 0]"
)

print(
    "  Reason: bitwise-identical probability and label vectors."
)

print()


# ==============================================================================
# 17. SAVE RAW BOOTSTRAP REPLICATES
# ==============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


np.savez_compressed(
    BOOTSTRAP_PATH,

    replicate_index=
        np.arange(
            BOOTSTRAP_REPLICATES,
            dtype=np.int32,
        ),

    # Individual metrics.
    **bootstrap_metrics,

    # Paired differences.
    **differences,
)


bootstrap_file_sha = sha256_file(
    BOOTSTRAP_PATH
)


print("=" * 118)
print("BOOTSTRAP ARTIFACT")
print("=" * 118)

print(
    "File:",
    BOOTSTRAP_PATH.relative_to(
        REPO
    ),
)

print(
    "Bytes:",
    f"{BOOTSTRAP_PATH.stat().st_size:,}",
)

print(
    "SHA256:",
    bootstrap_file_sha,
)

print()


# ==============================================================================
# 18. PRIMARY RESULTS FREEZE RECEIPT
# ==============================================================================

primary_summary = {
    "bridge62_PUBLISHED": {
        "probability_sha256":
            EXPECTED_P62_SHA,

        "pr_auc":
            point62[
                "pr_auc"
            ],

        "roc_auc":
            point62[
                "roc_auc"
            ],

        "brier":
            point62[
                "brier"
            ],
    },

    "bridge62_FLAG_CORRECTED": {
        "probability_sha256":
            EXPECTED_P62_SHA,

        "identity":
            "BITWISE_IDENTICAL_TO_BRIDGE62_PUBLISHED",

        "pr_auc":
            point62[
                "pr_auc"
            ],

        "roc_auc":
            point62[
                "roc_auc"
            ],

        "brier":
            point62[
                "brier"
            ],
    },

    "bridge70_PUBLISHED": {
        "probability_sha256":
            EXPECTED_P70_PUBLISHED_SHA,

        "pr_auc":
            point70_pub[
                "pr_auc"
            ],

        "roc_auc":
            point70_pub[
                "roc_auc"
            ],

        "brier":
            point70_pub[
                "brier"
            ],
    },

    "bridge70_FLAG_CORRECTED": {
        "probability_sha256":
            EXPECTED_P70_CORRECTED_SHA,

        "pr_auc":
            point70_corr[
                "pr_auc"
            ],

        "roc_auc":
            point70_corr[
                "roc_auc"
            ],

        "brier":
            point70_corr[
                "brier"
            ],
    },

    "bridge62_GROUNDED_S4": {
        "status":
            "ADMINISTRATIVELY_CANCELLED_BEFORE_OPENING",

        "reason":
            "GROUNDED_S4_DURABLE_ARTIFACT_INFEASIBILITY",
    },

    "bridge70_GROUNDED_S4": {
        "status":
            "ADMINISTRATIVELY_CANCELLED_BEFORE_OPENING",

        "reason":
            "GROUNDED_S4_DURABLE_ARTIFACT_INFEASIBILITY",
    },
}


result = {
    "stage":
        "Stage24-3A",

    "status":
        "PRIMARY_DIRECTION_RESULTS_FROZEN",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "direction":
        "IDS2018_TO_CICIDS2017",

    "parent_commit":
        EXPECTED_PARENT,

    "target_population": {
        "rows":
            EXPECTED_ROWS,

        "benign":
            EXPECTED_BENIGN,

        "attack":
            EXPECTED_ATTACK,

        "binary_label_sha256":
            EXPECTED_LABEL_SHA,
    },

    "primary_cells":
        primary_summary,

    "bootstrap_protocol": {
        "method":
            "PAIRED_STRATIFIED_ROW_BOOTSTRAP",

        "implementation":
            (
                "EXACT_JOINT_PREDICTION_LABEL_EQUIVALENCE_GROUP_"
                "MULTINOMIAL_SUFFICIENT_STATISTIC"
            ),

        "implementation_is_approximation":
            False,

        "equivalence_key":
            [
                "binary_label",
                "bridge62_float32_probability",
                "bridge70_PUBLISHED_float32_probability",
                "bridge70_FLAG_CORRECTED_float32_probability",
            ],

        "replicates":
            BOOTSTRAP_REPLICATES,

        "seed":
            BOOTSTRAP_SEED,

        "strata":
            [
                "BENIGN",
                "ATTACK",
            ],

        "sampling":
            "WITH_REPLACEMENT_WITHIN_EACH_STRATUM",

        "stratum_sizes_preserved":
            True,

        "pairing":
            (
                "SAME_BOOTSTRAP_MULTIPLICITIES_USED_FOR_ALL_"
                "PRIMARY_PREDICTION_VECTORS"
            ),

        "confidence_interval":
            "PERCENTILE_2.5_97.5",

        "percentile_method":
            "NUMPY_LINEAR",

        "metrics":
            [
                "PR_AUC",
                "ROC_AUC",
                "Brier",
            ],

        "auc_semantics": {
            "PR_AUC":
                (
                    "DISTINCT_SCORE_GROUPED_NONINTERPOLATED_"
                    "AVERAGE_PRECISION"
                ),

            "ROC_AUC":
                (
                    "DISTINCT_SCORE_GROUPED_TRAPEZOIDAL_ROC_AREA"
                ),

            "scores":
                "PERSISTED_FLOAT32_PROBABILITIES",
        },

        "exact_grouping": {
            "benign_rows":
                EXPECTED_BENIGN,

            "benign_groups":
                G0,

            "attack_rows":
                EXPECTED_ATTACK,

            "attack_groups":
                G1,

            "total_rows":
                EXPECTED_ROWS,

            "total_groups":
                G,

            "compression_ratio":
                float(
                    EXPECTED_ROWS
                    /
                    G
                ),
        },

        "runtime_seconds":
            bootstrap_seconds,
    },

    "paired_comparisons": [
        comparison_1,
        comparison_2,
        comparison_3,
    ],

    "bridge62_structural_identity": {
        "comparison":
            "bridge62_FLAG_CORRECTED_minus_bridge62_PUBLISHED",

        "probability_arrays_bitwise_identical":
            True,

        "pr_auc_difference":
            0.0,

        "pr_auc_ci":
            [
                0.0,
                0.0,
            ],

        "roc_auc_difference":
            0.0,

        "roc_auc_ci":
            [
                0.0,
                0.0,
            ],

        "brier_difference":
            0.0,

        "brier_ci":
            [
                0.0,
                0.0,
            ],
    },

    "bootstrap_artifact": {
        "file":
            str(
                BOOTSTRAP_PATH.relative_to(
                    REPO
                )
            ),

        "sha256":
            bootstrap_file_sha,
    },

    "receipt_hashes":
        receipt_hashes,

    "anti_adaptation": {
        "new_target_opening":
            False,

        "new_target_inference":
            False,

        "new_model_fit":
            False,

        "target_threshold_selection":
            False,

        "feature_search":
            False,

        "feature_remapping":
            False,

        "target_calibration":
            False,

        "target_fitted_preprocessing":
            False,

        "post_result_population_change":
            False,

        "performance_based_protocol_change":
            False,
    },

    "opening_accounting": {
        "original_budget":
            8,

        "primary_openings_consumed":
            4,

        "primary_grounded_cells_cancelled_before_opening":
            2,

        "secondary_openings_remaining":
            2,

        "total_openings_consumed":
            4,

        "total_administratively_cancelled":
            2,

        "remaining_evaluable_frozen_cells":
            2,

        "cancelled_slots_reassignable":
            False,
    },

    "scientific_fit_accounting": {
        "budget":
            4,

        "completed":
            2,

        "remaining":
            2,

        "new_fits_this_step":
            0,
    },

    "next_authorized_step":
        (
            "TRAIN_SECONDARY_CICIDS2017_TO_IDS2018_XGBOOST_"
            "BRIDGE62_AND_BRIDGE70_ON_FROZEN_MON_WED_SOURCE_"
            "WITH_FROZEN_THURSDAY_SOURCE_VALIDATION"
        ),
}


RESULT_PATH.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


result_sha = sha256_file(
    RESULT_PATH
)


RESULT_SHA_PATH.write_text(
    f"{result_sha}  {RESULT_PATH.name}\n",
    encoding="utf-8",
)


CHECKSUMS_PATH.write_text(
    (
        f"{result_sha}  {RESULT_PATH.name}\n"
        f"{bootstrap_file_sha}  {BOOTSTRAP_PATH.name}\n"
    ),
    encoding="utf-8",
)


print("=" * 118)
print("PRIMARY FREEZE ARTIFACT")
print("=" * 118)

print(
    "Result:",
    RESULT_PATH.relative_to(
        REPO
    ),
)

print(
    "Result SHA256:",
    result_sha,
)

print()


# ==============================================================================
# 19. GITHUB CREDENTIAL / REMOTE PARENT
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if value and value.strip():

            github_token = value.strip()
            token_source = (
                "ENV:"
                +
                name
            )

            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main changed before primary freeze.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main still equals Stage24-2D."
)

print()


# ==============================================================================
# 20. GIT AUTHOR SAFETY
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    git_cmd(
        "config",
        "--local",
        "user.name",
        git_cmd(
            "log",
            "-1",
            "--format=%an",
        ),
    )


if not author_email:

    git_cmd(
        "config",
        "--local",
        "user.email",
        git_cmd(
            "log",
            "-1",
            "--format=%ae",
        ),
    )


# ==============================================================================
# 21. GIT FREEZE
# ==============================================================================

print("=" * 118)
print("GIT FREEZE")
print("=" * 118)


git_cmd(
    "add",
    "--",
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        BOOTSTRAP_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


expected_staged = {
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        BOOTSTRAP_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: freeze primary results and paired bootstrap",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage24-3A commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 22. PUSH / REMOTE VERIFY
# ==============================================================================

print("=" * 118)
print("PUSH + REMOTE VERIFY")
print("=" * 118)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository is not clean after primary freeze:\n"
        + final_status
    )


print(
    "[PASS] Remote main == local Stage24-3A commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 23. FINAL
# ==============================================================================

print("=" * 118)
print("STAGE24-3A PRIMARY DIRECTION FREEZE: PASS")
print("=" * 118)

print()

print(
    "Direction:                       IDS2018 -> CICIDS2017"
)

print(
    "Target rows:                     ",
    f"{EXPECTED_ROWS:,}",
)

print()

print(
    "Primary evaluable cells frozen:   4"
)

print(
    "Primary GROUNDED_S4 cancelled:    2"
)

print()

print(
    "Bootstrap replicates:             2000"
)

print(
    "Bootstrap seed:                   42"
)

print(
    "Bootstrap pairing:                PASS"
)

print(
    "Bootstrap stratification:         BENIGN / ATTACK"
)

print(
    "Exact grouped implementation:     YES"
)

print()

print(
    "New target openings:              0"
)

print(
    "Target openings consumed total:   4 / 8"
)

print(
    "Secondary openings remaining:     2"
)

print()

print(
    "New scientific fits:              0"
)

print(
    "Scientific fits completed:        2 / 4"
)

print(
    "Scientific fits remaining:        2"
)

print()

print(
    "Bootstrap artifact SHA:"
)

print(
    " ",
    bootstrap_file_sha,
)

print()

print(
    "Primary freeze result SHA:"
)

print(
    " ",
    result_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "NEXT:"
)

print(
    "  Stage24 secondary direction:"
)

print(
    "  CICIDS2017 -> IDS2018"
)

print(
    "  Train frozen XGBoost bridge62 + bridge70 source models"
)

print(
    "  on CICIDS2017 Monday-Wednesday, validate Thursday."
)

print("=" * 118)

STAGE24-3A — PRIMARY RESULTS FREEZE + PAIRED STRATIFIED BOOTSTRAP

GOVERNANCE
HEAD: a2a197ec01d30f09f140c157cdddc2e1e870512a
[PASS] Repository clean.
[PASS] No prior Stage24-3A artifacts.

PRIMARY RECEIPT VERIFICATION
stage24_2a_bridge62_published_result.json 1d4b3010fdc8b0f7f62a0fed2c373824ce99cb2f16936c021756749a9ca54749
stage24_2b_bridge62_flag_corrected_identity_result.json f894c63a0b6aa3ac0a616fe07aafa7e94f963fbcfd83408df43e37f52fdefde1
stage24_2c_bridge70_published_result.json c06823a29303b5c4e5934fc585993af3e2c4b477d079934c8e48afd9cacb257b
stage24_2d_bridge70_flag_corrected_result.json 5b819f03d2f336dde1415cf08f48f1963fba15d725661dcebe3c37f5f5aaf53c

[PASS] All four evaluable primary opening receipts byte-verified.

[PASS] Primary opening statuses exact.
[PASS] bridge62 PUBLISHED == FLAG_CORRECTED remains bitwise exact.

PRIMARY PREDICTION VECTOR VERIFICATION
Binary label SHA:         0e4edd0670c9cedde571e08ab92fedbe6acd6e2aa605d7fc62d286791c00740a
bridge62 SHA:             e928

In [37]:
# ==============================================================================
# STAGE24-4A — SECONDARY SOURCE MODEL #1
# CICIDS2017 -> IDS2018
# bridge62 XGBOOST
#
# SCIENTIFIC FIT #3 / 4
#
# Frozen secondary design:
#   learner               = XGBOOST_ONLY
#   configuration         = INHERIT_STAGE22R_XGB_11
#   source train          = CICIDS2017 Monday + Tuesday + Wednesday
#   source validation     = CICIDS2017 Thursday
#   source unused         = Friday
#   representation        = bridge62
#   feature semantics     = frozen semantic bridge
#   hyperparameter search = NONE
#
# Frozen XGB_11:
#   objective        = binary:logistic
#   n_estimators     = 400
#   max_depth        = 7
#   learning_rate    = 0.06
#   subsample        = 0.9
#   colsample_bytree = 1.0
#   min_child_weight = 1
#   gamma            = 0
#   reg_alpha        = 0
#   reg_lambda       = 3
#   tree_method      = hist
#   device           = cuda
#   eval_metric      = logloss
#   random_state     = 42
#   n_jobs           = -1
#
# Numeric policy:
#   parse float64
#   +inf -> NaN
#   -inf -> NaN
#   explicit imputation = NONE
#   scaling = NONE
#
# Threshold selection:
#   SOURCE VALIDATION ONLY
#   integer-percent grid 5..95
#
#   BALANCED:
#       maximize F1
#       ties:
#         lower FPR
#         higher recall
#         closer to 0.50
#         lower threshold
#
#   SECURITY:
#       require FPR <= 0.05
#       maximize F2
#       ties:
#         lower FPR
#         higher recall
#         lower threshold
#
# THIS CELL:
#   - reads NO IDS2018 target features
#   - performs NO target inference
#   - consumes NO target opening
#   - performs exactly ONE scientific fit
# ==============================================================================

from __future__ import annotations

import os
import gc
import csv
import json
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import duckdb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


print("=" * 118)
print("STAGE24-4A — SECONDARY SOURCE MODEL #1 — bridge62 XGBOOST")
print("=" * 118)
print()


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "d624db8f14f96f9d2bde176df99653555cea842c"
)

EXPECTED_BRIDGE_SPEC_SHA = (
    "b7ff1d8563aafc69145be3acc827fbc59ae75d880b9804bad727f408299fce2f"
)

EXPECTED_PROTOCOL_LOCK_SHA = (
    "8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2"
)

EXPECTED_TRAIN_ROWS = 1_668_530
EXPECTED_TRAIN_BENIGN = 1_402_023
EXPECTED_TRAIN_ATTACK = 266_507

EXPECTED_VAL_ROWS = 458_968
EXPECTED_VAL_BENIGN = 456_752
EXPECTED_VAL_ATTACK = 2_216

EXPECTED_TARGET_OPENINGS_CONSUMED = 4
EXPECTED_FITS_BEFORE = 2
EXPECTED_FITS_AFTER = 3


# ==============================================================================
# 1. PATHS
# ==============================================================================

LOCK_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
)

BRIDGE_SPEC_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.json"
)

BRIDGE_SPEC_SHA_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.sha256"
)

FINAL_LOCK_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.json"
)

FINAL_LOCK_SHA_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.sha256"
)


PRIMARY_FREEZE_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_3_primary_results_freeze"
    / "stage24_3a_primary_paired_bootstrap"
)

PRIMARY_FREEZE_PATH = (
    PRIMARY_FREEZE_DIR
    / "stage24_3a_primary_results_freeze.json"
)

PRIMARY_FREEZE_SHA_PATH = (
    PRIMARY_FREEZE_DIR
    / "stage24_3a_primary_results_freeze.sha256"
)


TARGET_ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/"
    "snapshots/b7e532345512edcd530cb1770dc76636aeb52802/"
    "traffic_labels"
)


OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_4_secondary_source_training"
    / "stage24_4a_bridge62_xgboost"
)

MODEL_PATH = (
    OUT
    / "secondary_bridge62_xgboost_model.json"
)

VAL_PRED_PATH = (
    OUT
    / "secondary_bridge62_validation_probabilities.npz"
)

THRESHOLD_GRID_PATH = (
    OUT
    / "secondary_bridge62_validation_threshold_grid.csv"
)

RESULT_PATH = (
    OUT
    / "stage24_4a_secondary_bridge62_result.json"
)

RESULT_SHA_PATH = (
    OUT
    / "stage24_4a_secondary_bridge62_result.sha256"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


# Fit marker exists only in /kaggle/working.
# If it exists after a crash, DO NOT rerun the fit blindly.
FIT_LEDGER = Path(
    "/kaggle/working/stage24_secondary_fit_runtime_ledger.json"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(args, *, cwd=None, check=True):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(*args, auth_header=None, check=True):

    cmd = ["git"]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [
        str(x)
        for x in args
    ]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array(array):

    arr = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        arr.view(
            np.uint8
        )
    ).hexdigest()


def verify_sidecar(artifact, sidecar):

    if not artifact.is_file():

        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )

    if not sidecar.is_file():

        raise RuntimeError(
            f"Missing SHA sidecar:\n{sidecar}"
        )


    actual = sha256_file(
        artifact
    )

    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )


    if actual != expected:

        raise RuntimeError(
            "\nSHA mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


    return actual


def qident(value):

    return (
        '"'
        +
        str(value).replace(
            '"',
            '""',
        )
        +
        '"'
    )


def sql_path(path):

    return str(path).replace(
        "'",
        "''",
    )


# ==============================================================================
# 3. GOVERNANCE GATE
# ==============================================================================

print("=" * 118)
print("GOVERNANCE")
print("=" * 118)


head = git_cmd(
    "rev-parse",
    "HEAD",
)

status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


if FIT_LEDGER.exists():

    raise RuntimeError(
        "\nA secondary scientific-fit runtime ledger already exists:\n"
        f"{FIT_LEDGER}\n\n"
        "DO NOT rerun the fit blindly."
    )


if OUT.exists() and any(
    OUT.iterdir()
):

    raise RuntimeError(
        "\nStage24-4A output already exists.\n"
        "Refusing accidental duplicate scientific fit."
    )


print(
    "[PASS] Repository clean."
)

print(
    "[PASS] No unresolved secondary-fit marker."
)

print(
    "[PASS] Stage24-4A has not previously been fit."
)

print()


# ==============================================================================
# 4. VERIFY FROZEN PROTOCOL + PRIMARY FREEZE
# ==============================================================================

print("=" * 118)
print("FROZEN CONTRACT")
print("=" * 118)


bridge_sha = verify_sidecar(
    BRIDGE_SPEC_PATH,
    BRIDGE_SPEC_SHA_PATH,
)

lock_sha = verify_sidecar(
    FINAL_LOCK_PATH,
    FINAL_LOCK_SHA_PATH,
)

primary_freeze_sha = verify_sidecar(
    PRIMARY_FREEZE_PATH,
    PRIMARY_FREEZE_SHA_PATH,
)


if bridge_sha != EXPECTED_BRIDGE_SPEC_SHA:

    raise RuntimeError(
        "Semantic bridge SHA changed."
    )


if lock_sha != EXPECTED_PROTOCOL_LOCK_SHA:

    raise RuntimeError(
        "Protocol lock SHA changed."
    )


final_lock = json.loads(
    FINAL_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

primary_freeze = json.loads(
    PRIMARY_FREEZE_PATH.read_text(
        encoding="utf-8"
    )
)


secondary = final_lock[
    "secondary_direction"
]


if secondary[
    "direction"
] != "CICIDS2017_TO_IDS2018":

    raise RuntimeError(
        "Secondary direction changed."
    )


if secondary[
    "learner"
] != "XGBOOST_ONLY":

    raise RuntimeError(
        "Secondary learner changed."
    )


if secondary[
    "learner_configuration"
] != "INHERIT_STAGE22R_XGB_11":

    raise RuntimeError(
        "Secondary learner configuration changed."
    )


if secondary[
    "source_train_days"
] != [
    "Monday",
    "Tuesday",
    "Wednesday",
]:

    raise RuntimeError(
        "Secondary source-train days changed."
    )


if secondary[
    "source_validation_days"
] != [
    "Thursday"
]:

    raise RuntimeError(
        "Secondary validation day changed."
    )


if secondary[
    "source_unused_days"
] != [
    "Friday"
]:

    raise RuntimeError(
        "Secondary unused source day changed."
    )


if secondary[
    "target"
] != "IDS2018_02-28-2018":

    raise RuntimeError(
        "Secondary target changed."
    )


if int(
    secondary[
        "target_expected_clean_rows"
    ]
) != 593_780:

    raise RuntimeError(
        "Secondary target expected-row count changed."
    )


if secondary[
    "hyperparameter_search"
] is not False:

    raise RuntimeError(
        "Secondary hyperparameter-search contract changed."
    )


if int(
    primary_freeze[
        "opening_accounting"
    ][
        "total_openings_consumed"
    ]
) != EXPECTED_TARGET_OPENINGS_CONSUMED:

    raise RuntimeError(
        "Primary freeze opening accounting changed."
    )


if int(
    primary_freeze[
        "scientific_fit_accounting"
    ][
        "completed"
    ]
) != EXPECTED_FITS_BEFORE:

    raise RuntimeError(
        "Primary freeze scientific-fit accounting changed."
    )


print(
    "Semantic bridge SHA:",
    bridge_sha,
)

print(
    "Protocol lock SHA:  ",
    lock_sha,
)

print(
    "Primary freeze SHA:  ",
    primary_freeze_sha,
)

print()

print(
    "[PASS] Secondary direction = CICIDS2017 -> IDS2018."
)

print(
    "[PASS] XGBOOST_ONLY / INHERIT_STAGE22R_XGB_11."
)

print(
    "[PASS] Train = Monday-Wednesday; validation = Thursday; Friday unused."
)

print(
    "[PASS] IDS2018 Feb-28 target remains unopened."
)

print()


# ==============================================================================
# 5. FROZEN BRIDGE62 SOURCE REPRESENTATION
#
# For bridge62:
#     PUBLISHED == FLAG_CORRECTED
#
# We use the FLAG_CORRECTED semantic mapping explicitly because CICIDS2017 is
# now the SOURCE domain and IDS2018 semantic feature order is the canonical
# bridge order.
# ==============================================================================

bridge_spec = json.loads(
    BRIDGE_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)


b62 = bridge_spec[
    "bridge62"
]


FEATURE_ORDER = list(
    b62[
        "source_feature_order"
    ]
)


SOURCE_MAPPING = dict(
    b62[
        "FLAG_CORRECTED_mapping"
    ]
)


if len(
    FEATURE_ORDER
) != 62:

    raise RuntimeError(
        "bridge62 feature count changed."
    )


if (
    b62[
        "PUBLISHED_mapping"
    ]
    !=
    b62[
        "FLAG_CORRECTED_mapping"
    ]
):

    raise RuntimeError(
        "bridge62 source semantics unexpectedly variant-dependent."
    )


if set(
    FEATURE_ORDER
) != set(
    SOURCE_MAPPING
):

    raise RuntimeError(
        "bridge62 source mapping incomplete."
    )


print("=" * 118)
print("BRIDGE62 SOURCE REPRESENTATION")
print("=" * 118)

print(
    "Features:",
    len(
        FEATURE_ORDER
    ),
)

print(
    "CICIDS2017 source semantics:",
    "FLAG_CORRECTED == PUBLISHED for bridge62",
)

print()

print(
    "[PASS] Frozen 62-feature source representation."
)

print()


# ==============================================================================
# 6. FROZEN SOURCE MEMBERSHIPS
# ==============================================================================

TRAIN_FILES = [
    (
        "Monday",
        "Monday-WorkingHours.pcap_ISCX.csv.parquet",
        529_918,
    ),
    (
        "Tuesday",
        "Tuesday-WorkingHours.pcap_ISCX.csv.parquet",
        445_909,
    ),
    (
        "Wednesday",
        "Wednesday-workingHours.pcap_ISCX.csv.parquet",
        692_703,
    ),
]


VALIDATION_FILES = [
    (
        "Thursday-Afternoon",
        "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet",
        288_602,
    ),
    (
        "Thursday-Morning",
        "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet",
        170_366,
    ),
]


print("=" * 118)
print("SECONDARY SOURCE MEMBERSHIP")
print("=" * 118)


print(
    "TRAIN"
)

for day, filename, rows in TRAIN_FILES:

    print(
        f"  {day:20s} {rows:9,d}  {filename}"
    )


print(
    "  TOTAL",
    f"{sum(x[2] for x in TRAIN_FILES):,}",
)

print()

print(
    "VALIDATION"
)

for day, filename, rows in VALIDATION_FILES:

    print(
        f"  {day:20s} {rows:9,d}  {filename}"
    )


print(
    "  TOTAL",
    f"{sum(x[2] for x in VALIDATION_FILES):,}",
)

print()


if sum(
    x[2]
    for x in TRAIN_FILES
) != EXPECTED_TRAIN_ROWS:

    raise RuntimeError(
        "Frozen secondary training row arithmetic changed."
    )


if sum(
    x[2]
    for x in VALIDATION_FILES
) != EXPECTED_VAL_ROWS:

    raise RuntimeError(
        "Frozen secondary validation row arithmetic changed."
    )


# ==============================================================================
# 7. PREALLOCATE SOURCE MATRICES
# ==============================================================================

print("=" * 118)
print("SOURCE MATERIALIZATION")
print("=" * 118)


X_train = np.empty(
    (
        EXPECTED_TRAIN_ROWS,
        62,
    ),
    dtype=np.float64,
)

y_train = np.empty(
    EXPECTED_TRAIN_ROWS,
    dtype=np.uint8,
)


X_val = np.empty(
    (
        EXPECTED_VAL_ROWS,
        62,
    ),
    dtype=np.float64,
)

y_val = np.empty(
    EXPECTED_VAL_ROWS,
    dtype=np.uint8,
)


# ==============================================================================
# 8. LABEL RULE
# ==============================================================================

DASH_TRANSLATION = str.maketrans(
    {
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2015": "-",
        "\u2212": "-",
    }
)


def canonicalize_labels(series):

    if series.isna().any():

        raise RuntimeError(
            "NULL label survived frozen effective-row rule."
        )


    s = (
        series
        .astype(
            "string"
        )
        .str.strip()
        .str.translate(
            DASH_TRANSLATION
        )
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.casefold()
    )


    if s.isna().any():

        raise RuntimeError(
            "Label canonicalization generated NULL."
        )


    if (
        s.str.len()
        ==
        0
    ).any():

        raise RuntimeError(
            "Empty label encountered."
        )


    return s


# ==============================================================================
# 9. MATERIALIZER
# ==============================================================================

conn = duckdb.connect(
    database=":memory:"
)


projection_sql = ",\n".join(
    (
        f"CAST({qident(SOURCE_MAPPING[source_feature])} AS DOUBLE) "
        f"AS {qident(source_feature)}"
    )
    for source_feature in FEATURE_ORDER
)


def materialize_files(
    file_specs,
    X_out,
    y_out,
    expected_total,
    split_name,
):

    cursor = 0

    numeric_audit = []

    label_counter = Counter()


    for day, filename, expected_rows in file_specs:

        path = (
            TARGET_ROOT
            /
            filename
        )


        if not path.is_file():

            raise RuntimeError(
                f"Missing source Parquet:\n{path}"
            )


        query = f"""
            SELECT
                {projection_sql},
                CAST("Label" AS VARCHAR) AS "__label__"
            FROM read_parquet('{sql_path(path)}')
            WHERE "Label" IS NOT NULL
        """


        t0 = time.time()


        df = conn.execute(
            query
        ).df()


        observed_rows = len(
            df
        )


        if observed_rows != expected_rows:

            raise RuntimeError(
                "\nFrozen effective row mismatch.\n"
                f"Split:    {split_name}\n"
                f"File:     {filename}\n"
                f"Expected: {expected_rows:,}\n"
                f"Actual:   {observed_rows:,}"
            )


        X = df[
            FEATURE_ORDER
        ].to_numpy(
            dtype=np.float64,
            copy=True,
        )


        if X.shape != (
            expected_rows,
            62,
        ):

            raise RuntimeError(
                f"Unexpected matrix shape: {X.shape}"
            )


        positive_inf = int(
            np.isposinf(
                X
            ).sum()
        )

        negative_inf = int(
            np.isneginf(
                X
            ).sum()
        )


        inf_mask = np.isinf(
            X
        )


        if inf_mask.any():

            X[
                inf_mask
            ] = np.nan


        nan_count = int(
            np.isnan(
                X
            ).sum()
        )


        canonical = canonicalize_labels(
            df[
                "__label__"
            ]
        )


        local_counter = Counter(
            canonical.tolist()
        )

        label_counter.update(
            local_counter
        )


        y = (
            canonical
            !=
            "benign"
        ).to_numpy(
            dtype=np.uint8
        )


        start = cursor

        stop = (
            start
            +
            expected_rows
        )


        X_out[
            start:stop,
            :
        ] = X

        y_out[
            start:stop
        ] = y


        print(
            f"{split_name:10s} "
            f"{day:20s}"
            f" rows={expected_rows:9,d}"
            f" benign={int((y == 0).sum()):9,d}"
            f" attack={int((y == 1).sum()):8,d}"
            f" +inf={positive_inf:6,d}"
            f" nan={nan_count:6,d}"
            f" seconds={time.time() - t0:6.2f}"
        )


        numeric_audit.append(
            {
                "split":
                    split_name,

                "day":
                    day,

                "file":
                    filename,

                "rows":
                    expected_rows,

                "positive_inf_to_nan":
                    positive_inf,

                "negative_inf_to_nan":
                    negative_inf,

                "nan_cells_after_inf_conversion":
                    nan_count,
            }
        )


        cursor = stop


        del df
        del X
        del y
        del canonical
        del inf_mask

        gc.collect()


    if cursor != expected_total:

        raise RuntimeError(
            f"{split_name} final row count mismatch."
        )


    return (
        numeric_audit,
        label_counter,
    )


train_numeric_audit, train_label_counts = materialize_files(
    TRAIN_FILES,
    X_train,
    y_train,
    EXPECTED_TRAIN_ROWS,
    "TRAIN",
)


val_numeric_audit, val_label_counts = materialize_files(
    VALIDATION_FILES,
    X_val,
    y_val,
    EXPECTED_VAL_ROWS,
    "VALIDATION",
)


conn.close()


train_attack = int(
    y_train.sum()
)

train_benign = int(
    len(
        y_train
    )
    -
    train_attack
)


val_attack = int(
    y_val.sum()
)

val_benign = int(
    len(
        y_val
    )
    -
    val_attack
)


print()
print(
    "TRAIN totals:"
)

print(
    "  rows:  ",
    f"{len(y_train):,}",
)

print(
    "  benign:",
    f"{train_benign:,}",
)

print(
    "  attack:",
    f"{train_attack:,}",
)

print()

print(
    "VALIDATION totals:"
)

print(
    "  rows:  ",
    f"{len(y_val):,}",
)

print(
    "  benign:",
    f"{val_benign:,}",
)

print(
    "  attack:",
    f"{val_attack:,}",
)

print()


if (
    train_benign
    !=
    EXPECTED_TRAIN_BENIGN
):

    raise RuntimeError(
        "\nUnexpected frozen train benign count.\n"
        f"Expected: {EXPECTED_TRAIN_BENIGN:,}\n"
        f"Actual:   {train_benign:,}"
    )


if (
    train_attack
    !=
    EXPECTED_TRAIN_ATTACK
):

    raise RuntimeError(
        "\nUnexpected frozen train attack count.\n"
        f"Expected: {EXPECTED_TRAIN_ATTACK:,}\n"
        f"Actual:   {train_attack:,}"
    )


if (
    val_benign
    !=
    EXPECTED_VAL_BENIGN
):

    raise RuntimeError(
        "\nUnexpected frozen validation benign count.\n"
        f"Expected: {EXPECTED_VAL_BENIGN:,}\n"
        f"Actual:   {val_benign:,}"
    )


if (
    val_attack
    !=
    EXPECTED_VAL_ATTACK
):

    raise RuntimeError(
        "\nUnexpected frozen validation attack count.\n"
        f"Expected: {EXPECTED_VAL_ATTACK:,}\n"
        f"Actual:   {val_attack:,}"
    )


source_prior = float(
    train_attack
    /
    EXPECTED_TRAIN_ROWS
)


print(
    "Frozen secondary source prior:",
    f"{source_prior:.15f}",
)

print()

print(
    "[PASS] Secondary source membership and labels exact."
)

print()


# ==============================================================================
# 10. SOURCE MATRIX HASHES
# ==============================================================================

train_feature_sha = sha256_array(
    X_train
)

train_label_sha = sha256_array(
    y_train
)

val_feature_sha = sha256_array(
    X_val
)

val_label_sha = sha256_array(
    y_val
)


print("=" * 118)
print("SOURCE MATRIX IDENTITIES")
print("=" * 118)

print(
    "Train feature SHA:",
    train_feature_sha,
)

print(
    "Train label SHA:  ",
    train_label_sha,
)

print(
    "Val feature SHA:  ",
    val_feature_sha,
)

print(
    "Val label SHA:    ",
    val_label_sha,
)

print()


# ==============================================================================
# 11. CUDA PRE-FIT GATE
#
# Frozen secondary learner is XGBoost CUDA.
# No silent CPU fallback is allowed here.
# ==============================================================================

print("=" * 118)
print("CUDA PRE-FIT GATE")
print("=" * 118)


nvidia_output = run_cmd(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader",
    ],
    check=False,
)


if not nvidia_output:

    raise RuntimeError(
        "\nNo CUDA GPU detected BEFORE scientific fit #3.\n"
        "Fit has NOT been consumed."
    )


print(
    nvidia_output
)

print()


# ==============================================================================
# 12. GITHUB CREDENTIAL + REMOTE PARENT BEFORE FIT
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if value and value.strip():

            github_token = value.strip()
            token_source = (
                "ENV:"
                +
                name
            )

            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable BEFORE scientific fit."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved before scientific fit #3.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main == Stage24-3A."
)

print()


# ==============================================================================
# 13. FROZEN XGB_11 CONFIGURATION
# ==============================================================================

XGB_PARAMS = {
    "objective":
        "binary:logistic",

    "n_estimators":
        400,

    "max_depth":
        7,

    "learning_rate":
        0.06,

    "subsample":
        0.9,

    "colsample_bytree":
        1.0,

    "min_child_weight":
        1,

    "gamma":
        0.0,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "tree_method":
        "hist",

    "device":
        "cuda",

    "eval_metric":
        "logloss",

    "random_state":
        42,

    "n_jobs":
        -1,
}


print("=" * 118)
print("FROZEN XGB_11")
print("=" * 118)


for key in sorted(
    XGB_PARAMS
):

    print(
        f"{key:20s} = {XGB_PARAMS[key]}"
    )


print()


# ==============================================================================
# 14. SCIENTIFIC FIT #3 / 4
# ==============================================================================

import xgboost as xgb


print(
    "XGBoost version:",
    xgb.__version__,
)

print()


# Marker is written immediately before model.fit().
fit_started_at = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


FIT_LEDGER.write_text(
    json.dumps(
        {
            "stage":
                "Stage24-4A",

            "direction":
                "CICIDS2017_TO_IDS2018",

            "bridge":
                "bridge62",

            "learner":
                "XGBOOST_ONLY",

            "scientific_fit_number":
                3,

            "scientific_fit_budget":
                4,

            "fit_attempt_started":
                True,

            "fit_started_at_utc":
                fit_started_at,

            "parent_commit":
                EXPECTED_PARENT,

            "target_openings_consumed":
                4,

            "target_features_read":
                False,

            "target_predictions":
                0,
        },
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


print("=" * 118)
print("SCIENTIFIC FIT #3 / 4 — bridge62 XGBOOST")
print("=" * 118)

print()


model = xgb.XGBClassifier(
    **XGB_PARAMS
)


fit_start = time.time()


model.fit(
    X_train,
    y_train,
    verbose=False,
)


fit_seconds = (
    time.time()
    -
    fit_start
)


print(
    "Fit seconds:",
    f"{fit_seconds:.3f}",
)

print()


booster = model.get_booster()


if booster.num_features() != 62:

    raise RuntimeError(
        "\nFitted model dimension mismatch."
    )


if booster.num_boosted_rounds() != 400:

    raise RuntimeError(
        "\nFitted boosted-round count mismatch."
    )


config_text = booster.save_config()


if '"device":"cuda' not in config_text.replace(
    " ",
    ""
):

    raise RuntimeError(
        "\nFitted XGBoost configuration does not resolve to CUDA."
    )


print(
    "[PASS] Scientific fit #3 completed on CUDA."
)

print(
    "[PASS] 62 features / 400 boosted rounds."
)

print()


# Free train matrix before validation prediction.
del X_train

gc.collect()


# ==============================================================================
# 15. SOURCE-VALIDATION INFERENCE
#
# This is CICIDS2017 SOURCE validation, not IDS2018 target inference.
# ==============================================================================

print("=" * 118)
print("SOURCE VALIDATION — THURSDAY")
print("=" * 118)


val_start = time.time()


p_val_raw = model.predict_proba(
    X_val
)[
    :,
    1
]


p_val = np.asarray(
    p_val_raw,
    dtype=np.float32,
)


val_seconds = (
    time.time()
    -
    val_start
)


if p_val.shape != (
    EXPECTED_VAL_ROWS,
):

    raise RuntimeError(
        "Validation probability shape mismatch."
    )


if not np.isfinite(
    p_val
).all():

    raise RuntimeError(
        "Non-finite source-validation probabilities."
    )


if (
    (p_val < 0).any()
    or
    (p_val > 1).any()
):

    raise RuntimeError(
        "Source-validation probabilities outside [0,1]."
    )


val_probability_sha = sha256_array(
    p_val
)


print(
    "Validation inference seconds:",
    f"{val_seconds:.3f}",
)

print(
    "Probability min/max:",
    f"{float(p_val.min()):.9f}",
    "/",
    f"{float(p_val.max()):.9f}",
)

print(
    "Validation probability SHA:",
    val_probability_sha,
)

print()


# ==============================================================================
# 16. SOURCE-VALIDATION PR / ROC
# ==============================================================================

pr_auc = float(
    average_precision_score(
        y_val,
        p_val,
    )
)


roc_auc = float(
    roc_auc_score(
        y_val,
        p_val,
    )
)


val_prevalence = float(
    EXPECTED_VAL_ATTACK
    /
    EXPECTED_VAL_ROWS
)


pr_excess = float(
    pr_auc
    -
    val_prevalence
)


pr_normalized = float(
    (
        pr_auc
        -
        val_prevalence
    )
    /
    (
        1.0
        -
        val_prevalence
    )
)


print(
    "Validation prevalence:",
    f"{val_prevalence:.15f}",
)

print(
    "PR-AUC:              ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:             ",
    f"{roc_auc:.15f}",
)

print(
    "PR excess:           ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized:       ",
    f"{pr_normalized:.15f}",
)

print()


# ==============================================================================
# 17. FROZEN THRESHOLD GRID
# ==============================================================================

def threshold_row(
    pct,
):

    threshold = np.float32(
        pct
        /
        100.0
    )


    pred = (
        p_val
        >=
        threshold
    )


    y1 = (
        y_val
        ==
        1
    )

    y0 = ~y1


    tp = int(
        np.sum(
            pred
            &
            y1
        )
    )

    fp = int(
        np.sum(
            pred
            &
            y0
        )
    )

    fn = int(
        EXPECTED_VAL_ATTACK
        -
        tp
    )

    tn = int(
        EXPECTED_VAL_BENIGN
        -
        fp
    )


    return {
        "threshold_integer_percent":
            int(
                pct
            ),

        "threshold_float32_runtime":
            float(
                threshold
            ),

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn,
    }


grid = [
    threshold_row(
        pct
    )
    for pct in range(
        5,
        96,
    )
]


# ==============================================================================
# 18. EXACT RATIONAL OBJECTIVE HELPERS
# ==============================================================================

def f1_num_den(row):

    numerator = (
        2
        *
        row[
            "tp"
        ]
    )

    denominator = (
        2
        *
        row[
            "tp"
        ]
        +
        row[
            "fp"
        ]
        +
        row[
            "fn"
        ]
    )


    return (
        numerator,
        denominator,
    )


def f2_num_den(row):

    numerator = (
        5
        *
        row[
            "tp"
        ]
    )

    denominator = (
        5
        *
        row[
            "tp"
        ]
        +
        4
        *
        row[
            "fn"
        ]
        +
        row[
            "fp"
        ]
    )


    return (
        numerator,
        denominator,
    )


def rational_compare(
    n1,
    d1,
    n2,
    d2,
):

    left = (
        n1
        *
        d2
    )

    right = (
        n2
        *
        d1
    )


    if left > right:
        return 1

    if left < right:
        return -1

    return 0


# ==============================================================================
# 19. BALANCED THRESHOLD — EXACT FROZEN TIE BREAK
# ==============================================================================

balanced = None


for row in grid:

    if balanced is None:

        balanced = row
        continue


    n_new, d_new = f1_num_den(
        row
    )

    n_old, d_old = f1_num_den(
        balanced
    )


    objective_cmp = rational_compare(
        n_new,
        d_new,
        n_old,
        d_old,
    )


    if objective_cmp > 0:

        balanced = row
        continue


    if objective_cmp < 0:

        continue


    # Tie 1: LOWER_FPR.
    # Same validation benign denominator => lower FP.
    if row[
        "fp"
    ] < balanced[
        "fp"
    ]:

        balanced = row
        continue


    if row[
        "fp"
    ] > balanced[
        "fp"
    ]:

        continue


    # Tie 2: HIGHER_RECALL.
    # Same validation attack denominator => higher TP.
    if row[
        "tp"
    ] > balanced[
        "tp"
    ]:

        balanced = row
        continue


    if row[
        "tp"
    ] < balanced[
        "tp"
    ]:

        continue


    # Tie 3: CLOSER_TO_0_50.
    new_distance = abs(
        row[
            "threshold_integer_percent"
        ]
        -
        50
    )

    old_distance = abs(
        balanced[
            "threshold_integer_percent"
        ]
        -
        50
    )


    if new_distance < old_distance:

        balanced = row
        continue


    if new_distance > old_distance:

        continue


    # Tie 4: LOWER_THRESHOLD.
    if (
        row[
            "threshold_integer_percent"
        ]
        <
        balanced[
            "threshold_integer_percent"
        ]
    ):

        balanced = row


# ==============================================================================
# 20. SECURITY THRESHOLD
# ==============================================================================

security_candidates = [
    row
    for row in grid
    if (
        20
        *
        row[
            "fp"
        ]
    )
    <=
    EXPECTED_VAL_BENIGN
]


security = None


for row in security_candidates:

    if security is None:

        security = row
        continue


    n_new, d_new = f2_num_den(
        row
    )

    n_old, d_old = f2_num_den(
        security
    )


    objective_cmp = rational_compare(
        n_new,
        d_new,
        n_old,
        d_old,
    )


    if objective_cmp > 0:

        security = row
        continue


    if objective_cmp < 0:

        continue


    # Tie 1: lower FPR -> lower FP.
    if row[
        "fp"
    ] < security[
        "fp"
    ]:

        security = row
        continue


    if row[
        "fp"
    ] > security[
        "fp"
    ]:

        continue


    # Tie 2: higher recall -> higher TP.
    if row[
        "tp"
    ] > security[
        "tp"
    ]:

        security = row
        continue


    if row[
        "tp"
    ] < security[
        "tp"
    ]:

        continue


    # Tie 3: lower threshold.
    if (
        row[
            "threshold_integer_percent"
        ]
        <
        security[
            "threshold_integer_percent"
        ]
    ):

        security = row


# ==============================================================================
# 21. THRESHOLDED METRICS
# ==============================================================================

def expand_metrics(row):

    tp = row[
        "tp"
    ]

    tn = row[
        "tn"
    ]

    fp = row[
        "fp"
    ]

    fn = row[
        "fn"
    ]


    precision = (
        tp
        /
        (
            tp
            +
            fp
        )
        if (
            tp + fp
        ) > 0
        else 0.0
    )


    recall = (
        tp
        /
        (
            tp
            +
            fn
        )
        if (
            tp + fn
        ) > 0
        else 0.0
    )


    fpr = (
        fp
        /
        (
            fp
            +
            tn
        )
        if (
            fp + tn
        ) > 0
        else 0.0
    )


    fnr = (
        fn
        /
        (
            fn
            +
            tp
        )
        if (
            fn + tp
        ) > 0
        else 0.0
    )


    f1_num, f1_den = f1_num_den(
        row
    )

    f2_num, f2_den = f2_num_den(
        row
    )


    return {
        **row,

        "accuracy":
            float(
                (
                    tp
                    +
                    tn
                )
                /
                EXPECTED_VAL_ROWS
            ),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                f1_num
                /
                f1_den
                if f1_den
                else 0.0
            ),

        "f2":
            float(
                f2_num
                /
                f2_den
                if f2_den
                else 0.0
            ),

        "fpr":
            float(
                fpr
            ),

        "fnr":
            float(
                fnr
            ),
    }


standard = expand_metrics(
    grid[
        50
        -
        5
    ]
)


balanced_metrics = expand_metrics(
    balanced
)


security_metrics = (
    expand_metrics(
        security
    )
    if security is not None
    else None
)


print("=" * 118)
print("SOURCE-VALIDATION THRESHOLD SELECTION")
print("=" * 118)


print(
    "STANDARD:"
)

print(
    "  threshold:",
    standard[
        "threshold_float32_runtime"
    ],
)

print(
    "  TP/TN/FP/FN:",
    standard[
        "tp"
    ],
    standard[
        "tn"
    ],
    standard[
        "fp"
    ],
    standard[
        "fn"
    ],
)

print()


print(
    "BALANCED:"
)

print(
    "  threshold:",
    balanced_metrics[
        "threshold_float32_runtime"
    ],
)

print(
    "  integer percent:",
    balanced_metrics[
        "threshold_integer_percent"
    ],
)

print(
    "  F1:",
    f"{balanced_metrics['f1']:.15f}",
)

print(
    "  FPR:",
    f"{balanced_metrics['fpr']:.15f}",
)

print(
    "  recall:",
    f"{balanced_metrics['recall']:.15f}",
)

print(
    "  TP/TN/FP/FN:",
    balanced_metrics[
        "tp"
    ],
    balanced_metrics[
        "tn"
    ],
    balanced_metrics[
        "fp"
    ],
    balanced_metrics[
        "fn"
    ],
)

print()


print(
    "SECURITY:"
)


if security_metrics is None:

    print(
        "  UNAVAILABLE — no FPR <= 0.05 candidate."
    )

else:

    print(
        "  threshold:",
        security_metrics[
            "threshold_float32_runtime"
        ],
    )

    print(
        "  integer percent:",
        security_metrics[
            "threshold_integer_percent"
        ],
    )

    print(
        "  F2:",
        f"{security_metrics['f2']:.15f}",
    )

    print(
        "  FPR:",
        f"{security_metrics['fpr']:.15f}",
    )

    print(
        "  recall:",
        f"{security_metrics['recall']:.15f}",
    )

    print(
        "  TP/TN/FP/FN:",
        security_metrics[
            "tp"
        ],
        security_metrics[
            "tn"
        ],
        security_metrics[
            "fp"
        ],
        security_metrics[
            "fn"
        ],
    )


print()

print(
    "[PASS] Thresholds selected from CICIDS2017 Thursday source validation only."
)

print()


# ==============================================================================
# 22. PERSIST ARTIFACTS
# ==============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


# Save exact booster, not sklearn wrapper metadata only.
booster.save_model(
    MODEL_PATH
)


model_sha = sha256_file(
    MODEL_PATH
)


np.savez_compressed(
    VAL_PRED_PATH,

    probability=
        p_val,

    binary_label=
        y_val,
)


val_pred_file_sha = sha256_file(
    VAL_PRED_PATH
)


with THRESHOLD_GRID_PATH.open(
    "w",
    newline="",
    encoding="utf-8",
) as fh:

    writer = csv.DictWriter(
        fh,
        fieldnames=[
            "threshold_integer_percent",
            "threshold_float32_runtime",
            "tp",
            "tn",
            "fp",
            "fn",
        ],
    )

    writer.writeheader()

    writer.writerows(
        grid
    )


threshold_grid_sha = sha256_file(
    THRESHOLD_GRID_PATH
)


print("=" * 118)
print("PERSISTED SCIENTIFIC ARTIFACTS")
print("=" * 118)

print(
    "Model SHA:",
    model_sha,
)

print(
    "Validation probability array SHA:",
    val_probability_sha,
)

print(
    "Validation NPZ SHA:",
    val_pred_file_sha,
)

print(
    "Threshold grid SHA:",
    threshold_grid_sha,
)

print()


# ==============================================================================
# 23. RESULT RECEIPT
# ==============================================================================

result = {
    "stage":
        "Stage24-4A",

    "status":
        "SECONDARY_BRIDGE62_SOURCE_MODEL_FROZEN",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "direction":
        "CICIDS2017_TO_IDS2018",

    "bridge":
        "bridge62",

    "learner":
        "XGBOOST_ONLY",

    "learner_configuration":
        "INHERIT_STAGE22R_XGB_11",

    "parent_commit":
        EXPECTED_PARENT,

    "scientific_fit": {
        "fit_number":
            3,

        "fit_budget":
            4,

        "completed_before":
            2,

        "completed_after":
            3,

        "fit_started_at_utc":
            fit_started_at,

        "fit_seconds":
            fit_seconds,

        "hyperparameter_search":
            False,
    },

    "xgboost": {
        "library_version":
            xgb.__version__,

        "parameters":
            XGB_PARAMS,

        "resolved_device":
            "cuda",

        "feature_count":
            62,

        "boosted_rounds":
            400,

        "model_file":
            str(
                MODEL_PATH.relative_to(
                    REPO
                )
            ),

        "model_sha256":
            model_sha,
    },

    "representation": {
        "feature_count":
            62,

        "feature_order":
            FEATURE_ORDER,

        "cicids2017_source_mapping":
            SOURCE_MAPPING,

        "bridge62_published_equals_flag_corrected":
            True,

        "parse_dtype":
            "float64",

        "positive_infinity":
            "CONVERT_TO_NAN",

        "negative_infinity":
            "CONVERT_TO_NAN",

        "explicit_imputation":
            "NONE",

        "scaling":
            "NONE",
    },

    "source_train": {
        "dataset":
            "CICIDS2017",

        "days":
            [
                "Monday",
                "Tuesday",
                "Wednesday",
            ],

        "rows":
            EXPECTED_TRAIN_ROWS,

        "benign":
            train_benign,

        "attack":
            train_attack,

        "attack_prevalence":
            source_prior,

        "feature_matrix_sha256":
            train_feature_sha,

        "binary_label_sha256":
            train_label_sha,

        "canonical_label_counts":
            dict(
                sorted(
                    train_label_counts.items()
                )
            ),
    },

    "source_validation": {
        "dataset":
            "CICIDS2017",

        "days":
            [
                "Thursday",
            ],

        "rows":
            EXPECTED_VAL_ROWS,

        "benign":
            val_benign,

        "attack":
            val_attack,

        "prevalence":
            val_prevalence,

        "feature_matrix_sha256":
            val_feature_sha,

        "binary_label_sha256":
            val_label_sha,

        "canonical_label_counts":
            dict(
                sorted(
                    val_label_counts.items()
                )
            ),

        "probability_array_sha256":
            val_probability_sha,

        "probability_artifact":
            str(
                VAL_PRED_PATH.relative_to(
                    REPO
                )
            ),

        "probability_artifact_sha256":
            val_pred_file_sha,

        "pr_auc":
            pr_auc,

        "roc_auc":
            roc_auc,

        "pr_excess":
            pr_excess,

        "pr_normalized":
            pr_normalized,

        "standard":
            standard,

        "balanced":
            balanced_metrics,

        "security": {
            "status":
                (
                    "AVAILABLE"
                    if security_metrics is not None
                    else "UNAVAILABLE_NO_RELAXATION"
                ),

            "result":
                security_metrics,
        },

        "threshold_grid_file":
            str(
                THRESHOLD_GRID_PATH.relative_to(
                    REPO
                )
            ),

        "threshold_grid_sha256":
            threshold_grid_sha,

        "threshold_selection_data":
            "SOURCE_VALIDATION_ONLY",
    },

    "numeric_audit": (
        train_numeric_audit
        +
        val_numeric_audit
    ),

    "secondary_source_prior_constant_predictor": {
        "fit":
            False,

        "probability":
            source_prior,

        "computed_on":
            "FROZEN_CICIDS2017_MONDAY_WEDNESDAY_SOURCE_TRAIN_ONLY",
    },

    "unused_source": {
        "days":
            [
                "Friday",
            ],

        "read_for_this_fit":
            False,
    },

    "target_access": {
        "dataset":
            "IDS2018",

        "target":
            "02-28-2018",

        "target_features_read":
            False,

        "target_labels_read":
            False,

        "target_model_predictions":
            0,

        "target_metrics":
            0,

        "new_target_openings":
            0,

        "total_target_openings_consumed":
            4,
    },

    "anti_adaptation": {
        "hyperparameter_search":
            False,

        "target_guided_refit":
            False,

        "target_guided_threshold":
            False,

        "target_feature_search":
            False,

        "target_preprocessing_fit":
            False,

        "Friday_used":
            False,
    },

    "fit_accounting": {
        "scientific_fit_budget":
            4,

        "scientific_fits_completed":
            3,

        "scientific_fits_remaining":
            1,
    },

    "next_authorized_step":
        (
            "STAGE24_4B_SECONDARY_BRIDGE70_XGBOOST_SOURCE_FIT_"
            "ON_CICIDS2017_MON_WED_VALIDATE_THURSDAY"
        ),
}


RESULT_PATH.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


result_sha = sha256_file(
    RESULT_PATH
)


RESULT_SHA_PATH.write_text(
    f"{result_sha}  {RESULT_PATH.name}\n",
    encoding="utf-8",
)


CHECKSUMS_PATH.write_text(
    (
        f"{result_sha}  {RESULT_PATH.name}\n"
        f"{model_sha}  {MODEL_PATH.name}\n"
        f"{val_pred_file_sha}  {VAL_PRED_PATH.name}\n"
        f"{threshold_grid_sha}  {THRESHOLD_GRID_PATH.name}\n"
    ),
    encoding="utf-8",
)


print(
    "Result SHA:",
    result_sha,
)

print()


# ==============================================================================
# 24. GIT AUTHOR SAFETY
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    git_cmd(
        "config",
        "--local",
        "user.name",
        git_cmd(
            "log",
            "-1",
            "--format=%an",
        ),
    )


if not author_email:

    git_cmd(
        "config",
        "--local",
        "user.email",
        git_cmd(
            "log",
            "-1",
            "--format=%ae",
        ),
    )


# ==============================================================================
# 25. GIT FREEZE
# ==============================================================================

print("=" * 118)
print("GIT FREEZE")
print("=" * 118)


git_cmd(
    "add",
    "--",
    str(
        MODEL_PATH.relative_to(
            REPO
        )
    ),
    str(
        VAL_PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        THRESHOLD_GRID_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


expected_staged = {
    str(
        MODEL_PATH.relative_to(
            REPO
        )
    ),
    str(
        VAL_PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        THRESHOLD_GRID_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: freeze secondary bridge62 source model",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage24-4A commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 26. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 118)
print("PUSH + REMOTE VERIFY")
print("=" * 118)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository not clean after Stage24-4A:\n"
        + final_status
    )


# Scientific fit is now durable.
if FIT_LEDGER.exists():

    FIT_LEDGER.unlink()


print(
    "[PASS] Remote main == local Stage24-4A commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 27. FINAL
# ==============================================================================

print("=" * 118)
print("STAGE24-4A SECONDARY BRIDGE62 SOURCE FIT: PASS")
print("=" * 118)

print()

print(
    "Direction:                    CICIDS2017 -> IDS2018"
)

print(
    "Bridge:                       bridge62"
)

print(
    "Learner:                      XGBoost"
)

print()

print(
    "Train rows:                   ",
    f"{EXPECTED_TRAIN_ROWS:,}",
)

print(
    "Train benign:                 ",
    f"{train_benign:,}",
)

print(
    "Train attack:                 ",
    f"{train_attack:,}",
)

print(
    "Source prior:                 ",
    f"{source_prior:.12f}",
)

print()

print(
    "Validation rows:              ",
    f"{EXPECTED_VAL_ROWS:,}",
)

print(
    "Validation benign:            ",
    f"{val_benign:,}",
)

print(
    "Validation attack:            ",
    f"{val_attack:,}",
)

print()

print(
    "Validation PR-AUC:            ",
    f"{pr_auc:.15f}",
)

print(
    "Validation ROC-AUC:           ",
    f"{roc_auc:.15f}",
)

print()

print(
    "Balanced threshold:           ",
    balanced_metrics[
        "threshold_float32_runtime"
    ],
)

print(
    "Security threshold:           ",
    (
        security_metrics[
            "threshold_float32_runtime"
        ]
        if security_metrics is not None
        else "UNAVAILABLE"
    ),
)

print()

print(
    "Model SHA:"
)

print(
    " ",
    model_sha,
)

print()

print(
    "Validation probability SHA:"
)

print(
    " ",
    val_probability_sha,
)

print()

print(
    "Scientific fits completed:    3 / 4"
)

print(
    "Scientific fits remaining:    1"
)

print()

print(
    "New target openings:          0"
)

print(
    "Target openings consumed:     4 / 8"
)

print(
    "IDS2018 Feb-28 target opened: NO"
)

print()

print(
    "Result SHA:"
)

print(
    " ",
    result_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "NEXT:"
)

print(
    "  Stage24-4B — secondary bridge70 XGBoost source fit"
)

print(
    "  (scientific fit #4 / 4)."
)

print("=" * 118)

STAGE24-4A — SECONDARY SOURCE MODEL #1 — bridge62 XGBOOST

GOVERNANCE
HEAD: d624db8f14f96f9d2bde176df99653555cea842c
[PASS] Repository clean.
[PASS] No unresolved secondary-fit marker.
[PASS] Stage24-4A has not previously been fit.

FROZEN CONTRACT
Semantic bridge SHA: b7ff1d8563aafc69145be3acc827fbc59ae75d880b9804bad727f408299fce2f
Protocol lock SHA:   8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2
Primary freeze SHA:   bdb6471be8b154662149eeb616dfdfb78f978dd3f337659e773ad9a21b3ba42f

[PASS] Secondary direction = CICIDS2017 -> IDS2018.
[PASS] XGBOOST_ONLY / INHERIT_STAGE22R_XGB_11.
[PASS] Train = Monday-Wednesday; validation = Thursday; Friday unused.
[PASS] IDS2018 Feb-28 target remains unopened.

BRIDGE62 SOURCE REPRESENTATION
Features: 62
CICIDS2017 source semantics: FLAG_CORRECTED == PUBLISHED for bridge62

[PASS] Frozen 62-feature source representation.

SECONDARY SOURCE MEMBERSHIP
TRAIN
  Monday                 529,918  Monday-WorkingHours.pcap_ISCX.csv.parquet

FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

In [38]:
# ==============================================================================
# STAGE24-4A-EX1 — SECONDARY XGBOOST CPU BACKEND AMENDMENT
#
# Context:
#   Stage24-4A bridge62 secondary fit was NOT attempted.
#   Execution stopped at the pre-fit CUDA gate because this Kaggle session
#   intentionally has GPU disabled and `nvidia-smi` is unavailable.
#
# Scientific state BEFORE this amendment:
#   fits completed        = 2 / 4
#   fit #3 attempted      = NO
#   target openings       = 4 / 8
#   IDS2018 Feb-28 opened = NO
#
# Amendment:
#   XGBoost execution device:
#
#       cuda -> cpu
#
# Everything else remains frozen:
#   learner               = XGBOOST_ONLY
#   configuration         = INHERIT_STAGE22R_XGB_11
#   tree_method           = hist
#   n_estimators          = 400
#   max_depth             = 7
#   learning_rate         = 0.06
#   subsample             = 0.9
#   colsample_bytree      = 1.0
#   min_child_weight      = 1
#   gamma                 = 0
#   reg_alpha             = 0
#   reg_lambda            = 3
#   eval_metric           = logloss
#   random_state          = 42
#   n_jobs                = -1
#
# NO FIT IS PERFORMED IN THIS CELL.
# NO TARGET IS READ.
# NO TARGET OPENING IS CONSUMED.
# ==============================================================================

from __future__ import annotations

import os
import json
import base64
import hashlib
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone


print("=" * 118)
print("STAGE24-4A-EX1 — SECONDARY XGBOOST CPU BACKEND AMENDMENT")
print("=" * 118)
print()


# ==============================================================================
# 0. FROZEN STATE
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "d624db8f14f96f9d2bde176df99653555cea842c"
)


FIT_LEDGER = Path(
    "/kaggle/working/stage24_secondary_fit_runtime_ledger.json"
)


OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_4_secondary_source_training"
    / "stage24_4a_ex1_backend_amendment"
)


AMENDMENT_PATH = (
    OUT
    / "stage24_4a_ex1_xgboost_cpu_backend_amendment.json"
)


AMENDMENT_SHA_PATH = (
    OUT
    / "stage24_4a_ex1_xgboost_cpu_backend_amendment.sha256"
)


# ==============================================================================
# 1. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    try:

        p = subprocess.run(
            [str(x) for x in args],
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

    except FileNotFoundError as exc:

        if check:
            raise

        return ""


    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )


    return (p.stdout or "").strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    cmd = [
        "git"
    ]


    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]


    cmd += [
        str(x)
        for x in args
    ]


    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
):

    h = hashlib.sha256()


    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                8 * 1024 * 1024
            )


            if not block:
                break


            h.update(
                block
            )


    return h.hexdigest()


# ==============================================================================
# 2. GOVERNANCE
# ==============================================================================

print("=" * 118)
print("GOVERNANCE")
print("=" * 118)


head = git_cmd(
    "rev-parse",
    "HEAD",
)


status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


print(
    "[PASS] Repository clean at Stage24-3A."
)

print()


# ==============================================================================
# 3. PROVE FIT #3 WAS NOT CONSUMED
# ==============================================================================

print("=" * 118)
print("SCIENTIFIC FIT LEDGER")
print("=" * 118)


print(
    "Fit ledger exists:",
    FIT_LEDGER.exists(),
)


if FIT_LEDGER.exists():

    raise RuntimeError(
        "\nA scientific-fit ledger exists.\n"
        "Do NOT classify the previous failure as pre-fit."
    )


print()
print(
    "[PASS] No Stage24 secondary-fit ledger exists."
)

print(
    "[PASS] Previous Stage24-4A execution failed BEFORE fit-attempt marker."
)

print(
    "[PASS] Scientific fits remain 2 / 4."
)

print()


# ==============================================================================
# 4. CONFIRM CURRENT EXECUTION ENVIRONMENT
# ==============================================================================

print("=" * 118)
print("CURRENT EXECUTION BACKEND")
print("=" * 118)


nvidia_smi_path = shutil.which(
    "nvidia-smi"
)


cuda_visible = os.environ.get(
    "CUDA_VISIBLE_DEVICES"
)


dev_nvidia0 = Path(
    "/dev/nvidia0"
).exists()


print(
    "nvidia-smi path:",
    nvidia_smi_path,
)

print(
    "CUDA_VISIBLE_DEVICES:",
    repr(
        cuda_visible
    ),
)

print(
    "/dev/nvidia0 exists:",
    dev_nvidia0,
)

print()


if nvidia_smi_path is not None:

    nvidia_output = run_cmd(
        [
            nvidia_smi_path,
            "--query-gpu=name,memory.total",
            "--format=csv,noheader",
        ],
        check=False,
    )

else:

    nvidia_output = ""


print(
    "nvidia-smi output:",
    (
        nvidia_output
        if nvidia_output
        else "<UNAVAILABLE>"
    ),
)

print()


# The immediate failure evidence from the Stage24-4A run is:
# FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'
#
# The session was intentionally configured CPU-only before this fit.

if nvidia_smi_path is not None and nvidia_output:

    raise RuntimeError(
        "\nCUDA GPU is currently visible.\n"
        "CPU fallback amendment is therefore not justified in this runtime."
    )


print(
    "[PASS] CUDA execution backend unavailable in current notebook runtime."
)

print()


# ==============================================================================
# 5. VERIFY NO PARTIAL STAGE24-4A SCIENTIFIC ARTIFACT
# ==============================================================================

FIT_OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_4_secondary_source_training"
    / "stage24_4a_bridge62_xgboost"
)


partial_files = []


if FIT_OUT.exists():

    partial_files = [
        str(
            path.relative_to(
                REPO
            )
        )
        for path in FIT_OUT.rglob(
            "*"
        )
        if path.is_file()
    ]


print(
    "Stage24-4A scientific output files:",
    len(
        partial_files
    ),
)


if partial_files:

    print()

    for path in partial_files:

        print(
            " ",
            path,
        )


    raise RuntimeError(
        "\nPartial Stage24-4A scientific artifacts exist.\n"
        "Do not freeze backend amendment until those are audited."
    )


print(
    "[PASS] No model/result/validation artifact exists from failed Stage24-4A run."
)

print()


# ==============================================================================
# 6. FREEZE BACKEND AMENDMENT
# ==============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


amendment = {
    "stage":
        "Stage24-4A-EX1",

    "type":
        "EXECUTION_BACKEND_AMENDMENT",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_PARENT,

    "direction":
        "CICIDS2017_TO_IDS2018",

    "affected_secondary_fits":
        [
            "bridge62_xgboost",
            "bridge70_xgboost",
        ],

    "trigger": {
        "stage24_4a_failure_location":
            "CUDA_PRE_FIT_GATE",

        "observed_exception":
            (
                "FileNotFoundError: [Errno 2] "
                "No such file or directory: 'nvidia-smi'"
            ),

        "nvidia_smi_available_now":
            False,

        "dev_nvidia0_exists":
            bool(
                dev_nvidia0
            ),

        "fit_attempt_marker_existed_after_failure":
            False,

        "scientific_fit_was_attempted":
            False,

        "scientific_fit_consumed":
            False,
    },

    "amendment": {
        "parameter_changed":
            "device",

        "previous":
            "cuda",

        "replacement":
            "cpu",

        "tree_method":
            "hist",

        "scope":
            "EXECUTION_BACKEND_ONLY",

        "scientific_model_family_changed":
            False,

        "feature_representation_changed":
            False,

        "training_population_changed":
            False,

        "validation_population_changed":
            False,

        "hyperparameters_changed_other_than_device":
            False,

        "threshold_protocol_changed":
            False,

        "target_protocol_changed":
            False,

        "target_guided":
            False,

        "performance_guided":
            False,
    },

    "frozen_xgb_11_after_amendment": {
        "objective":
            "binary:logistic",

        "n_estimators":
            400,

        "max_depth":
            7,

        "learning_rate":
            0.06,

        "subsample":
            0.9,

        "colsample_bytree":
            1.0,

        "min_child_weight":
            1,

        "gamma":
            0.0,

        "reg_alpha":
            0.0,

        "reg_lambda":
            3.0,

        "tree_method":
            "hist",

        "device":
            "cpu",

        "eval_metric":
            "logloss",

        "random_state":
            42,

        "n_jobs":
            -1,
    },

    "scientific_fit_accounting": {
        "budget":
            4,

        "completed_before_amendment":
            2,

        "completed_after_amendment":
            2,

        "remaining":
            2,

        "fit_3_consumed":
            False,
    },

    "target_opening_accounting": {
        "budget":
            8,

        "consumed":
            4,

        "new_openings":
            0,

        "secondary_IDS2018_Feb28_opened":
            False,
    },

    "authorization_after_commit": {
        "next_step":
            (
                "REEXECUTE_STAGE24_4A_SECONDARY_BRIDGE62_XGBOOST_"
                "USING_CPU_BACKEND_WITH_ALL_OTHER_FROZEN_XGB_11_"
                "PARAMETERS_UNCHANGED"
            ),

        "bridge70_secondary_fit_backend":
            "CPU_SAME_AMENDMENT",

        "additional_backend_amendment_required_for_bridge70":
            False,
    },
}


AMENDMENT_PATH.write_text(
    json.dumps(
        amendment,
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


amendment_sha = sha256_file(
    AMENDMENT_PATH
)


AMENDMENT_SHA_PATH.write_text(
    f"{amendment_sha}  {AMENDMENT_PATH.name}\n",
    encoding="utf-8",
)


print("=" * 118)
print("AMENDMENT ARTIFACT")
print("=" * 118)

print(
    "Artifact:",
    AMENDMENT_PATH.relative_to(
        REPO
    ),
)

print(
    "SHA256:",
    amendment_sha,
)

print()


# ==============================================================================
# 7. GITHUB CREDENTIAL
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()


    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )


            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )


        if value and value.strip():

            github_token = value.strip()
            token_source = (
                "ENV:"
                +
                name
            )

            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved before amendment freeze.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main == Stage24-3A."
)

print()


# ==============================================================================
# 8. GIT AUTHOR SAFETY
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    git_cmd(
        "config",
        "--local",
        "user.name",
        git_cmd(
            "log",
            "-1",
            "--format=%an",
        ),
    )


if not author_email:

    git_cmd(
        "config",
        "--local",
        "user.email",
        git_cmd(
            "log",
            "-1",
            "--format=%ae",
        ),
    )


# ==============================================================================
# 9. COMMIT
# ==============================================================================

print("=" * 118)
print("GIT FREEZE")
print("=" * 118)


git_cmd(
    "add",
    "--",
    str(
        AMENDMENT_PATH.relative_to(
            REPO
        )
    ),
    str(
        AMENDMENT_SHA_PATH.relative_to(
            REPO
        )
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


expected_staged = {
    str(
        AMENDMENT_PATH.relative_to(
            REPO
        )
    ),
    str(
        AMENDMENT_SHA_PATH.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: amend secondary xgboost execution backend to cpu",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Backend amendment commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 10. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 118)
print("PUSH + REMOTE VERIFY")
print("=" * 118)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository not clean after amendment:\n"
        + final_status
    )


print(
    "[PASS] Remote main == local amendment commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 11. FINAL
# ==============================================================================

print("=" * 118)
print("STAGE24-4A-EX1 CPU BACKEND AMENDMENT: PASS")
print("=" * 118)

print()

print(
    "Previous backend:               CUDA"
)

print(
    "Amended backend:                CPU"
)

print(
    "Tree method:                    hist"
)

print(
    "Other XGB_11 parameters changed:NO"
)

print()

print(
    "Scientific fit #3 attempted:    NO"
)

print(
    "Scientific fits completed:      2 / 4"
)

print(
    "Scientific fits remaining:      2"
)

print()

print(
    "New target openings:            0"
)

print(
    "Target openings consumed:       4 / 8"
)

print(
    "IDS2018 Feb-28 opened:          NO"
)

print()

print(
    "Amendment SHA:"
)

print(
    " ",
    amendment_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "NEXT:"
)

print(
    "  Stage24-4A-R1 — bridge62 XGBoost scientific fit #3 / 4"
)

print(
    "  using CPU with all other frozen XGB_11 parameters unchanged."
)

print("=" * 118)

STAGE24-4A-EX1 — SECONDARY XGBOOST CPU BACKEND AMENDMENT

GOVERNANCE
HEAD: d624db8f14f96f9d2bde176df99653555cea842c
[PASS] Repository clean at Stage24-3A.

SCIENTIFIC FIT LEDGER
Fit ledger exists: False

[PASS] No Stage24 secondary-fit ledger exists.
[PASS] Previous Stage24-4A execution failed BEFORE fit-attempt marker.
[PASS] Scientific fits remain 2 / 4.

CURRENT EXECUTION BACKEND
nvidia-smi path: None
CUDA_VISIBLE_DEVICES: None
/dev/nvidia0 exists: False

nvidia-smi output: <UNAVAILABLE>

[PASS] CUDA execution backend unavailable in current notebook runtime.

Stage24-4A scientific output files: 0
[PASS] No model/result/validation artifact exists from failed Stage24-4A run.

AMENDMENT ARTIFACT
Artifact: results/stage24_cross_dataset/stage24_4_secondary_source_training/stage24_4a_ex1_backend_amendment/stage24_4a_ex1_xgboost_cpu_backend_amendment.json
SHA256: 2c356c8763c567702d14a74c042df42356f360e559e574c3d9a4abfa652e5c14

GitHub credential: GITHUB_TOKEN
[PASS] Remote main == Stage24-

In [39]:
# ==============================================================================
# STAGE24-4A-R1 — SECONDARY BRIDGE62 XGBOOST CPU FIT
#
# Direction:
#   CICIDS2017 -> IDS2018
#
# Scientific fit:
#   #3 / 4
#
# Frozen source:
#   TRAIN      Monday + Tuesday + Wednesday
#   VALIDATION Thursday
#   UNUSED     Friday
#
# Frozen representation:
#   bridge62
#
# Frozen learner:
#   XGBOOST_ONLY
#   INHERIT_STAGE22R_XGB_11
#
# Frozen backend amendment:
#   device: cuda -> cpu
#   tree_method: hist
#   ALL OTHER PARAMETERS UNCHANGED
#
# IMPORTANT:
#   This cell will REUSE the already-materialized runtime arrays from the
#   failed Stage24-4A execution if and only if their exact SHA identities
#   match the previously observed frozen matrices.
#
#   If those arrays are absent, it reconstructs them from frozen Parquets.
#
# NO IDS2018 TARGET READ.
# NO TARGET INFERENCE.
# NO TARGET OPENING.
#
# Once:
#
#     >>> SCIENTIFIC FIT #3 CONSUMED <<<
#
# appears, DO NOT rerun blindly after any later failure.
# ==============================================================================

from __future__ import annotations

import os
import gc
import csv
import json
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import duckdb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


print("=" * 118)
print("STAGE24-4A-R1 — SECONDARY BRIDGE62 XGBOOST CPU FIT")
print("=" * 118)
print()


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "c8c97de2a6088b67f07727608d2a97d33b90dda9"
)

EXPECTED_AMENDMENT_SHA = (
    "2c356c8763c567702d14a74c042df42356f360e559e574c3d9a4abfa652e5c14"
)

EXPECTED_BRIDGE_SPEC_SHA = (
    "b7ff1d8563aafc69145be3acc827fbc59ae75d880b9804bad727f408299fce2f"
)

EXPECTED_PROTOCOL_LOCK_SHA = (
    "8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2"
)


EXPECTED_TRAIN_ROWS = 1_668_530
EXPECTED_TRAIN_BENIGN = 1_402_023
EXPECTED_TRAIN_ATTACK = 266_507

EXPECTED_VAL_ROWS = 458_968
EXPECTED_VAL_BENIGN = 456_752
EXPECTED_VAL_ATTACK = 2_216


# Exact matrices observed immediately before the CUDA pre-fit failure.
EXPECTED_TRAIN_FEATURE_SHA = (
    "0058030b9684e917a403ff09ad6524016dc9c38c38352ff0070643a4e8bed9b0"
)

EXPECTED_TRAIN_LABEL_SHA = (
    "7b30dd8aa0a60e6c8d8715a44fae5c2019469c56624466841854daca8c0d6c8f"
)

EXPECTED_VAL_FEATURE_SHA = (
    "7da9498e27faf2acbbacc0999b007b0ec50d0e911fa13fa57cbbec9be378ba4c"
)

EXPECTED_VAL_LABEL_SHA = (
    "b1be032e4bde759b59a935476d7b440f6f0934f348ed0302af13d310137d0e95"
)


EXPECTED_OPENINGS = 4
EXPECTED_FITS_BEFORE = 2
EXPECTED_FITS_AFTER = 3


# ==============================================================================
# 1. PATHS
# ==============================================================================

LOCK_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
)


BRIDGE_SPEC_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.json"
)

BRIDGE_SPEC_SHA_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.sha256"
)


FINAL_LOCK_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.json"
)

FINAL_LOCK_SHA_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.sha256"
)


AMENDMENT_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_4_secondary_source_training"
    / "stage24_4a_ex1_backend_amendment"
)

AMENDMENT_PATH = (
    AMENDMENT_DIR
    / "stage24_4a_ex1_xgboost_cpu_backend_amendment.json"
)

AMENDMENT_SHA_PATH = (
    AMENDMENT_DIR
    / "stage24_4a_ex1_xgboost_cpu_backend_amendment.sha256"
)


PRIMARY_FREEZE_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_3_primary_results_freeze"
    / "stage24_3a_primary_paired_bootstrap"
)

PRIMARY_FREEZE_PATH = (
    PRIMARY_FREEZE_DIR
    / "stage24_3a_primary_results_freeze.json"
)


TARGET_ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/"
    "snapshots/b7e532345512edcd530cb1770dc76636aeb52802/"
    "traffic_labels"
)


OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_4_secondary_source_training"
    / "stage24_4a_bridge62_xgboost"
)


MODEL_PATH = (
    OUT
    / "secondary_bridge62_xgboost_model.json"
)


VAL_PRED_PATH = (
    OUT
    / "secondary_bridge62_validation_probabilities.npz"
)


THRESHOLD_GRID_PATH = (
    OUT
    / "secondary_bridge62_validation_threshold_grid.csv"
)


RESULT_PATH = (
    OUT
    / "stage24_4a_secondary_bridge62_result.json"
)


RESULT_SHA_PATH = (
    OUT
    / "stage24_4a_secondary_bridge62_result.sha256"
)


CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


FIT_LEDGER = Path(
    "/kaggle/working/stage24_secondary_fit_runtime_ledger.json"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    cmd = [
        "git"
    ]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [
        str(x)
        for x in args
    ]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_array(
    array,
):

    arr = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        arr.view(
            np.uint8
        )
    ).hexdigest()


def verify_sidecar(
    artifact,
    sidecar,
):

    if not artifact.is_file():

        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )

    if not sidecar.is_file():

        raise RuntimeError(
            f"Missing SHA sidecar:\n{sidecar}"
        )


    actual = sha256_file(
        artifact
    )


    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )


    if actual != expected:

        raise RuntimeError(
            "\nSHA mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


    return actual


def qident(
    value,
):

    return (
        '"'
        +
        str(
            value
        ).replace(
            '"',
            '""',
        )
        +
        '"'
    )


def sql_path(
    path,
):

    return str(
        path
    ).replace(
        "'",
        "''",
    )


# ==============================================================================
# 3. GOVERNANCE
# ==============================================================================

print("=" * 118)
print("GOVERNANCE")
print("=" * 118)


head = git_cmd(
    "rev-parse",
    "HEAD",
)


status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


if FIT_LEDGER.exists():

    raise RuntimeError(
        "\nA secondary fit ledger already exists:\n"
        f"{FIT_LEDGER}\n\n"
        "DO NOT rerun scientific fit #3 blindly."
    )


if OUT.exists() and any(
    OUT.iterdir()
):

    raise RuntimeError(
        "\nStage24-4A scientific artifacts already exist.\n"
        "Refusing duplicate scientific fit."
    )


print(
    "[PASS] Repository clean."
)

print(
    "[PASS] No unresolved scientific-fit ledger."
)

print(
    "[PASS] No prior Stage24-4A scientific artifacts."
)

print()


# ==============================================================================
# 4. VERIFY CPU AMENDMENT
# ==============================================================================

print("=" * 118)
print("CPU BACKEND AMENDMENT")
print("=" * 118)


amendment_sha = verify_sidecar(
    AMENDMENT_PATH,
    AMENDMENT_SHA_PATH,
)


if amendment_sha != EXPECTED_AMENDMENT_SHA:

    raise RuntimeError(
        "CPU backend amendment SHA changed."
    )


amendment = json.loads(
    AMENDMENT_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    amendment[
        "amendment"
    ][
        "parameter_changed"
    ]
    !=
    "device"
):

    raise RuntimeError(
        "Backend amendment parameter changed."
    )


if (
    amendment[
        "amendment"
    ][
        "previous"
    ]
    !=
    "cuda"
):

    raise RuntimeError(
        "Backend amendment previous value changed."
    )


if (
    amendment[
        "amendment"
    ][
        "replacement"
    ]
    !=
    "cpu"
):

    raise RuntimeError(
        "Backend amendment replacement changed."
    )


if (
    amendment[
        "amendment"
    ][
        "scope"
    ]
    !=
    "EXECUTION_BACKEND_ONLY"
):

    raise RuntimeError(
        "Backend amendment scope changed."
    )


if (
    amendment[
        "scientific_fit_accounting"
    ][
        "fit_3_consumed"
    ]
    is not False
):

    raise RuntimeError(
        "Amendment says scientific fit #3 was already consumed."
    )


print(
    "Amendment SHA:",
    amendment_sha,
)

print(
    "Execution device:",
    "CPU",
)

print()

print(
    "[PASS] Scientific fit #3 remains unconsumed."
)

print(
    "[PASS] CPU-only execution amendment is frozen."
)

print()


# ==============================================================================
# 5. VERIFY FROZEN PROTOCOL
# ==============================================================================

bridge_sha = verify_sidecar(
    BRIDGE_SPEC_PATH,
    BRIDGE_SPEC_SHA_PATH,
)


lock_sha = verify_sidecar(
    FINAL_LOCK_PATH,
    FINAL_LOCK_SHA_PATH,
)


if bridge_sha != EXPECTED_BRIDGE_SPEC_SHA:

    raise RuntimeError(
        "Semantic bridge SHA changed."
    )


if lock_sha != EXPECTED_PROTOCOL_LOCK_SHA:

    raise RuntimeError(
        "Protocol lock SHA changed."
    )


bridge_spec = json.loads(
    BRIDGE_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)


final_lock = json.loads(
    FINAL_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


primary_freeze = json.loads(
    PRIMARY_FREEZE_PATH.read_text(
        encoding="utf-8"
    )
)


secondary = final_lock[
    "secondary_direction"
]


if secondary[
    "learner"
] != "XGBOOST_ONLY":

    raise RuntimeError(
        "Secondary learner changed."
    )


if secondary[
    "learner_configuration"
] != "INHERIT_STAGE22R_XGB_11":

    raise RuntimeError(
        "Secondary XGBoost configuration changed."
    )


if secondary[
    "source_train_days"
] != [
    "Monday",
    "Tuesday",
    "Wednesday",
]:

    raise RuntimeError(
        "Secondary training days changed."
    )


if secondary[
    "source_validation_days"
] != [
    "Thursday"
]:

    raise RuntimeError(
        "Secondary validation day changed."
    )


if secondary[
    "source_unused_days"
] != [
    "Friday"
]:

    raise RuntimeError(
        "Friday unused-source contract changed."
    )


if int(
    primary_freeze[
        "opening_accounting"
    ][
        "total_openings_consumed"
    ]
) != EXPECTED_OPENINGS:

    raise RuntimeError(
        "Target opening accounting changed."
    )


print(
    "[PASS] Frozen secondary source/validation contract exact."
)

print(
    "[PASS] Target openings remain 4 / 8."
)

print()


# ==============================================================================
# 6. BRIDGE62
# ==============================================================================

b62 = bridge_spec[
    "bridge62"
]


FEATURE_ORDER = list(
    b62[
        "source_feature_order"
    ]
)


SOURCE_MAPPING = dict(
    b62[
        "FLAG_CORRECTED_mapping"
    ]
)


if len(
    FEATURE_ORDER
) != 62:

    raise RuntimeError(
        "bridge62 feature count changed."
    )


if (
    b62[
        "PUBLISHED_mapping"
    ]
    !=
    b62[
        "FLAG_CORRECTED_mapping"
    ]
):

    raise RuntimeError(
        "bridge62 source representation changed."
    )


if set(
    FEATURE_ORDER
) != set(
    SOURCE_MAPPING
):

    raise RuntimeError(
        "bridge62 source mapping incomplete."
    )


print("=" * 118)
print("BRIDGE62 REPRESENTATION")
print("=" * 118)

print(
    "Features:",
    len(
        FEATURE_ORDER
    ),
)

print(
    "[PASS] PUBLISHED == FLAG_CORRECTED for bridge62."
)

print()


# ==============================================================================
# 7. FROZEN SOURCE FILES
# ==============================================================================

TRAIN_FILES = [
    (
        "Monday",
        "Monday-WorkingHours.pcap_ISCX.csv.parquet",
        529_918,
    ),
    (
        "Tuesday",
        "Tuesday-WorkingHours.pcap_ISCX.csv.parquet",
        445_909,
    ),
    (
        "Wednesday",
        "Wednesday-workingHours.pcap_ISCX.csv.parquet",
        692_703,
    ),
]


VALIDATION_FILES = [
    (
        "Thursday-Afternoon",
        "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet",
        288_602,
    ),
    (
        "Thursday-Morning",
        "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet",
        170_366,
    ),
]


# ==============================================================================
# 8. TRY TO REUSE EXACT MATRICES FROM PREVIOUS FAILED CELL
# ==============================================================================

print("=" * 118)
print("RUNTIME MATRIX RECOVERY")
print("=" * 118)


required_runtime_names = [
    "X_train",
    "y_train",
    "X_val",
    "y_val",
]


runtime_presence = {
    name:
        (
            name
            in
            globals()
        )
    for name in required_runtime_names
}


for name in required_runtime_names:

    print(
        f"{name:10s}:",
        runtime_presence[
            name
        ],
    )


all_present = all(
    runtime_presence.values()
)

none_present = not any(
    runtime_presence.values()
)


if not all_present and not none_present:

    raise RuntimeError(
        "\nPartial stale source matrices exist in runtime.\n"
        "Refusing ambiguous recovery."
    )


source_materialization_mode = None


if all_present:

    X_train_local = globals()[
        "X_train"
    ]

    y_train_local = globals()[
        "y_train"
    ]

    X_val_local = globals()[
        "X_val"
    ]

    y_val_local = globals()[
        "y_val"
    ]


    expected_shapes = {
        "X_train":
            (
                EXPECTED_TRAIN_ROWS,
                62,
            ),

        "y_train":
            (
                EXPECTED_TRAIN_ROWS,
            ),

        "X_val":
            (
                EXPECTED_VAL_ROWS,
                62,
            ),

        "y_val":
            (
                EXPECTED_VAL_ROWS,
            ),
    }


    observed_shapes = {
        "X_train":
            X_train_local.shape,

        "y_train":
            y_train_local.shape,

        "X_val":
            X_val_local.shape,

        "y_val":
            y_val_local.shape,
    }


    if observed_shapes != expected_shapes:

        raise RuntimeError(
            "\nRuntime source matrix shape mismatch.\n"
            f"Expected: {expected_shapes}\n"
            f"Actual:   {observed_shapes}"
        )


    train_feature_sha = sha256_array(
        X_train_local
    )

    train_label_sha = sha256_array(
        y_train_local
    )

    val_feature_sha = sha256_array(
        X_val_local
    )

    val_label_sha = sha256_array(
        y_val_local
    )


    print()
    print(
        "Train feature SHA:",
        train_feature_sha,
    )

    print(
        "Train label SHA:  ",
        train_label_sha,
    )

    print(
        "Val feature SHA:  ",
        val_feature_sha,
    )

    print(
        "Val label SHA:    ",
        val_label_sha,
    )


    if train_feature_sha != EXPECTED_TRAIN_FEATURE_SHA:

        raise RuntimeError(
            "Runtime X_train SHA mismatch."
        )


    if train_label_sha != EXPECTED_TRAIN_LABEL_SHA:

        raise RuntimeError(
            "Runtime y_train SHA mismatch."
        )


    if val_feature_sha != EXPECTED_VAL_FEATURE_SHA:

        raise RuntimeError(
            "Runtime X_val SHA mismatch."
        )


    if val_label_sha != EXPECTED_VAL_LABEL_SHA:

        raise RuntimeError(
            "Runtime y_val SHA mismatch."
        )


    X_train = X_train_local
    y_train = y_train_local
    X_val = X_val_local
    y_val = y_val_local


    source_materialization_mode = (
        "REUSED_EXACT_PREVIOUS_RUNTIME_MATRICES"
    )


    print()
    print(
        "[PASS] Reusing exact source matrices already in RAM."
    )


else:

    print()
    print(
        "[INFO] Previous matrices are absent; reconstructing from frozen Parquets."
    )


    X_train = np.empty(
        (
            EXPECTED_TRAIN_ROWS,
            62,
        ),
        dtype=np.float64,
    )


    y_train = np.empty(
        EXPECTED_TRAIN_ROWS,
        dtype=np.uint8,
    )


    X_val = np.empty(
        (
            EXPECTED_VAL_ROWS,
            62,
        ),
        dtype=np.float64,
    )


    y_val = np.empty(
        EXPECTED_VAL_ROWS,
        dtype=np.uint8,
    )


    DASH_TRANSLATION = str.maketrans(
        {
            "\u2010": "-",
            "\u2011": "-",
            "\u2012": "-",
            "\u2013": "-",
            "\u2014": "-",
            "\u2015": "-",
            "\u2212": "-",
        }
    )


    def canonicalize_labels(
        series,
    ):

        if series.isna().any():

            raise RuntimeError(
                "NULL label survived frozen effective-row rule."
            )


        s = (
            series
            .astype(
                "string"
            )
            .str.strip()
            .str.translate(
                DASH_TRANSLATION
            )
            .str.replace(
                r"\s+",
                " ",
                regex=True,
            )
            .str.casefold()
        )


        if s.isna().any():

            raise RuntimeError(
                "Label canonicalization produced NULL."
            )


        if (
            s.str.len()
            ==
            0
        ).any():

            raise RuntimeError(
                "Empty source label."
            )


        return s


    projection_sql = ",\n".join(
        (
            f"CAST({qident(SOURCE_MAPPING[source_feature])} AS DOUBLE) "
            f"AS {qident(source_feature)}"
        )
        for source_feature in FEATURE_ORDER
    )


    conn = duckdb.connect(
        database=":memory:"
    )


    def materialize(
        specs,
        X_out,
        y_out,
        expected_total,
        split_name,
    ):

        cursor = 0


        for day, filename, expected_rows in specs:

            path = (
                TARGET_ROOT
                /
                filename
            )


            if not path.is_file():

                raise RuntimeError(
                    f"Missing source Parquet:\n{path}"
                )


            query = f"""
                SELECT
                    {projection_sql},
                    CAST("Label" AS VARCHAR) AS "__label__"
                FROM read_parquet('{sql_path(path)}')
                WHERE "Label" IS NOT NULL
            """


            t0 = time.time()


            df = conn.execute(
                query
            ).df()


            if len(
                df
            ) != expected_rows:

                raise RuntimeError(
                    f"{split_name}/{day}: row count mismatch."
                )


            X = df[
                FEATURE_ORDER
            ].to_numpy(
                dtype=np.float64,
                copy=True,
            )


            inf_mask = np.isinf(
                X
            )


            if inf_mask.any():

                X[
                    inf_mask
                ] = np.nan


            canonical = canonicalize_labels(
                df[
                    "__label__"
                ]
            )


            y = (
                canonical
                !=
                "benign"
            ).to_numpy(
                dtype=np.uint8,
            )


            start = cursor

            stop = (
                start
                +
                expected_rows
            )


            X_out[
                start:stop
            ] = X


            y_out[
                start:stop
            ] = y


            print(
                f"{split_name:10s}"
                f" {day:20s}"
                f" rows={expected_rows:9,d}"
                f" seconds={time.time() - t0:6.2f}"
            )


            cursor = stop


            del df
            del X
            del y
            del canonical
            del inf_mask

            gc.collect()


        if cursor != expected_total:

            raise RuntimeError(
                f"{split_name}: final row-count mismatch."
            )


    materialize(
        TRAIN_FILES,
        X_train,
        y_train,
        EXPECTED_TRAIN_ROWS,
        "TRAIN",
    )


    materialize(
        VALIDATION_FILES,
        X_val,
        y_val,
        EXPECTED_VAL_ROWS,
        "VALIDATION",
    )


    conn.close()


    train_feature_sha = sha256_array(
        X_train
    )

    train_label_sha = sha256_array(
        y_train
    )

    val_feature_sha = sha256_array(
        X_val
    )

    val_label_sha = sha256_array(
        y_val
    )


    if train_feature_sha != EXPECTED_TRAIN_FEATURE_SHA:

        raise RuntimeError(
            "\nReconstructed X_train differs from frozen pre-fit matrix."
        )


    if train_label_sha != EXPECTED_TRAIN_LABEL_SHA:

        raise RuntimeError(
            "\nReconstructed y_train differs from frozen pre-fit labels."
        )


    if val_feature_sha != EXPECTED_VAL_FEATURE_SHA:

        raise RuntimeError(
            "\nReconstructed X_val differs from frozen pre-fit matrix."
        )


    if val_label_sha != EXPECTED_VAL_LABEL_SHA:

        raise RuntimeError(
            "\nReconstructed y_val differs from frozen pre-fit labels."
        )


    source_materialization_mode = (
        "RECONSTRUCTED_FROM_FROZEN_CICIDS2017_PARQUETS"
    )


    print()
    print(
        "[PASS] Reconstructed matrices reproduce all four frozen SHAs."
    )


print()


# ==============================================================================
# 9. SOURCE POPULATION AUDIT
# ==============================================================================

train_attack = int(
    y_train.sum()
)


train_benign = int(
    EXPECTED_TRAIN_ROWS
    -
    train_attack
)


val_attack = int(
    y_val.sum()
)


val_benign = int(
    EXPECTED_VAL_ROWS
    -
    val_attack
)


if train_benign != EXPECTED_TRAIN_BENIGN:

    raise RuntimeError(
        "Train benign count mismatch."
    )


if train_attack != EXPECTED_TRAIN_ATTACK:

    raise RuntimeError(
        "Train attack count mismatch."
    )


if val_benign != EXPECTED_VAL_BENIGN:

    raise RuntimeError(
        "Validation benign count mismatch."
    )


if val_attack != EXPECTED_VAL_ATTACK:

    raise RuntimeError(
        "Validation attack count mismatch."
    )


source_prior = float(
    train_attack
    /
    EXPECTED_TRAIN_ROWS
)


val_prevalence = float(
    val_attack
    /
    EXPECTED_VAL_ROWS
)


print("=" * 118)
print("SOURCE POPULATION AUDIT")
print("=" * 118)


print(
    "Materialization mode:",
    source_materialization_mode,
)

print()

print(
    "TRAIN rows:  ",
    f"{EXPECTED_TRAIN_ROWS:,}",
)

print(
    "TRAIN benign:",
    f"{train_benign:,}",
)

print(
    "TRAIN attack:",
    f"{train_attack:,}",
)

print(
    "TRAIN prior: ",
    f"{source_prior:.15f}",
)

print()

print(
    "VAL rows:    ",
    f"{EXPECTED_VAL_ROWS:,}",
)

print(
    "VAL benign:  ",
    f"{val_benign:,}",
)

print(
    "VAL attack:  ",
    f"{val_attack:,}",
)

print(
    "VAL prior:   ",
    f"{val_prevalence:.15f}",
)

print()

print(
    "[PASS] Frozen secondary bridge62 source population exact."
)

print()


# ==============================================================================
# 10. GITHUB CREDENTIAL / REMOTE GATE
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()


    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )


            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )


        if value and value.strip():

            github_token = value.strip()
            token_source = (
                "ENV:"
                +
                name
            )

            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable BEFORE scientific fit #3."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved before scientific fit #3.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main == CPU amendment commit."
)

print()


# ==============================================================================
# 11. FROZEN XGB_11 CPU CONFIGURATION
# ==============================================================================

XGB_PARAMS = {
    "objective":
        "binary:logistic",

    "n_estimators":
        400,

    "max_depth":
        7,

    "learning_rate":
        0.06,

    "subsample":
        0.9,

    "colsample_bytree":
        1.0,

    "min_child_weight":
        1,

    "gamma":
        0.0,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "tree_method":
        "hist",

    "device":
        "cpu",

    "eval_metric":
        "logloss",

    "random_state":
        42,

    "n_jobs":
        -1,
}


print("=" * 118)
print("FROZEN XGB_11 — CPU AMENDMENT")
print("=" * 118)


for key in sorted(
    XGB_PARAMS
):

    print(
        f"{key:20s} = {XGB_PARAMS[key]}"
    )


print()


# ==============================================================================
# 12. SCIENTIFIC FIT #3 / 4
# ==============================================================================

import xgboost as xgb


print(
    "XGBoost version:",
    xgb.__version__,
)

print()


fit_started_at = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


FIT_LEDGER.write_text(
    json.dumps(
        {
            "stage":
                "Stage24-4A-R1",

            "direction":
                "CICIDS2017_TO_IDS2018",

            "bridge":
                "bridge62",

            "learner":
                "XGBOOST_ONLY",

            "backend":
                "CPU_FROZEN_AMENDMENT",

            "scientific_fit_number":
                3,

            "scientific_fit_budget":
                4,

            "fit_attempt_started":
                True,

            "fit_completed":
                False,

            "fit_started_at_utc":
                fit_started_at,

            "parent_commit":
                EXPECTED_PARENT,

            "target_openings_consumed":
                4,

            "IDS2018_target_features_read":
                False,

            "IDS2018_target_predictions":
                0,
        },
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


print("=" * 118)
print("SCIENTIFIC FIT #3 / 4")
print("=" * 118)

print()
print(
    ">>> SCIENTIFIC FIT #3 CONSUMED <<<"
)
print()


model = xgb.XGBClassifier(
    **XGB_PARAMS
)


fit_start = time.time()


model.fit(
    X_train,
    y_train,
    verbose=False,
)


fit_seconds = (
    time.time()
    -
    fit_start
)


print(
    "Fit seconds:",
    f"{fit_seconds:.3f}",
)

print()


booster = model.get_booster()


if booster.num_features() != 62:

    raise RuntimeError(
        "\nFitted model dimension mismatch."
    )


if booster.num_boosted_rounds() != 400:

    raise RuntimeError(
        "\nFitted boosted-round count mismatch."
    )


config_text = (
    booster
    .save_config()
    .replace(
        " ",
        ""
    )
)


if '"device":"cpu"' not in config_text:

    raise RuntimeError(
        "\nFitted XGBoost configuration does not resolve to CPU."
    )


# ==============================================================================
# 13. SAVE MODEL IMMEDIATELY AFTER SUCCESSFUL FIT
#
# If anything fails after this point, recovery can use this model artifact
# instead of fitting again.
# ==============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


booster.save_model(
    MODEL_PATH
)


model_sha = sha256_file(
    MODEL_PATH
)


FIT_LEDGER.write_text(
    json.dumps(
        {
            "stage":
                "Stage24-4A-R1",

            "direction":
                "CICIDS2017_TO_IDS2018",

            "bridge":
                "bridge62",

            "learner":
                "XGBOOST_ONLY",

            "backend":
                "CPU_FROZEN_AMENDMENT",

            "scientific_fit_number":
                3,

            "scientific_fit_budget":
                4,

            "fit_attempt_started":
                True,

            "fit_completed":
                True,

            "fit_started_at_utc":
                fit_started_at,

            "fit_seconds":
                fit_seconds,

            "model_path":
                str(
                    MODEL_PATH
                ),

            "model_sha256":
                model_sha,

            "parent_commit":
                EXPECTED_PARENT,

            "target_openings_consumed":
                4,

            "IDS2018_target_features_read":
                False,

            "IDS2018_target_predictions":
                0,
        },
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


print(
    "[PASS] Scientific fit #3 completed on CPU."
)

print(
    "[PASS] 62 features / 400 boosted rounds."
)

print(
    "Model SHA:",
    model_sha,
)

print()


# Train matrix no longer needed.
del X_train

gc.collect()


# ==============================================================================
# 14. THURSDAY SOURCE-VALIDATION INFERENCE
# ==============================================================================

print("=" * 118)
print("SOURCE VALIDATION — THURSDAY")
print("=" * 118)


val_start = time.time()


p_val = np.asarray(
    model.predict_proba(
        X_val
    )[
        :,
        1
    ],
    dtype=np.float32,
)


val_seconds = (
    time.time()
    -
    val_start
)


if p_val.shape != (
    EXPECTED_VAL_ROWS,
):

    raise RuntimeError(
        "Validation probability shape mismatch."
    )


if not np.isfinite(
    p_val
).all():

    raise RuntimeError(
        "Non-finite validation probability."
    )


if (
    (p_val < 0).any()
    or
    (p_val > 1).any()
):

    raise RuntimeError(
        "Validation probabilities outside [0,1]."
    )


val_probability_sha = sha256_array(
    p_val
)


print(
    "Inference seconds:",
    f"{val_seconds:.3f}",
)

print(
    "Probability min/max:",
    f"{float(p_val.min()):.9f}",
    "/",
    f"{float(p_val.max()):.9f}",
)

print(
    "Probability SHA:",
    val_probability_sha,
)

print()


# ==============================================================================
# 15. SOURCE-VALIDATION GLOBAL METRICS
# ==============================================================================

pr_auc = float(
    average_precision_score(
        y_val,
        p_val,
    )
)


roc_auc = float(
    roc_auc_score(
        y_val,
        p_val,
    )
)


pr_excess = float(
    pr_auc
    -
    val_prevalence
)


pr_normalized = float(
    (
        pr_auc
        -
        val_prevalence
    )
    /
    (
        1.0
        -
        val_prevalence
    )
)


p64 = p_val.astype(
    np.float64,
    copy=False,
)


y64 = y_val.astype(
    np.float64,
    copy=False,
)


brier = float(
    np.mean(
        (
            p64
            -
            y64
        ) ** 2
    )
)


clipped = np.clip(
    p64,
    1e-15,
    1.0 - 1e-15,
)


log_loss = float(
    -np.mean(
        y64
        *
        np.log(
            clipped
        )
        +
        (
            1.0
            -
            y64
        )
        *
        np.log(
            1.0
            -
            clipped
        )
    )
)


bins = np.floor(
    p64
    *
    10.0
).astype(
    np.int8
)


bins = np.clip(
    bins,
    0,
    9,
)


ece = 0.0


for b in range(
    10
):

    mask = (
        bins
        ==
        b
    )


    count = int(
        mask.sum()
    )


    if count == 0:

        continue


    confidence = float(
        p64[
            mask
        ].mean()
    )


    empirical = float(
        y64[
            mask
        ].mean()
    )


    ece += (
        count
        /
        EXPECTED_VAL_ROWS
    ) * abs(
        confidence
        -
        empirical
    )


ece = float(
    ece
)


print("=" * 118)
print("SOURCE-VALIDATION METRICS")
print("=" * 118)


print(
    "Prevalence:    ",
    f"{val_prevalence:.15f}",
)

print(
    "PR-AUC:        ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:       ",
    f"{roc_auc:.15f}",
)

print(
    "PR excess:     ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized: ",
    f"{pr_normalized:.15f}",
)

print()

print(
    "Brier:         ",
    f"{brier:.15f}",
)

print(
    "Log loss:      ",
    f"{log_loss:.15f}",
)

print(
    "ECE-10:        ",
    f"{ece:.15f}",
)

print()


# ==============================================================================
# 16. FROZEN THRESHOLD GRID 5..95
# ==============================================================================

def threshold_counts(
    pct,
):

    threshold = np.float32(
        pct
        /
        100.0
    )


    pred = (
        p_val
        >=
        threshold
    )


    y1 = (
        y_val
        ==
        1
    )


    y0 = ~y1


    tp = int(
        np.sum(
            pred
            &
            y1
        )
    )


    fp = int(
        np.sum(
            pred
            &
            y0
        )
    )


    fn = (
        EXPECTED_VAL_ATTACK
        -
        tp
    )


    tn = (
        EXPECTED_VAL_BENIGN
        -
        fp
    )


    return {
        "threshold_integer_percent":
            int(
                pct
            ),

        "threshold_float32_runtime":
            float(
                threshold
            ),

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn,
    }


grid = [
    threshold_counts(
        pct
    )
    for pct in range(
        5,
        96,
    )
]


# ==============================================================================
# 17. EXACT RATIONAL SELECTION
# ==============================================================================

def f1_fraction(
    row,
):

    return (
        2
        *
        row[
            "tp"
        ],
        2
        *
        row[
            "tp"
        ]
        +
        row[
            "fp"
        ]
        +
        row[
            "fn"
        ],
    )


def f2_fraction(
    row,
):

    return (
        5
        *
        row[
            "tp"
        ],
        5
        *
        row[
            "tp"
        ]
        +
        4
        *
        row[
            "fn"
        ]
        +
        row[
            "fp"
        ],
    )


def rational_cmp(
    n1,
    d1,
    n2,
    d2,
):

    left = (
        n1
        *
        d2
    )


    right = (
        n2
        *
        d1
    )


    if left > right:
        return 1


    if left < right:
        return -1


    return 0


# ------------------------------------------------------------------------------
# BALANCED
# ------------------------------------------------------------------------------

balanced = None


for row in grid:

    if balanced is None:

        balanced = row
        continue


    n1, d1 = f1_fraction(
        row
    )


    n2, d2 = f1_fraction(
        balanced
    )


    cmp = rational_cmp(
        n1,
        d1,
        n2,
        d2,
    )


    if cmp > 0:

        balanced = row
        continue


    if cmp < 0:

        continue


    # Lower FPR.
    if row[
        "fp"
    ] < balanced[
        "fp"
    ]:

        balanced = row
        continue


    if row[
        "fp"
    ] > balanced[
        "fp"
    ]:

        continue


    # Higher recall.
    if row[
        "tp"
    ] > balanced[
        "tp"
    ]:

        balanced = row
        continue


    if row[
        "tp"
    ] < balanced[
        "tp"
    ]:

        continue


    # Closer to 0.50.
    d_new = abs(
        row[
            "threshold_integer_percent"
        ]
        -
        50
    )


    d_old = abs(
        balanced[
            "threshold_integer_percent"
        ]
        -
        50
    )


    if d_new < d_old:

        balanced = row
        continue


    if d_new > d_old:

        continue


    # Lower threshold.
    if (
        row[
            "threshold_integer_percent"
        ]
        <
        balanced[
            "threshold_integer_percent"
        ]
    ):

        balanced = row


# ------------------------------------------------------------------------------
# SECURITY
# ------------------------------------------------------------------------------

security_candidates = [
    row
    for row in grid
    if (
        20
        *
        row[
            "fp"
        ]
    )
    <=
    EXPECTED_VAL_BENIGN
]


security = None


for row in security_candidates:

    if security is None:

        security = row
        continue


    n1, d1 = f2_fraction(
        row
    )


    n2, d2 = f2_fraction(
        security
    )


    cmp = rational_cmp(
        n1,
        d1,
        n2,
        d2,
    )


    if cmp > 0:

        security = row
        continue


    if cmp < 0:

        continue


    if row[
        "fp"
    ] < security[
        "fp"
    ]:

        security = row
        continue


    if row[
        "fp"
    ] > security[
        "fp"
    ]:

        continue


    if row[
        "tp"
    ] > security[
        "tp"
    ]:

        security = row
        continue


    if row[
        "tp"
    ] < security[
        "tp"
    ]:

        continue


    if (
        row[
            "threshold_integer_percent"
        ]
        <
        security[
            "threshold_integer_percent"
        ]
    ):

        security = row


# ==============================================================================
# 18. EXPANDED THRESHOLD METRICS
# ==============================================================================

def expand(
    row,
):

    tp = row[
        "tp"
    ]

    tn = row[
        "tn"
    ]

    fp = row[
        "fp"
    ]

    fn = row[
        "fn"
    ]


    precision = (
        tp
        /
        (
            tp + fp
        )
        if (
            tp + fp
        )
        else 0.0
    )


    recall = (
        tp
        /
        (
            tp + fn
        )
        if (
            tp + fn
        )
        else 0.0
    )


    fpr = (
        fp
        /
        (
            fp + tn
        )
        if (
            fp + tn
        )
        else 0.0
    )


    fnr = (
        fn
        /
        (
            fn + tp
        )
        if (
            fn + tp
        )
        else 0.0
    )


    f1n, f1d = f1_fraction(
        row
    )


    f2n, f2d = f2_fraction(
        row
    )


    return {
        **row,

        "accuracy":
            float(
                (
                    tp + tn
                )
                /
                EXPECTED_VAL_ROWS
            ),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                f1n / f1d
                if f1d
                else 0.0
            ),

        "f2":
            float(
                f2n / f2d
                if f2d
                else 0.0
            ),

        "fpr":
            float(
                fpr
            ),

        "fnr":
            float(
                fnr
            ),
    }


standard = expand(
    grid[
        45
    ]
)


balanced_metrics = expand(
    balanced
)


security_metrics = (
    expand(
        security
    )
    if security is not None
    else None
)


print("=" * 118)
print("SOURCE-VALIDATION OPERATING POINTS")
print("=" * 118)


for name, metrics in [
    (
        "STANDARD",
        standard,
    ),
    (
        "BALANCED",
        balanced_metrics,
    ),
]:

    print(
        name,
        "@",
        metrics[
            "threshold_float32_runtime"
        ],
    )

    print(
        "  TP/TN/FP/FN:",
        metrics[
            "tp"
        ],
        metrics[
            "tn"
        ],
        metrics[
            "fp"
        ],
        metrics[
            "fn"
        ],
    )

    print(
        "  F1/F2/FPR/Recall:",
        f"{metrics['f1']:.9f}",
        f"{metrics['f2']:.9f}",
        f"{metrics['fpr']:.9f}",
        f"{metrics['recall']:.9f}",
    )

    print()


print(
    "SECURITY:"
)


if security_metrics is None:

    print(
        "  UNAVAILABLE_NO_RELAXATION"
    )

else:

    print(
        "  threshold:",
        security_metrics[
            "threshold_float32_runtime"
        ],
    )

    print(
        "  TP/TN/FP/FN:",
        security_metrics[
            "tp"
        ],
        security_metrics[
            "tn"
        ],
        security_metrics[
            "fp"
        ],
        security_metrics[
            "fn"
        ],
    )

    print(
        "  F2/FPR/Recall:",
        f"{security_metrics['f2']:.9f}",
        f"{security_metrics['fpr']:.9f}",
        f"{security_metrics['recall']:.9f}",
    )


print()

print(
    "[PASS] Threshold selection used Thursday SOURCE validation only."
)

print()


# ==============================================================================
# 19. PERSIST VALIDATION ARTIFACTS
# ==============================================================================

np.savez_compressed(
    VAL_PRED_PATH,

    probability=
        p_val,

    binary_label=
        y_val,
)


val_pred_file_sha = sha256_file(
    VAL_PRED_PATH
)


with THRESHOLD_GRID_PATH.open(
    "w",
    newline="",
    encoding="utf-8",
) as fh:

    writer = csv.DictWriter(
        fh,
        fieldnames=[
            "threshold_integer_percent",
            "threshold_float32_runtime",
            "tp",
            "tn",
            "fp",
            "fn",
        ],
    )


    writer.writeheader()


    writer.writerows(
        grid
    )


threshold_grid_sha = sha256_file(
    THRESHOLD_GRID_PATH
)


print("=" * 118)
print("PERSISTED ARTIFACT IDENTITIES")
print("=" * 118)


print(
    "Model SHA:             ",
    model_sha,
)

print(
    "Validation array SHA:  ",
    val_probability_sha,
)

print(
    "Validation NPZ SHA:    ",
    val_pred_file_sha,
)

print(
    "Threshold grid SHA:    ",
    threshold_grid_sha,
)

print()


# ==============================================================================
# 20. RESULT RECEIPT
# ==============================================================================

result = {
    "stage":
        "Stage24-4A-R1",

    "status":
        "SECONDARY_BRIDGE62_SOURCE_MODEL_FROZEN",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "direction":
        "CICIDS2017_TO_IDS2018",

    "bridge":
        "bridge62",

    "learner":
        "XGBOOST_ONLY",

    "learner_configuration":
        "INHERIT_STAGE22R_XGB_11",

    "parent_commit":
        EXPECTED_PARENT,

    "backend_amendment": {
        "artifact":
            str(
                AMENDMENT_PATH.relative_to(
                    REPO
                )
            ),

        "sha256":
            amendment_sha,

        "original_device":
            "cuda",

        "execution_device":
            "cpu",

        "scope":
            "EXECUTION_BACKEND_ONLY",
    },

    "scientific_fit": {
        "fit_number":
            3,

        "fit_budget":
            4,

        "completed_before":
            EXPECTED_FITS_BEFORE,

        "completed_after":
            EXPECTED_FITS_AFTER,

        "fit_started_at_utc":
            fit_started_at,

        "fit_seconds":
            fit_seconds,

        "hyperparameter_search":
            False,
    },

    "xgboost": {
        "library_version":
            xgb.__version__,

        "parameters":
            XGB_PARAMS,

        "resolved_device":
            "cpu",

        "feature_count":
            62,

        "boosted_rounds":
            400,

        "model_file":
            str(
                MODEL_PATH.relative_to(
                    REPO
                )
            ),

        "model_sha256":
            model_sha,
    },

    "representation": {
        "feature_count":
            62,

        "feature_order":
            FEATURE_ORDER,

        "cicids2017_source_mapping":
            SOURCE_MAPPING,

        "published_equals_flag_corrected":
            True,

        "parse_dtype":
            "float64",

        "positive_infinity":
            "CONVERT_TO_NAN",

        "negative_infinity":
            "CONVERT_TO_NAN",

        "explicit_imputation":
            "NONE",

        "scaling":
            "NONE",
    },

    "source_materialization": {
        "mode":
            source_materialization_mode,

        "train_feature_matrix_sha256":
            train_feature_sha,

        "train_binary_label_sha256":
            train_label_sha,

        "validation_feature_matrix_sha256":
            val_feature_sha,

        "validation_binary_label_sha256":
            val_label_sha,
    },

    "source_train": {
        "dataset":
            "CICIDS2017",

        "days":
            [
                "Monday",
                "Tuesday",
                "Wednesday",
            ],

        "rows":
            EXPECTED_TRAIN_ROWS,

        "benign":
            train_benign,

        "attack":
            train_attack,

        "attack_prevalence":
            source_prior,
    },

    "source_validation": {
        "dataset":
            "CICIDS2017",

        "days":
            [
                "Thursday",
            ],

        "rows":
            EXPECTED_VAL_ROWS,

        "benign":
            val_benign,

        "attack":
            val_attack,

        "prevalence":
            val_prevalence,

        "probability_array_sha256":
            val_probability_sha,

        "probability_artifact":
            str(
                VAL_PRED_PATH.relative_to(
                    REPO
                )
            ),

        "probability_artifact_sha256":
            val_pred_file_sha,

        "metrics": {
            "pr_auc":
                pr_auc,

            "roc_auc":
                roc_auc,

            "pr_excess":
                pr_excess,

            "pr_normalized":
                pr_normalized,

            "brier":
                brier,

            "log_loss":
                log_loss,

            "ece_10":
                ece,
        },

        "operating_points": {
            "standard":
                standard,

            "balanced":
                balanced_metrics,

            "security": {
                "status":
                    (
                        "AVAILABLE"
                        if security_metrics is not None
                        else "UNAVAILABLE_NO_RELAXATION"
                    ),

                "result":
                    security_metrics,
            },
        },

        "threshold_grid_file":
            str(
                THRESHOLD_GRID_PATH.relative_to(
                    REPO
                )
            ),

        "threshold_grid_sha256":
            threshold_grid_sha,

        "threshold_selection_data":
            "SOURCE_VALIDATION_ONLY",
    },

    "secondary_source_prior_constant_predictor": {
        "fit":
            False,

        "probability":
            source_prior,

        "computed_on":
            "CICIDS2017_MONDAY_WEDNESDAY_SOURCE_TRAIN_ONLY",
    },

    "unused_source": {
        "days":
            [
                "Friday",
            ],

        "read_for_model_training":
            False,

        "read_for_validation":
            False,
    },

    "target_access": {
        "dataset":
            "IDS2018",

        "target":
            "02-28-2018",

        "target_features_read":
            False,

        "target_labels_read":
            False,

        "target_model_predictions":
            0,

        "target_metrics":
            0,

        "new_target_openings":
            0,

        "total_target_openings_consumed":
            4,
    },

    "anti_adaptation": {
        "hyperparameter_search":
            False,

        "target_guided_refit":
            False,

        "target_guided_threshold":
            False,

        "target_feature_search":
            False,

        "target_preprocessing_fit":
            False,

        "performance_based_change":
            False,

        "backend_change_only":
            True,
    },

    "fit_accounting": {
        "scientific_fit_budget":
            4,

        "scientific_fits_completed":
            3,

        "scientific_fits_remaining":
            1,
    },

    "next_authorized_step":
        (
            "STAGE24_4B_SECONDARY_BRIDGE70_XGBOOST_CPU_SOURCE_FIT"
        ),
}


RESULT_PATH.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


result_sha = sha256_file(
    RESULT_PATH
)


RESULT_SHA_PATH.write_text(
    f"{result_sha}  {RESULT_PATH.name}\n",
    encoding="utf-8",
)


CHECKSUMS_PATH.write_text(
    (
        f"{result_sha}  {RESULT_PATH.name}\n"
        f"{model_sha}  {MODEL_PATH.name}\n"
        f"{val_pred_file_sha}  {VAL_PRED_PATH.name}\n"
        f"{threshold_grid_sha}  {THRESHOLD_GRID_PATH.name}\n"
    ),
    encoding="utf-8",
)


print(
    "Result SHA:",
    result_sha,
)

print()


# ==============================================================================
# 21. GIT AUTHOR SAFETY
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    git_cmd(
        "config",
        "--local",
        "user.name",
        git_cmd(
            "log",
            "-1",
            "--format=%an",
        ),
    )


if not author_email:

    git_cmd(
        "config",
        "--local",
        "user.email",
        git_cmd(
            "log",
            "-1",
            "--format=%ae",
        ),
    )


# ==============================================================================
# 22. GIT FREEZE
# ==============================================================================

print("=" * 118)
print("GIT FREEZE")
print("=" * 118)


git_cmd(
    "add",
    "--",
    str(
        MODEL_PATH.relative_to(
            REPO
        )
    ),
    str(
        VAL_PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        THRESHOLD_GRID_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


expected_staged = {
    str(
        MODEL_PATH.relative_to(
            REPO
        )
    ),
    str(
        VAL_PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        THRESHOLD_GRID_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: freeze secondary bridge62 cpu source model",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage24-4A-R1 commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 23. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 118)
print("PUSH + REMOTE VERIFY")
print("=" * 118)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository not clean after Stage24-4A-R1:\n"
        + final_status
    )


# Fit is now durable remotely.
if FIT_LEDGER.exists():

    FIT_LEDGER.unlink()


print(
    "[PASS] Remote main == local Stage24-4A-R1 commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 24. FINAL
# ==============================================================================

print("=" * 118)
print("STAGE24-4A-R1 SECONDARY BRIDGE62 SOURCE FIT: PASS")
print("=" * 118)

print()

print(
    "Direction:                    CICIDS2017 -> IDS2018"
)

print(
    "Bridge:                       bridge62"
)

print(
    "Learner:                      XGBoost"
)

print(
    "Execution backend:            CPU"
)

print()

print(
    "Train rows:                   ",
    f"{EXPECTED_TRAIN_ROWS:,}",
)

print(
    "Train benign:                 ",
    f"{train_benign:,}",
)

print(
    "Train attack:                 ",
    f"{train_attack:,}",
)

print(
    "Source prior:                 ",
    f"{source_prior:.12f}",
)

print()

print(
    "Validation rows:              ",
    f"{EXPECTED_VAL_ROWS:,}",
)

print(
    "Validation benign:            ",
    f"{val_benign:,}",
)

print(
    "Validation attack:            ",
    f"{val_attack:,}",
)

print()

print(
    "Validation PR-AUC:            ",
    f"{pr_auc:.15f}",
)

print(
    "Validation ROC-AUC:           ",
    f"{roc_auc:.15f}",
)

print(
    "Validation Brier:             ",
    f"{brier:.15f}",
)

print()

print(
    "Balanced threshold:           ",
    balanced_metrics[
        "threshold_float32_runtime"
    ],
)

print(
    "Security threshold:           ",
    (
        security_metrics[
            "threshold_float32_runtime"
        ]
        if security_metrics is not None
        else "UNAVAILABLE"
    ),
)

print()

print(
    "Model SHA:"
)

print(
    " ",
    model_sha,
)

print()

print(
    "Validation probability SHA:"
)

print(
    " ",
    val_probability_sha,
)

print()

print(
    "Scientific fits completed:    3 / 4"
)

print(
    "Scientific fits remaining:    1"
)

print()

print(
    "New target openings:          0"
)

print(
    "Target openings consumed:     4 / 8"
)

print(
    "IDS2018 Feb-28 target opened: NO"
)

print()

print(
    "Result SHA:"
)

print(
    " ",
    result_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "NEXT:"
)

print(
    "  Stage24-4B — secondary bridge70 XGBoost CPU fit"
)

print(
    "  scientific fit #4 / 4."
)

print("=" * 118)

STAGE24-4A-R1 — SECONDARY BRIDGE62 XGBOOST CPU FIT

GOVERNANCE
HEAD: c8c97de2a6088b67f07727608d2a97d33b90dda9
[PASS] Repository clean.
[PASS] No unresolved scientific-fit ledger.
[PASS] No prior Stage24-4A scientific artifacts.

CPU BACKEND AMENDMENT
Amendment SHA: 2c356c8763c567702d14a74c042df42356f360e559e574c3d9a4abfa652e5c14
Execution device: CPU

[PASS] Scientific fit #3 remains unconsumed.
[PASS] CPU-only execution amendment is frozen.

[PASS] Frozen secondary source/validation contract exact.
[PASS] Target openings remain 4 / 8.

BRIDGE62 REPRESENTATION
Features: 62
[PASS] PUBLISHED == FLAG_CORRECTED for bridge62.

RUNTIME MATRIX RECOVERY
X_train   : True
y_train   : True
X_val     : True
y_val     : True

Train feature SHA: 0058030b9684e917a403ff09ad6524016dc9c38c38352ff0070643a4e8bed9b0
Train label SHA:   7b30dd8aa0a60e6c8d8715a44fae5c2019469c56624466841854daca8c0d6c8f
Val feature SHA:   7da9498e27faf2acbbacc0999b007b0ec50d0e911fa13fa57cbbec9be378ba4c
Val label SHA:     b1be03

In [40]:
# ==============================================================================
# STAGE24-4B — SECONDARY BRIDGE70 XGBOOST CPU SOURCE FIT
#
# Direction:
#   CICIDS2017 -> IDS2018
#
# Scientific fit:
#   #4 / 4   <-- FINAL SCIENTIFIC FIT IN STAGE24
#
# Frozen source:
#   TRAIN      Monday + Tuesday + Wednesday
#   VALIDATION Thursday
#   UNUSED     Friday
#
# Frozen representation:
#   bridge70
#
# CICIDS2017 source semantic projection:
#   FLAG_CORRECTED mapping from frozen Stage24 semantic bridge.
#
# Frozen learner:
#   XGBOOST_ONLY
#   INHERIT_STAGE22R_XGB_11
#
# Frozen CPU amendment:
#   device = cpu
#   tree_method = hist
#   all other XGB_11 hyperparameters unchanged.
#
# NO IDS2018 FEB-28 TARGET READ.
# NO TARGET INFERENCE.
# NO TARGET OPENING.
#
# Once:
#
#     >>> SCIENTIFIC FIT #4 CONSUMED <<<
#
# appears, DO NOT rerun this cell after any later failure.
# ==============================================================================

from __future__ import annotations

import os
import gc
import csv
import json
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import duckdb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


print("=" * 118)
print("STAGE24-4B — SECONDARY BRIDGE70 XGBOOST CPU SOURCE FIT")
print("=" * 118)
print()


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "e58019523237d8055c044f65f76d28ccf568e20c"
)

EXPECTED_AMENDMENT_SHA = (
    "2c356c8763c567702d14a74c042df42356f360e559e574c3d9a4abfa652e5c14"
)

EXPECTED_BRIDGE_SPEC_SHA = (
    "b7ff1d8563aafc69145be3acc827fbc59ae75d880b9804bad727f408299fce2f"
)

EXPECTED_PROTOCOL_LOCK_SHA = (
    "8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2"
)


EXPECTED_TRAIN_ROWS = 1_668_530
EXPECTED_TRAIN_BENIGN = 1_402_023
EXPECTED_TRAIN_ATTACK = 266_507

EXPECTED_VAL_ROWS = 458_968
EXPECTED_VAL_BENIGN = 456_752
EXPECTED_VAL_ATTACK = 2_216


# Canonical labels already frozen during bridge62 source construction.
EXPECTED_TRAIN_LABEL_SHA = (
    "7b30dd8aa0a60e6c8d8715a44fae5c2019469c56624466841854daca8c0d6c8f"
)

EXPECTED_VAL_LABEL_SHA = (
    "b1be032e4bde759b59a935476d7b440f6f0934f348ed0302af13d310137d0e95"
)


EXPECTED_OPENINGS = 4
EXPECTED_FITS_BEFORE = 3
EXPECTED_FITS_AFTER = 4


# ==============================================================================
# 1. PATHS
# ==============================================================================

LOCK_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
)


BRIDGE_SPEC_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.json"
)

BRIDGE_SPEC_SHA_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.sha256"
)


FINAL_LOCK_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.json"
)

FINAL_LOCK_SHA_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.sha256"
)


AMENDMENT_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_4_secondary_source_training"
    / "stage24_4a_ex1_backend_amendment"
)

AMENDMENT_PATH = (
    AMENDMENT_DIR
    / "stage24_4a_ex1_xgboost_cpu_backend_amendment.json"
)

AMENDMENT_SHA_PATH = (
    AMENDMENT_DIR
    / "stage24_4a_ex1_xgboost_cpu_backend_amendment.sha256"
)


B62_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_4_secondary_source_training"
    / "stage24_4a_bridge62_xgboost"
)

B62_RESULT_PATH = (
    B62_DIR
    / "stage24_4a_secondary_bridge62_result.json"
)

B62_RESULT_SHA_PATH = (
    B62_DIR
    / "stage24_4a_secondary_bridge62_result.sha256"
)


TARGET_ROOT = Path(
    "/kaggle/working/stage24_cicids2017_hf_cache/"
    "datasets--bvsam--cic-ids-2017/"
    "snapshots/b7e532345512edcd530cb1770dc76636aeb52802/"
    "traffic_labels"
)


OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_4_secondary_source_training"
    / "stage24_4b_bridge70_xgboost"
)


MODEL_PATH = (
    OUT
    / "secondary_bridge70_xgboost_model.json"
)


VAL_PRED_PATH = (
    OUT
    / "secondary_bridge70_validation_probabilities.npz"
)


THRESHOLD_GRID_PATH = (
    OUT
    / "secondary_bridge70_validation_threshold_grid.csv"
)


RESULT_PATH = (
    OUT
    / "stage24_4b_secondary_bridge70_result.json"
)


RESULT_SHA_PATH = (
    OUT
    / "stage24_4b_secondary_bridge70_result.sha256"
)


CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


FIT_LEDGER = Path(
    "/kaggle/working/stage24_secondary_fit_runtime_ledger.json"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    cmd = [
        "git"
    ]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [
        str(x)
        for x in args
    ]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_array(
    array,
):

    arr = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        arr.view(
            np.uint8
        )
    ).hexdigest()


def verify_sidecar(
    artifact,
    sidecar,
):

    if not artifact.is_file():

        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )

    if not sidecar.is_file():

        raise RuntimeError(
            f"Missing SHA sidecar:\n{sidecar}"
        )


    actual = sha256_file(
        artifact
    )


    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )


    if actual != expected:

        raise RuntimeError(
            "\nSHA mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


    return actual


def qident(
    value,
):

    return (
        '"'
        +
        str(
            value
        ).replace(
            '"',
            '""',
        )
        +
        '"'
    )


def sql_path(
    path,
):

    return str(
        path
    ).replace(
        "'",
        "''",
    )


# ==============================================================================
# 3. GOVERNANCE
# ==============================================================================

print("=" * 118)
print("GOVERNANCE")
print("=" * 118)


head = git_cmd(
    "rev-parse",
    "HEAD",
)


status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


if FIT_LEDGER.exists():

    raise RuntimeError(
        "\nA secondary scientific-fit ledger exists:\n"
        f"{FIT_LEDGER}\n\n"
        "DO NOT rerun fit #4 blindly."
    )


if OUT.exists() and any(
    OUT.iterdir()
):

    raise RuntimeError(
        "\nStage24-4B scientific artifacts already exist.\n"
        "Refusing duplicate scientific fit."
    )


print(
    "[PASS] Repository clean."
)

print(
    "[PASS] No unresolved scientific-fit ledger."
)

print(
    "[PASS] Stage24-4B has not previously been fit."
)

print()


# ==============================================================================
# 4. VERIFY BRIDGE62 FIT #3 IS DURABLE
# ==============================================================================

print("=" * 118)
print("SCIENTIFIC FIT #3 REFERENCE")
print("=" * 118)


b62_result_sha = verify_sidecar(
    B62_RESULT_PATH,
    B62_RESULT_SHA_PATH,
)


b62_result = json.loads(
    B62_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    b62_result[
        "status"
    ]
    !=
    "SECONDARY_BRIDGE62_SOURCE_MODEL_FROZEN"
):

    raise RuntimeError(
        "Stage24-4A status mismatch."
    )


if int(
    b62_result[
        "fit_accounting"
    ][
        "scientific_fits_completed"
    ]
) != 3:

    raise RuntimeError(
        "Stage24-4A fit accounting mismatch."
    )


if int(
    b62_result[
        "target_access"
    ][
        "total_target_openings_consumed"
    ]
) != EXPECTED_OPENINGS:

    raise RuntimeError(
        "Target opening accounting changed."
    )


if (
    b62_result[
        "target_access"
    ][
        "target_features_read"
    ]
    is not False
):

    raise RuntimeError(
        "IDS2018 target was unexpectedly read during fit #3."
    )


print(
    "Stage24-4A result SHA:",
    b62_result_sha,
)

print(
    "[PASS] Scientific fit #3 durable."
)

print(
    "[PASS] Target openings remain 4 / 8."
)

print()


# ==============================================================================
# 5. VERIFY CPU AMENDMENT
# ==============================================================================

print("=" * 118)
print("CPU BACKEND AMENDMENT")
print("=" * 118)


amendment_sha = verify_sidecar(
    AMENDMENT_PATH,
    AMENDMENT_SHA_PATH,
)


if amendment_sha != EXPECTED_AMENDMENT_SHA:

    raise RuntimeError(
        "CPU amendment SHA changed."
    )


amendment = json.loads(
    AMENDMENT_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    amendment[
        "authorization_after_commit"
    ][
        "bridge70_secondary_fit_backend"
    ]
    !=
    "CPU_SAME_AMENDMENT"
):

    raise RuntimeError(
        "CPU amendment does not authorize bridge70."
    )


if (
    amendment[
        "authorization_after_commit"
    ][
        "additional_backend_amendment_required_for_bridge70"
    ]
    is not False
):

    raise RuntimeError(
        "Unexpected bridge70 backend-amendment requirement."
    )


if (
    amendment[
        "frozen_xgb_11_after_amendment"
    ][
        "device"
    ]
    !=
    "cpu"
):

    raise RuntimeError(
        "Frozen amended backend is not CPU."
    )


print(
    "Amendment SHA:",
    amendment_sha,
)

print(
    "[PASS] Existing CPU amendment explicitly covers bridge70."
)

print()


# ==============================================================================
# 6. VERIFY FROZEN PROTOCOL / SEMANTIC BRIDGE
# ==============================================================================

bridge_sha = verify_sidecar(
    BRIDGE_SPEC_PATH,
    BRIDGE_SPEC_SHA_PATH,
)


lock_sha = verify_sidecar(
    FINAL_LOCK_PATH,
    FINAL_LOCK_SHA_PATH,
)


if bridge_sha != EXPECTED_BRIDGE_SPEC_SHA:

    raise RuntimeError(
        "Semantic bridge SHA changed."
    )


if lock_sha != EXPECTED_PROTOCOL_LOCK_SHA:

    raise RuntimeError(
        "Protocol lock SHA changed."
    )


bridge_spec = json.loads(
    BRIDGE_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)


final_lock = json.loads(
    FINAL_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


secondary = final_lock[
    "secondary_direction"
]


if secondary[
    "direction"
] != "CICIDS2017_TO_IDS2018":

    raise RuntimeError(
        "Secondary direction changed."
    )


if secondary[
    "learner"
] != "XGBOOST_ONLY":

    raise RuntimeError(
        "Secondary learner changed."
    )


if secondary[
    "learner_configuration"
] != "INHERIT_STAGE22R_XGB_11":

    raise RuntimeError(
        "Secondary learner configuration changed."
    )


if secondary[
    "source_train_days"
] != [
    "Monday",
    "Tuesday",
    "Wednesday",
]:

    raise RuntimeError(
        "Secondary training membership changed."
    )


if secondary[
    "source_validation_days"
] != [
    "Thursday"
]:

    raise RuntimeError(
        "Secondary validation membership changed."
    )


if secondary[
    "source_unused_days"
] != [
    "Friday"
]:

    raise RuntimeError(
        "Friday unused-source contract changed."
    )


print(
    "[PASS] Frozen secondary contract exact."
)

print()


# ==============================================================================
# 7. BRIDGE70 FLAG-CORRECTED SOURCE SEMANTICS
# ==============================================================================

print("=" * 118)
print("BRIDGE70 SOURCE REPRESENTATION")
print("=" * 118)


b70 = bridge_spec[
    "bridge70"
]


FEATURE_ORDER = list(
    b70[
        "source_feature_order"
    ]
)


PUBLISHED_MAPPING = dict(
    b70[
        "PUBLISHED_mapping"
    ]
)


SOURCE_MAPPING = dict(
    b70[
        "FLAG_CORRECTED_mapping"
    ]
)


if len(
    FEATURE_ORDER
) != 70:

    raise RuntimeError(
        "bridge70 feature count changed."
    )


if set(
    FEATURE_ORDER
) != set(
    SOURCE_MAPPING
):

    raise RuntimeError(
        "bridge70 FLAG_CORRECTED mapping incomplete."
    )


EXPECTED_FLAG_MAPPING = {
    "FIN Flag Cnt":
        "URG Flag Count",

    "SYN Flag Cnt":
        "PSH Flag Count",

    "RST Flag Cnt":
        "FIN Flag Count",

    "PSH Flag Cnt":
        "SYN Flag Count",

    "ACK Flag Cnt":
        "ACK Flag Count",

    "URG Flag Cnt":
        "CWE Flag Count",

    "CWE Flag Count":
        "ECE Flag Count",

    "ECE Flag Cnt":
        "RST Flag Count",
}


for feature, physical in EXPECTED_FLAG_MAPPING.items():

    if SOURCE_MAPPING[
        feature
    ] != physical:

        raise RuntimeError(
            "\nFrozen corrected flag mapping mismatch.\n"
            f"{feature}: expected {physical}, "
            f"actual {SOURCE_MAPPING[feature]}"
        )


mapping_differences = [
    feature
    for feature in FEATURE_ORDER
    if (
        PUBLISHED_MAPPING[
            feature
        ]
        !=
        SOURCE_MAPPING[
            feature
        ]
    )
]


if len(
    mapping_differences
) != 7:

    raise RuntimeError(
        "\nExpected exactly 7 PUBLISHED-vs-CORRECTED physical mapping changes."
    )


print(
    "Feature count:",
    len(
        FEATURE_ORDER
    ),
)

print(
    "Corrected physical mapping changes:",
    len(
        mapping_differences
    ),
)

print(
    "ACK mapping invariant:",
    SOURCE_MAPPING[
        "ACK Flag Cnt"
    ]
    ==
    PUBLISHED_MAPPING[
        "ACK Flag Cnt"
    ],
)

print()

print(
    "[PASS] Frozen bridge70 FLAG_CORRECTED source semantics exact."
)

print()


# ==============================================================================
# 8. FROZEN SOURCE FILES
# ==============================================================================

TRAIN_FILES = [
    (
        "Monday",
        "Monday-WorkingHours.pcap_ISCX.csv.parquet",
        529_918,
    ),
    (
        "Tuesday",
        "Tuesday-WorkingHours.pcap_ISCX.csv.parquet",
        445_909,
    ),
    (
        "Wednesday",
        "Wednesday-workingHours.pcap_ISCX.csv.parquet",
        692_703,
    ),
]


VALIDATION_FILES = [
    (
        "Thursday-Afternoon",
        "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet",
        288_602,
    ),
    (
        "Thursday-Morning",
        "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet",
        170_366,
    ),
]


# ==============================================================================
# 9. RECOVER CANONICAL SOURCE LABELS FROM CURRENT KERNEL IF POSSIBLE
# ==============================================================================

print("=" * 118)
print("CANONICAL LABEL RECOVERY")
print("=" * 118)


labels_reused = False


if (
    "y_train"
    in globals()
    and
    "y_val"
    in globals()
):

    candidate_y_train = np.asarray(
        globals()[
            "y_train"
        ],
        dtype=np.uint8,
    )


    candidate_y_val = np.asarray(
        globals()[
            "y_val"
        ],
        dtype=np.uint8,
    )


    if (
        candidate_y_train.shape
        ==
        (
            EXPECTED_TRAIN_ROWS,
        )
        and
        candidate_y_val.shape
        ==
        (
            EXPECTED_VAL_ROWS,
        )
        and
        sha256_array(
            candidate_y_train
        )
        ==
        EXPECTED_TRAIN_LABEL_SHA
        and
        sha256_array(
            candidate_y_val
        )
        ==
        EXPECTED_VAL_LABEL_SHA
    ):

        y_train = candidate_y_train

        y_val = candidate_y_val

        labels_reused = True


if labels_reused:

    print(
        "[PASS] Reusing exact canonical source labels from fit #3 runtime."
    )


else:

    print(
        "[INFO] Canonical labels not available in runtime; reconstructing."
    )


    y_train = np.empty(
        EXPECTED_TRAIN_ROWS,
        dtype=np.uint8,
    )


    y_val = np.empty(
        EXPECTED_VAL_ROWS,
        dtype=np.uint8,
    )


print()


# ==============================================================================
# 10. FREE STALE LARGE RUNTIME OBJECTS
#
# Keep only y_train / y_val.
# ==============================================================================

print("=" * 118)
print("RUNTIME MEMORY HOUSEKEEPING")
print("=" * 118)


for stale_name in [
    "X_train",
    "X_val",
    "p_val",
    "p_ens_all",
    "corrected_probability",
    "published_probability",
    "bridge62_probability",
    "p62",
    "p70_pub",
    "p70_corr",
    "group_p62",
    "group_p70_pub",
    "group_p70_corr",
    "model",
    "booster",
]:

    if stale_name in globals():

        globals().pop(
            stale_name,
            None,
        )


gc.collect()


try:

    import psutil

    vm = psutil.virtual_memory()

    print(
        "Available RAM:",
        f"{vm.available / (1024**3):.2f} GiB",
    )

except Exception:

    print(
        "Available RAM: <psutil unavailable>"
    )


print(
    "[PASS] Stale fit #3 feature/model objects released where present."
)

print()


# ==============================================================================
# 11. MATERIALIZE FROZEN BRIDGE70 SOURCE MATRICES
# ==============================================================================

print("=" * 118)
print("BRIDGE70 SOURCE MATERIALIZATION")
print("=" * 118)


X_train70 = np.empty(
    (
        EXPECTED_TRAIN_ROWS,
        70,
    ),
    dtype=np.float64,
)


X_val70 = np.empty(
    (
        EXPECTED_VAL_ROWS,
        70,
    ),
    dtype=np.float64,
)


DASH_TRANSLATION = str.maketrans(
    {
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2015": "-",
        "\u2212": "-",
    }
)


def canonicalize_labels(
    series,
):

    if series.isna().any():

        raise RuntimeError(
            "NULL label survived frozen effective-row rule."
        )


    s = (
        series
        .astype(
            "string"
        )
        .str.strip()
        .str.translate(
            DASH_TRANSLATION
        )
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.casefold()
    )


    if s.isna().any():

        raise RuntimeError(
            "Label canonicalization generated NULL."
        )


    if (
        s.str.len()
        ==
        0
    ).any():

        raise RuntimeError(
            "Empty source label encountered."
        )


    return s


projection_sql = ",\n".join(
    (
        f"CAST({qident(SOURCE_MAPPING[source_feature])} AS DOUBLE) "
        f"AS {qident(source_feature)}"
    )
    for source_feature in FEATURE_ORDER
)


conn = duckdb.connect(
    database=":memory:"
)


numeric_audit = []


def materialize_split(
    specs,
    X_out,
    y_out,
    expected_total,
    split_name,
    labels_already_frozen,
):

    cursor = 0


    for day, filename, expected_rows in specs:

        path = (
            TARGET_ROOT
            /
            filename
        )


        if not path.is_file():

            raise RuntimeError(
                f"Missing source Parquet:\n{path}"
            )


        query = f"""
            SELECT
                {projection_sql},
                CAST("Label" AS VARCHAR) AS "__label__"
            FROM read_parquet('{sql_path(path)}')
            WHERE "Label" IS NOT NULL
        """


        t0 = time.time()


        df = conn.execute(
            query
        ).df()


        if len(
            df
        ) != expected_rows:

            raise RuntimeError(
                "\nFrozen source row mismatch.\n"
                f"Split:    {split_name}\n"
                f"File:     {filename}\n"
                f"Expected: {expected_rows:,}\n"
                f"Actual:   {len(df):,}"
            )


        X = df[
            FEATURE_ORDER
        ].to_numpy(
            dtype=np.float64,
            copy=True,
        )


        if X.shape != (
            expected_rows,
            70,
        ):

            raise RuntimeError(
                f"Unexpected bridge70 matrix shape: {X.shape}"
            )


        positive_inf = int(
            np.isposinf(
                X
            ).sum()
        )


        negative_inf = int(
            np.isneginf(
                X
            ).sum()
        )


        inf_mask = np.isinf(
            X
        )


        if inf_mask.any():

            X[
                inf_mask
            ] = np.nan


        nan_count = int(
            np.isnan(
                X
            ).sum()
        )


        canonical = canonicalize_labels(
            df[
                "__label__"
            ]
        )


        y_file = (
            canonical
            !=
            "benign"
        ).to_numpy(
            dtype=np.uint8,
        )


        start = cursor

        stop = (
            start
            +
            expected_rows
        )


        X_out[
            start:stop,
            :
        ] = X


        if labels_already_frozen:

            if not np.array_equal(
                y_file,
                y_out[
                    start:stop
                ],
            ):

                raise RuntimeError(
                    "\nBridge70 source row/label order differs from frozen bridge62 source order."
                )


        else:

            y_out[
                start:stop
            ] = y_file


        print(
            f"{split_name:10s}"
            f" {day:20s}"
            f" rows={expected_rows:9,d}"
            f" benign={int((y_file == 0).sum()):9,d}"
            f" attack={int((y_file == 1).sum()):8,d}"
            f" +inf={positive_inf:6,d}"
            f" -inf={negative_inf:6,d}"
            f" nan={nan_count:6,d}"
            f" sec={time.time() - t0:6.2f}"
        )


        numeric_audit.append(
            {
                "split":
                    split_name,

                "day":
                    day,

                "file":
                    filename,

                "rows":
                    expected_rows,

                "positive_inf_to_nan":
                    positive_inf,

                "negative_inf_to_nan":
                    negative_inf,

                "nan_cells_after_inf_conversion":
                    nan_count,
            }
        )


        cursor = stop


        del df
        del X
        del y_file
        del canonical
        del inf_mask

        gc.collect()


    if cursor != expected_total:

        raise RuntimeError(
            f"{split_name}: final source-row mismatch."
        )


materialize_split(
    TRAIN_FILES,
    X_train70,
    y_train,
    EXPECTED_TRAIN_ROWS,
    "TRAIN",
    labels_reused,
)


materialize_split(
    VALIDATION_FILES,
    X_val70,
    y_val,
    EXPECTED_VAL_ROWS,
    "VALIDATION",
    labels_reused,
)


conn.close()


# ==============================================================================
# 12. SOURCE POPULATION + MATRIX IDENTITIES
# ==============================================================================

train_attack = int(
    y_train.sum()
)


train_benign = int(
    EXPECTED_TRAIN_ROWS
    -
    train_attack
)


val_attack = int(
    y_val.sum()
)


val_benign = int(
    EXPECTED_VAL_ROWS
    -
    val_attack
)


if train_benign != EXPECTED_TRAIN_BENIGN:

    raise RuntimeError(
        "Training benign count mismatch."
    )


if train_attack != EXPECTED_TRAIN_ATTACK:

    raise RuntimeError(
        "Training attack count mismatch."
    )


if val_benign != EXPECTED_VAL_BENIGN:

    raise RuntimeError(
        "Validation benign count mismatch."
    )


if val_attack != EXPECTED_VAL_ATTACK:

    raise RuntimeError(
        "Validation attack count mismatch."
    )


train_label_sha = sha256_array(
    y_train
)


val_label_sha = sha256_array(
    y_val
)


if train_label_sha != EXPECTED_TRAIN_LABEL_SHA:

    raise RuntimeError(
        "Training label SHA differs from frozen source membership."
    )


if val_label_sha != EXPECTED_VAL_LABEL_SHA:

    raise RuntimeError(
        "Validation label SHA differs from frozen source membership."
    )


train_feature_sha = sha256_array(
    X_train70
)


val_feature_sha = sha256_array(
    X_val70
)


source_prior = float(
    train_attack
    /
    EXPECTED_TRAIN_ROWS
)


val_prevalence = float(
    val_attack
    /
    EXPECTED_VAL_ROWS
)


print()
print("=" * 118)
print("SOURCE MATRIX IDENTITIES")
print("=" * 118)


print(
    "Train feature SHA:",
    train_feature_sha,
)

print(
    "Train label SHA:  ",
    train_label_sha,
)

print(
    "Val feature SHA:  ",
    val_feature_sha,
)

print(
    "Val label SHA:    ",
    val_label_sha,
)

print()

print(
    "TRAIN rows:  ",
    f"{EXPECTED_TRAIN_ROWS:,}",
)

print(
    "TRAIN benign:",
    f"{train_benign:,}",
)

print(
    "TRAIN attack:",
    f"{train_attack:,}",
)

print(
    "TRAIN prior: ",
    f"{source_prior:.15f}",
)

print()

print(
    "VAL rows:    ",
    f"{EXPECTED_VAL_ROWS:,}",
)

print(
    "VAL benign:  ",
    f"{val_benign:,}",
)

print(
    "VAL attack:  ",
    f"{val_attack:,}",
)

print(
    "VAL prior:   ",
    f"{val_prevalence:.15f}",
)

print()

print(
    "[PASS] Frozen secondary bridge70 source population exact."
)

print()


# ==============================================================================
# 13. GITHUB CREDENTIAL + REMOTE GATE
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()


    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )


            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )


        if value and value.strip():

            github_token = value.strip()
            token_source = (
                "ENV:"
                +
                name
            )

            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable BEFORE scientific fit #4."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved before scientific fit #4.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main == Stage24-4A bridge62 fit commit."
)

print()


# ==============================================================================
# 14. FROZEN XGB_11 CPU CONFIGURATION
# ==============================================================================

XGB_PARAMS = {
    "objective":
        "binary:logistic",

    "n_estimators":
        400,

    "max_depth":
        7,

    "learning_rate":
        0.06,

    "subsample":
        0.9,

    "colsample_bytree":
        1.0,

    "min_child_weight":
        1,

    "gamma":
        0.0,

    "reg_alpha":
        0.0,

    "reg_lambda":
        3.0,

    "tree_method":
        "hist",

    "device":
        "cpu",

    "eval_metric":
        "logloss",

    "random_state":
        42,

    "n_jobs":
        -1,
}


print("=" * 118)
print("FROZEN XGB_11 — CPU")
print("=" * 118)


for key in sorted(
    XGB_PARAMS
):

    print(
        f"{key:20s} = {XGB_PARAMS[key]}"
    )


print()


# ==============================================================================
# 15. SCIENTIFIC FIT #4 / 4
# ==============================================================================

import xgboost as xgb


print(
    "XGBoost version:",
    xgb.__version__,
)

print()


fit_started_at = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


FIT_LEDGER.write_text(
    json.dumps(
        {
            "stage":
                "Stage24-4B",

            "direction":
                "CICIDS2017_TO_IDS2018",

            "bridge":
                "bridge70",

            "learner":
                "XGBOOST_ONLY",

            "backend":
                "CPU_FROZEN_AMENDMENT",

            "scientific_fit_number":
                4,

            "scientific_fit_budget":
                4,

            "fit_attempt_started":
                True,

            "fit_completed":
                False,

            "fit_started_at_utc":
                fit_started_at,

            "parent_commit":
                EXPECTED_PARENT,

            "target_openings_consumed":
                4,

            "IDS2018_target_features_read":
                False,

            "IDS2018_target_predictions":
                0,
        },
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


print("=" * 118)
print("SCIENTIFIC FIT #4 / 4")
print("=" * 118)

print()
print(
    ">>> SCIENTIFIC FIT #4 CONSUMED <<<"
)
print()


model70 = xgb.XGBClassifier(
    **XGB_PARAMS
)


fit_start = time.time()


model70.fit(
    X_train70,
    y_train,
    verbose=False,
)


fit_seconds = (
    time.time()
    -
    fit_start
)


print(
    "Fit seconds:",
    f"{fit_seconds:.3f}",
)

print()


booster70 = model70.get_booster()


if booster70.num_features() != 70:

    raise RuntimeError(
        "Fitted bridge70 model dimension mismatch."
    )


if booster70.num_boosted_rounds() != 400:

    raise RuntimeError(
        "Fitted bridge70 boosted-round count mismatch."
    )


config_text = (
    booster70
    .save_config()
    .replace(
        " ",
        ""
    )
)


if '"device":"cpu"' not in config_text:

    raise RuntimeError(
        "Fitted bridge70 XGBoost configuration does not resolve to CPU."
    )


# ==============================================================================
# 16. SAVE MODEL IMMEDIATELY AFTER FIT
# ==============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


booster70.save_model(
    MODEL_PATH
)


model_sha = sha256_file(
    MODEL_PATH
)


FIT_LEDGER.write_text(
    json.dumps(
        {
            "stage":
                "Stage24-4B",

            "direction":
                "CICIDS2017_TO_IDS2018",

            "bridge":
                "bridge70",

            "learner":
                "XGBOOST_ONLY",

            "backend":
                "CPU_FROZEN_AMENDMENT",

            "scientific_fit_number":
                4,

            "scientific_fit_budget":
                4,

            "fit_attempt_started":
                True,

            "fit_completed":
                True,

            "fit_started_at_utc":
                fit_started_at,

            "fit_seconds":
                fit_seconds,

            "model_path":
                str(
                    MODEL_PATH
                ),

            "model_sha256":
                model_sha,

            "parent_commit":
                EXPECTED_PARENT,

            "target_openings_consumed":
                4,

            "IDS2018_target_features_read":
                False,

            "IDS2018_target_predictions":
                0,
        },
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


print(
    "[PASS] Scientific fit #4 completed on CPU."
)

print(
    "[PASS] 70 features / 400 boosted rounds."
)

print(
    "Model SHA:",
    model_sha,
)

print()


# Training matrix no longer needed.
del X_train70

gc.collect()


# ==============================================================================
# 17. THURSDAY SOURCE-VALIDATION INFERENCE
# ==============================================================================

print("=" * 118)
print("SOURCE VALIDATION — THURSDAY")
print("=" * 118)


val_start = time.time()


p_val70 = np.asarray(
    model70.predict_proba(
        X_val70
    )[
        :,
        1
    ],
    dtype=np.float32,
)


val_seconds = (
    time.time()
    -
    val_start
)


if p_val70.shape != (
    EXPECTED_VAL_ROWS,
):

    raise RuntimeError(
        "Validation probability shape mismatch."
    )


if not np.isfinite(
    p_val70
).all():

    raise RuntimeError(
        "Non-finite validation probability."
    )


if (
    (p_val70 < 0).any()
    or
    (p_val70 > 1).any()
):

    raise RuntimeError(
        "Validation probabilities outside [0,1]."
    )


val_probability_sha = sha256_array(
    p_val70
)


print(
    "Inference seconds:",
    f"{val_seconds:.3f}",
)

print(
    "Probability min/max:",
    f"{float(p_val70.min()):.9f}",
    "/",
    f"{float(p_val70.max()):.9f}",
)

print(
    "Probability SHA:",
    val_probability_sha,
)

print()


# ==============================================================================
# 18. SOURCE-VALIDATION GLOBAL METRICS
# ==============================================================================

pr_auc = float(
    average_precision_score(
        y_val,
        p_val70,
    )
)


roc_auc = float(
    roc_auc_score(
        y_val,
        p_val70,
    )
)


pr_excess = float(
    pr_auc
    -
    val_prevalence
)


pr_normalized = float(
    (
        pr_auc
        -
        val_prevalence
    )
    /
    (
        1.0
        -
        val_prevalence
    )
)


p64 = p_val70.astype(
    np.float64,
    copy=False,
)


y64 = y_val.astype(
    np.float64,
    copy=False,
)


brier = float(
    np.mean(
        (
            p64
            -
            y64
        ) ** 2
    )
)


clipped = np.clip(
    p64,
    1e-15,
    1.0 - 1e-15,
)


log_loss = float(
    -np.mean(
        y64
        *
        np.log(
            clipped
        )
        +
        (
            1.0
            -
            y64
        )
        *
        np.log(
            1.0
            -
            clipped
        )
    )
)


bins = np.floor(
    p64
    *
    10.0
).astype(
    np.int8
)


bins = np.clip(
    bins,
    0,
    9,
)


ece = 0.0


for b in range(
    10
):

    mask = (
        bins
        ==
        b
    )


    count = int(
        mask.sum()
    )


    if count == 0:

        continue


    confidence = float(
        p64[
            mask
        ].mean()
    )


    empirical = float(
        y64[
            mask
        ].mean()
    )


    ece += (
        count
        /
        EXPECTED_VAL_ROWS
    ) * abs(
        confidence
        -
        empirical
    )


ece = float(
    ece
)


print("=" * 118)
print("SOURCE-VALIDATION METRICS")
print("=" * 118)


print(
    "Prevalence:    ",
    f"{val_prevalence:.15f}",
)

print(
    "PR-AUC:        ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:       ",
    f"{roc_auc:.15f}",
)

print(
    "PR excess:     ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized: ",
    f"{pr_normalized:.15f}",
)

print()

print(
    "Brier:         ",
    f"{brier:.15f}",
)

print(
    "Log loss:      ",
    f"{log_loss:.15f}",
)

print(
    "ECE-10:        ",
    f"{ece:.15f}",
)

print()


# ==============================================================================
# 19. FROZEN THRESHOLD GRID
# ==============================================================================

def threshold_counts(
    pct,
):

    threshold = np.float32(
        pct
        /
        100.0
    )


    pred = (
        p_val70
        >=
        threshold
    )


    y1 = (
        y_val
        ==
        1
    )


    y0 = ~y1


    tp = int(
        np.sum(
            pred
            &
            y1
        )
    )


    fp = int(
        np.sum(
            pred
            &
            y0
        )
    )


    fn = (
        EXPECTED_VAL_ATTACK
        -
        tp
    )


    tn = (
        EXPECTED_VAL_BENIGN
        -
        fp
    )


    return {
        "threshold_integer_percent":
            int(
                pct
            ),

        "threshold_float32_runtime":
            float(
                threshold
            ),

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn,
    }


grid = [
    threshold_counts(
        pct
    )
    for pct in range(
        5,
        96,
    )
]


# ==============================================================================
# 20. EXACT RATIONAL OBJECTIVES
# ==============================================================================

def f1_fraction(
    row,
):

    return (
        2
        *
        row[
            "tp"
        ],
        2
        *
        row[
            "tp"
        ]
        +
        row[
            "fp"
        ]
        +
        row[
            "fn"
        ],
    )


def f2_fraction(
    row,
):

    return (
        5
        *
        row[
            "tp"
        ],
        5
        *
        row[
            "tp"
        ]
        +
        4
        *
        row[
            "fn"
        ]
        +
        row[
            "fp"
        ],
    )


def rational_cmp(
    n1,
    d1,
    n2,
    d2,
):

    left = (
        n1
        *
        d2
    )


    right = (
        n2
        *
        d1
    )


    if left > right:
        return 1


    if left < right:
        return -1


    return 0


# ==============================================================================
# 21. BALANCED THRESHOLD
# ==============================================================================

balanced = None


for row in grid:

    if balanced is None:

        balanced = row
        continue


    n1, d1 = f1_fraction(
        row
    )


    n2, d2 = f1_fraction(
        balanced
    )


    cmp = rational_cmp(
        n1,
        d1,
        n2,
        d2,
    )


    if cmp > 0:

        balanced = row
        continue


    if cmp < 0:

        continue


    # Lower FPR.
    if row[
        "fp"
    ] < balanced[
        "fp"
    ]:

        balanced = row
        continue


    if row[
        "fp"
    ] > balanced[
        "fp"
    ]:

        continue


    # Higher recall.
    if row[
        "tp"
    ] > balanced[
        "tp"
    ]:

        balanced = row
        continue


    if row[
        "tp"
    ] < balanced[
        "tp"
    ]:

        continue


    # Closer to 0.50.
    new_distance = abs(
        row[
            "threshold_integer_percent"
        ]
        -
        50
    )


    old_distance = abs(
        balanced[
            "threshold_integer_percent"
        ]
        -
        50
    )


    if new_distance < old_distance:

        balanced = row
        continue


    if new_distance > old_distance:

        continue


    # Lower threshold.
    if (
        row[
            "threshold_integer_percent"
        ]
        <
        balanced[
            "threshold_integer_percent"
        ]
    ):

        balanced = row


# ==============================================================================
# 22. SECURITY THRESHOLD
# ==============================================================================

security_candidates = [
    row
    for row in grid
    if (
        20
        *
        row[
            "fp"
        ]
    )
    <=
    EXPECTED_VAL_BENIGN
]


security = None


for row in security_candidates:

    if security is None:

        security = row
        continue


    n1, d1 = f2_fraction(
        row
    )


    n2, d2 = f2_fraction(
        security
    )


    cmp = rational_cmp(
        n1,
        d1,
        n2,
        d2,
    )


    if cmp > 0:

        security = row
        continue


    if cmp < 0:

        continue


    if row[
        "fp"
    ] < security[
        "fp"
    ]:

        security = row
        continue


    if row[
        "fp"
    ] > security[
        "fp"
    ]:

        continue


    if row[
        "tp"
    ] > security[
        "tp"
    ]:

        security = row
        continue


    if row[
        "tp"
    ] < security[
        "tp"
    ]:

        continue


    if (
        row[
            "threshold_integer_percent"
        ]
        <
        security[
            "threshold_integer_percent"
        ]
    ):

        security = row


# ==============================================================================
# 23. EXPAND THRESHOLD METRICS
# ==============================================================================

def expand(
    row,
):

    tp = row[
        "tp"
    ]

    tn = row[
        "tn"
    ]

    fp = row[
        "fp"
    ]

    fn = row[
        "fn"
    ]


    precision = (
        tp
        /
        (
            tp + fp
        )
        if (
            tp + fp
        )
        else 0.0
    )


    recall = (
        tp
        /
        (
            tp + fn
        )
        if (
            tp + fn
        )
        else 0.0
    )


    fpr = (
        fp
        /
        (
            fp + tn
        )
        if (
            fp + tn
        )
        else 0.0
    )


    fnr = (
        fn
        /
        (
            fn + tp
        )
        if (
            fn + tp
        )
        else 0.0
    )


    f1n, f1d = f1_fraction(
        row
    )


    f2n, f2d = f2_fraction(
        row
    )


    return {
        **row,

        "accuracy":
            float(
                (
                    tp + tn
                )
                /
                EXPECTED_VAL_ROWS
            ),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                f1n
                /
                f1d
                if f1d
                else 0.0
            ),

        "f2":
            float(
                f2n
                /
                f2d
                if f2d
                else 0.0
            ),

        "fpr":
            float(
                fpr
            ),

        "fnr":
            float(
                fnr
            ),
    }


standard = expand(
    grid[
        45
    ]
)


balanced_metrics = expand(
    balanced
)


security_metrics = (
    expand(
        security
    )
    if security is not None
    else None
)


print("=" * 118)
print("SOURCE-VALIDATION OPERATING POINTS")
print("=" * 118)


for name, metrics in [
    (
        "STANDARD",
        standard,
    ),
    (
        "BALANCED",
        balanced_metrics,
    ),
]:

    print(
        name,
        "@",
        metrics[
            "threshold_float32_runtime"
        ],
    )

    print(
        "  TP/TN/FP/FN:",
        metrics[
            "tp"
        ],
        metrics[
            "tn"
        ],
        metrics[
            "fp"
        ],
        metrics[
            "fn"
        ],
    )

    print(
        "  F1/F2/FPR/Recall:",
        f"{metrics['f1']:.9f}",
        f"{metrics['f2']:.9f}",
        f"{metrics['fpr']:.9f}",
        f"{metrics['recall']:.9f}",
    )

    print()


print(
    "SECURITY:"
)


if security_metrics is None:

    print(
        "  UNAVAILABLE_NO_RELAXATION"
    )

else:

    print(
        "  threshold:",
        security_metrics[
            "threshold_float32_runtime"
        ],
    )

    print(
        "  TP/TN/FP/FN:",
        security_metrics[
            "tp"
        ],
        security_metrics[
            "tn"
        ],
        security_metrics[
            "fp"
        ],
        security_metrics[
            "fn"
        ],
    )

    print(
        "  F2/FPR/Recall:",
        f"{security_metrics['f2']:.9f}",
        f"{security_metrics['fpr']:.9f}",
        f"{security_metrics['recall']:.9f}",
    )


print()

print(
    "[PASS] Threshold selection used Thursday SOURCE validation only."
)

print()


# ==============================================================================
# 24. PERSIST VALIDATION ARTIFACTS
# ==============================================================================

np.savez_compressed(
    VAL_PRED_PATH,

    probability=
        p_val70,

    binary_label=
        y_val,
)


val_pred_file_sha = sha256_file(
    VAL_PRED_PATH
)


with THRESHOLD_GRID_PATH.open(
    "w",
    newline="",
    encoding="utf-8",
) as fh:

    writer = csv.DictWriter(
        fh,
        fieldnames=[
            "threshold_integer_percent",
            "threshold_float32_runtime",
            "tp",
            "tn",
            "fp",
            "fn",
        ],
    )


    writer.writeheader()


    writer.writerows(
        grid
    )


threshold_grid_sha = sha256_file(
    THRESHOLD_GRID_PATH
)


print("=" * 118)
print("PERSISTED ARTIFACT IDENTITIES")
print("=" * 118)


print(
    "Model SHA:             ",
    model_sha,
)

print(
    "Validation array SHA:  ",
    val_probability_sha,
)

print(
    "Validation NPZ SHA:    ",
    val_pred_file_sha,
)

print(
    "Threshold grid SHA:    ",
    threshold_grid_sha,
)

print()


# ==============================================================================
# 25. RESULT RECEIPT
# ==============================================================================

result = {
    "stage":
        "Stage24-4B",

    "status":
        "SECONDARY_BRIDGE70_SOURCE_MODEL_FROZEN",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "direction":
        "CICIDS2017_TO_IDS2018",

    "bridge":
        "bridge70",

    "learner":
        "XGBOOST_ONLY",

    "learner_configuration":
        "INHERIT_STAGE22R_XGB_11",

    "parent_commit":
        EXPECTED_PARENT,

    "backend_amendment": {
        "artifact":
            str(
                AMENDMENT_PATH.relative_to(
                    REPO
                )
            ),

        "sha256":
            amendment_sha,

        "original_device":
            "cuda",

        "execution_device":
            "cpu",

        "scope":
            "EXECUTION_BACKEND_ONLY",

        "additional_bridge70_amendment":
            False,
    },

    "scientific_fit": {
        "fit_number":
            4,

        "fit_budget":
            4,

        "completed_before":
            EXPECTED_FITS_BEFORE,

        "completed_after":
            EXPECTED_FITS_AFTER,

        "fit_started_at_utc":
            fit_started_at,

        "fit_seconds":
            fit_seconds,

        "hyperparameter_search":
            False,
    },

    "xgboost": {
        "library_version":
            xgb.__version__,

        "parameters":
            XGB_PARAMS,

        "resolved_device":
            "cpu",

        "feature_count":
            70,

        "boosted_rounds":
            400,

        "model_file":
            str(
                MODEL_PATH.relative_to(
                    REPO
                )
            ),

        "model_sha256":
            model_sha,
    },

    "representation": {
        "feature_count":
            70,

        "feature_order":
            FEATURE_ORDER,

        "cicids2017_source_semantics":
            "FLAG_CORRECTED",

        "cicids2017_source_mapping":
            SOURCE_MAPPING,

        "published_mapping":
            PUBLISHED_MAPPING,

        "published_vs_corrected_mapping_difference_count":
            7,

        "ack_mapping_invariant":
            True,

        "parse_dtype":
            "float64",

        "positive_infinity":
            "CONVERT_TO_NAN",

        "negative_infinity":
            "CONVERT_TO_NAN",

        "explicit_imputation":
            "NONE",

        "scaling":
            "NONE",
    },

    "source_train": {
        "dataset":
            "CICIDS2017",

        "days":
            [
                "Monday",
                "Tuesday",
                "Wednesday",
            ],

        "rows":
            EXPECTED_TRAIN_ROWS,

        "benign":
            train_benign,

        "attack":
            train_attack,

        "attack_prevalence":
            source_prior,

        "feature_matrix_sha256":
            train_feature_sha,

        "binary_label_sha256":
            train_label_sha,
    },

    "source_validation": {
        "dataset":
            "CICIDS2017",

        "days":
            [
                "Thursday",
            ],

        "rows":
            EXPECTED_VAL_ROWS,

        "benign":
            val_benign,

        "attack":
            val_attack,

        "prevalence":
            val_prevalence,

        "feature_matrix_sha256":
            val_feature_sha,

        "binary_label_sha256":
            val_label_sha,

        "probability_array_sha256":
            val_probability_sha,

        "probability_artifact":
            str(
                VAL_PRED_PATH.relative_to(
                    REPO
                )
            ),

        "probability_artifact_sha256":
            val_pred_file_sha,

        "metrics": {
            "pr_auc":
                pr_auc,

            "roc_auc":
                roc_auc,

            "pr_excess":
                pr_excess,

            "pr_normalized":
                pr_normalized,

            "brier":
                brier,

            "log_loss":
                log_loss,

            "ece_10":
                ece,
        },

        "operating_points": {
            "standard":
                standard,

            "balanced":
                balanced_metrics,

            "security": {
                "status":
                    (
                        "AVAILABLE"
                        if security_metrics is not None
                        else "UNAVAILABLE_NO_RELAXATION"
                    ),

                "result":
                    security_metrics,
            },
        },

        "threshold_grid_file":
            str(
                THRESHOLD_GRID_PATH.relative_to(
                    REPO
                )
            ),

        "threshold_grid_sha256":
            threshold_grid_sha,

        "threshold_selection_data":
            "SOURCE_VALIDATION_ONLY",
    },

    "secondary_source_prior_constant_predictor": {
        "fit":
            False,

        "probability":
            source_prior,

        "computed_on":
            "CICIDS2017_MONDAY_WEDNESDAY_SOURCE_TRAIN_ONLY",
    },

    "unused_source": {
        "days":
            [
                "Friday",
            ],

        "read_for_model_training":
            False,

        "read_for_validation":
            False,
    },

    "target_access": {
        "dataset":
            "IDS2018",

        "target":
            "02-28-2018",

        "target_features_read":
            False,

        "target_labels_read":
            False,

        "target_model_predictions":
            0,

        "target_metrics":
            0,

        "new_target_openings":
            0,

        "total_target_openings_consumed":
            4,
    },

    "anti_adaptation": {
        "hyperparameter_search":
            False,

        "target_guided_refit":
            False,

        "target_guided_threshold":
            False,

        "target_feature_search":
            False,

        "target_preprocessing_fit":
            False,

        "performance_based_change":
            False,

        "backend_change_only":
            True,
    },

    "fit_accounting": {
        "scientific_fit_budget":
            4,

        "scientific_fits_completed":
            4,

        "scientific_fits_remaining":
            0,

        "additional_fits_authorized":
            0,
    },

    "next_authorized_step":
        (
            "OPEN_SECONDARY_IDS2018_FEB28_TARGET_CELLS_"
            "BRIDGE62_THEN_BRIDGE70"
        ),
}


RESULT_PATH.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


result_sha = sha256_file(
    RESULT_PATH
)


RESULT_SHA_PATH.write_text(
    f"{result_sha}  {RESULT_PATH.name}\n",
    encoding="utf-8",
)


CHECKSUMS_PATH.write_text(
    (
        f"{result_sha}  {RESULT_PATH.name}\n"
        f"{model_sha}  {MODEL_PATH.name}\n"
        f"{val_pred_file_sha}  {VAL_PRED_PATH.name}\n"
        f"{threshold_grid_sha}  {THRESHOLD_GRID_PATH.name}\n"
    ),
    encoding="utf-8",
)


print(
    "Result SHA:",
    result_sha,
)

print()


# ==============================================================================
# 26. GIT AUTHOR SAFETY
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    git_cmd(
        "config",
        "--local",
        "user.name",
        git_cmd(
            "log",
            "-1",
            "--format=%an",
        ),
    )


if not author_email:

    git_cmd(
        "config",
        "--local",
        "user.email",
        git_cmd(
            "log",
            "-1",
            "--format=%ae",
        ),
    )


# ==============================================================================
# 27. GIT FREEZE
# ==============================================================================

print("=" * 118)
print("GIT FREEZE")
print("=" * 118)


git_cmd(
    "add",
    "--",
    str(
        MODEL_PATH.relative_to(
            REPO
        )
    ),
    str(
        VAL_PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        THRESHOLD_GRID_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


expected_staged = {
    str(
        MODEL_PATH.relative_to(
            REPO
        )
    ),
    str(
        VAL_PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        THRESHOLD_GRID_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: freeze secondary bridge70 cpu source model",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage24-4B commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 28. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 118)
print("PUSH + REMOTE VERIFY")
print("=" * 118)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository not clean after Stage24-4B:\n"
        + final_status
    )


if FIT_LEDGER.exists():

    FIT_LEDGER.unlink()


print(
    "[PASS] Remote main == local Stage24-4B commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 29. FINAL
# ==============================================================================

print("=" * 118)
print("STAGE24-4B SECONDARY BRIDGE70 SOURCE FIT: PASS")
print("=" * 118)

print()

print(
    "Direction:                    CICIDS2017 -> IDS2018"
)

print(
    "Bridge:                       bridge70"
)

print(
    "Source semantics:             FLAG_CORRECTED"
)

print(
    "Learner:                      XGBoost"
)

print(
    "Execution backend:            CPU"
)

print()

print(
    "Train rows:                   ",
    f"{EXPECTED_TRAIN_ROWS:,}",
)

print(
    "Train benign:                 ",
    f"{train_benign:,}",
)

print(
    "Train attack:                 ",
    f"{train_attack:,}",
)

print(
    "Source prior:                 ",
    f"{source_prior:.12f}",
)

print()

print(
    "Validation rows:              ",
    f"{EXPECTED_VAL_ROWS:,}",
)

print(
    "Validation benign:            ",
    f"{val_benign:,}",
)

print(
    "Validation attack:            ",
    f"{val_attack:,}",
)

print()

print(
    "Validation PR-AUC:            ",
    f"{pr_auc:.15f}",
)

print(
    "Validation ROC-AUC:           ",
    f"{roc_auc:.15f}",
)

print(
    "Validation Brier:             ",
    f"{brier:.15f}",
)

print()

print(
    "Balanced threshold:           ",
    balanced_metrics[
        "threshold_float32_runtime"
    ],
)

print(
    "Security threshold:           ",
    (
        security_metrics[
            "threshold_float32_runtime"
        ]
        if security_metrics is not None
        else "UNAVAILABLE"
    ),
)

print()

print(
    "Model SHA:"
)

print(
    " ",
    model_sha,
)

print()

print(
    "Validation probability SHA:"
)

print(
    " ",
    val_probability_sha,
)

print()

print(
    "Scientific fits completed:    4 / 4"
)

print(
    "Scientific fits remaining:    0"
)

print(
    "Additional fits authorized:   0"
)

print()

print(
    "New target openings:          0"
)

print(
    "Target openings consumed:     4 / 8"
)

print(
    "IDS2018 Feb-28 target opened: NO"
)

print()

print(
    "Result SHA:"
)

print(
    " ",
    result_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "ALL STAGE24 SCIENTIFIC FITS COMPLETE."
)

print()

print(
    "NEXT:"
)

print(
    "  SECONDARY TARGET OPENING #5:"
)

print(
    "  bridge62 / IDS2018 Feb-28"
)

print("=" * 118)

STAGE24-4B — SECONDARY BRIDGE70 XGBOOST CPU SOURCE FIT

GOVERNANCE
HEAD: e58019523237d8055c044f65f76d28ccf568e20c
[PASS] Repository clean.
[PASS] No unresolved scientific-fit ledger.
[PASS] Stage24-4B has not previously been fit.

SCIENTIFIC FIT #3 REFERENCE
Stage24-4A result SHA: fc496c6342269d089eeaa4c1ec043741644ce5fe379f92e267e2d6689787335c
[PASS] Scientific fit #3 durable.
[PASS] Target openings remain 4 / 8.

CPU BACKEND AMENDMENT
Amendment SHA: 2c356c8763c567702d14a74c042df42356f360e559e574c3d9a4abfa652e5c14
[PASS] Existing CPU amendment explicitly covers bridge70.

[PASS] Frozen secondary contract exact.

BRIDGE70 SOURCE REPRESENTATION
Feature count: 70
Corrected physical mapping changes: 7
ACK mapping invariant: True

[PASS] Frozen bridge70 FLAG_CORRECTED source semantics exact.

CANONICAL LABEL RECOVERY
[PASS] Reusing exact canonical source labels from fit #3 runtime.

RUNTIME MEMORY HOUSEKEEPING
Available RAM: 26.38 GiB
[PASS] Stale fit #3 feature/model objects released wher

In [41]:
# ==============================================================================
# STAGE24-5A — SECONDARY TARGET OPENING #5
# CICIDS2017 -> IDS2018
# bridge62 / IDS2018 Feb-28
#
# Scientific fits:
#   COMPLETE 4 / 4
#   NO MORE FITS AUTHORIZED
#
# Target opening:
#   #5 / 8
#
# Frozen secondary model:
#   Stage24-4A-R1 bridge62 XGBoost
#   SHA256:
#     8040bc7909429736c95ca396ebebf18636ce5d1f5c2232d12042c6ffadd47c5d
#
# Frozen source-validation operating points:
#   STANDARD = 0.50
#   BALANCED = 0.17
#   SECURITY = 0.05
#
# Frozen IDS2018 Feb-28 target:
#   retained rows = 593,780
#   benign        = 531,524
#   attack        = 62,256
#
# Preferred target source:
#   frozen Stage22R day-07 70F cache
#
#   day_07_02-28-2018.parquet
#   SHA256:
#     3ab933a7988af562f49f63b8975882b7c66e843e9c4b967a38d5d875d062f62b
#
# Fallback:
#   byte-exact raw 02-28-2018.csv
#   +
#   frozen exact K79 exclusion membership
#
# NO FIT.
# NO TARGET THRESHOLD SEARCH.
# NO TARGET CALIBRATION.
# NO TARGET-FITTED PREPROCESSING.
#
# Once:
#
#     >>> TARGET OPENING #5 CONSUMED <<<
#
# appears, DO NOT RERUN BLINDLY after a later failure.
# ==============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


print("=" * 118)
print("STAGE24-5A — SECONDARY TARGET OPENING #5 — bridge62 / IDS2018 Feb-28")
print("=" * 118)
print()


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "1390cf5c3f2a5c6ec6c20cf486ca6ee6c84d4769"
)

EXPECTED_TARGET_ROWS = 593_780
EXPECTED_TARGET_BENIGN = 531_524
EXPECTED_TARGET_ATTACK = 62_256

EXPECTED_PHYSICAL_ROWS = 613_104
EXPECTED_EMBEDDED_HEADERS = 33
EXPECTED_EFFECTIVE_PRE_K79 = 613_071

EXPECTED_K79_DAY7_EXCLUSIONS = 19_291
EXPECTED_K79_DAY7_EXCLUDED_BENIGN = 12_676
EXPECTED_K79_DAY7_EXCLUDED_ATTACK = 6_615

EXPECTED_RAW_SHA = (
    "f15e2a12304446058a0186c8ad67de2bd15735a9ba5c70c9a1f4c4242ab06771"
)

EXPECTED_K79_SHA = (
    "3e979841468f9d42a8de7049ed50063d5907c0753fe5b1179dac948163d93134"
)

EXPECTED_DAY7_CACHE_SHA = (
    "3ab933a7988af562f49f63b8975882b7c66e843e9c4b967a38d5d875d062f62b"
)

EXPECTED_STAGE22_1C_MANIFEST_SHA = (
    "05fe6226afd37c3ab434140e1164643eecc7abc75c295a2d6fb0280d4f42127e"
)

EXPECTED_STAGE22_VALIDATION_NPZ_SHA = (
    "3fe6c468a0653ac5ee488da8d6586fd86628e9586eb01f6f5d962e35fff65e3f"
)

EXPECTED_BRIDGE62_MODEL_SHA = (
    "8040bc7909429736c95ca396ebebf18636ce5d1f5c2232d12042c6ffadd47c5d"
)

EXPECTED_BRIDGE62_RESULT_SHA = (
    "fc496c6342269d089eeaa4c1ec043741644ce5fe379f92e267e2d6689787335c"
)

EXPECTED_BRIDGE_SPEC_SHA = (
    "b7ff1d8563aafc69145be3acc827fbc59ae75d880b9804bad727f408299fce2f"
)

EXPECTED_PROTOCOL_LOCK_SHA = (
    "8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2"
)

EXPECTED_70F_POS_INF = 6_085
EXPECTED_70F_NEG_INF = 0
EXPECTED_70F_OUTPUT_NAN = 7_936

EXPECTED_CLEAN_POSITION_START = 13_818_623
EXPECTED_CLEAN_POSITION_STOP = 14_412_403

EXPECTED_FITS = 4
OPENINGS_BEFORE = 4
OPENINGS_AFTER = 5


# ==============================================================================
# 1. PATHS
# ==============================================================================

LOCK_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
)

BRIDGE_SPEC_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.json"
)

BRIDGE_SPEC_SHA_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.sha256"
)

FINAL_LOCK_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.json"
)

FINAL_LOCK_SHA_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.sha256"
)


B62_SOURCE_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_4_secondary_source_training"
    / "stage24_4a_bridge62_xgboost"
)

B62_MODEL_PATH = (
    B62_SOURCE_DIR
    / "secondary_bridge62_xgboost_model.json"
)

B62_RESULT_PATH = (
    B62_SOURCE_DIR
    / "stage24_4a_secondary_bridge62_result.json"
)

B62_RESULT_SHA_PATH = (
    B62_SOURCE_DIR
    / "stage24_4a_secondary_bridge62_result.sha256"
)


B70_SOURCE_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_4_secondary_source_training"
    / "stage24_4b_bridge70_xgboost"
)

B70_RESULT_PATH = (
    B70_SOURCE_DIR
    / "stage24_4b_secondary_bridge70_result.json"
)

B70_RESULT_SHA_PATH = (
    B70_SOURCE_DIR
    / "stage24_4b_secondary_bridge70_result.sha256"
)


K79_DIR = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1a3_k79_development_freeze"
)

K79_PATH = (
    K79_DIR
    / "stage22r_k79_development_exclusions.parquet"
)


STAGE22_1C_DIR = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
)

STAGE22_1C_MANIFEST_PATH = (
    STAGE22_1C_DIR
    / "stage22r_1c_development_model_input_manifest.json"
)


STAGE22_TRAIN_DIR = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
)

STAGE22_VALIDATION_NPZ_PATH = (
    STAGE22_TRAIN_DIR
    / "chronological_natural_validation_ensemble_probabilities.npz"
)


PRIMARY_1A_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_1_primary_source_sanity"
    / "stage24_1a_full_source_70f_sanity.json"
)


PREFERRED_DAY7_CACHE = Path(
    "/kaggle/working/"
    "stage22r_1c_70f_development_cache/"
    "day_07_02-28-2018.parquet"
)


OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_5_secondary_target_openings"
    / "stage24_5a_bridge62_ids2018_feb28"
)

PRED_PATH = (
    OUT
    / "secondary_bridge62_ids2018_feb28_predictions.npz"
)

RESULT_PATH = (
    OUT
    / "stage24_5a_secondary_bridge62_ids2018_feb28_result.json"
)

RESULT_SHA_PATH = (
    OUT
    / "stage24_5a_secondary_bridge62_ids2018_feb28_result.sha256"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


OPENING_LEDGER = Path(
    "/kaggle/working/stage24_target_opening_runtime_ledger.json"
)

FIT_LEDGER = Path(
    "/kaggle/working/stage24_secondary_fit_runtime_ledger.json"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(args, *, cwd=None, check=True):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(str(x) for x in args)
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(*args, auth_header=None, check=True):

    cmd = ["git"]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [str(x) for x in args]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array(array):

    arr = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        arr.view(np.uint8)
    ).hexdigest()


def verify_sidecar(artifact, sidecar):

    if not artifact.is_file():

        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )

    if not sidecar.is_file():

        raise RuntimeError(
            f"Missing SHA sidecar:\n{sidecar}"
        )

    actual = sha256_file(
        artifact
    )

    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )

    if actual != expected:

        raise RuntimeError(
            "\nSHA mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )

    return actual


DASH_TRANSLATION = str.maketrans(
    {
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2015": "-",
        "\u2212": "-",
    }
)


def canonicalize_labels(series):

    if series.isna().any():

        raise RuntimeError(
            "NULL target label encountered."
        )

    s = (
        series
        .astype("string")
        .str.strip()
        .str.translate(DASH_TRANSLATION)
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.casefold()
    )

    if s.isna().any():

        raise RuntimeError(
            "Label canonicalization produced NULL."
        )

    if (s.str.len() == 0).any():

        raise RuntimeError(
            "Empty target label encountered."
        )

    return s


# ==============================================================================
# 3. GOVERNANCE
# ==============================================================================

print("=" * 118)
print("GOVERNANCE")
print("=" * 118)


head = git_cmd(
    "rev-parse",
    "HEAD",
)

status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


if FIT_LEDGER.exists():

    raise RuntimeError(
        "\nUnexpected scientific-fit ledger exists:\n"
        f"{FIT_LEDGER}"
    )


if OPENING_LEDGER.exists():

    raise RuntimeError(
        "\nA target-opening runtime ledger already exists:\n"
        f"{OPENING_LEDGER}\n\n"
        "DO NOT rerun a target opening blindly."
    )


if OUT.exists() and any(
    OUT.iterdir()
):

    raise RuntimeError(
        "\nStage24-5A artifacts already exist.\n"
        "Refusing duplicate target opening."
    )


print(
    "[PASS] Repository clean."
)

print(
    "[PASS] No unresolved scientific-fit ledger."
)

print(
    "[PASS] No unresolved target-opening ledger."
)

print(
    "[PASS] Stage24-5A has not previously been opened."
)

print()


# ==============================================================================
# 4. VERIFY ALL SCIENTIFIC FITS ARE COMPLETE
# ==============================================================================

print("=" * 118)
print("SCIENTIFIC FIT ACCOUNTING")
print("=" * 118)


b62_result_sha = verify_sidecar(
    B62_RESULT_PATH,
    B62_RESULT_SHA_PATH,
)

b70_result_sha = verify_sidecar(
    B70_RESULT_PATH,
    B70_RESULT_SHA_PATH,
)


if b62_result_sha != EXPECTED_BRIDGE62_RESULT_SHA:

    raise RuntimeError(
        "Bridge62 source-fit receipt SHA changed."
    )


b62_result = json.loads(
    B62_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)

b70_result = json.loads(
    B70_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


if int(
    b62_result[
        "fit_accounting"
    ][
        "scientific_fits_completed"
    ]
) != 3:

    raise RuntimeError(
        "Bridge62 fit accounting changed."
    )


if int(
    b70_result[
        "fit_accounting"
    ][
        "scientific_fits_completed"
    ]
) != EXPECTED_FITS:

    raise RuntimeError(
        "Stage24 scientific fits are not 4/4."
    )


if int(
    b70_result[
        "fit_accounting"
    ][
        "scientific_fits_remaining"
    ]
) != 0:

    raise RuntimeError(
        "Unexpected scientific fits remain."
    )


if int(
    b70_result[
        "fit_accounting"
    ][
        "additional_fits_authorized"
    ]
) != 0:

    raise RuntimeError(
        "Unexpected additional fits authorized."
    )


print(
    "Bridge62 result SHA:",
    b62_result_sha,
)

print(
    "Bridge70 result SHA:",
    b70_result_sha,
)

print()

print(
    "[PASS] Scientific fits completed: 4 / 4."
)

print(
    "[PASS] Scientific fits remaining: 0."
)

print(
    "[PASS] Additional fits authorized: 0."
)

print()


# ==============================================================================
# 5. VERIFY FROZEN PROTOCOL / BRIDGE
# ==============================================================================

print("=" * 118)
print("FROZEN PROTOCOL")
print("=" * 118)


bridge_sha = verify_sidecar(
    BRIDGE_SPEC_PATH,
    BRIDGE_SPEC_SHA_PATH,
)

lock_sha = verify_sidecar(
    FINAL_LOCK_PATH,
    FINAL_LOCK_SHA_PATH,
)


if bridge_sha != EXPECTED_BRIDGE_SPEC_SHA:

    raise RuntimeError(
        "Semantic bridge SHA changed."
    )


if lock_sha != EXPECTED_PROTOCOL_LOCK_SHA:

    raise RuntimeError(
        "Protocol lock SHA changed."
    )


bridge_spec = json.loads(
    BRIDGE_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)

final_lock = json.loads(
    FINAL_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


secondary = final_lock[
    "secondary_direction"
]


if secondary[
    "direction"
] != "CICIDS2017_TO_IDS2018":

    raise RuntimeError(
        "Secondary direction changed."
    )


if secondary[
    "target"
] != "IDS2018_02-28-2018":

    raise RuntimeError(
        "Secondary target changed."
    )


if int(
    secondary[
        "target_expected_clean_rows"
    ]
) != EXPECTED_TARGET_ROWS:

    raise RuntimeError(
        "Secondary target row count changed."
    )


FEATURE_ORDER_62 = list(
    bridge_spec[
        "bridge62"
    ][
        "source_feature_order"
    ]
)


FEATURE_ORDER_70 = list(
    bridge_spec[
        "bridge70"
    ][
        "source_feature_order"
    ]
)


if len(FEATURE_ORDER_62) != 62:

    raise RuntimeError(
        "bridge62 feature count changed."
    )


if len(FEATURE_ORDER_70) != 70:

    raise RuntimeError(
        "bridge70 feature count changed."
    )


FLAG_FEATURES = {
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "ACK Flag Cnt",
    "URG Flag Cnt",
    "CWE Flag Count",
    "ECE Flag Cnt",
}


if [
    f
    for f in FEATURE_ORDER_70
    if f not in FLAG_FEATURES
] != FEATURE_ORDER_62:

    raise RuntimeError(
        "bridge62 is no longer bridge70 minus exactly the 8 flag fields."
    )


print(
    "[PASS] Secondary target = IDS2018 Feb-28."
)

print(
    "[PASS] bridge62 feature order exact."
)

print()


# ==============================================================================
# 6. VERIFY BRIDGE62 MODEL + SOURCE THRESHOLDS
# ==============================================================================

print("=" * 118)
print("FROZEN BRIDGE62 SECONDARY MODEL")
print("=" * 118)


model_sha = sha256_file(
    B62_MODEL_PATH
)


print(
    "Model SHA:",
    model_sha,
)


if model_sha != EXPECTED_BRIDGE62_MODEL_SHA:

    raise RuntimeError(
        "Bridge62 secondary model SHA changed."
    )


source_prior = float(
    b62_result[
        "source_train"
    ][
        "attack_prevalence"
    ]
)


ops = (
    b62_result[
        "source_validation"
    ][
        "operating_points"
    ]
)


STANDARD_THRESHOLD = np.float32(
    ops[
        "standard"
    ][
        "threshold_float32_runtime"
    ]
)


BALANCED_THRESHOLD = np.float32(
    ops[
        "balanced"
    ][
        "threshold_float32_runtime"
    ]
)


security_record = ops[
    "security"
]


if security_record[
    "status"
] != "AVAILABLE":

    raise RuntimeError(
        "Frozen bridge62 SECURITY operating point unavailable."
    )


SECURITY_THRESHOLD = np.float32(
    security_record[
        "result"
    ][
        "threshold_float32_runtime"
    ]
)


if STANDARD_THRESHOLD != np.float32(0.50):

    raise RuntimeError(
        "Standard threshold changed."
    )


if BALANCED_THRESHOLD != np.float32(0.17):

    raise RuntimeError(
        "Balanced threshold changed."
    )


if SECURITY_THRESHOLD != np.float32(0.05):

    raise RuntimeError(
        "Security threshold changed."
    )


print(
    "Source prior:",
    f"{source_prior:.15f}",
)

print(
    "STANDARD:",
    float(STANDARD_THRESHOLD),
)

print(
    "BALANCED:",
    float(BALANCED_THRESHOLD),
)

print(
    "SECURITY:",
    float(SECURITY_THRESHOLD),
)

print()

print(
    "[PASS] All target operating points come from CICIDS2017 Thursday validation only."
)

print()


# ==============================================================================
# 7. VERIFY STAGE22R DAY-07 MANIFEST
# ==============================================================================

print("=" * 118)
print("FROZEN IDS2018 FEB-28 PROVENANCE")
print("=" * 118)


manifest_sha = sha256_file(
    STAGE22_1C_MANIFEST_PATH
)


if manifest_sha != EXPECTED_STAGE22_1C_MANIFEST_SHA:

    raise RuntimeError(
        "\nStage22R model-input manifest SHA mismatch.\n"
        f"Expected: {EXPECTED_STAGE22_1C_MANIFEST_SHA}\n"
        f"Actual:   {manifest_sha}"
    )


manifest = json.loads(
    STAGE22_1C_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


day7_entries = [
    item
    for item in manifest[
        "cache"
    ][
        "files"
    ]
    if int(
        item[
            "day_id"
        ]
    ) == 7
]


if len(day7_entries) != 1:

    raise RuntimeError(
        "Frozen Stage22R manifest does not contain exactly one day-07 entry."
    )


day7_manifest = day7_entries[0]


if day7_manifest[
    "cache_file"
] != "day_07_02-28-2018.parquet":

    raise RuntimeError(
        "Frozen day-07 cache filename changed."
    )


if day7_manifest[
    "sha256"
] != EXPECTED_DAY7_CACHE_SHA:

    raise RuntimeError(
        "Frozen day-07 cache SHA changed."
    )


if int(
    day7_manifest[
        "rows"
    ]
) != EXPECTED_TARGET_ROWS:

    raise RuntimeError(
        "Frozen day-07 row count changed."
    )


if int(
    day7_manifest[
        "benign"
    ]
) != EXPECTED_TARGET_BENIGN:

    raise RuntimeError(
        "Frozen day-07 benign count changed."
    )


if int(
    day7_manifest[
        "attack"
    ]
) != EXPECTED_TARGET_ATTACK:

    raise RuntimeError(
        "Frozen day-07 attack count changed."
    )


if int(
    day7_manifest[
        "clean_start"
    ]
) != EXPECTED_CLEAN_POSITION_START:

    raise RuntimeError(
        "Frozen day-07 clean_start changed."
    )


if int(
    day7_manifest[
        "clean_stop_exclusive"
    ]
) != EXPECTED_CLEAN_POSITION_STOP:

    raise RuntimeError(
        "Frozen day-07 clean_stop changed."
    )


print(
    "Stage22R-1C manifest SHA:",
    manifest_sha,
)

print(
    "Frozen day-07 cache SHA:",
    EXPECTED_DAY7_CACHE_SHA,
)

print(
    "Canonical row order:",
    manifest[
        "row_schema"
    ][
        "canonical_row_order"
    ],
)

print()

print(
    "[PASS] Frozen Feb-28 population provenance exact."
)

print()


# ==============================================================================
# 8. REMOTE GATE BEFORE TARGET READ
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if value and value.strip():

            github_token = value.strip()
            token_source = f"ENV:{name}"
            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable BEFORE target read."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved before target opening #5.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main == Stage24-4B."
)

print()


# ==============================================================================
# 9. RELEASE LARGE SECONDARY-SOURCE RUNTIME MATRICES
# ==============================================================================

print("=" * 118)
print("RUNTIME MEMORY HOUSEKEEPING")
print("=" * 118)


for stale_name in [
    "X_train70",
    "X_val70",
    "p_val70",
    "model70",
    "booster70",
    "X_train",
    "X_val",
    "p_val",
    "model",
    "booster",
]:

    globals().pop(
        stale_name,
        None,
    )


gc.collect()


try:

    import psutil

    print(
        "Available RAM:",
        f"{psutil.virtual_memory().available / (1024**3):.2f} GiB",
    )

except Exception:

    print(
        "Available RAM: <psutil unavailable>"
    )


print()


# ==============================================================================
# 10. LOCATE BYTE-EXACT FROZEN DAY-07 CACHE
# ==============================================================================

print("=" * 118)
print("TARGET MATERIALIZATION")
print("=" * 118)


cache_candidates = []


if PREFERRED_DAY7_CACHE.is_file():

    cache_candidates.append(
        PREFERRED_DAY7_CACHE
    )


for root in [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]:

    if not root.exists():
        continue

    try:

        for path in root.rglob(
            "day_07_02-28-2018.parquet"
        ):

            if path.is_file():

                cache_candidates.append(
                    path
                )

    except Exception:
        pass


dedup_cache = []

seen_cache = set()


for path in cache_candidates:

    try:
        key = str(
            path.resolve()
        )
    except Exception:
        key = str(path)

    if key not in seen_cache:

        seen_cache.add(key)
        dedup_cache.append(path)


exact_cache_path = None


for path in dedup_cache:

    digest = sha256_file(
        path
    )

    print(
        "Cache candidate:",
        path,
    )

    print(
        "  SHA:",
        digest,
    )


    if digest == EXPECTED_DAY7_CACHE_SHA:

        exact_cache_path = path
        break


# ==============================================================================
# 11A. PREFERRED PATH — EXACT FROZEN STAGE22R CACHE
# ==============================================================================

if exact_cache_path is not None:

    print()
    print(
        "[PASS] Exact frozen Stage22R day-07 cache recovered."
    )

    print(
        "Using:",
        exact_cache_path,
    )

    print()


    target_df = pd.read_parquet(
        exact_cache_path
    )


    required_metadata = [
        "clean_position",
        "day_id",
        "original_row_index",
        "binary_label",
    ]


    required_columns = (
        required_metadata
        +
        FEATURE_ORDER_70
    )


    missing = [
        col
        for col in required_columns
        if col not in target_df.columns
    ]


    if missing:

        raise RuntimeError(
            "\nFrozen day-07 cache missing required columns:\n"
            + "\n".join(missing)
        )


    if len(
        target_df
    ) != EXPECTED_TARGET_ROWS:

        raise RuntimeError(
            "Frozen day-07 cache row count mismatch."
        )


    if not np.all(
        target_df[
            "day_id"
        ].to_numpy()
        ==
        7
    ):

        raise RuntimeError(
            "Frozen day-07 cache contains non-day7 rows."
        )


    clean_position = target_df[
        "clean_position"
    ].to_numpy(
        dtype=np.int64,
        copy=True,
    )


    original_row_index = target_df[
        "original_row_index"
    ].to_numpy(
        dtype=np.int64,
        copy=True,
    )


    y_target = target_df[
        "binary_label"
    ].to_numpy(
        dtype=np.uint8,
        copy=True,
    )


    if not np.array_equal(
        clean_position,
        np.arange(
            EXPECTED_CLEAN_POSITION_START,
            EXPECTED_CLEAN_POSITION_STOP,
            dtype=np.int64,
        ),
    ):

        raise RuntimeError(
            "Frozen day-07 clean_position ordering mismatch."
        )


    if (
        len(original_row_index) > 1
        and
        not np.all(
            original_row_index[
                1:
            ]
            >
            original_row_index[
                :-1
            ]
        )
    ):

        raise RuntimeError(
            "Frozen original_row_index is not strictly ascending."
        )


    X70 = target_df[
        FEATURE_ORDER_70
    ].to_numpy(
        dtype=np.float64,
        copy=True,
    )


    if np.isinf(
        X70
    ).any():

        raise RuntimeError(
            "Frozen Stage22R cache contains infinity; expected already-converted NaN."
        )


    nan70 = int(
        np.isnan(
            X70
        ).sum()
    )


    if nan70 != EXPECTED_70F_OUTPUT_NAN:

        raise RuntimeError(
            "\nFrozen day-07 NaN count mismatch.\n"
            f"Expected: {EXPECTED_70F_OUTPUT_NAN:,}\n"
            f"Actual:   {nan70:,}"
        )


    target_materialization_mode = (
        "BYTE_EXACT_FROZEN_STAGE22R_DAY07_CACHE"
    )


    target_materialization_source = str(
        exact_cache_path
    )


    raw_source_path = None


    del target_df

    gc.collect()


# ==============================================================================
# 11B. FALLBACK — RAW CSV + FROZEN K79 LOCATORS
# ==============================================================================

else:

    print()
    print(
        "[INFO] Exact Stage22R day-07 cache not present."
    )

    print(
        "[INFO] Reconstructing from byte-exact raw CSV + frozen K79 membership."
    )

    print()


    k79_sha = sha256_file(
        K79_PATH
    )


    if k79_sha != EXPECTED_K79_SHA:

        raise RuntimeError(
            "\nK79 exclusion artifact SHA mismatch.\n"
            f"Expected: {EXPECTED_K79_SHA}\n"
            f"Actual:   {k79_sha}"
        )


    raw_candidates = []


    for root in [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]:

        if not root.exists():
            continue

        try:

            for path in root.rglob(
                "02-28-2018.csv"
            ):

                if path.is_file():

                    raw_candidates.append(
                        path
                    )

        except Exception:
            pass


    dedup_raw = []

    seen_raw = set()


    for path in raw_candidates:

        try:
            key = str(
                path.resolve()
            )
        except Exception:
            key = str(path)

        if key not in seen_raw:

            seen_raw.add(key)
            dedup_raw.append(path)


    exact_raw_path = None


    for path in dedup_raw:

        digest = sha256_file(
            path
        )

        print(
            "Raw candidate:",
            path,
        )

        print(
            "  SHA:",
            digest,
        )


        if digest == EXPECTED_RAW_SHA:

            exact_raw_path = path
            break


    if exact_raw_path is None:

        raise RuntimeError(
            "\nUnable to locate byte-exact raw 02-28-2018.csv.\n"
            "Target opening has NOT been consumed."
        )


    raw_source_path = str(
        exact_raw_path
    )


    print()
    print(
        "[PASS] Byte-exact raw Feb-28 source located."
    )

    print(
        "Using:",
        exact_raw_path,
    )

    print()


    raw = pd.read_csv(
        exact_raw_path,
        low_memory=False,
    )


    raw.columns = [
        str(col).strip()
        for col in raw.columns
    ]


    if len(
        set(
            raw.columns
        )
    ) != len(
        raw.columns
    ):

        raise RuntimeError(
            "Column-name stripping created duplicates."
        )


    if len(
        raw
    ) != EXPECTED_PHYSICAL_ROWS:

        raise RuntimeError(
            "\nRaw physical-row count mismatch.\n"
            f"Expected: {EXPECTED_PHYSICAL_ROWS:,}\n"
            f"Actual:   {len(raw):,}"
        )


    for col in (
        FEATURE_ORDER_70
        +
        [
            "Label"
        ]
    ):

        if col not in raw.columns:

            raise RuntimeError(
                f"Raw Feb-28 missing required column: {col}"
            )


    embedded_mask = (
        raw[
            "Label"
        ]
        .astype("string")
        .str.strip()
        .str.casefold()
        ==
        "label"
    ).to_numpy()


    embedded_count = int(
        embedded_mask.sum()
    )


    if embedded_count != EXPECTED_EMBEDDED_HEADERS:

        raise RuntimeError(
            "\nEmbedded-header count mismatch.\n"
            f"Expected: {EXPECTED_EMBEDDED_HEADERS}\n"
            f"Actual:   {embedded_count}"
        )


    effective = raw.loc[
        ~embedded_mask
    ].copy()


    if len(
        effective
    ) != EXPECTED_EFFECTIVE_PRE_K79:

        raise RuntimeError(
            "Effective pre-K79 row count mismatch."
        )


    effective_labels = canonicalize_labels(
        effective[
            "Label"
        ]
    )


    y_effective = (
        effective_labels
        !=
        "benign"
    ).to_numpy(
        dtype=np.uint8,
    )


    if int(
        (y_effective == 0).sum()
    ) != 544_200:

        raise RuntimeError(
            "Pre-K79 benign count mismatch."
        )


    if int(
        (y_effective == 1).sum()
    ) != 68_871:

        raise RuntimeError(
            "Pre-K79 attack count mismatch."
        )


    k79 = pd.read_parquet(
        K79_PATH
    )


    required_k79_cols = {
        "day_id",
        "row_index",
        "binary_label",
    }


    if not required_k79_cols.issubset(
        k79.columns
    ):

        raise RuntimeError(
            "K79 artifact missing required locator columns."
        )


    k79_day7 = (
        k79.loc[
            k79[
                "day_id"
            ].astype(
                np.int64
            )
            ==
            7
        ]
        .copy()
    )


    if len(
        k79_day7
    ) != EXPECTED_K79_DAY7_EXCLUSIONS:

        raise RuntimeError(
            "Day-7 K79 exclusion count mismatch."
        )


    if k79_day7[
        "row_index"
    ].nunique() != EXPECTED_K79_DAY7_EXCLUSIONS:

        raise RuntimeError(
            "Day-7 K79 row locators are not unique."
        )


    exclusion_indices = k79_day7[
        "row_index"
    ].to_numpy(
        dtype=np.int64,
    )


    if (
        exclusion_indices.min() < 0
        or
        exclusion_indices.max() >= EXPECTED_PHYSICAL_ROWS
    ):

        raise RuntimeError(
            "K79 original row locator outside physical Feb-28 range."
        )


    embedded_indices = set(
        np.flatnonzero(
            embedded_mask
        ).tolist()
    )


    if any(
        int(i) in embedded_indices
        for i in exclusion_indices
    ):

        raise RuntimeError(
            "K79 exclusion locator unexpectedly points to an embedded header."
        )


    exclusion_raw_labels = canonicalize_labels(
        raw.loc[
            exclusion_indices,
            "Label"
        ]
    )


    exclusion_binary_from_raw = (
        exclusion_raw_labels
        !=
        "benign"
    ).to_numpy(
        dtype=np.uint8,
    )


    exclusion_binary_frozen = k79_day7[
        "binary_label"
    ].to_numpy(
        dtype=np.uint8,
    )


    if not np.array_equal(
        exclusion_binary_from_raw,
        exclusion_binary_frozen,
    ):

        mismatch = int(
            np.count_nonzero(
                exclusion_binary_from_raw
                !=
                exclusion_binary_frozen
            )
        )

        raise RuntimeError(
            "\nK79 original_zero_based_row_index locator semantics failed.\n"
            f"Binary-label mismatches: {mismatch:,}\n"
            "Target opening has NOT been consumed."
        )


    if int(
        (exclusion_binary_frozen == 0).sum()
    ) != EXPECTED_K79_DAY7_EXCLUDED_BENIGN:

        raise RuntimeError(
            "K79 excluded benign count mismatch."
        )


    if int(
        (exclusion_binary_frozen == 1).sum()
    ) != EXPECTED_K79_DAY7_EXCLUDED_ATTACK:

        raise RuntimeError(
            "K79 excluded attack count mismatch."
        )


    retained = effective.drop(
        index=exclusion_indices,
    )


    if len(
        retained
    ) != EXPECTED_TARGET_ROWS:

        raise RuntimeError(
            "K79-clean Feb-28 retained-row count mismatch."
        )


    original_row_index = retained.index.to_numpy(
        dtype=np.int64,
        copy=True,
    )


    if (
        len(original_row_index) > 1
        and
        not np.all(
            original_row_index[
                1:
            ]
            >
            original_row_index[
                :-1
            ]
        )
    ):

        raise RuntimeError(
            "Reconstructed original_row_index is not strictly ascending."
        )


    retained_labels = canonicalize_labels(
        retained[
            "Label"
        ]
    )


    y_target = (
        retained_labels
        !=
        "benign"
    ).to_numpy(
        dtype=np.uint8,
    )


    clean_position = np.arange(
        EXPECTED_CLEAN_POSITION_START,
        EXPECTED_CLEAN_POSITION_STOP,
        dtype=np.int64,
    )


    X70 = np.empty(
        (
            EXPECTED_TARGET_ROWS,
            70,
        ),
        dtype=np.float64,
    )


    for j, feature in enumerate(
        FEATURE_ORDER_70
    ):

        source_series = retained[
            feature
        ]


        numeric = pd.to_numeric(
            source_series,
            errors="coerce",
        )


        invalid = (
            numeric.isna()
            &
            source_series.notna()
        )


        if invalid.any():

            invalid_tokens = (
                source_series.loc[
                    invalid
                ]
                .astype("string")
                .str.strip()
                .str.casefold()
            )


            truly_invalid = ~invalid_tokens.isin(
                [
                    "",
                    "nan",
                ]
            )


            if truly_invalid.any():

                examples = (
                    invalid_tokens.loc[
                        truly_invalid
                    ]
                    .drop_duplicates()
                    .head(10)
                    .tolist()
                )

                raise RuntimeError(
                    "\nFrozen numeric FAIL_CLOSED rule triggered.\n"
                    f"Feature: {feature}\n"
                    f"Tokens: {examples}"
                )


        X70[
            :,
            j
        ] = numeric.to_numpy(
            dtype=np.float64,
        )


    positive_inf = int(
        np.isposinf(
            X70
        ).sum()
    )


    negative_inf = int(
        np.isneginf(
            X70
        ).sum()
    )


    if positive_inf != EXPECTED_70F_POS_INF:

        raise RuntimeError(
            "\nFeb-28 +inf count mismatch.\n"
            f"Expected: {EXPECTED_70F_POS_INF:,}\n"
            f"Actual:   {positive_inf:,}"
        )


    if negative_inf != EXPECTED_70F_NEG_INF:

        raise RuntimeError(
            "Feb-28 -inf count mismatch."
        )


    inf_mask = np.isinf(
        X70
    )


    if inf_mask.any():

        X70[
            inf_mask
        ] = np.nan


    nan70 = int(
        np.isnan(
            X70
        ).sum()
    )


    if nan70 != EXPECTED_70F_OUTPUT_NAN:

        raise RuntimeError(
            "\nFeb-28 final NaN count mismatch.\n"
            f"Expected: {EXPECTED_70F_OUTPUT_NAN:,}\n"
            f"Actual:   {nan70:,}"
        )


    target_materialization_mode = (
        "RECONSTRUCTED_FROM_BYTE_EXACT_RAW_PLUS_FROZEN_K79"
    )


    target_materialization_source = str(
        exact_raw_path
    )


    del raw
    del effective
    del retained
    del effective_labels
    del retained_labels
    del y_effective
    del k79
    del k79_day7
    del exclusion_raw_labels
    del exclusion_binary_from_raw
    del exclusion_binary_frozen
    del inf_mask

    gc.collect()


# ==============================================================================
# 12. COMPLETE TARGET IDENTITY GATE
# ==============================================================================

print()
print("=" * 118)
print("IDS2018 FEB-28 TARGET IDENTITY")
print("=" * 118)


if X70.shape != (
    EXPECTED_TARGET_ROWS,
    70,
):

    raise RuntimeError(
        f"Unexpected frozen 70F target shape: {X70.shape}"
    )


if y_target.shape != (
    EXPECTED_TARGET_ROWS,
):

    raise RuntimeError(
        "Target binary-label shape mismatch."
    )


target_attack = int(
    y_target.sum()
)


target_benign = int(
    EXPECTED_TARGET_ROWS
    -
    target_attack
)


if target_benign != EXPECTED_TARGET_BENIGN:

    raise RuntimeError(
        "\nTarget benign count mismatch.\n"
        f"Expected: {EXPECTED_TARGET_BENIGN:,}\n"
        f"Actual:   {target_benign:,}"
    )


if target_attack != EXPECTED_TARGET_ATTACK:

    raise RuntimeError(
        "\nTarget attack count mismatch.\n"
        f"Expected: {EXPECTED_TARGET_ATTACK:,}\n"
        f"Actual:   {target_attack:,}"
    )


if np.isinf(
    X70
).any():

    raise RuntimeError(
        "Infinity survived frozen target numeric policy."
    )


if int(
    np.isnan(
        X70
    ).sum()
) != EXPECTED_70F_OUTPUT_NAN:

    raise RuntimeError(
        "Frozen 70F target NaN count changed."
    )


target_label_sha = sha256_array(
    y_target
)

original_row_index_sha = sha256_array(
    original_row_index
)

clean_position_sha = sha256_array(
    clean_position
)

target_70f_sha = sha256_array(
    X70
)


print(
    "Materialization:",
    target_materialization_mode,
)

print(
    "Source:",
    target_materialization_source,
)

print()

print(
    "Rows:   ",
    f"{EXPECTED_TARGET_ROWS:,}",
)

print(
    "Benign: ",
    f"{target_benign:,}",
)

print(
    "Attack: ",
    f"{target_attack:,}",
)

print(
    "NaNs 70F:",
    f"{int(np.isnan(X70).sum()):,}",
)

print()

print(
    "70F matrix SHA:       ",
    target_70f_sha,
)

print(
    "Binary label SHA:     ",
    target_label_sha,
)

print(
    "Original row-index SHA:",
    original_row_index_sha,
)

print(
    "Clean-position SHA:   ",
    clean_position_sha,
)

print()


# ==============================================================================
# 13. OPTIONAL CROSS-CHECK AGAINST FROZEN STAGE22 VALIDATION LABEL ARTIFACT
#
# This performs NO model inference.
# ==============================================================================

if STAGE22_VALIDATION_NPZ_PATH.is_file():

    stage22_npz_sha = sha256_file(
        STAGE22_VALIDATION_NPZ_PATH
    )


    if stage22_npz_sha != EXPECTED_STAGE22_VALIDATION_NPZ_SHA:

        raise RuntimeError(
            "Stage22 validation probability NPZ SHA changed."
        )


    ref_npz = np.load(
        STAGE22_VALIDATION_NPZ_PATH,
        allow_pickle=False,
    )


    print(
        "Stage22 validation NPZ keys:",
        list(
            ref_npz.files
        ),
    )


    binary_reference_candidates = []


    for key in ref_npz.files:

        arr = np.asarray(
            ref_npz[
                key
            ]
        )


        if arr.shape != (
            EXPECTED_TARGET_ROWS,
        ):

            continue


        if arr.dtype.kind not in (
            "b",
            "i",
            "u",
        ):

            continue


        uniques = np.unique(
            arr
        )


        if np.all(
            np.isin(
                uniques,
                [
                    0,
                    1,
                ],
            )
        ):

            binary_reference_candidates.append(
                (
                    key,
                    arr.astype(
                        np.uint8,
                        copy=False,
                    ),
                )
            )


    if len(
        binary_reference_candidates
    ) == 1:

        ref_key, ref_y = binary_reference_candidates[
            0
        ]


        if not np.array_equal(
            y_target,
            ref_y,
        ):

            raise RuntimeError(
                "\nReconstructed target labels differ from frozen Stage22 validation labels."
            )


        print(
            f"[PASS] Binary target labels exactly match Stage22 NPZ key '{ref_key}'."
        )


    elif len(
        binary_reference_candidates
    ) == 0:

        print(
            "[INFO] Stage22 NPZ contains no unique binary-label candidate."
        )


    else:

        print(
            "[INFO] Multiple binary-looking Stage22 arrays; no automatic label-key inference used."
        )


    del ref_npz

    gc.collect()


print()
print(
    "[PASS] IDS2018 Feb-28 frozen target membership reconstructed."
)

print(
    "[PASS] No target feature has yet been supplied to the secondary model."
)

print()


# ==============================================================================
# 14. PROJECT 70F -> FROZEN BRIDGE62
# ==============================================================================

bridge62_indices = [
    FEATURE_ORDER_70.index(
        feature
    )
    for feature in FEATURE_ORDER_62
]


X_target = np.ascontiguousarray(
    X70[
        :,
        bridge62_indices
    ],
    dtype=np.float64,
)


if X_target.shape != (
    EXPECTED_TARGET_ROWS,
    62,
):

    raise RuntimeError(
        "bridge62 target projection shape mismatch."
    )


target_62f_sha = sha256_array(
    X_target
)


target_62f_nan = int(
    np.isnan(
        X_target
    ).sum()
)


print("=" * 118)
print("BRIDGE62 TARGET PROJECTION")
print("=" * 118)

print(
    "Shape:",
    X_target.shape,
)

print(
    "NaN cells:",
    f"{target_62f_nan:,}",
)

print(
    "Matrix SHA:",
    target_62f_sha,
)

print()

print(
    "[PASS] Frozen bridge62 target projection ready."
)

print()


del X70

gc.collect()


# ==============================================================================
# 15. LOAD FROZEN SECONDARY MODEL
#
# Model loading does not consume opening #5.
# ==============================================================================

import xgboost as xgb


print("=" * 118)
print("MODEL LOAD")
print("=" * 118)

print(
    "XGBoost:",
    xgb.__version__,
)


booster = xgb.Booster()

booster.load_model(
    str(
        B62_MODEL_PATH
    )
)


booster.set_param(
    {
        "device":
            "cpu",

        "nthread":
            -1,
    }
)


if booster.num_features() != 62:

    raise RuntimeError(
        "Frozen secondary bridge62 model feature count mismatch."
    )


if booster.num_boosted_rounds() != 400:

    raise RuntimeError(
        "Frozen secondary bridge62 model round count mismatch."
    )


print(
    "[PASS] Frozen bridge62 secondary model loaded."
)

print()


# ==============================================================================
# 16. TARGET OPENING #5
# ==============================================================================

print("=" * 118)
print("SECONDARY TARGET INFERENCE")
print("=" * 118)

print()
print(
    "The next prediction consumes SECONDARY TARGET OPENING #5."
)

print()


opening_started_at = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


OPENING_LEDGER.write_text(
    json.dumps(
        {
            "stage":
                "Stage24-5A",

            "direction":
                "CICIDS2017_TO_IDS2018",

            "bridge":
                "bridge62",

            "target":
                "IDS2018_02-28-2018",

            "opening_number":
                5,

            "opening_consumed":
                True,

            "opening_consumed_at_utc":
                opening_started_at,

            "trigger":
                (
                    "FIRST_IDS2018_FEB28_BRIDGE62_TARGET_FEATURE_VALUES_"
                    "SUPPLIED_TO_FROZEN_SECONDARY_XGBOOST"
                ),

            "parent_commit":
                EXPECTED_PARENT,

            "scientific_fits_completed":
                4,

            "additional_fits_authorized":
                0,
        },
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


print(
    ">>> TARGET OPENING #5 CONSUMED <<<"
)

print()


inference_start = time.time()


dtarget = xgb.DMatrix(
    X_target,
    missing=np.nan,
)


p_target = np.asarray(
    booster.predict(
        dtarget,
        validate_features=False,
    ),
    dtype=np.float32,
)


inference_seconds = (
    time.time()
    -
    inference_start
)


if p_target.shape != (
    EXPECTED_TARGET_ROWS,
):

    raise RuntimeError(
        "Target probability shape mismatch."
    )


if not np.isfinite(
    p_target
).all():

    raise RuntimeError(
        "Non-finite target probability."
    )


if (
    (p_target < 0).any()
    or
    (p_target > 1).any()
):

    raise RuntimeError(
        "Target probability outside [0,1]."
    )


probability_sha = sha256_array(
    p_target
)


print(
    "Inference seconds:",
    f"{inference_seconds:.3f}",
)

print(
    "Probability min/max:",
    f"{float(p_target.min()):.9f}",
    "/",
    f"{float(p_target.max()):.9f}",
)

print(
    "Probability SHA:",
    probability_sha,
)

print()


del dtarget
del X_target

gc.collect()


# ==============================================================================
# 17. FROZEN TARGET METRICS
# ==============================================================================

target_prevalence = float(
    EXPECTED_TARGET_ATTACK
    /
    EXPECTED_TARGET_ROWS
)


pr_auc = float(
    average_precision_score(
        y_target,
        p_target,
    )
)


roc_auc = float(
    roc_auc_score(
        y_target,
        p_target,
    )
)


pr_excess = float(
    pr_auc
    -
    target_prevalence
)


pr_normalized = float(
    (
        pr_auc
        -
        target_prevalence
    )
    /
    (
        1.0
        -
        target_prevalence
    )
)


p64 = p_target.astype(
    np.float64,
    copy=False,
)

y64 = y_target.astype(
    np.float64,
    copy=False,
)


brier = float(
    np.mean(
        (
            p64
            -
            y64
        ) ** 2
    )
)


clipped = np.clip(
    p64,
    1e-15,
    1.0 - 1e-15,
)


log_loss = float(
    -np.mean(
        y64
        *
        np.log(
            clipped
        )
        +
        (
            1.0
            -
            y64
        )
        *
        np.log(
            1.0
            -
            clipped
        )
    )
)


bins = np.floor(
    p64
    *
    10.0
).astype(
    np.int8
)

bins = np.clip(
    bins,
    0,
    9,
)


ece = 0.0

ece_bins = []


for b in range(
    10
):

    mask = (
        bins
        ==
        b
    )

    count = int(
        mask.sum()
    )


    if count == 0:

        ece_bins.append(
            {
                "bin":
                    b,

                "count":
                    0,

                "mean_probability":
                    None,

                "empirical_rate":
                    None,

                "absolute_gap":
                    None,
            }
        )

        continue


    confidence = float(
        p64[
            mask
        ].mean()
    )

    empirical = float(
        y64[
            mask
        ].mean()
    )

    gap = abs(
        confidence
        -
        empirical
    )


    ece += (
        count
        /
        EXPECTED_TARGET_ROWS
    ) * gap


    ece_bins.append(
        {
            "bin":
                b,

            "count":
                count,

            "mean_probability":
                confidence,

            "empirical_rate":
                empirical,

            "absolute_gap":
                float(
                    gap
                ),
        }
    )


ece = float(
    ece
)


print("=" * 118)
print("SECONDARY TARGET METRICS — bridge62")
print("=" * 118)

print(
    "Target prevalence:",
    f"{target_prevalence:.15f}",
)

print(
    "PR-AUC:          ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:         ",
    f"{roc_auc:.15f}",
)

print(
    "PR excess:       ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized:   ",
    f"{pr_normalized:.15f}",
)

print()

print(
    "Brier:           ",
    f"{brier:.15f}",
)

print(
    "Log loss:        ",
    f"{log_loss:.15f}",
)

print(
    "ECE-10:          ",
    f"{ece:.15f}",
)

print()


# ==============================================================================
# 18. FROZEN SOURCE-VALIDATION OPERATING POINTS ON TARGET
# ==============================================================================

def threshold_metrics(
    threshold,
):

    threshold = np.float32(
        threshold
    )


    pred = (
        p_target
        >=
        threshold
    )


    y1 = (
        y_target
        ==
        1
    )

    y0 = ~y1


    tp = int(
        np.sum(
            pred
            &
            y1
        )
    )

    fp = int(
        np.sum(
            pred
            &
            y0
        )
    )

    fn = int(
        EXPECTED_TARGET_ATTACK
        -
        tp
    )

    tn = int(
        EXPECTED_TARGET_BENIGN
        -
        fp
    )


    precision = (
        tp
        /
        (
            tp + fp
        )
        if (
            tp + fp
        )
        else 0.0
    )


    recall = (
        tp
        /
        (
            tp + fn
        )
        if (
            tp + fn
        )
        else 0.0
    )


    fpr = (
        fp
        /
        (
            fp + tn
        )
        if (
            fp + tn
        )
        else 0.0
    )


    fnr = (
        fn
        /
        (
            fn + tp
        )
        if (
            fn + tp
        )
        else 0.0
    )


    f1_den = (
        2 * tp
        +
        fp
        +
        fn
    )


    f2_den = (
        5 * tp
        +
        4 * fn
        +
        fp
    )


    return {
        "threshold":
            float(
                threshold
            ),

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn,

        "accuracy":
            float(
                (
                    tp + tn
                )
                /
                EXPECTED_TARGET_ROWS
            ),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                2 * tp / f1_den
                if f1_den
                else 0.0
            ),

        "f2":
            float(
                5 * tp / f2_den
                if f2_den
                else 0.0
            ),

        "fpr":
            float(
                fpr
            ),

        "fnr":
            float(
                fnr
            ),
    }


standard = threshold_metrics(
    STANDARD_THRESHOLD
)

balanced = threshold_metrics(
    BALANCED_THRESHOLD
)

security = threshold_metrics(
    SECURITY_THRESHOLD
)


print("=" * 118)
print("SOURCE-VALIDATION THRESHOLDS APPLIED TO TARGET")
print("=" * 118)


for name, metrics in [
    (
        "STANDARD",
        standard,
    ),
    (
        "BALANCED",
        balanced,
    ),
    (
        "SECURITY",
        security,
    ),
]:

    print(
        name,
        "@",
        metrics[
            "threshold"
        ],
    )

    print(
        "  TP/TN/FP/FN:",
        metrics[
            "tp"
        ],
        metrics[
            "tn"
        ],
        metrics[
            "fp"
        ],
        metrics[
            "fn"
        ],
    )

    print(
        "  precision/recall/F1/F2/FPR/FNR:",
        f"{metrics['precision']:.9f}",
        f"{metrics['recall']:.9f}",
        f"{metrics['f1']:.9f}",
        f"{metrics['f2']:.9f}",
        f"{metrics['fpr']:.9f}",
        f"{metrics['fnr']:.9f}",
    )

    print()


# ==============================================================================
# 19. FROZEN NULL ANCHORS
# ==============================================================================

target_prior_brier = float(
    np.mean(
        (
            np.float64(
                target_prevalence
            )
            -
            y64
        ) ** 2
    )
)


source_prior_brier = float(
    np.mean(
        (
            np.float64(
                source_prior
            )
            -
            y64
        ) ** 2
    )
)


null_anchors = {
    "target_prevalence_chance_anchor": {
        "probability":
            target_prevalence,

        "pr_auc_anchor":
            target_prevalence,

        "roc_auc_anchor":
            0.5,

        "brier":
            target_prior_brier,
    },

    "secondary_source_prior_constant_predictor": {
        "probability":
            source_prior,

        "fit_on_target":
            False,

        "probability_source":
            "CICIDS2017_MONDAY_WEDNESDAY_SOURCE_TRAIN_ONLY",

        "pr_auc":
            target_prevalence,

        "roc_auc":
            0.5,

        "brier":
            source_prior_brier,
    },
}


print("=" * 118)
print("NULL ANCHORS")
print("=" * 118)

print(
    "Target prevalence PR anchor:",
    f"{target_prevalence:.15f}",
)

print(
    "Target-prior Brier:           ",
    f"{target_prior_brier:.15f}",
)

print(
    "Source prior probability:     ",
    f"{source_prior:.15f}",
)

print(
    "Source-prior Brier on target: ",
    f"{source_prior_brier:.15f}",
)

print()


# ==============================================================================
# 20. PERSIST TARGET VECTOR + CANONICAL ROW LOCATORS
# ==============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


np.savez_compressed(
    PRED_PATH,

    probability=
        p_target,

    binary_label=
        y_target,

    original_row_index=
        original_row_index,

    clean_position=
        clean_position,
)


pred_file_sha = sha256_file(
    PRED_PATH
)


print("=" * 118)
print("PERSISTED TARGET OPENING #5")
print("=" * 118)

print(
    "Prediction artifact:",
    PRED_PATH.relative_to(
        REPO
    ),
)

print(
    "Bytes:",
    f"{PRED_PATH.stat().st_size:,}",
)

print(
    "File SHA:",
    pred_file_sha,
)

print(
    "Probability array SHA:",
    probability_sha,
)

print(
    "Binary label SHA:",
    target_label_sha,
)

print(
    "Original row-index SHA:",
    original_row_index_sha,
)

print()


# ==============================================================================
# 21. RESULT RECEIPT
# ==============================================================================

result = {
    "stage":
        "Stage24-5A",

    "status":
        "SECONDARY_BRIDGE62_IDS2018_FEB28_TARGET_OPENING_COMPLETE",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "direction":
        "CICIDS2017_TO_IDS2018",

    "bridge":
        "bridge62",

    "target":
        "IDS2018_02-28-2018",

    "parent_commit":
        EXPECTED_PARENT,

    "target_opening": {
        "opening_number":
            5,

        "original_budget":
            8,

        "consumed_before":
            OPENINGS_BEFORE,

        "consumed_after":
            OPENINGS_AFTER,

        "primary_grounded_cells_cancelled_before_opening":
            2,

        "opening_consumed_at_utc":
            opening_started_at,

        "new_scientific_fits":
            0,
    },

    "scientific_fit_accounting": {
        "budget":
            4,

        "completed":
            4,

        "remaining":
            0,

        "additional_fits_authorized":
            0,
    },

    "model": {
        "learner":
            "XGBOOST_ONLY",

        "source_domain":
            "CICIDS2017",

        "bridge":
            "bridge62",

        "model_file":
            str(
                B62_MODEL_PATH.relative_to(
                    REPO
                )
            ),

        "model_sha256":
            model_sha,

        "execution_device":
            "cpu",

        "feature_count":
            62,

        "boosted_rounds":
            400,
    },

    "target_population": {
        "dataset":
            "CSE-CIC-IDS2018",

        "file":
            "02-28-2018.csv",

        "rows":
            EXPECTED_TARGET_ROWS,

        "benign":
            target_benign,

        "attack":
            target_attack,

        "prevalence":
            target_prevalence,

        "raw_source_expected_sha256":
            EXPECTED_RAW_SHA,

        "materialization_mode":
            target_materialization_mode,

        "materialization_source":
            target_materialization_source,

        "stage22r_day07_cache_expected_sha256":
            EXPECTED_DAY7_CACHE_SHA,

        "binary_label_sha256":
            target_label_sha,

        "original_row_index_sha256":
            original_row_index_sha,

        "clean_position_sha256":
            clean_position_sha,

        "canonical_row_order":
            (
                "clean_position ASC; day_id=7; "
                "original_zero_based_row_index ASC after frozen K79 exclusions"
            ),
    },

    "representation": {
        "feature_count":
            62,

        "feature_order":
            FEATURE_ORDER_62,

        "input_dtype":
            "float64",

        "target_70f_matrix_sha256":
            target_70f_sha,

        "target_bridge62_matrix_sha256":
            target_62f_sha,

        "target_70f_nan_cells":
            EXPECTED_70F_OUTPUT_NAN,

        "target_bridge62_nan_cells":
            target_62f_nan,

        "explicit_imputation":
            "NONE",

        "scaling":
            "NONE",

        "positive_infinity":
            "CONVERT_TO_NAN",

        "negative_infinity":
            "CONVERT_TO_NAN",
    },

    "prediction": {
        "storage_dtype":
            "float32",

        "probability_array_sha256":
            probability_sha,

        "inference_seconds":
            inference_seconds,

        "artifact":
            str(
                PRED_PATH.relative_to(
                    REPO
                )
            ),

        "artifact_sha256":
            pred_file_sha,
    },

    "metrics": {
        "primary": {
            "pr_auc":
                pr_auc,

            "roc_auc":
                roc_auc,

            "prevalence_chance_anchor":
                target_prevalence,

            "pr_excess":
                pr_excess,

            "pr_normalized":
                pr_normalized,
        },

        "calibration": {
            "brier":
                brier,

            "log_loss":
                log_loss,

            "ece_10":
                ece,

            "ece_bins":
                ece_bins,
        },

        "thresholded": {
            "standard":
                standard,

            "balanced":
                balanced,

            "security":
                security,
        },

        "null_anchors":
            null_anchors,
    },

    "threshold_provenance": {
        "selection_dataset":
            "CICIDS2017_THURSDAY_SOURCE_VALIDATION",

        "standard":
            float(
                STANDARD_THRESHOLD
            ),

        "balanced":
            float(
                BALANCED_THRESHOLD
            ),

        "security":
            float(
                SECURITY_THRESHOLD
            ),

        "IDS2018_target_threshold_search":
            False,
    },

    "bootstrap": {
        "performed":
            False,

        "reason":
            (
                "Frozen paired bridge62-vs-bridge70 secondary comparison "
                "is deferred until target opening #6 is durable."
            ),

        "frozen_replicates":
            2000,

        "frozen_seed":
            42,
    },

    "anti_adaptation": {
        "new_fit":
            False,

        "target_guided_refit":
            False,

        "target_threshold_search":
            False,

        "target_feature_search":
            False,

        "target_feature_remapping":
            False,

        "target_imputation_fit":
            False,

        "target_scaling_fit":
            False,

        "target_calibration":
            False,

        "performance_based_change":
            False,
    },

    "opening_accounting": {
        "budget":
            8,

        "consumed":
            5,

        "administratively_cancelled":
            2,

        "remaining_evaluable":
            1,

        "next_evaluable_cell":
            "SECONDARY_BRIDGE70_IDS2018_FEB28",
    },

    "next_authorized_step":
        (
            "STAGE24_5B_SECONDARY_BRIDGE70_IDS2018_FEB28_TARGET_OPENING"
        ),
}


RESULT_PATH.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


result_sha = sha256_file(
    RESULT_PATH
)


RESULT_SHA_PATH.write_text(
    f"{result_sha}  {RESULT_PATH.name}\n",
    encoding="utf-8",
)


CHECKSUMS_PATH.write_text(
    (
        f"{result_sha}  {RESULT_PATH.name}\n"
        f"{pred_file_sha}  {PRED_PATH.name}\n"
    ),
    encoding="utf-8",
)


print(
    "Result SHA:",
    result_sha,
)

print()


# ==============================================================================
# 22. GIT AUTHOR SAFETY
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    git_cmd(
        "config",
        "--local",
        "user.name",
        git_cmd(
            "log",
            "-1",
            "--format=%an",
        ),
    )


if not author_email:

    git_cmd(
        "config",
        "--local",
        "user.email",
        git_cmd(
            "log",
            "-1",
            "--format=%ae",
        ),
    )


# ==============================================================================
# 23. GIT FREEZE
# ==============================================================================

print("=" * 118)
print("GIT FREEZE")
print("=" * 118)


git_cmd(
    "add",
    "--",
    str(
        PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


expected_staged = {
    str(
        PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: open secondary bridge62 ids2018 feb28 target",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage24-5A commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 24. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 118)
print("PUSH + REMOTE VERIFY")
print("=" * 118)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository not clean after Stage24-5A:\n"
        + final_status
    )


if OPENING_LEDGER.exists():

    OPENING_LEDGER.unlink()


print(
    "[PASS] Remote main == local Stage24-5A commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 25. FINAL
# ==============================================================================

print("=" * 118)
print("STAGE24-5A SECONDARY TARGET OPENING #5: PASS")
print("=" * 118)

print()

print(
    "Direction:                    CICIDS2017 -> IDS2018"
)

print(
    "Bridge:                       bridge62"
)

print(
    "Target:                       IDS2018 Feb-28"
)

print()

print(
    "Target rows:                  ",
    f"{EXPECTED_TARGET_ROWS:,}",
)

print(
    "Benign:                       ",
    f"{target_benign:,}",
)

print(
    "Attack:                       ",
    f"{target_attack:,}",
)

print(
    "Prevalence:                   ",
    f"{target_prevalence:.12f}",
)

print()

print(
    "PR-AUC:                       ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:                      ",
    f"{roc_auc:.15f}",
)

print(
    "PR excess:                    ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized:                ",
    f"{pr_normalized:.15f}",
)

print()

print(
    "Brier:                        ",
    f"{brier:.15f}",
)

print(
    "Log loss:                     ",
    f"{log_loss:.15f}",
)

print(
    "ECE-10:                       ",
    f"{ece:.15f}",
)

print()

print(
    "STANDARD threshold:           ",
    float(
        STANDARD_THRESHOLD
    ),
)

print(
    "BALANCED threshold:           ",
    float(
        BALANCED_THRESHOLD
    ),
)

print(
    "SECURITY threshold:           ",
    float(
        SECURITY_THRESHOLD
    ),
)

print()

print(
    "Scientific fits completed:    4 / 4"
)

print(
    "Scientific fits remaining:    0"
)

print(
    "Additional fits authorized:   0"
)

print()

print(
    "Target openings consumed:     5 / 8"
)

print(
    "Administratively cancelled:   2"
)

print(
    "Remaining evaluable opening:  1"
)

print()

print(
    "Probability SHA:"
)

print(
    " ",
    probability_sha,
)

print()

print(
    "Canonical binary-label SHA:"
)

print(
    " ",
    target_label_sha,
)

print()

print(
    "Original-row-index SHA:"
)

print(
    " ",
    original_row_index_sha,
)

print()

print(
    "Result SHA:"
)

print(
    " ",
    result_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "NEXT:"
)

print(
    "  SECONDARY TARGET OPENING #6:"
)

print(
    "  bridge70 / IDS2018 Feb-28"
)

print(
    "  then freeze secondary paired bootstrap + final Stage24 synthesis."
)

print("=" * 118)

STAGE24-5A — SECONDARY TARGET OPENING #5 — bridge62 / IDS2018 Feb-28

GOVERNANCE
HEAD: 1390cf5c3f2a5c6ec6c20cf486ca6ee6c84d4769
[PASS] Repository clean.
[PASS] No unresolved scientific-fit ledger.
[PASS] No unresolved target-opening ledger.
[PASS] Stage24-5A has not previously been opened.

SCIENTIFIC FIT ACCOUNTING
Bridge62 result SHA: fc496c6342269d089eeaa4c1ec043741644ce5fe379f92e267e2d6689787335c
Bridge70 result SHA: 7ad3789d670442d10e203fd35bcee6c84c52e009d7e162947825a08df50e74f1

[PASS] Scientific fits completed: 4 / 4.
[PASS] Scientific fits remaining: 0.
[PASS] Additional fits authorized: 0.

FROZEN PROTOCOL
[PASS] Secondary target = IDS2018 Feb-28.
[PASS] bridge62 feature order exact.

FROZEN BRIDGE62 SECONDARY MODEL
Model SHA: 8040bc7909429736c95ca396ebebf18636ce5d1f5c2232d12042c6ffadd47c5d
Source prior: 0.159725626749294
STANDARD: 0.5
BALANCED: 0.17000000178813934
SECURITY: 0.05000000074505806

[PASS] All target operating points come from CICIDS2017 Thursday validation only.

In [42]:
# ==============================================================================
# STAGE24-5B — SECONDARY TARGET OPENING #6
# CICIDS2017 -> IDS2018
# bridge70 / IDS2018 Feb-28
#
# FINAL EVALUABLE TARGET OPENING
#
# Scientific fits:
#   COMPLETE = 4 / 4
#   remaining = 0
#   additional fits authorized = 0
#
# Target openings:
#   before = 5 / 8
#   this   = #6
#   after  = 6 / 8
#
# Two frozen GROUNDED_S4 primary cells remain administratively cancelled
# and MUST NOT be reallocated.
#
# Target membership/order is inherited exactly from Stage24-5A:
#   rows              = 593,780
#   benign            = 531,524
#   attack            = 62,256
#   binary-label SHA  =
#     17369d505c1ce11f425a7c14c7552c95b39aad3492ae643c31f43a8a66a35177
#   physical-row SHA  =
#     374de0e9d74040581ddc5977fd4c78f8f8959df610d20d1775415f10f3e92dfc
#
# Frozen target 70F matrix identity from Stage24-5A:
#   97d0e4406b07b563b062b68b26676a57f36584befce00d4c0c9b71fe73a6ca33
#
# Frozen bridge70 secondary model:
#   c8e658832e1f4a389bccbf32a00f1677463b0c9b4a8070965cc7d3a98c409621
#
# Source-validation-only thresholds:
#   STANDARD = 0.50
#   BALANCED = 0.18
#   SECURITY = 0.05
#
# NO FIT.
# NO TARGET THRESHOLD SEARCH.
# NO TARGET FEATURE SEARCH.
# NO TARGET CALIBRATION.
#
# Once:
#
#     >>> TARGET OPENING #6 CONSUMED <<<
#
# appears, DO NOT RERUN BLINDLY if a later step fails.
# ==============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


print("=" * 118)
print("STAGE24-5B — SECONDARY TARGET OPENING #6 — bridge70 / IDS2018 Feb-28")
print("=" * 118)
print()


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "e09ff89f582f972a9641705f0850a9ba99516902"
)


EXPECTED_STAGE24_5A_RESULT_SHA = (
    "54ea37bd6cc5ea82b77762ee672afcecf9584d6fb115187df9040cebac715515"
)

EXPECTED_STAGE24_5A_PRED_FILE_SHA = (
    "a4fdb7c0d8b7f92065664d879c62174b854a2d8832cac627c985458c5f857671"
)

EXPECTED_STAGE24_5A_PROB_SHA = (
    "4c7cc6af61b8c4813c9241d526b6a473e46e4f69a058305eceb7fb4c30621640"
)


EXPECTED_TARGET_ROWS = 593_780
EXPECTED_TARGET_BENIGN = 531_524
EXPECTED_TARGET_ATTACK = 62_256

EXPECTED_PHYSICAL_ROWS = 613_104


EXPECTED_TARGET_LABEL_SHA = (
    "17369d505c1ce11f425a7c14c7552c95b39aad3492ae643c31f43a8a66a35177"
)

EXPECTED_ORIGINAL_ROW_INDEX_SHA = (
    "374de0e9d74040581ddc5977fd4c78f8f8959df610d20d1775415f10f3e92dfc"
)

EXPECTED_CLEAN_POSITION_SHA = (
    "d8a090f4c1667d62dd81de1ce8be9b0d68062adc2799c67fb7b55ad74a1f8b9b"
)


EXPECTED_TARGET_70F_SHA = (
    "97d0e4406b07b563b062b68b26676a57f36584befce00d4c0c9b71fe73a6ca33"
)

EXPECTED_TARGET_70F_NAN = 7_936
EXPECTED_TARGET_70F_POS_INF = 6_085
EXPECTED_TARGET_70F_NEG_INF = 0


EXPECTED_RAW_SHA = (
    "f15e2a12304446058a0186c8ad67de2bd15735a9ba5c70c9a1f4c4242ab06771"
)


EXPECTED_BRIDGE70_RESULT_SHA = (
    "7ad3789d670442d10e203fd35bcee6c84c52e009d7e162947825a08df50e74f1"
)

EXPECTED_BRIDGE70_MODEL_SHA = (
    "c8e658832e1f4a389bccbf32a00f1677463b0c9b4a8070965cc7d3a98c409621"
)


EXPECTED_BRIDGE_SPEC_SHA = (
    "b7ff1d8563aafc69145be3acc827fbc59ae75d880b9804bad727f408299fce2f"
)

EXPECTED_PROTOCOL_LOCK_SHA = (
    "8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2"
)


EXPECTED_CLEAN_START = 13_818_623
EXPECTED_CLEAN_STOP = 14_412_403


SCIENTIFIC_FITS = 4

OPENINGS_BEFORE = 5
OPENINGS_AFTER = 6


# ==============================================================================
# 1. PATHS
# ==============================================================================

LOCK_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
)


BRIDGE_SPEC_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.json"
)

BRIDGE_SPEC_SHA_PATH = (
    LOCK_DIR
    / "stage24_0c_semantic_bridge_spec.sha256"
)


FINAL_LOCK_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.json"
)

FINAL_LOCK_SHA_PATH = (
    LOCK_DIR
    / "stage24_0c_final_preopening_protocol_lock.sha256"
)


# ------------------------------------------------------------------------------
# Stage24-5A reference
# ------------------------------------------------------------------------------

B62_TARGET_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_5_secondary_target_openings"
    / "stage24_5a_bridge62_ids2018_feb28"
)


B62_TARGET_RESULT_PATH = (
    B62_TARGET_DIR
    / "stage24_5a_secondary_bridge62_ids2018_feb28_result.json"
)


B62_TARGET_RESULT_SHA_PATH = (
    B62_TARGET_DIR
    / "stage24_5a_secondary_bridge62_ids2018_feb28_result.sha256"
)


B62_TARGET_PRED_PATH = (
    B62_TARGET_DIR
    / "secondary_bridge62_ids2018_feb28_predictions.npz"
)


# ------------------------------------------------------------------------------
# Stage24-4B bridge70 model
# ------------------------------------------------------------------------------

B70_SOURCE_DIR = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_4_secondary_source_training"
    / "stage24_4b_bridge70_xgboost"
)


B70_RESULT_PATH = (
    B70_SOURCE_DIR
    / "stage24_4b_secondary_bridge70_result.json"
)


B70_RESULT_SHA_PATH = (
    B70_SOURCE_DIR
    / "stage24_4b_secondary_bridge70_result.sha256"
)


B70_MODEL_PATH = (
    B70_SOURCE_DIR
    / "secondary_bridge70_xgboost_model.json"
)


# ------------------------------------------------------------------------------
# Stage24-5B output
# ------------------------------------------------------------------------------

OUT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_5_secondary_target_openings"
    / "stage24_5b_bridge70_ids2018_feb28"
)


PRED_PATH = (
    OUT
    / "secondary_bridge70_ids2018_feb28_predictions.npz"
)


RESULT_PATH = (
    OUT
    / "stage24_5b_secondary_bridge70_ids2018_feb28_result.json"
)


RESULT_SHA_PATH = (
    OUT
    / "stage24_5b_secondary_bridge70_ids2018_feb28_result.sha256"
)


CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


OPENING_LEDGER = Path(
    "/kaggle/working/stage24_target_opening_runtime_ledger.json"
)


FIT_LEDGER = Path(
    "/kaggle/working/stage24_secondary_fit_runtime_ledger.json"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    cmd = [
        "git"
    ]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [
        str(x)
        for x in args
    ]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_array(
    array,
):

    arr = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        arr.view(
            np.uint8
        )
    ).hexdigest()


def verify_sidecar(
    artifact,
    sidecar,
):

    if not artifact.is_file():

        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )

    if not sidecar.is_file():

        raise RuntimeError(
            f"Missing SHA sidecar:\n{sidecar}"
        )

    actual = sha256_file(
        artifact
    )

    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )

    if actual != expected:

        raise RuntimeError(
            "\nSHA mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )

    return actual


DASH_TRANSLATION = str.maketrans(
    {
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2015": "-",
        "\u2212": "-",
    }
)


def canonicalize_labels(
    series,
):

    if series.isna().any():

        raise RuntimeError(
            "NULL target label encountered."
        )

    s = (
        series
        .astype(
            "string"
        )
        .str.strip()
        .str.translate(
            DASH_TRANSLATION
        )
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.casefold()
    )

    if s.isna().any():

        raise RuntimeError(
            "Label canonicalization produced NULL."
        )

    if (
        s.str.len()
        ==
        0
    ).any():

        raise RuntimeError(
            "Empty target label encountered."
        )

    return s


# ==============================================================================
# 3. GOVERNANCE
# ==============================================================================

print("=" * 118)
print("GOVERNANCE")
print("=" * 118)


head = git_cmd(
    "rev-parse",
    "HEAD",
)


status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


if FIT_LEDGER.exists():

    raise RuntimeError(
        "\nUnexpected scientific-fit ledger exists:\n"
        f"{FIT_LEDGER}"
    )


if OPENING_LEDGER.exists():

    raise RuntimeError(
        "\nA target-opening ledger already exists:\n"
        f"{OPENING_LEDGER}\n\n"
        "DO NOT rerun target opening #6 blindly."
    )


if OUT.exists() and any(
    OUT.iterdir()
):

    raise RuntimeError(
        "\nStage24-5B artifacts already exist.\n"
        "Refusing duplicate target opening."
    )


print(
    "[PASS] Repository clean."
)

print(
    "[PASS] No unresolved scientific-fit ledger."
)

print(
    "[PASS] No unresolved target-opening ledger."
)

print(
    "[PASS] Stage24-5B has not previously been opened."
)

print()


# ==============================================================================
# 4. VERIFY STAGE24-5A TARGET REFERENCE
# ==============================================================================

print("=" * 118)
print("STAGE24-5A TARGET REFERENCE")
print("=" * 118)


stage5a_result_sha = verify_sidecar(
    B62_TARGET_RESULT_PATH,
    B62_TARGET_RESULT_SHA_PATH,
)


if stage5a_result_sha != EXPECTED_STAGE24_5A_RESULT_SHA:

    raise RuntimeError(
        "Stage24-5A result SHA changed."
    )


stage5a_pred_file_sha = sha256_file(
    B62_TARGET_PRED_PATH
)


if stage5a_pred_file_sha != EXPECTED_STAGE24_5A_PRED_FILE_SHA:

    raise RuntimeError(
        "\nStage24-5A prediction artifact SHA changed.\n"
        f"Expected: {EXPECTED_STAGE24_5A_PRED_FILE_SHA}\n"
        f"Actual:   {stage5a_pred_file_sha}"
    )


stage5a = json.loads(
    B62_TARGET_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    stage5a[
        "status"
    ]
    !=
    "SECONDARY_BRIDGE62_IDS2018_FEB28_TARGET_OPENING_COMPLETE"
):

    raise RuntimeError(
        "Stage24-5A status changed."
    )


if int(
    stage5a[
        "opening_accounting"
    ][
        "consumed"
    ]
) != OPENINGS_BEFORE:

    raise RuntimeError(
        "Stage24-5A opening accounting changed."
    )


if int(
    stage5a[
        "opening_accounting"
    ][
        "remaining_evaluable"
    ]
) != 1:

    raise RuntimeError(
        "Stage24-5A does not authorize exactly one final evaluable opening."
    )


if (
    stage5a[
        "opening_accounting"
    ][
        "next_evaluable_cell"
    ]
    !=
    "SECONDARY_BRIDGE70_IDS2018_FEB28"
):

    raise RuntimeError(
        "Stage24-5A next cell changed."
    )


if (
    stage5a[
        "representation"
    ][
        "target_70f_matrix_sha256"
    ]
    !=
    EXPECTED_TARGET_70F_SHA
):

    raise RuntimeError(
        "Stage24-5A frozen 70F target SHA changed."
    )


print(
    "Result SHA:",
    stage5a_result_sha,
)

print(
    "Prediction file SHA:",
    stage5a_pred_file_sha,
)

print()

print(
    "[PASS] Opening #5 durable."
)

print(
    "[PASS] Opening #6 is the only remaining evaluable target cell."
)

print()


# ==============================================================================
# 5. LOAD EXACT CANONICAL ROW MEMBERSHIP FROM OPENING #5
# ==============================================================================

print("=" * 118)
print("CANONICAL IDS2018 FEB-28 MEMBERSHIP")
print("=" * 118)


ref_npz = np.load(
    B62_TARGET_PRED_PATH,
    allow_pickle=False,
)


required_keys = {
    "probability",
    "binary_label",
    "original_row_index",
    "clean_position",
}


if set(
    ref_npz.files
) != required_keys:

    raise RuntimeError(
        "\nUnexpected Stage24-5A NPZ keys.\n"
        f"Expected: {sorted(required_keys)}\n"
        f"Actual:   {sorted(ref_npz.files)}"
    )


p62_reference = np.asarray(
    ref_npz[
        "probability"
    ],
    dtype=np.float32,
)


y_target = np.asarray(
    ref_npz[
        "binary_label"
    ],
    dtype=np.uint8,
)


original_row_index = np.asarray(
    ref_npz[
        "original_row_index"
    ],
    dtype=np.int64,
)


clean_position = np.asarray(
    ref_npz[
        "clean_position"
    ],
    dtype=np.int64,
)


ref_npz.close()


for name, arr in [
    (
        "probability",
        p62_reference,
    ),
    (
        "binary_label",
        y_target,
    ),
    (
        "original_row_index",
        original_row_index,
    ),
    (
        "clean_position",
        clean_position,
    ),
]:

    if arr.shape != (
        EXPECTED_TARGET_ROWS,
    ):

        raise RuntimeError(
            f"{name} shape mismatch: {arr.shape}"
        )


p62_sha = sha256_array(
    p62_reference
)


target_label_sha = sha256_array(
    y_target
)


row_index_sha = sha256_array(
    original_row_index
)


clean_position_sha = sha256_array(
    clean_position
)


if p62_sha != EXPECTED_STAGE24_5A_PROB_SHA:

    raise RuntimeError(
        "Opening #5 probability-array SHA changed."
    )


if target_label_sha != EXPECTED_TARGET_LABEL_SHA:

    raise RuntimeError(
        "Canonical target binary-label SHA changed."
    )


if row_index_sha != EXPECTED_ORIGINAL_ROW_INDEX_SHA:

    raise RuntimeError(
        "Canonical original-row-index SHA changed."
    )


if clean_position_sha != EXPECTED_CLEAN_POSITION_SHA:

    raise RuntimeError(
        "Canonical clean-position SHA changed."
    )


if not np.array_equal(
    clean_position,
    np.arange(
        EXPECTED_CLEAN_START,
        EXPECTED_CLEAN_STOP,
        dtype=np.int64,
    ),
):

    raise RuntimeError(
        "Canonical clean_position sequence changed."
    )


if (
    original_row_index.min()
    <
    0
    or
    original_row_index.max()
    >=
    EXPECTED_PHYSICAL_ROWS
):

    raise RuntimeError(
        "Canonical physical-row locator outside Feb-28 range."
    )


if (
    len(
        original_row_index
    )
    >
    1
    and
    not np.all(
        original_row_index[
            1:
        ]
        >
        original_row_index[
            :-1
        ]
    )
):

    raise RuntimeError(
        "Canonical original-row-index sequence is not strictly increasing."
    )


target_attack = int(
    y_target.sum()
)


target_benign = int(
    EXPECTED_TARGET_ROWS
    -
    target_attack
)


if target_benign != EXPECTED_TARGET_BENIGN:

    raise RuntimeError(
        "Canonical target benign count changed."
    )


if target_attack != EXPECTED_TARGET_ATTACK:

    raise RuntimeError(
        "Canonical target attack count changed."
    )


print(
    "Rows:            ",
    f"{EXPECTED_TARGET_ROWS:,}",
)

print(
    "Benign:          ",
    f"{target_benign:,}",
)

print(
    "Attack:          ",
    f"{target_attack:,}",
)

print()

print(
    "Opening #5 probability SHA:",
    p62_sha,
)

print(
    "Binary-label SHA:          ",
    target_label_sha,
)

print(
    "Original-row-index SHA:    ",
    row_index_sha,
)

print(
    "Clean-position SHA:        ",
    clean_position_sha,
)

print()

print(
    "[PASS] Exact Stage24-5A target membership/order recovered."
)

print()


# ==============================================================================
# 6. VERIFY BRIDGE70 SOURCE MODEL + OPERATING POINTS
# ==============================================================================

print("=" * 118)
print("FROZEN BRIDGE70 SECONDARY MODEL")
print("=" * 118)


b70_result_sha = verify_sidecar(
    B70_RESULT_PATH,
    B70_RESULT_SHA_PATH,
)


if b70_result_sha != EXPECTED_BRIDGE70_RESULT_SHA:

    raise RuntimeError(
        "\nBridge70 source-fit receipt SHA changed.\n"
        f"Expected: {EXPECTED_BRIDGE70_RESULT_SHA}\n"
        f"Actual:   {b70_result_sha}"
    )


b70_result = json.loads(
    B70_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


model_sha = sha256_file(
    B70_MODEL_PATH
)


if model_sha != EXPECTED_BRIDGE70_MODEL_SHA:

    raise RuntimeError(
        "\nBridge70 model SHA changed.\n"
        f"Expected: {EXPECTED_BRIDGE70_MODEL_SHA}\n"
        f"Actual:   {model_sha}"
    )


if (
    b70_result[
        "status"
    ]
    !=
    "SECONDARY_BRIDGE70_SOURCE_MODEL_FROZEN"
):

    raise RuntimeError(
        "Bridge70 source-fit status changed."
    )


if int(
    b70_result[
        "fit_accounting"
    ][
        "scientific_fits_completed"
    ]
) != SCIENTIFIC_FITS:

    raise RuntimeError(
        "Scientific-fit accounting is not 4/4."
    )


if int(
    b70_result[
        "fit_accounting"
    ][
        "scientific_fits_remaining"
    ]
) != 0:

    raise RuntimeError(
        "Unexpected scientific fits remain."
    )


if int(
    b70_result[
        "fit_accounting"
    ][
        "additional_fits_authorized"
    ]
) != 0:

    raise RuntimeError(
        "Unexpected additional fits authorized."
    )


source_prior = float(
    b70_result[
        "secondary_source_prior_constant_predictor"
    ][
        "probability"
    ]
)


ops = (
    b70_result[
        "source_validation"
    ][
        "operating_points"
    ]
)


STANDARD_THRESHOLD = np.float32(
    ops[
        "standard"
    ][
        "threshold_float32_runtime"
    ]
)


BALANCED_THRESHOLD = np.float32(
    ops[
        "balanced"
    ][
        "threshold_float32_runtime"
    ]
)


if (
    ops[
        "security"
    ][
        "status"
    ]
    !=
    "AVAILABLE"
):

    raise RuntimeError(
        "Frozen bridge70 SECURITY threshold unavailable."
    )


SECURITY_THRESHOLD = np.float32(
    ops[
        "security"
    ][
        "result"
    ][
        "threshold_float32_runtime"
    ]
)


if STANDARD_THRESHOLD != np.float32(
    0.50
):

    raise RuntimeError(
        "Bridge70 standard threshold changed."
    )


if BALANCED_THRESHOLD != np.float32(
    0.18
):

    raise RuntimeError(
        "Bridge70 balanced threshold changed."
    )


if SECURITY_THRESHOLD != np.float32(
    0.05
):

    raise RuntimeError(
        "Bridge70 security threshold changed."
    )


print(
    "Result SHA:",
    b70_result_sha,
)

print(
    "Model SHA: ",
    model_sha,
)

print()

print(
    "STANDARD:",
    float(
        STANDARD_THRESHOLD
    ),
)

print(
    "BALANCED:",
    float(
        BALANCED_THRESHOLD
    ),
)

print(
    "SECURITY:",
    float(
        SECURITY_THRESHOLD
    ),
)

print()

print(
    "[PASS] Bridge70 model and source-only operating points exact."
)

print()


# ==============================================================================
# 7. VERIFY PROTOCOL / CANONICAL 70F ORDER
# ==============================================================================

bridge_sha = verify_sidecar(
    BRIDGE_SPEC_PATH,
    BRIDGE_SPEC_SHA_PATH,
)


lock_sha = verify_sidecar(
    FINAL_LOCK_PATH,
    FINAL_LOCK_SHA_PATH,
)


if bridge_sha != EXPECTED_BRIDGE_SPEC_SHA:

    raise RuntimeError(
        "Semantic bridge SHA changed."
    )


if lock_sha != EXPECTED_PROTOCOL_LOCK_SHA:

    raise RuntimeError(
        "Protocol lock SHA changed."
    )


bridge_spec = json.loads(
    BRIDGE_SPEC_PATH.read_text(
        encoding="utf-8"
    )
)


final_lock = json.loads(
    FINAL_LOCK_PATH.read_text(
        encoding="utf-8"
    )
)


secondary = final_lock[
    "secondary_direction"
]


if (
    secondary[
        "direction"
    ]
    !=
    "CICIDS2017_TO_IDS2018"
):

    raise RuntimeError(
        "Secondary direction changed."
    )


if (
    secondary[
        "target"
    ]
    !=
    "IDS2018_02-28-2018"
):

    raise RuntimeError(
        "Secondary target changed."
    )


FEATURE_ORDER = list(
    bridge_spec[
        "bridge70"
    ][
        "source_feature_order"
    ]
)


if len(
    FEATURE_ORDER
) != 70:

    raise RuntimeError(
        "Frozen bridge70 feature count changed."
    )


if (
    FEATURE_ORDER
    !=
    b70_result[
        "representation"
    ][
        "feature_order"
    ]
):

    raise RuntimeError(
        "Bridge70 canonical target order differs from fitted model order."
    )


print(
    "[PASS] Canonical bridge70 target feature order exact."
)

print()


# ==============================================================================
# 8. REMOTE GATE BEFORE TARGET FEATURE MATERIALIZATION
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )

            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )

        if value and value.strip():

            github_token = value.strip()

            token_source = (
                "ENV:"
                +
                name
            )

            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable BEFORE target opening #6."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved before opening #6.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main == Stage24-5A."
)

print()


# ==============================================================================
# 9. LOCATE BYTE-EXACT IDS2018 FEB-28 RAW SOURCE
# ==============================================================================

print("=" * 118)
print("IDS2018 FEB-28 70F MATERIALIZATION")
print("=" * 118)


preferred_raw = Path(
    "/kaggle/input/datasets/"
    "solarmainframe/ids-intrusion-csv/"
    "02-28-2018.csv"
)


raw_candidates = []


if preferred_raw.is_file():

    raw_candidates.append(
        preferred_raw
    )


for root in [
    Path(
        "/kaggle/input"
    ),
    Path(
        "/kaggle/working"
    ),
]:

    if not root.exists():
        continue

    try:

        for path in root.rglob(
            "02-28-2018.csv"
        ):

            if path.is_file():

                raw_candidates.append(
                    path
                )

    except Exception:
        pass


dedup = []
seen = set()


for path in raw_candidates:

    try:

        key = str(
            path.resolve()
        )

    except Exception:

        key = str(
            path
        )


    if key not in seen:

        seen.add(
            key
        )

        dedup.append(
            path
        )


exact_raw_path = None


for path in dedup:

    digest = sha256_file(
        path
    )


    print(
        "Candidate:",
        path,
    )

    print(
        "  SHA:",
        digest,
    )


    if digest == EXPECTED_RAW_SHA:

        exact_raw_path = path
        break


if exact_raw_path is None:

    raise RuntimeError(
        "\nUnable to locate byte-exact IDS2018 02-28-2018.csv.\n"
        "Target opening #6 has NOT been consumed."
    )


print()
print(
    "[PASS] Byte-exact raw IDS2018 Feb-28 located."
)

print(
    "Using:",
    exact_raw_path,
)

print()


# ==============================================================================
# 10. READ RAW FEB-28 AND SELECT EXACT STAGE24-5A ROW LOCATORS
#
# We intentionally use the same full-read semantics as Stage24-5A to maximize
# byte-for-byte reconstruction fidelity.
# ==============================================================================

t0 = time.time()


raw = pd.read_csv(
    exact_raw_path,
    low_memory=False,
)


raw.columns = [
    str(
        col
    ).strip()
    for col in raw.columns
]


if len(
    set(
        raw.columns
    )
) != len(
    raw.columns
):

    raise RuntimeError(
        "Column-name stripping created duplicate names."
    )


if len(
    raw
) != EXPECTED_PHYSICAL_ROWS:

    raise RuntimeError(
        "\nPhysical Feb-28 row count mismatch.\n"
        f"Expected: {EXPECTED_PHYSICAL_ROWS:,}\n"
        f"Actual:   {len(raw):,}"
    )


required_columns = (
    FEATURE_ORDER
    +
    [
        "Label"
    ]
)


missing = [
    col
    for col in required_columns
    if col not in raw.columns
]


if missing:

    raise RuntimeError(
        "\nRaw Feb-28 missing frozen columns:\n"
        + "\n".join(
            missing
        )
    )


retained = raw.iloc[
    original_row_index
].copy()


if len(
    retained
) != EXPECTED_TARGET_ROWS:

    raise RuntimeError(
        "Exact Stage24-5A retained-row locator count mismatch."
    )


print(
    "Raw read + exact row selection seconds:",
    f"{time.time() - t0:.2f}",
)

print()


# ==============================================================================
# 11. VERIFY ROW LABELS AGAINST STAGE24-5A
# ==============================================================================

labels = canonicalize_labels(
    retained[
        "Label"
    ]
)


y_reconstructed = (
    labels
    !=
    "benign"
).to_numpy(
    dtype=np.uint8,
)


if not np.array_equal(
    y_reconstructed,
    y_target,
):

    mismatch_count = int(
        np.count_nonzero(
            y_reconstructed
            !=
            y_target
        )
    )

    raise RuntimeError(
        "\nPhysical target rows no longer reproduce Stage24-5A labels.\n"
        f"Mismatches: {mismatch_count:,}\n"
        "Opening #6 has NOT been consumed."
    )


print(
    "[PASS] Exact physical rows reproduce Stage24-5A binary labels."
)

print()


# ==============================================================================
# 12. MATERIALIZE 70F TARGET MATRIX
# ==============================================================================

X_target = np.empty(
    (
        EXPECTED_TARGET_ROWS,
        70,
    ),
    dtype=np.float64,
)


for j, feature in enumerate(
    FEATURE_ORDER
):

    source_series = retained[
        feature
    ]


    numeric = pd.to_numeric(
        source_series,
        errors="coerce",
    )


    invalid = (
        numeric.isna()
        &
        source_series.notna()
    )


    if invalid.any():

        invalid_tokens = (
            source_series.loc[
                invalid
            ]
            .astype(
                "string"
            )
            .str.strip()
            .str.casefold()
        )


        truly_invalid = ~invalid_tokens.isin(
            [
                "",
                "nan",
            ]
        )


        if truly_invalid.any():

            examples = (
                invalid_tokens.loc[
                    truly_invalid
                ]
                .drop_duplicates()
                .head(
                    10
                )
                .tolist()
            )


            raise RuntimeError(
                "\nFrozen numeric FAIL_CLOSED rule triggered.\n"
                f"Feature: {feature}\n"
                f"Examples: {examples}\n"
                "Opening #6 has NOT been consumed."
            )


    X_target[
        :,
        j
    ] = numeric.to_numpy(
        dtype=np.float64,
    )


positive_inf = int(
    np.isposinf(
        X_target
    ).sum()
)


negative_inf = int(
    np.isneginf(
        X_target
    ).sum()
)


if positive_inf != EXPECTED_TARGET_70F_POS_INF:

    raise RuntimeError(
        "\n70F +inf count differs from Stage24-5A.\n"
        f"Expected: {EXPECTED_TARGET_70F_POS_INF:,}\n"
        f"Actual:   {positive_inf:,}"
    )


if negative_inf != EXPECTED_TARGET_70F_NEG_INF:

    raise RuntimeError(
        "\n70F -inf count differs from Stage24-5A."
    )


inf_mask = np.isinf(
    X_target
)


if inf_mask.any():

    X_target[
        inf_mask
    ] = np.nan


nan_count = int(
    np.isnan(
        X_target
    ).sum()
)


if nan_count != EXPECTED_TARGET_70F_NAN:

    raise RuntimeError(
        "\n70F final NaN count differs from Stage24-5A.\n"
        f"Expected: {EXPECTED_TARGET_70F_NAN:,}\n"
        f"Actual:   {nan_count:,}"
    )


target_70f_sha = sha256_array(
    X_target
)


print("=" * 118)
print("70F TARGET IDENTITY")
print("=" * 118)

print(
    "Shape:",
    X_target.shape,
)

print(
    "+inf -> NaN:",
    f"{positive_inf:,}",
)

print(
    "-inf -> NaN:",
    f"{negative_inf:,}",
)

print(
    "Final NaN cells:",
    f"{nan_count:,}",
)

print()

print(
    "70F matrix SHA:",
    target_70f_sha,
)

print()


if target_70f_sha != EXPECTED_TARGET_70F_SHA:

    raise RuntimeError(
        "\n70F target matrix does not reproduce Stage24-5A.\n"
        f"Expected: {EXPECTED_TARGET_70F_SHA}\n"
        f"Actual:   {target_70f_sha}\n"
        "Opening #6 has NOT been consumed."
    )


print(
    "[PASS] 70F matrix is byte-identical to the matrix independently frozen in Stage24-5A."
)

print()


del raw
del retained
del labels
del y_reconstructed
del inf_mask

gc.collect()


# ==============================================================================
# 13. LOAD FROZEN BRIDGE70 MODEL
#
# Loading does NOT consume target opening #6.
# ==============================================================================

import xgboost as xgb


print("=" * 118)
print("MODEL LOAD")
print("=" * 118)


print(
    "XGBoost:",
    xgb.__version__,
)


booster = xgb.Booster()


booster.load_model(
    str(
        B70_MODEL_PATH
    )
)


booster.set_param(
    {
        "device":
            "cpu",

        "nthread":
            -1,
    }
)


if booster.num_features() != 70:

    raise RuntimeError(
        "Frozen bridge70 model dimension mismatch."
    )


if booster.num_boosted_rounds() != 400:

    raise RuntimeError(
        "Frozen bridge70 model round count mismatch."
    )


print(
    "[PASS] Frozen bridge70 secondary model loaded."
)

print()


# ==============================================================================
# 14. PREPARE DMATRIX
#
# Creating the DMatrix is target preprocessing, but the frozen opening
# definition is triggered when target feature values are supplied to the model.
# The ledger is written immediately before booster.predict().
# ==============================================================================

dtarget = xgb.DMatrix(
    X_target,
    missing=np.nan,
)


print(
    "[PASS] Target DMatrix ready."
)

print(
    "[PASS] No target prediction created yet."
)

print()


# ==============================================================================
# 15. TARGET OPENING #6
# ==============================================================================

print("=" * 118)
print("SECONDARY TARGET INFERENCE")
print("=" * 118)

print()
print(
    "The next booster.predict() consumes SECONDARY TARGET OPENING #6."
)

print()


opening_started_at = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


OPENING_LEDGER.write_text(
    json.dumps(
        {
            "stage":
                "Stage24-5B",

            "direction":
                "CICIDS2017_TO_IDS2018",

            "bridge":
                "bridge70",

            "target":
                "IDS2018_02-28-2018",

            "opening_number":
                6,

            "opening_consumed":
                True,

            "opening_consumed_at_utc":
                opening_started_at,

            "trigger":
                (
                    "FIRST_IDS2018_FEB28_BRIDGE70_TARGET_FEATURE_VALUES_"
                    "SUPPLIED_TO_FROZEN_SECONDARY_XGBOOST"
                ),

            "parent_commit":
                EXPECTED_PARENT,

            "scientific_fits_completed":
                4,

            "scientific_fits_remaining":
                0,

            "additional_fits_authorized":
                0,

            "previous_openings_consumed":
                5,

            "administratively_cancelled":
                2,
        },
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


print(
    ">>> TARGET OPENING #6 CONSUMED <<<"
)

print()


inference_start = time.time()


p_target = np.asarray(
    booster.predict(
        dtarget,
        validate_features=False,
    ),
    dtype=np.float32,
)


inference_seconds = (
    time.time()
    -
    inference_start
)


if p_target.shape != (
    EXPECTED_TARGET_ROWS,
):

    raise RuntimeError(
        "Target probability shape mismatch."
    )


if not np.isfinite(
    p_target
).all():

    raise RuntimeError(
        "Non-finite target probability encountered."
    )


if (
    (p_target < 0).any()
    or
    (p_target > 1).any()
):

    raise RuntimeError(
        "Target probability outside [0,1]."
    )


probability_sha = sha256_array(
    p_target
)


print(
    "Inference seconds:",
    f"{inference_seconds:.3f}",
)

print(
    "Probability min/max:",
    f"{float(p_target.min()):.9f}",
    "/",
    f"{float(p_target.max()):.9f}",
)

print(
    "Probability SHA:",
    probability_sha,
)

print()


del dtarget
del X_target

gc.collect()


# ==============================================================================
# 16. TARGET METRICS
# ==============================================================================

target_prevalence = float(
    EXPECTED_TARGET_ATTACK
    /
    EXPECTED_TARGET_ROWS
)


pr_auc = float(
    average_precision_score(
        y_target,
        p_target,
    )
)


roc_auc = float(
    roc_auc_score(
        y_target,
        p_target,
    )
)


pr_excess = float(
    pr_auc
    -
    target_prevalence
)


pr_normalized = float(
    (
        pr_auc
        -
        target_prevalence
    )
    /
    (
        1.0
        -
        target_prevalence
    )
)


p64 = p_target.astype(
    np.float64,
    copy=False,
)


y64 = y_target.astype(
    np.float64,
    copy=False,
)


brier = float(
    np.mean(
        (
            p64
            -
            y64
        ) ** 2
    )
)


clipped = np.clip(
    p64,
    1e-15,
    1.0 - 1e-15,
)


log_loss = float(
    -np.mean(
        y64
        *
        np.log(
            clipped
        )
        +
        (
            1.0
            -
            y64
        )
        *
        np.log(
            1.0
            -
            clipped
        )
    )
)


bins = np.floor(
    p64
    *
    10.0
).astype(
    np.int8,
)


bins = np.clip(
    bins,
    0,
    9,
)


ece = 0.0

ece_bins = []


for b in range(
    10
):

    mask = (
        bins
        ==
        b
    )


    count = int(
        mask.sum()
    )


    if count == 0:

        ece_bins.append(
            {
                "bin":
                    b,

                "count":
                    0,

                "mean_probability":
                    None,

                "empirical_rate":
                    None,

                "absolute_gap":
                    None,
            }
        )

        continue


    confidence = float(
        p64[
            mask
        ].mean()
    )


    empirical = float(
        y64[
            mask
        ].mean()
    )


    gap = abs(
        confidence
        -
        empirical
    )


    ece += (
        count
        /
        EXPECTED_TARGET_ROWS
    ) * gap


    ece_bins.append(
        {
            "bin":
                b,

            "count":
                count,

            "mean_probability":
                confidence,

            "empirical_rate":
                empirical,

            "absolute_gap":
                float(
                    gap
                ),
        }
    )


ece = float(
    ece
)


print("=" * 118)
print("SECONDARY TARGET METRICS — bridge70")
print("=" * 118)


print(
    "Target prevalence:",
    f"{target_prevalence:.15f}",
)

print(
    "PR-AUC:          ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:         ",
    f"{roc_auc:.15f}",
)

print(
    "PR excess:       ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized:   ",
    f"{pr_normalized:.15f}",
)

print()

print(
    "Brier:           ",
    f"{brier:.15f}",
)

print(
    "Log loss:        ",
    f"{log_loss:.15f}",
)

print(
    "ECE-10:          ",
    f"{ece:.15f}",
)

print()


# ==============================================================================
# 17. APPLY ONLY SOURCE-VALIDATION THRESHOLDS
# ==============================================================================

def threshold_metrics(
    threshold,
):

    threshold = np.float32(
        threshold
    )


    pred = (
        p_target
        >=
        threshold
    )


    y1 = (
        y_target
        ==
        1
    )


    y0 = ~y1


    tp = int(
        np.sum(
            pred
            &
            y1
        )
    )


    fp = int(
        np.sum(
            pred
            &
            y0
        )
    )


    fn = int(
        EXPECTED_TARGET_ATTACK
        -
        tp
    )


    tn = int(
        EXPECTED_TARGET_BENIGN
        -
        fp
    )


    precision = (
        tp
        /
        (
            tp
            +
            fp
        )
        if (
            tp + fp
        ) > 0
        else 0.0
    )


    recall = (
        tp
        /
        (
            tp
            +
            fn
        )
        if (
            tp + fn
        ) > 0
        else 0.0
    )


    fpr = (
        fp
        /
        (
            fp
            +
            tn
        )
        if (
            fp + tn
        ) > 0
        else 0.0
    )


    fnr = (
        fn
        /
        (
            fn
            +
            tp
        )
        if (
            fn + tp
        ) > 0
        else 0.0
    )


    f1_den = (
        2
        *
        tp
        +
        fp
        +
        fn
    )


    f2_den = (
        5
        *
        tp
        +
        4
        *
        fn
        +
        fp
    )


    return {
        "threshold":
            float(
                threshold
            ),

        "tp":
            tp,

        "tn":
            tn,

        "fp":
            fp,

        "fn":
            fn,

        "accuracy":
            float(
                (
                    tp + tn
                )
                /
                EXPECTED_TARGET_ROWS
            ),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                (
                    2
                    *
                    tp
                    /
                    f1_den
                )
                if f1_den
                else 0.0
            ),

        "f2":
            float(
                (
                    5
                    *
                    tp
                    /
                    f2_den
                )
                if f2_den
                else 0.0
            ),

        "fpr":
            float(
                fpr
            ),

        "fnr":
            float(
                fnr
            ),
    }


standard = threshold_metrics(
    STANDARD_THRESHOLD
)


balanced = threshold_metrics(
    BALANCED_THRESHOLD
)


security = threshold_metrics(
    SECURITY_THRESHOLD
)


print("=" * 118)
print("SOURCE-VALIDATION THRESHOLDS APPLIED TO TARGET")
print("=" * 118)


for name, metrics in [
    (
        "STANDARD",
        standard,
    ),
    (
        "BALANCED",
        balanced,
    ),
    (
        "SECURITY",
        security,
    ),
]:

    print(
        name,
        "@",
        metrics[
            "threshold"
        ],
    )


    print(
        "  TP/TN/FP/FN:",
        metrics[
            "tp"
        ],
        metrics[
            "tn"
        ],
        metrics[
            "fp"
        ],
        metrics[
            "fn"
        ],
    )


    print(
        "  precision/recall/F1/F2/FPR/FNR:",
        f"{metrics['precision']:.9f}",
        f"{metrics['recall']:.9f}",
        f"{metrics['f1']:.9f}",
        f"{metrics['f2']:.9f}",
        f"{metrics['fpr']:.9f}",
        f"{metrics['fnr']:.9f}",
    )


    print()


# ==============================================================================
# 18. FROZEN NULL ANCHORS
# ==============================================================================

target_prior_brier = float(
    np.mean(
        (
            np.float64(
                target_prevalence
            )
            -
            y64
        ) ** 2
    )
)


source_prior_brier = float(
    np.mean(
        (
            np.float64(
                source_prior
            )
            -
            y64
        ) ** 2
    )
)


null_anchors = {
    "target_prevalence_chance_anchor": {
        "probability":
            target_prevalence,

        "pr_auc_anchor":
            target_prevalence,

        "roc_auc_anchor":
            0.5,

        "brier":
            target_prior_brier,
    },

    "secondary_source_prior_constant_predictor": {
        "probability":
            source_prior,

        "fit_on_target":
            False,

        "probability_source":
            "CICIDS2017_MONDAY_WEDNESDAY_SOURCE_TRAIN_ONLY",

        "pr_auc":
            target_prevalence,

        "roc_auc":
            0.5,

        "brier":
            source_prior_brier,
    },
}


# ==============================================================================
# 19. PERSIST OPENING #6
# ==============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


np.savez_compressed(
    PRED_PATH,

    probability=
        p_target,

    binary_label=
        y_target,

    original_row_index=
        original_row_index,

    clean_position=
        clean_position,
)


pred_file_sha = sha256_file(
    PRED_PATH
)


# Ensure paired-comparison locators/labels are still byte-identical.
saved = np.load(
    PRED_PATH,
    allow_pickle=False,
)


if sha256_array(
    np.asarray(
        saved[
            "binary_label"
        ],
        dtype=np.uint8,
    )
) != EXPECTED_TARGET_LABEL_SHA:

    raise RuntimeError(
        "Persisted opening #6 label SHA mismatch."
    )


if sha256_array(
    np.asarray(
        saved[
            "original_row_index"
        ],
        dtype=np.int64,
    )
) != EXPECTED_ORIGINAL_ROW_INDEX_SHA:

    raise RuntimeError(
        "Persisted opening #6 row-index SHA mismatch."
    )


if sha256_array(
    np.asarray(
        saved[
            "clean_position"
        ],
        dtype=np.int64,
    )
) != EXPECTED_CLEAN_POSITION_SHA:

    raise RuntimeError(
        "Persisted opening #6 clean-position SHA mismatch."
    )


saved.close()


print("=" * 118)
print("PERSISTED TARGET OPENING #6")
print("=" * 118)


print(
    "Prediction artifact:",
    PRED_PATH.relative_to(
        REPO
    ),
)

print(
    "Bytes:",
    f"{PRED_PATH.stat().st_size:,}",
)

print(
    "File SHA:",
    pred_file_sha,
)

print(
    "Probability SHA:",
    probability_sha,
)

print()


# ==============================================================================
# 20. RESULT RECEIPT
# ==============================================================================

result = {
    "stage":
        "Stage24-5B",

    "status":
        "SECONDARY_BRIDGE70_IDS2018_FEB28_TARGET_OPENING_COMPLETE",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "direction":
        "CICIDS2017_TO_IDS2018",

    "bridge":
        "bridge70",

    "target":
        "IDS2018_02-28-2018",

    "parent_commit":
        EXPECTED_PARENT,

    "target_opening": {
        "opening_number":
            6,

        "original_budget":
            8,

        "consumed_before":
            OPENINGS_BEFORE,

        "consumed_after":
            OPENINGS_AFTER,

        "administratively_cancelled":
            2,

        "remaining_evaluable":
            0,

        "opening_consumed_at_utc":
            opening_started_at,

        "new_scientific_fits":
            0,
    },

    "scientific_fit_accounting": {
        "budget":
            4,

        "completed":
            4,

        "remaining":
            0,

        "additional_fits_authorized":
            0,
    },

    "model": {
        "learner":
            "XGBOOST_ONLY",

        "source_domain":
            "CICIDS2017",

        "source_semantics":
            "FLAG_CORRECTED",

        "bridge":
            "bridge70",

        "model_file":
            str(
                B70_MODEL_PATH.relative_to(
                    REPO
                )
            ),

        "model_sha256":
            model_sha,

        "execution_device":
            "cpu",

        "feature_count":
            70,

        "boosted_rounds":
            400,
    },

    "target_population": {
        "dataset":
            "CSE-CIC-IDS2018",

        "file":
            "02-28-2018.csv",

        "raw_source_sha256":
            EXPECTED_RAW_SHA,

        "rows":
            EXPECTED_TARGET_ROWS,

        "benign":
            target_benign,

        "attack":
            target_attack,

        "prevalence":
            target_prevalence,

        "membership_source":
            (
                "STAGE24_5A_PERSISTED_CANONICAL_ROW_LOCATORS"
            ),

        "stage24_5a_prediction_artifact_sha256":
            stage5a_pred_file_sha,

        "binary_label_sha256":
            target_label_sha,

        "original_row_index_sha256":
            row_index_sha,

        "clean_position_sha256":
            clean_position_sha,

        "canonical_row_order":
            (
                "clean_position ASC; day_id=7; "
                "original_zero_based_row_index ASC after frozen K79 exclusions"
            ),
    },

    "representation": {
        "feature_count":
            70,

        "feature_order":
            FEATURE_ORDER,

        "input_dtype":
            "float64",

        "target_70f_matrix_sha256":
            target_70f_sha,

        "stage24_5a_expected_target_70f_matrix_sha256":
            EXPECTED_TARGET_70F_SHA,

        "stage24_5a_70f_matrix_bitwise_identity":
            True,

        "positive_infinity_to_nan":
            positive_inf,

        "negative_infinity_to_nan":
            negative_inf,

        "nan_cells_after_conversion":
            nan_count,

        "explicit_imputation":
            "NONE",

        "scaling":
            "NONE",
    },

    "prediction": {
        "storage_dtype":
            "float32",

        "probability_array_sha256":
            probability_sha,

        "inference_seconds":
            inference_seconds,

        "artifact":
            str(
                PRED_PATH.relative_to(
                    REPO
                )
            ),

        "artifact_sha256":
            pred_file_sha,
    },

    "metrics": {
        "primary": {
            "pr_auc":
                pr_auc,

            "roc_auc":
                roc_auc,

            "prevalence_chance_anchor":
                target_prevalence,

            "pr_excess":
                pr_excess,

            "pr_normalized":
                pr_normalized,
        },

        "calibration": {
            "brier":
                brier,

            "log_loss":
                log_loss,

            "ece_10":
                ece,

            "ece_bins":
                ece_bins,
        },

        "thresholded": {
            "standard":
                standard,

            "balanced":
                balanced,

            "security":
                security,
        },

        "null_anchors":
            null_anchors,
    },

    "threshold_provenance": {
        "selection_dataset":
            "CICIDS2017_THURSDAY_SOURCE_VALIDATION",

        "standard":
            float(
                STANDARD_THRESHOLD
            ),

        "balanced":
            float(
                BALANCED_THRESHOLD
            ),

        "security":
            float(
                SECURITY_THRESHOLD
            ),

        "IDS2018_target_threshold_search":
            False,
    },

    "paired_secondary_comparison_readiness": {
        "bridge62_artifact":
            str(
                B62_TARGET_PRED_PATH.relative_to(
                    REPO
                )
            ),

        "bridge62_artifact_sha256":
            stage5a_pred_file_sha,

        "bridge62_probability_sha256":
            p62_sha,

        "bridge70_artifact":
            str(
                PRED_PATH.relative_to(
                    REPO
                )
            ),

        "bridge70_artifact_sha256":
            pred_file_sha,

        "bridge70_probability_sha256":
            probability_sha,

        "binary_labels_identical":
            True,

        "original_row_locators_identical":
            True,

        "clean_positions_identical":
            True,

        "paired_rows":
            EXPECTED_TARGET_ROWS,

        "bootstrap_performed":
            False,

        "frozen_bootstrap_replicates":
            2000,

        "frozen_bootstrap_seed":
            42,
    },

    "anti_adaptation": {
        "new_fit":
            False,

        "target_guided_refit":
            False,

        "target_threshold_search":
            False,

        "target_feature_search":
            False,

        "target_feature_remapping":
            False,

        "target_imputation_fit":
            False,

        "target_scaling_fit":
            False,

        "target_calibration":
            False,

        "performance_based_change":
            False,
    },

    "opening_accounting": {
        "budget":
            8,

        "consumed":
            6,

        "administratively_cancelled":
            2,

        "remaining_evaluable":
            0,

        "unused_cancelled_slots_reallocated":
            False,
    },

    "next_authorized_step":
        (
            "STAGE24_6_SECONDARY_PAIRED_BOOTSTRAP_AND_FINAL_SYNTHESIS"
        ),
}


RESULT_PATH.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


result_sha = sha256_file(
    RESULT_PATH
)


RESULT_SHA_PATH.write_text(
    f"{result_sha}  {RESULT_PATH.name}\n",
    encoding="utf-8",
)


CHECKSUMS_PATH.write_text(
    (
        f"{result_sha}  {RESULT_PATH.name}\n"
        f"{pred_file_sha}  {PRED_PATH.name}\n"
    ),
    encoding="utf-8",
)


print(
    "Result SHA:",
    result_sha,
)

print()


# ==============================================================================
# 21. GIT AUTHOR SAFETY
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    git_cmd(
        "config",
        "--local",
        "user.name",
        git_cmd(
            "log",
            "-1",
            "--format=%an",
        ),
    )


if not author_email:

    git_cmd(
        "config",
        "--local",
        "user.email",
        git_cmd(
            "log",
            "-1",
            "--format=%ae",
        ),
    )


# ==============================================================================
# 22. GIT FREEZE
# ==============================================================================

print("=" * 118)
print("GIT FREEZE")
print("=" * 118)


git_cmd(
    "add",
    "--",
    str(
        PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


expected_staged = {
    str(
        PRED_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: open secondary bridge70 ids2018 feb28 target",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage24-5B commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 23. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 118)
print("PUSH + REMOTE VERIFY")
print("=" * 118)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository not clean after Stage24-5B:\n"
        + final_status
    )


if OPENING_LEDGER.exists():

    OPENING_LEDGER.unlink()


print(
    "[PASS] Remote main == local Stage24-5B commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 24. FINAL
# ==============================================================================

print("=" * 118)
print("STAGE24-5B SECONDARY TARGET OPENING #6: PASS")
print("=" * 118)

print()

print(
    "Direction:                    CICIDS2017 -> IDS2018"
)

print(
    "Bridge:                       bridge70"
)

print(
    "Target:                       IDS2018 Feb-28"
)

print()

print(
    "Target rows:                  ",
    f"{EXPECTED_TARGET_ROWS:,}",
)

print(
    "Benign:                       ",
    f"{target_benign:,}",
)

print(
    "Attack:                       ",
    f"{target_attack:,}",
)

print(
    "Prevalence:                   ",
    f"{target_prevalence:.12f}",
)

print()

print(
    "PR-AUC:                       ",
    f"{pr_auc:.15f}",
)

print(
    "ROC-AUC:                      ",
    f"{roc_auc:.15f}",
)

print(
    "PR excess:                    ",
    f"{pr_excess:.15f}",
)

print(
    "PR normalized:                ",
    f"{pr_normalized:.15f}",
)

print()

print(
    "Brier:                        ",
    f"{brier:.15f}",
)

print(
    "Log loss:                     ",
    f"{log_loss:.15f}",
)

print(
    "ECE-10:                       ",
    f"{ece:.15f}",
)

print()

print(
    "STANDARD threshold:           ",
    float(
        STANDARD_THRESHOLD
    ),
)

print(
    "BALANCED threshold:           ",
    float(
        BALANCED_THRESHOLD
    ),
)

print(
    "SECURITY threshold:           ",
    float(
        SECURITY_THRESHOLD
    ),
)

print()

print(
    "Scientific fits completed:    4 / 4"
)

print(
    "Scientific fits remaining:    0"
)

print(
    "Additional fits authorized:   0"
)

print()

print(
    "Target openings consumed:     6 / 8"
)

print(
    "Administratively cancelled:   2"
)

print(
    "Remaining evaluable openings: 0"
)

print(
    "Cancelled slots reallocated:  NO"
)

print()

print(
    "70F target SHA:"
)

print(
    " ",
    target_70f_sha,
)

print()

print(
    "Probability SHA:"
)

print(
    " ",
    probability_sha,
)

print()

print(
    "Result SHA:"
)

print(
    " ",
    result_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "ALL PREREGISTERED EVALUABLE TARGET OPENINGS COMPLETE."
)

print()

print(
    "NEXT:"
)

print(
    "  Stage24-6 — secondary paired bridge62-vs-bridge70 bootstrap"
)

print(
    "  + final cross-direction Stage24 synthesis."
)

print("=" * 118)

STAGE24-5B — SECONDARY TARGET OPENING #6 — bridge70 / IDS2018 Feb-28

GOVERNANCE
HEAD: e09ff89f582f972a9641705f0850a9ba99516902
[PASS] Repository clean.
[PASS] No unresolved scientific-fit ledger.
[PASS] No unresolved target-opening ledger.
[PASS] Stage24-5B has not previously been opened.

STAGE24-5A TARGET REFERENCE
Result SHA: 54ea37bd6cc5ea82b77762ee672afcecf9584d6fb115187df9040cebac715515
Prediction file SHA: a4fdb7c0d8b7f92065664d879c62174b854a2d8832cac627c985458c5f857671

[PASS] Opening #5 durable.
[PASS] Opening #6 is the only remaining evaluable target cell.

CANONICAL IDS2018 FEB-28 MEMBERSHIP
Rows:             593,780
Benign:           531,524
Attack:           62,256

Opening #5 probability SHA: 4c7cc6af61b8c4813c9241d526b6a473e46e4f69a058305eceb7fb4c30621640
Binary-label SHA:           17369d505c1ce11f425a7c14c7552c95b39aad3492ae643c31f43a8a66a35177
Original-row-index SHA:     374de0e9d74040581ddc5977fd4c78f8f8959df610d20d1775415f10f3e92dfc
Clean-position SHA:         d8a0

In [43]:
# ==============================================================================
# STAGE24-6 — SECONDARY PAIRED BOOTSTRAP + FINAL CROSS-DIRECTION SYNTHESIS
#
# FINAL ANALYTICAL FREEZE FOR STAGE24
#
# Direction 1 already frozen:
#   IDS2018 -> CICIDS2017
#
# Direction 2 target vectors now frozen:
#   CICIDS2017 -> IDS2018
#       bridge62
#       bridge70
#
# THIS CELL:
#   - performs NO model inference
#   - performs NO model fit
#   - performs NO target opening
#   - performs NO threshold search
#   - performs NO calibration
#   - performs NO feature remapping
#
# Frozen secondary bootstrap:
#   method       = paired stratified row bootstrap
#   strata       = BENIGN / ATTACK
#   replicates   = 2000
#   seed         = 42
#   metrics      = PR-AUC / ROC-AUC / Brier
#   comparison   = bridge70 - bridge62
#   CI           = percentile [2.5%, 97.5%]
#
# Exact computational compression:
#   rows sharing identical
#
#       (binary label,
#        bridge62 float32 probability,
#        bridge70 float32 probability)
#
#   are collapsed into sufficient-statistic groups.
#
# Multinomial resampling of those group multiplicities is EXACTLY equivalent
# to the frozen paired stratified row bootstrap for the frozen metrics.
#
# FINAL GOVERNANCE:
#   scientific fits        = 4 / 4 COMPLETE
#   target openings        = 6 / 8 CONSUMED
#   cancelled GROUNDED_S4  = 2
#   evaluable openings left= 0
#   cancelled reallocation = NO
#
# Directions MUST remain separate.
# NO averaging across directions.
# ==============================================================================

from __future__ import annotations

import os
import gc
import json
import time
import base64
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np


print("=" * 118)
print("STAGE24-6 — SECONDARY PAIRED BOOTSTRAP + FINAL CROSS-DIRECTION SYNTHESIS")
print("=" * 118)
print()


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "ce2913dd7a040b400be58acdacab4017bc4b45f7"
)


# ------------------------------------------------------------------------------
# Governance
# ------------------------------------------------------------------------------

EXPECTED_FITS = 4

EXPECTED_OPENINGS_CONSUMED = 6

EXPECTED_CANCELLED = 2

EXPECTED_REMAINING_EVALUABLE = 0


# ------------------------------------------------------------------------------
# Primary freeze
# ------------------------------------------------------------------------------

EXPECTED_PRIMARY_FREEZE_SHA = (
    "bdb6471be8b154662149eeb616dfdfb78f978dd3f337659e773ad9a21b3ba42f"
)

EXPECTED_PRIMARY_BOOTSTRAP_SHA = (
    "96732023e0a1c9b52a79fafa59ea4b851ef7f8bae1814e4b4b4fde0ef0df09aa"
)


# ------------------------------------------------------------------------------
# Secondary opening #5 — bridge62
# ------------------------------------------------------------------------------

EXPECTED_STAGE5A_RESULT_SHA = (
    "54ea37bd6cc5ea82b77762ee672afcecf9584d6fb115187df9040cebac715515"
)

EXPECTED_STAGE5A_PRED_FILE_SHA = (
    "a4fdb7c0d8b7f92065664d879c62174b854a2d8832cac627c985458c5f857671"
)

EXPECTED_P62_SHA = (
    "4c7cc6af61b8c4813c9241d526b6a473e46e4f69a058305eceb7fb4c30621640"
)


# ------------------------------------------------------------------------------
# Secondary opening #6 — bridge70
# ------------------------------------------------------------------------------

EXPECTED_STAGE5B_RESULT_SHA = (
    "86d5ff8d15f2f55543738c6a8697937643c50d48c3b5d894c2032e3981de5afa"
)

EXPECTED_STAGE5B_PRED_FILE_SHA = (
    "47ce16eeab4d2d059148a68388d624a62f4b524f429eb7dcbb8ba4b21fd0b5c1"
)

EXPECTED_P70_SHA = (
    "badf2e99f7b6a2aad9687ca30a2056271d18b47f089d29093151f6331adf8c3b"
)


# ------------------------------------------------------------------------------
# Canonical secondary target
# ------------------------------------------------------------------------------

EXPECTED_ROWS = 593_780

EXPECTED_BENIGN = 531_524

EXPECTED_ATTACK = 62_256

EXPECTED_LABEL_SHA = (
    "17369d505c1ce11f425a7c14c7552c95b39aad3492ae643c31f43a8a66a35177"
)

EXPECTED_ROW_INDEX_SHA = (
    "374de0e9d74040581ddc5977fd4c78f8f8959df610d20d1775415f10f3e92dfc"
)

EXPECTED_CLEAN_POSITION_SHA = (
    "d8a090f4c1667d62dd81de1ce8be9b0d68062adc2799c67fb7b55ad74a1f8b9b"
)


BOOTSTRAP_REPLICATES = 2000

BOOTSTRAP_SEED = 42

BOOTSTRAP_BATCH = 8


# ==============================================================================
# 1. PATHS
# ==============================================================================

BASE = (
    REPO
    / "results"
    / "stage24_cross_dataset"
)


# ------------------------------------------------------------------------------
# Primary freeze
# ------------------------------------------------------------------------------

PRIMARY_DIR = (
    BASE
    / "stage24_3_primary_results_freeze"
    / "stage24_3a_primary_paired_bootstrap"
)


PRIMARY_RESULT_PATH = (
    PRIMARY_DIR
    / "stage24_3a_primary_results_freeze.json"
)


PRIMARY_RESULT_SHA_PATH = (
    PRIMARY_DIR
    / "stage24_3a_primary_results_freeze.sha256"
)


PRIMARY_BOOTSTRAP_PATH = (
    PRIMARY_DIR
    / "stage24_3a_primary_paired_bootstrap_replicates.npz"
)


# ------------------------------------------------------------------------------
# Secondary bridge62
# ------------------------------------------------------------------------------

B62_DIR = (
    BASE
    / "stage24_5_secondary_target_openings"
    / "stage24_5a_bridge62_ids2018_feb28"
)


B62_RESULT_PATH = (
    B62_DIR
    / "stage24_5a_secondary_bridge62_ids2018_feb28_result.json"
)


B62_RESULT_SHA_PATH = (
    B62_DIR
    / "stage24_5a_secondary_bridge62_ids2018_feb28_result.sha256"
)


B62_PRED_PATH = (
    B62_DIR
    / "secondary_bridge62_ids2018_feb28_predictions.npz"
)


# ------------------------------------------------------------------------------
# Secondary bridge70
# ------------------------------------------------------------------------------

B70_DIR = (
    BASE
    / "stage24_5_secondary_target_openings"
    / "stage24_5b_bridge70_ids2018_feb28"
)


B70_RESULT_PATH = (
    B70_DIR
    / "stage24_5b_secondary_bridge70_ids2018_feb28_result.json"
)


B70_RESULT_SHA_PATH = (
    B70_DIR
    / "stage24_5b_secondary_bridge70_ids2018_feb28_result.sha256"
)


B70_PRED_PATH = (
    B70_DIR
    / "secondary_bridge70_ids2018_feb28_predictions.npz"
)


# ------------------------------------------------------------------------------
# Final output
# ------------------------------------------------------------------------------

OUT = (
    BASE
    / "stage24_6_final_synthesis"
)


BOOTSTRAP_PATH = (
    OUT
    / "stage24_6_secondary_paired_bootstrap_replicates.npz"
)


RESULT_PATH = (
    OUT
    / "stage24_6_final_synthesis.json"
)


RESULT_SHA_PATH = (
    OUT
    / "stage24_6_final_synthesis.sha256"
)


REPORT_PATH = (
    OUT
    / "stage24_6_final_synthesis.md"
)


CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


FIT_LEDGER = Path(
    "/kaggle/working/stage24_secondary_fit_runtime_ledger.json"
)


OPENING_LEDGER = Path(
    "/kaggle/working/stage24_target_opening_runtime_ledger.json"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    cmd = [
        "git"
    ]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [
        str(x)
        for x in args
    ]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_array(
    array,
):

    arr = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        arr.view(
            np.uint8
        )
    ).hexdigest()


def verify_sidecar(
    artifact,
    sidecar,
):

    if not artifact.is_file():

        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )

    if not sidecar.is_file():

        raise RuntimeError(
            f"Missing SHA sidecar:\n{sidecar}"
        )

    actual = sha256_file(
        artifact
    )

    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
        .lower()
    )

    if actual != expected:

        raise RuntimeError(
            "\nSHA mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )

    return actual


# ==============================================================================
# 3. GOVERNANCE
# ==============================================================================

print("=" * 118)
print("GOVERNANCE")
print("=" * 118)


head = git_cmd(
    "rev-parse",
    "HEAD",
)


status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


if FIT_LEDGER.exists():

    raise RuntimeError(
        "\nUnexpected scientific-fit ledger exists:\n"
        f"{FIT_LEDGER}"
    )


if OPENING_LEDGER.exists():

    raise RuntimeError(
        "\nUnexpected target-opening ledger exists:\n"
        f"{OPENING_LEDGER}"
    )


if OUT.exists() and any(
    OUT.iterdir()
):

    raise RuntimeError(
        "\nStage24-6 output already exists.\n"
        "Refusing accidental duplicate final freeze."
    )


print(
    "[PASS] Repository clean."
)

print(
    "[PASS] No unresolved fit ledger."
)

print(
    "[PASS] No unresolved opening ledger."
)

print(
    "[PASS] Stage24-6 has not previously been frozen."
)

print()


# ==============================================================================
# 4. VERIFY PRIMARY DIRECTION FREEZE
# ==============================================================================

print("=" * 118)
print("PRIMARY DIRECTION FREEZE")
print("=" * 118)


primary_result_sha = verify_sidecar(
    PRIMARY_RESULT_PATH,
    PRIMARY_RESULT_SHA_PATH,
)


if primary_result_sha != EXPECTED_PRIMARY_FREEZE_SHA:

    raise RuntimeError(
        "\nPrimary freeze SHA mismatch.\n"
        f"Expected: {EXPECTED_PRIMARY_FREEZE_SHA}\n"
        f"Actual:   {primary_result_sha}"
    )


primary_bootstrap_sha = sha256_file(
    PRIMARY_BOOTSTRAP_PATH
)


if primary_bootstrap_sha != EXPECTED_PRIMARY_BOOTSTRAP_SHA:

    raise RuntimeError(
        "\nPrimary bootstrap SHA mismatch.\n"
        f"Expected: {EXPECTED_PRIMARY_BOOTSTRAP_SHA}\n"
        f"Actual:   {primary_bootstrap_sha}"
    )


primary = json.loads(
    PRIMARY_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    primary[
        "status"
    ]
    !=
    "PRIMARY_DIRECTION_RESULTS_FROZEN"
):

    raise RuntimeError(
        "Primary direction status changed."
    )


if int(
    primary[
        "bootstrap_protocol"
    ][
        "replicates"
    ]
) != 2000:

    raise RuntimeError(
        "Primary bootstrap replicate count changed."
    )


if int(
    primary[
        "bootstrap_protocol"
    ][
        "seed"
    ]
) != 42:

    raise RuntimeError(
        "Primary bootstrap seed changed."
    )


if int(
    primary[
        "opening_accounting"
    ][
        "primary_openings_consumed"
    ]
) != 4:

    raise RuntimeError(
        "Primary opening accounting changed."
    )


if int(
    primary[
        "opening_accounting"
    ][
        "primary_grounded_cells_cancelled_before_opening"
    ]
) != EXPECTED_CANCELLED:

    raise RuntimeError(
        "Primary GROUNDED cancellation accounting changed."
    )


print(
    "Primary result SHA:",
    primary_result_sha,
)

print(
    "Primary bootstrap SHA:",
    primary_bootstrap_sha,
)

print()

print(
    "[PASS] Primary direction remains byte-frozen."
)

print()


# ==============================================================================
# 5. VERIFY SECONDARY TARGET RECEIPTS
# ==============================================================================

print("=" * 118)
print("SECONDARY TARGET RECEIPTS")
print("=" * 118)


b62_result_sha = verify_sidecar(
    B62_RESULT_PATH,
    B62_RESULT_SHA_PATH,
)


b70_result_sha = verify_sidecar(
    B70_RESULT_PATH,
    B70_RESULT_SHA_PATH,
)


if b62_result_sha != EXPECTED_STAGE5A_RESULT_SHA:

    raise RuntimeError(
        "Stage24-5A result SHA mismatch."
    )


if b70_result_sha != EXPECTED_STAGE5B_RESULT_SHA:

    raise RuntimeError(
        "Stage24-5B result SHA mismatch."
    )


b62_pred_file_sha = sha256_file(
    B62_PRED_PATH
)


b70_pred_file_sha = sha256_file(
    B70_PRED_PATH
)


if b62_pred_file_sha != EXPECTED_STAGE5A_PRED_FILE_SHA:

    raise RuntimeError(
        "Stage24-5A prediction artifact SHA mismatch."
    )


if b70_pred_file_sha != EXPECTED_STAGE5B_PRED_FILE_SHA:

    raise RuntimeError(
        "Stage24-5B prediction artifact SHA mismatch."
    )


b62_result = json.loads(
    B62_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


b70_result = json.loads(
    B70_RESULT_PATH.read_text(
        encoding="utf-8"
    )
)


if int(
    b70_result[
        "opening_accounting"
    ][
        "consumed"
    ]
) != EXPECTED_OPENINGS_CONSUMED:

    raise RuntimeError(
        "Final target-opening accounting changed."
    )


if int(
    b70_result[
        "opening_accounting"
    ][
        "administratively_cancelled"
    ]
) != EXPECTED_CANCELLED:

    raise RuntimeError(
        "Final cancellation accounting changed."
    )


if int(
    b70_result[
        "opening_accounting"
    ][
        "remaining_evaluable"
    ]
) != EXPECTED_REMAINING_EVALUABLE:

    raise RuntimeError(
        "Unexpected evaluable target opening remains."
    )


if (
    b70_result[
        "opening_accounting"
    ][
        "unused_cancelled_slots_reallocated"
    ]
    is not False
):

    raise RuntimeError(
        "Cancelled slots were unexpectedly reallocated."
    )


if int(
    b70_result[
        "scientific_fit_accounting"
    ][
        "completed"
    ]
) != EXPECTED_FITS:

    raise RuntimeError(
        "Final scientific-fit accounting changed."
    )


if int(
    b70_result[
        "scientific_fit_accounting"
    ][
        "remaining"
    ]
) != 0:

    raise RuntimeError(
        "Unexpected scientific fit remains."
    )


print(
    "Stage24-5A result SHA:",
    b62_result_sha,
)

print(
    "Stage24-5B result SHA:",
    b70_result_sha,
)

print()

print(
    "[PASS] Scientific fits complete: 4 / 4."
)

print(
    "[PASS] Evaluable target openings complete: 6."
)

print(
    "[PASS] GROUNDED_S4 cancellations: 2."
)

print(
    "[PASS] Cancelled slots not reallocated."
)

print()


# ==============================================================================
# 6. LOAD SECONDARY PAIRED VECTORS
# ==============================================================================

print("=" * 118)
print("SECONDARY PAIRED VECTOR IDENTITY")
print("=" * 118)


npz62 = np.load(
    B62_PRED_PATH,
    allow_pickle=False,
)


npz70 = np.load(
    B70_PRED_PATH,
    allow_pickle=False,
)


EXPECTED_KEYS = {
    "probability",
    "binary_label",
    "original_row_index",
    "clean_position",
}


if set(
    npz62.files
) != EXPECTED_KEYS:

    raise RuntimeError(
        "Unexpected bridge62 NPZ schema."
    )


if set(
    npz70.files
) != EXPECTED_KEYS:

    raise RuntimeError(
        "Unexpected bridge70 NPZ schema."
    )


p62 = np.asarray(
    npz62[
        "probability"
    ],
    dtype=np.float32,
)


p70 = np.asarray(
    npz70[
        "probability"
    ],
    dtype=np.float32,
)


y62 = np.asarray(
    npz62[
        "binary_label"
    ],
    dtype=np.uint8,
)


y70 = np.asarray(
    npz70[
        "binary_label"
    ],
    dtype=np.uint8,
)


row62 = np.asarray(
    npz62[
        "original_row_index"
    ],
    dtype=np.int64,
)


row70 = np.asarray(
    npz70[
        "original_row_index"
    ],
    dtype=np.int64,
)


clean62 = np.asarray(
    npz62[
        "clean_position"
    ],
    dtype=np.int64,
)


clean70 = np.asarray(
    npz70[
        "clean_position"
    ],
    dtype=np.int64,
)


npz62.close()
npz70.close()


for name, array in [
    (
        "bridge62 probability",
        p62,
    ),
    (
        "bridge70 probability",
        p70,
    ),
    (
        "binary label",
        y62,
    ),
    (
        "row locator",
        row62,
    ),
    (
        "clean position",
        clean62,
    ),
]:

    if array.shape != (
        EXPECTED_ROWS,
    ):

        raise RuntimeError(
            f"{name} shape mismatch: {array.shape}"
        )


if not np.array_equal(
    y62,
    y70,
):

    raise RuntimeError(
        "Secondary target labels are not paired identically."
    )


if not np.array_equal(
    row62,
    row70,
):

    raise RuntimeError(
        "Secondary physical row locators differ."
    )


if not np.array_equal(
    clean62,
    clean70,
):

    raise RuntimeError(
        "Secondary clean-position locators differ."
    )


y = y62


p62_sha = sha256_array(
    p62
)


p70_sha = sha256_array(
    p70
)


label_sha = sha256_array(
    y
)


row_sha = sha256_array(
    row62
)


clean_sha = sha256_array(
    clean62
)


if p62_sha != EXPECTED_P62_SHA:

    raise RuntimeError(
        "bridge62 probability SHA changed."
    )


if p70_sha != EXPECTED_P70_SHA:

    raise RuntimeError(
        "bridge70 probability SHA changed."
    )


if label_sha != EXPECTED_LABEL_SHA:

    raise RuntimeError(
        "Secondary binary-label SHA changed."
    )


if row_sha != EXPECTED_ROW_INDEX_SHA:

    raise RuntimeError(
        "Secondary row-locator SHA changed."
    )


if clean_sha != EXPECTED_CLEAN_POSITION_SHA:

    raise RuntimeError(
        "Secondary clean-position SHA changed."
    )


n_attack = int(
    y.sum()
)


n_benign = int(
    EXPECTED_ROWS
    -
    n_attack
)


if n_attack != EXPECTED_ATTACK:

    raise RuntimeError(
        "Secondary attack population changed."
    )


if n_benign != EXPECTED_BENIGN:

    raise RuntimeError(
        "Secondary benign population changed."
    )


print(
    "Rows:       ",
    f"{EXPECTED_ROWS:,}",
)

print(
    "Benign:     ",
    f"{n_benign:,}",
)

print(
    "Attack:     ",
    f"{n_attack:,}",
)

print()

print(
    "bridge62 SHA:",
    p62_sha,
)

print(
    "bridge70 SHA:",
    p70_sha,
)

print(
    "label SHA:   ",
    label_sha,
)

print(
    "row SHA:     ",
    row_sha,
)

print()

print(
    "[PASS] Secondary vectors are exactly paired."
)

print()


# ==============================================================================
# 7. EXACT JOINT SUFFICIENT-STATISTIC GROUPING
# ==============================================================================

print("=" * 118)
print("EXACT SECONDARY BOOTSTRAP COMPRESSION")
print("=" * 118)


PAIR_DTYPE = np.dtype(
    [
        (
            "p62",
            "<u4",
        ),
        (
            "p70",
            "<u4",
        ),
    ]
)


def group_stratum(
    label_value,
):

    mask = (
        y
        ==
        np.uint8(
            label_value
        )
    )


    rows = int(
        mask.sum()
    )


    pairs = np.empty(
        rows,
        dtype=PAIR_DTYPE,
    )


    pairs[
        "p62"
    ] = (
        np.ascontiguousarray(
            p62[
                mask
            ]
        )
        .view(
            np.uint32
        )
    )


    pairs[
        "p70"
    ] = (
        np.ascontiguousarray(
            p70[
                mask
            ]
        )
        .view(
            np.uint32
        )
    )


    t0 = time.time()


    unique_pairs, multiplicity = np.unique(
        pairs,
        return_counts=True,
    )


    seconds = (
        time.time()
        -
        t0
    )


    p62_group = (
        unique_pairs[
            "p62"
        ]
        .astype(
            np.uint32,
            copy=True,
        )
        .view(
            np.float32
        )
    )


    p70_group = (
        unique_pairs[
            "p70"
        ]
        .astype(
            np.uint32,
            copy=True,
        )
        .view(
            np.float32
        )
    )


    multiplicity = multiplicity.astype(
        np.int64,
        copy=False,
    )


    if int(
        multiplicity.sum()
    ) != rows:

        raise RuntimeError(
            "Grouped multiplicity does not reconstruct stratum."
        )


    del pairs
    del unique_pairs
    del mask

    gc.collect()


    return {
        "rows":
            rows,

        "groups":
            len(
                multiplicity
            ),

        "multiplicity":
            multiplicity,

        "p62":
            p62_group,

        "p70":
            p70_group,

        "seconds":
            seconds,
    }


benign_group = group_stratum(
    0
)


attack_group = group_stratum(
    1
)


if benign_group[
    "rows"
] != EXPECTED_BENIGN:

    raise RuntimeError(
        "Benign grouping count changed."
    )


if attack_group[
    "rows"
] != EXPECTED_ATTACK:

    raise RuntimeError(
        "Attack grouping count changed."
    )


G0 = int(
    benign_group[
        "groups"
    ]
)


G1 = int(
    attack_group[
        "groups"
    ]
)


G = (
    G0
    +
    G1
)


print(
    "Benign rows/groups:",
    f"{EXPECTED_BENIGN:,}",
    "/",
    f"{G0:,}",
)

print(
    "Attack rows/groups:",
    f"{EXPECTED_ATTACK:,}",
    "/",
    f"{G1:,}",
)

print(
    "Total exact groups:",
    f"{G:,}",
)

print(
    "Compression ratio:",
    f"{EXPECTED_ROWS / G:.3f}x",
)

print()


# ==============================================================================
# 8. COMBINE GROUPS
# ==============================================================================

group_label = np.concatenate(
    [
        np.zeros(
            G0,
            dtype=np.uint8,
        ),
        np.ones(
            G1,
            dtype=np.uint8,
        ),
    ]
)


group_multiplicity = np.concatenate(
    [
        benign_group[
            "multiplicity"
        ],
        attack_group[
            "multiplicity"
        ],
    ]
)


group_p62 = np.concatenate(
    [
        benign_group[
            "p62"
        ],
        attack_group[
            "p62"
        ],
    ]
).astype(
    np.float32,
    copy=False,
)


group_p70 = np.concatenate(
    [
        benign_group[
            "p70"
        ],
        attack_group[
            "p70"
        ],
    ]
).astype(
    np.float32,
    copy=False,
)


if int(
    group_multiplicity.sum()
) != EXPECTED_ROWS:

    raise RuntimeError(
        "Combined grouping does not reconstruct target population."
    )


prob_benign = (
    benign_group[
        "multiplicity"
    ].astype(
        np.float64
    )
    /
    float(
        EXPECTED_BENIGN
    )
)


prob_attack = (
    attack_group[
        "multiplicity"
    ].astype(
        np.float64
    )
    /
    float(
        EXPECTED_ATTACK
    )
)


prob_benign /= prob_benign.sum()

prob_attack /= prob_attack.sum()


# ==============================================================================
# 9. METRIC CONFIGURATION
# ==============================================================================

def make_metric_config(
    name,
    score,
):

    if score.dtype != np.float32:

        raise RuntimeError(
            f"{name}: probability storage is not float32."
        )


    if not np.isfinite(
        score
    ).all():

        raise RuntimeError(
            f"{name}: non-finite probability."
        )


    order = np.argsort(
        score,
        kind="stable",
    )[
        ::-1
    ]


    sorted_score = score[
        order
    ]


    starts = np.concatenate(
        [
            np.array(
                [
                    0
                ],
                dtype=np.int64,
            ),

            (
                np.flatnonzero(
                    sorted_score[
                        1:
                    ]
                    !=
                    sorted_score[
                        :-1
                    ]
                )
                +
                1
            ).astype(
                np.int64
            ),
        ]
    )


    sorted_label = group_label[
        order
    ].astype(
        np.uint8,
        copy=False,
    )


    score64 = score.astype(
        np.float64,
        copy=False,
    )


    loss = (
        score64
        -
        group_label.astype(
            np.float64,
            copy=False,
        )
    ) ** 2


    print(
        f"{name:12s}"
        f" groups={len(score):9,d}"
        f" distinct_scores={len(starts):9,d}"
    )


    return {
        "name":
            name,

        "order":
            order,

        "starts":
            starts,

        "sorted_label":
            sorted_label,

        "loss":
            loss,
    }


config62 = make_metric_config(
    "bridge62",
    group_p62,
)


config70 = make_metric_config(
    "bridge70",
    group_p70,
)


print()


# ==============================================================================
# 10. EXACT WEIGHTED METRIC ENGINE
# ==============================================================================

def metric_batch(
    bootstrap_group_counts,
    config,
):

    order = config[
        "order"
    ]


    starts = config[
        "starts"
    ]


    sorted_label = config[
        "sorted_label"
    ]


    w = bootstrap_group_counts[
        :,
        order
    ]


    total_by_score = np.add.reduceat(
        w,
        starts,
        axis=1,
    ).astype(
        np.float64,
        copy=False,
    )


    positive_by_score = np.add.reduceat(
        w
        *
        sorted_label[
            None,
            :
        ],
        starts,
        axis=1,
    ).astype(
        np.float64,
        copy=False,
    )


    negative_by_score = (
        total_by_score
        -
        positive_by_score
    )


    cumulative_positive = np.cumsum(
        positive_by_score,
        axis=1,
        dtype=np.float64,
    )


    cumulative_total = np.cumsum(
        total_by_score,
        axis=1,
        dtype=np.float64,
    )


    precision = np.divide(
        cumulative_positive,
        cumulative_total,
        out=np.zeros_like(
            cumulative_positive,
            dtype=np.float64,
        ),
        where=(
            cumulative_total
            >
            0
        ),
    )


    # Non-interpolated AP.
    pr_auc = np.sum(
        precision
        *
        (
            positive_by_score
            /
            float(
                EXPECTED_ATTACK
            )
        ),
        axis=1,
    )


    previous_positive = (
        cumulative_positive
        -
        positive_by_score
    )


    # Distinct-score trapezoidal ROC-AUC.
    roc_auc = np.sum(
        (
            (
                previous_positive
                +
                cumulative_positive
            )
            /
            (
                2.0
                *
                float(
                    EXPECTED_ATTACK
                )
            )
        )
        *
        (
            negative_by_score
            /
            float(
                EXPECTED_BENIGN
            )
        ),
        axis=1,
    )


    brier = (
        bootstrap_group_counts
        @
        config[
            "loss"
        ]
    ) / float(
        EXPECTED_ROWS
    )


    return {
        "pr_auc":
            np.asarray(
                pr_auc,
                dtype=np.float64,
            ),

        "roc_auc":
            np.asarray(
                roc_auc,
                dtype=np.float64,
            ),

        "brier":
            np.asarray(
                brier,
                dtype=np.float64,
            ),
    }


# ==============================================================================
# 11. VERIFY GROUPED POINT ESTIMATES AGAINST DURABLE RESULTS
# ==============================================================================

print("=" * 118)
print("POINT-ESTIMATE VALIDATION")
print("=" * 118)


original_counts = group_multiplicity[
    None,
    :
]


point62_batch = metric_batch(
    original_counts,
    config62,
)


point70_batch = metric_batch(
    original_counts,
    config70,
)


point62 = {
    key:
        float(
            value[
                0
            ]
        )
    for key, value in point62_batch.items()
}


point70 = {
    key:
        float(
            value[
                0
            ]
        )
    for key, value in point70_batch.items()
}


expected62 = {
    "pr_auc":
        float(
            b62_result[
                "metrics"
            ][
                "primary"
            ][
                "pr_auc"
            ]
        ),

    "roc_auc":
        float(
            b62_result[
                "metrics"
            ][
                "primary"
            ][
                "roc_auc"
            ]
        ),

    "brier":
        float(
            b62_result[
                "metrics"
            ][
                "calibration"
            ][
                "brier"
            ]
        ),
}


expected70 = {
    "pr_auc":
        float(
            b70_result[
                "metrics"
            ][
                "primary"
            ][
                "pr_auc"
            ]
        ),

    "roc_auc":
        float(
            b70_result[
                "metrics"
            ][
                "primary"
            ][
                "roc_auc"
            ]
        ),

    "brier":
        float(
            b70_result[
                "metrics"
            ][
                "calibration"
            ][
                "brier"
            ]
        ),
}


def verify_point(
    name,
    actual,
    expected,
    tolerance=5e-12,
):

    print(
        name
    )


    for metric in [
        "pr_auc",
        "roc_auc",
        "brier",
    ]:

        delta = abs(
            actual[
                metric
            ]
            -
            expected[
                metric
            ]
        )


        print(
            f"  {metric:8s}"
            f" grouped={actual[metric]:.15f}"
            f" durable={expected[metric]:.15f}"
            f" |Δ|={delta:.3e}"
        )


        if delta > tolerance:

            raise RuntimeError(
                "\nGrouped metric engine failed durable point-estimate validation.\n"
                f"Vector: {name}\n"
                f"Metric: {metric}\n"
                f"|Δ|: {delta}"
            )


verify_point(
    "bridge62",
    point62,
    expected62,
)


verify_point(
    "bridge70",
    point70,
    expected70,
)


print()
print(
    "[PASS] Grouped metric engine reproduces all six secondary durable metrics."
)

print()


# ==============================================================================
# 12. BOOTSTRAP STORAGE
# ==============================================================================

bridge62_pr = np.empty(
    BOOTSTRAP_REPLICATES,
    dtype=np.float64,
)

bridge62_roc = np.empty(
    BOOTSTRAP_REPLICATES,
    dtype=np.float64,
)

bridge62_brier = np.empty(
    BOOTSTRAP_REPLICATES,
    dtype=np.float64,
)


bridge70_pr = np.empty(
    BOOTSTRAP_REPLICATES,
    dtype=np.float64,
)

bridge70_roc = np.empty(
    BOOTSTRAP_REPLICATES,
    dtype=np.float64,
)

bridge70_brier = np.empty(
    BOOTSTRAP_REPLICATES,
    dtype=np.float64,
)


# ==============================================================================
# 13. EXECUTE FROZEN BOOTSTRAP
# ==============================================================================

print("=" * 118)
print("SECONDARY PAIRED STRATIFIED BOOTSTRAP")
print("=" * 118)


print(
    "Replicates:",
    BOOTSTRAP_REPLICATES,
)

print(
    "Seed:",
    BOOTSTRAP_SEED,
)

print(
    "Pairing:",
    "same sampled multiplicities for bridge62 and bridge70",
)

print(
    "Strata:",
    "BENIGN / ATTACK",
)

print(
    "Implementation:",
    "exact multinomial sufficient-statistic row bootstrap",
)

print()


rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


bootstrap_start = time.time()


for start_rep in range(
    0,
    BOOTSTRAP_REPLICATES,
    BOOTSTRAP_BATCH,
):

    batch = min(
        BOOTSTRAP_BATCH,
        BOOTSTRAP_REPLICATES
        -
        start_rep,
    )


    sampled_benign = rng.multinomial(
        EXPECTED_BENIGN,
        prob_benign,
        size=batch,
    )


    sampled_attack = rng.multinomial(
        EXPECTED_ATTACK,
        prob_attack,
        size=batch,
    )


    counts = np.empty(
        (
            batch,
            G,
        ),
        dtype=np.int64,
    )


    counts[
        :,
        :G0
    ] = sampled_benign


    counts[
        :,
        G0:
    ] = sampled_attack


    if not np.all(
        counts[
            :,
            :G0
        ].sum(
            axis=1
        )
        ==
        EXPECTED_BENIGN
    ):

        raise RuntimeError(
            "Bootstrap benign stratum size changed."
        )


    if not np.all(
        counts[
            :,
            G0:
        ].sum(
            axis=1
        )
        ==
        EXPECTED_ATTACK
    ):

        raise RuntimeError(
            "Bootstrap attack stratum size changed."
        )


    m62 = metric_batch(
        counts,
        config62,
    )


    m70 = metric_batch(
        counts,
        config70,
    )


    stop_rep = (
        start_rep
        +
        batch
    )


    bridge62_pr[
        start_rep:stop_rep
    ] = m62[
        "pr_auc"
    ]


    bridge62_roc[
        start_rep:stop_rep
    ] = m62[
        "roc_auc"
    ]


    bridge62_brier[
        start_rep:stop_rep
    ] = m62[
        "brier"
    ]


    bridge70_pr[
        start_rep:stop_rep
    ] = m70[
        "pr_auc"
    ]


    bridge70_roc[
        start_rep:stop_rep
    ] = m70[
        "roc_auc"
    ]


    bridge70_brier[
        start_rep:stop_rep
    ] = m70[
        "brier"
    ]


    del sampled_benign
    del sampled_attack
    del counts
    del m62
    del m70

    gc.collect()


    if (
        stop_rep % 100 == 0
        or
        stop_rep == BOOTSTRAP_REPLICATES
    ):

        elapsed = (
            time.time()
            -
            bootstrap_start
        )


        rate = (
            stop_rep
            /
            elapsed
            if elapsed
            else 0.0
        )


        print(
            f"  completed {stop_rep:4d} / {BOOTSTRAP_REPLICATES}"
            f"  elapsed={elapsed:8.2f}s"
            f"  rate={rate:6.2f} reps/s"
        )


bootstrap_seconds = (
    time.time()
    -
    bootstrap_start
)


print()
print(
    "[PASS] Frozen 2000-replicate secondary bootstrap complete."
)

print(
    "Seconds:",
    f"{bootstrap_seconds:.2f}",
)

print()


# ==============================================================================
# 14. PAIRED DIFFERENCES
#
# Frozen orientation:
#   bridge70 - bridge62
# ==============================================================================

delta_pr = (
    bridge70_pr
    -
    bridge62_pr
)


delta_roc = (
    bridge70_roc
    -
    bridge62_roc
)


delta_brier = (
    bridge70_brier
    -
    bridge62_brier
)


point_delta = {
    "pr_auc":
        float(
            point70[
                "pr_auc"
            ]
            -
            point62[
                "pr_auc"
            ]
        ),

    "roc_auc":
        float(
            point70[
                "roc_auc"
            ]
            -
            point62[
                "roc_auc"
            ]
        ),

    "brier":
        float(
            point70[
                "brier"
            ]
            -
            point62[
                "brier"
            ]
        ),
}


def percentile_ci(
    values,
):

    low, high = np.percentile(
        values,
        [
            2.5,
            97.5,
        ],
        method="linear",
    )

    return (
        float(
            low
        ),
        float(
            high
        ),
    )


secondary_bootstrap = {}


for metric, values in [
    (
        "pr_auc",
        delta_pr,
    ),
    (
        "roc_auc",
        delta_roc,
    ),
    (
        "brier",
        delta_brier,
    ),
]:

    low, high = percentile_ci(
        values
    )


    secondary_bootstrap[
        metric
    ] = {
        "point_difference":
            point_delta[
                metric
            ],

        "bootstrap_mean_difference":
            float(
                values.mean()
            ),

        "ci_2_5":
            low,

        "ci_97_5":
            high,

        "ci_excludes_zero":
            bool(
                (
                    low > 0.0
                )
                or
                (
                    high < 0.0
                )
            ),
    }


print("=" * 118)
print("SECONDARY PAIRED COMPARISON — bridge70 minus bridge62")
print("=" * 118)


for metric in [
    "pr_auc",
    "roc_auc",
    "brier",
]:

    r = secondary_bootstrap[
        metric
    ]


    print(
        f"{metric:8s}"
        f" Δ={r['point_difference']:+.15f}"
        f"  95% CI=[{r['ci_2_5']:+.15f}, {r['ci_97_5']:+.15f}]"
        f"  excludes zero={r['ci_excludes_zero']}"
    )


print()


# ==============================================================================
# 15. PERSIST RAW SECONDARY BOOTSTRAP
# ==============================================================================

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


np.savez_compressed(
    BOOTSTRAP_PATH,

    replicate_index=
        np.arange(
            BOOTSTRAP_REPLICATES,
            dtype=np.int32,
        ),

    bridge62_pr_auc=
        bridge62_pr,

    bridge62_roc_auc=
        bridge62_roc,

    bridge62_brier=
        bridge62_brier,

    bridge70_pr_auc=
        bridge70_pr,

    bridge70_roc_auc=
        bridge70_roc,

    bridge70_brier=
        bridge70_brier,

    bridge70_minus_bridge62_pr_auc=
        delta_pr,

    bridge70_minus_bridge62_roc_auc=
        delta_roc,

    bridge70_minus_bridge62_brier=
        delta_brier,
)


secondary_bootstrap_sha = sha256_file(
    BOOTSTRAP_PATH
)


print("=" * 118)
print("SECONDARY BOOTSTRAP ARTIFACT")
print("=" * 118)


print(
    "File:",
    BOOTSTRAP_PATH.relative_to(
        REPO
    ),
)

print(
    "Bytes:",
    f"{BOOTSTRAP_PATH.stat().st_size:,}",
)

print(
    "SHA256:",
    secondary_bootstrap_sha,
)

print()


# ==============================================================================
# 16. BUILD FINAL CROSS-DIRECTION SYNTHESIS
# ==============================================================================

primary_cells = primary[
    "primary_cells"
]


primary_comparisons = primary[
    "paired_comparisons"
]


secondary_cells = {
    "bridge62": {
        "target":
            "IDS2018_02-28-2018",

        "probability_sha256":
            p62_sha,

        "pr_auc":
            point62[
                "pr_auc"
            ],

        "roc_auc":
            point62[
                "roc_auc"
            ],

        "brier":
            point62[
                "brier"
            ],

        "pr_normalized":
            float(
                b62_result[
                    "metrics"
                ][
                    "primary"
                ][
                    "pr_normalized"
                ]
            ),
    },

    "bridge70": {
        "target":
            "IDS2018_02-28-2018",

        "probability_sha256":
            p70_sha,

        "pr_auc":
            point70[
                "pr_auc"
            ],

        "roc_auc":
            point70[
                "roc_auc"
            ],

        "brier":
            point70[
                "brier"
            ],

        "pr_normalized":
            float(
                b70_result[
                    "metrics"
                ][
                    "primary"
                ][
                    "pr_normalized"
                ]
            ),
    },
}


target_prevalence = float(
    EXPECTED_ATTACK
    /
    EXPECTED_ROWS
)


# Descriptive, not inferential, because directions use different targets/models.
directional_summary = {
    "averaging_performed":
        False,

    "averaging_prohibited":
        True,

    "reason":
        (
            "The frozen Stage24 protocol requires the two transfer directions "
            "to remain separate. They use different source models, target "
            "populations, and target prevalence structures."
        ),

    "IDS2018_TO_CICIDS2017": {
        "target_rows":
            int(
                primary[
                    "target_population"
                ][
                    "rows"
                ]
            ),

        "target_attack":
            int(
                primary[
                    "target_population"
                ][
                    "attack"
                ]
            ),

        "bridge62_published_pr_auc":
            float(
                primary_cells[
                    "bridge62_PUBLISHED"
                ][
                    "pr_auc"
                ]
            ),

        "bridge62_published_roc_auc":
            float(
                primary_cells[
                    "bridge62_PUBLISHED"
                ][
                    "roc_auc"
                ]
            ),

        "bridge70_published_pr_auc":
            float(
                primary_cells[
                    "bridge70_PUBLISHED"
                ][
                    "pr_auc"
                ]
            ),

        "bridge70_published_roc_auc":
            float(
                primary_cells[
                    "bridge70_PUBLISHED"
                ][
                    "roc_auc"
                ]
            ),

        "bridge70_flag_corrected_pr_auc":
            float(
                primary_cells[
                    "bridge70_FLAG_CORRECTED"
                ][
                    "pr_auc"
                ]
            ),

        "bridge70_flag_corrected_roc_auc":
            float(
                primary_cells[
                    "bridge70_FLAG_CORRECTED"
                ][
                    "roc_auc"
                ]
            ),
    },

    "CICIDS2017_TO_IDS2018": {
        "target_rows":
            EXPECTED_ROWS,

        "target_attack":
            EXPECTED_ATTACK,

        "target_prevalence":
            target_prevalence,

        "bridge62_pr_auc":
            point62[
                "pr_auc"
            ],

        "bridge62_roc_auc":
            point62[
                "roc_auc"
            ],

        "bridge70_pr_auc":
            point70[
                "pr_auc"
            ],

        "bridge70_roc_auc":
            point70[
                "roc_auc"
            ],
    },
}


# ==============================================================================
# 17. FINAL RESULT JSON
# ==============================================================================

result = {
    "stage":
        "Stage24-6",

    "status":
        "STAGE24_CROSS_DATASET_AUDIT_COMPLETE",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_PARENT,

    "title":
        "Cross-Dataset Generalization and Artifact-Sensitivity Audit",

    "datasets":
        [
            "CSE-CIC-IDS2018",
            "CICIDS2017",
        ],

    "governance": {
        "scientific_fit_budget":
            4,

        "scientific_fits_completed":
            4,

        "scientific_fits_remaining":
            0,

        "additional_fits_authorized":
            0,

        "target_opening_budget":
            8,

        "target_openings_consumed":
            6,

        "administratively_cancelled_GROUNDED_S4_cells":
            2,

        "remaining_evaluable_target_openings":
            0,

        "cancelled_slots_reallocated":
            False,

        "all_evaluable_target_cells_complete":
            True,
    },

    "primary_direction": {
        "direction":
            "IDS2018_TO_CICIDS2017",

        "freeze_artifact":
            str(
                PRIMARY_RESULT_PATH.relative_to(
                    REPO
                )
            ),

        "freeze_sha256":
            primary_result_sha,

        "bootstrap_artifact_sha256":
            primary_bootstrap_sha,

        "cells":
            primary_cells,

        "paired_comparisons":
            primary_comparisons,

        "GROUNDED_S4": {
            "status":
                "NOT_EVALUABLE",

            "reason":
                "DURABLE_ARTIFACT_INFEASIBILITY",

            "definition_changed":
                False,

            "heuristic_substitute_used":
                False,
        },
    },

    "secondary_direction": {
        "direction":
            "CICIDS2017_TO_IDS2018",

        "target":
            "IDS2018_02-28-2018",

        "target_population": {
            "rows":
                EXPECTED_ROWS,

            "benign":
                EXPECTED_BENIGN,

            "attack":
                EXPECTED_ATTACK,

            "prevalence":
                target_prevalence,

            "binary_label_sha256":
                label_sha,

            "original_row_index_sha256":
                row_sha,

            "clean_position_sha256":
                clean_sha,
        },

        "cells":
            secondary_cells,

        "paired_comparison": {
            "comparison":
                "bridge70_minus_bridge62",

            "orientation":
                "A_MINUS_B",

            "bootstrap":
                secondary_bootstrap,
        },

        "bootstrap_protocol": {
            "method":
                "PAIRED_STRATIFIED_ROW_BOOTSTRAP",

            "implementation":
                (
                    "EXACT_JOINT_PREDICTION_LABEL_EQUIVALENCE_GROUP_"
                    "MULTINOMIAL_SUFFICIENT_STATISTIC"
                ),

            "implementation_is_approximation":
                False,

            "replicates":
                BOOTSTRAP_REPLICATES,

            "seed":
                BOOTSTRAP_SEED,

            "strata":
                [
                    "BENIGN",
                    "ATTACK",
                ],

            "sampling":
                "WITH_REPLACEMENT_WITHIN_EACH_STRATUM",

            "pairing":
                (
                    "SAME_BOOTSTRAP_MULTIPLICITIES_USED_FOR_"
                    "BRIDGE62_AND_BRIDGE70"
                ),

            "metrics":
                [
                    "PR_AUC",
                    "ROC_AUC",
                    "Brier",
                ],

            "confidence_interval":
                "PERCENTILE_2.5_97.5",

            "auc_semantics": {
                "PR_AUC":
                    (
                        "DISTINCT_SCORE_GROUPED_NONINTERPOLATED_"
                        "AVERAGE_PRECISION"
                    ),

                "ROC_AUC":
                    (
                        "DISTINCT_SCORE_GROUPED_TRAPEZOIDAL_ROC_AREA"
                    ),

                "scores":
                    "PERSISTED_FLOAT32_PROBABILITIES",
            },

            "exact_grouping": {
                "benign_rows":
                    EXPECTED_BENIGN,

                "benign_groups":
                    G0,

                "attack_rows":
                    EXPECTED_ATTACK,

                "attack_groups":
                    G1,

                "total_rows":
                    EXPECTED_ROWS,

                "total_groups":
                    G,

                "compression_ratio":
                    float(
                        EXPECTED_ROWS
                        /
                        G
                    ),
            },

            "runtime_seconds":
                bootstrap_seconds,
        },

        "bootstrap_artifact": {
            "file":
                str(
                    BOOTSTRAP_PATH.relative_to(
                        REPO
                    )
                ),

            "sha256":
                secondary_bootstrap_sha,
        },
    },

    "cross_direction_synthesis":
        directional_summary,

    "interpretive_constraints": {
        "directions_averaged":
            False,

        "directions_must_remain_separate":
            True,

        "GROUNDED_S4_inferred":
            False,

        "GROUNDED_S4_fuzzy_substitute":
            False,

        "post_target_model_change":
            False,

        "post_target_threshold_change":
            False,

        "post_target_feature_change":
            False,
    },

    "anti_adaptation": {
        "new_model_fit":
            False,

        "new_target_inference":
            False,

        "new_target_opening":
            False,

        "target_threshold_selection":
            False,

        "target_feature_search":
            False,

        "target_feature_remapping":
            False,

        "target_fitted_preprocessing":
            False,

        "target_calibration":
            False,

        "performance_based_protocol_change":
            False,
    },

    "completion": {
        "Stage24_complete":
            True,

        "scientific_fits":
            "4/4",

        "evaluable_target_openings":
            "6/6",

        "original_opening_budget":
            8,

        "administratively_cancelled":
            2,

        "remaining_evaluable_work":
            0,
    },
}


RESULT_PATH.write_text(
    json.dumps(
        result,
        indent=2,
        sort_keys=True,
    )
    +
    "\n",
    encoding="utf-8",
)


result_sha = sha256_file(
    RESULT_PATH
)


RESULT_SHA_PATH.write_text(
    f"{result_sha}  {RESULT_PATH.name}\n",
    encoding="utf-8",
)


# ==============================================================================
# 18. HUMAN-READABLE FINAL SYNTHESIS
# ==============================================================================

def fmt_ci(
    item,
):

    return (
        f"{item['point_difference']:+.6f} "
        f"[{item['ci_2_5']:+.6f}, {item['ci_97_5']:+.6f}]"
    )


primary_pub_cmp = next(
    item
    for item in primary_comparisons
    if item[
        "comparison"
    ]
    ==
    "bridge70_PUBLISHED_minus_bridge62_PUBLISHED"
)


primary_corr_cmp = next(
    item
    for item in primary_comparisons
    if item[
        "comparison"
    ]
    ==
    "bridge70_FLAG_CORRECTED_minus_bridge62_FLAG_CORRECTED"
)


primary_artifact_cmp = next(
    item
    for item in primary_comparisons
    if item[
        "comparison"
    ]
    ==
    "bridge70_FLAG_CORRECTED_minus_bridge70_PUBLISHED"
)


report = f"""# Stage24 Final Freeze

## Cross-Dataset Generalization & Artifact-Sensitivity Audit

**Status:** COMPLETE

**Directions are reported separately and are not averaged.**

---

## 1. Governance

- Scientific fits completed: **4 / 4**
- Additional fits authorized: **0**
- Evaluable target openings completed: **6 / 6**
- Original target-opening budget: **8**
- GROUNDED_S4 cells cancelled before opening: **2**
- Cancelled openings reallocated: **No**
- Remaining evaluable target cells: **0**
- Target threshold tuning: **None**
- Target calibration: **None**
- Post-opening feature/model adaptation: **None**

---

## 2. Primary Direction — IDS2018 → CICIDS2017

Target population: **{primary['target_population']['rows']:,} rows**

| Cell | PR-AUC | ROC-AUC | Brier |
|---|---:|---:|---:|
| bridge62 PUBLISHED | {primary_cells['bridge62_PUBLISHED']['pr_auc']:.6f} | {primary_cells['bridge62_PUBLISHED']['roc_auc']:.6f} | {primary_cells['bridge62_PUBLISHED']['brier']:.6f} |
| bridge62 FLAG_CORRECTED | {primary_cells['bridge62_FLAG_CORRECTED']['pr_auc']:.6f} | {primary_cells['bridge62_FLAG_CORRECTED']['roc_auc']:.6f} | {primary_cells['bridge62_FLAG_CORRECTED']['brier']:.6f} |
| bridge70 PUBLISHED | {primary_cells['bridge70_PUBLISHED']['pr_auc']:.6f} | {primary_cells['bridge70_PUBLISHED']['roc_auc']:.6f} | {primary_cells['bridge70_PUBLISHED']['brier']:.6f} |
| bridge70 FLAG_CORRECTED | {primary_cells['bridge70_FLAG_CORRECTED']['pr_auc']:.6f} | {primary_cells['bridge70_FLAG_CORRECTED']['roc_auc']:.6f} | {primary_cells['bridge70_FLAG_CORRECTED']['brier']:.6f} |

bridge62 PUBLISHED and FLAG_CORRECTED are bitwise identical.

### Primary paired bootstrap — 2,000 replicates, seed 42

**bridge70 PUBLISHED − bridge62 PUBLISHED**

- PR-AUC: {fmt_ci(primary_pub_cmp['bootstrap']['pr_auc'])}
- ROC-AUC: {fmt_ci(primary_pub_cmp['bootstrap']['roc_auc'])}
- Brier: {fmt_ci(primary_pub_cmp['bootstrap']['brier'])}

**bridge70 FLAG_CORRECTED − bridge62 FLAG_CORRECTED**

- PR-AUC: {fmt_ci(primary_corr_cmp['bootstrap']['pr_auc'])}
- ROC-AUC: {fmt_ci(primary_corr_cmp['bootstrap']['roc_auc'])}
- Brier: {fmt_ci(primary_corr_cmp['bootstrap']['brier'])}

**bridge70 FLAG_CORRECTED − bridge70 PUBLISHED**

- PR-AUC: {fmt_ci(primary_artifact_cmp['bootstrap']['pr_auc'])}
- ROC-AUC: {fmt_ci(primary_artifact_cmp['bootstrap']['roc_auc'])}
- Brier: {fmt_ci(primary_artifact_cmp['bootstrap']['brier'])}

### GROUNDED_S4

Not evaluated.

Reason: the frozen Stage20 exact-S4 population could not be recovered as exact physical CICIDS2017 feature-table rows from the durable compact artifact without introducing a new heuristic matching rule.

No fuzzy or inferred substitute was used.

---

## 3. Secondary Direction — CICIDS2017 → IDS2018

Target: **IDS2018 Feb-28**

Target population:

- Rows: **{EXPECTED_ROWS:,}**
- Benign: **{EXPECTED_BENIGN:,}**
- Attack: **{EXPECTED_ATTACK:,}**
- Attack prevalence: **{target_prevalence:.6f}**

| Cell | PR-AUC | ROC-AUC | Brier |
|---|---:|---:|---:|
| bridge62 | {point62['pr_auc']:.6f} | {point62['roc_auc']:.6f} | {point62['brier']:.6f} |
| bridge70 | {point70['pr_auc']:.6f} | {point70['roc_auc']:.6f} | {point70['brier']:.6f} |

Chance PR-AUC anchor from target prevalence: **{target_prevalence:.6f}**

### Secondary paired bootstrap — bridge70 − bridge62

- PR-AUC: {fmt_ci(secondary_bootstrap['pr_auc'])}
- ROC-AUC: {fmt_ci(secondary_bootstrap['roc_auc'])}
- Brier: {fmt_ci(secondary_bootstrap['brier'])}

---

## 4. Cross-Direction Result

The two transfer directions produce materially different observed generalization behavior and are therefore retained separately, as preregistered.

### IDS2018 → CICIDS2017

The full-population CICIDS2017 evaluations retain substantial ranking signal:

- bridge62 PR-AUC: **{primary_cells['bridge62_PUBLISHED']['pr_auc']:.6f}**
- bridge62 ROC-AUC: **{primary_cells['bridge62_PUBLISHED']['roc_auc']:.6f}**
- bridge70 PUBLISHED PR-AUC: **{primary_cells['bridge70_PUBLISHED']['pr_auc']:.6f}**
- bridge70 PUBLISHED ROC-AUC: **{primary_cells['bridge70_PUBLISHED']['roc_auc']:.6f}**

The aggregate-flag serialization correction produces measurable changes in the bridge70 target predictions and in all three frozen paired-bootstrap metrics.

### CICIDS2017 → IDS2018

The reverse transfer is close to the IDS2018 Feb-28 prevalence/chance anchor:

- chance PR-AUC anchor: **{target_prevalence:.6f}**
- bridge62 PR-AUC: **{point62['pr_auc']:.6f}**
- bridge70 PR-AUC: **{point70['pr_auc']:.6f}**
- bridge62 ROC-AUC: **{point62['roc_auc']:.6f}**
- bridge70 ROC-AUC: **{point70['roc_auc']:.6f}**

The source-selected operating points also transferred with extremely low attack recall, which is retained as part of the frozen result rather than repaired post hoc.

---

## 5. Final Scientific Status

Stage24 demonstrates that cross-dataset IDS generalization is strongly direction-dependent under the frozen validation-safe protocol.

The study separately exposes:

1. **dataset/domain shift,**
2. **aggregate-flag extractor/serialization sensitivity,**
3. **bridge-feature sensitivity,**
4. **source-validation threshold transfer behavior,**
5. and the limitation that exact GROUNDED_S4 physical target membership was not durably recoverable.

No target-guided correction was introduced after any target result was observed.

---

## 6. Artifact Identities

- Primary freeze SHA256: `{primary_result_sha}`
- Primary bootstrap SHA256: `{primary_bootstrap_sha}`
- Secondary bridge62 probability SHA256: `{p62_sha}`
- Secondary bridge70 probability SHA256: `{p70_sha}`
- Secondary bootstrap SHA256: `{secondary_bootstrap_sha}`
- Final synthesis JSON SHA256: `{result_sha}`

"""


REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)


report_sha = sha256_file(
    REPORT_PATH
)


CHECKSUMS_PATH.write_text(
    (
        f"{result_sha}  {RESULT_PATH.name}\n"
        f"{report_sha}  {REPORT_PATH.name}\n"
        f"{secondary_bootstrap_sha}  {BOOTSTRAP_PATH.name}\n"
    ),
    encoding="utf-8",
)


print("=" * 118)
print("FINAL SYNTHESIS ARTIFACTS")
print("=" * 118)


print(
    "Result JSON:",
    RESULT_PATH.relative_to(
        REPO
    ),
)

print(
    "Result SHA:",
    result_sha,
)

print()

print(
    "Report:",
    REPORT_PATH.relative_to(
        REPO
    ),
)

print(
    "Report SHA:",
    report_sha,
)

print()

print(
    "Secondary bootstrap SHA:",
    secondary_bootstrap_sha,
)

print()


# ==============================================================================
# 19. GITHUB CREDENTIAL + REMOTE PARENT
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()


    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )


            if value and value.strip():

                github_token = value.strip()
                token_source = name
                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )


        if value and value.strip():

            github_token = value.strip()

            token_source = (
                "ENV:"
                +
                name
            )

            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved before final Stage24 freeze.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main == Stage24-5B."
)

print()


# ==============================================================================
# 20. GIT AUTHOR SAFETY
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    git_cmd(
        "config",
        "--local",
        "user.name",
        git_cmd(
            "log",
            "-1",
            "--format=%an",
        ),
    )


if not author_email:

    git_cmd(
        "config",
        "--local",
        "user.email",
        git_cmd(
            "log",
            "-1",
            "--format=%ae",
        ),
    )


# ==============================================================================
# 21. GIT FREEZE
# ==============================================================================

print("=" * 118)
print("GIT FREEZE")
print("=" * 118)


git_cmd(
    "add",
    "--",
    str(
        BOOTSTRAP_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        REPORT_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
)


staged = {
    line
    for line in git_cmd(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
    if line
}


expected_staged = {
    str(
        BOOTSTRAP_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_PATH.relative_to(
            REPO
        )
    ),
    str(
        RESULT_SHA_PATH.relative_to(
            REPO
        )
    ),
    str(
        REPORT_PATH.relative_to(
            REPO
        )
    ),
    str(
        CHECKSUMS_PATH.relative_to(
            REPO
        )
    ),
}


if staged != expected_staged:

    raise RuntimeError(
        "\nUnexpected staged files.\n"
        f"Expected: {sorted(expected_staged)}\n"
        f"Actual:   {sorted(staged)}"
    )


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: freeze secondary bootstrap and final synthesis",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage24-6 commit parent mismatch."
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 22. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 118)
print("PUSH + REMOTE VERIFY")
print("=" * 118)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository not clean after Stage24 final freeze:\n"
        + final_status
    )


print(
    "[PASS] Remote main == local Stage24-6 commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 23. FINAL
# ==============================================================================

print("=" * 118)
print("STAGE24 COMPLETE — FINAL FREEZE PASS")
print("=" * 118)

print()

print(
    "Study:"
)

print(
    "  Cross-Dataset Generalization & Artifact-Sensitivity Audit"
)

print()

print(
    "Scientific fits:                4 / 4 COMPLETE"
)

print(
    "Additional fits authorized:     0"
)

print()

print(
    "Evaluable target openings:      6 / 6 COMPLETE"
)

print(
    "Original opening budget:        8"
)

print(
    "GROUNDED_S4 cancelled:          2"
)

print(
    "Cancelled slots reallocated:    NO"
)

print(
    "Remaining evaluable openings:   0"
)

print()

print(
    "PRIMARY DIRECTION:"
)

print(
    "  IDS2018 -> CICIDS2017"
)

print(
    "  bridge62 PR-AUC:               ",
    f"{primary_cells['bridge62_PUBLISHED']['pr_auc']:.15f}",
)

print(
    "  bridge62 ROC-AUC:              ",
    f"{primary_cells['bridge62_PUBLISHED']['roc_auc']:.15f}",
)

print(
    "  bridge70 PUBLISHED PR-AUC:     ",
    f"{primary_cells['bridge70_PUBLISHED']['pr_auc']:.15f}",
)

print(
    "  bridge70 PUBLISHED ROC-AUC:    ",
    f"{primary_cells['bridge70_PUBLISHED']['roc_auc']:.15f}",
)

print(
    "  bridge70 CORRECTED PR-AUC:     ",
    f"{primary_cells['bridge70_FLAG_CORRECTED']['pr_auc']:.15f}",
)

print(
    "  bridge70 CORRECTED ROC-AUC:    ",
    f"{primary_cells['bridge70_FLAG_CORRECTED']['roc_auc']:.15f}",
)

print()

print(
    "SECONDARY DIRECTION:"
)

print(
    "  CICIDS2017 -> IDS2018"
)

print(
    "  target prevalence:             ",
    f"{target_prevalence:.15f}",
)

print(
    "  bridge62 PR-AUC:               ",
    f"{point62['pr_auc']:.15f}",
)

print(
    "  bridge62 ROC-AUC:              ",
    f"{point62['roc_auc']:.15f}",
)

print(
    "  bridge70 PR-AUC:               ",
    f"{point70['pr_auc']:.15f}",
)

print(
    "  bridge70 ROC-AUC:              ",
    f"{point70['roc_auc']:.15f}",
)

print()

print(
    "SECONDARY bridge70 - bridge62:"
)

print(
    "  PR-AUC Δ:",
    f"{secondary_bootstrap['pr_auc']['point_difference']:+.15f}",
)

print(
    "  PR 95% CI:",
    (
        f"[{secondary_bootstrap['pr_auc']['ci_2_5']:+.15f}, "
        f"{secondary_bootstrap['pr_auc']['ci_97_5']:+.15f}]"
    ),
)

print(
    "  ROC-AUC Δ:",
    f"{secondary_bootstrap['roc_auc']['point_difference']:+.15f}",
)

print(
    "  ROC 95% CI:",
    (
        f"[{secondary_bootstrap['roc_auc']['ci_2_5']:+.15f}, "
        f"{secondary_bootstrap['roc_auc']['ci_97_5']:+.15f}]"
    ),
)

print(
    "  Brier Δ:",
    f"{secondary_bootstrap['brier']['point_difference']:+.15f}",
)

print(
    "  Brier 95% CI:",
    (
        f"[{secondary_bootstrap['brier']['ci_2_5']:+.15f}, "
        f"{secondary_bootstrap['brier']['ci_97_5']:+.15f}]"
    ),
)

print()

print(
    "Directions averaged:            NO"
)

print(
    "Target-guided adaptation:       NO"
)

print()

print(
    "Secondary bootstrap SHA:"
)

print(
    " ",
    secondary_bootstrap_sha,
)

print()

print(
    "Final synthesis SHA:"
)

print(
    " ",
    result_sha,
)

print()

print(
    "Final report SHA:"
)

print(
    " ",
    report_sha,
)

print()

print(
    "Commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "STAGE24 SCIENTIFIC EXECUTION IS CLOSED."
)

print("=" * 118)

STAGE24-6 — SECONDARY PAIRED BOOTSTRAP + FINAL CROSS-DIRECTION SYNTHESIS

GOVERNANCE
HEAD: ce2913dd7a040b400be58acdacab4017bc4b45f7
[PASS] Repository clean.
[PASS] No unresolved fit ledger.
[PASS] No unresolved opening ledger.
[PASS] Stage24-6 has not previously been frozen.

PRIMARY DIRECTION FREEZE
Primary result SHA: bdb6471be8b154662149eeb616dfdfb78f978dd3f337659e773ad9a21b3ba42f
Primary bootstrap SHA: 96732023e0a1c9b52a79fafa59ea4b851ef7f8bae1814e4b4b4fde0ef0df09aa

[PASS] Primary direction remains byte-frozen.

SECONDARY TARGET RECEIPTS
Stage24-5A result SHA: 54ea37bd6cc5ea82b77762ee672afcecf9584d6fb115187df9040cebac715515
Stage24-5B result SHA: 86d5ff8d15f2f55543738c6a8697937643c50d48c3b5d894c2032e3981de5afa

[PASS] Scientific fits complete: 4 / 4.
[PASS] Evaluable target openings complete: 6.
[PASS] GROUNDED_S4 cancellations: 2.
[PASS] Cancelled slots not reallocated.

SECONDARY PAIRED VECTOR IDENTITY
Rows:        593,780
Benign:      531,524
Attack:      62,256

bridge62 SHA: 

In [44]:
# ==============================================================================
# STAGE24-PUB — PUBLICATION PACKAGE + NOTEBOOK/SCRIPT EXPORT + GITHUB CLOSEOUT
#
# NON-SCIENTIFIC POST-PROCESSING ONLY
#
# This cell:
#   1. Verifies the final frozen Stage24 commit and result artifacts.
#   2. Builds publication-ready CSV/Markdown/LaTeX tables.
#   3. Builds publication-ready PNG + SVG figures.
#   4. Writes manuscript-ready:
#        - Results
#        - Discussion
#        - Limitations / Threats to Validity
#        - Contributions
#        - Publication-safe claims
#   5. Exports the CURRENT Kaggle notebook:
#        - full downloadable notebook copy when recoverable
#        - sanitized code-only .ipynb for GitHub
#        - percent-cell .py script for GitHub
#   6. Creates downloadable ZIP archives in /kaggle/working.
#   7. Updates README + JOURNAL_EXTENSION_SUMMARY.
#   8. Commits and pushes everything to GitHub main.
#
# NO MODEL FIT.
# NO MODEL LOAD.
# NO MODEL INFERENCE.
# NO TARGET READ.
# NO TARGET OPENING.
# NO THRESHOLD SEARCH.
# NO SCIENTIFIC RESULT MODIFICATION.
#
# Expected starting commit:
#   a4b6a3854109ba3d85954fb4a40afe6fe1ee6114
#
# ==============================================================================

from __future__ import annotations

import os
import gc
import csv
import json
import math
import base64
import shutil
import hashlib
import subprocess
import textwrap
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import nbformat
except Exception:
    nbformat = None


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "a4b6a3854109ba3d85954fb4a40afe6fe1ee6114"
)

EXPECTED_FINAL_SYNTHESIS_SHA = (
    "785bbcb00140f4d7e07e9b49ad33924166b5995e66a229c986d5b1abcbcaee4b"
)

EXPECTED_FINAL_REPORT_SHA = (
    "f867289cec34eca6666d02deec01864395503cea8a59b7eac88e1759d5eb2a0b"
)

EXPECTED_SECONDARY_BOOTSTRAP_SHA = (
    "28005e0f13d86658ba81e6b19dc154b94eedb316ea5ce0348287265c00541953"
)

EXPECTED_PRIMARY_FREEZE_SHA = (
    "bdb6471be8b154662149eeb616dfdfb78f978dd3f337659e773ad9a21b3ba42f"
)

EXPECTED_PRIMARY_BOOTSTRAP_SHA = (
    "96732023e0a1c9b52a79fafa59ea4b851ef7f8bae1814e4b4b4fde0ef0df09aa"
)


print("=" * 120)
print("STAGE24-PUB — PUBLICATION PACKAGE + NOTEBOOK EXPORT + GITHUB CLOSEOUT")
print("=" * 120)
print()


# ==============================================================================
# 1. PATHS
# ==============================================================================

STAGE24_BASE = (
    REPO
    / "results"
    / "stage24_cross_dataset"
)

FINAL_DIR = (
    STAGE24_BASE
    / "stage24_6_final_synthesis"
)

FINAL_JSON = (
    FINAL_DIR
    / "stage24_6_final_synthesis.json"
)

FINAL_JSON_SHA = (
    FINAL_DIR
    / "stage24_6_final_synthesis.sha256"
)

FINAL_REPORT = (
    FINAL_DIR
    / "stage24_6_final_synthesis.md"
)

SECONDARY_BOOTSTRAP = (
    FINAL_DIR
    / "stage24_6_secondary_paired_bootstrap_replicates.npz"
)


PRIMARY_DIR = (
    STAGE24_BASE
    / "stage24_3_primary_results_freeze"
    / "stage24_3a_primary_paired_bootstrap"
)

PRIMARY_JSON = (
    PRIMARY_DIR
    / "stage24_3a_primary_results_freeze.json"
)

PRIMARY_BOOTSTRAP = (
    PRIMARY_DIR
    / "stage24_3a_primary_paired_bootstrap_replicates.npz"
)


B62_TARGET_RESULT = (
    STAGE24_BASE
    / "stage24_5_secondary_target_openings"
    / "stage24_5a_bridge62_ids2018_feb28"
    / "stage24_5a_secondary_bridge62_ids2018_feb28_result.json"
)

B70_TARGET_RESULT = (
    STAGE24_BASE
    / "stage24_5_secondary_target_openings"
    / "stage24_5b_bridge70_ids2018_feb28"
    / "stage24_5b_secondary_bridge70_ids2018_feb28_result.json"
)


# Publication outputs
PUB_RESULT_DIR = (
    STAGE24_BASE
    / "stage24_publication_package"
)

TABLE_DIR = (
    PUB_RESULT_DIR
    / "tables"
)

FIG_DIR = (
    REPO
    / "figures"
    / "stage24_cross_dataset"
)

DOC_MANUSCRIPT = (
    REPO
    / "docs"
    / "STAGE24_MANUSCRIPT_INTEGRATION.md"
)

DOC_TABLES_MD = (
    REPO
    / "docs"
    / "STAGE24_PUBLICATION_TABLES.md"
)

DOC_TABLES_TEX = (
    REPO
    / "docs"
    / "STAGE24_PUBLICATION_TABLES.tex"
)

DOC_MANUSCRIPT_TEX = (
    REPO
    / "docs"
    / "STAGE24_MANUSCRIPT_INTEGRATION.tex"
)

DOC_CLOSEOUT = (
    REPO
    / "docs"
    / "STAGE24_PUBLICATION_CLOSEOUT.md"
)


# Notebook/script exports
SCRIPT_DIR = (
    REPO
    / "scripts"
    / "stage24"
)

SCRIPT_IPYNB = (
    SCRIPT_DIR
    / "stage24_cross_dataset_generalization.ipynb"
)

SCRIPT_PY = (
    SCRIPT_DIR
    / "stage24_cross_dataset_generalization.py"
)

SCRIPT_README = (
    SCRIPT_DIR
    / "README.md"
)


# Downloadable copies outside repo.
DOWNLOAD_DIR = Path(
    "/kaggle/working/stage24_exports"
)

FULL_NOTEBOOK_DOWNLOAD = (
    DOWNLOAD_DIR
    / "stage24_cross_dataset_generalization_FULL.ipynb"
)

NOTEBOOK_ZIP_BASE = Path(
    "/kaggle/working/stage24_notebook_export"
)

PUBLICATION_ZIP_BASE = Path(
    "/kaggle/working/stage24_publication_package"
)


README_PATH = (
    REPO
    / "README.md"
)

JOURNAL_SUMMARY_PATH = (
    REPO
    / "docs"
    / "JOURNAL_EXTENSION_SUMMARY.md"
)


FIT_LEDGER = Path(
    "/kaggle/working/stage24_secondary_fit_runtime_ledger.json"
)

OPENING_LEDGER = Path(
    "/kaggle/working/stage24_target_opening_runtime_ledger.json"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    cmd = [
        "git"
    ]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [
        str(x)
        for x in args
    ]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
):

    path = Path(
        path
    )

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def verify_sidecar(
    artifact,
    sidecar,
):

    artifact = Path(
        artifact
    )

    sidecar = Path(
        sidecar
    )

    if not artifact.is_file():

        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )

    if not sidecar.is_file():

        raise RuntimeError(
            f"Missing sidecar:\n{sidecar}"
        )

    actual = sha256_file(
        artifact
    )

    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
    )

    if actual != expected:

        raise RuntimeError(
            "\nSHA mismatch:\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )

    return actual


def write_text(
    path,
    content,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        content.rstrip()
        +
        "\n",
        encoding="utf-8",
    )


def md_float(
    x,
    digits=6,
):

    if x is None:
        return "NA"

    return f"{float(x):.{digits}f}"


def latex_escape(
    value,
):

    value = str(
        value
    )

    replacements = {
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
    }

    for old, new in replacements.items():

        value = value.replace(
            old,
            new,
        )

    return value


def comparison_lookup(
    comparisons,
    name,
):

    matches = [
        x
        for x in comparisons
        if x.get(
            "comparison"
        ) == name
    ]

    if len(
        matches
    ) != 1:

        raise RuntimeError(
            f"Expected one comparison named {name}, got {len(matches)}"
        )

    return matches[
        0
    ]


def append_marked_section(
    path,
    begin_marker,
    end_marker,
    body,
):

    path = Path(
        path
    )

    original = (
        path.read_text(
            encoding="utf-8"
        )
        if path.is_file()
        else ""
    )

    block = (
        f"{begin_marker}\n\n"
        f"{body.strip()}\n\n"
        f"{end_marker}"
    )

    if begin_marker in original:

        start = original.index(
            begin_marker
        )

        if end_marker not in original[
            start:
        ]:

            raise RuntimeError(
                f"Found begin marker but no end marker in {path}"
            )

        end = (
            original.index(
                end_marker,
                start,
            )
            +
            len(
                end_marker
            )
        )

        updated = (
            original[
                :start
            ].rstrip()
            +
            "\n\n"
            +
            block
            +
            "\n"
            +
            original[
                end:
            ].lstrip(
                "\n"
            )
        )

    else:

        updated = (
            original.rstrip()
            +
            "\n\n"
            +
            block
            +
            "\n"
        )

    path.write_text(
        updated,
        encoding="utf-8",
    )


# ==============================================================================
# 3. GOVERNANCE — STAGE24 MUST ALREADY BE CLOSED
# ==============================================================================

print("=" * 120)
print("GOVERNANCE")
print("=" * 120)


head = git_cmd(
    "rev-parse",
    "HEAD",
)

status = git_cmd(
    "status",
    "--porcelain",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected repository HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}\n\n"
        "Do not run publication closeout on an unknown repository state."
    )


if status:

    raise RuntimeError(
        "\nRepository is not clean:\n"
        + status
    )


if FIT_LEDGER.exists():

    raise RuntimeError(
        f"Unexpected Stage24 fit ledger remains:\n{FIT_LEDGER}"
    )


if OPENING_LEDGER.exists():

    raise RuntimeError(
        f"Unexpected Stage24 target-opening ledger remains:\n{OPENING_LEDGER}"
    )


print(
    "[PASS] Repository starts from exact Stage24 final freeze."
)

print(
    "[PASS] Repository clean."
)

print(
    "[PASS] No unresolved fit/opening ledgers."
)

print()


# ==============================================================================
# 4. VERIFY FINAL FROZEN SCIENTIFIC ARTIFACTS
# ==============================================================================

print("=" * 120)
print("FROZEN SCIENTIFIC ARTIFACT VERIFICATION")
print("=" * 120)


final_sha = verify_sidecar(
    FINAL_JSON,
    FINAL_JSON_SHA,
)


if final_sha != EXPECTED_FINAL_SYNTHESIS_SHA:

    raise RuntimeError(
        "Unexpected final Stage24 synthesis SHA."
    )


final_report_sha = sha256_file(
    FINAL_REPORT
)


if final_report_sha != EXPECTED_FINAL_REPORT_SHA:

    raise RuntimeError(
        "\nFinal frozen report SHA changed.\n"
        f"Expected: {EXPECTED_FINAL_REPORT_SHA}\n"
        f"Actual:   {final_report_sha}"
    )


secondary_bootstrap_sha = sha256_file(
    SECONDARY_BOOTSTRAP
)


if secondary_bootstrap_sha != EXPECTED_SECONDARY_BOOTSTRAP_SHA:

    raise RuntimeError(
        "Secondary bootstrap SHA changed."
    )


primary_sha = sha256_file(
    PRIMARY_JSON
)


if primary_sha != EXPECTED_PRIMARY_FREEZE_SHA:

    raise RuntimeError(
        "Primary Stage24 freeze SHA changed."
    )


primary_bootstrap_sha = sha256_file(
    PRIMARY_BOOTSTRAP
)


if primary_bootstrap_sha != EXPECTED_PRIMARY_BOOTSTRAP_SHA:

    raise RuntimeError(
        "Primary bootstrap SHA changed."
    )


stage24 = json.loads(
    FINAL_JSON.read_text(
        encoding="utf-8"
    )
)

primary = json.loads(
    PRIMARY_JSON.read_text(
        encoding="utf-8"
    )
)

secondary62 = json.loads(
    B62_TARGET_RESULT.read_text(
        encoding="utf-8"
    )
)

secondary70 = json.loads(
    B70_TARGET_RESULT.read_text(
        encoding="utf-8"
    )
)


if stage24[
    "status"
] != "STAGE24_CROSS_DATASET_AUDIT_COMPLETE":

    raise RuntimeError(
        "Stage24 final status is not complete."
    )


if stage24[
    "completion"
][
    "Stage24_complete"
] is not True:

    raise RuntimeError(
        "Stage24 completion flag is false."
    )


if stage24[
    "completion"
][
    "scientific_fits"
] != "4/4":

    raise RuntimeError(
        "Stage24 scientific-fit accounting changed."
    )


if stage24[
    "completion"
][
    "evaluable_target_openings"
] != "6/6":

    raise RuntimeError(
        "Stage24 target-opening accounting changed."
    )


if int(
    stage24[
        "completion"
    ][
        "administratively_cancelled"
    ]
) != 2:

    raise RuntimeError(
        "Stage24 GROUNDED_S4 cancellation accounting changed."
    )


print(
    "Final synthesis SHA:    ",
    final_sha,
)

print(
    "Final report SHA:       ",
    final_report_sha,
)

print(
    "Primary bootstrap SHA:  ",
    primary_bootstrap_sha,
)

print(
    "Secondary bootstrap SHA:",
    secondary_bootstrap_sha,
)

print()

print(
    "[PASS] Stage24 frozen scientific state verified."
)

print()


# ==============================================================================
# 5. EXTRACT FINAL NUMBERS
# ==============================================================================

primary_cells = stage24[
    "primary_direction"
][
    "cells"
]

primary_comparisons = stage24[
    "primary_direction"
][
    "paired_comparisons"
]

secondary_cells = stage24[
    "secondary_direction"
][
    "cells"
]

secondary_comparison = stage24[
    "secondary_direction"
][
    "paired_comparison"
][
    "bootstrap"
]


primary_target_rows = int(
    primary[
        "target_population"
    ][
        "rows"
    ]
)

primary_target_attack = int(
    primary[
        "target_population"
    ][
        "attack"
    ]
)

primary_target_benign = int(
    primary[
        "target_population"
    ][
        "benign"
    ]
)

primary_prevalence = (
    primary_target_attack
    /
    primary_target_rows
)


secondary_target_rows = int(
    stage24[
        "secondary_direction"
    ][
        "target_population"
    ][
        "rows"
    ]
)

secondary_target_attack = int(
    stage24[
        "secondary_direction"
    ][
        "target_population"
    ][
        "attack"
    ]
)

secondary_target_benign = int(
    stage24[
        "secondary_direction"
    ][
        "target_population"
    ][
        "benign"
    ]
)

secondary_prevalence = float(
    stage24[
        "secondary_direction"
    ][
        "target_population"
    ][
        "prevalence"
    ]
)


# Primary PR-normalized values
for key in [
    "bridge62_PUBLISHED",
    "bridge62_FLAG_CORRECTED",
    "bridge70_PUBLISHED",
    "bridge70_FLAG_CORRECTED",
]:

    cell = primary_cells[
        key
    ]

    if "pr_auc" in cell:

        cell[
            "_pr_normalized_publication"
        ] = (
            (
                float(
                    cell[
                        "pr_auc"
                    ]
                )
                -
                primary_prevalence
            )
            /
            (
                1.0
                -
                primary_prevalence
            )
        )


pub_cmp = comparison_lookup(
    primary_comparisons,
    "bridge70_PUBLISHED_minus_bridge62_PUBLISHED",
)

corr_cmp = comparison_lookup(
    primary_comparisons,
    "bridge70_FLAG_CORRECTED_minus_bridge62_FLAG_CORRECTED",
)

artifact_cmp = comparison_lookup(
    primary_comparisons,
    "bridge70_FLAG_CORRECTED_minus_bridge70_PUBLISHED",
)


print("=" * 120)
print("FINAL NUMBERS")
print("=" * 120)

print(
    "IDS2018 -> CICIDS2017:"
)

print(
    "  bridge62 PR/ROC:",
    primary_cells[
        "bridge62_PUBLISHED"
    ][
        "pr_auc"
    ],
    primary_cells[
        "bridge62_PUBLISHED"
    ][
        "roc_auc"
    ],
)

print(
    "  bridge70 PUBLISHED PR/ROC:",
    primary_cells[
        "bridge70_PUBLISHED"
    ][
        "pr_auc"
    ],
    primary_cells[
        "bridge70_PUBLISHED"
    ][
        "roc_auc"
    ],
)

print(
    "  bridge70 CORRECTED PR/ROC:",
    primary_cells[
        "bridge70_FLAG_CORRECTED"
    ][
        "pr_auc"
    ],
    primary_cells[
        "bridge70_FLAG_CORRECTED"
    ][
        "roc_auc"
    ],
)

print()

print(
    "CICIDS2017 -> IDS2018:"
)

print(
    "  bridge62 PR/ROC:",
    secondary_cells[
        "bridge62"
    ][
        "pr_auc"
    ],
    secondary_cells[
        "bridge62"
    ][
        "roc_auc"
    ],
)

print(
    "  bridge70 PR/ROC:",
    secondary_cells[
        "bridge70"
    ][
        "pr_auc"
    ],
    secondary_cells[
        "bridge70"
    ][
        "roc_auc"
    ],
)

print()


# ==============================================================================
# 6. CREATE DIRECTORIES
# ==============================================================================

for path in [
    PUB_RESULT_DIR,
    TABLE_DIR,
    FIG_DIR,
    SCRIPT_DIR,
    DOWNLOAD_DIR,
]:

    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# ==============================================================================
# 7. PUBLICATION TABLE 1 — BIDIRECTIONAL GENERALIZATION
# ==============================================================================

table1_rows = []


def add_table1(
    direction,
    target,
    bridge,
    semantics,
    pr_auc,
    roc_auc,
    brier,
    prevalence,
    pr_normalized,
    note="",
):

    table1_rows.append(
        {
            "direction":
                direction,

            "target":
                target,

            "bridge":
                bridge,

            "semantics":
                semantics,

            "target_prevalence":
                float(
                    prevalence
                ),

            "pr_auc":
                float(
                    pr_auc
                ),

            "pr_excess":
                float(
                    pr_auc
                    -
                    prevalence
                ),

            "pr_normalized":
                float(
                    pr_normalized
                ),

            "roc_auc":
                float(
                    roc_auc
                ),

            "brier":
                float(
                    brier
                ),

            "note":
                note,
        }
    )


add_table1(
    "IDS2018 -> CICIDS2017",
    "CICIDS2017 full effective population",
    "bridge62",
    "PUBLISHED",
    primary_cells[
        "bridge62_PUBLISHED"
    ][
        "pr_auc"
    ],
    primary_cells[
        "bridge62_PUBLISHED"
    ][
        "roc_auc"
    ],
    primary_cells[
        "bridge62_PUBLISHED"
    ][
        "brier"
    ],
    primary_prevalence,
    primary_cells[
        "bridge62_PUBLISHED"
    ][
        "_pr_normalized_publication"
    ],
)


add_table1(
    "IDS2018 -> CICIDS2017",
    "CICIDS2017 full effective population",
    "bridge62",
    "FLAG_CORRECTED",
    primary_cells[
        "bridge62_FLAG_CORRECTED"
    ][
        "pr_auc"
    ],
    primary_cells[
        "bridge62_FLAG_CORRECTED"
    ][
        "roc_auc"
    ],
    primary_cells[
        "bridge62_FLAG_CORRECTED"
    ][
        "brier"
    ],
    primary_prevalence,
    primary_cells[
        "bridge62_FLAG_CORRECTED"
    ][
        "_pr_normalized_publication"
    ],
    "Bitwise identical to bridge62 PUBLISHED because bridge62 excludes all eight aggregate flag-count fields.",
)


add_table1(
    "IDS2018 -> CICIDS2017",
    "CICIDS2017 full effective population",
    "bridge70",
    "PUBLISHED",
    primary_cells[
        "bridge70_PUBLISHED"
    ][
        "pr_auc"
    ],
    primary_cells[
        "bridge70_PUBLISHED"
    ][
        "roc_auc"
    ],
    primary_cells[
        "bridge70_PUBLISHED"
    ][
        "brier"
    ],
    primary_prevalence,
    primary_cells[
        "bridge70_PUBLISHED"
    ][
        "_pr_normalized_publication"
    ],
)


add_table1(
    "IDS2018 -> CICIDS2017",
    "CICIDS2017 full effective population",
    "bridge70",
    "FLAG_CORRECTED",
    primary_cells[
        "bridge70_FLAG_CORRECTED"
    ][
        "pr_auc"
    ],
    primary_cells[
        "bridge70_FLAG_CORRECTED"
    ][
        "roc_auc"
    ],
    primary_cells[
        "bridge70_FLAG_CORRECTED"
    ][
        "brier"
    ],
    primary_prevalence,
    primary_cells[
        "bridge70_FLAG_CORRECTED"
    ][
        "_pr_normalized_publication"
    ],
)


add_table1(
    "CICIDS2017 -> IDS2018",
    "IDS2018 02-28-2018",
    "bridge62",
    "FLAG_CORRECTED source semantics",
    secondary_cells[
        "bridge62"
    ][
        "pr_auc"
    ],
    secondary_cells[
        "bridge62"
    ][
        "roc_auc"
    ],
    secondary_cells[
        "bridge62"
    ][
        "brier"
    ],
    secondary_prevalence,
    secondary_cells[
        "bridge62"
    ][
        "pr_normalized"
    ],
)


add_table1(
    "CICIDS2017 -> IDS2018",
    "IDS2018 02-28-2018",
    "bridge70",
    "FLAG_CORRECTED source semantics",
    secondary_cells[
        "bridge70"
    ][
        "pr_auc"
    ],
    secondary_cells[
        "bridge70"
    ][
        "roc_auc"
    ],
    secondary_cells[
        "bridge70"
    ][
        "brier"
    ],
    secondary_prevalence,
    secondary_cells[
        "bridge70"
    ][
        "pr_normalized"
    ],
)


table1 = pd.DataFrame(
    table1_rows
)


TABLE1_CSV = (
    TABLE_DIR
    / "table24_1_bidirectional_generalization.csv"
)


table1.to_csv(
    TABLE1_CSV,
    index=False,
)


# ==============================================================================
# 8. PUBLICATION TABLE 2 — PAIRED BOOTSTRAP CONTRASTS
# ==============================================================================

table2_rows = []


def add_bootstrap_comparison(
    direction,
    comparison_name,
    source,
):

    for metric in [
        "pr_auc",
        "roc_auc",
        "brier",
    ]:

        metric_result = source[
            metric
        ]

        table2_rows.append(
            {
                "direction":
                    direction,

                "comparison":
                    comparison_name,

                "metric":
                    metric.upper(),

                "point_difference":
                    float(
                        metric_result[
                            "point_difference"
                        ]
                    ),

                "ci_2_5":
                    float(
                        metric_result[
                            "ci_2_5"
                        ]
                    ),

                "ci_97_5":
                    float(
                        metric_result[
                            "ci_97_5"
                        ]
                    ),

                "ci_excludes_zero":
                    bool(
                        metric_result[
                            "ci_excludes_zero"
                        ]
                    ),

                "replicates":
                    2000,

                "seed":
                    42,
            }
        )


add_bootstrap_comparison(
    "IDS2018 -> CICIDS2017",
    "bridge70 PUBLISHED - bridge62 PUBLISHED",
    pub_cmp[
        "bootstrap"
    ],
)


add_bootstrap_comparison(
    "IDS2018 -> CICIDS2017",
    "bridge70 FLAG_CORRECTED - bridge62 FLAG_CORRECTED",
    corr_cmp[
        "bootstrap"
    ],
)


add_bootstrap_comparison(
    "IDS2018 -> CICIDS2017",
    "bridge70 FLAG_CORRECTED - bridge70 PUBLISHED",
    artifact_cmp[
        "bootstrap"
    ],
)


add_bootstrap_comparison(
    "CICIDS2017 -> IDS2018",
    "bridge70 - bridge62",
    secondary_comparison,
)


table2 = pd.DataFrame(
    table2_rows
)


TABLE2_CSV = (
    TABLE_DIR
    / "table24_2_paired_bootstrap_contrasts.csv"
)


table2.to_csv(
    TABLE2_CSV,
    index=False,
)


# ==============================================================================
# 9. PUBLICATION TABLE 3 — SECONDARY THRESHOLD TRANSFER
# ==============================================================================

table3_rows = []


for bridge_name, receipt in [
    (
        "bridge62",
        secondary62,
    ),
    (
        "bridge70",
        secondary70,
    ),
]:

    thresholded = receipt[
        "metrics"
    ][
        "thresholded"
    ]


    for op_name in [
        "standard",
        "balanced",
        "security",
    ]:

        metrics = thresholded[
            op_name
        ]


        table3_rows.append(
            {
                "bridge":
                    bridge_name,

                "operating_point":
                    op_name.upper(),

                "threshold":
                    float(
                        metrics[
                            "threshold"
                        ]
                    ),

                "tp":
                    int(
                        metrics[
                            "tp"
                        ]
                    ),

                "tn":
                    int(
                        metrics[
                            "tn"
                        ]
                    ),

                "fp":
                    int(
                        metrics[
                            "fp"
                        ]
                    ),

                "fn":
                    int(
                        metrics[
                            "fn"
                        ]
                    ),

                "precision":
                    float(
                        metrics[
                            "precision"
                        ]
                    ),

                "recall":
                    float(
                        metrics[
                            "recall"
                        ]
                    ),

                "f1":
                    float(
                        metrics[
                            "f1"
                        ]
                    ),

                "f2":
                    float(
                        metrics[
                            "f2"
                        ]
                    ),

                "fpr":
                    float(
                        metrics[
                            "fpr"
                        ]
                    ),

                "fnr":
                    float(
                        metrics[
                            "fnr"
                        ]
                    ),
            }
        )


table3 = pd.DataFrame(
    table3_rows
)


TABLE3_CSV = (
    TABLE_DIR
    / "table24_3_secondary_threshold_transfer.csv"
)


table3.to_csv(
    TABLE3_CSV,
    index=False,
)


# ==============================================================================
# 10. PUBLICATION TABLE 4 — GOVERNANCE / DATASET POPULATIONS
# ==============================================================================

table4 = pd.DataFrame(
    [
        {
            "item":
                "Primary target population",

            "value":
                primary_target_rows,

            "detail":
                (
                    f"CICIDS2017: "
                    f"{primary_target_benign:,} benign / "
                    f"{primary_target_attack:,} attack / "
                    f"prevalence={primary_prevalence:.12f}"
                ),
        },

        {
            "item":
                "Secondary target population",

            "value":
                secondary_target_rows,

            "detail":
                (
                    f"IDS2018 Feb-28: "
                    f"{secondary_target_benign:,} benign / "
                    f"{secondary_target_attack:,} attack / "
                    f"prevalence={secondary_prevalence:.12f}"
                ),
        },

        {
            "item":
                "Scientific fits",

            "value":
                4,

            "detail":
                "4/4 completed; no additional fits authorized",
        },

        {
            "item":
                "Evaluable target openings",

            "value":
                6,

            "detail":
                "6/6 completed",
        },

        {
            "item":
                "Administratively cancelled target cells",

            "value":
                2,

            "detail":
                "bridge62/bridge70 GROUNDED_S4; not reallocated",
        },

        {
            "item":
                "Bootstrap replicates",

            "value":
                2000,

            "detail":
                "Paired stratified bootstrap; seed 42",
        },
    ]
)


TABLE4_CSV = (
    TABLE_DIR
    / "table24_4_governance_and_populations.csv"
)


table4.to_csv(
    TABLE4_CSV,
    index=False,
)


print("=" * 120)
print("PUBLICATION TABLES")
print("=" * 120)

for path in [
    TABLE1_CSV,
    TABLE2_CSV,
    TABLE3_CSV,
    TABLE4_CSV,
]:

    print(
        "[CREATED]",
        path.relative_to(
            REPO
        ),
    )

print()


# ==============================================================================
# 11. MARKDOWN TABLE DOCUMENT
# ==============================================================================

table1_md = table1.copy()

for col in [
    "target_prevalence",
    "pr_auc",
    "pr_excess",
    "pr_normalized",
    "roc_auc",
    "brier",
]:

    table1_md[
        col
    ] = table1_md[
        col
    ].map(
        lambda x:
            f"{x:.6f}"
    )


table2_md = table2.copy()

for col in [
    "point_difference",
    "ci_2_5",
    "ci_97_5",
]:

    table2_md[
        col
    ] = table2_md[
        col
    ].map(
        lambda x:
            f"{x:+.6f}"
    )


table3_md = table3.copy()

for col in [
    "threshold",
    "precision",
    "recall",
    "f1",
    "f2",
    "fpr",
    "fnr",
]:

    table3_md[
        col
    ] = table3_md[
        col
    ].map(
        lambda x:
            f"{x:.6f}"
    )


tables_md = f"""# Stage24 Publication Tables

These tables are derived exclusively from the frozen Stage24 result artifacts.
No model fitting, inference, target threshold selection, calibration, feature
selection, or post-target adaptation is performed during publication packaging.

## Table 24-1. Bidirectional cross-dataset generalization

{table1_md.to_markdown(index=False)}

**Interpretation note.** PR-AUC is reported together with each target
prevalence and normalized PR-AUC because the two directions use different
target prevalences. The two transfer directions are not averaged.

## Table 24-2. Paired stratified bootstrap contrasts

{table2_md.to_markdown(index=False)}

All differences use the stated left-minus-right orientation. Confidence
intervals are percentile 95% intervals from 2,000 paired class-stratified
bootstrap replicates with seed 42.

## Table 24-3. Source-selected threshold transfer to IDS2018 Feb-28

{table3_md.to_markdown(index=False)}

Thresholds were selected exclusively on CICIDS2017 Thursday source validation.
No IDS2018 target threshold search was performed.

## Table 24-4. Governance and evaluation populations

{table4.to_markdown(index=False)}

## Frozen artifact identities

- Stage24 final synthesis SHA256: `{final_sha}`
- Primary bootstrap SHA256: `{primary_bootstrap_sha}`
- Secondary bootstrap SHA256: `{secondary_bootstrap_sha}`
"""


write_text(
    DOC_TABLES_MD,
    tables_md,
)


# ==============================================================================
# 12. LATEX PUBLICATION TABLES
# ==============================================================================

def ci_tex(
    metric_result,
):

    return (
        f"{float(metric_result['point_difference']):+.6f} "
        f"[{float(metric_result['ci_2_5']):+.6f}, "
        f"{float(metric_result['ci_97_5']):+.6f}]"
    )


tables_tex = rf"""
% =============================================================================
% Stage24 Publication Tables
% Auto-generated from frozen Stage24 artifacts.
% No scientific model execution is performed by this file.
% =============================================================================

\begin{{table*}}[t]
\centering
\caption{{Bidirectional cross-dataset generalization under the frozen Stage24 protocol.}}
\label{{tab:stage24_bidirectional}}
\begin{{tabular}}{{llrrrr}}
\hline
Direction & Representation & PR-AUC & PR$_{{norm}}$ & ROC-AUC & Brier \\
\hline
IDS2018$\rightarrow$CICIDS2017
& bridge62
& {primary_cells['bridge62_PUBLISHED']['pr_auc']:.6f}
& {primary_cells['bridge62_PUBLISHED']['_pr_normalized_publication']:.6f}
& {primary_cells['bridge62_PUBLISHED']['roc_auc']:.6f}
& {primary_cells['bridge62_PUBLISHED']['brier']:.6f} \\

IDS2018$\rightarrow$CICIDS2017
& bridge70 Published
& {primary_cells['bridge70_PUBLISHED']['pr_auc']:.6f}
& {primary_cells['bridge70_PUBLISHED']['_pr_normalized_publication']:.6f}
& {primary_cells['bridge70_PUBLISHED']['roc_auc']:.6f}
& {primary_cells['bridge70_PUBLISHED']['brier']:.6f} \\

IDS2018$\rightarrow$CICIDS2017
& bridge70 Corrected
& {primary_cells['bridge70_FLAG_CORRECTED']['pr_auc']:.6f}
& {primary_cells['bridge70_FLAG_CORRECTED']['_pr_normalized_publication']:.6f}
& {primary_cells['bridge70_FLAG_CORRECTED']['roc_auc']:.6f}
& {primary_cells['bridge70_FLAG_CORRECTED']['brier']:.6f} \\

CICIDS2017$\rightarrow$IDS2018
& bridge62
& {secondary_cells['bridge62']['pr_auc']:.6f}
& {secondary_cells['bridge62']['pr_normalized']:.6f}
& {secondary_cells['bridge62']['roc_auc']:.6f}
& {secondary_cells['bridge62']['brier']:.6f} \\

CICIDS2017$\rightarrow$IDS2018
& bridge70
& {secondary_cells['bridge70']['pr_auc']:.6f}
& {secondary_cells['bridge70']['pr_normalized']:.6f}
& {secondary_cells['bridge70']['roc_auc']:.6f}
& {secondary_cells['bridge70']['brier']:.6f} \\
\hline
\end{{tabular}}
\end{{table*}}


\begin{{table*}}[t]
\centering
\caption{{Paired stratified bootstrap contrasts (2,000 replicates, seed 42). Values are point difference [95\% percentile CI].}}
\label{{tab:stage24_bootstrap}}
\begin{{tabular}}{{lrrr}}
\hline
Comparison & PR-AUC & ROC-AUC & Brier \\
\hline
Primary: bridge70 Published $-$ bridge62 Published
& {ci_tex(pub_cmp['bootstrap']['pr_auc'])}
& {ci_tex(pub_cmp['bootstrap']['roc_auc'])}
& {ci_tex(pub_cmp['bootstrap']['brier'])} \\

Primary: bridge70 Corrected $-$ bridge62 Corrected
& {ci_tex(corr_cmp['bootstrap']['pr_auc'])}
& {ci_tex(corr_cmp['bootstrap']['roc_auc'])}
& {ci_tex(corr_cmp['bootstrap']['brier'])} \\

Primary artifact: bridge70 Corrected $-$ Published
& {ci_tex(artifact_cmp['bootstrap']['pr_auc'])}
& {ci_tex(artifact_cmp['bootstrap']['roc_auc'])}
& {ci_tex(artifact_cmp['bootstrap']['brier'])} \\

Secondary: bridge70 $-$ bridge62
& {ci_tex(secondary_comparison['pr_auc'])}
& {ci_tex(secondary_comparison['roc_auc'])}
& {ci_tex(secondary_comparison['brier'])} \\
\hline
\end{{tabular}}
\end{{table*}}


\begin{{table*}}[t]
\centering
\caption{{Transfer of CICIDS2017 source-validation operating points to the IDS2018 Feb-28 target.}}
\label{{tab:stage24_threshold_transfer}}
\begin{{tabular}}{{llrrrrrr}}
\hline
Bridge & Operating point & Threshold & Precision & Recall & F1 & FPR & FNR \\
\hline
"""


for _, row in table3.iterrows():

    tables_tex += (
        f"{latex_escape(row['bridge'])} & "
        f"{latex_escape(row['operating_point'])} & "
        f"{row['threshold']:.3f} & "
        f"{row['precision']:.6f} & "
        f"{row['recall']:.6f} & "
        f"{row['f1']:.6f} & "
        f"{row['fpr']:.6f} & "
        f"{row['fnr']:.6f} \\\\\n"
    )


tables_tex += r"""
\hline
\end{tabular}
\end{table*}

% PR-AUC must be interpreted with target prevalence:
%   IDS2018 -> CICIDS2017 prevalence = """ + f"{primary_prevalence:.12f}" + r"""
%   CICIDS2017 -> IDS2018 prevalence = """ + f"{secondary_prevalence:.12f}" + r"""
%
% The two transfer directions MUST NOT be averaged.
"""


write_text(
    DOC_TABLES_TEX,
    tables_tex,
)


# ==============================================================================
# 13. FIGURE STYLE
# ==============================================================================

plt.rcParams.update(
    {
        "figure.dpi":
            140,

        "savefig.dpi":
            600,

        "font.size":
            10,

        "axes.titlesize":
            11,

        "axes.labelsize":
            10,

        "legend.fontsize":
            8,

        "xtick.labelsize":
            9,

        "ytick.labelsize":
            9,

        "axes.spines.top":
            False,

        "axes.spines.right":
            False,

        "pdf.fonttype":
            42,

        "ps.fonttype":
            42,
    }
)


def save_figure(
    fig,
    stem,
):

    png_path = (
        FIG_DIR
        /
        f"{stem}.png"
    )

    svg_path = (
        FIG_DIR
        /
        f"{stem}.svg"
    )

    fig.savefig(
        png_path,
        bbox_inches="tight",
    )

    fig.savefig(
        svg_path,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

    return (
        png_path,
        svg_path,
    )


# ==============================================================================
# 14. FIGURE 24-1 — NORMALIZED PR-AUC DIRECTIONALITY
# ==============================================================================

labels = [
    "18→17\nB62",
    "18→17\nB70 Pub",
    "18→17\nB70 Corr",
    "17→18\nB62",
    "17→18\nB70",
]


normalized_pr = [
    primary_cells[
        "bridge62_PUBLISHED"
    ][
        "_pr_normalized_publication"
    ],

    primary_cells[
        "bridge70_PUBLISHED"
    ][
        "_pr_normalized_publication"
    ],

    primary_cells[
        "bridge70_FLAG_CORRECTED"
    ][
        "_pr_normalized_publication"
    ],

    secondary_cells[
        "bridge62"
    ][
        "pr_normalized"
    ],

    secondary_cells[
        "bridge70"
    ][
        "pr_normalized"
    ],
]


fig, ax = plt.subplots(
    figsize=(
        7.2,
        4.2,
    )
)


x = np.arange(
    len(
        labels
    )
)


bars = ax.bar(
    x,
    normalized_pr,
)


ax.axhline(
    0.0,
    linewidth=0.8,
)


ax.set_xticks(
    x
)

ax.set_xticklabels(
    labels
)

ax.set_ylabel(
    "Normalized PR-AUC"
)

ax.set_title(
    "Stage24 cross-dataset ranking transfer is strongly direction-dependent"
)


for bar, value in zip(
    bars,
    normalized_pr,
):

    ax.text(
        bar.get_x()
        +
        bar.get_width()
        /
        2,
        value
        +
        0.008,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )


ax.set_ylim(
    min(
        -0.03,
        min(
            normalized_pr
        )
        -
        0.02,
    ),
    max(
        normalized_pr
    )
    +
    0.09,
)


fig.tight_layout()


FIG1_PNG, FIG1_SVG = save_figure(
    fig,
    "fig24_1_normalized_pr_auc_directionality",
)


# ==============================================================================
# 15. FIGURE 24-2 — ROC-AUC DIRECTIONALITY
# ==============================================================================

roc_values = [
    primary_cells[
        "bridge62_PUBLISHED"
    ][
        "roc_auc"
    ],

    primary_cells[
        "bridge70_PUBLISHED"
    ][
        "roc_auc"
    ],

    primary_cells[
        "bridge70_FLAG_CORRECTED"
    ][
        "roc_auc"
    ],

    secondary_cells[
        "bridge62"
    ][
        "roc_auc"
    ],

    secondary_cells[
        "bridge70"
    ][
        "roc_auc"
    ],
]


fig, ax = plt.subplots(
    figsize=(
        7.2,
        4.2,
    )
)


bars = ax.bar(
    x,
    roc_values,
)


ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1.0,
    label="Chance ROC-AUC = 0.5",
)


ax.set_xticks(
    x
)

ax.set_xticklabels(
    labels
)

ax.set_ylabel(
    "ROC-AUC"
)

ax.set_ylim(
    0.45,
    0.80,
)

ax.set_title(
    "Bidirectional ROC-AUC under frozen cross-dataset transfer"
)

ax.legend(
    frameon=False
)


for bar, value in zip(
    bars,
    roc_values,
):

    ax.text(
        bar.get_x()
        +
        bar.get_width()
        /
        2,
        value
        +
        0.008,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )


fig.tight_layout()


FIG2_PNG, FIG2_SVG = save_figure(
    fig,
    "fig24_2_roc_auc_directionality",
)


# ==============================================================================
# 16. FIGURE 24-3 — FOREST PLOT OF PAIRED EFFECTS
# ==============================================================================

forest_rows = [
    (
        "Primary B70 Pub − B62",
        pub_cmp[
            "bootstrap"
        ],
    ),

    (
        "Primary B70 Corr − B62",
        corr_cmp[
            "bootstrap"
        ],
    ),

    (
        "Primary B70 Corr − Pub",
        artifact_cmp[
            "bootstrap"
        ],
    ),

    (
        "Secondary B70 − B62",
        secondary_comparison,
    ),
]


fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        12.0,
        4.8,
    ),
)


for ax, metric, title in [
    (
        axes[
            0
        ],
        "pr_auc",
        "PR-AUC difference",
    ),
    (
        axes[
            1
        ],
        "roc_auc",
        "ROC-AUC difference",
    ),
    (
        axes[
            2
        ],
        "brier",
        "Brier difference",
    ),
]:

    y_pos = np.arange(
        len(
            forest_rows
        )
    )


    points = []

    lower = []

    upper = []


    for _, source in forest_rows:

        r = source[
            metric
        ]

        point = float(
            r[
                "point_difference"
            ]
        )

        lo = float(
            r[
                "ci_2_5"
            ]
        )

        hi = float(
            r[
                "ci_97_5"
            ]
        )


        points.append(
            point
        )

        lower.append(
            point
            -
            lo
        )

        upper.append(
            hi
            -
            point
        )


    ax.errorbar(
        points,
        y_pos,
        xerr=np.array(
            [
                lower,
                upper,
            ]
        ),
        fmt="o",
        capsize=3,
    )


    ax.axvline(
        0.0,
        linestyle="--",
        linewidth=0.9,
    )


    ax.set_title(
        title
    )


    ax.set_yticks(
        y_pos
    )


    if metric == "pr_auc":

        ax.set_yticklabels(
            [
                name
                for name, _ in forest_rows
            ]
        )

    else:

        ax.set_yticklabels(
            []
        )


    ax.invert_yaxis()


fig.suptitle(
    "Paired Stage24 effect estimates with 95% bootstrap intervals",
    y=1.02,
)


fig.tight_layout()


FIG3_PNG, FIG3_SVG = save_figure(
    fig,
    "fig24_3_paired_effects_forest",
)


# ==============================================================================
# 17. FIGURE 24-4 — SECONDARY THRESHOLD TRANSFER
# ==============================================================================

ops_order = [
    "STANDARD",
    "BALANCED",
    "SECURITY",
]


b62_threshold = (
    table3[
        table3[
            "bridge"
        ]
        ==
        "bridge62"
    ]
    .set_index(
        "operating_point"
    )
    .loc[
        ops_order
    ]
)


b70_threshold = (
    table3[
        table3[
            "bridge"
        ]
        ==
        "bridge70"
    ]
    .set_index(
        "operating_point"
    )
    .loc[
        ops_order
    ]
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        10.2,
        4.3,
    ),
)


xop = np.arange(
    len(
        ops_order
    )
)


width = 0.36


axes[
    0
].bar(
    xop
    -
    width
    /
    2,
    b62_threshold[
        "recall"
    ].to_numpy()
    *
    100.0,
    width,
    label="bridge62",
)


axes[
    0
].bar(
    xop
    +
    width
    /
    2,
    b70_threshold[
        "recall"
    ].to_numpy()
    *
    100.0,
    width,
    label="bridge70",
)


axes[
    0
].set_xticks(
    xop
)

axes[
    0
].set_xticklabels(
    ops_order
)

axes[
    0
].set_ylabel(
    "Attack recall (%)"
)

axes[
    0
].set_title(
    "Frozen threshold recall"
)

axes[
    0
].legend(
    frameon=False
)


axes[
    1
].bar(
    xop
    -
    width
    /
    2,
    b62_threshold[
        "fpr"
    ].to_numpy()
    *
    100.0,
    width,
    label="bridge62",
)


axes[
    1
].bar(
    xop
    +
    width
    /
    2,
    b70_threshold[
        "fpr"
    ].to_numpy()
    *
    100.0,
    width,
    label="bridge70",
)


axes[
    1
].set_xticks(
    xop
)

axes[
    1
].set_xticklabels(
    ops_order
)

axes[
    1
].set_ylabel(
    "False-positive rate (%)"
)

axes[
    1
].set_title(
    "Frozen threshold FPR"
)


fig.suptitle(
    "CICIDS2017-selected operating points transferred to IDS2018 Feb-28",
    y=1.02,
)


fig.tight_layout()


FIG4_PNG, FIG4_SVG = save_figure(
    fig,
    "fig24_4_secondary_threshold_transfer",
)


print("=" * 120)
print("PUBLICATION FIGURES")
print("=" * 120)


for path in [
    FIG1_PNG,
    FIG1_SVG,
    FIG2_PNG,
    FIG2_SVG,
    FIG3_PNG,
    FIG3_SVG,
    FIG4_PNG,
    FIG4_SVG,
]:

    print(
        "[CREATED]",
        path.relative_to(
            REPO
        ),
    )


print()


# ==============================================================================
# 18. MANUSCRIPT-READY RESULTS / DISCUSSION / LIMITATIONS / CONTRIBUTIONS
# ==============================================================================

artifact_pr = artifact_cmp[
    "bootstrap"
][
    "pr_auc"
]

artifact_roc = artifact_cmp[
    "bootstrap"
][
    "roc_auc"
]

artifact_brier = artifact_cmp[
    "bootstrap"
][
    "brier"
]


secondary_pr = secondary_comparison[
    "pr_auc"
]

secondary_roc = secondary_comparison[
    "roc_auc"
]

secondary_brier = secondary_comparison[
    "brier"
]


b62_target_prior_brier = float(
    secondary62[
        "metrics"
    ][
        "null_anchors"
    ][
        "target_prevalence_chance_anchor"
    ][
        "brier"
    ]
)


b62_ece = float(
    secondary62[
        "metrics"
    ][
        "calibration"
    ][
        "ece_10"
    ]
)


b70_ece = float(
    secondary70[
        "metrics"
    ][
        "calibration"
    ][
        "ece_10"
    ]
)


manuscript_md = f"""# Stage24 Manuscript Integration

## Cross-Dataset Generalization and Artifact-Sensitivity Audit

Scientific execution for Stage24 is closed.

Final scientific commit before this publication package:

`{EXPECTED_PARENT}`

Frozen result SHA256:

`{final_sha}`

No section below authorizes additional model fitting, target inference,
threshold search, calibration, feature modification, or post-target model
selection.

---

# A. Results

## A.1 Bidirectional cross-dataset ranking generalization

Stage24 evaluated transfer in two separately reported directions. The
IDS2018-to-CICIDS2017 direction used the complete effective CICIDS2017
population of **{primary_target_rows:,} flows**, containing
**{primary_target_attack:,} attacks** and attack prevalence
**{primary_prevalence:.6f}**. The reciprocal CICIDS2017-to-IDS2018 direction
used the frozen K79-clean IDS2018 Feb-28 population of
**{secondary_target_rows:,} flows**, including **{secondary_target_attack:,}
attacks** and attack prevalence **{secondary_prevalence:.6f}**.

In the IDS2018-to-CICIDS2017 direction, bridge62 achieved PR-AUC
**{primary_cells['bridge62_PUBLISHED']['pr_auc']:.6f}** and ROC-AUC
**{primary_cells['bridge62_PUBLISHED']['roc_auc']:.6f}**. Including the eight
aggregate flag-count fields under the published bridge70 interpretation yielded
PR-AUC **{primary_cells['bridge70_PUBLISHED']['pr_auc']:.6f}** and ROC-AUC
**{primary_cells['bridge70_PUBLISHED']['roc_auc']:.6f}**. Under the frozen
flag-corrected interpretation, bridge70 achieved PR-AUC
**{primary_cells['bridge70_FLAG_CORRECTED']['pr_auc']:.6f}** and ROC-AUC
**{primary_cells['bridge70_FLAG_CORRECTED']['roc_auc']:.6f}**.

The reciprocal CICIDS2017-to-IDS2018 transfer was substantially weaker.
Bridge62 reached PR-AUC **{secondary_cells['bridge62']['pr_auc']:.6f}** and
ROC-AUC **{secondary_cells['bridge62']['roc_auc']:.6f}**, while bridge70 reached
PR-AUC **{secondary_cells['bridge70']['pr_auc']:.6f}** and ROC-AUC
**{secondary_cells['bridge70']['roc_auc']:.6f}**. The IDS2018 Feb-28 PR-AUC
chance anchor defined by target attack prevalence is **{secondary_prevalence:.6f}**.
Consequently, normalized PR-AUC is only
**{secondary_cells['bridge62']['pr_normalized']:.6f}** for bridge62 and
**{secondary_cells['bridge70']['pr_normalized']:.6f}** for bridge70, compared
with approximately
**{primary_cells['bridge62_PUBLISHED']['_pr_normalized_publication']:.6f}**
for bridge62 in the opposite direction.

These results establish a pronounced directional asymmetry under the frozen
protocol: the IDS2018-trained representation retains meaningful ranking signal
on CICIDS2017, whereas models trained on CICIDS2017 Monday-Wednesday transfer
only marginally above the Feb-28 prevalence/chance ranking anchor when evaluated
on IDS2018.

## A.2 Bridge-feature sensitivity

For IDS2018-to-CICIDS2017 under published target semantics, adding the eight
aggregate flag-count features changed PR-AUC by
**{pub_cmp['bootstrap']['pr_auc']['point_difference']:+.6f}**
(95% CI
[{pub_cmp['bootstrap']['pr_auc']['ci_2_5']:+.6f},
{pub_cmp['bootstrap']['pr_auc']['ci_97_5']:+.6f}]),
ROC-AUC by
**{pub_cmp['bootstrap']['roc_auc']['point_difference']:+.6f}**
([{pub_cmp['bootstrap']['roc_auc']['ci_2_5']:+.6f},
{pub_cmp['bootstrap']['roc_auc']['ci_97_5']:+.6f}]),
and Brier score by
**{pub_cmp['bootstrap']['brier']['point_difference']:+.6f}**
([{pub_cmp['bootstrap']['brier']['ci_2_5']:+.6f},
{pub_cmp['bootstrap']['brier']['ci_97_5']:+.6f}]).

In the reciprocal direction, bridge70-minus-bridge62 produced PR-AUC difference
**{secondary_pr['point_difference']:+.6f}**
(95% CI [{secondary_pr['ci_2_5']:+.6f}, {secondary_pr['ci_97_5']:+.6f}]),
ROC-AUC difference
**{secondary_roc['point_difference']:+.6f}**
([{secondary_roc['ci_2_5']:+.6f}, {secondary_roc['ci_97_5']:+.6f}]),
and Brier difference
**{secondary_brier['point_difference']:+.6f}**
([{secondary_brier['ci_2_5']:+.6f}, {secondary_brier['ci_97_5']:+.6f}]).

The reciprocal ROC-AUC interval includes zero, while the very small PR-AUC and
Brier differences exclude zero. Thus, the flag-count fields do not rescue the
weak reciprocal ranking transfer.

## A.3 Extractor and serialization sensitivity

Stage24 separately audited the known aggregate-flag serialization mismatch in
CICIDS2017. Holding the target population, source model, and bridge70 feature
count fixed, replacing the published aggregate-flag interpretation with the
frozen corrected mapping changed PR-AUC by
**{artifact_pr['point_difference']:+.6f}**
(95% CI [{artifact_pr['ci_2_5']:+.6f}, {artifact_pr['ci_97_5']:+.6f}]),
ROC-AUC by
**{artifact_roc['point_difference']:+.6f}**
([{artifact_roc['ci_2_5']:+.6f}, {artifact_roc['ci_97_5']:+.6f}]),
and Brier score by
**{artifact_brier['point_difference']:+.6f}**
([{artifact_brier['ci_2_5']:+.6f}, {artifact_brier['ci_97_5']:+.6f}]).

All three paired intervals exclude zero. Therefore, extractor/serialization
semantics are not merely a bookkeeping concern: they measurably change the
reported cross-dataset behavior of the same nominal bridge70 representation.

## A.4 Frozen operating-point transfer

The reciprocal experiment also exposes severe operating-point transfer
failure. Source-validation thresholds selected exclusively on CICIDS2017
Thursday were transferred without target retuning.

For bridge62, target recall is
**{secondary62['metrics']['thresholded']['standard']['recall']:.6f}** at the
standard threshold,
**{secondary62['metrics']['thresholded']['balanced']['recall']:.6f}** at the
balanced threshold, and
**{secondary62['metrics']['thresholded']['security']['recall']:.6f}** at the
security threshold.

For bridge70, corresponding recall is
**{secondary70['metrics']['thresholded']['standard']['recall']:.6f}**,
**{secondary70['metrics']['thresholded']['balanced']['recall']:.6f}**, and
**{secondary70['metrics']['thresholded']['security']['recall']:.6f}**.

Thus, even the source-selected security thresholds detect less than 0.14% of
IDS2018 Feb-28 attacks for bridge62 and less than 0.09% for bridge70. These poor
operating points are retained as results rather than repaired through target
threshold search.

## A.5 Calibration transfer

Reciprocal cross-dataset probability calibration also deteriorates. Bridge62
has Brier score **{secondary_cells['bridge62']['brier']:.6f}** and 10-bin ECE
**{b62_ece:.6f}**, while bridge70 has Brier score
**{secondary_cells['bridge70']['brier']:.6f}** and ECE **{b70_ece:.6f}**.
For comparison, the target-prevalence constant predictor has Brier score
**{b62_target_prior_brier:.6f}**. Both transferred models therefore have worse
Brier score than this target-prior reference, although the target-prior value is
reported as a diagnostic anchor rather than as a trained competing model.

---

# B. Discussion

Stage24 shows that cross-dataset IDS generalization cannot be summarized by a
single portability number. The strongest observation is directional asymmetry.
An IDS2018-derived representation retains substantial threshold-independent
ranking signal when transferred to the effective CICIDS2017 population, but the
reciprocal CICIDS2017-trained models evaluated on the frozen IDS2018 Feb-28
target are only slightly above chance in both normalized PR-AUC and ROC-AUC.
Because the two directions use different source models, target populations, and
prevalence structures, they are intentionally not averaged.

This asymmetry cautions against treating successful transfer from dataset A to
dataset B as evidence that the two datasets are interchangeable. A classifier
can exploit structures that survive one transfer direction but are absent,
reweighted, or encoded differently in the reciprocal direction. Cross-dataset
evaluation should therefore be bidirectional when both source/target roles are
scientifically meaningful.

The bridge ablation provides a second result. Adding the aggregate flag-count
features changes performance, but their effect is not uniformly beneficial.
Under IDS2018-to-CICIDS2017 transfer, bridge70 increases ROC-AUC but decreases
PR-AUC relative to bridge62. Under reciprocal transfer, bridge70 does not yield
a resolved ROC-AUC improvement and slightly decreases PR-AUC. More features are
therefore not automatically more portable features.

The serialization audit strengthens this interpretation. Correcting the
CICIDS2017 aggregate-flag semantics changes all three paired metrics in the
primary bridge70 comparison. A cross-dataset experiment can consequently
attribute performance changes to domain shift when part of the difference
actually arises from feature-extractor semantics. Cross-dataset IDS studies
should document semantic feature alignment rather than relying only on
similarly named columns.

Finally, Stage24 again separates ranking transfer from operating-point transfer.
The reciprocal models are already weak in ranking terms, but their source-frozen
probability thresholds degrade even more severely, missing virtually all
attacks on IDS2018 Feb-28. Thresholds and probability calibration therefore
require separate validation from threshold-independent discrimination.

---

# C. Limitations and Threats to Validity

## C.1 Dataset scope

The study evaluates two related benchmark families, CSE-CIC-IDS2018 and
CICIDS2017. Both were produced within the CIC research ecosystem and may share
traffic-generation or feature-extraction characteristics. The measured
asymmetry should therefore not be assumed to quantify transfer to unrelated
enterprise, ISP, cloud, IoT, or operational network traffic.

## C.2 Single frozen model strategy

The primary direction uses the frozen Stage22R classical ensemble lineage,
whereas the reciprocal direction uses preregistered XGBoost-only source models.
This design answers the declared Stage24 questions but does not establish that
the same directional asymmetry magnitude would occur for every model family,
representation learner, or neural architecture.

## C.3 Different target prevalences

CICIDS2017 and IDS2018 Feb-28 have different attack prevalences. PR-AUC is
therefore interpreted together with target prevalence, normalized PR-AUC,
ROC-AUC, and calibration metrics. Raw PR-AUC values from the two directions
must not be treated as directly interchangeable without this context.

## C.4 Aggregate-flag semantic correction

The flag-corrected bridge70 result is a preregistered semantic audit, not a
post-hoc optimization. Seven physical CICIDS2017 aggregate-flag mappings differ
between published and corrected representations, while ACK is invariant.
Results should not be generalized to other CICIDS2017 exports without
confirming their extractor lineage.

## C.5 GROUNDED_S4 structural unavailability

The two preregistered GROUNDED_S4 primary cells were not evaluated. Exact
Stage20 S4 membership could not be recovered as exact physical target-table rows
from durable artifacts without introducing a new heuristic matching rule after
protocol freeze. The cells were therefore administratively cancelled before
opening. No fuzzy or inferred substitute was used, and their two opening slots
were not reallocated.

## C.6 Threshold transfer

Target thresholds were deliberately not retuned. Poor target recall therefore
represents source-to-target operating-point transfer failure under the frozen
protocol, not the best threshold achievable if IDS2018 target labels were made
available for adaptation.

## C.7 Mechanistic interpretation

The experiment establishes distributional and semantic sensitivity but does not
uniquely identify the causal mechanism. The observed differences may include
covariate shift, attack-mixture shift, label-prior effects, extractor behavior,
environmental differences, or interactions among these factors. Stage24 does
not prove any one of these mechanisms in isolation.

## C.8 Bootstrap scope

The paired 95% intervals quantify uncertainty with respect to row-level
class-stratified resampling of the frozen target populations. They do not
represent uncertainty over independently collected networks, organizations,
capture campaigns, or dataset-generation processes.

---

# D. Contributions

Stage24 adds the following publication-level contributions:

1. **Bidirectional cross-dataset evaluation.** The study evaluates both
   IDS2018-to-CICIDS2017 and CICIDS2017-to-IDS2018 transfer rather than treating
   one direction as sufficient evidence of dataset portability.

2. **Directionality as an explicit result.** The frozen experiment demonstrates
   that cross-dataset generalization can be strongly asymmetric, with substantial
   ranking transfer in one direction and near-chance reciprocal transfer.

3. **Semantic bridge ablation.** The 62-feature bridge separates a core common
   representation from eight aggregate flag-count features, allowing their
   contribution to cross-dataset portability to be measured.

4. **Extractor/serialization artifact audit.** Published and corrected
   CICIDS2017 aggregate-flag interpretations are compared on the same frozen
   target rows, showing statistically resolved changes in PR-AUC, ROC-AUC, and
   Brier score.

5. **Validation-safe target governance.** Source-validation thresholds are
   transferred without target tuning, scientific fit and target-opening budgets
   are preregistered, and unavailable GROUNDED_S4 cells are cancelled rather
   than replaced with post-result heuristics.

6. **Paired uncertainty analysis.** Feature-bridge and semantic-artifact effects
   are quantified with 2,000-replicate paired class-stratified bootstrap
   intervals using frozen target prediction vectors.

7. **Separation of ranking, calibration, and operating-point transfer.**
   Stage24 reports PR-AUC/ROC-AUC, Brier/ECE, and fixed-threshold confusion
   behavior separately, exposing failures that would be hidden by a single
   metric.

---

# E. Contribution Text for Abstract / Introduction

A validation-safe bidirectional cross-dataset audit between CSE-CIC-IDS2018 and
CICIDS2017 revealed pronounced transfer asymmetry. IDS2018-derived models
retained substantial ranking discrimination on CICIDS2017
(PR-AUC {primary_cells['bridge62_PUBLISHED']['pr_auc']:.3f},
ROC-AUC {primary_cells['bridge62_PUBLISHED']['roc_auc']:.3f} for the 62-feature
bridge), whereas reciprocal CICIDS2017-to-IDS2018 transfer was near the Feb-28
chance/prevalence anchor
(PR-AUC {secondary_cells['bridge62']['pr_auc']:.3f},
ROC-AUC {secondary_cells['bridge62']['roc_auc']:.3f}). A semantic audit further
showed that correcting CICIDS2017 aggregate TCP-flag serialization changed
bridge70 PR-AUC by {artifact_pr['point_difference']:+.4f} and ROC-AUC by
{artifact_roc['point_difference']:+.4f}, with paired bootstrap intervals
excluding zero. The results show that cross-dataset IDS portability depends not
only on domain shift but also on transfer direction, feature-bridge design, and
extractor semantics.

---

# F. Publication-Safe Claims

1. Stage24 completed all four preregistered scientific fits.
2. Six evaluable target openings were completed; two GROUNDED_S4 cells were
   cancelled before opening and were not reallocated.
3. IDS2018-to-CICIDS2017 transfer substantially exceeds chance ranking.
4. CICIDS2017-to-IDS2018 transfer on IDS2018 Feb-28 is only marginally above
   chance/prevalence ranking.
5. Cross-dataset generalization is strongly direction-dependent in this frozen
   experiment.
6. bridge70 does not uniformly improve cross-dataset portability over bridge62.
7. Correcting aggregate flag serialization measurably changes primary bridge70
   PR-AUC, ROC-AUC, and Brier score.
8. Source-selected probability thresholds fail to transfer operationally to the
   reciprocal IDS2018 target.
9. No target threshold tuning, calibration, feature search, or target-guided
   retraining was performed.
10. The two transfer directions must remain separately reported.

---

# G. Claims That Must Not Appear

1. CICIDS2017 and IDS2018 are universally incompatible datasets.
2. Stage24 proves a unique causal mechanism such as concept drift.
3. The transfer directions can be averaged into one cross-dataset score.
4. GROUNDED_S4 was successfully reconstructed or evaluated.
5. A fuzzy substitute was used for unavailable GROUNDED_S4 membership.
6. bridge62 is universally superior to bridge70.
7. bridge70 is universally superior to bridge62.
8. Target thresholds were optimized on IDS2018.
9. Stage24 evaluates every possible IDS model family.
10. Row-bootstrap confidence intervals represent uncertainty across independent
    real-world organizations or deployment environments.

---

# H. Recommended Main-Manuscript Figures

**Figure 24-1**
`figures/stage24_cross_dataset/fig24_1_normalized_pr_auc_directionality.png`

Normalized PR-AUC reveals the directional asymmetry while accounting for
different target prevalence.

**Figure 24-2**
`figures/stage24_cross_dataset/fig24_2_roc_auc_directionality.png`

ROC-AUC provides the prevalence-insensitive companion view.

**Figure 24-3**
`figures/stage24_cross_dataset/fig24_3_paired_effects_forest.png`

Paired bootstrap intervals quantify bridge and extractor-semantic effects.

**Supplementary Figure 24-S1**
`figures/stage24_cross_dataset/fig24_4_secondary_threshold_transfer.png`

The source-selected operating points show severe reciprocal threshold-transfer
failure.

---

# I. Recommended Manuscript Placement

- **Introduction / Contributions:** Section D or E.
- **Results:** Section A + Tables 24-1 through 24-3 + Figures 24-1 through 24-3.
- **Discussion:** Section B.
- **Limitations / Threats to Validity:** Section C.
- **Supplementary material:** Figure 24-S1 and governance table.
- **Reproducibility package:** frozen Stage24 result JSONs, prediction hashes,
  bootstrap artifacts, publication tables, and exported Stage24 notebook/script.
"""


write_text(
    DOC_MANUSCRIPT,
    manuscript_md,
)


# ==============================================================================
# 19. LATEX MANUSCRIPT SNIPPET
# ==============================================================================

manuscript_tex = rf"""
% =============================================================================
% Stage24 Manuscript Integration
% Insert into an IEEE manuscript as appropriate.
% =============================================================================

\subsection{{Bidirectional Cross-Dataset Generalization}}

Stage24 evaluated transfer in two separately reported directions. The
IDS2018-to-CICIDS2017 direction used {primary_target_rows:,} effective
CICIDS2017 flows with attack prevalence {primary_prevalence:.6f}, whereas the
reciprocal CICIDS2017-to-IDS2018 direction used the frozen K79-clean IDS2018
Feb-28 population of {secondary_target_rows:,} flows with attack prevalence
{secondary_prevalence:.6f}.

For IDS2018-to-CICIDS2017, bridge62 achieved PR-AUC
{primary_cells['bridge62_PUBLISHED']['pr_auc']:.6f} and ROC-AUC
{primary_cells['bridge62_PUBLISHED']['roc_auc']:.6f}. Bridge70 under the
published semantics achieved PR-AUC
{primary_cells['bridge70_PUBLISHED']['pr_auc']:.6f} and ROC-AUC
{primary_cells['bridge70_PUBLISHED']['roc_auc']:.6f}, while the flag-corrected
bridge70 representation achieved PR-AUC
{primary_cells['bridge70_FLAG_CORRECTED']['pr_auc']:.6f} and ROC-AUC
{primary_cells['bridge70_FLAG_CORRECTED']['roc_auc']:.6f}.

The reciprocal transfer was substantially weaker. CICIDS2017-trained bridge62
reached PR-AUC {secondary_cells['bridge62']['pr_auc']:.6f} and ROC-AUC
{secondary_cells['bridge62']['roc_auc']:.6f}; bridge70 reached PR-AUC
{secondary_cells['bridge70']['pr_auc']:.6f} and ROC-AUC
{secondary_cells['bridge70']['roc_auc']:.6f}. The IDS2018 Feb-28 PR-AUC
prevalence anchor was {secondary_prevalence:.6f}. The two transfer directions
are therefore retained separately rather than averaged.

\subsection{{Extractor-Semantic Sensitivity}}

Within the primary bridge70 evaluation, replacing the published CICIDS2017
aggregate-flag interpretation with the frozen corrected mapping changed PR-AUC
by {artifact_pr['point_difference']:+.6f}
(95\% CI [{artifact_pr['ci_2_5']:+.6f}, {artifact_pr['ci_97_5']:+.6f}]),
ROC-AUC by {artifact_roc['point_difference']:+.6f}
([{artifact_roc['ci_2_5']:+.6f}, {artifact_roc['ci_97_5']:+.6f}]), and Brier
score by {artifact_brier['point_difference']:+.6f}
([{artifact_brier['ci_2_5']:+.6f}, {artifact_brier['ci_97_5']:+.6f}]).
All three intervals exclude zero, demonstrating measurable sensitivity to
extractor/serialization semantics.

\subsection{{Operating-Point Transfer}}

CICIDS2017 Thursday source-validation thresholds transferred poorly to IDS2018
Feb-28. For bridge62, standard, balanced, and security recall were
{secondary62['metrics']['thresholded']['standard']['recall']:.6f},
{secondary62['metrics']['thresholded']['balanced']['recall']:.6f}, and
{secondary62['metrics']['thresholded']['security']['recall']:.6f},
respectively. Corresponding bridge70 recall values were
{secondary70['metrics']['thresholded']['standard']['recall']:.6f},
{secondary70['metrics']['thresholded']['balanced']['recall']:.6f}, and
{secondary70['metrics']['thresholded']['security']['recall']:.6f}.
No target threshold search was permitted.

\subsection{{Discussion}}

The Stage24 results demonstrate that cross-dataset IDS portability can be
strongly directional even when feature names appear compatible across benchmark
families. Successful IDS2018-to-CICIDS2017 transfer did not imply reciprocal
CICIDS2017-to-IDS2018 generalization. The bridge ablation also shows that adding
aggregate flag-count fields is not uniformly beneficial, while the serialization
audit demonstrates that semantically misaligned extractor outputs can
materially change reported cross-dataset results. Ranking discrimination,
probability calibration, and operating-point transfer should therefore be
evaluated as separate properties.

\subsection{{Limitations and Threats to Validity}}

The experiment covers two related CIC benchmark families and one frozen model
strategy per direction, so the observed magnitude of asymmetry should not be
generalized to unrelated operational networks or all model families. Target
prevalence differs between directions, requiring PR-AUC to be interpreted
together with prevalence, normalized PR-AUC, and ROC-AUC. Two preregistered
GROUNDED\_S4 cells were administratively cancelled because exact durable
physical-row membership could not be reconstructed without a new post-freeze
heuristic; no substitute population was used. Finally, paired row-bootstrap
intervals characterize uncertainty for the frozen target populations and do
not represent variation across independently collected organizations or
capture campaigns.

\subsection{{Stage24 Contribution}}

A validation-safe bidirectional cross-dataset audit revealed pronounced
transfer asymmetry between CSE-CIC-IDS2018 and CICIDS2017, quantified the
contribution of a common 62-feature bridge versus aggregate TCP-flag fields,
demonstrated statistically resolved sensitivity to CICIDS2017 flag
serialization semantics, and showed that source-selected operating thresholds
can fail catastrophically under reciprocal transfer without target-guided
adaptation.
"""


write_text(
    DOC_MANUSCRIPT_TEX,
    manuscript_tex,
)


# ==============================================================================
# 20. PUBLICATION CLOSEOUT / REPRODUCIBILITY INDEX
# ==============================================================================

closeout_md = f"""# Stage24 Publication Closeout

## Status

**Stage24 scientific execution: CLOSED**

- Scientific fits: **4 / 4**
- Evaluable target openings: **6 / 6**
- Administrative GROUNDED_S4 cancellations: **2**
- Cancelled slots reallocated: **No**
- Remaining Stage24 model fits: **0**
- Remaining Stage24 evaluable target openings: **0**

## Scientific final freeze

Commit before publication packaging:

`{EXPECTED_PARENT}`

Stage24 final synthesis SHA256:

`{final_sha}`

## Publication package

### Manuscript

- `docs/STAGE24_MANUSCRIPT_INTEGRATION.md`
- `docs/STAGE24_MANUSCRIPT_INTEGRATION.tex`
- `docs/STAGE24_PUBLICATION_TABLES.md`
- `docs/STAGE24_PUBLICATION_TABLES.tex`

### Tables

- `results/stage24_cross_dataset/stage24_publication_package/tables/table24_1_bidirectional_generalization.csv`
- `results/stage24_cross_dataset/stage24_publication_package/tables/table24_2_paired_bootstrap_contrasts.csv`
- `results/stage24_cross_dataset/stage24_publication_package/tables/table24_3_secondary_threshold_transfer.csv`
- `results/stage24_cross_dataset/stage24_publication_package/tables/table24_4_governance_and_populations.csv`

### Figures

- `figures/stage24_cross_dataset/fig24_1_normalized_pr_auc_directionality.png`
- `figures/stage24_cross_dataset/fig24_2_roc_auc_directionality.png`
- `figures/stage24_cross_dataset/fig24_3_paired_effects_forest.png`
- `figures/stage24_cross_dataset/fig24_4_secondary_threshold_transfer.png`

Every figure is also exported as SVG.

### Reproducible notebook/script export

- `scripts/stage24/stage24_cross_dataset_generalization.ipynb`
- `scripts/stage24/stage24_cross_dataset_generalization.py`
- `scripts/stage24/README.md`

## Scientific interpretation

Stage24 demonstrates strongly asymmetric bidirectional cross-dataset
generalization. IDS2018-derived models retain substantial ranking signal on
CICIDS2017, whereas reciprocal CICIDS2017-to-IDS2018 transfer on the frozen
Feb-28 population is only marginally above chance/prevalence ranking.

It additionally shows that aggregate flag-count semantics materially influence
reported transfer performance and that source-validation operating thresholds
do not transfer reliably.

## Governance

Publication packaging is strictly post-processing of already-frozen Stage24
artifacts. It does not authorize any further Stage24 fitting, target inference,
threshold tuning, calibration, or feature modification.
"""


write_text(
    DOC_CLOSEOUT,
    closeout_md,
)


# ==============================================================================
# 21. EXPORT CURRENT KAGGLE NOTEBOOK
#
# Strategy:
#   A. Recover exact underlying .ipynb if Kaggle exposes one.
#   B. Otherwise reconstruct executed code cells from IPython history.
#
# GitHub receives a SANITIZED code notebook with outputs removed.
# /kaggle/working receives a full copy when the exact notebook file is found.
# ==============================================================================

print("=" * 120)
print("CURRENT KAGGLE NOTEBOOK EXPORT")
print("=" * 120)


def notebook_candidates():

    candidates = []


    preferred = [
        Path(
            "/kaggle/working/__notebook__.ipynb"
        ),
        Path(
            "/kaggle/working/notebook.ipynb"
        ),
    ]


    for path in preferred:

        if path.is_file():

            candidates.append(
                path
            )


    # Search working only. Avoid treating notebook inputs as current notebook.
    try:

        for path in Path(
            "/kaggle/working"
        ).rglob(
            "*.ipynb"
        ):

            if not path.is_file():
                continue

            if REPO in path.parents:
                continue

            if DOWNLOAD_DIR in path.parents:
                continue

            candidates.append(
                path
            )

    except Exception:
        pass


    dedup = {}

    for path in candidates:

        try:

            key = str(
                path.resolve()
            )

        except Exception:

            key = str(
                path
            )

        dedup[
            key
        ] = path


    result = list(
        dedup.values()
    )


    result.sort(
        key=lambda p:
            p.stat().st_mtime,
        reverse=True,
    )


    return result


def load_valid_notebook(
    path,
):

    if nbformat is None:

        return None


    try:

        nb = nbformat.read(
            path,
            as_version=4,
        )

    except Exception:

        return None


    if not getattr(
        nb,
        "cells",
        None,
    ):

        return None


    code_count = sum(
        1
        for cell in nb.cells
        if cell.cell_type == "code"
    )


    if code_count < 3:

        return None


    return nb


notebook_source = None

full_notebook = None

export_mode = None


for candidate in notebook_candidates():

    nb = load_valid_notebook(
        candidate
    )

    if nb is not None:

        notebook_source = candidate

        full_notebook = nb

        export_mode = (
            "EXACT_IPYNB_FILE_RECOVERED"
        )

        break


if full_notebook is None:

    if nbformat is None:

        raise RuntimeError(
            "\nCould not recover current notebook file and nbformat is unavailable."
        )


    try:

        ip = get_ipython()

    except Exception:

        ip = None


    if ip is None:

        raise RuntimeError(
            "\nCould not recover notebook file or IPython runtime history."
        )


    history = list(
        ip.user_ns.get(
            "In",
            []
        )
    )


    history = [
        cell
        for cell in history[
            1:
        ]
        if isinstance(
            cell,
            str,
        )
        and cell.strip()
    ]


    if not history:

        raise RuntimeError(
            "\nNo notebook source file and no usable executed-cell history."
        )


    full_notebook = nbformat.v4.new_notebook()


    full_notebook[
        "metadata"
    ][
        "stage24_export"
    ] = {
        "mode":
            "EXECUTED_CODE_HISTORY_FALLBACK",

        "warning":
            (
                "Exact Kaggle .ipynb was not exposed by the runtime. "
                "This notebook contains executed code-cell history only."
            ),
    }


    full_notebook.cells.append(
        nbformat.v4.new_markdown_cell(
            "# Stage24 Cross-Dataset Generalization\n\n"
            "Recovered from executed Kaggle code-cell history because the "
            "runtime did not expose the exact underlying `.ipynb` file."
        )
    )


    for index, source in enumerate(
        history,
        start=1,
    ):

        full_notebook.cells.append(
            nbformat.v4.new_code_cell(
                source=source,
                metadata={
                    "recovered_execution_order":
                        index,
                },
            )
        )


    export_mode = (
        "EXECUTED_CODE_HISTORY_FALLBACK"
    )


# ------------------------------------------------------------------------------
# Preserve downloadable full version
# ------------------------------------------------------------------------------

if export_mode == "EXACT_IPYNB_FILE_RECOVERED":

    shutil.copy2(
        notebook_source,
        FULL_NOTEBOOK_DOWNLOAD,
    )

else:

    nbformat.write(
        full_notebook,
        FULL_NOTEBOOK_DOWNLOAD,
    )


full_download_sha = sha256_file(
    FULL_NOTEBOOK_DOWNLOAD
)


# ------------------------------------------------------------------------------
# Create sanitized GitHub notebook
# ------------------------------------------------------------------------------

sanitized = nbformat.from_dict(
    json.loads(
        nbformat.writes(
            full_notebook
        )
    )
)


for cell in sanitized.cells:

    if cell.cell_type == "code":

        cell.outputs = []

        cell.execution_count = None


sanitized[
    "metadata"
][
    "stage24_publication_export"
] = {
    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "source_mode":
        export_mode,

    "scientific_stage_status":
        "CLOSED",

    "frozen_stage24_commit":
        EXPECTED_PARENT,

    "scientific_fits":
        "4/4",

    "evaluable_target_openings":
        "6/6",
}


nbformat.write(
    sanitized,
    SCRIPT_IPYNB,
)


sanitized_ipynb_sha = sha256_file(
    SCRIPT_IPYNB
)


# ------------------------------------------------------------------------------
# Convert sanitized notebook to percent-cell Python script.
# ------------------------------------------------------------------------------

script_lines = [
    "# -*- coding: utf-8 -*-",
    '"""',
    "Stage24 — Cross-Dataset Generalization & Artifact-Sensitivity Audit",
    "",
    "Auto-exported from the Stage24 Kaggle notebook.",
    "",
    f"Frozen scientific parent commit: {EXPECTED_PARENT}",
    f"Notebook export mode: {export_mode}",
    "",
    "IMPORTANT:",
    "This file preserves the notebook workflow for reproducibility.",
    "Do not interpret its presence as authorization to rerun closed target",
    "openings or exceed the frozen scientific fit/opening budgets.",
    '"""',
    "",
]


for cell_index, cell in enumerate(
    sanitized.cells,
    start=1,
):

    if cell.cell_type == "markdown":

        script_lines.append(
            "# %% [markdown]"
        )

        for line in cell.source.splitlines():

            script_lines.append(
                (
                    "# "
                    +
                    line
                )
                if line
                else "#"
            )

        script_lines.append(
            ""
        )


    elif cell.cell_type == "code":

        script_lines.append(
            "# %%"
        )

        script_lines.append(
            f"# Notebook cell {cell_index}"
        )

        script_lines.append(
            cell.source.rstrip()
        )

        script_lines.append(
            ""
        )


write_text(
    SCRIPT_PY,
    "\n".join(
        script_lines
    ),
)


script_py_sha = sha256_file(
    SCRIPT_PY
)


print(
    "Export mode:",
    export_mode,
)


if notebook_source is not None:

    print(
        "Recovered notebook source:",
        notebook_source,
    )


print(
    "Full downloadable notebook:",
    FULL_NOTEBOOK_DOWNLOAD,
)

print(
    "Full notebook SHA:",
    full_download_sha,
)

print()

print(
    "GitHub sanitized notebook:",
    SCRIPT_IPYNB.relative_to(
        REPO
    ),
)

print(
    "Sanitized notebook SHA:",
    sanitized_ipynb_sha,
)

print()

print(
    "GitHub Python script:",
    SCRIPT_PY.relative_to(
        REPO
    ),
)

print(
    "Python script SHA:",
    script_py_sha,
)

print()


# ==============================================================================
# 22. STAGE24 SCRIPT README
# ==============================================================================

script_readme = f"""# Stage24 Reproducibility Scripts

## Cross-Dataset Generalization & Artifact-Sensitivity Audit

This directory preserves the Stage24 Kaggle notebook workflow after scientific
execution was permanently closed.

## Files

### `stage24_cross_dataset_generalization.ipynb`

Sanitized notebook export with outputs removed for repository-friendly
reproducibility.

SHA256:

`{sanitized_ipynb_sha}`

### `stage24_cross_dataset_generalization.py`

Percent-cell Python export of the sanitized notebook.

SHA256:

`{script_py_sha}`

## Notebook source recovery

Export mode:

`{export_mode}`

Full downloadable notebook copy SHA256:

`{full_download_sha}`

The full notebook copy is generated under `/kaggle/working` for manual download
and is intentionally not committed with potentially large cell outputs.

## Frozen Stage24 scientific state

- Final pre-publication commit: `{EXPECTED_PARENT}`
- Scientific fits: `4/4`
- Evaluable target openings: `6/6`
- GROUNDED_S4 cancellations: `2`
- Cancelled openings reallocated: `No`
- Remaining Stage24 scientific fits: `0`
- Remaining Stage24 target openings: `0`

## Governance

These scripts are retained for reproducibility and auditability. Stage24 target
results are closed. Rerunning code must not be used to silently create new
target openings, additional scientific fits, target-selected thresholds, or
post-result model variants.
"""


write_text(
    SCRIPT_README,
    script_readme,
)


# ==============================================================================
# 23. UPDATE JOURNAL EXTENSION SUMMARY
# ==============================================================================

journal_stage24 = f"""## Stage24 — Bidirectional Cross-Dataset Generalization

Stage24 addresses the earlier cross-dataset future-work gap through a frozen
bidirectional audit between CSE-CIC-IDS2018 and CICIDS2017.

IDS2018-to-CICIDS2017 transfer retains substantial ranking signal. The
62-feature bridge achieves PR-AUC
**{primary_cells['bridge62_PUBLISHED']['pr_auc']:.6f}** and ROC-AUC
**{primary_cells['bridge62_PUBLISHED']['roc_auc']:.6f}** on
**{primary_target_rows:,}** effective CICIDS2017 rows.

The reciprocal CICIDS2017-to-IDS2018 transfer is much weaker. On the frozen
IDS2018 Feb-28 target, attack prevalence is **{secondary_prevalence:.6f}**,
while bridge62 reaches PR-AUC **{secondary_cells['bridge62']['pr_auc']:.6f}**
and ROC-AUC **{secondary_cells['bridge62']['roc_auc']:.6f}**. This produces
normalized PR-AUC of only **{secondary_cells['bridge62']['pr_normalized']:.6f}**.

A preregistered serialization audit further shows that correcting the CICIDS2017
aggregate flag mapping changes primary bridge70 PR-AUC by
**{artifact_pr['point_difference']:+.6f}**, ROC-AUC by
**{artifact_roc['point_difference']:+.6f}**, and Brier score by
**{artifact_brier['point_difference']:+.6f}**; all three 95% paired-bootstrap
intervals exclude zero.

The two transfer directions remain separate and are not averaged. No
target-guided fitting, threshold tuning, calibration, or feature search was
performed.

Two GROUNDED_S4 target cells were administratively cancelled before opening
because exact durable physical-row membership could not be reconstructed without
introducing a new post-freeze heuristic. No fuzzy substitute was used and the
two slots were not reallocated.

Stage24 therefore supersedes the earlier statement that cross-dataset
generalization remained untested.

See:

- `docs/STAGE24_MANUSCRIPT_INTEGRATION.md`
- `docs/STAGE24_PUBLICATION_TABLES.md`
- `docs/STAGE24_PUBLICATION_CLOSEOUT.md`
"""


append_marked_section(
    JOURNAL_SUMMARY_PATH,
    "<!-- BEGIN STAGE24 JOURNAL EXTENSION SUMMARY -->",
    "<!-- END STAGE24 JOURNAL EXTENSION SUMMARY -->",
    journal_stage24,
)


# ==============================================================================
# 24. UPDATE README
# ==============================================================================

readme_stage24 = f"""## Stage24 — Cross-Dataset Generalization

Stage24 is complete and scientifically closed.

The bidirectional audit evaluates CSE-CIC-IDS2018 and CICIDS2017 under frozen
feature bridges, extractor-semantic controls, source-only threshold selection,
and paired uncertainty analysis.

Key frozen observations:

- IDS2018 → CICIDS2017 bridge62:
  PR-AUC `{primary_cells['bridge62_PUBLISHED']['pr_auc']:.6f}`,
  ROC-AUC `{primary_cells['bridge62_PUBLISHED']['roc_auc']:.6f}`.
- CICIDS2017 → IDS2018 bridge62:
  PR-AUC `{secondary_cells['bridge62']['pr_auc']:.6f}`,
  ROC-AUC `{secondary_cells['bridge62']['roc_auc']:.6f}`.
- Reciprocal IDS2018 target prevalence:
  `{secondary_prevalence:.6f}`.
- Aggregate-flag serialization correction produces statistically resolved
  changes in primary bridge70 PR-AUC, ROC-AUC, and Brier score.
- Scientific fits: `4/4`.
- Evaluable target openings: `6/6`.
- GROUNDED_S4 cells cancelled before opening: `2`.
- Cancelled slots reallocated: `No`.

Publication package:

- `docs/STAGE24_MANUSCRIPT_INTEGRATION.md`
- `docs/STAGE24_PUBLICATION_TABLES.md`
- `docs/STAGE24_PUBLICATION_CLOSEOUT.md`
- `figures/stage24_cross_dataset/`
- `scripts/stage24/`
"""


append_marked_section(
    README_PATH,
    "<!-- BEGIN STAGE24 CROSS-DATASET SUMMARY -->",
    "<!-- END STAGE24 CROSS-DATASET SUMMARY -->",
    readme_stage24,
)


# ==============================================================================
# 25. CREATE PUBLICATION MANIFEST
# ==============================================================================

publication_files = [
    DOC_MANUSCRIPT,
    DOC_MANUSCRIPT_TEX,
    DOC_TABLES_MD,
    DOC_TABLES_TEX,
    DOC_CLOSEOUT,
    TABLE1_CSV,
    TABLE2_CSV,
    TABLE3_CSV,
    TABLE4_CSV,
    FIG1_PNG,
    FIG1_SVG,
    FIG2_PNG,
    FIG2_SVG,
    FIG3_PNG,
    FIG3_SVG,
    FIG4_PNG,
    FIG4_SVG,
    SCRIPT_IPYNB,
    SCRIPT_PY,
    SCRIPT_README,
]


manifest = {
    "stage":
        "Stage24-PUB",

    "type":
        "NON_SCIENTIFIC_PUBLICATION_PACKAGE",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent_commit":
        EXPECTED_PARENT,

    "stage24_final_synthesis_sha256":
        final_sha,

    "scientific_execution_closed":
        True,

    "scientific_fits":
        {
            "completed":
                4,

            "budget":
                4,

            "remaining":
                0,
        },

    "target_openings":
        {
            "evaluable_completed":
                6,

            "original_budget":
                8,

            "administratively_cancelled":
                2,

            "cancelled_slots_reallocated":
                False,

            "remaining_evaluable":
                0,
        },

    "notebook_export":
        {
            "mode":
                export_mode,

            "full_downloadable_copy":
                str(
                    FULL_NOTEBOOK_DOWNLOAD
                ),

            "full_downloadable_sha256":
                full_download_sha,

            "repo_sanitized_ipynb":
                str(
                    SCRIPT_IPYNB.relative_to(
                        REPO
                    )
                ),

            "repo_sanitized_ipynb_sha256":
                sanitized_ipynb_sha,

            "repo_python_script":
                str(
                    SCRIPT_PY.relative_to(
                        REPO
                    )
                ),

            "repo_python_script_sha256":
                script_py_sha,
        },

    "files":
        {
            str(
                path.relative_to(
                    REPO
                )
            ):
                sha256_file(
                    path
                )

            for path in publication_files
        },

    "anti_adaptation":
        {
            "model_fit":
                False,

            "model_inference":
                False,

            "target_opening":
                False,

            "target_threshold_search":
                False,

            "target_calibration":
                False,

            "target_feature_change":
                False,
        },
}


PUB_MANIFEST = (
    PUB_RESULT_DIR
    / "stage24_publication_package_manifest.json"
)


write_text(
    PUB_MANIFEST,
    json.dumps(
        manifest,
        indent=2,
        sort_keys=True,
    ),
)


PUB_MANIFEST_SHA = (
    PUB_RESULT_DIR
    / "stage24_publication_package_manifest.sha256"
)


manifest_sha = sha256_file(
    PUB_MANIFEST
)


write_text(
    PUB_MANIFEST_SHA,
    f"{manifest_sha}  {PUB_MANIFEST.name}",
)


# ==============================================================================
# 26. CREATE DOWNLOADABLE ZIP — NOTEBOOK
# ==============================================================================

notebook_zip_dir = (
    DOWNLOAD_DIR
    / "notebook"
)


if notebook_zip_dir.exists():

    shutil.rmtree(
        notebook_zip_dir
    )


notebook_zip_dir.mkdir(
    parents=True,
    exist_ok=True,
)


shutil.copy2(
    FULL_NOTEBOOK_DOWNLOAD,
    notebook_zip_dir
    /
    FULL_NOTEBOOK_DOWNLOAD.name,
)


shutil.copy2(
    SCRIPT_IPYNB,
    notebook_zip_dir
    /
    SCRIPT_IPYNB.name,
)


shutil.copy2(
    SCRIPT_PY,
    notebook_zip_dir
    /
    SCRIPT_PY.name,
)


notebook_zip_path = Path(
    shutil.make_archive(
        str(
            NOTEBOOK_ZIP_BASE
        ),
        "zip",
        root_dir=notebook_zip_dir,
    )
)


notebook_zip_sha = sha256_file(
    notebook_zip_path
)


# ==============================================================================
# 27. CREATE DOWNLOADABLE ZIP — PUBLICATION PACKAGE
# ==============================================================================

pub_zip_dir = (
    DOWNLOAD_DIR
    / "publication_package"
)


if pub_zip_dir.exists():

    shutil.rmtree(
        pub_zip_dir
    )


pub_zip_dir.mkdir(
    parents=True,
    exist_ok=True,
)


for path in [
    DOC_MANUSCRIPT,
    DOC_MANUSCRIPT_TEX,
    DOC_TABLES_MD,
    DOC_TABLES_TEX,
    DOC_CLOSEOUT,
    TABLE1_CSV,
    TABLE2_CSV,
    TABLE3_CSV,
    TABLE4_CSV,
    FIG1_PNG,
    FIG1_SVG,
    FIG2_PNG,
    FIG2_SVG,
    FIG3_PNG,
    FIG3_SVG,
    FIG4_PNG,
    FIG4_SVG,
    PUB_MANIFEST,
    PUB_MANIFEST_SHA,
]:

    destination = (
        pub_zip_dir
        /
        path.relative_to(
            REPO
        )
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        path,
        destination,
    )


publication_zip_path = Path(
    shutil.make_archive(
        str(
            PUBLICATION_ZIP_BASE
        ),
        "zip",
        root_dir=pub_zip_dir,
    )
)


publication_zip_sha = sha256_file(
    publication_zip_path
)


print("=" * 120)
print("DOWNLOADABLE EXPORTS")
print("=" * 120)

print(
    "Notebook ZIP:",
    notebook_zip_path,
)

print(
    "Notebook ZIP SHA:",
    notebook_zip_sha,
)

print()

print(
    "Publication ZIP:",
    publication_zip_path,
)

print(
    "Publication ZIP SHA:",
    publication_zip_sha,
)

print()


# ==============================================================================
# 28. GITHUB CREDENTIAL + REMOTE GATE
# ==============================================================================

github_token = None
token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()


    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )


            if value and value.strip():

                github_token = value.strip()

                token_source = name

                break

        except Exception:
            pass

except Exception:
    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )


        if value and value.strip():

            github_token = value.strip()

            token_source = (
                "ENV:"
                +
                name
            )

            break


if github_token is None:

    raise RuntimeError(
        "\nGitHub token unavailable."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved before Stage24 publication closeout.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main still equals frozen Stage24 commit."
)

print()


# ==============================================================================
# 29. GIT DIFF AUDIT
# ==============================================================================

print("=" * 120)
print("GIT DIFF AUDIT")
print("=" * 120)


status_before_commit = git_cmd(
    "status",
    "--porcelain",
)


print(
    status_before_commit
)

print()


if not status_before_commit:

    raise RuntimeError(
        "No publication-package changes detected."
    )


# Scientific result tree must remain untouched.
for line in status_before_commit.splitlines():

    path = line[
        3:
    ]


    forbidden_prefixes = [
        "results/stage24_cross_dataset/stage24_0_",
        "results/stage24_cross_dataset/stage24_1_",
        "results/stage24_cross_dataset/stage24_2_",
        "results/stage24_cross_dataset/stage24_3_",
        "results/stage24_cross_dataset/stage24_4_",
        "results/stage24_cross_dataset/stage24_5_",
        "results/stage24_cross_dataset/stage24_6_",
    ]


    if any(
        path.startswith(
            prefix
        )
        for prefix in forbidden_prefixes
    ):

        raise RuntimeError(
            "\nPublication closeout attempted to modify a frozen scientific artifact:\n"
            f"{path}"
        )


print(
    "[PASS] Frozen Stage24 scientific artifact directories unchanged."
)

print()


# ==============================================================================
# 30. GIT ADD
# ==============================================================================

git_cmd(
    "add",
    "--",
    "docs/STAGE24_MANUSCRIPT_INTEGRATION.md",
    "docs/STAGE24_MANUSCRIPT_INTEGRATION.tex",
    "docs/STAGE24_PUBLICATION_TABLES.md",
    "docs/STAGE24_PUBLICATION_TABLES.tex",
    "docs/STAGE24_PUBLICATION_CLOSEOUT.md",
    "docs/JOURNAL_EXTENSION_SUMMARY.md",
    "README.md",
    "figures/stage24_cross_dataset",
    "scripts/stage24",
    "results/stage24_cross_dataset/stage24_publication_package",
)


staged = git_cmd(
    "diff",
    "--cached",
    "--name-only",
)


print("=" * 120)
print("STAGED PUBLICATION PACKAGE")
print("=" * 120)

print(
    staged
)

print()


# Final forbidden-path check on staged files.
for path in staged.splitlines():

    forbidden_prefixes = [
        "results/stage24_cross_dataset/stage24_0_",
        "results/stage24_cross_dataset/stage24_1_",
        "results/stage24_cross_dataset/stage24_2_",
        "results/stage24_cross_dataset/stage24_3_",
        "results/stage24_cross_dataset/stage24_4_",
        "results/stage24_cross_dataset/stage24_5_",
        "results/stage24_cross_dataset/stage24_6_",
    ]


    if any(
        path.startswith(
            prefix
        )
        for prefix in forbidden_prefixes
    ):

        raise RuntimeError(
            "\nFrozen Stage24 scientific artifact accidentally staged:\n"
            f"{path}"
        )


print(
    "[PASS] Only publication/reproducibility artifacts are staged."
)

print()


# ==============================================================================
# 31. GIT AUTHOR SAFETY
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    git_cmd(
        "config",
        "--local",
        "user.name",
        git_cmd(
            "log",
            "-1",
            "--format=%an",
        ),
    )


if not author_email:

    git_cmd(
        "config",
        "--local",
        "user.email",
        git_cmd(
            "log",
            "-1",
            "--format=%ae",
        ),
    )


# ==============================================================================
# 32. COMMIT
# ==============================================================================

print("=" * 120)
print("GIT COMMIT")
print("=" * 120)


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: add publication package and reproducibility notebook",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "\nPublication commit parent mismatch.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {parent}"
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 33. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 120)
print("PUSH + REMOTE VERIFY")
print("=" * 120)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository not clean after publication closeout:\n"
        + final_status
    )


print(
    "[PASS] Remote main == publication closeout commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 34. OPTIONAL CLICKABLE KAGGLE DOWNLOAD LINKS
# ==============================================================================

try:

    from IPython.display import display, FileLink


    print(
        "Clickable downloads:"
    )


    display(
        FileLink(
            str(
                notebook_zip_path
            )
        )
    )


    display(
        FileLink(
            str(
                publication_zip_path
            )
        )
    )


except Exception as exc:

    print(
        "[INFO] Could not render FileLink widgets:",
        repr(
            exc
        ),
    )


# ==============================================================================
# 35. FINAL
# ==============================================================================

print()
print("=" * 120)
print("STAGE24 PUBLICATION + REPRODUCIBILITY CLOSEOUT: PASS")
print("=" * 120)

print()

print(
    "Scientific Stage24 status:      CLOSED"
)

print(
    "Scientific fits:                4 / 4"
)

print(
    "Evaluable target openings:      6 / 6"
)

print(
    "GROUNDED_S4 cancellations:      2"
)

print(
    "Cancelled slots reallocated:    NO"
)

print(
    "New fit performed here:         NO"
)

print(
    "New target inference here:      NO"
)

print()

print(
    "Publication tables:             COMPLETE"
)

print(
    "Publication figures:            COMPLETE"
)

print(
    "Results section:                COMPLETE"
)

print(
    "Discussion:                     COMPLETE"
)

print(
    "Limitations / Threats:          COMPLETE"
)

print(
    "Contributions:                  COMPLETE"
)

print()

print(
    "Notebook export mode:"
)

print(
    " ",
    export_mode,
)

print()

print(
    "GitHub Stage24 notebook:"
)

print(
    " ",
    SCRIPT_IPYNB.relative_to(
        REPO
    ),
)

print()

print(
    "GitHub Stage24 Python script:"
)

print(
    " ",
    SCRIPT_PY.relative_to(
        REPO
    ),
)

print()

print(
    "Download notebook ZIP:"
)

print(
    " ",
    notebook_zip_path,
)

print()

print(
    "Download publication ZIP:"
)

print(
    " ",
    publication_zip_path,
)

print()

print(
    "Publication manifest SHA:"
)

print(
    " ",
    manifest_sha,
)

print()

print(
    "GitHub commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "Remote main:"
)

print(
    " ",
    remote_after,
)

print()

print(
    "STAGE24 IS DONE DONE 😂"
)

print(
    "NEXT: Stage25."
)

print("=" * 120)

STAGE24-PUB — PUBLICATION PACKAGE + NOTEBOOK EXPORT + GITHUB CLOSEOUT

GOVERNANCE
HEAD: a4b6a3854109ba3d85954fb4a40afe6fe1ee6114
[PASS] Repository starts from exact Stage24 final freeze.
[PASS] Repository clean.
[PASS] No unresolved fit/opening ledgers.

FROZEN SCIENTIFIC ARTIFACT VERIFICATION
Final synthesis SHA:     785bbcb00140f4d7e07e9b49ad33924166b5995e66a229c986d5b1abcbcaee4b
Final report SHA:        f867289cec34eca6666d02deec01864395503cea8a59b7eac88e1759d5eb2a0b
Primary bootstrap SHA:   96732023e0a1c9b52a79fafa59ea4b851ef7f8bae1814e4b4b4fde0ef0df09aa
Secondary bootstrap SHA: 28005e0f13d86658ba81e6b19dc154b94eedb316ea5ce0348287265c00541953

[PASS] Stage24 frozen scientific state verified.

FINAL NUMBERS
IDS2018 -> CICIDS2017:
  bridge62 PR/ROC: 0.6674827010666808 0.7339464251813748
  bridge70 PUBLISHED PR/ROC: 0.6639810751980307 0.7418914673891956
  bridge70 CORRECTED PR/ROC: 0.6562523506197266 0.7440833611037603

CICIDS2017 -> IDS2018:
  bridge62 PR/ROC: 0.10817592573968819 0.5

AttributeError: 'list' object has no attribute 'splitlines'

In [45]:
# ==============================================================================
# STAGE24-PUB-R1
# RECOVER PUBLICATION CLOSEOUT AFTER NOTEBOOK SOURCE-LIST EXPORT FAILURE
#
# Previous failure:
#
#   AttributeError:
#       'list' object has no attribute 'splitlines'
#
# Failure location:
#   notebook -> Python percent-cell export
#
# IMPORTANT SCIENTIFIC STATE:
#   Stage24 scientific execution was ALREADY CLOSED before this packaging run.
#
#   scientific fits             = 4 / 4
#   evaluable target openings   = 6 / 6
#   GROUNDED_S4 cancellations   = 2
#   new fit in this recovery    = 0
#   new inference               = 0
#   new target opening          = 0
#
# THIS RECOVERY:
#   - verifies existing publication assets produced before the failure
#   - repairs notebook cell-source normalization
#   - exports sanitized .ipynb
#   - exports Python percent-cell script
#   - creates downloadable notebook/publication ZIPs
#   - updates README + JOURNAL_EXTENSION_SUMMARY
#   - writes publication manifest
#   - commits
#   - pushes to GitHub main
#
# Expected repository HEAD remains:
#   a4b6a3854109ba3d85954fb4a40afe6fe1ee6114
# ==============================================================================

from __future__ import annotations

import os
import gc
import json
import copy
import base64
import shutil
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np

try:
    import nbformat
except Exception as exc:
    raise RuntimeError(
        f"nbformat is required for Stage24 notebook recovery: {exc}"
    )


print("=" * 120)
print("STAGE24-PUB-R1 — NOTEBOOK EXPORT + PUBLICATION CLOSEOUT RECOVERY")
print("=" * 120)
print()


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "a4b6a3854109ba3d85954fb4a40afe6fe1ee6114"
)

EXPECTED_FINAL_SYNTHESIS_SHA = (
    "785bbcb00140f4d7e07e9b49ad33924166b5995e66a229c986d5b1abcbcaee4b"
)

EXPECTED_FINAL_REPORT_SHA = (
    "f867289cec34eca6666d02deec01864395503cea8a59b7eac88e1759d5eb2a0b"
)

EXPECTED_PRIMARY_BOOTSTRAP_SHA = (
    "96732023e0a1c9b52a79fafa59ea4b851ef7f8bae1814e4b4b4fde0ef0df09aa"
)

EXPECTED_SECONDARY_BOOTSTRAP_SHA = (
    "28005e0f13d86658ba81e6b19dc154b94eedb316ea5ce0348287265c00541953"
)


# ==============================================================================
# 1. PATHS
# ==============================================================================

STAGE24_BASE = (
    REPO
    / "results"
    / "stage24_cross_dataset"
)


FINAL_DIR = (
    STAGE24_BASE
    / "stage24_6_final_synthesis"
)

FINAL_JSON = (
    FINAL_DIR
    / "stage24_6_final_synthesis.json"
)

FINAL_JSON_SHA = (
    FINAL_DIR
    / "stage24_6_final_synthesis.sha256"
)

FINAL_REPORT = (
    FINAL_DIR
    / "stage24_6_final_synthesis.md"
)

SECONDARY_BOOTSTRAP = (
    FINAL_DIR
    / "stage24_6_secondary_paired_bootstrap_replicates.npz"
)


PRIMARY_DIR = (
    STAGE24_BASE
    / "stage24_3_primary_results_freeze"
    / "stage24_3a_primary_paired_bootstrap"
)

PRIMARY_JSON = (
    PRIMARY_DIR
    / "stage24_3a_primary_results_freeze.json"
)

PRIMARY_BOOTSTRAP = (
    PRIMARY_DIR
    / "stage24_3a_primary_paired_bootstrap_replicates.npz"
)


PUB_RESULT_DIR = (
    STAGE24_BASE
    / "stage24_publication_package"
)

TABLE_DIR = (
    PUB_RESULT_DIR
    / "tables"
)


FIG_DIR = (
    REPO
    / "figures"
    / "stage24_cross_dataset"
)


DOC_MANUSCRIPT = (
    REPO
    / "docs"
    / "STAGE24_MANUSCRIPT_INTEGRATION.md"
)

DOC_MANUSCRIPT_TEX = (
    REPO
    / "docs"
    / "STAGE24_MANUSCRIPT_INTEGRATION.tex"
)

DOC_TABLES_MD = (
    REPO
    / "docs"
    / "STAGE24_PUBLICATION_TABLES.md"
)

DOC_TABLES_TEX = (
    REPO
    / "docs"
    / "STAGE24_PUBLICATION_TABLES.tex"
)

DOC_CLOSEOUT = (
    REPO
    / "docs"
    / "STAGE24_PUBLICATION_CLOSEOUT.md"
)


README_PATH = (
    REPO
    / "README.md"
)

JOURNAL_SUMMARY_PATH = (
    REPO
    / "docs"
    / "JOURNAL_EXTENSION_SUMMARY.md"
)


SCRIPT_DIR = (
    REPO
    / "scripts"
    / "stage24"
)

SCRIPT_IPYNB = (
    SCRIPT_DIR
    / "stage24_cross_dataset_generalization.ipynb"
)

SCRIPT_PY = (
    SCRIPT_DIR
    / "stage24_cross_dataset_generalization.py"
)

SCRIPT_README = (
    SCRIPT_DIR
    / "README.md"
)


DOWNLOAD_DIR = Path(
    "/kaggle/working/stage24_exports"
)

FULL_NOTEBOOK_DOWNLOAD = (
    DOWNLOAD_DIR
    / "stage24_cross_dataset_generalization_FULL.ipynb"
)

NOTEBOOK_ZIP_BASE = Path(
    "/kaggle/working/stage24_notebook_export"
)

PUBLICATION_ZIP_BASE = Path(
    "/kaggle/working/stage24_publication_package"
)


PUB_MANIFEST = (
    PUB_RESULT_DIR
    / "stage24_publication_package_manifest.json"
)

PUB_MANIFEST_SHA = (
    PUB_RESULT_DIR
    / "stage24_publication_package_manifest.sha256"
)


FIT_LEDGER = Path(
    "/kaggle/working/stage24_secondary_fit_runtime_ledger.json"
)

OPENING_LEDGER = Path(
    "/kaggle/working/stage24_target_opening_runtime_ledger.json"
)


TABLE1_CSV = (
    TABLE_DIR
    / "table24_1_bidirectional_generalization.csv"
)

TABLE2_CSV = (
    TABLE_DIR
    / "table24_2_paired_bootstrap_contrasts.csv"
)

TABLE3_CSV = (
    TABLE_DIR
    / "table24_3_secondary_threshold_transfer.csv"
)

TABLE4_CSV = (
    TABLE_DIR
    / "table24_4_governance_and_populations.csv"
)


FIG1_PNG = (
    FIG_DIR
    / "fig24_1_normalized_pr_auc_directionality.png"
)

FIG1_SVG = (
    FIG_DIR
    / "fig24_1_normalized_pr_auc_directionality.svg"
)

FIG2_PNG = (
    FIG_DIR
    / "fig24_2_roc_auc_directionality.png"
)

FIG2_SVG = (
    FIG_DIR
    / "fig24_2_roc_auc_directionality.svg"
)

FIG3_PNG = (
    FIG_DIR
    / "fig24_3_paired_effects_forest.png"
)

FIG3_SVG = (
    FIG_DIR
    / "fig24_3_paired_effects_forest.svg"
)

FIG4_PNG = (
    FIG_DIR
    / "fig24_4_secondary_threshold_transfer.png"
)

FIG4_SVG = (
    FIG_DIR
    / "fig24_4_secondary_threshold_transfer.svg"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    return (p.stdout or "").strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    cmd = [
        "git"
    ]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [
        str(x)
        for x in args
    ]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
):

    path = Path(
        path
    )

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def verify_sidecar(
    artifact,
    sidecar,
):

    artifact = Path(
        artifact
    )

    sidecar = Path(
        sidecar
    )

    if not artifact.is_file():

        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )

    if not sidecar.is_file():

        raise RuntimeError(
            f"Missing SHA sidecar:\n{sidecar}"
        )

    actual = sha256_file(
        artifact
    )

    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
    )

    if actual != expected:

        raise RuntimeError(
            "\nSHA mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )

    return actual


def write_text(
    path,
    content,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        str(
            content
        ).rstrip()
        +
        "\n",
        encoding="utf-8",
    )


def append_marked_section(
    path,
    begin_marker,
    end_marker,
    body,
):

    path = Path(
        path
    )

    if not path.is_file():

        raise RuntimeError(
            f"Missing document to update:\n{path}"
        )

    original = path.read_text(
        encoding="utf-8"
    )

    block = (
        f"{begin_marker}\n\n"
        f"{body.strip()}\n\n"
        f"{end_marker}"
    )

    if begin_marker in original:

        start = original.index(
            begin_marker
        )

        if end_marker not in original[
            start:
        ]:

            raise RuntimeError(
                f"Found begin marker without end marker in {path}"
            )

        end = (
            original.index(
                end_marker,
                start,
            )
            +
            len(
                end_marker
            )
        )

        updated = (
            original[
                :start
            ].rstrip()
            +
            "\n\n"
            +
            block
            +
            "\n"
            +
            original[
                end:
            ].lstrip(
                "\n"
            )
        )

    else:

        updated = (
            original.rstrip()
            +
            "\n\n"
            +
            block
            +
            "\n"
        )

    path.write_text(
        updated,
        encoding="utf-8",
    )


def normalize_source(
    source,
):
    """
    nbformat normally exposes source as str, but some notebook JSON producers
    preserve it as a JSON list of lines. Convert both representations to str.
    """

    if source is None:

        return ""

    if isinstance(
        source,
        str,
    ):

        return source

    if isinstance(
        source,
        (
            list,
            tuple,
        ),
    ):

        return "".join(
            str(
                item
            )
            for item in source
        )

    return str(
        source
    )


def normalize_notebook_sources(
    notebook,
):

    for cell in notebook.cells:

        cell[
            "source"
        ] = normalize_source(
            cell.get(
                "source",
                "",
            )
        )

    return notebook


def comparison_lookup(
    comparisons,
    name,
):

    matches = [
        x
        for x in comparisons
        if x.get(
            "comparison"
        )
        ==
        name
    ]

    if len(
        matches
    ) != 1:

        raise RuntimeError(
            f"Expected exactly one comparison '{name}', got {len(matches)}."
        )

    return matches[
        0
    ]


# ==============================================================================
# 3. GOVERNANCE
# ==============================================================================

print("=" * 120)
print("GOVERNANCE")
print("=" * 120)


head = git_cmd(
    "rev-parse",
    "HEAD",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected repository HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if FIT_LEDGER.exists():

    raise RuntimeError(
        f"Unexpected Stage24 fit ledger:\n{FIT_LEDGER}"
    )


if OPENING_LEDGER.exists():

    raise RuntimeError(
        f"Unexpected Stage24 opening ledger:\n{OPENING_LEDGER}"
    )


staged_before = git_cmd(
    "diff",
    "--cached",
    "--name-only",
)


if staged_before:

    raise RuntimeError(
        "\nUnexpected pre-existing staged changes:\n"
        +
        staged_before
    )


print(
    "[PASS] Frozen Stage24 HEAD unchanged."
)

print(
    "[PASS] No fit/opening ledger."
)

print(
    "[PASS] No pre-existing staged changes."
)

print()


# ==============================================================================
# 4. VERIFY FROZEN SCIENTIFIC STATE
# ==============================================================================

print("=" * 120)
print("FROZEN SCIENTIFIC STATE")
print("=" * 120)


final_sha = verify_sidecar(
    FINAL_JSON,
    FINAL_JSON_SHA,
)


if final_sha != EXPECTED_FINAL_SYNTHESIS_SHA:

    raise RuntimeError(
        "Stage24 final synthesis SHA changed."
    )


final_report_sha = sha256_file(
    FINAL_REPORT
)


if final_report_sha != EXPECTED_FINAL_REPORT_SHA:

    raise RuntimeError(
        "Stage24 final report SHA changed."
    )


primary_bootstrap_sha = sha256_file(
    PRIMARY_BOOTSTRAP
)


if primary_bootstrap_sha != EXPECTED_PRIMARY_BOOTSTRAP_SHA:

    raise RuntimeError(
        "Primary bootstrap SHA changed."
    )


secondary_bootstrap_sha = sha256_file(
    SECONDARY_BOOTSTRAP
)


if secondary_bootstrap_sha != EXPECTED_SECONDARY_BOOTSTRAP_SHA:

    raise RuntimeError(
        "Secondary bootstrap SHA changed."
    )


stage24 = json.loads(
    FINAL_JSON.read_text(
        encoding="utf-8"
    )
)


primary = json.loads(
    PRIMARY_JSON.read_text(
        encoding="utf-8"
    )
)


if (
    stage24[
        "status"
    ]
    !=
    "STAGE24_CROSS_DATASET_AUDIT_COMPLETE"
):

    raise RuntimeError(
        "Stage24 final status changed."
    )


if stage24[
    "completion"
][
    "Stage24_complete"
] is not True:

    raise RuntimeError(
        "Stage24 completion flag changed."
    )


if stage24[
    "completion"
][
    "scientific_fits"
] != "4/4":

    raise RuntimeError(
        "Stage24 fit accounting changed."
    )


if stage24[
    "completion"
][
    "evaluable_target_openings"
] != "6/6":

    raise RuntimeError(
        "Stage24 opening accounting changed."
    )


print(
    "Final synthesis SHA:    ",
    final_sha,
)

print(
    "Final report SHA:       ",
    final_report_sha,
)

print(
    "Primary bootstrap SHA:  ",
    primary_bootstrap_sha,
)

print(
    "Secondary bootstrap SHA:",
    secondary_bootstrap_sha,
)

print()

print(
    "[PASS] Stage24 scientific execution remains frozen."
)

print()


# ==============================================================================
# 5. VERIFY ASSETS CREATED BEFORE PREVIOUS FAILURE
# ==============================================================================

print("=" * 120)
print("PREVIOUSLY GENERATED PUBLICATION ASSETS")
print("=" * 120)


preexisting_assets = [
    TABLE1_CSV,
    TABLE2_CSV,
    TABLE3_CSV,
    TABLE4_CSV,

    FIG1_PNG,
    FIG1_SVG,
    FIG2_PNG,
    FIG2_SVG,
    FIG3_PNG,
    FIG3_SVG,
    FIG4_PNG,
    FIG4_SVG,

    DOC_MANUSCRIPT,
    DOC_MANUSCRIPT_TEX,
    DOC_TABLES_MD,
    DOC_TABLES_TEX,
    DOC_CLOSEOUT,
]


missing_assets = [
    path
    for path in preexisting_assets
    if not path.is_file()
]


if missing_assets:

    raise RuntimeError(
        "\nThe previous packaging cell did not reach the notebook-export "
        "checkpoint as expected.\nMissing:\n"
        +
        "\n".join(
            str(
                x
            )
            for x in missing_assets
        )
    )


for path in preexisting_assets:

    print(
        "[FOUND]",
        path.relative_to(
            REPO
        ),
    )


print()

print(
    "[PASS] Tables, figures, Results, Discussion, Limitations and Contributions already exist."
)

print()


# ==============================================================================
# 6. LOAD FINAL PUBLICATION VALUES
# ==============================================================================

primary_cells = stage24[
    "primary_direction"
][
    "cells"
]


primary_comparisons = stage24[
    "primary_direction"
][
    "paired_comparisons"
]


secondary_cells = stage24[
    "secondary_direction"
][
    "cells"
]


primary_target_rows = int(
    primary[
        "target_population"
    ][
        "rows"
    ]
)


primary_target_attack = int(
    primary[
        "target_population"
    ][
        "attack"
    ]
)


primary_target_benign = int(
    primary[
        "target_population"
    ][
        "benign"
    ]
)


primary_prevalence = (
    primary_target_attack
    /
    primary_target_rows
)


secondary_target_rows = int(
    stage24[
        "secondary_direction"
    ][
        "target_population"
    ][
        "rows"
    ]
)


secondary_target_attack = int(
    stage24[
        "secondary_direction"
    ][
        "target_population"
    ][
        "attack"
    ]
)


secondary_target_benign = int(
    stage24[
        "secondary_direction"
    ][
        "target_population"
    ][
        "benign"
    ]
)


secondary_prevalence = float(
    stage24[
        "secondary_direction"
    ][
        "target_population"
    ][
        "prevalence"
    ]
)


artifact_cmp = comparison_lookup(
    primary_comparisons,
    "bridge70_FLAG_CORRECTED_minus_bridge70_PUBLISHED",
)


artifact_pr = artifact_cmp[
    "bootstrap"
][
    "pr_auc"
]


artifact_roc = artifact_cmp[
    "bootstrap"
][
    "roc_auc"
]


artifact_brier = artifact_cmp[
    "bootstrap"
][
    "brier"
]


# ==============================================================================
# 7. RECOVER / NORMALIZE CURRENT NOTEBOOK
# ==============================================================================

print("=" * 120)
print("NOTEBOOK RECOVERY")
print("=" * 120)


DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


SCRIPT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


full_notebook = None

export_mode = None

notebook_source = None


# ------------------------------------------------------------------------------
# Preferred recovery:
# use the full notebook export already written before the AttributeError.
# ------------------------------------------------------------------------------

if FULL_NOTEBOOK_DOWNLOAD.is_file():

    try:

        full_notebook = nbformat.read(
            FULL_NOTEBOOK_DOWNLOAD,
            as_version=4,
        )

        full_notebook = normalize_notebook_sources(
            full_notebook
        )

        export_mode = (
            "RECOVERED_FROM_PREVIOUS_STAGE24_EXPORT_CHECKPOINT"
        )

        notebook_source = (
            FULL_NOTEBOOK_DOWNLOAD
        )

    except Exception as exc:

        print(
            "[WARN] Existing full notebook export could not be loaded:",
            repr(
                exc
            ),
        )

        full_notebook = None


# ------------------------------------------------------------------------------
# Secondary recovery:
# locate a notebook file in /kaggle/working.
# ------------------------------------------------------------------------------

if full_notebook is None:

    candidates = []


    for preferred in [
        Path(
            "/kaggle/working/__notebook__.ipynb"
        ),
        Path(
            "/kaggle/working/notebook.ipynb"
        ),
    ]:

        if preferred.is_file():

            candidates.append(
                preferred
            )


    try:

        for path in Path(
            "/kaggle/working"
        ).rglob(
            "*.ipynb"
        ):

            if not path.is_file():

                continue


            if REPO in path.parents:

                continue


            if DOWNLOAD_DIR in path.parents:

                continue


            candidates.append(
                path
            )

    except Exception:
        pass


    seen = set()

    dedup_candidates = []


    for path in candidates:

        try:

            key = str(
                path.resolve()
            )

        except Exception:

            key = str(
                path
            )


        if key not in seen:

            seen.add(
                key
            )

            dedup_candidates.append(
                path
            )


    dedup_candidates.sort(
        key=lambda p:
            p.stat().st_mtime,
        reverse=True,
    )


    for candidate in dedup_candidates:

        try:

            nb = nbformat.read(
                candidate,
                as_version=4,
            )

            nb = normalize_notebook_sources(
                nb
            )


            code_cells = sum(
                1
                for cell in nb.cells
                if cell.cell_type == "code"
            )


            if code_cells < 3:

                continue


            full_notebook = nb

            notebook_source = candidate

            export_mode = (
                "EXACT_OR_RUNTIME_IPYNB_RECOVERED"
            )

            break

        except Exception:

            pass


# ------------------------------------------------------------------------------
# Final fallback:
# executed IPython input history.
# ------------------------------------------------------------------------------

if full_notebook is None:

    try:

        ip = get_ipython()

    except Exception:

        ip = None


    if ip is None:

        raise RuntimeError(
            "\nCould not recover notebook file or IPython history."
        )


    history = list(
        ip.user_ns.get(
            "In",
            [],
        )
    )


    history = [
        normalize_source(
            source
        )
        for source in history[
            1:
        ]
        if source
    ]


    history = [
        source
        for source in history
        if source.strip()
    ]


    if not history:

        raise RuntimeError(
            "No executed notebook code history available."
        )


    full_notebook = nbformat.v4.new_notebook()


    full_notebook.metadata[
        "stage24_export"
    ] = {
        "mode":
            "EXECUTED_CODE_HISTORY_FALLBACK",

        "warning":
            (
                "The exact Kaggle notebook file was not exposed. "
                "This export contains recovered executed code-cell history."
            ),
    }


    full_notebook.cells.append(
        nbformat.v4.new_markdown_cell(
            "# Stage24 Cross-Dataset Generalization\n\n"
            "Recovered from executed Kaggle code-cell history."
        )
    )


    for execution_order, source in enumerate(
        history,
        start=1,
    ):

        full_notebook.cells.append(
            nbformat.v4.new_code_cell(
                source=source,
                metadata={
                    "recovered_execution_order":
                        execution_order,
                },
            )
        )


    export_mode = (
        "EXECUTED_CODE_HISTORY_FALLBACK"
    )


full_notebook = normalize_notebook_sources(
    full_notebook
)


print(
    "Notebook recovery mode:",
    export_mode,
)


if notebook_source is not None:

    print(
        "Notebook source:",
        notebook_source,
    )


print(
    "Cells:",
    len(
        full_notebook.cells
    ),
)

print()


# ==============================================================================
# 8. ENSURE FULL DOWNLOADABLE NOTEBOOK EXISTS
# ==============================================================================

# Rewrite through nbformat so list-style `source` values become canonical strings.
nbformat.write(
    full_notebook,
    FULL_NOTEBOOK_DOWNLOAD,
)


full_download_sha = sha256_file(
    FULL_NOTEBOOK_DOWNLOAD
)


print(
    "Full downloadable notebook:"
)

print(
    " ",
    FULL_NOTEBOOK_DOWNLOAD,
)

print(
    "SHA256:",
    full_download_sha,
)

print()


# ==============================================================================
# 9. SANITIZED GITHUB NOTEBOOK
# ==============================================================================

sanitized = copy.deepcopy(
    full_notebook
)


sanitized = normalize_notebook_sources(
    sanitized
)


for cell in sanitized.cells:

    if cell.cell_type == "code":

        cell.outputs = []

        cell.execution_count = None


sanitized.metadata[
    "stage24_publication_export"
] = {
    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "source_mode":
        export_mode,

    "scientific_stage_status":
        "CLOSED",

    "frozen_stage24_commit":
        EXPECTED_PARENT,

    "scientific_fits":
        "4/4",

    "evaluable_target_openings":
        "6/6",

    "administratively_cancelled":
        2,
}


nbformat.write(
    sanitized,
    SCRIPT_IPYNB,
)


sanitized_ipynb_sha = sha256_file(
    SCRIPT_IPYNB
)


print(
    "Sanitized GitHub notebook:"
)

print(
    " ",
    SCRIPT_IPYNB.relative_to(
        REPO
    ),
)

print(
    "SHA256:",
    sanitized_ipynb_sha,
)

print()


# ==============================================================================
# 10. FIXED NOTEBOOK -> PYTHON EXPORT
#
# BUG FIX:
#   normalize_source(cell.source)
#
# instead of:
#   cell.source.splitlines()
# ==============================================================================

print("=" * 120)
print("PYTHON SCRIPT EXPORT — FIXED SOURCE NORMALIZATION")
print("=" * 120)


script_lines = [
    "# -*- coding: utf-8 -*-",
    '"""',
    "Stage24 — Cross-Dataset Generalization & Artifact-Sensitivity Audit",
    "",
    "Auto-exported from the Stage24 Kaggle notebook.",
    "",
    f"Frozen scientific parent commit: {EXPECTED_PARENT}",
    f"Notebook export mode: {export_mode}",
    "",
    "Stage24 scientific execution is CLOSED.",
    "",
    "Scientific fits:              4 / 4",
    "Evaluable target openings:    6 / 6",
    "GROUNDED_S4 cancellations:    2",
    "",
    "This script is retained for reproducibility and auditability.",
    "It does not authorize new scientific fits, target openings,",
    "target-selected thresholds, calibration, or post-result variants.",
    '"""',
    "",
]


for cell_index, cell in enumerate(
    sanitized.cells,
    start=1,
):

    source = normalize_source(
        cell.get(
            "source",
            "",
        )
    )


    if cell.cell_type == "markdown":

        script_lines.append(
            "# %% [markdown]"
        )

        script_lines.append(
            f"# Notebook cell {cell_index}"
        )


        for line in source.splitlines():

            if line:

                script_lines.append(
                    "# "
                    +
                    line
                )

            else:

                script_lines.append(
                    "#"
                )


        script_lines.append(
            ""
        )


    elif cell.cell_type == "code":

        script_lines.append(
            "# %%"
        )

        script_lines.append(
            f"# Notebook cell {cell_index}"
        )


        if source.strip():

            script_lines.append(
                source.rstrip()
            )

        else:

            script_lines.append(
                "pass"
            )


        script_lines.append(
            ""
        )


write_text(
    SCRIPT_PY,
    "\n".join(
        script_lines
    ),
)


script_py_sha = sha256_file(
    SCRIPT_PY
)


print(
    "[PASS] Python export completed."
)

print(
    "File:"
)

print(
    " ",
    SCRIPT_PY.relative_to(
        REPO
    ),
)

print(
    "Bytes:",
    f"{SCRIPT_PY.stat().st_size:,}",
)

print(
    "SHA256:",
    script_py_sha,
)

print()


# ==============================================================================
# 11. STAGE24 SCRIPT README
# ==============================================================================

script_readme = f"""# Stage24 Reproducibility Package

## Cross-Dataset Generalization & Artifact-Sensitivity Audit

This directory preserves the Stage24 Kaggle workflow after scientific
execution was permanently closed.

## Files

### `stage24_cross_dataset_generalization.ipynb`

Repository-friendly notebook export with execution outputs removed.

SHA256:

`{sanitized_ipynb_sha}`

### `stage24_cross_dataset_generalization.py`

Percent-cell Python export of the same sanitized notebook.

SHA256:

`{script_py_sha}`

## Full downloadable notebook

A full notebook copy is generated outside the Git repository at:

`{FULL_NOTEBOOK_DOWNLOAD}`

SHA256:

`{full_download_sha}`

Notebook recovery mode:

`{export_mode}`

## Frozen Stage24 state

- Final scientific commit: `{EXPECTED_PARENT}`
- Scientific fits: **4 / 4**
- Evaluable target openings: **6 / 6**
- GROUNDED_S4 cancellations: **2**
- Cancelled opening slots reallocated: **No**
- Remaining Stage24 fits: **0**
- Remaining evaluable target openings: **0**

## Governance

The notebook and Python script preserve the workflow for reproducibility.
Stage24 is closed. They must not be used to silently create additional target
openings, target-guided model variants, target-selected thresholds, or
additional scientific fits.
"""


write_text(
    SCRIPT_README,
    script_readme,
)


# ==============================================================================
# 12. UPDATE JOURNAL EXTENSION SUMMARY
# ==============================================================================

journal_stage24 = f"""## Stage24 — Bidirectional Cross-Dataset Generalization

Stage24 closes the earlier cross-dataset future-work gap through a frozen
bidirectional audit between CSE-CIC-IDS2018 and CICIDS2017.

IDS2018-to-CICIDS2017 transfer retained substantial ranking signal. The
62-feature bridge achieved PR-AUC
**{float(primary_cells['bridge62_PUBLISHED']['pr_auc']):.6f}**
and ROC-AUC
**{float(primary_cells['bridge62_PUBLISHED']['roc_auc']):.6f}**
on **{primary_target_rows:,}** effective CICIDS2017 rows.

The reciprocal CICIDS2017-to-IDS2018 transfer was much weaker. On the frozen
IDS2018 Feb-28 target, attack prevalence was
**{secondary_prevalence:.6f}**, while bridge62 reached PR-AUC
**{float(secondary_cells['bridge62']['pr_auc']):.6f}**
and ROC-AUC
**{float(secondary_cells['bridge62']['roc_auc']):.6f}**.

A preregistered serialization audit further showed that correcting the
CICIDS2017 aggregate flag mapping changed primary bridge70 PR-AUC by
**{float(artifact_pr['point_difference']):+.6f}**, ROC-AUC by
**{float(artifact_roc['point_difference']):+.6f}**, and Brier score by
**{float(artifact_brier['point_difference']):+.6f}**. The corresponding paired
95% bootstrap intervals excluded zero.

The two transfer directions are reported separately and are not averaged.
No target-guided fitting, target threshold tuning, calibration, or feature
search was performed.

Two GROUNDED_S4 target cells were administratively cancelled before opening
because exact durable physical-row membership could not be reconstructed
without introducing a new post-freeze heuristic. No fuzzy substitute was used
and the cancelled slots were not reallocated.

Stage24 therefore supersedes the earlier statement that cross-dataset
generalization remained untested.

Publication package:

- `docs/STAGE24_MANUSCRIPT_INTEGRATION.md`
- `docs/STAGE24_PUBLICATION_TABLES.md`
- `docs/STAGE24_PUBLICATION_CLOSEOUT.md`
- `figures/stage24_cross_dataset/`
- `scripts/stage24/`
"""


append_marked_section(
    JOURNAL_SUMMARY_PATH,
    "<!-- BEGIN STAGE24 JOURNAL EXTENSION SUMMARY -->",
    "<!-- END STAGE24 JOURNAL EXTENSION SUMMARY -->",
    journal_stage24,
)


# ==============================================================================
# 13. UPDATE README
# ==============================================================================

readme_stage24 = f"""## Stage24 — Cross-Dataset Generalization

Stage24 is complete and scientifically closed.

The bidirectional audit evaluates CSE-CIC-IDS2018 and CICIDS2017 using frozen
feature bridges, extractor-semantic controls, source-only threshold selection,
and paired uncertainty analysis.

Frozen headline results:

- IDS2018 → CICIDS2017 bridge62:
  PR-AUC `{float(primary_cells['bridge62_PUBLISHED']['pr_auc']):.6f}`,
  ROC-AUC `{float(primary_cells['bridge62_PUBLISHED']['roc_auc']):.6f}`.
- CICIDS2017 → IDS2018 bridge62:
  PR-AUC `{float(secondary_cells['bridge62']['pr_auc']):.6f}`,
  ROC-AUC `{float(secondary_cells['bridge62']['roc_auc']):.6f}`.
- Reciprocal IDS2018 target prevalence:
  `{secondary_prevalence:.6f}`.
- Aggregate-flag serialization correction produced statistically resolved
  changes in primary bridge70 PR-AUC, ROC-AUC, and Brier score.
- Scientific fits: `4/4`.
- Evaluable target openings: `6/6`.
- GROUNDED_S4 cells cancelled before opening: `2`.
- Cancelled slots reallocated: `No`.

Publication and reproducibility package:

- `docs/STAGE24_MANUSCRIPT_INTEGRATION.md`
- `docs/STAGE24_PUBLICATION_TABLES.md`
- `docs/STAGE24_PUBLICATION_CLOSEOUT.md`
- `figures/stage24_cross_dataset/`
- `scripts/stage24/`
"""


append_marked_section(
    README_PATH,
    "<!-- BEGIN STAGE24 CROSS-DATASET SUMMARY -->",
    "<!-- END STAGE24 CROSS-DATASET SUMMARY -->",
    readme_stage24,
)


print(
    "[PASS] README updated."
)

print(
    "[PASS] JOURNAL_EXTENSION_SUMMARY updated."
)

print()


# ==============================================================================
# 14. PUBLICATION MANIFEST
# ==============================================================================

publication_files = [
    DOC_MANUSCRIPT,
    DOC_MANUSCRIPT_TEX,
    DOC_TABLES_MD,
    DOC_TABLES_TEX,
    DOC_CLOSEOUT,

    TABLE1_CSV,
    TABLE2_CSV,
    TABLE3_CSV,
    TABLE4_CSV,

    FIG1_PNG,
    FIG1_SVG,
    FIG2_PNG,
    FIG2_SVG,
    FIG3_PNG,
    FIG3_SVG,
    FIG4_PNG,
    FIG4_SVG,

    SCRIPT_IPYNB,
    SCRIPT_PY,
    SCRIPT_README,
]


for path in publication_files:

    if not path.is_file():

        raise RuntimeError(
            f"Publication package file missing:\n{path}"
        )


manifest = {
    "stage":
        "Stage24-PUB-R1",

    "type":
        "NON_SCIENTIFIC_PUBLICATION_AND_REPRODUCIBILITY_CLOSEOUT",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent_commit":
        EXPECTED_PARENT,

    "stage24_final_synthesis_sha256":
        final_sha,

    "scientific_execution_closed":
        True,

    "scientific_fits": {
        "completed":
            4,

        "budget":
            4,

        "remaining":
            0,
    },

    "target_openings": {
        "evaluable_completed":
            6,

        "original_budget":
            8,

        "administratively_cancelled":
            2,

        "cancelled_slots_reallocated":
            False,

        "remaining_evaluable":
            0,
    },

    "notebook_export": {
        "mode":
            export_mode,

        "full_downloadable_copy":
            str(
                FULL_NOTEBOOK_DOWNLOAD
            ),

        "full_downloadable_sha256":
            full_download_sha,

        "repo_sanitized_ipynb":
            str(
                SCRIPT_IPYNB.relative_to(
                    REPO
                )
            ),

        "repo_sanitized_ipynb_sha256":
            sanitized_ipynb_sha,

        "repo_python_script":
            str(
                SCRIPT_PY.relative_to(
                    REPO
                )
            ),

        "repo_python_script_sha256":
            script_py_sha,
    },

    "publication_files": {
        str(
            path.relative_to(
                REPO
            )
        ):
            sha256_file(
                path
            )

        for path in publication_files
    },

    "frozen_scientific_artifacts": {
        "final_synthesis_sha256":
            final_sha,

        "final_report_sha256":
            final_report_sha,

        "primary_bootstrap_sha256":
            primary_bootstrap_sha,

        "secondary_bootstrap_sha256":
            secondary_bootstrap_sha,
    },

    "anti_adaptation": {
        "model_fit":
            False,

        "model_load":
            False,

        "model_inference":
            False,

        "target_opening":
            False,

        "target_threshold_search":
            False,

        "target_calibration":
            False,

        "target_feature_change":
            False,
    },
}


write_text(
    PUB_MANIFEST,
    json.dumps(
        manifest,
        indent=2,
        sort_keys=True,
    ),
)


manifest_sha = sha256_file(
    PUB_MANIFEST
)


write_text(
    PUB_MANIFEST_SHA,
    f"{manifest_sha}  {PUB_MANIFEST.name}",
)


print(
    "Publication manifest SHA:"
)

print(
    " ",
    manifest_sha,
)

print()


# ==============================================================================
# 15. DOWNLOADABLE NOTEBOOK ZIP
# ==============================================================================

print("=" * 120)
print("DOWNLOADABLE EXPORTS")
print("=" * 120)


notebook_zip_dir = (
    DOWNLOAD_DIR
    / "notebook_package"
)


if notebook_zip_dir.exists():

    shutil.rmtree(
        notebook_zip_dir
    )


notebook_zip_dir.mkdir(
    parents=True,
    exist_ok=True,
)


for source_path in [
    FULL_NOTEBOOK_DOWNLOAD,
    SCRIPT_IPYNB,
    SCRIPT_PY,
    SCRIPT_README,
]:

    shutil.copy2(
        source_path,
        notebook_zip_dir
        /
        source_path.name,
    )


notebook_zip_candidate = Path(
    str(
        NOTEBOOK_ZIP_BASE
    )
    +
    ".zip"
)


if notebook_zip_candidate.exists():

    notebook_zip_candidate.unlink()


notebook_zip_path = Path(
    shutil.make_archive(
        str(
            NOTEBOOK_ZIP_BASE
        ),
        "zip",
        root_dir=notebook_zip_dir,
    )
)


notebook_zip_sha = sha256_file(
    notebook_zip_path
)


print(
    "Notebook ZIP:"
)

print(
    " ",
    notebook_zip_path,
)

print(
    "SHA256:",
    notebook_zip_sha,
)

print()


# ==============================================================================
# 16. DOWNLOADABLE PUBLICATION ZIP
# ==============================================================================

publication_zip_dir = (
    DOWNLOAD_DIR
    / "publication_package"
)


if publication_zip_dir.exists():

    shutil.rmtree(
        publication_zip_dir
    )


publication_zip_dir.mkdir(
    parents=True,
    exist_ok=True,
)


for path in (
    publication_files
    +
    [
        PUB_MANIFEST,
        PUB_MANIFEST_SHA,
    ]
):

    relative_path = path.relative_to(
        REPO
    )

    destination = (
        publication_zip_dir
        /
        relative_path
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        path,
        destination,
    )


publication_zip_candidate = Path(
    str(
        PUBLICATION_ZIP_BASE
    )
    +
    ".zip"
)


if publication_zip_candidate.exists():

    publication_zip_candidate.unlink()


publication_zip_path = Path(
    shutil.make_archive(
        str(
            PUBLICATION_ZIP_BASE
        ),
        "zip",
        root_dir=publication_zip_dir,
    )
)


publication_zip_sha = sha256_file(
    publication_zip_path
)


print(
    "Publication ZIP:"
)

print(
    " ",
    publication_zip_path,
)

print(
    "SHA256:",
    publication_zip_sha,
)

print()


# ==============================================================================
# 17. GITHUB CREDENTIAL
# ==============================================================================

github_token = None

token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()


    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )


            if value and value.strip():

                github_token = value.strip()

                token_source = name

                break

        except Exception:

            pass

except Exception:

    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )


        if value and value.strip():

            github_token = value.strip()

            token_source = (
                "ENV:"
                +
                name
            )

            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved before publication closeout.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "[PASS] Remote main still equals frozen Stage24 commit."
)

print()


# ==============================================================================
# 18. REPOSITORY DIRTY-STATE SAFETY AUDIT
# ==============================================================================

print("=" * 120)
print("DIRTY-STATE AUDIT")
print("=" * 120)


status = git_cmd(
    "status",
    "--porcelain",
)


print(
    status
)

print()


if not status:

    raise RuntimeError(
        "No Stage24 publication changes found."
    )


allowed_exact = {
    "README.md",
    "docs/JOURNAL_EXTENSION_SUMMARY.md",
    "docs/STAGE24_MANUSCRIPT_INTEGRATION.md",
    "docs/STAGE24_MANUSCRIPT_INTEGRATION.tex",
    "docs/STAGE24_PUBLICATION_TABLES.md",
    "docs/STAGE24_PUBLICATION_TABLES.tex",
    "docs/STAGE24_PUBLICATION_CLOSEOUT.md",
}


allowed_prefixes = (
    "figures/stage24_cross_dataset/",
    "scripts/stage24/",
    "results/stage24_cross_dataset/stage24_publication_package/",
)


dirty_paths = []


for line in status.splitlines():

    if not line.strip():

        continue


    # Porcelain:
    # XY<space>path
    path = line[
        3:
    ].strip()


    # Handle rename-like output defensively.
    if " -> " in path:

        path = path.split(
            " -> "
        )[
            -1
        ].strip()


    dirty_paths.append(
        path
    )


unexpected_dirty = [
    path
    for path in dirty_paths
    if (
        path not in allowed_exact
        and
        not any(
            path.startswith(
                prefix
            )
            for prefix in allowed_prefixes
        )
    )
]


if unexpected_dirty:

    raise RuntimeError(
        "\nUnexpected repository modifications exist:\n"
        +
        "\n".join(
            unexpected_dirty
        )
    )


frozen_prefixes = (
    "results/stage24_cross_dataset/stage24_0_",
    "results/stage24_cross_dataset/stage24_1_",
    "results/stage24_cross_dataset/stage24_2_",
    "results/stage24_cross_dataset/stage24_3_",
    "results/stage24_cross_dataset/stage24_4_",
    "results/stage24_cross_dataset/stage24_5_",
    "results/stage24_cross_dataset/stage24_6_",
)


frozen_dirty = [
    path
    for path in dirty_paths
    if any(
        path.startswith(
            prefix
        )
        for prefix in frozen_prefixes
    )
]


if frozen_dirty:

    raise RuntimeError(
        "\nA frozen Stage24 scientific artifact was modified:\n"
        +
        "\n".join(
            frozen_dirty
        )
    )


print(
    "[PASS] All dirty files belong exclusively to publication/reproducibility closeout."
)

print(
    "[PASS] Frozen Stage24 scientific directories unchanged."
)

print()


# ==============================================================================
# 19. GIT ADD
# ==============================================================================

git_cmd(
    "add",
    "--",
    "README.md",
    "docs/JOURNAL_EXTENSION_SUMMARY.md",
    "docs/STAGE24_MANUSCRIPT_INTEGRATION.md",
    "docs/STAGE24_MANUSCRIPT_INTEGRATION.tex",
    "docs/STAGE24_PUBLICATION_TABLES.md",
    "docs/STAGE24_PUBLICATION_TABLES.tex",
    "docs/STAGE24_PUBLICATION_CLOSEOUT.md",
    "figures/stage24_cross_dataset",
    "scripts/stage24",
    "results/stage24_cross_dataset/stage24_publication_package",
)


staged = git_cmd(
    "diff",
    "--cached",
    "--name-only",
)


print("=" * 120)
print("STAGED FILES")
print("=" * 120)

print(
    staged
)

print()


if not staged:

    raise RuntimeError(
        "No files staged for Stage24 publication closeout."
    )


for path in staged.splitlines():

    if any(
        path.startswith(
            prefix
        )
        for prefix in frozen_prefixes
    ):

        raise RuntimeError(
            "\nFrozen Stage24 scientific file was accidentally staged:\n"
            f"{path}"
        )


print(
    "[PASS] Staging audit complete."
)

print()


# ==============================================================================
# 20. GIT AUTHOR SAFETY
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    git_cmd(
        "config",
        "--local",
        "user.name",
        git_cmd(
            "log",
            "-1",
            "--format=%an",
        ),
    )


if not author_email:

    git_cmd(
        "config",
        "--local",
        "user.email",
        git_cmd(
            "log",
            "-1",
            "--format=%ae",
        ),
    )


# ==============================================================================
# 21. COMMIT
# ==============================================================================

print("=" * 120)
print("GIT COMMIT")
print("=" * 120)


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: add publication package and reproducibility notebook",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "\nStage24 publication commit parent mismatch.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {parent}"
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 22. PUSH
# ==============================================================================

print("=" * 120)
print("PUSH + REMOTE VERIFY")
print("=" * 120)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


final_status = git_cmd(
    "status",
    "--porcelain",
)


if final_status:

    raise RuntimeError(
        "\nRepository is not clean after Stage24 publication closeout:\n"
        +
        final_status
    )


print(
    "[PASS] Remote main == local publication closeout commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 23. CLICKABLE KAGGLE DOWNLOADS
# ==============================================================================

print("=" * 120)
print("KAGGLE DOWNLOADS")
print("=" * 120)


try:

    from IPython.display import display, FileLink


    print(
        "Full Stage24 notebook:"
    )

    display(
        FileLink(
            str(
                FULL_NOTEBOOK_DOWNLOAD
            )
        )
    )


    print(
        "Stage24 notebook/script ZIP:"
    )

    display(
        FileLink(
            str(
                notebook_zip_path
            )
        )
    )


    print(
        "Stage24 publication package ZIP:"
    )

    display(
        FileLink(
            str(
                publication_zip_path
            )
        )
    )


except Exception as exc:

    print(
        "[INFO] FileLink rendering unavailable:",
        repr(
            exc
        ),
    )


print()


# ==============================================================================
# 24. FINAL
# ==============================================================================

print("=" * 120)
print("STAGE24 PUBLICATION + REPRODUCIBILITY CLOSEOUT: PASS")
print("=" * 120)

print()

print(
    "Scientific Stage24 status:       CLOSED"
)

print(
    "Scientific fits:                 4 / 4"
)

print(
    "Evaluable target openings:       6 / 6"
)

print(
    "GROUNDED_S4 cancellations:       2"
)

print(
    "Cancelled slots reallocated:     NO"
)

print()

print(
    "New model fit in recovery:       NO"
)

print(
    "New model inference:             NO"
)

print(
    "New target opening:              NO"
)

print()

print(
    "Publication tables:              COMPLETE"
)

print(
    "Publication figures:             COMPLETE"
)

print(
    "Results section:                 COMPLETE"
)

print(
    "Discussion:                      COMPLETE"
)

print(
    "Limitations / Threats:           COMPLETE"
)

print(
    "Contributions:                   COMPLETE"
)

print()

print(
    "Notebook export mode:"
)

print(
    " ",
    export_mode,
)

print()

print(
    "GitHub notebook:"
)

print(
    " ",
    SCRIPT_IPYNB.relative_to(
        REPO
    ),
)

print()

print(
    "GitHub Python script:"
)

print(
    " ",
    SCRIPT_PY.relative_to(
        REPO
    ),
)

print()

print(
    "Full downloadable notebook:"
)

print(
    " ",
    FULL_NOTEBOOK_DOWNLOAD,
)

print()

print(
    "Notebook/script ZIP:"
)

print(
    " ",
    notebook_zip_path,
)

print()

print(
    "Publication ZIP:"
)

print(
    " ",
    publication_zip_path,
)

print()

print(
    "Notebook SHA:"
)

print(
    " ",
    sanitized_ipynb_sha,
)

print()

print(
    "Python script SHA:"
)

print(
    " ",
    script_py_sha,
)

print()

print(
    "Publication manifest SHA:"
)

print(
    " ",
    manifest_sha,
)

print()

print(
    "GitHub commit:"
)

print(
    " ",
    commit,
)

print()

print(
    "Remote main:"
)

print(
    " ",
    remote_after,
)

print()

print(
    "STAGE24 = FINISHED 😂"
)

print(
    "TWO STAGES LEFT."
)

print("=" * 120)

STAGE24-PUB-R1 — NOTEBOOK EXPORT + PUBLICATION CLOSEOUT RECOVERY

GOVERNANCE
HEAD: a4b6a3854109ba3d85954fb4a40afe6fe1ee6114
[PASS] Frozen Stage24 HEAD unchanged.
[PASS] No fit/opening ledger.
[PASS] No pre-existing staged changes.

FROZEN SCIENTIFIC STATE
Final synthesis SHA:     785bbcb00140f4d7e07e9b49ad33924166b5995e66a229c986d5b1abcbcaee4b
Final report SHA:        f867289cec34eca6666d02deec01864395503cea8a59b7eac88e1759d5eb2a0b
Primary bootstrap SHA:   96732023e0a1c9b52a79fafa59ea4b851ef7f8bae1814e4b4b4fde0ef0df09aa
Secondary bootstrap SHA: 28005e0f13d86658ba81e6b19dc154b94eedb316ea5ce0348287265c00541953

[PASS] Stage24 scientific execution remains frozen.

PREVIOUSLY GENERATED PUBLICATION ASSETS
[FOUND] results/stage24_cross_dataset/stage24_publication_package/tables/table24_1_bidirectional_generalization.csv
[FOUND] results/stage24_cross_dataset/stage24_publication_package/tables/table24_2_paired_bootstrap_contrasts.csv
[FOUND] results/stage24_cross_dataset/stage24_publication_pa

RuntimeError: 
Unexpected repository modifications exist:
EADME.md

In [46]:
# ==============================================================================
# STAGE24-PUB-R2 — FINAL GIT CLOSEOUT RECOVERY
#
# Previous failure:
#
#   RuntimeError:
#       Unexpected repository modifications exist:
#       EADME.md
#
# Root cause:
#   `run_cmd(...).strip()` removed the leading whitespace from the FIRST line
#   of `git status --porcelain`, so:
#
#       " M README.md"
#
#   became:
#
#       "M README.md"
#
#   and line[3:] incorrectly became:
#
#       "EADME.md"
#
# SCIENTIFIC STATE:
#   Stage24 remains CLOSED.
#
#   scientific fits             = 4 / 4
#   evaluable target openings   = 6 / 6
#   GROUNDED_S4 cancellations   = 2
#   new fit                     = 0
#   new inference               = 0
#   new target opening          = 0
#
# THIS CELL DOES NOT REGENERATE PUBLICATION CONTENT.
#
# It only:
#   1. verifies the already-created Stage24 publication package,
#   2. audits changed paths using `git diff` + `git ls-files`,
#      avoiding porcelain-prefix parsing entirely,
#   3. stages the publication/reproducibility package,
#   4. commits,
#   5. pushes,
#   6. verifies remote main and clean working tree,
#   7. renders Kaggle download links.
#
# Expected HEAD:
#   a4b6a3854109ba3d85954fb4a40afe6fe1ee6114
# ==============================================================================

from __future__ import annotations

import os
import json
import base64
import hashlib
import subprocess
from pathlib import Path


print("=" * 120)
print("STAGE24-PUB-R2 — FINAL GIT CLOSEOUT RECOVERY")
print("=" * 120)
print()


# ==============================================================================
# 0. FROZEN ANCHORS
# ==============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "a4b6a3854109ba3d85954fb4a40afe6fe1ee6114"
)


EXPECTED_FINAL_SYNTHESIS_SHA = (
    "785bbcb00140f4d7e07e9b49ad33924166b5995e66a229c986d5b1abcbcaee4b"
)

EXPECTED_FINAL_REPORT_SHA = (
    "f867289cec34eca6666d02deec01864395503cea8a59b7eac88e1759d5eb2a0b"
)

EXPECTED_PRIMARY_BOOTSTRAP_SHA = (
    "96732023e0a1c9b52a79fafa59ea4b851ef7f8bae1814e4b4b4fde0ef0df09aa"
)

EXPECTED_SECONDARY_BOOTSTRAP_SHA = (
    "28005e0f13d86658ba81e6b19dc154b94eedb316ea5ce0348287265c00541953"
)


# Already-created publication/reproducibility identities from R1.
EXPECTED_FULL_NOTEBOOK_SHA = (
    "f51c48e29070b2b8b87bc098403b3cd40800317a4cb5c6750e7e31eb360784c5"
)

EXPECTED_SANITIZED_NOTEBOOK_SHA = (
    "9f2d54b4f36f516544026902e269d6601634ffda2c9c287a5ffef1758c9c4558"
)

EXPECTED_PYTHON_SCRIPT_SHA = (
    "104c8b3c3bde1eb81e7a70a05e15854e98d65f23d9d4799ea8ec22a064b106c6"
)

EXPECTED_PUBLICATION_MANIFEST_SHA = (
    "7cc628debf53e22ee0c71e21e307f7e9fa766cbeecdd527174db2dbe47e3bf82"
)

EXPECTED_NOTEBOOK_ZIP_SHA = (
    "c38b66637679e4123cfd84ac2c4b7c6c5b7e3701bb041ef250176fe7caa01647"
)

EXPECTED_PUBLICATION_ZIP_SHA = (
    "6f2e8d8ab033c9f730269e6d32f165c588c0e5598f842bf30c963a32abb049e5"
)


# ==============================================================================
# 1. PATHS
# ==============================================================================

STAGE24_BASE = (
    REPO
    / "results"
    / "stage24_cross_dataset"
)


FINAL_DIR = (
    STAGE24_BASE
    / "stage24_6_final_synthesis"
)

FINAL_JSON = (
    FINAL_DIR
    / "stage24_6_final_synthesis.json"
)

FINAL_JSON_SHA = (
    FINAL_DIR
    / "stage24_6_final_synthesis.sha256"
)

FINAL_REPORT = (
    FINAL_DIR
    / "stage24_6_final_synthesis.md"
)

SECONDARY_BOOTSTRAP = (
    FINAL_DIR
    / "stage24_6_secondary_paired_bootstrap_replicates.npz"
)


PRIMARY_BOOTSTRAP = (
    STAGE24_BASE
    / "stage24_3_primary_results_freeze"
    / "stage24_3a_primary_paired_bootstrap"
    / "stage24_3a_primary_paired_bootstrap_replicates.npz"
)


PUB_RESULT_DIR = (
    STAGE24_BASE
    / "stage24_publication_package"
)

PUB_MANIFEST = (
    PUB_RESULT_DIR
    / "stage24_publication_package_manifest.json"
)

PUB_MANIFEST_SHA = (
    PUB_RESULT_DIR
    / "stage24_publication_package_manifest.sha256"
)


TABLE_DIR = (
    PUB_RESULT_DIR
    / "tables"
)


FIG_DIR = (
    REPO
    / "figures"
    / "stage24_cross_dataset"
)


SCRIPT_DIR = (
    REPO
    / "scripts"
    / "stage24"
)

SCRIPT_IPYNB = (
    SCRIPT_DIR
    / "stage24_cross_dataset_generalization.ipynb"
)

SCRIPT_PY = (
    SCRIPT_DIR
    / "stage24_cross_dataset_generalization.py"
)

SCRIPT_README = (
    SCRIPT_DIR
    / "README.md"
)


DOC_MANUSCRIPT = (
    REPO
    / "docs"
    / "STAGE24_MANUSCRIPT_INTEGRATION.md"
)

DOC_MANUSCRIPT_TEX = (
    REPO
    / "docs"
    / "STAGE24_MANUSCRIPT_INTEGRATION.tex"
)

DOC_TABLES_MD = (
    REPO
    / "docs"
    / "STAGE24_PUBLICATION_TABLES.md"
)

DOC_TABLES_TEX = (
    REPO
    / "docs"
    / "STAGE24_PUBLICATION_TABLES.tex"
)

DOC_CLOSEOUT = (
    REPO
    / "docs"
    / "STAGE24_PUBLICATION_CLOSEOUT.md"
)

README_PATH = (
    REPO
    / "README.md"
)

JOURNAL_SUMMARY_PATH = (
    REPO
    / "docs"
    / "JOURNAL_EXTENSION_SUMMARY.md"
)


FULL_NOTEBOOK_DOWNLOAD = Path(
    "/kaggle/working/stage24_exports/"
    "stage24_cross_dataset_generalization_FULL.ipynb"
)

NOTEBOOK_ZIP = Path(
    "/kaggle/working/stage24_notebook_export.zip"
)

PUBLICATION_ZIP = Path(
    "/kaggle/working/stage24_publication_package.zip"
)


FIT_LEDGER = Path(
    "/kaggle/working/stage24_secondary_fit_runtime_ledger.json"
)

OPENING_LEDGER = Path(
    "/kaggle/working/stage24_target_opening_runtime_ledger.json"
)


# ==============================================================================
# 2. HELPERS
# ==============================================================================

def run_cmd(
    args,
    *,
    cwd=None,
    check=True,
):

    p = subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            "\nCommand failed:\n"
            + " ".join(
                str(x)
                for x in args
            )
            + "\n\n"
            + (p.stdout or "")
        )

    # Fine for normal commands.
    # We DO NOT parse `git status --porcelain` in this recovery.
    return (p.stdout or "").strip()


def git_cmd(
    *args,
    auth_header=None,
    check=True,
):

    cmd = [
        "git"
    ]

    if auth_header is not None:

        cmd += [
            "-c",
            "credential.helper=",
            "-c",
            f"http.extraHeader=AUTHORIZATION: Basic {auth_header}",
        ]

    cmd += [
        str(x)
        for x in args
    ]

    return run_cmd(
        cmd,
        cwd=REPO,
        check=check,
    )


def sha256_file(
    path,
):

    path = Path(
        path
    )

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def verify_sha(
    path,
    expected,
    label,
):

    path = Path(
        path
    )

    if not path.is_file():

        raise RuntimeError(
            f"Missing {label}:\n{path}"
        )

    actual = sha256_file(
        path
    )

    if actual != expected:

        raise RuntimeError(
            f"\n{label} SHA mismatch.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}\n"
            f"File:     {path}"
        )

    return actual


def verify_sidecar(
    artifact,
    sidecar,
):

    artifact = Path(
        artifact
    )

    sidecar = Path(
        sidecar
    )

    if not artifact.is_file():

        raise RuntimeError(
            f"Missing artifact:\n{artifact}"
        )

    if not sidecar.is_file():

        raise RuntimeError(
            f"Missing SHA sidecar:\n{sidecar}"
        )

    actual = sha256_file(
        artifact
    )

    expected = (
        sidecar
        .read_text(
            encoding="utf-8"
        )
        .strip()
        .split()[0]
    )

    if actual != expected:

        raise RuntimeError(
            "\nSHA mismatch.\n"
            f"Artifact: {artifact}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )

    return actual


def get_unstaged_tracked_paths():
    """
    Robust path discovery.

    Does NOT parse porcelain status prefixes.
    """

    output = git_cmd(
        "diff",
        "--name-only",
    )

    return {
        line.strip()
        for line in output.splitlines()
        if line.strip()
    }


def get_untracked_paths():
    """
    Robust untracked-file discovery.

    Does NOT parse porcelain XY prefixes.
    """

    output = git_cmd(
        "ls-files",
        "--others",
        "--exclude-standard",
    )

    return {
        line.strip()
        for line in output.splitlines()
        if line.strip()
    }


def get_staged_paths():

    output = git_cmd(
        "diff",
        "--cached",
        "--name-only",
    )

    return {
        line.strip()
        for line in output.splitlines()
        if line.strip()
    }


# ==============================================================================
# 3. GOVERNANCE
# ==============================================================================

print("=" * 120)
print("GOVERNANCE")
print("=" * 120)


head = git_cmd(
    "rev-parse",
    "HEAD",
)


print(
    "HEAD:",
    head,
)


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected HEAD.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


if FIT_LEDGER.exists():

    raise RuntimeError(
        f"Unexpected fit ledger:\n{FIT_LEDGER}"
    )


if OPENING_LEDGER.exists():

    raise RuntimeError(
        f"Unexpected target-opening ledger:\n{OPENING_LEDGER}"
    )


staged_before = get_staged_paths()


if staged_before:

    raise RuntimeError(
        "\nUnexpected pre-existing staged files:\n"
        +
        "\n".join(
            sorted(
                staged_before
            )
        )
    )


print(
    "[PASS] HEAD remains exact Stage24 scientific freeze."
)

print(
    "[PASS] No fit/opening ledger."
)

print(
    "[PASS] Nothing staged."
)

print()


# ==============================================================================
# 4. VERIFY FROZEN SCIENTIFIC ARTIFACTS AGAIN
# ==============================================================================

print("=" * 120)
print("FROZEN SCIENTIFIC ARTIFACTS")
print("=" * 120)


final_sha = verify_sidecar(
    FINAL_JSON,
    FINAL_JSON_SHA,
)


if final_sha != EXPECTED_FINAL_SYNTHESIS_SHA:

    raise RuntimeError(
        "Final synthesis SHA changed."
    )


verify_sha(
    FINAL_REPORT,
    EXPECTED_FINAL_REPORT_SHA,
    "Stage24 final report",
)


verify_sha(
    PRIMARY_BOOTSTRAP,
    EXPECTED_PRIMARY_BOOTSTRAP_SHA,
    "Stage24 primary bootstrap",
)


verify_sha(
    SECONDARY_BOOTSTRAP,
    EXPECTED_SECONDARY_BOOTSTRAP_SHA,
    "Stage24 secondary bootstrap",
)


stage24 = json.loads(
    FINAL_JSON.read_text(
        encoding="utf-8"
    )
)


if (
    stage24.get(
        "status"
    )
    !=
    "STAGE24_CROSS_DATASET_AUDIT_COMPLETE"
):

    raise RuntimeError(
        "Stage24 final status changed."
    )


if (
    stage24[
        "completion"
    ][
        "Stage24_complete"
    ]
    is not True
):

    raise RuntimeError(
        "Stage24 completion flag changed."
    )


if (
    stage24[
        "completion"
    ][
        "scientific_fits"
    ]
    !=
    "4/4"
):

    raise RuntimeError(
        "Scientific-fit accounting changed."
    )


if (
    stage24[
        "completion"
    ][
        "evaluable_target_openings"
    ]
    !=
    "6/6"
):

    raise RuntimeError(
        "Target-opening accounting changed."
    )


print(
    "Final synthesis SHA:    ",
    EXPECTED_FINAL_SYNTHESIS_SHA,
)

print(
    "Final report SHA:       ",
    EXPECTED_FINAL_REPORT_SHA,
)

print(
    "Primary bootstrap SHA:  ",
    EXPECTED_PRIMARY_BOOTSTRAP_SHA,
)

print(
    "Secondary bootstrap SHA:",
    EXPECTED_SECONDARY_BOOTSTRAP_SHA,
)

print()

print(
    "[PASS] Frozen scientific state untouched."
)

print()


# ==============================================================================
# 5. VERIFY ALREADY-CREATED PUBLICATION PACKAGE
# ==============================================================================

print("=" * 120)
print("PUBLICATION PACKAGE CHECKPOINT")
print("=" * 120)


required_repo_assets = [
    README_PATH,
    JOURNAL_SUMMARY_PATH,

    DOC_MANUSCRIPT,
    DOC_MANUSCRIPT_TEX,
    DOC_TABLES_MD,
    DOC_TABLES_TEX,
    DOC_CLOSEOUT,

    TABLE_DIR
    / "table24_1_bidirectional_generalization.csv",

    TABLE_DIR
    / "table24_2_paired_bootstrap_contrasts.csv",

    TABLE_DIR
    / "table24_3_secondary_threshold_transfer.csv",

    TABLE_DIR
    / "table24_4_governance_and_populations.csv",

    FIG_DIR
    / "fig24_1_normalized_pr_auc_directionality.png",

    FIG_DIR
    / "fig24_1_normalized_pr_auc_directionality.svg",

    FIG_DIR
    / "fig24_2_roc_auc_directionality.png",

    FIG_DIR
    / "fig24_2_roc_auc_directionality.svg",

    FIG_DIR
    / "fig24_3_paired_effects_forest.png",

    FIG_DIR
    / "fig24_3_paired_effects_forest.svg",

    FIG_DIR
    / "fig24_4_secondary_threshold_transfer.png",

    FIG_DIR
    / "fig24_4_secondary_threshold_transfer.svg",

    SCRIPT_IPYNB,
    SCRIPT_PY,
    SCRIPT_README,

    PUB_MANIFEST,
    PUB_MANIFEST_SHA,
]


missing = [
    path
    for path in required_repo_assets
    if not path.is_file()
]


if missing:

    raise RuntimeError(
        "\nPublication checkpoint is incomplete.\nMissing:\n"
        +
        "\n".join(
            str(
                path
            )
            for path in missing
        )
    )


verify_sha(
    SCRIPT_IPYNB,
    EXPECTED_SANITIZED_NOTEBOOK_SHA,
    "Stage24 sanitized GitHub notebook",
)


verify_sha(
    SCRIPT_PY,
    EXPECTED_PYTHON_SCRIPT_SHA,
    "Stage24 Python script",
)


manifest_sha = verify_sidecar(
    PUB_MANIFEST,
    PUB_MANIFEST_SHA,
)


if manifest_sha != EXPECTED_PUBLICATION_MANIFEST_SHA:

    raise RuntimeError(
        "\nPublication manifest SHA changed.\n"
        f"Expected: {EXPECTED_PUBLICATION_MANIFEST_SHA}\n"
        f"Actual:   {manifest_sha}"
    )


print(
    "Sanitized notebook SHA:",
    EXPECTED_SANITIZED_NOTEBOOK_SHA,
)

print(
    "Python script SHA:      ",
    EXPECTED_PYTHON_SCRIPT_SHA,
)

print(
    "Publication manifest SHA:",
    manifest_sha,
)

print()

print(
    "[PASS] Publication/reproducibility package checkpoint complete."
)

print()


# ==============================================================================
# 6. VERIFY DOWNLOADABLE NOTEBOOK + ZIPS
# ==============================================================================

print("=" * 120)
print("DOWNLOADABLE EXPORTS")
print("=" * 120)


verify_sha(
    FULL_NOTEBOOK_DOWNLOAD,
    EXPECTED_FULL_NOTEBOOK_SHA,
    "Full Stage24 notebook",
)


verify_sha(
    NOTEBOOK_ZIP,
    EXPECTED_NOTEBOOK_ZIP_SHA,
    "Stage24 notebook ZIP",
)


verify_sha(
    PUBLICATION_ZIP,
    EXPECTED_PUBLICATION_ZIP_SHA,
    "Stage24 publication ZIP",
)


print(
    "Full notebook:"
)

print(
    " ",
    FULL_NOTEBOOK_DOWNLOAD,
)

print(
    " SHA:",
    EXPECTED_FULL_NOTEBOOK_SHA,
)

print()

print(
    "Notebook/script ZIP:"
)

print(
    " ",
    NOTEBOOK_ZIP,
)

print(
    " SHA:",
    EXPECTED_NOTEBOOK_ZIP_SHA,
)

print()

print(
    "Publication ZIP:"
)

print(
    " ",
    PUBLICATION_ZIP,
)

print(
    " SHA:",
    EXPECTED_PUBLICATION_ZIP_SHA,
)

print()


# ==============================================================================
# 7. ROBUST DIRTY-PATH AUDIT
#
# DO NOT USE `git status --porcelain` PREFIX SLICING.
# ==============================================================================

print("=" * 120)
print("ROBUST DIRTY-PATH AUDIT")
print("=" * 120)


tracked_modified = get_unstaged_tracked_paths()

untracked = get_untracked_paths()


dirty_paths = (
    tracked_modified
    |
    untracked
)


print(
    "Tracked modified:"
)


for path in sorted(
    tracked_modified
):

    print(
        "  M",
        path,
    )


print()

print(
    "Untracked:"
)


for path in sorted(
    untracked
):

    print(
        "  ?",
        path,
    )


print()


if not dirty_paths:

    raise RuntimeError(
        "No Stage24 publication changes found."
    )


# ==============================================================================
# 8. ALLOWED PUBLICATION PATHS
# ==============================================================================

allowed_exact = {
    "README.md",
    "docs/JOURNAL_EXTENSION_SUMMARY.md",

    "docs/STAGE24_MANUSCRIPT_INTEGRATION.md",
    "docs/STAGE24_MANUSCRIPT_INTEGRATION.tex",

    "docs/STAGE24_PUBLICATION_TABLES.md",
    "docs/STAGE24_PUBLICATION_TABLES.tex",

    "docs/STAGE24_PUBLICATION_CLOSEOUT.md",
}


allowed_prefixes = (
    "figures/stage24_cross_dataset/",
    "scripts/stage24/",
    "results/stage24_cross_dataset/stage24_publication_package/",
)


unexpected_dirty = [
    path
    for path in sorted(
        dirty_paths
    )
    if (
        path not in allowed_exact
        and
        not any(
            path.startswith(
                prefix
            )
            for prefix in allowed_prefixes
        )
    )
]


if unexpected_dirty:

    raise RuntimeError(
        "\nUnexpected repository modifications exist:\n"
        +
        "\n".join(
            unexpected_dirty
        )
    )


# ==============================================================================
# 9. FROZEN SCIENTIFIC PATH PROTECTION
# ==============================================================================

frozen_prefixes = (
    "results/stage24_cross_dataset/stage24_0_",
    "results/stage24_cross_dataset/stage24_1_",
    "results/stage24_cross_dataset/stage24_2_",
    "results/stage24_cross_dataset/stage24_3_",
    "results/stage24_cross_dataset/stage24_4_",
    "results/stage24_cross_dataset/stage24_5_",
    "results/stage24_cross_dataset/stage24_6_",
)


frozen_dirty = [
    path
    for path in sorted(
        dirty_paths
    )
    if any(
        path.startswith(
            prefix
        )
        for prefix in frozen_prefixes
    )
]


if frozen_dirty:

    raise RuntimeError(
        "\nFrozen Stage24 scientific artifacts were modified:\n"
        +
        "\n".join(
            frozen_dirty
        )
    )


print(
    "[PASS] Every dirty path belongs to the Stage24 publication/reproducibility package."
)

print(
    "[PASS] Frozen Stage24 scientific result trees untouched."
)

print()


# ==============================================================================
# 10. GITHUB CREDENTIAL + REMOTE PARENT
# ==============================================================================

print("=" * 120)
print("GITHUB REMOTE GATE")
print("=" * 120)


github_token = None

token_source = None


try:

    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()


    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
    ]:

        try:

            value = secrets.get_secret(
                name
            )


            if value and value.strip():

                github_token = value.strip()

                token_source = name

                break

        except Exception:

            pass

except Exception:

    pass


if github_token is None:

    for name in [
        "GITHUB_TOKEN",
        "GH_TOKEN",
    ]:

        value = os.environ.get(
            name
        )


        if value and value.strip():

            github_token = value.strip()

            token_source = (
                "ENV:"
                +
                name
            )

            break


if github_token is None:

    raise RuntimeError(
        "GitHub token unavailable."
    )


auth_header = base64.b64encode(
    (
        "x-access-token:"
        +
        github_token
    ).encode()
).decode()


remote_before = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_before != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote main moved before Stage24 publication closeout.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_before}"
    )


print(
    "GitHub credential:",
    token_source,
)

print(
    "Remote main:",
    remote_before,
)

print()

print(
    "[PASS] Remote main still equals Stage24 scientific freeze."
)

print()


# ==============================================================================
# 11. STAGE PUBLICATION PACKAGE
# ==============================================================================

print("=" * 120)
print("GIT STAGE")
print("=" * 120)


git_cmd(
    "add",
    "--",
    "README.md",
    "docs/JOURNAL_EXTENSION_SUMMARY.md",

    "docs/STAGE24_MANUSCRIPT_INTEGRATION.md",
    "docs/STAGE24_MANUSCRIPT_INTEGRATION.tex",

    "docs/STAGE24_PUBLICATION_TABLES.md",
    "docs/STAGE24_PUBLICATION_TABLES.tex",

    "docs/STAGE24_PUBLICATION_CLOSEOUT.md",

    "figures/stage24_cross_dataset",

    "scripts/stage24",

    "results/stage24_cross_dataset/stage24_publication_package",
)


staged_paths = get_staged_paths()


print(
    "Staged files:"
)


for path in sorted(
    staged_paths
):

    print(
        " ",
        path,
    )


print()


if not staged_paths:

    raise RuntimeError(
        "No Stage24 publication files were staged."
    )


# Everything dirty before staging must now be staged.
unstaged_after_add = (
    get_unstaged_tracked_paths()
    |
    get_untracked_paths()
)


if unstaged_after_add:

    raise RuntimeError(
        "\nPublication changes remain unstaged:\n"
        +
        "\n".join(
            sorted(
                unstaged_after_add
            )
        )
    )


unexpected_staged = [
    path
    for path in sorted(
        staged_paths
    )
    if (
        path not in allowed_exact
        and
        not any(
            path.startswith(
                prefix
            )
            for prefix in allowed_prefixes
        )
    )
]


if unexpected_staged:

    raise RuntimeError(
        "\nUnexpected files were staged:\n"
        +
        "\n".join(
            unexpected_staged
        )
    )


frozen_staged = [
    path
    for path in sorted(
        staged_paths
    )
    if any(
        path.startswith(
            prefix
        )
        for prefix in frozen_prefixes
    )
]


if frozen_staged:

    raise RuntimeError(
        "\nFrozen Stage24 scientific artifacts were accidentally staged:\n"
        +
        "\n".join(
            frozen_staged
        )
    )


print(
    "[PASS] All publication changes staged."
)

print(
    "[PASS] No unrelated files staged."
)

print(
    "[PASS] No frozen scientific Stage24 files staged."
)

print()


# ==============================================================================
# 12. GIT AUTHOR SAFETY
# ==============================================================================

author_name = git_cmd(
    "config",
    "--local",
    "--get",
    "user.name",
    check=False,
)


author_email = git_cmd(
    "config",
    "--local",
    "--get",
    "user.email",
    check=False,
)


if not author_name:

    git_cmd(
        "config",
        "--local",
        "user.name",
        git_cmd(
            "log",
            "-1",
            "--format=%an",
        ),
    )


if not author_email:

    git_cmd(
        "config",
        "--local",
        "user.email",
        git_cmd(
            "log",
            "-1",
            "--format=%ae",
        ),
    )


# ==============================================================================
# 13. COMMIT
# ==============================================================================

print("=" * 120)
print("GIT COMMIT")
print("=" * 120)


commit_output = git_cmd(
    "commit",
    "-m",
    "stage24: add publication package and reproducibility notebook",
)


print(
    commit_output
)

print()


commit = git_cmd(
    "rev-parse",
    "HEAD",
)


parent = git_cmd(
    "rev-parse",
    "HEAD^",
)


if parent != EXPECTED_PARENT:

    raise RuntimeError(
        "\nPublication closeout commit parent mismatch.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {parent}"
    )


print(
    "Commit:",
    commit,
)

print()


# ==============================================================================
# 14. PUSH + REMOTE VERIFY
# ==============================================================================

print("=" * 120)
print("PUSH + REMOTE VERIFY")
print("=" * 120)


push_output = git_cmd(
    "push",
    "origin",
    "HEAD:main",
    auth_header=auth_header,
)


print(
    push_output
)

print()


remote_after = git_cmd(
    "ls-remote",
    "origin",
    "refs/heads/main",
    auth_header=auth_header,
).split()[0]


if remote_after != commit:

    raise RuntimeError(
        "\nRemote verification failed.\n"
        f"Local:  {commit}\n"
        f"Remote: {remote_after}"
    )


# Check both tracked and untracked state instead of relying on porcelain prefix parsing.
remaining_tracked = get_unstaged_tracked_paths()

remaining_untracked = get_untracked_paths()

remaining_staged = get_staged_paths()


if (
    remaining_tracked
    or
    remaining_untracked
    or
    remaining_staged
):

    raise RuntimeError(
        "\nRepository is not clean after Stage24 closeout.\n"
        f"Tracked modifications: {sorted(remaining_tracked)}\n"
        f"Untracked: {sorted(remaining_untracked)}\n"
        f"Staged: {sorted(remaining_staged)}"
    )


print(
    "[PASS] Remote main == publication closeout commit."
)

print(
    "[PASS] Repository clean."
)

print()


# ==============================================================================
# 15. CLICKABLE KAGGLE DOWNLOADS
# ==============================================================================

print("=" * 120)
print("KAGGLE DOWNLOADS")
print("=" * 120)


try:

    from IPython.display import display, FileLink


    print(
        "Full Stage24 notebook:"
    )

    display(
        FileLink(
            str(
                FULL_NOTEBOOK_DOWNLOAD
            )
        )
    )


    print(
        "Notebook + Python export ZIP:"
    )

    display(
        FileLink(
            str(
                NOTEBOOK_ZIP
            )
        )
    )


    print(
        "Publication package ZIP:"
    )

    display(
        FileLink(
            str(
                PUBLICATION_ZIP
            )
        )
    )


except Exception as exc:

    print(
        "[INFO] Could not render clickable FileLinks:",
        repr(
            exc
        ),
    )


print()


# ==============================================================================
# 16. FINAL
# ==============================================================================

print("=" * 120)
print("STAGE24 PUBLICATION + REPRODUCIBILITY CLOSEOUT: PASS")
print("=" * 120)

print()

print(
    "Scientific Stage24 status:       CLOSED"
)

print(
    "Scientific fits:                 4 / 4"
)

print(
    "Evaluable target openings:       6 / 6"
)

print(
    "GROUNDED_S4 cancellations:       2"
)

print(
    "Cancelled slots reallocated:     NO"
)

print()

print(
    "New scientific fit here:         NO"
)

print(
    "New model inference here:        NO"
)

print(
    "New target opening here:         NO"
)

print()

print(
    "Publication tables:              PUSHED"
)

print(
    "Publication figures:             PUSHED"
)

print(
    "Results section:                 PUSHED"
)

print(
    "Discussion:                      PUSHED"
)

print(
    "Limitations / Threats:           PUSHED"
)

print(
    "Contributions:                   PUSHED"
)

print()

print(
    "GitHub Stage24 notebook:"
)

print(
    " ",
    SCRIPT_IPYNB.relative_to(
        REPO
    ),
)

print()

print(
    "GitHub Stage24 Python script:"
)

print(
    " ",
    SCRIPT_PY.relative_to(
        REPO
    ),
)

print()

print(
    "Full downloadable notebook:"
)

print(
    " ",
    FULL_NOTEBOOK_DOWNLOAD
)

print()

print(
    "Notebook/script ZIP:"
)

print(
    " ",
    NOTEBOOK_ZIP
)

print()

print(
    "Publication ZIP:"
)

print(
    " ",
    PUBLICATION_ZIP
)

print()

print(
    "Publication manifest SHA:"
)

print(
    " ",
    EXPECTED_PUBLICATION_MANIFEST_SHA
)

print()

print(
    "GitHub commit:"
)

print(
    " ",
    commit
)

print()

print(
    "Remote main:"
)

print(
    " ",
    remote_after
)

print()

print(
    "STAGE24 = COMPLETELY FINISHED 😂"
)

print(
    "TWO STAGES LEFT."
)

print("=" * 120)

STAGE24-PUB-R2 — FINAL GIT CLOSEOUT RECOVERY

GOVERNANCE
HEAD: a4b6a3854109ba3d85954fb4a40afe6fe1ee6114
[PASS] HEAD remains exact Stage24 scientific freeze.
[PASS] No fit/opening ledger.
[PASS] Nothing staged.

FROZEN SCIENTIFIC ARTIFACTS
Final synthesis SHA:     785bbcb00140f4d7e07e9b49ad33924166b5995e66a229c986d5b1abcbcaee4b
Final report SHA:        f867289cec34eca6666d02deec01864395503cea8a59b7eac88e1759d5eb2a0b
Primary bootstrap SHA:   96732023e0a1c9b52a79fafa59ea4b851ef7f8bae1814e4b4b4fde0ef0df09aa
Secondary bootstrap SHA: 28005e0f13d86658ba81e6b19dc154b94eedb316ea5ce0348287265c00541953

[PASS] Frozen scientific state untouched.

PUBLICATION PACKAGE CHECKPOINT
Sanitized notebook SHA: 9f2d54b4f36f516544026902e269d6601634ffda2c9c287a5ffef1758c9c4558
Python script SHA:       104c8b3c3bde1eb81e7a70a05e15854e98d65f23d9d4799ea8ec22a064b106c6
Publication manifest SHA: 7cc628debf53e22ee0c71e21e307f7e9fa766cbeecdd527174db2dbe47e3bf82

[PASS] Publication/reproducibility package checkpoint c

/kaggle/working/stage24_exports/stage24_cross_dataset_generalization_FULL.ipynb

Notebook + Python export ZIP:


/kaggle/working/stage24_notebook_export.zip

Publication package ZIP:


/kaggle/working/stage24_publication_package.zip


STAGE24 PUBLICATION + REPRODUCIBILITY CLOSEOUT: PASS

Scientific Stage24 status:       CLOSED
Scientific fits:                 4 / 4
Evaluable target openings:       6 / 6
GROUNDED_S4 cancellations:       2
Cancelled slots reallocated:     NO

New scientific fit here:         NO
New model inference here:        NO
New target opening here:         NO

Publication tables:              PUSHED
Publication figures:             PUSHED
Results section:                 PUSHED
Discussion:                      PUSHED
Limitations / Threats:           PUSHED
Contributions:                   PUSHED

GitHub Stage24 notebook:
  scripts/stage24/stage24_cross_dataset_generalization.ipynb

GitHub Stage24 Python script:
  scripts/stage24/stage24_cross_dataset_generalization.py

Full downloadable notebook:
  /kaggle/working/stage24_exports/stage24_cross_dataset_generalization_FULL.ipynb

Notebook/script ZIP:
  /kaggle/working/stage24_notebook_export.zip

Publication ZIP:
  /kaggle/working/stage24_publica